# DS‑1 · офисы и часы работы

Kaggle → CPU → Internet → Run All. Предыдущие ноутбуки запускать не нужно.
База: лучший пользовательский score **0,88092** (holiday_service + профиль 11 + учебный профиль 28).
Ноутбук воспроизводит её по SHA, затем проверяет офисные признаки.

Три одинаковых по настройкам модели: контроль с образовательным календарём;
контроль + офисная география; география + известные часы работы.
Минимум 0,5% снижения ошибки в каждом из сентября и октября против базы и
соответствующего контроля. Дополнительная проверка после переобучения на сентябре.
Маршруты 5/7/50 защищены. В отсутствие выигрыша CSV полностью совпадает с 0,88092.

Исторический OSM на 01.01.2025: office=* и building=office.
Здания и организации — отдельные признаки, не оценка работников.
Расстояние 500 м от центра объекта до остановки, не пешеходный маршрут.
Часы работы взяты из opening_hours, неизвестные и сложные записи не угадываются.
Это публичные часы, а не подтверждённые смены сотрудников.
Праздники/переносы используют условный соответствующий день недели.
Окна 06–10 и 16–20 — проверяемая гипотеза о поездках, а не фактический график.

© OpenStreetMap contributors, ODbL: https://www.openstreetmap.org/copyright
Геометрия GTFS 2026 года; режим retrospective competition. Погода и POI встроены.
Ноябрь–декабрь не содержат фактических пассажирских меток.
Сентябрь–октябрь использовались ранее: результат исследовательский, не независимый тест.


## 1. Настройки
Ссылка уже указана. Режим `yandex` скачивает архив автоматически; `input` оставлен для ранее загруженных файлов.


In [ ]:
import os
import sys
import json
import csv
import hashlib
import subprocess
import tempfile
import zipfile
import importlib.metadata
from pathlib import Path

PROFILES_ONLY = True  # Базовые профили фиксированы; погодные поправки обучаются LightGBM.
RUN_TESTS = True
DATA_SOURCE = 'yandex'  # 'input' — необязательный ручной вариант через Add Input.
YANDEX_URL = 'https://disk.yandex.ru/d/DiFwlfMOauxjBg'
DOWNLOAD_CACHE = '/tmp/mthack_yandex'
INPUT_ROOT = Path(os.environ.get('MTHACK_INPUT_DIR', '/kaggle/input'))
WORK = Path(os.environ.get('MTHACK_WORK_DIR', '/kaggle/working'))
INPUT_FILES = {
    'train': '',   # например /kaggle/input/my-private-data/labels_day_train.csv
    'test': '',    # labels_day_test.csv — фактические метки сентября–октября
    'sample': '',  # исходный test_submission.csv за ноябрь–декабрь
}
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
if sys.version_info < (3, 11):
    raise RuntimeError('Нужен Python 3.11+; выберите актуальную Python-среду Kaggle.')
WORK.mkdir(parents=True, exist_ok=True)
RUNTIME = Path(tempfile.mkdtemp(prefix='mthack_ds1_code_'))
EXTRACTED = Path(tempfile.mkdtemp(prefix='mthack_ds1_inputs_'))
OUT = WORK / 'hourly_experts_results'
print('Python:', sys.version.split()[0])
if not PROFILES_ONLY:
    for package in ['numpy', 'scipy', 'scikit-learn', 'lightgbm', 'catboost']:
        try:
            print(package, importlib.metadata.version(package))
        except importlib.metadata.PackageNotFoundError:
            print(package, 'не установлен (для обучения бустинга установите пакет или PROFILES_ONLY=True)')
print('Результаты:', OUT)
# Install into the model environment using the notebook's pip.
# A target environment does not need its own pip or Kaggle startup hooks.
MODEL_READY = False
MODEL_ENV = Path('/tmp/mthack_hourly_pinned_env')
TRAIN_PYTHON = MODEL_ENV/'bin/python'
MODEL_PROCESS_ENV = {k: v for k, v in os.environ.items()
                     if k not in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP')}
MODEL_PROCESS_ENV['PYTHONNOUSERSITE'] = '1'
subprocess.check_call([sys.executable, '-I', '-m', 'venv', '--without-pip', str(MODEL_ENV)],
                      env=MODEL_PROCESS_ENV)
PINS = ['numpy==2.0.2', 'lightgbm==4.6.0', 'scikit-learn==1.6.1', 'scipy==1.14.1']
subprocess.check_call([sys.executable, '-I', '-m', 'pip', '--python', str(TRAIN_PYTHON),
                       'install', '--quiet', *PINS], env=MODEL_PROCESS_ENV)
probe = "import numpy, lightgbm, sklearn, scipy; import importlib.metadata as m; "
probe += "expected = " + repr(dict(x.split('==') for x in PINS)) + "; "
probe += "actual = {name: m.version(name) for name in expected}; "
probe += "assert actual == expected, (actual, expected); print(actual)"
subprocess.check_call([str(TRAIN_PYTHON), '-I', '-c', probe], env=MODEL_PROCESS_ENV)
MODEL_READY = True
print('Среда обучения готова; версии и импорт библиотек проверены.')


## 2. Встроенный код модели
Ячейка разворачивает точную копию модулей DS‑1 во временной папке.
Ничего не скачивается. Исходные данные не входят в ноутбук.


In [ ]:
SOURCES = {'ml/forecast/core.py': '"""Strict readers and full-grid scoring of organizer hourly labels (MSK)."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport math\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nKey = tuple[str, date, int]\nLabels = dict[Key, int]\n\nRUSSIAN_HOLIDAYS_2025 = frozenset({\n    date(2025, 1, 1), date(2025, 1, 2), date(2025, 1, 3), date(2025, 1, 4),\n    date(2025, 1, 5), date(2025, 1, 6), date(2025, 1, 7), date(2025, 1, 8),\n    date(2025, 2, 23), date(2025, 3, 8),\n    date(2025, 5, 1), date(2025, 5, 2), date(2025, 5, 8), date(2025, 5, 9),\n    date(2025, 6, 12), date(2025, 6, 13),\n    date(2025, 11, 3), date(2025, 11, 4),\n    date(2025, 12, 31),\n})\n\nRUSSIAN_WORKING_WEEKENDS_2025 = frozenset({\n    date(2025, 11, 1),\n})\n\n\ndef is_workday(d: date) -> bool:\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return True\n    if d in RUSSIAN_HOLIDAYS_2025:\n        return False\n    return d.weekday() < 5\n\n\ndef effective_weekday(d: date) -> int:\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return 4\n    if d in RUSSIAN_HOLIDAYS_2025:\n        return 6\n    return d.weekday()\n\n\ndef calendar_day_type(d: date) -> int:\n    if not is_workday(d):\n        return 3 if (effective_weekday(d) == 6 or d in RUSSIAN_HOLIDAYS_2025) else 2\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return 4\n    if d.weekday() == 4:\n        return 1\n    return 0\n\n\ndef days(start: date, end: date):\n    while start <= end:\n        yield start\n        start += timedelta(days=1)\n\n\ndef grid(routes, start, end):\n    return [(str(r), d, h) for r in routes for d in days(start, end) for h in range(24)]\n\n\ndef parse_key(row) -> Key:\n    route = row[\'route\']\n    if not route or not route.isascii() or not route.isdecimal() or int(route) <= 0:\n        raise ValueError(f\'Invalid route: {route!r}\')\n    day = date.fromisoformat(row[\'date\'])\n    if day.isoformat() != row[\'date\']:\n        raise ValueError(\'Date must be YYYY-MM-DD\')\n    hour = int(row[\'hour\'])\n    if str(hour) != row[\'hour\'] or not 0 <= hour <= 23:\n        raise ValueError(f\'Invalid hour: {row["hour"]!r}\')\n    return str(int(route)), day, hour\n\n\ndef load_labels(paths) -> Labels:\n    result = {}\n    for path in paths:\n        with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n            reader = csv.DictReader(f, delimiter=\';\')\n            if reader.fieldnames != [\'route\', \'date\', \'hour\', \'boardings\']:\n                raise ValueError(f\'Unexpected label schema: {reader.fieldnames}\')\n            for row in reader:\n                if None in row or any(v is None for v in row.values()):\n                    raise ValueError(\'Malformed CSV row\')\n                key = parse_key(row)\n                value = int(row[\'boardings\'])\n                if value < 0 or str(value) != row[\'boardings\']:\n                    raise ValueError(\'Boardings must be nonnegative integers\')\n                if key in result:\n                    raise ValueError(f\'Duplicate label key across input files: {key}\')\n                result[key] = value\n    if not result:\n        raise ValueError(\'Labels are empty\')\n    return result\n\n\ndef rounded(value) -> int:\n    value = float(value)\n    if not math.isfinite(value) or abs(value) > 10**12:\n        raise ValueError(\'Prediction must be finite and within 1e12\')\n    return max(0, round(value))  # Same half-to-even rule as DS-2 baseline.\n\n\ndef sha256(path):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef score(keys, truth, predictions, peak_hours=None):\n    """Score exactly the requested grid; absent label = 0 is an explicit assumption."""\n    if len(keys) != len(set(keys)) or set(keys) != set(predictions):\n        raise ValueError(\'Prediction keys must match the unique evaluation grid exactly\')\n    if set(truth) - set(keys):\n        raise ValueError(\'Truth contains keys outside evaluation grid\')\n    groups = defaultdict(lambda: {\'rows\': 0, \'actual_sum\': 0, \'absolute_error_sum\': 0,\n                                  \'pred_minus_actual\': 0, \'missing_label_rows\': 0})\n    for key in keys:\n        route, day, hour = key\n        actual, pred = truth.get(key, 0), rounded(predictions[key])\n        if actual < 0 or not math.isfinite(actual):\n            raise ValueError(\'Invalid truth\')\n        names = [\'all\', f\'route:{route}\', f\'month:{day:%Y-%m}\',\n                 \'hours:day_06_22\' if 6 <= hour < 23 else \'hours:night_23_05\',\n                 \'labels:observed\' if key in truth else \'labels:missing_assumed_zero\']\n        if peak_hours is not None:\n            names.append(\'hours:train_peaks\' if hour in peak_hours.get(route, ())\n                         else \'hours:other\')\n        for name in names:\n            m = groups[name]\n            m[\'rows\'] += 1\n            m[\'actual_sum\'] += actual\n            m[\'absolute_error_sum\'] += abs(pred - actual)\n            m[\'pred_minus_actual\'] += pred - actual\n            m[\'missing_label_rows\'] += key not in truth\n    if not groups[\'all\'][\'actual_sum\']:\n        raise ValueError(\'Global WAPE undefined: sum(y) must be positive\')\n    for m in groups.values():\n        m[\'wape\'] = m[\'absolute_error_sum\'] / m[\'actual_sum\'] if m[\'actual_sum\'] else None\n        m[\'wape_score\'] = max(0, 1 - m[\'wape\']) if m[\'wape\'] is not None else None\n    return dict(sorted(groups.items()))\n\n\ndef peak_hours(history, routes):\n    totals = defaultdict(int)\n    for (r, _, h), y in history.items():\n        totals[r, h] += y\n    return {r: sorted(range(24), key=lambda h: (-totals[r, h], h))[:4] for r in routes}\n', 'ml/forecast/models.py': '"""Models receive ONLY history through cutoff. Predict never accepts future truth."""\nfrom __future__ import annotations\n\nimport math\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom statistics import median, mean\n\nfrom .core import grid, rounded, is_workday, effective_weekday, calendar_day_type\n\n\nclass Profile:\n    def __init__(self, statistic=\'mean\', missing=\'zero\', shrink=0.2, window=None,\n                 use_calendar=False, route50_rule=False):\n        if statistic not in (\'mean\', \'median\') or missing not in (\'zero\', \'observed\'):\n            raise ValueError(\'Invalid profile configuration\')\n        self.statistic, self.missing, self.shrink, self.window = statistic, missing, shrink, window\n        self.use_calendar = use_calendar\n        self.route50_rule = route50_rule\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'Training history crosses cutoff or is empty\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history})\n        if self.window:\n            start = max(start, cutoff - timedelta(days=self.window - 1))\n        filtered = {k: v for k, v in history.items() if k[1] >= start}\n        local, hour, route = defaultdict(list), defaultdict(list), defaultdict(list)\n        keys = grid(self.routes, start, cutoff) if self.missing == \'zero\' else filtered\n        for r, d, h in keys:\n            y = filtered.get((r, d, h), 0)\n            w = effective_weekday(d) if self.use_calendar else d.weekday()\n            local[r, w, h].append(y)\n            hour[r, h].append(y)\n            route[r].append(y)\n        fn = mean if self.statistic == \'mean\' else median\n        self.local = {k: fn(v) for k, v in local.items()}\n        self.hour = {k: fn(v) for k, v in hour.items()}\n        self.route = {k: fn(v) for k, v in route.items()}\n        return self\n\n    def predict(self, keys):\n        if any(k[1] <= self.cutoff for k in keys):\n            raise ValueError(\'Prediction dates must be after cutoff\')\n        result = {}\n        for key in keys:\n            r, d, h = key\n            if self.route50_rule and r == \'50\' and not is_workday(d) and date(2025, 9, 6) <= d < date(2025, 11, 15):\n                result[key] = 0\n                continue\n            w = effective_weekday(d) if self.use_calendar else d.weekday()\n            parent = self.hour.get((r, h), self.route.get(r, 0))\n            local = self.local.get((r, w, h), parent)\n            result[key] = rounded((1 - self.shrink) * local + self.shrink * parent)\n        return result\n\n\nclass CalendarProfile:\n    """Optimized calendar profile: blends recent 56-day trend with full history on effective weekdays."""\n    def __init__(self, recent_weight=0.6, window=56, route50_rule=True):\n        self.recent_weight = recent_weight\n        self.window = window\n        self.route50_rule = route50_rule\n\n    def fit(self, history, start, cutoff):\n        self.cutoff = cutoff\n        self.full_profile = Profile(\'median\', shrink=0.0, use_calendar=True,\n                                    route50_rule=self.route50_rule).fit(history, start, cutoff)\n        self.recent_profile = Profile(\'median\', shrink=0.0, window=self.window,\n                                      use_calendar=True, route50_rule=self.route50_rule).fit(history, start, cutoff)\n        return self\n\n    def predict(self, keys):\n        p_full = self.full_profile.predict(keys)\n        p_recent = self.recent_profile.predict(keys)\n        w = self.recent_weight\n        return {k: rounded(w * p_recent[k] + (1 - w) * p_full[k]) for k in keys}\n\n\nclass Boosting:\n    def __init__(self, kind=\'hgb\', recursive=False, use_calendar=True, route50_rule=False):\n        self.kind = kind\n        self.recursive = recursive\n        self.use_calendar = use_calendar\n        self.route50_rule = route50_rule\n\n    def _extract_profiles(self, history, start, cutoff):\n        full_med, full_mean = defaultdict(list), defaultdict(list)\n        rec_med, rec_mean = defaultdict(list), defaultdict(list)\n        hour_med = defaultdict(list)\n        rec_start = max(start, cutoff - timedelta(days=55))\n\n        keys = grid(self.routes, start, cutoff)\n        for r, d, h in keys:\n            y = history.get((r, d, h), 0)\n            eff_w = effective_weekday(d) if self.use_calendar else d.weekday()\n            full_med[r, eff_w, h].append(y)\n            full_mean[r, eff_w, h].append(y)\n            hour_med[r, h].append(y)\n            if d >= rec_start:\n                rec_med[r, eff_w, h].append(y)\n                rec_mean[r, eff_w, h].append(y)\n\n        self.p_full_med = {k: median(v) for k, v in full_med.items()}\n        self.p_full_mean = {k: mean(v) for k, v in full_mean.items()}\n        self.p_rec_med = {k: median(v) for k, v in rec_med.items()}\n        self.p_rec_mean = {k: mean(v) for k, v in rec_mean.items()}\n        self.p_hour_med = {k: median(v) for k, v in hour_med.items()}\n\n    def features(self, keys, context=None):\n        import numpy as np\n        result = []\n        if self.recursive:\n            for r, d, h in keys:\n                t = d.timetuple().tm_yday\n                row = [self.route_codes[r], d.weekday(), h, d.month, t,\n                       int(d.weekday() >= 5), math.sin(2 * math.pi * t / 365),\n                       math.cos(2 * math.pi * t / 365)]\n                if context is not None:\n                    for lag in (1, 7):\n                        key = r, d - timedelta(days=lag), h\n                        row.extend([context.get(key, 0), int(key in context)])\n                result.append(row)\n            return np.asarray(result, dtype=float)\n\n        for r, d, h in keys:\n            t = d.timetuple().tm_yday\n            eff_w = effective_weekday(d) if self.use_calendar else d.weekday()\n            workday = int(is_workday(d)) if self.use_calendar else int(d.weekday() < 5)\n            dtype = calendar_day_type(d) if self.use_calendar else (2 if d.weekday() >= 5 else 0)\n\n            pf_med = getattr(self, \'p_full_med\', {}).get((r, eff_w, h), getattr(self, \'p_hour_med\', {}).get((r, h), 0))\n            pf_mean = getattr(self, \'p_full_mean\', {}).get((r, eff_w, h), pf_med)\n            pr_med = getattr(self, \'p_rec_med\', {}).get((r, eff_w, h), pf_med)\n            pr_mean = getattr(self, \'p_rec_mean\', {}).get((r, eff_w, h), pf_mean)\n            ph_med = getattr(self, \'p_hour_med\', {}).get((r, h), 0)\n            trend_ratio = pr_mean / (pf_mean + 1.0) if pf_mean > 0 else 1.0\n\n            is_r50_susp = int(self.route50_rule and r == \'50\' and not is_workday(d)\n                              and date(2025, 9, 6) <= d < date(2025, 11, 15))\n\n            row = [\n                h,\n                d.weekday(),\n                eff_w,\n                dtype,\n                workday,\n                d.month,\n                d.day,\n                t,\n                math.sin(2 * math.pi * h / 24),\n                math.cos(2 * math.pi * h / 24),\n                math.sin(2 * math.pi * t / 365),\n                math.cos(2 * math.pi * t / 365),\n                pf_med,\n                pf_mean,\n                pr_med,\n                pr_mean,\n                ph_med,\n                trend_ratio,\n                is_r50_susp,\n            ]\n            result.append(row)\n        return np.asarray(result, dtype=float)\n\n    def fit(self, history, start, cutoff):\n        import numpy as np\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'Training history crosses cutoff or is empty\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history})\n        self.route_codes = {r: i for i, r in enumerate(self.routes)}\n        self._extract_profiles(history, start, cutoff)\n        self.history = dict(history)\n\n        keys = grid(self.routes, start, cutoff)\n        if self.recursive:\n            from sklearn.ensemble import HistGradientBoostingRegressor\n            keys = [k for k in keys if k[1] >= start + timedelta(days=7)]\n            self.model = HistGradientBoostingRegressor(\n                loss=\'absolute_error\', learning_rate=.08, max_iter=150,\n                max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=2,\n                categorical_features=[0, 1, 2, 3, 5], early_stopping=False, random_state=2025)\n            self.model.fit(self.features(keys, history), [history.get(k, 0) for k in keys])\n            return self\n\n        # Out-of-time target aggregates: each monthly block uses ONLY prior days.\n        # Reuse the exact feature builder used at inference, with an earlier cutoff.\n        train_keys, train_features = [], []\n        block_start = start + timedelta(days=28)\n        while block_start <= cutoff:\n            next_month = (block_start.replace(day=28) + timedelta(days=4)).replace(day=1)\n            block_end = min(cutoff, next_month - timedelta(days=1))\n            prefix_end = block_start - timedelta(days=1)\n            prefix = {k:v for k,v in history.items() if k[1] <= prefix_end}\n            self._extract_profiles(prefix, start, prefix_end)\n            batch = grid(self.routes, block_start, block_end)\n            train_keys.extend(batch)\n            train_features.extend(self.features(batch))\n            block_start = block_end + timedelta(days=1)\n        if not train_keys:\n            raise ValueError(\'Direct boosting needs at least 29 days of history\')\n        # Final prediction profiles may now use all history up to the actual cutoff.\n        self._extract_profiles(history, start, cutoff)\n        route_keys, route_features = defaultdict(list), defaultdict(list)\n        for k, features in zip(train_keys, train_features):\n            route_keys[k[0]].append(k)\n            route_features[k[0]].append(features)\n\n        self.models = {}\n        for r in self.routes:\n            r_keys = route_keys[r]\n            X = np.asarray(route_features[r], dtype=float)\n            y = np.array([history.get(k, 0) for k in r_keys], dtype=float)\n\n            if self.kind == \'lgb\':\n                try:\n                    import lightgbm as lgb\n                    model = lgb.LGBMRegressor(\n                        objective=\'regression_l1\', metric=\'mae\', learning_rate=0.03,\n                        n_estimators=300, num_leaves=15, min_child_samples=10,\n                        subsample=0.85, colsample_bytree=0.85, random_state=2025,\n                        verbose=-1, n_jobs=2\n                    )\n                    model.fit(X, y)\n                    self.models[r] = model\n                    continue\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n            if self.kind == \'catboost\':\n                try:\n                    import catboost as cb\n                    model = cb.CatBoostRegressor(\n                        loss_function=\'MAE\', eval_metric=\'MAE\', iterations=300,\n                        learning_rate=0.04, depth=5, random_seed=2025, verbose=0,\n                        thread_count=2\n                    )\n                    model.fit(X, y)\n                    self.models[r] = model\n                    continue\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n            if self.kind == \'blend\':\n                submodels = []\n                try:\n                    import lightgbm as lgb\n                    m1 = lgb.LGBMRegressor(\n                        objective=\'regression_l1\', metric=\'mae\', learning_rate=0.03,\n                        n_estimators=300, num_leaves=15, min_child_samples=10,\n                        subsample=0.85, colsample_bytree=0.85, random_state=2025,\n                        verbose=-1, n_jobs=2\n                    )\n                    m1.fit(X, y)\n                    submodels.append((\'lgb\', m1, 0.5))\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n                try:\n                    import catboost as cb\n                    m2 = cb.CatBoostRegressor(\n                        loss_function=\'MAE\', eval_metric=\'MAE\', iterations=300,\n                        learning_rate=0.04, depth=5, random_seed=2025, verbose=0,\n                        thread_count=2\n                    )\n                    m2.fit(X, y)\n                    submodels.append((\'catboost\', m2, 0.5))\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n                if not submodels:\n                    from sklearn.ensemble import HistGradientBoostingRegressor\n                    m3 = HistGradientBoostingRegressor(\n                        loss=\'absolute_error\', learning_rate=0.04, max_iter=200,\n                        max_leaf_nodes=15, min_samples_leaf=10, l2_regularization=1,\n                        early_stopping=False, random_state=2025\n                    )\n                    m3.fit(X, y)\n                    submodels.append((\'hgb\', m3, 1.0))\n                self.models[r] = submodels\n                continue\n\n            from sklearn.ensemble import HistGradientBoostingRegressor\n            model = HistGradientBoostingRegressor(\n                loss=\'absolute_error\', learning_rate=0.04, max_iter=200,\n                max_leaf_nodes=15, min_samples_leaf=10, l2_regularization=1,\n                early_stopping=False, random_state=2025\n            )\n            model.fit(X, y)\n            self.models[r] = model\n\n        return self\n\n    def _predict_route(self, r, X):\n        m = getattr(self, \'models\', {}).get(r)\n        if m is None:\n            if hasattr(self, \'model\'):\n                return self.model.predict(X)\n            return [0] * len(X)\n        if isinstance(m, list):\n            total_w = sum(w for _, _, w in m)\n            preds = sum(w * sub.predict(X) for _, sub, w in m) / total_w\n            return preds\n        return m.predict(X)\n\n    def predict(self, keys):\n        if any(k[1] <= self.cutoff for k in keys):\n            raise ValueError(\'Prediction dates must be after cutoff\')\n        known = [k for k in keys if k[0] in self.route_codes]\n        result = {k: 0 for k in keys}\n        if not known:\n            return result\n\n        if not self.recursive:\n            route_known = defaultdict(list)\n            for k in known:\n                route_known[k[0]].append(k)\n            for r, r_keys in route_known.items():\n                X = self.features(r_keys)\n                raw_preds = self._predict_route(r, X)\n                for k, p in zip(r_keys, raw_preds):\n                    if self.route50_rule and r == \'50\' and not is_workday(k[1]) and date(2025, 9, 6) <= k[1] < date(2025, 11, 15):\n                        result[k] = 0\n                    else:\n                        result[k] = rounded(p)\n            return result\n\n        context = dict(self.history)\n        day, end = self.cutoff + timedelta(days=1), max(k[1] for k in known)\n        wanted = set(known)\n        while day <= end:\n            batch = grid(self.routes, day, day)\n            predictions = self.model.predict(self.features(batch, context))\n            for key, value in zip(batch, predictions):\n                context[key] = rounded(value)\n                if key in wanted:\n                    result[key] = context[key]\n            day += timedelta(days=1)\n        return result\n\n\ndef candidates(include_boosting=True):\n    result = {\n        \'mean_zero\': lambda: Profile(\'mean\'),\n        \'median_zero\': lambda: Profile(\'median\'),\n        \'mean_observed\': lambda: Profile(\'mean\', \'observed\'),\n        \'median_observed\': lambda: Profile(\'median\', \'observed\'),\n        \'mean_56d\': lambda: Profile(\'mean\', window=56),\n        \'profile_calendar\': lambda: CalendarProfile(recent_weight=0.6, window=56, route50_rule=True),\n    }\n    if include_boosting:\n        result.update(\n            hgb_direct=lambda: Boosting(kind=\'hgb\'),\n            lgb_direct=lambda: Boosting(kind=\'lgb\'),\n            catboost_direct=lambda: Boosting(kind=\'catboost\'),\n            blend_ensemble=lambda: Boosting(kind=\'blend\'),\n            hgb_recursive=lambda: Boosting(recursive=True),\n        )\n    return result\n\n', 'ml/forecast/route_specific.py': '"""DS-2-informed route-specific selection for a fixed 61-day forecast.\n\nNo validation labels enter fitted candidates. Selection scores are exploratory:\nSep-Oct has already been examined repeatedly by DS-2.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport pickle\nimport time\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom pathlib import Path\nimport numpy as np\nfrom .core import days, grid, load_labels, rounded, score, sha256, effective_weekday\nfrom .models import Profile, Boosting\nfrom .run import FIRST, FINAL_CUTOFF, FOLDS, dump\nfrom ml.submit.adapter import read_sample, write_submission, export_api, validate\n\nRESEARCH_COMMIT = \'b593ebb\'\nEXPECTED_HASHES = {\n    \'train\': \'20a61009b4ee00327784c6fa888109609fd3a1905ea3a8509b36964dd9baced7\',\n    \'test\': \'37479acea3e319ac613d72df7217bbe57b66af49644e538139174fd534ce2347\',\n    \'sample\': \'71c8126bb51068c936462a170da885bad860fc91a58b3d2ac731876d324d7ec3\',\n}\n\n\ndef verify_inputs(paths, synthetic=False):\n    result = {role: {\'name\': p.name, \'sha256\': sha256(p)} for role, p in paths.items()}\n    if not synthetic:\n        for role, item in result.items():\n            if item[\'sha256\'] != EXPECTED_HASHES[role]:\n                raise ValueError(f\'{role}: SHA256 does not match real DS-2 inputs. \'\n                                 \'Do not train a submission on synthetic or unverified files. \'\n                                 f"Received {item[\'sha256\']}; expected {EXPECTED_HASHES[role]}")\n    return result\n\n\nclass SeasonalProfile:\n    """Calendar regimes, robust hourly shapes, optional local history blend.\n\nSummer weekdays are a separate regime, not deleted unconditionally: if a\nforecast requests summer, use summer history when at least 3 dates exist.\nFor autumn this reproduces the DS-2 school-term exclusion candidate.\n"""\n    def __init__(self, statistic=\'median\', shrink=.2, window=None,\n                 seasonal=True, calendar=True, recent_weight=0., regime7=False):\n        self.statistic, self.shrink, self.window = statistic, shrink, window\n        self.seasonal, self.calendar = seasonal, calendar\n        self.recent_weight, self.regime7 = recent_weight, regime7\n\n    @staticmethod\n    def summer(d):\n        return d.month in (7, 8) and d.weekday() < 5\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'History empty or crosses cutoff\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history}, key=int)\n        self.tables = {}\n        # All keys are explicit, including absent training labels = zero assumption.\n        for r in self.routes:\n            for season in (None, False, True):\n                ds = [d for d in days(start, cutoff)\n                      if (not self.window or (cutoff-d).days < self.window)\n                      and (season is None or self.summer(d) == season)]\n                local, parent = defaultdict(list), defaultdict(list)\n                recent = defaultdict(list)\n                regime = defaultdict(list)\n                for d in ds:\n                    w = effective_weekday(d) if self.calendar else d.weekday()\n                    for h in range(24):\n                        y = history.get((r, d, h), 0)\n                        local[w,h].append(y); parent[h].append(y)\n                        if (cutoff-d).days < 56:\n                            recent[w,h].append(y)\n                        if r == \'7\' and cutoff >= date(2025,8,16) and d >= date(2025,8,16):\n                            regime[w,h].append(y)\n                fn = np.mean if self.statistic == \'mean\' else np.median\n                self.tables[r,season] = tuple({k: float(fn(v)) for k,v in t.items() if len(v) >= 3}\n                                             for t in (local,parent,recent,regime))\n        return self\n\n    def predict(self, keys):\n        if any(d <= self.cutoff for _,d,_ in keys):\n            raise ValueError(\'Prediction dates must follow cutoff\')\n        out = {}\n        for key in keys:\n            r,d,h = key\n            if r not in self.routes:\n                out[key] = 0; continue\n            season = self.summer(d) if self.seasonal else None\n            loc,par,rec,reg = self.tables[r,season]\n            fallback,fp,_,_ = self.tables[r,None]\n            w = effective_weekday(d) if self.calendar else d.weekday()\n            parent = par.get(h,fp.get(h,0))\n            local = loc.get((w,h),fallback.get((w,h),parent))\n            p = (1-self.shrink)*local + self.shrink*parent\n            if (w,h) in rec:\n                p = (1-self.recent_weight)*p + self.recent_weight*rec[w,h]\n            if self.regime7 and r == \'7\' and d.weekday() >= 5 and (w,h) in reg:\n                # DS-2 experiment 13: 25% sparse post-Aug16 profile, no future notice.\n                p = .75*p + .25*reg[w,h]\n            out[key] = rounded(p)\n        return out\n\n\ndef candidate_factories(profiles_only=False):\n    pool = {\n        \'median_zero\': lambda: Profile(\'median\'),\n        \'mean_zero\': lambda: Profile(\'mean\'),\n        \'school_ds2\': lambda: SeasonalProfile(calendar=False),\n        \'school_calendar\': lambda: SeasonalProfile(),\n        \'school_local\': lambda: SeasonalProfile(shrink=0),\n        \'school_mean\': lambda: SeasonalProfile(statistic=\'mean\'),\n        \'school_recent25\': lambda: SeasonalProfile(recent_weight=.25),\n        \'school_recent50\': lambda: SeasonalProfile(recent_weight=.5),\n        \'school_route7\': lambda: SeasonalProfile(regime7=True),\n        \'median_56d\': lambda: SeasonalProfile(window=56, seasonal=False),\n        \'median_112d\': lambda: SeasonalProfile(window=112, seasonal=False),\n        \'mean_56d\': lambda: SeasonalProfile(statistic=\'mean\',window=56,seasonal=False),\n        \'mean_112d\': lambda: SeasonalProfile(statistic=\'mean\',window=112,seasonal=False),\n    }\n    if not profiles_only:\n        pool.update({\n            \'hgb_causal\': lambda: Boosting(kind=\'hgb\',route50_rule=False),\n            \'lgb_causal\': lambda: Boosting(kind=\'lgb\',route50_rule=False),\n            \'catboost_causal\': lambda: Boosting(kind=\'catboost\',route50_rule=False),\n        })\n    return pool\n\n\ndef choose_routes(keys, truth, predictions, routes, min_gain=.01):\n    """Conservative route choice: beat school baseline in both calendar months.\n\nA predeclared 50/50 blend is also considered. No continuous weight fitting.\nFall back to school baseline unless each month improves by >= 1% in MAE.\n"""\n    months = sorted({d.month for _,d,_ in keys})\n    choices, diagnostics = {}, {}\n    for r in routes:\n        rk = [k for k in keys if k[0] == r]\n        by_month = [[k for k in rk if k[1].month == m] for m in months]\n        def losses(name, weight):\n            return [sum(abs(rounded(weight*predictions[name][k] +\n                                (1-weight)*predictions[\'school_calendar\'][k])-truth.get(k,0))\n                        for k in batch) for batch in by_month]\n        base = losses(\'school_calendar\',1.)\n        options = [(\'school_calendar\',1.,base)]\n        diagnostics[r] = {\'baseline_month_errors\':base,\'candidates\':{}}\n        for name in predictions:\n            for weight in (0.5,1.):\n                errs = losses(name,weight)\n                diagnostics[r][\'candidates\'][f\'{name}:{weight}\'] = errs\n                if all(e <= b*(1-min_gain) for e,b in zip(errs,base)) and sum(errs)<sum(base):\n                    options.append((name,weight,errs))\n        name,weight,errs = min(options,key=lambda item:(sum(item[2]),item[0],item[1]))\n        choices[r] = {\'candidate\':name,\'weight\':weight,\'month_errors\':errs}\n    return choices,diagnostics\n\n\ndef combine(keys, predictions, choices):\n    result = {}\n    for key in keys:\n        choice = choices.get(key[0])\n        if choice is None:\n            result[key] = 0\n        else:\n            name,w = choice[\'candidate\'],choice[\'weight\']\n            result[key] = rounded(w*predictions[name][key]+(1-w)*predictions[\'school_calendar\'][key])\n    return result\n\n\nclass RouteEnsemble:\n    def __init__(self, models, choices, cutoff):\n        self.models,self.choices,self.cutoff = models,choices,cutoff\n    def predict(self,keys):\n        return combine(keys,{name:m.predict(keys) for name,m in self.models.items()},self.choices)\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'train\',\'test\',\'sample\',\'out\'):p.add_argument(\'--\'+name,type=Path,required=True)\n    p.add_argument(\'--profiles-only\',action=\'store_true\')\n    p.add_argument(\'--synthetic\',action=\'store_true\')\n    args=p.parse_args(argv)\n    inputs=verify_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')},args.synthetic)\n    train,test=load_labels([args.train]),load_labels([args.test])\n    if any(not FIRST<=k[1]<=date(2025,8,31) for k in train):raise ValueError(\'Invalid train period\')\n    if any(not date(2025,9,1)<=k[1]<=FINAL_CUTOFF for k in test):raise ValueError(\'Invalid test period\')\n    values=load_labels([args.train,args.test]); _,sample=read_sample(args.sample)\n    routes=sorted({k[0] for k in values},key=int)\n    if set(routes)-{k[0] for k in sample}:raise ValueError(\'Sample misses known routes\')\n    for lo,hi,part in [(FIRST,date(2025,8,31),train),(date(2025,9,1),FINAL_CUTOFF,test)]:\n        if {k[1] for k in part} != set(days(lo,hi)):raise ValueError(\'Missing complete input dates\')\n    if not args.profiles_only:\n        missing=[n for n in (\'sklearn\',\'lightgbm\',\'catboost\') if importlib.util.find_spec(n) is None]\n        if missing:raise RuntimeError(\'Missing training packages: \'+\', \'.join(missing)+\'. Install them or use --profiles-only explicitly.\')\n    factories=candidate_factories(args.profiles_only)\n    args.out.mkdir(parents=True,exist_ok=True)\n    report={\'schema_version\':2,\'research_commit\':RESEARCH_COMMIT,\'inputs\':inputs,\n            \'data_kind\':\'synthetic_not_for_submission\' if args.synthetic else \'verified_organizer_labels\',\n            \'selection_warning\':\'Research validation repeatedly examined in DS-2; not an independent test or platform score.\',\n            \'missing_label_policy\':\'zero-filled full grid for training and scoring; not proof of zero physical demand\',\n            \'external_policy\':\'calendar only; no future network notices, weather or validation labels in features\',\n            \'folds\':{},\'platform_score\':None,\'library_versions\':{}}\n    for pkg in (\'numpy\',\'scikit-learn\',\'lightgbm\',\'catboost\'):\n        try:report[\'library_versions\'][pkg]=importlib.metadata.version(pkg)\n        except importlib.metadata.PackageNotFoundError:pass\n    for label,cutoff,start,end in FOLDS:\n        history={k:v for k,v in values.items() if k[1]<=cutoff}\n        keys=grid(routes,start,end); truth={k:v for k,v in values.items() if start<=k[1]<=end}\n        forecasts={};report[\'folds\'][label]={}\n        for name,factory in factories.items():\n            t=time.perf_counter();model=factory().fit(history,FIRST,cutoff)\n            forecasts[name]=model.predict(keys)\n            m=score(keys,truth,forecasts[name])\n            report[\'folds\'][label][name]={\'metrics\':m,\'seconds\':round(time.perf_counter()-t,3)}\n            print(f\'{label} {name}: score={m["all"]["wape_score"]:.6f}\',flush=True)\n        if label==\'sep_oct\':\n            # Forward diagnostic: choose using September only, evaluate October\n            # from the SAME Aug31 forecasts (no October/September refitting).\n            sep=[k for k in keys if k[1].month==9];october=[k for k in keys if k[1].month==10]\n            sept_choices,_=choose_routes(sep,truth,forecasts,routes)\n            op=combine(october,forecasts,sept_choices)\n            report[\'october_forward_diagnostic\']={\'selection\':\'September only; forecasts fixed at Aug31\',\n                \'choices\':sept_choices,\'metrics\':score(october,{k:truth[k] for k in october if k in truth},op)}\n            choices,diagnostics=choose_routes(keys,truth,forecasts,routes)\n            ensemble=combine(keys,forecasts,choices)\n            report[\'folds\'][label][\'route_ensemble\']={\'metrics\':score(keys,truth,ensemble)}\n            report[\'selection\']={\'selected\':\'route_ensemble\',\'by_route\':choices,\'diagnostics\':diagnostics,\n                \'rule\':\'Per route: lower pooled Sep-Oct error AND >=1% reduction in each month vs school_calendar; weights 0.5 or 1.0.\'}\n        dump(args.out/\'report.json\',report)\n    selected_names={\'school_calendar\'}|{c[\'candidate\'] for c in choices.values()}\n    fitted={n:factories[n]().fit(values,FIRST,FINAL_CUTOFF) for n in sorted(selected_names)}\n    model=RouteEnsemble(fitted,choices,FINAL_CUTOFF);predictions=model.predict(sample)\n    report[\'code_sha256\']={q.name:sha256(q) for q in sorted(Path(__file__).parent.glob(\'*.py\'))}\n    identity=json.dumps({\'inputs\':inputs,\'choices\':choices,\'code\':report[\'code_sha256\'],\n                         \'libraries\':report[\'library_versions\']},sort_keys=True).encode()\n    version=\'ds1-route-\'+hashlib.sha256(identity).hexdigest()[:12]\n    if args.synthetic:version=\'synthetic-NOT-FOR-SUBMISSION-\'+version\n    report[\'model_version\']=version\n    report[\'cold_start\']={\'routes\':sorted({k[0] for k in sample}-set(routes),key=int),\n                          \'policy\':\'zero placeholder; unvalidated, not a required true target\'}\n    report[\'submission\']=write_submission(args.sample,predictions,args.out/\'submission.csv\')\n    export_api(sample,predictions,args.out/\'forecast_api.csv\',version)\n    (args.out/\'model.local.pkl\').write_bytes(pickle.dumps(model,protocol=5))\n    validate(args.sample,args.out/\'submission.csv\')\n    report[\'outputs_sha256\']={n:sha256(args.out/n) for n in (\'submission.csv\',\'forecast_api.csv\',\'model.local.pkl\')}\n    report[\'matches_previous_submission\'] = report[\'outputs_sha256\'][\'submission.csv\'] == \'371d05c44e20b63db110f5a5e27b8da8fafb785fdbc66374307b0ab7304a75ac\'\n    dump(args.out/\'report.json\',report)\n    print(\'Selected route models:\',json.dumps(choices,ensure_ascii=False),flush=True)\n    print(\'Validated:\',version, len(sample),\'rows; platform score unknown.\',flush=True)\n    return report\n\nif __name__==\'__main__\':\n    from ml.forecast.route_specific import main as run_main\n    run_main()\n', 'ml/forecast/test_route_specific.py': "import tempfile\nimport importlib.util\nimport unittest\nfrom datetime import date,timedelta\nfrom pathlib import Path\nfrom unittest.mock import patch\nimport numpy as np\nfrom ml.forecast.core import grid\nfrom ml.forecast.models import Boosting\nfrom ml.forecast.route_specific import SeasonalProfile,choose_routes,combine,verify_inputs,candidate_factories\n\n\nclass RouteTests(unittest.TestCase):\n    def test_unverified_inputs_rejected_before_training(self):\n        with tempfile.TemporaryDirectory() as td:\n            p=Path(td)/'fake.csv';p.write_text('synthetic')\n            with self.assertRaisesRegex(ValueError,'does not match real DS-2'):\n                verify_inputs({'train':p})\n            self.assertIn('train',verify_inputs({'train':p},synthetic=True))\n\n    def test_autumn_does_not_inherit_summer_weekday_slump(self):\n        start,cutoff=date(2025,1,1),date(2025,8,31)\n        hist={k: (20 if k[1].month in (7,8) and k[1].weekday()<5 else 100)\n              for k in grid(['1'],start,cutoff)}\n        model=SeasonalProfile(calendar=False).fit(hist,start,cutoff)\n        k=('1',date(2025,9,1),12)\n        self.assertEqual(model.predict([k])[k],100)\n        self.assertEqual(model.predict([('5',date(2025,9,1),12)])[('5',date(2025,9,1),12)],0)\n        hist['1',date(2025,9,1),12]=999\n        with self.assertRaises(ValueError):model.fit(hist,start,cutoff)\n\n    def test_route_choices_differ_and_single_month_overfit_rejected(self):\n        keys=[(r,date(2025,m,1),12) for r in ('1','7') for m in (9,10)]\n        truth={k:100 for k in keys}\n        preds={'school_calendar':dict.fromkeys(keys,80),\n               'a':{k:100 if k[0]=='1' else 70 for k in keys},\n               'b':{k:100 if k[0]=='7' else 70 for k in keys},\n               'overfit':{k:100 if k[1].month==9 else 10 for k in keys}}\n        choices,_=choose_routes(keys,truth,preds,['1','7'])\n        self.assertEqual(choices['1']['candidate'],'a')\n        self.assertEqual(choices['7']['candidate'],'b')\n        self.assertEqual(combine(keys,preds,choices),truth)\n\n    def test_boosting_target_aggregates_are_strictly_past(self):\n        start,cutoff=date(2025,1,1),date(2025,3,31)\n        hist={k:(k[1]-start).days*100+k[2] for k in grid(['1'],start,cutoff)}\n        observed=[]\n        class Spy(Boosting):\n            def _extract_profiles(self,history,start,cutoff):\n                self.profile_end=cutoff\n                return super()._extract_profiles(history,start,cutoff)\n            def features(self,keys,context=None):\n                observed.append((self.profile_end,min(k[1] for k in keys)))\n                assert self.profile_end<min(k[1] for k in keys)\n                return super().features(keys,context)\n        Spy(kind='hgb').fit(hist,start,cutoff)\n        self.assertGreaterEqual(len(observed),3)\n\n    @unittest.skipUnless(importlib.util.find_spec('lightgbm'), 'optional LightGBM')\n    def test_training_error_cannot_silently_change_library(self):\n        hist={k:100 for k in grid(['1'],date(2025,1,1),date(2025,2,28))}\n        with patch('lightgbm.LGBMRegressor.fit',side_effect=ValueError('broken fit')):\n            with self.assertRaisesRegex(RuntimeError,'no silent fallback'):\n                Boosting(kind='lgb').fit(hist,date(2025,1,1),date(2025,2,28))\n\n    def test_candidates_have_no_future_route50_rule(self):\n        for name,factory in candidate_factories().items():\n            model=factory()\n            self.assertFalse(getattr(model,'route50_rule',False),name)\n\nif __name__=='__main__':unittest.main()\n", 'ml/forecast/run.py': '"""One-command chronological backtest, model selection and exact submission."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport pickle\nimport platform\nimport time\nfrom datetime import date\nfrom pathlib import Path\n\n# A bounded CPU budget also avoids OpenMP oversubscription on desktops.\nos.environ.setdefault(\'OMP_NUM_THREADS\', \'2\')\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\', \'2\')\n\nfrom .core import grid, load_labels, peak_hours, score, sha256\nfrom .models import candidates\nfrom ml.submit.adapter import export_api, read_sample, validate, write_submission\n\nFIRST, FINAL_CUTOFF = date(2025, 1, 1), date(2025, 10, 31)\nFOLDS = [\n    (\'may_jun\', date(2025, 4, 30), date(2025, 5, 1), date(2025, 6, 30)),\n    (\'jul_aug\', date(2025, 6, 30), date(2025, 7, 1), date(2025, 8, 31)),\n    (\'sep_oct\', date(2025, 8, 31), date(2025, 9, 1), date(2025, 10, 31)),\n]\n\n\ndef dump(path, data):\n    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\',\n                          encoding=\'utf-8\')\n\n\ndef evaluate_fold(values, cutoff, start, end, factory):\n    history = {k: v for k, v in values.items() if FIRST <= k[1] <= cutoff}\n    routes = sorted({k[0] for k in history}, key=int)\n    truth = {k: v for k, v in values.items() if start <= k[1] <= end}\n    if {k[0] for k in truth} - set(routes):\n        raise ValueError(\'Validation has new routes: define cold-start evaluation explicitly\')\n    keys = grid(routes, start, end)\n    t = time.perf_counter()\n    model = factory().fit(history, FIRST, cutoff)\n    predictions = model.predict(keys)\n    metrics = score(keys, truth, predictions, peak_hours(history, routes))\n    return {\'cutoff\': str(cutoff), \'from\': str(start), \'to_inclusive\': str(end),\n            \'routes\': routes, \'metrics\': metrics,\n            \'fit_predict_score_seconds\': round(time.perf_counter() - t, 3)}\n\n\ndef select_dev(folds, names):\n    """Select on May–August only; pool absolute errors, never average route scores."""\n    losses = {name: sum(folds[fold][name][\'metrics\'][\'all\'][\'absolute_error_sum\']\n                       for fold in (\'may_jun\', \'jul_aug\')) for name in names}\n    return min(losses, key=lambda n: (losses[n], n)), losses\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--train\', required=True, type=Path)\n    parser.add_argument(\'--test\', required=True, type=Path)\n    parser.add_argument(\'--sample\', required=True, type=Path)\n    parser.add_argument(\'--out\', required=True, type=Path)\n    parser.add_argument(\'--profiles-only\', action=\'store_true\',\n                        help=\'Skip optional scikit-learn candidates explicitly\')\n    parser.add_argument(\'--synthetic\', action=\'store_true\',\n                        help=\'Mark artificial smoke data and outputs as NOT FOR SUBMISSION\')\n    args = parser.parse_args(argv)\n    from .route_specific import verify_inputs\n    verify_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')}, args.synthetic)\n    # Fail on malformed input before training or creating any outputs.\n    _, sample_keys = read_sample(args.sample)\n    train, test = load_labels([args.train]), load_labels([args.test])\n    if any(not FIRST <= k[1] <= date(2025, 8, 31) for k in train):\n        raise ValueError(\'Train must be January–August 2025 only\')\n    if any(not date(2025, 9, 1) <= k[1] <= FINAL_CUTOFF for k in test):\n        raise ValueError(\'Test labels must be September–October 2025 only\')\n    values = load_labels([args.train, args.test])\n    # Missing hours are allowed, an entirely missing date is a data availability problem.\n    for a, b, data in ((FIRST, date(2025, 8, 31), train),\n                       (date(2025, 9, 1), FINAL_CUTOFF, test)):\n        if {k[1] for k in data} != {k[1] for k in grid([\'check\'], a, b)}:\n            raise ValueError(\'Labels have missing full dates; audit before running model selection\')\n    if {k[0] for k in values} - {k[0] for k in sample_keys}:\n        raise ValueError(\'Historical routes absent from sample\')\n    models = candidates(not args.profiles_only)\n    if not args.profiles_only:\n        import sklearn  # noqa: F401; fail early instead of silently skipping boosting.\n    args.out.mkdir(parents=True, exist_ok=True)\n    inputs = {name: {\'name\': path.name, \'sha256\': sha256(path)}\n              for name, path in [(\'train\', args.train), (\'test\', args.test), (\'sample\', args.sample)]}\n    code_files = sorted(Path(__file__).parent.glob(\'*.py\')) + [\n        Path(__file__).parents[1] / \'submit\' / \'adapter.py\']\n    code = {str(p.relative_to(Path(__file__).parents[2])): sha256(p) for p in code_files}\n    installed_libs = {}\n    if not args.profiles_only:\n        for pkg in (\'numpy\', \'scipy\', \'scikit-learn\', \'lightgbm\', \'catboost\'):\n            try:\n                installed_libs[pkg] = importlib.metadata.version(pkg)\n            except importlib.metadata.PackageNotFoundError:\n                pass\n    report = {\n        \'schema_version\': 1, \'inputs\': inputs, \'code_sha256\': code,\n        \'data_kind\': \'synthetic_not_for_submission\' if args.synthetic else \'organizer_labels\',\n        \'python_version\': platform.python_version(),\n        \'library_versions\': installed_libs,\n        \'source_mode\': \'as_of_2025_10_31\', \'timezone\': \'Europe/Moscow\',\n        \'missing_label_policy\': \'zero for scoring; compare zero-filled and observed-only profiles\',\n        \'rounding\': \'clip negatives to zero; Python round half-to-even\',\n        \'peak_definition\': \'four largest hour sums per route in each training window only\',\n        \'night_definition\': \'23:00–05:59 MSK diagnostic; not an operating-hours assertion\',\n        \'external_factors\': \'Russian 2025 production calendar, route 50 weekend resumption policy\',\n        \'parameters\': {\'profile_shrink\': .2, \'short_window_days\': 56, \'recent_weight\': 0.6,\n                       \'lgb\': {\'objective\': \'regression_l1\', \'learning_rate\': .03, \'n_estimators\': 300, \'num_leaves\': 15},\n                       \'catboost\': {\'loss_function\': \'MAE\', \'iterations\': 300, \'learning_rate\': .04, \'depth\': 5},\n                       \'calendar\': \'Russian 2025 production calendar with working Saturdays and state holidays\',\n                       \'route50_policy\': \'weekend repair 2025-09-06..2025-11-14; resumed 2025-11-15\'},\n        \'folds\': {}, \'selection\': {}, \'platform_score\': None,\n    }\n    for fold, cutoff, start, end in FOLDS[:2]:\n        report[\'folds\'][fold] = {}\n        for name, factory in models.items():\n            result = evaluate_fold(values, cutoff, start, end, factory)\n            report[\'folds\'][fold][name] = result\n            print(f\'{fold} {name}: WAPE-score={result["metrics"]["all"]["wape_score"]:.6f}\', flush=True)\n        dump(args.out / \'report.json\', report)\n    proposed, losses = select_dev(report[\'folds\'], models)\n    baseline, _ = select_dev(report[\'folds\'], [\'mean_zero\', \'median_zero\'])\n    report[\'selection\'].update(development_candidate=proposed, development_baseline=baseline,\n                               development_absolute_errors=losses)\n    # Preselected candidate plus controls and calendar profile see Sep–Oct.\n    report[\'folds\'][\'sep_oct\'] = {}\n    candidates_to_eval = dict.fromkeys([baseline, \'mean_zero\', \'median_zero\', \'profile_calendar\', proposed])\n    for name in candidates_to_eval:\n        result = evaluate_fold(values, *FOLDS[2][1:], models[name])\n        report[\'folds\'][\'sep_oct\'][name] = result\n        print(f\'sep_oct {name}: WAPE-score={result["metrics"]["all"]["wape_score"]:.6f}\', flush=True)\n    latest = report[\'folds\'][\'sep_oct\']\n    err = lambda name: latest[name][\'metrics\'][\'all\'][\'absolute_error_sum\']\n    final_baseline = min([\'mean_zero\', \'median_zero\'], key=lambda n: (err(n), n))\n    best_candidate = min([c for c in [proposed, \'profile_calendar\'] if c in latest], key=lambda n: (err(n), n))\n    selected = best_candidate if err(best_candidate) < err(final_baseline) else final_baseline\n    report[\'selection\'].update(selected=selected,\n        final_baseline=final_baseline,\n        rule=\'candidate from pooled May–August error; keep it only if better than both registered baselines in Sep–Oct\',\n        caveat=\'Sep–Oct is a validation gate used in selection, not an untouched test estimate\')\n    # Refit final model on all available history through Oct 31, independently of backtest fits.\n    model = models[selected]().fit(values, FIRST, FINAL_CUTOFF)\n    predictions = model.predict(sample_keys)\n    cold = sorted({k[0] for k in sample_keys} - {k[0] for k in values}, key=int)\n    report[\'cold_start\'] = {\'routes\': cold, \'policy\': \'zero; no evidence to calibrate unseen routes\',\n                            \'evaluated\': False, \'route5_rows_preserved\': True}\n    identity = json.dumps({\'inputs\': inputs, \'code\': code, \'selected\': selected,\n                           \'parameters\': report[\'parameters\'], \'libraries\': report[\'library_versions\'],\n                           \'python\': report[\'python_version\']}, sort_keys=True).encode()\n    version = f\'ds1-{selected}-\' + hashlib.sha256(identity).hexdigest()[:12]\n    if args.synthetic:\n        version = \'synthetic-NOT-FOR-SUBMISSION-\' + version\n    report[\'model_version\'] = version\n    # Local artifact only. Never load an untrusted pickle and never publish organizer-derived artifacts.\n    artifact_path = args.out / \'model.local.pkl\'\n    artifact_path.write_bytes(pickle.dumps(model, protocol=5))\n    output_path = args.out / \'submission.csv\'\n    report[\'submission\'] = write_submission(args.sample, predictions, output_path)\n    export_api(sample_keys, predictions, args.out / \'forecast_api.csv\', version)\n    report[\'outputs_sha256\'] = {p.name: sha256(p) for p in\n                                (artifact_path, output_path, args.out / \'forecast_api.csv\')}\n    if sha256(args.sample) != inputs[\'sample\'][\'sha256\']:\n        raise RuntimeError(\'Original sample changed during run\')\n    validate(args.sample, output_path)\n    dump(args.out / \'report.json\', report)\n    print(f\'Selected {version}; validated {len(sample_keys)} rows. Platform result unknown.\', flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    main()\n', 'ml/forecast/prepare_inputs.py': '"""Extract only labels/sample/description from a local organizer ZIP, never raw events."""\nimport argparse\nimport json\nimport zipfile\nfrom pathlib import Path\n\nfrom .core import sha256\n\n\ndef extract(archive, output):\n    required = {\'labels_day_train.csv\', \'labels_day_test.csv\', \'test_submission.csv\'}\n    output = Path(output)\n    with zipfile.ZipFile(archive) as z:\n        members = {}\n        for info in z.infolist():\n            name = Path(info.filename).name\n            if name not in required and name.lower() != \'readme.md\':\n                continue\n            if name in members:\n                raise ValueError(f\'Ambiguous archive: duplicate {name}\')\n            if info.file_size > 20_000_000:\n                raise ValueError(f\'Unexpectedly large label/description file: {name}\')\n            members[name] = info\n        if not required <= members.keys():\n            raise ValueError(f\'Missing archive members: {sorted(required - members.keys())}\')\n        if any((output / name).exists() for name in members):\n            raise ValueError(\'Refuse to overwrite original inputs; use an empty directory\')\n        output.mkdir(parents=True, exist_ok=True)\n        for name, info in members.items():\n            (output / name).write_bytes(z.read(info))\n    manifest = {\'archive_name\': Path(archive).name,\n                \'source_url\': \'https://disk.yandex.ru/d/DiFwlfMOauxjBg\',\n                \'files_sha256\': {name: sha256(output / name) for name in sorted(members)}}\n    (output / \'inputs_manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    return manifest\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--archive\', required=True, type=Path)\n    p.add_argument(\'--out\', required=True, type=Path)\n    a = p.parse_args()\n    print(json.dumps(extract(a.archive, a.out), indent=2))\n', 'ml/forecast/test_pipeline.py': '"""Self-contained synthetic contract/leakage tests; no organizer data required."""\nimport csv\nimport importlib.util\nimport tempfile\nimport unittest\nimport zipfile\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nfrom ml.forecast.core import grid, load_labels, rounded, score, sha256\nfrom ml.forecast.models import Boosting, Profile\nfrom ml.forecast.run import evaluate_fold, select_dev\nfrom ml.forecast.prepare_inputs import extract\nfrom ml.submit.adapter import export_api, read_sample, validate, write_submission\n\n\nclass PipelineTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.root = Path(self.tmp.name)\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def sample(self):\n        path = self.root / \'sample.csv\'\n        # Deliberately reverse order to detect implicit model sorting of sample rows.\n        keys = grid([str(i) for i in range(1, 11)], date(2025, 11, 1), date(2025, 12, 31))[::-1]\n        with path.open(\'w\', newline=\'\') as f:\n            writer = csv.writer(f, delimiter=\';\')\n            writer.writerow([\'route\', \'date\', \'hour\', \'prediction\'])\n            writer.writerows((r, d, h, \'\') for r, d, h in keys)\n        return path, keys\n\n    def test_exact_sample_roundtrip_api_sums_and_route5(self):\n        sample, keys = self.sample()\n        digest = sha256(sample)\n        preds = {k: (int(k[0]) * 10 + k[2] + .5) for k in keys}\n        out = self.root / \'out.csv\'\n        result = write_submission(sample, preds, out)\n        self.assertEqual(result[\'route5_rows\'], 1464)\n        self.assertEqual(read_sample(out)[1], keys)\n        self.assertEqual(sha256(sample), digest)\n        api = self.root / \'api.csv\'\n        export_api(keys, preds, api, \'synthetic-test\')\n        with api.open() as f:\n            rows = list(csv.DictReader(f))\n        self.assertEqual(sum(int(r[\'pred_validations\']) for r in rows), result[\'prediction_sum\'])\n        self.assertTrue(all(r[\'timestamp_msk\'].endswith(\'+03:00\') for r in rows))\n        daily = {}\n        monthly = {}\n        for row in rows:\n            d = row[\'timestamp_msk\'][:10]\n            daily[d] = daily.get(d, 0) + int(row[\'pred_validations\'])\n        for d, value in daily.items():\n            monthly[d[:7]] = monthly.get(d[:7], 0) + value\n        self.assertEqual(sum(monthly.values()), result[\'prediction_sum\'])\n\n    def test_reject_bad_submission_inputs(self):\n        sample, keys = self.sample()\n        predictions = dict.fromkeys(keys, 1)\n        out = self.root / \'out.csv\'\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, sample)\n        predictions.pop(keys[0])\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, out)\n        predictions[keys[0]] = float(\'nan\')\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, out)\n        predictions[keys[0]] = 1\n        write_submission(sample, predictions, out)\n        lines = out.read_text().splitlines()\n        lines[1], lines[2] = lines[2], lines[1]\n        out.write_text(\'\\n\'.join(lines) + \'\\n\')\n        with self.assertRaises(ValueError):\n            validate(sample, out)\n\n    def test_sample_missing_duplicate_wrong_period(self):\n        path, _ = self.sample()\n        lines = path.read_text().splitlines()\n        path.write_text(\'\\n\'.join(lines[:-1]) + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n        path.write_text(\'\\n\'.join(lines[:-1] + [lines[1]]) + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n        path.write_text(\'\\n\'.join(lines).replace(\'2025-11-01\', \'2025-10-31\') + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n\n    def test_metric_is_pooled_and_includes_missing_hours(self):\n        keys = [(\'1\', date(2025, 9, 1), 7), (\'2\', date(2025, 9, 1), 7)]\n        m = score(keys, {keys[0]: 100}, {keys[0]: 80, keys[1]: 10})\n        self.assertAlmostEqual(m[\'all\'][\'wape_score\'], .7)\n        self.assertIsNone(m[\'route:2\'][\'wape_score\'])\n        with self.assertRaises(ValueError):\n            score(keys, {}, dict.fromkeys(keys, 0))\n        with self.assertRaises(ValueError):\n            score(keys, {keys[0]: 1}, {keys[0]: 1})\n\n    def test_reader_rejects_duplicate_files_and_negative_counts(self):\n        p = self.root / \'labels.csv\'\n        p.write_text(\'route;date;hour;boardings\\n7;2025-01-01;0;2\\n\')\n        with self.assertRaises(ValueError):\n            load_labels([p, p])\n        p.write_text(\'route;date;hour;boardings\\n7;2025-01-01;0;-2\\n\')\n        with self.assertRaises(ValueError):\n            load_labels([p])\n\n    def test_fit_rejects_future_and_predict_rejects_past(self):\n        cutoff = date(2025, 1, 31)\n        history = {(\'7\', date(2025, 1, 1), 7): 20}\n        model = Profile().fit(history, date(2025, 1, 1), cutoff)\n        with self.assertRaises(ValueError):\n            model.predict(list(history))\n        history[\'7\', cutoff + timedelta(days=1), 7] = 1000\n        with self.assertRaises(ValueError):\n            Profile().fit(history, date(2025, 1, 1), cutoff)\n\n    def test_backtest_never_passes_future_labels_to_model(self):\n        values = {(\'7\', date(2025, 1, 1), 7): 10,\n                  (\'7\', date(2025, 2, 1), 7): 2000}\n        class Spy(Profile):\n            def fit(self, history, start, cutoff):\n                assert max(k[1] for k in history) <= cutoff\n                assert max(history.values()) == 10\n                return super().fit(history, start, cutoff)\n        result = evaluate_fold(values, date(2025, 1, 31), date(2025, 2, 1), date(2025, 2, 1), Spy)\n        self.assertEqual(result[\'metrics\'][\'all\'][\'rows\'], 24)\n\n    def test_zero_cold_start_and_missing_training_sensitivity(self):\n        history = {(\'7\', date(2025, 1, 1), 7): 40}\n        keys = [(\'7\', date(2025, 2, 5), 7), (\'5\', date(2025, 2, 5), 7)]\n        zero = Profile().fit(history, date(2025, 1, 1), date(2025, 1, 31)).predict(keys)\n        obs = Profile(missing=\'observed\').fit(history, date(2025, 1, 1), date(2025, 1, 31)).predict(keys)\n        self.assertEqual(zero[keys[1]], 0)\n        self.assertLess(zero[keys[0]], obs[keys[0]])\n\n    def test_rounding_is_finite_nonnegative(self):\n        self.assertEqual([rounded(x) for x in [-2, .5, 1.5, 2.5]], [0, 0, 2, 2])\n        for x in [float(\'inf\'), float(\'nan\'), 10**15]:\n            with self.assertRaises(ValueError):\n                rounded(x)\n\n    def test_selection_does_not_use_sep_oct(self):\n        def entry(n):\n            return {\'metrics\': {\'all\': {\'absolute_error_sum\': n}}}\n        folds = {\'may_jun\': {\'a\': entry(1), \'b\': entry(10)},\n                 \'jul_aug\': {\'a\': entry(5), \'b\': entry(1)},\n                 \'sep_oct\': {\'a\': entry(10000), \'b\': entry(0)}}\n        self.assertEqual(select_dev(folds, [\'a\', \'b\'])[0], \'a\')\n\n    def test_archive_extracts_only_needed_files_and_preserves_originals(self):\n        archive = self.root / \'data.zip\'\n        with zipfile.ZipFile(archive, \'w\') as z:\n            for name in [\'labels_day_train.csv\', \'labels_day_test.csv\', \'test_submission.csv\']:\n                z.writestr(\'nested/\' + name, \'synthetic\')\n            z.writestr(\'../../raw.csv\', \'must not be extracted\')\n            z.writestr(\'README.md\', \'Synthetic description\')\n        dest = self.root / \'inputs\'\n        manifest = extract(archive, dest)\n        self.assertEqual(len(manifest[\'files_sha256\']), 4)\n        self.assertFalse((self.root / \'raw.csv\').exists())\n        self.assertFalse((dest / \'raw.csv\').exists())\n        with self.assertRaises(ValueError):\n            extract(archive, dest)\n\n    @unittest.skipUnless(importlib.util.find_spec(\'numpy\'), \'optional boosting dependency\')\n    def test_recursive_lags_use_predictions_and_fill_intervening_days(self):\n        class FakeRegressor:\n            def __init__(self):\n                self.batches = []\n            def predict(self, features):\n                self.batches.append(features)\n                return [7] * len(features)\n        model = Boosting(recursive=True)\n        model.cutoff = date(2025, 10, 31)\n        model.routes, model.route_codes = [\'7\'], {\'7\': 0}\n        model.history = {(\'7\', model.cutoff, h): 99 for h in range(24)}\n        model.model = FakeRegressor()\n        key = (\'7\', date(2025, 11, 3), 12)\n        self.assertEqual(model.predict([key]), {key: 7})\n        self.assertEqual(len(model.model.batches), 3)\n        self.assertTrue(all(row[8] == 99 for row in model.model.batches[0]))\n        self.assertTrue(all(row[8] == 7 for row in model.model.batches[1]))\n        self.assertTrue(all(row[8] == 7 for row in model.model.batches[2]))\n        self.assertEqual(model.history[\'7\', model.cutoff, 12], 99)\n\n    def test_russian_calendar_2025(self):\n        from ml.forecast.core import is_workday, effective_weekday, calendar_day_type\n        # 2025-11-01 is a working Saturday\n        self.assertTrue(is_workday(date(2025, 11, 1)))\n        self.assertEqual(effective_weekday(date(2025, 11, 1)), 4)\n        self.assertEqual(calendar_day_type(date(2025, 11, 1)), 4)\n\n        # 2025-11-02 is Sunday\n        self.assertFalse(is_workday(date(2025, 11, 2)))\n        self.assertEqual(effective_weekday(date(2025, 11, 2)), 6)\n\n        # 2025-11-03 is non-working day (transferred holiday)\n        self.assertFalse(is_workday(date(2025, 11, 3)))\n        self.assertEqual(effective_weekday(date(2025, 11, 3)), 6)\n\n        # 2025-11-04 is National Unity Day\n        self.assertFalse(is_workday(date(2025, 11, 4)))\n        self.assertEqual(effective_weekday(date(2025, 11, 4)), 6)\n\n        # 2025-11-05 is normal Wednesday\n        self.assertTrue(is_workday(date(2025, 11, 5)))\n        self.assertEqual(effective_weekday(date(2025, 11, 5)), 2)\n\n        # 2025-12-31 is non-working day\n        self.assertFalse(is_workday(date(2025, 12, 31)))\n        self.assertEqual(effective_weekday(date(2025, 12, 31)), 6)\n\n    def test_calendar_profile_and_route50_rule(self):\n        from ml.forecast.models import CalendarProfile\n        history = {}\n        for d in [date(2025, 10, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'50\', d, h] = 50 if d.weekday() < 5 else 0\n                history[\'1\', d, h] = 100 if d.weekday() < 5 else 20\n        cutoff = date(2025, 10, 31)\n        model = CalendarProfile(recent_weight=0.5, window=28, route50_rule=True).fit(history, date(2025, 10, 1), cutoff)\n\n        # 2025-11-01 is working Saturday: route 1 should have weekday traffic (~100)\n        p1_work = model.predict([(\'1\', date(2025, 11, 1), 12)])[(\'1\', date(2025, 11, 1), 12)]\n        self.assertGreater(p1_work, 80)\n\n        # 2025-11-04 is holiday: route 1 should have weekend/Sunday traffic (~20)\n        p1_hol = model.predict([(\'1\', date(2025, 11, 4), 12)])[(\'1\', date(2025, 11, 4), 12)]\n        self.assertLess(p1_hol, 30)\n\n        # Route 50 suspended on weekend before Nov 15\n        p50_susp = model.predict([(\'50\', date(2025, 11, 2), 12)])[(\'50\', date(2025, 11, 2), 12)]\n        self.assertEqual(p50_susp, 0)\n\n    @unittest.skipUnless(importlib.util.find_spec(\'lightgbm\'), \'lightgbm required\')\n    def test_lgbm_direct_fit_and_predict(self):\n        history = {}\n        for d in [date(2025, 9, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'1\', d, h] = 100 + h * 2 if d.weekday() < 5 else 20 + h\n        cutoff = date(2025, 9, 30)\n        model = Boosting(kind=\'lgb\', recursive=False).fit(history, date(2025, 9, 1), cutoff)\n        preds = model.predict([(\'1\', date(2025, 10, 1), 10), (\'1\', date(2025, 10, 4), 10)])\n        self.assertGreater(preds[\'1\', date(2025, 10, 1), 10], preds[\'1\', date(2025, 10, 4), 10])\n\n    @unittest.skipUnless(importlib.util.find_spec(\'catboost\'), \'catboost required\')\n    def test_catboost_direct_fit_and_predict(self):\n        history = {}\n        for d in [date(2025, 9, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'1\', d, h] = 100 + h * 2 if d.weekday() < 5 else 20 + h\n        cutoff = date(2025, 9, 30)\n        model = Boosting(kind=\'catboost\', recursive=False).fit(history, date(2025, 9, 1), cutoff)\n        preds = model.predict([(\'1\', date(2025, 10, 1), 10)])\n        self.assertGreater(preds[\'1\', date(2025, 10, 1), 10], 50)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n\n', 'ml/forecast/download_yandex.py': '"""Download organizer ZIP through the public Yandex Disk API, without credentials."""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport time\nimport urllib.parse\nimport urllib.request\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nPUBLIC_URL = \'https://disk.yandex.ru/d/DiFwlfMOauxjBg\'\nAPI = \'https://cloud-api.yandex.net/v1/disk/public/resources\'\n\n\ndef api_json(suffix, params, attempts=3):\n    url = API + suffix + \'?\' + urllib.parse.urlencode(params)\n    for attempt in range(attempts):\n        try:\n            with urllib.request.urlopen(url, timeout=30) as response:\n                return json.load(response)\n        except (OSError, ValueError) as exc:\n            if attempt == attempts - 1:\n                raise RuntimeError(\'API Яндекс Диска недоступен. Включите Internet в Kaggle; \'\n                                   \'если уже включён, повторите ячейку позже. \'\n                                   \'Также возможен лимит скачивания публичной ссылки.\') from exc\n            print(f\'API: повторная попытка {attempt + 2}/{attempts}\', flush=True)\n            time.sleep(2 ** (attempt + 1))\n\n\ndef digest(path, algorithm=\'sha256\'):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, algorithm).hexdigest()\n\n\ndef download(public_url=PUBLIC_URL, cache_dir=\'/tmp/mthack_yandex\', attempts=3):\n    params = {\'public_key\': public_url}\n    meta = api_json(\'\', params)\n    if meta.get(\'type\') == \'dir\':\n        # Organizer link is a public folder containing dataset.zip.\n        params[\'path\'] = \'/dataset.zip\'\n        meta = api_json(\'\', params)\n    if meta.get(\'type\') != \'file\' or not meta.get(\'name\', \'\').lower().endswith(\'.zip\'):\n        raise ValueError(\'Ссылка должна вести на ZIP или папку с dataset.zip\')\n    expected_size = int(meta[\'size\'])\n    if expected_size <= 0:\n        raise ValueError(\'Пустой архив в метаданных Яндекс Диска\')\n    algorithm = \'sha256\' if meta.get(\'sha256\') else \'md5\'\n    expected_hash = meta.get(algorithm)\n    identity = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:16]\n    directory = Path(cache_dir) / identity\n    directory.mkdir(parents=True, exist_ok=True)\n    target, partial = directory / \'dataset.zip\', directory / \'dataset.zip.part\'\n\n    def valid(path):\n        return (path.exists() and path.stat().st_size == expected_size\n                and (not expected_hash or digest(path, algorithm) == expected_hash)\n                and zipfile.is_zipfile(path))\n\n    if valid(target):\n        print(\'Используется ранее проверенный архив:\', target, flush=True)\n        return target\n    print(f\'Архив: {expected_size / 1024**3:.2f} ГиБ. Потоковое скачивание во временную папку.\',\n          flush=True)\n    for attempt in range(attempts):\n        try:\n            offset = partial.stat().st_size if partial.exists() else 0\n            if offset >= expected_size:\n                if valid(partial):\n                    break\n                partial.unlink()\n                offset = 0\n            if shutil.disk_usage(directory).free < expected_size - offset + 64 * 1024**2:\n                raise RuntimeError(\'Недостаточно места для архива во временной папке\')\n            href = api_json(\'/download\', params)[\'href\']\n            if urllib.parse.urlparse(href).scheme != \'https\':\n                raise ValueError(\'API вернул незащищённую ссылку скачивания\')\n            headers = {\'Range\': f\'bytes={offset}-\'} if offset else {}\n            request = urllib.request.Request(href, headers=headers)\n            with urllib.request.urlopen(request, timeout=60) as response:\n                status = response.status\n                if status == 206:\n                    match = re.fullmatch(r\'bytes (\\d+)-(\\d+)/(\\d+)\',\n                                         response.headers.get(\'Content-Range\', \'\'))\n                    if not match or int(match[1]) != offset or int(match[3]) != expected_size:\n                        raise ValueError(\'Некорректный диапазон при докачке\')\n                    mode = \'ab\'\n                elif status == 200:\n                    offset, mode = 0, \'wb\'  # Server ignored Range: safely restart.\n                else:\n                    raise ValueError(f\'Неожиданный HTTP-статус: {status}\')\n                received, logged = offset, offset\n                with partial.open(mode) as f:\n                    while chunk := response.read(4 * 1024**2):\n                        f.write(chunk)\n                        received += len(chunk)\n                        if received > expected_size:\n                            raise ValueError(\'Размер загрузки превысил метаданные\')\n                        if received - logged >= 64 * 1024**2 or received == expected_size:\n                            print(f\'Скачано {received / expected_size:.1%} \'\n                                  f\'({received / 1024**2:.0f}/{expected_size / 1024**2:.0f} МиБ)\',\n                                  flush=True)\n                            logged = received\n            if partial.stat().st_size != expected_size:\n                raise OSError(\'Соединение прервано до конца файла\')\n            if not valid(partial):\n                partial.unlink()\n                raise ValueError(\'Проверка хеша или структуры ZIP не пройдена\')\n            break\n        except (OSError, ValueError) as exc:\n            if attempt == attempts - 1:\n                raise RuntimeError(\'Архив не скачан после повторных попыток. \'\n                                   \'Проверьте Internet и повторите ячейку: частичный файл сохранён \'\n                                   \'для докачки, если сервер поддерживает Range.\') from exc\n            print(f\'Скачивание: повторная попытка {attempt + 2}/{attempts}\', flush=True)\n            time.sleep(2 ** (attempt + 1))\n    partial.replace(target)\n    manifest = {\'source_url\': public_url, \'path\': params.get(\'path\'), \'size\': expected_size,\n                \'sha256\': digest(target), \'fetched_at_utc\': datetime.now(timezone.utc).isoformat(),\n                \'server_hash_algorithm\': algorithm if expected_hash else None,\n                \'server_hash_verified\': bool(expected_hash)}\n    (directory / \'download_manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    print(\'Архив скачан и проверен:\', target, flush=True)\n    return target\n', 'ml/forecast/test_download_yandex.py': '"""Network-independent tests: streaming, resume, cache and integrity checks."""\nimport hashlib\nimport io\nimport tempfile\nimport unittest\nimport zipfile\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nfrom ml.forecast import download_yandex as yd\n\n\nclass Response(io.BytesIO):\n    def __init__(self, data, status=200, headers=None):\n        super().__init__(data)\n        self.status = status\n        self.headers = headers or {}\n\n\nclass DownloadTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.root = Path(self.tmp.name)\n        buffer = io.BytesIO()\n        with zipfile.ZipFile(buffer, \'w\') as z:\n            z.writestr(\'labels/synthetic.txt\', \'artificial fixture\')\n        self.blob = buffer.getvalue()\n        self.meta = {\'type\': \'file\', \'name\': \'dataset.zip\', \'size\': len(self.blob),\n                     \'sha256\': hashlib.sha256(self.blob).hexdigest()}\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def api(self, suffix, params):\n        if suffix == \'/download\':\n            self.assertEqual(params[\'path\'], \'/dataset.zip\')\n            return {\'href\': \'https://download.example.invalid/data.zip\'}\n        return self.meta if \'path\' in params else {\'type\': \'dir\'}\n\n    def run_download(self, opener, attempts=2):\n        with patch.object(yd, \'api_json\', side_effect=self.api), \\\n             patch.object(yd.urllib.request, \'urlopen\', side_effect=opener), \\\n             patch.object(yd.time, \'sleep\'):\n            return yd.download(cache_dir=self.root, attempts=attempts)\n\n    def test_folder_download_and_verified_cache(self):\n        target = self.run_download(lambda *a, **kw: Response(self.blob))\n        self.assertEqual(target.read_bytes(), self.blob)\n        with patch.object(yd, \'api_json\', side_effect=self.api), \\\n             patch.object(yd.urllib.request, \'urlopen\') as opened:\n            self.assertEqual(yd.download(cache_dir=self.root), target)\n            opened.assert_not_called()\n\n    def test_interruption_resumes_with_range(self):\n        blob = self.blob\n        class Interrupted(Response):\n            def read(self, size=-1):\n                if self.tell():\n                    raise OSError(\'simulated disconnect\')\n                return super().read(40)\n        calls = []\n        def open_response(request, **kwargs):\n            calls.append(request)\n            if len(calls) == 1:\n                return Interrupted(blob)\n            self.assertEqual(request.get_header(\'Range\'), \'bytes=40-\')\n            return Response(blob[40:], 206,\n                            {\'Content-Range\': f\'bytes 40-{len(blob)-1}/{len(blob)}\'})\n        self.assertEqual(self.run_download(open_response).read_bytes(), self.blob)\n        self.assertEqual(len(calls), 2)\n\n    def test_ignored_range_restarts_instead_of_appending(self):\n        class Interrupted(Response):\n            def read(self, size=-1):\n                if self.tell():\n                    raise OSError(\'disconnect\')\n                return super().read(40)\n        responses = iter([Interrupted(self.blob), Response(self.blob)])\n        self.assertEqual(self.run_download(lambda *a, **kw: next(responses)).read_bytes(), self.blob)\n\n    def test_wrong_hash_never_becomes_completed_archive(self):\n        self.meta[\'sha256\'] = \'0\' * 64\n        with self.assertRaises(RuntimeError):\n            self.run_download(lambda *a, **kw: Response(self.blob), attempts=1)\n        self.assertFalse(list(self.root.rglob(\'dataset.zip\')))\n\n    def test_wrong_range_never_becomes_completed_archive(self):\n        with self.assertRaises(RuntimeError):\n            self.run_download(lambda *a, **kw: Response(self.blob, 206,\n                              {\'Content-Range\': f\'bytes 5-{len(self.blob)-1}/{len(self.blob)}\'}),\n                              attempts=1)\n        self.assertFalse(list(self.root.rglob(\'dataset.zip\')))\n\n    def test_api_retries_and_reports_network_problem(self):\n        with patch.object(yd.urllib.request, \'urlopen\', side_effect=OSError(\'timeout\')) as opened, \\\n             patch.object(yd.time, \'sleep\'):\n            with self.assertRaisesRegex(RuntimeError, \'Internet\'):\n                yd.api_json(\'\', {\'public_key\': yd.PUBLIC_URL}, attempts=2)\n            self.assertEqual(opened.call_count, 2)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'ml/submit/adapter.py': '"""Preserve the exact organizer sample keys, ordering and non-target text."""\nfrom __future__ import annotations\n\nimport csv\nfrom datetime import date, datetime\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nfrom ml.forecast.core import grid, parse_key, rounded\n\nFIELDS = [\'route\', \'date\', \'hour\', \'prediction\']\nSTART, END = date(2025, 11, 1), date(2025, 12, 31)\n\n\ndef read_sample(path):\n    with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n        reader = csv.DictReader(f, delimiter=\';\')\n        if reader.fieldnames != FIELDS:\n            raise ValueError(f\'Unexpected sample schema: {reader.fieldnames}\')\n        rows = list(reader)\n    if any(None in r or any(v is None for v in r.values()) for r in rows):\n        raise ValueError(\'Malformed sample row\')\n    keys = [parse_key(r) for r in rows]\n    routes = sorted({k[0] for k in keys}, key=int)\n    if len(keys) != 14640 or len(set(keys)) != len(keys):\n        raise ValueError(\'Sample must contain exactly 14640 unique keys\')\n    if len(routes) != 10 or \'5\' not in routes:\n        raise ValueError(\'Sample must contain ten routes, including route 5\')\n    if set(keys) != set(grid(routes, START, END)):\n        raise ValueError(\'Sample must cover every hour of November–December 2025\')\n    return rows, keys\n\n\ndef validate(sample_path, output_path):\n    rows, keys = read_sample(sample_path)\n    other, other_keys = read_sample(output_path)\n    if keys != other_keys:\n        raise ValueError(\'Submission changed sample ordering\')\n    for expected, actual in zip(rows, other):\n        if any(expected[k] != actual[k] for k in FIELDS[:-1]):\n            raise ValueError(\'Non-target sample text changed\')\n        value = actual[\'prediction\']\n        if not value.isascii() or not value.isdecimal() or str(int(value)) != value:\n            raise ValueError(\'Submission predictions must be canonical nonnegative integers\')\n        rounded(int(value))\n    return {\'rows\': len(keys), \'routes\': len({k[0] for k in keys}),\n            \'route5_rows\': sum(k[0] == \'5\' for k in keys), \'prediction_sum\':\n            sum(int(row[\'prediction\']) for row in other)}\n\n\ndef write_submission(sample_path, predictions, output_path):\n    if Path(sample_path).resolve() == Path(output_path).resolve():\n        raise ValueError(\'Never overwrite the original sample\')\n    rows, keys = read_sample(sample_path)\n    if set(predictions) != set(keys):\n        raise ValueError(\'Prediction keys must match sample exactly; no missing/extra rows\')\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    values = [rounded(predictions[k]) for k in keys]\n    with output_path.open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        writer = csv.DictWriter(f, fieldnames=FIELDS, delimiter=\';\', lineterminator=\'\\n\')\n        writer.writeheader()\n        for row, value in zip(rows, values):\n            writer.writerow({**row, \'prediction\': value})\n    return validate(sample_path, output_path)\n\n\ndef export_api(keys, predictions, output_path, model_version):\n    if len(set(keys)) != len(keys) or set(keys) != set(predictions):\n        raise ValueError(\'API export key mismatch\')\n    with Path(output_path).open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        writer = csv.writer(f, lineterminator=\'\\n\')\n        writer.writerow([\'route_id\', \'timestamp_msk\', \'pred_validations\',\n                         \'model_version\', \'source_mode\'])\n        for r, d, h in keys:\n            stamp = datetime(d.year, d.month, d.day, h, tzinfo=ZoneInfo(\'Europe/Moscow\'))\n            writer.writerow([r, stamp.isoformat(), rounded(predictions[r, d, h]),\n                             model_version, \'as_of_2025_10_31\'])\n\n\nif __name__ == \'__main__\':\n    import argparse\n    import json\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--sample\', required=True)\n    parser.add_argument(\'--submission\', required=True)\n    args = parser.parse_args()\n    print(json.dumps(validate(args.sample, args.submission), ensure_ascii=False, indent=2))\n', 'ml/forecast/weather_competition.py': '"""Paired weather ablation. Competition mode uses allowed retrospective weather.\n\nNever consumes future passenger counts. The frozen reference was selected in\nprior Sep-Oct research, so those scores are exploratory, not independent tests.\n"""\nfrom __future__ import annotations\nimport argparse\nimport calendar\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport pickle\nfrom datetime import date,datetime,timedelta,timezone\nfrom pathlib import Path\nimport numpy as np\nfrom .core import grid,load_labels,rounded,score,sha256,effective_weekday,calendar_day_type\nfrom .route_specific import verify_inputs,candidate_factories,RouteEnsemble\nfrom .run import FIRST,FINAL_CUTOFF,FOLDS,dump\nfrom ml.submit.adapter import read_sample,write_submission\n\nVARIABLES=(\'temperature_2m\',\'relative_humidity_2m\',\'precipitation\',\'rain\',\n           \'snowfall\',\'wind_speed_10m\',\'cloud_cover\')\nWEIGHTS=(.25,.5,1.)\n\n\ndef weather_rows(payload):\n    if payload.get(\'timezone\')!=\'Europe/Moscow\' or payload.get(\'utc_offset_seconds\')!=10800:\n        raise ValueError(\'Weather must use Europe/Moscow (+03:00)\')\n    h=payload[\'hourly\'];times=h[\'time\']\n    if len(set(times))!=len(times):raise ValueError(\'Duplicate weather timestamp\')\n    if any(len(h[v])!=len(times) for v in VARIABLES):raise ValueError(\'Weather lengths differ\')\n    result={};previous=None\n    for i,stamp in enumerate(times):\n        t=datetime.fromisoformat(stamp)\n        if t.minute or t.second or (previous and t-previous!=timedelta(hours=1)):\n            raise ValueError(\'Weather hours are not contiguous\')\n        previous=t\n        raw=[h[v][i] for v in VARIABLES]\n        if any(v is None or not math.isfinite(float(v)) for v in raw):\n            raise ValueError(f\'Missing/nonfinite weather at {stamp}; do not silently impute\')\n        raw=list(map(float,raw))\n        # Only preceding/current weather hours in rolling summaries.\n        j=max(0,i-23)\n        raw.extend([float(np.mean(h[\'temperature_2m\'][j:i+1])),\n                    float(np.sum(h[\'precipitation\'][j:i+1])),float(raw[0]<0)])\n        result[t.date(),t.hour]=raw\n    return result\n\n\ndef fit_reference(history,cutoff,choices):\n    factories=candidate_factories(profiles_only=True)\n    names={\'school_calendar\'}|{c[\'candidate\'] for c in choices.values()}\n    models={n:factories[n]().fit(history,FIRST,cutoff) for n in names}\n    return RouteEnsemble(models,choices,cutoff)\n\n\ndef matrix(keys,base,weather,use_weather):\n    rows=[]\n    for k in keys:\n        r,d,h=k\n        row=[int(r),h,d.weekday(),effective_weekday(d),calendar_day_type(d),\n             int(d.month in (7,8)),base[k],math.sin(2*math.pi*h/24),math.cos(2*math.pi*h/24)]\n        if use_weather:\n            if (d,h) not in weather:raise ValueError(f\'Weather missing for {d} {h}\')\n            row.extend(weather[d,h])\n        rows.append(row)\n    return np.asarray(rows,dtype=float)\n\n\ndef causal_residual_training(history,cutoff,choices):\n    """Monthly out-of-time reference predictions; no same-row target encoding."""\n    routes=sorted({k[0] for k in history},key=int)\n    keys=[];base={};blocks=[]\n    lo=date(2025,2,1)\n    while lo<=cutoff:\n        hi=min(cutoff,date(lo.year,lo.month,calendar.monthrange(lo.year,lo.month)[1]))\n        training_cutoff=lo-timedelta(days=1)\n        prefix={k:y for k,y in history.items() if k[1]<=training_cutoff}\n        model=fit_reference(prefix,training_cutoff,choices)\n        batch=grid(routes,lo,hi)\n        base.update(model.predict(batch));keys.extend(batch)\n        blocks.append({\'train_through\':str(training_cutoff),\'predict_from\':str(lo),\'predict_to\':str(hi)})\n        lo=hi+timedelta(days=1)\n    if not keys:raise ValueError(\'At least two months of history required\')\n    scale=np.asarray([max(50,base[k]) for k in keys],dtype=float)\n    target=np.asarray([(history.get(k,0)-base[k])/s for k,s in zip(keys,scale)],dtype=float)\n    return keys,base,scale,target,blocks\n\n\ndef fit_pair(history,cutoff,choices,weather):\n    import lightgbm as lgb\n    keys,base,scale,y,blocks=causal_residual_training(history,cutoff,choices)\n    models={}\n    for name,use_weather in [(\'calendar\',False),(\'weather\',True)]:\n        reg=lgb.LGBMRegressor(objective=\'regression_l1\',n_estimators=220,\n            learning_rate=.025,num_leaves=12,min_child_samples=120,\n            reg_lambda=10.,max_depth=5,n_jobs=2,verbosity=-1,random_state=2025)\n        reg.fit(matrix(keys,base,weather,use_weather),y,sample_weight=scale,\n                categorical_feature=[0,1,2,3,4,5])\n        models[name]=reg\n    return models,blocks\n\n\ndef corrected(keys,base,raw,weight):\n    """Bound changes to 35% of nonzero reference; do not invent service at zero."""\n    out={}\n    for k,v in zip(keys,raw):\n        b=base[k]\n        delta=float(np.clip(float(v)*max(50,b),-.35*b,.35*b))\n        out[k]=rounded(b+weight*delta)\n    return out\n\n\ndef predictions_pair(keys,base,models,weather):\n    preds={\'reference\':base}\n    for name,use_weather in [(\'calendar\',False),(\'weather\',True)]:\n        raw=models[name].predict(matrix(keys,base,weather,use_weather))\n        for weight in WEIGHTS:preds[f\'{name}_{weight}\']=corrected(keys,base,raw,weight)\n    return preds\n\n\ndef select(keys,truth,preds,routes):\n    """Require >=0.5% gain in every validation month, with matched weather control."""\n    months=sorted({k[1].month for k in keys})\n    chosen={};control_chosen={};diagnostics={}\n    for r in routes:\n        groups=[[k for k in keys if k[0]==r and k[1].month==m] for m in months]\n        errors={name:[sum(abs(p[k]-truth.get(k,0)) for k in g) for g in groups]\n                for name,p in preds.items()}\n        wins=lambda name,ref: all(a<=.995*b for a,b in zip(errors[name],errors[ref])) and sum(errors[name])<sum(errors[ref])\n        controls=[\'reference\']+[f\'calendar_{w}\' for w in WEIGHTS if wins(f\'calendar_{w}\',\'reference\')]\n        control=min(controls,key=lambda n:(sum(errors[n]),n));control_chosen[r]=control\n        candidates=list(controls)\n        for w in WEIGHTS:\n            name=f\'weather_{w}\'\n            if wins(name,\'reference\') and wins(name,f\'calendar_{w}\') and wins(name,control):\n                candidates.append(name)\n        chosen[r]=min(candidates,key=lambda n:(sum(errors[n]),n))\n        diagnostics[r]={\'errors_by_month\':errors,\'months\':months,\'calendar_control\':control,\'selected\':chosen[r]}\n    return chosen,control_chosen,diagnostics\n\n\ndef join(keys,preds,choice):\n    return {k:preds[choice.get(k[0],\'reference\')][k] if k[0] in choice else 0 for k in keys}\n\n\nclass WeatherEnsemble:\n    def __init__(self,reference,models,weather,choices):\n        self.reference,self.models,self.weather,self.choices=reference,models,weather,choices\n    def predict(self,keys):\n        base=self.reference.predict(keys)\n        return join(keys,predictions_pair(keys,base,self.models,self.weather),self.choices)\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    for n in [\'train\',\'test\',\'sample\',\'weather\',\'reference\',\'out\']:p.add_argument(\'--\'+n,type=Path,required=True)\n    args=p.parse_args(argv)\n    inputs=verify_inputs({n:getattr(args,n) for n in [\'train\',\'test\',\'sample\']})\n    train,test=load_labels([args.train]),load_labels([args.test])\n    if any(not FIRST<=k[1]<=date(2025,8,31) for k in train):raise ValueError(\'Bad train dates\')\n    if any(not date(2025,9,1)<=k[1]<=FINAL_CUTOFF for k in test):raise ValueError(\'Bad test dates\')\n    values=load_labels([args.train,args.test]);_,sample=read_sample(args.sample)\n    ref=json.loads(args.reference.read_text());choices=ref[\'choices\']\n    routes=sorted({k[0] for k in values},key=int)\n    if set(routes)!=set(choices):raise ValueError(\'Reference route set differs from real labels\')\n    weather=weather_rows(json.loads(args.weather.read_text()))\n    for _,d,h in grid([\'weather\'],FIRST,date(2025,12,31)):\n        if (d,h) not in weather:raise ValueError(\'Weather must cover the entire 2025 calendar year\')\n    args.out.mkdir(parents=True,exist_ok=True)\n    report={\'source_mode\':\'competition\',\'external_information\':\'Retrospective ERA5 weather, allowed by organizer Q&A; not an as-of Oct31 weather forecast.\',\n            \'weather_source\':\'https://open-meteo.com/en/docs/historical-weather-api\',\n            \'weather_source_metadata\':json.loads(Path(__file__).with_name(\'weather_source.json\').read_text()),\n            \'weather_sha256\':sha256(args.weather),\'reference_sha256\':sha256(args.reference),\n            \'reference\':ref,\'inputs\':inputs,\'data_kind\':\'verified_organizer_labels\',\n            \'selection_warning\':ref[\'selection_warning\'],\'folds\':{},\'platform_score\':None,\n            \'weather_features\':list(VARIABLES)+[\'temperature_24h_mean\',\'precipitation_24h_sum\',\'below_freezing\'],\n            \'excluded_variable\':\'snow_depth is all null in downloaded ERA5 response; not used\',\n            \'correction_limit\':.35,\'weights\':list(WEIGHTS),\n            \'fitted_at\':datetime.now(timezone.utc).isoformat(),\n            \'libraries\':{n:importlib.metadata.version(n) for n in [\'numpy\',\'lightgbm\',\'scikit-learn\']},\n            \'code_sha256\':{p.name:sha256(p) for p in sorted(Path(__file__).parent.glob(\'*.py\'))}}\n    for label,cutoff,lo,hi in FOLDS:\n        history={k:y for k,y in values.items() if k[1]<=cutoff}\n        keys=grid(routes,lo,hi);truth={k:y for k,y in values.items() if lo<=k[1]<=hi}\n        base=fit_reference(history,cutoff,choices).predict(keys)\n        models,blocks=fit_pair(history,cutoff,choices,weather)\n        preds=predictions_pair(keys,base,models,weather)\n        report[\'folds\'][label]={\'training_blocks\':blocks,\'variants\':{}}\n        for name,pred in preds.items():\n            m=score(keys,truth,pred);report[\'folds\'][label][\'variants\'][name]=m\n            print(f\'{label} {name}: WAPE-score={m["all"]["wape_score"]:.6f}\',flush=True)\n        if label==\'sep_oct\':\n            sep=[k for k in keys if k[1].month==9];october=[k for k in keys if k[1].month==10]\n            sept,_,_=select(sep,truth,preds,routes)\n            report[\'october_forward_diagnostic\']={\'choices_from_september\':sept,\n                \'metrics\':score(october,{k:truth[k] for k in october if k in truth},join(october,preds,sept))}\n            final_choice,calendar_choice,diag=select(keys,truth,preds,routes)\n            report[\'selection\']={\'by_route\':final_choice,\'calendar_control_by_route\':calendar_choice,\n                \'diagnostics\':diag,\'weather_accepted_routes\':[r for r,n in final_choice.items() if n.startswith(\'weather_\')]}\n            for name,ch in [(\'selected\',final_choice),(\'selected_calendar_control\',calendar_choice)]:\n                report[\'folds\'][label][\'variants\'][name]=score(keys,truth,join(keys,preds,ch))\n        dump(args.out/\'weather_report.json\',report)\n    reference=fit_reference(values,FINAL_CUTOFF,choices)\n    models,_=fit_pair(values,FINAL_CUTOFF,choices,weather)\n    model=WeatherEnsemble(reference,models,weather,final_choice)\n    base=reference.predict(sample);all_final=predictions_pair(sample,base,models,weather)\n    for name,ch in [(\'submission_competition.csv\',final_choice),(\'submission_calendar_control.csv\',calendar_choice)]:\n        report[name]=write_submission(args.sample,join(sample,all_final,ch),args.out/name)\n    # This must reproduce the protected 0.877 baseline byte-for-byte.\n    write_submission(args.sample,base,args.out/\'submission_reference.csv\')\n    if sha256(args.out/\'submission_reference.csv\')!=ref[\'submission_sha256\']:\n        raise RuntimeError(\'Frozen reference does not reproduce original CSV; stop and investigate\')\n    (args.out/\'model.local.pkl\').write_bytes(pickle.dumps(model,protocol=5))\n    report[\'outputs_sha256\']={n:sha256(args.out/n) for n in [\'submission_competition.csv\',\'submission_calendar_control.csv\',\'submission_reference.csv\',\'model.local.pkl\']}\n    final_predictions=join(sample,all_final,final_choice)\n    report[\'changed_rows_vs_reference\']=sum(final_predictions[k]!=base[k] for k in sample)\n    dump(args.out/\'weather_report.json\',report)\n    print(\'Weather accepted on routes:\',report[\'selection\'][\'weather_accepted_routes\'],flush=True)\n    print(\'Reference reproduced; 14640 rows exported. Competition score unknown.\',flush=True)\n    return report\n\nif __name__==\'__main__\':\n    from ml.forecast.weather_competition import main as run_main\n    run_main()\n', 'ml/forecast/reference_route_model.json': '{\n  "choices": {\n    "1": {\n      "candidate": "school_local",\n      "weight": 1.0,\n      "month_errors": [\n        41280,\n        55185\n      ]\n    },\n    "7": {\n      "candidate": "school_recent50",\n      "weight": 1.0,\n      "month_errors": [\n        83375,\n        78862\n      ]\n    },\n    "11": {\n      "candidate": "school_calendar",\n      "weight": 1.0,\n      "month_errors": [\n        90010,\n        92860\n      ]\n    },\n    "12": {\n      "candidate": "school_local",\n      "weight": 0.5,\n      "month_errors": [\n        87764,\n        84047\n      ]\n    },\n    "17": {\n      "candidate": "school_local",\n      "weight": 0.5,\n      "month_errors": [\n        86781,\n        120431\n      ]\n    },\n    "25": {\n      "candidate": "school_calendar",\n      "weight": 1.0,\n      "month_errors": [\n        29571,\n        49803\n      ]\n    },\n    "26": {\n      "candidate": "school_calendar",\n      "weight": 1.0,\n      "month_errors": [\n        43025,\n        59428\n      ]\n    },\n    "28": {\n      "candidate": "school_local",\n      "weight": 0.5,\n      "month_errors": [\n        42733,\n        39846\n      ]\n    },\n    "50": {\n      "candidate": "school_local",\n      "weight": 1.0,\n      "month_errors": [\n        133254,\n        136981\n      ]\n    }\n  },\n  "model_version": "ds1-route-63b89667a69c",\n  "validation_score": 0.8937443240536405,\n  "submission_sha256": "6efbbe7dc7dee64245b99573a7e92a1452a3816eb6e7cc4e2fc56e482663290f",\n  "selection_warning": "Choices were selected on Sep-Oct in prior research; all subsequent comparisons on that period remain exploratory."\n}\n', 'ml/forecast/test_weather_competition.py': '"""Small in-memory software tests; no persistent generated datasets."""\nimport unittest\nfrom datetime import date,timedelta\nfrom unittest.mock import patch\nimport numpy as np\nfrom ml.forecast.weather_competition import weather_rows,VARIABLES,corrected,select,join,causal_residual_training\n\nclass WeatherTests(unittest.TestCase):\n    def payload(self):\n        return {\'timezone\':\'Europe/Moscow\',\'utc_offset_seconds\':10800,\'hourly\':{\n            \'time\':[\'2025-01-01T00:00\',\'2025-01-01T01:00\'],\n            **{v:[1.,2.] for v in VARIABLES}}}\n\n    def test_missing_weather_is_not_silently_imputed(self):\n        p=self.payload();p[\'hourly\'][\'temperature_2m\'][1]=None\n        with self.assertRaisesRegex(ValueError,\'Missing\'):weather_rows(p)\n\n    def test_weather_time_alignment_and_rolling_values(self):\n        p=self.payload();r=weather_rows(p)\n        self.assertEqual(r[date(2025,1,1),1][-3:],[1.5,3.,0.])\n        p[\'hourly\'][\'time\'][1]=\'2025-01-01T02:00\'\n        with self.assertRaisesRegex(ValueError,\'contiguous\'):weather_rows(p)\n\n    def test_weather_must_beat_matched_control_in_both_months(self):\n        keys=[(\'1\',date(2025,m,1),12) for m in (9,10)]\n        truth=dict.fromkeys(keys,100)\n        preds={\'reference\':dict.fromkeys(keys,80)}\n        for w in (.25,.5,1.):\n            preds[f\'calendar_{w}\']=dict.fromkeys(keys,95)\n            preds[f\'weather_{w}\']={keys[0]:100,keys[1]:90}\n        chosen,control,_=select(keys,truth,preds,[\'1\'])\n        self.assertTrue(chosen[\'1\'].startswith(\'calendar\'))\n        preds[\'weather_0.5\']=dict.fromkeys(keys,99)\n        chosen,_,_=select(keys,truth,preds,[\'1\'])\n        self.assertEqual(chosen[\'1\'],\'weather_0.5\')\n\n    def test_corrections_are_bounded_and_route5_stays_zero(self):\n        keys=[(\'1\',date(2025,11,1),12),(\'5\',date(2025,11,1),12)]\n        base={keys[0]:100,keys[1]:0}\n        self.assertEqual(corrected(keys,base,[99,99],1),{keys[0]:135,keys[1]:0})\n        self.assertEqual(corrected(keys,base,[-99,-99],1),{keys[0]:65,keys[1]:0})\n\n    def test_residual_training_has_no_current_block_labels(self):\n        start=date(2025,1,1);end=date(2025,3,31)\n        history={(\'1\',start+timedelta(days=i),12):i+1 for i in range((end-start).days+1)}\n        seen=[]\n        def factory(prefix,cutoff,choices):\n            self.assertLessEqual(max(k[1] for k in prefix),cutoff)\n            class Reference:\n                def predict(_,keys):\n                    self.assertGreater(min(k[1] for k in keys),cutoff)\n                    seen.append((cutoff,min(k[1] for k in keys)))\n                    return dict.fromkeys(keys,10)\n            return Reference()\n        with patch(\'ml.forecast.weather_competition.fit_reference\',side_effect=factory):\n            keys,base,scale,target,blocks=causal_residual_training(history,end,{})\n        self.assertEqual(len(seen),2)\n        self.assertEqual(blocks[0][\'train_through\'],\'2025-01-31\')\n        self.assertEqual(keys[0][1],date(2025,2,1))\n\nif __name__==\'__main__\':unittest.main()\n', 'ml/forecast/weather_source.json': '{\n  "url": "https://archive-api.open-meteo.com/v1/archive?latitude=55.75&longitude=37.62&start_date=2025-01-01&end_date=2025-12-31&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation%2Crain%2Csnowfall%2Csnow_depth%2Cwind_speed_10m%2Ccloud_cover&timezone=Europe%2FMoscow&models=era5",\n  "source_mode": "competition",\n  "description": "Reanalysis, not available as a full future year at forecast cutoff",\n  "retrieved_at": "2026-09-26T19:25:02Z",\n  "attribution": "Weather data by Open-Meteo; ERA5 reanalysis by ECMWF/Copernicus."\n}\n', 'ml/forecast/network_regimes.py': '"""Route service states, sourced externally, with target-only causal profiles.\n\nCompetition mode deliberately allows retrospective transport notices. Dates in\nthese notices describe service, not a claim that recorded validations are zero.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom collections import defaultdict\nfrom datetime import date\nfrom pathlib import Path\nfrom statistics import median\n\nfrom .core import days, effective_weekday, rounded\n\nNORMAL, JULY_SHORT, WEST_AUG, WEST_SEP, BOTH_ENDS, CANCELLED = range(6)\nSTATE_NAMES = (\'normal\', \'sokolniki_july\', \'novoslobodskaya_august\',\n               \'kalanchyovskaya\', \'kalanchyovskaya_and_bogorodskoe\', \'cancelled\')\nEVENTS = json.loads(Path(__file__).with_name(\'network_sources.json\').read_text())\n\n\ndef state(route, day, *, mode=\'competition\', as_of=None):\n    """A publication cutoff applies only to as-of mode; ends have own notices.\n\n    Repair weekends are literal Saturday/Sunday, independent of the demand\n    calendar. Nov 1/3/4 are uncertain and explicitly listed in source metadata.\n    """\n    if mode not in (\'competition\', \'as_of\') or (mode == \'as_of\' and as_of is None):\n        raise ValueError(\'Choose competition or as_of with a publication cutoff\')\n    route = str(route)\n    known = lambda stamp: mode == \'competition\' or date.fromisoformat(stamp) <= as_of\n    if route not in (\'7\', \'50\'):\n        return NORMAL\n    if route == \'7\' and known(\'2025-07-02\') and day >= date(2025, 7, 10):\n        if not (known(\'2025-08-13\') and day >= date(2025, 8, 13)):\n            return JULY_SHORT\n    if day.weekday() < 5:\n        return NORMAL\n    if known(\'2025-11-15\') and day >= date(2025, 11, 15):\n        return NORMAL\n    if known(\'2025-09-05\') and day >= date(2025, 9, 6):\n        if route == \'50\':\n            return CANCELLED\n        east = known(\'2025-09-19\') and day >= date(2025, 9, 20)\n        restored = known(\'2025-10-04\') and day >= date(2025, 10, 4)\n        return BOTH_ENDS if east and not restored else WEST_SEP\n    if route == \'7\' and known(\'2025-08-15\') and day >= date(2025, 8, 16):\n        return WEST_AUG\n    return NORMAL\n\n\nclass RegimeProfile:\n    """25% same-service hourly median + 75% frozen route reference.\n\n    Distinct shortened routes never borrow a July reduction coefficient.\n    Unseen or sparse states leave the reference untouched. Only announced\n    route-50 cancellations use a zero prior, or a learned residual-service\n    median once three same-weekday or four pooled weekend days are available.\n    Positive labels survive.\n    """\n    def __init__(self, weight=.25, mode=\'competition\', as_of=None):\n        if not 0 <= weight <= .5:\n            raise ValueError(\'Regime pooling weight must be in [0, .5]\')\n        self.weight, self.mode, self.as_of = weight, mode, as_of\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'History is empty or crosses cutoff\')\n        self.cutoff = cutoff\n        self.routes = {k[0] for k in history}\n        groups = defaultdict(list)\n        for r in (\'7\', \'50\'):\n            if r not in self.routes:\n                continue\n            for d in days(start, cutoff):\n                s = state(r, d, mode=self.mode, as_of=self.as_of)\n                # Weekdays remain the established route model. Holiday demand\n                # is not assumed to imply a repair closure.\n                if d.weekday() < 5:\n                    continue\n                for h in range(24):\n                    groups[r, s, d.weekday(), h].append(history.get((r, d, h), 0))\n                    groups[r, s, -1, h].append(history.get((r, d, h), 0))\n        self.support = {k: len(v) for k, v in groups.items()}\n        self.profiles = {k: float(median(v)) for k, v in groups.items()\n                         if len(v) >= (4 if k[2] == -1 else 3)}\n        return self\n\n    def lookup(self, route, service, weekday, hour):\n        return self.profiles.get((route, service, weekday, hour),\n                                 self.profiles.get((route, service, -1, hour)))\n\n    def apply(self, keys, base, *, active=(\'7\', \'50\'), zero_only=False):\n        if any(d <= self.cutoff for _, d, _ in keys):\n            raise ValueError(\'Predictions must follow training cutoff\')\n        out = {}\n        for k in keys:\n            r, d, h = k\n            b = base[k]\n            if r not in self.routes:\n                out[k] = 0\n                continue\n            s = state(r, d, mode=self.mode, as_of=self.as_of)\n            local = self.lookup(r, s, d.weekday(), h)\n            if r in active and s == CANCELLED:\n                # This is a prediction, never an overwrite of known truth.\n                out[k] = 0 if zero_only or local is None else rounded(local)\n            elif r in active and d.weekday() >= 5 and effective_weekday(d) == d.weekday() and local is not None:\n                # <=50% change in the profile before 25% pooling; protects\n                # restoration from the rejected DS2 history-exclusion rule.\n                local = min(1.5 * b, max(.5 * b, local))\n                out[k] = rounded((1 - self.weight) * b + self.weight * local)\n            else:\n                out[k] = b\n        return out\n\n    def coverage(self, keys):\n        counts = defaultdict(lambda: {\'rows\': 0, \'supported_rows\': 0})\n        for r, d, h in keys:\n            if r in (\'7\', \'50\'):\n                s = state(r, d, mode=self.mode, as_of=self.as_of)\n                item = counts[f\'{r}:{STATE_NAMES[s]}\']\n                item[\'rows\'] += 1\n                item[\'supported_rows\'] += self.lookup(r, s, d.weekday(), h) is not None\n        return dict(counts)\n', 'ml/forecast/network_competition.py': '"""Fixed-horizon competition comparison: service regimes x actual weather.\n\nOrganizer labels are verified by SHA256. No generated dataset can pass the\nentry point. New scores are produced only by running on those real inputs.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport calendar\nimport importlib.metadata\nimport json\nimport pickle\nfrom collections import defaultdict\nfrom datetime import date, datetime, timedelta, timezone\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .core import grid, load_labels, score, sha256, peak_hours\nfrom .network_regimes import RegimeProfile, state, STATE_NAMES, EVENTS\nfrom .route_specific import verify_inputs\nfrom .run import FIRST, FINAL_CUTOFF, FOLDS, dump\nfrom .weather_competition import (fit_reference, matrix, weather_rows,\n                                 corrected, WEIGHTS, select as select_weather)\nfrom ml.submit.adapter import read_sample, write_submission\n\nVERIFIED_REFERENCE_CSV_SHA256 = \'6efbbe7dc7dee64245b99573a7e92a1452a3816eb6e7cc4e2fc56e482663290f\'\n\n\ndef verify_competition_inputs(paths, sample_from_reference=False):\n    if not sample_from_reference:\n        return verify_inputs(paths)\n    inputs = verify_inputs({n: paths[n] for n in (\'train\', \'test\')})\n    digest = sha256(paths[\'sample\'])\n    if digest != VERIFIED_REFERENCE_CSV_SHA256:\n        raise ValueError(\'Reference template must be the exact protected 0.877 submission\')\n    # The adapter also validates the full original key grid and schema. The\n    # previous prediction column is ignored by all training and export logic.\n    read_sample(paths[\'sample\'])\n    inputs[\'sample\'] = {\'name\': paths[\'sample\'].name, \'sha256\': digest,\n        \'role\': \'protected_previous_submission_as_key_template\',\n        \'note\': \'Original organizer sample not supplied locally; keys/order come from verified previous export, old predictions ignored.\'}\n    return inputs\n\n\ndef features(keys, base, weather, use_weather, network):\n    x = matrix(keys, base, weather, use_weather)\n    if network:\n        # Same service information is present in both matched weather arms.\n        x = np.column_stack((x, [state(r, d) for r, d, _ in keys]))\n    return x\n\n\nclass HistoryBank:\n    """Cache predictions for monthly blocks, fitted strictly before each block."""\n    def __init__(self, values, choices):\n        self.values, self.choices = values, choices\n        self.routes = sorted({k[0] for k in values}, key=int)\n        self.fits, self.blocks = {}, {}\n\n    def fitted(self, cutoff):\n        if cutoff not in self.fits:\n            history = {k: v for k, v in self.values.items() if FIRST <= k[1] <= cutoff}\n            self.fits[cutoff] = (fit_reference(history, cutoff, self.choices),\n                                RegimeProfile().fit(history, FIRST, cutoff))\n        return self.fits[cutoff]\n\n    def predict_bases(self, cutoff, keys):\n        ref, net = self.fitted(cutoff)\n        plain = ref.predict(keys)\n        return {\'plain\': plain, \'network\': net.apply(keys, plain)}\n\n    def training(self, cutoff):\n        keys, base = [], {\'plain\': {}, \'network\': {}}\n        audit = []\n        lo = date(2025, 2, 1)\n        while lo <= cutoff:\n            hi = min(cutoff, date(lo.year, lo.month, calendar.monthrange(lo.year, lo.month)[1]))\n            origin = lo - timedelta(days=1)\n            cache_key = (origin, hi)\n            if cache_key not in self.blocks:\n                batch = grid(self.routes, lo, hi)\n                self.blocks[cache_key] = (batch, self.predict_bases(origin, batch))\n            batch, predictions = self.blocks[cache_key]\n            keys.extend(batch)\n            for name in base:\n                base[name].update(predictions[name])\n            audit.append({\'train_through\': str(origin), \'predict_from\': str(lo), \'predict_to\': str(hi)})\n            lo = hi + timedelta(days=1)\n        if not keys:\n            raise ValueError(\'Need at least two months of history\')\n        # In this function the bank may contain future evaluation labels; only\n        # targets ending at cutoff are read. Each feature profile is earlier.\n        y = np.asarray([self.values.get(k, 0) for k in keys], dtype=float)\n        return keys, base, y, audit\n\n\ndef fit_models(bank, cutoff, weather):\n    import lightgbm as lgb\n    keys, bases, truth, audit = bank.training(cutoff)\n    models = {}\n    for group in (\'plain\', \'network\'):\n        base = bases[group]\n        scale = np.asarray([max(50, base[k]) for k in keys], dtype=float)\n        target = (truth - np.asarray([base[k] for k in keys])) / scale\n        models[group] = {}\n        for head, use_weather in ((\'calendar\', False), (\'weather\', True)):\n            x = features(keys, base, weather, use_weather, group == \'network\')\n            categorical = [0, 1, 2, 3, 4, 5] + ([x.shape[1]-1] if group == \'network\' else [])\n            reg = lgb.LGBMRegressor(objective=\'regression_l1\', n_estimators=220,\n                learning_rate=.025, num_leaves=12, min_child_samples=120,\n                reg_lambda=10., max_depth=5, n_jobs=2, verbosity=-1, random_state=2025)\n            reg.fit(x, target, sample_weight=scale, categorical_feature=categorical)\n            models[group][head] = reg\n    return models, audit\n\n\ndef predict_models(keys, bases, models, weather):\n    out = {}\n    for group, base in bases.items():\n        out[group] = {\'reference\': base}\n        for head, use_weather in ((\'calendar\', False), (\'weather\', True)):\n            raw = models[group][head].predict(features(keys, base, weather, use_weather, group == \'network\'))\n            for weight in WEIGHTS:\n                out[group][f\'{head}_{weight}\'] = corrected(keys, base, raw, weight)\n    return out\n\n\ndef metrics(keys, truth, predictions):\n    return score(keys, {k: truth[k] for k in keys if k in truth},\n                 {k: predictions[k] for k in keys})\n\n\ndef route_errors(keys, truth, prediction, route):\n    return sum(abs(prediction[k] - truth.get(k, 0)) for k in keys if k[0] == route)\n\n\ndef select_groups(keys, truth, preds, october_keys, october_truth, october_preds, routes):\n    """Finite, preregistered gate; no tuning of a state-specific effect size.\n\n    Route 50: >=0.5% gain in both long validation months. Route 7: the new\n    September endpoint is first learnable in the October refit, requiring\n    >=0.5% gain there and <=0.5% deterioration in each long validation month.\n    Both must also pass the October refit non-deterioration check.\n    """\n    chosen, audit = {}, {}\n    for r in routes:\n        monthly = [[k for k in keys if k[1].month == m] for m in (9, 10)]\n        errors = {g: [route_errors(x, truth, preds[g][\'reference\'], r) for x in monthly]\n                  for g in (\'plain\', \'network\')}\n        oct_errors = {g: route_errors(october_keys, october_truth, october_preds[g][\'reference\'], r)\n                      for g in (\'plain\', \'network\')}\n        wins = all(a <= .995*b for a, b in zip(errors[\'network\'], errors[\'plain\'])) and sum(errors[\'network\']) < sum(errors[\'plain\'])\n        learns_new_state = (r == \'7\' and oct_errors[\'network\'] < .995*oct_errors[\'plain\']\n                            and all(a <= 1.005*b for a, b in zip(errors[\'network\'], errors[\'plain\'])))\n        accepted = (r in (\'7\', \'50\') and (wins or learns_new_state)\n                    and oct_errors[\'network\'] <= 1.005*oct_errors[\'plain\'])\n        chosen[r] = \'network\' if accepted else \'plain\'\n        audit[r] = {\'long_month_errors\': errors, \'october_refit_errors\': oct_errors,\n                    \'passes_long_month_gate\': wins, \'passes_new_state_gate\': learns_new_state,\n                    \'selected\': chosen[r]}\n    return chosen, audit\n\n\ndef choose_heads(keys, truth, preds, groups, routes):\n    chosen, controls, audit = {}, {}, {}\n    for group in (\'plain\', \'network\'):\n        heads, no_weather, details = select_weather(keys, truth, preds[group], routes)\n        for r in routes:\n            if groups[r] == group:\n                chosen[r] = {\'group\': group, \'head\': heads[r]}\n                controls[r] = {\'group\': group, \'head\': no_weather[r]}\n                audit[r] = details[r]\n    return chosen, controls, audit\n\n\ndef assemble(keys, preds, choices):\n    return {k: preds[choices[k[0]][\'group\']][choices[k[0]][\'head\']][k]\n            if k[0] in choices else 0 for k in keys}\n\n\ndef slice_diagnostics(keys, truth, predictions, origin, history):\n    buckets = defaultdict(list)\n    peaks = peak_hours(history, sorted({k[0] for k in history}, key=int))\n    for k in keys:\n        r, d, h = k\n        lead = (d-origin).days\n        buckets[\'lead:\' + (\'01_14\' if lead <= 14 else \'15_30\' if lead <= 30 else \'31_61\')].append(k)\n        buckets[\'weekend\' if d.weekday() >= 5 else \'weekday\'].append(k)\n        if h in peaks.get(r, ()):\n            buckets[\'historical_peak_hours\'].append(k)\n        if r in (\'7\', \'50\'):\n            buckets[f\'route:{r}:state:{STATE_NAMES[state(r, d)]}\'].append(k)\n    out = {}\n    for name, subset in sorted(buckets.items()):\n        actual = sum(truth.get(k, 0) for k in subset)\n        error = sum(abs(predictions[k] - truth.get(k, 0)) for k in subset)\n        out[name] = {\'rows\': len(subset), \'actual_sum\': actual, \'absolute_error_sum\': error,\n                     \'wape_score\': max(0, 1-error/actual) if actual else None}\n    return out\n\n\nclass NetworkWeatherEnsemble:\n    def __init__(self, reference, regimes, models, weather, choices):\n        self.reference, self.regimes = reference, regimes\n        self.models, self.weather, self.choices = models, weather, choices\n\n    def predict(self, keys):\n        plain = self.reference.predict(keys)\n        bases = {\'plain\': plain, \'network\': self.regimes.apply(keys, plain)}\n        return assemble(keys, predict_models(keys, bases, self.models, self.weather), self.choices)\n\n\n\nclass HolidayServiceEnsemble:\n    """Reproduce the user-reported best holiday-service submission."""\n    def __init__(self, baseline, reference, regimes):\n        self.baseline, self.reference, self.regimes = baseline, reference, regimes\n\n    def predict(self, keys):\n        keys = list(keys)\n        result = self.baseline.predict(keys)\n        normal = [k for k in keys if k[0] in (\'7\', \'50\') and k[1] == date(2025, 11, 1)]\n        result.update(self.reference.predict(normal))\n        replacements = {k: (k[0], date(2025, 11, 2), k[2]) for k in keys\n                        if k[0] in (\'7\', \'50\') and k[1] in (date(2025, 11, 3), date(2025, 11, 4))}\n        proxies = list(dict.fromkeys(replacements.values()))\n        if proxies:\n            restricted = self.regimes.apply(proxies, self.reference.predict(proxies))\n            result.update({k: restricted[v] for k, v in replacements.items()})\n        return result\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    for name in (\'train\', \'test\', \'sample\', \'weather\', \'reference\', \'out\'):\n        p.add_argument(\'--\'+name, type=Path, required=True)\n    p.add_argument(\'--hourly-experts\', action=\'store_true\',\n                   help=\'Compare separate daily-volume/hour-shape experts for 17/11/12\')\n    p.add_argument(\'--sample-from-reference\', action=\'store_true\',\n                   help=\'Use only the exact SHA-verified prior 0.877 CSV as a key/order template\')\n    args = p.parse_args(argv)\n    if args.hourly_experts:\n        from . import hourly_experts as hourly\n    inputs = verify_competition_inputs({n: getattr(args, n) for n in (\'train\', \'test\', \'sample\')},\n                                      args.sample_from_reference)\n    values = load_labels([args.train, args.test])\n    if any(not FIRST <= k[1] <= FINAL_CUTOFF for k in values):\n        raise ValueError(\'Labels cross the allowed training interval\')\n    _, sample = read_sample(args.sample)\n    ref = json.loads(args.reference.read_text())\n    routes = sorted({k[0] for k in values}, key=int)\n    if set(routes) != set(ref[\'choices\']):\n        raise ValueError(\'Reference route set differs from labels\')\n    weather = weather_rows(json.loads(args.weather.read_text()))\n    if any((d, h) not in weather for _, d, h in grid([\'weather\'], FIRST, date(2025, 12, 31))):\n        raise ValueError(\'Weather must cover all 2025 hours\')\n    args.out.mkdir(parents=True, exist_ok=True)\n    report = {\'source_mode\': \'competition\', \'data_kind\': \'verified_organizer_labels\',\n        \'inputs\': inputs, \'reference\': ref, \'network_sources\': EVENTS,\n        \'research_audit\': json.loads(Path(__file__).with_name(\'research_audit.json\').read_text()),\n        \'weather_source\': json.loads(Path(__file__).with_name(\'weather_source.json\').read_text()),\n        \'weather_sha256\': sha256(args.weather), \'platform_score\': None,\n        \'selection_warning\': \'Exploratory: frozen route choices and new selection gates have seen Sep-Oct. October refit covers 31 days, not an independent 61-day test.\',\n        \'fitted_at\': datetime.now(timezone.utc).isoformat(),\n        \'libraries\': {n: importlib.metadata.version(n) for n in (\'numpy\', \'lightgbm\', \'scikit-learn\', \'scipy\')},\n        \'code_sha256\': {f.name: sha256(f) for f in sorted(Path(__file__).parent.glob(\'*.py\'))},\n        \'folds\': {}, \'regime_weight\': .25, \'residual_correction_limit\': .35}\n    bank = HistoryBank(values, ref[\'choices\'])\n    experiments = list(FOLDS) + [(\'october_refit_31d\', date(2025, 9, 30), date(2025, 10, 1), date(2025, 10, 31))]\n    kept, fitted_hourly = {}, {}\n    for label, cutoff, lo, hi in experiments:\n        print(f\'Fitting {label}; passenger labels through {cutoff}\', flush=True)\n        keys = grid(routes, lo, hi)\n        bases = bank.predict_bases(cutoff, keys)\n        models, training_audit = fit_models(bank, cutoff, weather)\n        preds = predict_models(keys, bases, models, weather)\n        variants = {f\'{g}/{n}\': metrics(keys, values, v) for g, heads in preds.items() for n, v in heads.items()}\n        net = bank.fitted(cutoff)[1]\n        for label_part, active in ((\'only_50\', (\'50\',)), (\'only_7\', (\'7\',))):\n            variants[label_part] = metrics(keys, values, net.apply(keys, bases[\'plain\'], active=active))\n        variants[\'50_zero_prior\'] = metrics(keys, values, net.apply(keys, bases[\'plain\'], active=(\'50\',), zero_only=True))\n        history = {k: y for k, y in values.items() if k[1] <= cutoff}\n        report[\'folds\'][label] = {\'cutoff\': str(cutoff), \'from\': str(lo), \'to\': str(hi),\n            \'days\': (hi-lo).days+1, \'training_blocks\': training_audit,\n            \'state_coverage\': net.coverage(keys), \'variants\': variants,\n            \'regime_slices\': slice_diagnostics(keys, values, bases[\'network\'], cutoff, history)}\n        if args.hourly_experts:\n            fitted_hourly[label] = hourly.HourlyExperts().fit(history, FIRST, cutoff)\n            candidate_predictions = fitted_hourly[label].variants(keys, bases[\'plain\'])\n            report[\'folds\'][label][\'hourly_vs_frozen_reference\'] = {\n                name: metrics(keys, values, pred) for name, pred in candidate_predictions.items()}\n        for n in (\'plain/reference\', \'network/reference\', \'only_50\', \'only_7\'):\n            print(f\'{label} {n}: {variants[n]["all"]["wape_score"]:.6f}\', flush=True)\n        if label in (\'sep_oct\', \'october_refit_31d\'):\n            kept[label] = keys, preds\n        dump(args.out/\'network_weather_report.json\', report)\n\n    keys, preds = kept[\'sep_oct\']\n    oct_keys, oct_preds = kept[\'october_refit_31d\']\n    groups, group_audit = select_groups(keys, values, preds, oct_keys, values, oct_preds, routes)\n    choice, control, head_audit = choose_heads(keys, values, preds, groups, routes)\n    report[\'selection\'] = {\'by_route\': choice, \'without_weather_by_route\': control,\n        \'network_gate\': group_audit, \'weather_gate\': head_audit,\n        \'network_accepted_routes\': [r for r in routes if groups[r] == \'network\'],\n        \'weather_accepted_routes\': [r for r in routes if choice[r][\'head\'].startswith(\'weather_\')]}\n    for label, (ks, ps) in kept.items():\n        report[\'folds\'][label][\'variants\'][\'selected\'] = metrics(ks, values, assemble(ks, ps, choice))\n        report[\'folds\'][label][\'variants\'][\'selected_no_weather\'] = metrics(ks, values, assemble(ks, ps, control))\n    # October selection using September alone and a fixed Aug31 model remains\n    # a separate forward diagnostic, without the October refit\'s label access.\n    sep_keys = [k for k in keys if k[1].month == 9]\n    fixed_oct_keys = [k for k in keys if k[1].month == 10]\n    sep_groups = {r: (\'network\' if r in (\'7\', \'50\') and\n        route_errors(sep_keys, values, preds[\'network\'][\'reference\'], r) <\n        .995*route_errors(sep_keys, values, preds[\'plain\'][\'reference\'], r) else \'plain\') for r in routes}\n    sep_choice, _, _ = choose_heads(sep_keys, values, preds, sep_groups, routes)\n    report[\'october_forward_from_september\'] = {\'choices\': sep_choice,\n        \'metrics\': metrics(fixed_oct_keys, values, assemble(fixed_oct_keys, preds, sep_choice)),\n        \'warning\': \'Frozen reference was previously selected using Sep-Oct; only additional choices use September alone.\'}\n    if args.hourly_experts:\n        hourly_predictions = {}\n        for label, (ks, ps) in kept.items():\n            incumbent = assemble(ks, ps, choice)\n            hourly_predictions[label] = fitted_hourly[label].variants(ks, incumbent)\n        hourly_choice, hourly_audit = hourly.select(keys, values, hourly_predictions[\'sep_oct\'],\n            refit_keys=oct_keys, refit_truth=values, refit_variants=hourly_predictions[\'october_refit_31d\'])\n        report[\'hourly_experts\'] = {\n            \'routes\': list(hourly.TARGET_ROUTES), \'choices\': hourly_choice,\n            \'variants\': {k: {\'daily_weight\': v[0], \'shape_weight\': v[1]} for k, v in hourly.VARIANTS.items()},\n            \'selection\': hourly_audit, \'minimum_comparable_days\': 4,\n            \'recent_comparable_weekdays\': 8,\n            \'warning\': \'Selection is exploratory on repeatedly used Sep-Oct. A refit-October guard is 31 days, not an independent long-horizon test.\'}\n        for label, (ks, _) in kept.items():\n            hp = hourly_predictions[label]\n            selected = hourly.combine(ks, hp, hourly_choice)\n            report[\'folds\'][label][\'hourly_on_network_weather\'] = {\n                name: metrics(ks, values, pred) for name, pred in {**hp, \'selected\': selected}.items()}\n            report[\'folds\'][label][\'hourly_diagnostics\'] = {\n                \'before\': hourly.diagnostics(ks, values, hp[\'reference\']),\n                \'after\': hourly.diagnostics(ks, values, selected)}\n        # Extra forward check uses only September for both the network/weather\n        # and hourly-head choices. The original frozen reference remains exploratory.\n        forward_base = assemble(keys, preds, sep_choice)\n        forward_variants = fitted_hourly[\'sep_oct\'].variants(keys, forward_base)\n        forward_choice, _ = hourly.select(sep_keys, values, forward_variants)\n        report[\'hourly_experts\'][\'october_forward_from_september\'] = {\n            \'choices\': forward_choice, \'metrics\': metrics(fixed_oct_keys, values,\n                hourly.combine(fixed_oct_keys, forward_variants, forward_choice))}\n        dump(args.out/\'network_weather_report.json\', report)\n    # Historical restoration check corresponding to DS2 experiment 04.\n    restore_cut = date(2025, 8, 13)\n    restore_keys = grid(routes, date(2025, 8, 14), date(2025, 8, 31))\n    restore = bank.predict_bases(restore_cut, restore_keys)\n    report[\'august_restoration_check\'] = {n: metrics(restore_keys, values, v) for n, v in restore.items()}\n\n    reference, regimes = bank.fitted(FINAL_CUTOFF)\n    models, _ = fit_models(bank, FINAL_CUTOFF, weather)\n    bases = bank.predict_bases(FINAL_CUTOFF, sample)\n    predictions = predict_models(sample, bases, models, weather)\n    outputs = {\n        \'submission_network_weather.csv\': assemble(sample, predictions, choice),\n        \'submission_network_no_weather.csv\': assemble(sample, predictions, control),\n        \'submission_regimes_candidate.csv\': bases[\'network\'],\n        \'submission_reference.csv\': bases[\'plain\']}\n    primary_submission = \'submission_network_weather.csv\'\n    if args.hourly_experts:\n        final_hourly = hourly.HourlyExperts().fit(values, FIRST, FINAL_CUTOFF)\n        final_variants = final_hourly.variants(sample, outputs[\'submission_network_weather.csv\'])\n        primary_submission = \'submission_hourly_experts.csv\'\n        outputs[primary_submission] = hourly.combine(sample, final_variants, hourly_choice)\n        report[\'hourly_experts\'][\'changed_rows_vs_network_weather\'] = {\n            r: sum(outputs[primary_submission][k] != outputs[\'submission_network_weather.csv\'][k]\n                   for k in sample if k[0] == r) for r in routes}\n        if any(outputs[primary_submission][k] != outputs[\'submission_network_weather.csv\'][k]\n               for k in sample if k[0] not in hourly.TARGET_ROUTES):\n            raise RuntimeError(\'Hourly expert changed an unauthorized route\')\n    # Keep a clearly labeled alternative for the uncertain repair schedule on\n    # the working Saturday and holiday weekdays, rather than silently asserting\n    # that the production calendar is a service timetable.\n    scenario = dict(outputs[\'submission_network_weather.csv\'])\n    mapping = {date(2025, 11, 1): date(2025, 11, 7),\n               date(2025, 11, 3): date(2025, 11, 2),\n               date(2025, 11, 4): date(2025, 11, 2)}\n    replacements = {k: (k[0], mapping[k[1]], k[2]) for k in sample if k[0] in (\'7\', \'50\') and k[1] in mapping}\n    # Deliberately simple demand proxies for this separate scenario: original\n    # Nov1 reference, and Nov2 restricted profile for Nov3/4. Weather corrections\n    # on those 144 rows are not retained. This is not the default submission.\n    for k, proxy in replacements.items():\n        if k[1] == date(2025, 11, 1):\n            scenario[k] = outputs[\'submission_reference.csv\'][k]\n        else:\n            scenario[k] = outputs[\'submission_regimes_candidate.csv\'][proxy]\n    outputs[\'submission_holiday_service_scenario.csv\'] = scenario\n    if args.hourly_experts:\n        primary_submission = \'submission_hourly_holiday_service.csv\'\n        outputs[primary_submission] = hourly.combine(\n            sample, final_hourly.variants(sample, scenario), hourly_choice)\n        report[\'hourly_experts\'][\'changed_rows_vs_holiday_service\'] = {\n            r: sum(outputs[primary_submission][k] != scenario[k] for k in sample if k[0] == r)\n            for r in routes}\n        if any(outputs[primary_submission][k] != scenario[k]\n               for k in sample if k[0] not in hourly.TARGET_ROUTES):\n            raise RuntimeError(\'Hourly expert changed a protected holiday-service route\')\n        report[\'user_reported_platform_scores\'] = {\n            \'submission_holiday_service_scenario.csv\': 0.87989,\n            \'submission_network_weather.csv\': 0.87755,\n            \'source\': \'User message; new hourly submission has no platform score yet\'}\n    report[\'outputs\'] = {}\n    for name, prediction in outputs.items():\n        report[\'outputs\'][name] = write_submission(args.sample, prediction, args.out/name)\n    if sha256(args.out/\'submission_reference.csv\') != ref[\'submission_sha256\']:\n        raise RuntimeError(\'Frozen 0.877 reference did not reproduce byte-for-byte\')\n    ensemble = NetworkWeatherEnsemble(reference, regimes, models, weather, choice)\n    if args.hourly_experts:\n        ensemble = hourly.HourlyExpertEnsemble(\n            HolidayServiceEnsemble(ensemble, reference, regimes), final_hourly, hourly_choice)\n    artifact = pickle.dumps(ensemble, protocol=5)\n    if pickle.loads(artifact).predict(sample) != outputs[primary_submission]:\n        raise RuntimeError(\'Saved model prediction mismatch\')\n    (args.out/\'model.local.pkl\').write_bytes(artifact)\n    report[\'outputs_sha256\'] = {n: sha256(args.out/n) for n in (*outputs, \'model.local.pkl\')}\n    report[\'changed_rows_vs_reference\'] = {n: sum(v[k] != bases[\'plain\'][k] for k in sample) for n, v in outputs.items()}\n    report[\'final_state_coverage\'] = regimes.coverage(sample)\n    report[\'restoration_warning\'] = \'Nov15 service restoration is confirmed externally. Its hourly demand profile is pooled with earlier normal-service history; no November demand truth is available.\'\n    report[\'holiday_service_scenario\'] = \'Alternative only: Nov1 normal service, Nov3/4 Sunday restricted service for 7/50. No claim that this unpublished repair schedule is correct.\'\n    report[\'primary_submission\'] = primary_submission\n    dump(args.out/\'network_weather_report.json\', report)\n    if args.hourly_experts:\n        dump(args.out/\'hourly_experts_report.json\', report)\n        print(\'Hourly expert choices:\', hourly_choice, flush=True)\n    print(\'Accepted service regimes:\', report[\'selection\'][\'network_accepted_routes\'], flush=True)\n    print(\'Accepted weather:\', report[\'selection\'][\'weather_accepted_routes\'], flush=True)\n    print(\'Saved 14640-row submissions; route 5 retained with zeros. Platform score unknown.\', flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    from ml.forecast.network_competition import main as run_main\n    run_main()\n', 'ml/forecast/test_network_competition.py': '"""In-memory correctness checks, not a demand-quality benchmark."""\nimport json\nimport pickle\nimport unittest\nfrom datetime import date, timedelta\nfrom unittest.mock import patch, Mock\n\nimport numpy as np\n\nfrom ml.forecast.core import grid\nfrom ml.forecast.network_regimes import (state, NORMAL, JULY_SHORT, WEST_AUG, WEST_SEP,\n                              BOTH_ENDS, CANCELLED, RegimeProfile)\nfrom ml.forecast.network_competition import HistoryBank, select_groups, assemble, features\n\n\nclass NetworkTests(unittest.TestCase):\n    def test_holiday_service_handles_partial_keys_and_preserves_other_routes(self):\n        from ml.forecast.network_competition import HolidayServiceEnsemble\n        keys = [(\'7\', date(2025, 11, 1), 12), (\'50\', date(2025, 11, 3), 9),\n                (\'7\', date(2025, 11, 4), 10), (\'11\', date(2025, 11, 3), 9),\n                (\'7\', date(2025, 11, 5), 9), (\'5\', date(2025, 11, 3), 9)]\n        baseline, reference, regimes = Mock(), Mock(), Mock()\n        baseline.predict.side_effect = lambda ks: {k: 0 if k[0] == \'5\' else 100 for k in ks}\n        reference.predict.side_effect = lambda ks: {k: k[2]*10 for k in ks}\n        regimes.apply.side_effect = lambda ks, pred: {k: pred[k]//2 for k in ks}\n        result = HolidayServiceEnsemble(baseline, reference, regimes).predict(keys)\n        self.assertEqual(list(result.values()), [120, 45, 50, 100, 100, 0])\n        proxy_keys = regimes.apply.call_args.args[0]\n        self.assertTrue(all(k[1] == date(2025, 11, 2) for k in proxy_keys))\n        self.assertEqual(set(result), set(keys))\n\n    def test_distinct_endpoints_and_restoration_boundaries(self):\n        for stamp, expected in [(\'2025-07-12\', JULY_SHORT), (\'2025-08-13\', NORMAL),\n                (\'2025-08-16\', WEST_AUG), (\'2025-09-06\', WEST_SEP),\n                (\'2025-09-20\', BOTH_ENDS), (\'2025-10-04\', WEST_SEP),\n                (\'2025-11-09\', WEST_SEP), (\'2025-11-15\', NORMAL)]:\n            self.assertEqual(state(\'7\', date.fromisoformat(stamp)), expected, stamp)\n        self.assertEqual(state(\'50\', date(2025, 9, 6)), CANCELLED)\n        self.assertEqual(state(\'50\', date(2025, 11, 15)), NORMAL)\n\n    def test_demand_holiday_does_not_imply_repair_closure(self):\n        self.assertEqual(state(\'50\', date(2025, 11, 1)), CANCELLED)  # literal Saturday\n        self.assertEqual(state(\'50\', date(2025, 11, 3)), NORMAL)  # demand holiday\n        self.assertEqual(state(\'50\', date(2025, 9, 8)), NORMAL)\n        self.assertEqual(state(\'17\', date(2025, 9, 6)), NORMAL)\n\n    def test_asof_uses_separate_publications_for_start_and_end(self):\n        d = date(2025, 11, 22)\n        self.assertEqual(state(\'50\', d, mode=\'as_of\', as_of=date(2025, 8, 31)), NORMAL)\n        self.assertEqual(state(\'50\', d, mode=\'as_of\', as_of=date(2025, 10, 31)), CANCELLED)\n        self.assertEqual(state(\'50\', d), NORMAL)\n        self.assertEqual(state(\'7\', date(2025, 10, 11), mode=\'as_of\', as_of=date(2025, 9, 30)), BOTH_ENDS)\n        with self.assertRaises(ValueError):\n            state(\'7\', d, mode=\'as_of\')\n\n    def test_unseen_state_does_not_transfer_july_factor(self):\n        keys = grid([\'7\', \'50\', \'1\'], date(2025, 1, 1), date(2025, 8, 31))\n        history = {k: 10 if state(k[0], k[1]) == JULY_SHORT else 100 for k in keys}\n        model = RegimeProfile().fit(history, date(2025, 1, 1), date(2025, 8, 31))\n        forecast = grid([\'7\', \'50\', \'1\', \'5\'], date(2025, 9, 6), date(2025, 9, 6))\n        base = dict.fromkeys(forecast, 100)\n        pred = model.apply(forecast, base)\n        self.assertEqual(pred[\'7\', date(2025, 9, 6), 12], 100)\n        self.assertEqual(pred[\'50\', date(2025, 9, 6), 12], 0)\n        self.assertEqual(pred[\'1\', date(2025, 9, 6), 12], 100)\n        self.assertEqual(pred[\'5\', date(2025, 9, 6), 12], 0)\n\n    def test_cancellation_keeps_observed_truth_and_learns_residual(self):\n        history = {k: 12 for k in grid([\'50\'], date(2025, 9, 6), date(2025, 10, 31))}\n        original = dict(history)\n        model = RegimeProfile().fit(history, date(2025, 9, 6), date(2025, 10, 31))\n        keys = [(\'50\', date(2025, 11, 2), 12)]\n        self.assertEqual(model.apply(keys, dict.fromkeys(keys, 100))[keys[0]], 12)\n        self.assertEqual(history, original)\n\n    def test_september_state_pools_four_weekend_days_for_october(self):\n        keys = grid([\'7\'], date(2025, 9, 1), date(2025, 9, 30))\n        history = {k: 40 if state(\'7\', k[1]) == WEST_SEP else 100 for k in keys}\n        model = RegimeProfile().fit(history, date(2025, 9, 1), date(2025, 9, 30))\n        future = [(\'7\', date(2025, 10, 4), 12)]\n        self.assertEqual(model.lookup(\'7\', WEST_SEP, 5, 12), 40)\n        self.assertEqual(model.apply(future, dict.fromkeys(future, 100))[future[0]], 88)\n\n    def test_restoration_removes_zero_rule_and_pooling_is_bounded(self):\n        keys = grid([\'50\'], date(2025, 1, 1), date(2025, 10, 31))\n        history = {k: 0 if state(k[0], k[1]) == CANCELLED else 1000 for k in keys}\n        model = RegimeProfile().fit(history, date(2025, 1, 1), date(2025, 10, 31))\n        future = [(\'50\', date(2025, 11, 9), 12), (\'50\', date(2025, 11, 15), 12)]\n        pred = model.apply(future, dict.fromkeys(future, 100))\n        self.assertEqual(pred[future[0]], 0)\n        self.assertEqual(pred[future[1]], 112)  # 25% of bounded 150 profile; half-even\n        self.assertEqual(pickle.loads(pickle.dumps(model)).apply(future, dict.fromkeys(future, 100)), pred)\n\n    def test_state_training_cannot_cross_cutoff(self):\n        with self.assertRaises(ValueError):\n            RegimeProfile().fit({(\'7\', date(2025, 9, 1), 12): 1}, date(2025, 1, 1), date(2025, 8, 31))\n\n    def test_training_cache_never_reads_future_labels_into_features_or_targets(self):\n        values = {(\'1\', date(2025, 1, 1)+timedelta(days=i), 12): i+1 for i in range(120)}\n        seen = []\n        def factory(prefix, cutoff, choices):\n            self.assertLessEqual(max(k[1] for k in prefix), cutoff)\n            class Model:\n                def predict(_, ks):\n                    self.assertGreater(min(k[1] for k in ks), cutoff)\n                    seen.append(cutoff)\n                    return dict.fromkeys(ks, sum(prefix.values()))\n            return Model()\n        with patch(\'ml.forecast.network_competition.fit_reference\', side_effect=factory):\n            a = HistoryBank(values, {}).training(date(2025, 3, 31))\n            perturbed = {k: (999999 if k[1] > date(2025, 3, 31) else v) for k, v in values.items()}\n            b = HistoryBank(perturbed, {}).training(date(2025, 3, 31))\n        self.assertEqual(a[0:2], b[0:2])\n        np.testing.assert_array_equal(a[2], b[2])\n        self.assertEqual(set(seen), {date(2025, 1, 31), date(2025, 2, 28)})\n\n    def test_bad_long_month_rejects_route50_even_with_good_october_refit(self):\n        ks = [(\'50\', date(2025, m, 6), 12) for m in (9, 10)]\n        truth = dict.fromkeys(ks, 100)\n        preds = {\'plain\': {\'reference\': dict.fromkeys(ks, 80)},\n                 \'network\': {\'reference\': {ks[0]: 100, ks[1]: 70}}}\n        recent = {\'plain\': {\'reference\': dict.fromkeys(ks, 80)},\n                  \'network\': {\'reference\': dict.fromkeys(ks, 100)}}\n        result, _ = select_groups(ks, truth, preds, ks, truth, recent, [\'50\'])\n        self.assertEqual(result[\'50\'], \'plain\')\n\n    def test_new_route7_state_requires_october_evidence(self):\n        ks = [(\'7\', date(2025, m, 6), 12) for m in (9, 10)]\n        truth = dict.fromkeys(ks, 100)\n        preds = {g: {\'reference\': dict.fromkeys(ks, 80)} for g in (\'plain\', \'network\')}\n        recent = {\'plain\': {\'reference\': dict.fromkeys(ks, 80)},\n                  \'network\': {\'reference\': dict.fromkeys(ks, 98)}}\n        result, _ = select_groups(ks, truth, preds, ks, truth, recent, [\'7\'])\n        self.assertEqual(result[\'7\'], \'network\')\n\n    def test_weather_pair_has_identical_service_feature(self):\n        ks = [(\'50\', date(2025, 9, 6), 12)]\n        weather = {(ks[0][1], 12): [0.]*10}\n        base = dict.fromkeys(ks, 0)\n        a = features(ks, base, weather, False, True)\n        b = features(ks, base, weather, True, True)\n        self.assertEqual(a[0, -1], CANCELLED)\n        self.assertEqual(b[0, -1], CANCELLED)\n        np.testing.assert_array_equal(a[:, :9], b[:, :9])\n\n    def test_unknown_route_preserved_as_zero_when_assembling(self):\n        ks = [(\'5\', date(2025, 11, 1), 12)]\n        self.assertEqual(assemble(ks, {}, {}), {ks[0]: 0})\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'ml/forecast/network_sources.json': '{\n  "source_mode": "competition",\n  "retrieved_at": "2026-09-27",\n  "research_branch": "ds2/research-protocol",\n  "research_commit": "b593ebbb5351bcc31881a99431d820ad97b5d140",\n  "authorization": "Organizer Q&A permits external facts published after 2025-10-31, including actual weather and transport news. Future passenger targets are never permitted.",\n  "events": [\n    {"routes": [7], "published_at": "2025-07-02", "start": "2025-07-10", "end_exclusive": "2025-08-13", "end_published_at": "2025-08-13", "state": "sokolniki_july", "days": "all", "source": "https://transport.mos.ru/mostrans/all_news/125169", "end_source": "https://transport.mos.ru/mostrans/all_news/125800"},\n    {"routes": [7], "published_at": "2025-08-15", "start": "2025-08-16", "end_exclusive": "2025-09-06", "end_published_at": "2025-09-05", "state": "novoslobodskaya_august", "days": "Saturday/Sunday", "source": "https://transport.mos.ru/mostrans/all_news/125846"},\n    {"routes": [7, 50], "published_at": "2025-09-05", "start": "2025-09-06", "end_exclusive": "2025-11-15", "end_published_at": "2025-11-15", "state_by_route": {"7": "kalanchyovskaya", "50": "cancelled"}, "days": "Saturday/Sunday", "source": "https://mosmetro.ru/news/details/7732", "readable_primary_mirror": "https://mosmetro.teletribe.ru/news/details/7732", "end_source": "https://t.me/DtOperativno/23565", "note": "DS2 registry omitted the simultaneous route-7 change in this route-50 notice; included here. Recorded positive route-50 labels are retained."},\n    {"routes": [7], "published_at": "2025-09-19", "start": "2025-09-20", "end_exclusive": "2025-10-04", "end_published_at": "2025-10-04", "state": "kalanchyovskaya_and_bogorodskoe", "days": "Saturday/Sunday", "source": "https://transport.mos.ru/mostrans/all_news/126405", "end_source": "https://mosmetro.ru/news/details/7973", "readable_end_source": "https://mosmetro.teletribe.ru/news/details/7973"}\n  ],\n  "limitations": [\n    "This is a partial daytime service registry, not a complete historical timetable.",\n    "Repair weekends are interpreted literally as Saturday/Sunday. Nov 1 is a working Saturday: whether repair closures followed the changed work calendar is unconfirmed. Nov 3/4 closures are not inferred from the public-holiday demand calendar. A separate holiday-service scenario CSV isolates this uncertainty.",\n    "Late-night changes reported Sep 12, Oct 28, Nov 5 and Dec 13 have no fully verified repeat schedule and hour-by-hour capacity. No invented zero-service mask is applied to them.",\n    "November restoration is externally confirmed, but no November labels exist locally; its passenger-demand effect is unvalidated.",\n    "July-shortening and September-east-detour magnitudes are not assumed identical. An unseen state falls back to the reference.",\n    "The 31-day October refit diagnostic can assess a state observed in September; it is not an independent 61-day test. The long Sep-Oct gate remains reported separately."\n  ]\n}\n', 'ml/forecast/research_audit.json': '{\n  "branch": "ds2/research-protocol",\n  "commit": "b593ebbb5351bcc31881a99431d820ad97b5d140",\n  "scope": "docs/research recursively, complete tracked files",\n  "review": "All Markdown reports and Python scripts inspected; all saved JSON reports parsed and metric identities checked. Experiments were not rerun without organizer labels.",\n  "file_count": 56,\n  "counts": {\n    ".md": 19,\n    ".py": 19,\n    ".json": 18\n  },\n  "metric_identity_checks": 1063,\n  "files": [\n    {\n      "path": "CASE_STUDIES_2025.md",\n      "sha256": "9a4984e5d807f4e63b16e20f03c2b5024b1a55af7ecd4588a40b9226ddb3d166",\n      "bytes": 7616\n    },\n    {\n      "path": "EXPERIMENT_02_HISTORY.md",\n      "sha256": "7740029d69ccb1ae6140aba1aad85af7a95146ab37125da37f584713c920ba49",\n      "bytes": 5964\n    },\n    {\n      "path": "EXPERIMENT_03_CALENDAR.md",\n      "sha256": "211abd3b6fb18c61d68cf396e1d51fd0d26bb61682cbe0adbddb54cdd73096c9",\n      "bytes": 4749\n    },\n    {\n      "path": "EXPERIMENT_04_ROUTE7_RESTORATION.md",\n      "sha256": "67b3b013deb111ed9091b5dcb4cfe804c2f384c105b493de466088265817908c",\n      "bytes": 5865\n    },\n    {\n      "path": "EXPERIMENT_05_ROUTE50_SHUTDOWN.md",\n      "sha256": "daf0259ba7a538a05762d53a7365f853b5179b8e8641013d1a9a8f36a8e014ae",\n      "bytes": 6491\n    },\n    {\n      "path": "EXPERIMENT_06_DS1_REAL_LABELS.md",\n      "sha256": "00ae445c9ce6a928baef74cf8484c330032d5527daf87e6394842b39607dce14",\n      "bytes": 5505\n    },\n    {\n      "path": "EXPERIMENT_07_ADAPTIVE_PROFILES.md",\n      "sha256": "309bb442b43676bbc67986ceff39f4e3422b64462e263b41defff3ac5b3cc728",\n      "bytes": 6083\n    },\n    {\n      "path": "EXPERIMENT_08_NETWORK_AND_SOURCE_AUDIT.md",\n      "sha256": "e7bc6d4096534a4bdee2a3635665f2329de618f1779d18d1941ec4a56c66c7bf",\n      "bytes": 10484\n    },\n    {\n      "path": "EXPERIMENT_09_SCHEDULE_AND_GAPS.md",\n      "sha256": "a976453ebb2b3ab9df9f592cab345faae37158d21fc7cd829facde9460283161",\n      "bytes": 9534\n    },\n    {\n      "path": "EXPERIMENT_10_SCHOOL_TERM.md",\n      "sha256": "274c2fa9ffcb55fc0f3dc1493bd38d4b1372b6d70abb44dc525bef8840470eb4",\n      "bytes": 5512\n    },\n    {\n      "path": "EXPERIMENT_11_ANNUAL_SEASONALITY.md",\n      "sha256": "72ba8ceb60b0824e79bda4248c342fb54e44fcdddb94ea5705449fbf7e16e2d7",\n      "bytes": 3243\n    },\n    {\n      "path": "EXPERIMENT_12_EXTERNAL_AND_YEAR.md",\n      "sha256": "6d23415deed545c7503a71cf5f9db0462f5ef215cd2c199260a9b23d411424a6",\n      "bytes": 8568\n    },\n    {\n      "path": "EXPERIMENT_13_AUGUST_REGIME.md",\n      "sha256": "6c0b1e0fbcfea3cb72c4475d1d4d5c8bf78b7d2630c6dfd5008db0c3756ba1c6",\n      "bytes": 5239\n    },\n    {\n      "path": "EXPERIMENT_14_NEAR_TERM.md",\n      "sha256": "f2024714ca248299d4daa3d933c722e012b77de27a570c5d41aeb367508c5c1e",\n      "bytes": 5498\n    },\n    {\n      "path": "EXPERIMENT_15_SCHOOL_POI_ASOF.md",\n      "sha256": "e54058d79116ef88f98cf5c5e769315e15dac1209876f60b86b6fe92166a954b",\n      "bytes": 6852\n    },\n    {\n      "path": "FACTOR_MATRIX.md",\n      "sha256": "888c73c456a94a51cb678dfca0031ebadd465783295b469feff22cffa0a9a1c4",\n      "bytes": 18276\n    },\n    {\n      "path": "MODEL_CRITERIA_2026-09-26.md",\n      "sha256": "737ccbdbbe4d434e99905688bc5cd282ef0ad83656f584238ba515e1603c34bf",\n      "bytes": 9542\n    },\n    {\n      "path": "README.md",\n      "sha256": "a8cd5e3b21a814ec8ad027866ff3c3218e552837a2f37e0deb427246c9fe75d4",\n      "bytes": 7066\n    },\n    {\n      "path": "RESULTS_2026-09-26.md",\n      "sha256": "f8dcd389b856420bcec8ad2ac4cdef60913245660c457ffdba718fc94debc85d",\n      "bytes": 8576\n    },\n    {\n      "path": "audit_missing_hours_raw.py",\n      "sha256": "4f8821f3d2ec9dd8ef26de8505bb81e0bfefcebf59124dec8c8e5db1635cfb45",\n      "bytes": 4478,\n      "functions": [\n        "days",\n        "main"\n      ]\n    },\n    {\n      "path": "audit_route50_sep13.py",\n      "sha256": "65e53f7b197309e06e4a62ef8e6a2e93999e77d8d5d344a43bfb118b5bb2f521",\n      "bytes": 3646,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "backtest_baseline.py",\n      "sha256": "3f28f66e3e2e095058838d0a94b3b38e7a8aea04e30c43fb99ecbb907a607f65",\n      "bytes": 5320,\n      "functions": [\n        "load",\n        "dates",\n        "fit",\n        "evaluate",\n        "main"\n      ]\n    },\n    {\n      "path": "diagnose_routes.py",\n      "sha256": "a2c24e59a3442bf3f39028926f0f313b92e38897f894794a5aaaa7f8f6f0ce80",\n      "bytes": 3848,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_14_near_term.py",\n      "sha256": "64063003ab70a52d6bd4d869f33821bd9ed7ed593a2e462a544ef2f404bc88e3",\n      "bytes": 5670,\n      "functions": [\n        "load",\n        "windows",\n        "measure",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_15_school_poi.py",\n      "sha256": "6367e8a2bd14608239f1d5f6299337a8a7efd8766f908ae39d114f79efac35b8",\n      "bytes": 5564,\n      "functions": [\n        "days",\n        "read_labels",\n        "profile_score",\n        "audited_stops",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_adaptive_profiles.py",\n      "sha256": "bea122b940756225faddab502da31d271b11c450134eed4c168fddcd91fed3f7",\n      "bytes": 6239,\n      "functions": [\n        "recent_means",\n        "predict",\n        "assess",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_annual_fourier.py",\n      "sha256": "dfc97e68ff989d510dd79cdbd42cf7d8c8119e63f94071e0baba6292f7f59d2d",\n      "bytes": 4192,\n      "functions": [\n        "vector",\n        "run_fold",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_august_regime.py",\n      "sha256": "beb45b83bc205430c431b95a269cfc700e17e7e8e8e3a6d32fff7c0fdb5e6523",\n      "bytes": 4057,\n      "functions": [\n        "run",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_calendar.py",\n      "sha256": "1a1dbb0747cbb789d3ef7f226ca73dc74073ef64147bbb6722a59f94afe543a1",\n      "bytes": 4858,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_ds1_candidates.py",\n      "sha256": "e7a199335e7e8b085037ce16510744dd1c3ebdd83d5a07b31df8fe39cc41ed4f",\n      "bytes": 4347,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_history_window.py",\n      "sha256": "fe83d4c96179fe0ecbd2605092a4e3a7ea946c685ef83651b2cfef948a35fb0b",\n      "bytes": 3105,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_network_sept19.py",\n      "sha256": "c777658e3881c929eb0f8fca860133e135b3747c8676ed1ee8f0c9d19a952ac5",\n      "bytes": 5157,\n      "functions": [\n        "predict",\n        "route7_transfer_factor",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_route17_restoration.py",\n      "sha256": "abe7b995edbcd2f6dbbb3407d20aa93c65d0de8c9921372f8579d0dcd763cae2",\n      "bytes": 4223,\n      "functions": [\n        "revised_route17_profile",\n        "metrics",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_route50_shutdown.py",\n      "sha256": "8ac9891d653652bb48b1e24ec8a03badc36072b54e8b54be62b3aec3f1878111",\n      "bytes": 4174,\n      "functions": [\n        "compare",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_route7_restoration.py",\n      "sha256": "7edd3ece4cc68f917722d4d7316acb5fa82ab43f13f96c35beedf793c29b5c4d",\n      "bytes": 4823,\n      "functions": [\n        "fit_with_known_status",\n        "score",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_school_term_profiles.py",\n      "sha256": "1a25b05c95b5c608d0a436b5257628e1a2fa0bd872764ad70c75c37974fa5e5e",\n      "bytes": 3821,\n      "functions": [\n        "score",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_year_horizon.py",\n      "sha256": "97831b55945f01de9984d3fbd869f60a1ff09a0387df5a5e1287ce2319741338",\n      "bytes": 3013,\n      "functions": [\n        "peak_metrics",\n        "main"\n      ]\n    },\n    {\n      "path": "profile_data.py",\n      "sha256": "30db2ba842ee830a8212a31edf8db048b049f7dbe1ebe0849e6a484c96d8d0e4",\n      "bytes": 10200,\n      "functions": [\n        "open_text",\n        "parse_timestamp",\n        "parse_route",\n        "write_csv",\n        "load_labels",\n        "inspect_submission",\n        "days",\n        "audit_raw",\n        "main"\n      ]\n    },\n    {\n      "path": "results/adaptive_profiles.json",\n      "sha256": "b87dae1e03381c8556911f08fbc926c7836fe71f5f406323e411b6e7cc1b5bfa",\n      "bytes": 44543,\n      "top_level_fields": [\n        "hypothesis",\n        "calendar_source",\n        "candidate_grid",\n        "missing_label_rule",\n        "folds",\n        "selection"\n      ]\n    },\n    {\n      "path": "results/annual_fourier.json",\n      "sha256": "d6cd86d8f4e911121539c21c93592299099767e77c874b91d036d75a3c0d1bda",\n      "bytes": 3458,\n      "top_level_fields": [\n        "calendar_features",\n        "network_scale_routes",\n        "excluded_from_scale",\n        "rules",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/august_regime.json",\n      "sha256": "79c0009fcd49255c4d69626c13b9438543bc83ce4d47e6d66248579eaf15d7e7",\n      "bytes": 5314,\n      "top_level_fields": [\n        "cutoff",\n        "validation",\n        "status",\n        "source",\n        "metrics"\n      ]\n    },\n    {\n      "path": "results/backtest.json",\n      "sha256": "f711379ea7e8ebb23b94d8b259a833c4f1cd7c40e6b00178d2a47f1003994589",\n      "bytes": 7279,\n      "top_level_fields": [\n        "method",\n        "missing_label_rule",\n        "routes",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/calendar.json",\n      "sha256": "d84c45e9612a3ed23f49f52cb306f55cdd5409f2ac5940fb411a1d25f8ed509e",\n      "bytes": 4178,\n      "top_level_fields": [\n        "source_url",\n        "published_at",\n        "hypothesis",\n        "new_feature",\n        "routes",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/ds1_real_candidates.json",\n      "sha256": "8bdfc73f7a63f1b70563b6ad84bfc932116897bc7c9ceee6184875af7b4dc6d1",\n      "bytes": 90696,\n      "top_level_fields": [\n        "source_branch",\n        "source_commit",\n        "ds1_code_sha256",\n        "input_sha256",\n        "metric",\n        "policy",\n        "folds",\n        "selection"\n      ]\n    },\n    {\n      "path": "results/experiment_15_school_poi_asof.json",\n      "sha256": "0becf8b5167656b2a82f9d074e98ea2f81a1a283291f2fdb1f2ac23b7f6731fd",\n      "bytes": 1083,\n      "top_level_fields": [\n        "cutoff",\n        "holdout",\n        "hour_grid",\n        "baseline",\n        "exclude_summer_weekdays",\n        "delta_score",\n        "selection_note",\n        "stops"\n      ]\n    },\n    {\n      "path": "results/history_window.json",\n      "sha256": "595313903aabbc72610249572c09d4b88ef331a99792faf063e8821001bcdf1c",\n      "bytes": 15558,\n      "top_level_fields": [\n        "hypothesis",\n        "changed_parameter",\n        "leakage_rule",\n        "routes",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/missing_hours_raw.json",\n      "sha256": "f41b898af29f924914683b71ed20049f419ca1844fec63e9f429749d6bf87554",\n      "bytes": 13378,\n      "top_level_fields": [\n        "raw_rows_read",\n        "scope",\n        "candidate_hours_not_in_labels",\n        "candidate_hours_with_any_raw_transaction",\n        "candidate_hours_with_success",\n        "oracle_125_hour_error_removed_from_aug31_median",\n        "oracle_score_gain_if_future_missing_keys_known",\n        "raw_status_totals",\n        "raw_transaction_types",\n        "missing_hours",\n        "limitations"\n      ]\n    },\n    {\n      "path": "results/network_events_2025.json",\n      "sha256": "8659d4aee1b2b4b7e17ab91b0099f885f048ce5b5bd98d5bdf573d3b50bcbd84",\n      "bytes": 3742,\n      "top_level_fields": [\n        "scope",\n        "time_rule",\n        "routes_in_labels",\n        "routes_with_events_in_this_partial_registry",\n        "events"\n      ]\n    },\n    {\n      "path": "results/network_sept19.json",\n      "sha256": "fc70fd54fc6eed72cba400ebb571fab6e95dfe61a20cd203ca35e723b420aef2",\n      "bytes": 11711,\n      "top_level_fields": [\n        "cutoff",\n        "validation",\n        "event_dates",\n        "news",\n        "route7_scenario_transfer",\n        "training_rule",\n        "predictions",\n        "metrics"\n      ]\n    },\n    {\n      "path": "results/route17_restoration.json",\n      "sha256": "ed58b4ad0ee8c338c23b4a6ca37947274b18d78cbe09dfd07412f1cf7b2ee00c",\n      "bytes": 17182,\n      "top_level_fields": [\n        "sources",\n        "restricted_weekends_excluded_from_train",\n        "effect_isolated_to_route",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/route50_sep13_raw_audit.json",\n      "sha256": "493cdf647c8dd8b50b78a55c953fe87e22a18d91acbf0112ace528a8b7c4a483",\n      "bytes": 4348,\n      "top_level_fields": [\n        "source",\n        "rows_scanned",\n        "route_50_invalid_event_time",\n        "sep13_status_counts",\n        "sep13_status_transaction_type_counts",\n        "sep13_hourly",\n        "weekend_daily"\n      ]\n    },\n    {\n      "path": "results/route50_shutdown.json",\n      "sha256": "12d4dfa2ee886b2e04fca1424442a2a0e83959b147e3d370405d32ed7e50a6fe",\n      "bytes": 9622,\n      "top_level_fields": [\n        "source",\n        "published_at",\n        "cutoff",\n        "validation",\n        "missing_label_rule",\n        "changed_rows_per_statistic",\n        "variants",\n        "affected_day_facts"\n      ]\n    },\n    {\n      "path": "results/route7_restoration.json",\n      "sha256": "d71ee09e86ed79bb323deac50b1feb7644ca3f3906be40eb32f0652bd4339e47",\n      "bytes": 8203,\n      "top_level_fields": [\n        "source_start",\n        "start_notice_published",\n        "source_restored",\n        "restoration_notice_published",\n        "cutoff",\n        "excluded_route_7_training_dates",\n        "validation",\n        "definition",\n        "variants"\n      ]\n    },\n    {\n      "path": "results/route_diagnostics.json",\n      "sha256": "a904ee4732a186530b9e1d47dad50867141d4e6b00a4083c43f26bce532bace3",\n      "bytes": 3603,\n      "top_level_fields": [\n        "method",\n        "candidate_service_hours",\n        "routes_in_labels",\n        "route_7_windows",\n        "missing_active_hours_count",\n        "missing_active_hours_by_route",\n        "missing_active_hours_by_date"\n      ]\n    },\n    {\n      "path": "results/school_term_profiles.json",\n      "sha256": "6a0d0b7d7c860b14b1f876fa7f0bee1fa0ef328d526bbee32f339b843903570d",\n      "bytes": 8347,\n      "top_level_fields": [\n        "train_cutoff",\n        "validation",\n        "rule",\n        "selection_warning",\n        "baseline_median",\n        "variants"\n      ]\n    },\n    {\n      "path": "results/year_horizon.json",\n      "sha256": "17d534587150c0b44d37c00eb85e4a7638c7e1c26a9eb7c1cf7e1bef0c1791dd",\n      "bytes": 13960,\n      "top_level_fields": [\n        "metric",\n        "model",\n        "external_sources",\n        "full_year_validated",\n        "folds"\n      ]\n    }\n  ]\n}\n', 'ml/forecast/hourly_experts.py': '"""Separable daily-volume/hour-shape candidates for routes 17, 11 and 12.\n\nThese are finite, testable alternatives to the existing hourly route model,\nnot an assertion that a finer profile necessarily improves competition WAPE.\n"""\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nfrom datetime import date\nimport numpy as np\n\nfrom .core import days, effective_weekday, rounded\nfrom .route_specific import SeasonalProfile\n\nTARGET_ROUTES = (\'17\', \'11\', \'12\')\nVARIANTS = {\'level_50\': (.5, 0.), \'shape_50\': (0., .5),\n            \'level_shape_25\': (.25, .25), \'level_shape_50\': (.5, .5)}\n\n\ndef allocate(total, proportions):\n    """Largest remainders: integer hours sum exactly to the rounded daily total."""\n    p = np.asarray(proportions, dtype=float)\n    if p.shape != (24,) or not np.isfinite(p).all() or (p < 0).any():\n        raise ValueError(\'Need 24 finite nonnegative hourly shares\')\n    total = rounded(total)\n    if not p.sum():\n        if total:\n            raise ValueError(\'Positive daily total has no supported hours\')\n        return [0]*24\n    raw = total*p/p.sum()\n    integer = np.floor(raw).astype(np.int64)\n    left = total-int(integer.sum())\n    order = sorted(range(24), key=lambda h: (-(raw[h]-integer[h]), h))\n    for h in order[:left]:\n        integer[h] += 1\n    return [int(x) for x in integer]\n\n\nclass HourlyExperts:\n    """Calendar-compatible volume and shape with shrinkage to the incumbent.\n\n    Recent means last eight comparable weekdays, not eight calendar days.\n    Summer weekdays are separate. Sparse groups fall back to longer history;\n    no future targets or predicted-period lag updates are used.\n    """\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'History is empty or crosses cutoff\')\n        self.cutoff = cutoff\n        self.routes = set(TARGET_ROUTES) & {k[0] for k in history}\n        groups = defaultdict(list)\n        for r in sorted(self.routes):\n            for d in days(start, cutoff):\n                values = np.asarray([history.get((r, d, h), 0) for h in range(24)], dtype=float)\n                if (values < 0).any() or not np.isfinite(values).all():\n                    raise ValueError(\'Invalid historical demand\')\n                season = SeasonalProfile.summer(d)\n                w = effective_weekday(d)\n                for s in (None, season):\n                    groups[r, s, w].append(values)\n        self.tables = {}\n        for key, rows in groups.items():\n            if len(rows) < 4:\n                continue\n            x = np.stack(rows)\n            totals = x.sum(axis=1)\n            recent_totals = totals[-8:]\n            # Two robust training estimates, not a global autumn decay rule.\n            volume = .5*float(np.median(totals)) + .5*float(np.median(recent_totals))\n            valid = totals > 0\n            shape = None\n            if valid.sum() >= 4:\n                shares = x[valid]/totals[valid, None]\n                long = np.median(shares, axis=0)\n                recent = np.median(shares[-8:], axis=0)\n                if long.sum() and recent.sum():\n                    shape = .5*long/long.sum() + .5*recent/recent.sum()\n            self.tables[key] = {\'volume\': volume, \'shape\': shape,\n                                \'days\': len(rows), \'positive_days\': int(valid.sum()),\n                                \'recent_days\': min(8, len(rows))}\n        return self\n\n    def profile(self, route, day):\n        w = effective_weekday(day)\n        return self.tables.get((route, SeasonalProfile.summer(day), w),\n                               self.tables.get((route, None, w)))\n\n    def variants(self, keys, base):\n        if len(set(keys)) != len(keys) or set(keys) != set(base):\n            raise ValueError(\'Base predictions must match unique requested keys\')\n        if any(d <= self.cutoff for _, d, _ in keys):\n            raise ValueError(\'Prediction dates must follow cutoff\')\n        out = {\'reference\': dict(base), **{name: dict(base) for name in VARIANTS}}\n        by_day = defaultdict(set)\n        for r, d, h in keys:\n            if r in self.routes:\n                by_day[r, d].add(h)\n        for (r, d), hours in by_day.items():\n            if hours != set(range(24)):\n                raise ValueError(\'Daily/shape experts require all 24 hours for each target route-day\')\n            profile = self.profile(r, d)\n            if profile is None:\n                continue\n            b = np.asarray([base[r, d, h] for h in range(24)], dtype=float)\n            if not np.isfinite(b).all() or (b < 0).any():\n                raise ValueError(\'Invalid base prediction\')\n            total = float(b.sum())\n            if total == 0:\n                continue\n            base_shape = b/total\n            # Limit the proposed daily estimate before blending; final 50%\n            # candidate changes daily volume by no more than 15% (rounding aside).\n            volume = float(np.clip(profile[\'volume\'], .7*total, 1.3*total))\n            shape = profile[\'shape\']\n            if shape is None:\n                shape = base_shape\n            else:\n                # Preserve unsupported zero hours, then normalize the bounded\n                # shape. This avoids inventing night service from pooled days.\n                shape = np.clip(shape, .5*base_shape, 1.5*base_shape)\n                shape = shape/shape.sum()\n            for name, (volume_weight, shape_weight) in VARIANTS.items():\n                daily = (1-volume_weight)*total + volume_weight*volume\n                shares = (1-shape_weight)*base_shape + shape_weight*shape\n                prediction = allocate(daily, shares)\n                for h, value in enumerate(prediction):\n                    out[name][r, d, h] = value\n        return out\n\n\ndef select(keys, truth, variants, *, refit_keys=None, refit_truth=None, refit_variants=None):\n    """Require a 0.5% gain in each month, with an optional October refit guard."""\n    months = sorted({(k[1].year, k[1].month) for k in keys})\n    choices, audit = {}, {}\n    for r in TARGET_ROUTES:\n        groups = [[k for k in keys if k[0] == r and (k[1].year, k[1].month) == m] for m in months]\n        if not any(groups):\n            continue\n        errors = {name: [sum(abs(pred[k]-truth.get(k, 0)) for k in g) for g in groups]\n                  for name, pred in variants.items()}\n        recent_errors = {}\n        if refit_keys is not None:\n            if refit_truth is None or refit_variants is None:\n                raise ValueError(\'Refit guard requires both truth and predictions\')\n            selected = [k for k in refit_keys if k[0] == r]\n            recent_errors = {name: sum(abs(pred[k]-refit_truth.get(k, 0)) for k in selected)\n                             for name, pred in refit_variants.items()}\n        accepted = [\'reference\']\n        for name in VARIANTS:\n            monthly_win = (all(a <= .995*b for a, b in zip(errors[name], errors[\'reference\']))\n                           and sum(errors[name]) < sum(errors[\'reference\']))\n            recent_ok = not recent_errors or recent_errors[name] <= 1.005*recent_errors[\'reference\']\n            if monthly_win and recent_ok:\n                accepted.append(name)\n        choices[r] = min(accepted, key=lambda n: (sum(errors[n]), n))\n        audit[r] = {\'months\': [f\'{y}-{m:02}\' for y, m in months],\n                    \'monthly_absolute_errors\': errors, \'refit_absolute_errors\': recent_errors,\n                    \'accepted\': accepted, \'selected\': choices[r]}\n    return choices, audit\n\n\ndef combine(keys, variants, choices):\n    return {k: variants[choices.get(k[0], \'reference\')][k] for k in keys}\n\n\ndef diagnostics(keys, truth, prediction):\n    """Hour-specific errors and daily-total errors, never used as fit features."""\n    result = {}\n    for r in TARGET_ROUTES:\n        subset = [k for k in keys if k[0] == r]\n        if not subset:\n            continue\n        rows, daily = [], defaultdict(lambda: [0, 0])\n        for h in range(24):\n            ks = [k for k in subset if k[2] == h]\n            actual = sum(truth.get(k, 0) for k in ks)\n            predicted = sum(prediction[k] for k in ks)\n            error = sum(abs(prediction[k]-truth.get(k, 0)) for k in ks)\n            rows.append({\'hour\': h, \'actual_sum\': actual, \'prediction_sum\': predicted,\n                         \'absolute_error_sum\': error, \'bias\': predicted-actual})\n        for k in subset:\n            daily[k[1]][0] += truth.get(k, 0)\n            daily[k[1]][1] += prediction[k]\n        dayparts = {}\n        for name, hours in [(\'morning_06_10\', range(6, 11)), (\'midday_11_15\', range(11, 16)),\n                            (\'evening_16_20\', range(16, 21)), (\'other_hours\', (0, 1, 2, 3, 4, 5, 21, 22, 23))]:\n            dayparts[name] = {field: sum(rows[h][field] for h in hours)\n                              for field in (\'actual_sum\', \'prediction_sum\', \'absolute_error_sum\', \'bias\')}\n        result[r] = {\'by_hour\': rows, \'by_daypart\': dayparts,\n                     \'daily_total_absolute_error\': sum(abs(p-y) for y, p in daily.values()),\n                     \'hourly_absolute_error\': sum(x[\'absolute_error_sum\'] for x in rows)}\n    return result\n\n\nclass HourlyExpertEnsemble:\n    def __init__(self, baseline, experts, choices):\n        self.baseline, self.experts, self.choices = baseline, experts, choices\n\n    def predict(self, keys):\n        base = self.baseline.predict(keys)\n        return combine(keys, self.experts.variants(keys, base), self.choices)\n', 'ml/forecast/test_hourly_experts.py': '"""Correctness fixtures in memory; no generated training files or quality scores."""\nimport pickle\nimport unittest\nfrom datetime import date\n\nimport numpy as np\n\nfrom ml.forecast.core import grid\nfrom ml.forecast.hourly_experts import (HourlyExperts, allocate, combine, diagnostics,\n                                       select, VARIANTS, TARGET_ROUTES)\n\n\nclass HourlyTests(unittest.TestCase):\n    def history(self):\n        return {k: (100 if 6 <= k[2] <= 10 else 200 if 16 <= k[2] <= 20 else 20)\n                for k in grid(TARGET_ROUTES, date(2025, 1, 1), date(2025, 4, 30))}\n\n    def fitted(self):\n        return HourlyExperts().fit(self.history(), date(2025, 1, 1), date(2025, 4, 30))\n\n    def forecast(self):\n        keys = grid(TARGET_ROUTES+(\'7\', \'50\', \'5\'), date(2025, 5, 5), date(2025, 5, 5))\n        base = {k: (0 if k[0] == \'5\' or k[2] < 5 else 300 if k[2] < 12 else 50) for k in keys}\n        return keys, base\n\n    def test_shape_preserves_daily_sum_and_zero_hours(self):\n        model = self.fitted()\n        keys, base = self.forecast()\n        variants = model.variants(keys, base)\n        for r in TARGET_ROUTES:\n            ks = [k for k in keys if k[0] == r]\n            self.assertEqual(sum(base[k] for k in ks), sum(variants[\'shape_50\'][k] for k in ks))\n            self.assertTrue(any(base[k] != variants[\'shape_50\'][k] for k in ks))\n            self.assertTrue(all(variants[\'shape_50\'][k] == 0 for k in ks if base[k] == 0))\n\n    def test_other_routes_are_identical_in_every_variant(self):\n        keys, base = self.forecast()\n        for name, pred in self.fitted().variants(keys, base).items():\n            self.assertEqual(set(pred), set(keys))\n            self.assertTrue(all(pred[k] == base[k] for k in keys if k[0] not in TARGET_ROUTES), name)\n            self.assertTrue(all(isinstance(v, int) and v >= 0 for v in pred.values()))\n\n    def test_daily_level_bounds(self):\n        keys, base = self.forecast()\n        variants = self.fitted().variants(keys, base)\n        for r in TARGET_ROUTES:\n            ks = [k for k in keys if k[0] == r]\n            total = sum(base[k] for k in ks)\n            for name in VARIANTS:\n                self.assertLessEqual(abs(sum(variants[name][k] for k in ks)-total), .15*total+1)\n\n    def test_proportions_do_not_depend_on_training_volume_units(self):\n        values = self.history()\n        a = self.fitted()\n        b = HourlyExperts().fit({k: 3*v for k, v in values.items()}, date(2025, 1, 1), date(2025, 4, 30))\n        d = date(2025, 5, 5)\n        np.testing.assert_allclose(a.profile(\'17\', d)[\'shape\'], b.profile(\'17\', d)[\'shape\'])\n        self.assertAlmostEqual(b.profile(\'17\', d)[\'volume\'], 3*a.profile(\'17\', d)[\'volume\'])\n\n    def test_summer_weekday_slump_is_not_autumn_level(self):\n        keys = grid([\'17\'], date(2025, 1, 1), date(2025, 8, 31))\n        history = {k: 10 if k[1].month in (7, 8) and k[1].weekday() < 5 else 100 for k in keys}\n        model = HourlyExperts().fit(history, date(2025, 1, 1), date(2025, 8, 31))\n        self.assertEqual(model.profile(\'17\', date(2025, 9, 1))[\'volume\'], 2400)\n        self.assertEqual(model.profile(\'17\', date(2026, 7, 6))[\'volume\'], 240)\n\n    def test_history_and_prediction_cutoffs_enforced(self):\n        values = self.history()\n        values[\'17\', date(2025, 5, 1), 12] = 99999\n        with self.assertRaises(ValueError):\n            HourlyExperts().fit(values, date(2025, 1, 1), date(2025, 4, 30))\n        k = (\'17\', date(2025, 4, 30), 12)\n        with self.assertRaises(ValueError):\n            self.fitted().variants([k], {k: 1})\n\n    def test_exact_sums_and_deterministic_rounding(self):\n        result = allocate(101, np.ones(24))\n        self.assertEqual(sum(result), 101)\n        self.assertEqual(result[:5], [5]*5)\n        self.assertEqual(result[5:], [4]*19)\n        with self.assertRaises(ValueError):\n            allocate(1, np.zeros(24))\n        with self.assertRaises(ValueError):\n            allocate(1, [float(\'nan\')]*24)\n\n    def test_no_silent_partial_days_or_duplicate_keys(self):\n        model = self.fitted()\n        keys, base = self.forecast()\n        del base[keys[0]]\n        with self.assertRaises(ValueError):\n            model.variants(keys[1:], base)\n        with self.assertRaises(ValueError):\n            model.variants(keys+keys, dict.fromkeys(keys, 1))\n\n    def test_order_and_pickle_invariance(self):\n        model = self.fitted()\n        keys, base = self.forecast()\n        expected = model.variants(keys, base)\n        self.assertEqual(model.variants(list(reversed(keys)), base), expected)\n        self.assertEqual(pickle.loads(pickle.dumps(model)).variants(keys, base), expected)\n\n    def test_monthly_gate_rejects_single_month_win(self):\n        keys = [(\'17\', date(2025, m, 1), 12) for m in (9, 10)]\n        truth = dict.fromkeys(keys, 100)\n        variants = {n: dict.fromkeys(keys, 80) for n in (\'reference\', *VARIANTS)}\n        variants[\'shape_50\'] = {keys[0]: 100, keys[1]: 79}\n        choices, _ = select(keys, truth, variants)\n        self.assertEqual(choices[\'17\'], \'reference\')\n        variants[\'shape_50\'][keys[1]] = 99\n        choices, _ = select(keys, truth, variants)\n        self.assertEqual(choices[\'17\'], \'shape_50\')\n\n    def test_october_refit_can_veto_a_long_holdout_winner(self):\n        keys = [(\'17\', date(2025, m, 1), 12) for m in (9, 10)]\n        truth = dict.fromkeys(keys, 100)\n        variants = {n: dict.fromkeys(keys, 80) for n in (\'reference\', *VARIANTS)}\n        variants[\'shape_50\'] = dict.fromkeys(keys, 99)\n        refit = {n: dict(v) for n, v in variants.items()}\n        refit[\'shape_50\'] = dict.fromkeys(keys, 50)\n        choices, _ = select(keys, truth, variants, refit_keys=keys, refit_truth=truth, refit_variants=refit)\n        self.assertEqual(choices[\'17\'], \'reference\')\n\n    def test_hourly_error_detects_misshapen_day_with_perfect_total(self):\n        keys = grid([\'17\'], date(2025, 9, 1), date(2025, 9, 1))\n        truth = dict.fromkeys(keys, 0)\n        truth[\'17\', date(2025, 9, 1), 8] = 100\n        prediction = dict.fromkeys(keys, 0)\n        prediction[\'17\', date(2025, 9, 1), 18] = 100\n        d = diagnostics(keys, truth, prediction)[\'17\']\n        self.assertEqual(d[\'daily_total_absolute_error\'], 0)\n        self.assertEqual(d[\'hourly_absolute_error\'], 200)\n        self.assertEqual(d[\'by_daypart\'][\'morning_06_10\'][\'bias\'], -100)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'ml/forecast/education_competition.py': '"""Historical school/daycare POIs and calendar scenarios, with separate ablations."""\nimport argparse\nimport json\nimport pickle\nfrom datetime import date\nfrom pathlib import Path\nimport numpy as np\nfrom . import network_competition as nc\nfrom .network_regimes import state,STATE_NAMES,NORMAL\nfrom .hourly_experts import HourlyExperts,HourlyExpertEnsemble,combine\nfrom .weather_competition import corrected,weather_rows,WEIGHTS\nfrom .core import grid,load_labels,sha256\nfrom .run import FIRST,FINAL_CUTOFF\nfrom ml.submit.adapter import read_sample,write_submission\n\nfrom .education_features import REGISTRY,SOURCES,calendar_features,exposure_features\nBEST_HASH=\'b7c3dcbabc8d0915ac57ba9932fb8ee75b4f97d052fdba011d58270e1edc24a2\'\nTARGETS=(\'1\',\'11\',\'12\',\'17\',\'25\',\'26\',\'28\')\n\ndef active(key):return key[0] in TARGETS\n\ndef matrix(keys,base,weather,arm):\n    x=nc.features(keys,base,weather,True,True)\n    if arm in (\'calendar\',\'education\'):x=np.column_stack((x,np.asarray([calendar_features(k) for k in keys])))\n    if arm==\'education\':x=np.column_stack((x,np.asarray([exposure_features(k) for k in keys])))\n    return x\n\ndef fit_pair(bank,cutoff,weather):\n    import lightgbm as lgb\n    keys,bases,y,blocks=bank.training(cutoff);base=bases[\'network\']\n    scale=np.asarray([max(50,base[k]) for k in keys]);target=(y-np.asarray([base[k] for k in keys]))/scale\n    models={}\n    state_column=matrix(keys[:1],base,weather,\'control\').shape[1]-1\n    for name in (\'control\',\'calendar\',\'education\'):\n        model=lgb.LGBMRegressor(objective=\'regression_l1\',n_estimators=220,learning_rate=.025,num_leaves=12,min_child_samples=120,reg_lambda=10.,max_depth=5,n_jobs=2,verbosity=-1,random_state=2025)\n        model.fit(matrix(keys,base,weather,name),target,sample_weight=scale,categorical_feature=[0,1,2,3,4,5,state_column])\n        models[name]=model\n    return models,blocks\n\ndef variants(keys,base,incumbent,models,weather):\n    result={\'incumbent\':incumbent}\n    for name in (\'control\',\'calendar\',\'education\'):\n        raw=models[name].predict(matrix(keys,base,weather,name))\n        for w in WEIGHTS:\n            p=corrected(keys,base,raw,w)\n            result[f\'{name}_{w}\']={k:p[k] if active(k) else incumbent[k] for k in keys}\n    return result\n\ndef choose(keys,truth,preds,oct_keys,oct_preds):\n    selected={};audit={}\n    for route in TARGETS:\n        groups=[[k for k in keys if k[0]==route and k[1].month==m] for m in (9,10)]\n        errors={n:[sum(abs(p[k]-truth.get(k,0)) for k in g) for g in groups] for n,p in preds.items()}\n        oe={n:sum(abs(p[k]-truth.get(k,0)) for k in oct_keys if k[0]==route) for n,p in oct_preds.items()}\n        def beats(n,c):return all(a<=.995*b for a,b in zip(errors[n],errors[c])) and oe[n]<=1.005*oe[c]\n        calendars=[f\'calendar_{w}\' for w in WEIGHTS if beats(f\'calendar_{w}\',\'incumbent\') and beats(f\'calendar_{w}\',f\'control_{w}\')]\n        best_calendar=min([\'incumbent\']+calendars,key=lambda n:sum(errors[n]))\n        education=[f\'education_{w}\' for w in WEIGHTS if beats(f\'education_{w}\',\'incumbent\') and beats(f\'education_{w}\',f\'calendar_{w}\') and beats(f\'education_{w}\',best_calendar)]\n        selected[route]=min([\'incumbent\']+calendars+education,key=lambda n:sum(errors[n]))\n        audit[route]={\'errors_sep_oct\':errors,\'errors_october_refit\':oe,\'accepted_calendar\':calendars,\'accepted_education\':education,\'selected\':selected[route]}\n    return selected,audit\n\ndef join(keys,preds,selected):return {k:preds[selected.get(k[0],\'incumbent\')][k] for k in keys}\n\nclass EducationEnsemble:\n    def __init__(self,incumbent,reference,regimes,models,weather,selected):\n        self.incumbent,self.reference,self.regimes=incumbent,reference,regimes\n        self.models,self.weather,self.selected=models,weather,selected\n    def predict(self,keys):\n        keys=list(keys);old=self.incumbent.predict(keys)\n        base=self.regimes.apply(keys,self.reference.predict(keys))\n        replacements={k:(k[0],date(2025,11,2),k[2]) for k in keys\n                      if k[0] in (\'7\',\'50\') and k[1] in (date(2025,11,3),date(2025,11,4))}\n        proxies=list(dict.fromkeys(replacements.values()))\n        if proxies:\n            restricted=self.regimes.apply(proxies,self.reference.predict(proxies))\n            base.update({k:restricted[v] for k,v in replacements.items()})\n        return join(keys,variants(keys,base,old,self.models,self.weather),self.selected)\n\ndef main(argv=None):\n    ap=argparse.ArgumentParser()\n    for name in (\'train\',\'test\',\'sample\',\'weather\',\'baseline-dir\',\'out\'):ap.add_argument(\'--\'+name,type=Path,required=True)\n    ap.add_argument(\'--sample-from-reference\',action=\'store_true\');args=ap.parse_args(argv)\n    inputs=nc.verify_competition_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')},args.sample_from_reference)\n    values=load_labels([args.train,args.test]);assert all(FIRST<=k[1]<=FINAL_CUTOFF for k in values)\n    previous=json.loads((args.baseline_dir/\'hourly_experts_report.json\').read_text())\n    if sha256(args.baseline_dir/\'submission_hourly_holiday_service.csv\')!=BEST_HASH:raise ValueError(\'Baseline differs from verified 0.88078 submission\')\n    ref=previous[\'reference\'];choices=previous[\'selection\'][\'by_route\'];hourly=previous[\'hourly_experts\'][\'choices\']\n    weather=weather_rows(json.loads(args.weather.read_text()));bank=nc.HistoryBank(values,ref[\'choices\'])\n    args.out.mkdir(parents=True,exist_ok=True);folds={};held={}\n    for label,cutoff,start,end in [(\'sep_oct\',date(2025,8,31),date(2025,9,1),date(2025,10,31)),(\'october_refit\',date(2025,9,30),date(2025,10,1),date(2025,10,31))]:\n        print(\'Education:\',label,flush=True)\n        keys=grid(bank.routes,start,end);bases=bank.predict_bases(cutoff,keys)\n        oldmodels,_=nc.fit_models(bank,cutoff,weather)\n        incumbent=nc.assemble(keys,nc.predict_models(keys,bases,oldmodels,weather),choices)\n        hp=HourlyExperts().fit({k:v for k,v in values.items() if k[1]<=cutoff},FIRST,cutoff)\n        incumbent=combine(keys,hp.variants(keys,incumbent),hourly)\n        expected=previous[\'folds\'][label if label==\'sep_oct\' else \'october_refit_31d\'][\'hourly_on_network_weather\'][\'selected\'][\'all\']\n        if nc.metrics(keys,values,incumbent)[\'all\']!=expected:raise RuntimeError(\'Historical baseline did not reproduce\')\n        models,blocks=fit_pair(bank,cutoff,weather)\n        preds=variants(keys,bases[\'network\'],incumbent,models,weather)\n        held[label]=(keys,preds)\n        folds[label]={\'metrics\':{n:nc.metrics(keys,values,p) for n,p in preds.items()},\'training_blocks\':blocks}\n    keys,preds=held[\'sep_oct\'];oct_keys,oct_preds=held[\'october_refit\']\n    selected,audit=choose(keys,values,preds,oct_keys,oct_preds)\n    for label,(ks,ps) in held.items():folds[label][\'selected\']=nc.metrics(ks,values,join(ks,ps,selected))\n    # Fit all final model components from organizer labels, never load supplied pickle files.\n    _,sample=read_sample(args.sample);reference,regimes=bank.fitted(FINAL_CUTOFF)\n    oldmodels,_=nc.fit_models(bank,FINAL_CUTOFF,weather)\n    incumbent=nc.HolidayServiceEnsemble(nc.NetworkWeatherEnsemble(reference,regimes,oldmodels,weather,choices),reference,regimes)\n    incumbent=HourlyExpertEnsemble(incumbent,HourlyExperts().fit(values,FIRST,FINAL_CUTOFF),hourly)\n    before=incumbent.predict(sample)\n    write_submission(args.sample,before,args.out/\'submission_best_088078.csv\')\n    if sha256(args.out/\'submission_best_088078.csv\')!=BEST_HASH:raise RuntimeError(\'Final baseline hash mismatch\')\n    models,_=fit_pair(bank,FINAL_CUTOFF,weather)\n    ensemble=EducationEnsemble(incumbent,reference,regimes,models,weather,selected);after=ensemble.predict(sample)\n    assert all(after[k]==before[k] for k in sample if not active(k))\n    artifact=pickle.dumps(ensemble,protocol=5)\n    assert pickle.loads(artifact).predict(sample)==after\n    (args.out/\'model.local.pkl\').write_bytes(artifact)\n    output=write_submission(args.sample,after,args.out/\'submission_education.csv\')\n    report={\'source_mode\':\'competition_retrospective\',\'inputs\':inputs,\'registry\':REGISTRY,\'calendar_sources\':SOURCES,\'selected\':selected,\'selection\':audit,\'folds\':folds,\'gate\':\'Calendar must beat incumbent and bare control by >=0.5% per month; POI must beat incumbent, matched calendar and best admitted calendar. Oct refit deterioration <=0.5%. Routes 7/50/5 protected.\',\'selection_warning\':\'Sep-Oct reused in research. This is exploratory validation, not an independent test.\',\'best_platform_score_user_reported\':.88078,\'new_platform_score\':None,\'primary_submission\':\'submission_education.csv\',\'output\':output,\'changed_rows\':{r:sum(before[k]!=after[k] for k in sample if k[0]==r) for r in sorted({k[0] for k in sample},key=int)},\'outputs_sha256\':{n:sha256(args.out/n) for n in (\'submission_best_088078.csv\',\'submission_education.csv\')},\'weather_sha256\':sha256(args.weather),\'libraries\':previous[\'libraries\']}\n    (args.out/\'education_report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\')\n    print(\'Selected:\',selected,\'changed rows:\',sum(report[\'changed_rows\'].values()),flush=True)\n    return report\n\nif __name__==\'__main__\':\n    from ml.forecast.education_competition import main as run_main\n    run_main()\n', 'ml/forecast/education_features.py': '"""School/daycare geographic exposure × explicitly approximate calendar scenarios."""\nimport json,math\nfrom datetime import date\nfrom pathlib import Path\nfrom .core import effective_weekday\nREGISTRY=json.loads(Path(__file__).with_name(\'education_registry.json\').read_text())\nSOURCES={\n \'quarter_2025_example\':\'https://petrovskaja-school.ru/files/education/godovoy_kalendarnyi_uchebnyi_grafic_na_2024_2025.pdf\',\n \'modular_2025_example\':\'https://st.educom.ru/eduoffices/gateways/get_file.php?id=%7BBF05FA0D-636E-CF38-639D-9C753CE71F23%7D&name=%D0%9Frotokol-16.01-03.pdf\',\n \'moscow_recommendations_2025_2026\':\'https://old-school.online/docs/op/holidays.pdf\'}\n# Separate scenario flags. No claim that any particular nearby school follows one.\nQUARTER=[(\'2025-01-01\',\'2025-01-08\'),(\'2025-03-29\',\'2025-04-06\'),(\'2025-05-27\',\'2025-08-31\'),(\'2025-10-25\',\'2025-11-02\'),(\'2025-12-31\',\'2025-12-31\')]\nMODULAR=[(\'2025-01-01\',\'2025-01-08\'),(\'2025-02-15\',\'2025-02-24\'),(\'2025-04-05\',\'2025-04-13\'),(\'2025-05-27\',\'2025-08-31\'),(\'2025-10-04\',\'2025-10-12\'),(\'2025-11-15\',\'2025-11-23\'),(\'2025-12-31\',\'2025-12-31\')]\ndef closed(d,spans):return any(date.fromisoformat(a)<=d<=date.fromisoformat(b) for a,b in spans)\ndef calendar_features(key):\n    _,d,h=key\n    if d.year!=2025:raise ValueError(\'Educational calendar defined only for 2025\')\n    work=float(effective_weekday(d)<5)\n    q=work*(not closed(d,QUARTER));m=work*(not closed(d,MODULAR))\n    # Daycare workday proxy persists through school vacations; these are hypotheses, not opening hours.\n    return [q,m,work,q*(6<=h<=9),m*(6<=h<=9),q*(12<=h<=16),m*(12<=h<=16),work*(6<=h<=9),work*(16<=h<=19)]\ndef exposure_features(key):\n    p=REGISTRY[\'routes\'].get(key[0],{\'school\':0,\'kindergarten\':0});s=math.log1p(p[\'school\']);k=math.log1p(p[\'kindergarten\']);c=calendar_features(key)\n    return [s,k]+[s*c[i] for i in (0,1,3,4,5,6)]+[k*c[i] for i in (2,7,8)]\n', 'ml/forecast/education_registry.json': '{\n  "source": "https://www.openstreetmap.org/copyright",\n  "license": "ODbL; © OpenStreetMap contributors",\n  "api": "https://overpass-api.de/api/interpreter",\n  "osm_snapshot": "2025-01-01T00:00:00Z",\n  "osm_sha256": "fd5db1a4376e94a3d7ec50e23042ffd733ae0a6fd0e83ff1cdba337cb26ae4b5",\n  "query": "[out:json][timeout:45][date:\\"2025-01-01T00:00:00Z\\"];nwr[\\"amenity\\"~\\"^(school|kindergarten)$\\"](55.584,37.44,55.898,37.834);out center tags;",\n  "gtfs_sha256": "dec3ec0ccc2efb7f1327ee6f26a0636a4a271494490e6debedcb2c1ccc579e76",\n  "radius_m": 500.0,\n  "raw_objects": 3828,\n  "retained_objects": 3717,\n  "deduplicated_ids": [\n    "relation/1284304",\n    "relation/1285682",\n    "relation/3612075",\n    "relation/7103027",\n    "relation/9897400",\n    "relation/13051361",\n    "relation/13496987",\n    "relation/17960168",\n    "way/29325329",\n    "way/29446296",\n    "way/31025605",\n    "way/37082736",\n    "way/39629328",\n    "way/40057242",\n    "way/52073302",\n    "way/96057466",\n    "way/96283968",\n    "way/96691325",\n    "way/96844626",\n    "way/98925503",\n    "way/98925579",\n    "way/105041950",\n    "way/113776912",\n    "way/117196204",\n    "way/118593455",\n    "way/129624462",\n    "way/129717682",\n    "way/130122951",\n    "way/136752940",\n    "way/154000080",\n    "way/155217711",\n    "way/162959648",\n    "way/171616601",\n    "way/179615797",\n    "way/223512281",\n    "way/227635648",\n    "way/234903720",\n    "way/282238823",\n    "way/292071887",\n    "way/304423709",\n    "way/347944650",\n    "way/377708696",\n    "way/386459645",\n    "way/398512858",\n    "way/425074014",\n    "way/459404197",\n    "way/460009813",\n    "way/460031920",\n    "way/488268384",\n    "way/568604304",\n    "way/581845124",\n    "way/591773583",\n    "way/596216286",\n    "way/596360350",\n    "way/596360354",\n    "way/596360356",\n    "way/596370524",\n    "way/632589852",\n    "way/635814043",\n    "way/641355975",\n    "way/647183330",\n    "way/650555423",\n    "way/655682580",\n    "way/656685218",\n    "way/656685220",\n    "way/690243296",\n    "way/707938467",\n    "way/881232953",\n    "way/1042681701",\n    "way/1130649958",\n    "way/1130649959",\n    "way/1144177518",\n    "way/1154845324",\n    "way/1155008743",\n    "way/1193335196",\n    "way/1194067581",\n    "way/1194073085",\n    "way/1211187706",\n    "way/1214119136",\n    "node/1094884834",\n    "node/1373636953",\n    "node/1373694625",\n    "node/1373755232",\n    "node/1373766030",\n    "node/1373778974",\n    "node/1428214514",\n    "node/1429367710",\n    "node/1456659417",\n    "node/2262304592",\n    "node/2273992710",\n    "node/2273992711",\n    "node/2942619205",\n    "node/2986435369",\n    "node/4381977804",\n    "node/4520379715",\n    "node/4752645623",\n    "node/5180531055",\n    "node/5199351212",\n    "node/5238076005",\n    "node/5732129330",\n    "node/6331788808",\n    "node/6413499948",\n    "node/6413499949",\n    "node/6794888867",\n    "node/7218108678",\n    "node/7223599185",\n    "node/7241782461",\n    "node/8154382975",\n    "node/8752581937",\n    "node/8752581938",\n    "node/9910450170"\n  ],\n  "routes": {\n    "17": {\n      "school": 27,\n      "kindergarten": 35,\n      "poi_ids": [\n        "relation/286862",\n        "relation/1270060",\n        "relation/1490894",\n        "relation/1535976",\n        "relation/1621860",\n        "relation/2108267",\n        "relation/2432227",\n        "relation/2432228",\n        "relation/2432229",\n        "relation/2589367",\n        "relation/3015734",\n        "relation/3015735",\n        "relation/3428535",\n        "relation/3428539",\n        "relation/5699016",\n        "relation/6010660",\n        "relation/9068415",\n        "relation/9068416",\n        "relation/9070447",\n        "relation/9070448",\n        "relation/9071344",\n        "relation/11550302",\n        "relation/12223764",\n        "relation/12223765",\n        "relation/17789214",\n        "way/48757559",\n        "way/59370210",\n        "way/59404034",\n        "way/59409572",\n        "way/60093986",\n        "way/69728519",\n        "way/85910107",\n        "way/102317311",\n        "way/102317313",\n        "way/102326794",\n        "way/117368268",\n        "way/226915647",\n        "way/226915648",\n        "way/227478963",\n        "way/227488072",\n        "way/241167631",\n        "way/255945625",\n        "way/270445570",\n        "way/302728535",\n        "way/325989403",\n        "way/378894408",\n        "way/378894409",\n        "way/400804215",\n        "way/445522765",\n        "way/445522767",\n        "way/469126286",\n        "way/499308094",\n        "way/532936533",\n        "way/532939500",\n        "way/618533762",\n        "way/709265694",\n        "way/709265695",\n        "way/709548475",\n        "way/867853165",\n        "way/1163425107",\n        "node/3794027238",\n        "node/4575871390"\n      ],\n      "known_opening_hours": 1\n    },\n    "1": {\n      "school": 33,\n      "kindergarten": 43,\n      "poi_ids": [\n        "relation/2074057",\n        "relation/2074058",\n        "relation/5713567",\n        "relation/5713568",\n        "relation/9310365",\n        "relation/10165007",\n        "relation/13211681",\n        "relation/13211682",\n        "relation/13372228",\n        "relation/13372229",\n        "relation/13524468",\n        "relation/13524469",\n        "relation/13606724",\n        "way/48227462",\n        "way/51812359",\n        "way/51832350",\n        "way/51832351",\n        "way/52015581",\n        "way/76593145",\n        "way/88609750",\n        "way/93181279",\n        "way/97716798",\n        "way/97730991",\n        "way/97730993",\n        "way/107728025",\n        "way/111538985",\n        "way/111538986",\n        "way/115613768",\n        "way/129475181",\n        "way/146603937",\n        "way/146603939",\n        "way/154047644",\n        "way/154047649",\n        "way/154331611",\n        "way/154331612",\n        "way/157210046",\n        "way/183948096",\n        "way/205680825",\n        "way/260177247",\n        "way/284385908",\n        "way/284401169",\n        "way/286082075",\n        "way/287304987",\n        "way/288601028",\n        "way/288609547",\n        "way/293518110",\n        "way/295308773",\n        "way/330781480",\n        "way/330783273",\n        "way/333465880",\n        "way/367879924",\n        "way/377728857",\n        "way/390483185",\n        "way/396923766",\n        "way/399595392",\n        "way/399595394",\n        "way/400514902",\n        "way/401282468",\n        "way/401282470",\n        "way/404373621",\n        "way/404373622",\n        "way/404379796",\n        "way/404896628",\n        "way/404896629",\n        "way/404896633",\n        "way/405311002",\n        "way/405413167",\n        "way/461030275",\n        "way/498121462",\n        "way/498175708",\n        "way/498314470",\n        "way/595122415",\n        "way/608127845",\n        "way/693839893",\n        "way/711304803",\n        "way/711304806"\n      ],\n      "known_opening_hours": 0\n    },\n    "12": {\n      "school": 51,\n      "kindergarten": 65,\n      "poi_ids": [\n        "relation/2018554",\n        "relation/2574596",\n        "relation/5713478",\n        "relation/6847000",\n        "relation/7329513",\n        "relation/7329517",\n        "relation/7334650",\n        "relation/7334651",\n        "relation/7814787",\n        "relation/8897320",\n        "relation/8897321",\n        "relation/8897322",\n        "relation/11186468",\n        "relation/11284691",\n        "relation/11342614",\n        "relation/12661655",\n        "relation/14237997",\n        "relation/15284663",\n        "relation/15285093",\n        "relation/18244733",\n        "way/23768961",\n        "way/27575988",\n        "way/29601128",\n        "way/43321808",\n        "way/47572148",\n        "way/48346415",\n        "way/49140332",\n        "way/50059100",\n        "way/50072669",\n        "way/52462301",\n        "way/52462352",\n        "way/52462356",\n        "way/52463398",\n        "way/64612074",\n        "way/78273780",\n        "way/78444429",\n        "way/78641368",\n        "way/79534444",\n        "way/80033791",\n        "way/93527752",\n        "way/94054910",\n        "way/105137093",\n        "way/128531064",\n        "way/128531081",\n        "way/129318472",\n        "way/129408541",\n        "way/129408542",\n        "way/129416914",\n        "way/129584044",\n        "way/129585077",\n        "way/130278483",\n        "way/130819428",\n        "way/133738328",\n        "way/145111581",\n        "way/145111582",\n        "way/145111583",\n        "way/150964760",\n        "way/168840466",\n        "way/186311028",\n        "way/190098593",\n        "way/190118996",\n        "way/196094633",\n        "way/204059721",\n        "way/297179085",\n        "way/309413338",\n        "way/362529929",\n        "way/362924386",\n        "way/372687745",\n        "way/395166265",\n        "way/395166267",\n        "way/395236550",\n        "way/395236552",\n        "way/395236553",\n        "way/399276902",\n        "way/425202129",\n        "way/425304688",\n        "way/429427494",\n        "way/444428136",\n        "way/444428137",\n        "way/450112055",\n        "way/454469587",\n        "way/461358380",\n        "way/482435459",\n        "way/524737858",\n        "way/640129189",\n        "way/824290373",\n        "way/878383228",\n        "way/878979417",\n        "way/878979512",\n        "way/1198830522",\n        "way/1285516011",\n        "node/1252450767",\n        "node/1444803462",\n        "node/1444819629",\n        "node/1456162724",\n        "node/1472892444",\n        "node/1504923799",\n        "node/1504923815",\n        "node/1588932818",\n        "node/1602410893",\n        "node/1719646868",\n        "node/1858733747",\n        "node/2146273545",\n        "node/2661667230",\n        "node/4243926389",\n        "node/4588995214",\n        "node/4792431751",\n        "node/5199351213",\n        "node/5273631003",\n        "node/5292569524",\n        "node/5305719149",\n        "node/6866820508",\n        "node/8920785773",\n        "node/9880328119",\n        "node/11085165913",\n        "node/11085236054"\n      ],\n      "known_opening_hours": 4\n    },\n    "11": {\n      "school": 41,\n      "kindergarten": 48,\n      "poi_ids": [\n        "relation/1262955",\n        "relation/1346265",\n        "relation/2108267",\n        "relation/2574596",\n        "relation/2752454",\n        "relation/7814787",\n        "relation/9068415",\n        "relation/9068416",\n        "relation/9070447",\n        "relation/9070448",\n        "relation/9071344",\n        "relation/11550302",\n        "relation/11679771",\n        "relation/11778070",\n        "relation/11805313",\n        "relation/14237997",\n        "relation/17087636",\n        "relation/17087637",\n        "relation/17789214",\n        "relation/18244733",\n        "way/23768961",\n        "way/28858683",\n        "way/50059100",\n        "way/50072669",\n        "way/51239305",\n        "way/51239411",\n        "way/51422928",\n        "way/64612074",\n        "way/78273780",\n        "way/78444429",\n        "way/78641368",\n        "way/79534444",\n        "way/94054910",\n        "way/96645733",\n        "way/105704716",\n        "way/128531064",\n        "way/128531081",\n        "way/133738328",\n        "way/148175913",\n        "way/154363995",\n        "way/154363996",\n        "way/154544725",\n        "way/155653757",\n        "way/168840466",\n        "way/186311028",\n        "way/190098593",\n        "way/190118996",\n        "way/204059721",\n        "way/302728535",\n        "way/362529929",\n        "way/362924386",\n        "way/395166265",\n        "way/395166267",\n        "way/395236550",\n        "way/395236552",\n        "way/395236553",\n        "way/425202129",\n        "way/425304688",\n        "way/450112055",\n        "way/454469587",\n        "way/461358380",\n        "way/504270193",\n        "way/520900483",\n        "way/524737858",\n        "way/555999588",\n        "way/555999622",\n        "way/709206068",\n        "way/711588628",\n        "way/867853165",\n        "way/878383228",\n        "way/878979417",\n        "way/878979512",\n        "node/1444803462",\n        "node/1444819629",\n        "node/1456162724",\n        "node/1472892444",\n        "node/1504923799",\n        "node/1504923815",\n        "node/1588932818",\n        "node/1719646868",\n        "node/4243926389",\n        "node/4393094689",\n        "node/4959152543",\n        "node/5199351213",\n        "node/5273631003",\n        "node/5305719149",\n        "node/6866820508",\n        "node/10044484701",\n        "node/11780110793"\n      ],\n      "known_opening_hours": 2\n    },\n    "25": {\n      "school": 16,\n      "kindergarten": 15,\n      "poi_ids": [\n        "relation/1262955",\n        "relation/1346265",\n        "relation/2108267",\n        "relation/7712039",\n        "relation/9068415",\n        "relation/9068416",\n        "relation/9070447",\n        "relation/9070448",\n        "relation/9071344",\n        "relation/10524778",\n        "relation/10524779",\n        "relation/10698703",\n        "relation/11126283",\n        "relation/11152601",\n        "relation/11257412",\n        "relation/11331272",\n        "relation/11331273",\n        "relation/11550302",\n        "relation/11778070",\n        "relation/11805313",\n        "relation/17789214",\n        "way/96645733",\n        "way/149908800",\n        "way/166477542",\n        "way/179615791",\n        "way/187448839",\n        "way/302728535",\n        "way/555999588",\n        "way/555999622",\n        "way/867853165",\n        "node/5465097276"\n      ],\n      "known_opening_hours": 0\n    },\n    "50": {\n      "school": 29,\n      "kindergarten": 37,\n      "poi_ids": [\n        "relation/1476090",\n        "relation/4729216",\n        "relation/6319979",\n        "relation/6476664",\n        "relation/6505043",\n        "relation/7103030",\n        "relation/7238772",\n        "relation/7268814",\n        "relation/7595822",\n        "relation/7895065",\n        "relation/9656985",\n        "relation/11186468",\n        "relation/11284691",\n        "relation/11370208",\n        "relation/11855555",\n        "relation/11997800",\n        "relation/12003323",\n        "relation/12476264",\n        "relation/13188995",\n        "relation/13470035",\n        "relation/13802230",\n        "relation/14956586",\n        "relation/14956589",\n        "relation/14977370",\n        "relation/15046320",\n        "relation/15285093",\n        "relation/15326688",\n        "relation/15451606",\n        "relation/15968755",\n        "relation/16270432",\n        "relation/16409205",\n        "relation/17057701",\n        "relation/17824771",\n        "relation/17940119",\n        "way/46740734",\n        "way/49908065",\n        "way/49908070",\n        "way/50548821",\n        "way/125849340",\n        "way/129760307",\n        "way/139617929",\n        "way/172288880",\n        "way/172839742",\n        "way/188079728",\n        "way/189136645",\n        "way/189136646",\n        "way/192393322",\n        "way/196952681",\n        "way/197907188",\n        "way/372687745",\n        "way/482435459",\n        "way/487787682",\n        "way/523390513",\n        "way/539236815",\n        "way/684216820",\n        "way/824290373",\n        "node/1429376733",\n        "node/1432138391",\n        "node/1858733747",\n        "node/1867012052",\n        "node/4802013987",\n        "node/5305719158",\n        "node/8920785773",\n        "node/9880328119",\n        "node/11085165913",\n        "node/11085236054"\n      ],\n      "known_opening_hours": 4\n    },\n    "7": {\n      "school": 43,\n      "kindergarten": 57,\n      "poi_ids": [\n        "relation/1283096",\n        "relation/1403786",\n        "relation/1407665",\n        "relation/1708757",\n        "relation/1708763",\n        "relation/2752454",\n        "relation/4719459",\n        "relation/4729216",\n        "relation/6505043",\n        "relation/7103029",\n        "relation/7238772",\n        "relation/7268814",\n        "relation/7595822",\n        "relation/7895065",\n        "relation/9656985",\n        "relation/10524778",\n        "relation/10524779",\n        "relation/10524782",\n        "relation/10698703",\n        "relation/11208335",\n        "relation/11257412",\n        "relation/11268549",\n        "relation/11268551",\n        "relation/11370208",\n        "relation/11679771",\n        "relation/11855555",\n        "relation/12106667",\n        "relation/12476264",\n        "relation/13188995",\n        "relation/13470035",\n        "relation/13936495",\n        "relation/14680298",\n        "relation/15046320",\n        "relation/15968755",\n        "relation/16270432",\n        "relation/17057701",\n        "relation/17087636",\n        "relation/17087637",\n        "relation/17940119",\n        "way/28858683",\n        "way/37927642",\n        "way/51239305",\n        "way/51239411",\n        "way/51422928",\n        "way/51480004",\n        "way/96300065",\n        "way/101264114",\n        "way/101264120",\n        "way/105704716",\n        "way/115707848",\n        "way/125849340",\n        "way/141643616",\n        "way/141643619",\n        "way/141643622",\n        "way/141643633",\n        "way/141643636",\n        "way/141643639",\n        "way/148175913",\n        "way/149908800",\n        "way/153236806",\n        "way/154000034",\n        "way/154000076",\n        "way/154000082",\n        "way/154000083",\n        "way/154343006",\n        "way/154363995",\n        "way/154363996",\n        "way/154544725",\n        "way/155551279",\n        "way/155653703",\n        "way/155653757",\n        "way/166477542",\n        "way/172288880",\n        "way/172839742",\n        "way/185426891",\n        "way/188079728",\n        "way/189136645",\n        "way/189136646",\n        "way/192393322",\n        "way/196952681",\n        "way/197907188",\n        "way/487787682",\n        "way/504270193",\n        "way/520900483",\n        "way/523390513",\n        "way/539236815",\n        "way/709206068",\n        "way/711588628",\n        "way/813559738",\n        "node/1427996071",\n        "node/1429376733",\n        "node/1432138391",\n        "node/4393094689",\n        "node/4959152543",\n        "node/5113395065",\n        "node/5334698649",\n        "node/5465097276",\n        "node/7257951974",\n        "node/10044484701",\n        "node/11780110793"\n      ],\n      "known_opening_hours": 3\n    },\n    "28": {\n      "school": 15,\n      "kindergarten": 20,\n      "poi_ids": [\n        "relation/1656086",\n        "relation/1697080",\n        "relation/1819043",\n        "relation/6376832",\n        "relation/6702161",\n        "relation/8119012",\n        "relation/8119013",\n        "relation/17232824",\n        "relation/17695569",\n        "way/27800688",\n        "way/27810681",\n        "way/27810687",\n        "way/37943640",\n        "way/52456628",\n        "way/59605079",\n        "way/66860549",\n        "way/85641529",\n        "way/103244527",\n        "way/105589602",\n        "way/106406165",\n        "way/108201816",\n        "way/108315230",\n        "way/108323089",\n        "way/108446906",\n        "way/114533462",\n        "way/114540677",\n        "way/120831644",\n        "way/146398902",\n        "way/146696317",\n        "way/244039366",\n        "way/305671071",\n        "way/643329688",\n        "way/952069499",\n        "node/8752557700",\n        "node/9687922817"\n      ],\n      "known_opening_hours": 1\n    },\n    "26": {\n      "school": 35,\n      "kindergarten": 47,\n      "poi_ids": [\n        "relation/6890548",\n        "relation/7220450",\n        "relation/7738645",\n        "relation/13176656",\n        "relation/13983628",\n        "relation/14606511",\n        "relation/14926150",\n        "relation/18134842",\n        "relation/18178325",\n        "relation/18212679",\n        "way/29127999",\n        "way/29128001",\n        "way/35977653",\n        "way/35977707",\n        "way/37030105",\n        "way/46657618",\n        "way/48399916",\n        "way/51684041",\n        "way/51684048",\n        "way/81950814",\n        "way/82137808",\n        "way/89297039",\n        "way/96447491",\n        "way/96447494",\n        "way/102781292",\n        "way/104635222",\n        "way/108715365",\n        "way/112390777",\n        "way/112390793",\n        "way/112390798",\n        "way/112390800",\n        "way/130630559",\n        "way/152795922",\n        "way/158895386",\n        "way/168744868",\n        "way/170304996",\n        "way/173206552",\n        "way/184132673",\n        "way/184132674",\n        "way/191905193",\n        "way/194309723",\n        "way/234744383",\n        "way/277070499",\n        "way/306239412",\n        "way/306239414",\n        "way/316630443",\n        "way/316835898",\n        "way/329889832",\n        "way/334658366",\n        "way/355662159",\n        "way/373614960",\n        "way/373734133",\n        "way/374271122",\n        "way/382029609",\n        "way/405103977",\n        "way/425074005",\n        "way/440610765",\n        "way/496457582",\n        "way/586808383",\n        "way/608667848",\n        "way/710644450",\n        "way/722407298",\n        "way/977876539",\n        "way/1122007526",\n        "way/1235854567",\n        "node/416876296",\n        "node/954912765",\n        "node/1277580872",\n        "node/1373669698",\n        "node/1373669718",\n        "node/1373669722",\n        "node/1373669725",\n        "node/1373669735",\n        "node/1373669736",\n        "node/1373768269",\n        "node/1373768284",\n        "node/1373768288",\n        "node/1427855785",\n        "node/1428076955",\n        "node/3741570155",\n        "node/3741570172",\n        "node/10810258626"\n      ],\n      "known_opening_hours": 2\n    }\n  },\n  "poi": [\n    {\n      "id": "relation/36980",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 6",\n      "lat": 55.7782163,\n      "lon": 37.5292095,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/62986",\n      "kind": "school",\n      "name": "Школа № 1141",\n      "lat": 55.6435543,\n      "lon": 37.7188551,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/62987",\n      "kind": "school",\n      "name": "Школа №499",\n      "lat": 55.643592,\n      "lon": 37.7205177,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/286862",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8536875,\n      "lon": 37.6593748,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/337465",\n      "kind": "school",\n      "name": "ГБОУ Школа в Капотне (Учебный корпус \\"Боровицкий\\")",\n      "lat": 55.631654,\n      "lon": 37.7968531,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/358784",\n      "kind": "school",\n      "name": "ГБОУ школа № 1363, корпус № 1",\n      "lat": 55.7077963,\n      "lon": 37.8237939,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/359070",\n      "kind": "school",\n      "name": "Школа-интернат им. Ю.В.Никулина",\n      "lat": 55.7043499,\n      "lon": 37.7902112,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1245766",\n      "kind": "school",\n      "name": "Центр образования №1470",\n      "lat": 55.8139487,\n      "lon": 37.644921,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1249599",\n      "kind": "school",\n      "name": "Центр педагогического мастерства",\n      "lat": 55.7898209,\n      "lon": 37.6212747,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1249600",\n      "kind": "kindergarten",\n      "name": "Детский сад №1189",\n      "lat": 55.7892559,\n      "lon": 37.619603,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1262955",\n      "kind": "school",\n      "name": "Школа № 1503",\n      "lat": 55.8181393,\n      "lon": 37.6628376,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1270058",\n      "kind": "school",\n      "name": "Школа №1095",\n      "lat": 55.8721435,\n      "lon": 37.6703797,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1270060",\n      "kind": "school",\n      "name": "Школа №757",\n      "lat": 55.8708491,\n      "lon": 37.6696544,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1270065",\n      "kind": "school",\n      "name": "Школа №1381",\n      "lat": 55.8729153,\n      "lon": 37.6912967,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1280704",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 12",\n      "lat": 55.8527736,\n      "lon": 37.6536023,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1280705",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 5",\n      "lat": 55.852089,\n      "lon": 37.6542093,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1283096",\n      "kind": "kindergarten",\n      "name": "СПДО № 1 школы № 1784",\n      "lat": 55.7779019,\n      "lon": 37.5762138,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1284256",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7798937,\n      "lon": 37.4917759,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1284257",\n      "kind": "school",\n      "name": "Школа №100",\n      "lat": 55.7743882,\n      "lon": 37.4855824,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1284265",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7795023,\n      "lon": 37.4850428,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1285664",\n      "kind": "school",\n      "name": "Гимназия №1522",\n      "lat": 55.7730106,\n      "lon": 37.4822545,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1285674",\n      "kind": "school",\n      "name": "ГБОУ СОШ Школа №1010 Корпус №3",\n      "lat": 55.7708868,\n      "lon": 37.4788264,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1285680",\n      "kind": "school",\n      "name": "",\n      "lat": 55.770442,\n      "lon": 37.4841783,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1285729",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7721677,\n      "lon": 37.4790474,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1317290",\n      "kind": "school",\n      "name": "Школа-интернат №11",\n      "lat": 55.7696609,\n      "lon": 37.4715224,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1338486",\n      "kind": "school",\n      "name": "Школа «Потенциал»",\n      "lat": 55.8051799,\n      "lon": 37.6405551,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1346265",\n      "kind": "school",\n      "name": "Школа № 1503",\n      "lat": 55.8186977,\n      "lon": 37.6597964,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1380084",\n      "kind": "kindergarten",\n      "name": "Детский сад №1824",\n      "lat": 55.7724937,\n      "lon": 37.4855377,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1380144",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7792391,\n      "lon": 37.4983553,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1403786",\n      "kind": "school",\n      "name": "Лицей № 1574. ОП № 1",\n      "lat": 55.7745603,\n      "lon": 37.5905282,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1407665",\n      "kind": "school",\n      "name": "Центр образования №1447",\n      "lat": 55.7797874,\n      "lon": 37.5880813,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1412042",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6761828,\n      "lon": 37.6954364,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1412043",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6773853,\n      "lon": 37.7000029,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1412051",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6809905,\n      "lon": 37.7061743,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1421038",\n      "kind": "kindergarten",\n      "name": "Школа № 2054. Дошкольный корпус № 3",\n      "lat": 55.7697175,\n      "lon": 37.5995896,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1421089",\n      "kind": "school",\n      "name": "Школа № 1574. Корпус Старопименовский",\n      "lat": 55.7692423,\n      "lon": 37.600585,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1427143",\n      "kind": "kindergarten",\n      "name": "Детский сад №513",\n      "lat": 55.7707493,\n      "lon": 37.6032283,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1450020",\n      "kind": "school",\n      "name": "ГБОУ Школа в Капотне (Учебный корпус \\"Камергерский\\")",\n      "lat": 55.6301933,\n      "lon": 37.7967704,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1459232",\n      "kind": "kindergarten",\n      "name": "Детский сад №2140",\n      "lat": 55.8146776,\n      "lon": 37.6445515,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1459233",\n      "kind": "kindergarten",\n      "name": "Детский сад №1320",\n      "lat": 55.8153467,\n      "lon": 37.6447063,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1459416",\n      "kind": "kindergarten",\n      "name": "Детский сад №1342 «Солнышко»",\n      "lat": 55.8150162,\n      "lon": 37.6505895,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1476090",\n      "kind": "school",\n      "name": "Школа «Содружество». Учебный корпус №1",\n      "lat": 55.7574312,\n      "lon": 37.7011875,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1490894",\n      "kind": "school",\n      "name": "Детская хоровая школа «Весна»",\n      "lat": 55.8706346,\n      "lon": 37.6480796,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1514803",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8776422,\n      "lon": 37.6718289,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1516929",\n      "kind": "school",\n      "name": "Школа №309",\n      "lat": 55.8632082,\n      "lon": 37.6793466,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1516932",\n      "kind": "school",\n      "name": "",\n      "lat": 55.866711,\n      "lon": 37.677224,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1535976",\n      "kind": "school",\n      "name": "Школа №281",\n      "lat": 55.8665021,\n      "lon": 37.6703141,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1587283",\n      "kind": "school",\n      "name": "Центр образования №1816",\n      "lat": 55.7367864,\n      "lon": 37.4937056,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1601011",\n      "kind": "kindergarten",\n      "name": "Школа № 1239, дошкольное отделение",\n      "lat": 55.7612391,\n      "lon": 37.5953884,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1607477",\n      "kind": "kindergarten",\n      "name": "Школа № 1950 с углубленным изучением отдельных предметов. Дошкольное отделение",\n      "lat": 55.7717618,\n      "lon": 37.5855241,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1621860",\n      "kind": "kindergarten",\n      "name": "Школа №1499 (дошкольное отделение №6)",\n      "lat": 55.8356163,\n      "lon": 37.6644287,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1640597",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8356414,\n      "lon": 37.6660974,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1642492",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6767585,\n      "lon": 37.5112879,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1650088",\n      "kind": "school",\n      "name": "Школа №426",\n      "lat": 55.6245364,\n      "lon": 37.7192144,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1655777",\n      "kind": "school",\n      "name": "Центр образования №117",\n      "lat": 55.6693832,\n      "lon": 37.534134,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1656086",\n      "kind": "school",\n      "name": "Гимназия №1517",\n      "lat": 55.7802492,\n      "lon": 37.4571477,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1664990",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.770405,\n      "lon": 37.4776106,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1697080",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8064323,\n      "lon": 37.5145754,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1708757",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7728531,\n      "lon": 37.5820939,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1708763",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 1 «Звёздочка»",\n      "lat": 55.7733392,\n      "lon": 37.5809738,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1751600",\n      "kind": "school",\n      "name": "Центр образования №1441«Бронная слобода»",\n      "lat": 55.7620395,\n      "lon": 37.5974549,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1751601",\n      "kind": "kindergarten",\n      "name": "Детский сад №1144",\n      "lat": 55.7603816,\n      "lon": 37.5956879,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1752165",\n      "kind": "kindergarten",\n      "name": "Детский сад №3",\n      "lat": 55.7630146,\n      "lon": 37.5994673,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1757153",\n      "kind": "kindergarten",\n      "name": "Детский сад №1684",\n      "lat": 55.8721575,\n      "lon": 37.6815924,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1757154",\n      "kind": "school",\n      "name": "Школа №298",\n      "lat": 55.8724788,\n      "lon": 37.6805273,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763003",\n      "kind": "school",\n      "name": "Школа №1929",\n      "lat": 55.6353589,\n      "lon": 37.743337,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763004",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6345315,\n      "lon": 37.743037,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763005",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6302235,\n      "lon": 37.6521784,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763006",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6320931,\n      "lon": 37.6519736,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763007",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6309712,\n      "lon": 37.6520773,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1819043",\n      "kind": "school",\n      "name": "Школа № 149 имени Героя Советского Союза Ю. Н. Зыкова",\n      "lat": 55.8057513,\n      "lon": 37.5009076,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1819916",\n      "kind": "school",\n      "name": "Школа №648",\n      "lat": 55.8554885,\n      "lon": 37.5022595,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1836516",\n      "kind": "school",\n      "name": "Школа №1043",\n      "lat": 55.6469914,\n      "lon": 37.7488536,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1840315",\n      "kind": "school",\n      "name": "Центр образования №1685",\n      "lat": 55.7475168,\n      "lon": 37.6526198,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1841866",\n      "kind": "kindergarten",\n      "name": "Детский сад №2324",\n      "lat": 55.6478415,\n      "lon": 37.7509163,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1873600",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7716519,\n      "lon": 37.4844162,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1881807",\n      "kind": "kindergarten",\n      "name": "Детский сад №2709 \\"Совёнок\\"",\n      "lat": 55.7745419,\n      "lon": 37.4902214,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930228",\n      "kind": "kindergarten",\n      "name": "Детский сад Феникс",\n      "lat": 55.5938026,\n      "lon": 37.6695414,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930229",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №8",\n      "lat": 55.5936898,\n      "lon": 37.6680902,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930230",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №11",\n      "lat": 55.591127,\n      "lon": 37.6681777,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930231",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №10",\n      "lat": 55.5910223,\n      "lon": 37.6667322,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930233",\n      "kind": "school",\n      "name": "Школа № 902 \\"Диалог\\", здание №3",\n      "lat": 55.5913274,\n      "lon": 37.6700217,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930234",\n      "kind": "school",\n      "name": "Школа № 902 \\"Диалог\\", здание №2",\n      "lat": 55.593533,\n      "lon": 37.6660668,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1998795",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa01368",\n      "lat": 55.6755258,\n      "lon": 37.6900136,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2018554",\n      "kind": "school",\n      "name": "Школа №1274",\n      "lat": 55.7233969,\n      "lon": 37.67137,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2074057",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Открытие»",\n      "lat": 55.6295375,\n      "lon": 37.5957873,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2074058",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус школьного образования №2",\n      "lat": 55.6300278,\n      "lon": 37.5945952,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2102794",\n      "kind": "school",\n      "name": "Школа №1314",\n      "lat": 55.7130695,\n      "lon": 37.7891315,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2102795",\n      "kind": "school",\n      "name": "Школа № 2090",\n      "lat": 55.713451,\n      "lon": 37.7879981,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2108267",\n      "kind": "school",\n      "name": "Школа №1415",\n      "lat": 55.8246563,\n      "lon": 37.6247623,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2140115",\n      "kind": "school",\n      "name": "ГБОУ школа № 507, здание № 3",\n      "lat": 55.6673747,\n      "lon": 37.6542195,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2145798",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.708118,\n      "lon": 37.4609493,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2145799",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7078435,\n      "lon": 37.4619846,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2145800",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7075471,\n      "lon": 37.4630171,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2168335",\n      "kind": "kindergarten",\n      "name": "Детский сад №1541",\n      "lat": 55.8908275,\n      "lon": 37.6768881,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2168336",\n      "kind": "kindergarten",\n      "name": "Детский сад №1495",\n      "lat": 55.8899927,\n      "lon": 37.6769625,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2183431",\n      "kind": "school",\n      "name": "Школа № 171 Школьное отделение № 2",\n      "lat": 55.7194291,\n      "lon": 37.5750929,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2183433",\n      "kind": "kindergarten",\n      "name": "Детский сад №1155",\n      "lat": 55.7176475,\n      "lon": 37.575196,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2235992",\n      "kind": "school",\n      "name": "Школа № 305",\n      "lat": 55.8755815,\n      "lon": 37.5798725,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2236017",\n      "kind": "kindergarten",\n      "name": "Детский сад №172",\n      "lat": 55.8058568,\n      "lon": 37.822233,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2236018",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №6 \\"Цветик-семицветик\\"",\n      "lat": 55.8056378,\n      "lon": 37.8206519,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2248443",\n      "kind": "school",\n      "name": "Школа-интернат для глухих детей №65",\n      "lat": 55.7162602,\n      "lon": 37.7377174,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2248444",\n      "kind": "school",\n      "name": "Школа-интернат №105",\n      "lat": 55.7165616,\n      "lon": 37.7402113,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2257315",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Дошкольное здание",\n      "lat": 55.816185,\n      "lon": 37.4961085,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2307304",\n      "kind": "kindergarten",\n      "name": "Детский сад №1633",\n      "lat": 55.7654883,\n      "lon": 37.490507,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2307305",\n      "kind": "school",\n      "name": "Школа №1114",\n      "lat": 55.7647688,\n      "lon": 37.4906759,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2378354",\n      "kind": "kindergarten",\n      "name": "ДОУ №1729",\n      "lat": 55.7668255,\n      "lon": 37.6678701,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2396447",\n      "kind": "school",\n      "name": "Школа № 996",\n      "lat": 55.6269965,\n      "lon": 37.7236231,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2425832",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР №32",\n      "lat": 55.6193083,\n      "lon": 37.7384342,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2425833",\n      "kind": "school",\n      "name": "Спецшкола №991",\n      "lat": 55.6193772,\n      "lon": 37.7395421,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2432227",\n      "kind": "kindergarten",\n      "name": "Детский сад №2420",\n      "lat": 55.8679673,\n      "lon": 37.6582333,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2432228",\n      "kind": "school",\n      "name": "ГБОУ школа № 1558 имени Росалии де Кастро",\n      "lat": 55.8681585,\n      "lon": 37.6603169,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2432229",\n      "kind": "kindergarten",\n      "name": "Детский сад №1870",\n      "lat": 55.8686113,\n      "lon": 37.6585889,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2465066",\n      "kind": "school",\n      "name": "ГБОУ школа № 507, здание № 9",\n      "lat": 55.6721224,\n      "lon": 37.6572188,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2465075",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, Радуга (здание № 6)",\n      "lat": 55.6697752,\n      "lon": 37.65829,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2574596",\n      "kind": "kindergarten",\n      "name": "Детский сад №777",\n      "lat": 55.7857976,\n      "lon": 37.734087,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2578431",\n      "kind": "kindergarten",\n      "name": "Детский сад №696",\n      "lat": 55.8751193,\n      "lon": 37.6931984,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2589367",\n      "kind": "kindergarten",\n      "name": "Детский сад №1120",\n      "lat": 55.867757,\n      "lon": 37.6684159,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2624116",\n      "kind": "kindergarten",\n      "name": "Школа №158, дошкольное отделение №5",\n      "lat": 55.8671514,\n      "lon": 37.4653512,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2624119",\n      "kind": "kindergarten",\n      "name": "Школа №158, дошкольное отделение №4",\n      "lat": 55.8664185,\n      "lon": 37.4657795,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2631944",\n      "kind": "kindergarten",\n      "name": "Центр развития творчества детей и юношества \\"Планета\\"",\n      "lat": 55.7044491,\n      "lon": 37.5789552,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2631945",\n      "kind": "school",\n      "name": "Школа № 1266",\n      "lat": 55.7048374,\n      "lon": 37.578306,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2639185",\n      "kind": "kindergarten",\n      "name": "Школа № 1266. Дошкольное отделение",\n      "lat": 55.7040403,\n      "lon": 37.5775092,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2699560",\n      "kind": "school",\n      "name": "Центр образования №1421",\n      "lat": 55.7135878,\n      "lon": 37.7530688,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2699561",\n      "kind": "school",\n      "name": "Школа №1367",\n      "lat": 55.7146528,\n      "lon": 37.7524919,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2699563",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.71317,\n      "lon": 37.7496638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2752454",\n      "kind": "school",\n      "name": "Школа №363 (начальные классы)",\n      "lat": 55.8087912,\n      "lon": 37.7119852,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2766945",\n      "kind": "school",\n      "name": "Центр образования № 1466 имени Надежды Рушевой",\n      "lat": 55.6293298,\n      "lon": 37.6749566,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2785111",\n      "kind": "school",\n      "name": "Школа №1257",\n      "lat": 55.7145546,\n      "lon": 37.6259982,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2838632",\n      "kind": "school",\n      "name": "Школа №878 отделение №2",\n      "lat": 55.6143271,\n      "lon": 37.6979107,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2880891",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7904238,\n      "lon": 37.5208434,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2880892",\n      "kind": "school",\n      "name": "Центр образования №1409",\n      "lat": 55.7840878,\n      "lon": 37.5315363,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2893419",\n      "kind": "school",\n      "name": "Школа №12",\n      "lat": 55.8846316,\n      "lon": 37.7432036,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2894192",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Звёздочка\\"",\n      "lat": 55.885443,\n      "lon": 37.7382802,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2896213",\n      "kind": "kindergarten",\n      "name": "Детский сад № 53",\n      "lat": 55.6947284,\n      "lon": 37.5746002,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2896247",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2243",\n      "lat": 55.6962045,\n      "lon": 37.561549,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2896248",\n      "kind": "school",\n      "name": "Школа № 199 (начальные классы)",\n      "lat": 55.692326,\n      "lon": 37.5657951,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3015734",\n      "kind": "kindergarten",\n      "name": "Детский сад №1255",\n      "lat": 55.875249,\n      "lon": 37.649616,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3015735",\n      "kind": "kindergarten",\n      "name": "Дом ребенка №23",\n      "lat": 55.8761197,\n      "lon": 37.6489728,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3049322",\n      "kind": "school",\n      "name": "ГБОУ школа №463, здание № 5",\n      "lat": 55.6756477,\n      "lon": 37.6705987,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "relation/3049425",\n      "kind": "school",\n      "name": "Лицей №1523",\n      "lat": 55.6773454,\n      "lon": 37.6817597,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3049511",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Дельфиненок",\n      "lat": 55.6804069,\n      "lon": 37.6860008,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3049512",\n      "kind": "school",\n      "name": "ГБОУ школа № 1527 Корпус 4",\n      "lat": 55.6808685,\n      "lon": 37.6876232,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3112266",\n      "kind": "kindergarten",\n      "name": "Детский сад №830",\n      "lat": 55.7746839,\n      "lon": 37.484466,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3113404",\n      "kind": "kindergarten",\n      "name": "Детский сад №756",\n      "lat": 55.7027047,\n      "lon": 37.8095237,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3113405",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7034962,\n      "lon": 37.8110557,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3160010",\n      "kind": "kindergarten",\n      "name": "Детский сад №2665",\n      "lat": 55.8657463,\n      "lon": 37.5296892,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3160011",\n      "kind": "kindergarten",\n      "name": "Детский сад №633",\n      "lat": 55.8663685,\n      "lon": 37.5297428,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3164563",\n      "kind": "school",\n      "name": "Школа №1213",\n      "lat": 55.8439594,\n      "lon": 37.5653435,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3175157",\n      "kind": "kindergarten",\n      "name": "ГБОУ Детский сад №9",\n      "lat": 55.7708635,\n      "lon": 37.4800582,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3201223",\n      "kind": "school",\n      "name": "Школа №1188",\n      "lat": 55.8802901,\n      "lon": 37.691191,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3305498",\n      "kind": "school",\n      "name": "Школа №396",\n      "lat": 55.7397949,\n      "lon": 37.6861473,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3325183",\n      "kind": "school",\n      "name": "Школа № 1273. Корпус школьного образования №1",\n      "lat": 55.6292478,\n      "lon": 37.5227549,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3325184",\n      "kind": "kindergarten",\n      "name": "Школа № 1273. Корпус дошкольного образования №3",\n      "lat": 55.6300119,\n      "lon": 37.5231402,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3344574",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус № 1",\n      "lat": 55.6383431,\n      "lon": 37.5280169,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3370903",\n      "kind": "school",\n      "name": "Гимназия №1570",\n      "lat": 55.7893227,\n      "lon": 37.5772352,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3425342",\n      "kind": "school",\n      "name": "Школа № 17. Школьное здание №4",\n      "lat": 55.6369476,\n      "lon": 37.5405138,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3425343",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус №3",\n      "lat": 55.6374659,\n      "lon": 37.5388157,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3428535",\n      "kind": "kindergarten",\n      "name": "Детский сад №1849",\n      "lat": 55.8845623,\n      "lon": 37.6497155,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3428539",\n      "kind": "school",\n      "name": "Детский юношеский центр",\n      "lat": 55.8838133,\n      "lon": 37.6499429,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3428541",\n      "kind": "school",\n      "name": "Школа №1380",\n      "lat": 55.8844527,\n      "lon": 37.6516384,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3498403",\n      "kind": "school",\n      "name": "Школа № 1260",\n      "lat": 55.6998705,\n      "lon": 37.5568668,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3500635",\n      "kind": "school",\n      "name": "Школа № 1273. Корпус школьного образования №2",\n      "lat": 55.6282814,\n      "lon": 37.5224043,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3509355",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус №6",\n      "lat": 55.6340578,\n      "lon": 37.5328202,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3509356",\n      "kind": "school",\n      "name": "Школа № 17. Школьное здание №2",\n      "lat": 55.6345233,\n      "lon": 37.5346001,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3792745",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 8",\n      "lat": 55.7787734,\n      "lon": 37.5213099,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3834185",\n      "kind": "school",\n      "name": "Школа №1950",\n      "lat": 55.7664959,\n      "lon": 37.5847443,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3951778",\n      "kind": "school",\n      "name": "Школа Сотрудничества",\n      "lat": 55.7430626,\n      "lon": 37.6565492,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4090263",\n      "kind": "school",\n      "name": "СП \\"Гимназия\\"",\n      "lat": 55.7163312,\n      "lon": 37.4608347,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4197824",\n      "kind": "kindergarten",\n      "name": "Школа № 2123 имени Мигеля Эрнандеса. Дошкольное отделение \\"Журавушка\\"",\n      "lat": 55.7618487,\n      "lon": 37.5997866,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4217693",\n      "kind": "school",\n      "name": "Школа № 2072",\n      "lat": 55.743314,\n      "lon": 37.8032021,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4233096",\n      "kind": "school",\n      "name": "Школа им. Н.М. Карамзина. Корпус школьного образования №1 «Петербургский»",\n      "lat": 55.5923521,\n      "lon": 37.5406021,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4245552",\n      "kind": "kindergarten",\n      "name": "Детский сад №1611",\n      "lat": 55.8717965,\n      "lon": 37.5172604,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4245553",\n      "kind": "school",\n      "name": "Кадетская школа №1702",\n      "lat": 55.8649025,\n      "lon": 37.5153349,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4523021",\n      "kind": "kindergarten",\n      "name": "Школа № 1950 с углубленным изучением отдельных предметов. Дошкольное отделение",\n      "lat": 55.7671184,\n      "lon": 37.5867091,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4580283",\n      "kind": "school",\n      "name": "Школа №91",\n      "lat": 55.7541659,\n      "lon": 37.5970008,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4681982",\n      "kind": "kindergarten",\n      "name": "Детский сад №2635",\n      "lat": 55.6908751,\n      "lon": 37.572006,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4681983",\n      "kind": "school",\n      "name": "Школа №199",\n      "lat": 55.6901594,\n      "lon": 37.571638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4716520",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение №2 «Рябинушка» школы №2123",\n      "lat": 55.7567566,\n      "lon": 37.5902411,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4719459",\n      "kind": "school",\n      "name": "Школа № 1525",\n      "lat": 55.7846628,\n      "lon": 37.6701265,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4729216",\n      "kind": "school",\n      "name": "Школа № 2107. Здание \\"На Мещанской\\"",\n      "lat": 55.7769562,\n      "lon": 37.6274505,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4742204",\n      "kind": "school",\n      "name": "Школа №1234",\n      "lat": 55.753599,\n      "lon": 37.5902878,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4831993",\n      "kind": "school",\n      "name": "Спецшкола №486",\n      "lat": 55.7417619,\n      "lon": 37.6585159,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4834971",\n      "kind": "school",\n      "name": "Школа №498",\n      "lat": 55.7377925,\n      "lon": 37.6521352,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4835145",\n      "kind": "kindergarten",\n      "name": "Детский сад №3",\n      "lat": 55.7365461,\n      "lon": 37.6531505,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4835322",\n      "kind": "kindergarten",\n      "name": "Детский сад №1827",\n      "lat": 55.7355475,\n      "lon": 37.6541212,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5359641",\n      "kind": "school",\n      "name": "Школа №57 (здание №2)",\n      "lat": 55.7300903,\n      "lon": 37.5808146,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5362168",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1618",\n      "lat": 55.6933192,\n      "lon": 37.4991307,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5362169",\n      "kind": "kindergarten",\n      "name": "Центр психолого-медико-социального сопровождения \\"Раменки\\"",\n      "lat": 55.6938073,\n      "lon": 37.4999947,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5362204",\n      "kind": "kindergarten",\n      "name": "Детский сад №2164",\n      "lat": 55.7320866,\n      "lon": 37.58664,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5441013",\n      "kind": "school",\n      "name": "Школа № 7 филиал № 3",\n      "lat": 55.6794451,\n      "lon": 37.5192988,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5513696",\n      "kind": "school",\n      "name": "Школа №1253",\n      "lat": 55.7346148,\n      "lon": 37.5922405,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5569527",\n      "kind": "school",\n      "name": "Школа с углубленным изучением иностранного языка № 1231 имени В. Д. Поленова",\n      "lat": 55.7387981,\n      "lon": 37.5923222,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573120",\n      "kind": "kindergarten",\n      "name": "Школа №1078 (Модуль №5)",\n      "lat": 55.8190706,\n      "lon": 37.822638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573122",\n      "kind": "school",\n      "name": "Школа №1598. Здание №2",\n      "lat": 55.8185248,\n      "lon": 37.829256,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573123",\n      "kind": "school",\n      "name": "Школа №1078 (Модуль №2)",\n      "lat": 55.8190333,\n      "lon": 37.8208288,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573124",\n      "kind": "kindergarten",\n      "name": "Школа №368 (Здание №3)",\n      "lat": 55.8190607,\n      "lon": 37.8148522,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573125",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №12",\n      "lat": 55.819322,\n      "lon": 37.8292055,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5594700",\n      "kind": "kindergarten",\n      "name": "Детский сад №233",\n      "lat": 55.668933,\n      "lon": 37.5490159,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5594701",\n      "kind": "kindergarten",\n      "name": "Детский сад №293",\n      "lat": 55.668595,\n      "lon": 37.5499694,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5594702",\n      "kind": "school",\n      "name": "Школа №121",\n      "lat": 55.6697291,\n      "lon": 37.5493704,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5623081",\n      "kind": "school",\n      "name": "Школа №1259",\n      "lat": 55.7319937,\n      "lon": 37.6320087,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5686625",\n      "kind": "school",\n      "name": "Школа №633",\n      "lat": 55.7546747,\n      "lon": 37.8270964,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5699016",\n      "kind": "school",\n      "name": "Школа №1097",\n      "lat": 55.8659895,\n      "lon": 37.6574187,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713477",\n      "kind": "kindergarten",\n      "name": "Детский сад №2287",\n      "lat": 55.8756377,\n      "lon": 37.5815872,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713478",\n      "kind": "kindergarten",\n      "name": "Детский сад №36",\n      "lat": 55.7720042,\n      "lon": 37.7305262,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713479",\n      "kind": "school",\n      "name": "Лицей №1575",\n      "lat": 55.8050537,\n      "lon": 37.5320355,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713481",\n      "kind": "school",\n      "name": "Школа №302",\n      "lat": 55.8804305,\n      "lon": 37.5794377,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713482",\n      "kind": "school",\n      "name": "Школа №635",\n      "lat": 55.7571779,\n      "lon": 37.7923071,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713483",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.818824,\n      "lon": 37.571378,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713567",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №4",\n      "lat": 55.6351639,\n      "lon": 37.6075809,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713568",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №3",\n      "lat": 55.6356447,\n      "lon": 37.6079979,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5736841",\n      "kind": "kindergarten",\n      "name": "Детский сад №2389",\n      "lat": 55.8847424,\n      "lon": 37.6844234,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5754384",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6651528,\n      "lon": 37.5575675,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5754385",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6644274,\n      "lon": 37.5580763,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5781856",\n      "kind": "kindergarten",\n      "name": "Детский сад №380",\n      "lat": 55.7377889,\n      "lon": 37.4665472,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5849302",\n      "kind": "kindergarten",\n      "name": "Детский сад №1159",\n      "lat": 55.8089016,\n      "lon": 37.6405263,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5851694",\n      "kind": "school",\n      "name": "Спортшкола № 46",\n      "lat": 55.6069056,\n      "lon": 37.6777051,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5892824",\n      "kind": "school",\n      "name": "Гимназия №45",\n      "lat": 55.6917983,\n      "lon": 37.5858468,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6010660",\n      "kind": "school",\n      "name": "Центр образования №1499",\n      "lat": 55.8357258,\n      "lon": 37.6605556,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6022234",\n      "kind": "kindergarten",\n      "name": "Детский сад № 11 «Незабудка»",\n      "lat": 55.8909247,\n      "lon": 37.7221782,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6116666",\n      "kind": "school",\n      "name": "Школа «Золотое сечение»",\n      "lat": 55.7385903,\n      "lon": 37.577139,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6228956",\n      "kind": "kindergarten",\n      "name": "Детский сад №869",\n      "lat": 55.8592521,\n      "lon": 37.4700925,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6282939",\n      "kind": "school",\n      "name": "Школа №692",\n      "lat": 55.8563497,\n      "lon": 37.4730511,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6319979",\n      "kind": "school",\n      "name": "Школа «Содружество». Учебный корпус №2",\n      "lat": 55.7607457,\n      "lon": 37.6915744,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6322609",\n      "kind": "kindergarten",\n      "name": "Детский сад №1262",\n      "lat": 55.8670411,\n      "lon": 37.5687592,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6322610",\n      "kind": "school",\n      "name": "Начальная школа №1826",\n      "lat": 55.8667093,\n      "lon": 37.5675411,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6334556",\n      "kind": "school",\n      "name": "Детский сад №1548",\n      "lat": 55.8611376,\n      "lon": 37.5604878,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6339395",\n      "kind": "kindergarten",\n      "name": "Школа № 1383. Корпус 3",\n      "lat": 55.8593028,\n      "lon": 37.5661993,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6339396",\n      "kind": "kindergarten",\n      "name": "Школа № 1383. Корпус 4",\n      "lat": 55.8586097,\n      "lon": 37.5662128,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6339397",\n      "kind": "school",\n      "name": "Школа №1383",\n      "lat": 55.8579072,\n      "lon": 37.5663031,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6344000",\n      "kind": "kindergarten",\n      "name": "Детский сад №1472",\n      "lat": 55.7280741,\n      "lon": 37.5872774,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6362542",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Юниор-парк\\"",\n      "lat": 55.8657052,\n      "lon": 37.5543541,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6362543",\n      "kind": "kindergarten",\n      "name": "Школа № 656 имени А. С. Макаренко, дошкольное отделение",\n      "lat": 55.8650348,\n      "lon": 37.5548924,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6376832",\n      "kind": "school",\n      "name": "Школа №\xa01252 имени Сервантеса",\n      "lat": 55.8102107,\n      "lon": 37.4963393,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6476664",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1951 комбинированного вида",\n      "lat": 55.7763858,\n      "lon": 37.6003314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6498938",\n      "kind": "school",\n      "name": "Школа №1285",\n      "lat": 55.8338476,\n      "lon": 37.4485177,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6505043",\n      "kind": "kindergarten",\n      "name": "Детский сад №49",\n      "lat": 55.7844535,\n      "lon": 37.6313068,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6505044",\n      "kind": "school",\n      "name": "Школа № 2107. Школа им. Ромена Роллана",\n      "lat": 55.7854975,\n      "lon": 37.6320923,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6517532",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1021",\n      "lat": 55.7897922,\n      "lon": 37.6257217,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6519019",\n      "kind": "kindergarten",\n      "name": "Детский сад №1517",\n      "lat": 55.7763258,\n      "lon": 37.4605473,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6519020",\n      "kind": "kindergarten",\n      "name": "Гимназия №1517",\n      "lat": 55.7754771,\n      "lon": 37.4658991,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6527144",\n      "kind": "school",\n      "name": "Школа №1231 имени В. Д. Поленова",\n      "lat": 55.7403285,\n      "lon": 37.5774833,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6528274",\n      "kind": "kindergarten",\n      "name": "Гимназия №1517",\n      "lat": 55.775982,\n      "lon": 37.4633056,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6533359",\n      "kind": "school",\n      "name": "Гимназия №1517",\n      "lat": 55.7751017,\n      "lon": 37.4629548,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6598494",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Сказка\\"",\n      "lat": 55.7275453,\n      "lon": 37.7611404,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6609198",\n      "kind": "school",\n      "name": "Психоневрологический интернат №4",\n      "lat": 55.7362311,\n      "lon": 37.4590944,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6618012",\n      "kind": "kindergarten",\n      "name": "Детский сад №2444",\n      "lat": 55.8712089,\n      "lon": 37.5673144,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6618014",\n      "kind": "school",\n      "name": "Центр образования №1487",\n      "lat": 55.8696222,\n      "lon": 37.5658385,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6618015",\n      "kind": "school",\n      "name": "Школа №1029",\n      "lat": 55.8700469,\n      "lon": 37.567721,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6696799",\n      "kind": "school",\n      "name": "Школа № 1323",\n      "lat": 55.7430375,\n      "lon": 37.625783,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6698208",\n      "kind": "kindergarten",\n      "name": "ДОУ № 729",\n      "lat": 55.7562398,\n      "lon": 37.586336,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6698209",\n      "kind": "school",\n      "name": "Школа №1234",\n      "lat": 55.7562998,\n      "lon": 37.5852289,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6702161",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8135142,\n      "lon": 37.4932529,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6712910",\n      "kind": "kindergarten",\n      "name": "Детский сад №1389",\n      "lat": 55.8714024,\n      "lon": 37.5591549,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6712911",\n      "kind": "kindergarten",\n      "name": "Детский сад №642",\n      "lat": 55.8712824,\n      "lon": 37.5574807,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6718174",\n      "kind": "school",\n      "name": "Школа №1223",\n      "lat": 55.8448107,\n      "lon": 37.5534485,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6728142",\n      "kind": "school",\n      "name": "Лицей № 1535",\n      "lat": 55.7260387,\n      "lon": 37.5650015,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6728144",\n      "kind": "kindergarten",\n      "name": "Филипповская школа",\n      "lat": 55.7256014,\n      "lon": 37.5641975,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6728834",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8037758,\n      "lon": 37.7772296,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6728835",\n      "kind": "school",\n      "name": "Школа №343",\n      "lat": 55.8034367,\n      "lon": 37.7770052,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6741550",\n      "kind": "school",\n      "name": "Школьное отделение № 22 Лицея № 1501",\n      "lat": 55.7699198,\n      "lon": 37.6066168,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751348",\n      "kind": "kindergarten",\n      "name": "Детский сад №1486",\n      "lat": 55.8758657,\n      "lon": 37.5444077,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751349",\n      "kind": "kindergarten",\n      "name": "Детский сад №2298",\n      "lat": 55.8755071,\n      "lon": 37.5435031,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751350",\n      "kind": "school",\n      "name": "Школа №849",\n      "lat": 55.8752844,\n      "lon": 37.5396717,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751351",\n      "kind": "school",\n      "name": "Школа №1121",\n      "lat": 55.8754473,\n      "lon": 37.5416211,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751356",\n      "kind": "kindergarten",\n      "name": "Детский сад №1578",\n      "lat": 55.8770485,\n      "lon": 37.5507131,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758020",\n      "kind": "kindergarten",\n      "name": "Детский сад №669",\n      "lat": 55.7414618,\n      "lon": 37.5747631,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758021",\n      "kind": "school",\n      "name": "Спецшкола №30",\n      "lat": 55.7370603,\n      "lon": 37.5714404,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758026",\n      "kind": "school",\n      "name": "Школа №57 (здание №3)",\n      "lat": 55.7205716,\n      "lon": 37.569045,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758127",\n      "kind": "school",\n      "name": "Школа №45",\n      "lat": 55.7240585,\n      "lon": 37.5602548,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758920",\n      "kind": "kindergarten",\n      "name": "Детский сад №2044",\n      "lat": 55.842186,\n      "lon": 37.504784,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6796156",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8441523,\n      "lon": 37.4870092,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6847000",\n      "kind": "school",\n      "name": "Школа № 647",\n      "lat": 55.763312,\n      "lon": 37.7377824,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6886515",\n      "kind": "school",\n      "name": "Школа № 1329",\n      "lat": 55.6689647,\n      "lon": 37.4649297,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6890548",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Хавской\\"",\n      "lat": 55.7133694,\n      "lon": 37.6146213,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6909136",\n      "kind": "school",\n      "name": "Школа №1409",\n      "lat": 55.7885255,\n      "lon": 37.5444968,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6913392",\n      "kind": "school",\n      "name": "Московский академический художественный лицей",\n      "lat": 55.7323416,\n      "lon": 37.6079558,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6913847",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1761",\n      "lat": 55.7534586,\n      "lon": 37.6784135,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6929207",\n      "kind": "kindergarten",\n      "name": "Детский сад №1080",\n      "lat": 55.7107297,\n      "lon": 37.7403293,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6947137",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1268",\n      "lat": 55.8913369,\n      "lon": 37.5350199,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6957072",\n      "kind": "kindergarten",\n      "name": "Школа №2054, Дошкольное отделение №1",\n      "lat": 55.7672766,\n      "lon": 37.6164575,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6968315",\n      "kind": "school",\n      "name": "Школа № 812",\n      "lat": 55.6698785,\n      "lon": 37.4602345,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6981012",\n      "kind": "school",\n      "name": "Школа №1236 имени С.В.Милашенкова. Отделение №250",\n      "lat": 55.817851,\n      "lon": 37.5867705,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7004980",\n      "kind": "kindergarten",\n      "name": "Детский сад №632",\n      "lat": 55.8709959,\n      "lon": 37.5496012,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7004981",\n      "kind": "school",\n      "name": "Школа № 656 им. А. С. Макаренко",\n      "lat": 55.8704118,\n      "lon": 37.5526854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7013096",\n      "kind": "school",\n      "name": "Школа №767",\n      "lat": 55.8209221,\n      "lon": 37.7948285,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7055165",\n      "kind": "kindergarten",\n      "name": "Детский сад №822",\n      "lat": 55.8751489,\n      "lon": 37.5522452,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7092147",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Б. Казенном",\n      "lat": 55.7605664,\n      "lon": 37.6555793,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103021",\n      "kind": "school",\n      "name": "Иракская школа",\n      "lat": 55.7563414,\n      "lon": 37.5948429,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103024",\n      "kind": "school",\n      "name": "Пушкинская гимназия на Пресне",\n      "lat": 55.7586151,\n      "lon": 37.5706237,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103026",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7605417,\n      "lon": 37.5657947,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103028",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7672908,\n      "lon": 37.5695366,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103029",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1570. Дошкольное отделение",\n      "lat": 55.7813388,\n      "lon": 37.5845225,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103030",\n      "kind": "kindergarten",\n      "name": "Школа «Содружество». Учебный корпус №4",\n      "lat": 55.7568008,\n      "lon": 37.6964894,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103031",\n      "kind": "kindergarten",\n      "name": "Школа № 1228 \\"Лефортово\\". Корпус \\"Цветик-семицветик\\"",\n      "lat": 55.7638631,\n      "lon": 37.7148521,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103032",\n      "kind": "kindergarten",\n      "name": "Школа № 1228 \\"Лефортово\\". Корпус \\"Аленький цветочек\\"",\n      "lat": 55.7635683,\n      "lon": 37.7156527,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103198",\n      "kind": "school",\n      "name": "Школа № 962",\n      "lat": 55.8553388,\n      "lon": 37.5877126,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103199",\n      "kind": "kindergarten",\n      "name": "Школа № 962. Дошкольное отделение № 5",\n      "lat": 55.8545481,\n      "lon": 37.5869014,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103200",\n      "kind": "kindergarten",\n      "name": "Лицей № 1575. Дошкольное отделение",\n      "lat": 55.7969242,\n      "lon": 37.5573494,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103202",\n      "kind": "school",\n      "name": "Инженерно-техническая школа им. дважды Героя Советского Союза П.Р. Поповича",\n      "lat": 55.8057821,\n      "lon": 37.5480073,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103203",\n      "kind": "kindergarten",\n      "name": "Инженерно-техническая школа имени дважды Героя Советского Союза П. Р. Поповича. Дошкольное отделение Коккинаки № 7",\n      "lat": 55.8054757,\n      "lon": 37.5470409,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103204",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1266",\n      "lat": 55.8664607,\n      "lon": 37.6181239,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103205",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1554. Дошкольное отделение № 1",\n      "lat": 55.8673444,\n      "lon": 37.6176933,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103206",\n      "kind": "kindergarten",\n      "name": "Школа № 152. Детский сад № 286",\n      "lat": 55.8031036,\n      "lon": 37.5309517,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103207",\n      "kind": "kindergarten",\n      "name": "Школа-сад № 1847",\n      "lat": 55.8064758,\n      "lon": 37.5332772,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103210",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1570. Дошкольное отделение",\n      "lat": 55.7825166,\n      "lon": 37.5776695,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103211",\n      "kind": "school",\n      "name": "Специальная (коррекционная) общеобразовательная школа-интернат № 2",\n      "lat": 55.8098255,\n      "lon": 37.5347202,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103213",\n      "kind": "school",\n      "name": "Кадетская школа № 1784",\n      "lat": 55.7803093,\n      "lon": 37.5684252,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7119651",\n      "kind": "school",\n      "name": "Школа № 2054. Школьное отделение №3",\n      "lat": 55.7650282,\n      "lon": 37.6142371,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7141808",\n      "kind": "school",\n      "name": "Школа № 2030",\n      "lat": 55.7598673,\n      "lon": 37.5557759,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7153404",\n      "kind": "school",\n      "name": "Планета успеха",\n      "lat": 55.8366676,\n      "lon": 37.5280176,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7156736",\n      "kind": "school",\n      "name": "Школа с углубленным изучением отдельных предметов № 1234",\n      "lat": 55.7651748,\n      "lon": 37.6013492,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7173366",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Здание 8",\n      "lat": 55.819026,\n      "lon": 37.4954988,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7184365",\n      "kind": "kindergarten",\n      "name": "Детский сад №1592",\n      "lat": 55.6735542,\n      "lon": 37.7427032,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7190748",\n      "kind": "school",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1631",\n      "lat": 55.8921625,\n      "lon": 37.5395776,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7220450",\n      "kind": "kindergarten",\n      "name": "Детский сад №2204",\n      "lat": 55.7234383,\n      "lon": 37.6124879,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7238772",\n      "kind": "school",\n      "name": "Гимназия Марьина Роща имени В. Ф. Орлова. Учебный корпус 35",\n      "lat": 55.7865131,\n      "lon": 37.6046806,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7268814",\n      "kind": "school",\n      "name": "Школа №1501",\n      "lat": 55.7796397,\n      "lon": 37.6141136,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7295266",\n      "kind": "school",\n      "name": "Интeрнат \\"Третий Московский кадетский корпус имени Александра Невского\\"",\n      "lat": 55.7328784,\n      "lon": 37.6522596,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7299944",\n      "kind": "kindergarten",\n      "name": "Детский сад №1497",\n      "lat": 55.7740821,\n      "lon": 37.6018766,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7307938",\n      "kind": "kindergarten",\n      "name": "ДОУ № 1465",\n      "lat": 55.7633218,\n      "lon": 37.5392107,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7307939",\n      "kind": "kindergarten",\n      "name": "ДОУ №1930",\n      "lat": 55.7634294,\n      "lon": 37.5382958,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7309883",\n      "kind": "kindergarten",\n      "name": "ДОУ № 749",\n      "lat": 55.7604989,\n      "lon": 37.5381105,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7309884",\n      "kind": "kindergarten",\n      "name": "ДОУ № 1927",\n      "lat": 55.7606072,\n      "lon": 37.5372349,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7325096",\n      "kind": "school",\n      "name": "Школа №\xa0340",\n      "lat": 55.7588422,\n      "lon": 37.5366524,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7326339",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 244",\n      "lat": 55.8964256,\n      "lon": 37.5388947,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7326343",\n      "kind": "school",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 184",\n      "lat": 55.8955124,\n      "lon": 37.5378709,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7329513",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7171504,\n      "lon": 37.6726984,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7329517",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7178924,\n      "lon": 37.6693535,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7330425",\n      "kind": "kindergarten",\n      "name": "Детский сад №390",\n      "lat": 55.8816791,\n      "lon": 37.5723914,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7334650",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7415593,\n      "lon": 37.6783755,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7334651",\n      "kind": "school",\n      "name": "Школа №1221",\n      "lat": 55.7407482,\n      "lon": 37.6789543,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7357058",\n      "kind": "school",\n      "name": "Школа № 56",\n      "lat": 55.7467017,\n      "lon": 37.5492835,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7359503",\n      "kind": "kindergarten",\n      "name": "Детский сад №703",\n      "lat": 55.8818296,\n      "lon": 37.5347146,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7359504",\n      "kind": "school",\n      "name": "Школа №771",\n      "lat": 55.8827959,\n      "lon": 37.5346768,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7402655",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №4",\n      "lat": 55.6688623,\n      "lon": 37.5057022,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7405009",\n      "kind": "school",\n      "name": "Школа №771",\n      "lat": 55.8865095,\n      "lon": 37.5373528,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7405010",\n      "kind": "school",\n      "name": "Школа №1291 с углублённым изучением английского языка",\n      "lat": 55.8872557,\n      "lon": 37.5373531,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7435037",\n      "kind": "school",\n      "name": "Инженерная школа № 1581. Корпус 2",\n      "lat": 55.768521,\n      "lon": 37.65502,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7484311",\n      "kind": "school",\n      "name": "Многопрофильный лицей № 1799. СП \\"Лицей на Полянке\\"",\n      "lat": 55.7369958,\n      "lon": 37.6180798,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7500830",\n      "kind": "school",\n      "name": "Многопрофильный лицей № 1799. СП «Школа № 19»",\n      "lat": 55.7441771,\n      "lon": 37.6217091,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7540984",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1143",\n      "lat": 55.8535421,\n      "lon": 37.5789539,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7568257",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7047134,\n      "lon": 37.4936867,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7595822",\n      "kind": "kindergarten",\n      "name": "Детский сад № 516",\n      "lat": 55.7801277,\n      "lon": 37.5958008,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7612096",\n      "kind": "school",\n      "name": "Лицей НИУ ВШЭ",\n      "lat": 55.7696296,\n      "lon": 37.6188347,\n      "opening_hours": "Mo-Sa 09:00-18:00"\n    },\n    {\n      "id": "relation/7612099",\n      "kind": "school",\n      "name": "Центр образования №1479",\n      "lat": 55.7708346,\n      "lon": 37.619002,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7612111",\n      "kind": "kindergarten",\n      "name": "Колледж малого бизнеса № 4. Дошкольный корпус 2",\n      "lat": 55.772255,\n      "lon": 37.6173229,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7613478",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8625097,\n      "lon": 37.6930833,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7621468",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Покровке",\n      "lat": 55.7585886,\n      "lon": 37.6444861,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7629953",\n      "kind": "kindergarten",\n      "name": "Детский сад № 395 Медицинского центра Управления делами Мэра и Правительства Москвы",\n      "lat": 55.7549191,\n      "lon": 37.6417214,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7637088",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"Маршал\\"",\n      "lat": 55.7917885,\n      "lon": 37.4985782,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7639829",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1428",\n      "lat": 55.6678026,\n      "lon": 37.4682709,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7641051",\n      "kind": "school",\n      "name": "Школа № 627",\n      "lat": 55.7330417,\n      "lon": 37.6360641,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7658483",\n      "kind": "kindergarten",\n      "name": "ДОУ №1982",\n      "lat": 55.7641954,\n      "lon": 37.6661706,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7658484",\n      "kind": "school",\n      "name": "Центр образования №1480",\n      "lat": 55.7645917,\n      "lon": 37.6659139,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7666664",\n      "kind": "kindergarten",\n      "name": "Детский сад №8",\n      "lat": 55.7389176,\n      "lon": 37.4996479,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7666666",\n      "kind": "school",\n      "name": "Школа №700",\n      "lat": 55.7389273,\n      "lon": 37.5020715,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7672222",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"В гостях у сказки\\"",\n      "lat": 55.734339,\n      "lon": 37.6309811,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7672224",\n      "kind": "kindergarten",\n      "name": "Школа № 1259. Дошкольное отделение",\n      "lat": 55.7342307,\n      "lon": 37.6294031,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7712039",\n      "kind": "school",\n      "name": "",\n      "lat": 55.806347,\n      "lon": 37.6895545,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7715842",\n      "kind": "school",\n      "name": "Гимназия №1257",\n      "lat": 55.7144854,\n      "lon": 37.6256635,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7716814",\n      "kind": "kindergarten",\n      "name": "Детский сад №2032",\n      "lat": 55.7599186,\n      "lon": 37.6356376,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7717004",\n      "kind": "school",\n      "name": "Лицей № 1535",\n      "lat": 55.7309714,\n      "lon": 37.5619802,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7719254",\n      "kind": "school",\n      "name": "Школа № 2124 \\"Центр развития и коррекции\\" Здание №3",\n      "lat": 55.7610715,\n      "lon": 37.6365513,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722663",\n      "kind": "school",\n      "name": "Школа № 554",\n      "lat": 55.6609508,\n      "lon": 37.5815156,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722664",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.6600307,\n      "lon": 37.5823025,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722665",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.6596599,\n      "lon": 37.5809665,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722666",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.6578611,\n      "lon": 37.5804658,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722668",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.658422,\n      "lon": 37.5818665,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7725968",\n      "kind": "school",\n      "name": "Школа № 1231 им. В. Д. Поленова",\n      "lat": 55.7485171,\n      "lon": 37.5966758,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7738645",\n      "kind": "kindergarten",\n      "name": "Детский сад №1170",\n      "lat": 55.7128498,\n      "lon": 37.614267,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7771762",\n      "kind": "kindergarten",\n      "name": "Школа №830 дошкольное отделение Незабудка-1",\n      "lat": 55.8239873,\n      "lon": 37.4447691,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7771763",\n      "kind": "kindergarten",\n      "name": "Школа №830 дошкольное отделение Незабудка-2",\n      "lat": 55.8235078,\n      "lon": 37.4442511,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7795518",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8041486,\n      "lon": 37.5489507,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7798211",\n      "kind": "kindergarten",\n      "name": "Школа № 345 имени А.С. Пушкина. Дошкольное отделение",\n      "lat": 55.7666126,\n      "lon": 37.6563882,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7799935",\n      "kind": "kindergarten",\n      "name": "Детский сад №2529",\n      "lat": 55.8070525,\n      "lon": 37.5199063,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7814787",\n      "kind": "school",\n      "name": "Школа №1301 им. Е.Т. Гайдара",\n      "lat": 55.7800317,\n      "lon": 37.7385422,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7839822",\n      "kind": "school",\n      "name": "Школа №1262",\n      "lat": 55.7305746,\n      "lon": 37.6205231,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7840004",\n      "kind": "kindergarten",\n      "name": "Школа № 1231 имени В. Д. Поленова. Дошкольный корпус",\n      "lat": 55.7298123,\n      "lon": 37.5617337,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7877996",\n      "kind": "school",\n      "name": "Школа №1231",\n      "lat": 55.7498071,\n      "lon": 37.5890358,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7891536",\n      "kind": "school",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 2",\n      "lat": 55.7451167,\n      "lon": 37.5925565,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7891541",\n      "kind": "school",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 4",\n      "lat": 55.7469797,\n      "lon": 37.595742,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7895065",\n      "kind": "school",\n      "name": "Школа № 1277",\n      "lat": 55.7838582,\n      "lon": 37.6020225,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7903980",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7851838,\n      "lon": 37.5361895,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7903981",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7857234,\n      "lon": 37.5370061,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7916369",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 3",\n      "lat": 55.8507959,\n      "lon": 37.635942,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7927700",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 10",\n      "lat": 55.7761138,\n      "lon": 37.551338,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7966560",\n      "kind": "school",\n      "name": "Школа № 1270 «Вектор»",\n      "lat": 55.7439793,\n      "lon": 37.6651545,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7997756",\n      "kind": "school",\n      "name": "Лицей НИУ ВШЭ",\n      "lat": 55.7636235,\n      "lon": 37.6440206,\n      "opening_hours": "Mo-Sa 09:00-18:00"\n    },\n    {\n      "id": "relation/8010397",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №6",\n      "lat": 55.8522782,\n      "lon": 37.6950801,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8016230",\n      "kind": "school",\n      "name": "Частная школа \\"Бнот Менахем\\"",\n      "lat": 55.7664398,\n      "lon": 37.6429479,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8025165",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8947465,\n      "lon": 37.7315412,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8062611",\n      "kind": "kindergarten",\n      "name": "Школа \\"Покровский квартал\\". Детский сад № 1",\n      "lat": 55.7551529,\n      "lon": 37.6514914,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8119012",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Ромашка\\"",\n      "lat": 55.8090257,\n      "lon": 37.4742969,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8119013",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Колокольчик\\"",\n      "lat": 55.8097715,\n      "lon": 37.4741278,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8157274",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8016983,\n      "lon": 37.4650659,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8178929",\n      "kind": "school",\n      "name": "Школа №123",\n      "lat": 55.7579814,\n      "lon": 37.6022541,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8197027",\n      "kind": "kindergarten",\n      "name": "Детский сад №298",\n      "lat": 55.8353875,\n      "lon": 37.533436,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8197028",\n      "kind": "kindergarten",\n      "name": "Детский сад №309",\n      "lat": 55.836043,\n      "lon": 37.5339836,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8197029",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8361141,\n      "lon": 37.5325804,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8209718",\n      "kind": "kindergarten",\n      "name": "Школа № 717. Дошкольное отделение",\n      "lat": 55.8202265,\n      "lon": 37.515923,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8224251",\n      "kind": "school",\n      "name": "Школа №1232",\n      "lat": 55.7501767,\n      "lon": 37.5612288,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8298825",\n      "kind": "school",\n      "name": "Британская Международная Школа № 2",\n      "lat": 55.8186452,\n      "lon": 37.5275612,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8298826",\n      "kind": "kindergarten",\n      "name": "Детский сад №2394",\n      "lat": 55.8181515,\n      "lon": 37.5289511,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8304326",\n      "kind": "school",\n      "name": "Школа №\xa067",\n      "lat": 55.7413846,\n      "lon": 37.5232664,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8318510",\n      "kind": "school",\n      "name": "Школа №1125",\n      "lat": 55.8829353,\n      "lon": 37.4921408,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8331322",\n      "kind": "school",\n      "name": "ГБОУ Школа № 1249",\n      "lat": 55.8012999,\n      "lon": 37.5185448,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8342319",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1576\\" Дошкольное отделение корпус № 11 (\\"Радуга\\")",\n      "lat": 55.8270129,\n      "lon": 37.531832,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8342560",\n      "kind": "school",\n      "name": "Британская Международная Школа № 1",\n      "lat": 55.8261245,\n      "lon": 37.5319324,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8361003",\n      "kind": "kindergarten",\n      "name": "Школа \\"Бескудниково\\". Дошкольный корпус №3",\n      "lat": 55.8700128,\n      "lon": 37.541388,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8361042",\n      "kind": "school",\n      "name": "Прогимназия №1661",\n      "lat": 55.8740524,\n      "lon": 37.5307217,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8361054",\n      "kind": "school",\n      "name": "Школа \\"Бескудниково\\". Корпус 2",\n      "lat": 55.8718693,\n      "lon": 37.5422574,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8367574",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7585504,\n      "lon": 37.5720282,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8439571",\n      "kind": "kindergarten",\n      "name": "Детский сад №34 «Колосок»",\n      "lat": 55.8967424,\n      "lon": 37.4535117,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8439572",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa02 «Журавушка»",\n      "lat": 55.8968427,\n      "lon": 37.4524757,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8444542",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.741603,\n      "lon": 37.6358854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8446521",\n      "kind": "kindergarten",\n      "name": "Школа № 1799. Дошкольное отделение Пятницкое",\n      "lat": 55.7340469,\n      "lon": 37.6266797,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8606729",\n      "kind": "kindergarten",\n      "name": "Школа № 1583 имени К. А. Керимова (Дошкольный корпус)",\n      "lat": 55.8517902,\n      "lon": 37.5071134,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8606732",\n      "kind": "school",\n      "name": "Школьная академия",\n      "lat": 55.8506129,\n      "lon": 37.5146245,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8650883",\n      "kind": "school",\n      "name": "Образовательный комплекс градостроительства \\"Столица\\". Отделение Саратовское",\n      "lat": 55.7116724,\n      "lon": 37.7364235,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8675635",\n      "kind": "school",\n      "name": "Школа № 179",\n      "lat": 55.7596561,\n      "lon": 37.614753,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8709255",\n      "kind": "school",\n      "name": "Школа № 1601 имени Героя Советского Союза Е. К. Лютикова. Основная и средняя школа (4)",\n      "lat": 55.8060143,\n      "lon": 37.5761265,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8710076",\n      "kind": "kindergarten",\n      "name": "slamiKIDS",\n      "lat": 55.8061541,\n      "lon": 37.5300521,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8712975",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Лялином",\n      "lat": 55.7589931,\n      "lon": 37.6516152,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8714875",\n      "kind": "school",\n      "name": "Школа № 1499 им. И. А. Докукина",\n      "lat": 55.8344476,\n      "lon": 37.6443723,\n      "opening_hours": "Mo-Fr 09:00-17:30"\n    },\n    {\n      "id": "relation/8748581",\n      "kind": "school",\n      "name": "ГБОУ Школа №1373, корпус №2",\n      "lat": 55.7782872,\n      "lon": 37.823674,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8748583",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1373, корпус №6",\n      "lat": 55.7772644,\n      "lon": 37.8222188,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8751770",\n      "kind": "kindergarten",\n      "name": "Школа № 1576. Корпус № 12",\n      "lat": 55.8434623,\n      "lon": 37.5493667,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8751771",\n      "kind": "kindergarten",\n      "name": "Школа № 1576. Корпус № 16",\n      "lat": 55.8430116,\n      "lon": 37.5498226,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8769465",\n      "kind": "school",\n      "name": "Школа №1361",\n      "lat": 55.6749953,\n      "lon": 37.5367835,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8775881",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Земляном валу",\n      "lat": 55.7539838,\n      "lon": 37.6551128,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8789251",\n      "kind": "kindergarten",\n      "name": "Школа №1231 им. В. Д. Поленова. Дошкольный корпус",\n      "lat": 55.7472311,\n      "lon": 37.5877923,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8791878",\n      "kind": "school",\n      "name": "Школа № 2123 имени Мигеля Эрнандеса. Корпус \\"Стодесятая\\"",\n      "lat": 55.7568883,\n      "lon": 37.5949346,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8796311",\n      "kind": "school",\n      "name": "Лицей НИУ ВШЭ",\n      "lat": 55.7608039,\n      "lon": 37.6520715,\n      "opening_hours": "Mo-Sa 09:00-18:00"\n    },\n    {\n      "id": "relation/8801932",\n      "kind": "kindergarten",\n      "name": "Детское поместье",\n      "lat": 55.7601523,\n      "lon": 37.5887812,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8801933",\n      "kind": "school",\n      "name": "Школа № 1239",\n      "lat": 55.7606239,\n      "lon": 37.588361,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8849889",\n      "kind": "school",\n      "name": "Школа №1113",\n      "lat": 55.7686461,\n      "lon": 37.6021606,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8897320",\n      "kind": "school",\n      "name": "Школа № 498. Школьный корпус Марксистская улица, дом 9, корпус 2",\n      "lat": 55.7364752,\n      "lon": 37.6679886,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8897321",\n      "kind": "kindergarten",\n      "name": "Школа № 498. Дошкольный корпус Добровольческая улица, дом 28, строение 1",\n      "lat": 55.7418933,\n      "lon": 37.66924,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8897322",\n      "kind": "school",\n      "name": "Школа № 498. Школьный корпус Большой Факельный переулок, дом 23",\n      "lat": 55.7413421,\n      "lon": 37.6680822,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8898272",\n      "kind": "kindergarten",\n      "name": "Школа № 1259. Дошкольный корпус",\n      "lat": 55.7348096,\n      "lon": 37.6343849,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8904131",\n      "kind": "kindergarten",\n      "name": "Школа № 138. Дошкольный корпус № 7",\n      "lat": 55.7877506,\n      "lon": 37.4666234,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8904132",\n      "kind": "kindergarten",\n      "name": "Школа № 138. Дошкольный корпус № 8",\n      "lat": 55.7876334,\n      "lon": 37.4686236,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8907712",\n      "kind": "school",\n      "name": "Школа № 1694 «Ясенево». Здание 3",\n      "lat": 55.6151733,\n      "lon": 37.5478396,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8907713",\n      "kind": "school",\n      "name": "Школа № 1694 «Ясенево». Здание 2",\n      "lat": 55.6143672,\n      "lon": 37.5478722,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9068415",\n      "kind": "school",\n      "name": "Школа №1220",\n      "lat": 55.8174236,\n      "lon": 37.621538,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9068416",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8188251,\n      "lon": 37.6234868,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9070447",\n      "kind": "school",\n      "name": "Школа №304",\n      "lat": 55.8186767,\n      "lon": 37.6338394,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9070448",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8194776,\n      "lon": 37.627331,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9071344",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8233012,\n      "lon": 37.6279675,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9072849",\n      "kind": "school",\n      "name": "Пироговская школа",\n      "lat": 55.7353638,\n      "lon": 37.6112445,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9080263",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №299",\n      "lat": 55.7089057,\n      "lon": 37.7586704,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081239",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8283747,\n      "lon": 37.5971789,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081441",\n      "kind": "school",\n      "name": "Школа № 1494",\n      "lat": 55.8260892,\n      "lon": 37.6002734,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081470",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8232807,\n      "lon": 37.5967986,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081474",\n      "kind": "kindergarten",\n      "name": "Детский сад №392",\n      "lat": 55.8234987,\n      "lon": 37.6002273,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081476",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8243167,\n      "lon": 37.6003037,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9083102",\n      "kind": "school",\n      "name": "Школа № 1950",\n      "lat": 55.7683739,\n      "lon": 37.5880679,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9093873",\n      "kind": "school",\n      "name": "Школа № 1494",\n      "lat": 55.8279267,\n      "lon": 37.5921387,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9097255",\n      "kind": "school",\n      "name": "Лицей №1571 корпус №1",\n      "lat": 55.857786,\n      "lon": 37.4472314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9143532",\n      "kind": "school",\n      "name": "Школа им. Н.М. Карамзина. Корпус школьного образования №2 «Преображенский»",\n      "lat": 55.5927681,\n      "lon": 37.5383377,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9143533",\n      "kind": "school",\n      "name": "Школа им. Н.М. Карамзина. Корпус школьного образования №3 «Московский»",\n      "lat": 55.5942859,\n      "lon": 37.5407904,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9154405",\n      "kind": "kindergarten",\n      "name": "P\'titCREF \\"1905 года\\"",\n      "lat": 55.7585429,\n      "lon": 37.5649543,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9170797",\n      "kind": "kindergarten",\n      "name": "Школа №1516. Дошкольный корпус",\n      "lat": 55.8191574,\n      "lon": 37.832088,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9170798",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №11",\n      "lat": 55.8184236,\n      "lon": 37.8321488,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9170799",\n      "kind": "school",\n      "name": "Школа №1598. Здание №3",\n      "lat": 55.8247403,\n      "lon": 37.8278634,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9174029",\n      "kind": "school",\n      "name": "Школа с углубленным изучением иностранного языка № 1241. Корпус № 2",\n      "lat": 55.7565362,\n      "lon": 37.5681522,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9223416",\n      "kind": "school",\n      "name": "Православная Свято-Петровская школа",\n      "lat": 55.7510844,\n      "lon": 37.6507548,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9310365",\n      "kind": "school",\n      "name": "Школа 1158",\n      "lat": 55.6343175,\n      "lon": 37.596299,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9332123",\n      "kind": "school",\n      "name": "Школа №110",\n      "lat": 55.7592727,\n      "lon": 37.5967673,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9355953",\n      "kind": "kindergarten",\n      "name": "Школа № 518. Дошкольные группы",\n      "lat": 55.7463645,\n      "lon": 37.6303216,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9384689",\n      "kind": "kindergarten",\n      "name": "Детский сад №33",\n      "lat": 55.8907739,\n      "lon": 37.439701,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9418965",\n      "kind": "kindergarten",\n      "name": "Детский сад № 82",\n      "lat": 55.7127797,\n      "lon": 37.5177828,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9453295",\n      "kind": "school",\n      "name": "Школа №590",\n      "lat": 55.7404377,\n      "lon": 37.4956145,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9464683",\n      "kind": "school",\n      "name": "Школа с углублённым изучением иностранного языка № 1231 имени В. Д. Поленова",\n      "lat": 55.7441076,\n      "lon": 37.5976314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9482182",\n      "kind": "school",\n      "name": "Школа № 627 имени генерала Д. Д. Лелюшенко. Учебный корпус № 6",\n      "lat": 55.7354889,\n      "lon": 37.6216001,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9489535",\n      "kind": "kindergarten",\n      "name": "Школа № 1799. Дошкольное отделение \\"Полянка\\"",\n      "lat": 55.7337689,\n      "lon": 37.6175331,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9569040",\n      "kind": "school",\n      "name": "Лицей № 1511",\n      "lat": 55.6484277,\n      "lon": 37.6522543,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9597254",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7677359,\n      "lon": 37.5563097,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9611653",\n      "kind": "school",\n      "name": "Школа №1284",\n      "lat": 55.7707099,\n      "lon": 37.6401554,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9628854",\n      "kind": "school",\n      "name": "Школа №\xa067. Отделение общего образования",\n      "lat": 55.7344278,\n      "lon": 37.5214083,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9628855",\n      "kind": "kindergarten",\n      "name": "Школа №\xa067. Отделение дошкольного образования",\n      "lat": 55.7348801,\n      "lon": 37.5209666,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9633273",\n      "kind": "school",\n      "name": "Школа №2054, Школьный корпус №1",\n      "lat": 55.7724802,\n      "lon": 37.6262314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9652869",\n      "kind": "school",\n      "name": "Школа № 1259",\n      "lat": 55.7397462,\n      "lon": 37.6398702,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9656985",\n      "kind": "kindergarten",\n      "name": "Школа № 345 им. А.С. Пушкина. Дошкольное отделение",\n      "lat": 55.7796764,\n      "lon": 37.6683067,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9661837",\n      "kind": "school",\n      "name": "Колледж малого бизнеса № 4. Школьный корпус 2",\n      "lat": 55.7396251,\n      "lon": 37.6419205,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9694333",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8052642,\n      "lon": 37.5905905,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9812322",\n      "kind": "school",\n      "name": "Brookes Moscow",\n      "lat": 55.8482212,\n      "lon": 37.6331828,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9830040",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\"",\n      "lat": 55.7633591,\n      "lon": 37.648703,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9830043",\n      "kind": "kindergarten",\n      "name": "Школа \\"Покровский квартал\\". Детский сад № 2",\n      "lat": 55.7634365,\n      "lon": 37.6497945,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865157",\n      "kind": "kindergarten",\n      "name": "Детский сад №1632",\n      "lat": 55.8527453,\n      "lon": 37.6119168,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865158",\n      "kind": "school",\n      "name": "Школа № 950",\n      "lat": 55.8512506,\n      "lon": 37.6148638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865159",\n      "kind": "school",\n      "name": "Центр образования № 264",\n      "lat": 55.8519235,\n      "lon": 37.6158783,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865160",\n      "kind": "kindergarten",\n      "name": "Детский сад №402",\n      "lat": 55.8583931,\n      "lon": 37.6160749,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865161",\n      "kind": "kindergarten",\n      "name": "Детский сад №1123",\n      "lat": 55.8589036,\n      "lon": 37.6144071,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865162",\n      "kind": "school",\n      "name": "Школа №950",\n      "lat": 55.8575581,\n      "lon": 37.6130821,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865163",\n      "kind": "school",\n      "name": "Школа №950",\n      "lat": 55.857052,\n      "lon": 37.6146396,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865164",\n      "kind": "kindergarten",\n      "name": "Детский сад №1125",\n      "lat": 55.7220863,\n      "lon": 37.8208855,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865165",\n      "kind": "school",\n      "name": "Школа №36",\n      "lat": 55.7212744,\n      "lon": 37.8199101,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865170",\n      "kind": "kindergarten",\n      "name": "Детский сад №761",\n      "lat": 55.7263712,\n      "lon": 37.8316232,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865171",\n      "kind": "kindergarten",\n      "name": "Школа №402 имени Алии Молдагуловой (дошкольный корпус \\"Семицветик\\")",\n      "lat": 55.726888,\n      "lon": 37.8319292,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9866094",\n      "kind": "school",\n      "name": "Школа №969",\n      "lat": 55.8654822,\n      "lon": 37.61946,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9866095",\n      "kind": "school",\n      "name": "Школа №240",\n      "lat": 55.8656146,\n      "lon": 37.6218607,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9897399",\n      "kind": "school",\n      "name": "Гбоу Гимназия № 1520 имени Капцовых",\n      "lat": 55.7564684,\n      "lon": 37.6029762,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10133035",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7246115,\n      "lon": 37.7419583,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10165007",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6508734,\n      "lon": 37.6022408,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10377236",\n      "kind": "school",\n      "name": "Московская экономическая школа",\n      "lat": 55.760909,\n      "lon": 37.5662638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10391671",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 3 «Мозаика»",\n      "lat": 55.7644286,\n      "lon": 37.5725901,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10523482",\n      "kind": "kindergarten",\n      "name": "Феникс",\n      "lat": 55.5841196,\n      "lon": 37.6031506,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10524778",\n      "kind": "kindergarten",\n      "name": "Школа № 1530 \\"Школа Ломоносова\\". Дошкольные группы на Егерской",\n      "lat": 55.7927301,\n      "lon": 37.683713,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10524779",\n      "kind": "school",\n      "name": "Школа № 1530 \\"Школа Ломоносова\\". Школа на Егерской",\n      "lat": 55.7933156,\n      "lon": 37.6849364,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10524782",\n      "kind": "kindergarten",\n      "name": "Пушкинская школа № 1500. Детский сад",\n      "lat": 55.7838364,\n      "lon": 37.6716538,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10545944",\n      "kind": "school",\n      "name": "Школа № 1601 имени Героя Советского Союза Е. К. Лютикова",\n      "lat": 55.8007293,\n      "lon": 37.5573277,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10559479",\n      "kind": "kindergarten",\n      "name": "Leaders International School",\n      "lat": 55.778889,\n      "lon": 37.5749262,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10563386",\n      "kind": "kindergarten",\n      "name": "Школа № 149 имени Героя Советского Союза Ю. Н. Зыкова. Корпус Левитана 3А",\n      "lat": 55.8008619,\n      "lon": 37.500553,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10563387",\n      "kind": "kindergarten",\n      "name": "Школа № 149 имени Героя Советского Союза Ю. Н. Зыкова. Корпус Левитана 5А",\n      "lat": 55.801186,\n      "lon": 37.4992869,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10617155",\n      "kind": "school",\n      "name": "Школа № 1621 Древо Жизни",\n      "lat": 55.7648533,\n      "lon": 37.6493763,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10698703",\n      "kind": "school",\n      "name": "Школа №1282 \\"Сокольники\\"",\n      "lat": 55.7895468,\n      "lon": 37.6846896,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10723137",\n      "kind": "school",\n      "name": "Спортивный интернат ЦСКА",\n      "lat": 55.7974366,\n      "lon": 37.5562853,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10723138",\n      "kind": "school",\n      "name": "Лицей № 1575. Структурное подразделение дошкольного отделения",\n      "lat": 55.7978021,\n      "lon": 37.5549136,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10846794",\n      "kind": "kindergarten",\n      "name": "Детский сад №631",\n      "lat": 55.8397845,\n      "lon": 37.531076,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10906781",\n      "kind": "kindergarten",\n      "name": "ГБОУ СОШ № 2105 Дошкольный корпус №8,9",\n      "lat": 55.7773505,\n      "lon": 37.7059496,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10918094",\n      "kind": "school",\n      "name": "Инженерно-техническая школа имени дважды Героя Советского Союза П. Р. Поповича",\n      "lat": 55.8038797,\n      "lon": 37.5415481,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10981163",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Совёнок\\"",\n      "lat": 55.7899501,\n      "lon": 37.4899315,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10981165",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"5-й элемент\\"",\n      "lat": 55.8055921,\n      "lon": 37.4824775,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086454",\n      "kind": "school",\n      "name": "Школа №939",\n      "lat": 55.6160556,\n      "lon": 37.7258166,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086455",\n      "kind": "school",\n      "name": "Школа №438",\n      "lat": 55.6162396,\n      "lon": 37.7239994,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086459",\n      "kind": "kindergarten",\n      "name": "Детский сад №1077",\n      "lat": 55.6078815,\n      "lon": 37.7347008,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086461",\n      "kind": "school",\n      "name": "Центр образования № 548",\n      "lat": 55.6027936,\n      "lon": 37.7378758,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086463",\n      "kind": "kindergarten",\n      "name": "Детский сад №1114",\n      "lat": 55.6037931,\n      "lon": 37.7418284,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086464",\n      "kind": "kindergarten",\n      "name": "Детский сад №1383",\n      "lat": 55.6040382,\n      "lon": 37.7431162,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11126283",\n      "kind": "school",\n      "name": "Центр образования №1404 \\"Гамма\\"",\n      "lat": 55.8039145,\n      "lon": 37.6881613,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11152601",\n      "kind": "school",\n      "name": "Центр образования №1404 \\"Гамма\\"",\n      "lat": 55.8079392,\n      "lon": 37.6869219,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11163001",\n      "kind": "school",\n      "name": "Школа № 1164",\n      "lat": 55.7976766,\n      "lon": 37.5742211,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11164803",\n      "kind": "school",\n      "name": "Центр образования №1601 (начальная школа)",\n      "lat": 55.7978457,\n      "lon": 37.5808804,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11181083",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"Вектор\\"",\n      "lat": 55.7948237,\n      "lon": 37.48661,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11186468",\n      "kind": "kindergarten",\n      "name": "Bright Children Academy",\n      "lat": 55.7523385,\n      "lon": 37.7090175,\n      "opening_hours": "Mo-Fr 08:00-20:00; Sa 10:00-17:00"\n    },\n    {\n      "id": "relation/11187072",\n      "kind": "kindergarten",\n      "name": "Школа №1208 им. Героя Советского Союза М.С. Шумилова",\n      "lat": 55.709847,\n      "lon": 37.7769369,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11187073",\n      "kind": "school",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa03",\n      "lat": 55.7084957,\n      "lon": 37.7761886,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11207186",\n      "kind": "school",\n      "name": "Школа №439 \\"Интеллект\\" (корпус начального образования №2)",\n      "lat": 55.7389672,\n      "lon": 37.7408622,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11208335",\n      "kind": "school",\n      "name": "Школа №1530",\n      "lat": 55.7860481,\n      "lon": 37.685283,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11257412",\n      "kind": "school",\n      "name": "Гимназия №1530",\n      "lat": 55.7880045,\n      "lon": 37.6836278,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11268549",\n      "kind": "kindergarten",\n      "name": "Детский сад № 74",\n      "lat": 55.7863591,\n      "lon": 37.5926044,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11268551",\n      "kind": "school",\n      "name": "Гимназия № 1540",\n      "lat": 55.786841,\n      "lon": 37.5921546,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11279390",\n      "kind": "kindergarten",\n      "name": "Школа № 1540. Дошкольные группы",\n      "lat": 55.7872697,\n      "lon": 37.590681,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11284691",\n      "kind": "kindergarten",\n      "name": "Школа № 1321 \\"Ковчег\\". Дошкольное отделение здание №2",\n      "lat": 55.7472332,\n      "lon": 37.7183324,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11284692",\n      "kind": "kindergarten",\n      "name": "Школа № 1321 \\"Ковчег\\". Дошкольное отделение здание №1",\n      "lat": 55.7445728,\n      "lon": 37.7201853,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11292978",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7894355,\n      "lon": 37.7370935,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11295523",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7895842,\n      "lon": 37.740588,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11297320",\n      "kind": "school",\n      "name": "Школа №1228. Корпус 1",\n      "lat": 55.7408011,\n      "lon": 37.7199991,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11331272",\n      "kind": "school",\n      "name": "Школа-интернат №30 им. К. А. Микаэльяна",\n      "lat": 55.7995763,\n      "lon": 37.6925662,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11331273",\n      "kind": "kindergarten",\n      "name": "Детский сад №1860",\n      "lat": 55.7986335,\n      "lon": 37.6922682,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11342614",\n      "kind": "school",\n      "name": "Школа №457",\n      "lat": 55.7451548,\n      "lon": 37.6769573,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11355615",\n      "kind": "school",\n      "name": "Школа №354 имени Д.М. Карбышева ШО-1/1",\n      "lat": 55.7691447,\n      "lon": 37.6680972,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11356961",\n      "kind": "school",\n      "name": "Школа №760 им. А. П. Маресьева. Школьное отделение",\n      "lat": 55.8810918,\n      "lon": 37.7203272,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11356962",\n      "kind": "kindergarten",\n      "name": "Школа №760 им. А. П. Маресьева. Дошкольное отделение",\n      "lat": 55.8816904,\n      "lon": 37.7189245,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11370208",\n      "kind": "kindergarten",\n      "name": "Детский сад №115",\n      "lat": 55.7825002,\n      "lon": 37.6621864,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11478852",\n      "kind": "kindergarten",\n      "name": "ЧДОУ Детский сад №36 ОАО РЖД",\n      "lat": 55.7991085,\n      "lon": 37.6519322,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11490679",\n      "kind": "kindergarten",\n      "name": "Школа № 118. Дошкольное отделение",\n      "lat": 55.6872676,\n      "lon": 37.5311302,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11545654",\n      "kind": "kindergarten",\n      "name": "Brookes Life",\n      "lat": 55.8484077,\n      "lon": 37.6300392,\n      "opening_hours": "Mo-Fr 07:30-18:00"\n    },\n    {\n      "id": "relation/11550302",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8272414,\n      "lon": 37.6524778,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11561771",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.6976818,\n      "lon": 37.4918342,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11565410",\n      "kind": "school",\n      "name": "Школа №2094. Школьная образовательная площадка №1",\n      "lat": 55.6207443,\n      "lon": 37.4765156,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11601689",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa0624",\n      "lat": 55.8111301,\n      "lon": 37.574118,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11601690",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8108034,\n      "lon": 37.5747916,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11601691",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.810404,\n      "lon": 37.5754206,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11648361",\n      "kind": "kindergarten",\n      "name": "ГБУ Школа №1540 дошкольное отделение",\n      "lat": 55.7906654,\n      "lon": 37.5928186,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11654990",\n      "kind": "school",\n      "name": "Школа №1250",\n      "lat": 55.8293519,\n      "lon": 37.4846268,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11655326",\n      "kind": "school",\n      "name": "Школа № 46",\n      "lat": 55.6578013,\n      "lon": 37.5110612,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11679771",\n      "kind": "school",\n      "name": "Лицей №1502. Корпус \\"Гамма\\"",\n      "lat": 55.7980676,\n      "lon": 37.707357,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11698101",\n      "kind": "school",\n      "name": "Школа с углубленным изучением иностранного языка № 1236 имени С.В. Милашенкова. Школьное Отделение №230",\n      "lat": 55.8235514,\n      "lon": 37.5825689,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11730496",\n      "kind": "kindergarten",\n      "name": "Школа № 627 имени генерала Д.Д. Лелюшенко. Учебный корпус № 9",\n      "lat": 55.7327132,\n      "lon": 37.616815,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11749423",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7987357,\n      "lon": 37.5407179,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11749425",\n      "kind": "school",\n      "name": "i-Школа",\n      "lat": 55.8001555,\n      "lon": 37.5419484,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11750181",\n      "kind": "school",\n      "name": "Петровская школа",\n      "lat": 55.7973568,\n      "lon": 37.5635229,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11753300",\n      "kind": "kindergarten",\n      "name": "Детский сад 1920",\n      "lat": 55.7931333,\n      "lon": 37.5834959,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11778070",\n      "kind": "school",\n      "name": "ГБОУ Школа № 293 им. А. Т. Твардовского",\n      "lat": 55.8220118,\n      "lon": 37.6530343,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11805313",\n      "kind": "school",\n      "name": "Частная школа «Академическая гимназия»",\n      "lat": 55.8127078,\n      "lon": 37.6828974,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11807076",\n      "kind": "school",\n      "name": "Школа Марьина Роща имени В. Ф. Орлова. Отделение Берендеево царство",\n      "lat": 55.7938728,\n      "lon": 37.6216194,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11855555",\n      "kind": "school",\n      "name": "Пушкинская школа № 1500",\n      "lat": 55.7809283,\n      "lon": 37.6635559,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11880178",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1797 \\"Богородская\\" Дошкольное отделение",\n      "lat": 55.811311,\n      "lon": 37.7139661,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11909217",\n      "kind": "school",\n      "name": "Школа № 2107. Здание школы \\"Школа им. В. М. Комарова\\"",\n      "lat": 55.7875347,\n      "lon": 37.6303219,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11913160",\n      "kind": "school",\n      "name": "Школа №2200 корпус 2",\n      "lat": 55.8088228,\n      "lon": 37.8011446,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11951886",\n      "kind": "school",\n      "name": "Школа № 1297 основного и среднего общего образования",\n      "lat": 55.7875579,\n      "lon": 37.6371635,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11956021",\n      "kind": "school",\n      "name": "Школа №2200 корпус 9",\n      "lat": 55.8083187,\n      "lon": 37.8147277,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11956022",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 12 (дошкольное отделение)",\n      "lat": 55.8093652,\n      "lon": 37.8146346,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11956023",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 13 (дошкольное отделение)",\n      "lat": 55.8094517,\n      "lon": 37.8159521,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11970107",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 4 Семицветик",\n      "lat": 55.7637842,\n      "lon": 37.5646012,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11977541",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8638641,\n      "lon": 37.5355138,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11996436",\n      "kind": "kindergarten",\n      "name": "ЦРР-д/с №780",\n      "lat": 55.8301113,\n      "lon": 37.7632098,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11997800",\n      "kind": "kindergarten",\n      "name": "Центр образования №345 (детский сад)",\n      "lat": 55.7743628,\n      "lon": 37.6757305,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12003323",\n      "kind": "kindergarten",\n      "name": "Школа №1429 имени Героя Советского Союза Н.А. Боброва (Дошкольное отделение)",\n      "lat": 55.7777536,\n      "lon": 37.682845,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12102755",\n      "kind": "school",\n      "name": "Школа \\"Венда\\"",\n      "lat": 55.8154275,\n      "lon": 37.6156453,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12106667",\n      "kind": "school",\n      "name": "Школа №315",\n      "lat": 55.7829817,\n      "lon": 37.6744233,\n      "opening_hours": "Mo-Su 07:30-21:30"\n    },\n    {\n      "id": "relation/12110737",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6009517,\n      "lon": 37.7308125,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12117271",\n      "kind": "kindergarten",\n      "name": "Центр образования №1601 (детский сад)",\n      "lat": 55.7963101,\n      "lon": 37.5613984,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12188454",\n      "kind": "kindergarten",\n      "name": "Детский сад AcademKids",\n      "lat": 55.8745656,\n      "lon": 37.5975229,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "relation/12217953",\n      "kind": "kindergarten",\n      "name": "Центр развития ребенка №430",\n      "lat": 55.7963431,\n      "lon": 37.5740919,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12223764",\n      "kind": "kindergarten",\n      "name": "Школа №1499 Дошкольное отделение",\n      "lat": 55.8380287,\n      "lon": 37.6517619,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12223765",\n      "kind": "school",\n      "name": "Школа №1499",\n      "lat": 55.8373491,\n      "lon": 37.6530058,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12231186",\n      "kind": "kindergarten",\n      "name": "Лучик",\n      "lat": 55.807892,\n      "lon": 37.5575178,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12250223",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.7017285,\n      "lon": 37.5027379,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12278837",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6936683,\n      "lon": 37.5727668,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12285338",\n      "kind": "kindergarten",\n      "name": "Детский сад №525",\n      "lat": 55.7966548,\n      "lon": 37.5648636,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12294630",\n      "kind": "kindergarten",\n      "name": "ДОП \\"Красностуденческая 2\\"",\n      "lat": 55.8268461,\n      "lon": 37.5675593,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12294632",\n      "kind": "school",\n      "name": "Кадетская школа-интернат № 1 \\"Первый Московский кадетский корпус",\n      "lat": 55.8151469,\n      "lon": 37.5590016,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12314255",\n      "kind": "school",\n      "name": "Школа №285 имени В.А. Молодцова",\n      "lat": 55.8771321,\n      "lon": 37.6182724,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12314803",\n      "kind": "kindergarten",\n      "name": "Остров сокровищ",\n      "lat": 55.8213995,\n      "lon": 37.5694061,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12344755",\n      "kind": "kindergarten",\n      "name": "Школа № 1259. Дошкольное отделение",\n      "lat": 55.7317743,\n      "lon": 37.6341318,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12347999",\n      "kind": "school",\n      "name": "Средняя общеобразовательная лингвистическая школа",\n      "lat": 55.7832479,\n      "lon": 37.6608182,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12360702",\n      "kind": "school",\n      "name": "Школа № 1228. Корпус № 4",\n      "lat": 55.7636365,\n      "lon": 37.7079688,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12367974",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа №1531\\", корпус 2",\n      "lat": 55.8085781,\n      "lon": 37.6312862,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12380317",\n      "kind": "school",\n      "name": "ГБОУ Инженерно-техническая школа имени дважды героя Советского Союза Павла Романовича Поповича",\n      "lat": 55.8097084,\n      "lon": 37.531497,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12380320",\n      "kind": "kindergarten",\n      "name": "Школа № 1601 имени Героя Советского Союза Е. К. Лютикова. Дошкольное отделение",\n      "lat": 55.8021578,\n      "lon": 37.556378,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12380321",\n      "kind": "school",\n      "name": "Гимназия Свет",\n      "lat": 55.8019017,\n      "lon": 37.5557474,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12417771",\n      "kind": "school",\n      "name": "Гимназия №1579",\n      "lat": 55.6468364,\n      "lon": 37.6595344,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12476264",\n      "kind": "kindergarten",\n      "name": "Детский сад №694",\n      "lat": 55.7746236,\n      "lon": 37.6461949,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12533163",\n      "kind": "kindergarten",\n      "name": "Детский сад №1727",\n      "lat": 55.7773447,\n      "lon": 37.5537523,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12580079",\n      "kind": "kindergarten",\n      "name": "Школа №1282 \\"Сокольники\\" (дошкольное отделение 3)",\n      "lat": 55.7950346,\n      "lon": 37.6561801,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12580080",\n      "kind": "kindergarten",\n      "name": "Школа №1282 \\"Сокольники\\" (дошкольное отделение 4)",\n      "lat": 55.7963166,\n      "lon": 37.655646,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12644180",\n      "kind": "kindergarten",\n      "name": "Школа № 345 имени А.С. Пушкина. Дошкольное отделение",\n      "lat": 55.7654735,\n      "lon": 37.6558909,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12661654",\n      "kind": "kindergarten",\n      "name": "Детский сад №306",\n      "lat": 55.7733982,\n      "lon": 37.7349016,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12661655",\n      "kind": "school",\n      "name": "Школа №1301 имени Е.Т. Гайдара",\n      "lat": 55.7731515,\n      "lon": 37.7338592,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12688429",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1290. Дошкольное отделение № 4/1",\n      "lat": 55.7982156,\n      "lon": 37.7635035,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12689325",\n      "kind": "school",\n      "name": "Школа № 547",\n      "lat": 55.5911562,\n      "lon": 37.4594967,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12704262",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е.В. Михайлова. Школьный корпус Рязанский",\n      "lat": 55.7187639,\n      "lon": 37.7933316,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12717206",\n      "kind": "kindergarten",\n      "name": "ДОУ №1733",\n      "lat": 55.7678399,\n      "lon": 37.6708091,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12730222",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8098502,\n      "lon": 37.7614967,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12739902",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Интеграл\\"",\n      "lat": 55.7083936,\n      "lon": 37.6192842,\n      "opening_hours": "Mo-Fr 07:30-19:30"\n    },\n    {\n      "id": "relation/12745351",\n      "kind": "school",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №1",\n      "lat": 55.8524168,\n      "lon": 37.6906082,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12745352",\n      "kind": "school",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №2",\n      "lat": 55.8515144,\n      "lon": 37.6883851,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12773236",\n      "kind": "kindergarten",\n      "name": "Детский сад №863",\n      "lat": 55.7093416,\n      "lon": 37.469002,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12921870",\n      "kind": "school",\n      "name": "ГБОУ школа на Юго-Востоке имени Маршала В.И. Чуйкова",\n      "lat": 55.6970664,\n      "lon": 37.7725554,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12966936",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №20",\n      "lat": 55.8253601,\n      "lon": 37.8105283,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12966937",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №18",\n      "lat": 55.8260773,\n      "lon": 37.8104018,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12966938",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №16",\n      "lat": 55.8249313,\n      "lon": 37.818907,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12966939",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №13",\n      "lat": 55.8257435,\n      "lon": 37.8188648,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12969784",\n      "kind": "school",\n      "name": "",\n      "lat": 55.65662,\n      "lon": 37.7559216,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12969785",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №10",\n      "lat": 55.6561894,\n      "lon": 37.7584447,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12969787",\n      "kind": "school",\n      "name": "Школа № 1357 «На Братиславской». Здание №15",\n      "lat": 55.666917,\n      "lon": 37.7775141,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980861",\n      "kind": "school",\n      "name": "Шуваловская школа № 1448. Корпус «Мосфильмовский»",\n      "lat": 55.7208063,\n      "lon": 37.5116901,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980862",\n      "kind": "kindergarten",\n      "name": "Шуваловская школа № 1448. Корпус «Мосфильмовский»",\n      "lat": 55.7213245,\n      "lon": 37.5126873,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980864",\n      "kind": "school",\n      "name": "Школа №37. Корпус школьного образования",\n      "lat": 55.6964498,\n      "lon": 37.4909324,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980875",\n      "kind": "kindergarten",\n      "name": "Шуваловская школа № 1448. Корпус «Шуваловский»",\n      "lat": 55.7030495,\n      "lon": 37.5130607,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980879",\n      "kind": "school",\n      "name": "Шуваловская школа № 1448. Основной корпус «Шуваловский»",\n      "lat": 55.7025401,\n      "lon": 37.5126036,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980882",\n      "kind": "kindergarten",\n      "name": "Шуваловская школа № 1448. Корпус «Академия детства»",\n      "lat": 55.7017511,\n      "lon": 37.5130444,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994086",\n      "kind": "school",\n      "name": "Школа № 1248. Центр «Давыдково»",\n      "lat": 55.7238334,\n      "lon": 37.4664046,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994088",\n      "kind": "kindergarten",\n      "name": "Детский сад №553",\n      "lat": 55.7231641,\n      "lon": 37.4636622,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994089",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Здание дошкольных групп 7",\n      "lat": 55.7232068,\n      "lon": 37.4646303,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994090",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Здание дошкольных групп 5",\n      "lat": 55.7233884,\n      "lon": 37.4746995,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994098",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Здание дошкольных групп 4",\n      "lat": 55.7252384,\n      "lon": 37.4766505,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994102",\n      "kind": "school",\n      "name": "Школа №1248. Школьное здание 1",\n      "lat": 55.7256354,\n      "lon": 37.4736251,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12998410",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение 3/5",\n      "lat": 55.6297755,\n      "lon": 37.4880568,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12998411",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение 3/4",\n      "lat": 55.6301752,\n      "lon": 37.4894585,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12998414",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение 3/8",\n      "lat": 55.6344312,\n      "lon": 37.4864553,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004333",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №4",\n      "lat": 55.6122302,\n      "lon": 37.548642,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004334",\n      "kind": "kindergarten",\n      "name": "Школа № 1694 «Ясенево». Дошкольное отделение 6",\n      "lat": 55.6119901,\n      "lon": 37.5478155,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004335",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус школьного образования №10 «Наш дом»",\n      "lat": 55.6134269,\n      "lon": 37.5517052,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004336",\n      "kind": "kindergarten",\n      "name": "Школа № 1694 «Ясенево». Дошкольное отделение 5",\n      "lat": 55.613174,\n      "lon": 37.5509399,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004340",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6142246,\n      "lon": 37.542505,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004341",\n      "kind": "kindergarten",\n      "name": "Школа № 1694 «Ясенево». Дошкольное отделение 4",\n      "lat": 55.6147396,\n      "lon": 37.5428393,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13008017",\n      "kind": "school",\n      "name": "Школа № 17. Школьное здание №3",\n      "lat": 55.6353393,\n      "lon": 37.5283704,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13008020",\n      "kind": "school",\n      "name": "",\n      "lat": 55.636754,\n      "lon": 37.5315408,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13010684",\n      "kind": "school",\n      "name": "Школа №49. Школьный корпус",\n      "lat": 55.645723,\n      "lon": 37.5360213,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13011079",\n      "kind": "school",\n      "name": "Школа № 1273. Корпус школьного образования №8",\n      "lat": 55.6307482,\n      "lon": 37.5239063,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13024797",\n      "kind": "school",\n      "name": "Школа № 113. Корпус №2",\n      "lat": 55.6425203,\n      "lon": 37.5143281,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13024800",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №5",\n      "lat": 55.6439453,\n      "lon": 37.5176007,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13025525",\n      "kind": "school",\n      "name": "Школа № 1507. Корпус школьного образования",\n      "lat": 55.6270586,\n      "lon": 37.5042155,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13025527",\n      "kind": "school",\n      "name": "Школа № 1507. Корпус школьного образования",\n      "lat": 55.6303727,\n      "lon": 37.5065736,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13037120",\n      "kind": "kindergarten",\n      "name": "Детский сад «Белый кролик»",\n      "lat": 55.5914152,\n      "lon": 37.4622392,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13051360",\n      "kind": "kindergarten",\n      "name": "Школа №1995. Дошкольный уровень",\n      "lat": 55.6467494,\n      "lon": 37.4961936,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13051973",\n      "kind": "school",\n      "name": "Школа №109. Основная школа",\n      "lat": 55.6407698,\n      "lon": 37.4753687,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13057844",\n      "kind": "school",\n      "name": "Школа на Яузе корпус 7",\n      "lat": 55.7735435,\n      "lon": 37.7074005,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13075779",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6233417,\n      "lon": 37.4867878,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13075780",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/7",\n      "lat": 55.6236575,\n      "lon": 37.4878849,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13101201",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.7029865,\n      "lon": 37.5005314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151519",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №1",\n      "lat": 55.6048443,\n      "lon": 37.5476809,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151520",\n      "kind": "school",\n      "name": "Британская Международная школа",\n      "lat": 55.6046949,\n      "lon": 37.5487683,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151521",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №4",\n      "lat": 55.6040143,\n      "lon": 37.5517382,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151522",\n      "kind": "kindergarten",\n      "name": "детский сад №1461",\n      "lat": 55.6038862,\n      "lon": 37.5528075,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151527",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №7",\n      "lat": 55.6060392,\n      "lon": 37.5522041,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151528",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №8",\n      "lat": 55.6058516,\n      "lon": 37.5539773,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151529",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №9",\n      "lat": 55.6081561,\n      "lon": 37.5423683,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151530",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №10",\n      "lat": 55.607317,\n      "lon": 37.5420445,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13176656",\n      "kind": "kindergarten",\n      "name": "Школа № 627 имени генерала Д. Д. Лелюшенко. Учебный корпус № 8",\n      "lat": 55.7261421,\n      "lon": 37.620965,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179273",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус школьного образования №2",\n      "lat": 55.6101516,\n      "lon": 37.5290254,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179274",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус школьного образования №1",\n      "lat": 55.610966,\n      "lon": 37.5295453,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179275",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус начального образования №3 «Ресурсный центр»",\n      "lat": 55.6164806,\n      "lon": 37.5372548,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179276",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №6",\n      "lat": 55.6165483,\n      "lon": 37.5363215,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179803",\n      "kind": "kindergarten",\n      "name": "Школа Тропарево. Корпус дошкольного образования",\n      "lat": 55.6524151,\n      "lon": 37.4685564,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179804",\n      "kind": "school",\n      "name": "Школа Тропарево. Корпус школьного образования",\n      "lat": 55.6534828,\n      "lon": 37.4692649,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13180366",\n      "kind": "school",\n      "name": "Гимназия №201",\n      "lat": 55.8213088,\n      "lon": 37.5049687,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13188995",\n      "kind": "school",\n      "name": "Школа № 2107. Физико-математическая школа",\n      "lat": 55.7821585,\n      "lon": 37.6350583,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13211681",\n      "kind": "kindergarten",\n      "name": "Школа №556. Корпус дошкольного образования",\n      "lat": 55.6215044,\n      "lon": 37.5898665,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13211682",\n      "kind": "kindergarten",\n      "name": "Школа №556. Корпус дошкольного образования",\n      "lat": 55.6212446,\n      "lon": 37.5915229,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13234192",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО №3",\n      "lat": 55.7078821,\n      "lon": 37.675786,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13248115",\n      "kind": "school",\n      "name": "Школа №875. Корпус школьного образования",\n      "lat": 55.6638663,\n      "lon": 37.4911326,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13262831",\n      "kind": "kindergarten",\n      "name": "Детское посольство",\n      "lat": 55.8311798,\n      "lon": 37.6162923,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13347424",\n      "kind": "school",\n      "name": "Школа № 324 «Жар-птица». Корпус школьного образования «Журавушка»",\n      "lat": 55.6745677,\n      "lon": 37.4930911,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13347425",\n      "kind": "kindergarten",\n      "name": "Школа № 324 «Жар-птица». Корпус дошкольного образования «Соловушка»",\n      "lat": 55.6738448,\n      "lon": 37.4930525,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13372228",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №6",\n      "lat": 55.6103211,\n      "lon": 37.594732,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13372229",\n      "kind": "school",\n      "name": "Школа № 1582. Корпус школьного образования",\n      "lat": 55.6093183,\n      "lon": 37.5969963,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13401623",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №9",\n      "lat": 55.59885,\n      "lon": 37.6115738,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13401624",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5986382,\n      "lon": 37.6129509,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13422176",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6054818,\n      "lon": 37.6181761,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13443141",\n      "kind": "kindergarten",\n      "name": "Гимназия № 2027. Дошкольное отделение № 3",\n      "lat": 55.7476059,\n      "lon": 37.7956939,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13470035",\n      "kind": "school",\n      "name": "Школа № 2107. Физико-математическая школа",\n      "lat": 55.7809762,\n      "lon": 37.6427738,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13472246",\n      "kind": "school",\n      "name": "Школа №1179. Корпус школьного образования №3",\n      "lat": 55.6378887,\n      "lon": 37.6132711,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13472247",\n      "kind": "kindergarten",\n      "name": "Детский сад № 853",\n      "lat": 55.6377669,\n      "lon": 37.6145602,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13492626",\n      "kind": "school",\n      "name": "Школа на Юго-Востоке имени Маршала В.И. Чуйкова",\n      "lat": 55.7118079,\n      "lon": 37.7669083,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13492627",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа имени Маршала Чуйкова, здание № 3",\n      "lat": 55.7119586,\n      "lon": 37.7644685,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13495476",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6499319,\n      "lon": 37.5740018,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13495477",\n      "kind": "school",\n      "name": "Школа № 538 имени С.В. Гришина. Корпус школьного образования",\n      "lat": 55.6500286,\n      "lon": 37.572277,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13496928",\n      "kind": "school",\n      "name": "Школа № 538 имени С.В. Гришина. Корпус школьного образования",\n      "lat": 55.6519599,\n      "lon": 37.5763527,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13496929",\n      "kind": "school",\n      "name": "Школа № 538 имени С.В. Гришина. Главный корпус школьного образования",\n      "lat": 55.6524845,\n      "lon": 37.577197,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13496986",\n      "kind": "kindergarten",\n      "name": "Школа № 538 имени С.В. Гришина. Корпус дошкольного образования",\n      "lat": 55.6531645,\n      "lon": 37.5731206,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13512774",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.841191,\n      "lon": 37.4974225,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13524468",\n      "kind": "school",\n      "name": "Школа № 1245. Корпус школьного образования №1",\n      "lat": 55.5997889,\n      "lon": 37.593382,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13524469",\n      "kind": "school",\n      "name": "Центр образования №504",\n      "lat": 55.6000823,\n      "lon": 37.5916094,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13549475",\n      "kind": "school",\n      "name": "Школа № 667. Корпус школьного образования №4",\n      "lat": 55.5837166,\n      "lon": 37.6427537,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13549476",\n      "kind": "kindergarten",\n      "name": "Школа № 667. Корпус дошкольного образования №6",\n      "lat": 55.5850103,\n      "lon": 37.6428602,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13549477",\n      "kind": "school",\n      "name": "Школа № 667. Корпус школьного образования №5",\n      "lat": 55.5848141,\n      "lon": 37.6416728,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13575969",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №3",\n      "lat": 55.6464016,\n      "lon": 37.6144554,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13575970",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №2",\n      "lat": 55.6478098,\n      "lon": 37.6144253,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13606724",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус школьного образования №1",\n      "lat": 55.6315816,\n      "lon": 37.5945193,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13653023",\n      "kind": "school",\n      "name": "Центр образования и спорта \\"Москва-98\\". Школа № 1",\n      "lat": 55.6846361,\n      "lon": 37.4943333,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13688258",\n      "kind": "kindergarten",\n      "name": "Школа № 1536. Корпус дошкольного образования",\n      "lat": 55.6810882,\n      "lon": 37.5435718,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13688266",\n      "kind": "kindergarten",\n      "name": "Школа № 1536. Корпус дошкольного образования",\n      "lat": 55.6815899,\n      "lon": 37.5421721,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13688421",\n      "kind": "school",\n      "name": "Школа № 7. Подразделение ул. Панфёрова, дом 8 к1",\n      "lat": 55.6811038,\n      "lon": 37.540903,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13719712",\n      "kind": "school",\n      "name": "Школа № 324 «Жар-птица». Корпус школьного образования «Жар-птица»",\n      "lat": 55.6730663,\n      "lon": 37.4928015,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13719919",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №1",\n      "lat": 55.6757263,\n      "lon": 37.5133521,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13719951",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №2",\n      "lat": 55.6751362,\n      "lon": 37.5143975,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13768991",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6697985,\n      "lon": 37.7222908,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13778835",\n      "kind": "kindergarten",\n      "name": "Детский сад «Горница-узорница»",\n      "lat": 55.7613907,\n      "lon": 37.5491346,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13802230",\n      "kind": "school",\n      "name": "ЦО №354",\n      "lat": 55.7694638,\n      "lon": 37.6779451,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13851281",\n      "kind": "school",\n      "name": "Школа №436",\n      "lat": 55.7121531,\n      "lon": 37.8057751,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13851369",\n      "kind": "kindergarten",\n      "name": "Центр образования №1998",\n      "lat": 55.6345523,\n      "lon": 37.7394872,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13874935",\n      "kind": "school",\n      "name": "Школа при посольстве Индии",\n      "lat": 55.7467413,\n      "lon": 37.5632821,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13936495",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8117601,\n      "lon": 37.739361,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13947476",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8036435,\n      "lon": 37.5964321,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13950665",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8012808,\n      "lon": 37.5994677,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13983628",\n      "kind": "school",\n      "name": "Школа №522",\n      "lat": 55.6823227,\n      "lon": 37.5967786,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14060443",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение № 6",\n      "lat": 55.8621308,\n      "lon": 37.6981965,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14062188",\n      "kind": "kindergarten",\n      "name": "Школа №1503 Дошкольное отделение",\n      "lat": 55.8093945,\n      "lon": 37.6522298,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14075840",\n      "kind": "school",\n      "name": "Школа № 1503",\n      "lat": 55.8131357,\n      "lon": 37.6548427,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14075841",\n      "kind": "kindergarten",\n      "name": "Школа № 1503 Дошкольное отделение",\n      "lat": 55.8137677,\n      "lon": 37.6546553,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14080267",\n      "kind": "school",\n      "name": "Гимназия №1551 корпус №8 (начальная школа и основное общее)",\n      "lat": 55.8395614,\n      "lon": 37.4566219,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14097751",\n      "kind": "school",\n      "name": "Лицей № 1581 имени Ивана Бунина",\n      "lat": 55.7789394,\n      "lon": 37.6955034,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14097752",\n      "kind": "school",\n      "name": "Школа №1204",\n      "lat": 55.7728316,\n      "lon": 37.6914015,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14108489",\n      "kind": "kindergarten",\n      "name": "Планета детства",\n      "lat": 55.7326403,\n      "lon": 37.6316154,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14124596",\n      "kind": "school",\n      "name": "Школа № 518",\n      "lat": 55.7438014,\n      "lon": 37.6371272,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14134782",\n      "kind": "kindergarten",\n      "name": "Детский сад №30",\n      "lat": 55.776776,\n      "lon": 37.6857492,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14134784",\n      "kind": "kindergarten",\n      "name": "Детский сад №1308",\n      "lat": 55.772204,\n      "lon": 37.7104349,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14179520",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7504759,\n      "lon": 37.5166289,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14227468",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6580704,\n      "lon": 37.5583023,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14237997",\n      "kind": "school",\n      "name": "Школа № 1362",\n      "lat": 55.7819004,\n      "lon": 37.7323858,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14275411",\n      "kind": "school",\n      "name": "КиберШкола",\n      "lat": 55.8070538,\n      "lon": 37.7865574,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14275412",\n      "kind": "kindergarten",\n      "name": "Центр \\"Исток\\"",\n      "lat": 55.8070472,\n      "lon": 37.7852393,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14328396",\n      "kind": "school",\n      "name": "Школа № 1236 имени С.В. Милашенкова. Учебный корпус № 963",\n      "lat": 55.8248631,\n      "lon": 37.5758929,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14328397",\n      "kind": "kindergarten",\n      "name": "Школа № 1236 имени С.В. Милашенкова. Учебный корпус № 567",\n      "lat": 55.8252687,\n      "lon": 37.5763221,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14331020",\n      "kind": "kindergarten",\n      "name": "Школа № 158. Дошкольное отделение",\n      "lat": 55.8686474,\n      "lon": 37.4703356,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14331021",\n      "kind": "school",\n      "name": "Школа № 158. Школьное отделение № 4",\n      "lat": 55.867572,\n      "lon": 37.4707942,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14331022",\n      "kind": "school",\n      "name": "Школа № 158. Школьное отделение № 1",\n      "lat": 55.8681307,\n      "lon": 37.4684484,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14331375",\n      "kind": "school",\n      "name": "Школа № 1474. Подразделение 4",\n      "lat": 55.8721899,\n      "lon": 37.4900897,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14333603",\n      "kind": "school",\n      "name": "Школа №2098 корпус 5",\n      "lat": 55.8788068,\n      "lon": 37.5104288,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14336312",\n      "kind": "school",\n      "name": "Школа №\xa067. Подразделение №\xa02",\n      "lat": 55.7367153,\n      "lon": 37.528245,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14352898",\n      "kind": "kindergarten",\n      "name": "Школа № 2104 на Таганке. Корпус 1",\n      "lat": 55.7445795,\n      "lon": 37.6458444,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14358472",\n      "kind": "school",\n      "name": "Школа №1474",\n      "lat": 55.8707703,\n      "lon": 37.4913039,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14374244",\n      "kind": "school",\n      "name": "Пушкинская школа № 1500. Отделение \\"Сретенка\\"",\n      "lat": 55.769069,\n      "lon": 37.6328317,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14374917",\n      "kind": "school",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 3",\n      "lat": 55.7355099,\n      "lon": 37.5979576,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14389122",\n      "kind": "kindergarten",\n      "name": "Детский сад «Горница-Узорница»",\n      "lat": 55.7906043,\n      "lon": 37.4760979,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14397087",\n      "kind": "kindergarten",\n      "name": "Маленькая страна",\n      "lat": 55.8789834,\n      "lon": 37.539014,\n      "opening_hours": "Mo-Fr 07:00-20:00"\n    },\n    {\n      "id": "relation/14397259",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Чудо городок\\"",\n      "lat": 55.8779173,\n      "lon": 37.5386431,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14418711",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8160949,\n      "lon": 37.7524524,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14573448",\n      "kind": "school",\n      "name": "Школа №57 (здание №1)",\n      "lat": 55.7487141,\n      "lon": 37.6050056,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14585413",\n      "kind": "school",\n      "name": "W.I.D.E. School",\n      "lat": 55.8755887,\n      "lon": 37.6008585,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14585594",\n      "kind": "kindergarten",\n      "name": "Школа № 1465 имени адмирала Н.Г. Кузнецова. Дошкольное отделение № 4",\n      "lat": 55.7415939,\n      "lon": 37.5418569,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14598740",\n      "kind": "kindergarten",\n      "name": "W.I.D.E.",\n      "lat": 55.8516147,\n      "lon": 37.6191599,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14599775",\n      "kind": "kindergarten",\n      "name": "Школа № 1499 имени Героя Советского Союза Ивана Архиповича Докукина. Дошкольное отделение № 8",\n      "lat": 55.8354344,\n      "lon": 37.6414028,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14606511",\n      "kind": "school",\n      "name": "Школа № 120",\n      "lat": 55.6881157,\n      "lon": 37.5510905,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14635035",\n      "kind": "kindergarten",\n      "name": "Школа им. Ф. М. Достоевского. Учебный корпус № 9",\n      "lat": 55.6616762,\n      "lon": 37.7278496,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14663499",\n      "kind": "kindergarten",\n      "name": "Школа № 1465 имени адмирала Н.Г. Кузнецова. Здание на Студенческой, 29",\n      "lat": 55.7395482,\n      "lon": 37.5467823,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14663500",\n      "kind": "school",\n      "name": "Школа № 1465 имени адмирала Н.Г. Кузнецова. Здание на Студенческой, 29",\n      "lat": 55.7400871,\n      "lon": 37.5456408,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14680298",\n      "kind": "kindergarten",\n      "name": "Школа № 1530 Дошкольная группа",\n      "lat": 55.7849156,\n      "lon": 37.6764424,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14926150",\n      "kind": "kindergarten",\n      "name": "Детский сад «Sun School»",\n      "lat": 55.6828952,\n      "lon": 37.5824489,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14956586",\n      "kind": "school",\n      "name": "Центр образования №345",\n      "lat": 55.7731486,\n      "lon": 37.6724634,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14956589",\n      "kind": "school",\n      "name": "Школа №\xa0345 им. А.\xa0С.\xa0Пушкина",\n      "lat": 55.7707133,\n      "lon": 37.6775854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14962273",\n      "kind": "school",\n      "name": "Школа № 158. Школьное отделение № 3",\n      "lat": 55.8735021,\n      "lon": 37.4674782,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14962274",\n      "kind": "school",\n      "name": "Школа № 158. Школьное отделение № 2",\n      "lat": 55.8735256,\n      "lon": 37.4652311,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14962275",\n      "kind": "kindergarten",\n      "name": "Школа № 158. Дошкольное отделение № 2",\n      "lat": 55.8729118,\n      "lon": 37.4659243,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14962523",\n      "kind": "kindergarten",\n      "name": "Детский сад «Академическая гимназия»",\n      "lat": 55.6010535,\n      "lon": 37.7261168,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14977370",\n      "kind": "school",\n      "name": "Школа № 1501. Многопрофильные классы на Тихвинской",\n      "lat": 55.7916642,\n      "lon": 37.5982605,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14978852",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6014696,\n      "lon": 37.7335755,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14983573",\n      "kind": "kindergarten",\n      "name": "Частный детский сад «Горница-Узорница»",\n      "lat": 55.6691946,\n      "lon": 37.7717367,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14989885",\n      "kind": "school",\n      "name": "Лицей №1550, подр №2",\n      "lat": 55.7853699,\n      "lon": 37.5586262,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15001788",\n      "kind": "school",\n      "name": "Школа №152",\n      "lat": 55.8008186,\n      "lon": 37.5352032,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15013450",\n      "kind": "school",\n      "name": "Школа №1690",\n      "lat": 55.7926731,\n      "lon": 37.7251975,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15040197",\n      "kind": "kindergarten",\n      "name": "ONE! International School",\n      "lat": 55.7870994,\n      "lon": 37.4961103,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "relation/15045970",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7870087,\n      "lon": 37.619206,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15046320",\n      "kind": "kindergarten",\n      "name": "Детский сад №2718",\n      "lat": 55.7871198,\n      "lon": 37.599392,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15049348",\n      "kind": "kindergarten",\n      "name": "ONE! International School",\n      "lat": 55.786609,\n      "lon": 37.4937838,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "relation/15049712",\n      "kind": "school",\n      "name": "Школа №1287 (2)",\n      "lat": 55.7988898,\n      "lon": 37.5254497,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15049713",\n      "kind": "school",\n      "name": "Школа №1384 имени А. А. Леманского",\n      "lat": 55.7982404,\n      "lon": 37.5234806,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15087742",\n      "kind": "kindergarten",\n      "name": "Детский сад №864",\n      "lat": 55.8328953,\n      "lon": 37.5852464,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15087743",\n      "kind": "kindergarten",\n      "name": "Детский сад №2154",\n      "lat": 55.8328628,\n      "lon": 37.5871574,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15284663",\n      "kind": "school",\n      "name": "Школа №1228 \\"Лефортово\\"",\n      "lat": 55.7453136,\n      "lon": 37.7039961,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15285093",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7504735,\n      "lon": 37.7040729,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15326688",\n      "kind": "kindergarten",\n      "name": "Детский сад №2663",\n      "lat": 55.7747433,\n      "lon": 37.6800078,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15348984",\n      "kind": "kindergarten",\n      "name": "Детский сад",\n      "lat": 55.8610406,\n      "lon": 37.5398508,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15349064",\n      "kind": "school",\n      "name": "Школа №1236",\n      "lat": 55.8174682,\n      "lon": 37.582646,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15450200",\n      "kind": "school",\n      "name": "Центр образования №1480",\n      "lat": 55.7672761,\n      "lon": 37.6709812,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15451606",\n      "kind": "kindergarten",\n      "name": "ДОУ №\xa02555",\n      "lat": 55.7672952,\n      "lon": 37.6760037,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15518043",\n      "kind": "kindergarten",\n      "name": "Детский сад №1253",\n      "lat": 55.8185327,\n      "lon": 37.5635915,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15543373",\n      "kind": "school",\n      "name": "Бауманская инженерная школа № 1580. Корпус 3",\n      "lat": 55.7137027,\n      "lon": 37.5953765,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15671989",\n      "kind": "school",\n      "name": "Школа «Алгоритм»",\n      "lat": 55.7647665,\n      "lon": 37.6332242,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15671999",\n      "kind": "school",\n      "name": "Французский лицей им. А. Дюма при посольстве Франции",\n      "lat": 55.7630565,\n      "lon": 37.6316021,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15691315",\n      "kind": "school",\n      "name": "Школа №285 имени В.А. Молодцова, дополнительное образование",\n      "lat": 55.8780127,\n      "lon": 37.6245895,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15697831",\n      "kind": "school",\n      "name": "Школа № 2104 на Таганке. Корпус 2",\n      "lat": 55.7436285,\n      "lon": 37.6462823,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15700997",\n      "kind": "school",\n      "name": "Школа № 1271",\n      "lat": 55.7421669,\n      "lon": 37.6488379,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15710120",\n      "kind": "school",\n      "name": "Кадетский корпус Следственного комитета РФ им. Александра Невского",\n      "lat": 55.6977469,\n      "lon": 37.7582214,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15710121",\n      "kind": "school",\n      "name": "Школа №466",\n      "lat": 55.6987181,\n      "lon": 37.757896,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15737346",\n      "kind": "kindergarten",\n      "name": "Частное общеобразовательное учреждение «Венда»",\n      "lat": 55.8491512,\n      "lon": 37.5781226,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15767588",\n      "kind": "kindergarten",\n      "name": "Школа № 56 имени академика В.А. Легасова. Дошкольный корпус",\n      "lat": 55.7435864,\n      "lon": 37.5406149,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15898872",\n      "kind": "kindergarten",\n      "name": "Детский сад №2144",\n      "lat": 55.799081,\n      "lon": 37.6172375,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15913197",\n      "kind": "kindergarten",\n      "name": "Детский сад №133",\n      "lat": 55.7192268,\n      "lon": 37.7959831,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15922271",\n      "kind": "kindergarten",\n      "name": "Школа № 1317. Дошкольное отделение",\n      "lat": 55.6707721,\n      "lon": 37.4696285,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15922272",\n      "kind": "school",\n      "name": "Школа № 1317. Начальное отделение",\n      "lat": 55.6713862,\n      "lon": 37.4705109,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15968755",\n      "kind": "school",\n      "name": "Школа № 1501. Инженерные и ИТ-классы",\n      "lat": 55.7877555,\n      "lon": 37.5957341,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16094457",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение В4",\n      "lat": 55.735661,\n      "lon": 37.8184396,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16125998",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Д.А.Р.»",\n      "lat": 55.6320413,\n      "lon": 37.5875136,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16145301",\n      "kind": "school",\n      "name": "Школа № 2099. Здание 2",\n      "lat": 55.8199537,\n      "lon": 37.4910476,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16257777",\n      "kind": "school",\n      "name": "Школа №\xa01203",\n      "lat": 55.7688158,\n      "lon": 37.6880488,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16270432",\n      "kind": "school",\n      "name": "Лицей №1574",\n      "lat": 55.7797743,\n      "lon": 37.5968479,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16330701",\n      "kind": "kindergarten",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 1",\n      "lat": 55.7431445,\n      "lon": 37.5947904,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16409165",\n      "kind": "kindergarten",\n      "name": "Школа \\"Покровский квартал\\", Детский сад №2",\n      "lat": 55.7638996,\n      "lon": 37.653529,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16409205",\n      "kind": "school",\n      "name": "Школа №\xa01388",\n      "lat": 55.776072,\n      "lon": 37.6047918,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16508347",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.681964,\n      "lon": 37.7198444,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16542182",\n      "kind": "school",\n      "name": "Татьянинская школа",\n      "lat": 55.7339582,\n      "lon": 37.7481277,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16617553",\n      "kind": "school",\n      "name": "Французский лицей им. Александра Дюма",\n      "lat": 55.7336784,\n      "lon": 37.6157276,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16769249",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7040582,\n      "lon": 37.6707193,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16844710",\n      "kind": "kindergarten",\n      "name": "Детский сад №23",\n      "lat": 55.6162384,\n      "lon": 37.6987172,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16867327",\n      "kind": "kindergarten",\n      "name": "Детский сад №2724",\n      "lat": 55.7106797,\n      "lon": 37.7393524,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16983105",\n      "kind": "kindergarten",\n      "name": "Школа № 1799. Дошкольный корпус",\n      "lat": 55.7326525,\n      "lon": 37.6099083,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17034920",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7244379,\n      "lon": 37.748801,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17057701",\n      "kind": "school",\n      "name": "Греко-латинский кабинет Ю. А. Шичалина",\n      "lat": 55.7839377,\n      "lon": 37.6402746,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17087636",\n      "kind": "kindergarten",\n      "name": "Детский сад №1392",\n      "lat": 55.8122522,\n      "lon": 37.7037854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17087637",\n      "kind": "kindergarten",\n      "name": "Детский сад №1299",\n      "lat": 55.8117206,\n      "lon": 37.7048854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17138480",\n      "kind": "school",\n      "name": "Школа № 1297 для детей с особыми образовательными потребностями",\n      "lat": 55.789311,\n      "lon": 37.6377293,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17138481",\n      "kind": "kindergarten",\n      "name": "Школа № 1297 дошкольные группы",\n      "lat": 55.7892993,\n      "lon": 37.638729,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17232824",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01252 имени Сервантеса. Начальная школа",\n      "lat": 55.8103018,\n      "lon": 37.4948845,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17237320",\n      "kind": "kindergarten",\n      "name": "Протон - дошкольные группы",\n      "lat": 55.7480479,\n      "lon": 37.5023391,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17237321",\n      "kind": "kindergarten",\n      "name": "Детский сад №514",\n      "lat": 55.7471283,\n      "lon": 37.5032456,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17287634",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Brosco Junior\\"",\n      "lat": 55.7687016,\n      "lon": 37.6253893,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17386620",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7525562,\n      "lon": 37.5101156,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17473831",\n      "kind": "school",\n      "name": "Школа № 627 имени генерала Д. Д. Лелюшенко. Учебный корпус № 4",\n      "lat": 55.7260108,\n      "lon": 37.6275672,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17499698",\n      "kind": "school",\n      "name": "Школа № 1576",\n      "lat": 55.841583,\n      "lon": 37.5419044,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17499699",\n      "kind": "kindergarten",\n      "name": "Школа №1576, дошкольное отделение",\n      "lat": 55.8408704,\n      "lon": 37.5411516,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17501327",\n      "kind": "school",\n      "name": "Школа № 1241 «На Красной Пресне». Корпус 1",\n      "lat": 55.7601254,\n      "lon": 37.5636358,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561174",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa04",\n      "lat": 55.7084812,\n      "lon": 37.7702523,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561175",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa06",\n      "lat": 55.7063442,\n      "lon": 37.7749862,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561176",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa07",\n      "lat": 55.7070212,\n      "lon": 37.7738603,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561177",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7072764,\n      "lon": 37.7717679,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561178",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa05",\n      "lat": 55.7062367,\n      "lon": 37.7737886,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17577569",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2",\n      "lat": 55.8815516,\n      "lon": 37.4949445,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17620396",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7000616,\n      "lon": 37.555451,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17626072",\n      "kind": "kindergarten",\n      "name": "Школа №1550 Дошкольное отделение",\n      "lat": 55.7929019,\n      "lon": 37.7278789,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17626073",\n      "kind": "kindergarten",\n      "name": "Детский сад №2518",\n      "lat": 55.7935209,\n      "lon": 37.7273127,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17671292",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01231 имени В.\xa0Д.\xa0Поленова. Дошкольный корпус",\n      "lat": 55.7438277,\n      "lon": 37.5801976,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17682458",\n      "kind": "school",\n      "name": "Школа №\xa01231 имени В. Д. Поленова. Школьный корпус",\n      "lat": 55.7418873,\n      "lon": 37.5807267,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17695569",\n      "kind": "kindergarten",\n      "name": "Детский сад №890",\n      "lat": 55.8074198,\n      "lon": 37.4554918,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17734461",\n      "kind": "kindergarten",\n      "name": "Детский сад №271",\n      "lat": 55.8177234,\n      "lon": 37.6423024,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17734508",\n      "kind": "school",\n      "name": "Школа №293 имени Твардовского",\n      "lat": 55.8178531,\n      "lon": 37.6404351,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17789214",\n      "kind": "kindergarten",\n      "name": "Детский сад №1132",\n      "lat": 55.8176934,\n      "lon": 37.6243185,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17807277",\n      "kind": "kindergarten",\n      "name": "Детский сад № 116",\n      "lat": 55.8468676,\n      "lon": 37.5659983,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17824771",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.756781,\n      "lon": 37.7068846,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17852619",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7195742,\n      "lon": 37.6899939,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17855597",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7185499,\n      "lon": 37.6564335,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17865216",\n      "kind": "kindergarten",\n      "name": "Детское посольство",\n      "lat": 55.8321029,\n      "lon": 37.6174232,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17865577",\n      "kind": "school",\n      "name": "Школа №1699",\n      "lat": 55.727948,\n      "lon": 37.484443,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17866091",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7184766,\n      "lon": 37.637083,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17940119",\n      "kind": "school",\n      "name": "Школа № 2107. Школа на Гиляровского",\n      "lat": 55.7776722,\n      "lon": 37.6298123,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17958851",\n      "kind": "school",\n      "name": "Школа № 1465 имени адмирала Н.Г. Кузнецова. Корпус 3",\n      "lat": 55.7428186,\n      "lon": 37.5470449,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17987677",\n      "kind": "school",\n      "name": "Лингвистический лицей № 1555 при МГЛУ",\n      "lat": 55.7374677,\n      "lon": 37.598308,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18096256",\n      "kind": "school",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 5",\n      "lat": 55.7414824,\n      "lon": 37.6033899,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18122613",\n      "kind": "school",\n      "name": "Школа №319 (здание №1)",\n      "lat": 55.8067124,\n      "lon": 37.7591769,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18133893",\n      "kind": "school",\n      "name": "Школа № 215",\n      "lat": 55.8361889,\n      "lon": 37.527125,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18134842",\n      "kind": "school",\n      "name": "Школа № 1553 имени В.И. Вернадского",\n      "lat": 55.720123,\n      "lon": 37.6138718,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18163741",\n      "kind": "kindergarten",\n      "name": "Школа № 1223. Отделение дошкольного образования №1",\n      "lat": 55.8378584,\n      "lon": 37.5291441,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18178325",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7191932,\n      "lon": 37.606956,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18212679",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Варшавке\\"",\n      "lat": 55.6969219,\n      "lon": 37.6193182,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18244733",\n      "kind": "school",\n      "name": "Школа № 444. Начальная школа",\n      "lat": 55.7900437,\n      "lon": 37.7926185,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18282668",\n      "kind": "school",\n      "name": "Школа им. Ф. М. Достоевского. Учебный корпус № 10",\n      "lat": 55.6599807,\n      "lon": 37.7265512,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18302142",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6994816,\n      "lon": 37.6329135,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18396247",\n      "kind": "school",\n      "name": "Школа №745",\n      "lat": 55.865519,\n      "lon": 37.587801,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18451116",\n      "kind": "school",\n      "name": "Школа №\xa01282 \\"Сокольники\\" (главное здание)",\n      "lat": 55.789805,\n      "lon": 37.6697848,\n      "opening_hours": null\n    },\n    {\n      "id": "way/23426808",\n      "kind": "kindergarten",\n      "name": "Д/C № 597",\n      "lat": 55.6352633,\n      "lon": 37.664568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/23768961",\n      "kind": "school",\n      "name": "Детский сад №970",\n      "lat": 55.7905444,\n      "lon": 37.8040977,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27256911",\n      "kind": "kindergarten",\n      "name": "Детский сад №1267",\n      "lat": 55.6178306,\n      "lon": 37.7315572,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27256934",\n      "kind": "kindergarten",\n      "name": "Детский сад №1966",\n      "lat": 55.6176794,\n      "lon": 37.7328893,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27474375",\n      "kind": "school",\n      "name": "Школа-интернат \\"Интеллектуал\\"",\n      "lat": 55.7181295,\n      "lon": 37.4647244,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27546250",\n      "kind": "school",\n      "name": "школа №138",\n      "lat": 55.7812025,\n      "lon": 37.4644698,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27548429",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.720522,\n      "lon": 37.4580375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27548438",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7199881,\n      "lon": 37.4583296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27548539",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7198279,\n      "lon": 37.4536159,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27549008",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7229605,\n      "lon": 37.4550246,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27549026",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7232209,\n      "lon": 37.4560019,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27560499",\n      "kind": "school",\n      "name": "Школа №467",\n      "lat": 55.7289891,\n      "lon": 37.6713802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27569168",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7286567,\n      "lon": 37.4832217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27575988",\n      "kind": "kindergarten",\n      "name": "Детский сад №2210",\n      "lat": 55.7301274,\n      "lon": 37.6714715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27651219",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 5",\n      "lat": 55.7776147,\n      "lon": 37.5291545,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27728924",\n      "kind": "kindergarten",\n      "name": "Детский сад №1888",\n      "lat": 55.799537,\n      "lon": 37.4900072,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27728931",\n      "kind": "school",\n      "name": "Школа №1212",\n      "lat": 55.7991425,\n      "lon": 37.487259,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27760624",\n      "kind": "kindergarten",\n      "name": "ДОУ 1506",\n      "lat": 55.6610066,\n      "lon": 37.5261642,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27787667",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7073227,\n      "lon": 37.4680125,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27787837",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7056023,\n      "lon": 37.4740525,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27787841",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7062635,\n      "lon": 37.4728918,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27793471",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7118038,\n      "lon": 37.4828666,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27793477",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7113993,\n      "lon": 37.4841003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27800688",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8074512,\n      "lon": 37.4796295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27804313",\n      "kind": "school",\n      "name": "Школа 1910",\n      "lat": 55.650675,\n      "lon": 37.7620407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27810681",\n      "kind": "school",\n      "name": "Дом-интернат №21 для умственно отсталых детей",\n      "lat": 55.8074948,\n      "lon": 37.4808947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27810687",\n      "kind": "school",\n      "name": "Школа Ювенес Начальные классы",\n      "lat": 55.8067105,\n      "lon": 37.4828153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27811752",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6550565,\n      "lon": 37.750546,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27811792",\n      "kind": "school",\n      "name": "Школа №1967",\n      "lat": 55.6629901,\n      "lon": 37.7507997,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27811803",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6634465,\n      "lon": 37.7476784,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28115021",\n      "kind": "school",\n      "name": "Центр ЮССТ",\n      "lat": 55.7157512,\n      "lon": 37.4403522,\n      "opening_hours": "Mo-Fr 07:00-18:00; Sa-Su 00:00-24:00"\n    },\n    {\n      "id": "way/28150399",\n      "kind": "school",\n      "name": "Школа №90 им. Героя Советского Союза Е.Г. Ларикова",\n      "lat": 55.8341134,\n      "lon": 37.5256301,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28150828",\n      "kind": "kindergarten",\n      "name": "Детский сад №831",\n      "lat": 55.8390877,\n      "lon": 37.5313954,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28151565",\n      "kind": "kindergarten",\n      "name": "Центр образования №1679 Школа здоровья (детский сад)",\n      "lat": 55.8283668,\n      "lon": 37.5213005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28187032",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8699297,\n      "lon": 37.7057741,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28187033",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8706284,\n      "lon": 37.7044591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28194282",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8660446,\n      "lon": 37.5635627,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28276380",\n      "kind": "school",\n      "name": "ГБОУ г. Москвы Школа № 1159",\n      "lat": 55.8592247,\n      "lon": 37.5231898,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28276418",\n      "kind": "kindergarten",\n      "name": "ГБОУ г. Москвы Школа № 1159 (ДО)",\n      "lat": 55.8602149,\n      "lon": 37.5229593,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28287586",\n      "kind": "kindergarten",\n      "name": "Детский сад №1938",\n      "lat": 55.7339579,\n      "lon": 37.486005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28287588",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7351321,\n      "lon": 37.4836517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28305744",\n      "kind": "school",\n      "name": "школа №887",\n      "lat": 55.7321563,\n      "lon": 37.4396771,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28317427",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1628",\n      "lat": 55.6508731,\n      "lon": 37.7312477,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28377590",\n      "kind": "school",\n      "name": "Школа № 1321. Филиал № 3",\n      "lat": 55.6721357,\n      "lon": 37.7500955,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28379114",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7246288,\n      "lon": 37.442373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28379124",\n      "kind": "school",\n      "name": "Школа №888",\n      "lat": 55.7248085,\n      "lon": 37.443918,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28395095",\n      "kind": "kindergarten",\n      "name": "Детский сад №2",\n      "lat": 55.67084,\n      "lon": 37.7498762,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28587796",\n      "kind": "school",\n      "name": "Гимназия №1583",\n      "lat": 55.8499085,\n      "lon": 37.4945518,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28683132",\n      "kind": "school",\n      "name": "Школа №830 корпус №4",\n      "lat": 55.8141499,\n      "lon": 37.4593779,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28858683",\n      "kind": "school",\n      "name": "Школа №1360",\n      "lat": 55.8126546,\n      "lon": 37.702082,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28859009",\n      "kind": "school",\n      "name": "Школа №386",\n      "lat": 55.8095089,\n      "lon": 37.7203059,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28879187",\n      "kind": "school",\n      "name": "Школа №1315",\n      "lat": 55.8460062,\n      "lon": 37.4811199,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28896147",\n      "kind": "school",\n      "name": "Лицей №1571 корпус №6",\n      "lat": 55.8567494,\n      "lon": 37.4440687,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28938786",\n      "kind": "school",\n      "name": "Школа № 830. Корпус № 3",\n      "lat": 55.8251923,\n      "lon": 37.4493216,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28944074",\n      "kind": "kindergarten",\n      "name": "Детский сад №335 Института естественных наук и экологии (ИНЕСНЭК)",\n      "lat": 55.8022883,\n      "lon": 37.4784098,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28950132",\n      "kind": "school",\n      "name": "Школа №224",\n      "lat": 55.8291421,\n      "lon": 37.4910137,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28950143",\n      "kind": "school",\n      "name": "Школа №224",\n      "lat": 55.8378457,\n      "lon": 37.4843581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28950763",\n      "kind": "school",\n      "name": "Центр образования №1679 Школа здоровья",\n      "lat": 55.8277391,\n      "lon": 37.5192129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29008845",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6955402,\n      "lon": 37.4573312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29019244",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6735931,\n      "lon": 37.4684407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29019245",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6731947,\n      "lon": 37.4675836,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29019247",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6702971,\n      "lon": 37.4624472,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29019273",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6756642,\n      "lon": 37.4548492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29034683",\n      "kind": "school",\n      "name": "Воскресная школа",\n      "lat": 55.7572129,\n      "lon": 37.8139517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29092187",\n      "kind": "school",\n      "name": "Интернат №108",\n      "lat": 55.681983,\n      "lon": 37.5265689,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29114815",\n      "kind": "school",\n      "name": "Школа № 1514",\n      "lat": 55.686297,\n      "lon": 37.5283695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29127999",\n      "kind": "kindergarten",\n      "name": "Школа № 199. Дошкольное отделение № 7",\n      "lat": 55.6825082,\n      "lon": 37.5578716,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29128001",\n      "kind": "kindergarten",\n      "name": "Школа № 199. Дошкольное отделение № 7",\n      "lat": 55.6830478,\n      "lon": 37.5584461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29172983",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8682239,\n      "lon": 37.5323734,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29243290",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6937689,\n      "lon": 37.4941239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29265266",\n      "kind": "school",\n      "name": "Школа №115",\n      "lat": 55.6532486,\n      "lon": 37.5270247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29277434",\n      "kind": "school",\n      "name": "Специальная школа №3",\n      "lat": 55.6699349,\n      "lon": 37.5760101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29277435",\n      "kind": "school",\n      "name": "Школа-интернат №61",\n      "lat": 55.6696845,\n      "lon": 37.5767271,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29277495",\n      "kind": "school",\n      "name": "Интернат № 56",\n      "lat": 55.6708326,\n      "lon": 37.5962506,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29325314",\n      "kind": "school",\n      "name": "Центр внешкольной работы \\"Зюзино\\" (УПК №2)",\n      "lat": 55.6479616,\n      "lon": 37.5905415,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386096",\n      "kind": "school",\n      "name": "Школа №398",\n      "lat": 55.7175457,\n      "lon": 37.8048564,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386727",\n      "kind": "kindergarten",\n      "name": "Школа №1420. Корпус №1",\n      "lat": 55.7021253,\n      "lon": 37.8189579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386728",\n      "kind": "kindergarten",\n      "name": "Школа №1420. Корпус №5",\n      "lat": 55.7019684,\n      "lon": 37.8175424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386752",\n      "kind": "school",\n      "name": "Спецшкола №895",\n      "lat": 55.7003552,\n      "lon": 37.8137004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386753",\n      "kind": "school",\n      "name": "Центр образования №1420",\n      "lat": 55.7009762,\n      "lon": 37.8148969,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386777",\n      "kind": "kindergarten",\n      "name": "Детский сад №1734",\n      "lat": 55.6986182,\n      "lon": 37.8107168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386778",\n      "kind": "kindergarten",\n      "name": "Детский сад №2118",\n      "lat": 55.6984591,\n      "lon": 37.8124211,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29388396",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1393 ДО",\n      "lat": 55.6960676,\n      "lon": 37.8150581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29388405",\n      "kind": "kindergarten",\n      "name": "Детский сад №326",\n      "lat": 55.6965439,\n      "lon": 37.8237105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29396623",\n      "kind": "school",\n      "name": "Школа имени Ф. М. Достоевского",\n      "lat": 55.6865233,\n      "lon": 37.7510571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29396677",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6824659,\n      "lon": 37.7529292,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29400194",\n      "kind": "kindergarten",\n      "name": "Детский сад №1333",\n      "lat": 55.7053222,\n      "lon": 37.7900766,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29400195",\n      "kind": "kindergarten",\n      "name": "Детский сад №1677",\n      "lat": 55.7058003,\n      "lon": 37.7890398,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29400198",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7065067,\n      "lon": 37.7833545,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29400566",\n      "kind": "school",\n      "name": "Школа №623",\n      "lat": 55.7055473,\n      "lon": 37.7871892,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29430157",\n      "kind": "school",\n      "name": "Школа №2090",\n      "lat": 55.7144209,\n      "lon": 37.7921939,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29430159",\n      "kind": "school",\n      "name": "Школа №776",\n      "lat": 55.7144452,\n      "lon": 37.7927937,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29430209",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1883",\n      "lat": 55.7184371,\n      "lon": 37.8030253,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29430212",\n      "kind": "kindergarten",\n      "name": "Детский сад №581",\n      "lat": 55.719514,\n      "lon": 37.7937924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29431666",\n      "kind": "school",\n      "name": "Школа №797",\n      "lat": 55.7548712,\n      "lon": 37.8224523,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446097",\n      "kind": "kindergarten",\n      "name": "Детский сад №724",\n      "lat": 55.7568191,\n      "lon": 37.7858978,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446098",\n      "kind": "school",\n      "name": "Центр образования №1637",\n      "lat": 55.7560708,\n      "lon": 37.7864197,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446190",\n      "kind": "kindergarten",\n      "name": "Детский сад №1900",\n      "lat": 55.7580628,\n      "lon": 37.7998375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446272",\n      "kind": "kindergarten",\n      "name": "Центр развития ребенка №1908",\n      "lat": 55.7550447,\n      "lon": 37.7807216,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446275",\n      "kind": "school",\n      "name": "Центр образования №1852",\n      "lat": 55.7560122,\n      "lon": 37.7792385,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446294",\n      "kind": "kindergarten",\n      "name": "Детский сад №1216",\n      "lat": 55.7585526,\n      "lon": 37.7773452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446299",\n      "kind": "school",\n      "name": "Школа №681",\n      "lat": 55.7589275,\n      "lon": 37.7757171,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446339",\n      "kind": "kindergarten",\n      "name": "Московская международная школа. Подразделение \\"Планета Земля\\"",\n      "lat": 55.7597229,\n      "lon": 37.7857198,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446344",\n      "kind": "school",\n      "name": "Московская международная школа. Подразделение \\"Планета Уран\\"",\n      "lat": 55.7633918,\n      "lon": 37.7813803,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446378",\n      "kind": "school",\n      "name": "Московская международная школа. Подразделение \\"Планета Солнце\\"",\n      "lat": 55.7623385,\n      "lon": 37.7867543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449572",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.650583,\n      "lon": 37.7238597,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449574",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.650188,\n      "lon": 37.7209587,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449575",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6508963,\n      "lon": 37.7216145,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449576",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6487741,\n      "lon": 37.7182695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449577",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6479391,\n      "lon": 37.717653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29450186",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6435458,\n      "lon": 37.7161641,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29450778",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6433703,\n      "lon": 37.7126553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29450878",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6537227,\n      "lon": 37.7145432,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29450880",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6541151,\n      "lon": 37.7134389,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29451029",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6508977,\n      "lon": 37.7129595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29601128",\n      "kind": "school",\n      "name": "Интернат №53",\n      "lat": 55.7767955,\n      "lon": 37.728609,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29689770",\n      "kind": "school",\n      "name": "Школа №1324",\n      "lat": 55.755097,\n      "lon": 37.8201818,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29696444",\n      "kind": "school",\n      "name": "ЦРР №837",\n      "lat": 55.7998586,\n      "lon": 37.785213,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29728292",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8090605,\n      "lon": 37.8210452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29728293",\n      "kind": "school",\n      "name": "",\n      "lat": 55.807993,\n      "lon": 37.8222138,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30049930",\n      "kind": "school",\n      "name": "Школа \\"Вита\\"",\n      "lat": 55.7343311,\n      "lon": 37.4798654,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30132609",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7472919,\n      "lon": 37.7716329,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30414740",\n      "kind": "school",\n      "name": "Школа №22",\n      "lat": 55.6986533,\n      "lon": 37.5553316,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30414751",\n      "kind": "school",\n      "name": "Школа №1265",\n      "lat": 55.6991038,\n      "lon": 37.5582731,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30414756",\n      "kind": "school",\n      "name": "Лицей \\"Вторая Школа\\"",\n      "lat": 55.6978181,\n      "lon": 37.5563936,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30433167",\n      "kind": "school",\n      "name": "Школа №171",\n      "lat": 55.7222711,\n      "lon": 37.5795407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30853347",\n      "kind": "school",\n      "name": "Центр образования №775",\n      "lat": 55.6752927,\n      "lon": 37.7535069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30853467",\n      "kind": "school",\n      "name": "НОУ Школа \\"Эрудит-2\\"",\n      "lat": 55.6757566,\n      "lon": 37.7499035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30990497",\n      "kind": "kindergarten",\n      "name": "Детский сад № 621",\n      "lat": 55.6268177,\n      "lon": 37.6502032,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31024968",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2680",\n      "lat": 55.6323548,\n      "lon": 37.6743605,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31025606",\n      "kind": "school",\n      "name": "ЦДТ \\"Царицыно\\"",\n      "lat": 55.6308695,\n      "lon": 37.6522876,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31025623",\n      "kind": "school",\n      "name": "Школа №1640",\n      "lat": 55.6323645,\n      "lon": 37.6519217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31172614",\n      "kind": "kindergarten",\n      "name": "ДОУ №1873",\n      "lat": 55.6808136,\n      "lon": 37.6413283,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31650953",\n      "kind": "kindergarten",\n      "name": "Детский сад №674",\n      "lat": 55.6871685,\n      "lon": 37.7285679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31653611",\n      "kind": "school",\n      "name": "Лицей №504",\n      "lat": 55.6678229,\n      "lon": 37.6270009,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31663291",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6781056,\n      "lon": 37.6744201,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31663292",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6781643,\n      "lon": 37.672463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31663877",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, Светлячок  (здание № 4)",\n      "lat": 55.6691759,\n      "lon": 37.6516045,\n      "opening_hours": null\n    },\n    {\n      "id": "way/32605841",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Ленинском\\"",\n      "lat": 55.7180444,\n      "lon": 37.593162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/32611554",\n      "kind": "school",\n      "name": "Школа №730",\n      "lat": 55.6599999,\n      "lon": 37.6266505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/33060422",\n      "kind": "school",\n      "name": "Школа №183",\n      "lat": 55.8740976,\n      "lon": 37.5560007,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34321752",\n      "kind": "school",\n      "name": "Школа 1040",\n      "lat": 55.6438454,\n      "lon": 37.7374497,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34321753",\n      "kind": "school",\n      "name": "Школа 1147",\n      "lat": 55.6441597,\n      "lon": 37.7361151,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34329690",\n      "kind": "school",\n      "name": "Центр образования №2010",\n      "lat": 55.6673041,\n      "lon": 37.748292,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34585487",\n      "kind": "school",\n      "name": "Школа № 296",\n      "lat": 55.8312193,\n      "lon": 37.5914633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34873806",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 1",\n      "lat": 55.7730658,\n      "lon": 37.5415792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34908291",\n      "kind": "school",\n      "name": "Школа №1770",\n      "lat": 55.6897412,\n      "lon": 37.6886911,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34908295",\n      "kind": "school",\n      "name": "Кадетская школа №1770",\n      "lat": 55.6861271,\n      "lon": 37.6886855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34911740",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6784852,\n      "lon": 37.6998653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34911744",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6807628,\n      "lon": 37.7034581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34911748",\n      "kind": "school",\n      "name": "Детский оздоровительно-образовательный центр \\"Южный\\"",\n      "lat": 55.6828058,\n      "lon": 37.7049521,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34911749",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6824531,\n      "lon": 37.7064009,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34923064",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Медвежонок",\n      "lat": 55.6793522,\n      "lon": 37.6806842,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34923065",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6803916,\n      "lon": 37.6811359,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34990988",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Сказка",\n      "lat": 55.6783257,\n      "lon": 37.6837186,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35013513",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8012961,\n      "lon": 37.5181267,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069953",\n      "kind": "kindergarten",\n      "name": "Детский сад №1047",\n      "lat": 55.8913307,\n      "lon": 37.6697913,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069955",\n      "kind": "kindergarten",\n      "name": "Детский сад №112",\n      "lat": 55.8913889,\n      "lon": 37.667136,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069957",\n      "kind": "kindergarten",\n      "name": "Детский сад №1856",\n      "lat": 55.8896475,\n      "lon": 37.6659392,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069958",\n      "kind": "kindergarten",\n      "name": "Детский сад №2576",\n      "lat": 55.8896526,\n      "lon": 37.6685231,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069959",\n      "kind": "school",\n      "name": "Школа № 283. Корпус № 2",\n      "lat": 55.8914151,\n      "lon": 37.6643298,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069960",\n      "kind": "school",\n      "name": "Школа №283",\n      "lat": 55.8895329,\n      "lon": 37.671121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35134626",\n      "kind": "school",\n      "name": "Школа №950",\n      "lat": 55.8602701,\n      "lon": 37.6046609,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35158327",\n      "kind": "school",\n      "name": "Социально-ребилитационный центр Отрадное",\n      "lat": 55.8613273,\n      "lon": 37.6059881,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35172004",\n      "kind": "school",\n      "name": "Школа № 536",\n      "lat": 55.6436935,\n      "lon": 37.5956243,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35379957",\n      "kind": "school",\n      "name": "Центр образования №548 Царицыно",\n      "lat": 55.6020586,\n      "lon": 37.7169951,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35403477",\n      "kind": "school",\n      "name": "Школа №544, корпус 3",\n      "lat": 55.602306,\n      "lon": 37.7190308,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35403511",\n      "kind": "kindergarten",\n      "name": "Детский сад №1166",\n      "lat": 55.6031909,\n      "lon": 37.7163099,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35403550",\n      "kind": "school",\n      "name": "Школа №544",\n      "lat": 55.6068626,\n      "lon": 37.7146404,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35680206",\n      "kind": "school",\n      "name": "Школа № 370",\n      "lat": 55.6902904,\n      "lon": 37.5845344,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35680207",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1216",\n      "lat": 55.6904938,\n      "lon": 37.5839383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35763140",\n      "kind": "school",\n      "name": "Школа №1205 \\"Единство\\"",\n      "lat": 55.6770512,\n      "lon": 37.5434015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35960354",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6647818,\n      "lon": 37.4944948,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35960355",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6644009,\n      "lon": 37.4954367,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35977653",\n      "kind": "school",\n      "name": "Лицей им. Софьи Нюберг",\n      "lat": 55.6763331,\n      "lon": 37.5817519,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35977707",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2015",\n      "lat": 55.6759253,\n      "lon": 37.5782656,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36243218",\n      "kind": "school",\n      "name": "Школа №585",\n      "lat": 55.5985768,\n      "lon": 37.7191269,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36306984",\n      "kind": "kindergarten",\n      "name": "Детский сад №1018",\n      "lat": 55.8641133,\n      "lon": 37.4904729,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36358903",\n      "kind": "school",\n      "name": "Школа-вуз «Современное образование»",\n      "lat": 55.7687625,\n      "lon": 37.6355598,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36476905",\n      "kind": "school",\n      "name": "Высшая коммерческая школа ВАВТ Минэкономразвития России",\n      "lat": 55.8789001,\n      "lon": 37.4680313,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515109",\n      "kind": "school",\n      "name": "Центр образования №117",\n      "lat": 55.6736121,\n      "lon": 37.531363,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515110",\n      "kind": "kindergarten",\n      "name": "Детский сад №1505",\n      "lat": 55.6719177,\n      "lon": 37.5356426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515111",\n      "kind": "kindergarten",\n      "name": "Детский сад №1609",\n      "lat": 55.6719208,\n      "lon": 37.5387011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515112",\n      "kind": "school",\n      "name": "Школа №117",\n      "lat": 55.6720714,\n      "lon": 37.5340263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515113",\n      "kind": "kindergarten",\n      "name": "Детский сад №515",\n      "lat": 55.6725719,\n      "lon": 37.5374105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36746759",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8640461,\n      "lon": 37.4877039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36868611",\n      "kind": "school",\n      "name": "Школа №2000 корпус №3",\n      "lat": 55.6411232,\n      "lon": 37.6598579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36953234",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №9 (дошкольное отделение)",\n      "lat": 55.8405956,\n      "lon": 37.4618454,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37030105",\n      "kind": "school",\n      "name": "ЦПМСС \\"Взаимодействие\\"",\n      "lat": 55.6910606,\n      "lon": 37.6118144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37082803",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7141953,\n      "lon": 37.5145849,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37330394",\n      "kind": "school",\n      "name": "Школа №49. Школьный корпус",\n      "lat": 55.6420096,\n      "lon": 37.546718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37330635",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.64255,\n      "lon": 37.5450393,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37870407",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6821161,\n      "lon": 37.6877681,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37927642",\n      "kind": "kindergarten",\n      "name": "Детский сад №66",\n      "lat": 55.8157405,\n      "lon": 37.7123382,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37943640",\n      "kind": "kindergarten",\n      "name": "Детский сад №526",\n      "lat": 55.7863181,\n      "lon": 37.4520041,\n      "opening_hours": null\n    },\n    {\n      "id": "way/38298522",\n      "kind": "school",\n      "name": "Цетр образования №1601",\n      "lat": 55.792534,\n      "lon": 37.573236,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39254578",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №3",\n      "lat": 55.6404443,\n      "lon": 37.5195128,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39393807",\n      "kind": "school",\n      "name": "Школа № 489",\n      "lat": 55.7176541,\n      "lon": 37.7368662,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39393808",\n      "kind": "school",\n      "name": "Школа № 841",\n      "lat": 55.7173734,\n      "lon": 37.735468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39505342",\n      "kind": "school",\n      "name": "Школа №597",\n      "lat": 55.8658618,\n      "lon": 37.4895788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39524490",\n      "kind": "kindergarten",\n      "name": "Детский сад №51",\n      "lat": 55.7112872,\n      "lon": 37.7329596,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39577157",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №264",\n      "lat": 55.671468,\n      "lon": 37.7218289,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39577161",\n      "kind": "school",\n      "name": "Школа №918",\n      "lat": 55.6702373,\n      "lon": 37.7224202,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39627110",\n      "kind": "school",\n      "name": "Центр образования №773",\n      "lat": 55.6894716,\n      "lon": 37.7288475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39627114",\n      "kind": "school",\n      "name": "Школа №773",\n      "lat": 55.6881679,\n      "lon": 37.7293032,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39627119",\n      "kind": "kindergarten",\n      "name": "Детский сад №671",\n      "lat": 55.6879414,\n      "lon": 37.7275966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39629314",\n      "kind": "kindergarten",\n      "name": "Детский сад №2224",\n      "lat": 55.675903,\n      "lon": 37.7188544,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39629327",\n      "kind": "kindergarten",\n      "name": "Детский сад № 829",\n      "lat": 55.682038,\n      "lon": 37.7191801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39651080",\n      "kind": "school",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (корпус 1)",\n      "lat": 55.6803772,\n      "lon": 37.7270089,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39651117",\n      "kind": "kindergarten",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (дошкольное отделение, корпус 2)",\n      "lat": 55.6810646,\n      "lon": 37.7248692,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39651120",\n      "kind": "school",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (корпус 2)",\n      "lat": 55.6818471,\n      "lon": 37.7275647,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39651153",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1550",\n      "lat": 55.6859908,\n      "lon": 37.7285406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39665778",\n      "kind": "kindergarten",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (дошкольное отделение, корпус 1)",\n      "lat": 55.6788088,\n      "lon": 37.725612,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39894769",\n      "kind": "school",\n      "name": "Центр образования №1465 имени адмирала Н.Г. Кузнецова",\n      "lat": 55.7451143,\n      "lon": 37.5598952,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39958817",\n      "kind": "school",\n      "name": "Школа № 2101",\n      "lat": 55.7415895,\n      "lon": 37.4777674,\n      "opening_hours": "Mo-Sa 08:00-19:00"\n    },\n    {\n      "id": "way/40057230",\n      "kind": "kindergarten",\n      "name": "Школа №1100 (дошкольное отделение)",\n      "lat": 55.6623467,\n      "lon": 37.5288537,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40057245",\n      "kind": "school",\n      "name": "Школа №1100",\n      "lat": 55.6628789,\n      "lon": 37.5317421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40445965",\n      "kind": "kindergarten",\n      "name": "Школа \\"Покровский квартал\\", Детский сад 1",\n      "lat": 55.7555356,\n      "lon": 37.6524788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40607358",\n      "kind": "kindergarten",\n      "name": "Детский сад № 63",\n      "lat": 55.6885878,\n      "lon": 37.564302,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40792993",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.743724,\n      "lon": 37.7561889,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40793349",\n      "kind": "school",\n      "name": "Школа № 2126 \\"Перово\\"",\n      "lat": 55.7428651,\n      "lon": 37.7590491,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40983857",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8234461,\n      "lon": 37.599908,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41088548",\n      "kind": "kindergarten",\n      "name": "Детский сад №1855",\n      "lat": 55.7369908,\n      "lon": 37.4639043,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41088549",\n      "kind": "school",\n      "name": "Школа №2101",\n      "lat": 55.7360516,\n      "lon": 37.4624765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41347146",\n      "kind": "kindergarten",\n      "name": "Детский сад №103",\n      "lat": 55.6991646,\n      "lon": 37.7821279,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41347147",\n      "kind": "kindergarten",\n      "name": "Детский сад №139",\n      "lat": 55.6989681,\n      "lon": 37.7835401,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41657209",\n      "kind": "school",\n      "name": "Гимназия №1592. Отделение №2043",\n      "lat": 55.8749129,\n      "lon": 37.5544011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41667074",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6781004,\n      "lon": 37.4876995,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41667088",\n      "kind": "school",\n      "name": "Центр образования и спорта \\"Москва-98\\". Школа № 2",\n      "lat": 55.6802679,\n      "lon": 37.4849926,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41826105",\n      "kind": "school",\n      "name": "Школа №964",\n      "lat": 55.8971022,\n      "lon": 37.6161054,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41826342",\n      "kind": "school",\n      "name": "Школа №1413",\n      "lat": 55.8966871,\n      "lon": 37.6174649,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41870646",\n      "kind": "school",\n      "name": "Центр образования №1852",\n      "lat": 55.7570633,\n      "lon": 37.8035144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41981684",\n      "kind": "school",\n      "name": "Центр образования №1497",\n      "lat": 55.7446262,\n      "lon": 37.4927263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41981695",\n      "kind": "kindergarten",\n      "name": "Детский сад №1328",\n      "lat": 55.7435007,\n      "lon": 37.4919555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42031082",\n      "kind": "school",\n      "name": "Мытищинская детская дом-школа музыкального воспитания",\n      "lat": 55.8893147,\n      "lon": 37.7536061,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42032375",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа Глория\\"",\n      "lat": 55.8958252,\n      "lon": 37.6157141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42102028",\n      "kind": "school",\n      "name": "Школа Глория, корпус № 10",\n      "lat": 55.8951174,\n      "lon": 37.6250771,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42170724",\n      "kind": "school",\n      "name": "Школа №2101",\n      "lat": 55.7347513,\n      "lon": 37.4804259,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42170744",\n      "kind": "kindergarten",\n      "name": "Детский сад №545",\n      "lat": 55.7330745,\n      "lon": 37.4731726,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42213016",\n      "kind": "kindergarten",\n      "name": "ГУ МВД РФ по МО",\n      "lat": 55.7123225,\n      "lon": 37.6602774,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237425",\n      "kind": "school",\n      "name": "Школа №73",\n      "lat": 55.7328756,\n      "lon": 37.4672408,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237426",\n      "kind": "school",\n      "name": "Школа №262",\n      "lat": 55.732406,\n      "lon": 37.4647222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237493",\n      "kind": "kindergarten",\n      "name": "Детский сад №1394",\n      "lat": 55.7322756,\n      "lon": 37.4582185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237542",\n      "kind": "kindergarten",\n      "name": "Детский сад №77",\n      "lat": 55.7312015,\n      "lon": 37.4668464,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237543",\n      "kind": "kindergarten",\n      "name": "Детский сад №659",\n      "lat": 55.7315904,\n      "lon": 37.4692174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42239788",\n      "kind": "kindergarten",\n      "name": "Детский сад №1240",\n      "lat": 55.8654668,\n      "lon": 37.6840473,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42244977",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Солнышко\\"",\n      "lat": 55.7328614,\n      "lon": 37.4512911,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42368435",\n      "kind": "school",\n      "name": "Центр образования №1854 (8-11 классы)",\n      "lat": 55.8025995,\n      "lon": 37.4717526,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42394177",\n      "kind": "school",\n      "name": "Навигацкая школа",\n      "lat": 55.7310613,\n      "lon": 37.4555855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42484743",\n      "kind": "kindergarten",\n      "name": "Ясли - сад №262",\n      "lat": 55.7093657,\n      "lon": 37.7551514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42484744",\n      "kind": "school",\n      "name": "Школа №336",\n      "lat": 55.7095905,\n      "lon": 37.7571555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42485986",\n      "kind": "kindergarten",\n      "name": "Детский сад №1130",\n      "lat": 55.6344211,\n      "lon": 37.7939016,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42563777",\n      "kind": "kindergarten",\n      "name": "Детский сад №2457",\n      "lat": 55.6324949,\n      "lon": 37.7974361,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42708942",\n      "kind": "kindergarten",\n      "name": "Детский сад №2556",\n      "lat": 55.7414863,\n      "lon": 37.6662469,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42774926",\n      "kind": "school",\n      "name": "Московская богословская семинария евангельских христиан-баптистов",\n      "lat": 55.7461854,\n      "lon": 37.7561497,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42774928",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7461639,\n      "lon": 37.7575604,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42855891",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6522714,\n      "lon": 37.7127087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42946013",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1135",\n      "lat": 55.6577756,\n      "lon": 37.7045004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43175483",\n      "kind": "kindergarten",\n      "name": "Детский сад №911",\n      "lat": 55.652137,\n      "lon": 37.7011088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43175547",\n      "kind": "kindergarten",\n      "name": "Детский сад №774",\n      "lat": 55.6546546,\n      "lon": 37.7038672,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43198474",\n      "kind": "school",\n      "name": "Школа №493",\n      "lat": 55.6573417,\n      "lon": 37.6995277,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43198475",\n      "kind": "school",\n      "name": "Центр Непрерывного Ускоренного Обучения",\n      "lat": 55.6575068,\n      "lon": 37.7021117,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43321808",\n      "kind": "school",\n      "name": "Школа №1229",\n      "lat": 55.7504613,\n      "lon": 37.6793611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43566667",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7175522,\n      "lon": 37.7361626,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43790668",\n      "kind": "kindergarten",\n      "name": "Детский сад №1541",\n      "lat": 55.7002506,\n      "lon": 37.7350172,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43790866",\n      "kind": "kindergarten",\n      "name": "Детский сад №1275",\n      "lat": 55.7034525,\n      "lon": 37.7349309,\n      "opening_hours": null\n    },\n    {\n      "id": "way/44146666",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8687457,\n      "lon": 37.7120246,\n      "opening_hours": null\n    },\n    {\n      "id": "way/44384266",\n      "kind": "kindergarten",\n      "name": "Детский сад №1838",\n      "lat": 55.7990431,\n      "lon": 37.7729884,\n      "opening_hours": null\n    },\n    {\n      "id": "way/44502324",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8903859,\n      "lon": 37.4758059,\n      "opening_hours": null\n    },\n    {\n      "id": "way/44503057",\n      "kind": "school",\n      "name": "Школа №707",\n      "lat": 55.7971548,\n      "lon": 37.7735426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45138130",\n      "kind": "kindergarten",\n      "name": "Школа №1056 (дошкольное отделение №1)",\n      "lat": 55.8465531,\n      "lon": 37.4429213,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45249027",\n      "kind": "kindergarten",\n      "name": "Центр образования № 1858 - д/с \\"Сказка\\"",\n      "lat": 55.6394707,\n      "lon": 37.7944318,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45249666",\n      "kind": "kindergarten",\n      "name": "Центр образования № 1858 - д/с \\"Львёнок\\"",\n      "lat": 55.6390333,\n      "lon": 37.7944197,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45249984",\n      "kind": "kindergarten",\n      "name": "Детский сад №692",\n      "lat": 55.6351427,\n      "lon": 37.7999839,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45250579",\n      "kind": "kindergarten",\n      "name": "Детский сад №1504",\n      "lat": 55.6325715,\n      "lon": 37.8001879,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45256626",\n      "kind": "school",\n      "name": "ГБОУ Школа в Капотне (Учебный корпус \\"Кузнецкий\\")",\n      "lat": 55.6359101,\n      "lon": 37.7976899,\n      "opening_hours": "Mo-Sa 08:00-19:00"\n    },\n    {\n      "id": "way/45398742",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6838216,\n      "lon": 37.4570798,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45399802",\n      "kind": "kindergarten",\n      "name": "Детский сад №874",\n      "lat": 55.8710096,\n      "lon": 37.673611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45463106",\n      "kind": "school",\n      "name": "Воскресная школа при храме Св. Николая",\n      "lat": 55.7320652,\n      "lon": 37.5904566,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45487360",\n      "kind": "kindergarten",\n      "name": "Школа № 1363, ДО Радуга",\n      "lat": 55.7057657,\n      "lon": 37.8252415,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45739177",\n      "kind": "school",\n      "name": "НОУ \\"Развитие\\" (школа компенсирующего обучения)",\n      "lat": 55.6715593,\n      "lon": 37.5647551,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45778892",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Б. Трехсвятительском",\n      "lat": 55.7549631,\n      "lon": 37.6460396,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45820293",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1084",\n      "lat": 55.7156782,\n      "lon": 37.6188496,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45823256",\n      "kind": "kindergarten",\n      "name": "Детский сад №469",\n      "lat": 55.672203,\n      "lon": 37.59147,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45826027",\n      "kind": "kindergarten",\n      "name": "Школа Эрудит",\n      "lat": 55.7309182,\n      "lon": 37.7147858,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45829275",\n      "kind": "kindergarten",\n      "name": "Школа № 1222 имени Маршала Советского Союза И. Х. Баграмяна. Дошкольное  отделение № 1",\n      "lat": 55.7321958,\n      "lon": 37.7100014,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45831433",\n      "kind": "school",\n      "name": "Школа №1344",\n      "lat": 55.6951689,\n      "lon": 37.7999825,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45850376",\n      "kind": "school",\n      "name": "ГБОУ школа № 1363, корпус 2",\n      "lat": 55.7066325,\n      "lon": 37.8233965,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45850682",\n      "kind": "school",\n      "name": "Школа №1937",\n      "lat": 55.6997432,\n      "lon": 37.8146225,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45875704",\n      "kind": "school",\n      "name": "Школа №394",\n      "lat": 55.70516,\n      "lon": 37.7859633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45876203",\n      "kind": "kindergarten",\n      "name": "Детский сад №779",\n      "lat": 55.7096469,\n      "lon": 37.7880116,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45876992",\n      "kind": "school",\n      "name": "Школа № 468",\n      "lat": 55.7129864,\n      "lon": 37.7710257,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45897345",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7317013,\n      "lon": 37.7410536,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45934474",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1315",\n      "lat": 55.6333705,\n      "lon": 37.4788494,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45936847",\n      "kind": "school",\n      "name": "ЦО \\"Самбо-70\\"",\n      "lat": 55.6308846,\n      "lon": 37.4872392,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46166572",\n      "kind": "kindergarten",\n      "name": "Детский сад № 313",\n      "lat": 55.6496975,\n      "lon": 37.5897187,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46166579",\n      "kind": "kindergarten",\n      "name": "Детский сад № 314",\n      "lat": 55.6488776,\n      "lon": 37.5900296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46578351",\n      "kind": "school",\n      "name": "Лицей №1571 корпус №2",\n      "lat": 55.8618588,\n      "lon": 37.4455613,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46581741",\n      "kind": "school",\n      "name": "Лицей №\xa01571 корпус 3",\n      "lat": 55.8626615,\n      "lon": 37.4417625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46604063",\n      "kind": "kindergarten",\n      "name": "Детский сад №1265",\n      "lat": 55.6994677,\n      "lon": 37.5596529,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46657069",\n      "kind": "kindergarten",\n      "name": "Детский сад «Исток 2»",\n      "lat": 55.6783991,\n      "lon": 37.5550965,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46657618",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1335",\n      "lat": 55.6877506,\n      "lon": 37.5489992,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46657631",\n      "kind": "school",\n      "name": "Школа № 26",\n      "lat": 55.6944192,\n      "lon": 37.5504454,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46657632",\n      "kind": "school",\n      "name": "Школа № 25",\n      "lat": 55.6952633,\n      "lon": 37.5489463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46671693",\n      "kind": "kindergarten",\n      "name": "Детский сад №549",\n      "lat": 55.6725462,\n      "lon": 37.5330648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46671697",\n      "kind": "kindergarten",\n      "name": "детский сад 942",\n      "lat": 55.6758118,\n      "lon": 37.534604,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46671699",\n      "kind": "kindergarten",\n      "name": "Детский сад №1511",\n      "lat": 55.6753419,\n      "lon": 37.5299693,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46740734",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7873449,\n      "lon": 37.6151356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46830401",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №01",\n      "lat": 55.6219218,\n      "lon": 37.4914937,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46831048",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/1",\n      "lat": 55.6224351,\n      "lon": 37.4933496,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46922890",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 8",\n      "lat": 55.8483814,\n      "lon": 37.6430853,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47067455",\n      "kind": "school",\n      "name": "ГБОУ школа на Яузе",\n      "lat": 55.7736771,\n      "lon": 37.707104,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47067457",\n      "kind": "kindergarten",\n      "name": "Детский сад №1979",\n      "lat": 55.7733274,\n      "lon": 37.7078097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47346175",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6514957,\n      "lon": 37.73924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47514087",\n      "kind": "school",\n      "name": "Школа №951",\n      "lat": 55.8920145,\n      "lon": 37.6769757,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47572148",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7330721,\n      "lon": 37.6610855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47584412",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6440596,\n      "lon": 37.7367633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48036691",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №4",\n      "lat": 55.880319,\n      "lon": 37.6542186,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48038280",\n      "kind": "kindergarten",\n      "name": "Детский сад №2101",\n      "lat": 55.6446565,\n      "lon": 37.7402371,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48038333",\n      "kind": "kindergarten",\n      "name": "Детский сад №2162",\n      "lat": 55.6457544,\n      "lon": 37.7402389,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48038725",\n      "kind": "kindergarten",\n      "name": "Детский сад №3 Сказка",\n      "lat": 55.8910272,\n      "lon": 37.4454591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48042142",\n      "kind": "kindergarten",\n      "name": "Детский сад №1797",\n      "lat": 55.6455114,\n      "lon": 37.7328499,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48042227",\n      "kind": "kindergarten",\n      "name": "Детский сад №1799",\n      "lat": 55.6444231,\n      "lon": 37.733024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48057819",\n      "kind": "kindergarten",\n      "name": "Детский сад № 735",\n      "lat": 55.7240301,\n      "lon": 37.6283509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48125946",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.7999928,\n      "lon": 37.5081815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48126019",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.8006034,\n      "lon": 37.5085065,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48167354",\n      "kind": "kindergarten",\n      "name": "детский сад №1070",\n      "lat": 55.601711,\n      "lon": 37.5450614,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48174162",\n      "kind": "kindergarten",\n      "name": "ОСЗН района Ясенево",\n      "lat": 55.5964108,\n      "lon": 37.5417582,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48178990",\n      "kind": "kindergarten",\n      "name": "Школа №1429 имени Героя Советского Союза Н.А. Боброва (Дошкольное отделение)",\n      "lat": 55.7783694,\n      "lon": 37.6876405,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48179027",\n      "kind": "kindergarten",\n      "name": "Детский сад №1061",\n      "lat": 55.6076334,\n      "lon": 37.7273522,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48222931",\n      "kind": "school",\n      "name": "НОУ \\"Школа св. Георгия\\"",\n      "lat": 55.6868672,\n      "lon": 37.7454194,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48227462",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1512",\n      "lat": 55.657015,\n      "lon": 37.6155436,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48239580",\n      "kind": "school",\n      "name": "Школа №1429 имени Героя Советского Союза Н.А. Боброва",\n      "lat": 55.77892,\n      "lon": 37.6867318,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48298114",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1490",\n      "lat": 55.7170934,\n      "lon": 37.730874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48298115",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1941",\n      "lat": 55.7174803,\n      "lon": 37.732506,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48298364",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1304",\n      "lat": 55.7151618,\n      "lon": 37.7327344,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48303492",\n      "kind": "kindergarten",\n      "name": "Детский сад № 295",\n      "lat": 55.7362894,\n      "lon": 37.6437021,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48337556",\n      "kind": "school",\n      "name": "Школа №478",\n      "lat": 55.71377,\n      "lon": 37.741121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48337573",\n      "kind": "school",\n      "name": "ЦТРиГО \\"Жемчужина\\"",\n      "lat": 55.7134945,\n      "lon": 37.7377764,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48346415",\n      "kind": "kindergarten",\n      "name": "Детский сад № 492",\n      "lat": 55.7327459,\n      "lon": 37.6564484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48399916",\n      "kind": "school",\n      "name": "Центр образования №1496",\n      "lat": 55.7234639,\n      "lon": 37.6057452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48524187",\n      "kind": "school",\n      "name": "Школа №954",\n      "lat": 55.8952138,\n      "lon": 37.6353503,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48587459",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6435928,\n      "lon": 37.7196933,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48599543",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Яблонька»",\n      "lat": 55.8652504,\n      "lon": 37.4419269,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48634158",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1791",\n      "lat": 55.7035769,\n      "lon": 37.7501424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48637694",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6969813,\n      "lon": 37.7574855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48637695",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6967204,\n      "lon": 37.7588024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48757559",\n      "kind": "kindergarten",\n      "name": "Детский сад «Горница-Узорница»",\n      "lat": 55.8677108,\n      "lon": 37.6657108,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48892974",\n      "kind": "kindergarten",\n      "name": "Детский сад №2617",\n      "lat": 55.7718249,\n      "lon": 37.704149,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49059813",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.7960365,\n      "lon": 37.5056424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49140332",\n      "kind": "school",\n      "name": "Школа XXI век",\n      "lat": 55.7459077,\n      "lon": 37.701383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49251892",\n      "kind": "kindergarten",\n      "name": "Детский сад №84",\n      "lat": 55.7376797,\n      "lon": 37.7072893,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49251909",\n      "kind": "school",\n      "name": "Школа №459",\n      "lat": 55.7388464,\n      "lon": 37.7060765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49260760",\n      "kind": "school",\n      "name": "Рогожская Древлеправославная общеобразовательная школа",\n      "lat": 55.7387866,\n      "lon": 37.7012173,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49432199",\n      "kind": "school",\n      "name": "ГБПОУ ДЗМ Медицинский колледж № 1 СП № 1",\n      "lat": 55.6662877,\n      "lon": 37.6315817,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49869363",\n      "kind": "kindergarten",\n      "name": "Детский сад №502",\n      "lat": 55.8488484,\n      "lon": 37.4791162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49869365",\n      "kind": "kindergarten",\n      "name": "Детский сад №787",\n      "lat": 55.8474061,\n      "lon": 37.4801653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49906444",\n      "kind": "school",\n      "name": "Лицей №1303",\n      "lat": 55.7518616,\n      "lon": 37.6791705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49908065",\n      "kind": "school",\n      "name": "Школа №415",\n      "lat": 55.7574042,\n      "lon": 37.6897461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49908070",\n      "kind": "school",\n      "name": "Школа № 1321 \\"Ковчег\\"",\n      "lat": 55.755963,\n      "lon": 37.689715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50055625",\n      "kind": "school",\n      "name": "Детский сад №617",\n      "lat": 55.7987844,\n      "lon": 37.7951505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50059100",\n      "kind": "kindergarten",\n      "name": "Детский сад №687",\n      "lat": 55.7955806,\n      "lon": 37.7954579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50072669",\n      "kind": "school",\n      "name": "Школа №444",\n      "lat": 55.7914234,\n      "lon": 37.7963295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123050",\n      "kind": "school",\n      "name": "Школа 1379",\n      "lat": 55.673814,\n      "lon": 37.5510975,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123057",\n      "kind": "kindergarten",\n      "name": "Детский сад 78 РАН",\n      "lat": 55.6743555,\n      "lon": 37.5503543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123061",\n      "kind": "kindergarten",\n      "name": "Детский сад 2242 Института Физической Химии и Электрохимии им. Фрумкина А.Н.",\n      "lat": 55.6733306,\n      "lon": 37.5534475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123062",\n      "kind": "kindergarten",\n      "name": "Детский сад 1780 РАН",\n      "lat": 55.6730881,\n      "lon": 37.5517606,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123064",\n      "kind": "school",\n      "name": "Школа 1205",\n      "lat": 55.676182,\n      "lon": 37.5566572,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123075",\n      "kind": "school",\n      "name": "Школа 1205",\n      "lat": 55.6769536,\n      "lon": 37.5544505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50319730",\n      "kind": "kindergarten",\n      "name": "Семейный центр раннего развития и творчества \\"Умняша\\"",\n      "lat": 55.6365119,\n      "lon": 37.5254485,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50343593",\n      "kind": "school",\n      "name": "",\n      "lat": 55.668754,\n      "lon": 37.6537782,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50548821",\n      "kind": "kindergarten",\n      "name": "Детский сад №1280 Общевойсковой академии ВC РФ",\n      "lat": 55.7583177,\n      "lon": 37.7029153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50584301",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7430308,\n      "lon": 37.7714362,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50584411",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7460384,\n      "lon": 37.7702369,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50672832",\n      "kind": "school",\n      "name": "Школа №751 (начальные классы)",\n      "lat": 55.8843317,\n      "lon": 37.6969641,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51036156",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус №5",\n      "lat": 55.6328197,\n      "lon": 37.5345137,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51036163",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус №4",\n      "lat": 55.6340595,\n      "lon": 37.53756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51239305",\n      "kind": "kindergarten",\n      "name": "Гимназия №1799, дошкольное отделение 4",\n      "lat": 55.801374,\n      "lon": 37.7183797,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51239312",\n      "kind": "school",\n      "name": "Гимназия № 1799, Школьное отделение 1",\n      "lat": 55.8016238,\n      "lon": 37.7211325,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51239411",\n      "kind": "kindergarten",\n      "name": "Гимназия №1080, дошкольное отделение 1",\n      "lat": 55.7992081,\n      "lon": 37.7206594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51244872",\n      "kind": "kindergarten",\n      "name": "Гимназия №1799, дошкольное отделение 2",\n      "lat": 55.8003305,\n      "lon": 37.7291861,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51244893",\n      "kind": "kindergarten",\n      "name": "Гимназия №1799, дошкольное отделение 3",\n      "lat": 55.8002886,\n      "lon": 37.7257222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51273173",\n      "kind": "school",\n      "name": "Школа №1310",\n      "lat": 55.7426753,\n      "lon": 37.7825148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51273194",\n      "kind": "kindergarten",\n      "name": "Детский сад №2001",\n      "lat": 55.7581305,\n      "lon": 37.7922867,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51304603",\n      "kind": "school",\n      "name": "Учебно-производственный центр \\"МКМ\\"",\n      "lat": 55.7449401,\n      "lon": 37.713418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51357392",\n      "kind": "school",\n      "name": "Школа № 1273. Корпус школьного образования №6",\n      "lat": 55.627384,\n      "lon": 37.5241024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51357394",\n      "kind": "kindergarten",\n      "name": "Школа № 1273. Корпус дошкольного образования №5",\n      "lat": 55.6265796,\n      "lon": 37.5234083,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51357399",\n      "kind": "kindergarten",\n      "name": "Школа № 1273. Корпус дошкольного образования №4",\n      "lat": 55.6261186,\n      "lon": 37.5227277,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51422928",\n      "kind": "school",\n      "name": "Гимназия №1799 Школьное отделение 3",\n      "lat": 55.8018196,\n      "lon": 37.7168169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51422970",\n      "kind": "school",\n      "name": "Школа 1080",\n      "lat": 55.8041024,\n      "lon": 37.72746,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51480004",\n      "kind": "kindergarten",\n      "name": "Детский сад №75",\n      "lat": 55.8102594,\n      "lon": 37.7323787,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51550808",\n      "kind": "school",\n      "name": "",\n      "lat": 55.73437,\n      "lon": 37.5667191,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51675683",\n      "kind": "school",\n      "name": "Школа № 1987. Начальная школа",\n      "lat": 55.6597354,\n      "lon": 37.7758,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51684041",\n      "kind": "school",\n      "name": "Лицей № 1533",\n      "lat": 55.6807249,\n      "lon": 37.5617659,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51684048",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6814392,\n      "lon": 37.5604461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51690266",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6510804,\n      "lon": 37.713926,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51690287",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6519557,\n      "lon": 37.7134848,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51812359",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №4",\n      "lat": 55.650346,\n      "lon": 37.6082315,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51832350",\n      "kind": "school",\n      "name": "Школа № 2016. Корпус школьного образования №1",\n      "lat": 55.651927,\n      "lon": 37.6080146,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51832351",\n      "kind": "kindergarten",\n      "name": "Школа № 1862. Корпус дошкольного образования №7",\n      "lat": 55.6519774,\n      "lon": 37.6097511,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51858795",\n      "kind": "kindergarten",\n      "name": "Детский сад № 62 «Жемчужина» (второе здание)",\n      "lat": 55.8945296,\n      "lon": 37.7236455,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51858798",\n      "kind": "school",\n      "name": "МБОУ СОШ №5",\n      "lat": 55.8912361,\n      "lon": 37.7183033,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52015581",\n      "kind": "kindergarten",\n      "name": "Школа № 1862. Корпус дошкольного образования №6",\n      "lat": 55.6471299,\n      "lon": 37.6081545,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52018812",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8465755,\n      "lon": 37.487374,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52025650",\n      "kind": "school",\n      "name": "Школа №\xa01315",\n      "lat": 55.8478303,\n      "lon": 37.4861106,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52047452",\n      "kind": "kindergarten",\n      "name": "Детский сад № 62 «Жемчужина» (первое здание)",\n      "lat": 55.8966812,\n      "lon": 37.7109465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52454805",\n      "kind": "kindergarten",\n      "name": "Детский сад №1942",\n      "lat": 55.7313906,\n      "lon": 37.6803359,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52456628",\n      "kind": "school",\n      "name": "Школа раннего развития «Приоритет»",\n      "lat": 55.8009232,\n      "lon": 37.4526391,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52462301",\n      "kind": "school",\n      "name": "МЦО \\"Атлантик\\"",\n      "lat": 55.7174892,\n      "lon": 37.6722807,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52462352",\n      "kind": "kindergarten",\n      "name": "Детский сад №476",\n      "lat": 55.7168388,\n      "lon": 37.6731276,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52462356",\n      "kind": "school",\n      "name": "Школа № 500",\n      "lat": 55.7133713,\n      "lon": 37.6727839,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52463398",\n      "kind": "kindergarten",\n      "name": "Детский сад №77",\n      "lat": 55.723668,\n      "lon": 37.6652131,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52476785",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7276914,\n      "lon": 37.6751008,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52476786",\n      "kind": "school",\n      "name": "Интернат №4",\n      "lat": 55.7271974,\n      "lon": 37.6752442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564709",\n      "kind": "school",\n      "name": "Кадетская школа-интернат №69",\n      "lat": 55.6704931,\n      "lon": 37.523219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564710",\n      "kind": "school",\n      "name": "Школа-интернат №62",\n      "lat": 55.6706078,\n      "lon": 37.5254816,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564712",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6709763,\n      "lon": 37.5248203,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564713",\n      "kind": "school",\n      "name": "Центр образования №1311 Тхия",\n      "lat": 55.6716899,\n      "lon": 37.5236035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564722",\n      "kind": "school",\n      "name": "Школа-интернат №62",\n      "lat": 55.6713823,\n      "lon": 37.5251384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52577428",\n      "kind": "school",\n      "name": "СУНЦ МГУ",\n      "lat": 55.7194408,\n      "lon": 37.464605,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52582033",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7680701,\n      "lon": 37.5918264,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53213796",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус школьного образования №9",\n      "lat": 55.615628,\n      "lon": 37.5333957,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53445879",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7435004,\n      "lon": 37.6639174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53587339",\n      "kind": "school",\n      "name": "Школа №967, школьное отделение №1",\n      "lat": 55.8791233,\n      "lon": 37.6522682,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53722254",\n      "kind": "school",\n      "name": "ГБОУ ШМК Бибирево",\n      "lat": 55.8882138,\n      "lon": 37.6191407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53816214",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7340621,\n      "lon": 37.6279421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54255822",\n      "kind": "school",\n      "name": "Школа № 2126 \\"Перово\\"",\n      "lat": 55.7425888,\n      "lon": 37.7623087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54365845",\n      "kind": "school",\n      "name": "Лицей №1571 корпус №5",\n      "lat": 55.8560964,\n      "lon": 37.4442916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54571601",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.875911,\n      "lon": 37.6762791,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54571602",\n      "kind": "school",\n      "name": "Центр дополнительного образования 1095",\n      "lat": 55.8768013,\n      "lon": 37.6775765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54625418",\n      "kind": "school",\n      "name": "",\n      "lat": 55.65413,\n      "lon": 37.5573682,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54675243",\n      "kind": "school",\n      "name": "Специальная (коррекционная) школа-интернат № 102",\n      "lat": 55.8534073,\n      "lon": 37.6517343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54830058",\n      "kind": "school",\n      "name": "Центр образования № 1828 \\"Сабурово\\"",\n      "lat": 55.639843,\n      "lon": 37.6971665,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54834851",\n      "kind": "school",\n      "name": "Спорткомплекс «Борисово»",\n      "lat": 55.636303,\n      "lon": 37.726978,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54864830",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №433",\n      "lat": 55.7227709,\n      "lon": 37.7370463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54905924",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение Ж5",\n      "lat": 55.7328546,\n      "lon": 37.8244261,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55340453",\n      "kind": "school",\n      "name": "Школа №\xa0519",\n      "lat": 55.6839751,\n      "lon": 37.7201125,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55503336",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 8 «Ласточка»",\n      "lat": 55.7562818,\n      "lon": 37.5209215,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55884817",\n      "kind": "kindergarten",\n      "name": "Детский сад №1932",\n      "lat": 55.6498974,\n      "lon": 37.7366286,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55914705",\n      "kind": "school",\n      "name": "Школа №1223",\n      "lat": 55.8386699,\n      "lon": 37.5272356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55914757",\n      "kind": "school",\n      "name": "Школа №90 \\"Многопрофильный образовательный комплекс\\" имени Героя Советского Союза Е.Г.Ларикова",\n      "lat": 55.8390658,\n      "lon": 37.5259885,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55970598",\n      "kind": "kindergarten",\n      "name": "Начальная школа - детский сад №1645",\n      "lat": 55.7145319,\n      "lon": 37.7737869,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55970606",\n      "kind": "kindergarten",\n      "name": "Детский сад №934",\n      "lat": 55.7132453,\n      "lon": 37.768591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55974513",\n      "kind": "school",\n      "name": "Знайка",\n      "lat": 55.8421141,\n      "lon": 37.5314573,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56077411",\n      "kind": "school",\n      "name": "Школа №891",\n      "lat": 55.7211327,\n      "lon": 37.8292776,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56080280",\n      "kind": "kindergarten",\n      "name": "Детский сад №856",\n      "lat": 55.7220138,\n      "lon": 37.8301334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56460825",\n      "kind": "school",\n      "name": "Лицей №1553",\n      "lat": 55.7210683,\n      "lon": 37.6491053,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56460850",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7266649,\n      "lon": 37.650446,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56461516",\n      "kind": "kindergarten",\n      "name": "Детский сад №1091",\n      "lat": 55.6042732,\n      "lon": 37.7284765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56677805",\n      "kind": "kindergarten",\n      "name": "Детский сад №2542",\n      "lat": 55.7141247,\n      "lon": 37.7560949,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56677826",\n      "kind": "kindergarten",\n      "name": "Детский сад №2612",\n      "lat": 55.7153842,\n      "lon": 37.7490634,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56677833",\n      "kind": "kindergarten",\n      "name": "Детский сад",\n      "lat": 55.7180951,\n      "lon": 37.7459423,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56677839",\n      "kind": "school",\n      "name": "Школа №1386",\n      "lat": 55.7164928,\n      "lon": 37.7488785,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56725006",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1713",\n      "lat": 55.7126456,\n      "lon": 37.7469512,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56757597",\n      "kind": "school",\n      "name": "Школа №488",\n      "lat": 55.7121068,\n      "lon": 37.742159,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56759631",\n      "kind": "school",\n      "name": "Центр образования №1404 Гамма",\n      "lat": 55.7161704,\n      "lon": 37.8271856,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56759689",\n      "kind": "school",\n      "name": "Гимназия №1512",\n      "lat": 55.7184103,\n      "lon": 37.8302964,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56871236",\n      "kind": "kindergarten",\n      "name": "Детский сад №755",\n      "lat": 55.7127134,\n      "lon": 37.7856828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/57393909",\n      "kind": "kindergarten",\n      "name": "Начальная школа - Детский сад №1767",\n      "lat": 55.8853988,\n      "lon": 37.6586488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/58742202",\n      "kind": "school",\n      "name": "Школа №751",\n      "lat": 55.8841648,\n      "lon": 37.698794,\n      "opening_hours": null\n    },\n    {\n      "id": "way/58780525",\n      "kind": "school",\n      "name": "Школа №947",\n      "lat": 55.5847305,\n      "lon": 37.6739715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/58987047",\n      "kind": "kindergarten",\n      "name": "Детский сад №92",\n      "lat": 55.8644922,\n      "lon": 37.6714101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59370210",\n      "kind": "school",\n      "name": "Школа №233",\n      "lat": 55.8746037,\n      "lon": 37.6504494,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59374931",\n      "kind": "school",\n      "name": "Школа № 1375",\n      "lat": 55.6813942,\n      "lon": 37.6566679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59404034",\n      "kind": "school",\n      "name": "Школа №311",\n      "lat": 55.8742486,\n      "lon": 37.6624178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59409572",\n      "kind": "kindergarten",\n      "name": "Детский сад №598",\n      "lat": 55.8667985,\n      "lon": 37.6605097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59603291",\n      "kind": "kindergarten",\n      "name": "Детский сад №2082",\n      "lat": 55.7971147,\n      "lon": 37.5112565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59605079",\n      "kind": "kindergarten",\n      "name": "Детский сад № 325",\n      "lat": 55.8034453,\n      "lon": 37.5154065,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59684131",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №14",\n      "lat": 55.8266437,\n      "lon": 37.8188488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59966660",\n      "kind": "school",\n      "name": "",\n      "lat": 55.793969,\n      "lon": 37.5071484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/60093986",\n      "kind": "kindergarten",\n      "name": "Детский сад №1591",\n      "lat": 55.8669753,\n      "lon": 37.6525592,\n      "opening_hours": null\n    },\n    {\n      "id": "way/60335496",\n      "kind": "school",\n      "name": "Гимназия №1505",\n      "lat": 55.7955071,\n      "lon": 37.7283299,\n      "opening_hours": null\n    },\n    {\n      "id": "way/60417707",\n      "kind": "kindergarten",\n      "name": "Детский сад 2688",\n      "lat": 55.649298,\n      "lon": 37.5802359,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61240218",\n      "kind": "kindergarten",\n      "name": "Детский сад №1744",\n      "lat": 55.8733695,\n      "lon": 37.6707086,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61299004",\n      "kind": "school",\n      "name": "Школа №368 (Здание №1)",\n      "lat": 55.8221012,\n      "lon": 37.8180648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61724032",\n      "kind": "school",\n      "name": "Школа №953, ШО 4",\n      "lat": 55.8966981,\n      "lon": 37.5960061,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61732784",\n      "kind": "school",\n      "name": "Школа №249",\n      "lat": 55.8944131,\n      "lon": 37.5941337,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61733498",\n      "kind": "school",\n      "name": "Школа №953, ШО 1",\n      "lat": 55.8966844,\n      "lon": 37.5896343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61891745",\n      "kind": "kindergarten",\n      "name": "Детский сад №1765",\n      "lat": 55.8954586,\n      "lon": 37.5900087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61891748",\n      "kind": "kindergarten",\n      "name": "Детский сад №938",\n      "lat": 55.896791,\n      "lon": 37.5924669,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61893179",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 13",\n      "lat": 55.8607097,\n      "lon": 37.6442328,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61915162",\n      "kind": "school",\n      "name": "Школа №139",\n      "lat": 55.8914523,\n      "lon": 37.5930648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/62112479",\n      "kind": "school",\n      "name": "Общеобразовательная школа «Локомотив»",\n      "lat": 55.8040485,\n      "lon": 37.74375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/62224282",\n      "kind": "school",\n      "name": "Лицей №1571",\n      "lat": 55.8565268,\n      "lon": 37.4440965,\n      "opening_hours": null\n    },\n    {\n      "id": "way/63089977",\n      "kind": "school",\n      "name": "Школа совхоза им. Ленина",\n      "lat": 55.5848122,\n      "lon": 37.7281802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/63090016",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.5858246,\n      "lon": 37.7306579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/63090060",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.5863722,\n      "lon": 37.7306297,\n      "opening_hours": null\n    },\n    {\n      "id": "way/64612074",\n      "kind": "school",\n      "name": "Центр образования №1483 (встр)",\n      "lat": 55.790625,\n      "lon": 37.7936681,\n      "opening_hours": null\n    },\n    {\n      "id": "way/66860549",\n      "kind": "kindergarten",\n      "name": "Детский сад №76",\n      "lat": 55.8035138,\n      "lon": 37.4514418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/67071359",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1129",\n      "lat": 55.6816,\n      "lon": 37.6593821,\n      "opening_hours": null\n    },\n    {\n      "id": "way/68001687",\n      "kind": "school",\n      "name": "Школа №878, отделение №1",\n      "lat": 55.6097946,\n      "lon": 37.7011671,\n      "opening_hours": null\n    },\n    {\n      "id": "way/68260689",\n      "kind": "school",\n      "name": "ЦО №627",\n      "lat": 55.7242809,\n      "lon": 37.6355556,\n      "opening_hours": null\n    },\n    {\n      "id": "way/68260701",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8930185,\n      "lon": 37.621498,\n      "opening_hours": null\n    },\n    {\n      "id": "way/68260702",\n      "kind": "kindergarten",\n      "name": "Детский сад №2152",\n      "lat": 55.893079,\n      "lon": 37.6239456,\n      "opening_hours": null\n    },\n    {\n      "id": "way/69323123",\n      "kind": "school",\n      "name": "Школа №\xa01527",\n      "lat": 55.6822055,\n      "lon": 37.6689442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/69728519",\n      "kind": "school",\n      "name": "Школа №956",\n      "lat": 55.8678901,\n      "lon": 37.6426931,\n      "opening_hours": null\n    },\n    {\n      "id": "way/73854348",\n      "kind": "school",\n      "name": "ГБОУ СОШ № 978 здание № 2",\n      "lat": 55.6816762,\n      "lon": 37.6427718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/73855179",\n      "kind": "school",\n      "name": "ГБОУ школа № 463 корпус №1",\n      "lat": 55.6793275,\n      "lon": 37.6726477,\n      "opening_hours": "Mo-Sa 08:00-20:00"\n    },\n    {\n      "id": "way/75392188",\n      "kind": "kindergarten",\n      "name": "Школа № 1503 Дошкольное отделение",\n      "lat": 55.8158418,\n      "lon": 37.6601449,\n      "opening_hours": null\n    },\n    {\n      "id": "way/75649333",\n      "kind": "kindergarten",\n      "name": "Детский сад №1360",\n      "lat": 55.6068614,\n      "lon": 37.7075582,\n      "opening_hours": null\n    },\n    {\n      "id": "way/76212064",\n      "kind": "school",\n      "name": "Школа №218",\n      "lat": 55.8085809,\n      "lon": 37.5608891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/76345960",\n      "kind": "school",\n      "name": "Центр образования №\xa0218",\n      "lat": 55.8121972,\n      "lon": 37.5747946,\n      "opening_hours": null\n    },\n    {\n      "id": "way/76593145",\n      "kind": "kindergarten",\n      "name": "Детский сад № 706",\n      "lat": 55.6115097,\n      "lon": 37.5873778,\n      "opening_hours": null\n    },\n    {\n      "id": "way/77352043",\n      "kind": "school",\n      "name": "Школа №88",\n      "lat": 55.7368012,\n      "lon": 37.6812459,\n      "opening_hours": null\n    },\n    {\n      "id": "way/77352047",\n      "kind": "kindergarten",\n      "name": "Детский сад №2639",\n      "lat": 55.7366715,\n      "lon": 37.6827566,\n      "opening_hours": null\n    },\n    {\n      "id": "way/77611759",\n      "kind": "school",\n      "name": "Школа №646",\n      "lat": 55.7976044,\n      "lon": 37.7853647,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78273780",\n      "kind": "kindergarten",\n      "name": "Детский сад №1517",\n      "lat": 55.7810407,\n      "lon": 37.7265749,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78287618",\n      "kind": "school",\n      "name": "Школа №1955",\n      "lat": 55.8796704,\n      "lon": 37.68846,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78287658",\n      "kind": "kindergarten",\n      "name": "Детский сад №769",\n      "lat": 55.882664,\n      "lon": 37.6880101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78444429",\n      "kind": "kindergarten",\n      "name": "Детский сад №1755",\n      "lat": 55.7957967,\n      "lon": 37.7923881,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78446879",\n      "kind": "kindergarten",\n      "name": "Детский сад №398",\n      "lat": 55.7349748,\n      "lon": 37.4925174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78628390",\n      "kind": "school",\n      "name": "Школа № 1524 к.2",\n      "lat": 55.6927455,\n      "lon": 37.7200943,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78641368",\n      "kind": "kindergarten",\n      "name": "Детский сад №660",\n      "lat": 55.7945997,\n      "lon": 37.7971727,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78712467",\n      "kind": "school",\n      "name": "Гармония",\n      "lat": 55.5968405,\n      "lon": 37.7467676,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78719017",\n      "kind": "school",\n      "name": "Образовательный центр \\"Газпром\\"",\n      "lat": 55.6628519,\n      "lon": 37.5586915,\n      "opening_hours": "Mo-Fr 08:00-18:30"\n    },\n    {\n      "id": "way/78730245",\n      "kind": "school",\n      "name": "Развилковская средняя школа",\n      "lat": 55.5876329,\n      "lon": 37.7606004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79054130",\n      "kind": "school",\n      "name": "Школа №419",\n      "lat": 55.799679,\n      "lon": 37.7963295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79181708",\n      "kind": "kindergarten",\n      "name": "Детский сад №184",\n      "lat": 55.7370857,\n      "lon": 37.4790318,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79394593",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Ладушки»",\n      "lat": 55.8644779,\n      "lon": 37.4443528,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79529590",\n      "kind": "kindergarten",\n      "name": "Детский сад №1435",\n      "lat": 55.8012194,\n      "lon": 37.7868468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79534444",\n      "kind": "school",\n      "name": "Школа №1301 имени Е. Т. Гайдара",\n      "lat": 55.7805738,\n      "lon": 37.7237752,\n      "opening_hours": null\n    },\n    {\n      "id": "way/80033791",\n      "kind": "school",\n      "name": "Центр образования №429",\n      "lat": 55.7754703,\n      "lon": 37.7248509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/80632666",\n      "kind": "kindergarten",\n      "name": "Школа - интернат №1",\n      "lat": 55.8173565,\n      "lon": 37.6540721,\n      "opening_hours": null\n    },\n    {\n      "id": "way/81020739",\n      "kind": "kindergarten",\n      "name": "Школа №760 им. А. П. Маресьева. Дошкольное отделение",\n      "lat": 55.8752972,\n      "lon": 37.713858,\n      "opening_hours": null\n    },\n    {\n      "id": "way/81240143",\n      "kind": "kindergarten",\n      "name": "Детский сад №766 присмотра и оздоровления",\n      "lat": 55.8105527,\n      "lon": 37.561927,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/81950814",\n      "kind": "school",\n      "name": "Школа №11",\n      "lat": 55.6887606,\n      "lon": 37.538679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82083171",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.6400957,\n      "lon": 37.544641,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82083420",\n      "kind": "school",\n      "name": "Начальная школа №2553",\n      "lat": 55.7522833,\n      "lon": 37.8284181,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82083424",\n      "kind": "kindergarten",\n      "name": "Детский сад №1559",\n      "lat": 55.753749,\n      "lon": 37.8290121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82137808",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.687607,\n      "lon": 37.5373639,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82210430",\n      "kind": "kindergarten",\n      "name": "Детский сад №1773",\n      "lat": 55.7396047,\n      "lon": 37.4919244,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83015042",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Радуга»",\n      "lat": 55.8639723,\n      "lon": 37.4397699,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83059652",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7714574,\n      "lon": 37.4889188,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83103906",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение В6",\n      "lat": 55.7359062,\n      "lon": 37.8232353,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83138645",\n      "kind": "school",\n      "name": "Школа №1139",\n      "lat": 55.8609718,\n      "lon": 37.6771567,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83138648",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8595676,\n      "lon": 37.6761092,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83147492",\n      "kind": "kindergarten",\n      "name": "Детский сад № 317",\n      "lat": 55.8139278,\n      "lon": 37.5616617,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83147494",\n      "kind": "school",\n      "name": "Школа № 1454 «Тимирязевская»",\n      "lat": 55.8125141,\n      "lon": 37.5605762,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83168208",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1738",\n      "lat": 55.7705664,\n      "lon": 37.4911491,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83168243",\n      "kind": "school",\n      "name": "Школа № 1010",\n      "lat": 55.770877,\n      "lon": 37.492178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83238455",\n      "kind": "school",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ШП-3)",\n      "lat": 55.6743875,\n      "lon": 37.8097393,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83257895",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7779766,\n      "lon": 37.4831405,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83257957",\n      "kind": "school",\n      "name": "Школа 1605",\n      "lat": 55.7789532,\n      "lon": 37.4819083,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83258043",\n      "kind": "school",\n      "name": "Лицей №1560 Многопрофильный",\n      "lat": 55.7803493,\n      "lon": 37.4840806,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83290498",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7801254,\n      "lon": 37.4958345,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83389681",\n      "kind": "kindergarten",\n      "name": "Д/C № 367",\n      "lat": 55.6205707,\n      "lon": 37.6510102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83389694",\n      "kind": "school",\n      "name": "Школа № 868",\n      "lat": 55.6212498,\n      "lon": 37.6533046,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83461703",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8717013,\n      "lon": 37.6291789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83686950",\n      "kind": "school",\n      "name": "Школа №606",\n      "lat": 55.8718432,\n      "lon": 37.6135458,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83686964",\n      "kind": "kindergarten",\n      "name": "Школа № 1411. Структурное подразделение Детский сад № 370",\n      "lat": 55.8708899,\n      "lon": 37.6022976,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83686968",\n      "kind": "kindergarten",\n      "name": "Школа № 1411. Структурное подразделение Детский сад № 939",\n      "lat": 55.8699699,\n      "lon": 37.5981217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83686971",\n      "kind": "school",\n      "name": "Школа №1411 (СП-761)",\n      "lat": 55.8687417,\n      "lon": 37.5961069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83687271",\n      "kind": "school",\n      "name": "Школа №970",\n      "lat": 55.8726849,\n      "lon": 37.588898,\n      "opening_hours": null\n    },\n    {\n      "id": "way/84435931",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Почемучки»",\n      "lat": 55.8584314,\n      "lon": 37.4396335,\n      "opening_hours": null\n    },\n    {\n      "id": "way/84973400",\n      "kind": "school",\n      "name": "Специальная (коррекционная) общеобразовательная школа-интернат № 52",\n      "lat": 55.8135341,\n      "lon": 37.5655981,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85122909",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8114369,\n      "lon": 37.5636635,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85152965",\n      "kind": "kindergarten",\n      "name": "СПДО № 2 школы № 1784",\n      "lat": 55.7826534,\n      "lon": 37.5640949,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85282869",\n      "kind": "school",\n      "name": "Школа № 2115. Корпус школьного образования № 3",\n      "lat": 55.6693903,\n      "lon": 37.5638749,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85641529",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1384 имени А. А. Леманского\\"",\n      "lat": 55.8036484,\n      "lon": 37.5112167,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85642991",\n      "kind": "school",\n      "name": "Школа № 870",\n      "lat": 55.6238793,\n      "lon": 37.6639606,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85666971",\n      "kind": "school",\n      "name": "Школа №1363",\n      "lat": 55.6978026,\n      "lon": 37.8194295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85711998",\n      "kind": "kindergarten",\n      "name": "Школа №1310 (дошкольный корпус №1)",\n      "lat": 55.7446156,\n      "lon": 37.7866135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85851078",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение школы № 547",\n      "lat": 55.7099323,\n      "lon": 37.6408752,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85910107",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 15",\n      "lat": 55.8528525,\n      "lon": 37.6599388,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85982043",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7979461,\n      "lon": 37.5042209,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85982044",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7972561,\n      "lon": 37.504341,\n      "opening_hours": null\n    },\n    {\n      "id": "way/86163498",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №3",\n      "lat": 55.6594257,\n      "lon": 37.7625936,\n      "opening_hours": null\n    },\n    {\n      "id": "way/86251431",\n      "kind": "school",\n      "name": "Лицей №\xa01571, корпус 7 «Созвездие»",\n      "lat": 55.8588754,\n      "lon": 37.4441623,\n      "opening_hours": null\n    },\n    {\n      "id": "way/86251581",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Ручеёк»",\n      "lat": 55.8587131,\n      "lon": 37.442513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/86552971",\n      "kind": "school",\n      "name": "Школа № 950",\n      "lat": 55.8589225,\n      "lon": 37.6185548,\n      "opening_hours": null\n    },\n    {\n      "id": "way/87035878",\n      "kind": "kindergarten",\n      "name": "детский сад 1948 РАН",\n      "lat": 55.6738059,\n      "lon": 37.5497594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/87296431",\n      "kind": "school",\n      "name": "Школа №2000 корпус №1",\n      "lat": 55.6421954,\n      "lon": 37.6600432,\n      "opening_hours": null\n    },\n    {\n      "id": "way/87297154",\n      "kind": "kindergarten",\n      "name": "Детский сад №800",\n      "lat": 55.641486,\n      "lon": 37.6587902,\n      "opening_hours": null\n    },\n    {\n      "id": "way/87681246",\n      "kind": "school",\n      "name": "ГОУ ЦО «Технологии обучения»",\n      "lat": 55.7411982,\n      "lon": 37.5501732,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88175214",\n      "kind": "school",\n      "name": "Школа надомного обучения №388",\n      "lat": 55.8938225,\n      "lon": 37.5896932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88301012",\n      "kind": "school",\n      "name": "Школа № 547",\n      "lat": 55.7092631,\n      "lon": 37.6404573,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88306971",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №4",\n      "lat": 55.61007,\n      "lon": 37.6005377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88397811",\n      "kind": "school",\n      "name": "Центр образования № 1408",\n      "lat": 55.7006786,\n      "lon": 37.7750219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88609750",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №5",\n      "lat": 55.6108738,\n      "lon": 37.5966035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89153919",\n      "kind": "school",\n      "name": "Интернат №73",\n      "lat": 55.6647519,\n      "lon": 37.6081418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89177699",\n      "kind": "kindergarten",\n      "name": "Детский сад №1046",\n      "lat": 55.8925978,\n      "lon": 37.5926686,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89177701",\n      "kind": "kindergarten",\n      "name": "Детский сад №908",\n      "lat": 55.8944979,\n      "lon": 37.5924914,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89177709",\n      "kind": "kindergarten",\n      "name": "Детский сад №1751",\n      "lat": 55.8959559,\n      "lon": 37.6008785,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89297039",\n      "kind": "school",\n      "name": "Воскресная школа при храме",\n      "lat": 55.6912113,\n      "lon": 37.5938082,\n      "opening_hours": null\n    },\n    {\n      "id": "way/90834744",\n      "kind": "school",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 3.",\n      "lat": 55.651123,\n      "lon": 37.5190565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/91211981",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8129805,\n      "lon": 37.639327,\n      "opening_hours": null\n    },\n    {\n      "id": "way/91233158",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8063841,\n      "lon": 37.6381811,\n      "opening_hours": null\n    },\n    {\n      "id": "way/92139340",\n      "kind": "school",\n      "name": "Гимназия №1539",\n      "lat": 55.8105066,\n      "lon": 37.6418595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/93181279",\n      "kind": "school",\n      "name": "Школа №880",\n      "lat": 55.609456,\n      "lon": 37.5888457,\n      "opening_hours": null\n    },\n    {\n      "id": "way/93476938",\n      "kind": "school",\n      "name": "Школа №\xa01474, подразделение 12",\n      "lat": 55.8707138,\n      "lon": 37.4931824,\n      "opening_hours": null\n    },\n    {\n      "id": "way/93527752",\n      "kind": "school",\n      "name": "Школа № 688",\n      "lat": 55.7685831,\n      "lon": 37.7331741,\n      "opening_hours": null\n    },\n    {\n      "id": "way/93911126",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №4",\n      "lat": 55.6612938,\n      "lon": 37.7621642,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94054910",\n      "kind": "school",\n      "name": "Гимназия №1508",\n      "lat": 55.7920862,\n      "lon": 37.8026258,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94448275",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Росток\\"",\n      "lat": 55.7937162,\n      "lon": 37.488517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94527801",\n      "kind": "kindergarten",\n      "name": "Детский сад №2348",\n      "lat": 55.6532855,\n      "lon": 37.7615797,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94527802",\n      "kind": "school",\n      "name": "Школа №1319",\n      "lat": 55.6525027,\n      "lon": 37.7603479,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94528469",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"Факультет\\"",\n      "lat": 55.7973449,\n      "lon": 37.4884448,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94837685",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Винни Пух\\"",\n      "lat": 55.7907943,\n      "lon": 37.4946621,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95172319",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8842057,\n      "lon": 37.6980489,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95172322",\n      "kind": "school",\n      "name": "Школа №763",\n      "lat": 55.8837346,\n      "lon": 37.694233,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95172323",\n      "kind": "kindergarten",\n      "name": "Детский сад №1313",\n      "lat": 55.8852125,\n      "lon": 37.6947536,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95406391",\n      "kind": "school",\n      "name": "Школа №51. Корпус №4",\n      "lat": 55.6442337,\n      "lon": 37.482071,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95692120",\n      "kind": "kindergarten",\n      "name": "Школа 1948 \\"Лингвист-М\\" (д/с 2481)",\n      "lat": 55.6644719,\n      "lon": 37.5749833,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95951840",\n      "kind": "school",\n      "name": "Школа №1955",\n      "lat": 55.8849929,\n      "lon": 37.6831031,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96018896",\n      "kind": "school",\n      "name": "Школа №1207",\n      "lat": 55.6058069,\n      "lon": 37.7097687,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96059113",\n      "kind": "kindergarten",\n      "name": "Детский сад №1241",\n      "lat": 55.607824,\n      "lon": 37.7118112,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96107115",\n      "kind": "kindergarten",\n      "name": "Детский сад №903",\n      "lat": 55.6183098,\n      "lon": 37.721841,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96109435",\n      "kind": "kindergarten",\n      "name": "Д/C № 563",\n      "lat": 55.6259636,\n      "lon": 37.6496628,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96112681",\n      "kind": "kindergarten",\n      "name": "Д/C № 591",\n      "lat": 55.6211785,\n      "lon": 37.6511437,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96114968",\n      "kind": "school",\n      "name": "Школа № 869",\n      "lat": 55.6155795,\n      "lon": 37.6468005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96124576",\n      "kind": "school",\n      "name": "Школа №575",\n      "lat": 55.6012632,\n      "lon": 37.7131464,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96124582",\n      "kind": "kindergarten",\n      "name": "Детский сад №1021",\n      "lat": 55.6027384,\n      "lon": 37.7100135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96124599",\n      "kind": "kindergarten",\n      "name": "Детский сад №857",\n      "lat": 55.6020808,\n      "lon": 37.7112422,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96202738",\n      "kind": "school",\n      "name": "Школа №916",\n      "lat": 55.6118493,\n      "lon": 37.7128652,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96208205",\n      "kind": "kindergarten",\n      "name": "Детский сад № 895",\n      "lat": 55.611131,\n      "lon": 37.7014884,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96264956",\n      "kind": "kindergarten",\n      "name": "Детский сад №986",\n      "lat": 55.6173257,\n      "lon": 37.7227532,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96283955",\n      "kind": "kindergarten",\n      "name": "Д/C № 163",\n      "lat": 55.6317878,\n      "lon": 37.6765845,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96283958",\n      "kind": "kindergarten",\n      "name": "Школа № 982 им. маршала бронетанковых войск П.П. Полубоярова. Здание № 4",\n      "lat": 55.6341199,\n      "lon": 37.6766468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96283965",\n      "kind": "school",\n      "name": "ЦЛПиДО",\n      "lat": 55.6299408,\n      "lon": 37.6753011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96300065",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8121897,\n      "lon": 37.7301897,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96324790",\n      "kind": "kindergarten",\n      "name": "Д/C № 600",\n      "lat": 55.6318039,\n      "lon": 37.6649141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96371786",\n      "kind": "kindergarten",\n      "name": "Детский сад № 24",\n      "lat": 55.8087556,\n      "lon": 37.7242701,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96447491",\n      "kind": "school",\n      "name": "Гимназия №1534 Школьное отделение №2",\n      "lat": 55.6820925,\n      "lon": 37.5707995,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96447494",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6752217,\n      "lon": 37.5785207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96455034",\n      "kind": "school",\n      "name": "ГБОУ Гимназия № 1534",\n      "lat": 55.6851349,\n      "lon": 37.5741934,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96597526",\n      "kind": "kindergarten",\n      "name": "Детский сад №38",\n      "lat": 55.5985097,\n      "lon": 37.7164217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96597533",\n      "kind": "kindergarten",\n      "name": "Детский сад №2375",\n      "lat": 55.5977136,\n      "lon": 37.7176618,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96597535",\n      "kind": "school",\n      "name": "Школа №544",\n      "lat": 55.5961737,\n      "lon": 37.7195198,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96638899",\n      "kind": "kindergarten",\n      "name": "Детский сад №2269",\n      "lat": 55.8852757,\n      "lon": 37.6928876,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96638900",\n      "kind": "kindergarten",\n      "name": "Детский сад №925",\n      "lat": 55.884937,\n      "lon": 37.6909481,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96644706",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7925475,\n      "lon": 37.4846736,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96645733",\n      "kind": "kindergarten",\n      "name": "Школа № 1503 Дошкольное отделение",\n      "lat": 55.8199575,\n      "lon": 37.6545599,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96675547",\n      "kind": "school",\n      "name": "Школа № 770",\n      "lat": 55.6348918,\n      "lon": 37.6821351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96709458",\n      "kind": "school",\n      "name": "Школа №7",\n      "lat": 55.8922821,\n      "lon": 37.7346039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96759949",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1874",\n      "lat": 55.7330247,\n      "lon": 37.6553346,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96844581",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1777",\n      "lat": 55.6270008,\n      "lon": 37.6720864,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96844598",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1974",\n      "lat": 55.6264337,\n      "lon": 37.6705947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96927392",\n      "kind": "school",\n      "name": "ЦДТ \\"Логос\\"",\n      "lat": 55.6304467,\n      "lon": 37.6799684,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96960537",\n      "kind": "kindergarten",\n      "name": "Детский сад №1344",\n      "lat": 55.8777739,\n      "lon": 37.7250149,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97145781",\n      "kind": "kindergarten",\n      "name": "Детский сад №1570",\n      "lat": 55.6080444,\n      "lon": 37.7146544,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97226291",\n      "kind": "kindergarten",\n      "name": "Детский сад № 529",\n      "lat": 55.6357515,\n      "lon": 37.6600789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97337335",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6319089,\n      "lon": 37.664614,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97631336",\n      "kind": "school",\n      "name": "ЦДТ \\"Царицыно\\"",\n      "lat": 55.6341756,\n      "lon": 37.6671377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97716798",\n      "kind": "school",\n      "name": "Школа №1179. Корпус школьного образования №2",\n      "lat": 55.6365862,\n      "lon": 37.6090692,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97716807",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №5",\n      "lat": 55.6360687,\n      "lon": 37.6158065,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97730991",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №1",\n      "lat": 55.639098,\n      "lon": 37.6113294,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97730993",\n      "kind": "school",\n      "name": "Школа №1179. Корпус школьного образования №1 (Главное здание)",\n      "lat": 55.6382055,\n      "lon": 37.6092185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97911511",\n      "kind": "kindergarten",\n      "name": "Детский сад №1787",\n      "lat": 55.6397391,\n      "lon": 37.6614493,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98185508",\n      "kind": "kindergarten",\n      "name": "Детский сад №1153",\n      "lat": 55.6105722,\n      "lon": 37.7036912,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98212210",\n      "kind": "kindergarten",\n      "name": "Детский сад №1092",\n      "lat": 55.6107658,\n      "lon": 37.7049468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98229514",\n      "kind": "kindergarten",\n      "name": "Детский сад №935",\n      "lat": 55.614347,\n      "lon": 37.7003563,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98229537",\n      "kind": "kindergarten",\n      "name": "Д/C №1969",\n      "lat": 55.6148465,\n      "lon": 37.7024205,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98229550",\n      "kind": "kindergarten",\n      "name": "Детский сад №842",\n      "lat": 55.6154347,\n      "lon": 37.7005803,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98247341",\n      "kind": "school",\n      "name": "Школа №979",\n      "lat": 55.6033917,\n      "lon": 37.7459276,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98330543",\n      "kind": "school",\n      "name": "Школа №832",\n      "lat": 55.605545,\n      "lon": 37.7043986,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98343467",\n      "kind": "school",\n      "name": "Школа №1771 (начальная)",\n      "lat": 55.6062206,\n      "lon": 37.7271051,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98354226",\n      "kind": "kindergarten",\n      "name": "Дом ребенка №25 (неврологический)",\n      "lat": 55.6081123,\n      "lon": 37.730773,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98354245",\n      "kind": "kindergarten",\n      "name": "Детский сад №1275",\n      "lat": 55.605061,\n      "lon": 37.7320255,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98599498",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Кораблик",\n      "lat": 55.6775586,\n      "lon": 37.6836862,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98607374",\n      "kind": "kindergarten",\n      "name": "Детский сад №1234",\n      "lat": 55.7175321,\n      "lon": 37.6290721,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98668679",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2679",\n      "lat": 55.6336053,\n      "lon": 37.6511293,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98668689",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1970",\n      "lat": 55.6334128,\n      "lon": 37.6523947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98925501",\n      "kind": "kindergarten",\n      "name": "Д/C № 1516",\n      "lat": 55.6252525,\n      "lon": 37.6535702,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98925570",\n      "kind": "school",\n      "name": "Школа №840",\n      "lat": 55.6260928,\n      "lon": 37.6527135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98943164",\n      "kind": "school",\n      "name": "Школа № 904",\n      "lat": 55.633515,\n      "lon": 37.6658515,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99166330",\n      "kind": "school",\n      "name": "Курчатовская школа \\"ФОК\\"",\n      "lat": 55.8050206,\n      "lon": 37.4699896,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99373841",\n      "kind": "kindergarten",\n      "name": "Детский сад школы №1370 корпус 5",\n      "lat": 55.8855669,\n      "lon": 37.5933237,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99373886",\n      "kind": "kindergarten",\n      "name": "Детский сад школы №1370 корпус 8",\n      "lat": 55.8848474,\n      "lon": 37.5889489,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99510199",\n      "kind": "school",\n      "name": "Школа № 733",\n      "lat": 55.7694685,\n      "lon": 37.7127513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99598008",\n      "kind": "school",\n      "name": "Школа №871",\n      "lat": 55.6174085,\n      "lon": 37.7357033,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99598020",\n      "kind": "school",\n      "name": "Центр образования №1433",\n      "lat": 55.61935,\n      "lon": 37.7356804,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99622644",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8097933,\n      "lon": 37.7211013,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99752573",\n      "kind": "school",\n      "name": "Школа №90 (СП-1)",\n      "lat": 55.7942603,\n      "lon": 37.5162208,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99780170",\n      "kind": "school",\n      "name": "Школа №1287 (1)",\n      "lat": 55.7962258,\n      "lon": 37.5205717,\n      "opening_hours": "Mo-Fr 09:00-18:00; Sa 09:00-15:00"\n    },\n    {\n      "id": "way/100169483",\n      "kind": "kindergarten",\n      "name": "Детский сад №1574",\n      "lat": 55.6109423,\n      "lon": 37.7067648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101161295",\n      "kind": "kindergarten",\n      "name": "Детский сад №1551",\n      "lat": 55.7513543,\n      "lon": 37.807635,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101264114",\n      "kind": "kindergarten",\n      "name": "Детский сад №1041",\n      "lat": 55.8186345,\n      "lon": 37.7172154,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101264120",\n      "kind": "school",\n      "name": "Школа №390",\n      "lat": 55.8176903,\n      "lon": 37.7158655,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101311726",\n      "kind": "kindergarten",\n      "name": "Детский сад №1823",\n      "lat": 55.6395708,\n      "lon": 37.6661035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101321916",\n      "kind": "school",\n      "name": "Школа №2000 корпус №2",\n      "lat": 55.641803,\n      "lon": 37.6608917,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101321942",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6415465,\n      "lon": 37.6607226,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101321948",\n      "kind": "kindergarten",\n      "name": "Детский сад №577",\n      "lat": 55.6414308,\n      "lon": 37.6625542,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101502907",\n      "kind": "kindergarten",\n      "name": "Детский сад №753",\n      "lat": 55.7494983,\n      "lon": 37.8103495,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101502964",\n      "kind": "school",\n      "name": "Школа № 2093",\n      "lat": 55.7504036,\n      "lon": 37.8037409,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101540604",\n      "kind": "kindergarten",\n      "name": "Школа №2094. Дошкольная образовательная площадка №1",\n      "lat": 55.6231653,\n      "lon": 37.4741144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101616652",\n      "kind": "kindergarten",\n      "name": "Д/C № 2371",\n      "lat": 55.6251211,\n      "lon": 37.6595298,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101696224",\n      "kind": "kindergarten",\n      "name": "Детский сад №1909",\n      "lat": 55.7515251,\n      "lon": 37.8043551,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101696225",\n      "kind": "kindergarten",\n      "name": "Гимназия №1798 - дошкольное образование «Фениксёнок»",\n      "lat": 55.7512753,\n      "lon": 37.8132308,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101696226",\n      "kind": "school",\n      "name": "Школа №787",\n      "lat": 55.746301,\n      "lon": 37.8108187,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101830726",\n      "kind": "school",\n      "name": "Школа № 501",\n      "lat": 55.6210528,\n      "lon": 37.663093,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101830737",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2071",\n      "lat": 55.6229417,\n      "lon": 37.6607776,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102000128",\n      "kind": "school",\n      "name": "Школа №1370 корпус 3",\n      "lat": 55.8829323,\n      "lon": 37.5911914,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102000135",\n      "kind": "kindergarten",\n      "name": "Детский сад школы №1370 корпус 6",\n      "lat": 55.8830588,\n      "lon": 37.59274,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102199974",\n      "kind": "kindergarten",\n      "name": "Детский сад №548",\n      "lat": 55.6198018,\n      "lon": 37.7025383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102199982",\n      "kind": "kindergarten",\n      "name": "Детский сад №1635",\n      "lat": 55.6196054,\n      "lon": 37.7035951,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102317311",\n      "kind": "kindergarten",\n      "name": "Школа №1568, дошкольное отделение",\n      "lat": 55.8774177,\n      "lon": 37.6301756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102317312",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.877194,\n      "lon": 37.621689,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102317313",\n      "kind": "school",\n      "name": "Школа №285 имени В.А. Молодцова",\n      "lat": 55.8762763,\n      "lon": 37.6302045,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102326794",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8732048,\n      "lon": 37.6317888,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102334393",\n      "kind": "kindergarten",\n      "name": "Детский сад №1082",\n      "lat": 55.6177507,\n      "lon": 37.725568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102334394",\n      "kind": "kindergarten",\n      "name": "Детский сад №995",\n      "lat": 55.6188684,\n      "lon": 37.7241093,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102714124",\n      "kind": "kindergarten",\n      "name": "Детский сад №1754",\n      "lat": 55.6389506,\n      "lon": 37.6756264,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102714130",\n      "kind": "school",\n      "name": "Школа №982",\n      "lat": 55.6407751,\n      "lon": 37.6745832,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102714133",\n      "kind": "kindergarten",\n      "name": "Детский сад №922",\n      "lat": 55.6385253,\n      "lon": 37.6784813,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102781292",\n      "kind": "kindergarten",\n      "name": "Детский сад №2620",\n      "lat": 55.6809863,\n      "lon": 37.5760262,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103134505",\n      "kind": "school",\n      "name": "Школа №975",\n      "lat": 55.640089,\n      "lon": 37.7741036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103134518",\n      "kind": "kindergarten",\n      "name": "Детский сад №1824",\n      "lat": 55.6410917,\n      "lon": 37.7728452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103244527",\n      "kind": "school",\n      "name": "Гимназия №1517",\n      "lat": 55.7858566,\n      "lon": 37.4580566,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103267450",\n      "kind": "school",\n      "name": "Школа №2200 корпус 4",\n      "lat": 55.8047933,\n      "lon": 37.7906194,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103467375",\n      "kind": "school",\n      "name": "Школа №2200 корпус 1",\n      "lat": 55.8073783,\n      "lon": 37.8063305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103562938",\n      "kind": "school",\n      "name": "Школа № 1384 имени А. А. Леманского",\n      "lat": 55.7982326,\n      "lon": 37.515783,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103566503",\n      "kind": "kindergarten",\n      "name": "Детский сад №1579",\n      "lat": 55.6390562,\n      "lon": 37.76572,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103566512",\n      "kind": "kindergarten",\n      "name": "Детский сад №1663",\n      "lat": 55.6368448,\n      "lon": 37.7677669,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103566513",\n      "kind": "kindergarten",\n      "name": "Детский сад №858",\n      "lat": 55.6375917,\n      "lon": 37.7670916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103568126",\n      "kind": "kindergarten",\n      "name": "Детский сад №1803",\n      "lat": 55.6432358,\n      "lon": 37.7736275,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103858727",\n      "kind": "kindergarten",\n      "name": "Детский сад №985",\n      "lat": 55.6154952,\n      "lon": 37.7390374,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103858738",\n      "kind": "school",\n      "name": "Школа №945",\n      "lat": 55.6153101,\n      "lon": 37.7366679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103858760",\n      "kind": "kindergarten",\n      "name": "Детский сад-начальная школа Премьер",\n      "lat": 55.6157908,\n      "lon": 37.7400707,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103884054",\n      "kind": "school",\n      "name": "Школа № 1370. Корпус 4",\n      "lat": 55.8815106,\n      "lon": 37.5942509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103884059",\n      "kind": "kindergarten",\n      "name": "Детский сад №1233 Теремок",\n      "lat": 55.8823738,\n      "lon": 37.5968444,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103884064",\n      "kind": "kindergarten",\n      "name": "Детский сад школы №1370 корпус 7",\n      "lat": 55.8811043,\n      "lon": 37.5902274,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104128400",\n      "kind": "kindergarten",\n      "name": "Центр развития ребенка \\"Дошколёнок\\"",\n      "lat": 55.8149763,\n      "lon": 37.4624898,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104222879",\n      "kind": "school",\n      "name": "Школа №1352 (корпус на Щёлковском шс., 77А)",\n      "lat": 55.813087,\n      "lon": 37.8042638,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104337973",\n      "kind": "kindergarten",\n      "name": "ДОУ №2661",\n      "lat": 55.723152,\n      "lon": 37.5570874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104531903",\n      "kind": "school",\n      "name": "Школа №1370 корпус 1",\n      "lat": 55.8837855,\n      "lon": 37.5940064,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104531904",\n      "kind": "school",\n      "name": "Школа №1370 корпус 2",\n      "lat": 55.8841771,\n      "lon": 37.5958879,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104539332",\n      "kind": "kindergarten",\n      "name": "Детский сад №914",\n      "lat": 55.6072533,\n      "lon": 37.7041397,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104635222",\n      "kind": "school",\n      "name": "Школа № 1536. Корпус школьного образования № 3",\n      "lat": 55.6839078,\n      "lon": 37.5458227,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104662200",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6450672,\n      "lon": 37.7165202,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104720652",\n      "kind": "kindergarten",\n      "name": "Детский сад №1515",\n      "lat": 55.720801,\n      "lon": 37.567976,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104978925",\n      "kind": "school",\n      "name": "Детская школа искусств «Родник»",\n      "lat": 55.6067823,\n      "lon": 37.7369376,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104978930",\n      "kind": "school",\n      "name": "Школа №1636, здание 10",\n      "lat": 55.6077717,\n      "lon": 37.7384529,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104978931",\n      "kind": "school",\n      "name": "Школа №1636, здание 9",\n      "lat": 55.6085976,\n      "lon": 37.7372412,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105078029",\n      "kind": "kindergarten",\n      "name": "Школа № 1636 \\"НИКА\\", дошкольное отделение",\n      "lat": 55.6056211,\n      "lon": 37.735715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105137093",\n      "kind": "school",\n      "name": "Школа № 433 им. И.И.Якушкина",\n      "lat": 55.7627029,\n      "lon": 37.7319887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105197703",\n      "kind": "kindergarten",\n      "name": "Детский сад №1843",\n      "lat": 55.6097727,\n      "lon": 37.7381162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105199600",\n      "kind": "school",\n      "name": "Православная гимназия Сабурово",\n      "lat": 55.6063226,\n      "lon": 37.7388712,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105200501",\n      "kind": "kindergarten",\n      "name": "Детский сад №69",\n      "lat": 55.6075388,\n      "lon": 37.7415555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105208194",\n      "kind": "kindergarten",\n      "name": "Детский сад №1277",\n      "lat": 55.6024077,\n      "lon": 37.7428534,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105208196",\n      "kind": "kindergarten",\n      "name": "Детский сад №1274",\n      "lat": 55.6093594,\n      "lon": 37.7425,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105208197",\n      "kind": "kindergarten",\n      "name": "Детский сад №1719",\n      "lat": 55.6010046,\n      "lon": 37.7431306,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105219753",\n      "kind": "kindergarten",\n      "name": "Детский сад № 840",\n      "lat": 55.6160749,\n      "lon": 37.7074916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105589602",\n      "kind": "school",\n      "name": "Школа №1005",\n      "lat": 55.8072707,\n      "lon": 37.4522987,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105704716",\n      "kind": "school",\n      "name": "Центр образования №1452 Богородский",\n      "lat": 55.8071051,\n      "lon": 37.708861,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105845924",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №2",\n      "lat": 55.5919539,\n      "lon": 37.6115347,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105897910",\n      "kind": "school",\n      "name": "Гимназия \\"Эллада\\"",\n      "lat": 55.643315,\n      "lon": 37.6714518,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105906586",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1768",\n      "lat": 55.6385246,\n      "lon": 37.7001222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105906603",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1762",\n      "lat": 55.6405388,\n      "lon": 37.6942483,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105906667",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1767",\n      "lat": 55.6390277,\n      "lon": 37.6988902,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105906681",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6399842,\n      "lon": 37.6969484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105917632",\n      "kind": "school",\n      "name": "Школа № 2017",\n      "lat": 55.6361605,\n      "lon": 37.7178026,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105917633",\n      "kind": "kindergarten",\n      "name": "Детский сад №2502",\n      "lat": 55.6370202,\n      "lon": 37.7174325,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105917639",\n      "kind": "school",\n      "name": "Наследник",\n      "lat": 55.6360298,\n      "lon": 37.7221514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105920324",\n      "kind": "kindergarten",\n      "name": "Детский сад №2470",\n      "lat": 55.6369044,\n      "lon": 37.7220333,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105933133",\n      "kind": "kindergarten",\n      "name": "Детский сад №2378",\n      "lat": 55.6327237,\n      "lon": 37.7360331,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105937693",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6336124,\n      "lon": 37.7464629,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105938066",\n      "kind": "kindergarten",\n      "name": "Детский сад №1612",\n      "lat": 55.6336305,\n      "lon": 37.7566248,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105938070",\n      "kind": "kindergarten",\n      "name": "Детский сад №1552",\n      "lat": 55.6350374,\n      "lon": 37.7553313,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105938072",\n      "kind": "kindergarten",\n      "name": "Детский сад №1554",\n      "lat": 55.635758,\n      "lon": 37.7547774,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105938073",\n      "kind": "kindergarten",\n      "name": "Детский сад №1577",\n      "lat": 55.634333,\n      "lon": 37.755985,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106182903",\n      "kind": "school",\n      "name": "Школа №1034",\n      "lat": 55.6329595,\n      "lon": 37.7581561,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106406165",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8072453,\n      "lon": 37.4541005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106445340",\n      "kind": "kindergarten",\n      "name": "Детский сад №928",\n      "lat": 55.8777319,\n      "lon": 37.5736031,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106445366",\n      "kind": "kindergarten",\n      "name": "Детский сад №708",\n      "lat": 55.8755833,\n      "lon": 37.5755957,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106535939",\n      "kind": "school",\n      "name": "Школа № 1034",\n      "lat": 55.6316908,\n      "lon": 37.7584732,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106535942",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2356",\n      "lat": 55.6317389,\n      "lon": 37.7608247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106549295",\n      "kind": "kindergarten",\n      "name": "Детский сад №2365",\n      "lat": 55.6335593,\n      "lon": 37.7683241,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106549299",\n      "kind": "school",\n      "name": "Гимназия №1587",\n      "lat": 55.6353301,\n      "lon": 37.7674789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106573641",\n      "kind": "school",\n      "name": "Начальная школа-детский сад № 1708",\n      "lat": 55.6758483,\n      "lon": 37.5238924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106573647",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад № 1708",\n      "lat": 55.6754142,\n      "lon": 37.5232704,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106576502",\n      "kind": "kindergarten",\n      "name": "Детский сад №1613",\n      "lat": 55.6395444,\n      "lon": 37.7688964,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106576512",\n      "kind": "kindergarten",\n      "name": "Детский сад №1576",\n      "lat": 55.6405957,\n      "lon": 37.7679666,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106577838",\n      "kind": "school",\n      "name": "Школа №998",\n      "lat": 55.640508,\n      "lon": 37.7663596,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106906402",\n      "kind": "kindergarten",\n      "name": "Детский сад №1664",\n      "lat": 55.8903418,\n      "lon": 37.5255754,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106918211",\n      "kind": "school",\n      "name": "Школа-интернат №5 \\"Преображенский кадетский корпус\\"",\n      "lat": 55.8253329,\n      "lon": 37.710076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106995473",\n      "kind": "kindergarten",\n      "name": "Центр образования №1858 (Детский сад Стрелец)",\n      "lat": 55.6294937,\n      "lon": 37.7957492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107101977",\n      "kind": "school",\n      "name": "Школа №999",\n      "lat": 55.639643,\n      "lon": 37.7640597,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107101982",\n      "kind": "school",\n      "name": "Начальная школа №992",\n      "lat": 55.6389934,\n      "lon": 37.7628296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107715061",\n      "kind": "kindergarten",\n      "name": "Детский сад №1498",\n      "lat": 55.5839058,\n      "lon": 37.6845133,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107715062",\n      "kind": "kindergarten",\n      "name": "Детский дом №18",\n      "lat": 55.5840872,\n      "lon": 37.6874257,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107728025",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Притяжение»",\n      "lat": 55.6393722,\n      "lon": 37.5985664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108093619",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №6)",\n      "lat": 55.8077904,\n      "lon": 37.7632804,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108201816",\n      "kind": "school",\n      "name": "Школа №1210",\n      "lat": 55.8003725,\n      "lon": 37.4567543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108202805",\n      "kind": "school",\n      "name": "Коррекционная спецшкола №442",\n      "lat": 55.8015355,\n      "lon": 37.8021316,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108295810",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа № 1454 \\"Тимирязевская\\"",\n      "lat": 55.834045,\n      "lon": 37.5609513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108315230",\n      "kind": "school",\n      "name": "Школа № 1210",\n      "lat": 55.7929162,\n      "lon": 37.4556715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108322388",\n      "kind": "school",\n      "name": "Специальная общеобразовательная школа №7",\n      "lat": 55.8012596,\n      "lon": 37.490871,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108323089",\n      "kind": "kindergarten",\n      "name": "Детский сад №265",\n      "lat": 55.7947611,\n      "lon": 37.4578265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108446906",\n      "kind": "kindergarten",\n      "name": "Центр развития ребёнка — Детский сад №2452",\n      "lat": 55.8084872,\n      "lon": 37.4526996,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108447327",\n      "kind": "school",\n      "name": "Центр Образования №1874 (Старшая школа)",\n      "lat": 55.8033777,\n      "lon": 37.4635543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108581514",\n      "kind": "kindergarten",\n      "name": "Детский сад №412",\n      "lat": 55.7960671,\n      "lon": 37.7723484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108680565",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1672",\n      "lat": 55.7041605,\n      "lon": 37.662774,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108715365",\n      "kind": "school",\n      "name": "Школа № 104",\n      "lat": 55.6841211,\n      "lon": 37.5646599,\n      "opening_hours": null\n    },\n    {\n      "id": "way/109308297",\n      "kind": "school",\n      "name": "Школа № 1232",\n      "lat": 55.743093,\n      "lon": 37.5387829,\n      "opening_hours": null\n    },\n    {\n      "id": "way/109562529",\n      "kind": "school",\n      "name": "Школа №1269",\n      "lat": 55.7550073,\n      "lon": 37.7868144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/109562535",\n      "kind": "kindergarten",\n      "name": "Детский сад №1910",\n      "lat": 55.7544601,\n      "lon": 37.7853713,\n      "opening_hours": null\n    },\n    {\n      "id": "way/109562536",\n      "kind": "school",\n      "name": "Школа №450",\n      "lat": 55.7539862,\n      "lon": 37.7863316,\n      "opening_hours": null\n    },\n    {\n      "id": "way/110298846",\n      "kind": "school",\n      "name": "Школа №74",\n      "lat": 55.727616,\n      "lon": 37.523302,\n      "opening_hours": null\n    },\n    {\n      "id": "way/110298855",\n      "kind": "kindergarten",\n      "name": "Детский сад № 805",\n      "lat": 55.7283238,\n      "lon": 37.5228253,\n      "opening_hours": null\n    },\n    {\n      "id": "way/110362226",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №1",\n      "lat": 55.5930469,\n      "lon": 37.608581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/110780000",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6803577,\n      "lon": 37.6008963,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111088772",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6802407,\n      "lon": 37.605767,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111088775",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6802494,\n      "lon": 37.6046395,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111090419",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6810181,\n      "lon": 37.6071642,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111136360",\n      "kind": "school",\n      "name": "Елизаветинская гимназия",\n      "lat": 55.7373544,\n      "lon": 37.6239743,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111538985",\n      "kind": "school",\n      "name": "Средняя школа № 536, отделение № 2",\n      "lat": 55.6495675,\n      "lon": 37.5998812,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111538986",\n      "kind": "school",\n      "name": "ГБПОУ Технический пожарно-спасательный колледж им. В.М. Максимчука",\n      "lat": 55.649517,\n      "lon": 37.6029872,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111543362",\n      "kind": "school",\n      "name": "Школа №1623",\n      "lat": 55.6275861,\n      "lon": 37.6137114,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111543367",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1711",\n      "lat": 55.6282777,\n      "lon": 37.6138966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111609675",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №7",\n      "lat": 55.6152411,\n      "lon": 37.5314247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111859126",\n      "kind": "kindergarten",\n      "name": "Детский сад №2580",\n      "lat": 55.7381186,\n      "lon": 37.6916792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111988381",\n      "kind": "school",\n      "name": "Школа № 56 им. академика В.А. Легасова. Корпус 2",\n      "lat": 55.7450747,\n      "lon": 37.544995,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112152372",\n      "kind": "kindergarten",\n      "name": "Школа №283, дошкольное отделение 9",\n      "lat": 55.88623,\n      "lon": 37.6706752,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112165344",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа № 67\\", отделение общего образования",\n      "lat": 55.7385691,\n      "lon": 37.5183621,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112390777",\n      "kind": "kindergarten",\n      "name": "Детский сад №6",\n      "lat": 55.6915852,\n      "lon": 37.5476242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112390793",\n      "kind": "school",\n      "name": "Начальная школа №2086",\n      "lat": 55.6917126,\n      "lon": 37.5448426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112390798",\n      "kind": "kindergarten",\n      "name": "Детский сад №1260",\n      "lat": 55.6924925,\n      "lon": 37.5454649,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112390800",\n      "kind": "school",\n      "name": "Лицей №1533",\n      "lat": 55.6922876,\n      "lon": 37.5438838,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112398203",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6949539,\n      "lon": 37.5498379,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112398251",\n      "kind": "kindergarten",\n      "name": "Детский сад №2042",\n      "lat": 55.6944455,\n      "lon": 37.5489711,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112427703",\n      "kind": "school",\n      "name": "Школа №1099",\n      "lat": 55.8743686,\n      "lon": 37.724078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112433183",\n      "kind": "school",\n      "name": "Лицей №1537",\n      "lat": 55.8742268,\n      "lon": 37.719222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112586642",\n      "kind": "kindergarten",\n      "name": "Детский сад №1473",\n      "lat": 55.6292784,\n      "lon": 37.7376793,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112586645",\n      "kind": "school",\n      "name": "Школа № 534 Корпус 1",\n      "lat": 55.6281801,\n      "lon": 37.7382903,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112587135",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6279543,\n      "lon": 37.7356076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112594626",\n      "kind": "kindergarten",\n      "name": "Детский сад №1745",\n      "lat": 55.6275318,\n      "lon": 37.7477033,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112594627",\n      "kind": "school",\n      "name": "Школа № 534 Корпус 3",\n      "lat": 55.6285027,\n      "lon": 37.7474005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112594628",\n      "kind": "kindergarten",\n      "name": "Детский сад №1752",\n      "lat": 55.6274827,\n      "lon": 37.7449991,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610213",\n      "kind": "school",\n      "name": "Специальная (коррекционная) школа №991",\n      "lat": 55.6248291,\n      "lon": 37.7498145,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610214",\n      "kind": "school",\n      "name": "Гимназия № 1569 \\"Созвездие\\"",\n      "lat": 55.6237287,\n      "lon": 37.7478421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610216",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР №1271",\n      "lat": 55.6232964,\n      "lon": 37.7462482,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610217",\n      "kind": "school",\n      "name": "Школа № 1552, здание №2",\n      "lat": 55.6250583,\n      "lon": 37.7448679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610218",\n      "kind": "kindergarten",\n      "name": "Детский дом №9",\n      "lat": 55.6255205,\n      "lon": 37.7456952,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610219",\n      "kind": "kindergarten",\n      "name": "Школа № 1552, здание №7",\n      "lat": 55.6248228,\n      "lon": 37.7463153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112628689",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6255601,\n      "lon": 37.7394427,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112628690",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР №1271",\n      "lat": 55.6245258,\n      "lon": 37.7382005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112688156",\n      "kind": "school",\n      "name": "Вешняковская школа. Школьное отделение Ж7",\n      "lat": 55.7332515,\n      "lon": 37.8269055,\n      "opening_hours": null\n    },\n    {\n      "id": "way/113558368",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Маграритка\\"",\n      "lat": 55.7888997,\n      "lon": 37.4930419,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114421932",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2176",\n      "lat": 55.7967979,\n      "lon": 37.484242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114421935",\n      "kind": "kindergarten",\n      "name": "Детский сад №1106",\n      "lat": 55.8021799,\n      "lon": 37.4890696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114421948",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"Интеграция\\"",\n      "lat": 55.7900674,\n      "lon": 37.477882,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114533462",\n      "kind": "school",\n      "name": "Школа-лицей Возрождение",\n      "lat": 55.7947491,\n      "lon": 37.4596483,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114533471",\n      "kind": "kindergarten",\n      "name": "Детский сад №1679",\n      "lat": 55.8032203,\n      "lon": 37.4683426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114540677",\n      "kind": "kindergarten",\n      "name": "Центр образования №1874 (детский сад)",\n      "lat": 55.8068079,\n      "lon": 37.4610261,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114553687",\n      "kind": "school",\n      "name": "Школа № 1454 «Тимирязевская»",\n      "lat": 55.8240813,\n      "lon": 37.5676768,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114600871",\n      "kind": "kindergarten",\n      "name": "ДОП \\"Дмитровская 29\\"",\n      "lat": 55.8278722,\n      "lon": 37.5702414,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114671844",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1397",\n      "lat": 55.7980618,\n      "lon": 37.4618443,\n      "opening_hours": null\n    },\n    {\n      "id": "way/115613768",\n      "kind": "kindergarten",\n      "name": "Детский сад №768",\n      "lat": 55.6508747,\n      "lon": 37.5996555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/115646586",\n      "kind": "school",\n      "name": "Кадетская школа № 1784 имени генерала армии В.А. Матросова",\n      "lat": 55.7848327,\n      "lon": 37.5744311,\n      "opening_hours": null\n    },\n    {\n      "id": "way/115707848",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7733715,\n      "lon": 37.5843441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/115976870",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7910999,\n      "lon": 37.5000854,\n      "opening_hours": null\n    },\n    {\n      "id": "way/116299193",\n      "kind": "kindergarten",\n      "name": "Детский сад №1462",\n      "lat": 55.6665749,\n      "lon": 37.5893849,\n      "opening_hours": null\n    },\n    {\n      "id": "way/116299212",\n      "kind": "school",\n      "name": "Экспериментальная школа (центр образования МЭШ)",\n      "lat": 55.6545965,\n      "lon": 37.5642906,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117196191",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8724675,\n      "lon": 37.7162291,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117348087",\n      "kind": "school",\n      "name": "Школа №1598. Здание №4",\n      "lat": 55.8281604,\n      "lon": 37.8214628,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117348099",\n      "kind": "school",\n      "name": "Школа №1598. Здание №5",\n      "lat": 55.8280996,\n      "lon": 37.8180492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117348110",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №15",\n      "lat": 55.8291357,\n      "lon": 37.8219177,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117368268",\n      "kind": "kindergarten",\n      "name": "Школа №1499 (дошкольное отделение)",\n      "lat": 55.8360441,\n      "lon": 37.6625174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117763674",\n      "kind": "school",\n      "name": "Школа №285 имени В. А. Молодцова",\n      "lat": 55.8769452,\n      "lon": 37.6244087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117904490",\n      "kind": "school",\n      "name": "Школа № 656 им. А. С. Макаренко",\n      "lat": 55.8676155,\n      "lon": 37.5550262,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118013507",\n      "kind": "school",\n      "name": "Школа № 814. Учебный корпус № 1",\n      "lat": 55.7068301,\n      "lon": 37.4742622,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118034079",\n      "kind": "kindergarten",\n      "name": "Детский сад",\n      "lat": 55.8680186,\n      "lon": 37.48728,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118037513",\n      "kind": "school",\n      "name": "Школа №815",\n      "lat": 55.708096,\n      "lon": 37.4755349,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118037514",\n      "kind": "school",\n      "name": "ГБОУ школа № 814 Учебный корпус 2",\n      "lat": 55.7079276,\n      "lon": 37.4757859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118055399",\n      "kind": "kindergarten",\n      "name": "Школа № 158. Дошкольное отделение № 3",\n      "lat": 55.8712906,\n      "lon": 37.4665694,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118193796",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №177",\n      "lat": 55.6729938,\n      "lon": 37.7233517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118271893",\n      "kind": "kindergarten",\n      "name": "Центр развития ребенка. Детский сад №2558",\n      "lat": 55.6835407,\n      "lon": 37.5360968,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118555415",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7184905,\n      "lon": 37.5799125,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118555424",\n      "kind": "kindergarten",\n      "name": "Детский сад №2107",\n      "lat": 55.7187486,\n      "lon": 37.5737354,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118570499",\n      "kind": "school",\n      "name": "Центр духовного развития молодежи",\n      "lat": 55.7119819,\n      "lon": 37.6318634,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118641707",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7218279,\n      "lon": 37.578717,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118648732",\n      "kind": "kindergarten",\n      "name": "ДОУ №1975",\n      "lat": 55.7764558,\n      "lon": 37.7102975,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118738013",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №05/3ш",\n      "lat": 55.6205856,\n      "lon": 37.4941386,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118792575",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №04",\n      "lat": 55.6202798,\n      "lon": 37.4924015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119072133",\n      "kind": "school",\n      "name": "Школа №352",\n      "lat": 55.8319484,\n      "lon": 37.6639127,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119072246",\n      "kind": "kindergarten",\n      "name": "Детский сад №1247",\n      "lat": 55.8303266,\n      "lon": 37.661106,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119335328",\n      "kind": "school",\n      "name": "Школа №603",\n      "lat": 55.8250739,\n      "lon": 37.5238508,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119917726",\n      "kind": "school",\n      "name": "Школа №1515",\n      "lat": 55.7787929,\n      "lon": 37.4742991,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119917732",\n      "kind": "kindergarten",\n      "name": "Детский сад №1021",\n      "lat": 55.777652,\n      "lon": 37.4737507,\n      "opening_hours": null\n    },\n    {\n      "id": "way/120831644",\n      "kind": "kindergarten",\n      "name": "Детский сад №564",\n      "lat": 55.7831392,\n      "lon": 37.4596205,\n      "opening_hours": null\n    },\n    {\n      "id": "way/122231613",\n      "kind": "school",\n      "name": "Школа №947, Кадетский корпус государственной дипломатической службы им. И.А. Рогачева",\n      "lat": 55.5841777,\n      "lon": 37.6651589,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123099143",\n      "kind": "kindergarten",\n      "name": "Детский сад 1783",\n      "lat": 55.6748188,\n      "lon": 37.5951549,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123099144",\n      "kind": "kindergarten",\n      "name": "Детский сад 1835",\n      "lat": 55.6768991,\n      "lon": 37.5954003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123099148",\n      "kind": "kindergarten",\n      "name": "Детский сад 577",\n      "lat": 55.6761518,\n      "lon": 37.5967154,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123186332",\n      "kind": "kindergarten",\n      "name": "Детский сад №1",\n      "lat": 55.8844592,\n      "lon": 37.4893802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123186335",\n      "kind": "kindergarten",\n      "name": "Детский сад № 3",\n      "lat": 55.8834557,\n      "lon": 37.4936137,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123226397",\n      "kind": "school",\n      "name": "Интернат №24",\n      "lat": 55.6528795,\n      "lon": 37.5907662,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124188649",\n      "kind": "kindergarten",\n      "name": "Детский сад №1236 Солнышко",\n      "lat": 55.8082842,\n      "lon": 37.5639872,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124792732",\n      "kind": "school",\n      "name": "Частная школа детский сад «Лотос»",\n      "lat": 55.8929323,\n      "lon": 37.5899646,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124797767",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад №1817",\n      "lat": 55.890783,\n      "lon": 37.5834239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124797790",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 6",\n      "lat": 55.8925464,\n      "lon": 37.5788197,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124797791",\n      "kind": "school",\n      "name": "ГБОУ Школа № 1416 корпус № 2",\n      "lat": 55.8925641,\n      "lon": 37.5810127,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125070311",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Счастливое детство\\"",\n      "lat": 55.8045714,\n      "lon": 37.5720934,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "way/125105442",\n      "kind": "school",\n      "name": "ГБОУ Школа №1797 \\"Богородская\\"",\n      "lat": 55.8119017,\n      "lon": 37.7152192,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125105473",\n      "kind": "kindergarten",\n      "name": "Детский сад №1809",\n      "lat": 55.8097851,\n      "lon": 37.7190151,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125105474",\n      "kind": "kindergarten",\n      "name": "Детский сад №2065",\n      "lat": 55.808692,\n      "lon": 37.7189908,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200345",\n      "kind": "kindergarten",\n      "name": "Детский сад Светлячок",\n      "lat": 55.8969798,\n      "lon": 37.5698469,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200349",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №3)",\n      "lat": 55.896814,\n      "lon": 37.5687375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200362",\n      "kind": "school",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №5)",\n      "lat": 55.8925968,\n      "lon": 37.5711057,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200363",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №4)",\n      "lat": 55.897039,\n      "lon": 37.5664383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200365",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №6)",\n      "lat": 55.8939503,\n      "lon": 37.5703869,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200366",\n      "kind": "school",\n      "name": "Знак",\n      "lat": 55.8940178,\n      "lon": 37.5757154,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200368",\n      "kind": "school",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №1)",\n      "lat": 55.8950656,\n      "lon": 37.570219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200370",\n      "kind": "school",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №2)",\n      "lat": 55.8971171,\n      "lon": 37.5715637,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204825",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 8",\n      "lat": 55.8925835,\n      "lon": 37.5843378,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204840",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 5",\n      "lat": 55.8933469,\n      "lon": 37.5843465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204842",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 4",\n      "lat": 55.8962966,\n      "lon": 37.5840013,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204843",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 7",\n      "lat": 55.8953805,\n      "lon": 37.5782693,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204844",\n      "kind": "school",\n      "name": "ГБОУ Школа № 1416 корпус № 1",\n      "lat": 55.8953733,\n      "lon": 37.5814333,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204845",\n      "kind": "school",\n      "name": "Школа №1416",\n      "lat": 55.8934769,\n      "lon": 37.5813015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125849340",\n      "kind": "school",\n      "name": "Гимназия №1540",\n      "lat": 55.7853154,\n      "lon": 37.5969853,\n      "opening_hours": null\n    },\n    {\n      "id": "way/126326179",\n      "kind": "school",\n      "name": "Школа № 108",\n      "lat": 55.6761274,\n      "lon": 37.5510441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/126326596",\n      "kind": "school",\n      "name": "Школа № 58",\n      "lat": 55.6783805,\n      "lon": 37.5393496,\n      "opening_hours": null\n    },\n    {\n      "id": "way/127417192",\n      "kind": "kindergarten",\n      "name": "Школа № 962. Дошкольное отделение",\n      "lat": 55.8645895,\n      "lon": 37.5961427,\n      "opening_hours": null\n    },\n    {\n      "id": "way/127868208",\n      "kind": "school",\n      "name": "Школа \\"Ника\\"",\n      "lat": 55.6543635,\n      "lon": 37.595068,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128056567",\n      "kind": "kindergarten",\n      "name": "Детский сад №937",\n      "lat": 55.8896508,\n      "lon": 37.6130377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128056568",\n      "kind": "school",\n      "name": "Школа №180",\n      "lat": 55.8896209,\n      "lon": 37.6106233,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128111685",\n      "kind": "kindergarten",\n      "name": "Детский сад № 18 \\"Кораблик\\"",\n      "lat": 55.5922283,\n      "lon": 37.7512445,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128116005",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.5891372,\n      "lon": 37.7629338,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128116767",\n      "kind": "kindergarten",\n      "name": "Детский сад № 45 \\"Сказка\\"",\n      "lat": 55.5898354,\n      "lon": 37.7565267,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128263509",\n      "kind": "school",\n      "name": "Школа №225",\n      "lat": 55.824626,\n      "lon": 37.5153224,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128531064",\n      "kind": "school",\n      "name": "Центр образования № 1811 «Измайлово» – начальная школа",\n      "lat": 55.7960544,\n      "lon": 37.8088873,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128531081",\n      "kind": "kindergarten",\n      "name": "Центр образования №1811 Измайлово (детский сад)",\n      "lat": 55.7956776,\n      "lon": 37.8068841,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128594640",\n      "kind": "school",\n      "name": "Школа № 1383. Корпус 2",\n      "lat": 55.8623681,\n      "lon": 37.5657504,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754875",\n      "kind": "school",\n      "name": "Школа №2200 корпус 3",\n      "lat": 55.8060903,\n      "lon": 37.7967024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754879",\n      "kind": "school",\n      "name": "Школа №619",\n      "lat": 55.8077222,\n      "lon": 37.7885859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754887",\n      "kind": "kindergarten",\n      "name": "Детский сад №211",\n      "lat": 55.8052689,\n      "lon": 37.7936768,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754894",\n      "kind": "kindergarten",\n      "name": "Детский сад №1457",\n      "lat": 55.8045424,\n      "lon": 37.7936956,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754895",\n      "kind": "school",\n      "name": "Центр развития творчества детей и юношества им. А.В. Косарева",\n      "lat": 55.8059688,\n      "lon": 37.7892282,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129239209",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 6 (дошкольное отделение)",\n      "lat": 55.8083601,\n      "lon": 37.7963659,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129239218",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 5 (дошкольное отделение)",\n      "lat": 55.8082133,\n      "lon": 37.7913644,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129307720",\n      "kind": "kindergarten",\n      "name": "Детский сад № 841",\n      "lat": 55.7251804,\n      "lon": 37.6321433,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129318472",\n      "kind": "kindergarten",\n      "name": "Детский сад №1828",\n      "lat": 55.7335274,\n      "lon": 37.6749952,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129357954",\n      "kind": "school",\n      "name": "Гимназия №1562 им. Артема Боровика",\n      "lat": 55.6654393,\n      "lon": 37.7507745,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129405498",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6682715,\n      "lon": 37.6510233,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129408334",\n      "kind": "kindergarten",\n      "name": "ДОУ №144",\n      "lat": 55.6773572,\n      "lon": 37.6280809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129408541",\n      "kind": "kindergarten",\n      "name": "Детский сад №2161",\n      "lat": 55.7239539,\n      "lon": 37.6609878,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129408542",\n      "kind": "kindergarten",\n      "name": "Школа-сад № 494 им. Героя Российской Федерации А. Н. Рожкова",\n      "lat": 55.7255936,\n      "lon": 37.6607453,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129413586",\n      "kind": "kindergarten",\n      "name": "Детский сад №1186",\n      "lat": 55.7313266,\n      "lon": 37.6758185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129416914",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7396872,\n      "lon": 37.6780399,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129474366",\n      "kind": "school",\n      "name": "Интернат №72",\n      "lat": 55.6656442,\n      "lon": 37.6127148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129475181",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6497603,\n      "lon": 37.6040084,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129475373",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6639979,\n      "lon": 37.621583,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129529144",\n      "kind": "school",\n      "name": "Школа №473",\n      "lat": 55.6389719,\n      "lon": 37.7976845,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129531218",\n      "kind": "kindergarten",\n      "name": "Центр образования № 1858 - центр дошкольного дополнительного образования",\n      "lat": 55.6391644,\n      "lon": 37.7938312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129584044",\n      "kind": "school",\n      "name": "Школа №462",\n      "lat": 55.715703,\n      "lon": 37.673287,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129585077",\n      "kind": "school",\n      "name": "Вечерняя школа № 66 ЮВАО",\n      "lat": 55.7289623,\n      "lon": 37.6627126,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129624464",\n      "kind": "kindergarten",\n      "name": "Детский сад №962",\n      "lat": 55.867025,\n      "lon": 37.7012276,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129691146",\n      "kind": "school",\n      "name": "Школа-интернат №9",\n      "lat": 55.695253,\n      "lon": 37.7522334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129704196",\n      "kind": "kindergarten",\n      "name": "Школа № 1331 Молодая Иверия. Дошкольное отделение",\n      "lat": 55.7481725,\n      "lon": 37.5944135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129704229",\n      "kind": "kindergarten",\n      "name": "ЦВР \\"На Сумском\\" (филиал)",\n      "lat": 55.677459,\n      "lon": 37.6295932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129747822",\n      "kind": "school",\n      "name": "Школа №1321 \\"Ковчег\\"",\n      "lat": 55.7445935,\n      "lon": 37.7192621,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129749845",\n      "kind": "kindergarten",\n      "name": "Детский сад №1946",\n      "lat": 55.76828,\n      "lon": 37.7127492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129760307",\n      "kind": "school",\n      "name": "Учебный центр \\"МИР\\"",\n      "lat": 55.7591469,\n      "lon": 37.7002674,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129760407",\n      "kind": "kindergarten",\n      "name": "Ясли-Сад №1",\n      "lat": 55.7963767,\n      "lon": 37.4621816,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129831577",\n      "kind": "kindergarten",\n      "name": "Детский сад №2651",\n      "lat": 55.7488992,\n      "lon": 37.8127391,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130011189",\n      "kind": "kindergarten",\n      "name": "Школа № 2030. Дошкольное отделение",\n      "lat": 55.7649536,\n      "lon": 37.5590889,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130065838",\n      "kind": "school",\n      "name": "Школа №2048",\n      "lat": 55.7593184,\n      "lon": 37.5177825,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130105062",\n      "kind": "kindergarten",\n      "name": "Детский сад №354",\n      "lat": 55.7565427,\n      "lon": 37.7740169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130109360",\n      "kind": "school",\n      "name": "Школа № 2115. Корпус школьного образования № 1",\n      "lat": 55.6724679,\n      "lon": 37.5667449,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130278483",\n      "kind": "school",\n      "name": "Школа №1216",\n      "lat": 55.7350969,\n      "lon": 37.6725978,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130512523",\n      "kind": "kindergarten",\n      "name": "Школа №368 (Здание №5)",\n      "lat": 55.8213288,\n      "lon": 37.818129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130537875",\n      "kind": "kindergarten",\n      "name": "ДОУ №53",\n      "lat": 55.7729035,\n      "lon": 37.7036471,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130630559",\n      "kind": "school",\n      "name": "Гимназия № 45 имени Л. И. Мильграма Общеобразовательное отделение Новочеремушкинская 11а",\n      "lat": 55.6861165,\n      "lon": 37.5866625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130643279",\n      "kind": "kindergarten",\n      "name": "Детский сад №793",\n      "lat": 55.7979832,\n      "lon": 37.8035868,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130688441",\n      "kind": "school",\n      "name": "Школа №1515",\n      "lat": 55.7803156,\n      "lon": 37.4769756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130688456",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7821353,\n      "lon": 37.4746938,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130753111",\n      "kind": "school",\n      "name": "Школа №1381 ШО 3",\n      "lat": 55.8762716,\n      "lon": 37.6971147,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130753112",\n      "kind": "kindergarten",\n      "name": "Детский сад №2379",\n      "lat": 55.8778973,\n      "lon": 37.697789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130819428",\n      "kind": "school",\n      "name": "Детский сад №210",\n      "lat": 55.7768112,\n      "lon": 37.7261106,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130998381",\n      "kind": "school",\n      "name": "Школа №313",\n      "lat": 55.8792135,\n      "lon": 37.7038627,\n      "opening_hours": null\n    },\n    {\n      "id": "way/131000314",\n      "kind": "school",\n      "name": "Московский колледж градостроительства и предпринимательства",\n      "lat": 55.8841624,\n      "lon": 37.7136584,\n      "opening_hours": null\n    },\n    {\n      "id": "way/131483181",\n      "kind": "kindergarten",\n      "name": "Школа №1284",\n      "lat": 55.769578,\n      "lon": 37.6375994,\n      "opening_hours": null\n    },\n    {\n      "id": "way/131663201",\n      "kind": "kindergarten",\n      "name": "Детский сад №1958",\n      "lat": 55.8760183,\n      "lon": 37.6994323,\n      "opening_hours": null\n    },\n    {\n      "id": "way/131751466",\n      "kind": "school",\n      "name": "Свято-Владимирская школа",\n      "lat": 55.7536688,\n      "lon": 37.6406339,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132083765",\n      "kind": "kindergarten",\n      "name": "Детский сад №1518",\n      "lat": 55.7463821,\n      "lon": 37.7852695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132084593",\n      "kind": "kindergarten",\n      "name": "Детский сад №1617",\n      "lat": 55.7475809,\n      "lon": 37.7882575,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132084698",\n      "kind": "kindergarten",\n      "name": "Детский сад №527",\n      "lat": 55.7487245,\n      "lon": 37.7944755,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132117555",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №4 \\"Белоснежка\\"",\n      "lat": 55.8055241,\n      "lon": 37.8247901,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132725687",\n      "kind": "school",\n      "name": "Школа №245",\n      "lat": 55.8911685,\n      "lon": 37.6130868,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132832339",\n      "kind": "kindergarten",\n      "name": "Детский сад №1108",\n      "lat": 55.8929269,\n      "lon": 37.6135705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133408372",\n      "kind": "school",\n      "name": "Гимназия №1551 корпус №3 (начальная школа)",\n      "lat": 55.8460173,\n      "lon": 37.4513321,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133408558",\n      "kind": "school",\n      "name": "Гимназия №1551 корпус №2 (основное общее)",\n      "lat": 55.8455084,\n      "lon": 37.449048,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133487277",\n      "kind": "kindergarten",\n      "name": "Школа №1056 (дошкольное отделение №4)",\n      "lat": 55.8479132,\n      "lon": 37.4480763,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133487311",\n      "kind": "kindergarten",\n      "name": "Школа №1056 (дошкольное отделение №5)",\n      "lat": 55.8485331,\n      "lon": 37.4484511,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133487388",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №4 (дошкольное отделение)",\n      "lat": 55.8486631,\n      "lon": 37.4497719,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133487448",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №5 (дошкольное отделение)",\n      "lat": 55.848926,\n      "lon": 37.4519575,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133738328",\n      "kind": "kindergarten",\n      "name": "Детский сад №140",\n      "lat": 55.7851111,\n      "lon": 37.7379828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/134901767",\n      "kind": "school",\n      "name": "АНО \\"Школа \\"Премьер\\"",\n      "lat": 55.6196757,\n      "lon": 37.7256132,\n      "opening_hours": null\n    },\n    {\n      "id": "way/134901780",\n      "kind": "kindergarten",\n      "name": "Детский сад №1134",\n      "lat": 55.614081,\n      "lon": 37.7265053,\n      "opening_hours": null\n    },\n    {\n      "id": "way/134901781",\n      "kind": "kindergarten",\n      "name": "Детский сад №939",\n      "lat": 55.6143531,\n      "lon": 37.727616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/134901782",\n      "kind": "school",\n      "name": "Школа №939",\n      "lat": 55.6134721,\n      "lon": 37.7236603,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135398211",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 435",\n      "lat": 55.8956901,\n      "lon": 37.5327341,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135398212",\n      "kind": "school",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 668",\n      "lat": 55.8962697,\n      "lon": 37.5323568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135640537",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7467461,\n      "lon": 37.5641221,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135742368",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение школы №712",\n      "lat": 55.7266731,\n      "lon": 37.4574602,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135854614",\n      "kind": "kindergarten",\n      "name": "Детский сад №1650",\n      "lat": 55.8883133,\n      "lon": 37.6100126,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135854615",\n      "kind": "kindergarten",\n      "name": "Детский сад №1245",\n      "lat": 55.8868506,\n      "lon": 37.6066122,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135854616",\n      "kind": "school",\n      "name": "Школа №960",\n      "lat": 55.8884301,\n      "lon": 37.6074706,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135854617",\n      "kind": "kindergarten",\n      "name": "Детский сад №909",\n      "lat": 55.8870355,\n      "lon": 37.6048788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135985865",\n      "kind": "school",\n      "name": "ГБОУ Марьинская школа № 1566 корпус 6",\n      "lat": 55.6483386,\n      "lon": 37.7564702,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135993453",\n      "kind": "kindergarten",\n      "name": "ГБОУ Марьинская школа № 1566 ДО 2 Теремок",\n      "lat": 55.6487485,\n      "lon": 37.7574976,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136179391",\n      "kind": "kindergarten",\n      "name": "Прогимназия №1855 (начальная школа – детский сад)",\n      "lat": 55.8942577,\n      "lon": 37.5984969,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136341087",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2323",\n      "lat": 55.64554,\n      "lon": 37.7491263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136473981",\n      "kind": "kindergarten",\n      "name": "Детский сад №2494",\n      "lat": 55.6580937,\n      "lon": 37.7743893,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136473982",\n      "kind": "kindergarten",\n      "name": "Детский сад №2423",\n      "lat": 55.6582758,\n      "lon": 37.7725845,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136473983",\n      "kind": "kindergarten",\n      "name": "Детский сад №2427",\n      "lat": 55.6588218,\n      "lon": 37.774535,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136752933",\n      "kind": "school",\n      "name": "Школа № 1357 «На Братиславской». Здание №11",\n      "lat": 55.6663031,\n      "lon": 37.7577239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136752938",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №12",\n      "lat": 55.664775,\n      "lon": 37.7560194,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136752941",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6602526,\n      "lon": 37.7454029,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136752947",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6615265,\n      "lon": 37.7503677,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884279",\n      "kind": "school",\n      "name": "Школа №1412",\n      "lat": 55.8910098,\n      "lon": 37.6013674,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884281",\n      "kind": "kindergarten",\n      "name": "Детский сад №1113",\n      "lat": 55.8894334,\n      "lon": 37.6014801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884282",\n      "kind": "kindergarten",\n      "name": "Детский сад №2153",\n      "lat": 55.8930637,\n      "lon": 37.6076579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884283",\n      "kind": "kindergarten",\n      "name": "Детский сад №1014",\n      "lat": 55.8896192,\n      "lon": 37.5995616,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/136884284",\n      "kind": "kindergarten",\n      "name": "Детский сад №1115",\n      "lat": 55.8960358,\n      "lon": 37.605869,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884285",\n      "kind": "school",\n      "name": "Школа №255 (бывшая №195)",\n      "lat": 55.8950083,\n      "lon": 37.6056973,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884286",\n      "kind": "kindergarten",\n      "name": "Детский сад №1640",\n      "lat": 55.895026,\n      "lon": 37.6093112,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884287",\n      "kind": "school",\n      "name": "Школа №1412",\n      "lat": 55.8932066,\n      "lon": 37.6040258,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537398",\n      "kind": "kindergarten",\n      "name": "Детский сад №1040",\n      "lat": 55.6204674,\n      "lon": 37.7317515,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537400",\n      "kind": "kindergarten",\n      "name": "Центр образования № 548",\n      "lat": 55.6202134,\n      "lon": 37.7332599,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537401",\n      "kind": "school",\n      "name": "Школа №534 корпус 2",\n      "lat": 55.6237996,\n      "lon": 37.7344389,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537404",\n      "kind": "kindergarten",\n      "name": "Детский сад №1167",\n      "lat": 55.623089,\n      "lon": 37.7383628,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537405",\n      "kind": "kindergarten",\n      "name": "Детский сад №1398",\n      "lat": 55.6228081,\n      "lon": 37.7336169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139617929",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7711207,\n      "lon": 37.6765969,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139853868",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №17",\n      "lat": 55.82896,\n      "lon": 37.8145966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042486",\n      "kind": "school",\n      "name": "Школа № 2116, школьное отделение № 2",\n      "lat": 55.6179107,\n      "lon": 37.7512138,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042487",\n      "kind": "school",\n      "name": "Школа № 1552, здание №1",\n      "lat": 55.6197632,\n      "lon": 37.7470344,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042488",\n      "kind": "school",\n      "name": "Школа № 2116, Классика",\n      "lat": 55.6203789,\n      "lon": 37.749712,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042489",\n      "kind": "school",\n      "name": "Школа № 1552, здание №3",\n      "lat": 55.6180298,\n      "lon": 37.7468973,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042490",\n      "kind": "kindergarten",\n      "name": "Школа № 2116, дошкольное отделение № 9",\n      "lat": 55.6162099,\n      "lon": 37.7508286,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042491",\n      "kind": "kindergarten",\n      "name": "Школа № 1552, здание №5",\n      "lat": 55.621001,\n      "lon": 37.7470633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042492",\n      "kind": "kindergarten",\n      "name": "Школа № 1552, здание №6",\n      "lat": 55.6210318,\n      "lon": 37.7491874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042493",\n      "kind": "school",\n      "name": "Школа № 1552, здание №4",\n      "lat": 55.6167638,\n      "lon": 37.747139,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042494",\n      "kind": "kindergarten",\n      "name": "Школа № 2116, дошкольное отделение № 4",\n      "lat": 55.6155153,\n      "lon": 37.7505724,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140563393",\n      "kind": "school",\n      "name": "Вечерняя сменная общеобразовательная школа №17",\n      "lat": 55.8077002,\n      "lon": 37.6433932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140563396",\n      "kind": "school",\n      "name": "Школа №308",\n      "lat": 55.8072416,\n      "lon": 37.6421924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140563398",\n      "kind": "school",\n      "name": "Специальная школа-интернат №1",\n      "lat": 55.8073825,\n      "lon": 37.6441465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141040653",\n      "kind": "school",\n      "name": "Школа № 524",\n      "lat": 55.6462674,\n      "lon": 37.584172,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141067210",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6147101,\n      "lon": 37.6139895,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643616",\n      "kind": "school",\n      "name": "Школа №318",\n      "lat": 55.8145672,\n      "lon": 37.723657,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643619",\n      "kind": "kindergarten",\n      "name": "Детский сад №534",\n      "lat": 55.8142259,\n      "lon": 37.7245145,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643622",\n      "kind": "kindergarten",\n      "name": "Детский сад №1475",\n      "lat": 55.8148942,\n      "lon": 37.7302036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643633",\n      "kind": "kindergarten",\n      "name": "Детский сад №564",\n      "lat": 55.8156609,\n      "lon": 37.726792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643636",\n      "kind": "school",\n      "name": "Школа №362",\n      "lat": 55.8138368,\n      "lon": 37.7286248,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643639",\n      "kind": "kindergarten",\n      "name": "Детский сад №535",\n      "lat": 55.813324,\n      "lon": 37.725322,\n      "opening_hours": null\n    },\n    {\n      "id": "way/143013486",\n      "kind": "school",\n      "name": "Школа №222 (Корпус 2)",\n      "lat": 55.8862611,\n      "lon": 37.5610346,\n      "opening_hours": null\n    },\n    {\n      "id": "way/143142432",\n      "kind": "school",\n      "name": "Школа №192",\n      "lat": 55.706648,\n      "lon": 37.574794,\n      "opening_hours": null\n    },\n    {\n      "id": "way/143422559",\n      "kind": "school",\n      "name": "Центр образования №118",\n      "lat": 55.6834934,\n      "lon": 37.5345025,\n      "opening_hours": null\n    },\n    {\n      "id": "way/143422563",\n      "kind": "kindergarten",\n      "name": "Детский сад №1225",\n      "lat": 55.6842738,\n      "lon": 37.5365974,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144131944",\n      "kind": "school",\n      "name": "Школа №847",\n      "lat": 55.882246,\n      "lon": 37.5222784,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144131946",\n      "kind": "school",\n      "name": "Школа №683",\n      "lat": 55.8895383,\n      "lon": 37.5245824,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144138168",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1485",\n      "lat": 55.8938578,\n      "lon": 37.5273598,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144138172",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 656",\n      "lat": 55.89282,\n      "lon": 37.5279553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144138173",\n      "kind": "kindergarten",\n      "name": "Детский сад №1487",\n      "lat": 55.8889314,\n      "lon": 37.5235105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144284830",\n      "kind": "kindergarten",\n      "name": "Детский сад №929",\n      "lat": 55.8838237,\n      "lon": 37.5250693,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144284837",\n      "kind": "kindergarten",\n      "name": "Детский сад №2246",\n      "lat": 55.8807658,\n      "lon": 37.5233492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144647442",\n      "kind": "kindergarten",\n      "name": "Гимназия №1517",\n      "lat": 55.7731176,\n      "lon": 37.471677,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144884018",\n      "kind": "kindergarten",\n      "name": "Школа 1362 дошкольное отделение",\n      "lat": 55.7746688,\n      "lon": 37.733034,\n      "opening_hours": null\n    },\n    {\n      "id": "way/145111581",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1176",\n      "lat": 55.7684204,\n      "lon": 37.7313753,\n      "opening_hours": null\n    },\n    {\n      "id": "way/145111582",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2090",\n      "lat": 55.7688224,\n      "lon": 37.7346461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/145111583",\n      "kind": "kindergarten",\n      "name": "Детский сад №2566",\n      "lat": 55.769186,\n      "lon": 37.7306148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146398902",\n      "kind": "school",\n      "name": "Эрудит",\n      "lat": 55.80417,\n      "lon": 37.4599662,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146603847",\n      "kind": "kindergarten",\n      "name": "Детский сад №907",\n      "lat": 55.6248061,\n      "lon": 37.607711,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146603937",\n      "kind": "kindergarten",\n      "name": "Детский сад №698",\n      "lat": 55.6273358,\n      "lon": 37.6001565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146603939",\n      "kind": "kindergarten",\n      "name": "Детский сад №705",\n      "lat": 55.6256899,\n      "lon": 37.6014653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146696317",\n      "kind": "kindergarten",\n      "name": "Детский сад №212",\n      "lat": 55.7836253,\n      "lon": 37.4516603,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146767851",\n      "kind": "school",\n      "name": "Школа № 2098 им. Героя Советского Союза Л.М. Доватора",\n      "lat": 55.8789895,\n      "lon": 37.5141932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/147003347",\n      "kind": "school",\n      "name": "Школа № 446",\n      "lat": 55.7838189,\n      "lon": 37.7059941,\n      "opening_hours": null\n    },\n    {\n      "id": "way/147043593",\n      "kind": "kindergarten",\n      "name": "Детский сад №151",\n      "lat": 55.6604771,\n      "lon": 37.5998884,\n      "opening_hours": null\n    },\n    {\n      "id": "way/147043594",\n      "kind": "school",\n      "name": "Школа № 516",\n      "lat": 55.6600948,\n      "lon": 37.597322,\n      "opening_hours": null\n    },\n    {\n      "id": "way/147998246",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2112",\n      "lat": 55.7531211,\n      "lon": 37.5373254,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148175692",\n      "kind": "kindergarten",\n      "name": "Детский сад №2247",\n      "lat": 55.8653629,\n      "lon": 37.5689045,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148175699",\n      "kind": "kindergarten",\n      "name": "Детский сад №1823",\n      "lat": 55.8678313,\n      "lon": 37.5668569,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148175701",\n      "kind": "school",\n      "name": "Школа № 1794",\n      "lat": 55.8727403,\n      "lon": 37.5664443,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148175913",\n      "kind": "kindergarten",\n      "name": "Детский сад №1521",\n      "lat": 55.8078002,\n      "lon": 37.7082547,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148190333",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa01704",\n      "lat": 55.6758614,\n      "lon": 37.6914285,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148698553",\n      "kind": "school",\n      "name": "Школа №272",\n      "lat": 55.8686323,\n      "lon": 37.5612752,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148751530",\n      "kind": "school",\n      "name": "Школа № 1245. Корпус школьного образования №3",\n      "lat": 55.6034944,\n      "lon": 37.6016657,\n      "opening_hours": null\n    },\n    {\n      "id": "way/149573325",\n      "kind": "school",\n      "name": "Школа № 1541. Корпус школьного образования",\n      "lat": 55.6722034,\n      "lon": 37.510608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/149865608",\n      "kind": "school",\n      "name": "Школа № 937",\n      "lat": 55.6231737,\n      "lon": 37.6971075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/149865610",\n      "kind": "school",\n      "name": "Московский городской детский телевизионный учебный центр",\n      "lat": 55.6222491,\n      "lon": 37.6978722,\n      "opening_hours": null\n    },\n    {\n      "id": "way/149908800",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1530 \\"Школа Ломоносова\\" Дошкольные группы на Охотничьей",\n      "lat": 55.795574,\n      "lon": 37.6846887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150185126",\n      "kind": "school",\n      "name": "ГБОУ Школа 1420, Корпус № 12",\n      "lat": 55.7001639,\n      "lon": 37.7872112,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150185142",\n      "kind": "kindergarten",\n      "name": "Детский сад №321",\n      "lat": 55.6999627,\n      "lon": 37.7889057,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150206818",\n      "kind": "kindergarten",\n      "name": "ДОУ №1407",\n      "lat": 55.6628468,\n      "lon": 37.6476168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150470231",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №03",\n      "lat": 55.6213366,\n      "lon": 37.4846388,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150881647",\n      "kind": "school",\n      "name": "Школа №708",\n      "lat": 55.8033541,\n      "lon": 37.7675175,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150964760",\n      "kind": "school",\n      "name": "Центр образования №1468",\n      "lat": 55.7427716,\n      "lon": 37.6809029,\n      "opening_hours": null\n    },\n    {\n      "id": "way/151230707",\n      "kind": "school",\n      "name": "Центр Образования №1874 (Начальная школа)",\n      "lat": 55.805346,\n      "lon": 37.4673533,\n      "opening_hours": null\n    },\n    {\n      "id": "way/151230712",\n      "kind": "kindergarten",\n      "name": "Детский сад №333",\n      "lat": 55.8047605,\n      "lon": 37.4669519,\n      "opening_hours": null\n    },\n    {\n      "id": "way/152795922",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1204",\n      "lat": 55.6837241,\n      "lon": 37.5734087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/153236806",\n      "kind": "school",\n      "name": "Школа №365",\n      "lat": 55.8173651,\n      "lon": 37.7287664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/153479091",\n      "kind": "kindergarten",\n      "name": "Дом ребёнка № 9",\n      "lat": 55.6363278,\n      "lon": 37.5184343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154000034",\n      "kind": "kindergarten",\n      "name": "Детский сад №437 (компенсирующего типа)",\n      "lat": 55.8157708,\n      "lon": 37.7194102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154000076",\n      "kind": "school",\n      "name": "Школа №442",\n      "lat": 55.8154922,\n      "lon": 37.7223619,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154000082",\n      "kind": "kindergarten",\n      "name": "Детский сад №35",\n      "lat": 55.8171879,\n      "lon": 37.721919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154000083",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8135927,\n      "lon": 37.7325946,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154047644",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Развитие»",\n      "lat": 55.6320968,\n      "lon": 37.5993351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154047649",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Гармония»",\n      "lat": 55.6312741,\n      "lon": 37.5967687,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154143461",\n      "kind": "kindergarten",\n      "name": "Ясли-сад \\"Остров детства\\"",\n      "lat": 55.8013716,\n      "lon": 37.5690702,\n      "opening_hours": "Mo-Su 07:00-21:00"\n    },\n    {\n      "id": "way/154331611",\n      "kind": "school",\n      "name": "Школа № 1450 Олимп",\n      "lat": 55.6586424,\n      "lon": 37.6113636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154331612",\n      "kind": "school",\n      "name": "Школа №1450 Олимп",\n      "lat": 55.6578417,\n      "lon": 37.6123671,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154343006",\n      "kind": "kindergarten",\n      "name": "Детский сад №1040",\n      "lat": 55.8227834,\n      "lon": 37.7173988,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154357103",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7123096,\n      "lon": 37.668503,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154363995",\n      "kind": "kindergarten",\n      "name": "Детский сад №367",\n      "lat": 55.8154176,\n      "lon": 37.7006334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154363996",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8128319,\n      "lon": 37.7013467,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154368654",\n      "kind": "school",\n      "name": "Школа №1349",\n      "lat": 55.8095204,\n      "lon": 37.7357242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154430931",\n      "kind": "school",\n      "name": "Школа №1561, корпус 2",\n      "lat": 55.5996604,\n      "lon": 37.5423868,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154544725",\n      "kind": "kindergarten",\n      "name": "Школа 1795 \\"Лосиноостровская\\" Дошкольное отделение",\n      "lat": 55.8178352,\n      "lon": 37.694897,\n      "opening_hours": "Mo-Sa 07:00-19:00"\n    },\n    {\n      "id": "way/154980275",\n      "kind": "school",\n      "name": "",\n      "lat": 55.676499,\n      "lon": 37.5581504,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154980276",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1165",\n      "lat": 55.6770954,\n      "lon": 37.558801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/155340396",\n      "kind": "kindergarten",\n      "name": "СПДО № 2 школы № 1784",\n      "lat": 55.7755299,\n      "lon": 37.5565296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/155551279",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8191332,\n      "lon": 37.6999845,\n      "opening_hours": null\n    },\n    {\n      "id": "way/155653703",\n      "kind": "school",\n      "name": "Школа №1360",\n      "lat": 55.815756,\n      "lon": 37.7110323,\n      "opening_hours": null\n    },\n    {\n      "id": "way/155653757",\n      "kind": "school",\n      "name": "Школа №374",\n      "lat": 55.8163905,\n      "lon": 37.6974535,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156236091",\n      "kind": "kindergarten",\n      "name": "Детский сад №249",\n      "lat": 55.7811408,\n      "lon": 37.5127382,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156236175",\n      "kind": "school",\n      "name": "Школа №141",\n      "lat": 55.7790134,\n      "lon": 37.5144449,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156334837",\n      "kind": "school",\n      "name": "Школа №1284",\n      "lat": 55.7678791,\n      "lon": 37.6384005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156378577",\n      "kind": "school",\n      "name": "Школа №2121 – школьный корпус №1",\n      "lat": 55.6846108,\n      "lon": 37.7539906,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156527145",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Солнышко",\n      "lat": 55.6813463,\n      "lon": 37.6713146,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156830473",\n      "kind": "school",\n      "name": "ГБОУ школа № 463 Корпус №2",\n      "lat": 55.6843454,\n      "lon": 37.6815914,\n      "opening_hours": "Mo-Sa 08:00-20:00"\n    },\n    {\n      "id": "way/157140955",\n      "kind": "school",\n      "name": "Школа \\"Самсон\\"",\n      "lat": 55.6838128,\n      "lon": 37.6767863,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157167712",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6491647,\n      "lon": 37.7225809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157210046",\n      "kind": "school",\n      "name": "Лицей №1580",\n      "lat": 55.642702,\n      "lon": 37.6129445,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157566692",\n      "kind": "kindergarten",\n      "name": "Детский сад №1183",\n      "lat": 55.7443151,\n      "lon": 37.5774122,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157600516",\n      "kind": "school",\n      "name": "Центр образования №1865",\n      "lat": 55.7835789,\n      "lon": 37.5124591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157617101",\n      "kind": "school",\n      "name": "Школа №684",\n      "lat": 55.8078239,\n      "lon": 37.5158862,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157730576",\n      "kind": "kindergarten",\n      "name": "Детский сад №2136",\n      "lat": 55.7529594,\n      "lon": 37.5048761,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157730583",\n      "kind": "school",\n      "name": "Образовательный центр «Протон». Школа на Новозаводской",\n      "lat": 55.7521711,\n      "lon": 37.5044171,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158029421",\n      "kind": "school",\n      "name": "Школа №1884",\n      "lat": 55.867521,\n      "lon": 37.5296699,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158036921",\n      "kind": "kindergarten",\n      "name": "Пушкинская школа № 1500 Дошкольное отделение",\n      "lat": 55.7698575,\n      "lon": 37.6495632,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158182125",\n      "kind": "school",\n      "name": "Класс-центр",\n      "lat": 55.8219276,\n      "lon": 37.5245478,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158434309",\n      "kind": "kindergarten",\n      "name": "Детский сад №1608",\n      "lat": 55.8687143,\n      "lon": 37.5314946,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158895386",\n      "kind": "kindergarten",\n      "name": "Детский сад №2017",\n      "lat": 55.6875482,\n      "lon": 37.5919705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/159085693",\n      "kind": "kindergarten",\n      "name": "Школа № 1770 \\"Детство\\"",\n      "lat": 55.685896,\n      "lon": 37.6783438,\n      "opening_hours": null\n    },\n    {\n      "id": "way/159918976",\n      "kind": "kindergarten",\n      "name": "ДОУ №1330",\n      "lat": 55.6635315,\n      "lon": 37.6482227,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160150768",\n      "kind": "kindergarten",\n      "name": "ДОУ №67",\n      "lat": 55.6811795,\n      "lon": 37.6391218,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160199979",\n      "kind": "school",\n      "name": "школа №574",\n      "lat": 55.6790395,\n      "lon": 37.6543892,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160283590",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-10)",\n      "lat": 55.6751953,\n      "lon": 37.8078488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160340355",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7056664,\n      "lon": 37.4723526,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160417652",\n      "kind": "school",\n      "name": "школа №1251",\n      "lat": 55.7941805,\n      "lon": 37.5068391,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160674739",\n      "kind": "kindergarten",\n      "name": "Детский сад №526",\n      "lat": 55.6472563,\n      "lon": 37.651918,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161413502",\n      "kind": "school",\n      "name": "Школа № 2099. Здание 1",\n      "lat": 55.8177179,\n      "lon": 37.4930151,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161413517",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Здание 4",\n      "lat": 55.8179892,\n      "lon": 37.4911934,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161416878",\n      "kind": "kindergarten",\n      "name": "Детский сад комбинированного вида № 8 «Лебёдушка»",\n      "lat": 55.8924454,\n      "lon": 37.4747833,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161416884",\n      "kind": "kindergarten",\n      "name": "Детский сад №10 Колобок",\n      "lat": 55.8899557,\n      "lon": 37.4742221,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161416892",\n      "kind": "school",\n      "name": "Гимназия №16",\n      "lat": 55.8905784,\n      "lon": 37.4763473,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161420365",\n      "kind": "kindergarten",\n      "name": "Детский сад «Золотой ключик»",\n      "lat": 55.8886522,\n      "lon": 37.4753242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161849900",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №13",\n      "lat": 55.5927099,\n      "lon": 37.6615374,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161863169",\n      "kind": "kindergarten",\n      "name": "Школа №508 с дошкольным отделением",\n      "lat": 55.5837918,\n      "lon": 37.6956312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161863170",\n      "kind": "school",\n      "name": "Школа №508 с дошкольным отделением",\n      "lat": 55.5844758,\n      "lon": 37.6974443,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161863171",\n      "kind": "kindergarten",\n      "name": "Детский сад №2472",\n      "lat": 55.5839739,\n      "lon": 37.6890365,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161872281",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 14",\n      "lat": 55.8492998,\n      "lon": 37.6442441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161893388",\n      "kind": "kindergarten",\n      "name": "Детский сад №2675",\n      "lat": 55.7484788,\n      "lon": 37.7786734,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161956483",\n      "kind": "kindergarten",\n      "name": "Детский сад №1436",\n      "lat": 55.6794885,\n      "lon": 37.7572409,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161956486",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6787899,\n      "lon": 37.757046,\n      "opening_hours": null\n    },\n    {\n      "id": "way/162037557",\n      "kind": "kindergarten",\n      "name": "Школа №283, дошкольное отделение 8",\n      "lat": 55.8862434,\n      "lon": 37.6641712,\n      "opening_hours": null\n    },\n    {\n      "id": "way/162037559",\n      "kind": "school",\n      "name": "Школа №283, отделение 7",\n      "lat": 55.885776,\n      "lon": 37.6676882,\n      "opening_hours": null\n    },\n    {\n      "id": "way/162811503",\n      "kind": "school",\n      "name": "Школа №1056 (начальная школа и общее образование)",\n      "lat": 55.8481568,\n      "lon": 37.4438749,\n      "opening_hours": null\n    },\n    {\n      "id": "way/164433805",\n      "kind": "kindergarten",\n      "name": "Детский сад №1477",\n      "lat": 55.584667,\n      "lon": 37.6842664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/165254296",\n      "kind": "kindergarten",\n      "name": "Школа №51. Дошкольные группы",\n      "lat": 55.6417229,\n      "lon": 37.4837087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/165254299",\n      "kind": "school",\n      "name": "Школа №109. Начальная школа",\n      "lat": 55.6426295,\n      "lon": 37.478506,\n      "opening_hours": null\n    },\n    {\n      "id": "way/165467022",\n      "kind": "kindergarten",\n      "name": "ЭЦДиЮТ \\"Родина\\"",\n      "lat": 55.7782153,\n      "lon": 37.8220447,\n      "opening_hours": null\n    },\n    {\n      "id": "way/165706724",\n      "kind": "kindergarten",\n      "name": "Д/C № 2618",\n      "lat": 55.6339332,\n      "lon": 37.6798809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/166477542",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1530 \\"Школа Ломоносова\\" Дошкольное отделение",\n      "lat": 55.7911611,\n      "lon": 37.6816168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/167637695",\n      "kind": "school",\n      "name": "Школа №1598. Здание №6",\n      "lat": 55.8280182,\n      "lon": 37.8142915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168049902",\n      "kind": "school",\n      "name": "Школа №1514",\n      "lat": 55.6657508,\n      "lon": 37.5249624,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168385808",\n      "kind": "school",\n      "name": "Школа №1374",\n      "lat": 55.8688359,\n      "lon": 37.7037404,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168744868",\n      "kind": "school",\n      "name": "Гимназия №45",\n      "lat": 55.6852034,\n      "lon": 37.5982635,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168840466",\n      "kind": "school",\n      "name": "Школа №445",\n      "lat": 55.7935435,\n      "lon": 37.7811988,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168992391",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Зарайской",\n      "lat": 55.7273187,\n      "lon": 37.7784155,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170304996",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Развитие\\"",\n      "lat": 55.6866343,\n      "lon": 37.5967276,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170333997",\n      "kind": "kindergarten",\n      "name": "Детский сад №281",\n      "lat": 55.8843784,\n      "lon": 37.5246873,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170334002",\n      "kind": "kindergarten",\n      "name": "Детский сад №2664",\n      "lat": 55.8847624,\n      "lon": 37.5242061,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170334003",\n      "kind": "school",\n      "name": "Школа №236",\n      "lat": 55.8841757,\n      "lon": 37.5228733,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170945825",\n      "kind": "kindergarten",\n      "name": "Школа № 657. Корпус дошкольного образования №1",\n      "lat": 55.6056535,\n      "lon": 37.6125019,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170945868",\n      "kind": "kindergarten",\n      "name": "Школа № 657. Корпус дошкольного образования №2",\n      "lat": 55.6052753,\n      "lon": 37.6148223,\n      "opening_hours": null\n    },\n    {\n      "id": "way/171694384",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2667",\n      "lat": 55.8823821,\n      "lon": 37.5255291,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172228479",\n      "kind": "kindergarten",\n      "name": "Детский сад №497",\n      "lat": 55.8810008,\n      "lon": 37.5174284,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172278760",\n      "kind": "kindergarten",\n      "name": "Школа № 2054. Дошкольный корпус № 1",\n      "lat": 55.7723434,\n      "lon": 37.6242077,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172288880",\n      "kind": "kindergarten",\n      "name": "Детский сад для ослабленных детей",\n      "lat": 55.7882984,\n      "lon": 37.5960732,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172805530",\n      "kind": "kindergarten",\n      "name": "Детский сад №241",\n      "lat": 55.7658755,\n      "lon": 37.4833836,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172805531",\n      "kind": "kindergarten",\n      "name": "Детский сад №1414",\n      "lat": 55.7644863,\n      "lon": 37.4813463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172805534",\n      "kind": "school",\n      "name": "Школа №696",\n      "lat": 55.7646651,\n      "lon": 37.4843043,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172839742",\n      "kind": "kindergarten",\n      "name": "Школа № 1297. Дошкольное отделение",\n      "lat": 55.7823177,\n      "lon": 37.6405325,\n      "opening_hours": null\n    },\n    {\n      "id": "way/173206552",\n      "kind": "kindergarten",\n      "name": "Детский сад 2007",\n      "lat": 55.6863958,\n      "lon": 37.6010518,\n      "opening_hours": null\n    },\n    {\n      "id": "way/173567631",\n      "kind": "kindergarten",\n      "name": "Лицей №1571 дошкольное отделение Веснушки",\n      "lat": 55.8545842,\n      "lon": 37.4430321,\n      "opening_hours": null\n    },\n    {\n      "id": "way/174679496",\n      "kind": "kindergarten",\n      "name": "Школа №1352 (дошкольная группа на Щёлковском шс., 91с2)",\n      "lat": 55.8144211,\n      "lon": 37.8244129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/174679502",\n      "kind": "school",\n      "name": "Школа №1352 (корпус на Новосибирской, 8А)",\n      "lat": 55.8158567,\n      "lon": 37.8227454,\n      "opening_hours": null\n    },\n    {\n      "id": "way/175916326",\n      "kind": "kindergarten",\n      "name": "Детский сад №2668",\n      "lat": 55.8691537,\n      "lon": 37.5295515,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176025435",\n      "kind": "school",\n      "name": "Пушкинская школа № 1500. Здание на Скорняжном",\n      "lat": 55.7735481,\n      "lon": 37.6425694,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176491329",\n      "kind": "kindergarten",\n      "name": "Детский сад №1064",\n      "lat": 55.7145643,\n      "lon": 37.7982918,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176751809",\n      "kind": "kindergarten",\n      "name": "Детский сад №1540",\n      "lat": 55.7254551,\n      "lon": 37.8059508,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176751812",\n      "kind": "school",\n      "name": "Школа №407",\n      "lat": 55.7239431,\n      "lon": 37.8121755,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176751813",\n      "kind": "school",\n      "name": "Школа №813",\n      "lat": 55.7200325,\n      "lon": 37.8137074,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176751989",\n      "kind": "kindergarten",\n      "name": "Детский сад №1906",\n      "lat": 55.7171706,\n      "lon": 37.8249971,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176752007",\n      "kind": "kindergarten",\n      "name": "Детский сад №62",\n      "lat": 55.7175272,\n      "lon": 37.8240275,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176752014",\n      "kind": "kindergarten",\n      "name": "Детский сад №1150",\n      "lat": 55.71956,\n      "lon": 37.8265935,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176752045",\n      "kind": "kindergarten",\n      "name": "Прогимназия №1753",\n      "lat": 55.7190542,\n      "lon": 37.8282025,\n      "opening_hours": null\n    },\n    {\n      "id": "way/178441138",\n      "kind": "kindergarten",\n      "name": "Школа №1505 Дошкольные группы",\n      "lat": 55.7974773,\n      "lon": 37.7364447,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/178441142",\n      "kind": "kindergarten",\n      "name": "Детский сад №1343",\n      "lat": 55.7971331,\n      "lon": 37.730984,\n      "opening_hours": null\n    },\n    {\n      "id": "way/178441147",\n      "kind": "school",\n      "name": "Школа №1032",\n      "lat": 55.7958595,\n      "lon": 37.7309944,\n      "opening_hours": null\n    },\n    {\n      "id": "way/178441148",\n      "kind": "school",\n      "name": "Школа №1505 (начальное)",\n      "lat": 55.7969714,\n      "lon": 37.7384377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/179464291",\n      "kind": "kindergarten",\n      "name": "Детский сад №1011",\n      "lat": 55.6250984,\n      "lon": 37.7243024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/179488798",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8092378,\n      "lon": 37.6330871,\n      "opening_hours": null\n    },\n    {\n      "id": "way/179615791",\n      "kind": "kindergarten",\n      "name": "Частный детский сад «Олимпик»",\n      "lat": 55.8068203,\n      "lon": 37.6896419,\n      "opening_hours": null\n    },\n    {\n      "id": "way/181158039",\n      "kind": "kindergarten",\n      "name": "Школа 949 (дошкольное отделение, здание 6)",\n      "lat": 55.6094234,\n      "lon": 37.7483112,\n      "opening_hours": null\n    },\n    {\n      "id": "way/181158040",\n      "kind": "kindergarten",\n      "name": "Школа №949 (дошкольное отделение, здание №7)",\n      "lat": 55.6088728,\n      "lon": 37.7479897,\n      "opening_hours": null\n    },\n    {\n      "id": "way/181333557",\n      "kind": "kindergarten",\n      "name": "Школа №158, дошкольное отделение №7",\n      "lat": 55.8624346,\n      "lon": 37.4760164,\n      "opening_hours": null\n    },\n    {\n      "id": "way/181999177",\n      "kind": "kindergarten",\n      "name": "Школа №949 (дошкольное отделение, здание №4)",\n      "lat": 55.6066217,\n      "lon": 37.7482978,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182027507",\n      "kind": "school",\n      "name": "Гимназия №1569 Созвездие",\n      "lat": 55.6174472,\n      "lon": 37.7392698,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182027508",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1632",\n      "lat": 55.6149475,\n      "lon": 37.7342948,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182027900",\n      "kind": "kindergarten",\n      "name": "Школа 949 (дошкольное отделение, здание №3)",\n      "lat": 55.6101142,\n      "lon": 37.7569828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182027903",\n      "kind": "school",\n      "name": "Школа № 949 (средняя школа, здание №1)",\n      "lat": 55.6088366,\n      "lon": 37.7539915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182028220",\n      "kind": "kindergarten",\n      "name": "Школа № 2116, дошкольное отделение № 5",\n      "lat": 55.618126,\n      "lon": 37.7587734,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182028221",\n      "kind": "kindergarten",\n      "name": "Школа № 2116, дошкольное отделение № 8",\n      "lat": 55.6175227,\n      "lon": 37.7549141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182028222",\n      "kind": "school",\n      "name": "Школа № 2116, школьное отделение № 1",\n      "lat": 55.6207859,\n      "lon": 37.754642,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182028223",\n      "kind": "school",\n      "name": "Школа № 2116, школьное отделение № 3",\n      "lat": 55.6207962,\n      "lon": 37.7567888,\n      "opening_hours": null\n    },\n    {\n      "id": "way/183539979",\n      "kind": "school",\n      "name": "Школа № 1465 им. Н. Г. Кузнецова. Школьное отделение № 2",\n      "lat": 55.7478433,\n      "lon": 37.5679935,\n      "opening_hours": null\n    },\n    {\n      "id": "way/183644378",\n      "kind": "kindergarten",\n      "name": "Детский сад №733",\n      "lat": 55.7991673,\n      "lon": 37.4653919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/183768721",\n      "kind": "kindergarten",\n      "name": "Детский сад №64/1 «Яблонька»",\n      "lat": 55.8887579,\n      "lon": 37.7208656,\n      "opening_hours": null\n    },\n    {\n      "id": "way/183948096",\n      "kind": "school",\n      "name": "Лицей №1580",\n      "lat": 55.6415767,\n      "lon": 37.6126431,\n      "opening_hours": null\n    },\n    {\n      "id": "way/184132673",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1305",\n      "lat": 55.7086369,\n      "lon": 37.6121409,\n      "opening_hours": null\n    },\n    {\n      "id": "way/184132674",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР № 1662",\n      "lat": 55.708453,\n      "lon": 37.6167661,\n      "opening_hours": null\n    },\n    {\n      "id": "way/185251618",\n      "kind": "kindergarten",\n      "name": "Школа №1352 (дошкольная группа на Байкальской, 46А)",\n      "lat": 55.8151546,\n      "lon": 37.8180203,\n      "opening_hours": null\n    },\n    {\n      "id": "way/185254551",\n      "kind": "kindergarten",\n      "name": "Детский сад №2565",\n      "lat": 55.806861,\n      "lon": 37.7801568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/185254552",\n      "kind": "school",\n      "name": "Школа №2033",\n      "lat": 55.8079228,\n      "lon": 37.7794539,\n      "opening_hours": null\n    },\n    {\n      "id": "way/185426891",\n      "kind": "school",\n      "name": "Школа №378",\n      "lat": 55.7949577,\n      "lon": 37.696326,\n      "opening_hours": null\n    },\n    {\n      "id": "way/186311028",\n      "kind": "school",\n      "name": "Школа №1290 корпус №2",\n      "lat": 55.7888593,\n      "lon": 37.7795474,\n      "opening_hours": null\n    },\n    {\n      "id": "way/187243714",\n      "kind": "school",\n      "name": "Школа № 1987. Основная школа",\n      "lat": 55.6605666,\n      "lon": 37.7763621,\n      "opening_hours": null\n    },\n    {\n      "id": "way/187244540",\n      "kind": "school",\n      "name": "Центр образования №1426",\n      "lat": 55.5965294,\n      "lon": 37.6600262,\n      "opening_hours": null\n    },\n    {\n      "id": "way/187448839",\n      "kind": "kindergarten",\n      "name": "Центр образования №1404 \\"Гамма\\"",\n      "lat": 55.7982219,\n      "lon": 37.6859968,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188079728",\n      "kind": "school",\n      "name": "Школа №1275",\n      "lat": 55.7837857,\n      "lon": 37.6070414,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188330084",\n      "kind": "school",\n      "name": "Школа №1035",\n      "lat": 55.8202276,\n      "lon": 37.7539524,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188330310",\n      "kind": "kindergarten",\n      "name": "Детский сад №1332",\n      "lat": 55.8217216,\n      "lon": 37.7567904,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188434483",\n      "kind": "school",\n      "name": "Школа №373",\n      "lat": 55.8287111,\n      "lon": 37.7572137,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188434606",\n      "kind": "school",\n      "name": "Школа №1246",\n      "lat": 55.8292877,\n      "lon": 37.7605578,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188435372",\n      "kind": "kindergarten",\n      "name": "Детский сад №605",\n      "lat": 55.8244308,\n      "lon": 37.763765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188575249",\n      "kind": "kindergarten",\n      "name": "Детский сад №989",\n      "lat": 55.8263258,\n      "lon": 37.7541493,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188575301",\n      "kind": "school",\n      "name": "Школа №835",\n      "lat": 55.8271118,\n      "lon": 37.7524487,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188576666",\n      "kind": "kindergarten",\n      "name": "Детский сад №2669",\n      "lat": 55.82346,\n      "lon": 37.7618518,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188584996",\n      "kind": "kindergarten",\n      "name": "Детский сад №357",\n      "lat": 55.8252612,\n      "lon": 37.7537113,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188584997",\n      "kind": "kindergarten",\n      "name": "Детский сад №1872",\n      "lat": 55.826902,\n      "lon": 37.757298,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188593423",\n      "kind": "kindergarten",\n      "name": "Детский сад №273",\n      "lat": 55.8256595,\n      "lon": 37.7631857,\n      "opening_hours": null\n    },\n    {\n      "id": "way/189136645",\n      "kind": "school",\n      "name": "Гимназия №1572",\n      "lat": 55.7866458,\n      "lon": 37.6101903,\n      "opening_hours": null\n    },\n    {\n      "id": "way/189136646",\n      "kind": "kindergarten",\n      "name": "Детский сад №1918",\n      "lat": 55.7823876,\n      "lon": 37.60879,\n      "opening_hours": null\n    },\n    {\n      "id": "way/189725485",\n      "kind": "school",\n      "name": "СОШ № 198",\n      "lat": 55.7064943,\n      "lon": 37.5835591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/190098593",\n      "kind": "school",\n      "name": "Центр образования №449",\n      "lat": 55.7872543,\n      "lon": 37.7405437,\n      "opening_hours": null\n    },\n    {\n      "id": "way/190118996",\n      "kind": "kindergarten",\n      "name": "Детский сад №573",\n      "lat": 55.7917745,\n      "lon": 37.7724748,\n      "opening_hours": null\n    },\n    {\n      "id": "way/190430436",\n      "kind": "school",\n      "name": "Гимназия №\xa09",\n      "lat": 55.8975721,\n      "lon": 37.454051,\n      "opening_hours": null\n    },\n    {\n      "id": "way/190430438",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa020 «Лукоморье»",\n      "lat": 55.8982473,\n      "lon": 37.4545036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191278153",\n      "kind": "kindergarten",\n      "name": "Детский сад №449",\n      "lat": 55.8590909,\n      "lon": 37.4810238,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191278154",\n      "kind": "kindergarten",\n      "name": "Детский сад №503",\n      "lat": 55.8584909,\n      "lon": 37.481665,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191278155",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.861878,\n      "lon": 37.4783044,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191318881",\n      "kind": "school",\n      "name": "ДДТ \\"Исток\\"",\n      "lat": 55.6341603,\n      "lon": 37.5897241,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191905193",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6820855,\n      "lon": 37.5734275,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191971338",\n      "kind": "kindergarten",\n      "name": "Детский сад №1435",\n      "lat": 55.8725379,\n      "lon": 37.6685796,\n      "opening_hours": null\n    },\n    {\n      "id": "way/192393322",\n      "kind": "kindergarten",\n      "name": "Детский сад №284",\n      "lat": 55.7776387,\n      "lon": 37.6736783,\n      "opening_hours": null\n    },\n    {\n      "id": "way/192897006",\n      "kind": "kindergarten",\n      "name": "Детский сад №1892",\n      "lat": 55.7413514,\n      "lon": 37.5483953,\n      "opening_hours": null\n    },\n    {\n      "id": "way/192917712",\n      "kind": "kindergarten",\n      "name": "Детский сад №1957",\n      "lat": 55.8300026,\n      "lon": 37.7606707,\n      "opening_hours": null\n    },\n    {\n      "id": "way/193282033",\n      "kind": "kindergarten",\n      "name": "Школа №1499 Дошкольное отделение",\n      "lat": 55.8359403,\n      "lon": 37.6394328,\n      "opening_hours": null\n    },\n    {\n      "id": "way/193443974",\n      "kind": "school",\n      "name": "Школа №1499",\n      "lat": 55.8369942,\n      "lon": 37.6427461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/193637899",\n      "kind": "kindergarten",\n      "name": "Гимназия РУТ (МИИТ)",\n      "lat": 55.8013399,\n      "lon": 37.6548101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/193747059",\n      "kind": "kindergarten",\n      "name": "Школа №158. Дошкольное отделение № 1",\n      "lat": 55.8721214,\n      "lon": 37.4660128,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194080481",\n      "kind": "kindergarten",\n      "name": "Школа №1285 структурное подразделение Золотая рыбка",\n      "lat": 55.8310319,\n      "lon": 37.4490797,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194309723",\n      "kind": "school",\n      "name": "Гимназия 625",\n      "lat": 55.6923343,\n      "lon": 37.5928813,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194309724",\n      "kind": "kindergarten",\n      "name": "Детский сад № 219",\n      "lat": 55.7054076,\n      "lon": 37.5828157,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194309725",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2486",\n      "lat": 55.6940045,\n      "lon": 37.5798872,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194898350",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8290764,\n      "lon": 37.665629,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194898502",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8278859,\n      "lon": 37.6643253,\n      "opening_hours": null\n    },\n    {\n      "id": "way/195523838",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО № 1",\n      "lat": 55.700345,\n      "lon": 37.6827521,\n      "opening_hours": "07:30-21:00"\n    },\n    {\n      "id": "way/195529430",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО \\"Начальная школа\\"",\n      "lat": 55.7038274,\n      "lon": 37.6813608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/195584571",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129 ДО \\"Колобок\\"",\n      "lat": 55.701554,\n      "lon": 37.6842476,\n      "opening_hours": null\n    },\n    {\n      "id": "way/195584573",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129 ДО \\"Теремок\\"",\n      "lat": 55.7013056,\n      "lon": 37.6854823,\n      "opening_hours": null\n    },\n    {\n      "id": "way/195726456",\n      "kind": "kindergarten",\n      "name": "Детский сад школы с углубленным изучением отдельных предметов № 1794\\". Здание 11",\n      "lat": 55.8628428,\n      "lon": 37.5707068,\n      "opening_hours": null\n    },\n    {\n      "id": "way/196094633",\n      "kind": "school",\n      "name": "Школа им. Маяковского. Учебный корпус «Пролетарка»",\n      "lat": 55.7294242,\n      "lon": 37.6662235,\n      "opening_hours": null\n    },\n    {\n      "id": "way/196482821",\n      "kind": "kindergarten",\n      "name": "Детский сад №1761",\n      "lat": 55.7515823,\n      "lon": 37.8278759,\n      "opening_hours": null\n    },\n    {\n      "id": "way/196482823",\n      "kind": "school",\n      "name": "Лицей № 1502 при МЭИ",\n      "lat": 55.7602998,\n      "lon": 37.8304282,\n      "opening_hours": null\n    },\n    {\n      "id": "way/196952681",\n      "kind": "school",\n      "name": "Школа-интернат №35",\n      "lat": 55.7872785,\n      "lon": 37.602729,\n      "opening_hours": null\n    },\n    {\n      "id": "way/197131575",\n      "kind": "school",\n      "name": "Школа с углубленным изучением отдельных предметов № 1794. Здание 12",\n      "lat": 55.8611353,\n      "lon": 37.5708535,\n      "opening_hours": null\n    },\n    {\n      "id": "way/197879615",\n      "kind": "kindergarten",\n      "name": "Детский сад №1763",\n      "lat": 55.7660534,\n      "lon": 37.7044463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/197907188",\n      "kind": "kindergarten",\n      "name": "Школа № 2107. Здание \\"На Васнецова\\"",\n      "lat": 55.7756811,\n      "lon": 37.6260571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/198366816",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6463836,\n      "lon": 37.5031408,\n      "opening_hours": null\n    },\n    {\n      "id": "way/198366833",\n      "kind": "school",\n      "name": "Школа №1995",\n      "lat": 55.6459557,\n      "lon": 37.4984602,\n      "opening_hours": null\n    },\n    {\n      "id": "way/198660320",\n      "kind": "school",\n      "name": "Школа №167",\n      "lat": 55.8605304,\n      "lon": 37.4793384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/200141633",\n      "kind": "school",\n      "name": "Школа №922 (учебный корпус \\"Ивановский\\")",\n      "lat": 55.7630246,\n      "lon": 37.8270038,\n      "opening_hours": null\n    },\n    {\n      "id": "way/200657576",\n      "kind": "kindergarten",\n      "name": "Школа 1530 дошкольная группа",\n      "lat": 55.7861336,\n      "lon": 37.6974786,\n      "opening_hours": null\n    },\n    {\n      "id": "way/201014765",\n      "kind": "school",\n      "name": "Гимназия МГУПС",\n      "lat": 55.8031976,\n      "lon": 37.6452612,\n      "opening_hours": null\n    },\n    {\n      "id": "way/202018161",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8018156,\n      "lon": 37.6320398,\n      "opening_hours": null\n    },\n    {\n      "id": "way/202195837",\n      "kind": "school",\n      "name": "Школа №259",\n      "lat": 55.7954085,\n      "lon": 37.600514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/202768060",\n      "kind": "kindergarten",\n      "name": "Детский сад №2716",\n      "lat": 55.6900955,\n      "lon": 37.7191721,\n      "opening_hours": null\n    },\n    {\n      "id": "way/202768061",\n      "kind": "school",\n      "name": "Физико-математический лицей № 1524",\n      "lat": 55.6910038,\n      "lon": 37.7197552,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204059721",\n      "kind": "school",\n      "name": "Школа №444",\n      "lat": 55.7906822,\n      "lon": 37.7964766,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204090333",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №7)",\n      "lat": 55.8086988,\n      "lon": 37.7667312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204347256",\n      "kind": "kindergarten",\n      "name": "Детский сад №2386",\n      "lat": 55.8060555,\n      "lon": 37.7714204,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204356436",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №8)",\n      "lat": 55.8116714,\n      "lon": 37.7799702,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204498542",\n      "kind": "kindergarten",\n      "name": "Школа и детский сад «Колибри»",\n      "lat": 55.8063743,\n      "lon": 37.775343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204500876",\n      "kind": "school",\n      "name": "Школа №319 (здание №5)",\n      "lat": 55.8107836,\n      "lon": 37.778981,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205589387",\n      "kind": "kindergarten",\n      "name": "Школа №51. Дошкольные группы",\n      "lat": 55.6449675,\n      "lon": 37.4834666,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205590474",\n      "kind": "school",\n      "name": "Школа №319 (здание №4)",\n      "lat": 55.8124387,\n      "lon": 37.7893134,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205680825",\n      "kind": "kindergarten",\n      "name": "Детский сад № 260",\n      "lat": 55.643116,\n      "lon": 37.6017362,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205723070",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №9)",\n      "lat": 55.8120881,\n      "lon": 37.7926665,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205723592",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №10)",\n      "lat": 55.8122372,\n      "lon": 37.7947956,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205833586",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7933307,\n      "lon": 37.57062,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205871984",\n      "kind": "kindergarten",\n      "name": "Детский нефрологический санаторий № 9",\n      "lat": 55.8219747,\n      "lon": 37.7916483,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205878083",\n      "kind": "school",\n      "name": "Школа \\"Эдвайс\\"",\n      "lat": 55.8153211,\n      "lon": 37.791076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205880108",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №11)",\n      "lat": 55.8138312,\n      "lon": 37.7904489,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205880825",\n      "kind": "school",\n      "name": "Школа №319 (здание №3)",\n      "lat": 55.8147538,\n      "lon": 37.7926827,\n      "opening_hours": null\n    },\n    {\n      "id": "way/206079411",\n      "kind": "school",\n      "name": "Лад",\n      "lat": 55.819317,\n      "lon": 37.7954182,\n      "opening_hours": null\n    },\n    {\n      "id": "way/206226399",\n      "kind": "kindergarten",\n      "name": "Детский сад №1276",\n      "lat": 55.6845961,\n      "lon": 37.5360134,\n      "opening_hours": null\n    },\n    {\n      "id": "way/206362274",\n      "kind": "school",\n      "name": "Гимназия №\xa01590 им. Героя Советского Союза В.\xa0В.\xa0Колесника",\n      "lat": 55.8627047,\n      "lon": 37.5029639,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207155606",\n      "kind": "kindergarten",\n      "name": "Детский сад №575",\n      "lat": 55.8086448,\n      "lon": 37.7863197,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207156932",\n      "kind": "kindergarten",\n      "name": "Детский сад №2385",\n      "lat": 55.8082542,\n      "lon": 37.7833293,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207161306",\n      "kind": "kindergarten",\n      "name": "Детский сад №55",\n      "lat": 55.8055319,\n      "lon": 37.7852462,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207161871",\n      "kind": "school",\n      "name": "Школа №399",\n      "lat": 55.8059439,\n      "lon": 37.7866808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207163906",\n      "kind": "kindergarten",\n      "name": "Детский сад №2517",\n      "lat": 55.8040276,\n      "lon": 37.7865481,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207289431",\n      "kind": "school",\n      "name": "Кадетская школа №1381 ШО2",\n      "lat": 55.8740201,\n      "lon": 37.6963064,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207606311",\n      "kind": "school",\n      "name": "Школа №1956",\n      "lat": 55.8028548,\n      "lon": 37.6105094,\n      "opening_hours": null\n    },\n    {\n      "id": "way/208552947",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №8",\n      "lat": 55.6086448,\n      "lon": 37.5285579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/210205065",\n      "kind": "kindergarten",\n      "name": "Школа 1236 Дошкольное отделение",\n      "lat": 55.8151472,\n      "lon": 37.5965992,\n      "opening_hours": null\n    },\n    {\n      "id": "way/210508753",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1497",\n      "lat": 55.7719394,\n      "lon": 37.6083963,\n      "opening_hours": null\n    },\n    {\n      "id": "way/213490165",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 1",\n      "lat": 55.6519138,\n      "lon": 37.5573247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/213490168",\n      "kind": "school",\n      "name": "Школа № 15. Корпус школьного образования № 1",\n      "lat": 55.6526899,\n      "lon": 37.5599967,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216024137",\n      "kind": "kindergarten",\n      "name": "Детский сад №1879",\n      "lat": 55.7455538,\n      "lon": 37.7728428,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216065798",\n      "kind": "school",\n      "name": "Школа №920",\n      "lat": 55.746911,\n      "lon": 37.7780788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216252169",\n      "kind": "school",\n      "name": "Школа №660",\n      "lat": 55.8673079,\n      "lon": 37.534496,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216253188",\n      "kind": "school",\n      "name": "Школа № 656 имени А.С. Макаренко",\n      "lat": 55.8620898,\n      "lon": 37.5573871,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216266765",\n      "kind": "kindergarten",\n      "name": "Детский сад №1026",\n      "lat": 55.878866,\n      "lon": 37.5347724,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216268349",\n      "kind": "school",\n      "name": "Лицей №1594",\n      "lat": 55.8641515,\n      "lon": 37.5405072,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216268350",\n      "kind": "school",\n      "name": "Детский сад №1278",\n      "lat": 55.8651654,\n      "lon": 37.5345097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269282",\n      "kind": "school",\n      "name": "Школа №662",\n      "lat": 55.8743239,\n      "lon": 37.5205347,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269283",\n      "kind": "kindergarten",\n      "name": "Детский сад №2311",\n      "lat": 55.8778417,\n      "lon": 37.5132436,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269285",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1851",\n      "lat": 55.8950557,\n      "lon": 37.541291,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269717",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1631",\n      "lat": 55.8911136,\n      "lon": 37.5396505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269825",\n      "kind": "school",\n      "name": "Детский сад №1776",\n      "lat": 55.8758559,\n      "lon": 37.5142108,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216647969",\n      "kind": "kindergarten",\n      "name": "Детский сад №1965",\n      "lat": 55.8829083,\n      "lon": 37.5372014,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216913382",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6639861,\n      "lon": 37.4947696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216919488",\n      "kind": "kindergarten",\n      "name": "Детский сад №2003",\n      "lat": 55.7181944,\n      "lon": 37.7998232,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216919490",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1883",\n      "lat": 55.7172537,\n      "lon": 37.8024957,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217369343",\n      "kind": "kindergarten",\n      "name": "Детский сад №2335",\n      "lat": 55.7825631,\n      "lon": 37.5578006,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217369344",\n      "kind": "school",\n      "name": "Лицей №1550",\n      "lat": 55.7824683,\n      "lon": 37.5596722,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217552197",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 2",\n      "lat": 55.7843327,\n      "lon": 37.5275135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217552202",\n      "kind": "kindergarten",\n      "name": "Детский сад №2508",\n      "lat": 55.7849796,\n      "lon": 37.5253089,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217552415",\n      "kind": "kindergarten",\n      "name": "Детский сад №2526",\n      "lat": 55.7870466,\n      "lon": 37.5219908,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217552491",\n      "kind": "kindergarten",\n      "name": "Детский сад №2527",\n      "lat": 55.7883655,\n      "lon": 37.5210882,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217563272",\n      "kind": "kindergarten",\n      "name": "Детский сад №1374",\n      "lat": 55.698004,\n      "lon": 37.7494159,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217566510",\n      "kind": "kindergarten",\n      "name": "Детский сад №2117",\n      "lat": 55.6999258,\n      "lon": 37.7564173,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217573182",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1494",\n      "lat": 55.6995824,\n      "lon": 37.7609847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218506936",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №5",\n      "lat": 55.6109553,\n      "lon": 37.533306,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218576191",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 1",\n      "lat": 55.8527443,\n      "lon": 37.64264,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218576192",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 10",\n      "lat": 55.8539559,\n      "lon": 37.6457207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218576193",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 6",\n      "lat": 55.8533937,\n      "lon": 37.6389169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218780937",\n      "kind": "kindergarten",\n      "name": "Детский сад №2705",\n      "lat": 55.882469,\n      "lon": 37.6894582,\n      "opening_hours": null\n    },\n    {\n      "id": "way/220747230",\n      "kind": "kindergarten",\n      "name": "Школа №830 дошкольное отделение Умка",\n      "lat": 55.8309813,\n      "lon": 37.4550988,\n      "opening_hours": null\n    },\n    {\n      "id": "way/220941601",\n      "kind": "kindergarten",\n      "name": "Школа №2033 (дошкольные группы)",\n      "lat": 55.8066731,\n      "lon": 37.8082801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/220952575",\n      "kind": "kindergarten",\n      "name": "Детский сад №185",\n      "lat": 55.8755934,\n      "lon": 37.5221162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221105974",\n      "kind": "kindergarten",\n      "name": "Детский сад №438",\n      "lat": 55.659293,\n      "lon": 37.5109541,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221106410",\n      "kind": "school",\n      "name": "Школа № 1541. Корпус школьного образования",\n      "lat": 55.6671995,\n      "lon": 37.5088826,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221246676",\n      "kind": "school",\n      "name": "Школа № 1541. Корпус школьного образования",\n      "lat": 55.6726963,\n      "lon": 37.5045078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221251555",\n      "kind": "school",\n      "name": "English International School of Moscow",\n      "lat": 55.6674115,\n      "lon": 37.5062793,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221261337",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №3",\n      "lat": 55.6734106,\n      "lon": 37.5086919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221263466",\n      "kind": "kindergarten",\n      "name": "Детский сад №629",\n      "lat": 55.6570771,\n      "lon": 37.5148594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221600360",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №5",\n      "lat": 55.6683966,\n      "lon": 37.5101775,\n      "opening_hours": null\n    },\n    {\n      "id": "way/222343517",\n      "kind": "kindergarten",\n      "name": "Детский сад №375",\n      "lat": 55.664523,\n      "lon": 37.5226013,\n      "opening_hours": null\n    },\n    {\n      "id": "way/222343526",\n      "kind": "kindergarten",\n      "name": "Детский сад №406",\n      "lat": 55.6653857,\n      "lon": 37.5225782,\n      "opening_hours": null\n    },\n    {\n      "id": "way/222944753",\n      "kind": "kindergarten",\n      "name": "Детский сад №2521",\n      "lat": 55.8739076,\n      "lon": 37.5181397,\n      "opening_hours": null\n    },\n    {\n      "id": "way/225271999",\n      "kind": "school",\n      "name": "Школа № 1526. Корпус школьного образования",\n      "lat": 55.6028283,\n      "lon": 37.6202075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/225272001",\n      "kind": "kindergarten",\n      "name": "Школа № 657. Корпус дошкольного образования №4",\n      "lat": 55.602798,\n      "lon": 37.6186413,\n      "opening_hours": null\n    },\n    {\n      "id": "way/226915647",\n      "kind": "kindergarten",\n      "name": "Детский сад №1312",\n      "lat": 55.8722461,\n      "lon": 37.6630477,\n      "opening_hours": null\n    },\n    {\n      "id": "way/226915648",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8733562,\n      "lon": 37.6636091,\n      "opening_hours": null\n    },\n    {\n      "id": "way/227478963",\n      "kind": "kindergarten",\n      "name": "Первый Московский Образовательный Комплекс",\n      "lat": 55.8865625,\n      "lon": 37.6499783,\n      "opening_hours": null\n    },\n    {\n      "id": "way/227488072",\n      "kind": "school",\n      "name": "Школа №274",\n      "lat": 55.8838799,\n      "lon": 37.6479545,\n      "opening_hours": null\n    },\n    {\n      "id": "way/227619070",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6912034,\n      "lon": 37.5797537,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228071832",\n      "kind": "kindergarten",\n      "name": "Школа № 950. Структурное подразделение № 12",\n      "lat": 55.8552724,\n      "lon": 37.6110146,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228137029",\n      "kind": "kindergarten",\n      "name": "Детский сад №37 ОАО \\"РЖД\\"",\n      "lat": 55.7907007,\n      "lon": 37.6273513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228407946",\n      "kind": "kindergarten",\n      "name": "Детский сад №33",\n      "lat": 55.7389998,\n      "lon": 37.6878092,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228627305",\n      "kind": "kindergarten",\n      "name": "Детский сад №1413",\n      "lat": 55.7111828,\n      "lon": 37.8110097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228627321",\n      "kind": "kindergarten",\n      "name": "Центр развития ребёнка №2402",\n      "lat": 55.7116161,\n      "lon": 37.8095175,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228627322",\n      "kind": "kindergarten",\n      "name": "Центр развития ребёнка №894",\n      "lat": 55.712108,\n      "lon": 37.8080554,\n      "opening_hours": null\n    },\n    {\n      "id": "way/230771354",\n      "kind": "school",\n      "name": "ЦО №1481",\n      "lat": 55.8744474,\n      "lon": 37.5189514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/230787749",\n      "kind": "school",\n      "name": "Прогимназия №1882",\n      "lat": 55.7056962,\n      "lon": 37.8123139,\n      "opening_hours": null\n    },\n    {\n      "id": "way/232169524",\n      "kind": "school",\n      "name": "Московский городской детский морской центр им. Петра Великого",\n      "lat": 55.8386875,\n      "lon": 37.4843099,\n      "opening_hours": null\n    },\n    {\n      "id": "way/234744383",\n      "kind": "kindergarten",\n      "name": "Детский сад №651",\n      "lat": 55.703062,\n      "lon": 37.6184751,\n      "opening_hours": null\n    },\n    {\n      "id": "way/234883539",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6898616,\n      "lon": 37.4691162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/237560828",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №8 «Елизаветинский»",\n      "lat": 55.6087131,\n      "lon": 37.5305501,\n      "opening_hours": null\n    },\n    {\n      "id": "way/238304835",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2273",\n      "lat": 55.6484251,\n      "lon": 37.6601075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239185830",\n      "kind": "kindergarten",\n      "name": "Школа №1995. Дошкольный уровень",\n      "lat": 55.6446271,\n      "lon": 37.4754571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239870094",\n      "kind": "school",\n      "name": "Школа № 267",\n      "lat": 55.8721182,\n      "lon": 37.6092383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239870098",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1350",\n      "lat": 55.8729235,\n      "lon": 37.6082516,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239872122",\n      "kind": "kindergarten",\n      "name": "Детский сад комбинированного вида № 1105",\n      "lat": 55.8718275,\n      "lon": 37.6067908,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239933215",\n      "kind": "kindergarten",\n      "name": "ДОУ №300",\n      "lat": 55.7648793,\n      "lon": 37.6532127,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239977947",\n      "kind": "kindergarten",\n      "name": "Детский сад №1288",\n      "lat": 55.6449616,\n      "lon": 37.7125384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241164188",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 2 «Страна Ромашково»",\n      "lat": 55.7666389,\n      "lon": 37.5752176,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241167631",\n      "kind": "school",\n      "name": "Лицей № 1568 имени Пабло Неруды",\n      "lat": 55.8746603,\n      "lon": 37.6420919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241354689",\n      "kind": "kindergarten",\n      "name": "ГБОУ г. Москвы Школа № 1159 (ДО)",\n      "lat": 55.8612811,\n      "lon": 37.511966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241360759",\n      "kind": "school",\n      "name": "Школа №222",\n      "lat": 55.8818245,\n      "lon": 37.5615509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241361187",\n      "kind": "school",\n      "name": "Школа №1678",\n      "lat": 55.8917945,\n      "lon": 37.5603014,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241361651",\n      "kind": "school",\n      "name": "Школа №1678",\n      "lat": 55.881696,\n      "lon": 37.5553788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241361719",\n      "kind": "kindergarten",\n      "name": "Детский сад №1611",\n      "lat": 55.8808516,\n      "lon": 37.5533875,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241362470",\n      "kind": "school",\n      "name": "Центр образования №1419",\n      "lat": 55.8762384,\n      "lon": 37.563215,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241362598",\n      "kind": "kindergarten",\n      "name": "Школа № 1794. Дошкольное отделение",\n      "lat": 55.8737582,\n      "lon": 37.5651843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241363249",\n      "kind": "school",\n      "name": "Школа №1296",\n      "lat": 55.8773774,\n      "lon": 37.562658,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241363401",\n      "kind": "kindergarten",\n      "name": "Детский сад №2201",\n      "lat": 55.8787325,\n      "lon": 37.561327,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241363402",\n      "kind": "kindergarten",\n      "name": "Детский сад №2104",\n      "lat": 55.8787335,\n      "lon": 37.5628056,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241364237",\n      "kind": "kindergarten",\n      "name": "Детский сад №22",\n      "lat": 55.8850932,\n      "lon": 37.5605904,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241364238",\n      "kind": "kindergarten",\n      "name": "Детский сад №1230",\n      "lat": 55.8851114,\n      "lon": 37.5616103,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241365040",\n      "kind": "kindergarten",\n      "name": "Детский сад №1241",\n      "lat": 55.8903375,\n      "lon": 37.560558,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241365041",\n      "kind": "kindergarten",\n      "name": "Детский сад №1323",\n      "lat": 55.8903394,\n      "lon": 37.5617939,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241365042",\n      "kind": "kindergarten",\n      "name": "Центр образования №1678",\n      "lat": 55.892004,\n      "lon": 37.5626638,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241366995",\n      "kind": "kindergarten",\n      "name": "Детский сад №1020",\n      "lat": 55.8876309,\n      "lon": 37.5606067,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241366996",\n      "kind": "kindergarten",\n      "name": "Детский сад № 14",\n      "lat": 55.8876311,\n      "lon": 37.5615798,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241371253",\n      "kind": "school",\n      "name": "Гимназия №1592",\n      "lat": 55.8749908,\n      "lon": 37.5548261,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241374319",\n      "kind": "kindergarten",\n      "name": "Детский сад №154",\n      "lat": 55.8659715,\n      "lon": 37.5631375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241375900",\n      "kind": "school",\n      "name": "Школа № 1213. Корпус № 2",\n      "lat": 55.8424324,\n      "lon": 37.573629,\n      "opening_hours": null\n    },\n    {\n      "id": "way/243539757",\n      "kind": "school",\n      "name": "Школа 117, дошкольное отделение",\n      "lat": 55.674477,\n      "lon": 37.5335477,\n      "opening_hours": "07:00-19:00"\n    },\n    {\n      "id": "way/244039366",\n      "kind": "school",\n      "name": "Seymour House School",\n      "lat": 55.802443,\n      "lon": 37.504716,\n      "opening_hours": null\n    },\n    {\n      "id": "way/244467130",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7366467,\n      "lon": 37.6852351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/244467372",\n      "kind": "kindergarten",\n      "name": "Детский сад №2731",\n      "lat": 55.6123525,\n      "lon": 37.5762322,\n      "opening_hours": null\n    },\n    {\n      "id": "way/246898126",\n      "kind": "school",\n      "name": "Гимназия №1597",\n      "lat": 55.7092981,\n      "lon": 37.8163047,\n      "opening_hours": null\n    },\n    {\n      "id": "way/246898127",\n      "kind": "kindergarten",\n      "name": "Детский сад №2717",\n      "lat": 55.7100719,\n      "lon": 37.8129302,\n      "opening_hours": null\n    },\n    {\n      "id": "way/246898128",\n      "kind": "kindergarten",\n      "name": "Детский сад №931",\n      "lat": 55.7093926,\n      "lon": 37.8143746,\n      "opening_hours": null\n    },\n    {\n      "id": "way/247696842",\n      "kind": "kindergarten",\n      "name": "Детский сад №576",\n      "lat": 55.7993281,\n      "lon": 37.7434797,\n      "opening_hours": null\n    },\n    {\n      "id": "way/247982892",\n      "kind": "school",\n      "name": "Гимназия №1583",\n      "lat": 55.8541414,\n      "lon": 37.5081683,\n      "opening_hours": null\n    },\n    {\n      "id": "way/248420121",\n      "kind": "school",\n      "name": "Школа №1096",\n      "lat": 55.8664031,\n      "lon": 37.698543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/248885411",\n      "kind": "school",\n      "name": "Лицей №1581",\n      "lat": 55.7529689,\n      "lon": 37.6629017,\n      "opening_hours": null\n    },\n    {\n      "id": "way/249169573",\n      "kind": "school",\n      "name": "ГБОУ СОШ 480 Дошкольное отделение",\n      "lat": 55.7330202,\n      "lon": 37.6803981,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/250184978",\n      "kind": "kindergarten",\n      "name": "Детский сад №1 «Солнышко»",\n      "lat": 55.8906491,\n      "lon": 37.479794,\n      "opening_hours": null\n    },\n    {\n      "id": "way/251684295",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа №224 им. П.Н. Еремеева. Дошкольное отделение",\n      "lat": 55.8325349,\n      "lon": 37.4911004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/252200000",\n      "kind": "kindergarten",\n      "name": "Детский сад №1246",\n      "lat": 55.7355913,\n      "lon": 37.5812554,\n      "opening_hours": null\n    },\n    {\n      "id": "way/255945625",\n      "kind": "kindergarten",\n      "name": "Первый Московский Образовательный Комплекс",\n      "lat": 55.8856463,\n      "lon": 37.6499498,\n      "opening_hours": null\n    },\n    {\n      "id": "way/256044990",\n      "kind": "school",\n      "name": "Школа-интернат МГУ",\n      "lat": 55.6944382,\n      "lon": 37.5251815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/256445730",\n      "kind": "school",\n      "name": "Гимназия №1520 им. Капцовых",\n      "lat": 55.7610236,\n      "lon": 37.6049075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/258254558",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.6467675,\n      "lon": 37.5392193,\n      "opening_hours": null\n    },\n    {\n      "id": "way/258753558",\n      "kind": "school",\n      "name": "Школа №491",\n      "lat": 55.6489192,\n      "lon": 37.7220664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/259256790",\n      "kind": "kindergarten",\n      "name": "Школа №556. Корпус дошкольного образования",\n      "lat": 55.6178179,\n      "lon": 37.5825525,\n      "opening_hours": null\n    },\n    {\n      "id": "way/259550332",\n      "kind": "school",\n      "name": "Школа «УНА»",\n      "lat": 55.8780841,\n      "lon": 37.6494028,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260112101",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.796754,\n      "lon": 37.6073217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260177247",\n      "kind": "kindergarten",\n      "name": "Школа № 629. Корпус дошкольного образования №1",\n      "lat": 55.5911265,\n      "lon": 37.5936224,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260452178",\n      "kind": "school",\n      "name": "Школа №237",\n      "lat": 55.7958019,\n      "lon": 37.6120739,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260600593",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8031016,\n      "lon": 37.6136497,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260608752",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус № 2",\n      "lat": 55.6398143,\n      "lon": 37.5302679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/261158934",\n      "kind": "school",\n      "name": "Школа №9",\n      "lat": 55.6630993,\n      "lon": 37.5237737,\n      "opening_hours": null\n    },\n    {\n      "id": "way/261158935",\n      "kind": "school",\n      "name": "Школа №9",\n      "lat": 55.6634794,\n      "lon": 37.524874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/261460294",\n      "kind": "school",\n      "name": "Специализированная детско-юношеская школа олимпийского резерва №28",\n      "lat": 55.7587815,\n      "lon": 37.561584,\n      "opening_hours": null\n    },\n    {\n      "id": "way/262982881",\n      "kind": "school",\n      "name": "Школа № 113. Корпус №1",\n      "lat": 55.6374416,\n      "lon": 37.5163211,\n      "opening_hours": null\n    },\n    {\n      "id": "way/263955306",\n      "kind": "kindergarten",\n      "name": "Школа № 14. Корпус дошкольного образования №4 «Цветик-семицветик»",\n      "lat": 55.6574404,\n      "lon": 37.4701631,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264149139",\n      "kind": "kindergarten",\n      "name": "Школа № 14. Корпус дошкольного образования №3 «Теремок»",\n      "lat": 55.6600184,\n      "lon": 37.4709156,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264154345",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Цветочный городок»",\n      "lat": 55.6318001,\n      "lon": 37.4755183,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264154346",\n      "kind": "school",\n      "name": "ЦО ОЦ \\"Солнечный ветер\\"",\n      "lat": 55.6319114,\n      "lon": 37.4770452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264155007",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Сказочный городок»",\n      "lat": 55.6331354,\n      "lon": 37.4725964,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264155008",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Белоснежка»",\n      "lat": 55.6336279,\n      "lon": 37.4738725,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264644427",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1623",\n      "lat": 55.6897571,\n      "lon": 37.4961502,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264644428",\n      "kind": "school",\n      "name": "Школа № 1119",\n      "lat": 55.6891002,\n      "lon": 37.4974201,\n      "opening_hours": null\n    },\n    {\n      "id": "way/265442692",\n      "kind": "school",\n      "name": "ГБОУ Школа №825",\n      "lat": 55.7109869,\n      "lon": 37.7574233,\n      "opening_hours": null\n    },\n    {\n      "id": "way/267390971",\n      "kind": "school",\n      "name": "Школа на проспекте Вернадского. Корпус школьного образования 1",\n      "lat": 55.6535796,\n      "lon": 37.4836595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/267625741",\n      "kind": "kindergarten",\n      "name": "Детский сад №2209",\n      "lat": 55.6491877,\n      "lon": 37.740232,\n      "opening_hours": null\n    },\n    {\n      "id": "way/270445570",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №5",\n      "lat": 55.8820482,\n      "lon": 37.6432623,\n      "opening_hours": null\n    },\n    {\n      "id": "way/271091464",\n      "kind": "school",\n      "name": "Школа № 814",\n      "lat": 55.7053586,\n      "lon": 37.4661088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/272980100",\n      "kind": "kindergarten",\n      "name": "Детский сад №2282",\n      "lat": 55.6664515,\n      "lon": 37.4527006,\n      "opening_hours": null\n    },\n    {\n      "id": "way/272980137",\n      "kind": "kindergarten",\n      "name": "Детский сад №2252",\n      "lat": 55.6660049,\n      "lon": 37.4542323,\n      "opening_hours": null\n    },\n    {\n      "id": "way/272980138",\n      "kind": "school",\n      "name": "Школа №1741",\n      "lat": 55.6665912,\n      "lon": 37.4554356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/274246811",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2061",\n      "lat": 55.7417004,\n      "lon": 37.4795672,\n      "opening_hours": null\n    },\n    {\n      "id": "way/275801691",\n      "kind": "school",\n      "name": "Школа №484",\n      "lat": 55.7060609,\n      "lon": 37.742303,\n      "opening_hours": null\n    },\n    {\n      "id": "way/276226416",\n      "kind": "kindergarten",\n      "name": "Кембриджская международная школа",\n      "lat": 55.8556729,\n      "lon": 37.4737931,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "way/276880867",\n      "kind": "school",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №1",\n      "lat": 55.8051041,\n      "lon": 37.8280267,\n      "opening_hours": null\n    },\n    {\n      "id": "way/276975541",\n      "kind": "kindergarten",\n      "name": "Школа №439 \\"Интеллект\\" (дошкольное отделение №4)",\n      "lat": 55.7389262,\n      "lon": 37.7382263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/277070499",\n      "kind": "school",\n      "name": "Средняя общеобразовательная школа «Столичный - КИТ»",\n      "lat": 55.7173804,\n      "lon": 37.6146828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/277815333",\n      "kind": "school",\n      "name": "Гимназия № 1570. Дошкольное отделение",\n      "lat": 55.7866261,\n      "lon": 37.5759701,\n      "opening_hours": null\n    },\n    {\n      "id": "way/280429324",\n      "kind": "kindergarten",\n      "name": "Детский сад №2436",\n      "lat": 55.6689648,\n      "lon": 37.7553539,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282238821",\n      "kind": "school",\n      "name": "Школа #1493",\n      "lat": 55.6536248,\n      "lon": 37.7495045,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282497054",\n      "kind": "kindergarten",\n      "name": "Детский сад №1621",\n      "lat": 55.6216372,\n      "lon": 37.7009571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282750312",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8723561,\n      "lon": 37.5609147,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282873404",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129  ДО \\"Терем-Теремок\\"",\n      "lat": 55.7062587,\n      "lon": 37.6780758,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282875256",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129  ДО \\"Солнышко\\"",\n      "lat": 55.7055103,\n      "lon": 37.678256,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282885417",\n      "kind": "kindergarten",\n      "name": "Детский сад №48",\n      "lat": 55.6542956,\n      "lon": 37.6152244,\n      "opening_hours": null\n    },\n    {\n      "id": "way/283341336",\n      "kind": "school",\n      "name": "Школа №858",\n      "lat": 55.6158049,\n      "lon": 37.6021169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/284376076",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6495378,\n      "lon": 37.711225,\n      "opening_hours": null\n    },\n    {\n      "id": "way/284385908",\n      "kind": "school",\n      "name": "Школа № 1582. Корпус школьного образования",\n      "lat": 55.617823,\n      "lon": 37.6007872,\n      "opening_hours": null\n    },\n    {\n      "id": "way/284401169",\n      "kind": "school",\n      "name": "НП ЦО \\"Лицей \\"Столичный\\"",\n      "lat": 55.6168761,\n      "lon": 37.588356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/284423448",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО №2",\n      "lat": 55.7063838,\n      "lon": 37.6722458,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285186415",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №3",\n      "lat": 55.6087959,\n      "lon": 37.5159587,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285186416",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa01986",\n      "lat": 55.6085806,\n      "lon": 37.5170265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285794541",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №7 «Александровский»",\n      "lat": 55.5909684,\n      "lon": 37.5478788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285875625",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №2",\n      "lat": 55.6051207,\n      "lon": 37.5136272,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285875626",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №7",\n      "lat": 55.6048991,\n      "lon": 37.5146559,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286082075",\n      "kind": "kindergarten",\n      "name": "Школа №851",\n      "lat": 55.6333488,\n      "lon": 37.6059369,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286482971",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №6",\n      "lat": 55.602692,\n      "lon": 37.5187104,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286482972",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №4",\n      "lat": 55.6029143,\n      "lon": 37.5176385,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584497",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №3",\n      "lat": 55.6048558,\n      "lon": 37.5175329,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584498",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №1",\n      "lat": 55.6068056,\n      "lon": 37.5169351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584499",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №2",\n      "lat": 55.6058422,\n      "lon": 37.5172227,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584500",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №5",\n      "lat": 55.6072267,\n      "lon": 37.5205718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584501",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №1",\n      "lat": 55.6069813,\n      "lon": 37.5216692,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287005456",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6514576,\n      "lon": 37.7361533,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287005459",\n      "kind": "kindergarten",\n      "name": "Детский сад №1881",\n      "lat": 55.6529243,\n      "lon": 37.7367682,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287198704",\n      "kind": "kindergarten",\n      "name": "СПДО № 3 школы № 1784",\n      "lat": 55.7870238,\n      "lon": 37.5708603,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287304987",\n      "kind": "school",\n      "name": "Школа №1450 Олимп",\n      "lat": 55.6539415,\n      "lon": 37.6123343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287306189",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №3",\n      "lat": 55.608345,\n      "lon": 37.5505637,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287306190",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №2",\n      "lat": 55.6083003,\n      "lon": 37.5493128,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287495346",\n      "kind": "kindergarten",\n      "name": "ДОУ №1177",\n      "lat": 55.6809564,\n      "lon": 37.6534372,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287553004",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №5",\n      "lat": 55.6001688,\n      "lon": 37.5117202,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287553005",\n      "kind": "school",\n      "name": "Православная гимназия \\"Радонеж\\"",\n      "lat": 55.5991123,\n      "lon": 37.5135717,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288601028",\n      "kind": "school",\n      "name": "Школа \\"Выбор\\"",\n      "lat": 55.65794,\n      "lon": 37.6102343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288604118",\n      "kind": "school",\n      "name": "Школа № 1212. Корпус школьного образования №2",\n      "lat": 55.5993401,\n      "lon": 37.5175476,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288604121",\n      "kind": "school",\n      "name": "Школа № 1212. Корпус школьного образования №1",\n      "lat": 55.5981508,\n      "lon": 37.5170961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288609539",\n      "kind": "kindergarten",\n      "name": "Школа № 1862. Корпус дошкольного образования №8",\n      "lat": 55.6504413,\n      "lon": 37.617549,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288609547",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №10",\n      "lat": 55.6461253,\n      "lon": 37.6121285,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288820295",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №8",\n      "lat": 55.6055779,\n      "lon": 37.5270247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288820296",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №9",\n      "lat": 55.6058261,\n      "lon": 37.5258843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288938254",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2514",\n      "lat": 55.6560092,\n      "lon": 37.5883948,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288938260",\n      "kind": "school",\n      "name": "Школа № 2042",\n      "lat": 55.6567778,\n      "lon": 37.588673,\n      "opening_hours": null\n    },\n    {\n      "id": "way/289210857",\n      "kind": "kindergarten",\n      "name": "Детский сад №1343",\n      "lat": 55.7030274,\n      "lon": 37.7468863,\n      "opening_hours": null\n    },\n    {\n      "id": "way/289860985",\n      "kind": "school",\n      "name": "Школа № 1212. Корпус школьного образования №4",\n      "lat": 55.6040732,\n      "lon": 37.5286177,\n      "opening_hours": null\n    },\n    {\n      "id": "way/290495398",\n      "kind": "school",\n      "name": "Школа № 1212. Корпус школьного образования №3",\n      "lat": 55.6015108,\n      "lon": 37.5267612,\n      "opening_hours": null\n    },\n    {\n      "id": "way/290650733",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №6",\n      "lat": 55.6017092,\n      "lon": 37.523109,\n      "opening_hours": null\n    },\n    {\n      "id": "way/290650734",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №7",\n      "lat": 55.6014793,\n      "lon": 37.5241883,\n      "opening_hours": null\n    },\n    {\n      "id": "way/291091227",\n      "kind": "kindergarten",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 7.",\n      "lat": 55.6527581,\n      "lon": 37.5214677,\n      "opening_hours": null\n    },\n    {\n      "id": "way/291431905",\n      "kind": "school",\n      "name": "Школа №937",\n      "lat": 55.6192688,\n      "lon": 37.7003505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/291677793",\n      "kind": "kindergarten",\n      "name": "Детский сад № 391",\n      "lat": 55.7805982,\n      "lon": 37.4734718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/293518110",\n      "kind": "school",\n      "name": "Школа №556. Корпус школьного образования",\n      "lat": 55.6172075,\n      "lon": 37.5853168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294746097",\n      "kind": "school",\n      "name": "ГБОУ Школа №1476, корпус №1",\n      "lat": 55.7771975,\n      "lon": 37.8315407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294746098",\n      "kind": "school",\n      "name": "ГБОУ Школа №1476, корпус №2",\n      "lat": 55.7760923,\n      "lon": 37.8264817,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294767909",\n      "kind": "kindergarten",\n      "name": "Детский сад 1944",\n      "lat": 55.681542,\n      "lon": 37.5203513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294767910",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2186",\n      "lat": 55.6775543,\n      "lon": 37.5266007,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294767923",\n      "kind": "school",\n      "name": "Москвич",\n      "lat": 55.6780027,\n      "lon": 37.5272994,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294767929",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6755472,\n      "lon": 37.5235789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/295165515",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 9",\n      "lat": 55.7786723,\n      "lon": 37.5154246,\n      "opening_hours": null\n    },\n    {\n      "id": "way/295308773",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Самсон\\"",\n      "lat": 55.646955,\n      "lon": 37.6118135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/296404036",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8227432,\n      "lon": 37.5133024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/296470165",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №5",\n      "lat": 55.6509066,\n      "lon": 37.6144295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/296766987",\n      "kind": "school",\n      "name": "Школа №726",\n      "lat": 55.8178257,\n      "lon": 37.5071475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/296899004",\n      "kind": "school",\n      "name": "Центр образования №1685",\n      "lat": 55.7488656,\n      "lon": 37.6510548,\n      "opening_hours": null\n    },\n    {\n      "id": "way/297179085",\n      "kind": "kindergarten",\n      "name": "Детский сад №1162",\n      "lat": 55.7500579,\n      "lon": 37.6780667,\n      "opening_hours": null\n    },\n    {\n      "id": "way/297419999",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7057586,\n      "lon": 37.7829519,\n      "opening_hours": null\n    },\n    {\n      "id": "way/297787530",\n      "kind": "kindergarten",\n      "name": "Школа № 224. Дошкольное отделение",\n      "lat": 55.8359911,\n      "lon": 37.4867076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/299334055",\n      "kind": "school",\n      "name": "Гимназия №1583",\n      "lat": 55.8444754,\n      "lon": 37.4995778,\n      "opening_hours": null\n    },\n    {\n      "id": "way/299439198",\n      "kind": "kindergarten",\n      "name": "Детский сад №2",\n      "lat": 55.8435487,\n      "lon": 37.4932647,\n      "opening_hours": null\n    },\n    {\n      "id": "way/299618941",\n      "kind": "kindergarten",\n      "name": "Школа №1282 \\"Сокольники\\" (дошкольное отделение 6)",\n      "lat": 55.7907656,\n      "lon": 37.6631069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/299959836",\n      "kind": "kindergarten",\n      "name": "Детский сад 1432",\n      "lat": 55.6804001,\n      "lon": 37.5207961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/300862782",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6785969,\n      "lon": 37.5165494,\n      "opening_hours": null\n    },\n    {\n      "id": "way/301225747",\n      "kind": "school",\n      "name": "Технорама на Юго-Востоке",\n      "lat": 55.714684,\n      "lon": 37.7804105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/301257076",\n      "kind": "kindergarten",\n      "name": "Детский сад №436",\n      "lat": 55.7140392,\n      "lon": 37.7743356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/301261065",\n      "kind": "kindergarten",\n      "name": "Детский сад №2608",\n      "lat": 55.714085,\n      "lon": 37.7736867,\n      "opening_hours": null\n    },\n    {\n      "id": "way/301827317",\n      "kind": "kindergarten",\n      "name": "Школа № 1315. Дошкольный корпус",\n      "lat": 55.8488197,\n      "lon": 37.482704,\n      "opening_hours": null\n    },\n    {\n      "id": "way/302728535",\n      "kind": "school",\n      "name": "Кадетская школа №1783",\n      "lat": 55.8174951,\n      "lon": 37.627102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/302731045",\n      "kind": "kindergarten",\n      "name": "ГБОУ Многопрофильная школа № 1220 Дошкольное отделение № 7",\n      "lat": 55.8164639,\n      "lon": 37.6280538,\n      "opening_hours": null\n    },\n    {\n      "id": "way/302731046",\n      "kind": "school",\n      "name": "Гимназия №1518",\n      "lat": 55.8166099,\n      "lon": 37.6294587,\n      "opening_hours": null\n    },\n    {\n      "id": "way/302874395",\n      "kind": "kindergarten",\n      "name": "Школа № 1220. Дошкольное отделение 7",\n      "lat": 55.8155766,\n      "lon": 37.6260994,\n      "opening_hours": null\n    },\n    {\n      "id": "way/304423695",\n      "kind": "school",\n      "name": "Школа №7",\n      "lat": 55.6833967,\n      "lon": 37.5241144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/304423701",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад №1620",\n      "lat": 55.6829149,\n      "lon": 37.5246628,\n      "opening_hours": null\n    },\n    {\n      "id": "way/304943996",\n      "kind": "school",\n      "name": "Школа №1264",\n      "lat": 55.6749372,\n      "lon": 37.5288398,\n      "opening_hours": null\n    },\n    {\n      "id": "way/305509989",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1311",\n      "lat": 55.7874154,\n      "lon": 37.4621433,\n      "opening_hours": null\n    },\n    {\n      "id": "way/305512014",\n      "kind": "kindergarten",\n      "name": "Школа № 138. Дошкольный корпус № 9",\n      "lat": 55.7873837,\n      "lon": 37.4706843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/305671071",\n      "kind": "school",\n      "name": "Курчатовская школа \\"Школа-интернат №101 (для слабослышащих)\\"",\n      "lat": 55.7878736,\n      "lon": 37.4502528,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306230576",\n      "kind": "school",\n      "name": "Школа №547",\n      "lat": 55.7090957,\n      "lon": 37.619606,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306239412",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР №208",\n      "lat": 55.710361,\n      "lon": 37.61135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306239414",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Роще\\"",\n      "lat": 55.7104506,\n      "lon": 37.6125525,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306457185",\n      "kind": "school",\n      "name": "Школа № 14. Корпус школьного образования №1",\n      "lat": 55.662527,\n      "lon": 37.4657222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306457186",\n      "kind": "school",\n      "name": "Школа № 14. Корпус школьного образования №2",\n      "lat": 55.6588522,\n      "lon": 37.4673021,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306467403",\n      "kind": "school",\n      "name": "Ломоносовская школа",\n      "lat": 55.7144431,\n      "lon": 37.4559731,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306467404",\n      "kind": "school",\n      "name": "Ломоносовская школа",\n      "lat": 55.7133511,\n      "lon": 37.4564043,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306467405",\n      "kind": "school",\n      "name": "Школа №97",\n      "lat": 55.7122262,\n      "lon": 37.4562088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306577777",\n      "kind": "kindergarten",\n      "name": "Детский сад №407",\n      "lat": 55.6614087,\n      "lon": 37.5182497,\n      "opening_hours": null\n    },\n    {\n      "id": "way/307231038",\n      "kind": "school",\n      "name": "Классическое образование",\n      "lat": 55.7184322,\n      "lon": 37.4565557,\n      "opening_hours": null\n    },\n    {\n      "id": "way/307912086",\n      "kind": "kindergarten",\n      "name": "Детский сад №1708",\n      "lat": 55.6799912,\n      "lon": 37.5371016,\n      "opening_hours": null\n    },\n    {\n      "id": "way/307912089",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6744783,\n      "lon": 37.5335792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/307912091",\n      "kind": "school",\n      "name": "English Playschool Moscow",\n      "lat": 55.6790374,\n      "lon": 37.5366821,\n      "opening_hours": null\n    },\n    {\n      "id": "way/309031590",\n      "kind": "school",\n      "name": "Школа №830 корпус №1",\n      "lat": 55.8330794,\n      "lon": 37.4596075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/309031610",\n      "kind": "kindergarten",\n      "name": "Школа №830 дошкольное отделение Росток",\n      "lat": 55.8335138,\n      "lon": 37.4582196,\n      "opening_hours": null\n    },\n    {\n      "id": "way/309413338",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7661583,\n      "lon": 37.7295241,\n      "opening_hours": null\n    },\n    {\n      "id": "way/310484266",\n      "kind": "kindergarten",\n      "name": "Школа № 1989. Дошкольное отделение № 1",\n      "lat": 55.6635856,\n      "lon": 37.7780625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/311745513",\n      "kind": "school",\n      "name": "Лицей № 1547",\n      "lat": 55.6623122,\n      "lon": 37.7768224,\n      "opening_hours": null\n    },\n    {\n      "id": "way/311874351",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6526951,\n      "lon": 37.7160265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/313374880",\n      "kind": "kindergarten",\n      "name": "Детский сад №1798 \\"Марьино\\"",\n      "lat": 55.6497054,\n      "lon": 37.7328339,\n      "opening_hours": null\n    },\n    {\n      "id": "way/313678454",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №02",\n      "lat": 55.6178991,\n      "lon": 37.493707,\n      "opening_hours": null\n    },\n    {\n      "id": "way/313972931",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7435428,\n      "lon": 37.8014098,\n      "opening_hours": null\n    },\n    {\n      "id": "way/314426206",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №6",\n      "lat": 55.6062815,\n      "lon": 37.5498466,\n      "opening_hours": null\n    },\n    {\n      "id": "way/314889731",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №4 «Софийский»",\n      "lat": 55.5948356,\n      "lon": 37.5376778,\n      "opening_hours": null\n    },\n    {\n      "id": "way/314889732",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №5 «Екатерининский»",\n      "lat": 55.5950594,\n      "lon": 37.5365957,\n      "opening_hours": null\n    },\n    {\n      "id": "way/315006942",\n      "kind": "school",\n      "name": "Британская Международная школа",\n      "lat": 55.5935377,\n      "lon": 37.5437821,\n      "opening_hours": null\n    },\n    {\n      "id": "way/316630443",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1661",\n      "lat": 55.6841287,\n      "lon": 37.5632973,\n      "opening_hours": null\n    },\n    {\n      "id": "way/316835898",\n      "kind": "kindergarten",\n      "name": "Детский сад №2241",\n      "lat": 55.6924597,\n      "lon": 37.6103943,\n      "opening_hours": null\n    },\n    {\n      "id": "way/317239775",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №10",\n      "lat": 55.5990282,\n      "lon": 37.6098207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/317584503",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, Гномики (здание №10)",\n      "lat": 55.6776182,\n      "lon": 37.6703317,\n      "opening_hours": null\n    },\n    {\n      "id": "way/317792794",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №5",\n      "lat": 55.5995573,\n      "lon": 37.6192564,\n      "opening_hours": null\n    },\n    {\n      "id": "way/317792796",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5996094,\n      "lon": 37.6170891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/318098302",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №7",\n      "lat": 55.5982049,\n      "lon": 37.6149972,\n      "opening_hours": null\n    },\n    {\n      "id": "way/318098304",\n      "kind": "kindergarten",\n      "name": "Детский сад №725",\n      "lat": 55.5979473,\n      "lon": 37.617163,\n      "opening_hours": null\n    },\n    {\n      "id": "way/318866391",\n      "kind": "kindergarten",\n      "name": "Школа №654 имени А. Д. Фридмана. Дошкольное отделение №2",\n      "lat": 55.6987433,\n      "lon": 37.7455939,\n      "opening_hours": null\n    },\n    {\n      "id": "way/318866392",\n      "kind": "school",\n      "name": "Школа №355",\n      "lat": 55.6951349,\n      "lon": 37.8230844,\n      "opening_hours": null\n    },\n    {\n      "id": "way/319196588",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8104228,\n      "lon": 37.5716682,\n      "opening_hours": null\n    },\n    {\n      "id": "way/319196606",\n      "kind": "school",\n      "name": "Школа № 1454",\n      "lat": 55.8134542,\n      "lon": 37.5688665,\n      "opening_hours": null\n    },\n    {\n      "id": "way/319902984",\n      "kind": "school",\n      "name": "Школа № 843",\n      "lat": 55.6711324,\n      "lon": 37.4579789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/320747994",\n      "kind": "school",\n      "name": "Школа №672",\n      "lat": 55.7495847,\n      "lon": 37.7822088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/321358664",\n      "kind": "school",\n      "name": "Гимназия №1798 \\"Феникс\\"",\n      "lat": 55.752896,\n      "lon": 37.8139613,\n      "opening_hours": null\n    },\n    {\n      "id": "way/322896806",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6023601,\n      "lon": 37.5349225,\n      "opening_hours": null\n    },\n    {\n      "id": "way/324740801",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Здание дошкольных групп 6",\n      "lat": 55.7226459,\n      "lon": 37.4710084,\n      "opening_hours": null\n    },\n    {\n      "id": "way/325925347",\n      "kind": "kindergarten",\n      "name": "Детский сад № 489",\n      "lat": 55.6492145,\n      "lon": 37.5817353,\n      "opening_hours": null\n    },\n    {\n      "id": "way/325989403",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В. А. Молодцова, дополнительное образование",\n      "lat": 55.8761052,\n      "lon": 37.6320678,\n      "opening_hours": null\n    },\n    {\n      "id": "way/326986434",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6869049,\n      "lon": 37.5799478,\n      "opening_hours": null\n    },\n    {\n      "id": "way/326986435",\n      "kind": "school",\n      "name": "Гимназия №1534 Школьное отделение №4",\n      "lat": 55.6877374,\n      "lon": 37.5812575,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329291176",\n      "kind": "school",\n      "name": "Гимназия №1551 корпус №1 (полное общее и гимназическое)",\n      "lat": 55.8467929,\n      "lon": 37.4564324,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329291638",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №6 (дошкольное отделение)",\n      "lat": 55.8479145,\n      "lon": 37.4564773,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329353098",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №7 (дошкольное отделение)",\n      "lat": 55.8449866,\n      "lon": 37.4569229,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329535556",\n      "kind": "school",\n      "name": "Школа № 1561. Корпус №3",\n      "lat": 55.6002397,\n      "lon": 37.533737,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329630118",\n      "kind": "school",\n      "name": "Школа № 1561. Корпус №1",\n      "lat": 55.6007025,\n      "lon": 37.5374328,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329630119",\n      "kind": "kindergarten",\n      "name": "Школа № 1561. Дошкольное отделение корпус №7",\n      "lat": 55.599153,\n      "lon": 37.5352235,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329630120",\n      "kind": "kindergarten",\n      "name": "Школа № 1561. Дошкольное отделение корпус №9",\n      "lat": 55.5989161,\n      "lon": 37.5363265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329889832",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6811381,\n      "lon": 37.5649205,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330589924",\n      "kind": "kindergarten",\n      "name": "Детский сад №1614",\n      "lat": 55.6675506,\n      "lon": 37.7457501,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330589925",\n      "kind": "kindergarten",\n      "name": "Центр образования №1877 \\"Люблино\\" - дошкольное отделение",\n      "lat": 55.6687898,\n      "lon": 37.7479076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330589926",\n      "kind": "school",\n      "name": "Центр образования №1877 \\"Люблино\\"",\n      "lat": 55.6685781,\n      "lon": 37.7461851,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330781480",\n      "kind": "school",\n      "name": "Школа №728",\n      "lat": 55.629689,\n      "lon": 37.6075114,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330782053",\n      "kind": "kindergarten",\n      "name": "Детский сад №668",\n      "lat": 55.6294453,\n      "lon": 37.6101111,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330783273",\n      "kind": "school",\n      "name": "Школа №851",\n      "lat": 55.6279349,\n      "lon": 37.6062266,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331403705",\n      "kind": "school",\n      "name": "Лицей №138, отделение начальная школа",\n      "lat": 55.782451,\n      "lon": 37.4665567,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445723",\n      "kind": "kindergarten",\n      "name": "Детский сад №1375",\n      "lat": 55.673436,\n      "lon": 37.7693615,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445725",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6714522,\n      "lon": 37.7709225,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445726",\n      "kind": "kindergarten",\n      "name": "Лицей № 2010. Дошкольное отделение Радуга",\n      "lat": 55.6717546,\n      "lon": 37.7631003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445728",\n      "kind": "school",\n      "name": "Центр образования №2010",\n      "lat": 55.6725204,\n      "lon": 37.7641608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445729",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6694269,\n      "lon": 37.7604846,\n      "opening_hours": null\n    },\n    {\n      "id": "way/332105703",\n      "kind": "school",\n      "name": "Гимназия №1576",\n      "lat": 55.8229834,\n      "lon": 37.5291763,\n      "opening_hours": null\n    },\n    {\n      "id": "way/332122466",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 3",\n      "lat": 55.7735963,\n      "lon": 37.542007,\n      "opening_hours": null\n    },\n    {\n      "id": "way/332446075",\n      "kind": "kindergarten",\n      "name": "Школа № 1561. Дошкольное отделение корпус №8",\n      "lat": 55.6029686,\n      "lon": 37.5393923,\n      "opening_hours": null\n    },\n    {\n      "id": "way/332446076",\n      "kind": "school",\n      "name": "Школа № 1561. Начальная школа - корпус №10",\n      "lat": 55.6031895,\n      "lon": 37.5383104,\n      "opening_hours": null\n    },\n    {\n      "id": "way/333465880",\n      "kind": "kindergarten",\n      "name": "Детский сад №2456",\n      "lat": 55.643294,\n      "lon": 37.5987303,\n      "opening_hours": null\n    },\n    {\n      "id": "way/333465883",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2456",\n      "lat": 55.6444922,\n      "lon": 37.5966251,\n      "opening_hours": null\n    },\n    {\n      "id": "way/333533055",\n      "kind": "school",\n      "name": "Школа № 2126 \\"Перово\\"",\n      "lat": 55.741556,\n      "lon": 37.7625178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/334253493",\n      "kind": "school",\n      "name": "Школа №958",\n      "lat": 55.8687633,\n      "lon": 37.6091067,\n      "opening_hours": null\n    },\n    {\n      "id": "way/334458773",\n      "kind": "school",\n      "name": "ГБУ ЦССВ \\"Наш дом\\"",\n      "lat": 55.7514212,\n      "lon": 37.5036492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/334658366",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6783464,\n      "lon": 37.568421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/335073907",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 5",\n      "lat": 55.6687127,\n      "lon": 37.5693168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/335195464",\n      "kind": "school",\n      "name": "Школа № 15. Корпус школьного образования № 4",\n      "lat": 55.661952,\n      "lon": 37.5704672,\n      "opening_hours": null\n    },\n    {\n      "id": "way/335317974",\n      "kind": "kindergarten",\n      "name": "Детский сад №574 ФГУП ГосНИИАС",\n      "lat": 55.7958354,\n      "lon": 37.5232028,\n      "opening_hours": "Mo-Fr 07:30-19:30"\n    },\n    {\n      "id": "way/335598252",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 6",\n      "lat": 55.6640312,\n      "lon": 37.5688032,\n      "opening_hours": null\n    },\n    {\n      "id": "way/336472571",\n      "kind": "school",\n      "name": "Школа № 1329. Учебный корпус № 2",\n      "lat": 55.6735197,\n      "lon": 37.4731658,\n      "opening_hours": null\n    },\n    {\n      "id": "way/336720753",\n      "kind": "school",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ШП-2)",\n      "lat": 55.6725105,\n      "lon": 37.7701674,\n      "opening_hours": null\n    },\n    {\n      "id": "way/336883714",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 5",\n      "lat": 55.6617853,\n      "lon": 37.5678305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/336890069",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 8",\n      "lat": 55.66635,\n      "lon": 37.568822,\n      "opening_hours": null\n    },\n    {\n      "id": "way/337152319",\n      "kind": "kindergarten",\n      "name": "Детский сад №2419",\n      "lat": 55.7554655,\n      "lon": 37.8170242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/338052419",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №7",\n      "lat": 55.6454751,\n      "lon": 37.5118452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/339123293",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 4",\n      "lat": 55.6713752,\n      "lon": 37.5733048,\n      "opening_hours": null\n    },\n    {\n      "id": "way/339923948",\n      "kind": "school",\n      "name": "English International School",\n      "lat": 55.7491892,\n      "lon": 37.8224728,\n      "opening_hours": null\n    },\n    {\n      "id": "way/340910737",\n      "kind": "kindergarten",\n      "name": "Детский сад №1290",\n      "lat": 55.7481424,\n      "lon": 37.8244796,\n      "opening_hours": null\n    },\n    {\n      "id": "way/343922114",\n      "kind": "school",\n      "name": "Школа №54",\n      "lat": 55.7237568,\n      "lon": 37.5703509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/343924854",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7234189,\n      "lon": 37.5590935,\n      "opening_hours": null\n    },\n    {\n      "id": "way/345118620",\n      "kind": "school",\n      "name": "Центр образования \\"Личность\\"",\n      "lat": 55.7193582,\n      "lon": 37.7892985,\n      "opening_hours": null\n    },\n    {\n      "id": "way/345628413",\n      "kind": "school",\n      "name": "Лицей №138, отделение средняя школа",\n      "lat": 55.7810707,\n      "lon": 37.4673229,\n      "opening_hours": null\n    },\n    {\n      "id": "way/349929585",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №152 СП №2",\n      "lat": 55.8090438,\n      "lon": 37.5232553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350090313",\n      "kind": "school",\n      "name": "Лицей №1575",\n      "lat": 55.8013012,\n      "lon": 37.5469938,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350097515",\n      "kind": "kindergarten",\n      "name": "Инженерно-техническая школа имени дважды Героя Советского Союза П.Р. Поповича",\n      "lat": 55.8076753,\n      "lon": 37.5367688,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350100413",\n      "kind": "kindergarten",\n      "name": "ГБОУ г. Москвы \\"Инженерно-техническая школа имени дважды Героя Советского Союза П.Р. Поповича\\"",\n      "lat": 55.8096465,\n      "lon": 37.536859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350105151",\n      "kind": "school",\n      "name": "Школа №152",\n      "lat": 55.7958955,\n      "lon": 37.5452135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350320195",\n      "kind": "kindergarten",\n      "name": "Школа №1420. Корпус №11",\n      "lat": 55.6991996,\n      "lon": 37.8229847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/352135092",\n      "kind": "school",\n      "name": "Школа № 15. Корпус школьного образования № 3",\n      "lat": 55.6604236,\n      "lon": 37.5693012,\n      "opening_hours": null\n    },\n    {\n      "id": "way/352807461",\n      "kind": "kindergarten",\n      "name": "Детский сад № 421",\n      "lat": 55.6606324,\n      "lon": 37.5708153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/353049778",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 4",\n      "lat": 55.6620471,\n      "lon": 37.573247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/353198790",\n      "kind": "kindergarten",\n      "name": "Детский сад №797",\n      "lat": 55.7313037,\n      "lon": 37.5350636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/353198792",\n      "kind": "school",\n      "name": "Школа № 12",\n      "lat": 55.7306206,\n      "lon": 37.5327473,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355262402",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Надежда»",\n      "lat": 55.6581095,\n      "lon": 37.4968887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355321688",\n      "kind": "school",\n      "name": "Центр образования № 1434",\n      "lat": 55.6907361,\n      "lon": 37.5003688,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355321689",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1565",\n      "lat": 55.6905524,\n      "lon": 37.4976442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355321690",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1544",\n      "lat": 55.6913973,\n      "lon": 37.49915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355466540",\n      "kind": "kindergarten",\n      "name": "Детский сад №226",\n      "lat": 55.8083912,\n      "lon": 37.5256277,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355662159",\n      "kind": "school",\n      "name": "Школа им. Хинксона",\n      "lat": 55.6748036,\n      "lon": 37.5765673,\n      "opening_hours": null\n    },\n    {\n      "id": "way/356023465",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 7",\n      "lat": 55.6664651,\n      "lon": 37.5722384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/358194039",\n      "kind": "school",\n      "name": "Школа №1352 (корпус на Щёлковском шс., 79А)",\n      "lat": 55.8130511,\n      "lon": 37.8077561,\n      "opening_hours": null\n    },\n    {\n      "id": "way/358616273",\n      "kind": "school",\n      "name": "Гимназия №1514",\n      "lat": 55.6850391,\n      "lon": 37.5265823,\n      "opening_hours": null\n    },\n    {\n      "id": "way/358616274",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад №1851",\n      "lat": 55.6858704,\n      "lon": 37.5272578,\n      "opening_hours": null\n    },\n    {\n      "id": "way/358807892",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7727287,\n      "lon": 37.5779719,\n      "opening_hours": null\n    },\n    {\n      "id": "way/359730497",\n      "kind": "school",\n      "name": "Школа № 49",\n      "lat": 55.6472878,\n      "lon": 37.5369015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/360180382",\n      "kind": "kindergarten",\n      "name": "Центр образования и спорта \\"Москва-98\\". Дошкольное отделение № 3",\n      "lat": 55.6838348,\n      "lon": 37.4840979,\n      "opening_hours": null\n    },\n    {\n      "id": "way/360180383",\n      "kind": "kindergarten",\n      "name": "Центр образования и спорта \\"Москва-98\\". Дошкольное отделение № 2",\n      "lat": 55.685457,\n      "lon": 37.4869063,\n      "opening_hours": null\n    },\n    {\n      "id": "way/360180385",\n      "kind": "school",\n      "name": "Центр образования и спорта \\"Москва-98\\". Школа № 3",\n      "lat": 55.6847896,\n      "lon": 37.4852296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/361018891",\n      "kind": "school",\n      "name": "Школа №494 им. А. Н. Рожкова",\n      "lat": 55.704784,\n      "lon": 37.6557062,\n      "opening_hours": null\n    },\n    {\n      "id": "way/361313197",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6658724,\n      "lon": 37.5549265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/362529929",\n      "kind": "kindergarten",\n      "name": "Set International School",\n      "lat": 55.7908035,\n      "lon": 37.7871961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/362924386",\n      "kind": "school",\n      "name": "Средняя школа №723",\n      "lat": 55.7917401,\n      "lon": 37.8128954,\n      "opening_hours": null\n    },\n    {\n      "id": "way/363405576",\n      "kind": "school",\n      "name": "Центр образования № 628",\n      "lat": 55.6681172,\n      "lon": 37.6083376,\n      "opening_hours": null\n    },\n    {\n      "id": "way/363467163",\n      "kind": "school",\n      "name": "Школа №423",\n      "lat": 55.7586268,\n      "lon": 37.7975102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/363470439",\n      "kind": "kindergarten",\n      "name": "Школа № 243. Дошкольные отделения",\n      "lat": 55.7632594,\n      "lon": 37.7897795,\n      "opening_hours": null\n    },\n    {\n      "id": "way/363943988",\n      "kind": "kindergarten",\n      "name": "Гимназия 1570 Дошкольное отделение",\n      "lat": 55.790683,\n      "lon": 37.5834186,\n      "opening_hours": null\n    },\n    {\n      "id": "way/364436597",\n      "kind": "kindergarten",\n      "name": "Детский сад №60",\n      "lat": 55.6677193,\n      "lon": 37.6173746,\n      "opening_hours": null\n    },\n    {\n      "id": "way/364719268",\n      "kind": "school",\n      "name": "Школа №1576",\n      "lat": 55.838709,\n      "lon": 37.5340906,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365268317",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №5",\n      "lat": 55.8533787,\n      "lon": 37.6884056,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365268318",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №4",\n      "lat": 55.8525706,\n      "lon": 37.6878097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365268319",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №3",\n      "lat": 55.8517964,\n      "lon": 37.6864865,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365355137",\n      "kind": "kindergarten",\n      "name": "Детский сад 1489",\n      "lat": 55.6795738,\n      "lon": 37.5166667,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365490472",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1431",\n      "lat": 55.6773901,\n      "lon": 37.5237762,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365533380",\n      "kind": "school",\n      "name": "Школа №1352 (корпус на Байкальской, 44)",\n      "lat": 55.8152539,\n      "lon": 37.8147714,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365668976",\n      "kind": "school",\n      "name": "ДШИ им. Ф.Шуберта",\n      "lat": 55.6773707,\n      "lon": 37.4956891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/366295826",\n      "kind": "school",\n      "name": "Школа №1253 (начальные классы)",\n      "lat": 55.7325984,\n      "lon": 37.5904657,\n      "opening_hours": null\n    },\n    {\n      "id": "way/366476184",\n      "kind": "school",\n      "name": "Школа № 7 филиал № 2",\n      "lat": 55.6783242,\n      "lon": 37.5222297,\n      "opening_hours": null\n    },\n    {\n      "id": "way/366476185",\n      "kind": "school",\n      "name": "Олимпия",\n      "lat": 55.6775725,\n      "lon": 37.5215053,\n      "opening_hours": null\n    },\n    {\n      "id": "way/366561360",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2466",\n      "lat": 55.6642355,\n      "lon": 37.7753399,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/366786670",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 7 «Радуга»",\n      "lat": 55.7655704,\n      "lon": 37.5736484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/367879924",\n      "kind": "school",\n      "name": "Школа №881",\n      "lat": 55.6018094,\n      "lon": 37.5839302,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368059893",\n      "kind": "kindergarten",\n      "name": "ДОУ №638",\n      "lat": 55.6797655,\n      "lon": 37.6610759,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368261204",\n      "kind": "school",\n      "name": "Школа № 1248. Школьное здание 2",\n      "lat": 55.7215291,\n      "lon": 37.4678696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368262333",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7250567,\n      "lon": 37.4648214,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368267185",\n      "kind": "school",\n      "name": "Школа №97",\n      "lat": 55.7204483,\n      "lon": 37.460397,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368275200",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6318409,\n      "lon": 37.5116148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368275202",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6313366,\n      "lon": 37.5131849,\n      "opening_hours": null\n    },\n    {\n      "id": "way/369050718",\n      "kind": "school",\n      "name": "НОУ \\"Британская международная школа\\"",\n      "lat": 55.6721614,\n      "lon": 37.5742829,\n      "opening_hours": null\n    },\n    {\n      "id": "way/369872117",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.686673,\n      "lon": 37.5806099,\n      "opening_hours": null\n    },\n    {\n      "id": "way/370556590",\n      "kind": "kindergarten",\n      "name": "Школа №51. Дошкольные группы",\n      "lat": 55.643659,\n      "lon": 37.4873683,\n      "opening_hours": null\n    },\n    {\n      "id": "way/370556591",\n      "kind": "kindergarten",\n      "name": "Школа №51. Дошкольные группы",\n      "lat": 55.6427025,\n      "lon": 37.4854944,\n      "opening_hours": null\n    },\n    {\n      "id": "way/371851060",\n      "kind": "kindergarten",\n      "name": "Школа №1352 (дошкольная группа на Уральской, 6)",\n      "lat": 55.8125084,\n      "lon": 37.8006892,\n      "opening_hours": null\n    },\n    {\n      "id": "way/371930812",\n      "kind": "school",\n      "name": "Вальдорфская семейная инициатива «Школа под солнцем»",\n      "lat": 55.6286849,\n      "lon": 37.4534843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/372219915",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Единство»",\n      "lat": 55.6375279,\n      "lon": 37.5930864,\n      "opening_hours": null\n    },\n    {\n      "id": "way/372687745",\n      "kind": "school",\n      "name": "Школа № 1228. Корпус № 2",\n      "lat": 55.7539392,\n      "lon": 37.7113082,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373222414",\n      "kind": "kindergarten",\n      "name": "Детский сад №288",\n      "lat": 55.7396176,\n      "lon": 37.6828883,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373614960",\n      "kind": "kindergarten",\n      "name": "Гимназия № 625 Дошкольное отделение Созвездие",\n      "lat": 55.6908704,\n      "lon": 37.5925088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373734133",\n      "kind": "kindergarten",\n      "name": "Детский сад № 775",\n      "lat": 55.6851542,\n      "lon": 37.58927,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373783441",\n      "kind": "school",\n      "name": "Школа №962",\n      "lat": 55.8596609,\n      "lon": 37.5934956,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373873844",\n      "kind": "school",\n      "name": "Школа №1516 (Основной корпус)",\n      "lat": 55.8166613,\n      "lon": 37.831409,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374145748",\n      "kind": "kindergarten",\n      "name": "Школа №1078 (Модуль №4)",\n      "lat": 55.8187335,\n      "lon": 37.8060185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374145750",\n      "kind": "school",\n      "name": "Школа №1078 (Модуль №1)",\n      "lat": 55.8186512,\n      "lon": 37.8079475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374185846",\n      "kind": "school",\n      "name": "Школа №49. Школьный корпус",\n      "lat": 55.6436293,\n      "lon": 37.5340588,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374260412",\n      "kind": "kindergarten",\n      "name": "Детский сад № 737",\n      "lat": 55.6434527,\n      "lon": 37.5365573,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374271122",\n      "kind": "kindergarten",\n      "name": "Академическая гимназия",\n      "lat": 55.6843339,\n      "lon": 37.5943692,\n      "opening_hours": "Mo-Su 07:30-19:30"\n    },\n    {\n      "id": "way/374629533",\n      "kind": "school",\n      "name": "Школа №1076",\n      "lat": 55.8134114,\n      "lon": 37.8182652,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374939352",\n      "kind": "kindergarten",\n      "name": "Школа №109. Дошкольные группы",\n      "lat": 55.6416672,\n      "lon": 37.4762207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374951049",\n      "kind": "kindergarten",\n      "name": "Школа №1995. Дошкольный уровень",\n      "lat": 55.6433273,\n      "lon": 37.4798488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374958831",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Замок детства\\"",\n      "lat": 55.5841738,\n      "lon": 37.7334101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/376499093",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8893178,\n      "lon": 37.6585919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708650",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6731937,\n      "lon": 37.4529157,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708651",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6767323,\n      "lon": 37.4488698,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708652",\n      "kind": "school",\n      "name": "Школа № 2025",\n      "lat": 55.6777208,\n      "lon": 37.4462442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708653",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6883637,\n      "lon": 37.4667261,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708654",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6754746,\n      "lon": 37.4481651,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377728857",\n      "kind": "school",\n      "name": "Школа №1173",\n      "lat": 55.6168244,\n      "lon": 37.5945999,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378038130",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6694457,\n      "lon": 37.5758566,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378124074",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7554576,\n      "lon": 37.5699743,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378186606",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 9",\n      "lat": 55.8549232,\n      "lon": 37.6410485,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378186608",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 7",\n      "lat": 55.8544926,\n      "lon": 37.6433601,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378418006",\n      "kind": "school",\n      "name": "Школа «Содружество». Учебный корпус №3",\n      "lat": 55.7635152,\n      "lon": 37.7032461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378894408",\n      "kind": "school",\n      "name": "Школа № 967, школьное отделение №2",\n      "lat": 55.8803451,\n      "lon": 37.6421581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378894409",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №6",\n      "lat": 55.8788117,\n      "lon": 37.641468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/379502769",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6613381,\n      "lon": 37.5940904,\n      "opening_hours": null\n    },\n    {\n      "id": "way/379502770",\n      "kind": "kindergarten",\n      "name": "Детский сад №2514",\n      "lat": 55.660408,\n      "lon": 37.5927616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/382029609",\n      "kind": "kindergarten",\n      "name": "Детский сад №2021",\n      "lat": 55.7141857,\n      "lon": 37.6117815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/383149167",\n      "kind": "kindergarten",\n      "name": "Детский сад №1899",\n      "lat": 55.7493118,\n      "lon": 37.8196775,\n      "opening_hours": null\n    },\n    {\n      "id": "way/383157041",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7460199,\n      "lon": 37.8206962,\n      "opening_hours": null\n    },\n    {\n      "id": "way/383319619",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7685951,\n      "lon": 37.5695966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/385816697",\n      "kind": "school",\n      "name": "Школа № 461",\n      "lat": 55.70263,\n      "lon": 37.7692074,\n      "opening_hours": null\n    },\n    {\n      "id": "way/385965951",\n      "kind": "school",\n      "name": "Школа №368 (Здание №2)",\n      "lat": 55.8222076,\n      "lon": 37.8200132,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386178150",\n      "kind": "school",\n      "name": "Школа №1411",\n      "lat": 55.8669381,\n      "lon": 37.6007503,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386345936",\n      "kind": "kindergarten",\n      "name": "Школа «Содружество». Учебный корпус №5",\n      "lat": 55.758656,\n      "lon": 37.7135594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386779191",\n      "kind": "school",\n      "name": "Школа № 1514",\n      "lat": 55.6865191,\n      "lon": 37.5292895,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386779744",\n      "kind": "kindergarten",\n      "name": "Детский сад №63",\n      "lat": 55.7989267,\n      "lon": 37.6201377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386906994",\n      "kind": "kindergarten",\n      "name": "Школа №760 им. А. П. Маресьева. Дошкольное отделение",\n      "lat": 55.8762201,\n      "lon": 37.7101421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387057250",\n      "kind": "kindergarten",\n      "name": "Детский сад №1903",\n      "lat": 55.7433006,\n      "lon": 37.7622113,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387463901",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1626",\n      "lat": 55.8611164,\n      "lon": 37.5940337,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387463902",\n      "kind": "kindergarten",\n      "name": "Школа № 962. Дошкольное отделение",\n      "lat": 55.8616045,\n      "lon": 37.5925913,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387467935",\n      "kind": "kindergarten",\n      "name": "Школа №1310 (дошкольный корпус №3)",\n      "lat": 55.743555,\n      "lon": 37.7801359,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387467936",\n      "kind": "kindergarten",\n      "name": "Школа №1310 (дошкольный корпус №2)",\n      "lat": 55.7427871,\n      "lon": 37.7801718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/388407978",\n      "kind": "kindergarten",\n      "name": "Школа №1352 (дошкольная группа на Чусовской, 6)",\n      "lat": 55.8132975,\n      "lon": 37.815292,\n      "opening_hours": null\n    },\n    {\n      "id": "way/389025261",\n      "kind": "kindergarten",\n      "name": "Детский сад № 682 Аленький цветочек",\n      "lat": 55.7384988,\n      "lon": 37.4696899,\n      "opening_hours": null\n    },\n    {\n      "id": "way/389190468",\n      "kind": "kindergarten",\n      "name": "Школа на проспекте Вернадского. Корпус дошкольного образования 6",\n      "lat": 55.6589259,\n      "lon": 37.4842515,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390376376",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2476",\n      "lat": 55.6811384,\n      "lon": 37.4511423,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390483182",\n      "kind": "kindergarten",\n      "name": "Школа № 629. Корпус дошкольного образования №3",\n      "lat": 55.5890295,\n      "lon": 37.5958941,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390483183",\n      "kind": "school",\n      "name": "Школа № 629. Корпус школьного образования №5",\n      "lat": 55.5876227,\n      "lon": 37.5950827,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390483184",\n      "kind": "school",\n      "name": "Школа № 629. Корпус школьного образования №4",\n      "lat": 55.591432,\n      "lon": 37.5965467,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390483185",\n      "kind": "kindergarten",\n      "name": "Школа № 629. Корпус дошкольного образования №2",\n      "lat": 55.5923739,\n      "lon": 37.5940998,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390485001",\n      "kind": "kindergarten",\n      "name": "Детский сад №1431",\n      "lat": 55.8593986,\n      "lon": 37.6705996,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390485005",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8596248,\n      "lon": 37.6686069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/393266311",\n      "kind": "kindergarten",\n      "name": "Школа № 1474, дошкольное отделение",\n      "lat": 55.8735371,\n      "lon": 37.4915173,\n      "opening_hours": null\n    },\n    {\n      "id": "way/394363818",\n      "kind": "school",\n      "name": "Школа №760 им. А. П. Маресьева. Школьное отделение",\n      "lat": 55.87765,\n      "lon": 37.7149576,\n      "opening_hours": null\n    },\n    {\n      "id": "way/394589999",\n      "kind": "school",\n      "name": "Школа №760 им. А. П. Маресьева. Школьное отделение",\n      "lat": 55.8773705,\n      "lon": 37.7111649,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395166265",\n      "kind": "kindergarten",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 14",\n      "lat": 55.7964263,\n      "lon": 37.8227056,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395166267",\n      "kind": "school",\n      "name": "Училище олимпийского резерва № 1",\n      "lat": 55.7981421,\n      "lon": 37.8219185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395236550",\n      "kind": "kindergarten",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 12",\n      "lat": 55.7985003,\n      "lon": 37.8201491,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395236552",\n      "kind": "kindergarten",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 16",\n      "lat": 55.7956005,\n      "lon": 37.8157633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395236553",\n      "kind": "school",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 4",\n      "lat": 55.7962876,\n      "lon": 37.8157306,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395430224",\n      "kind": "school",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпуc №3 (для детей с особыми образовательными потребностями)",\n      "lat": 55.7991279,\n      "lon": 37.8259144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395430226",\n      "kind": "school",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус 2",\n      "lat": 55.8023486,\n      "lon": 37.8332336,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395756677",\n      "kind": "kindergarten",\n      "name": "Детский сад №423",\n      "lat": 55.5993408,\n      "lon": 37.6558297,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396444371",\n      "kind": "kindergarten",\n      "name": "Школа самоопределения № 734 имени А. Н. Тубельского (корпус №2)",\n      "lat": 55.7997801,\n      "lon": 37.8215733,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396444372",\n      "kind": "kindergarten",\n      "name": "Школа №1811 (дошкольный корпус №10)",\n      "lat": 55.8003633,\n      "lon": 37.8176821,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396444374",\n      "kind": "school",\n      "name": "Школа №1811 (школьный корпус №6)",\n      "lat": 55.8001185,\n      "lon": 37.8155549,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396444375",\n      "kind": "school",\n      "name": "Школа самоопределения № 734 имени А. Н. Тубельского (корпус №1)",\n      "lat": 55.8011975,\n      "lon": 37.8240129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396864913",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Сказка\\"",\n      "lat": 55.8750218,\n      "lon": 37.7107411,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396923766",\n      "kind": "school",\n      "name": "Школа № 1245. Корпус школьного образования №2",\n      "lat": 55.5994146,\n      "lon": 37.5954815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749563",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2148",\n      "lat": 55.8698656,\n      "lon": 37.6087129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749564",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1554. Дошкольное отделение",\n      "lat": 55.8694,\n      "lon": 37.6056915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749565",\n      "kind": "kindergarten",\n      "name": "Школа № 1411. Структурное подразделение № 978",\n      "lat": 55.8685277,\n      "lon": 37.6030916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749567",\n      "kind": "school",\n      "name": "Детско-юношеский центр «Отрадное»",\n      "lat": 55.870072,\n      "lon": 37.6150961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749568",\n      "kind": "kindergarten",\n      "name": "Начальная школа -детский сад №1879",\n      "lat": 55.8709316,\n      "lon": 37.6144368,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397934095",\n      "kind": "kindergarten",\n      "name": "Детский сад №2495",\n      "lat": 55.670626,\n      "lon": 37.757475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/398512694",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8658295,\n      "lon": 37.4871452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/398512988",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8625257,\n      "lon": 37.4883318,\n      "opening_hours": null\n    },\n    {\n      "id": "way/398981887",\n      "kind": "kindergarten",\n      "name": "Школа №368 (Здание №4)",\n      "lat": 55.8217372,\n      "lon": 37.81568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399065542",\n      "kind": "school",\n      "name": "Хорошёвская прогимназия",\n      "lat": 55.7835468,\n      "lon": 37.4657546,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399258980",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5922383,\n      "lon": 37.6061252,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399276902",\n      "kind": "kindergarten",\n      "name": "Детский сад №2640",\n      "lat": 55.7442809,\n      "lon": 37.6822947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399584298",\n      "kind": "school",\n      "name": "Школа № 1518",\n      "lat": 55.8036181,\n      "lon": 37.6347402,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399595392",\n      "kind": "kindergarten",\n      "name": "Детский сад №957",\n      "lat": 55.6163535,\n      "lon": 37.5970222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399595394",\n      "kind": "kindergarten",\n      "name": "Детский сад № 897",\n      "lat": 55.618628,\n      "lon": 37.5886996,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399595402",\n      "kind": "school",\n      "name": "Школа №556. Корпус начального образования",\n      "lat": 55.6208187,\n      "lon": 37.5858025,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400514902",\n      "kind": "school",\n      "name": "Школа № 1582. Корпус школьного образования",\n      "lat": 55.6183662,\n      "lon": 37.5971795,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400616531",\n      "kind": "school",\n      "name": "Финансово-экономическая школа",\n      "lat": 55.8242532,\n      "lon": 37.5821131,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400702131",\n      "kind": "kindergarten",\n      "name": "Школа \\"Марьина роща\\" имени В.Ф. Фролова",\n      "lat": 55.8075121,\n      "lon": 37.6137209,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400717278",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8095632,\n      "lon": 37.6299132,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400717280",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8099871,\n      "lon": 37.619417,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400717281",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8097293,\n      "lon": 37.620565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400761385",\n      "kind": "school",\n      "name": "Школа № 1531 имени С. К. Годовикова",\n      "lat": 55.8038384,\n      "lon": 37.6327084,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400804215",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8751568,\n      "lon": 37.6295166,\n      "opening_hours": null\n    },\n    {\n      "id": "way/401282468",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №1",\n      "lat": 55.6193035,\n      "lon": 37.6005617,\n      "opening_hours": null\n    },\n    {\n      "id": "way/401282470",\n      "kind": "school",\n      "name": "Школа № 1582. Корпус школьного образования",\n      "lat": 55.6199163,\n      "lon": 37.5968862,\n      "opening_hours": null\n    },\n    {\n      "id": "way/401791217",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №2",\n      "lat": 55.6185972,\n      "lon": 37.6047427,\n      "opening_hours": null\n    },\n    {\n      "id": "way/401791223",\n      "kind": "school",\n      "name": "ДШИ \\"Тутти\\"",\n      "lat": 55.6127838,\n      "lon": 37.6005689,\n      "opening_hours": null\n    },\n    {\n      "id": "way/403258384",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №3",\n      "lat": 55.6174214,\n      "lon": 37.6040874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404189291",\n      "kind": "kindergarten",\n      "name": "Детский сад №1158",\n      "lat": 55.6278953,\n      "lon": 37.6081784,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404373621",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1820",\n      "lat": 55.5979379,\n      "lon": 37.5948011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404373622",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №6",\n      "lat": 55.5983338,\n      "lon": 37.5925095,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404379796",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №4",\n      "lat": 55.6036467,\n      "lon": 37.5955711,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404896628",\n      "kind": "kindergarten",\n      "name": "Детский сад №740",\n      "lat": 55.6003145,\n      "lon": 37.5842174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404896629",\n      "kind": "school",\n      "name": "Школа №932",\n      "lat": 55.5998095,\n      "lon": 37.5856551,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404896633",\n      "kind": "kindergarten",\n      "name": "Детский сад №1855",\n      "lat": 55.5984043,\n      "lon": 37.5823008,\n      "opening_hours": null\n    },\n    {\n      "id": "way/405103977",\n      "kind": "kindergarten",\n      "name": "Детский сад №364",\n      "lat": 55.7094654,\n      "lon": 37.6038531,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/405119704",\n      "kind": "school",\n      "name": "Школа №409",\n      "lat": 55.7460849,\n      "lon": 37.8190331,\n      "opening_hours": null\n    },\n    {\n      "id": "way/405306213",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/5",\n      "lat": 55.6210344,\n      "lon": 37.4973385,\n      "opening_hours": null\n    },\n    {\n      "id": "way/405311002",\n      "kind": "kindergarten",\n      "name": "Н.ш.-д.с. №1820",\n      "lat": 55.5987228,\n      "lon": 37.5902983,\n      "opening_hours": null\n    },\n    {\n      "id": "way/405413167",\n      "kind": "kindergarten",\n      "name": "Детский сад №2160",\n      "lat": 55.6020157,\n      "lon": 37.5869107,\n      "opening_hours": null\n    },\n    {\n      "id": "way/406178595",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.6559922,\n      "lon": 37.5790847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/407197238",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6733256,\n      "lon": 37.5625679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/407197239",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6727681,\n      "lon": 37.5619856,\n      "opening_hours": null\n    },\n    {\n      "id": "way/409972622",\n      "kind": "kindergarten",\n      "name": "Детский сад № 859",\n      "lat": 55.7281841,\n      "lon": 37.6328063,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410016274",\n      "kind": "kindergarten",\n      "name": "Детский сад №1949",\n      "lat": 55.7610584,\n      "lon": 37.7149582,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410016275",\n      "kind": "kindergarten",\n      "name": "Детский сад №471",\n      "lat": 55.7654898,\n      "lon": 37.7165835,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410329463",\n      "kind": "school",\n      "name": "Школа № 554",\n      "lat": 55.6571011,\n      "lon": 37.5798305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410648326",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa0430",\n      "lat": 55.8598473,\n      "lon": 37.5023723,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410648327",\n      "kind": "kindergarten",\n      "name": "Психоневрологический детский санаторий № 30",\n      "lat": 55.858768,\n      "lon": 37.5042696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410673515",\n      "kind": "school",\n      "name": "Гимназия № 1590",\n      "lat": 55.859617,\n      "lon": 37.4968942,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410873677",\n      "kind": "kindergarten",\n      "name": "Школа №760 им. А. П. Маресьева. Дошкольное отделение",\n      "lat": 55.8798861,\n      "lon": 37.7173487,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055126",\n      "kind": "school",\n      "name": "Вешняковская школа. Школьное отделение Р2",\n      "lat": 55.7412471,\n      "lon": 37.833085,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055128",\n      "kind": "school",\n      "name": "Вешняковская школа. Отделение В2",\n      "lat": 55.7368693,\n      "lon": 37.8194826,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055131",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение Р2",\n      "lat": 55.7402024,\n      "lon": 37.832808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055132",\n      "kind": "kindergarten",\n      "name": "Вешняковское школа. Дошкольное отделение Г12",\n      "lat": 55.7388634,\n      "lon": 37.8266828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055133",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольная группа №8",\n      "lat": 55.7415112,\n      "lon": 37.8246743,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055135",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение Р24",\n      "lat": 55.7333339,\n      "lon": 37.8343378,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055136",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение В3",\n      "lat": 55.739526,\n      "lon": 37.8207923,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411734363",\n      "kind": "school",\n      "name": "Школа № 1234",\n      "lat": 55.7513284,\n      "lon": 37.5809227,\n      "opening_hours": null\n    },\n    {\n      "id": "way/413720755",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8607172,\n      "lon": 37.5046593,\n      "opening_hours": null\n    },\n    {\n      "id": "way/413720759",\n      "kind": "kindergarten",\n      "name": "Детский сад №174",\n      "lat": 55.8574882,\n      "lon": 37.4999611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/416474001",\n      "kind": "kindergarten",\n      "name": "Детский сад №2593",\n      "lat": 55.6431986,\n      "lon": 37.6518222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/418714324",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8904965,\n      "lon": 37.6581524,\n      "opening_hours": null\n    },\n    {\n      "id": "way/418949050",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 7",\n      "lat": 55.779392,\n      "lon": 37.5286584,\n      "opening_hours": null\n    },\n    {\n      "id": "way/418959509",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 4",\n      "lat": 55.7780677,\n      "lon": 37.5326836,\n      "opening_hours": null\n    },\n    {\n      "id": "way/422615145",\n      "kind": "kindergarten",\n      "name": "Школа № 1948 \\"Лингвист-М\\". Учебный корпус 6 и 7",\n      "lat": 55.6636299,\n      "lon": 37.5825885,\n      "opening_hours": null\n    },\n    {\n      "id": "way/423811043",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6947786,\n      "lon": 37.4585915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/424314736",\n      "kind": "school",\n      "name": "Школа №1636 НИКА",\n      "lat": 55.6061841,\n      "lon": 37.729846,\n      "opening_hours": null\n    },\n    {\n      "id": "way/424822341",\n      "kind": "school",\n      "name": "Школа № 2090, структурное подразделение № 6",\n      "lat": 55.7131876,\n      "lon": 37.7845159,\n      "opening_hours": null\n    },\n    {\n      "id": "way/424822344",\n      "kind": "kindergarten",\n      "name": "Детский сад №1638",\n      "lat": 55.7111778,\n      "lon": 37.7907619,\n      "opening_hours": null\n    },\n    {\n      "id": "way/425074005",\n      "kind": "kindergarten",\n      "name": "Школа № 199. Дошкольное отделение № 7",\n      "lat": 55.6862245,\n      "lon": 37.56161,\n      "opening_hours": null\n    },\n    {\n      "id": "way/425202129",\n      "kind": "school",\n      "name": "Школа №741",\n      "lat": 55.7954249,\n      "lon": 37.80177,\n      "opening_hours": null\n    },\n    {\n      "id": "way/425304688",\n      "kind": "kindergarten",\n      "name": "Международная школа \\"Концепт\\". Детский сад",\n      "lat": 55.7942319,\n      "lon": 37.8041269,\n      "opening_hours": null\n    },\n    {\n      "id": "way/425378025",\n      "kind": "school",\n      "name": "Школа № 1948 \\"Лингвист-М\\". Центр дополнительного образования",\n      "lat": 55.6643429,\n      "lon": 37.583951,\n      "opening_hours": null\n    },\n    {\n      "id": "way/429116890",\n      "kind": "school",\n      "name": "Центр образования № 1272",\n      "lat": 55.7079002,\n      "lon": 37.663441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/429427494",\n      "kind": "kindergarten",\n      "name": "ДОУ",\n      "lat": 55.7632531,\n      "lon": 37.7434553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/430865710",\n      "kind": "kindergarten",\n      "name": "Детский сад №964",\n      "lat": 55.5883382,\n      "lon": 37.6537433,\n      "opening_hours": null\n    },\n    {\n      "id": "way/431163355",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 11",\n      "lat": 55.7761318,\n      "lon": 37.5301211,\n      "opening_hours": null\n    },\n    {\n      "id": "way/431566577",\n      "kind": "kindergarten",\n      "name": "Знайка",\n      "lat": 55.6038283,\n      "lon": 37.6608775,\n      "opening_hours": null\n    },\n    {\n      "id": "way/434375819",\n      "kind": "kindergarten",\n      "name": "Школа № 727 (Детский сад)",\n      "lat": 55.8505995,\n      "lon": 37.5160843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/435300297",\n      "kind": "school",\n      "name": "Школа №2033 (средняя и старшая школа)",\n      "lat": 55.8074721,\n      "lon": 37.8104311,\n      "opening_hours": null\n    },\n    {\n      "id": "way/435937342",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7548671,\n      "lon": 37.8211338,\n      "opening_hours": null\n    },\n    {\n      "id": "way/436009599",\n      "kind": "school",\n      "name": "Школа №511",\n      "lat": 55.6459039,\n      "lon": 37.6632881,\n      "opening_hours": null\n    },\n    {\n      "id": "way/436601516",\n      "kind": "school",\n      "name": "Гимназия №1798 \\"Феникс\\"",\n      "lat": 55.7539544,\n      "lon": 37.8046638,\n      "opening_hours": null\n    },\n    {\n      "id": "way/438153842",\n      "kind": "kindergarten",\n      "name": "Школа №2107, дошкольное отделение",\n      "lat": 55.7869319,\n      "lon": 37.6243553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/439928256",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6716065,\n      "lon": 37.6124307,\n      "opening_hours": null\n    },\n    {\n      "id": "way/440610765",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2019",\n      "lat": 55.6896976,\n      "lon": 37.5909215,\n      "opening_hours": null\n    },\n    {\n      "id": "way/441522066",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7511141,\n      "lon": 37.5240103,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443086824",\n      "kind": "school",\n      "name": "Школа №1222",\n      "lat": 55.7327777,\n      "lon": 37.7062437,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443369267",\n      "kind": "kindergarten",\n      "name": "ДМШ №35",\n      "lat": 55.7088467,\n      "lon": 37.6759961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443375876",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129 ДО \\"Детство\\"",\n      "lat": 55.707895,\n      "lon": 37.679498,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443391981",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Жемчужинка\\"",\n      "lat": 55.7279498,\n      "lon": 37.7611351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443470161",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129 ДО \\"Непоседы\\"",\n      "lat": 55.7063069,\n      "lon": 37.6803725,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443480698",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО для детей с ОВЗ",\n      "lat": 55.7056836,\n      "lon": 37.6722072,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443593271",\n      "kind": "school",\n      "name": "ГБОУ Школа №799, корпус 1",\n      "lat": 55.7691695,\n      "lon": 37.8253811,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443593272",\n      "kind": "school",\n      "name": "ГБОУ Школа №799, корпус 2",\n      "lat": 55.7701493,\n      "lon": 37.8288644,\n      "opening_hours": null\n    },\n    {\n      "id": "way/444428136",\n      "kind": "kindergarten",\n      "name": "Детский сад №1777",\n      "lat": 55.7450525,\n      "lon": 37.6742782,\n      "opening_hours": null\n    },\n    {\n      "id": "way/444428137",\n      "kind": "kindergarten",\n      "name": "Детский сад №1939",\n      "lat": 55.7429692,\n      "lon": 37.6727728,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445022081",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8729713,\n      "lon": 37.6259949,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445022088",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8722345,\n      "lon": 37.6252449,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445522765",\n      "kind": "kindergarten",\n      "name": "Лицей № 1568 имени Пабло Неруды, дошкольные отделения №2, 5, 6",\n      "lat": 55.8764058,\n      "lon": 37.6412513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445522767",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №5",\n      "lat": 55.8811752,\n      "lon": 37.6430426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445971629",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №3",\n      "lat": 55.8787132,\n      "lon": 37.6475385,\n      "opening_hours": null\n    },\n    {\n      "id": "way/447984762",\n      "kind": "kindergarten",\n      "name": "Детский сад №1818",\n      "lat": 55.8830796,\n      "lon": 37.711249,\n      "opening_hours": null\n    },\n    {\n      "id": "way/447984779",\n      "kind": "kindergarten",\n      "name": "Детский сад №278",\n      "lat": 55.8815306,\n      "lon": 37.7087141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/448154309",\n      "kind": "kindergarten",\n      "name": "Детский сад №406",\n      "lat": 55.6657735,\n      "lon": 37.5194747,\n      "opening_hours": null\n    },\n    {\n      "id": "way/449497270",\n      "kind": "kindergarten",\n      "name": "Детский сад №240",\n      "lat": 55.8007876,\n      "lon": 37.8021527,\n      "opening_hours": null\n    },\n    {\n      "id": "way/449684295",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6735244,\n      "lon": 37.4678958,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450112051",\n      "kind": "kindergarten",\n      "name": "Школа №1811 (дошкольный корпус №15)",\n      "lat": 55.8018115,\n      "lon": 37.8108373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450112055",\n      "kind": "kindergarten",\n      "name": "Детский сад №1522",\n      "lat": 55.7929112,\n      "lon": 37.8123178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450112057",\n      "kind": "school",\n      "name": "Кладезь",\n      "lat": 55.8018366,\n      "lon": 37.800374,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450357369",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 4",\n      "lat": 55.8603655,\n      "lon": 37.6391282,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450357372",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 16",\n      "lat": 55.8593541,\n      "lon": 37.6370481,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450362997",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение школы №717",\n      "lat": 55.8185974,\n      "lon": 37.5050245,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450366849",\n      "kind": "school",\n      "name": "Школа №717",\n      "lat": 55.8172835,\n      "lon": 37.5050391,\n      "opening_hours": null\n    },\n    {\n      "id": "way/452897582",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, Щелкунчик (здание №11)",\n      "lat": 55.673897,\n      "lon": 37.6672121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/453368828",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8387752,\n      "lon": 37.5750259,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454021189",\n      "kind": "kindergarten",\n      "name": "Школа №1223, дошкольный корпус №4",\n      "lat": 55.8410143,\n      "lon": 37.5466998,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454055460",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа \\"Кузьминки\\"",\n      "lat": 55.7028199,\n      "lon": 37.7741658,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454055462",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.703316,\n      "lon": 37.770527,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454064408",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7007361,\n      "lon": 37.7702559,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454311146",\n      "kind": "kindergarten",\n      "name": "Школа № 656 им. А. С. Макаренко. Дошкольный корпус",\n      "lat": 55.869375,\n      "lon": 37.5513991,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454469587",\n      "kind": "kindergarten",\n      "name": "Школа №1290 дошкольный корпус №1",\n      "lat": 55.7929082,\n      "lon": 37.7783216,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454743223",\n      "kind": "school",\n      "name": "Школа № 1285. Школьный корпус",\n      "lat": 55.8257017,\n      "lon": 37.4496465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454743226",\n      "kind": "kindergarten",\n      "name": "Школа №654 имени А. Д. Фридмана. Дошкольное отделение №1",\n      "lat": 55.6977138,\n      "lon": 37.7417608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454844150",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №14",\n      "lat": 55.6672437,\n      "lon": 37.7795248,\n      "opening_hours": null\n    },\n    {\n      "id": "way/455510927",\n      "kind": "kindergarten",\n      "name": "Детский сад №1187",\n      "lat": 55.7542167,\n      "lon": 37.6753953,\n      "opening_hours": null\n    },\n    {\n      "id": "way/455697968",\n      "kind": "kindergarten",\n      "name": "Детский сад №962",\n      "lat": 55.8673923,\n      "lon": 37.7001389,\n      "opening_hours": null\n    },\n    {\n      "id": "way/456669987",\n      "kind": "kindergarten",\n      "name": "Детский сад №2445",\n      "lat": 55.8805037,\n      "lon": 37.5520859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459404196",\n      "kind": "kindergarten",\n      "name": "Детский сад №643",\n      "lat": 55.7424834,\n      "lon": 37.6874653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459574973",\n      "kind": "school",\n      "name": "Школа №1716 \\"Эврика-Огонёк\\"",\n      "lat": 55.6654085,\n      "lon": 37.774361,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459575997",\n      "kind": "kindergarten",\n      "name": "Школа № 1716 \\"Эврика-Огонек\\". Дошкольное подразделение",\n      "lat": 55.6660439,\n      "lon": 37.7724445,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459584374",\n      "kind": "kindergarten",\n      "name": "Школа № 1989. Дошкольное отделение № 1",\n      "lat": 55.6627951,\n      "lon": 37.7775865,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459586029",\n      "kind": "school",\n      "name": "Центр образования №1989",\n      "lat": 55.6636276,\n      "lon": 37.7759043,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459837877",\n      "kind": "school",\n      "name": "Школа №2011",\n      "lat": 55.6677537,\n      "lon": 37.7887291,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459838512",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6684037,\n      "lon": 37.7881625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459979623",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6718643,\n      "lon": 37.7917963,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459980354",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6723095,\n      "lon": 37.7925643,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459980356",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6717319,\n      "lon": 37.7931056,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460006530",\n      "kind": "school",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ШП-1)",\n      "lat": 55.6712605,\n      "lon": 37.7690968,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460010094",\n      "kind": "school",\n      "name": "Школа №1186",\n      "lat": 55.6720073,\n      "lon": 37.7673517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460014006",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-3)",\n      "lat": 55.6699962,\n      "lon": 37.7677221,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460237676",\n      "kind": "kindergarten",\n      "name": "ГБОУ СОШ № 572 ДО № 8",\n      "lat": 55.6801952,\n      "lon": 37.7749467,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460240331",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6789643,\n      "lon": 37.773561,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460240720",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6791372,\n      "lon": 37.770448,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460240932",\n      "kind": "school",\n      "name": "Школа №572",\n      "lat": 55.6801343,\n      "lon": 37.7699305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460245030",\n      "kind": "school",\n      "name": "Школа № 1547",\n      "lat": 55.663779,\n      "lon": 37.7673599,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460511550",\n      "kind": "kindergarten",\n      "name": "Детский сад №2437",\n      "lat": 55.6538686,\n      "lon": 37.7701124,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460512501",\n      "kind": "school",\n      "name": "Школа №1959 \\"Школа здоровья\\"",\n      "lat": 55.655738,\n      "lon": 37.7698676,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460514084",\n      "kind": "kindergarten",\n      "name": "Детский сад №2398",\n      "lat": 55.655292,\n      "lon": 37.7683679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460514466",\n      "kind": "school",\n      "name": "Школа №1959 \\"Школа здоровья\\" (Гимназические классы)",\n      "lat": 55.6543652,\n      "lon": 37.7714188,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460560229",\n      "kind": "kindergarten",\n      "name": "Школа №1547. Дошкольное отделение",\n      "lat": 55.6631947,\n      "lon": 37.7661136,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460564113",\n      "kind": "kindergarten",\n      "name": "Школа №1423. Дошкольное отделение",\n      "lat": 55.6642222,\n      "lon": 37.7688131,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460993900",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.5892129,\n      "lon": 37.4574601,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461030275",\n      "kind": "school",\n      "name": "Международная школа завтрашнего дня",\n      "lat": 55.6329431,\n      "lon": 37.6088891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461109423",\n      "kind": "school",\n      "name": "Школа №1904",\n      "lat": 55.649823,\n      "lon": 37.7475564,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461110609",\n      "kind": "kindergarten",\n      "name": "Детский сад №2325",\n      "lat": 55.649498,\n      "lon": 37.7502884,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461111582",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6520484,\n      "lon": 37.7458941,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461358380",\n      "kind": "kindergarten",\n      "name": "Детский сад №2280",\n      "lat": 55.7892197,\n      "lon": 37.7872809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461703728",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №3",\n      "lat": 55.5902005,\n      "lon": 37.6109967,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461836412",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №6",\n      "lat": 55.6511098,\n      "lon": 37.7639551,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461836978",\n      "kind": "school",\n      "name": "Школа № 1357 «На Братиславской». Здание №5",\n      "lat": 55.6505489,\n      "lon": 37.7624974,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461846655",\n      "kind": "kindergarten",\n      "name": "Детский сад №2397",\n      "lat": 55.6593661,\n      "lon": 37.7449486,\n      "opening_hours": null\n    },\n    {\n      "id": "way/467769169",\n      "kind": "school",\n      "name": "Школа \\"Ника\\"",\n      "lat": 55.6566093,\n      "lon": 37.566925,\n      "opening_hours": null\n    },\n    {\n      "id": "way/467769170",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 3",\n      "lat": 55.6564087,\n      "lon": 37.5617166,\n      "opening_hours": null\n    },\n    {\n      "id": "way/468860751",\n      "kind": "school",\n      "name": "ГБОУ г. Москвы Школа № 1159 (ДО)",\n      "lat": 55.8626661,\n      "lon": 37.5159729,\n      "opening_hours": null\n    },\n    {\n      "id": "way/469126286",\n      "kind": "kindergarten",\n      "name": "Лицей № 1568 имени Пабло Неруды, дошкольные отделения №3, 4, 7",\n      "lat": 55.8731054,\n      "lon": 37.6421695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/472691717",\n      "kind": "kindergarten",\n      "name": "Школа №949 (дошкольное отделение, здание №5)",\n      "lat": 55.607815,\n      "lon": 37.750898,\n      "opening_hours": null\n    },\n    {\n      "id": "way/474041399",\n      "kind": "school",\n      "name": "Школа №949 (начальная школа, здание №2)",\n      "lat": 55.6089946,\n      "lon": 37.7518158,\n      "opening_hours": null\n    },\n    {\n      "id": "way/474041402",\n      "kind": "kindergarten",\n      "name": "Детский сад №1779",\n      "lat": 55.6861192,\n      "lon": 37.7550947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/474206141",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1703",\n      "lat": 55.6220269,\n      "lon": 37.7041333,\n      "opening_hours": null\n    },\n    {\n      "id": "way/475303188",\n      "kind": "kindergarten",\n      "name": "Школа №2033 (дошкольные группы)",\n      "lat": 55.8050198,\n      "lon": 37.8086258,\n      "opening_hours": null\n    },\n    {\n      "id": "way/475662142",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №13",\n      "lat": 55.6669792,\n      "lon": 37.7649866,\n      "opening_hours": null\n    },\n    {\n      "id": "way/476155610",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-9)",\n      "lat": 55.6781352,\n      "lon": 37.8006441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/476412270",\n      "kind": "school",\n      "name": "Школа № 572, Отделение № 9",\n      "lat": 55.6774152,\n      "lon": 37.7710418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/476903701",\n      "kind": "kindergarten",\n      "name": "Детский сад №1547",\n      "lat": 55.7555245,\n      "lon": 37.7805331,\n      "opening_hours": null\n    },\n    {\n      "id": "way/477778886",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.6426509,\n      "lon": 37.5309424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/477787991",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.6443387,\n      "lon": 37.5399307,\n      "opening_hours": "Mo-Fr 08:30-20:00, Tu-Su 08:00-15:00; Sa 08:00-12:00"\n    },\n    {\n      "id": "way/477908122",\n      "kind": "school",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 5.",\n      "lat": 55.6472517,\n      "lon": 37.5237555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/479152965",\n      "kind": "school",\n      "name": "Школа №37. Корпус школьного образования",\n      "lat": 55.7027906,\n      "lon": 37.5025093,\n      "opening_hours": null\n    },\n    {\n      "id": "way/480525994",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.6972358,\n      "lon": 37.4945335,\n      "opening_hours": null\n    },\n    {\n      "id": "way/480699878",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7532888,\n      "lon": 37.6599928,\n      "opening_hours": null\n    },\n    {\n      "id": "way/481075429",\n      "kind": "school",\n      "name": "Структурное подразделение № 28 школы № 2124 «Центр развития и коррекции»",\n      "lat": 55.7655501,\n      "lon": 37.6719508,\n      "opening_hours": null\n    },\n    {\n      "id": "way/481379779",\n      "kind": "kindergarten",\n      "name": "Детский сад/ясли №1837",\n      "lat": 55.8658035,\n      "lon": 37.5922666,\n      "opening_hours": null\n    },\n    {\n      "id": "way/481547265",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №9",\n      "lat": 55.6581803,\n      "lon": 37.7548634,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482435437",\n      "kind": "kindergarten",\n      "name": "Школа № 1465, дошкольное отделение № 3",\n      "lat": 55.7453437,\n      "lon": 37.5617271,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482435442",\n      "kind": "kindergarten",\n      "name": "Школа №1465 имени адмирала Н. Г. Кузнецова, дошкольное подразделение №7",\n      "lat": 55.7493641,\n      "lon": 37.5688828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482435458",\n      "kind": "school",\n      "name": "Школа № 1228 с углубленным изучением английского языка. Корпус 5",\n      "lat": 55.7675145,\n      "lon": 37.7117966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482435459",\n      "kind": "school",\n      "name": "Школа № 1228 Корпус № 2",\n      "lat": 55.750002,\n      "lon": 37.7127725,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447167",\n      "kind": "kindergarten",\n      "name": "Школа № 1411. Структурное подразделение № 1055 детский сад \\"Ручеек счастья\\"",\n      "lat": 55.8680473,\n      "lon": 37.6002271,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447168",\n      "kind": "school",\n      "name": "Школа № 962. Структурное подразделение № 3",\n      "lat": 55.8615952,\n      "lon": 37.5877115,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447172",\n      "kind": "kindergarten",\n      "name": "Школа № 962, дошкольное отделение № 4",\n      "lat": 55.8580891,\n      "lon": 37.5875808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447186",\n      "kind": "kindergarten",\n      "name": "Школа № 950, дошкольное отделение",\n      "lat": 55.8592374,\n      "lon": 37.6077987,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447187",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1554. Дошкольное отделение № 3",\n      "lat": 55.8641495,\n      "lon": 37.6189017,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482740277",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6723848,\n      "lon": 37.5377625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/483201827",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.889964,\n      "lon": 37.4399616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/483709852",\n      "kind": "kindergarten",\n      "name": "Детский сад № 712",\n      "lat": 55.6202141,\n      "lon": 37.66659,\n      "opening_hours": null\n    },\n    {\n      "id": "way/484519282",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.619309,\n      "lon": 37.7599569,\n      "opening_hours": null\n    },\n    {\n      "id": "way/484519295",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6192405,\n      "lon": 37.7585881,\n      "opening_hours": null\n    },\n    {\n      "id": "way/484692544",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6719262,\n      "lon": 37.5895967,\n      "opening_hours": null\n    },\n    {\n      "id": "way/484924150",\n      "kind": "kindergarten",\n      "name": "Детский сад №2068",\n      "lat": 55.6466236,\n      "lon": 37.6641687,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485063924",\n      "kind": "kindergarten",\n      "name": "Российское Дошкольное Отделение \\"Филиппок\\"",\n      "lat": 55.6963003,\n      "lon": 37.5087398,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485063925",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6980742,\n      "lon": 37.5086194,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485063931",\n      "kind": "school",\n      "name": "Школа №1498 «ММШ-15». Корпус школьного образования",\n      "lat": 55.6974493,\n      "lon": 37.5101101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485063932",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6988892,\n      "lon": 37.5078553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485382207",\n      "kind": "school",\n      "name": "Новая гуманитарная школа",\n      "lat": 55.8023622,\n      "lon": 37.5410002,\n      "opening_hours": null\n    },\n    {\n      "id": "way/487787682",\n      "kind": "kindergarten",\n      "name": "Многопрофильный лицей № 1501. 1-е дошкольное отделение № 1921",\n      "lat": 55.7792124,\n      "lon": 37.6039133,\n      "opening_hours": null\n    },\n    {\n      "id": "way/487951922",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6602417,\n      "lon": 37.7756397,\n      "opening_hours": null\n    },\n    {\n      "id": "way/487957328",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6598351,\n      "lon": 37.7642406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/488261721",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6778493,\n      "lon": 37.7673625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/488268383",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-5)",\n      "lat": 55.6794431,\n      "lon": 37.7668384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/488482961",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Здание 3",\n      "lat": 55.8167738,\n      "lon": 37.4918663,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489066386",\n      "kind": "school",\n      "name": "Центр образования №1524. Корпус 3",\n      "lat": 55.6896469,\n      "lon": 37.7220748,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489272879",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6371021,\n      "lon": 37.7549869,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489393313",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8935527,\n      "lon": 37.4475519,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489393314",\n      "kind": "school",\n      "name": "Гимназия №4",\n      "lat": 55.8933318,\n      "lon": 37.4458706,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489393337",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa045",\n      "lat": 55.8894213,\n      "lon": 37.4540406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489393339",\n      "kind": "school",\n      "name": "Лицей №\xa011",\n      "lat": 55.89046,\n      "lon": 37.453744,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489409755",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8891392,\n      "lon": 37.4502244,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489409759",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8901645,\n      "lon": 37.4469646,\n      "opening_hours": null\n    },\n    {\n      "id": "way/490122227",\n      "kind": "school",\n      "name": "Школа №393",\n      "lat": 55.6775483,\n      "lon": 37.7609936,\n      "opening_hours": null\n    },\n    {\n      "id": "way/490420670",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa050",\n      "lat": 55.8883121,\n      "lon": 37.4545854,\n      "opening_hours": null\n    },\n    {\n      "id": "way/490420686",\n      "kind": "school",\n      "name": "Лицей №7",\n      "lat": 55.8867949,\n      "lon": 37.4555735,\n      "opening_hours": null\n    },\n    {\n      "id": "way/491821617",\n      "kind": "school",\n      "name": "Школа № 654 им. А. Д. Фридмана, корпус 3",\n      "lat": 55.6956907,\n      "lon": 37.7498411,\n      "opening_hours": null\n    },\n    {\n      "id": "way/492713772",\n      "kind": "school",\n      "name": "НОУ Школа \\"Апогей\\"",\n      "lat": 55.6795447,\n      "lon": 37.753285,\n      "opening_hours": null\n    },\n    {\n      "id": "way/492974181",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6504446,\n      "lon": 37.7555789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/492974505",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6507007,\n      "lon": 37.7546706,\n      "opening_hours": null\n    },\n    {\n      "id": "way/493001319",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Лесная сказка»",\n      "lat": 55.62595,\n      "lon": 37.4810562,\n      "opening_hours": null\n    },\n    {\n      "id": "way/494132822",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6558453,\n      "lon": 37.7471003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/494136258",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6540924,\n      "lon": 37.7459609,\n      "opening_hours": null\n    },\n    {\n      "id": "way/496457582",\n      "kind": "school",\n      "name": "Младшая школа №1496",\n      "lat": 55.7219792,\n      "lon": 37.6035283,\n      "opening_hours": null\n    },\n    {\n      "id": "way/497088018",\n      "kind": "school",\n      "name": "Школа № 553",\n      "lat": 55.7242137,\n      "lon": 37.6235887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498121462",\n      "kind": "school",\n      "name": "Школа №5",\n      "lat": 55.6602874,\n      "lon": 37.6062387,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498175708",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1403",\n      "lat": 55.6619129,\n      "lon": 37.6063568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498299188",\n      "kind": "school",\n      "name": "Школа №1623",\n      "lat": 55.6250771,\n      "lon": 37.6140342,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498304285",\n      "kind": "kindergarten",\n      "name": "Детский сад №1834",\n      "lat": 55.6241119,\n      "lon": 37.6126637,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498314470",\n      "kind": "kindergarten",\n      "name": "Детский сад №917",\n      "lat": 55.6137791,\n      "lon": 37.5954438,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498572996",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Мечта»",\n      "lat": 55.6354819,\n      "lon": 37.5905932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/499308094",\n      "kind": "school",\n      "name": "Школа №285 имени В.А. Молодцова",\n      "lat": 55.8763892,\n      "lon": 37.6290788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/500981449",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8059641,\n      "lon": 37.5470221,\n      "opening_hours": null\n    },\n    {\n      "id": "way/502645356",\n      "kind": "school",\n      "name": "Школа №\xa083",\n      "lat": 55.7598113,\n      "lon": 37.5400412,\n      "opening_hours": null\n    },\n    {\n      "id": "way/502790155",\n      "kind": "kindergarten",\n      "name": "Детский сад №229",\n      "lat": 55.7106995,\n      "lon": 37.6615929,\n      "opening_hours": null\n    },\n    {\n      "id": "way/502828348",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2091",\n      "lat": 55.7291203,\n      "lon": 37.6788028,\n      "opening_hours": null\n    },\n    {\n      "id": "way/503995297",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-6)",\n      "lat": 55.6808544,\n      "lon": 37.7667484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/504270193",\n      "kind": "kindergarten",\n      "name": "Детский сад №381",\n      "lat": 55.8080275,\n      "lon": 37.7112367,\n      "opening_hours": null\n    },\n    {\n      "id": "way/504521532",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6810519,\n      "lon": 37.6113237,\n      "opening_hours": null\n    },\n    {\n      "id": "way/504548447",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8854302,\n      "lon": 37.4432532,\n      "opening_hours": null\n    },\n    {\n      "id": "way/505041657",\n      "kind": "kindergarten",\n      "name": "Детский сад №720",\n      "lat": 55.8802785,\n      "lon": 37.7064247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/506470757",\n      "kind": "kindergarten",\n      "name": "Детский сад №2704",\n      "lat": 55.875651,\n      "lon": 37.7202259,\n      "opening_hours": null\n    },\n    {\n      "id": "way/507033427",\n      "kind": "kindergarten",\n      "name": "Лицей № 38. Дошкольное отделение № 10",\n      "lat": 55.7830639,\n      "lon": 37.4682924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/511924009",\n      "kind": "kindergarten",\n      "name": "Центр образования №1998",\n      "lat": 55.636983,\n      "lon": 37.7520557,\n      "opening_hours": null\n    },\n    {\n      "id": "way/515690177",\n      "kind": "school",\n      "name": "Школа №962. Начальная школа",\n      "lat": 55.8635333,\n      "lon": 37.5959611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/520900483",\n      "kind": "school",\n      "name": "Школа №1797",\n      "lat": 55.8073519,\n      "lon": 37.7122562,\n      "opening_hours": null\n    },\n    {\n      "id": "way/523390513",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1387",\n      "lat": 55.7789975,\n      "lon": 37.6110774,\n      "opening_hours": null\n    },\n    {\n      "id": "way/523731664",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7715368,\n      "lon": 37.4709838,\n      "opening_hours": null\n    },\n    {\n      "id": "way/524737858",\n      "kind": "school",\n      "name": "Школа №440 (Гимназия № 1811 «Восточное Измайлово»)",\n      "lat": 55.7896737,\n      "lon": 37.8127305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/526745593",\n      "kind": "kindergarten",\n      "name": "Школа на проспекте Вернадского. Корпус дошкольного образования 5",\n      "lat": 55.6563965,\n      "lon": 37.4901662,\n      "opening_hours": null\n    },\n    {\n      "id": "way/528700091",\n      "kind": "kindergarten",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (дошкольное отделение, корпус 3)",\n      "lat": 55.6784765,\n      "lon": 37.7260469,\n      "opening_hours": null\n    },\n    {\n      "id": "way/529572336",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\", Отделение ОВЗ",\n      "lat": 55.7548179,\n      "lon": 37.6464144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/530563722",\n      "kind": "kindergarten",\n      "name": "Детский сад №1043",\n      "lat": 55.8007834,\n      "lon": 37.4846858,\n      "opening_hours": null\n    },\n    {\n      "id": "way/531958078",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7031226,\n      "lon": 37.4904059,\n      "opening_hours": null\n    },\n    {\n      "id": "way/532935146",\n      "kind": "kindergarten",\n      "name": "Гимназия №1506, дошкольное отделение",\n      "lat": 55.8911872,\n      "lon": 37.6485366,\n      "opening_hours": null\n    },\n    {\n      "id": "way/532935469",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Умка\\"",\n      "lat": 55.8925842,\n      "lon": 37.6572345,\n      "opening_hours": null\n    },\n    {\n      "id": "way/532936533",\n      "kind": "school",\n      "name": "Гимназия №1558",\n      "lat": 55.8685922,\n      "lon": 37.6533191,\n      "opening_hours": null\n    },\n    {\n      "id": "way/532939500",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 2",\n      "lat": 55.8538772,\n      "lon": 37.6591808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/539236815",\n      "kind": "kindergarten",\n      "name": "Детский сад № 879",\n      "lat": 55.7840475,\n      "lon": 37.6386054,\n      "opening_hours": null\n    },\n    {\n      "id": "way/539723370",\n      "kind": "kindergarten",\n      "name": "Детский сад Развитие 21век",\n      "lat": 55.6761008,\n      "lon": 37.5945524,\n      "opening_hours": null\n    },\n    {\n      "id": "way/544107259",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.799077,\n      "lon": 37.5530406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/547279582",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8710569,\n      "lon": 37.6923745,\n      "opening_hours": null\n    },\n    {\n      "id": "way/547624723",\n      "kind": "school",\n      "name": "Школа №750",\n      "lat": 55.8647481,\n      "lon": 37.6960994,\n      "opening_hours": null\n    },\n    {\n      "id": "way/548236327",\n      "kind": "school",\n      "name": "Школа № 1536. Корпус школьного образования № 1",\n      "lat": 55.6800806,\n      "lon": 37.5393311,\n      "opening_hours": null\n    },\n    {\n      "id": "way/548239829",\n      "kind": "kindergarten",\n      "name": "Детский сад 97",\n      "lat": 55.6781345,\n      "lon": 37.5417093,\n      "opening_hours": null\n    },\n    {\n      "id": "way/552386633",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 11",\n      "lat": 55.8504273,\n      "lon": 37.6412022,\n      "opening_hours": null\n    },\n    {\n      "id": "way/552591097",\n      "kind": "school",\n      "name": "Специальная (коррекционная) школа-интернат № 102",\n      "lat": 55.8506431,\n      "lon": 37.6329417,\n      "opening_hours": null\n    },\n    {\n      "id": "way/552786569",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8004074,\n      "lon": 37.5266772,\n      "opening_hours": null\n    },\n    {\n      "id": "way/553048211",\n      "kind": "kindergarten",\n      "name": "Детский сад №690",\n      "lat": 55.8613423,\n      "lon": 37.4977078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/553048213",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2085",\n      "lat": 55.8592747,\n      "lon": 37.5066078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554494298",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1384 имени А.А. Леманского\\"",\n      "lat": 55.7988268,\n      "lon": 37.50608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554647030",\n      "kind": "kindergarten",\n      "name": "Детский сад №2574",\n      "lat": 55.8757158,\n      "lon": 37.4867373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554648356",\n      "kind": "kindergarten",\n      "name": "Школа № 1474. Дошкольное отделение № 8",\n      "lat": 55.8768195,\n      "lon": 37.487539,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554648358",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8720704,\n      "lon": 37.4949596,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554648359",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01474, дошкольное отделение",\n      "lat": 55.8672474,\n      "lon": 37.4985507,\n      "opening_hours": null\n    },\n    {\n      "id": "way/555198691",\n      "kind": "school",\n      "name": "Школа № 185",\n      "lat": 55.8367731,\n      "lon": 37.5353569,\n      "opening_hours": null\n    },\n    {\n      "id": "way/555999588",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8232333,\n      "lon": 37.6625466,\n      "opening_hours": null\n    },\n    {\n      "id": "way/555999622",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8230246,\n      "lon": 37.6626039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/556568449",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1646",\n      "lat": 55.8977464,\n      "lon": 37.6112121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/556568491",\n      "kind": "school",\n      "name": "Школа №1413",\n      "lat": 55.8991537,\n      "lon": 37.6119921,\n      "opening_hours": null\n    },\n    {\n      "id": "way/557818356",\n      "kind": "school",\n      "name": "Школа №224",\n      "lat": 55.8324713,\n      "lon": 37.5023435,\n      "opening_hours": null\n    },\n    {\n      "id": "way/558782674",\n      "kind": "school",\n      "name": "Школа № 1474. Подразделение 2",\n      "lat": 55.87335,\n      "lon": 37.4890118,\n      "opening_hours": null\n    },\n    {\n      "id": "way/558782684",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8646341,\n      "lon": 37.4964523,\n      "opening_hours": null\n    },\n    {\n      "id": "way/558785509",\n      "kind": "school",\n      "name": "ГБОУ г. Москвы Школа № 1159",\n      "lat": 55.8626161,\n      "lon": 37.5096781,\n      "opening_hours": null\n    },\n    {\n      "id": "way/561260076",\n      "kind": "kindergarten",\n      "name": "Детский сад №1960",\n      "lat": 55.8635938,\n      "lon": 37.6942832,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568303123",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 9 Дошкольное отделение Семицветик",\n      "lat": 55.6658089,\n      "lon": 37.5208623,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568709881",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №799, корпус 5",\n      "lat": 55.7714964,\n      "lon": 37.8284763,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568709882",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №799, корпус 3",\n      "lat": 55.7705827,\n      "lon": 37.8249913,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568709883",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №799, корпус 4",\n      "lat": 55.770562,\n      "lon": 37.8260427,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568709885",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №799, корпус 6",\n      "lat": 55.7715139,\n      "lon": 37.8295964,\n      "opening_hours": null\n    },\n    {\n      "id": "way/575895642",\n      "kind": "kindergarten",\n      "name": "Детский сад №1327",\n      "lat": 55.8713826,\n      "lon": 37.5887819,\n      "opening_hours": null\n    },\n    {\n      "id": "way/576198916",\n      "kind": "school",\n      "name": "Школа №727",\n      "lat": 55.8471612,\n      "lon": 37.5166713,\n      "opening_hours": null\n    },\n    {\n      "id": "way/577047930",\n      "kind": "kindergarten",\n      "name": "Детский  сад «Ладушки»",\n      "lat": 55.8520529,\n      "lon": 37.5048415,\n      "opening_hours": null\n    },\n    {\n      "id": "way/577047933",\n      "kind": "kindergarten",\n      "name": "Академия  им. М.Ю. Лермонтова (Детский сад)",\n      "lat": 55.852289,\n      "lon": 37.5029705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/577139276",\n      "kind": "kindergarten",\n      "name": "Школа № 1583 имени К. А. Керимова (Дошкольный корпус)",\n      "lat": 55.8534448,\n      "lon": 37.5110635,\n      "opening_hours": null\n    },\n    {\n      "id": "way/577139278",\n      "kind": "school",\n      "name": "Детская музыкальная школа №98",\n      "lat": 55.8529296,\n      "lon": 37.5109268,\n      "opening_hours": null\n    },\n    {\n      "id": "way/579520474",\n      "kind": "kindergarten",\n      "name": "Школа № 1862. Корпус дошкольного образования №9",\n      "lat": 55.6435195,\n      "lon": 37.6172907,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948080",\n      "kind": "school",\n      "name": "Школа №922 (учебный корпус \\"Терлецкий\\")",\n      "lat": 55.7598996,\n      "lon": 37.8273607,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948085",\n      "kind": "kindergarten",\n      "name": "Школа №922 (учебный корпус \\"Измайловский\\")",\n      "lat": 55.7628859,\n      "lon": 37.824001,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948086",\n      "kind": "kindergarten",\n      "name": "Дошкольный корпус ГБОУ Школа №922",\n      "lat": 55.7639292,\n      "lon": 37.8243023,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948087",\n      "kind": "kindergarten",\n      "name": "Школа №922 (учебный корпус \\"Екатерининский\\")",\n      "lat": 55.7641655,\n      "lon": 37.8302441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948088",\n      "kind": "kindergarten",\n      "name": "Школа №922 (учебный корпус \\"Мещерский\\")",\n      "lat": 55.7647033,\n      "lon": 37.8285922,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948089",\n      "kind": "kindergarten",\n      "name": "Школа №922 (учебный корпус \\"Воронцовский\\")",\n      "lat": 55.7649301,\n      "lon": 37.8266818,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929543",\n      "kind": "kindergarten",\n      "name": "Дошкольный корпус №3 ГБОУ Школа №1476",\n      "lat": 55.7777351,\n      "lon": 37.8338801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929546",\n      "kind": "kindergarten",\n      "name": "Дошкольный корпус №1 ГБОУ Школа №1476",\n      "lat": 55.7742978,\n      "lon": 37.8214623,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929558",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1373, корпус №3",\n      "lat": 55.7806757,\n      "lon": 37.8334947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929561",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1373, корпус №5",\n      "lat": 55.7794952,\n      "lon": 37.8310343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929562",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1373, корпус №4",\n      "lat": 55.7801494,\n      "lon": 37.8301517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586387137",\n      "kind": "kindergarten",\n      "name": "Бывшая территория детского сада 1727",\n      "lat": 55.7776719,\n      "lon": 37.5548097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586790386",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6753566,\n      "lon": 37.6016615,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586790387",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6774801,\n      "lon": 37.603928,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586790388",\n      "kind": "kindergarten",\n      "name": "Детский сад 1406",\n      "lat": 55.6779575,\n      "lon": 37.6103831,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586805962",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6696608,\n      "lon": 37.5947719,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586808380",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6983863,\n      "lon": 37.5574606,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586808381",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.696696,\n      "lon": 37.5571806,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586808383",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6944149,\n      "lon": 37.6047447,\n      "opening_hours": null\n    },\n    {\n      "id": "way/587445099",\n      "kind": "school",\n      "name": "Московский образовательный комплекс имени Виктора Талалихина. Структурное подразделение № 5",\n      "lat": 55.7472248,\n      "lon": 37.7211595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/588597975",\n      "kind": "kindergarten",\n      "name": "ДОУ №2009",\n      "lat": 55.7527536,\n      "lon": 37.5223742,\n      "opening_hours": null\n    },\n    {\n      "id": "way/588598072",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7516926,\n      "lon": 37.5229583,\n      "opening_hours": null\n    },\n    {\n      "id": "way/590221545",\n      "kind": "kindergarten",\n      "name": "Детский сад №965",\n      "lat": 55.8622524,\n      "lon": 37.6178373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/592726680",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/8",\n      "lat": 55.6244529,\n      "lon": 37.486332,\n      "opening_hours": null\n    },\n    {\n      "id": "way/593891735",\n      "kind": "kindergarten",\n      "name": "Школа № 504. Дошкольное отделение «Радуга-1»",\n      "lat": 55.6672844,\n      "lon": 37.6250138,\n      "opening_hours": null\n    },\n    {\n      "id": "way/595122415",\n      "kind": "kindergarten",\n      "name": "Детский сад №26",\n      "lat": 55.6006711,\n      "lon": 37.596141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/595420149",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6452053,\n      "lon": 37.722634,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596215157",\n      "kind": "kindergarten",\n      "name": "Школа № 1770. Дошкольное отделение \\"Непоседы\\"",\n      "lat": 55.6878141,\n      "lon": 37.6864357,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596216287",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6866057,\n      "lon": 37.6885882,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596360348",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6829692,\n      "lon": 37.7051047,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596370511",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 463 корпус №1 «Жемчужинка»",\n      "lat": 55.6780005,\n      "lon": 37.6722907,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596370512",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 463 корпус №2 «Жемчужинка»",\n      "lat": 55.6779398,\n      "lon": 37.674095,\n      "opening_hours": null\n    },\n    {\n      "id": "way/598588576",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7469855,\n      "lon": 37.792693,\n      "opening_hours": null\n    },\n    {\n      "id": "way/601108319",\n      "kind": "kindergarten",\n      "name": "Школа № 657. Корпус дошкольного образования №3",\n      "lat": 55.6015997,\n      "lon": 37.6236916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/605601044",\n      "kind": "school",\n      "name": "Школа № 1454 «Тимирязевская»",\n      "lat": 55.8333738,\n      "lon": 37.5701,\n      "opening_hours": null\n    },\n    {\n      "id": "way/607334891",\n      "kind": "kindergarten",\n      "name": "Детский сад №923",\n      "lat": 55.6919248,\n      "lon": 37.7247782,\n      "opening_hours": null\n    },\n    {\n      "id": "way/607339934",\n      "kind": "kindergarten",\n      "name": "Детский сад №968",\n      "lat": 55.6898083,\n      "lon": 37.7241153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/608127845",\n      "kind": "kindergarten",\n      "name": "Детский сад № 789",\n      "lat": 55.610534,\n      "lon": 37.5871809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/608378970",\n      "kind": "school",\n      "name": "Европейская гимназия с дошкольным отделением",\n      "lat": 55.7911068,\n      "lon": 37.6682859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/608667848",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Мытной\\"",\n      "lat": 55.7186688,\n      "lon": 37.6164258,\n      "opening_hours": null\n    },\n    {\n      "id": "way/609317059",\n      "kind": "school",\n      "name": "Школа № 1770. Корпус школьного образования «Фрегат»",\n      "lat": 55.6824788,\n      "lon": 37.6957807,\n      "opening_hours": null\n    },\n    {\n      "id": "way/613303406",\n      "kind": "kindergarten",\n      "name": "Детский сад №64/2 “Яблонька”",\n      "lat": 55.8935497,\n      "lon": 37.7160892,\n      "opening_hours": null\n    },\n    {\n      "id": "way/614889264",\n      "kind": "kindergarten",\n      "name": "Школа №\xa067. Отделение дошкольного образования",\n      "lat": 55.7418818,\n      "lon": 37.5207871,\n      "opening_hours": null\n    },\n    {\n      "id": "way/617465057",\n      "kind": "kindergarten",\n      "name": "Детский сад «Светлые Горы»",\n      "lat": 55.5896809,\n      "lon": 37.6028563,\n      "opening_hours": null\n    },\n    {\n      "id": "way/618533762",\n      "kind": "kindergarten",\n      "name": "Детский сад №2035",\n      "lat": 55.8685276,\n      "lon": 37.6329645,\n      "opening_hours": null\n    },\n    {\n      "id": "way/623564253",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8196852,\n      "lon": 37.5131849,\n      "opening_hours": null\n    },\n    {\n      "id": "way/624186249",\n      "kind": "school",\n      "name": "Школа №\xa0252",\n      "lat": 55.8149961,\n      "lon": 37.5944483,\n      "opening_hours": null\n    },\n    {\n      "id": "way/624783579",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.877561,\n      "lon": 37.5084158,\n      "opening_hours": null\n    },\n    {\n      "id": "way/627999917",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Здание 5",\n      "lat": 55.8175812,\n      "lon": 37.4897852,\n      "opening_hours": null\n    },\n    {\n      "id": "way/628537533",\n      "kind": "kindergarten",\n      "name": "Школа №1576, дошкольный корпус №14",\n      "lat": 55.8418732,\n      "lon": 37.5458069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/632644537",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6723207,\n      "lon": 37.5338421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/632661091",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6721082,\n      "lon": 37.5356219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/632661092",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.673893,\n      "lon": 37.5310235,\n      "opening_hours": null\n    },\n    {\n      "id": "way/635814039",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8928432,\n      "lon": 37.4438674,\n      "opening_hours": null\n    },\n    {\n      "id": "way/637885652",\n      "kind": "kindergarten",\n      "name": "Детский сад №358",\n      "lat": 55.6132369,\n      "lon": 37.7108268,\n      "opening_hours": null\n    },\n    {\n      "id": "way/637885653",\n      "kind": "school",\n      "name": "Интеллект XXI",\n      "lat": 55.6133997,\n      "lon": 37.7118931,\n      "opening_hours": null\n    },\n    {\n      "id": "way/640129189",\n      "kind": "kindergarten",\n      "name": "Школа № 498. Дошкольный корпус Тихий тупик, дом 2",\n      "lat": 55.7378528,\n      "lon": 37.6651454,\n      "opening_hours": null\n    },\n    {\n      "id": "way/641355965",\n      "kind": "school",\n      "name": "Кадетская школа №1770 (Московский кадетский музыкальный корпус)",\n      "lat": 55.676287,\n      "lon": 37.695895,\n      "opening_hours": null\n    },\n    {\n      "id": "way/642317749",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8843445,\n      "lon": 37.5610928,\n      "opening_hours": null\n    },\n    {\n      "id": "way/642492630",\n      "kind": "school",\n      "name": "Школа № 138. Корпус \\"Коллаж\\"",\n      "lat": 55.786609,\n      "lon": 37.4766223,\n      "opening_hours": null\n    },\n    {\n      "id": "way/643329688",\n      "kind": "school",\n      "name": "Гимназия №1517",\n      "lat": 55.7849844,\n      "lon": 37.4528217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644322701",\n      "kind": "kindergarten",\n      "name": "Детский сад №346",\n      "lat": 55.695234,\n      "lon": 37.7627508,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644322704",\n      "kind": "school",\n      "name": "ГБОУ СОШ № 479 Дошкольное отделение",\n      "lat": 55.6960289,\n      "lon": 37.7624681,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644512212",\n      "kind": "kindergarten",\n      "name": "Школа №641 им. С. Есенина с дошкольным отделением",\n      "lat": 55.6972906,\n      "lon": 37.7542788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644895848",\n      "kind": "kindergarten",\n      "name": "Школа №947, Начальная школа им. А. де Сент-Экзюпери",\n      "lat": 55.5856179,\n      "lon": 37.671176,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644895854",\n      "kind": "kindergarten",\n      "name": "Школа №947, Дошкольное отделение \\"Изумрудный город\\"",\n      "lat": 55.5848112,\n      "lon": 37.6701145,\n      "opening_hours": null\n    },\n    {\n      "id": "way/645305236",\n      "kind": "kindergarten",\n      "name": "Школа №641 им. С. Есенина с дошкольным отделением",\n      "lat": 55.7037793,\n      "lon": 37.7621648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/645707233",\n      "kind": "kindergarten",\n      "name": "Школа №1420 с дошкольным отделением",\n      "lat": 55.7007703,\n      "lon": 37.7891796,\n      "opening_hours": null\n    },\n    {\n      "id": "way/645707237",\n      "kind": "school",\n      "name": "Школа №1208 им. Героя Советского Союза М.С. Шумилова",\n      "lat": 55.6987748,\n      "lon": 37.7848959,\n      "opening_hours": null\n    },\n    {\n      "id": "way/647308761",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 6",\n      "lat": 55.670781,\n      "lon": 37.5638596,\n      "opening_hours": null\n    },\n    {\n      "id": "way/647423465",\n      "kind": "kindergarten",\n      "name": "Детский сад №817",\n      "lat": 55.716691,\n      "lon": 37.7795046,\n      "opening_hours": null\n    },\n    {\n      "id": "way/648098841",\n      "kind": "school",\n      "name": "Школа № 962. Структурное подразделение № 4",\n      "lat": 55.8650286,\n      "lon": 37.590761,\n      "opening_hours": null\n    },\n    {\n      "id": "way/648540752",\n      "kind": "kindergarten",\n      "name": "Детский сад №1013",\n      "lat": 55.6856803,\n      "lon": 37.7389819,\n      "opening_hours": null\n    },\n    {\n      "id": "way/650087440",\n      "kind": "school",\n      "name": "Школа №810",\n      "lat": 55.7244069,\n      "lon": 37.8054149,\n      "opening_hours": null\n    },\n    {\n      "id": "way/651507354",\n      "kind": "kindergarten",\n      "name": "Школа №1078 (Модуль №3)",\n      "lat": 55.8185884,\n      "lon": 37.8004966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/656569636",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6452536,\n      "lon": 37.7252021,\n      "opening_hours": null\n    },\n    {\n      "id": "way/656685223",\n      "kind": "school",\n      "name": "ГБОУ Школа №825",\n      "lat": 55.7100413,\n      "lon": 37.7540253,\n      "opening_hours": null\n    },\n    {\n      "id": "way/656929750",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1613",\n      "lat": 55.7209396,\n      "lon": 37.7809325,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500454",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №19",\n      "lat": 55.8241823,\n      "lon": 37.8060986,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500458",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №8",\n      "lat": 55.8269188,\n      "lon": 37.8269476,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500459",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №9",\n      "lat": 55.8224627,\n      "lon": 37.8315203,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500462",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №7",\n      "lat": 55.8224695,\n      "lon": 37.8298585,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500463",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №10",\n      "lat": 55.8224754,\n      "lon": 37.8283767,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659608154",\n      "kind": "kindergarten",\n      "name": "Школа № 1363, ДО Мозаика",\n      "lat": 55.7090544,\n      "lon": 37.820927,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659608346",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1363, корпус № 3",\n      "lat": 55.7083047,\n      "lon": 37.8217562,\n      "opening_hours": null\n    },\n    {\n      "id": "way/661148268",\n      "kind": "kindergarten",\n      "name": "Дошкольное подразделение школы №258",\n      "lat": 55.8644397,\n      "lon": 37.6108039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/661148271",\n      "kind": "school",\n      "name": "Гимназия №1554",\n      "lat": 55.8659096,\n      "lon": 37.6118085,\n      "opening_hours": null\n    },\n    {\n      "id": "way/661593268",\n      "kind": "kindergarten",\n      "name": "Школа № 1770. Корпус дошкольного образования «Бригантина»",\n      "lat": 55.6836191,\n      "lon": 37.6965265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/669759734",\n      "kind": "school",\n      "name": "Школа № 1101. Здание 2/2",\n      "lat": 55.6347404,\n      "lon": 37.4871522,\n      "opening_hours": null\n    },\n    {\n      "id": "way/674812006",\n      "kind": "school",\n      "name": "Школа № 814. Корпус 5",\n      "lat": 55.7140342,\n      "lon": 37.4814636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/682070168",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №152 дошкольное отделение",\n      "lat": 55.8068595,\n      "lon": 37.5337671,\n      "opening_hours": null\n    },\n    {\n      "id": "way/683111302",\n      "kind": "school",\n      "name": "Школа №\xa02025. Здание 3",\n      "lat": 55.6743777,\n      "lon": 37.4550086,\n      "opening_hours": null\n    },\n    {\n      "id": "way/684216820",\n      "kind": "school",\n      "name": "Школа Марьина роща им. В. Ф. Орлова",\n      "lat": 55.789119,\n      "lon": 37.6158138,\n      "opening_hours": null\n    },\n    {\n      "id": "way/684938116",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.6980046,\n      "lon": 37.4892972,\n      "opening_hours": null\n    },\n    {\n      "id": "way/686660334",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №2 \\"Улыбка\\"",\n      "lat": 55.8082904,\n      "lon": 37.8276224,\n      "opening_hours": null\n    },\n    {\n      "id": "way/686669814",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №3 \\"Умка\\"",\n      "lat": 55.8064227,\n      "lon": 37.8277862,\n      "opening_hours": null\n    },\n    {\n      "id": "way/686938091",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №1 \\"Теремок\\"",\n      "lat": 55.8103014,\n      "lon": 37.8274037,\n      "opening_hours": null\n    },\n    {\n      "id": "way/690243295",\n      "kind": "school",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №4",\n      "lat": 55.8083844,\n      "lon": 37.8216907,\n      "opening_hours": null\n    },\n    {\n      "id": "way/690812150",\n      "kind": "school",\n      "name": "Школа № 1494. Классы начального, основного и среднего общего образования",\n      "lat": 55.8450925,\n      "lon": 37.5816088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/691065085",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1353",\n      "lat": 55.8444875,\n      "lon": 37.5862854,\n      "opening_hours": null\n    },\n    {\n      "id": "way/693839893",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №5",\n      "lat": 55.6009517,\n      "lon": 37.5937766,\n      "opening_hours": null\n    },\n    {\n      "id": "way/694729851",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.7045586,\n      "lon": 37.5083282,\n      "opening_hours": null\n    },\n    {\n      "id": "way/704215334",\n      "kind": "school",\n      "name": "Школа № 830. Корпус № 2",\n      "lat": 55.8269759,\n      "lon": 37.4546993,\n      "opening_hours": null\n    },\n    {\n      "id": "way/704357047",\n      "kind": "school",\n      "name": "Московский кадетский корпус полиции",\n      "lat": 55.8441505,\n      "lon": 37.5445622,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708821993",\n      "kind": "kindergarten",\n      "name": "Детский сад №1907 (ортопедический)",\n      "lat": 55.7224953,\n      "lon": 37.8177979,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708821994",\n      "kind": "kindergarten",\n      "name": "Детский сад №1307",\n      "lat": 55.7226758,\n      "lon": 37.8154802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708821995",\n      "kind": "kindergarten",\n      "name": "Детский сад №1287",\n      "lat": 55.7190083,\n      "lon": 37.8168477,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708833031",\n      "kind": "kindergarten",\n      "name": "Дом ребенка №15",\n      "lat": 55.7361265,\n      "lon": 37.8335538,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708833032",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение К6",\n      "lat": 55.7429381,\n      "lon": 37.8235109,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708833033",\n      "kind": "school",\n      "name": "Вешняковская школа. Школьное отделение К8",\n      "lat": 55.7429746,\n      "lon": 37.825451,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708833035",\n      "kind": "school",\n      "name": "Вешняковская школа. Школьное отделение К10",\n      "lat": 55.7427065,\n      "lon": 37.8280743,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708929428",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8984304,\n      "lon": 37.6212814,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708929429",\n      "kind": "kindergarten",\n      "name": "Детский сад №820",\n      "lat": 55.8921888,\n      "lon": 37.5961004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709206068",\n      "kind": "school",\n      "name": "Школа №1502 \\"Энергия\\"",\n      "lat": 55.7919408,\n      "lon": 37.7132723,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709227297",\n      "kind": "kindergarten",\n      "name": "Детский сад №1195",\n      "lat": 55.8558715,\n      "lon": 37.6175702,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709227298",\n      "kind": "kindergarten",\n      "name": "Детский сад №1195",\n      "lat": 55.8565466,\n      "lon": 37.6184917,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709227304",\n      "kind": "school",\n      "name": "Школа №959",\n      "lat": 55.8606277,\n      "lon": 37.6087716,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709231665",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1128",\n      "lat": 55.8635375,\n      "lon": 37.6166178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709231668",\n      "kind": "school",\n      "name": "Гимназия №1554",\n      "lat": 55.8648387,\n      "lon": 37.6124939,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709231669",\n      "kind": "school",\n      "name": "Школа №258",\n      "lat": 55.8655541,\n      "lon": 37.6097456,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709236497",\n      "kind": "school",\n      "name": "Пенаты",\n      "lat": 55.8542638,\n      "lon": 37.5880798,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709265694",\n      "kind": "kindergarten",\n      "name": "Детский сад №1282",\n      "lat": 55.8692612,\n      "lon": 37.6414764,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709265695",\n      "kind": "school",\n      "name": "Разум-Л",\n      "lat": 55.8694434,\n      "lon": 37.6431078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709265696",\n      "kind": "school",\n      "name": "Центр образования №1482",\n      "lat": 55.8672369,\n      "lon": 37.6357424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709276384",\n      "kind": "kindergarten",\n      "name": "Детский сад №665",\n      "lat": 55.8918395,\n      "lon": 37.6571627,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709276385",\n      "kind": "school",\n      "name": "Детский дом № 5",\n      "lat": 55.89074,\n      "lon": 37.6502373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709548475",\n      "kind": "school",\n      "name": "Гимназия №1506",\n      "lat": 55.889765,\n      "lon": 37.6504661,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709548479",\n      "kind": "school",\n      "name": "ЧУ СОШ \\"Надежда\\"",\n      "lat": 55.8842774,\n      "lon": 37.609371,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709548480",\n      "kind": "kindergarten",\n      "name": "Детский сад Надежда",\n      "lat": 55.8846808,\n      "lon": 37.610653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709548485",\n      "kind": "kindergarten",\n      "name": "Детский сад №932",\n      "lat": 55.8919058,\n      "lon": 37.6021049,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709568857",\n      "kind": "kindergarten",\n      "name": "Социальный приют",\n      "lat": 55.6269318,\n      "lon": 37.7252742,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709568859",\n      "kind": "school",\n      "name": "Центр образования № 548",\n      "lat": 55.6237667,\n      "lon": 37.7015439,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580265",\n      "kind": "kindergarten",\n      "name": "Школа № 667. Корпус дошкольного образования №7",\n      "lat": 55.5856782,\n      "lon": 37.6448971,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580266",\n      "kind": "school",\n      "name": "Школа №933",\n      "lat": 55.5837199,\n      "lon": 37.6493815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580269",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.592495,\n      "lon": 37.6445906,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580271",\n      "kind": "kindergarten",\n      "name": "Детский сад №2471",\n      "lat": 55.5927515,\n      "lon": 37.6419358,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580272",\n      "kind": "school",\n      "name": "Школа №2001",\n      "lat": 55.5908387,\n      "lon": 37.6401232,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580273",\n      "kind": "kindergarten",\n      "name": "Детский сад №972",\n      "lat": 55.5897339,\n      "lon": 37.6430669,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709588441",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5913847,\n      "lon": 37.6051539,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709647798",\n      "kind": "school",\n      "name": "Гимназия №1554",\n      "lat": 55.8645463,\n      "lon": 37.6234977,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709647799",\n      "kind": "kindergarten",\n      "name": "Детский сад №1524",\n      "lat": 55.8627249,\n      "lon": 37.5875968,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709647800",\n      "kind": "kindergarten",\n      "name": "Детский сад №1933",\n      "lat": 55.7939269,\n      "lon": 37.7229808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/710644450",\n      "kind": "kindergarten",\n      "name": "Школа № 627 имени генерала Д. Д. Лелюшенко. Учебный корпус № 7",\n      "lat": 55.7272136,\n      "lon": 37.6210533,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279699",\n      "kind": "kindergarten",\n      "name": "Школа №947, Дошкольное отделение \\"Цветочный город\\"",\n      "lat": 55.5850066,\n      "lon": 37.6760841,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279700",\n      "kind": "school",\n      "name": "Школа №947, Начальная школа им Л. да Винчи",\n      "lat": 55.5879099,\n      "lon": 37.6673035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279701",\n      "kind": "kindergarten",\n      "name": "Школа №947, Дошкольное отделение \\"Лукоморье\\"",\n      "lat": 55.5875779,\n      "lon": 37.6713781,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279703",\n      "kind": "school",\n      "name": "Школа № 902 \\"Диалог\\", здание №1",\n      "lat": 55.5963153,\n      "lon": 37.6654733,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279704",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №6",\n      "lat": 55.5964405,\n      "lon": 37.6676727,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279705",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №7",\n      "lat": 55.5965485,\n      "lon": 37.6690315,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279706",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №4",\n      "lat": 55.5991447,\n      "lon": 37.6669189,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279707",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №5",\n      "lat": 55.5992447,\n      "lon": 37.6684337,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279708",\n      "kind": "kindergarten",\n      "name": "Детский сад №960",\n      "lat": 55.597429,\n      "lon": 37.6574516,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279709",\n      "kind": "kindergarten",\n      "name": "Детский сад №1972",\n      "lat": 55.5983525,\n      "lon": 37.6569542,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279710",\n      "kind": "school",\n      "name": "Школа №508",\n      "lat": 55.6005267,\n      "lon": 37.6554268,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287578",\n      "kind": "kindergarten",\n      "name": "Детский сад №2232",\n      "lat": 55.5838878,\n      "lon": 37.6513664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287579",\n      "kind": "kindergarten",\n      "name": "Детский сад №667",\n      "lat": 55.5853673,\n      "lon": 37.6511133,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287580",\n      "kind": "kindergarten",\n      "name": "Детский сад №834",\n      "lat": 55.5858187,\n      "lon": 37.6532842,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287581",\n      "kind": "kindergarten",\n      "name": "Детский сад №901",\n      "lat": 55.5904482,\n      "lon": 37.649388,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287582",\n      "kind": "kindergarten",\n      "name": "Детский сад №486",\n      "lat": 55.5931886,\n      "lon": 37.647639,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287583",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5999399,\n      "lon": 37.6149476,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287584",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №4",\n      "lat": 55.600395,\n      "lon": 37.6129039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287585",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.6005052,\n      "lon": 37.6109571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711301314",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7248448,\n      "lon": 37.801857,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304801",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №7",\n      "lat": 55.6049098,\n      "lon": 37.6032411,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304802",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №8",\n      "lat": 55.6078665,\n      "lon": 37.604734,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304803",\n      "kind": "kindergarten",\n      "name": "Детский сад №1318",\n      "lat": 55.6063973,\n      "lon": 37.5849161,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304804",\n      "kind": "kindergarten",\n      "name": "Детский сад №990",\n      "lat": 55.6094234,\n      "lon": 37.5811713,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304805",\n      "kind": "school",\n      "name": "Школа №652",\n      "lat": 55.6110709,\n      "lon": 37.5814015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304806",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №2",\n      "lat": 55.6391561,\n      "lon": 37.6128891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306802",\n      "kind": "school",\n      "name": "Школа №878 отделение №3",\n      "lat": 55.6169734,\n      "lon": 37.7087544,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306803",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1502",\n      "lat": 55.6223831,\n      "lon": 37.708636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306804",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1513",\n      "lat": 55.6246774,\n      "lon": 37.7047102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306805",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1637",\n      "lat": 55.626826,\n      "lon": 37.715952,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306806",\n      "kind": "kindergarten",\n      "name": "Детский сад №905",\n      "lat": 55.6238256,\n      "lon": 37.72416,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306807",\n      "kind": "school",\n      "name": "Школа №939",\n      "lat": 55.6223236,\n      "lon": 37.723614,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711388921",\n      "kind": "kindergarten",\n      "name": "Детский сад №1709",\n      "lat": 55.6106533,\n      "lon": 37.7265286,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711388922",\n      "kind": "school",\n      "name": "Гимназия №1636 НИКА",\n      "lat": 55.6110785,\n      "lon": 37.7288174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711588628",\n      "kind": "kindergarten",\n      "name": "Детский сад №413",\n      "lat": 55.8077059,\n      "lon": 37.7116342,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711588631",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8967409,\n      "lon": 37.6288236,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711588632",\n      "kind": "kindergarten",\n      "name": "Детский сад №531",\n      "lat": 55.8964179,\n      "lon": 37.6302669,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711588633",\n      "kind": "kindergarten",\n      "name": "Школа Глория, корпус № 8",\n      "lat": 55.8953672,\n      "lon": 37.6278932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/713630942",\n      "kind": "kindergarten",\n      "name": "Школа №2094. Дошкольная образовательная площадка №3",\n      "lat": 55.5906259,\n      "lon": 37.453851,\n      "opening_hours": null\n    },\n    {\n      "id": "way/722407298",\n      "kind": "school",\n      "name": "ОК \\"Юго-Запад\\"",\n      "lat": 55.68364,\n      "lon": 37.5932506,\n      "opening_hours": null\n    },\n    {\n      "id": "way/722633303",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1272 Академия детства",\n      "lat": 55.701769,\n      "lon": 37.6613565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/723104569",\n      "kind": "school",\n      "name": "Школа № 548 «Царицыно». Корпус школьного образования «ЗилАрт»",\n      "lat": 55.7010224,\n      "lon": 37.6387796,\n      "opening_hours": null\n    },\n    {\n      "id": "way/734561897",\n      "kind": "kindergarten",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г. П. Кравченко. Здание № 4",\n      "lat": 55.7113463,\n      "lon": 37.5979744,\n      "opening_hours": null\n    },\n    {\n      "id": "way/744280212",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение школы №224",\n      "lat": 55.8316249,\n      "lon": 37.5025618,\n      "opening_hours": null\n    },\n    {\n      "id": "way/744569171",\n      "kind": "kindergarten",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 8.",\n      "lat": 55.6470461,\n      "lon": 37.5209714,\n      "opening_hours": null\n    },\n    {\n      "id": "way/746494190",\n      "kind": "school",\n      "name": "Школа №978",\n      "lat": 55.6767103,\n      "lon": 37.6314956,\n      "opening_hours": null\n    },\n    {\n      "id": "way/756873354",\n      "kind": "school",\n      "name": "Школа № 15. Корпус школьного образования № 2",\n      "lat": 55.6552954,\n      "lon": 37.5624595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/756877742",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 2",\n      "lat": 55.6517768,\n      "lon": 37.5627347,\n      "opening_hours": null\n    },\n    {\n      "id": "way/760941887",\n      "kind": "school",\n      "name": "Школа №1601 им. Героя Советского Союза Е. К. Лютикова",\n      "lat": 55.7981571,\n      "lon": 37.5595217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/773586915",\n      "kind": "school",\n      "name": "ГБОУ города Москвы «Школа № 1601 имени Героя Советского Союза Е. К. Лютикова»",\n      "lat": 55.7918953,\n      "lon": 37.567021,\n      "opening_hours": null\n    },\n    {\n      "id": "way/777144344",\n      "kind": "kindergarten",\n      "name": "Школа № 1222 имени Маршала Советского Союза И. Х. Баграмяна. Дошкольное  отделение № 2",\n      "lat": 55.7322736,\n      "lon": 37.7219154,\n      "opening_hours": "Mo-Su 08:00-19:00"\n    },\n    {\n      "id": "way/784611808",\n      "kind": "school",\n      "name": "Школа на Яузе",\n      "lat": 55.7748598,\n      "lon": 37.7085442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/785862762",\n      "kind": "school",\n      "name": "Школа №\xa0568",\n      "lat": 55.6756404,\n      "lon": 37.5959945,\n      "opening_hours": null\n    },\n    {\n      "id": "way/805810950",\n      "kind": "kindergarten",\n      "name": "Детский сад №1108",\n      "lat": 55.7208497,\n      "lon": 37.5770152,\n      "opening_hours": null\n    },\n    {\n      "id": "way/805810951",\n      "kind": "kindergarten",\n      "name": "Детский сад №1093",\n      "lat": 55.720705,\n      "lon": 37.575624,\n      "opening_hours": null\n    },\n    {\n      "id": "way/806227829",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7187018,\n      "lon": 37.5786616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/806227830",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7176747,\n      "lon": 37.5772695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/813559738",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Европейская гимназия\\"",\n      "lat": 55.7865816,\n      "lon": 37.6840973,\n      "opening_hours": null\n    },\n    {\n      "id": "way/814267642",\n      "kind": "school",\n      "name": "СААШ \\"Марина\\"",\n      "lat": 55.6820311,\n      "lon": 37.5415069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/814995890",\n      "kind": "kindergarten",\n      "name": "Детский сад №1708",\n      "lat": 55.6795924,\n      "lon": 37.535807,\n      "opening_hours": null\n    },\n    {\n      "id": "way/820333794",\n      "kind": "school",\n      "name": "Школа № 1507. Корпус школьного образования",\n      "lat": 55.6329838,\n      "lon": 37.509036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/824290373",\n      "kind": "kindergarten",\n      "name": "Школа № 1321 \\"Ковчег\\". Дошкольное отделение здание №3",\n      "lat": 55.750761,\n      "lon": 37.7186581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/845403776",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №5",\n      "lat": 55.6061184,\n      "lon": 37.5404105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/851409274",\n      "kind": "school",\n      "name": "Прогимназия №1774",\n      "lat": 55.8197589,\n      "lon": 37.5111443,\n      "opening_hours": null\n    },\n    {\n      "id": "way/853678162",\n      "kind": "kindergarten",\n      "name": "Детский сад №1446",\n      "lat": 55.8275213,\n      "lon": 37.575913,\n      "opening_hours": null\n    },\n    {\n      "id": "way/860122160",\n      "kind": "kindergarten",\n      "name": "Детский сад №1746",\n      "lat": 55.6822542,\n      "lon": 37.7417905,\n      "opening_hours": null\n    },\n    {\n      "id": "way/867853165",\n      "kind": "kindergarten",\n      "name": "Детский сад №2142",\n      "lat": 55.8235066,\n      "lon": 37.6492761,\n      "opening_hours": null\n    },\n    {\n      "id": "way/870571581",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №7 \\"Солнышко\\"",\n      "lat": 55.810496,\n      "lon": 37.8218343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/874223859",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 10 (дошкольное отделение)",\n      "lat": 55.8060129,\n      "lon": 37.814694,\n      "opening_hours": null\n    },\n    {\n      "id": "way/874415684",\n      "kind": "kindergarten",\n      "name": "Школа № 2107. Здание школы \\"На Напрудном\\"",\n      "lat": 55.7886075,\n      "lon": 37.63108,\n      "opening_hours": null\n    },\n    {\n      "id": "way/874475981",\n      "kind": "school",\n      "name": "Школа №2200 корпус 8",\n      "lat": 55.8039545,\n      "lon": 37.8153901,\n      "opening_hours": null\n    },\n    {\n      "id": "way/874983433",\n      "kind": "school",\n      "name": "Школа №2033 (начальная школа)",\n      "lat": 55.8059649,\n      "lon": 37.8104058,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878383228",\n      "kind": "kindergarten",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 11",\n      "lat": 55.7977501,\n      "lon": 37.8199268,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878384754",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8003224,\n      "lon": 37.813406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878548821",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №8 \\"Золотая рыбка\\"",\n      "lat": 55.8095872,\n      "lon": 37.82048,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878690744",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 11 (дошкольное отделение)",\n      "lat": 55.8060318,\n      "lon": 37.815974,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878979417",\n      "kind": "school",\n      "name": "Школа №1290",\n      "lat": 55.7939863,\n      "lon": 37.7931653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878979512",\n      "kind": "school",\n      "name": "Измайловская гимназия №1508",\n      "lat": 55.7939085,\n      "lon": 37.7962294,\n      "opening_hours": null\n    },\n    {\n      "id": "way/879568393",\n      "kind": "school",\n      "name": "Школа № 1222 имени Маршала Советского Союза И. Х. Баграмяна. Школьное отделение № 2",\n      "lat": 55.7322892,\n      "lon": 37.7200179,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "way/879949379",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Яснополянской",\n      "lat": 55.7231819,\n      "lon": 37.7903059,\n      "opening_hours": null\n    },\n    {\n      "id": "way/880734466",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Радость\\"",\n      "lat": 55.7306236,\n      "lon": 37.766762,\n      "opening_hours": null\n    },\n    {\n      "id": "way/880734467",\n      "kind": "kindergarten",\n      "name": "Образовательная школа \\"Академическая гимназия\\"",\n      "lat": 55.7301615,\n      "lon": 37.7684411,\n      "opening_hours": null\n    },\n    {\n      "id": "way/880735669",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Маёвок",\n      "lat": 55.7285241,\n      "lon": 37.7696011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881052775",\n      "kind": "kindergarten",\n      "name": "Детский сад №872",\n      "lat": 55.6878981,\n      "lon": 37.7212016,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881124918",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Шатурской",\n      "lat": 55.7300836,\n      "lon": 37.765602,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881223214",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Фокус\\"",\n      "lat": 55.7222851,\n      "lon": 37.7853544,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881223348",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Семицветик\\"",\n      "lat": 55.721046,\n      "lon": 37.7843286,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881231366",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Михайлова",\n      "lat": 55.725454,\n      "lon": 37.7682396,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881232227",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Звёздочка\\"",\n      "lat": 55.7248986,\n      "lon": 37.7699148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881232954",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Лукоморье\\"",\n      "lat": 55.7274065,\n      "lon": 37.760757,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881325381",\n      "kind": "kindergarten",\n      "name": "Детский сад №2027",\n      "lat": 55.7514368,\n      "lon": 37.8289433,\n      "opening_hours": null\n    },\n    {\n      "id": "way/908462514",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6518802,\n      "lon": 37.7114786,\n      "opening_hours": null\n    },\n    {\n      "id": "way/929277115",\n      "kind": "school",\n      "name": "ГБОУ СОШ № 2105 \\"Школа на Яузе\\" корпус №1",\n      "lat": 55.7789746,\n      "lon": 37.7062422,\n      "opening_hours": null\n    },\n    {\n      "id": "way/937806650",\n      "kind": "kindergarten",\n      "name": "Школа № 1362. Корпус дошкольного образования",\n      "lat": 55.7754397,\n      "lon": 37.736683,\n      "opening_hours": "Mo-Fr 07:00-20:00"\n    },\n    {\n      "id": "way/943940531",\n      "kind": "school",\n      "name": "Школа №1498 «ММШ-23». Корпус школьного образования",\n      "lat": 55.6946266,\n      "lon": 37.506344,\n      "opening_hours": null\n    },\n    {\n      "id": "way/945400139",\n      "kind": "kindergarten",\n      "name": "Лицей №1571 дошкольное отделение Лучики",\n      "lat": 55.8546815,\n      "lon": 37.4409175,\n      "opening_hours": null\n    },\n    {\n      "id": "way/946129394",\n      "kind": "kindergarten",\n      "name": "Школа №1056 (дошкольное отделение №2)",\n      "lat": 55.8470934,\n      "lon": 37.4410136,\n      "opening_hours": null\n    },\n    {\n      "id": "way/952069499",\n      "kind": "kindergarten",\n      "name": "Школа № 1517. Дошкольное отделение",\n      "lat": 55.7799074,\n      "lon": 37.4588802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/952784277",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6218133,\n      "lon": 37.6534386,\n      "opening_hours": null\n    },\n    {\n      "id": "way/952784421",\n      "kind": "kindergarten",\n      "name": "Д/C № 1168",\n      "lat": 55.6221106,\n      "lon": 37.6549616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/958193582",\n      "kind": "kindergarten",\n      "name": "Школа № 1272. Корпус дошкольного образования",\n      "lat": 55.7013421,\n      "lon": 37.6362218,\n      "opening_hours": null\n    },\n    {\n      "id": "way/964951015",\n      "kind": "kindergarten",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 10.",\n      "lat": 55.6503467,\n      "lon": 37.5277066,\n      "opening_hours": null\n    },\n    {\n      "id": "way/964951021",\n      "kind": "school",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 5.",\n      "lat": 55.6502546,\n      "lon": 37.5252847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/964951027",\n      "kind": "kindergarten",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 9.",\n      "lat": 55.6510375,\n      "lon": 37.5241165,\n      "opening_hours": null\n    },\n    {\n      "id": "way/965961813",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Фруктовая радуга»",\n      "lat": 55.6266,\n      "lon": 37.4800534,\n      "opening_hours": null\n    },\n    {\n      "id": "way/965961825",\n      "kind": "school",\n      "name": "Школа № 1101. Здание 2/3",\n      "lat": 55.6286959,\n      "lon": 37.4816317,\n      "opening_hours": null\n    },\n    {\n      "id": "way/965961888",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6336695,\n      "lon": 37.4767149,\n      "opening_hours": null\n    },\n    {\n      "id": "way/966197845",\n      "kind": "school",\n      "name": "Школа № 1694 «Ясенево». Здание 1",\n      "lat": 55.6130815,\n      "lon": 37.5450963,\n      "opening_hours": null\n    },\n    {\n      "id": "way/966227910",\n      "kind": "kindergarten",\n      "name": "Школа № 1561. Дошкольное отделение корпус №6",\n      "lat": 55.5971756,\n      "lon": 37.5430883,\n      "opening_hours": null\n    },\n    {\n      "id": "way/966227926",\n      "kind": "school",\n      "name": "Школа №1561, начальная школа, корпус 4",\n      "lat": 55.6019397,\n      "lon": 37.5438795,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967847042",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №4",\n      "lat": 55.6383727,\n      "lon": 37.5136462,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967917167",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6232121,\n      "lon": 37.5037784,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967917171",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6237342,\n      "lon": 37.5022421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967917190",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6273145,\n      "lon": 37.5084937,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967917191",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6278657,\n      "lon": 37.5069219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972027575",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/2",\n      "lat": 55.6150753,\n      "lon": 37.4935368,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972027578",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/3",\n      "lat": 55.6179576,\n      "lon": 37.4985487,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972027586",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/4",\n      "lat": 55.6176937,\n      "lon": 37.4964275,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972027597",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/6",\n      "lat": 55.6198426,\n      "lon": 37.4898063,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972047403",\n      "kind": "kindergarten",\n      "name": "Школа №2094. Дошкольная образовательная площадка №2",\n      "lat": 55.6125285,\n      "lon": 37.4724735,\n      "opening_hours": null\n    },\n    {\n      "id": "way/975646976",\n      "kind": "school",\n      "name": "Школа №90 «Многопрофильный образовательный комплекс» имени Героя Советского Союза Е. Г. Ларикова",\n      "lat": 55.8220524,\n      "lon": 37.5041657,\n      "opening_hours": null\n    },\n    {\n      "id": "way/976037691",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №6 «Андреевский»",\n      "lat": 55.59211,\n      "lon": 37.5376585,\n      "opening_hours": null\n    },\n    {\n      "id": "way/976678942",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 224 дошкольное отделение",\n      "lat": 55.8348462,\n      "lon": 37.5067296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/977876539",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Загородке\\"",\n      "lat": 55.6933174,\n      "lon": 37.6036611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/980596512",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №11 «Наш дом»",\n      "lat": 55.6094806,\n      "lon": 37.5399204,\n      "opening_hours": null\n    },\n    {\n      "id": "way/985558075",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Мир детства»",\n      "lat": 55.6655656,\n      "lon": 37.4913243,\n      "opening_hours": null\n    },\n    {\n      "id": "way/985558079",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Содружество»",\n      "lat": 55.6651349,\n      "lon": 37.4973504,\n      "opening_hours": null\n    },\n    {\n      "id": "way/985565307",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Радуга»",\n      "lat": 55.6601085,\n      "lon": 37.4931627,\n      "opening_hours": null\n    },\n    {\n      "id": "way/985565313",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Сказка»",\n      "lat": 55.6614491,\n      "lon": 37.4905988,\n      "opening_hours": null\n    },\n    {\n      "id": "way/986753456",\n      "kind": "kindergarten",\n      "name": "Детский сад №299",\n      "lat": 55.7754649,\n      "lon": 37.6929529,\n      "opening_hours": null\n    },\n    {\n      "id": "way/987757025",\n      "kind": "school",\n      "name": "Школа на проспекте Вернадского. Корпус школьного образования 3",\n      "lat": 55.6538607,\n      "lon": 37.4856074,\n      "opening_hours": null\n    },\n    {\n      "id": "way/987757026",\n      "kind": "school",\n      "name": "Школа на проспекте Вернадского. Корпус школьного образования 2",\n      "lat": 55.6526647,\n      "lon": 37.4851661,\n      "opening_hours": null\n    },\n    {\n      "id": "way/991278392",\n      "kind": "school",\n      "name": "Школа №\xa01247",\n      "lat": 55.7710673,\n      "lon": 37.6879321,\n      "opening_hours": null\n    },\n    {\n      "id": "way/993985457",\n      "kind": "school",\n      "name": "Лицей №6",\n      "lat": 55.8914902,\n      "lon": 37.4719645,\n      "opening_hours": "Mo-Sa 08:00-18:00"\n    },\n    {\n      "id": "way/995333442",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Островок\\"",\n      "lat": 55.8012865,\n      "lon": 37.8154756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/995337699",\n      "kind": "kindergarten",\n      "name": "Детский сад №1651",\n      "lat": 55.815968,\n      "lon": 37.8023462,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1007452119",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6485861,\n      "lon": 37.7321824,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1022286575",\n      "kind": "school",\n      "name": "Школа № 2115. Корпус школьного образования № 2",\n      "lat": 55.6695321,\n      "lon": 37.5714823,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1024554009",\n      "kind": "kindergarten",\n      "name": "Школа № 324 «Жар-птица». Корпус дошкольного образования «Солнышко»",\n      "lat": 55.6785907,\n      "lon": 37.4975735,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1024555865",\n      "kind": "kindergarten",\n      "name": "Школа № 324 «Жар-птица». Корпус дошкольного образования «Садко»",\n      "lat": 55.6754806,\n      "lon": 37.4968372,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1029616464",\n      "kind": "school",\n      "name": "Школа № 2055. Учебный корпус",\n      "lat": 55.7627449,\n      "lon": 37.5467382,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1033315598",\n      "kind": "school",\n      "name": "Центр образования № 1699",\n      "lat": 55.7350632,\n      "lon": 37.7457263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1039436694",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольная группа №9",\n      "lat": 55.7420548,\n      "lon": 37.8243971,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1052414608",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7619091,\n      "lon": 37.5152847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1056570456",\n      "kind": "kindergarten",\n      "name": "Детский сад №1865",\n      "lat": 55.7753786,\n      "lon": 37.689577,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1056570460",\n      "kind": "kindergarten",\n      "name": "Детский сад №1976",\n      "lat": 55.7732997,\n      "lon": 37.6895622,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1068412729",\n      "kind": "kindergarten",\n      "name": "Школа № 1454 «Тимирязевская», дошкольное отделение «Дубки»",\n      "lat": 55.8233506,\n      "lon": 37.5615985,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1069628566",\n      "kind": "kindergarten",\n      "name": "Детский сад №2543",\n      "lat": 55.6938567,\n      "lon": 37.7207985,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1076569790",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8713653,\n      "lon": 37.4887808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1078433003",\n      "kind": "kindergarten",\n      "name": "Школа № 830. Дошкольное отделение \\"Крепыш\\"",\n      "lat": 55.8247815,\n      "lon": 37.4532705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1079522044",\n      "kind": "kindergarten",\n      "name": "Школа № 2121 имени Маршала Советского Союза С.К. Куркоткина. Дошкольный корпус № 4 \\"Кораблик\\"",\n      "lat": 55.6744005,\n      "lon": 37.7508414,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1086788732",\n      "kind": "kindergarten",\n      "name": "Школа № 1222 имени Маршала Советского Союза И. Х. Баграмяна. Дошкольное отделение № 3",\n      "lat": 55.7334197,\n      "lon": 37.7201807,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1090983005",\n      "kind": "school",\n      "name": "Второй корпус школы № 1384",\n      "lat": 55.8025114,\n      "lon": 37.5158186,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1093229206",\n      "kind": "school",\n      "name": "Школа №641",\n      "lat": 55.7042448,\n      "lon": 37.7572998,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1094253413",\n      "kind": "kindergarten",\n      "name": "Лицей №1571 дошкольное отделение Конфетти",\n      "lat": 55.8532562,\n      "lon": 37.4470517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562107",\n      "kind": "school",\n      "name": "Лицей №1524",\n      "lat": 55.6740384,\n      "lon": 37.7570109,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562108",\n      "kind": "kindergarten",\n      "name": "Детский сад №616",\n      "lat": 55.675768,\n      "lon": 37.7554346,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562109",\n      "kind": "kindergarten",\n      "name": "Детский сад №1821",\n      "lat": 55.6749303,\n      "lon": 37.7470481,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562110",\n      "kind": "school",\n      "name": "НОУ Школа \\"Карьера\\"",\n      "lat": 55.6776009,\n      "lon": 37.7429604,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562111",\n      "kind": "school",\n      "name": "Школа №1148",\n      "lat": 55.6783193,\n      "lon": 37.7424465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562112",\n      "kind": "school",\n      "name": "НОУ Школа \\"Интеллект\\"",\n      "lat": 55.6801717,\n      "lon": 37.7481319,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562113",\n      "kind": "kindergarten",\n      "name": "Детский сад №2129",\n      "lat": 55.6829347,\n      "lon": 37.7470999,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563380",\n      "kind": "kindergarten",\n      "name": "Детский сад №499",\n      "lat": 55.7016697,\n      "lon": 37.7439589,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563381",\n      "kind": "school",\n      "name": "Школа №654 имени А. Д. Фридмана. Учебный корпус №2",\n      "lat": 55.7010935,\n      "lon": 37.7447165,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563382",\n      "kind": "school",\n      "name": "Школа №687",\n      "lat": 55.701199,\n      "lon": 37.7409905,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563383",\n      "kind": "kindergarten",\n      "name": "Детский сад №884",\n      "lat": 55.7028332,\n      "lon": 37.7395109,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563384",\n      "kind": "kindergarten",\n      "name": "Детский сад №88",\n      "lat": 55.7023772,\n      "lon": 37.7401763,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096324017",\n      "kind": "kindergarten",\n      "name": "Школа №654 имени А. Д. Фридмана. Дошкольное отделение №4",\n      "lat": 55.6952483,\n      "lon": 37.7413111,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096324021",\n      "kind": "kindergarten",\n      "name": "Школа №654 имени А. Д. Фридмана. Дошкольное отделение №5",\n      "lat": 55.6958816,\n      "lon": 37.7415304,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096324034",\n      "kind": "school",\n      "name": "Школа №654 имени А. Д. Фридмана. Учебный корпус №1",\n      "lat": 55.6978464,\n      "lon": 37.7399458,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096326152",\n      "kind": "school",\n      "name": "Спортшкола № 64",\n      "lat": 55.7009693,\n      "lon": 37.7810595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096326153",\n      "kind": "kindergarten",\n      "name": "Детский сад № 149",\n      "lat": 55.7003701,\n      "lon": 37.772826,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096326154",\n      "kind": "school",\n      "name": "Школа № 333",\n      "lat": 55.7037588,\n      "lon": 37.7602107,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097355893",\n      "kind": "school",\n      "name": "Школа №338",\n      "lat": 55.6752795,\n      "lon": 37.7415239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097358480",\n      "kind": "school",\n      "name": "Школа №482",\n      "lat": 55.7018149,\n      "lon": 37.7363211,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097447006",\n      "kind": "school",\n      "name": "Школа № 1469",\n      "lat": 55.7021938,\n      "lon": 37.754003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097447008",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7019521,\n      "lon": 37.7555023,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097448183",\n      "kind": "kindergarten",\n      "name": "Детский сад №183",\n      "lat": 55.7052188,\n      "lon": 37.7516711,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097682346",\n      "kind": "school",\n      "name": "Центр образования №2121-начальная школа",\n      "lat": 55.6798684,\n      "lon": 37.7519334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097682347",\n      "kind": "school",\n      "name": "Центр образования №2121",\n      "lat": 55.6786958,\n      "lon": 37.7520085,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097701062",\n      "kind": "school",\n      "name": "Школа №480 им. В.В. Талалихина",\n      "lat": 55.7354192,\n      "lon": 37.6846312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1118849806",\n      "kind": "kindergarten",\n      "name": "Школа №1362 дошкольный корпус",\n      "lat": 55.7918528,\n      "lon": 37.7343115,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1120376459",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa0832",\n      "lat": 55.827139,\n      "lon": 37.5834897,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1120376460",\n      "kind": "school",\n      "name": "Школа №\xa0968",\n      "lat": 55.826211,\n      "lon": 37.5851886,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1120517398",\n      "kind": "kindergarten",\n      "name": "Детский сад №425",\n      "lat": 55.8164913,\n      "lon": 37.5854418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1122007526",\n      "kind": "school",\n      "name": "Школа № 625. Начальная и средняя школа",\n      "lat": 55.6894775,\n      "lon": 37.5888729,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1134331221",\n      "kind": "kindergarten",\n      "name": "Детский сад №5",\n      "lat": 55.8201318,\n      "lon": 37.5889146,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1152214665",\n      "kind": "school",\n      "name": "Школа № 1586. Учебный корпус № 1",\n      "lat": 55.7092526,\n      "lon": 37.5134592,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1154845325",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7432396,\n      "lon": 37.802756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1156606030",\n      "kind": "kindergarten",\n      "name": "Школа № 1236 имени С.В. Милашенкова. Учебный корпус №121",\n      "lat": 55.8219526,\n      "lon": 37.5816334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1157204034",\n      "kind": "kindergarten",\n      "name": "ДОУ №64",\n      "lat": 55.6784299,\n      "lon": 37.6279123,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1163425107",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8827442,\n      "lon": 37.6377306,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1168924918",\n      "kind": "kindergarten",\n      "name": "Детский сад №154 МЖД МПС РФ",\n      "lat": 55.7401194,\n      "lon": 37.5483008,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1176375533",\n      "kind": "school",\n      "name": "Школа № 1508. Корпус Софьи Ковалевской",\n      "lat": 55.8018027,\n      "lon": 37.79074,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1178180614",\n      "kind": "kindergarten",\n      "name": "Детский сад №461",\n      "lat": 55.802211,\n      "lon": 37.7962243,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1180159484",\n      "kind": "school",\n      "name": "начальная школа «Интеграция XXI век»",\n      "lat": 55.8009481,\n      "lon": 37.4898015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1182571822",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6391524,\n      "lon": 37.798654,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1188713166",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7150868,\n      "lon": 37.754811,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1198830522",\n      "kind": "school",\n      "name": "Бауманская инженерная школа № 1580",\n      "lat": 55.7502931,\n      "lon": 37.7018673,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1198908691",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №2099",\n      "lat": 55.6792155,\n      "lon": 37.7188036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1198908692",\n      "kind": "school",\n      "name": "Школа №\xa01041",\n      "lat": 55.677969,\n      "lon": 37.72018,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1198909608",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6800556,\n      "lon": 37.7199422,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1199316488",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6647467,\n      "lon": 37.7318461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1214119130",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад № 1620",\n      "lat": 55.6822661,\n      "lon": 37.5334921,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1216010923",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8256143,\n      "lon": 37.5248514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1226935123",\n      "kind": "kindergarten",\n      "name": "Протон",\n      "lat": 55.7394294,\n      "lon": 37.5020085,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1230105741",\n      "kind": "school",\n      "name": "Новое образование",\n      "lat": 55.7337701,\n      "lon": 37.7163769,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1235854567",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6862176,\n      "lon": 37.5393207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1242441869",\n      "kind": "kindergarten",\n      "name": "Детский сад №2149",\n      "lat": 55.8229795,\n      "lon": 37.5767771,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1243054430",\n      "kind": "kindergarten",\n      "name": "Детский сад №1832",\n      "lat": 55.8209024,\n      "lon": 37.5769887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1246044684",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7474048,\n      "lon": 37.5516422,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1249918852",\n      "kind": "school",\n      "name": "Путь зерна",\n      "lat": 55.6435668,\n      "lon": 37.5189347,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1255000953",\n      "kind": "school",\n      "name": "Школа №1543. Корпус школьного образования",\n      "lat": 55.6556246,\n      "lon": 37.486239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1257008884",\n      "kind": "school",\n      "name": "Школа № 1213",\n      "lat": 55.8397827,\n      "lon": 37.5761873,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1259771713",\n      "kind": "school",\n      "name": "Школа № 1324",\n      "lat": 55.7586314,\n      "lon": 37.8237309,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1259810311",\n      "kind": "kindergarten",\n      "name": "Школа № 185, дошкольное отделение",\n      "lat": 55.8391121,\n      "lon": 37.5419756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1262490343",\n      "kind": "school",\n      "name": "Школа №439 \\"Интеллект\\" (корпус №1)",\n      "lat": 55.7311773,\n      "lon": 37.7453032,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1263868352",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7318894,\n      "lon": 37.7586297,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1267863905",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7552717,\n      "lon": 37.5148694,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1268336549",\n      "kind": "school",\n      "name": "Кадетская школа №1785",\n      "lat": 55.7358086,\n      "lon": 37.6917636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1274403627",\n      "kind": "school",\n      "name": "Новая школа",\n      "lat": 55.7049909,\n      "lon": 37.4899723,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1276348994",\n      "kind": "school",\n      "name": "Школа №\xa01579",\n      "lat": 55.6503976,\n      "lon": 37.632768,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1278312025",\n      "kind": "kindergarten",\n      "name": "Детский сад №1174",\n      "lat": 55.7586329,\n      "lon": 37.7842092,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1285516011",\n      "kind": "school",\n      "name": "Школа № 463 им. Д. Н. Медведева",\n      "lat": 55.7219225,\n      "lon": 37.6617855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1289119543",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6516216,\n      "lon": 37.6311792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1302584894",\n      "kind": "kindergarten",\n      "name": "Школа № 1363, ДО Незабудка",\n      "lat": 55.69349,\n      "lon": 37.821866,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1303268971",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6747159,\n      "lon": 37.8094696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1305195718",\n      "kind": "school",\n      "name": "Школа №1637",\n      "lat": 55.7594673,\n      "lon": 37.7837123,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1312881247",\n      "kind": "school",\n      "name": "Школа № 2090",\n      "lat": 55.7093043,\n      "lon": 37.7924488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1313956567",\n      "kind": "school",\n      "name": "Школа № 463 имени Героя Советского Союза Д.Н. Медведева",\n      "lat": 55.6900128,\n      "lon": 37.6682002,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1329403077",\n      "kind": "kindergarten",\n      "name": "Детский сад №1770, Школа \\"Наши Пенаты\\"",\n      "lat": 55.6492061,\n      "lon": 37.5275594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1334478416",\n      "kind": "kindergarten",\n      "name": "Школа №1576. Дошкольное отделение №10",\n      "lat": 55.8315226,\n      "lon": 37.5286559,\n      "opening_hours": null\n    },\n    {\n      "id": "node/416876296",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6839716,\n      "lon": 37.5734071,\n      "opening_hours": null\n    },\n    {\n      "id": "node/517247653",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8852577,\n      "lon": 37.7469699,\n      "opening_hours": null\n    },\n    {\n      "id": "node/643737879",\n      "kind": "kindergarten",\n      "name": "Школа №439 \\"Интеллект\\" (дошкольное отделение №3)",\n      "lat": 55.735387,\n      "lon": 37.7470929,\n      "opening_hours": null\n    },\n    {\n      "id": "node/745608832",\n      "kind": "kindergarten",\n      "name": "Детский сад №1071",\n      "lat": 55.7931586,\n      "lon": 37.5080654,\n      "opening_hours": null\n    },\n    {\n      "id": "node/949916341",\n      "kind": "kindergarten",\n      "name": "Детский сад №2143",\n      "lat": 55.8059612,\n      "lon": 37.6377796,\n      "opening_hours": null\n    },\n    {\n      "id": "node/954912765",\n      "kind": "school",\n      "name": "Школа №1",\n      "lat": 55.6890655,\n      "lon": 37.5378735,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1013879417",\n      "kind": "school",\n      "name": "Московская международная гимназия",\n      "lat": 55.755926,\n      "lon": 37.8012033,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1072766433",\n      "kind": "kindergarten",\n      "name": "Центр Родник",\n      "lat": 55.707117,\n      "lon": 37.6588989,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1072770629",\n      "kind": "kindergarten",\n      "name": "Жемчужина",\n      "lat": 55.7070518,\n      "lon": 37.6585598,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1106954251",\n      "kind": "kindergarten",\n      "name": "Детский сад №1309",\n      "lat": 55.8133975,\n      "lon": 37.639607,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1141776015",\n      "kind": "kindergarten",\n      "name": "Школа 1574 Дошкольное отделение",\n      "lat": 55.7733226,\n      "lon": 37.6004213,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1142158796",\n      "kind": "school",\n      "name": "Школа №1362",\n      "lat": 55.7898108,\n      "lon": 37.7410624,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1145370005",\n      "kind": "school",\n      "name": "Средняя школа №72",\n      "lat": 55.7475855,\n      "lon": 37.4985787,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1147951174",\n      "kind": "kindergarten",\n      "name": "Детский сад №252",\n      "lat": 55.6934079,\n      "lon": 37.585788,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1192496379",\n      "kind": "school",\n      "name": "Школа надомного обучения №381",\n      "lat": 55.8047269,\n      "lon": 37.7928053,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1252450767",\n      "kind": "kindergarten",\n      "name": "Детский сад №1321",\n      "lat": 55.7222138,\n      "lon": 37.6662232,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1274329843",\n      "kind": "kindergarten",\n      "name": "Школа № 67. Отделение дошкольного образования № 4",\n      "lat": 55.7372044,\n      "lon": 37.5328738,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1277580872",\n      "kind": "kindergarten",\n      "name": "Детский сад №1123",\n      "lat": 55.6904012,\n      "lon": 37.5451365,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1288895160",\n      "kind": "kindergarten",\n      "name": "Детский сад №3",\n      "lat": 55.793622,\n      "lon": 37.4818594,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1333238486",\n      "kind": "kindergarten",\n      "name": "Детский сад компенсирующего вида №1093",\n      "lat": 55.7207776,\n      "lon": 37.5750012,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1333238570",\n      "kind": "kindergarten",\n      "name": "Детский сад №1359",\n      "lat": 55.7185682,\n      "lon": 37.5796113,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1355396778",\n      "kind": "kindergarten",\n      "name": "Почемучка",\n      "lat": 55.6446753,\n      "lon": 37.5198495,\n      "opening_hours": "Mo-Su 10:00-18:00"\n    },\n    {\n      "id": "node/1373666635",\n      "kind": "school",\n      "name": "Школа № 7, филиал № 1",\n      "lat": 55.6791957,\n      "lon": 37.5253409,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669698",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1136",\n      "lat": 55.6925084,\n      "lon": 37.5397693,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669716",\n      "kind": "kindergarten",\n      "name": "Детский сад №2042",\n      "lat": 55.6940211,\n      "lon": 37.554457,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669718",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2558",\n      "lat": 55.6862785,\n      "lon": 37.5379804,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669722",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2558",\n      "lat": 55.6872354,\n      "lon": 37.5396966,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669725",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2558",\n      "lat": 55.6899587,\n      "lon": 37.5391276,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669729",\n      "kind": "kindergarten",\n      "name": "Детский сад №4",\n      "lat": 55.6998837,\n      "lon": 37.5549115,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669735",\n      "kind": "school",\n      "name": "Н.ш. - д.с. № 1708",\n      "lat": 55.6853591,\n      "lon": 37.5386444,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669736",\n      "kind": "kindergarten",\n      "name": "Начальная школа, д/с №1851",\n      "lat": 55.6914932,\n      "lon": 37.5361552,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694626",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад №1620",\n      "lat": 55.6812095,\n      "lon": 37.5279342,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694627",\n      "kind": "school",\n      "name": "Начальная школа-детский сад №1620",\n      "lat": 55.6814973,\n      "lon": 37.5295702,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694628",\n      "kind": "school",\n      "name": "Начальная школа-детский сад №1620",\n      "lat": 55.6824634,\n      "lon": 37.5250336,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694629",\n      "kind": "school",\n      "name": "Школа №7 (филиал №2)",\n      "lat": 55.6829241,\n      "lon": 37.5270539,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694630",\n      "kind": "school",\n      "name": "Школа №119",\n      "lat": 55.6798473,\n      "lon": 37.5213993,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373755234",\n      "kind": "school",\n      "name": "СААШ \\"Марина\\"",\n      "lat": 55.6763945,\n      "lon": 37.5207685,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373756112",\n      "kind": "kindergarten",\n      "name": "Детский сад №375",\n      "lat": 55.668598,\n      "lon": 37.5233243,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373756113",\n      "kind": "school",\n      "name": "Школа 1181 (для пациентов больницы)",\n      "lat": 55.6510102,\n      "lon": 37.493042,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373768269",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1041",\n      "lat": 55.6854919,\n      "lon": 37.5637465,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373768284",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2171",\n      "lat": 55.6798608,\n      "lon": 37.5641675,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373768288",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2171",\n      "lat": 55.6810221,\n      "lon": 37.565399,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373776071",\n      "kind": "school",\n      "name": "Лицей №1525 \\"Воробьевы горы\\"",\n      "lat": 55.7029962,\n      "lon": 37.5544024,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373778976",\n      "kind": "school",\n      "name": "Школа № 1318 (киношкола)",\n      "lat": 55.6870452,\n      "lon": 37.5298416,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373825239",\n      "kind": "school",\n      "name": "Центр дополнительного образования детей \\"Диалог наук\\"",\n      "lat": 55.6950395,\n      "lon": 37.5518003,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373831939",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1462",\n      "lat": 55.6665921,\n      "lon": 37.5851997,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374506143",\n      "kind": "school",\n      "name": "Школа \\"Промо-М\\"",\n      "lat": 55.643379,\n      "lon": 37.47992,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374543558",\n      "kind": "kindergarten",\n      "name": "ДОУ 2415",\n      "lat": 55.6521405,\n      "lon": 37.5192693,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374741490",\n      "kind": "school",\n      "name": "ДООЦ \\"Юго-Западный\\"",\n      "lat": 55.6085997,\n      "lon": 37.5287946,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374741494",\n      "kind": "school",\n      "name": "Лингвистический центр \\"Маленький принц\\"",\n      "lat": 55.6018739,\n      "lon": 37.5452392,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374741534",\n      "kind": "school",\n      "name": "Школа \\"Ясенево\\"",\n      "lat": 55.5950379,\n      "lon": 37.5381405,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374876546",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2480",\n      "lat": 55.6514906,\n      "lon": 37.5973955,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1376740671",\n      "kind": "kindergarten",\n      "name": "Детский сад №2694",\n      "lat": 55.6769338,\n      "lon": 37.547569,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1376761322",\n      "kind": "kindergarten",\n      "name": "Школа 1948 \\"Лингвист-М\\"",\n      "lat": 55.6641266,\n      "lon": 37.5549242,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1426996766",\n      "kind": "school",\n      "name": "Православная школа во имя иконы божией Матери \\"Скоропослушница\\"",\n      "lat": 55.5899136,\n      "lon": 37.6517472,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427761505",\n      "kind": "kindergarten",\n      "name": "ДОУ №87",\n      "lat": 55.6696127,\n      "lon": 37.6289439,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427761506",\n      "kind": "school",\n      "name": "Государственное бюджетное образовательное учреждение города Москвы центр лечебной педагогики и дифференцированного обучения \\"Возрождение\\"",\n      "lat": 55.6632193,\n      "lon": 37.6485723,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427798458",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, здание № 14 Семицветик",\n      "lat": 55.6616667,\n      "lon": 37.6322709,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427798459",\n      "kind": "school",\n      "name": "ННОУ \\"Эрудит\\"",\n      "lat": 55.6793447,\n      "lon": 37.6545815,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427798462",\n      "kind": "school",\n      "name": "Негосударственное образовательное учреждение русская гуманитарная школа",\n      "lat": 55.6655864,\n      "lon": 37.6431838,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427803906",\n      "kind": "kindergarten",\n      "name": "Детский сад №67",\n      "lat": 55.7999182,\n      "lon": 37.5265136,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427853539",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа № 1073",\n      "lat": 55.6622784,\n      "lon": 37.6261054,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855779",\n      "kind": "kindergarten",\n      "name": "Детский сад № 131",\n      "lat": 55.7079922,\n      "lon": 37.661336,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855782",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1672",\n      "lat": 55.7035467,\n      "lon": 37.6639557,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855785",\n      "kind": "kindergarten",\n      "name": "Школа № 630 им. дважды Героя Советского Союза Г.П. Кравченко. Дошкольное отделение",\n      "lat": 55.7112246,\n      "lon": 37.6111345,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855794",\n      "kind": "kindergarten",\n      "name": "Детский сад № 307",\n      "lat": 55.7192549,\n      "lon": 37.6484758,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855795",\n      "kind": "kindergarten",\n      "name": "Детский сад №852",\n      "lat": 55.7195446,\n      "lon": 37.6607312,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427956880",\n      "kind": "kindergarten",\n      "name": "Детский сад №2172",\n      "lat": 55.7221258,\n      "lon": 37.5768038,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427982444",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1257",\n      "lat": 55.7634798,\n      "lon": 37.5567572,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427982445",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 9 «Анютины глазки»",\n      "lat": 55.7502211,\n      "lon": 37.5235882,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427982447",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7580315,\n      "lon": 37.5528012,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427996071",\n      "kind": "kindergarten",\n      "name": "ДО №6 Улыбка",\n      "lat": 55.7725055,\n      "lon": 37.5832127,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1428076955",\n      "kind": "kindergarten",\n      "name": "Детский сад №281",\n      "lat": 55.7235473,\n      "lon": 37.6068753,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429235509",\n      "kind": "school",\n      "name": "ЦПМСС \\"Взаимодействие\\"",\n      "lat": 55.6603447,\n      "lon": 37.6252917,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429376733",\n      "kind": "kindergarten",\n      "name": "Детский сад №37",\n      "lat": 55.7758727,\n      "lon": 37.6468774,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429379251",\n      "kind": "school",\n      "name": "ЦДТ \\"Логос\\"",\n      "lat": 55.7098599,\n      "lon": 37.6427218,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429406139",\n      "kind": "kindergarten",\n      "name": "Детский сад №65",\n      "lat": 55.7132417,\n      "lon": 37.6679007,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429495670",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7034523,\n      "lon": 37.6735585,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429499809",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7033024,\n      "lon": 37.6691982,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430419667",\n      "kind": "kindergarten",\n      "name": "ДОУ №729",\n      "lat": 55.7540076,\n      "lon": 37.5844648,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430428669",\n      "kind": "kindergarten",\n      "name": "НОУ \\"Атлант\\"",\n      "lat": 55.624655,\n      "lon": 37.6074245,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430457015",\n      "kind": "kindergarten",\n      "name": "Детский сад №2500",\n      "lat": 55.6936898,\n      "lon": 37.7376052,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430522302",\n      "kind": "school",\n      "name": "АНО СОШ \\"Учебный центр Перспектива\\"",\n      "lat": 55.6374278,\n      "lon": 37.6143473,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430546453",\n      "kind": "school",\n      "name": "Школа №435",\n      "lat": 55.7746135,\n      "lon": 37.7048526,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430612438",\n      "kind": "kindergarten",\n      "name": "Детский сад №1609",\n      "lat": 55.7236994,\n      "lon": 37.7697169,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430804068",\n      "kind": "school",\n      "name": "Школа №1113",\n      "lat": 55.7691104,\n      "lon": 37.6030203,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1432138391",\n      "kind": "kindergarten",\n      "name": "Детский сад №1192",\n      "lat": 55.780008,\n      "lon": 37.615793,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1433444316",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 4 «Солнышко»",\n      "lat": 55.7650358,\n      "lon": 37.5621154,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1433744179",\n      "kind": "kindergarten",\n      "name": "Детский сад №1548",\n      "lat": 55.7214707,\n      "lon": 37.578384,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1434207695",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1786",\n      "lat": 55.7573124,\n      "lon": 37.5420359,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1434221468",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.767322,\n      "lon": 37.5560413,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1434221469",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7678856,\n      "lon": 37.5558112,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1434238335",\n      "kind": "kindergarten",\n      "name": "ЦРР детский сад № 1842",\n      "lat": 55.7706674,\n      "lon": 37.586669,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1435421467",\n      "kind": "kindergarten",\n      "name": "Школа №1621 \\"Древо жизни\\". Дошкольное отделение",\n      "lat": 55.7665753,\n      "lon": 37.6536785,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1435673590",\n      "kind": "kindergarten",\n      "name": "Детский сад № 26",\n      "lat": 55.7085092,\n      "lon": 37.6595251,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1436508002",\n      "kind": "kindergarten",\n      "name": "Центр развития ребёнка №2364",\n      "lat": 55.7671373,\n      "lon": 37.5829561,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1436607111",\n      "kind": "kindergarten",\n      "name": "ДОУ №2237",\n      "lat": 55.6793049,\n      "lon": 37.646904,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1437387424",\n      "kind": "kindergarten",\n      "name": "ДОУ №6",\n      "lat": 55.7714842,\n      "lon": 37.667638,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1437658381",\n      "kind": "kindergarten",\n      "name": "Московская международная школа. Подразделение \\"Планета Марс\\"",\n      "lat": 55.7644746,\n      "lon": 37.7862673,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1437658382",\n      "kind": "kindergarten",\n      "name": "Московская международная школа. Подразделение \\"Планета Венера\\"",\n      "lat": 55.7638021,\n      "lon": 37.7828847,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1443197494",\n      "kind": "kindergarten",\n      "name": "Школа № 1231 им. В. Д. Поленова. Дошкольный корпус",\n      "lat": 55.7278139,\n      "lon": 37.5606804,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1443307436",\n      "kind": "school",\n      "name": "Школа №654 имени А. Д. Фридмана. Учебный корпус №4",\n      "lat": 55.694833,\n      "lon": 37.7362385,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1444803462",\n      "kind": "school",\n      "name": "Центр психолого-педагогической реабилитации и коррекции \\"Восточный\\"",\n      "lat": 55.7820621,\n      "lon": 37.7276916,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1444819629",\n      "kind": "kindergarten",\n      "name": "Детский сад №118",\n      "lat": 55.7801928,\n      "lon": 37.730691,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1456162724",\n      "kind": "kindergarten",\n      "name": "Детский сад №227 (встр)",\n      "lat": 55.7837038,\n      "lon": 37.7331763,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1456659414",\n      "kind": "kindergarten",\n      "name": "Детский сад №1089",\n      "lat": 55.7268272,\n      "lon": 37.5619022,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1456659419",\n      "kind": "kindergarten",\n      "name": "Детский сад компенсирующего вида №1476",\n      "lat": 55.7239375,\n      "lon": 37.5823418,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1456659423",\n      "kind": "kindergarten",\n      "name": "Детский сад №676",\n      "lat": 55.72473,\n      "lon": 37.5664425,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1460061975",\n      "kind": "kindergarten",\n      "name": "Московская международная школа. Подразделение \\"Планета Меркурий\\"",\n      "lat": 55.7589128,\n      "lon": 37.7886511,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1472892444",\n      "kind": "kindergarten",\n      "name": "Детский сад №1107",\n      "lat": 55.7822439,\n      "lon": 37.7422118,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1504923799",\n      "kind": "kindergarten",\n      "name": "Детский сад №960",\n      "lat": 55.794034,\n      "lon": 37.7905245,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1504923815",\n      "kind": "kindergarten",\n      "name": "Детский сад №2251",\n      "lat": 55.7936427,\n      "lon": 37.7982305,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1560232958",\n      "kind": "school",\n      "name": "Центр образования № 118",\n      "lat": 55.6869036,\n      "lon": 37.5295957,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1561070468",\n      "kind": "school",\n      "name": "Школа №213",\n      "lat": 55.8167885,\n      "lon": 37.5198901,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1566443974",\n      "kind": "kindergarten",\n      "name": "Детский сад №1500",\n      "lat": 55.7061792,\n      "lon": 37.5725787,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1588932818",\n      "kind": "school",\n      "name": "Центр образования \\"Измайлово\\" № 1811",\n      "lat": 55.7939167,\n      "lon": 37.8147527,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1599296100",\n      "kind": "school",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa01",\n      "lat": 55.7077593,\n      "lon": 37.7712348,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1602410893",\n      "kind": "kindergarten",\n      "name": "Детский сад №1905 (встр)",\n      "lat": 55.7619833,\n      "lon": 37.7347428,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1603692724",\n      "kind": "kindergarten",\n      "name": "Вальдорфовская школа",\n      "lat": 55.6411221,\n      "lon": 37.5223934,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1618108202",\n      "kind": "school",\n      "name": "Прогимназия № 130",\n      "lat": 55.7796573,\n      "lon": 37.4960053,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1635684783",\n      "kind": "school",\n      "name": "Спортивно-технический центр \\"Икар\\"",\n      "lat": 55.7585593,\n      "lon": 37.7115837,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1635684784",\n      "kind": "school",\n      "name": "Спортивно-технический центр \\"Икар\\"",\n      "lat": 55.7718921,\n      "lon": 37.7111201,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1719646868",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2416",\n      "lat": 55.7821094,\n      "lon": 37.736209,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1727533933",\n      "kind": "school",\n      "name": "Знаменская гимназия",\n      "lat": 55.8575416,\n      "lon": 37.4997164,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1735446417",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2296",\n      "lat": 55.7920113,\n      "lon": 37.5674501,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1792256966",\n      "kind": "school",\n      "name": "Детская музыкально-хоровая школа \\"Радость\\"",\n      "lat": 55.8395702,\n      "lon": 37.5294679,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1841073860",\n      "kind": "kindergarten",\n      "name": "Школа №439 \\"Интеллект\\" (дошкольное отделение №1)",\n      "lat": 55.7345153,\n      "lon": 37.745368,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1858733747",\n      "kind": "kindergarten",\n      "name": "Детский сад №1945",\n      "lat": 55.7540246,\n      "lon": 37.7102476,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1867012052",\n      "kind": "school",\n      "name": "Центр еврейского образования \\"Бейт швидлер\\"",\n      "lat": 55.7897637,\n      "lon": 37.6089304,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1945951836",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №6",\n      "lat": 55.6425327,\n      "lon": 37.5030157,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1947167782",\n      "kind": "school",\n      "name": "Школа №423",\n      "lat": 55.7571365,\n      "lon": 37.8020241,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2055643773",\n      "kind": "kindergarten",\n      "name": "Школа №218. Дошкольное отделение",\n      "lat": 55.8127657,\n      "lon": 37.5732614,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2107834140",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1284",\n      "lat": 55.7247929,\n      "lon": 37.4408259,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2107834141",\n      "kind": "kindergarten",\n      "name": "Детский сад № 943",\n      "lat": 55.7237007,\n      "lon": 37.4403719,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2146273545",\n      "kind": "school",\n      "name": "Школа №77",\n      "lat": 55.7668171,\n      "lon": 37.7275087,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2239554842",\n      "kind": "kindergarten",\n      "name": "ЛУЧиК",\n      "lat": 55.7427543,\n      "lon": 37.6009831,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2298658284",\n      "kind": "kindergarten",\n      "name": "Школа 1601 дошкольное отделение",\n      "lat": 55.8004005,\n      "lon": 37.5683713,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2298842448",\n      "kind": "kindergarten",\n      "name": "Детский сад №2509",\n      "lat": 55.6595529,\n      "lon": 37.5179687,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2352319437",\n      "kind": "school",\n      "name": "Детский досуговый центр «Аист»",\n      "lat": 55.7153262,\n      "lon": 37.779866,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2367806239",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7341456,\n      "lon": 37.7051295,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2525214326",\n      "kind": "kindergarten",\n      "name": "Детский развивающий центр",\n      "lat": 55.7085875,\n      "lon": 37.8097785,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2558642751",\n      "kind": "school",\n      "name": "Воскресная школа церкви Евангельских христиан-баптистов Воскресение",\n      "lat": 55.6427918,\n      "lon": 37.6760494,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2661065760",\n      "kind": "school",\n      "name": "Молодежный центр «Звездный»",\n      "lat": 55.8960402,\n      "lon": 37.7094401,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2661667230",\n      "kind": "kindergarten",\n      "name": "Детский сад №1422",\n      "lat": 55.7616574,\n      "lon": 37.7364917,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2828438665",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8975524,\n      "lon": 37.4796576,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2834722596",\n      "kind": "kindergarten",\n      "name": "Детский сад № 232",\n      "lat": 55.8151625,\n      "lon": 37.5670383,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2942619202",\n      "kind": "kindergarten",\n      "name": "Лицей № 1586, дошкольное отделение 2",\n      "lat": 55.7151446,\n      "lon": 37.5173189,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2942619203",\n      "kind": "kindergarten",\n      "name": "Лицей № 1586, дошкольное отделение 3",\n      "lat": 55.7130673,\n      "lon": 37.5169617,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2942619204",\n      "kind": "kindergarten",\n      "name": "Лицей №1586, дошкольное отделение",\n      "lat": 55.7127864,\n      "lon": 37.5206021,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3005725110",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8182793,\n      "lon": 37.5054235,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3008237259",\n      "kind": "kindergarten",\n      "name": "Школа с углубленным изучением английского языка № 1270, дошкольное отделение № 2",\n      "lat": 55.749667,\n      "lon": 37.659845,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3008237260",\n      "kind": "school",\n      "name": "Школа № 2104",\n      "lat": 55.7495507,\n      "lon": 37.6585411,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "node/3041391839",\n      "kind": "school",\n      "name": "Филипповская Школа",\n      "lat": 55.7293782,\n      "lon": 37.5708461,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3139685557",\n      "kind": "school",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa04",\n      "lat": 55.7071909,\n      "lon": 37.7710406,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3363053807",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №3 (дошкольное отделение)",\n      "lat": 55.8457996,\n      "lon": 37.4503084,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3478996648",\n      "kind": "school",\n      "name": "Детская музыкально-хоровая школа имени И.И.Радченко",\n      "lat": 55.7357379,\n      "lon": 37.6598187,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3558549610",\n      "kind": "kindergarten",\n      "name": "Лукоморье",\n      "lat": 55.8044871,\n      "lon": 37.5385491,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3558834930",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7992699,\n      "lon": 37.5535226,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3632408193",\n      "kind": "kindergarten",\n      "name": "P\'titCREF",\n      "lat": 55.750418,\n      "lon": 37.597092,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3741570155",\n      "kind": "school",\n      "name": "",\n      "lat": 55.686451,\n      "lon": 37.5382072,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3741570172",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6871486,\n      "lon": 37.5394425,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3794027238",\n      "kind": "school",\n      "name": "Школа Английского",\n      "lat": 55.8871929,\n      "lon": 37.6509059,\n      "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-18:00"\n    },\n    {\n      "id": "node/3894595446",\n      "kind": "kindergarten",\n      "name": "Ч.У.Д.О. Детский сад №\xa01",\n      "lat": 55.8171088,\n      "lon": 37.5074547,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3955821430",\n      "kind": "kindergarten",\n      "name": "Детский центр \\"Вешняки\\"",\n      "lat": 55.7402605,\n      "lon": 37.8243783,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4107853391",\n      "kind": "kindergarten",\n      "name": "Sun School",\n      "lat": 55.8008773,\n      "lon": 37.6337751,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4119301621",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7703871,\n      "lon": 37.5868692,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4243926389",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение ГБОУ Школы №444",\n      "lat": 55.7896286,\n      "lon": 37.7966572,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4257318691",\n      "kind": "kindergarten",\n      "name": "Талантино",\n      "lat": 55.7729464,\n      "lon": 37.4645667,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4334671589",\n      "kind": "kindergarten",\n      "name": "Мамин Садик Seasons",\n      "lat": 55.7696732,\n      "lon": 37.6078508,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4351546398",\n      "kind": "kindergarten",\n      "name": "Даниил",\n      "lat": 55.7186774,\n      "lon": 37.6234095,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4381937594",\n      "kind": "school",\n      "name": "О\'кей инглиш",\n      "lat": 55.7131516,\n      "lon": 37.7554185,\n      "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-16:00"\n    },\n    {\n      "id": "node/4393094689",\n      "kind": "kindergarten",\n      "name": "Семейный центр \\"Веста\\"",\n      "lat": 55.800881,\n      "lon": 37.7199409,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4429110930",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7313423,\n      "lon": 37.7581814,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4518521699",\n      "kind": "school",\n      "name": "Мини-школа",\n      "lat": 55.6942016,\n      "lon": 37.7877321,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4518521717",\n      "kind": "kindergarten",\n      "name": "Голицынский",\n      "lat": 55.6894547,\n      "lon": 37.7908161,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4520379714",\n      "kind": "kindergarten",\n      "name": "Лукошко",\n      "lat": 55.8205812,\n      "lon": 37.5822579,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4575871390",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8366297,\n      "lon": 37.6602361,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4588995214",\n      "kind": "kindergarten",\n      "name": "Детский сад №2566",\n      "lat": 55.7633007,\n      "lon": 37.7366041,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4642004020",\n      "kind": "school",\n      "name": "Ховринская гимназия \\"Лампада\\"",\n      "lat": 55.8649026,\n      "lon": 37.5089872,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4683882102",\n      "kind": "kindergarten",\n      "name": "Домик Клуб",\n      "lat": 55.8650457,\n      "lon": 37.5039408,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4732895821",\n      "kind": "kindergarten",\n      "name": "ЛучиК",\n      "lat": 55.7988505,\n      "lon": 37.5009718,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4742086648",\n      "kind": "kindergarten",\n      "name": "Детский развивающий центр Катюша",\n      "lat": 55.7042701,\n      "lon": 37.6686927,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750851597",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7652441,\n      "lon": 37.5637221,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750851599",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7698779,\n      "lon": 37.5858852,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750904980",\n      "kind": "kindergarten",\n      "name": "Радуга",\n      "lat": 55.7201055,\n      "lon": 37.5745181,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750904981",\n      "kind": "school",\n      "name": "Школа Виктория-2000",\n      "lat": 55.7169438,\n      "lon": 37.5770098,\n      "opening_hours": "Mo-Fr 08:00-18:30"\n    },\n    {\n      "id": "node/4750904982",\n      "kind": "kindergarten",\n      "name": "Школа № 192. Дошкольное отделение",\n      "lat": 55.702035,\n      "lon": 37.5751536,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "node/4750904983",\n      "kind": "school",\n      "name": "Школа № 19",\n      "lat": 55.7025627,\n      "lon": 37.5761214,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750941909",\n      "kind": "kindergarten",\n      "name": "Центр образования и спорта \\"Москва-98\\". Дошкольное отделение № 1",\n      "lat": 55.6851927,\n      "lon": 37.49421,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4751092803",\n      "kind": "kindergarten",\n      "name": "Звездочет",\n      "lat": 55.6521046,\n      "lon": 37.7645701,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4752529949",\n      "kind": "school",\n      "name": "Школа № 1331 \\"Молодая Иверия\\"",\n      "lat": 55.7478298,\n      "lon": 37.5942907,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4752529957",\n      "kind": "kindergarten",\n      "name": "Школа № 1465, дошкольное отделение № 2",\n      "lat": 55.7489559,\n      "lon": 37.5713427,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4752529964",\n      "kind": "kindergarten",\n      "name": "СОШ № 1232. Дошкольное отделение № 1",\n      "lat": 55.751215,\n      "lon": 37.5618039,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4752922517",\n      "kind": "kindergarten",\n      "name": "Детский сад №58",\n      "lat": 55.8973585,\n      "lon": 37.4778193,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4760975781",\n      "kind": "school",\n      "name": "Школа № 887. Отделение дополнительного образования детей",\n      "lat": 55.7351452,\n      "lon": 37.4406248,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4760975806",\n      "kind": "kindergarten",\n      "name": "Центр раннего развития «Тиша & Маруся»",\n      "lat": 55.6959326,\n      "lon": 37.5011659,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4792431751",\n      "kind": "kindergarten",\n      "name": "Лицей № 429 «Соколиная гора»",\n      "lat": 55.7678304,\n      "lon": 37.7325179,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4802013987",\n      "kind": "kindergarten",\n      "name": "Божья коровка",\n      "lat": 55.7770639,\n      "lon": 37.6010791,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4857880422",\n      "kind": "school",\n      "name": "Утро",\n      "lat": 55.7253324,\n      "lon": 37.5256742,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "node/4924865509",\n      "kind": "kindergarten",\n      "name": "Вгорошек",\n      "lat": 55.6458458,\n      "lon": 37.616462,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4959152543",\n      "kind": "school",\n      "name": "Коррекционная школа-интернат VIII вида №29",\n      "lat": 55.8015246,\n      "lon": 37.7016433,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5045257699",\n      "kind": "school",\n      "name": "Школа баристы и барменов",\n      "lat": 55.8052147,\n      "lon": 37.528516,\n      "opening_hours": "Mo-Su 11:00-23:00"\n    },\n    {\n      "id": "node/5054611710",\n      "kind": "school",\n      "name": "Школа \\"Виктория\\"",\n      "lat": 55.7573563,\n      "lon": 37.5723477,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5054611711",\n      "kind": "school",\n      "name": "Свято-Димитриевская школа",\n      "lat": 55.7232298,\n      "lon": 37.6002213,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5092714240",\n      "kind": "school",\n      "name": "Дополнительное образование школы № 601",\n      "lat": 55.8034934,\n      "lon": 37.5409607,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5113395065",\n      "kind": "school",\n      "name": "Школа №370 им. генерала П. И. Батова",\n      "lat": 55.810851,\n      "lon": 37.7292703,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5161282123",\n      "kind": "kindergarten",\n      "name": "Парк детского периода",\n      "lat": 55.7023809,\n      "lon": 37.5042457,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5180531053",\n      "kind": "kindergarten",\n      "name": "Земляне",\n      "lat": 55.8045196,\n      "lon": 37.5445216,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5180531089",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8061398,\n      "lon": 37.5466437,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5199351213",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Глобус\\"",\n      "lat": 55.7898956,\n      "lon": 37.7710953,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5223728204",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7184332,\n      "lon": 37.6243684,\n      "opening_hours": "Mo-Su 08:00-19:00"\n    },\n    {\n      "id": "node/5241859752",\n      "kind": "kindergarten",\n      "name": "Детский центр \\"Умка\\"",\n      "lat": 55.6464796,\n      "lon": 37.7585041,\n      "opening_hours": "Mo-Su 07:00-20:00"\n    },\n    {\n      "id": "node/5241878658",\n      "kind": "kindergarten",\n      "name": "Детский центр \\"Антей\\"",\n      "lat": 55.6462993,\n      "lon": 37.7566806,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5251043166",\n      "kind": "kindergarten",\n      "name": "Маленький гений",\n      "lat": 55.652822,\n      "lon": 37.7559923,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5251072777",\n      "kind": "kindergarten",\n      "name": "Детский развивающий центр \\"Радуга\\"",\n      "lat": 55.6528416,\n      "lon": 37.7547201,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5260578062",\n      "kind": "kindergarten",\n      "name": "Детский сад №988",\n      "lat": 55.8015282,\n      "lon": 37.789439,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5273631003",\n      "kind": "kindergarten",\n      "name": "Детский сад №1221",\n      "lat": 55.7965665,\n      "lon": 37.8146034,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5274311218",\n      "kind": "kindergarten",\n      "name": "Детский сад № 457",\n      "lat": 55.8045805,\n      "lon": 37.5503065,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5292569524",\n      "kind": "kindergarten",\n      "name": "Школа № 429 - детский сад",\n      "lat": 55.7612682,\n      "lon": 37.7333849,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5293959915",\n      "kind": "kindergarten",\n      "name": "ДОУ №2233",\n      "lat": 55.6784207,\n      "lon": 37.6585405,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5300129067",\n      "kind": "kindergarten",\n      "name": "Детский сад №1155",\n      "lat": 55.8003112,\n      "lon": 37.7893176,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719149",\n      "kind": "kindergarten",\n      "name": "Детский сад №2455",\n      "lat": 55.7959106,\n      "lon": 37.7835327,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719152",\n      "kind": "kindergarten",\n      "name": "Детский сад №1127 (встр)",\n      "lat": 55.7977305,\n      "lon": 37.7867031,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719153",\n      "kind": "kindergarten",\n      "name": "Детский сад №1084",\n      "lat": 55.8016157,\n      "lon": 37.7962437,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719154",\n      "kind": "kindergarten",\n      "name": "Детский сад №461",\n      "lat": 55.802389,\n      "lon": 37.7975526,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719155",\n      "kind": "kindergarten",\n      "name": "Детский сад №461",\n      "lat": 55.8023453,\n      "lon": 37.7948651,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719158",\n      "kind": "kindergarten",\n      "name": "Школа «Содружество». Учебный корпус №1-1",\n      "lat": 55.7615986,\n      "lon": 37.7011981,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719159",\n      "kind": "kindergarten",\n      "name": "Детский сад №1798",\n      "lat": 55.7631259,\n      "lon": 37.7117421,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719162",\n      "kind": "kindergarten",\n      "name": "ДОО Розовый Слоненок",\n      "lat": 55.6690273,\n      "lon": 37.7535564,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5332088797",\n      "kind": "kindergarten",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 1",\n      "lat": 55.7464549,\n      "lon": 37.5927941,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5334698649",\n      "kind": "kindergarten",\n      "name": "Детский сад №367",\n      "lat": 55.8168407,\n      "lon": 37.7037691,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5336819124",\n      "kind": "kindergarten",\n      "name": "Центр дополнительного образования \\"Наши дети\\"",\n      "lat": 55.5854041,\n      "lon": 37.7336197,\n      "opening_hours": "Mo-Su 09:00-21:00"\n    },\n    {\n      "id": "node/5336819321",\n      "kind": "school",\n      "name": "Школа искусств \\"Перспектива\\"",\n      "lat": 55.5854819,\n      "lon": 37.7335077,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5338565872",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.7967049,\n      "lon": 37.5051915,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5338693858",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.7984227,\n      "lon": 37.5053771,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5392473021",\n      "kind": "school",\n      "name": "Перспектива",\n      "lat": 55.6966698,\n      "lon": 37.5075131,\n      "opening_hours": "Su-Sa 09:00-21:00"\n    },\n    {\n      "id": "node/5418157015",\n      "kind": "kindergarten",\n      "name": "Самый лучший",\n      "lat": 55.6254039,\n      "lon": 37.6646795,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "node/5465097276",\n      "kind": "kindergarten",\n      "name": "Школа № 1530 \\"Школа Ломоносова\\". Дошкольная группа",\n      "lat": 55.7913991,\n      "lon": 37.6842642,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5477281121",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7276692,\n      "lon": 37.5339605,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5601185228",\n      "kind": "school",\n      "name": "Международный инновационный образовательный центр",\n      "lat": 55.691929,\n      "lon": 37.6243869,\n      "opening_hours": "Mo-Fr 10:00-20:00"\n    },\n    {\n      "id": "node/5634481148",\n      "kind": "school",\n      "name": "НОУ «Первая Школа»",\n      "lat": 55.8829735,\n      "lon": 37.4915995,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5660523622",\n      "kind": "school",\n      "name": "Православный центр непрерывного образования во имя прп. Серафима Саровского",\n      "lat": 55.5903957,\n      "lon": 37.7598343,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5800842228",\n      "kind": "school",\n      "name": "Центр немецкого языка DE Lingua",\n      "lat": 55.7063408,\n      "lon": 37.5965042,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5919855496",\n      "kind": "kindergarten",\n      "name": "ONE! International School",\n      "lat": 55.7681478,\n      "lon": 37.5907513,\n      "opening_hours": "Mo-Fr 08:30-18:30"\n    },\n    {\n      "id": "node/6015545790",\n      "kind": "kindergarten",\n      "name": "Умка",\n      "lat": 55.5894948,\n      "lon": 37.7621307,\n      "opening_hours": "Mo-Fr 09:00-20:00"\n    },\n    {\n      "id": "node/6040072881",\n      "kind": "kindergarten",\n      "name": "Большая Черепаха",\n      "lat": 55.6844441,\n      "lon": 37.5342768,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6050114811",\n      "kind": "school",\n      "name": "Big Ben",\n      "lat": 55.6649753,\n      "lon": 37.7545153,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6052716488",\n      "kind": "school",\n      "name": "АНО СДЦ \\"Барс\\"",\n      "lat": 55.6977391,\n      "lon": 37.7738466,\n      "opening_hours": "Mo-Fr 11:00-21:00; Sa 12:00-18:00"\n    },\n    {\n      "id": "node/6067290559",\n      "kind": "school",\n      "name": "Барс",\n      "lat": 55.7029264,\n      "lon": 37.7502666,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6100272958",\n      "kind": "school",\n      "name": "Образовательный центр \\"Юниум\\"",\n      "lat": 55.74963,\n      "lon": 37.8232839,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6299092036",\n      "kind": "school",\n      "name": "Лицей НИУ ВШЭ",\n      "lat": 55.7523702,\n      "lon": 37.6374355,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6341309524",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Смышлёнок\\"",\n      "lat": 55.6230445,\n      "lon": 37.7051784,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6413499947",\n      "kind": "school",\n      "name": "Итальянская школа им. Итало Кальвино",\n      "lat": 55.682994,\n      "lon": 37.5341919,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6413499950",\n      "kind": "school",\n      "name": "Московская японская школа",\n      "lat": 55.6830015,\n      "lon": 37.5337172,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6415850521",\n      "kind": "kindergarten",\n      "name": "Sun School",\n      "lat": 55.8408588,\n      "lon": 37.4942243,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6435567397",\n      "kind": "kindergarten",\n      "name": "Центр детского развития Сундучок",\n      "lat": 55.7654354,\n      "lon": 37.7028599,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6452268986",\n      "kind": "kindergarten",\n      "name": "Школа № 1531 имени С. К. Годовикова. Дошкольное отделение",\n      "lat": 55.8012686,\n      "lon": 37.6330007,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6625005385",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6984981,\n      "lon": 37.6361301,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6632780514",\n      "kind": "kindergarten",\n      "name": "Английский детский сад",\n      "lat": 55.8856659,\n      "lon": 37.6780571,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6659929215",\n      "kind": "kindergarten",\n      "name": "Happy Elephant",\n      "lat": 55.7993168,\n      "lon": 37.5207014,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6665428945",\n      "kind": "school",\n      "name": "Центр спорта и образования \\"Локомотив\\"",\n      "lat": 55.8061697,\n      "lon": 37.7423295,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6841199202",\n      "kind": "kindergarten",\n      "name": "Послушайка",\n      "lat": 55.670609,\n      "lon": 37.6117617,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6866820508",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7961228,\n      "lon": 37.8017832,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6868031109",\n      "kind": "kindergarten",\n      "name": "iQ-Сад",\n      "lat": 55.6671613,\n      "lon": 37.7885163,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6988316009",\n      "kind": "school",\n      "name": "Первый экстернат",\n      "lat": 55.7429862,\n      "lon": 37.5374481,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7018057539",\n      "kind": "school",\n      "name": "Русская классическая гимназия",\n      "lat": 55.7711404,\n      "lon": 37.6166484,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7047225544",\n      "kind": "kindergarten",\n      "name": "Школа № 1231 дошкольное отделение",\n      "lat": 55.7428661,\n      "lon": 37.5803554,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7057170497",\n      "kind": "kindergarten",\n      "name": "Китенок",\n      "lat": 55.6290768,\n      "lon": 37.7411759,\n      "opening_hours": "Mo-Su 09:00-21:00"\n    },\n    {\n      "id": "node/7118093215",\n      "kind": "school",\n      "name": "Общеобразовательная православная школа искусств",\n      "lat": 55.781397,\n      "lon": 37.5767317,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7219021062",\n      "kind": "kindergarten",\n      "name": "Smile Fish",\n      "lat": 55.7875385,\n      "lon": 37.6333371,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7223451181",\n      "kind": "kindergarten",\n      "name": "Совёнок",\n      "lat": 55.6049805,\n      "lon": 37.7245161,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7223585386",\n      "kind": "kindergarten",\n      "name": "Детский сад Kidsberry",\n      "lat": 55.7548657,\n      "lon": 37.5977827,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "node/7223585387",\n      "kind": "kindergarten",\n      "name": "Детский сад Chekins",\n      "lat": 55.7493615,\n      "lon": 37.5654449,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "node/7223585388",\n      "kind": "kindergarten",\n      "name": "Детский сад Kidsberry",\n      "lat": 55.7496902,\n      "lon": 37.5768652,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "node/7257951974",\n      "kind": "kindergarten",\n      "name": "KidStarter",\n      "lat": 55.7880531,\n      "lon": 37.6710359,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7296709837",\n      "kind": "kindergarten",\n      "name": "Островок",\n      "lat": 55.6919141,\n      "lon": 37.5582648,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7315431992",\n      "kind": "kindergarten",\n      "name": "Мой садик",\n      "lat": 55.6038138,\n      "lon": 37.7253462,\n      "opening_hours": "Mo-Su 08:00-20:00"\n    },\n    {\n      "id": "node/7366466241",\n      "kind": "kindergarten",\n      "name": "Чудесная лужайка",\n      "lat": 55.828081,\n      "lon": 37.6646787,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7388381455",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Тигрёнок\\"",\n      "lat": 55.7912608,\n      "lon": 37.5010622,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7388383892",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Василёк\\"",\n      "lat": 55.7928782,\n      "lon": 37.48464,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7388383895",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Незабудка\\"",\n      "lat": 55.7968576,\n      "lon": 37.4838136,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7403153966",\n      "kind": "kindergarten",\n      "name": "Непоседы",\n      "lat": 55.7441279,\n      "lon": 37.8031669,\n      "opening_hours": "Mo-Fr 07:00-20:00; Sa-Su 08:00-19:30"\n    },\n    {\n      "id": "node/7403181580",\n      "kind": "kindergarten",\n      "name": "Карапузики",\n      "lat": 55.7438525,\n      "lon": 37.8045053,\n      "opening_hours": "Mo-Fr 07:00-20:00; Sa-Su 08:00-19:30"\n    },\n    {\n      "id": "node/7632284524",\n      "kind": "kindergarten",\n      "name": "Алые паруса",\n      "lat": 55.8723226,\n      "lon": 37.4856616,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7714534120",\n      "kind": "kindergarten",\n      "name": "Kids World International Preschool",\n      "lat": 55.7676724,\n      "lon": 37.6605467,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7732273582",\n      "kind": "kindergarten",\n      "name": "Bambini-Club",\n      "lat": 55.7982687,\n      "lon": 37.5686797,\n      "opening_hours": "Mo-Fr 07:30-20:00"\n    },\n    {\n      "id": "node/7832365986",\n      "kind": "school",\n      "name": "Bambini-Club",\n      "lat": 55.6697134,\n      "lon": 37.4738556,\n      "opening_hours": "Mo-Fr 07:30-19:00;Sa-Su 09:00-18:00"\n    },\n    {\n      "id": "node/7876939796",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.730467,\n      "lon": 37.5801543,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7934421513",\n      "kind": "kindergarten",\n      "name": "Гений",\n      "lat": 55.8178886,\n      "lon": 37.8217582,\n      "opening_hours": "Mo,We 11:00-12:15,16:00-20:00; Tu,Th,Fr 16:00-20:00; Su 11:00-17:00"\n    },\n    {\n      "id": "node/7986070109",\n      "kind": "kindergarten",\n      "name": "People&kids",\n      "lat": 55.7329437,\n      "lon": 37.6177575,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7992280518",\n      "kind": "kindergarten",\n      "name": "Мозаика",\n      "lat": 55.7316491,\n      "lon": 37.5856852,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8033690833",\n      "kind": "school",\n      "name": "Лицей \\"Москвич\\"",\n      "lat": 55.6778188,\n      "lon": 37.5178345,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8082242566",\n      "kind": "school",\n      "name": "Ирида",\n      "lat": 55.7301696,\n      "lon": 37.8118483,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8329707709",\n      "kind": "kindergarten",\n      "name": "Alertum Kids",\n      "lat": 55.8120963,\n      "lon": 37.6584774,\n      "opening_hours": "Mo-Su 08:00-20:00"\n    },\n    {\n      "id": "node/8369406538",\n      "kind": "kindergarten",\n      "name": "Школа 152 дошкольное отделение",\n      "lat": 55.7981345,\n      "lon": 37.5453217,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8429777503",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Сфера\\"",\n      "lat": 55.8927808,\n      "lon": 37.6611203,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8445715562",\n      "kind": "kindergarten",\n      "name": "Кембриджуниор",\n      "lat": 55.7831029,\n      "lon": 37.6604274,\n      "opening_hours": "Mo-Su 08:00-20:00"\n    },\n    {\n      "id": "node/8606067848",\n      "kind": "kindergarten",\n      "name": "Умный малыш",\n      "lat": 55.7427861,\n      "lon": 37.6608029,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8628476525",\n      "kind": "kindergarten",\n      "name": "Начальная школа — детский сад «Радуга»",\n      "lat": 55.7178006,\n      "lon": 37.5766106,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8628476532",\n      "kind": "kindergarten",\n      "name": "Начальная школа — детский сад «Радуга»",\n      "lat": 55.7188998,\n      "lon": 37.5781032,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8635443934",\n      "kind": "kindergarten",\n      "name": "Бемби-клуб",\n      "lat": 55.8145182,\n      "lon": 37.5621486,\n      "opening_hours": "Mo-Fr 08:00-19:00"\n    },\n    {\n      "id": "node/8715324604",\n      "kind": "school",\n      "name": "Школа №1505 \\"Преображенская\\"",\n      "lat": 55.8039734,\n      "lon": 37.7354515,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8715324619",\n      "kind": "school",\n      "name": "Школа №1505 \\"Преображенская\\"",\n      "lat": 55.8008166,\n      "lon": 37.737962,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8752557700",\n      "kind": "kindergarten",\n      "name": "Avokado kids",\n      "lat": 55.780932,\n      "lon": 37.44999,\n      "opening_hours": "Mo-Fr 07:45-20:15"\n    },\n    {\n      "id": "node/8839210217",\n      "kind": "school",\n      "name": "Русский Гарвард",\n      "lat": 55.6359375,\n      "lon": 37.617555,\n      "opening_hours": "Mo-Su 09:00-20:00"\n    },\n    {\n      "id": "node/8840934476",\n      "kind": "school",\n      "name": "Православная школа им. Преподобного Сергия Радонежского",\n      "lat": 55.8537147,\n      "lon": 37.6316461,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8920785773",\n      "kind": "kindergarten",\n      "name": "Школа «Содружество». Учебный корпус №7",\n      "lat": 55.7538675,\n      "lon": 37.7138382,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8933385375",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Центр «Ватутино»",\n      "lat": 55.7209908,\n      "lon": 37.4565554,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9008304817",\n      "kind": "school",\n      "name": "Полиглот",\n      "lat": 55.6474353,\n      "lon": 37.7496871,\n      "opening_hours": "Mo-Fr 09:00-21:00; Sa 09:00-18:00; Su 09:00-15:00"\n    },\n    {\n      "id": "node/9024732566",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8542844,\n      "lon": 37.5879219,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9288072222",\n      "kind": "kindergarten",\n      "name": "Детвариум",\n      "lat": 55.8548335,\n      "lon": 37.4484108,\n      "opening_hours": "Mo-Su 10:00-22:00"\n    },\n    {\n      "id": "node/9506713321",\n      "kind": "kindergarten",\n      "name": "Хорошие дети",\n      "lat": 55.879124,\n      "lon": 37.6933034,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9580996533",\n      "kind": "kindergarten",\n      "name": "Совёнок",\n      "lat": 55.6053532,\n      "lon": 37.7224629,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9687922817",\n      "kind": "kindergarten",\n      "name": "Sun School",\n      "lat": 55.7843624,\n      "lon": 37.4484654,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9725167200",\n      "kind": "kindergarten",\n      "name": "Magic Castle",\n      "lat": 55.7719793,\n      "lon": 37.6118258,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9806916428",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8231654,\n      "lon": 37.5685573,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9880328119",\n      "kind": "kindergarten",\n      "name": "Чудо чердачок",\n      "lat": 55.7533666,\n      "lon": 37.7057881,\n      "opening_hours": "Mo-Su 10:00-22:00"\n    },\n    {\n      "id": "node/9886687621",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8740544,\n      "lon": 37.5990769,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10044484701",\n      "kind": "kindergarten",\n      "name": "Обнимая солнце",\n      "lat": 55.8009863,\n      "lon": 37.7089167,\n      "opening_hours": "Mo-Sa 09:00-20:00"\n    },\n    {\n      "id": "node/10064274923",\n      "kind": "kindergarten",\n      "name": "Смарт скул",\n      "lat": 55.7436924,\n      "lon": 37.5105308,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10649421167",\n      "kind": "kindergarten",\n      "name": "Дарвиль",\n      "lat": 55.8810075,\n      "lon": 37.5260088,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "node/10709028671",\n      "kind": "school",\n      "name": "Подсолнух",\n      "lat": 55.8184515,\n      "lon": 37.5900516,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10741456003",\n      "kind": "kindergarten",\n      "name": "Многогранник Тропарёво",\n      "lat": 55.6443406,\n      "lon": 37.4907795,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10741504207",\n      "kind": "kindergarten",\n      "name": "Многогранник Раменки",\n      "lat": 55.7047156,\n      "lon": 37.4996105,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10795032875",\n      "kind": "kindergarten",\n      "name": "1450 Олимп",\n      "lat": 55.6517294,\n      "lon": 37.6166643,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10810258626",\n      "kind": "kindergarten",\n      "name": "Горница-узорница",\n      "lat": 55.7099574,\n      "lon": 37.6090493,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10821765498",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.6641116,\n      "lon": 37.5157208,\n      "opening_hours": "Mo-Fr 08:00-20:00; Sa 10:00-16:00"\n    },\n    {\n      "id": "node/10821842294",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.6888531,\n      "lon": 37.5809841,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10821944261",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.825085,\n      "lon": 37.5974592,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10821994773",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.8533145,\n      "lon": 37.4974823,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10823789312",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.6577986,\n      "lon": 37.5877138,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10823804270",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.8704288,\n      "lon": 37.5933234,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10936228694",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2243",\n      "lat": 55.7166404,\n      "lon": 37.5956299,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10977220398",\n      "kind": "kindergarten",\n      "name": "Sky Garden",\n      "lat": 55.853089,\n      "lon": 37.6234719,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10994921689",\n      "kind": "kindergarten",\n      "name": "Sky Garden",\n      "lat": 55.8517153,\n      "lon": 37.6242535,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11043096771",\n      "kind": "kindergarten",\n      "name": "Космо Кидс Отрадное",\n      "lat": 55.8510603,\n      "lon": 37.6176533,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11085165913",\n      "kind": "kindergarten",\n      "name": "Домик счастья",\n      "lat": 55.7530238,\n      "lon": 37.7084085,\n      "opening_hours": "Mo-Fr 09:00-20:00"\n    },\n    {\n      "id": "node/11085236054",\n      "kind": "kindergarten",\n      "name": "Ласточка",\n      "lat": 55.7510438,\n      "lon": 37.7093043,\n      "opening_hours": "Mo-Fr 10:00-19:30; Sa 10:00-18:00"\n    },\n    {\n      "id": "node/11108426366",\n      "kind": "kindergarten",\n      "name": "Киндерли",\n      "lat": 55.8107715,\n      "lon": 37.5573152,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11361866977",\n      "kind": "kindergarten",\n      "name": "Moscow International School",\n      "lat": 55.7476556,\n      "lon": 37.5188023,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11361932184",\n      "kind": "kindergarten",\n      "name": "Academkids",\n      "lat": 55.7506234,\n      "lon": 37.5162488,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11505302813",\n      "kind": "kindergarten",\n      "name": "Школа №1251 имени генерала Шарля де Голля, корпус №9",\n      "lat": 55.7938781,\n      "lon": 37.512356,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11505302818",\n      "kind": "kindergarten",\n      "name": "Школа №1251 имени генерала Шарля де Голля, корпус №1",\n      "lat": 55.7927389,\n      "lon": 37.5123118,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11584098151",\n      "kind": "kindergarten",\n      "name": "Школа № 56 имени Легасова. Дошкольное отделение",\n      "lat": 55.7480058,\n      "lon": 37.5514124,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11598702097",\n      "kind": "kindergarten",\n      "name": "Школа №\xa067. Отделение дошкольного образования",\n      "lat": 55.7393167,\n      "lon": 37.515432,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11618355902",\n      "kind": "kindergarten",\n      "name": "Discovery",\n      "lat": 55.7361332,\n      "lon": 37.5010461,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11618734695",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6434032,\n      "lon": 37.5196015,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11634229389",\n      "kind": "kindergarten",\n      "name": "Могу - Детская Архитектурная Студия",\n      "lat": 55.7550089,\n      "lon": 37.6477373,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11760867026",\n      "kind": "school",\n      "name": "Академическая Гимназия",\n      "lat": 55.755475,\n      "lon": 37.5085821,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11775013424",\n      "kind": "kindergarten",\n      "name": "Старт",\n      "lat": 55.7567506,\n      "lon": 37.5105635,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11780110793",\n      "kind": "kindergarten",\n      "name": "Хэппи Нэппи",\n      "lat": 55.795978,\n      "lon": 37.7070808,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12043951605",\n      "kind": "kindergarten",\n      "name": "Детская Академия Развития",\n      "lat": 55.6505559,\n      "lon": 37.5715486,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12215008301",\n      "kind": "school",\n      "name": "Школа №1636, здание 17",\n      "lat": 55.6017334,\n      "lon": 37.731282,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12241062907",\n      "kind": "kindergarten",\n      "name": "Космо Кидс",\n      "lat": 55.7254195,\n      "lon": 37.7429032,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12241615196",\n      "kind": "kindergarten",\n      "name": "Discovery",\n      "lat": 55.7405802,\n      "lon": 37.6012324,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12346514390",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7634623,\n      "lon": 37.5112229,\n      "opening_hours": null\n    }\n  ],\n  "limitations": [\n    "Counts are approximate mapped sites, not enrollment or daily attendance.",\n    "Distance uses OSM point/area center to GTFS stop, not walking distance.",\n    "OSM missing sites and mislabeled objects remain possible.",\n    "POIs use Jan 2025 snapshot; GTFS is retrospective June 2026 geometry.",\n    "Opening-hours tags are too sparse to assign verified timetables to every institution."\n  ]\n}\n', 'ml/forecast/build_education_registry.py': '"""Build a small attributed historical OSM exposure registry, without passenger imputation."""\nimport json,math,hashlib\nfrom pathlib import Path\nimport numpy as np\nROOT=Path(__file__).resolve().parents[2]\nRADIUS=500.\ndef distances(lat,lon,coords):\n    a=np.radians(coords);p=math.radians(lat);dl=a[:,0]-p;dn=a[:,1]-math.radians(lon)\n    return 6371000*2*np.arcsin(np.sqrt(np.clip(np.sin(dl/2)**2+np.cos(p)*np.cos(a[:,0])*np.sin(dn/2)**2,0,1)))\ndef main():\n    path=ROOT/\'artifacts/education/osm_school_kindergarten_20250101.json\';raw=json.loads(path.read_text());feed=json.loads((ROOT/\'artifacts/metro_audit/gtfs_routes.json\').read_text())\n    points=[];dropped=[]\n    # Prefer areas to nodes where both map the same object; conservative geometric dedup.\n    for e in sorted(raw[\'elements\'],key=lambda x:({\'relation\':0,\'way\':1,\'node\':2}[x[\'type\']],x[\'id\'])):\n        tags=e[\'tags\'];pos=e.get(\'center\',e);lat,lon=pos[\'lat\'],pos[\'lon\'];kind=tags[\'amenity\'];name=tags.get(\'name\',\'\').casefold().strip()\n        same=[p for p in points if p[\'kind\']==kind]\n        duplicate=False\n        if same:\n            ds=distances(lat,lon,np.array([[p[\'lat\'],p[\'lon\']] for p in same]))\n            duplicate=any(d<=20 or (name and name==p[\'name\'].casefold().strip() and d<=60) for p,d in zip(same,ds))\n        if duplicate:dropped.append(f"{e[\'type\']}/{e[\'id\']}");continue\n        points.append({\'id\':f"{e[\'type\']}/{e[\'id\']}",\'kind\':kind,\'name\':tags.get(\'name\',\'\'),\'lat\':lat,\'lon\':lon,\'opening_hours\':tags.get(\'opening_hours\')})\n    routes={}\n    for route,item in feed[\'routes\'].items():\n        stops=np.array([[float(s[\'stop_lat\']),float(s[\'stop_lon\'])] for s in item[\'stops\'].values()]);near=[]\n        for p in points:\n            if min(distances(p[\'lat\'],p[\'lon\'],stops))<=RADIUS:near.append(p)\n        routes[route]={\'school\':sum(p[\'kind\']==\'school\' for p in near),\'kindergarten\':sum(p[\'kind\']==\'kindergarten\' for p in near),\'poi_ids\':[p[\'id\'] for p in near],\'known_opening_hours\':sum(bool(p[\'opening_hours\']) for p in near)}\n    out={\'source\':\'https://www.openstreetmap.org/copyright\',\'license\':\'ODbL; © OpenStreetMap contributors\',\'api\':\'https://overpass-api.de/api/interpreter\',\'osm_snapshot\':\'2025-01-01T00:00:00Z\',\'osm_sha256\':hashlib.file_digest(path.open(\'rb\'),\'sha256\').hexdigest(),\'query\':(ROOT/\'artifacts/education/overpass_query.txt\').read_text(),\'gtfs_sha256\':feed[\'archive_sha256\'],\'radius_m\':RADIUS,\'raw_objects\':len(raw[\'elements\']),\'retained_objects\':len(points),\'deduplicated_ids\':dropped,\'routes\':routes,\'poi\':points,\'limitations\':[\'Counts are approximate mapped sites, not enrollment or daily attendance.\',\'Distance uses OSM point/area center to GTFS stop, not walking distance.\',\'OSM missing sites and mislabeled objects remain possible.\',\'POIs use Jan 2025 snapshot; GTFS is retrospective June 2026 geometry.\',\'Opening-hours tags are too sparse to assign verified timetables to every institution.\']}\n    (ROOT/\'ml/forecast/education_registry.json\').write_text(json.dumps(out,ensure_ascii=False,indent=2)+\'\\n\')\n    print(\'Raw\',len(raw[\'elements\']),\'retained\',len(points),\'routes\',routes)\nif __name__==\'__main__\':main()\n', 'ml/forecast/test_education.py': "import unittest\nfrom datetime import date\nimport numpy as np\nfrom ml.forecast.education_features import calendar_features,exposure_features\nfrom ml.forecast.education_competition import matrix,active,choose\nfrom ml.forecast.build_education_registry import distances\n\nclass EducationTests(unittest.TestCase):\n    def test_daycare_not_closed_for_school_summer(self):\n        c=calendar_features(('17',date(2025,7,8),8))\n        self.assertEqual(c[:3],[0,0,1]);self.assertEqual(c[7],1)\n\n    def test_calendar_scenarios_remain_separate(self):\n        c=calendar_features(('11',date(2025,10,6),8))\n        self.assertEqual(c[:3],[1,0,1])\n        c=calendar_features(('11',date(2025,10,27),8))\n        self.assertEqual(c[:3],[0,1,1])\n\n    def test_holidays_and_unknown_year(self):\n        self.assertEqual(calendar_features(('11',date(2025,11,4),8)),[0]*9)\n        with self.assertRaises(ValueError):calendar_features(('11',date(2026,1,1),8))\n\n    def test_unknown_route_no_fake_pois_and_service_routes_protected(self):\n        self.assertEqual(exposure_features(('5',date(2025,9,1),8)),[0]*11)\n        for r in ('5','7','50'):self.assertFalse(active((r,date(2025,9,1),8)))\n\n    def test_matched_arms_preserve_control_columns(self):\n        k=('17',date(2025,10,6),8);b={k:200};w={(k[1],8):[1.]*10}\n        a=matrix([k],b,w,'control');c=matrix([k],b,w,'calendar');e=matrix([k],b,w,'education')\n        np.testing.assert_equal(a,c[:,:a.shape[1]]);np.testing.assert_equal(c,e[:,:c.shape[1]])\n\n    def test_distance_units(self):\n        d=distances(55,37,np.array([[55,37],[55.01,37]]))\n        self.assertAlmostEqual(d[0],0);self.assertTrue(1100<d[1]<1120)\n\n    def test_poi_not_credited_for_calendar_gain(self):\n        ks=[('17',date(2025,m,6),8) for m in (9,10)];truth=dict.fromkeys(ks,100)\n        p={'incumbent':dict.fromkeys(ks,80)}\n        for w in (.25,.5,1.):\n            p[f'control_{w}']=dict.fromkeys(ks,80)\n            p[f'calendar_{w}']=dict.fromkeys(ks,90)\n            p[f'education_{w}']=dict.fromkeys(ks,90)\n        choices,_=choose(ks,truth,p,ks,p)\n        self.assertTrue(choices['17'].startswith('calendar_'))\n        for w in (.25,.5,1.):p[f'education_{w}']=dict.fromkeys(ks,95)\n        choices,_=choose(ks,truth,p,ks,p)\n        self.assertTrue(choices['17'].startswith('education_'))\n\nclass EducationProfileTests(unittest.TestCase):\n    def test_day_totals_and_protected_routes(self):\n        from ml.forecast.education_profiles import EducationProfiles\n        from ml.forecast.core import grid\n        history={k:(100 if k[2]==8 else 10) for k in grid(['17'],date(2025,1,1),date(2025,4,30))}\n        model=EducationProfiles().fit(history,date(2025,4,30))\n        keys=grid(['17','7','50','5'],date(2025,5,5),date(2025,5,5));base={k:(0 if k[0]=='5' or k[2]<5 else 30) for k in keys}\n        for name,p in model.variants(keys,base).items():\n            self.assertEqual(sum(p.values()),sum(base.values()))\n            self.assertTrue(all(p[k]==base[k] for k in keys if k[0]!='17'))\n            self.assertTrue(all(p[k]==0 for k in keys if base[k]==0))\n\n    def test_profile_rejects_future_history_and_partial_days(self):\n        from ml.forecast.education_profiles import EducationProfiles\n        with self.assertRaises(ValueError):EducationProfiles().fit({('17',date(2025,2,1),8):100},date(2025,1,31))\n        model=EducationProfiles().fit({('17',date(2025,1,1),8):100},date(2025,1,31))\n        k=('17',date(2025,2,1),8)\n        with self.assertRaises(ValueError):model.variants([k],{k:100})\n", 'ml/forecast/education_profiles.py': '"""Second preregistered experiment: calendar-conditioned hour shares; daily totals fixed."""\nfrom collections import defaultdict\nfrom datetime import date\nimport json,pickle,argparse\nfrom pathlib import Path\nimport numpy as np\nfrom .education_features import closed,QUARTER,MODULAR\nfrom .education_competition import TARGETS\nfrom .hourly_experts import allocate,HourlyExperts,HourlyExpertEnsemble,combine\nfrom .core import days,effective_weekday,grid,load_labels,sha256\nfrom .run import FIRST,FINAL_CUTOFF\nfrom . import network_competition as nc\nfrom .weather_competition import weather_rows\nfrom ml.submit.adapter import read_sample,write_submission\n\nWEIGHTS=(.25,.5)\n\ndef summer(d):return date(2025,5,27)<=d<=date(2025,8,31)\ndef condition(d,mode):\n    if mode==\'control\':return 0\n    return int(closed(d,QUARTER if mode==\'quarter\' else MODULAR))\n\nclass EducationProfiles:\n    def fit(self,history,cutoff):\n        if any(k[1]>cutoff or k[1]<FIRST for k in history):raise ValueError(\'History crosses cutoff\')\n        self.cutoff=cutoff;groups=defaultdict(list)\n        for r in TARGETS:\n            for d in days(FIRST,cutoff):\n                y=np.array([history.get((r,d,h),0) for h in range(24)],dtype=float)\n                if y.sum()<=0:continue\n                for mode in (\'control\',\'quarter\',\'modular\'):\n                    groups[r,effective_weekday(d),summer(d),mode,condition(d,mode)].append(y/y.sum())\n        self.shapes={}\n        for k,rows in groups.items():\n            if len(rows)<4:continue\n            a=np.median(rows,axis=0);b=np.median(rows[-8:],axis=0)\n            if a.sum()>0 and b.sum()>0:self.shapes[k]=.5*a/a.sum()+.5*b/b.sum()\n        return self\n    def shape(self,r,d,mode):\n        return self.shapes.get((r,effective_weekday(d),summer(d),mode,condition(d,mode)),self.shapes.get((r,effective_weekday(d),summer(d),\'control\',0)))\n    def variants(self,keys,base):\n        if any(k[1]<=self.cutoff for k in keys):raise ValueError(\'Predict after cutoff\')\n        result={\'incumbent\':dict(base)}\n        for mode in (\'control\',\'quarter\',\'modular\',\'mixed\'):\n            for w in WEIGHTS:result[f\'{mode}_{w}\']=dict(base)\n        grouped=defaultdict(list)\n        for k in keys:\n            if k[0] in TARGETS:grouped[k[0],k[1]].append(k)\n        for (r,d),ks in grouped.items():\n            if len(ks)!=24 or {k[2] for k in ks}!=set(range(24)):raise ValueError(\'Require full unique daily hours\')\n            ks=sorted(ks,key=lambda k:k[2]);v=np.array([base[k] for k in ks],float);total=int(v.sum())\n            if total<=0:continue\n            old=v/total\n            for mode in (\'control\',\'quarter\',\'modular\',\'mixed\'):\n                if mode==\'mixed\':\n                    a=self.shape(r,d,\'quarter\');b=self.shape(r,d,\'modular\');shape=(a+b)/2 if a is not None and b is not None else None\n                else:shape=self.shape(r,d,mode)\n                if shape is None:continue\n                shape=np.clip(shape,.5*old,1.5*old)\n                if shape.sum()<=0:continue\n                shape=shape/shape.sum()\n                for w in WEIGHTS:\n                    values=allocate(total,(1-w)*old+w*shape)\n                    result[f\'{mode}_{w}\'].update(zip(ks,values))\n        return result\n\ndef choose(keys,truth,preds,oks,op):\n    choices={};audit={}\n    for r in TARGETS:\n        parts=[[k for k in keys if k[0]==r and k[1].month==m] for m in (9,10)]\n        errors={n:[sum(abs(p[k]-truth.get(k,0)) for k in ks) for ks in parts] for n,p in preds.items()}\n        oe={n:sum(abs(p[k]-truth.get(k,0)) for k in oks if k[0]==r) for n,p in op.items()}\n        accepted=[]\n        for mode in (\'quarter\',\'modular\',\'mixed\'):\n            for w in WEIGHTS:\n                n=f\'{mode}_{w}\';c=f\'control_{w}\'\n                if all(a<=.995*b for a,b in zip(errors[n],errors[\'incumbent\'])) and all(a<=.995*b for a,b in zip(errors[n],errors[c])) and oe[n]<=1.005*oe[\'incumbent\'] and oe[n]<=1.005*oe[c]:accepted.append(n)\n        choices[r]=min(accepted,key=lambda n:sum(errors[n])) if accepted else \'incumbent\'\n        audit[r]={\'errors_sep_oct\':errors,\'errors_october_refit\':oe,\'accepted\':accepted,\'selected\':choices[r]}\n    return choices,audit\n\ndef join(keys,ps,choice):return {k:ps[choice.get(k[0],\'incumbent\')][k] for k in keys}\nclass Ensemble:\n    def __init__(self,baseline,profiles,choices):self.baseline,self.profiles,self.choices=baseline,profiles,choices\n    def predict(self,keys):\n        keys=list(keys);return join(keys,self.profiles.variants(keys,self.baseline.predict(keys)),self.choices)\n\ndef main(argv=None):\n    ap=argparse.ArgumentParser()\n    for name in (\'train\',\'test\',\'sample\',\'weather\',\'baseline-dir\',\'out\'):ap.add_argument(\'--\'+name,type=Path,required=True)\n    ap.add_argument(\'--sample-from-reference\',action=\'store_true\');args=ap.parse_args(argv)\n    nc.verify_competition_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')},args.sample_from_reference)\n    previous=json.loads((args.baseline_dir/\'hourly_experts_report.json\').read_text());values=load_labels([args.train,args.test]);ref=previous[\'reference\'];route_choices=previous[\'selection\'][\'by_route\'];hourly=previous[\'hourly_experts\'][\'choices\']\n    weather=weather_rows(json.loads(args.weather.read_text()));bank=nc.HistoryBank(values,ref[\'choices\']);held={};folds={};args.out.mkdir(parents=True,exist_ok=True)\n    for label,cutoff,lo,hi in [(\'sep_oct\',date(2025,8,31),date(2025,9,1),date(2025,10,31)),(\'october_refit\',date(2025,9,30),date(2025,10,1),date(2025,10,31))]:\n        print(\'Calendar hour profiles\',label,flush=True)\n        history={k:v for k,v in values.items() if k[1]<=cutoff};keys=grid(bank.routes,lo,hi);bases=bank.predict_bases(cutoff,keys);models,_=nc.fit_models(bank,cutoff,weather)\n        base=nc.assemble(keys,nc.predict_models(keys,bases,models,weather),route_choices);hp=HourlyExperts().fit(history,FIRST,cutoff);base=combine(keys,hp.variants(keys,base),hourly)\n        ps=EducationProfiles().fit(history,cutoff).variants(keys,base);held[label]=(keys,ps);folds[label]={\'metrics\':{n:nc.metrics(keys,values,p) for n,p in ps.items()}}\n    choices,audit=choose(held[\'sep_oct\'][0],values,held[\'sep_oct\'][1],*held[\'october_refit\'])\n    for label,(ks,ps) in held.items():folds[label][\'selected\']=nc.metrics(ks,values,join(ks,ps,choices))\n    _,sample=read_sample(args.sample);reference,regimes=bank.fitted(FINAL_CUTOFF);models,_=nc.fit_models(bank,FINAL_CUTOFF,weather)\n    baseline=nc.HolidayServiceEnsemble(nc.NetworkWeatherEnsemble(reference,regimes,models,weather,route_choices),reference,regimes)\n    baseline=HourlyExpertEnsemble(baseline,HourlyExperts().fit(values,FIRST,FINAL_CUTOFF),hourly)\n    before=baseline.predict(sample);write_submission(args.sample,before,args.out/\'submission_best_088078.csv\')\n    from .education_competition import BEST_HASH\n    assert sha256(args.out/\'submission_best_088078.csv\')==BEST_HASH\n    ensemble=Ensemble(baseline,EducationProfiles().fit(values,FINAL_CUTOFF),choices);after=ensemble.predict(sample);artifact=pickle.dumps(ensemble,protocol=5);assert pickle.loads(artifact).predict(sample)==after\n    write_submission(args.sample,after,args.out/\'submission_education_profiles.csv\');(args.out/\'model.local.pkl\').write_bytes(artifact)\n    report={\'experiment\':\'Second exploratory experiment: calendar-conditioned daily shares; not a proven POI effect.\',\'source_mode\':\'competition\',\'selected\':choices,\'selection\':audit,\'folds\':folds,\'primary_submission\':\'submission_education_profiles.csv\',\'changed_rows\':{r:sum(before[k]!=after[k] for k in sample if k[0]==r) for r in sorted({k[0] for k in sample},key=int)},\'daily_totals_preserved\':True,\'outputs_sha256\':{n:sha256(args.out/n) for n in (\'submission_education_profiles.csv\',\'submission_best_088078.csv\')},\'platform_score\':None}\n    (args.out/\'education_profiles_report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\');print(\'Selected:\',choices,flush=True)\nif __name__==\'__main__\':\n    from ml.forecast.education_profiles import main as run_main\n    run_main()\n', 'ml/forecast/office_hours.py': '"""Conservative weekly opening-hours parser. Unsupported syntax stays unknown."""\nimport re\nimport numpy as np\nDAYS=(\'Mo\',\'Tu\',\'We\',\'Th\',\'Fr\',\'Sa\',\'Su\')\ndef parse_hours(text):\n    if not text:return None\n    if text.strip()==\'24/7\':return np.ones((7,1440),dtype=bool)\n    result=np.zeros((7,1440),dtype=bool);seen=set()\n    for part in text.split(\';\'):\n        part=part.strip()\n        # Holiday exceptions require dated evaluation; leave unknown in this first experiment.\n        m=re.fullmatch(r\'(?:(Mo|Tu|We|Th|Fr|Sa|Su)([-,A-Za-z]*)\\s+)?(off|\\d{2}:\\d{2}-\\d{2}:\\d{2}(?:,\\s*\\d{2}:\\d{2}-\\d{2}:\\d{2})*)\',part)\n        if not m:return None\n        spec=(m[1] or \'\')+(m[2] or \'\')\n        selected=[]\n        if not spec:selected=list(range(7))\n        else:\n            for group in spec.split(\',\'):\n                ds=group.split(\'-\')\n                if any(d not in DAYS for d in ds) or len(ds)>2:return None\n                a=DAYS.index(ds[0]);b=DAYS.index(ds[-1])\n                selected.extend((a+i)%7 for i in range((b-a)%7+1))\n        if set(selected)&seen:return None  # ambiguous override semantics\n        seen.update(selected)\n        if m[3]==\'off\':continue\n        for interval in m[3].split(\',\'):\n            ends=[]\n            for t in interval.strip().split(\'-\'):\n                h,minute=map(int,t.split(\':\'))\n                if h>24 or minute>59 or (h==24 and minute):return None\n                ends.append(h*60+minute)\n            a,b=ends\n            if a>=b:return None  # overnight rules remain unknown\n            for day in selected:result[day,a:b]=True\n    return result\n\ndef hourly_signals(schedule):\n    opened=schedule.reshape(7,24,60).mean(axis=2)\n    previous=np.roll(schedule.reshape(-1),1).reshape(7,1440)\n    starts=(schedule & ~previous).reshape(7,24,60).sum(axis=2)\n    ends=(~schedule & previous).reshape(7,24,60).sum(axis=2)\n    return np.stack((opened,starts,ends),axis=2)\n', 'ml/forecast/office_features.py': '"""Office exposure and documented public weekly hours, with explicit missingness."""\nimport json,math\nfrom datetime import timedelta\nfrom pathlib import Path\nfrom .core import effective_weekday\nREGISTRY=json.loads(Path(__file__).with_name(\'office_registry.json\').read_text())\ndef density_features(key):\n    r,d,h=key;p=REGISTRY[\'routes\'].get(r,{});work=effective_weekday(d)<5\n    # Windows are commute hypotheses, not asserted office hours.\n    values=[math.log1p(p.get(c,0)) for c in (\'buildings\',\'organisations\')]\n    return values+[v*f for v in values for f in (work,work*(6<=h<=10),work*(16<=h<=20))]\ndef hours_features(key):\n    r,d,h=key;p=REGISTRY[\'routes\'].get(r)\n    if not p:return [0.]*8\n    weekday=effective_weekday(d);s=p[\'hour_signals\'];n=p[\'organisations\'];known=p[\'parsed_hours\']\n    return [math.log1p(known),known/max(1,n),math.log1p(n-known)]+[math.log1p(v) for v in s[weekday][h]]+[math.log1p(s[effective_weekday(d+timedelta(days=1)) if h==23 else weekday][(h+1)%24][1]),math.log1p(s[effective_weekday(d-timedelta(days=1)) if h==0 else weekday][(h-1)%24][2])]\n', 'ml/forecast/office_registry.json': '{\n "source": "https://www.openstreetmap.org/copyright",\n "license": "© OpenStreetMap contributors, ODbL",\n "snapshot": "2025-01-01T00:00:00Z",\n "access_date": "2026-09-27",\n "query": "[out:json][timeout:180][date:\\"2025-01-01T00:00:00Z\\"];(nwr[\\"office\\"][\\"office\\"!=\\"no\\"](55.584,37.44,55.898,37.834);nwr[\\"building\\"=\\"office\\"](55.584,37.44,55.898,37.834););out center tags;",\n "raw_sha256": "910f528bb198639038ae877274afe9446d9db3b3c6ff3b7b9b129e75616b0d56",\n "gtfs_sha256": "dec3ec0ccc2efb7f1327ee6f26a0636a4a271494490e6debedcb2c1ccc579e76",\n "radius_m": 500,\n "raw_objects": 10854,\n "deduplicated_ids": [\n  "relation/3939128",\n  "relation/14763979",\n  "way/28709590",\n  "way/52933367",\n  "way/1240432008",\n  "node/798569024",\n  "node/1233280554",\n  "node/6061295313",\n  "node/6153225181",\n  "node/6511504785",\n  "node/6554658685",\n  "node/6720763086",\n  "node/8270161407",\n  "node/8270161409",\n  "node/9612448451",\n  "node/9612450172",\n  "node/9612450372",\n  "node/9612450483",\n  "node/9612450597",\n  "node/9612452419",\n  "node/9612452421",\n  "node/9612453416",\n  "node/9612453568",\n  "node/9612454095",\n  "node/9718273055",\n  "node/9718274564",\n  "node/9718274662",\n  "node/9718274733",\n  "node/9718274926",\n  "node/9825485714",\n  "node/9867035891",\n  "node/9867035892",\n  "node/9867035903",\n  "node/9867035908",\n  "node/9867036324",\n  "node/9867036332",\n  "node/9867036336",\n  "node/9867036384",\n  "node/9867036388",\n  "node/10603591533",\n  "node/11375112007"\n ],\n "poi": [\n  {\n   "id": "relation/57729",\n   "kind": "organisation",\n   "name": "Российский государственный военный архив",\n   "office_type": "government",\n   "lat": 55.8341421,\n   "lon": 37.4896844,\n   "opening_hours": "Mo,We,Th 10:00-17:00; Tu 12:00-20:00; Fr 10:00-16:00"\n  },\n  {\n   "id": "relation/66913",\n   "kind": "organisation",\n   "name": "Федеральная служба по надзору в сфере транспорта (Ространснадзор)",\n   "office_type": "government",\n   "lat": 55.792281,\n   "lon": 37.5466227,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/85639",\n   "kind": "building",\n   "name": "Правительство Москвы",\n   "office_type": "government",\n   "lat": 55.7612458,\n   "lon": 37.6086384,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/85760",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588964,\n   "lon": 37.6244031,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/85763",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7611993,\n   "lon": 37.6262892,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/107124",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7200047,\n   "lon": 37.6236372,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/149242",\n   "kind": "building",\n   "name": "Военный комиссариат города Москвы",\n   "office_type": null,\n   "lat": 55.7771811,\n   "lon": 37.6318092,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/149243",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7803672,\n   "lon": 37.6400402,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/149244",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835515,\n   "lon": 37.6305804,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/153792",\n   "kind": "building",\n   "name": "Дом Правительства Российской Федерации",\n   "office_type": null,\n   "lat": 55.7550273,\n   "lon": 37.5732785,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/163593",\n   "kind": "building",\n   "name": "Бизнес-центр Валлекс",\n   "office_type": null,\n   "lat": 55.6607446,\n   "lon": 37.5380526,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/164345",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Лесная 43\\"",\n   "office_type": null,\n   "lat": 55.7806639,\n   "lon": 37.5906315,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/165294",\n   "kind": "organisation",\n   "name": "Посольство КНДР",\n   "office_type": "diplomatic",\n   "lat": 55.7127833,\n   "lon": 37.5084165,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/175337",\n   "kind": "organisation",\n   "name": "НИИ \\"Восход\\"",\n   "office_type": "research",\n   "lat": 55.6835705,\n   "lon": 37.4874263,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/361992",\n   "kind": "building",\n   "name": "ФГУП МИА «Россия сегодня»",\n   "office_type": "news_agency",\n   "lat": 55.7376805,\n   "lon": 37.5904924,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/368313",\n   "kind": "organisation",\n   "name": "Федеральное агентство по печати и массовым коммуникациям",\n   "office_type": "government",\n   "lat": 55.7667112,\n   "lon": 37.6085844,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/380619",\n   "kind": "organisation",\n   "name": "Министерство внутренних дел РФ",\n   "office_type": "government",\n   "lat": 55.7310235,\n   "lon": 37.613863,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/385807",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.68536,\n   "lon": 37.7364119,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/391481",\n   "kind": "organisation",\n   "name": "Федеральное агенство воздушного транспорта (Росавиация)",\n   "office_type": "government",\n   "lat": 55.7915251,\n   "lon": 37.5463885,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/418691",\n   "kind": "building",\n   "name": "Московский почтамт",\n   "office_type": null,\n   "lat": 55.7639856,\n   "lon": 37.6371544,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/421694",\n   "kind": "organisation",\n   "name": "Государственный архив Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7327437,\n   "lon": 37.5744794,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/448227",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7613242,\n   "lon": 37.6068559,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/451293",\n   "kind": "building",\n   "name": "Саввинское архиерейское подворье",\n   "office_type": null,\n   "lat": 55.76058,\n   "lon": 37.6118812,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/451743",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7558592,\n   "lon": 37.6296727,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/451744",\n   "kind": "building",\n   "name": "Администрация Президента Российской Федерации",\n   "office_type": "government",\n   "lat": 55.75549,\n   "lon": 37.6302134,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/451745",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7554226,\n   "lon": 37.6297491,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/451746",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7545656,\n   "lon": 37.6315931,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/454730",\n   "kind": "organisation",\n   "name": "Институт машиноведения им. А.А. Благонравова РАН",\n   "office_type": "research",\n   "lat": 55.7669372,\n   "lon": 37.6436214,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/454731",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7678027,\n   "lon": 37.6474492,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/455522",\n   "kind": "building",\n   "name": "Прокуратура Московской области",\n   "office_type": "government",\n   "lat": 55.7656901,\n   "lon": 37.6262903,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/456962",\n   "kind": "building",\n   "name": "Российский государственный архив социально-политической истории",\n   "office_type": "government",\n   "lat": 55.76251,\n   "lon": 37.6122702,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/530583",\n   "kind": "building",\n   "name": "Министерство транспорта Российской Федерации",\n   "office_type": "government",\n   "lat": 55.759838,\n   "lon": 37.6230807,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/530604",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7608517,\n   "lon": 37.6280738,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/530605",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623272,\n   "lon": 37.6271575,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/531869",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Ситидел\\"",\n   "office_type": null,\n   "lat": 55.7615903,\n   "lon": 37.6586253,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/532348",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7596592,\n   "lon": 37.6374032,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/532944",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7587862,\n   "lon": 37.6297104,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/532946",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7577444,\n   "lon": 37.6328982,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1103792",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7272169,\n   "lon": 37.8054676,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1112174",\n   "kind": "organisation",\n   "name": "Институт Океанологии РАН им. П.П.Ширшова",\n   "office_type": "research",\n   "lat": 55.6768894,\n   "lon": 37.5689267,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1187658",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Бэлрайс\\"",\n   "office_type": null,\n   "lat": 55.8205019,\n   "lon": 37.7820426,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1204804",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8199587,\n   "lon": 37.6640695,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1204805",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8197926,\n   "lon": 37.6637223,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1265068",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.754971,\n   "lon": 37.6248152,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1266364",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7566563,\n   "lon": 37.6273565,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1266365",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7562734,\n   "lon": 37.6280939,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1266607",\n   "kind": "building",\n   "name": "Федеральное агентство по государственным резервам",\n   "office_type": "government",\n   "lat": 55.756693,\n   "lon": 37.6263661,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1267765",\n   "kind": "building",\n   "name": "Посольство Словакии",\n   "office_type": "diplomatic",\n   "lat": 55.7719361,\n   "lon": 37.5898067,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1278302",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7897749,\n   "lon": 37.5868535,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1278304",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7838944,\n   "lon": 37.585678,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1286264",\n   "kind": "building",\n   "name": "Ямское Плаза",\n   "office_type": null,\n   "lat": 55.7853457,\n   "lon": 37.5846572,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1317288",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7708484,\n   "lon": 37.466125,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1339262",\n   "kind": "organisation",\n   "name": "Аэроэлектромаш",\n   "office_type": "research",\n   "lat": 55.7977034,\n   "lon": 37.5860169,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1359233",\n   "kind": "building",\n   "name": "Сенатский дворец",\n   "office_type": null,\n   "lat": 55.7532404,\n   "lon": 37.618783,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1360641",\n   "kind": "organisation",\n   "name": "Московский Кремль",\n   "office_type": "government",\n   "lat": 55.7516181,\n   "lon": 37.6178228,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1376656",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7699997,\n   "lon": 37.5929132,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1376680",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7707106,\n   "lon": 37.5942714,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1376682",\n   "kind": "organisation",\n   "name": "Минэкономразвития РФ",\n   "office_type": "government",\n   "lat": 55.7701748,\n   "lon": 37.5949871,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1383032",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7824169,\n   "lon": 37.583657,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1386032",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7715694,\n   "lon": 37.5952042,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1386035",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.771401,\n   "lon": 37.5954144,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1386041",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7711466,\n   "lon": 37.5958271,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1386065",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7713232,\n   "lon": 37.5958326,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1391936",\n   "kind": "organisation",\n   "name": "Наш центр",\n   "office_type": "yes",\n   "lat": 55.5903372,\n   "lon": 37.6703701,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1393275",\n   "kind": "building",\n   "name": "Дом Прошина",\n   "office_type": null,\n   "lat": 55.7738967,\n   "lon": 37.5901993,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1394790",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.778523,\n   "lon": 37.5860689,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1397821",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7778725,\n   "lon": 37.6581263,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1410498",\n   "kind": "organisation",\n   "name": "ГКУ ЦОДД",\n   "office_type": "government",\n   "lat": 55.7809961,\n   "lon": 37.5871274,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1414615",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Маяковская плаза\\"",\n   "office_type": null,\n   "lat": 55.772228,\n   "lon": 37.6006128,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1421004",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.770832,\n   "lon": 37.6006784,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1421015",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7704348,\n   "lon": 37.600818,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1427106",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7710634,\n   "lon": 37.602346,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1427131",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7709086,\n   "lon": 37.6046097,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1427141",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7705762,\n   "lon": 37.6047974,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428192",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7701159,\n   "lon": 37.604878,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428820",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7671288,\n   "lon": 37.6053748,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428831",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7668988,\n   "lon": 37.6051529,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428850",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7661986,\n   "lon": 37.6044022,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428852",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7667918,\n   "lon": 37.606197,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428859",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7668422,\n   "lon": 37.6056477,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428863",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7664697,\n   "lon": 37.6060485,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428869",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7666716,\n   "lon": 37.6044189,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428871",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7672914,\n   "lon": 37.6058989,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1428877",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7663429,\n   "lon": 37.605319,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1434414",\n   "kind": "building",\n   "name": "Очаково",\n   "office_type": null,\n   "lat": 55.6736695,\n   "lon": 37.440821,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1451195",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7507192,\n   "lon": 37.6070778,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1517647",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8527625,\n   "lon": 37.6687034,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1518697",\n   "kind": "building",\n   "name": "Комплекс \\"Башня на набережной\\"",\n   "office_type": null,\n   "lat": 55.7467501,\n   "lon": 37.5367624,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1529891",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "research",\n   "lat": 55.8034961,\n   "lon": 37.6486606,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1543687",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7249932,\n   "lon": 37.6429475,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1606951",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7742267,\n   "lon": 37.5846035,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1662024",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7691025,\n   "lon": 37.4800238,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1708760",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7739859,\n   "lon": 37.5840259,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1740134",\n   "kind": "building",\n   "name": "Олимпийский комитет России",\n   "office_type": null,\n   "lat": 55.7143992,\n   "lon": 37.5672499,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1749678",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7707769,\n   "lon": 37.6382577,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1752365",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7602279,\n   "lon": 37.5943158,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1752366",\n   "kind": "organisation",\n   "name": "Генеральное консульство Греции",\n   "office_type": "diplomatic",\n   "lat": 55.7602708,\n   "lon": 37.5944573,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1758250",\n   "kind": "organisation",\n   "name": "Издательство \\"Просвещение\\"",\n   "office_type": "publisher",\n   "lat": 55.79601,\n   "lon": 37.6110748,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1759236",\n   "kind": "building",\n   "name": "территория ГБУ \\"Автомобильные дороги\\"",\n   "office_type": null,\n   "lat": 55.7848028,\n   "lon": 37.480829,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1761513",\n   "kind": "building",\n   "name": "Бизнес-центр «Каланчёвский»",\n   "office_type": null,\n   "lat": 55.7722871,\n   "lon": 37.6516242,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1763581",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7577345,\n   "lon": 37.6674215,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1787566",\n   "kind": "building",\n   "name": "Капитал Плаза",\n   "office_type": null,\n   "lat": 55.7791744,\n   "lon": 37.5871465,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1819045",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8064852,\n   "lon": 37.5014955,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1824846",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.738012,\n   "lon": 37.5334723,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1839408",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7501235,\n   "lon": 37.6481595,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1839418",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7480429,\n   "lon": 37.6471928,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1839419",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Яузская, 5\\"",\n   "office_type": null,\n   "lat": 55.7495354,\n   "lon": 37.6451058,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1850032",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7376936,\n   "lon": 37.5959248,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1850037",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7413434,\n   "lon": 37.5971562,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1850045",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7384379,\n   "lon": 37.5938084,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1850049",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7401486,\n   "lon": 37.5958751,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1850050",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7401772,\n   "lon": 37.5956301,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1850056",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7416346,\n   "lon": 37.5950479,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1850058",\n   "kind": "organisation",\n   "name": "Государственный океанографический институт им. Н. Н. Зубова",\n   "office_type": "research",\n   "lat": 55.7378525,\n   "lon": 37.5938968,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1850065",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7412372,\n   "lon": 37.5947736,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1850072",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7370615,\n   "lon": 37.5971076,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1853036",\n   "kind": "organisation",\n   "name": "Российские космические системы",\n   "office_type": "engineer",\n   "lat": 55.7678293,\n   "lon": 37.6813304,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1856842",\n   "kind": "building",\n   "name": "АтомСтройЭкспорт",\n   "office_type": "company",\n   "lat": 55.809345,\n   "lon": 37.5816766,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1934067",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7471003,\n   "lon": 37.6075767,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1934069",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.746742,\n   "lon": 37.6086636,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1934070",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7471251,\n   "lon": 37.6079113,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1934072",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7470174,\n   "lon": 37.6081927,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1939138",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7597014,\n   "lon": 37.6165762,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/1939143",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7607089,\n   "lon": 37.6161783,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2018324",\n   "kind": "organisation",\n   "name": "Гидропроект",\n   "office_type": "research",\n   "lat": 55.8078453,\n   "lon": 37.5052848,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2019683",\n   "kind": "organisation",\n   "name": "ЦИАМ им. Баранова",\n   "office_type": "research",\n   "lat": 55.7608941,\n   "lon": 37.7093277,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2024252",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7004913,\n   "lon": 37.6216686,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2024253",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7000029,\n   "lon": 37.6220209,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2071256",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6652736,\n   "lon": 37.4865213,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2098703",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8746398,\n   "lon": 37.588026,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2104071",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7679591,\n   "lon": 37.6056166,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2142862",\n   "kind": "organisation",\n   "name": "Главное архитектурно-планировочное управление Москомархитектуры",\n   "office_type": "government",\n   "lat": 55.7695172,\n   "lon": 37.5924768,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2150657",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.745526,\n   "lon": 37.6032996,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2150659",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7455032,\n   "lon": 37.6027877,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2150661",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7453531,\n   "lon": 37.6019699,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2150664",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7453691,\n   "lon": 37.6025294,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2150666",\n   "kind": "building",\n   "name": "Институт русского языка РАН",\n   "office_type": "research",\n   "lat": 55.7451956,\n   "lon": 37.6029327,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2154490",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7459094,\n   "lon": 37.6009519,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2154491",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461437,\n   "lon": 37.602457,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2156223",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7468707,\n   "lon": 37.6020326,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2156224",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.74646,\n   "lon": 37.6018311,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2156295",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7483313,\n   "lon": 37.6024171,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2156519",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7486061,\n   "lon": 37.6013327,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2162284",\n   "kind": "building",\n   "name": "Лабораторный корпус",\n   "office_type": null,\n   "lat": 55.825469,\n   "lon": 37.4602899,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2167545",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Ферро-плаза\\"",\n   "office_type": null,\n   "lat": 55.6796506,\n   "lon": 37.567001,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2169831",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7566396,\n   "lon": 37.5955131,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2170469",\n   "kind": "organisation",\n   "name": "Ростест-Москва",\n   "office_type": "government",\n   "lat": 55.6693544,\n   "lon": 37.5833067,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2178941",\n   "kind": "organisation",\n   "name": "АйТеко",\n   "office_type": "it",\n   "lat": 55.6836338,\n   "lon": 37.5742571,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2236140",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.732792,\n   "lon": 37.7673753,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2351001",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7609181,\n   "lon": 37.6330624,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2369032",\n   "kind": "organisation",\n   "name": "ДОК 1",\n   "office_type": "company",\n   "lat": 55.6073797,\n   "lon": 37.6459514,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2386975",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7796716,\n   "lon": 37.5943466,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2400253",\n   "kind": "organisation",\n   "name": "Гохран России",\n   "office_type": "government",\n   "lat": 55.7429045,\n   "lon": 37.5214479,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2427995",\n   "kind": "organisation",\n   "name": "Институт органической химии им. Н. Д. Зелинского РАН",\n   "office_type": "research",\n   "lat": 55.7015681,\n   "lon": 37.5728268,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2427996",\n   "kind": "organisation",\n   "name": "Институт элементоорганических соединений им. А. Н. Несмеянова РАН",\n   "office_type": "research",\n   "lat": 55.7005952,\n   "lon": 37.5748815,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2669505",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7624158,\n   "lon": 37.6230131,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2669547",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7615341,\n   "lon": 37.6107684,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2669553",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7597406,\n   "lon": 37.6124342,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2746582",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7768503,\n   "lon": 37.7211199,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2761707",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6720935,\n   "lon": 37.6185103,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2801667",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7925547,\n   "lon": 37.6982819,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2834106",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623365,\n   "lon": 37.629941,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2849452",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7086464,\n   "lon": 37.6081816,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2849453",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7089955,\n   "lon": 37.6078272,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2849454",\n   "kind": "building",\n   "name": "Специальное конструкторское бюро сго",\n   "office_type": "company",\n   "lat": 55.708467,\n   "lon": 37.6080913,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2849458",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7086956,\n   "lon": 37.6081159,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2861680",\n   "kind": "organisation",\n   "name": "Посольство Индии. Консульский отдел, экономический и коммерческий отдел.",\n   "office_type": "diplomatic",\n   "lat": 55.7528442,\n   "lon": 37.6484194,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/2898652",\n   "kind": "organisation",\n   "name": "Роснано",\n   "office_type": "company",\n   "lat": 55.692743,\n   "lon": 37.5744744,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3146442",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7584322,\n   "lon": 37.6239214,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3159022",\n   "kind": "building",\n   "name": "Институт проблем безопасного развития атомной энергетики",\n   "office_type": "research",\n   "lat": 55.7041241,\n   "lon": 37.6210696,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3259584",\n   "kind": "organisation",\n   "name": "Телецентр Russia Today",\n   "office_type": "company",\n   "lat": 55.7722487,\n   "lon": 37.7147911,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3266537",\n   "kind": "organisation",\n   "name": "Центр эксплуатации обьектов космической инфраструктуры",\n   "office_type": "engineer",\n   "lat": 55.7444122,\n   "lon": 37.7313896,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3321177",\n   "kind": "organisation",\n   "name": "Совет Федерации",\n   "office_type": "government",\n   "lat": 55.7645862,\n   "lon": 37.6125137,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3325783",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7661913,\n   "lon": 37.6287029,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3369084",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7699532,\n   "lon": 37.5566737,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3369085",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7686441,\n   "lon": 37.5589068,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3458029",\n   "kind": "organisation",\n   "name": "Телевизионный технический центр Останкино им.\xa050-летия Октября",\n   "office_type": "telecommunication",\n   "lat": 55.8215516,\n   "lon": 37.6063131,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3493352",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6599146,\n   "lon": 37.5380955,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3551173",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7747347,\n   "lon": 37.5854558,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3596814",\n   "kind": "organisation",\n   "name": "посольство Пакистана",\n   "office_type": "diplomatic",\n   "lat": 55.7647889,\n   "lon": 37.5879527,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3612070",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8641486,\n   "lon": 37.6769917,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3618426",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7833216,\n   "lon": 37.6319788,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3683969",\n   "kind": "organisation",\n   "name": "Шаболовский телецентр",\n   "office_type": "telecommunication",\n   "lat": 55.7178135,\n   "lon": 37.6112449,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3745052",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7685309,\n   "lon": 37.590276,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3760043",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.738012,\n   "lon": 37.5102465,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3789120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7751969,\n   "lon": 37.6118946,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3818407",\n   "kind": "building",\n   "name": "Телецентр «Останкино» АСК-1",\n   "office_type": null,\n   "lat": 55.8226942,\n   "lon": 37.606286,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3881262",\n   "kind": "building",\n   "name": "МИД России центральное здание",\n   "office_type": null,\n   "lat": 55.7461772,\n   "lon": 37.5844084,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3881263",\n   "kind": "building",\n   "name": "Новый корпус МИД России",\n   "office_type": null,\n   "lat": 55.7447413,\n   "lon": 37.5857498,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3884587",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7949891,\n   "lon": 37.587417,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3893034",\n   "kind": "organisation",\n   "name": "Институт физических проблем имени П. Л. Капицы РАН",\n   "office_type": "research",\n   "lat": 55.7090722,\n   "lon": 37.5768482,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3905188",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7475233,\n   "lon": 37.6768697,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3940974",\n   "kind": "organisation",\n   "name": "Отделение № 3 Главного управления Банка России по Центральному федеральному округу",\n   "office_type": "bank",\n   "lat": 55.7722972,\n   "lon": 37.5483635,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3951747",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449574,\n   "lon": 37.6588853,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/3951751",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432489,\n   "lon": 37.6589695,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4036811",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Морозов\\"",\n   "office_type": null,\n   "lat": 55.7341361,\n   "lon": 37.5879565,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4036812",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7336473,\n   "lon": 37.5875717,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4036817",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7342169,\n   "lon": 37.5879565,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4036821",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.734183,\n   "lon": 37.5883628,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4036824",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7341183,\n   "lon": 37.5877956,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4036831",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7340859,\n   "lon": 37.5880035,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4124853",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7869403,\n   "lon": 37.584467,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4179785",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7213036,\n   "lon": 37.5731619,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4227357",\n   "kind": "organisation",\n   "name": "Государственный астрономический институт им. П. К. Штернберга МГУ",\n   "office_type": "research",\n   "lat": 55.7009656,\n   "lon": 37.5426879,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4237433",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7519099,\n   "lon": 37.5858978,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4237454",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7519117,\n   "lon": 37.5887155,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4237468",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7520451,\n   "lon": 37.5957232,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4237469",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.751975,\n   "lon": 37.5925922,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4240736",\n   "kind": "organisation",\n   "name": "Институт нанотехнологий микроэлектроники",\n   "office_type": "research",\n   "lat": 55.6758148,\n   "lon": 37.6445882,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4485804",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7638634,\n   "lon": 37.630126,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4485805",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7641214,\n   "lon": 37.6300002,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4494169",\n   "kind": "organisation",\n   "name": "Московский институт теплотехники",\n   "office_type": "research",\n   "lat": 55.8558278,\n   "lon": 37.6040172,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4839934",\n   "kind": "organisation",\n   "name": "Центральная таможенная служба Российской Федерации",\n   "office_type": "government",\n   "lat": 55.8073127,\n   "lon": 37.5909258,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/4867297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7946462,\n   "lon": 37.5878502,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5364656",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.774678,\n   "lon": 37.6085009,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5369311",\n   "kind": "organisation",\n   "name": "Русклимат",\n   "office_type": "company",\n   "lat": 55.842133,\n   "lon": 37.501498,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5371134",\n   "kind": "building",\n   "name": "Усадьба Шаховского-Краузе-Осиповых",\n   "office_type": null,\n   "lat": 55.7533076,\n   "lon": 37.602669,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5371135",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7548502,\n   "lon": 37.6011026,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5424719",\n   "kind": "organisation",\n   "name": "посольство Японии",\n   "office_type": "diplomatic",\n   "lat": 55.7778596,\n   "lon": 37.6418127,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5428217",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.776899,\n   "lon": 37.6538505,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5428219",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7757694,\n   "lon": 37.6540813,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5428220",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7756576,\n   "lon": 37.6531005,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5511732",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7429847,\n   "lon": 37.5259927,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5511733",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7425366,\n   "lon": 37.5274031,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5559920",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.824946,\n   "lon": 37.6164589,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5567733",\n   "kind": "organisation",\n   "name": "посольство Финляндии",\n   "office_type": "diplomatic",\n   "lat": 55.7388678,\n   "lon": 37.5904417,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5569529",\n   "kind": "organisation",\n   "name": "посольство Египта",\n   "office_type": "diplomatic",\n   "lat": 55.7391504,\n   "lon": 37.5916634,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5586727",\n   "kind": "organisation",\n   "name": "Теплотехнический научно-исследовательский институт",\n   "office_type": "research",\n   "lat": 55.7050663,\n   "lon": 37.6486428,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5642647",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7885243,\n   "lon": 37.6920401,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5703841",\n   "kind": "organisation",\n   "name": "Посольство Польши",\n   "office_type": "diplomatic",\n   "lat": 55.7680882,\n   "lon": 37.5776165,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5710903",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7616678,\n   "lon": 37.5757272,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5754110",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7944466,\n   "lon": 37.6326011,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5767908",\n   "kind": "organisation",\n   "name": "ЦНИИ стоматологии и челюстно-лицевой хирургии",\n   "office_type": "research",\n   "lat": 55.7348401,\n   "lon": 37.5906667,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5767909",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7342897,\n   "lon": 37.5904724,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5818982",\n   "kind": "building",\n   "name": "Правительство Москвы",\n   "office_type": null,\n   "lat": 55.7537707,\n   "lon": 37.57727,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5850957",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8286085,\n   "lon": 37.5804485,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5943029",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7604951,\n   "lon": 37.5793567,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5943035",\n   "kind": "building",\n   "name": "Группа Компаний ПИК",\n   "office_type": "company",\n   "lat": 55.7603303,\n   "lon": 37.5799054,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5943159",\n   "kind": "organisation",\n   "name": "Минэкономразвития России",\n   "office_type": "government",\n   "lat": 55.7604135,\n   "lon": 37.5747566,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5966462",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт по изысканию новых антибиотиков им. Г. Ф. Гаузе",\n   "office_type": "research",\n   "lat": 55.7346697,\n   "lon": 37.5787377,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/5968416",\n   "kind": "building",\n   "name": "Государственная Дума",\n   "office_type": "government",\n   "lat": 55.7579196,\n   "lon": 37.6157998,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6036052",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7565404,\n   "lon": 37.5428918,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6036056",\n   "kind": "building",\n   "name": "Служба эксплуатации",\n   "office_type": null,\n   "lat": 55.7567735,\n   "lon": 37.5433349,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6049578",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6637375,\n   "lon": 37.5633958,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6074086",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6687384,\n   "lon": 37.5775314,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6074089",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6654544,\n   "lon": 37.5793018,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6090248",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Серпуховской двор\\"",\n   "office_type": null,\n   "lat": 55.7059712,\n   "lon": 37.6025233,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6093554",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "diplomatic",\n   "lat": 55.7694817,\n   "lon": 37.5896215,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6141294",\n   "kind": "building",\n   "name": "Министерство природных ресурсов и экологии Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7618437,\n   "lon": 37.5800005,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6141299",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7617582,\n   "lon": 37.5808977,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6141309",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7614772,\n   "lon": 37.5816509,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6141321",\n   "kind": "building",\n   "name": "Управление собственной безопасности Москвы",\n   "office_type": "government",\n   "lat": 55.7623404,\n   "lon": 37.582328,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6155073",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7382962,\n   "lon": 37.5906287,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6172516",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.755803,\n   "lon": 37.5830537,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6193706",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.729055,\n   "lon": 37.5891649,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6193992",\n   "kind": "building",\n   "name": "Департамент экономической политики и развития города Москвы",\n   "office_type": "government",\n   "lat": 55.7615448,\n   "lon": 37.6078749,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6210566",\n   "kind": "building",\n   "name": "МЧС России",\n   "office_type": "government",\n   "lat": 55.759539,\n   "lon": 37.622061,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6210571",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7600203,\n   "lon": 37.6226726,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6210572",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7598083,\n   "lon": 37.6221249,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6210578",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7606291,\n   "lon": 37.6224143,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6228039",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7556528,\n   "lon": 37.614162,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "relation/6316684",\n   "kind": "organisation",\n   "name": "ФГУП «Гидроспецгеология»",\n   "office_type": "research",\n   "lat": 55.7974539,\n   "lon": 37.4922027,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6320281",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8672138,\n   "lon": 37.5613564,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6381244",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7407434,\n   "lon": 37.6242126,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6398017",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7519542,\n   "lon": 37.6709822,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6406060",\n   "kind": "organisation",\n   "name": "Патриаршая резиденция в Чистом переулке",\n   "office_type": "religion",\n   "lat": 55.7422593,\n   "lon": 37.5914037,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6414818",\n   "kind": "organisation",\n   "name": "Бизнес-центр \\"Каскад\\"",\n   "office_type": "company",\n   "lat": 55.7731643,\n   "lon": 37.5882483,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6446664",\n   "kind": "organisation",\n   "name": "Институт физической химии и электрохимии им А.Н Фрумкина РАН",\n   "office_type": "research",\n   "lat": 55.6512906,\n   "lon": 37.5420026,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6495522",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7403585,\n   "lon": 37.5886307,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6495523",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7410372,\n   "lon": 37.5874592,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6544636",\n   "kind": "organisation",\n   "name": "НИИ социологии медицины, экономики здравоохранения и медицинского страхования",\n   "office_type": "research",\n   "lat": 55.7344823,\n   "lon": 37.5800648,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6618016",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8641286,\n   "lon": 37.552448,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6675398",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6799059,\n   "lon": 37.5684944,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6700701",\n   "kind": "organisation",\n   "name": "ВНИИ организации, управления и экономики нефтегазовой промышленности",\n   "office_type": "research",\n   "lat": 55.6611092,\n   "lon": 37.5554567,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6700895",\n   "kind": "organisation",\n   "name": "НИИ Транснефть",\n   "office_type": "research",\n   "lat": 55.6705488,\n   "lon": 37.5870659,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6702158",\n   "kind": "organisation",\n   "name": "Управление кабельных сетей Северо-западного округа",\n   "office_type": "company",\n   "lat": 55.8142933,\n   "lon": 37.4887285,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6740205",\n   "kind": "organisation",\n   "name": "Центральный научно-исследовательский автомобильный и автомоторный институт",\n   "office_type": "research",\n   "lat": 55.8482503,\n   "lon": 37.5421709,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6780160",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7791261,\n   "lon": 37.5885723,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6780161",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.778731,\n   "lon": 37.5879471,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6849323",\n   "kind": "organisation",\n   "name": "Посольство Ирака",\n   "office_type": "diplomatic",\n   "lat": 55.7322369,\n   "lon": 37.5663152,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6858292",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7400835,\n   "lon": 37.636776,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6913337",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7326341,\n   "lon": 37.6009304,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6913343",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Крымский Вал\\"",\n   "office_type": null,\n   "lat": 55.7292626,\n   "lon": 37.6091901,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6913849",\n   "kind": "organisation",\n   "name": "ВНИИ пищевой биотехнологии",\n   "office_type": "research",\n   "lat": 55.7568399,\n   "lon": 37.6743921,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6913850",\n   "kind": "organisation",\n   "name": "ОАО «Биопрепарат-Центр»",\n   "office_type": "research",\n   "lat": 55.7566103,\n   "lon": 37.6731045,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6915200",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8726743,\n   "lon": 37.5187634,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/6957069",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7675887,\n   "lon": 37.6163707,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7006925",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.8232919,\n   "lon": 37.6457568,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7029390",\n   "kind": "organisation",\n   "name": "Московские кабельные сети",\n   "office_type": "company",\n   "lat": 55.7456303,\n   "lon": 37.6348438,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7029392",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7460114,\n   "lon": 37.6340496,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7029393",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7456936,\n   "lon": 37.6340138,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7037890",\n   "kind": "organisation",\n   "name": "Объединенный институт высоких температур Российской академии наук",\n   "office_type": "research",\n   "lat": 55.8929522,\n   "lon": 37.5194508,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7061943",\n   "kind": "organisation",\n   "name": "Красный Куб",\n   "office_type": "company",\n   "lat": 55.8426731,\n   "lon": 37.502065,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7100433",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7415111,\n   "lon": 37.6279653,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7100434",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7413507,\n   "lon": 37.6275415,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7103022",\n   "kind": "organisation",\n   "name": "Посольство Ганы",\n   "office_type": "diplomatic",\n   "lat": 55.7560854,\n   "lon": 37.5947968,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7119666",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7657679,\n   "lon": 37.6120304,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7119667",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7659564,\n   "lon": 37.6120844,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7119668",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7660207,\n   "lon": 37.6117814,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7121405",\n   "kind": "organisation",\n   "name": "Российская академия архитектуры и строительных наук",\n   "office_type": "ngo",\n   "lat": 55.7638735,\n   "lon": 37.6130473,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7149139",\n   "kind": "building",\n   "name": "апарт-комплекс «Metropolis Loft»",\n   "office_type": null,\n   "lat": 55.7630801,\n   "lon": 37.6368503,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7152416",\n   "kind": "organisation",\n   "name": "Генеральная прокуратура Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7633024,\n   "lon": 37.6117378,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7154322",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.762759,\n   "lon": 37.6055469,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7175428",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7306888,\n   "lon": 37.7358837,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7175429",\n   "kind": "building",\n   "name": "Бизнес–центр \\"Карачарово\\"",\n   "office_type": null,\n   "lat": 55.7304464,\n   "lon": 37.7368602,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7175430",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.730339,\n   "lon": 37.7380914,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7239516",\n   "kind": "building",\n   "name": "Информационное телеграфное агентство России «ТАСС»",\n   "office_type": "news_agency",\n   "lat": 55.7578901,\n   "lon": 37.5997015,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259780",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7586859,\n   "lon": 37.6328131,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259787",\n   "kind": "organisation",\n   "name": "Департамент капитального ремонта города Москвы",\n   "office_type": "government",\n   "lat": 55.7586171,\n   "lon": 37.6349837,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259809",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7584846,\n   "lon": 37.632318,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259811",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7579641,\n   "lon": 37.6311287,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259812",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7581285,\n   "lon": 37.6315146,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259815",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7584355,\n   "lon": 37.6314914,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259819",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7575722,\n   "lon": 37.6317765,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259820",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7579597,\n   "lon": 37.6320023,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259822",\n   "kind": "building",\n   "name": "Российский союз молодежи",\n   "office_type": "ngo",\n   "lat": 55.7575742,\n   "lon": 37.6313893,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259823",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7584082,\n   "lon": 37.6320438,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7259826",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7583488,\n   "lon": 37.6317983,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7270174",\n   "kind": "building",\n   "name": "Всероссийский научно-исследовательский институт технической эстетики",\n   "office_type": "research",\n   "lat": 55.8300884,\n   "lon": 37.6214314,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7302411",\n   "kind": "organisation",\n   "name": "Институт ядерных исследований Российской академии наук",\n   "office_type": "research",\n   "lat": 55.6982526,\n   "lon": 37.5825522,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7303363",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.748677,\n   "lon": 37.5837014,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7326540",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7841889,\n   "lon": 37.7162063,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7328019",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835223,\n   "lon": 37.7138923,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7330295",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7833567,\n   "lon": 37.7076005,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7330297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7833362,\n   "lon": 37.7068,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7332453",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7802683,\n   "lon": 37.6958718,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7333938",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7790407,\n   "lon": 37.6888879,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7334011",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7519076,\n   "lon": 37.6414771,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7384310",\n   "kind": "organisation",\n   "name": "ПАТЕРО Девелопмент",\n   "office_type": "company",\n   "lat": 55.7348087,\n   "lon": 37.45076,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7394180",\n   "kind": "building",\n   "name": "Российская академия наук",\n   "office_type": "research",\n   "lat": 55.7107949,\n   "lon": 37.5779208,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7418769",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7393914,\n   "lon": 37.6311695,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7418770",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7391751,\n   "lon": 37.6308281,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7418773",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7384528,\n   "lon": 37.6308379,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7418774",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7382082,\n   "lon": 37.6308877,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7424314",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6706851,\n   "lon": 37.4742502,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7442263",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7371464,\n   "lon": 37.6335042,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7442266",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7357723,\n   "lon": 37.6347421,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7442268",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7354798,\n   "lon": 37.6349833,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7442272",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7361707,\n   "lon": 37.6323275,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7442274",\n   "kind": "organisation",\n   "name": "Замоскворецкая межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7353729,\n   "lon": 37.6335204,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7442282",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7360659,\n   "lon": 37.6336966,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7453714",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7371804,\n   "lon": 37.6289721,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7473219",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.718861,\n   "lon": 37.6270738,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7499651",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7556791,\n   "lon": 37.6364286,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7500835",\n   "kind": "building",\n   "name": "ФГУП \\"Росатомстрой\\"",\n   "office_type": null,\n   "lat": 55.7445733,\n   "lon": 37.6209687,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7500836",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447856,\n   "lon": 37.6221298,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7558664",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7685941,\n   "lon": 37.6079982,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7606631",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7697397,\n   "lon": 37.614751,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7606632",\n   "kind": "organisation",\n   "name": "Главное управление МВД России по городу Москве",\n   "office_type": "police",\n   "lat": 55.7699676,\n   "lon": 37.6140155,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7609336",\n   "kind": "building",\n   "name": "Экспертно-криминалистический центр ГУ МВД России по г. Москве",\n   "office_type": null,\n   "lat": 55.7704824,\n   "lon": 37.6178023,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7623553",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7544278,\n   "lon": 37.6441914,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7627802",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.764132,\n   "lon": 37.6288217,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7639816",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6662639,\n   "lon": 37.4631952,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7665799",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7431142,\n   "lon": 37.5041758,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7722017",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7380831,\n   "lon": 37.5908735,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7751611",\n   "kind": "organisation",\n   "name": "Бизнес-центр Лето",\n   "office_type": "yes",\n   "lat": 55.6816826,\n   "lon": 37.5165247,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7761398",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7733561,\n   "lon": 37.6781408,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7763757",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Северное Медведково",\n   "office_type": "government",\n   "lat": 55.8889773,\n   "lon": 37.639724,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "relation/7771744",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Аврора\\"",\n   "office_type": null,\n   "lat": 55.7365115,\n   "lon": 37.6406545,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7771745",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Аврора\\"",\n   "office_type": null,\n   "lat": 55.7354255,\n   "lon": 37.6414037,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7771761",\n   "kind": "organisation",\n   "name": "ЦССК ДОСААФ России",\n   "office_type": "quango",\n   "lat": 55.8239264,\n   "lon": 37.4430254,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7793081",\n   "kind": "organisation",\n   "name": "НИИ вакцин и сывороток им. И. И. Мечникова",\n   "office_type": "research",\n   "lat": 55.762075,\n   "lon": 37.6550504,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7793487",\n   "kind": "building",\n   "name": "Филиал ФГУП \\"ЦЭНКИ\\" - НИИСК им. В.П. Бармина",\n   "office_type": "research",\n   "lat": 55.6647373,\n   "lon": 37.4880373,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7809894",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7861497,\n   "lon": 37.5805566,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7809900",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7880413,\n   "lon": 37.5836408,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7809901",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7870968,\n   "lon": 37.5838421,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7840001",\n   "kind": "organisation",\n   "name": "АО «Всерегиональное объединение «Изотоп»",\n   "office_type": "company",\n   "lat": 55.7294411,\n   "lon": 37.5609263,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7840005",\n   "kind": "organisation",\n   "name": "Издательский Совет Русской Православной Церкви",\n   "office_type": "religion",\n   "lat": 55.7298968,\n   "lon": 37.5611562,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7895066",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт автоматики им. Н. Л. Духова",\n   "office_type": "research",\n   "lat": 55.7831702,\n   "lon": 37.6037841,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7895073",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.783514,\n   "lon": 37.6035848,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7909759",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7478461,\n   "lon": 37.6622744,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7916401",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7486357,\n   "lon": 37.661268,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7916402",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7489484,\n   "lon": 37.6614137,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7916403",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7488238,\n   "lon": 37.6612455,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/7932048",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8029545,\n   "lon": 37.5189041,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8001762",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7665279,\n   "lon": 37.6438773,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8004426",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7663238,\n   "lon": 37.6383541,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8016233",\n   "kind": "building",\n   "name": "Росстат",\n   "office_type": "government",\n   "lat": 55.767527,\n   "lon": 37.6411165,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8062614",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7559233,\n   "lon": 37.6516092,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8062617",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7593689,\n   "lon": 37.6487723,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8082302",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "diplomatic",\n   "lat": 55.7543232,\n   "lon": 37.653003,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8097460",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8444147,\n   "lon": 37.5720902,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8118911",\n   "kind": "organisation",\n   "name": "Научно-технический центр радиационно-химической безопасности и гигиены",\n   "office_type": "research",\n   "lat": 55.8087222,\n   "lon": 37.4652962,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8152662",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7613811,\n   "lon": 37.6242598,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8157277",\n   "kind": "organisation",\n   "name": "НИИ эпидемиологии и микробиологии им. Н. Ф. Гамалеи",\n   "office_type": "research",\n   "lat": 55.8016291,\n   "lon": 37.4608493,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8167981",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7097173,\n   "lon": 37.6238595,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8167983",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7096562,\n   "lon": 37.6243992,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8288384",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7673609,\n   "lon": 37.6444127,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8304320",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.741985,\n   "lon": 37.5292752,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8304324",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7399504,\n   "lon": 37.5287747,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8356566",\n   "kind": "organisation",\n   "name": "Краснопресненская обсерватория ГАИШ МГУ",\n   "office_type": "research",\n   "lat": 55.7581928,\n   "lon": 37.5685699,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8367575",\n   "kind": "organisation",\n   "name": "Научно-исследовательский центр космической гидрометеорологии \\"Планета\\"",\n   "office_type": "research",\n   "lat": 55.7590783,\n   "lon": 37.5697555,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8368008",\n   "kind": "organisation",\n   "name": "Федеральная служба по гидрометеорологии и мониторингу окружающей среды",\n   "office_type": "government",\n   "lat": 55.7589977,\n   "lon": 37.5681585,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8368009",\n   "kind": "organisation",\n   "name": "Гидрометцентр России",\n   "office_type": "research",\n   "lat": 55.7591903,\n   "lon": 37.5690059,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8378065",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт строительной физики",\n   "office_type": "research",\n   "lat": 55.845278,\n   "lon": 37.5723785,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8446526",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7334332,\n   "lon": 37.6311349,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8463134",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7541748,\n   "lon": 37.7151446,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8463583",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.75701,\n   "lon": 37.6044691,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8463584",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7573158,\n   "lon": 37.6040605,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8469406",\n   "kind": "building",\n   "name": "Деловой центр «Лубянский»",\n   "office_type": null,\n   "lat": 55.7566075,\n   "lon": 37.6328271,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8495825",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.75304,\n   "lon": 37.609933,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8511101",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7435313,\n   "lon": 37.5238815,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8511102",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432127,\n   "lon": 37.5226608,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8511103",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7428917,\n   "lon": 37.5211892,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8511104",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7429117,\n   "lon": 37.5201012,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8511105",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7428748,\n   "lon": 37.5191935,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8513516",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7409147,\n   "lon": 37.5244455,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8513517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7404963,\n   "lon": 37.5243573,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8589843",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8236005,\n   "lon": 37.5019798,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8625749",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7403587,\n   "lon": 37.5274366,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8642542",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7576788,\n   "lon": 37.6316506,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8642543",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7584934,\n   "lon": 37.6299842,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8685001",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.734917,\n   "lon": 37.6595753,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8714888",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8341843,\n   "lon": 37.6491901,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8718673",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8313927,\n   "lon": 37.6453923,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8722473",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8344131,\n   "lon": 37.6463268,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8732927",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.804986,\n   "lon": 37.6198366,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8742734",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7984822,\n   "lon": 37.6313331,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8747382",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7703635,\n   "lon": 37.6086704,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8747385",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7709517,\n   "lon": 37.606707,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8747386",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7710334,\n   "lon": 37.6070341,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763458",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7554405,\n   "lon": 37.655009,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763459",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7552352,\n   "lon": 37.6550451,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763460",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7553401,\n   "lon": 37.6547472,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763461",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7551572,\n   "lon": 37.6548185,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763462",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7549536,\n   "lon": 37.654249,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763463",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7550497,\n   "lon": 37.6541972,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763464",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7552059,\n   "lon": 37.6544924,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763465",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7554999,\n   "lon": 37.6545547,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763466",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7554011,\n   "lon": 37.6540145,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8763467",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.755572,\n   "lon": 37.654181,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8770862",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7570289,\n   "lon": 37.6403111,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8782294",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7527315,\n   "lon": 37.6524531,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8782301",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7529857,\n   "lon": 37.6507388,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8782313",\n   "kind": "building",\n   "name": "Национальный НИИ общественного здоровья им. Н.А. Семашко",\n   "office_type": "research",\n   "lat": 55.753345,\n   "lon": 37.6526436,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8782314",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7528834,\n   "lon": 37.6529897,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8807045",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7664731,\n   "lon": 37.5886318,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8840439",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7610278,\n   "lon": 37.6306359,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8878388",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7388775,\n   "lon": 37.6673207,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8888850",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7558301,\n   "lon": 37.6387099,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8888852",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7572326,\n   "lon": 37.6270014,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8888865",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7585792,\n   "lon": 37.6390082,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8888886",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7587076,\n   "lon": 37.630432,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8888889",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7593671,\n   "lon": 37.6314638,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8888890",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588699,\n   "lon": 37.6313631,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8888891",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7594518,\n   "lon": 37.6303057,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8888894",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7596684,\n   "lon": 37.6300211,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8888901",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7584869,\n   "lon": 37.630938,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8898273",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7345691,\n   "lon": 37.6350817,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8901350",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7368641,\n   "lon": 37.6598107,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8901449",\n   "kind": "organisation",\n   "name": "Посольство Брунея",\n   "office_type": "diplomatic",\n   "lat": 55.7395406,\n   "lon": 37.6145908,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8910972",\n   "kind": "organisation",\n   "name": "Федеральная налоговая служба",\n   "office_type": "government",\n   "lat": 55.7645989,\n   "lon": 37.6192468,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8910974",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7631114,\n   "lon": 37.6290649,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8910976",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626924,\n   "lon": 37.6289115,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8910977",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626954,\n   "lon": 37.6293732,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8910979",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7628759,\n   "lon": 37.6294747,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8910980",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7632032,\n   "lon": 37.6296338,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8910981",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7628992,\n   "lon": 37.6290985,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8910985",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.763325,\n   "lon": 37.6291476,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8910987",\n   "kind": "organisation",\n   "name": "Генеральная прокуратура Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7632822,\n   "lon": 37.6183321,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8911006",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7642893,\n   "lon": 37.6336708,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8911010",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7631349,\n   "lon": 37.6328033,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/8911013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7634551,\n   "lon": 37.632945,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9084214",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7214255,\n   "lon": 37.7411755,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9090749",\n   "kind": "organisation",\n   "name": "Институт проблем управления им. В.А. Трапезникова РАН",\n   "office_type": "research",\n   "lat": 55.6513341,\n   "lon": 37.5377094,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9113070",\n   "kind": "organisation",\n   "name": "ПАО «Лукойл»",\n   "office_type": "company",\n   "lat": 55.7672912,\n   "lon": 37.6362178,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9125994",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7671521,\n   "lon": 37.6361289,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9201158",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819755,\n   "lon": 37.6127165,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9241422",\n   "kind": "building",\n   "name": "Федеральная служба государственной регистрации, кадастра и картографии",\n   "office_type": "government",\n   "lat": 55.7634864,\n   "lon": 37.6396015,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9302998",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7580186,\n   "lon": 37.6232095,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9307522",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7555972,\n   "lon": 37.6264404,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9333444",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591083,\n   "lon": 37.5984081,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9341579",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7602969,\n   "lon": 37.599309,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9357304",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7445833,\n   "lon": 37.6381621,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9360768",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7431987,\n   "lon": 37.6403454,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9390359",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8449377,\n   "lon": 37.6345345,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9402392",\n   "kind": "organisation",\n   "name": "Лаборатория электронной микроскопии",\n   "office_type": "educational_institution",\n   "lat": 55.7073599,\n   "lon": 37.5216441,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9472317",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7314012,\n   "lon": 37.6219689,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9475059",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.732156,\n   "lon": 37.6209041,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9482185",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7354218,\n   "lon": 37.6228122,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9482186",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7356646,\n   "lon": 37.6228907,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9482188",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7351951,\n   "lon": 37.620999,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9484529",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7326799,\n   "lon": 37.6209908,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9527437",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7466765,\n   "lon": 37.6353865,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9527696",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7347616,\n   "lon": 37.7329543,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9527702",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7344983,\n   "lon": 37.7328502,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9551871",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7869062,\n   "lon": 37.5346785,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9551875",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7873023,\n   "lon": 37.533287,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9593880",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390554,\n   "lon": 37.5169118,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9612625",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7599839,\n   "lon": 37.6374739,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9612626",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7601142,\n   "lon": 37.6380985,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9620168",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7768573,\n   "lon": 37.6114108,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9798995",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7383519,\n   "lon": 37.619304,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9825007",\n   "kind": "organisation",\n   "name": "Посольство Латвийской республики",\n   "office_type": "diplomatic",\n   "lat": 55.7641358,\n   "lon": 37.6471112,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9825008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7644864,\n   "lon": 37.6471447,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9830036",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7628016,\n   "lon": 37.6484628,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9830044",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7630463,\n   "lon": 37.6480641,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9830045",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7633479,\n   "lon": 37.6476993,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9880631",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7633499,\n   "lon": 37.6532689,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9897397",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7566669,\n   "lon": 37.6036477,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9897398",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7568141,\n   "lon": 37.6024996,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9903335",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7304271,\n   "lon": 37.6346831,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9906547",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7562943,\n   "lon": 37.6077216,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9920534",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588758,\n   "lon": 37.6020776,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9920535",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7589574,\n   "lon": 37.6019601,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9920538",\n   "kind": "building",\n   "name": "Представительство Объединённых Наций",\n   "office_type": "diplomatic",\n   "lat": 55.7592176,\n   "lon": 37.6021157,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9920539",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591324,\n   "lon": 37.6015365,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9920540",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7590875,\n   "lon": 37.6010561,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9920541",\n   "kind": "building",\n   "name": "ИТАР-ТАСС",\n   "office_type": "news_agency",\n   "lat": 55.7593158,\n   "lon": 37.6008361,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9920542",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7590713,\n   "lon": 37.6005938,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9920547",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7583805,\n   "lon": 37.5999301,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9941129",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7658749,\n   "lon": 37.5897786,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9941135",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7633165,\n   "lon": 37.5862611,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9960278",\n   "kind": "organisation",\n   "name": "НИИ гигиены и охраны здоровья детей и подростков",\n   "office_type": "research",\n   "lat": 55.7607675,\n   "lon": 37.6533662,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9969073",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7483661,\n   "lon": 37.6290508,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9975361",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7479829,\n   "lon": 37.6323026,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9975365",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7479905,\n   "lon": 37.6316664,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9979657",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7515097,\n   "lon": 37.6433788,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/9979659",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7511266,\n   "lon": 37.6436212,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10045082",\n   "kind": "building",\n   "name": "Лихоборское подразделение Московского УЦПК ОАО «РЖД»",\n   "office_type": "railway",\n   "lat": 55.8408808,\n   "lon": 37.5192024,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10142465",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Сильвер Стоун\\"",\n   "office_type": null,\n   "lat": 55.8464669,\n   "lon": 37.6554977,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10161990",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7575265,\n   "lon": 37.607658,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10165677",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7587881,\n   "lon": 37.6102498,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10165678",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7580304,\n   "lon": 37.608298,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10312433",\n   "kind": "organisation",\n   "name": "Научно-технический центр",\n   "office_type": "research",\n   "lat": 55.761067,\n   "lon": 37.5196828,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10335884",\n   "kind": "building",\n   "name": "Деловой центр \\"на Цветном бульваре\\"",\n   "office_type": null,\n   "lat": 55.7713304,\n   "lon": 37.6227264,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10340517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.75788,\n   "lon": 37.6260041,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10375133",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7742552,\n   "lon": 37.6121534,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10409017",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7458829,\n   "lon": 37.5865972,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10409025",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Плотников\\"",\n   "office_type": null,\n   "lat": 55.7466599,\n   "lon": 37.5881064,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10409026",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7465632,\n   "lon": 37.5873565,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10409027",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7462728,\n   "lon": 37.5868847,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10409028",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7462272,\n   "lon": 37.5860891,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10409037",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.746206,\n   "lon": 37.5979429,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10433069",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7779878,\n   "lon": 37.6636632,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10441094",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7668942,\n   "lon": 37.6497571,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10460404",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт стандартизации и унификации",\n   "office_type": "research",\n   "lat": 55.7919141,\n   "lon": 37.6515352,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10461929",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7675965,\n   "lon": 37.6488188,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10461930",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7680909,\n   "lon": 37.6497524,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10461932",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7673554,\n   "lon": 37.6503653,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10474317",\n   "kind": "organisation",\n   "name": "NextGIS",\n   "office_type": "GIS",\n   "lat": 55.7710651,\n   "lon": 37.6619905,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10485563",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7578834,\n   "lon": 37.6307008,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10485565",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7579452,\n   "lon": 37.6304476,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10492243",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7660317,\n   "lon": 37.6487178,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10492245",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7667806,\n   "lon": 37.6510679,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10492246",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7669642,\n   "lon": 37.6512096,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10492247",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7672362,\n   "lon": 37.650988,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10492249",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7664462,\n   "lon": 37.6512393,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10492252",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7666216,\n   "lon": 37.6507935,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10559481",\n   "kind": "organisation",\n   "name": "Калибр",\n   "office_type": "coworking",\n   "lat": 55.8086936,\n   "lon": 37.629538,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10595748",\n   "kind": "organisation",\n   "name": "Федеральная служба безопасности Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7609702,\n   "lon": 37.6271739,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10595749",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.760457,\n   "lon": 37.6286046,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10617158",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7649062,\n   "lon": 37.6501943,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10701996",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7688849,\n   "lon": 37.6249478,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10721838",\n   "kind": "building",\n   "name": "Арбитражный суд г. Москвы",\n   "office_type": null,\n   "lat": 55.707555,\n   "lon": 37.6236722,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10782393",\n   "kind": "organisation",\n   "name": "Научно-исследовательский и конструкторский институт энерготехники им. Н. А. Доллежаля",\n   "office_type": "research",\n   "lat": 55.7877105,\n   "lon": 37.6536611,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10842713",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7620987,\n   "lon": 37.6330943,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10952288",\n   "kind": "organisation",\n   "name": "Министерство иностранных дел Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7456543,\n   "lon": 37.5847815,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10952289",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7441745,\n   "lon": 37.5846359,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10969772",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.72596,\n   "lon": 37.6109763,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10976811",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7857425,\n   "lon": 37.6737217,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10976813",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7863612,\n   "lon": 37.6746366,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10976815",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7854069,\n   "lon": 37.6752673,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10976817",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7857858,\n   "lon": 37.6745818,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10976819",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7856662,\n   "lon": 37.6741753,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/10991376",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7869312,\n   "lon": 37.6735366,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11003505",\n   "kind": "organisation",\n   "name": "Посольство Гренады",\n   "office_type": "diplomatic",\n   "lat": 55.7564613,\n   "lon": 37.5896267,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11007551",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7164061,\n   "lon": 37.6308039,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11035803",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7894186,\n   "lon": 37.5018437,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11078410",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Лотос\\"",\n   "office_type": null,\n   "lat": 55.6641223,\n   "lon": 37.5989509,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11084072",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7086999,\n   "lon": 37.6454217,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11084392",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.78649,\n   "lon": 37.6755158,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11084394",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7859507,\n   "lon": 37.6758362,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11092579",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7860896,\n   "lon": 37.6753786,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11093180",\n   "kind": "organisation",\n   "name": "ГАОУ ДПО Центр педагогического мастерства",\n   "office_type": "government",\n   "lat": 55.7171364,\n   "lon": 37.5762218,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11156164",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8051603,\n   "lon": 37.5803195,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11208825",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7871109,\n   "lon": 37.6849257,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11208831",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7847792,\n   "lon": 37.6884946,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11224840",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7976984,\n   "lon": 37.5846486,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11224841",\n   "kind": "organisation",\n   "name": "НИ и ПИ Градплан города Москвы",\n   "office_type": "research",\n   "lat": 55.7989576,\n   "lon": 37.5847294,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11227983",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8011439,\n   "lon": 37.5841021,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11234556",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8005744,\n   "lon": 37.5861574,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11234557",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7994358,\n   "lon": 37.5862969,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11254965",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8122976,\n   "lon": 37.4613995,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11256188",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7406639,\n   "lon": 37.7352187,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11257406",\n   "kind": "organisation",\n   "name": "Государственный научный центр дерматовенерологии и косметологии",\n   "office_type": "research",\n   "lat": 55.7966419,\n   "lon": 37.6921612,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11259766",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8005562,\n   "lon": 37.5846586,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11259767",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7999264,\n   "lon": 37.5847772,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11259769",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8027576,\n   "lon": 37.5844537,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11259770",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8031609,\n   "lon": 37.5843739,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11259771",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8035447,\n   "lon": 37.5843246,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11259772",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8041502,\n   "lon": 37.5843015,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11259775",\n   "kind": "building",\n   "name": "Бизнес-центр «Дмитровский»",\n   "office_type": null,\n   "lat": 55.8042505,\n   "lon": 37.5834314,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11259777",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Z-plaza\\"",\n   "office_type": null,\n   "lat": 55.8028365,\n   "lon": 37.5837552,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11292283",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7405117,\n   "lon": 37.6959396,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11292284",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7411508,\n   "lon": 37.6953004,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11292289",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7402714,\n   "lon": 37.695456,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11292828",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432055,\n   "lon": 37.6915714,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11296305",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7398294,\n   "lon": 37.694014,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11296306",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7401727,\n   "lon": 37.6936143,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11310358",\n   "kind": "organisation",\n   "name": "ПАО \\"НПП Импульс\\"",\n   "office_type": "research",\n   "lat": 55.8016151,\n   "lon": 37.6395883,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11313551",\n   "kind": "organisation",\n   "name": "НИИ хлебопекарной промышлености",\n   "office_type": "research",\n   "lat": 55.7951762,\n   "lon": 37.7401144,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11325715",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт железнодорожного транспорта",\n   "office_type": "research",\n   "lat": 55.8028507,\n   "lon": 37.6479213,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11335632",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7703094,\n   "lon": 37.6663807,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11340216",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.766922,\n   "lon": 37.6555265,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11341175",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7691914,\n   "lon": 37.6529436,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11341178",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7677292,\n   "lon": 37.6516411,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11341179",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7675039,\n   "lon": 37.6520469,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11341180",\n   "kind": "building",\n   "name": "Российские железные дороги",\n   "office_type": "company",\n   "lat": 55.7685242,\n   "lon": 37.6519433,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11344013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7693149,\n   "lon": 37.6544811,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11344016",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.76779,\n   "lon": 37.6523176,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11344329",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7725554,\n   "lon": 37.6619359,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11345933",\n   "kind": "organisation",\n   "name": "Министерство энергетики РФ",\n   "office_type": "government",\n   "lat": 55.7835215,\n   "lon": 37.6305804,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11345934",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7831917,\n   "lon": 37.6303154,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11351986",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7702468,\n   "lon": 37.6570549,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11351990",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7702231,\n   "lon": 37.6551736,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11351994",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7705731,\n   "lon": 37.6554228,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11351995",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7704632,\n   "lon": 37.6559934,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11352001",\n   "kind": "building",\n   "name": "Российский государственный университет инновационных технологий и предпринимательства",\n   "office_type": null,\n   "lat": 55.7698317,\n   "lon": 37.6556078,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11369154",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7607673,\n   "lon": 37.6250711,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11380679",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7073976,\n   "lon": 37.6139316,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11385219",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8550369,\n   "lon": 37.6717007,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11478847",\n   "kind": "building",\n   "name": "Московская дирекция по энергообеспечению",\n   "office_type": "railway",\n   "lat": 55.7842044,\n   "lon": 37.6434347,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11486847",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8145101,\n   "lon": 37.6066668,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11581983",\n   "kind": "building",\n   "name": "Межрайонный отдел лицензионно-разрешительной работы УВД",\n   "office_type": "government",\n   "lat": 55.7313615,\n   "lon": 37.5802832,\n   "opening_hours": "Tu,Th 10:00-17:00"\n  },\n  {\n   "id": "relation/11647935",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт шинной промышленности",\n   "office_type": "research",\n   "lat": 55.75822,\n   "lon": 37.7361518,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11679775",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7960227,\n   "lon": 37.7071193,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11722526",\n   "kind": "organisation",\n   "name": "Всероссийская государственная телевизионная и радиовещательная компания",\n   "office_type": "media",\n   "lat": 55.7880074,\n   "lon": 37.5785635,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724065",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.772522,\n   "lon": 37.610094,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724072",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7717904,\n   "lon": 37.6108321,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724074",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7714943,\n   "lon": 37.6110963,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724075",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7720445,\n   "lon": 37.6113099,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724076",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7718735,\n   "lon": 37.6113923,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724077",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7717171,\n   "lon": 37.611585,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724078",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7720228,\n   "lon": 37.6117809,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724079",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7717911,\n   "lon": 37.612024,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724080",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7716061,\n   "lon": 37.6118661,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11724081",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7715155,\n   "lon": 37.6120971,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11749528",\n   "kind": "organisation",\n   "name": "Московский институт электромеханики и автоматики",\n   "office_type": "research",\n   "lat": 55.8006748,\n   "lon": 37.5387289,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11749723",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8078203,\n   "lon": 37.5206997,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11757544",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8112929,\n   "lon": 37.6245289,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11840862",\n   "kind": "organisation",\n   "name": "Управление кабельных сетей Юго-Восточного округа",\n   "office_type": "engineer",\n   "lat": 55.7456148,\n   "lon": 37.68679,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11855557",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7796172,\n   "lon": 37.6638232,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11855559",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7791853,\n   "lon": 37.6631133,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11863749",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7761856,\n   "lon": 37.66337,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11909218",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7878181,\n   "lon": 37.6308427,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11909219",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7876287,\n   "lon": 37.6311169,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11978680",\n   "kind": "organisation",\n   "name": "Наш Солнечный Мир",\n   "office_type": "association",\n   "lat": 55.7299149,\n   "lon": 37.7233956,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11993059",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8221987,\n   "lon": 37.7985962,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/11993878",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623448,\n   "lon": 37.5681386,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12085584",\n   "kind": "organisation",\n   "name": "Российская телевизионная и радиовещательная сеть",\n   "office_type": "telecommunication",\n   "lat": 55.8197323,\n   "lon": 37.612726,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12231243",\n   "kind": "organisation",\n   "name": "Концерн \\"Радиотехнические и информационные системы\\"",\n   "office_type": "research",\n   "lat": 55.8036099,\n   "lon": 37.5545955,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12231247",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.802803,\n   "lon": 37.554145,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12272023",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7883417,\n   "lon": 37.641053,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12307630",\n   "kind": "organisation",\n   "name": "Московский городской центр дезинфекции",\n   "office_type": "government",\n   "lat": 55.8540522,\n   "lon": 37.6773346,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "relation/12362912",\n   "kind": "organisation",\n   "name": "Главное управление федеральной службы судебных приставов",\n   "office_type": "government",\n   "lat": 55.7753723,\n   "lon": 37.7065657,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12397969",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7027505,\n   "lon": 37.6491625,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12432667",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт фтизиопульмонологии ММА им. И. М. Сеченова",\n   "office_type": "research",\n   "lat": 55.7844343,\n   "lon": 37.6098806,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12452847",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7392881,\n   "lon": 37.5082151,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12503161",\n   "kind": "organisation",\n   "name": "Управление кабельных сетей САО",\n   "office_type": "engineer",\n   "lat": 55.7997314,\n   "lon": 37.5827286,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12554294",\n   "kind": "organisation",\n   "name": "Центр информационных технологий и систем органов исполнительной власти",\n   "office_type": "it",\n   "lat": 55.7711831,\n   "lon": 37.5675543,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12554302",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7705804,\n   "lon": 37.5665222,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565003",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835114,\n   "lon": 37.5839686,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565010",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7840112,\n   "lon": 37.5847196,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565011",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7827674,\n   "lon": 37.5834647,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565012",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7828424,\n   "lon": 37.5831033,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.782833,\n   "lon": 37.5824472,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565019",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7857158,\n   "lon": 37.580739,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565020",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7860714,\n   "lon": 37.5812061,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565022",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7865072,\n   "lon": 37.5813495,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565527",\n   "kind": "building",\n   "name": "",\n   "office_type": "telecommunication",\n   "lat": 55.7861837,\n   "lon": 37.5774217,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12565528",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7873126,\n   "lon": 37.5778115,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12593853",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8082424,\n   "lon": 37.6080795,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12593854",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8075484,\n   "lon": 37.605401,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12593855",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8072973,\n   "lon": 37.6044973,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12593891",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8021621,\n   "lon": 37.6076841,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12593905",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8028235,\n   "lon": 37.6073659,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12604607",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7942357,\n   "lon": 37.6576945,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12612455",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7590859,\n   "lon": 37.6404425,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12618513",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8411139,\n   "lon": 37.6503177,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12618514",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8418674,\n   "lon": 37.6515747,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12622816",\n   "kind": "organisation",\n   "name": "Всероссийский НИИ сельскохозяйственной биотехнологии РАСХН",\n   "office_type": "research",\n   "lat": 55.8270198,\n   "lon": 37.5571822,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12681154",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7895976,\n   "lon": 37.7540532,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12681155",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7894652,\n   "lon": 37.7538694,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12681158",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.790154,\n   "lon": 37.7540643,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12695735",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7611521,\n   "lon": 37.5710534,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12695736",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.760928,\n   "lon": 37.5695319,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12695737",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7607691,\n   "lon": 37.5699259,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12695738",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7606804,\n   "lon": 37.5705247,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12695739",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7610489,\n   "lon": 37.5701816,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12701603",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8042278,\n   "lon": 37.5706477,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12701604",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8043929,\n   "lon": 37.5700219,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12703923",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8495786,\n   "lon": 37.4539436,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12703924",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8494,\n   "lon": 37.4548154,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12717207",\n   "kind": "organisation",\n   "name": "Союздорпроект",\n   "office_type": "company",\n   "lat": 55.7675302,\n   "lon": 37.6701769,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12721167",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8053364,\n   "lon": 37.7494868,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12721169",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "research",\n   "lat": 55.8006013,\n   "lon": 37.7375114,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12721170",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "research",\n   "lat": 55.8002514,\n   "lon": 37.7372625,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12730192",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8127255,\n   "lon": 37.7736781,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12730242",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8093369,\n   "lon": 37.7751307,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12743120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7793188,\n   "lon": 37.6711589,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12743122",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7793894,\n   "lon": 37.6692197,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12743123",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7794675,\n   "lon": 37.6694261,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12770663",\n   "kind": "organisation",\n   "name": "Район по эксплуатации водопроводной сети №2",\n   "office_type": "property_management",\n   "lat": 55.8628684,\n   "lon": 37.7014901,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12771684",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7631365,\n   "lon": 37.5744174,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12811682",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7301878,\n   "lon": 37.6358023,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12817787",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7726024,\n   "lon": 37.5481946,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/12900987",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7691273,\n   "lon": 37.6730075,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13069740",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8123206,\n   "lon": 37.6319672,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13161365",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7425386,\n   "lon": 37.5309748,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13311708",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Академик\\"",\n   "office_type": null,\n   "lat": 55.6745342,\n   "lon": 37.5033794,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13353101",\n   "kind": "building",\n   "name": "Бизнес-центр \\"AFI2B\\"",\n   "office_type": null,\n   "lat": 55.7749307,\n   "lon": 37.5847375,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13393226",\n   "kind": "organisation",\n   "name": "Институт синтетических полимерных материалов им. Н.С. Ениколопова РАН",\n   "office_type": "research",\n   "lat": 55.6634497,\n   "lon": 37.5461071,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13398224",\n   "kind": "organisation",\n   "name": "Резиденция посла Ирана",\n   "office_type": "diplomatic",\n   "lat": 55.6646822,\n   "lon": 37.5280476,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13450864",\n   "kind": "organisation",\n   "name": "Сибур",\n   "office_type": "company",\n   "lat": 55.6799547,\n   "lon": 37.5688252,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "relation/13450866",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6799984,\n   "lon": 37.5693492,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13465821",\n   "kind": "organisation",\n   "name": "Мосводоканал",\n   "office_type": "company",\n   "lat": 55.7713724,\n   "lon": 37.674583,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13465823",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7714056,\n   "lon": 37.6744124,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13486046",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8116009,\n   "lon": 37.4566552,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13486048",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8123154,\n   "lon": 37.4562393,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13486054",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8134273,\n   "lon": 37.4571298,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13533473",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7760911,\n   "lon": 37.6543698,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13560918",\n   "kind": "building",\n   "name": "Блок С",\n   "office_type": null,\n   "lat": 55.7470539,\n   "lon": 37.5366308,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13560919",\n   "kind": "building",\n   "name": "Блок В",\n   "office_type": null,\n   "lat": 55.7464551,\n   "lon": 37.5363494,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13560920",\n   "kind": "building",\n   "name": "Блок А",\n   "office_type": null,\n   "lat": 55.7466594,\n   "lon": 37.5373757,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13561876",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7699046,\n   "lon": 37.6129416,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13588106",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8814631,\n   "lon": 37.5480527,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13626693",\n   "kind": "building",\n   "name": "Росатом",\n   "office_type": "company",\n   "lat": 55.7398653,\n   "lon": 37.6232752,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13687010",\n   "kind": "building",\n   "name": "",\n   "office_type": "research",\n   "lat": 55.801701,\n   "lon": 37.4577227,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13783336",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7768301,\n   "lon": 37.649782,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13838105",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7651105,\n   "lon": 37.6785519,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13852090",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7434273,\n   "lon": 37.5997794,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13852091",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.743553,\n   "lon": 37.5999946,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13876669",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8228486,\n   "lon": 37.5702974,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13883096",\n   "kind": "organisation",\n   "name": "Со. здание",\n   "office_type": "employment_agency",\n   "lat": 55.7696694,\n   "lon": 37.6769264,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13900808",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Нагатинский\\"",\n   "office_type": null,\n   "lat": 55.6830102,\n   "lon": 37.661348,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13968086",\n   "kind": "organisation",\n   "name": "Рособоронэкспорт",\n   "office_type": "government",\n   "lat": 55.7956946,\n   "lon": 37.7017375,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/13968260",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7926788,\n   "lon": 37.7048372,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14033236",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Новосущёвский\\"",\n   "office_type": null,\n   "lat": 55.7920477,\n   "lon": 37.6034483,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14056468",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7910718,\n   "lon": 37.5745081,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14056470",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7908377,\n   "lon": 37.5744606,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14059101",\n   "kind": "building",\n   "name": "Технодинамика",\n   "office_type": "company",\n   "lat": 55.7365336,\n   "lon": 37.6362783,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14062415",\n   "kind": "organisation",\n   "name": "Желдорэкспедиция",\n   "office_type": "logistics",\n   "lat": 55.8049837,\n   "lon": 37.6554091,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14097875",\n   "kind": "organisation",\n   "name": "Российский государственный военно-исторический архив",\n   "office_type": "government",\n   "lat": 55.767836,\n   "lon": 37.6858889,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14110459",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7763868,\n   "lon": 37.6682924,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14124599",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7442156,\n   "lon": 37.6369167,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14124600",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7443612,\n   "lon": 37.6367928,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14124601",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7442456,\n   "lon": 37.6362725,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14124602",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7441646,\n   "lon": 37.6370409,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14134783",\n   "kind": "organisation",\n   "name": "Аппарат атташе по вопросам обороны Вьетнама",\n   "office_type": "diplomatic",\n   "lat": 55.7724822,\n   "lon": 37.7096933,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14141119",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.790401,\n   "lon": 37.5766385,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14141120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7909436,\n   "lon": 37.5773607,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14141121",\n   "kind": "building",\n   "name": "Бизнес-центр \\"На Масловке\\"",\n   "office_type": null,\n   "lat": 55.7915049,\n   "lon": 37.5811348,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14141122",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7911308,\n   "lon": 37.5806056,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14141123",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7910583,\n   "lon": 37.5795626,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14141124",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7909657,\n   "lon": 37.5782655,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14153409",\n   "kind": "building",\n   "name": "Бизнес-центр \\"На Спартаковской\\"",\n   "office_type": null,\n   "lat": 55.7757803,\n   "lon": 37.6776468,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14178075",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7947948,\n   "lon": 37.5283067,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14209002",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7839381,\n   "lon": 37.6429759,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14212251",\n   "kind": "organisation",\n   "name": "посольство Португалии",\n   "office_type": "diplomatic",\n   "lat": 55.7774533,\n   "lon": 37.6373672,\n   "opening_hours": "Mo-Fr 10:15-13:00"\n  },\n  {\n   "id": "relation/14212253",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7783417,\n   "lon": 37.6378126,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14213232",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7777188,\n   "lon": 37.6390995,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14214913",\n   "kind": "organisation",\n   "name": "посольство Ирландии",\n   "office_type": "diplomatic",\n   "lat": 55.7773438,\n   "lon": 37.6383334,\n   "opening_hours": "Mo-Fr 09:30-13:00, 14:30-17:30"\n  },\n  {\n   "id": "relation/14239439",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Красный Богатырь\\"",\n   "office_type": null,\n   "lat": 55.8146471,\n   "lon": 37.6937054,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14255022",\n   "kind": "organisation",\n   "name": "Центр специального назначения в области обеспечения безопасности дорожного движения, МВД РФ",\n   "office_type": "government",\n   "lat": 55.7329437,\n   "lon": 37.520227,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14271463",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7831371,\n   "lon": 37.6511366,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14271726",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7834249,\n   "lon": 37.6513418,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14271754",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7784133,\n   "lon": 37.6588725,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14274930",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7661107,\n   "lon": 37.6614163,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14274936",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7663432,\n   "lon": 37.660965,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14274937",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7666655,\n   "lon": 37.6608693,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14274938",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7670387,\n   "lon": 37.6606751,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14277629",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7693031,\n   "lon": 37.6623827,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14277905",\n   "kind": "building",\n   "name": "Дом Румянцева",\n   "office_type": null,\n   "lat": 55.7525033,\n   "lon": 37.6434167,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14277909",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7672304,\n   "lon": 37.679155,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14310470",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7520045,\n   "lon": 37.6414654,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14310471",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7521115,\n   "lon": 37.6414979,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14310472",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7522711,\n   "lon": 37.6416144,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14310473",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7521188,\n   "lon": 37.6417462,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14310474",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7520987,\n   "lon": 37.6418563,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14325108",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7441487,\n   "lon": 37.6479108,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14325115",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7437268,\n   "lon": 37.6530724,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14348593",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6724281,\n   "lon": 37.6314701,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14348594",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6720047,\n   "lon": 37.6316334,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14352894",\n   "kind": "building",\n   "name": "Представительство Орловской области при Правительстве РФ",\n   "office_type": "government",\n   "lat": 55.7456835,\n   "lon": 37.6457854,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14352899",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449372,\n   "lon": 37.6458525,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14356204",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7893457,\n   "lon": 37.6667559,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14374916",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7358412,\n   "lon": 37.5992258,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14580210",\n   "kind": "building",\n   "name": "Бизнес.Техноград",\n   "office_type": null,\n   "lat": 55.8395111,\n   "lon": 37.6215434,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14654134",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7630726,\n   "lon": 37.6610122,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14679718",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6605393,\n   "lon": 37.5370669,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14679719",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6612013,\n   "lon": 37.5378744,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14683956",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7560161,\n   "lon": 37.6668647,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14694503",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6601161,\n   "lon": 37.5383386,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14714875",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7616816,\n   "lon": 37.6406219,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14714877",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7625908,\n   "lon": 37.6417024,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14714878",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626076,\n   "lon": 37.6414288,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14714880",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7625355,\n   "lon": 37.6411994,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14714889",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619454,\n   "lon": 37.6411822,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14760679",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7877171,\n   "lon": 37.6820202,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14763978",\n   "kind": "organisation",\n   "name": "Ростех",\n   "office_type": "company",\n   "lat": 55.7278346,\n   "lon": 37.5684985,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14923956",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7777111,\n   "lon": 37.6696551,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14924020",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7779361,\n   "lon": 37.6725909,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14930869",\n   "kind": "organisation",\n   "name": "Центр занятости населения",\n   "office_type": "government",\n   "lat": 55.7909274,\n   "lon": 37.710207,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14930872",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7909001,\n   "lon": 37.7103054,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14930873",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.79051,\n   "lon": 37.7100693,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14946967",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7789692,\n   "lon": 37.6682235,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14946968",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7789301,\n   "lon": 37.6678031,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14946969",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7790818,\n   "lon": 37.6679997,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14956592",\n   "kind": "organisation",\n   "name": "Теплоэлектропроект",\n   "office_type": "architect",\n   "lat": 55.7697043,\n   "lon": 37.6723345,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14966578",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7925762,\n   "lon": 37.6772029,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14976625",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7960534,\n   "lon": 37.5869406,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14977215",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8024376,\n   "lon": 37.4923601,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14982711",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7615706,\n   "lon": 37.6344322,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14987149",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.767158,\n   "lon": 37.680048,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14987150",\n   "kind": "building",\n   "name": "ГЛОНАСС",\n   "office_type": "engineer",\n   "lat": 55.7678692,\n   "lon": 37.6799498,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14989887",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7847723,\n   "lon": 37.5575348,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14989888",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7861356,\n   "lon": 37.5578813,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/14990073",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7842301,\n   "lon": 37.5566376,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15014411",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7899614,\n   "lon": 37.5701469,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15015905",\n   "kind": "organisation",\n   "name": "ЦУС АО \\"ОЭК\\" Объединенная Энергетическая Компания Центр управления сетями.",\n   "office_type": "engineer",\n   "lat": 55.7967584,\n   "lon": 37.7444967,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15019422",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7768692,\n   "lon": 37.6774848,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15025366",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.777638,\n   "lon": 37.6638333,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15025368",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7783511,\n   "lon": 37.6669829,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15043796",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7833231,\n   "lon": 37.7147317,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15043807",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7817474,\n   "lon": 37.7134431,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15043808",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.781683,\n   "lon": 37.7128813,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15043809",\n   "kind": "organisation",\n   "name": "ОАО ВНИИ инструмент",\n   "office_type": "research",\n   "lat": 55.7814185,\n   "lon": 37.7153796,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15045974",\n   "kind": "organisation",\n   "name": "Villagio Estate",\n   "office_type": "estate_agent",\n   "lat": 55.786999,\n   "lon": 37.6210512,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "relation/15046087",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7866328,\n   "lon": 37.607796,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15046094",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7868352,\n   "lon": 37.6072278,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15046096",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7862389,\n   "lon": 37.6068839,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15047543",\n   "kind": "organisation",\n   "name": "Главное управление экономической безопасности и противодействия коррупции",\n   "office_type": "government",\n   "lat": 55.771046,\n   "lon": 37.656767,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15047549",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7710491,\n   "lon": 37.6566182,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15047550",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7709739,\n   "lon": 37.6555673,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15047552",\n   "kind": "building",\n   "name": "Главный вычислительный центр РЖД",\n   "office_type": null,\n   "lat": 55.7701616,\n   "lon": 37.6523896,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15047559",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Красноворотский\\"",\n   "office_type": null,\n   "lat": 55.7703703,\n   "lon": 37.6542707,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15052650",\n   "kind": "organisation",\n   "name": "Администрация Измайловского лесопарка",\n   "office_type": "government",\n   "lat": 55.787577,\n   "lon": 37.7895712,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15053170",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7669294,\n   "lon": 37.6444093,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15053172",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7670495,\n   "lon": 37.6443027,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15053603",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8153098,\n   "lon": 37.5125813,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15053605",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8153856,\n   "lon": 37.5147672,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15053608",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8155211,\n   "lon": 37.5182686,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15053610",\n   "kind": "organisation",\n   "name": "Проектно-конструкторское бюро по инфраструктуре",\n   "office_type": "engineer",\n   "lat": 55.8151623,\n   "lon": 37.5109784,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15057044",\n   "kind": "organisation",\n   "name": "Управа района Сокольники",\n   "office_type": "government",\n   "lat": 55.7908697,\n   "lon": 37.6833309,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15068806",\n   "kind": "organisation",\n   "name": "ФКУЗ «Противочумный центр»",\n   "office_type": "government",\n   "lat": 55.8610001,\n   "lon": 37.6209796,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15080641",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7910215,\n   "lon": 37.7056374,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15080647",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7914956,\n   "lon": 37.7069889,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15080649",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7914277,\n   "lon": 37.7055357,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15088763",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7284905,\n   "lon": 37.6386339,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15097714",\n   "kind": "building",\n   "name": "ДС станции Черкизово",\n   "office_type": null,\n   "lat": 55.8062061,\n   "lon": 37.7456366,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15257667",\n   "kind": "organisation",\n   "name": "ОБП МОВО по ВАО ФГКУ УВО ВНГ России по городу Москве",\n   "office_type": "security",\n   "lat": 55.7907436,\n   "lon": 37.7828469,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15268234",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7580961,\n   "lon": 37.7398282,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15268237",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7587034,\n   "lon": 37.7424951,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15268248",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7572519,\n   "lon": 37.7409026,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15270544",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7699276,\n   "lon": 37.6576927,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15315419",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7706005,\n   "lon": 37.6535858,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15315420",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7698705,\n   "lon": 37.6543938,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15321216",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.770687,\n   "lon": 37.6607249,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15321224",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7707902,\n   "lon": 37.6621787,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15321281",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7709026,\n   "lon": 37.6597917,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15321323",\n   "kind": "organisation",\n   "name": "Центральный аппарат ЛДПР",\n   "office_type": "political_party",\n   "lat": 55.7707811,\n   "lon": 37.662853,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "relation/15321324",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7701404,\n   "lon": 37.6577433,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15321540",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7704268,\n   "lon": 37.6596643,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15322772",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7754038,\n   "lon": 37.6700234,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15322773",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7752949,\n   "lon": 37.6699541,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15322775",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7752846,\n   "lon": 37.6710428,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15322795",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7739052,\n   "lon": 37.6714545,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15322817",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7720547,\n   "lon": 37.6724844,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15322818",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7721099,\n   "lon": 37.6722489,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15322820",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7719017,\n   "lon": 37.6719411,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15322821",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7720649,\n   "lon": 37.6719681,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15326525",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7741309,\n   "lon": 37.6637399,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15327986",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7787938,\n   "lon": 37.6690612,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15342436",\n   "kind": "building",\n   "name": "Информационно-вычислительный центр МЖД",\n   "office_type": "telecommunication",\n   "lat": 55.7774042,\n   "lon": 37.6629613,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15353116",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7649921,\n   "lon": 37.6211229,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15353127",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623962,\n   "lon": 37.6239221,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15353129",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7628956,\n   "lon": 37.6240162,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15353130",\n   "kind": "building",\n   "name": "Деловой центр \\"Кузнецкий мост 17\\"",\n   "office_type": null,\n   "lat": 55.7624678,\n   "lon": 37.6247943,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15353135",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7627434,\n   "lon": 37.6257964,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15360853",\n   "kind": "organisation",\n   "name": "Дизайн-клуб",\n   "office_type": "union",\n   "lat": 55.7649746,\n   "lon": 37.6253817,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15360854",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7656484,\n   "lon": 37.6257969,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15392529",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.784869,\n   "lon": 37.7092854,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15392530",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7849075,\n   "lon": 37.7096717,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15392541",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7848554,\n   "lon": 37.7035111,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15392545",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7847761,\n   "lon": 37.7074518,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15392546",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7847294,\n   "lon": 37.7046546,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15392553",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.784621,\n   "lon": 37.7051286,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15393022",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7661457,\n   "lon": 37.6283382,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15394670",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7810024,\n   "lon": 37.7072837,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15430478",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7905249,\n   "lon": 37.6601734,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15430479",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7905761,\n   "lon": 37.6594277,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15449241",\n   "kind": "building",\n   "name": "Консульский отдел посольства Ирана",\n   "office_type": "diplomatic",\n   "lat": 55.7556532,\n   "lon": 37.6490741,\n   "opening_hours": "Mo-Fr 09:00-12:00 open \\"приём документов\\", Mo-Fr 12:00-13:00 open \\"выдача документов\\""\n  },\n  {\n   "id": "relation/15451609",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7678552,\n   "lon": 37.6746313,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15537987",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7861429,\n   "lon": 37.6748391,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15537988",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7859916,\n   "lon": 37.6744911,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15537989",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7866898,\n   "lon": 37.6734773,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15540951",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7632806,\n   "lon": 37.6097917,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15597420",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.770455,\n   "lon": 37.6021878,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15608048",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7575662,\n   "lon": 37.6659253,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15620035",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7702896,\n   "lon": 37.6033934,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15621765",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7665107,\n   "lon": 37.6394028,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15661174",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8029717,\n   "lon": 37.5725573,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15671460",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7639705,\n   "lon": 37.6351574,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15671462",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7635149,\n   "lon": 37.635376,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15671992",\n   "kind": "organisation",\n   "name": "Московская городская нотариальная палата",\n   "office_type": "notary",\n   "lat": 55.7647719,\n   "lon": 37.6341852,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15671998",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7646389,\n   "lon": 37.6330275,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15672001",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7634903,\n   "lon": 37.6315984,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15673978",\n   "kind": "organisation",\n   "name": "Посольство Ирана",\n   "office_type": "diplomatic",\n   "lat": 55.755398,\n   "lon": 37.64962,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15673979",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7553116,\n   "lon": 37.6489818,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15673980",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7549702,\n   "lon": 37.64895,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15673988",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7549607,\n   "lon": 37.6494147,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15673989",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7556513,\n   "lon": 37.6494764,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15692038",\n   "kind": "organisation",\n   "name": "Социальный фонд России",\n   "office_type": "government",\n   "lat": 55.8518169,\n   "lon": 37.6705199,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15692039",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8519299,\n   "lon": 37.6702433,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15692040",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8518169,\n   "lon": 37.6708546,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15697832",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7440472,\n   "lon": 37.6466097,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15697835",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7438346,\n   "lon": 37.6456191,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15700998",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7431348,\n   "lon": 37.6484323,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15700999",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7429614,\n   "lon": 37.6486548,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15701004",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7429709,\n   "lon": 37.6480735,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15701005",\n   "kind": "organisation",\n   "name": "Ростелеком",\n   "office_type": "company",\n   "lat": 55.7424429,\n   "lon": 37.6480143,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15701006",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.742549,\n   "lon": 37.6486675,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15701007",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7427054,\n   "lon": 37.6487391,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15701008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.742586,\n   "lon": 37.6480971,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15701009",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7424231,\n   "lon": 37.647395,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15701013",\n   "kind": "organisation",\n   "name": "Представительство Республики Татарстан",\n   "office_type": "diplomatic",\n   "lat": 55.742019,\n   "lon": 37.646212,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15705147",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447514,\n   "lon": 37.6436817,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15716189",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7383191,\n   "lon": 37.6264879,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15716190",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7380291,\n   "lon": 37.6265702,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15737498",\n   "kind": "building",\n   "name": "Счётная палата Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7381779,\n   "lon": 37.5836607,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15791431",\n   "kind": "building",\n   "name": "ПАО «Новатэк»",\n   "office_type": "company",\n   "lat": 55.672827,\n   "lon": 37.520597,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15842825",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7426846,\n   "lon": 37.5975472,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15844900",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819693,\n   "lon": 37.5944225,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/15867259",\n   "kind": "building",\n   "name": "Neva Towers (Башня 1)",\n   "office_type": null,\n   "lat": 55.7511029,\n   "lon": 37.5337174,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16109093",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7943729,\n   "lon": 37.7079914,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16109097",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7932372,\n   "lon": 37.7072477,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16109098",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7932263,\n   "lon": 37.7078502,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16109099",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.793489,\n   "lon": 37.7075049,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16109101",\n   "kind": "organisation",\n   "name": "Институт геотехники и инженерных изысканий в строительстве",\n   "office_type": "research",\n   "lat": 55.7939728,\n   "lon": 37.7093686,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16138996",\n   "kind": "organisation",\n   "name": "Посольство Армении",\n   "office_type": "diplomatic",\n   "lat": 55.7604165,\n   "lon": 37.6347038,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16173223",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7593614,\n   "lon": 37.6310996,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16190693",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7569943,\n   "lon": 37.7387169,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16238171",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7803538,\n   "lon": 37.5872572,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16254272",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7735131,\n   "lon": 37.6070879,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16284531",\n   "kind": "organisation",\n   "name": "\\"Мосгаз\\" Управление ГВСД и ГРС",\n   "office_type": "engineer",\n   "lat": 55.7611418,\n   "lon": 37.6647103,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16330700",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7426853,\n   "lon": 37.5948032,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16330702",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7426488,\n   "lon": 37.5952345,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16349266",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7400138,\n   "lon": 37.5926647,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16374674",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7727402,\n   "lon": 37.6379187,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16374675",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7725389,\n   "lon": 37.637864,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16374695",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7730752,\n   "lon": 37.6329481,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16374696",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7731791,\n   "lon": 37.6330932,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16374697",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7732199,\n   "lon": 37.633263,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16374699",\n   "kind": "building",\n   "name": "Meliora place",\n   "office_type": null,\n   "lat": 55.7734352,\n   "lon": 37.6329722,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16374772",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7735742,\n   "lon": 37.6318578,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16374773",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7731687,\n   "lon": 37.6317423,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16374885",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Вишневый Сад\\"",\n   "office_type": null,\n   "lat": 55.7724798,\n   "lon": 37.6306998,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16380393",\n   "kind": "organisation",\n   "name": "Федеральная таможенная служба РФ",\n   "office_type": "government",\n   "lat": 55.7742033,\n   "lon": 37.6234383,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16380400",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7744278,\n   "lon": 37.6270742,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16380406",\n   "kind": "building",\n   "name": "Дом 15",\n   "office_type": null,\n   "lat": 55.7743021,\n   "lon": 37.626033,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16386205",\n   "kind": "organisation",\n   "name": "Министерство сельского хозяйства РФ",\n   "office_type": "government",\n   "lat": 55.7705175,\n   "lon": 37.64581,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16386285",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7724536,\n   "lon": 37.6391452,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16421395",\n   "kind": "organisation",\n   "name": "Центр услуг связи МГТС Люблинский 4",\n   "office_type": "telecommunication",\n   "lat": 55.7936161,\n   "lon": 37.7092187,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "relation/16421396",\n   "kind": "organisation",\n   "name": "Восточное окружное управление образования",\n   "office_type": "government",\n   "lat": 55.7937117,\n   "lon": 37.7105626,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16421397",\n   "kind": "building",\n   "name": "Восточное окружное управление образования",\n   "office_type": "government",\n   "lat": 55.7937441,\n   "lon": 37.7104715,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16448845",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7809488,\n   "lon": 37.5809664,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16479645",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7653021,\n   "lon": 37.612925,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16545914",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7260052,\n   "lon": 37.6884214,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16545932",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7281424,\n   "lon": 37.6872827,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16545933",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Калитники\\"",\n   "office_type": null,\n   "lat": 55.7277603,\n   "lon": 37.6876802,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16561994",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7338665,\n   "lon": 37.6132781,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16561998",\n   "kind": "organisation",\n   "name": "посольство Франции",\n   "office_type": "diplomatic",\n   "lat": 55.7324541,\n   "lon": 37.6136626,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16564531",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7699805,\n   "lon": 37.6046079,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16622535",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7872439,\n   "lon": 37.632363,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16622536",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7879042,\n   "lon": 37.6315981,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16672013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7953136,\n   "lon": 37.7271505,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16674147",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8122017,\n   "lon": 37.5936492,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16682979",\n   "kind": "building",\n   "name": "Дом Г. А. Тарасова",\n   "office_type": null,\n   "lat": 55.7623875,\n   "lon": 37.5909325,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16685820",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623816,\n   "lon": 37.6004394,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16696511",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7592116,\n   "lon": 37.5926143,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16696512",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591669,\n   "lon": 37.5929663,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16696517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7599965,\n   "lon": 37.5868384,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16753201",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7718845,\n   "lon": 37.5752483,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16753205",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7722678,\n   "lon": 37.5757904,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16753206",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7727755,\n   "lon": 37.5750348,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16766933",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7707066,\n   "lon": 37.6393532,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16780432",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7858509,\n   "lon": 37.7085269,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16788910",\n   "kind": "organisation",\n   "name": "Бизнес центр Инкапстройинвест",\n   "office_type": "company",\n   "lat": 55.7619975,\n   "lon": 37.6068708,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16869122",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7705337,\n   "lon": 37.5754741,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16893857",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7707711,\n   "lon": 37.6600659,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16930258",\n   "kind": "organisation",\n   "name": "Туполев",\n   "office_type": "engineer",\n   "lat": 55.7608159,\n   "lon": 37.6772981,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16930262",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626634,\n   "lon": 37.6795764,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16930275",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7621681,\n   "lon": 37.6795857,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16930276",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619459,\n   "lon": 37.6795036,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16930278",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7611699,\n   "lon": 37.6766558,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16930282",\n   "kind": "organisation",\n   "name": "РЖДстрой",\n   "office_type": "company",\n   "lat": 55.7607344,\n   "lon": 37.6720107,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/16930283",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт физической культуры и спорта",\n   "office_type": "research",\n   "lat": 55.7615112,\n   "lon": 37.6721705,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17001357",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7478435,\n   "lon": 37.598207,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17001358",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7476423,\n   "lon": 37.5980949,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17024308",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7828112,\n   "lon": 37.6001709,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17024309",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7832944,\n   "lon": 37.600597,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17051423",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7805568,\n   "lon": 37.6409943,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17051424",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.78056,\n   "lon": 37.6414317,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17051425",\n   "kind": "building",\n   "name": "Постоянное представительство Чеченской Республики при президенте РФ",\n   "office_type": "government",\n   "lat": 55.7799808,\n   "lon": 37.641024,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "relation/17051426",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7809546,\n   "lon": 37.6388842,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17064750",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7179624,\n   "lon": 37.6195959,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17064763",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.71884,\n   "lon": 37.6188405,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17088305",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт рыбного хозяйства и океанографии",\n   "office_type": "research",\n   "lat": 55.7825176,\n   "lon": 37.6639209,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17088308",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.782458,\n   "lon": 37.6639451,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17096996",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.76388,\n   "lon": 37.6259396,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17096998",\n   "kind": "organisation",\n   "name": "Институт востоковедения РАН",\n   "office_type": "research",\n   "lat": 55.7640564,\n   "lon": 37.6240029,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17173562",\n   "kind": "organisation",\n   "name": "ООО Палмира Электроникс",\n   "office_type": "it",\n   "lat": 55.7150556,\n   "lon": 37.6927529,\n   "opening_hours": "Mo-Fr 10:00-18:30"\n  },\n  {\n   "id": "relation/17173565",\n   "kind": "organisation",\n   "name": "ВНИИ Химических Средств и Защиты Растений",\n   "office_type": "research",\n   "lat": 55.716677,\n   "lon": 37.6951464,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17194863",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "newspaper",\n   "lat": 55.7883392,\n   "lon": 37.5800246,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17229963",\n   "kind": "organisation",\n   "name": "Научно-исследовательский центр МГМУ им. И. М. Сеченова",\n   "office_type": "research",\n   "lat": 55.7283475,\n   "lon": 37.5738536,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17305839",\n   "kind": "organisation",\n   "name": "ВТБ Страхование",\n   "office_type": "insurance",\n   "lat": 55.7627283,\n   "lon": 37.640849,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "relation/17305840",\n   "kind": "organisation",\n   "name": "МосТрансПроект",\n   "office_type": "research",\n   "lat": 55.7622291,\n   "lon": 37.6404869,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17305856",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7647714,\n   "lon": 37.6374373,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17305858",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7642945,\n   "lon": 37.6378713,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17305860",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7639367,\n   "lon": 37.6385565,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17305925",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Селезнёвская, 11\\"",\n   "office_type": null,\n   "lat": 55.7812388,\n   "lon": 37.603825,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17309820",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.774235,\n   "lon": 37.5776759,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17329527",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7494744,\n   "lon": 37.7108662,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17345231",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7330127,\n   "lon": 37.6190714,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17361996",\n   "kind": "organisation",\n   "name": "ФИЦ питания, биотехнологии и безопасности пищи",\n   "office_type": "research",\n   "lat": 55.7497183,\n   "lon": 37.6403403,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17362005",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7516204,\n   "lon": 37.6366033,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17362006",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7513798,\n   "lon": 37.6353091,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17362008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.751908,\n   "lon": 37.6346793,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17362012",\n   "kind": "organisation",\n   "name": "Федеральная служба по надзору в сфере связи, информационных технологий и массовых коммуникаций",\n   "office_type": "government",\n   "lat": 55.7515578,\n   "lon": 37.6366519,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17386576",\n   "kind": "organisation",\n   "name": "Федеральная таможенная служба",\n   "office_type": "government",\n   "lat": 55.7495526,\n   "lon": 37.5079009,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17404365",\n   "kind": "building",\n   "name": "1",\n   "office_type": null,\n   "lat": 55.7653634,\n   "lon": 37.6272106,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17404367",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7648114,\n   "lon": 37.6286667,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17443824",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7500558,\n   "lon": 37.6443513,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17443829",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7515923,\n   "lon": 37.6485582,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17443832",\n   "kind": "building",\n   "name": "Посольство Перу",\n   "office_type": "diplomatic",\n   "lat": 55.7524021,\n   "lon": 37.6539087,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "relation/17443838",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7516824,\n   "lon": 37.6539149,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17458347",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7667375,\n   "lon": 37.6352728,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17459376",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7338354,\n   "lon": 37.6200507,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17474925",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8470478,\n   "lon": 37.5927535,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17474927",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8470967,\n   "lon": 37.5916694,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17479607",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.744084,\n   "lon": 37.6473657,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17573183",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.776887,\n   "lon": 37.693612,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17573185",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7772067,\n   "lon": 37.694317,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17611020",\n   "kind": "organisation",\n   "name": "Дворец Бракосочетания ВДНХ",\n   "office_type": "government",\n   "lat": 55.8357168,\n   "lon": 37.6266262,\n   "opening_hours": "Tu-Sa 09:00-14:00,15:00-18:00"\n  },\n  {\n   "id": "relation/17647229",\n   "kind": "organisation",\n   "name": "Жилищник района Якиманка",\n   "office_type": "property_management",\n   "lat": 55.7294681,\n   "lon": 37.6073724,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17682463",\n   "kind": "organisation",\n   "name": "31 Государственный проектный институт специального строительства",\n   "office_type": "engineering",\n   "lat": 55.7431605,\n   "lon": 37.5830931,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17682465",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7428844,\n   "lon": 37.5824175,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17698111",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.845439,\n   "lon": 37.6337075,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17704247",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7852045,\n   "lon": 37.6747874,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17704846",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8188879,\n   "lon": 37.6440516,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17723011",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7927076,\n   "lon": 37.6519178,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17723012",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7928828,\n   "lon": 37.6511821,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17733417",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619224,\n   "lon": 37.5578204,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17734525",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8086507,\n   "lon": 37.5072375,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17734526",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8083721,\n   "lon": 37.507469,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17734527",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8079898,\n   "lon": 37.5080005,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17734537",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.807405,\n   "lon": 37.5100183,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17802943",\n   "kind": "building",\n   "name": "Лабораторный корпус",\n   "office_type": "research",\n   "lat": 55.8442746,\n   "lon": 37.5899193,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17873319",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6333694,\n   "lon": 37.4536872,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17887534",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7585544,\n   "lon": 37.6781093,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17940120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7780348,\n   "lon": 37.629767,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17950695",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8271704,\n   "lon": 37.6595138,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17950696",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8266709,\n   "lon": 37.660674,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17950697",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8270334,\n   "lon": 37.6609951,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/17991370",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7381343,\n   "lon": 37.6038231,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18005706",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8138365,\n   "lon": 37.6043787,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18005730",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8133244,\n   "lon": 37.603286,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18008364",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8137646,\n   "lon": 37.6028102,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18009739",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8128213,\n   "lon": 37.6044149,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18086077",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7730069,\n   "lon": 37.6407392,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18086078",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7734228,\n   "lon": 37.6404823,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18086079",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7733663,\n   "lon": 37.639773,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18086080",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7729298,\n   "lon": 37.6394291,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18096258",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7414585,\n   "lon": 37.6025506,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18096259",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7419012,\n   "lon": 37.6030617,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18116992",\n   "kind": "building",\n   "name": "Бизнес-центр «Союз деловых людей»",\n   "office_type": null,\n   "lat": 55.8086449,\n   "lon": 37.7719764,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18134838",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7206262,\n   "lon": 37.6134981,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18134840",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7192787,\n   "lon": 37.6131496,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18147637",\n   "kind": "building",\n   "name": "Морозовская Детская Городская Клиническая Больница",\n   "office_type": null,\n   "lat": 55.7410761,\n   "lon": 37.602074,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18148143",\n   "kind": "organisation",\n   "name": "Федеральный центр гигиены и эпидемиологии",\n   "office_type": "government",\n   "lat": 55.6956693,\n   "lon": 37.6233177,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18148153",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6952608,\n   "lon": 37.6253636,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18155769",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6994191,\n   "lon": 37.6244857,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18155771",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6991136,\n   "lon": 37.6235329,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18176576",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7182617,\n   "lon": 37.6130576,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18178315",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7188166,\n   "lon": 37.6051753,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18196457",\n   "kind": "building",\n   "name": "Институт космических исследований Российской академии наук",\n   "office_type": "research",\n   "lat": 55.6551157,\n   "lon": 37.5335788,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18245710",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7562798,\n   "lon": 37.6254979,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18267726",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7832016,\n   "lon": 37.5979705,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18274494",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.781207,\n   "lon": 37.6057773,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18274495",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7810271,\n   "lon": 37.6055048,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18274496",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7812468,\n   "lon": 37.6050467,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18293383",\n   "kind": "building",\n   "name": "Тверская застава",\n   "office_type": null,\n   "lat": 55.7753771,\n   "lon": 37.5819023,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18302580",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.757172,\n   "lon": 37.7562444,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18328881",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7924837,\n   "lon": 37.6516414,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18347473",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8034786,\n   "lon": 37.6184023,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18347475",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8035393,\n   "lon": 37.6184609,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18347484",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.803636,\n   "lon": 37.6185967,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "relation/18347485",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8036458,\n   "lon": 37.618531,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "relation/18352287",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6902189,\n   "lon": 37.6253141,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18352774",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7404293,\n   "lon": 37.5320567,\n   "opening_hours": null\n  },\n  {\n   "id": "relation/18372355",\n   "kind": "organisation",\n   "name": "IQ Парк",\n   "office_type": "company",\n   "lat": 55.7114904,\n   "lon": 37.6835119,\n   "opening_hours": null\n  },\n  {\n   "id": "way/23372126",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.727645,\n   "lon": 37.6620439,\n   "opening_hours": null\n  },\n  {\n   "id": "way/23410658",\n   "kind": "organisation",\n   "name": "Царицынский ЗАГС",\n   "office_type": "government",\n   "lat": 55.636125,\n   "lon": 37.671452,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "way/23414592",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7288897,\n   "lon": 37.6634045,\n   "opening_hours": null\n  },\n  {\n   "id": "way/23768262",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7959656,\n   "lon": 37.800646,\n   "opening_hours": null\n  },\n  {\n   "id": "way/23769066",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7902921,\n   "lon": 37.8100397,\n   "opening_hours": null\n  },\n  {\n   "id": "way/23769784",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6738234,\n   "lon": 37.5052334,\n   "opening_hours": null\n  },\n  {\n   "id": "way/24913831",\n   "kind": "organisation",\n   "name": "Альтоника",\n   "office_type": "company",\n   "lat": 55.6607372,\n   "lon": 37.6049987,\n   "opening_hours": null\n  },\n  {\n   "id": "way/25418521",\n   "kind": "organisation",\n   "name": "Центральный государственный архив Московской области",\n   "office_type": "government",\n   "lat": 55.6554879,\n   "lon": 37.598751,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "way/26232926",\n   "kind": "organisation",\n   "name": "Госкорпорация по ОрВД",\n   "office_type": "government",\n   "lat": 55.7924832,\n   "lon": 37.5397999,\n   "opening_hours": null\n  },\n  {\n   "id": "way/26232931",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.792566,\n   "lon": 37.5416265,\n   "opening_hours": null\n  },\n  {\n   "id": "way/26233007",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7955059,\n   "lon": 37.5485555,\n   "opening_hours": null\n  },\n  {\n   "id": "way/26233780",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8631338,\n   "lon": 37.6095114,\n   "opening_hours": null\n  },\n  {\n   "id": "way/26622966",\n   "kind": "building",\n   "name": "Башня «Запад»",\n   "office_type": null,\n   "lat": 55.7494535,\n   "lon": 37.5370726,\n   "opening_hours": null\n  },\n  {\n   "id": "way/26756776",\n   "kind": "building",\n   "name": "Единый диспетчерский центр Московского метрополитена",\n   "office_type": null,\n   "lat": 55.8650604,\n   "lon": 37.5494391,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27254201",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7575203,\n   "lon": 37.6101223,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27528574",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7861245,\n   "lon": 37.4797797,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27528613",\n   "kind": "organisation",\n   "name": "Управа района Хорошёво-Мнёвники",\n   "office_type": "government",\n   "lat": 55.7863368,\n   "lon": 37.4804932,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27528615",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7859483,\n   "lon": 37.480914,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27528622",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7855991,\n   "lon": 37.4844514,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27555530",\n   "kind": "organisation",\n   "name": "Центр разрешительной документации «Документик»",\n   "office_type": "company",\n   "lat": 55.731227,\n   "lon": 37.6693585,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27568333",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7229271,\n   "lon": 37.4688147,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27568572",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7259785,\n   "lon": 37.4700483,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27651410",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6973337,\n   "lon": 37.5025328,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27715421",\n   "kind": "organisation",\n   "name": "Роспотребнадзор СЗАО",\n   "office_type": "government",\n   "lat": 55.8091175,\n   "lon": 37.4766465,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27716733",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8236946,\n   "lon": 37.570116,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27728746",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Тушинского района СЗАО",\n   "office_type": "government",\n   "lat": 55.8011552,\n   "lon": 37.4810367,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27742960",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8442635,\n   "lon": 37.5739237,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27742962",\n   "kind": "building",\n   "name": "ОМВД России по Тимирязевскому району г. Москвы",\n   "office_type": null,\n   "lat": 55.8436726,\n   "lon": 37.5737601,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27744812",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7966725,\n   "lon": 37.5388103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27752791",\n   "kind": "building",\n   "name": "Бизнес-центр \\"РТС\\"",\n   "office_type": null,\n   "lat": 55.7331602,\n   "lon": 37.6974275,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27753681",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7859735,\n   "lon": 37.4854146,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27753682",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7861319,\n   "lon": 37.486139,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27753925",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7894326,\n   "lon": 37.4890809,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27754329",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7920965,\n   "lon": 37.4866187,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27768038",\n   "kind": "building",\n   "name": "Ростэк",\n   "office_type": "company",\n   "lat": 55.8147915,\n   "lon": 37.7104905,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27793261",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7102244,\n   "lon": 37.4769011,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27793289",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7094343,\n   "lon": 37.4795071,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27800666",\n   "kind": "organisation",\n   "name": "Пенсионный фонд",\n   "office_type": "government",\n   "lat": 55.8090744,\n   "lon": 37.4774702,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27811814",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6631945,\n   "lon": 37.7445752,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27830835",\n   "kind": "building",\n   "name": "",\n   "office_type": "research",\n   "lat": 55.8029675,\n   "lon": 37.4583106,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27830836",\n   "kind": "building",\n   "name": "",\n   "office_type": "research",\n   "lat": 55.8024905,\n   "lon": 37.4605595,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27873305",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8105572,\n   "lon": 37.6347922,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27873309",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8119227,\n   "lon": 37.6363014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27873310",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8119633,\n   "lon": 37.6356318,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27887048",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8027971,\n   "lon": 37.637016,\n   "opening_hours": null\n  },\n  {\n   "id": "way/27937526",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7889288,\n   "lon": 37.4890716,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28030422",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8031717,\n   "lon": 37.5607761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28030427",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8000532,\n   "lon": 37.5640494,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28033285",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7812988,\n   "lon": 37.4988576,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28033287",\n   "kind": "organisation",\n   "name": "ОАО \\"Гипросвязь\\"",\n   "office_type": "research",\n   "lat": 55.7817974,\n   "lon": 37.4998899,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28064652",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8035124,\n   "lon": 37.547737,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28114793",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7184042,\n   "lon": 37.447521,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28119648",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8374756,\n   "lon": 37.5234389,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28119660",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8383968,\n   "lon": 37.5294801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28127764",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8148568,\n   "lon": 37.5911939,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28127766",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8144241,\n   "lon": 37.5922869,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28127771",\n   "kind": "building",\n   "name": "РЭМ",\n   "office_type": "company",\n   "lat": 55.8158182,\n   "lon": 37.5950798,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28137603",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8665913,\n   "lon": 37.5507099,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28172855",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8757972,\n   "lon": 37.7142185,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28226874",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7624588,\n   "lon": 37.6577886,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28226925",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Путейский тупик, 6\\"",\n   "office_type": null,\n   "lat": 55.7612537,\n   "lon": 37.6597877,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28270701",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8556004,\n   "lon": 37.5061963,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28395179",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.672376,\n   "lon": 37.7540467,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28395205",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт по социальным и кадровым проблемам агропромышленного комплекса",\n   "office_type": "research",\n   "lat": 55.6719095,\n   "lon": 37.7577668,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28414180",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8977868,\n   "lon": 37.6199848,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28427304",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8871515,\n   "lon": 37.6577418,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28444110",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8776493,\n   "lon": 37.6848931,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28456113",\n   "kind": "building",\n   "name": "Главное управление МВД РФ по Центральному федеральному округу",\n   "office_type": "government",\n   "lat": 55.8119124,\n   "lon": 37.5905597,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28461854",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8641815,\n   "lon": 37.566632,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28475443",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.865118,\n   "lon": 37.5612119,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28529238",\n   "kind": "building",\n   "name": "Эдтроникторг",\n   "office_type": null,\n   "lat": 55.8083821,\n   "lon": 37.518531,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28568800",\n   "kind": "organisation",\n   "name": "Всероссийский электротехнический институт имени В. И. Ленина",\n   "office_type": "research",\n   "lat": 55.754594,\n   "lon": 37.7025569,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28587922",\n   "kind": "organisation",\n   "name": "Налоговая инспекция №43",\n   "office_type": "government",\n   "lat": 55.8508027,\n   "lon": 37.4935713,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28678508",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8847931,\n   "lon": 37.563018,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28682364",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7880888,\n   "lon": 37.5750733,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28682368",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.787526,\n   "lon": 37.5785635,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28684103",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7820574,\n   "lon": 37.5835353,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28684104",\n   "kind": "building",\n   "name": "ВНИИ Эталон",\n   "office_type": "research",\n   "lat": 55.7827312,\n   "lon": 37.5840529,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28684108",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7839475,\n   "lon": 37.5838,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28684112",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7851575,\n   "lon": 37.5832639,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28708496",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8684117,\n   "lon": 37.4983737,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28709589",\n   "kind": "organisation",\n   "name": "Российский государственный архив литературы и искусства",\n   "office_type": "government",\n   "lat": 55.8340556,\n   "lon": 37.4881211,\n   "opening_hours": "Mo-Th 09:00-17:00; Fr 09:00-16:00"\n  },\n  {\n   "id": "way/28733007",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7956589,\n   "lon": 37.5794436,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28733012",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7963011,\n   "lon": 37.5801631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28733054",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7958316,\n   "lon": 37.5803786,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28733057",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7957701,\n   "lon": 37.579837,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28768035",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8724413,\n   "lon": 37.5581801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28822242",\n   "kind": "organisation",\n   "name": "Физический институт им. П. Н. Лебедева",\n   "office_type": "research",\n   "lat": 55.697732,\n   "lon": 37.565508,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28828645",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8307927,\n   "lon": 37.6478852,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28828694",\n   "kind": "building",\n   "name": "ГУ \\"ИПГ\\"",\n   "office_type": null,\n   "lat": 55.8332608,\n   "lon": 37.665797,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28835617",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.852635,\n   "lon": 37.6484249,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28836552",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8497271,\n   "lon": 37.6699366,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28847435",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8064138,\n   "lon": 37.629692,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28847439",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8054686,\n   "lon": 37.6334013,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28847442",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8052506,\n   "lon": 37.6323224,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28852214",\n   "kind": "organisation",\n   "name": "Павильон 74. Управление по работе со странами СНГ и внешним связям",\n   "office_type": "company",\n   "lat": 55.8339423,\n   "lon": 37.6326428,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28856942",\n   "kind": "organisation",\n   "name": "Дирекция Измайловского парка",\n   "office_type": "government",\n   "lat": 55.7688103,\n   "lon": 37.7508255,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28860303",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8011326,\n   "lon": 37.7147356,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28870428",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8763748,\n   "lon": 37.5897688,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28875004",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7759505,\n   "lon": 37.510925,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28875029",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7770653,\n   "lon": 37.5196421,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28938762",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8293213,\n   "lon": 37.4490423,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28938812",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8325578,\n   "lon": 37.4464893,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28944065",\n   "kind": "organisation",\n   "name": "Головной центр гигиены и эпидемиологии ФМБА России",\n   "office_type": "research",\n   "lat": 55.8053767,\n   "lon": 37.481105,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28944203",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8032181,\n   "lon": 37.490963,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28950139",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8318433,\n   "lon": 37.4920419,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28950857",\n   "kind": "organisation",\n   "name": "Центральное экспертно криминалистическое таможенное управление",\n   "office_type": "government",\n   "lat": 55.8251045,\n   "lon": 37.5133228,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28965443",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7576813,\n   "lon": 37.808729,\n   "opening_hours": null\n  },\n  {\n   "id": "way/28999329",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6829107,\n   "lon": 37.4578729,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29036196",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7503535,\n   "lon": 37.7872486,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29070586",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7924583,\n   "lon": 37.6089369,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29071146",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7991576,\n   "lon": 37.6131259,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29080642",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8615932,\n   "lon": 37.5262513,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29080759",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8622792,\n   "lon": 37.5397364,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29128003",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6821735,\n   "lon": 37.5626102,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29128009",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6819355,\n   "lon": 37.56235,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29157156",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6486564,\n   "lon": 37.6140006,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29198273",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8560598,\n   "lon": 37.5588776,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29199425",\n   "kind": "organisation",\n   "name": "Госцентр «Природа»",\n   "office_type": "geodesist",\n   "lat": 55.7470089,\n   "lon": 37.8033053,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29243311",\n   "kind": "building",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7000407,\n   "lon": 37.4956983,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29265265",\n   "kind": "organisation",\n   "name": "Институт высшей нервной деятельности и нейрофизиологии РАН",\n   "office_type": "research",\n   "lat": 55.6541426,\n   "lon": 37.5276594,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29265318",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6438559,\n   "lon": 37.5124557,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29265363",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6452754,\n   "lon": 37.5297161,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29265371",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.648909,\n   "lon": 37.533305,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29265383",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6561579,\n   "lon": 37.5265752,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29276623",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8064504,\n   "lon": 37.5047598,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29292714",\n   "kind": "organisation",\n   "name": "Группа компаний Компьюлинк",\n   "office_type": "company",\n   "lat": 55.6877855,\n   "lon": 37.4827999,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29311565",\n   "kind": "organisation",\n   "name": "Мосэнергосбыт",\n   "office_type": "government",\n   "lat": 55.6078226,\n   "lon": 37.713668,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29313052",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8463102,\n   "lon": 37.5854625,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29325295",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6448913,\n   "lon": 37.599817,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29325346",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6519255,\n   "lon": 37.6046132,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29325348",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6551804,\n   "lon": 37.6014484,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29325350",\n   "kind": "building",\n   "name": "Управление по контролю за оборотом наркотиков Главного управления МВД России по г. Москве",\n   "office_type": "government",\n   "lat": 55.6544416,\n   "lon": 37.599171,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29355015",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Орехово-Борисово Северное",\n   "office_type": "government",\n   "lat": 55.5995944,\n   "lon": 37.7178007,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29396671",\n   "kind": "building",\n   "name": "Мосводоканал",\n   "office_type": "water_utility",\n   "lat": 55.6819755,\n   "lon": 37.7503904,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29400193",\n   "kind": "organisation",\n   "name": "Мосэнергосбыт \\"Кузьминки\\"",\n   "office_type": "company",\n   "lat": 55.7048702,\n   "lon": 37.7885376,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29430167",\n   "kind": "organisation",\n   "name": "Управа района Рязанский",\n   "office_type": "government",\n   "lat": 55.7158042,\n   "lon": 37.7847934,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29430491",\n   "kind": "organisation",\n   "name": "Налоговая инспекция № 21",\n   "office_type": "government",\n   "lat": 55.716277,\n   "lon": 37.7859407,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29431686",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7475054,\n   "lon": 37.8114215,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29446253",\n   "kind": "building",\n   "name": "Узел связи",\n   "office_type": null,\n   "lat": 55.7499768,\n   "lon": 37.7754904,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29446254",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7501547,\n   "lon": 37.7762066,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29447885",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7535925,\n   "lon": 37.8147491,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29450194",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6513968,\n   "lon": 37.7232408,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29544221",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7695378,\n   "lon": 37.7270665,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29544297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7671899,\n   "lon": 37.7324289,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29544371",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7608357,\n   "lon": 37.7351351,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29544372",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7609291,\n   "lon": 37.7337426,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29544871",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7704451,\n   "lon": 37.7270379,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29601418",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.795086,\n   "lon": 37.725205,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29612027",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8075688,\n   "lon": 37.7199308,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29612138",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8031062,\n   "lon": 37.7196089,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29688992",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7331774,\n   "lon": 37.8158366,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29690884",\n   "kind": "organisation",\n   "name": "Управа района Вешняки",\n   "office_type": "government",\n   "lat": 55.7363838,\n   "lon": 37.8291533,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29854157",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.806966,\n   "lon": 37.6912736,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29854159",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8073396,\n   "lon": 37.6907887,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29854193",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7966701,\n   "lon": 37.694116,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29854197",\n   "kind": "organisation",\n   "name": "Городской методический центр",\n   "office_type": "educational_institution",\n   "lat": 55.7997886,\n   "lon": 37.70527,\n   "opening_hours": null\n  },\n  {\n   "id": "way/29854216",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.805134,\n   "lon": 37.7062762,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30036352",\n   "kind": "organisation",\n   "name": "Главное архивное управление г. Москвы",\n   "office_type": "government",\n   "lat": 55.6566374,\n   "lon": 37.5388351,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30049120",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7393815,\n   "lon": 37.4738898,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30049122",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7398118,\n   "lon": 37.4734875,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30049923",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7344553,\n   "lon": 37.4820818,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30049926",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7338256,\n   "lon": 37.4813817,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30049929",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7336142,\n   "lon": 37.480315,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30051587",\n   "kind": "organisation",\n   "name": "Дирекция по обеспечению деятельности государственных учреждений здравоохранения Западного административного округа города Москвы",\n   "office_type": "government",\n   "lat": 55.7366766,\n   "lon": 37.4882125,\n   "opening_hours": "08:30-17:15"\n  },\n  {\n   "id": "way/30051612",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7367,\n   "lon": 37.5017395,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30051637",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7379666,\n   "lon": 37.488225,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30051700",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7433173,\n   "lon": 37.5009996,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30098150",\n   "kind": "organisation",\n   "name": "Московский центр качества образования",\n   "office_type": "educational_institution",\n   "lat": 55.7081066,\n   "lon": 37.6032504,\n   "opening_hours": "Mo-Fr 09:00-17:30"\n  },\n  {\n   "id": "way/30098648",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7063781,\n   "lon": 37.6179568,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30131690",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8003863,\n   "lon": 37.7923564,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30153015",\n   "kind": "building",\n   "name": "Посольство Туркменистана",\n   "office_type": "diplomatic",\n   "lat": 55.75055,\n   "lon": 37.5990445,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30178517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6513446,\n   "lon": 37.548303,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30185148",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7248797,\n   "lon": 37.6655121,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30185173",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7245554,\n   "lon": 37.66569,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30280881",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6939955,\n   "lon": 37.5642839,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30280883",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6940561,\n   "lon": 37.5627178,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30280885",\n   "kind": "organisation",\n   "name": "Математический институт имени В. А. Стеклова РАН",\n   "office_type": "research",\n   "lat": 55.6947728,\n   "lon": 37.5644395,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30350695",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6890338,\n   "lon": 37.5726138,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30384122",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7079498,\n   "lon": 37.5228735,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30384151",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7082745,\n   "lon": 37.5227125,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30384156",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.707994,\n   "lon": 37.5225114,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30384159",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.708121,\n   "lon": 37.5223045,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30384169",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7083315,\n   "lon": 37.5222372,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30384201",\n   "kind": "building",\n   "name": "Гарант",\n   "office_type": null,\n   "lat": 55.7085801,\n   "lon": 37.5218508,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30385694",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6886639,\n   "lon": 37.5932102,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30389838",\n   "kind": "organisation",\n   "name": "Главное управление пенсионного фонда РФ №3",\n   "office_type": "government",\n   "lat": 55.7245462,\n   "lon": 37.6921531,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30415236",\n   "kind": "building",\n   "name": "Институт геологии и разработки горючих ископаемых (ИГиРГИ)",\n   "office_type": "research",\n   "lat": 55.6958933,\n   "lon": 37.5721761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30423968",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7534989,\n   "lon": 37.4907106,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30423994",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7531716,\n   "lon": 37.4811158,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30425604",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8169055,\n   "lon": 37.6440853,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30425611",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8166379,\n   "lon": 37.6496117,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30425666",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8176184,\n   "lon": 37.6497463,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30425669",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8168637,\n   "lon": 37.6474869,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30425898",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8431542,\n   "lon": 37.564967,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30426002",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8193675,\n   "lon": 37.647916,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30426292",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8434158,\n   "lon": 37.5784449,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30426293",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.843862,\n   "lon": 37.578439,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30426860",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.817821,\n   "lon": 37.6310447,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30433160",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7214599,\n   "lon": 37.5724431,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30433214",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.720187,\n   "lon": 37.5686194,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30433418",\n   "kind": "organisation",\n   "name": "Интер РАО ЕЭС",\n   "office_type": "company",\n   "lat": 55.7291483,\n   "lon": 37.5686583,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30433425",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.725995,\n   "lon": 37.5756655,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30435600",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7888572,\n   "lon": 37.5732978,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30435721",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7894928,\n   "lon": 37.5800399,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30435747",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7899463,\n   "lon": 37.5811142,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30435750",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.791198,\n   "lon": 37.5769851,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30437389",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7942159,\n   "lon": 37.6083425,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30437399",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7984395,\n   "lon": 37.6046588,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30437402",\n   "kind": "organisation",\n   "name": "Издательский дом «Бурда»",\n   "office_type": "publisher",\n   "lat": 55.7996007,\n   "lon": 37.6069694,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30437404",\n   "kind": "building",\n   "name": "Диасофт",\n   "office_type": "company",\n   "lat": 55.7983274,\n   "lon": 37.6061044,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30528810",\n   "kind": "building",\n   "name": "Всероссийский научно-исследовстельский институт труда",\n   "office_type": "research",\n   "lat": 55.7572322,\n   "lon": 37.6544181,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30639334",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7986851,\n   "lon": 37.5707891,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30649756",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7913817,\n   "lon": 37.5909039,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30853905",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6935646,\n   "lon": 37.7421004,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30854047",\n   "kind": "organisation",\n   "name": "Управление ФНС России по г. Москве, ЮВАО, №23",\n   "office_type": "government",\n   "lat": 55.6860566,\n   "lon": 37.7569716,\n   "opening_hours": null\n  },\n  {\n   "id": "way/30858430",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6844576,\n   "lon": 37.6742098,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31019687",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8388808,\n   "lon": 37.6540846,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31091545",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7635369,\n   "lon": 37.5669789,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31303271",\n   "kind": "organisation",\n   "name": "Научно-Методический Центр ЮВОУО ДО",\n   "office_type": "educational_institution",\n   "lat": 55.6806795,\n   "lon": 37.7404985,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31482137",\n   "kind": "building",\n   "name": "Институт радиотехники и электроники им. В. А. Котельникова РАН",\n   "office_type": "research",\n   "lat": 55.7561603,\n   "lon": 37.6113491,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31482139",\n   "kind": "building",\n   "name": "НИИ нормальной физиологии им. П. К. Анохина",\n   "office_type": "research",\n   "lat": 55.755953,\n   "lon": 37.6102817,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31482157",\n   "kind": "building",\n   "name": "Мосгоризбирком",\n   "office_type": "government",\n   "lat": 55.7565086,\n   "lon": 37.6119528,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31482159",\n   "kind": "building",\n   "name": "Институт Европы РАН",\n   "office_type": "research",\n   "lat": 55.7566763,\n   "lon": 37.6101688,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31482166",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.757407,\n   "lon": 37.6110883,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31482170",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.757151,\n   "lon": 37.6098093,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31482173",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7571224,\n   "lon": 37.6054182,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31496301",\n   "kind": "organisation",\n   "name": "Институт языкознания РАН",\n   "office_type": "research",\n   "lat": 55.7532839,\n   "lon": 37.6052191,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31496312",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7555298,\n   "lon": 37.6057938,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31496320",\n   "kind": "building",\n   "name": "Пожарная часть №33",\n   "office_type": null,\n   "lat": 55.7553672,\n   "lon": 37.6029251,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31497214",\n   "kind": "organisation",\n   "name": "Психологический институт РАО",\n   "office_type": "research",\n   "lat": 55.7535913,\n   "lon": 37.6101362,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31497223",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Воздвиженка центр\\"",\n   "office_type": null,\n   "lat": 55.7533204,\n   "lon": 37.6063465,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31525641",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7580804,\n   "lon": 37.6089739,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31525644",\n   "kind": "building",\n   "name": "Министерство образования и науки России",\n   "office_type": "government",\n   "lat": 55.7602499,\n   "lon": 37.6091118,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31525648",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7594921,\n   "lon": 37.6098507,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31550347",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7583126,\n   "lon": 37.6092714,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31550350",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Экоофис\\"",\n   "office_type": null,\n   "lat": 55.7593059,\n   "lon": 37.6103849,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31550358",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7612298,\n   "lon": 37.6150715,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31608140",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7643153,\n   "lon": 37.6270316,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31608155",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7635549,\n   "lon": 37.6284452,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31608156",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7639348,\n   "lon": 37.6276539,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31660128",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8077322,\n   "lon": 37.6107571,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31660130",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8112794,\n   "lon": 37.6037363,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31660675",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7707934,\n   "lon": 37.6129877,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31660677",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.77017,\n   "lon": 37.6139691,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31712854",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8185039,\n   "lon": 37.6052964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31712855",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.818552,\n   "lon": 37.6046556,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31712856",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.819042,\n   "lon": 37.6039631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31712857",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8190262,\n   "lon": 37.6051291,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31717818",\n   "kind": "organisation",\n   "name": "Римско-католическая Архиепархия Божией Матери в Москве",\n   "office_type": "religion",\n   "lat": 55.7673969,\n   "lon": 37.570568,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31724088",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7665623,\n   "lon": 37.646401,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31724090",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7665325,\n   "lon": 37.6487518,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31724095",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7674428,\n   "lon": 37.6496159,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31724102",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7672743,\n   "lon": 37.6370584,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31724103",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7671204,\n   "lon": 37.637548,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31724117",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7624612,\n   "lon": 37.647423,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31793989",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7679112,\n   "lon": 37.6292992,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31794008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.768143,\n   "lon": 37.6275294,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31794026",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7706211,\n   "lon": 37.6310426,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31891521",\n   "kind": "organisation",\n   "name": "ГКУ \\"Соцэнерго\\"",\n   "office_type": "company",\n   "lat": 55.777611,\n   "lon": 37.5893658,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31950856",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7772123,\n   "lon": 37.5994046,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31951022",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.772145,\n   "lon": 37.6071843,\n   "opening_hours": null\n  },\n  {\n   "id": "way/31951024",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7717439,\n   "lon": 37.6079241,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32013463",\n   "kind": "organisation",\n   "name": "ЦНИИ «Циклон»",\n   "office_type": "research",\n   "lat": 55.8112942,\n   "lon": 37.8043428,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32045633",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7557767,\n   "lon": 37.596141,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32045634",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7558845,\n   "lon": 37.5976538,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32045646",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7566523,\n   "lon": 37.5973532,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32045647",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7563399,\n   "lon": 37.5976539,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32045649",\n   "kind": "organisation",\n   "name": "Посольство Испании",\n   "office_type": "diplomatic",\n   "lat": 55.7581307,\n   "lon": 37.5899241,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32045650",\n   "kind": "building",\n   "name": "Резиденция посла Турции",\n   "office_type": "diplomatic",\n   "lat": 55.7573125,\n   "lon": 37.5920277,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32066738",\n   "kind": "organisation",\n   "name": "Официальная резиденция Посла Республики Куба",\n   "office_type": "diplomatic",\n   "lat": 55.7620101,\n   "lon": 37.5912131,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32113465",\n   "kind": "building",\n   "name": "Бронная плаза",\n   "office_type": null,\n   "lat": 55.765411,\n   "lon": 37.5909011,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32113483",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7650503,\n   "lon": 37.5987636,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32113485",\n   "kind": "organisation",\n   "name": "Посольство Иордании",\n   "office_type": "diplomatic",\n   "lat": 55.7658967,\n   "lon": 37.5974047,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32203084",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7844287,\n   "lon": 37.4676244,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32203093",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7913247,\n   "lon": 37.4555286,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32235315",\n   "kind": "building",\n   "name": "Финансово-казначейское управление ЮАО",\n   "office_type": "government",\n   "lat": 55.7108755,\n   "lon": 37.6593848,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32244153",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7542797,\n   "lon": 37.7121446,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32276135",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7569953,\n   "lon": 37.7088094,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32496411",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8464004,\n   "lon": 37.5639951,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32499985",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8487772,\n   "lon": 37.5790595,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32500035",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.7148897,\n   "lon": 37.6212205,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32605826",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7157669,\n   "lon": 37.5947677,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32606552",\n   "kind": "organisation",\n   "name": "Моя работа",\n   "office_type": "government",\n   "lat": 55.7143071,\n   "lon": 37.6063963,\n   "opening_hours": "Mo-Fr 09:00-20-00"\n  },\n  {\n   "id": "way/32607670",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7052953,\n   "lon": 37.6207986,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32608694",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6923963,\n   "lon": 37.6227317,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32609212",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6942818,\n   "lon": 37.6072594,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32610846",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6789525,\n   "lon": 37.6034803,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32610847",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6792616,\n   "lon": 37.6033231,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32611555",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6603935,\n   "lon": 37.6252696,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32611865",\n   "kind": "building",\n   "name": "МОЭСК - Центральные электрические сети",\n   "office_type": null,\n   "lat": 55.6609135,\n   "lon": 37.6308689,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32616458",\n   "kind": "building",\n   "name": "Усадьба Салтыковых - Чертковых",\n   "office_type": null,\n   "lat": 55.7605531,\n   "lon": 37.631055,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32616459",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7609018,\n   "lon": 37.6315288,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32616462",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7611646,\n   "lon": 37.6301613,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32650828",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.7183247,\n   "lon": 37.6239015,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32692803",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7517123,\n   "lon": 37.61073,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32693721",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7480728,\n   "lon": 37.6017335,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32693723",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7479434,\n   "lon": 37.6014952,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32693726",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7468766,\n   "lon": 37.6027263,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32722777",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7534106,\n   "lon": 37.5418193,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32722779",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7529232,\n   "lon": 37.541799,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32783676",\n   "kind": "organisation",\n   "name": "Главное следственное управление Следственного комитета РФ по г. Москве",\n   "office_type": "government",\n   "lat": 55.7511698,\n   "lon": 37.5943424,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32783696",\n   "kind": "organisation",\n   "name": "Посольство Бельгии",\n   "office_type": "diplomatic",\n   "lat": 55.753721,\n   "lon": 37.5924675,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32783700",\n   "kind": "organisation",\n   "name": "Посольство Монголии",\n   "office_type": "diplomatic",\n   "lat": 55.7546722,\n   "lon": 37.5883459,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32783702",\n   "kind": "organisation",\n   "name": "Национальный медицинский исследовательский центр реабилитации и курортологии",\n   "office_type": "research",\n   "lat": 55.7539423,\n   "lon": 37.5893143,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32783704",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7530162,\n   "lon": 37.5921272,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32784128",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7556309,\n   "lon": 37.5970647,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32786114",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6554101,\n   "lon": 37.557569,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32791294",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8352728,\n   "lon": 37.5693407,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32868214",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7593011,\n   "lon": 37.5567396,\n   "opening_hours": null\n  },\n  {\n   "id": "way/32903280",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7587728,\n   "lon": 37.6047244,\n   "opening_hours": null\n  },\n  {\n   "id": "way/33127310",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Омега-2\\" корп. C",\n   "office_type": null,\n   "lat": 55.7100817,\n   "lon": 37.6555477,\n   "opening_hours": null\n  },\n  {\n   "id": "way/33456225",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7529672,\n   "lon": 37.7157463,\n   "opening_hours": null\n  },\n  {\n   "id": "way/33645378",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8217989,\n   "lon": 37.5707541,\n   "opening_hours": null\n  },\n  {\n   "id": "way/33816325",\n   "kind": "organisation",\n   "name": "Мортон. Офис продаж",\n   "office_type": "estate_agent",\n   "lat": 55.7600922,\n   "lon": 37.717114,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34319560",\n   "kind": "building",\n   "name": "Федеральное агентство железнодорожного транспорта",\n   "office_type": "government",\n   "lat": 55.7656813,\n   "lon": 37.6585103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34319561",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7665239,\n   "lon": 37.6580642,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34319562",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7669936,\n   "lon": 37.6579878,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34319563",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7662387,\n   "lon": 37.6586597,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34319576",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7711761,\n   "lon": 37.6499237,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34319583",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7670378,\n   "lon": 37.646607,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34319623",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7697147,\n   "lon": 37.6761116,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34319969",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7606372,\n   "lon": 37.598726,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34320668",\n   "kind": "building",\n   "name": "Российский центр судебно-медицинской экспертизы",\n   "office_type": null,\n   "lat": 55.7777203,\n   "lon": 37.5514922,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34417779",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7563783,\n   "lon": 37.548743,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34585736",\n   "kind": "building",\n   "name": "Служба финансового контроля Департамента здравоохранения города Москвы",\n   "office_type": "government",\n   "lat": 55.8224228,\n   "lon": 37.5791599,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:45"\n  },\n  {\n   "id": "way/34844420",\n   "kind": "organisation",\n   "name": "Бизнес-центр \\"Савва\\"",\n   "office_type": "company",\n   "lat": 55.7605787,\n   "lon": 37.7035378,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34873845",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7736052,\n   "lon": 37.5260297,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34873846",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7731525,\n   "lon": 37.5269357,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34911451",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6763498,\n   "lon": 37.6928516,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34922709",\n   "kind": "organisation",\n   "name": "1 СПП ГУ МВД России по г. Москве",\n   "office_type": "government",\n   "lat": 55.6785316,\n   "lon": 37.6669988,\n   "opening_hours": null\n  },\n  {\n   "id": "way/34996665",\n   "kind": "organisation",\n   "name": "Южное окружное управление образования",\n   "office_type": "government",\n   "lat": 55.6791396,\n   "lon": 37.65926,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35012974",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8030006,\n   "lon": 37.5164477,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35024691",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7983856,\n   "lon": 37.5305464,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35024815",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7973324,\n   "lon": 37.5336667,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35024816",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7970732,\n   "lon": 37.532893,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35024971",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Авиатор\\"",\n   "office_type": null,\n   "lat": 55.793667,\n   "lon": 37.5451209,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35029689",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8889958,\n   "lon": 37.5943508,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35091018",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8218737,\n   "lon": 37.5621847,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35091019",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8212037,\n   "lon": 37.5624798,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35159721",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7867296,\n   "lon": 37.6807733,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168339",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7845519,\n   "lon": 37.63553,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168341",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7824881,\n   "lon": 37.6346691,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168343",\n   "kind": "building",\n   "name": "ГУП \\"Московский метрополитен\\"",\n   "office_type": "company",\n   "lat": 55.7817474,\n   "lon": 37.6326162,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168352",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7806619,\n   "lon": 37.6338557,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168355",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7810036,\n   "lon": 37.6329025,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168374",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7784942,\n   "lon": 37.6301979,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168375",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7779429,\n   "lon": 37.6309044,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168378",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7787002,\n   "lon": 37.6319633,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168379",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7789872,\n   "lon": 37.6294822,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168380",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7799985,\n   "lon": 37.6296145,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168383",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7780465,\n   "lon": 37.6321223,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168391",\n   "kind": "organisation",\n   "name": "посольство Шри-Ланки",\n   "office_type": "diplomatic",\n   "lat": 55.7778316,\n   "lon": 37.6292983,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168393",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7765218,\n   "lon": 37.6317882,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168396",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7771071,\n   "lon": 37.6331234,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168399",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7798317,\n   "lon": 37.6344347,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168404",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7807089,\n   "lon": 37.6393756,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168409",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7808046,\n   "lon": 37.6402525,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168417",\n   "kind": "building",\n   "name": "Посольство Эфиопии",\n   "office_type": "diplomatic",\n   "lat": 55.7832521,\n   "lon": 37.6401733,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35168803",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7871367,\n   "lon": 37.6362491,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35222440",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6817976,\n   "lon": 37.5997062,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35355247",\n   "kind": "organisation",\n   "name": "МГТС",\n   "office_type": "telecommunication",\n   "lat": 55.6159442,\n   "lon": 37.70321,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35543227",\n   "kind": "building",\n   "name": "Северная башня",\n   "office_type": null,\n   "lat": 55.7518355,\n   "lon": 37.5326582,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35547549",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7511032,\n   "lon": 37.7118165,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35547550",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.750993,\n   "lon": 37.7078283,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35547552",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7506087,\n   "lon": 37.7095766,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35559239",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6766485,\n   "lon": 37.5765034,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35559313",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6788416,\n   "lon": 37.583045,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35559537",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6788387,\n   "lon": 37.5748232,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35559674",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.6839895,\n   "lon": 37.5784413,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35559683",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6802696,\n   "lon": 37.5706263,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35559684",\n   "kind": "organisation",\n   "name": "Российская академия сельскохозяйственных наух",\n   "office_type": "research",\n   "lat": 55.6801747,\n   "lon": 37.5714642,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35560443",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6841417,\n   "lon": 37.5537266,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35560595",\n   "kind": "organisation",\n   "name": "Институт кристаллографии им. А. В. Шубникова РАН",\n   "office_type": "research",\n   "lat": 55.6955899,\n   "lon": 37.5601946,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574498",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.679008,\n   "lon": 37.5899697,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574613",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6804392,\n   "lon": 37.5682288,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574639",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6796674,\n   "lon": 37.5701047,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574640",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6794011,\n   "lon": 37.5677962,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574646",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6794371,\n   "lon": 37.5689678,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574731",\n   "kind": "building",\n   "name": "Институт \\"Каналстройпроект\\"",\n   "office_type": "company",\n   "lat": 55.6790426,\n   "lon": 37.5722768,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574739",\n   "kind": "building",\n   "name": "УВД по ЮЗАО",\n   "office_type": null,\n   "lat": 55.6785587,\n   "lon": 37.5715306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574741",\n   "kind": "building",\n   "name": "Черёмушкинский районный суд",\n   "office_type": null,\n   "lat": 55.6784655,\n   "lon": 37.5738613,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:45"\n  },\n  {\n   "id": "way/35574743",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6781153,\n   "lon": 37.5723883,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574830",\n   "kind": "building",\n   "name": "Архив РАН",\n   "office_type": null,\n   "lat": 55.6769502,\n   "lon": 37.5723016,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35574912",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6774463,\n   "lon": 37.5698414,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35575084",\n   "kind": "organisation",\n   "name": "Фирма-Курьер",\n   "office_type": "logistics",\n   "lat": 55.6878157,\n   "lon": 37.5664396,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35589278",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7126521,\n   "lon": 37.6170374,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35590380",\n   "kind": "building",\n   "name": "Мосэнергострой",\n   "office_type": "company",\n   "lat": 55.7135732,\n   "lon": 37.6171992,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35604034",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6918637,\n   "lon": 37.5647185,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35604043",\n   "kind": "building",\n   "name": "Бизнес-центр",\n   "office_type": null,\n   "lat": 55.6929514,\n   "lon": 37.5667869,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35604077",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6918367,\n   "lon": 37.5725955,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35610495",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7763587,\n   "lon": 37.6216073,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35610503",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7745683,\n   "lon": 37.6285994,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35610504",\n   "kind": "building",\n   "name": "ОЦ \\"Synergy development\\"",\n   "office_type": null,\n   "lat": 55.7762287,\n   "lon": 37.6272314,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35610506",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7762984,\n   "lon": 37.6252918,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35610513",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7755074,\n   "lon": 37.6284019,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35667968",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6835338,\n   "lon": 37.5871801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35680155",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6869103,\n   "lon": 37.5838444,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35680217",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6937039,\n   "lon": 37.5848362,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35683824",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7705551,\n   "lon": 37.6686962,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35683825",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.771116,\n   "lon": 37.6645934,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35693472",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.768091,\n   "lon": 37.662066,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35693484",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7676862,\n   "lon": 37.6621357,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35693490",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7664647,\n   "lon": 37.6612771,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35693533",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7708182,\n   "lon": 37.6712729,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35763195",\n   "kind": "organisation",\n   "name": "ЖЭУ",\n   "office_type": "property_management",\n   "lat": 55.6732876,\n   "lon": 37.5391878,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35766555",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6651353,\n   "lon": 37.5169238,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35766604",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6596235,\n   "lon": 37.5244796,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35766627",\n   "kind": "organisation",\n   "name": "Кристаллография и фотоника РАН",\n   "office_type": "research",\n   "lat": 55.6636146,\n   "lon": 37.5277177,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35766632",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6676704,\n   "lon": 37.5242339,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35774131",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7067189,\n   "lon": 37.6007512,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35804416",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447487,\n   "lon": 37.5991085,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35804418",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7433117,\n   "lon": 37.5982588,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35804427",\n   "kind": "organisation",\n   "name": "Московский центр развития кадрового потенциала образования",\n   "office_type": "educational_institution",\n   "lat": 55.7432534,\n   "lon": 37.5927835,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35804440",\n   "kind": "organisation",\n   "name": "посольство Мексики",\n   "office_type": "diplomatic",\n   "lat": 55.743189,\n   "lon": 37.5897904,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35814527",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.748473,\n   "lon": 37.5975436,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35814548",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Альфа Арбат Центр\\"",\n   "office_type": null,\n   "lat": 55.7516909,\n   "lon": 37.5992966,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35814552",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7499386,\n   "lon": 37.598185,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35814554",\n   "kind": "organisation",\n   "name": "Резиденция посла Азербайджана",\n   "office_type": "diplomatic",\n   "lat": 55.7502324,\n   "lon": 37.5994401,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35814581",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7564404,\n   "lon": 37.5932739,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35814643",\n   "kind": "organisation",\n   "name": "Посольство Кипра",\n   "office_type": "diplomatic",\n   "lat": 55.7539615,\n   "lon": 37.5953853,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35948519",\n   "kind": "building",\n   "name": "Мосгоргеотрест",\n   "office_type": "architect",\n   "lat": 55.8633739,\n   "lon": 37.4833194,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35948531",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8664309,\n   "lon": 37.4831151,\n   "opening_hours": null\n  },\n  {\n   "id": "way/35977722",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.678519,\n   "lon": 37.5776385,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36072197",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.710498,\n   "lon": 37.5871124,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36358879",\n   "kind": "building",\n   "name": "Бизнес-центр «Даев Плаза»",\n   "office_type": null,\n   "lat": 55.7693806,\n   "lon": 37.638418,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36358911",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7688976,\n   "lon": 37.633527,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36358916",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7697539,\n   "lon": 37.6345912,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36417313",\n   "kind": "organisation",\n   "name": "Московское речное пароходство",\n   "office_type": "company",\n   "lat": 55.8577082,\n   "lon": 37.4658636,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36419089",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7795043,\n   "lon": 37.5940009,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36438939",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8656087,\n   "lon": 37.4845046,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36438942",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8665654,\n   "lon": 37.4812451,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36455507",\n   "kind": "building",\n   "name": "Институт проблем экологии и эволюции им. А. Н. Северцова",\n   "office_type": "research",\n   "lat": 55.6944655,\n   "lon": 37.5671897,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36489508",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6324058,\n   "lon": 37.5375248,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36507990",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.679211,\n   "lon": 37.5844494,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36508120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6789971,\n   "lon": 37.5851931,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36515106",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.671334,\n   "lon": 37.5309285,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36743674",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6495109,\n   "lon": 37.6076069,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36743698",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6558571,\n   "lon": 37.6073672,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36780013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7556194,\n   "lon": 37.5549211,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36793070",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8447466,\n   "lon": 37.5009553,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36812846",\n   "kind": "building",\n   "name": "Общественная палата Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7788407,\n   "lon": 37.5930407,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36814750",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.822143,\n   "lon": 37.4918701,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36814753",\n   "kind": "organisation",\n   "name": "МосОблСтат",\n   "office_type": "government",\n   "lat": 55.8218193,\n   "lon": 37.4920854,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36815237",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8876943,\n   "lon": 37.5971334,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36852892",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7440252,\n   "lon": 37.5810631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36938482",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6620907,\n   "lon": 37.5347814,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36938947",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6601153,\n   "lon": 37.5375521,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36938951",\n   "kind": "organisation",\n   "name": "Российский государственный архив научно-технической документации (РГАНТД)",\n   "office_type": "government",\n   "lat": 55.6574177,\n   "lon": 37.5389739,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36938952",\n   "kind": "organisation",\n   "name": "Главное архивное управление г. Москвы",\n   "office_type": "government",\n   "lat": 55.6579281,\n   "lon": 37.5400534,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36999675",\n   "kind": "organisation",\n   "name": "посольство Чехии",\n   "office_type": "diplomatic",\n   "lat": 55.7701188,\n   "lon": 37.5885157,\n   "opening_hours": "Mo-Fr 08:30-17:00"\n  },\n  {\n   "id": "way/36999680",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7709079,\n   "lon": 37.5766144,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36999695",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7708091,\n   "lon": 37.566014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/36999737",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7716713,\n   "lon": 37.5764657,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37001618",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Парус\\"",\n   "office_type": null,\n   "lat": 55.7751578,\n   "lon": 37.5867298,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37001671",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7793557,\n   "lon": 37.6058247,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37001672",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7791714,\n   "lon": 37.6064115,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37002213",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7805128,\n   "lon": 37.5898133,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37045054",\n   "kind": "organisation",\n   "name": "Посольство Малайзии",\n   "office_type": "diplomatic",\n   "lat": 55.7203698,\n   "lon": 37.5164432,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "way/37045056",\n   "kind": "organisation",\n   "name": "Посольство Сербии",\n   "office_type": "diplomatic",\n   "lat": 55.7184605,\n   "lon": 37.5188455,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37045057",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7195483,\n   "lon": 37.5198456,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37082707",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Воробьевский\\"",\n   "office_type": null,\n   "lat": 55.715514,\n   "lon": 37.52265,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37082708",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7164153,\n   "lon": 37.5217834,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37082729",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7197515,\n   "lon": 37.5194792,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37082739",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7147122,\n   "lon": 37.5164826,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37082749",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7247179,\n   "lon": 37.5255718,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37082806",\n   "kind": "organisation",\n   "name": "Посольство Румынии",\n   "office_type": "diplomatic",\n   "lat": 55.7149797,\n   "lon": 37.5122047,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37082807",\n   "kind": "organisation",\n   "name": "Посольство Венгрии",\n   "office_type": "diplomatic",\n   "lat": 55.7156602,\n   "lon": 37.5137378,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37082809",\n   "kind": "organisation",\n   "name": "Посольство Болгарии",\n   "office_type": "diplomatic",\n   "lat": 55.7137546,\n   "lon": 37.5097297,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37082844",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7209205,\n   "lon": 37.5186624,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130923",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7778227,\n   "lon": 37.6015867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130969",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.780862,\n   "lon": 37.6052476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130977",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7801348,\n   "lon": 37.6067355,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130979",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7794072,\n   "lon": 37.6062932,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130980",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7790462,\n   "lon": 37.6059334,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130982",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7793803,\n   "lon": 37.6078989,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130984",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7802346,\n   "lon": 37.6077405,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130985",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7804883,\n   "lon": 37.608074,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130990",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7815074,\n   "lon": 37.6059637,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130991",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7789296,\n   "lon": 37.6063884,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130993",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7787999,\n   "lon": 37.606614,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37130999",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7778605,\n   "lon": 37.601076,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37131000",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.777693,\n   "lon": 37.6006917,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37191279",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7074615,\n   "lon": 37.6597548,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37191282",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7073746,\n   "lon": 37.6617527,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37191297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7053909,\n   "lon": 37.6582426,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37191307",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7044654,\n   "lon": 37.6552625,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37191308",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7036193,\n   "lon": 37.6554831,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37214071",\n   "kind": "building",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Отрадное",\n   "office_type": "government",\n   "lat": 55.8625416,\n   "lon": 37.58986,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37446384",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7241387,\n   "lon": 37.5047041,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37447151",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7524712,\n   "lon": 37.6047423,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37448944",\n   "kind": "building",\n   "name": "Город Столиц - Санкт-Петербург",\n   "office_type": null,\n   "lat": 55.7473447,\n   "lon": 37.5385673,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37453476",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Четыре ветра\\"",\n   "office_type": null,\n   "lat": 55.774574,\n   "lon": 37.587118,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37543686",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7319999,\n   "lon": 37.5522297,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37543691",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7313096,\n   "lon": 37.5500528,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37543702",\n   "kind": "building",\n   "name": "Западное окружное управление образования",\n   "office_type": null,\n   "lat": 55.7383623,\n   "lon": 37.5401268,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37611506",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7408439,\n   "lon": 37.5453255,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37611510",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7423157,\n   "lon": 37.5433918,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37612496",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7483759,\n   "lon": 37.5671542,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37612519",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7399619,\n   "lon": 37.551907,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37645136",\n   "kind": "organisation",\n   "name": "Торгпредство США",\n   "office_type": "diplomatic",\n   "lat": 55.7559827,\n   "lon": 37.5786691,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37645148",\n   "kind": "organisation",\n   "name": "Посольство США",\n   "office_type": "diplomatic",\n   "lat": 55.7560167,\n   "lon": 37.5794226,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37645229",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7624881,\n   "lon": 37.5691666,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37645231",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7632182,\n   "lon": 37.5652941,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37645240",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7604951,\n   "lon": 37.5724741,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37647620",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7622674,\n   "lon": 37.5686441,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37647621",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7621778,\n   "lon": 37.5691337,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37647624",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626545,\n   "lon": 37.5688163,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37647628",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7632818,\n   "lon": 37.563956,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37647643",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.767431,\n   "lon": 37.5594332,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37647651",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7615034,\n   "lon": 37.5632155,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37647657",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7657996,\n   "lon": 37.558846,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37735415",\n   "kind": "building",\n   "name": "Российские железные дороги",\n   "office_type": "company",\n   "lat": 55.7767979,\n   "lon": 37.647906,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37756919",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7745336,\n   "lon": 37.647895,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37756923",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7753644,\n   "lon": 37.6505174,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37756924",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7757633,\n   "lon": 37.6496105,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37756966",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7744359,\n   "lon": 37.6454856,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37756975",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7725372,\n   "lon": 37.6418694,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37757008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7778378,\n   "lon": 37.6418125,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37757037",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7778724,\n   "lon": 37.646839,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37757039",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7781877,\n   "lon": 37.6439324,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37757051",\n   "kind": "organisation",\n   "name": "Центральный аппарат ППК «Роскадастр»",\n   "office_type": "government",\n   "lat": 55.7716621,\n   "lon": 37.6505843,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37759663",\n   "kind": "building",\n   "name": "Страховой Дом ВСК",\n   "office_type": "insurance",\n   "lat": 55.7568105,\n   "lon": 37.4477564,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37760885",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7389531,\n   "lon": 37.5125549,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37760897",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7400282,\n   "lon": 37.5263543,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37760898",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7423067,\n   "lon": 37.5237188,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37760899",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7426841,\n   "lon": 37.5230761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37760919",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7393938,\n   "lon": 37.5276168,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37760923",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7359837,\n   "lon": 37.5262294,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37762428",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7684658,\n   "lon": 37.512898,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37762433",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7688675,\n   "lon": 37.5070048,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37762439",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7647864,\n   "lon": 37.5099644,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37762443",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7645099,\n   "lon": 37.5090475,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37764334",\n   "kind": "building",\n   "name": "Офисно-деловой центр \\"Бородино Плаза\\"",\n   "office_type": null,\n   "lat": 55.7844577,\n   "lon": 37.6734077,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769462",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7451069,\n   "lon": 37.6584043,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769471",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7453143,\n   "lon": 37.6635505,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769472",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7446453,\n   "lon": 37.6632613,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769490",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7463929,\n   "lon": 37.6646645,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769493",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7486487,\n   "lon": 37.6597992,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769496",\n   "kind": "organisation",\n   "name": "ГСПИ радио и телевидения филиал РСВО",\n   "office_type": "research",\n   "lat": 55.748353,\n   "lon": 37.658761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769497",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7481961,\n   "lon": 37.6596776,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769505",\n   "kind": "building",\n   "name": "Бизнес-центр «Мосэнка 5»",\n   "office_type": null,\n   "lat": 55.7467503,\n   "lon": 37.6639446,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769508",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7486447,\n   "lon": 37.6602633,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769516",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7476283,\n   "lon": 37.6565504,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769522",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449292,\n   "lon": 37.6524474,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769527",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7440179,\n   "lon": 37.6507596,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769992",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.744457,\n   "lon": 37.650555,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769996",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7451326,\n   "lon": 37.6478785,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37769997",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7446246,\n   "lon": 37.646973,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37770003",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7433391,\n   "lon": 37.6514217,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37770004",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.743682,\n   "lon": 37.6514691,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37770005",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7436388,\n   "lon": 37.6519761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37770011",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.743562,\n   "lon": 37.6497548,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37770013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7430768,\n   "lon": 37.6465099,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37770017",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7423222,\n   "lon": 37.6463498,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37771099",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7412544,\n   "lon": 37.6397793,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37771106",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7406591,\n   "lon": 37.6335208,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37771152",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7397533,\n   "lon": 37.640561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37771167",\n   "kind": "organisation",\n   "name": "Институт философии РАН",\n   "office_type": "research",\n   "lat": 55.745229,\n   "lon": 37.645129,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37771185",\n   "kind": "building",\n   "name": "Росреестр",\n   "office_type": "government",\n   "lat": 55.7525768,\n   "lon": 37.6489789,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37771186",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7523644,\n   "lon": 37.648548,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37771204",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447408,\n   "lon": 37.6384599,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37771217",\n   "kind": "organisation",\n   "name": "Постоянное Представительство Республики Адыгея при Президенте РФ",\n   "office_type": "government",\n   "lat": 55.7401285,\n   "lon": 37.631624,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37786966",\n   "kind": "building",\n   "name": "Дирекция железнодорожных вокзалов",\n   "office_type": "railway",\n   "lat": 55.7378168,\n   "lon": 37.6960538,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37786992",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7107485,\n   "lon": 37.5773483,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37790290",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6585326,\n   "lon": 37.6060914,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37791113",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7879045,\n   "lon": 37.5110165,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37841917",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6772854,\n   "lon": 37.6743484,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37914835",\n   "kind": "organisation",\n   "name": "Восточный район электрических сетей",\n   "office_type": "company",\n   "lat": 55.8150261,\n   "lon": 37.7112432,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37932816",\n   "kind": "organisation",\n   "name": "Московское машиностроительное предприятние имени В.В. Чернышева",\n   "office_type": "company",\n   "lat": 55.8291665,\n   "lon": 37.4444145,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37975542",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7784508,\n   "lon": 37.689038,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37994555",\n   "kind": "building",\n   "name": "Бизнес-центр «Метрополис» (корпус 2)",\n   "office_type": null,\n   "lat": 55.8216315,\n   "lon": 37.4985827,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37998228",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8157925,\n   "lon": 37.5939476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37998232",\n   "kind": "building",\n   "name": "Управление образования Северо-Восточного административного округа города Москвы",\n   "office_type": "government",\n   "lat": 55.812543,\n   "lon": 37.5918498,\n   "opening_hours": null\n  },\n  {\n   "id": "way/37998550",\n   "kind": "building",\n   "name": "МГТС",\n   "office_type": "telecommunication",\n   "lat": 55.8181525,\n   "lon": 37.5790895,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-18:00; PH off"\n  },\n  {\n   "id": "way/37998552",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8178273,\n   "lon": 37.5782178,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38017841",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8506031,\n   "lon": 37.5895306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38026838",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8069745,\n   "lon": 37.62233,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38026839",\n   "kind": "organisation",\n   "name": "Проектмашприбор",\n   "office_type": "research",\n   "lat": 55.8122589,\n   "lon": 37.6266002,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38062721",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6550757,\n   "lon": 37.6230106,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38300183",\n   "kind": "building",\n   "name": "СК МПО им. И. И. Румянцева",\n   "office_type": null,\n   "lat": 55.7981826,\n   "lon": 37.5773238,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38344036",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7753187,\n   "lon": 37.499559,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38344058",\n   "kind": "organisation",\n   "name": "ГУП Мосгортранс филиал западный",\n   "office_type": "transport",\n   "lat": 55.7712581,\n   "lon": 37.5029554,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38344069",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7770588,\n   "lon": 37.5019404,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38573248",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7947116,\n   "lon": 37.7358506,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38573818",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7963766,\n   "lon": 37.7063276,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38573823",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7923523,\n   "lon": 37.7064413,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38573825",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7933003,\n   "lon": 37.7044286,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38672736",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7365475,\n   "lon": 37.4792988,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38692001",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7397106,\n   "lon": 37.4816278,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38695477",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7416948,\n   "lon": 37.4838148,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38749536",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8175824,\n   "lon": 37.510779,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38866818",\n   "kind": "building",\n   "name": "НИИ Автоматической аппаратуры им. академика В.С. Семинихина",\n   "office_type": "research",\n   "lat": 55.6595311,\n   "lon": 37.5412077,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38938880",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7567457,\n   "lon": 37.5626482,\n   "opening_hours": null\n  },\n  {\n   "id": "way/38966159",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7797331,\n   "lon": 37.607241,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39129711",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6685539,\n   "lon": 37.5659376,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39129746",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "research",\n   "lat": 55.6572561,\n   "lon": 37.5516284,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39129833",\n   "kind": "organisation",\n   "name": "Федеральное агентство водных ресурсов",\n   "office_type": "government",\n   "lat": 55.6850722,\n   "lon": 37.5675105,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39208692",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7612505,\n   "lon": 37.5631658,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39251637",\n   "kind": "building",\n   "name": "Лабораторно-производственный комплекс НИИ «Дельта»",\n   "office_type": null,\n   "lat": 55.8032017,\n   "lon": 37.7580544,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39344995",\n   "kind": "building",\n   "name": "Пинскдрев",\n   "office_type": null,\n   "lat": 55.7113489,\n   "lon": 37.6567961,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "way/39393811",\n   "kind": "organisation",\n   "name": "Военкомат",\n   "office_type": "military",\n   "lat": 55.7160995,\n   "lon": 37.7397432,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39462423",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6792922,\n   "lon": 37.656701,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39464385",\n   "kind": "organisation",\n   "name": "АО «ВНИИХТ»",\n   "office_type": "research",\n   "lat": 55.6485605,\n   "lon": 37.6710123,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39496334",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7609168,\n   "lon": 37.6324666,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39532089",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6241983,\n   "lon": 37.6591145,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39532103",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6228615,\n   "lon": 37.6663774,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39539511",\n   "kind": "organisation",\n   "name": "Комитет труда и занятости",\n   "office_type": "government",\n   "lat": 55.7115258,\n   "lon": 37.7395612,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39577890",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "educational_institution",\n   "lat": 55.7264263,\n   "lon": 37.6346626,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39577896",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7264992,\n   "lon": 37.630747,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39578051",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7294824,\n   "lon": 37.6324107,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39578053",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7290116,\n   "lon": 37.6339096,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39578278",\n   "kind": "organisation",\n   "name": "Генеральное Консульство Испании",\n   "office_type": "diplomatic",\n   "lat": 55.7269859,\n   "lon": 37.6290727,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39578434",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7109229,\n   "lon": 37.7479327,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39578452",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7272058,\n   "lon": 37.6257794,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39630263",\n   "kind": "organisation",\n   "name": "Управа района Печатники ЮВАО",\n   "office_type": "government",\n   "lat": 55.6676657,\n   "lon": 37.7196252,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39634163",\n   "kind": "organisation",\n   "name": "ТСЖ «НИКА»",\n   "office_type": "property_management",\n   "lat": 55.7152248,\n   "lon": 37.4609634,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39634180",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7176213,\n   "lon": 37.4630906,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39645516",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7445097,\n   "lon": 37.5914331,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39645517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7448054,\n   "lon": 37.5900044,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39646432",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.736479,\n   "lon": 37.5900805,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39690107",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7336276,\n   "lon": 37.6220337,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39690157",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7358663,\n   "lon": 37.6175254,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39690189",\n   "kind": "building",\n   "name": "ГУП Мосгортранс",\n   "office_type": "government",\n   "lat": 55.7474933,\n   "lon": 37.6340396,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39851083",\n   "kind": "organisation",\n   "name": "НИИ глазных болезней",\n   "office_type": "research",\n   "lat": 55.7342062,\n   "lon": 37.5825734,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39854799",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7273443,\n   "lon": 37.5322276,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39894632",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7438258,\n   "lon": 37.5318013,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39898966",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7750702,\n   "lon": 37.8242781,\n   "opening_hours": null\n  },\n  {\n   "id": "way/39963178",\n   "kind": "building",\n   "name": "ДЕЗ Ивановское",\n   "office_type": null,\n   "lat": 55.7621254,\n   "lon": 37.8331024,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40008890",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6669091,\n   "lon": 37.5241686,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40009388",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7206396,\n   "lon": 37.6109232,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40032722",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7617754,\n   "lon": 37.8304128,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40314940",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8330496,\n   "lon": 37.5888156,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40398160",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7813387,\n   "lon": 37.6773118,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40398164",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7817317,\n   "lon": 37.6773211,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40431036",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8166423,\n   "lon": 37.5945291,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40444257",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.752569,\n   "lon": 37.6496442,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40444262",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7516108,\n   "lon": 37.6471158,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40444274",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7506295,\n   "lon": 37.6439131,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40444285",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7526974,\n   "lon": 37.6506442,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40444287",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7519133,\n   "lon": 37.6477014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40444297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.753014,\n   "lon": 37.6515776,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40445939",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7533254,\n   "lon": 37.6491496,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40445947",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7537388,\n   "lon": 37.6520717,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40445949",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7537908,\n   "lon": 37.6526789,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40445954",\n   "kind": "organisation",\n   "name": "Посольство Индии. Культурный центр имени Джавахарлала Неру",\n   "office_type": "diplomatic",\n   "lat": 55.754122,\n   "lon": 37.6535283,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40445989",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7553766,\n   "lon": 37.6501103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40445995",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7560883,\n   "lon": 37.6514974,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40446722",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7570197,\n   "lon": 37.6491439,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40446980",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.756994,\n   "lon": 37.6525669,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40451408",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7041583,\n   "lon": 37.7405014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40469057",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7571952,\n   "lon": 37.5686363,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40580991",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6430712,\n   "lon": 37.5238813,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40604727",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7909248,\n   "lon": 37.6297801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40604732",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7848908,\n   "lon": 37.6302422,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40604734",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7872261,\n   "lon": 37.6317806,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40687051",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8017595,\n   "lon": 37.4531783,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40693063",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8277227,\n   "lon": 37.5834953,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40693162",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8246758,\n   "lon": 37.5869555,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40694244",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8247146,\n   "lon": 37.5896,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40700054",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8214981,\n   "lon": 37.5842307,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40703530",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6847841,\n   "lon": 37.5602487,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40735575",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6884392,\n   "lon": 37.6018968,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40813032",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.716013,\n   "lon": 37.6152282,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859055",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7552994,\n   "lon": 37.6557976,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859056",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7552604,\n   "lon": 37.6553103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859059",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.754614,\n   "lon": 37.6544319,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859060",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7546653,\n   "lon": 37.6547135,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859066",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7561546,\n   "lon": 37.6553849,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859071",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.756244,\n   "lon": 37.6546029,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859072",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7560925,\n   "lon": 37.6541583,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859073",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7575767,\n   "lon": 37.6568993,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859075",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7586654,\n   "lon": 37.6552765,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859082",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7575572,\n   "lon": 37.6536138,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859427",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7586864,\n   "lon": 37.6538051,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859428",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588786,\n   "lon": 37.6537607,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859430",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7589155,\n   "lon": 37.6534235,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859431",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7586259,\n   "lon": 37.6534295,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859432",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588309,\n   "lon": 37.6532075,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859444",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7600546,\n   "lon": 37.6558955,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40859995",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7611079,\n   "lon": 37.6549804,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40950856",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.819062,\n   "lon": 37.5863722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40955105",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8120972,\n   "lon": 37.5929145,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40955106",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8116594,\n   "lon": 37.5930484,\n   "opening_hours": null\n  },\n  {\n   "id": "way/40999582",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8242229,\n   "lon": 37.6130902,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41084337",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619672,\n   "lon": 37.6523502,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41084338",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623635,\n   "lon": 37.6531755,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41183746",\n   "kind": "organisation",\n   "name": "Центральный экономико-математический институт РАН",\n   "office_type": "research",\n   "lat": 55.6747985,\n   "lon": 37.5677133,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41667042",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6767087,\n   "lon": 37.496411,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41667061",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6821786,\n   "lon": 37.4938168,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41667063",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6824684,\n   "lon": 37.4951707,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41854723",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7466413,\n   "lon": 37.4995902,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41854815",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7472476,\n   "lon": 37.5008776,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41863342",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7518428,\n   "lon": 37.8266921,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41863394",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7489028,\n   "lon": 37.8076466,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41863422",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7540982,\n   "lon": 37.8224957,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41874221",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7554262,\n   "lon": 37.829468,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41874222",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7551134,\n   "lon": 37.8296801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/41900540",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7479207,\n   "lon": 37.8029718,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42065515",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.72497,\n   "lon": 37.8016213,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42065516",\n   "kind": "building",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7255357,\n   "lon": 37.8026482,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42201643",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7071667,\n   "lon": 37.6259062,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42201650",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7064366,\n   "lon": 37.6250816,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42201654",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7068019,\n   "lon": 37.6255172,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42201656",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7065743,\n   "lon": 37.6259578,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42201661",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Center-T\\"",\n   "office_type": null,\n   "lat": 55.7076294,\n   "lon": 37.6253415,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42236139",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8724165,\n   "lon": 37.6793183,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42367060",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.885077,\n   "lon": 37.7084565,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42381112",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8554504,\n   "lon": 37.681134,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42437851",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8886686,\n   "lon": 37.5947686,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42437852",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8890082,\n   "lon": 37.5950743,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42454094",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8903339,\n   "lon": 37.6106128,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42455035",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8817952,\n   "lon": 37.6017634,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42474101",\n   "kind": "building",\n   "name": "Управа района Южное Медведково",\n   "office_type": "government",\n   "lat": 55.8756367,\n   "lon": 37.6197047,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42476203",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8943981,\n   "lon": 37.604055,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42477289",\n   "kind": "organisation",\n   "name": "Дирекция единого заказчика",\n   "office_type": "property_management",\n   "lat": 55.8954644,\n   "lon": 37.5949839,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42490813",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7197907,\n   "lon": 37.7275951,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42503273",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7622212,\n   "lon": 37.6477684,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42503284",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7644669,\n   "lon": 37.6448746,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42503306",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7614315,\n   "lon": 37.6470018,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42503307",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.761543,\n   "lon": 37.6467501,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42503313",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7618878,\n   "lon": 37.645067,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42503314",\n   "kind": "building",\n   "name": "Бизнес-центр «Бульварное Кольцо»",\n   "office_type": null,\n   "lat": 55.7620175,\n   "lon": 37.6455488,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42503341",\n   "kind": "building",\n   "name": "Академия детского мюзикла",\n   "office_type": null,\n   "lat": 55.762732,\n   "lon": 37.6463044,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42503342",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7625675,\n   "lon": 37.6464749,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42503401",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7609995,\n   "lon": 37.6499968,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42528933",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8754432,\n   "lon": 37.6225847,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42677056",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.708576,\n   "lon": 37.7274319,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42677090",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7047915,\n   "lon": 37.7257208,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42677096",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7043563,\n   "lon": 37.7252144,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42690788",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7032985,\n   "lon": 37.7446151,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42690869",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7035846,\n   "lon": 37.7438041,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42723619",\n   "kind": "organisation",\n   "name": "Центральная дирекция единого заказчика",\n   "office_type": "government",\n   "lat": 55.8928734,\n   "lon": 37.60245,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42766183",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7296659,\n   "lon": 37.6362534,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42806671",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7698173,\n   "lon": 37.6441964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42806672",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7693173,\n   "lon": 37.6454636,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42857988",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6608212,\n   "lon": 37.7099275,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42921011",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7831786,\n   "lon": 37.6769716,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42921013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7831697,\n   "lon": 37.678168,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42921014",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7828328,\n   "lon": 37.6777953,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42944570",\n   "kind": "building",\n   "name": "Торгово-деловой центр Гульден",\n   "office_type": null,\n   "lat": 55.8046943,\n   "lon": 37.5940552,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42944577",\n   "kind": "building",\n   "name": "Goodman",\n   "office_type": null,\n   "lat": 55.8049898,\n   "lon": 37.5899228,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42944584",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8058917,\n   "lon": 37.5923641,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42945342",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7967395,\n   "lon": 37.6044385,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42945393",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8099527,\n   "lon": 37.6029455,\n   "opening_hours": null\n  },\n  {\n   "id": "way/42987779",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7612436,\n   "lon": 37.7399444,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43129134",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.758646,\n   "lon": 37.6772304,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43175460",\n   "kind": "organisation",\n   "name": "Технопарк \\"Мосмедпарк\\"",\n   "office_type": "company",\n   "lat": 55.6528166,\n   "lon": 37.6968026,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43175461",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6536655,\n   "lon": 37.6948211,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43175463",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6537963,\n   "lon": 37.6952943,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43175471",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.651809,\n   "lon": 37.6963344,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43175473",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6522944,\n   "lon": 37.6985033,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43175474",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6525172,\n   "lon": 37.6973732,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43175486",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6516441,\n   "lon": 37.7000081,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43175491",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6531899,\n   "lon": 37.7006507,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43175502",\n   "kind": "building",\n   "name": "Приёмник-распределитель УВД ЮВАО",\n   "office_type": null,\n   "lat": 55.654212,\n   "lon": 37.7014893,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43201928",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6503957,\n   "lon": 37.6832568,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43202709",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6510263,\n   "lon": 37.6961464,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43302463",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6550959,\n   "lon": 37.5216894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43321798",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7539164,\n   "lon": 37.6679534,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43321811",\n   "kind": "organisation",\n   "name": "Федеральный центр детско-юношеского туризма и краеведения",\n   "office_type": "educational_institution",\n   "lat": 55.7523251,\n   "lon": 37.6754344,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43321812",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7513297,\n   "lon": 37.6749103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43401859",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8404346,\n   "lon": 37.6511852,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43513840",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7273694,\n   "lon": 37.7544284,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43539393",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7171261,\n   "lon": 37.6333992,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43683652",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7406857,\n   "lon": 37.6470951,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43683656",\n   "kind": "building",\n   "name": "Космическая связь",\n   "office_type": "telecommunication",\n   "lat": 55.7413049,\n   "lon": 37.6486069,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43684287",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7393743,\n   "lon": 37.6516487,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43684867",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7358675,\n   "lon": 37.653446,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43790719",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.700833,\n   "lon": 37.7396993,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43790749",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7035591,\n   "lon": 37.7363687,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43790807",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7023394,\n   "lon": 37.7363851,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43790817",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7028191,\n   "lon": 37.7359127,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43790851",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7032128,\n   "lon": 37.7356842,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43827572",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6360521,\n   "lon": 37.7964374,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43827740",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6395069,\n   "lon": 37.7897443,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43831137",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7014457,\n   "lon": 37.7465823,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43907792",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6896899,\n   "lon": 37.737112,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43909378",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7036686,\n   "lon": 37.7313053,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43909414",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7033694,\n   "lon": 37.7313764,\n   "opening_hours": null\n  },\n  {\n   "id": "way/43909612",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7034765,\n   "lon": 37.730982,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44139222",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8609682,\n   "lon": 37.686345,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44141483",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8611904,\n   "lon": 37.694558,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44146668",\n   "kind": "organisation",\n   "name": "Инженерная служба района Ярославский",\n   "office_type": "property_management",\n   "lat": 55.8697401,\n   "lon": 37.7129286,\n   "opening_hours": "08:00-17:00; 12:00-13:00 off"\n  },\n  {\n   "id": "way/44167828",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7994016,\n   "lon": 37.4614414,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44169066",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.790911,\n   "lon": 37.4830248,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44185734",\n   "kind": "building",\n   "name": "Балчуг-плаза",\n   "office_type": null,\n   "lat": 55.7467898,\n   "lon": 37.6266199,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44353019",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8158521,\n   "lon": 37.7045211,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44363973",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7393665,\n   "lon": 37.5385071,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44448154",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8004761,\n   "lon": 37.7718951,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44530189",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7541723,\n   "lon": 37.4915962,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44581783",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7896854,\n   "lon": 37.8111587,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44619619",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7460675,\n   "lon": 37.5115988,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44642920",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7562492,\n   "lon": 37.6132649,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44642922",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7481002,\n   "lon": 37.6225972,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44661134",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8677513,\n   "lon": 37.6671806,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44718594",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7874602,\n   "lon": 37.4993875,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44771613",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7794912,\n   "lon": 37.6678036,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44771617",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7795265,\n   "lon": 37.6710612,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44778324",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7797611,\n   "lon": 37.669875,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44778327",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.778652,\n   "lon": 37.6680124,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44794976",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7053529,\n   "lon": 37.6479054,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44847758",\n   "kind": "organisation",\n   "name": "Институт биологии развития им. Н. К. Кольцова",\n   "office_type": "research",\n   "lat": 55.7011452,\n   "lon": 37.576177,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44847759",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7013205,\n   "lon": 37.5757894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44848420",\n   "kind": "building",\n   "name": "Институт молекулярной биологии им. В. А. Энгельгардта",\n   "office_type": "research",\n   "lat": 55.6986426,\n   "lon": 37.5720374,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44848422",\n   "kind": "organisation",\n   "name": "Академснаб",\n   "office_type": "company",\n   "lat": 55.699338,\n   "lon": 37.5735468,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44848923",\n   "kind": "building",\n   "name": "Фосагро",\n   "office_type": "company",\n   "lat": 55.6972688,\n   "lon": 37.5634879,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44854817",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Очаково-Матвеевское",\n   "office_type": "government",\n   "lat": 55.6828469,\n   "lon": 37.4594369,\n   "opening_hours": null\n  },\n  {\n   "id": "way/44856822",\n   "kind": "organisation",\n   "name": "Сбербанк",\n   "office_type": "bank",\n   "lat": 55.6999651,\n   "lon": 37.5801207,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "way/45066504",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7846215,\n   "lon": 37.6421335,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45074997",\n   "kind": "organisation",\n   "name": "Институт общей генетики им. Н.И.Вавилова РАН",\n   "office_type": "research",\n   "lat": 55.696215,\n   "lon": 37.564363,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45075018",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6967831,\n   "lon": 37.5682114,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45105693",\n   "kind": "organisation",\n   "name": "ГУП \\"Московский метрополитен\\"",\n   "office_type": "company",\n   "lat": 55.7822403,\n   "lon": 37.6313411,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45111252",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7633618,\n   "lon": 37.6587047,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45112485",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.726175,\n   "lon": 37.5767457,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45112489",\n   "kind": "building",\n   "name": "Бизнес-центр «Аврора»",\n   "office_type": null,\n   "lat": 55.7353426,\n   "lon": 37.6421294,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45129978",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7629086,\n   "lon": 37.6530428,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45129990",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7613413,\n   "lon": 37.6506046,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45130012",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.761425,\n   "lon": 37.6502976,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45130526",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7640058,\n   "lon": 37.651826,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45243241",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7640513,\n   "lon": 37.681251,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45248659",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.640012,\n   "lon": 37.7990829,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45248697",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6400074,\n   "lon": 37.7996861,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45248749",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6400364,\n   "lon": 37.7942903,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45249091",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6380973,\n   "lon": 37.8009172,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45256542",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591901,\n   "lon": 37.6428394,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45257645",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7625836,\n   "lon": 37.6396424,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45257661",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7613543,\n   "lon": 37.6411455,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45257685",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7636038,\n   "lon": 37.6381174,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45257687",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7638147,\n   "lon": 37.6383618,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45257704",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626566,\n   "lon": 37.6377956,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45402019",\n   "kind": "building",\n   "name": "Бизнес-центр \\"На Харитоньевском\\"",\n   "office_type": null,\n   "lat": 55.765934,\n   "lon": 37.6505179,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45402028",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7642258,\n   "lon": 37.6488214,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45402046",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7641587,\n   "lon": 37.6423428,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45404836",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7344389,\n   "lon": 37.5913421,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45404855",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7320158,\n   "lon": 37.583615,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45404879",\n   "kind": "organisation",\n   "name": "Посольство Южной Кореи",\n   "office_type": "diplomatic",\n   "lat": 55.7379454,\n   "lon": 37.5751456,\n   "opening_hours": "Mo-Fr 09:00-12:30, 14:00-18:00"\n  },\n  {\n   "id": "way/45418204",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6517238,\n   "lon": 37.5313819,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463093",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7333743,\n   "lon": 37.5904669,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463099",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7330615,\n   "lon": 37.5909338,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463100",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7330515,\n   "lon": 37.5901719,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463102",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7325305,\n   "lon": 37.5900015,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463108",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7323227,\n   "lon": 37.590952,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463109",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7321974,\n   "lon": 37.590679,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463111",\n   "kind": "building",\n   "name": "Жилые палаты Хамовного двора",\n   "office_type": null,\n   "lat": 55.732551,\n   "lon": 37.589267,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463574",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7378436,\n   "lon": 37.5864759,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463578",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7366187,\n   "lon": 37.5881774,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463579",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7368422,\n   "lon": 37.5888523,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463581",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7358991,\n   "lon": 37.5871085,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463586",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7356829,\n   "lon": 37.5914953,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45463590",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7362364,\n   "lon": 37.5908748,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466579",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7358296,\n   "lon": 37.588677,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466581",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7352875,\n   "lon": 37.5885297,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466582",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7355413,\n   "lon": 37.5886498,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466584",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7363419,\n   "lon": 37.5897834,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466585",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7359754,\n   "lon": 37.5901913,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466586",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7357252,\n   "lon": 37.5902486,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466591",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7356535,\n   "lon": 37.5899145,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466596",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.731355,\n   "lon": 37.5874897,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466597",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7312007,\n   "lon": 37.5869105,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466600",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7311361,\n   "lon": 37.5872088,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466602",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7304212,\n   "lon": 37.5854003,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466608",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7300496,\n   "lon": 37.5841298,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466616",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7291585,\n   "lon": 37.5820759,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466624",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7288935,\n   "lon": 37.5815333,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466631",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7309545,\n   "lon": 37.5874504,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45466658",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт пивоваренной, безалкогольной и винодельческой промышленности",\n   "office_type": "research",\n   "lat": 55.735713,\n   "lon": 37.5827326,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45468554",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7318513,\n   "lon": 37.5885019,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45468555",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7317786,\n   "lon": 37.5880856,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45468556",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7319454,\n   "lon": 37.5850683,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45468569",\n   "kind": "organisation",\n   "name": "Посольство Вьетнама",\n   "office_type": "diplomatic",\n   "lat": 55.7341751,\n   "lon": 37.5776396,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45543320",\n   "kind": "building",\n   "name": "Префектура ЮАО",\n   "office_type": null,\n   "lat": 55.7059862,\n   "lon": 37.654067,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45744009",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.6998791,\n   "lon": 37.5616689,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45767511",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.764719,\n   "lon": 37.6691565,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45767512",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7648519,\n   "lon": 37.669576,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45767516",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7647389,\n   "lon": 37.670546,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45768169",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7563289,\n   "lon": 37.6342858,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45773140",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8216601,\n   "lon": 37.4999341,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45777820",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591776,\n   "lon": 37.6397623,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45778224",\n   "kind": "building",\n   "name": "Бизнес-центр «Яковоапостольский»",\n   "office_type": null,\n   "lat": 55.7588717,\n   "lon": 37.6563388,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45779329",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8394815,\n   "lon": 37.5428429,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45780008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.766915,\n   "lon": 37.6544113,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45781692",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.842695,\n   "lon": 37.5343041,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45782178",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7818574,\n   "lon": 37.6779483,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45782708",\n   "kind": "building",\n   "name": "Пенсионный фонд Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7177419,\n   "lon": 37.6201497,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45818759",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7396598,\n   "lon": 37.5916506,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45818762",\n   "kind": "organisation",\n   "name": "Посольство Сирии",\n   "office_type": "diplomatic",\n   "lat": 55.7398527,\n   "lon": 37.5952434,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45818763",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7402472,\n   "lon": 37.5964729,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45819339",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7384674,\n   "lon": 37.5913056,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45819341",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.738673,\n   "lon": 37.590799,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45819752",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.713373,\n   "lon": 37.5688461,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45819754",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7159372,\n   "lon": 37.5687457,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45819755",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7162644,\n   "lon": 37.5709696,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45819989",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7099545,\n   "lon": 37.6009204,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45819990",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7092269,\n   "lon": 37.6025072,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45820297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7156324,\n   "lon": 37.6139255,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45820299",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7161636,\n   "lon": 37.6145919,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45823390",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7303811,\n   "lon": 37.7152964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45826025",\n   "kind": "building",\n   "name": "ФГУП \\"Госземкадастрсъемка\\"-ВИСХАГИ",\n   "office_type": null,\n   "lat": 55.7291036,\n   "lon": 37.7201221,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45827900",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.729379,\n   "lon": 37.7234831,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45829255",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.781942,\n   "lon": 37.6383587,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45829258",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7815857,\n   "lon": 37.6382425,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45829280",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7340896,\n   "lon": 37.7054122,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45829284",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7339387,\n   "lon": 37.7057554,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45830772",\n   "kind": "building",\n   "name": "Управа района \\"Гольяново\\"",\n   "office_type": "government",\n   "lat": 55.8294745,\n   "lon": 37.8198946,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45831631",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7658979,\n   "lon": 37.6763586,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45831773",\n   "kind": "organisation",\n   "name": "Московский центр качества образования - офис №1",\n   "office_type": "government",\n   "lat": 55.7812335,\n   "lon": 37.7211203,\n   "opening_hours": "Mo-Fr 09:00-17:30"\n  },\n  {\n   "id": "way/45831961",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7813012,\n   "lon": 37.7155512,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45831963",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7833675,\n   "lon": 37.7245384,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45832828",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626486,\n   "lon": 37.7367026,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45834934",\n   "kind": "organisation",\n   "name": "Московский Эндокринный Завод",\n   "office_type": "company",\n   "lat": 55.7264274,\n   "lon": 37.7175748,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45848071",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7105703,\n   "lon": 37.6249298,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45848072",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7105031,\n   "lon": 37.6238398,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45848075",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7111978,\n   "lon": 37.6251817,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45848078",\n   "kind": "building",\n   "name": "АРТ-Информ Экспресс",\n   "office_type": "company",\n   "lat": 55.7105756,\n   "lon": 37.6258984,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45848079",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7099863,\n   "lon": 37.6252362,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45848495",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7137619,\n   "lon": 37.6124668,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45860703",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7683462,\n   "lon": 37.6435266,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45861093",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7711586,\n   "lon": 37.6330155,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45861126",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7697752,\n   "lon": 37.6341759,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45861127",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7697096,\n   "lon": 37.6338786,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45892322",\n   "kind": "organisation",\n   "name": "Аналитический центр при Правительстве Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7691905,\n   "lon": 37.6432296,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45900447",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.734635,\n   "lon": 37.7114181,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45903938",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7714708,\n   "lon": 37.6146435,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45903939",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.771375,\n   "lon": 37.6152396,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45903942",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7716561,\n   "lon": 37.6150653,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45903949",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7716983,\n   "lon": 37.6131823,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45903952",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7716552,\n   "lon": 37.6125782,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45903957",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7724776,\n   "lon": 37.6147228,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45903960",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7726569,\n   "lon": 37.6149061,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45903961",\n   "kind": "organisation",\n   "name": "Институт проблем передачи информации им. А.А. Харкевича",\n   "office_type": "research",\n   "lat": 55.7725963,\n   "lon": 37.6158284,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45903967",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7726564,\n   "lon": 37.6125681,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45942208",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6028098,\n   "lon": 37.5280932,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45980575",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "telecommunication",\n   "lat": 55.6212223,\n   "lon": 37.745609,\n   "opening_hours": null\n  },\n  {\n   "id": "way/45981226",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Зябликово",\n   "office_type": "government",\n   "lat": 55.6228643,\n   "lon": 37.7441247,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "way/46094867",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7193858,\n   "lon": 37.6122758,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46094874",\n   "kind": "building",\n   "name": "Федеральная служба по надзору в сфере образования и науки",\n   "office_type": "government",\n   "lat": 55.7193994,\n   "lon": 37.610051,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46094878",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7182118,\n   "lon": 37.610105,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46096539",\n   "kind": "building",\n   "name": "Мосгорстат",\n   "office_type": "government",\n   "lat": 55.7554808,\n   "lon": 37.6472382,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46096541",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7579693,\n   "lon": 37.6457405,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46178375",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6587849,\n   "lon": 37.5566944,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46178378",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6587212,\n   "lon": 37.5558206,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46178379",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6582197,\n   "lon": 37.555959,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46178380",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6589606,\n   "lon": 37.5579849,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46178381",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.658487,\n   "lon": 37.5574493,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46277336",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7280347,\n   "lon": 37.5704713,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46277338",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7288152,\n   "lon": 37.5704317,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46277367",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7229377,\n   "lon": 37.5608597,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46277368",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7228523,\n   "lon": 37.5615207,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46277369",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7230801,\n   "lon": 37.5616639,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46277374",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7203284,\n   "lon": 37.5698977,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46277376",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.722711,\n   "lon": 37.5716561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46278272",\n   "kind": "building",\n   "name": "МФИ софт",\n   "office_type": "it",\n   "lat": 55.7284633,\n   "lon": 37.5854568,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46278334",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7249972,\n   "lon": 37.5845872,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353933",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7642908,\n   "lon": 37.5683914,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353934",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7641205,\n   "lon": 37.5689515,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353945",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7645993,\n   "lon": 37.5734326,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353948",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7631496,\n   "lon": 37.5701491,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353950",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Сибирь\\"",\n   "office_type": null,\n   "lat": 55.762325,\n   "lon": 37.569898,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353954",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7628891,\n   "lon": 37.5681177,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353960",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.763616,\n   "lon": 37.5678027,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353963",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7633752,\n   "lon": 37.5691334,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353964",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7635884,\n   "lon": 37.5696018,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353965",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7647549,\n   "lon": 37.571578,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46353971",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7633247,\n   "lon": 37.5723143,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46414092",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7871314,\n   "lon": 37.6055456,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46414098",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.785602,\n   "lon": 37.6026584,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46414205",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7894349,\n   "lon": 37.611781,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46414225",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7908641,\n   "lon": 37.6094095,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46414229",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7914905,\n   "lon": 37.6088783,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46414232",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7917941,\n   "lon": 37.6084124,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46414240",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7922322,\n   "lon": 37.6093192,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46414241",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7919987,\n   "lon": 37.6092865,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46414248",\n   "kind": "building",\n   "name": "Отдел ФСБ по СВАО",\n   "office_type": "government",\n   "lat": 55.7890443,\n   "lon": 37.6141798,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46505053",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7568902,\n   "lon": 37.5873409,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46512994",\n   "kind": "building",\n   "name": "Палаты Троекуровых",\n   "office_type": null,\n   "lat": 55.7585053,\n   "lon": 37.6157455,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46513008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7607693,\n   "lon": 37.6143608,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46513026",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7608286,\n   "lon": 37.6147373,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46521826",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8026502,\n   "lon": 37.4913656,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46567583",\n   "kind": "building",\n   "name": "Институт геологии рудных месторождений, петрографии, минералогии и геохимии РАН",\n   "office_type": "research",\n   "lat": 55.7374904,\n   "lon": 37.6210911,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46567602",\n   "kind": "building",\n   "name": "Бизнес-центр \\"На Ордынке\\"",\n   "office_type": null,\n   "lat": 55.7351616,\n   "lon": 37.6239962,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46567642",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7351396,\n   "lon": 37.6222873,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46567647",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7351512,\n   "lon": 37.6227833,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46570571",\n   "kind": "organisation",\n   "name": "Управление Федерального казначейства по Москве",\n   "office_type": "government",\n   "lat": 55.7076843,\n   "lon": 37.6149663,\n   "opening_hours": "Mo-Fr 09:00-13:00"\n  },\n  {\n   "id": "way/46581054",\n   "kind": "building",\n   "name": "РОДП \\"Яблоко\\"",\n   "office_type": "political_party",\n   "lat": 55.7403824,\n   "lon": 37.6288058,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46581055",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.739953,\n   "lon": 37.6286841,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46603828",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7838284,\n   "lon": 37.5997015,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46603838",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7837589,\n   "lon": 37.6003683,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46604049",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7030991,\n   "lon": 37.5643676,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46615634",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8395362,\n   "lon": 37.6358226,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46620303",\n   "kind": "organisation",\n   "name": "Центр «Биоинженерия» РАН",\n   "office_type": "research",\n   "lat": 55.6995045,\n   "lon": 37.5818617,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46620304",\n   "kind": "organisation",\n   "name": "Институт микробиологии им. С.Н. Виноградского РАН",\n   "office_type": "research",\n   "lat": 55.6989331,\n   "lon": 37.581648,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46657148",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6846472,\n   "lon": 37.5544813,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46657149",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6844513,\n   "lon": 37.5549511,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46657649",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6919556,\n   "lon": 37.5583315,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46661628",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.779776,\n   "lon": 37.6892961,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46737429",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.784584,\n   "lon": 37.5978604,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46737470",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7901086,\n   "lon": 37.5916631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46737486",\n   "kind": "organisation",\n   "name": "Апостольская Нунциатура",\n   "office_type": "diplomatic",\n   "lat": 55.790897,\n   "lon": 37.5994036,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46738282",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7881364,\n   "lon": 37.5889125,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46738291",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7890511,\n   "lon": 37.5887465,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46739091",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7821596,\n   "lon": 37.5927614,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46739092",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7821206,\n   "lon": 37.5925108,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46739093",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7821151,\n   "lon": 37.5919751,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46739099",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819656,\n   "lon": 37.5928429,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740665",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7815605,\n   "lon": 37.6083299,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740669",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7813247,\n   "lon": 37.6084942,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740674",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7821249,\n   "lon": 37.6078016,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740675",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7823192,\n   "lon": 37.6075611,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740679",\n   "kind": "building",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.7827525,\n   "lon": 37.6086143,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740695",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7834911,\n   "lon": 37.6089146,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740699",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7836012,\n   "lon": 37.608165,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740700",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7833538,\n   "lon": 37.6085848,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740701",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7837803,\n   "lon": 37.6079654,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740714",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7860938,\n   "lon": 37.6093519,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740716",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7864697,\n   "lon": 37.6092975,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46740725",\n   "kind": "organisation",\n   "name": "Прокуратура СВАО",\n   "office_type": "government",\n   "lat": 55.7851241,\n   "lon": 37.6126328,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837201",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7702249,\n   "lon": 37.6144255,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837203",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7710788,\n   "lon": 37.6143801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837204",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7708758,\n   "lon": 37.6144728,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837205",\n   "kind": "building",\n   "name": "Управление ЗАГС по Московской области",\n   "office_type": "government",\n   "lat": 55.7708341,\n   "lon": 37.6140512,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837206",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7707663,\n   "lon": 37.6136308,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837207",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7696213,\n   "lon": 37.6139783,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837208",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7710849,\n   "lon": 37.6135985,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837213",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7699246,\n   "lon": 37.6152978,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837222",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7722,\n   "lon": 37.619494,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837243",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7678503,\n   "lon": 37.6242977,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46837245",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Миллениум Хаус\\"",\n   "office_type": null,\n   "lat": 55.7683624,\n   "lon": 37.6248941,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46843803",\n   "kind": "building",\n   "name": "Центральное территориальное управление имущественных отношений Минобороны РФ",\n   "office_type": "government",\n   "lat": 55.7767235,\n   "lon": 37.5396355,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46846646",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.768094,\n   "lon": 37.609599,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46846696",\n   "kind": "organisation",\n   "name": "Объединение \\"Росинкас\\"",\n   "office_type": "company",\n   "lat": 55.7689689,\n   "lon": 37.6193279,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46846721",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.767799,\n   "lon": 37.6278093,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46854271",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7852892,\n   "lon": 37.615155,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46854290",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7858961,\n   "lon": 37.6249461,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46871650",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7693414,\n   "lon": 37.5674912,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46874402",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7617873,\n   "lon": 37.5738111,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46891143",\n   "kind": "organisation",\n   "name": "Мособлгаз",\n   "office_type": "yes",\n   "lat": 55.8981395,\n   "lon": 37.4458511,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46915602",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.78598,\n   "lon": 37.5590876,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46916650",\n   "kind": "building",\n   "name": "Московская городская военная прокуратура",\n   "office_type": null,\n   "lat": 55.7813479,\n   "lon": 37.5386924,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46931798",\n   "kind": "building",\n   "name": "Особняк А. К. Ферстер — Михельсона",\n   "office_type": null,\n   "lat": 55.7588543,\n   "lon": 37.5903179,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46932109",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7586993,\n   "lon": 37.588911,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46932126",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7590342,\n   "lon": 37.5889583,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46934098",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7253778,\n   "lon": 37.6231304,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46946857",\n   "kind": "organisation",\n   "name": "Прокуратура Северного АО",\n   "office_type": "government",\n   "lat": 55.7899662,\n   "lon": 37.5095187,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46952709",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.761203,\n   "lon": 37.5794446,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46961012",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623213,\n   "lon": 37.5801921,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46963118",\n   "kind": "building",\n   "name": "БЦ \\"Банный 9\\"",\n   "office_type": null,\n   "lat": 55.7861824,\n   "lon": 37.6391876,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46963586",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7877458,\n   "lon": 37.6364331,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46965453",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7632915,\n   "lon": 37.5886236,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46965811",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7657059,\n   "lon": 37.5913209,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46966527",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7523962,\n   "lon": 37.6055049,\n   "opening_hours": null\n  },\n  {\n   "id": "way/46966552",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7525279,\n   "lon": 37.606458,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47030995",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7602326,\n   "lon": 37.5875131,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47031034",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7584507,\n   "lon": 37.5859376,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47031092",\n   "kind": "building",\n   "name": "Городская усадьба Долгоруких",\n   "office_type": null,\n   "lat": 55.7582248,\n   "lon": 37.5869251,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47031131",\n   "kind": "building",\n   "name": "Центральный дом литераторов",\n   "office_type": null,\n   "lat": 55.7579478,\n   "lon": 37.5877696,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47031147",\n   "kind": "building",\n   "name": "Посольство Бразилии",\n   "office_type": "diplomatic",\n   "lat": 55.7583643,\n   "lon": 37.5879221,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47031231",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7581326,\n   "lon": 37.5892386,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47031428",\n   "kind": "building",\n   "name": "Посольство Танзании",\n   "office_type": "diplomatic",\n   "lat": 55.7578561,\n   "lon": 37.5885234,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47031455",\n   "kind": "building",\n   "name": "Посольство Египта",\n   "office_type": "diplomatic",\n   "lat": 55.7585603,\n   "lon": 37.5870368,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47066458",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7957193,\n   "lon": 37.7014734,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47066460",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Сокольники",\n   "office_type": "government",\n   "lat": 55.7895824,\n   "lon": 37.6831089,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "way/47067456",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7752567,\n   "lon": 37.7088662,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47138743",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7649075,\n   "lon": 37.6154085,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47186447",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7510647,\n   "lon": 37.5822888,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47191476",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7171075,\n   "lon": 37.5971028,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47191477",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7175452,\n   "lon": 37.5975878,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47215757",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7549698,\n   "lon": 37.5665605,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47216747",\n   "kind": "organisation",\n   "name": "Представительство администрации Брянской области при правительстве Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7548684,\n   "lon": 37.5654094,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47217906",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7468588,\n   "lon": 37.5690809,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47218702",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7474941,\n   "lon": 37.5834362,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47219251",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7480531,\n   "lon": 37.5841057,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47219587",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.748181,\n   "lon": 37.5856996,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47219979",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461832,\n   "lon": 37.5856512,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47220182",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7454033,\n   "lon": 37.5881924,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47270476",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7602792,\n   "lon": 37.5664857,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47290813",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7580788,\n   "lon": 37.5726467,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47291006",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7597345,\n   "lon": 37.568808,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47291017",\n   "kind": "building",\n   "name": "Миусский телефонный узел",\n   "office_type": null,\n   "lat": 55.7597393,\n   "lon": 37.5713295,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47291923",\n   "kind": "organisation",\n   "name": "Резиденция посла Мексики",\n   "office_type": "diplomatic",\n   "lat": 55.750524,\n   "lon": 37.5900871,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47294368",\n   "kind": "organisation",\n   "name": "Посольство Норвегии",\n   "office_type": "diplomatic",\n   "lat": 55.7537677,\n   "lon": 37.5962674,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47294691",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7545655,\n   "lon": 37.598159,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47294701",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7546522,\n   "lon": 37.5976754,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47296208",\n   "kind": "organisation",\n   "name": "ФГКУ \\"ВНИИ МВД России\\"",\n   "office_type": "research",\n   "lat": 55.7557881,\n   "lon": 37.5898885,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47296531",\n   "kind": "organisation",\n   "name": "Посольство Афганистана",\n   "office_type": "diplomatic",\n   "lat": 55.7566671,\n   "lon": 37.5892292,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47296549",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7570367,\n   "lon": 37.5882632,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47359221",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7486332,\n   "lon": 37.5854401,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47359563",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.74871,\n   "lon": 37.5868449,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47359873",\n   "kind": "organisation",\n   "name": "Посольство Сингапура",\n   "office_type": "diplomatic",\n   "lat": 55.7499008,\n   "lon": 37.5855907,\n   "opening_hours": "Mo-Fr 09:00-13:00, 14:00-17:30"\n  },\n  {\n   "id": "way/47360141",\n   "kind": "organisation",\n   "name": "Посольство Филиппин",\n   "office_type": "diplomatic",\n   "lat": 55.7494976,\n   "lon": 37.5857077,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "way/47387343",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7557815,\n   "lon": 37.5869462,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47387434",\n   "kind": "building",\n   "name": "Территориальный центр социального обслуживания Арбат",\n   "office_type": null,\n   "lat": 55.7555098,\n   "lon": 37.5868505,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47388013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7575697,\n   "lon": 37.5874093,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47388048",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7573077,\n   "lon": 37.5876049,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47486389",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7377764,\n   "lon": 37.6605737,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47507780",\n   "kind": "organisation",\n   "name": "Посольство Канады",\n   "office_type": "diplomatic",\n   "lat": 55.7467661,\n   "lon": 37.5949379,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47511245",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.741425,\n   "lon": 37.5826092,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47511261",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7408179,\n   "lon": 37.5824766,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47511408",\n   "kind": "organisation",\n   "name": "Министерство Иностранных Дел РФ, Консульский департамент",\n   "office_type": "government",\n   "lat": 55.7412161,\n   "lon": 37.5796507,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47512357",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7423955,\n   "lon": 37.5858059,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47512393",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432419,\n   "lon": 37.5854757,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47512416",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7434142,\n   "lon": 37.5858793,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47512468",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7435998,\n   "lon": 37.5865988,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47513223",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7430395,\n   "lon": 37.5847155,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47513349",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7434612,\n   "lon": 37.5873512,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47513393",\n   "kind": "building",\n   "name": "посольство Италии",\n   "office_type": "diplomatic",\n   "lat": 55.7431668,\n   "lon": 37.5875687,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47572010",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.732426,\n   "lon": 37.6626512,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47575424",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7445798,\n   "lon": 37.5799519,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47575466",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7446159,\n   "lon": 37.5816241,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47575483",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7444684,\n   "lon": 37.5821569,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47575510",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.74431,\n   "lon": 37.5818639,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47578513",\n   "kind": "building",\n   "name": "посольство Палестины",\n   "office_type": "diplomatic",\n   "lat": 55.7414811,\n   "lon": 37.5877187,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47584266",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7436277,\n   "lon": 37.5889391,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47584795",\n   "kind": "organisation",\n   "name": "посольство Дании",\n   "office_type": "diplomatic",\n   "lat": 55.7434681,\n   "lon": 37.5919805,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47585045",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7427643,\n   "lon": 37.5957658,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47623851",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7397518,\n   "lon": 37.5863509,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47625595",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7392709,\n   "lon": 37.5867819,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47625603",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Мосаларко Плаза Один\\"",\n   "office_type": null,\n   "lat": 55.7390399,\n   "lon": 37.6586935,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47625731",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7401515,\n   "lon": 37.5881526,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47626380",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7389991,\n   "lon": 37.6579988,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47626390",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7400737,\n   "lon": 37.5890733,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642614",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7710721,\n   "lon": 37.6224197,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642615",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7714986,\n   "lon": 37.6223608,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642616",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Инженер\\"",\n   "office_type": null,\n   "lat": 55.7706656,\n   "lon": 37.6249357,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642619",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7706886,\n   "lon": 37.6243731,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642620",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7704681,\n   "lon": 37.6232681,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642632",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7713146,\n   "lon": 37.624494,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642636",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7709193,\n   "lon": 37.6250924,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642638",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7712962,\n   "lon": 37.6249467,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642644",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7732563,\n   "lon": 37.6220567,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "way/47642645",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7725433,\n   "lon": 37.6221237,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642646",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.772866,\n   "lon": 37.6220365,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47642666",\n   "kind": "building",\n   "name": "Рособрнадзор",\n   "office_type": "government",\n   "lat": 55.7724129,\n   "lon": 37.6277348,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47680372",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7638936,\n   "lon": 37.6022105,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47680646",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7639955,\n   "lon": 37.6010038,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47682979",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7644886,\n   "lon": 37.5990735,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47703711",\n   "kind": "building",\n   "name": "Бизнес-центр \\"На Трубной\\"",\n   "office_type": null,\n   "lat": 55.771257,\n   "lon": 37.6255567,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47703712",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7709883,\n   "lon": 37.6255065,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47703717",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7707448,\n   "lon": 37.6278267,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47703722",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7711728,\n   "lon": 37.6260149,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47703730",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7708442,\n   "lon": 37.6301374,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47703731",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7708238,\n   "lon": 37.6297625,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47704671",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7727066,\n   "lon": 37.6282348,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47704677",\n   "kind": "building",\n   "name": "Офисный центр «Ян-Рон»",\n   "office_type": null,\n   "lat": 55.7700666,\n   "lon": 37.6272299,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47705393",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7695045,\n   "lon": 37.6281129,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47705406",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7695547,\n   "lon": 37.6288531,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47705407",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7695478,\n   "lon": 37.6293159,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47705410",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7698122,\n   "lon": 37.6291115,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47705411",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7697952,\n   "lon": 37.62845,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47705412",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7695915,\n   "lon": 37.629858,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47705417",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7698166,\n   "lon": 37.6314274,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47707919",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7080433,\n   "lon": 37.8302863,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47846239",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.768587,\n   "lon": 37.5963732,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47916237",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7450784,\n   "lon": 37.5990987,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47916253",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7451602,\n   "lon": 37.5998917,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47916362",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7458529,\n   "lon": 37.5987282,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47917378",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7463113,\n   "lon": 37.5975931,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47918612",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7468717,\n   "lon": 37.5979685,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47918653",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7468703,\n   "lon": 37.5976405,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47922021",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7556661,\n   "lon": 37.5855322,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47924407",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7397513,\n   "lon": 37.6013047,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47925526",\n   "kind": "building",\n   "name": "Управление по образованию",\n   "office_type": "government",\n   "lat": 55.8930728,\n   "lon": 37.4433681,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47941894",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8100388,\n   "lon": 37.8322688,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47973329",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7657271,\n   "lon": 37.6086363,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47973330",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Дмитровка, 23\\"",\n   "office_type": null,\n   "lat": 55.7659718,\n   "lon": 37.6093513,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47973338",\n   "kind": "building",\n   "name": "Государственный институт искусствознания",\n   "office_type": "research",\n   "lat": 55.7650062,\n   "lon": 37.6094014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47978250",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.740758,\n   "lon": 37.5989608,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47978315",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7414214,\n   "lon": 37.5960528,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47978474",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7410989,\n   "lon": 37.5961023,\n   "opening_hours": null\n  },\n  {\n   "id": "way/47978481",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7415114,\n   "lon": 37.5957631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48014531",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7333685,\n   "lon": 37.6185493,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48014653",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7344541,\n   "lon": 37.6154294,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48014890",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7319725,\n   "lon": 37.6137471,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48014936",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7316962,\n   "lon": 37.6174363,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48014960",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7318422,\n   "lon": 37.6182629,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48017063",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6481201,\n   "lon": 37.7331703,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48041839",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6454097,\n   "lon": 37.7312456,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48044632",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8510347,\n   "lon": 37.447103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48056959",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7264619,\n   "lon": 37.6150048,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48068564",\n   "kind": "building",\n   "name": "Бизнес-центр «Лиман»",\n   "office_type": null,\n   "lat": 55.669584,\n   "lon": 37.5466502,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48122553",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6666922,\n   "lon": 37.7412702,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48122564",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6657933,\n   "lon": 37.7416032,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48123597",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6155028,\n   "lon": 37.5363886,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48134860",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.726401,\n   "lon": 37.6260749,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48134905",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7264382,\n   "lon": 37.6264384,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48135360",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6751177,\n   "lon": 37.5810561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48136223",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6233476,\n   "lon": 37.6686457,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48136225",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6218097,\n   "lon": 37.6704317,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48136242",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6437701,\n   "lon": 37.669955,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48138364",\n   "kind": "organisation",\n   "name": "Страховая компания «МАКС»",\n   "office_type": "insurance",\n   "lat": 55.641374,\n   "lon": 37.6816963,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48164005",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6065982,\n   "lon": 37.5478741,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48164662",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7794306,\n   "lon": 37.6879356,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48172598",\n   "kind": "organisation",\n   "name": "Посольство Аргентины",\n   "office_type": "diplomatic",\n   "lat": 55.7309764,\n   "lon": 37.6237436,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48172727",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7325689,\n   "lon": 37.6238063,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48172847",\n   "kind": "organisation",\n   "name": "Посольство Кыргызской Республики",\n   "office_type": "diplomatic",\n   "lat": 55.7323177,\n   "lon": 37.6239193,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48172859",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Domino\\"",\n   "office_type": null,\n   "lat": 55.7315425,\n   "lon": 37.6249673,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48172988",\n   "kind": "organisation",\n   "name": "Учебный центр Микроинформ",\n   "office_type": "company",\n   "lat": 55.7324833,\n   "lon": 37.6251767,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48174162",\n   "kind": "organisation",\n   "name": "ОСЗН района Ясенево",\n   "office_type": "government",\n   "lat": 55.5964108,\n   "lon": 37.5417582,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48232564",\n   "kind": "building",\n   "name": "Бизнес-центр Диапазон",\n   "office_type": "yes",\n   "lat": 55.8046159,\n   "lon": 37.4903763,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48232565",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8045594,\n   "lon": 37.49128,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48232566",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8037323,\n   "lon": 37.4905083,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48236691",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7410081,\n   "lon": 37.6179253,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48244044",\n   "kind": "building",\n   "name": "Бизнес-центр «Алтуфьевский»",\n   "office_type": null,\n   "lat": 55.8761852,\n   "lon": 37.588836,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48274534",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7736915,\n   "lon": 37.6813401,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48287957",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7358195,\n   "lon": 37.6309099,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48288174",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7339187,\n   "lon": 37.6357191,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48288204",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7343473,\n   "lon": 37.6353041,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48288333",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7315035,\n   "lon": 37.6369849,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48296120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7936964,\n   "lon": 37.6916726,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48296123",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7943683,\n   "lon": 37.6927868,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48302992",\n   "kind": "building",\n   "name": "Московский Радиозавод",\n   "office_type": null,\n   "lat": 55.7374932,\n   "lon": 37.6354663,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48303271",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7387157,\n   "lon": 37.6404098,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48303289",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7381085,\n   "lon": 37.6405307,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48303676",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7368898,\n   "lon": 37.6425112,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48303679",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7374832,\n   "lon": 37.6426211,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48303681",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7375872,\n   "lon": 37.6422241,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346029",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7288709,\n   "lon": 37.6474189,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346030",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7278673,\n   "lon": 37.6477888,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346143",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7272776,\n   "lon": 37.6470662,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346144",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7279386,\n   "lon": 37.6503472,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346156",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.728543,\n   "lon": 37.6477321,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346200",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7294907,\n   "lon": 37.648443,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346201",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.729778,\n   "lon": 37.6472905,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346307",\n   "kind": "organisation",\n   "name": "BBDO",\n   "office_type": "advertising_agency",\n   "lat": 55.7236986,\n   "lon": 37.652085,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346313",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7258519,\n   "lon": 37.6496357,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48346402",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7292494,\n   "lon": 37.6576678,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48380726",\n   "kind": "building",\n   "name": "Бизнес-центр \\"На Газетном\\"",\n   "office_type": null,\n   "lat": 55.758955,\n   "lon": 37.6105625,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48399932",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7185626,\n   "lon": 37.5980919,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48399949",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7190444,\n   "lon": 37.6016188,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48400215",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7181865,\n   "lon": 37.6055261,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48400331",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7152814,\n   "lon": 37.6165945,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48400598",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7147894,\n   "lon": 37.6050276,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48400620",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7129401,\n   "lon": 37.5980699,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48401008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7078179,\n   "lon": 37.6082707,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48401017",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.707901,\n   "lon": 37.6086784,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48401043",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7071753,\n   "lon": 37.6102944,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48417405",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7681732,\n   "lon": 37.5959544,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48420624",\n   "kind": "building",\n   "name": "НИИ эпидемиологии и микробиологии им. Г.Н. Габричевского \\"",\n   "office_type": "research",\n   "lat": 55.8374871,\n   "lon": 37.4894239,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48461230",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7169254,\n   "lon": 37.6288534,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48504331",\n   "kind": "building",\n   "name": "Росстандарт",\n   "office_type": "government",\n   "lat": 55.7241178,\n   "lon": 37.6055207,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "way/48504405",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7251647,\n   "lon": 37.6106386,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48560525",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.708477,\n   "lon": 37.507232,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582102",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7859726,\n   "lon": 37.7047989,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582126",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.785548,\n   "lon": 37.7099311,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582135",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7865075,\n   "lon": 37.7077633,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582140",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7881477,\n   "lon": 37.7067615,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582143",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7867492,\n   "lon": 37.7094224,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582162",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7890465,\n   "lon": 37.7069914,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582163",\n   "kind": "building",\n   "name": "Административно-деловой центр \\"Преображенский\\"",\n   "office_type": null,\n   "lat": 55.7889205,\n   "lon": 37.7080154,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582165",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7889483,\n   "lon": 37.7076648,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582166",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7897373,\n   "lon": 37.707381,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582172",\n   "kind": "building",\n   "name": "Alstom Grid",\n   "office_type": "company",\n   "lat": 55.7902106,\n   "lon": 37.7077123,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582174",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7895357,\n   "lon": 37.7094723,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582176",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7925399,\n   "lon": 37.7099039,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582179",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7900735,\n   "lon": 37.7130224,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582184",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7893567,\n   "lon": 37.7112897,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582185",\n   "kind": "organisation",\n   "name": "Преображенская межрайонная прокуратура г. Москвы",\n   "office_type": "government",\n   "lat": 55.7905394,\n   "lon": 37.7126985,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582186",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.790348,\n   "lon": 37.7116401,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48582193",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7926244,\n   "lon": 37.7136505,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585124",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7920538,\n   "lon": 37.6510411,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585134",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7863686,\n   "lon": 37.6595152,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585172",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7936808,\n   "lon": 37.6561932,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585175",\n   "kind": "building",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7912748,\n   "lon": 37.6543345,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585189",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7876736,\n   "lon": 37.672058,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585194",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7869148,\n   "lon": 37.6717287,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585197",\n   "kind": "building",\n   "name": "БЦ \\"На Шумкина\\"",\n   "office_type": null,\n   "lat": 55.7872269,\n   "lon": 37.6708366,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585219",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7883032,\n   "lon": 37.6623539,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585248",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7892231,\n   "lon": 37.6647978,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585258",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7908198,\n   "lon": 37.6590803,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48585341",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7907421,\n   "lon": 37.6610944,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48610526",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт холодильной промышленности",\n   "office_type": "research",\n   "lat": 55.8163331,\n   "lon": 37.5711538,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48637709",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6984525,\n   "lon": 37.7547256,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48648090",\n   "kind": "organisation",\n   "name": "РусТрансКом",\n   "office_type": "company",\n   "lat": 55.7830696,\n   "lon": 37.6624747,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48648098",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7827537,\n   "lon": 37.6580773,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48668514",\n   "kind": "organisation",\n   "name": "Инспекция ФНС России № 14",\n   "office_type": "government",\n   "lat": 55.7825121,\n   "lon": 37.5548953,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:45"\n  },\n  {\n   "id": "way/48673212",\n   "kind": "building",\n   "name": "Атом-охрана",\n   "office_type": null,\n   "lat": 55.7904109,\n   "lon": 37.4800676,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48673226",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.788832,\n   "lon": 37.4752283,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48700745",\n   "kind": "building",\n   "name": "",\n   "office_type": "telecommunication",\n   "lat": 55.7811275,\n   "lon": 37.7087128,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48702288",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7636684,\n   "lon": 37.5739069,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48719859",\n   "kind": "organisation",\n   "name": "Московская объединенная энергетическая компания",\n   "office_type": "company",\n   "lat": 55.7556423,\n   "lon": 37.7513823,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48752298",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6782329,\n   "lon": 37.6899672,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48752535",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6799996,\n   "lon": 37.6927192,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48830577",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.791786,\n   "lon": 37.8159511,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48840538",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7015883,\n   "lon": 37.6299255,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48840552",\n   "kind": "building",\n   "name": "Вторая проходная",\n   "office_type": null,\n   "lat": 55.703451,\n   "lon": 37.6419537,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48841547",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7034806,\n   "lon": 37.639976,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48841570",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7023129,\n   "lon": 37.640905,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48842058",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7034575,\n   "lon": 37.6373667,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48842066",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7037735,\n   "lon": 37.6363157,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48842070",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7037347,\n   "lon": 37.6379949,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48868516",\n   "kind": "organisation",\n   "name": "Перовский военкомат",\n   "office_type": "government",\n   "lat": 55.7453038,\n   "lon": 37.7753424,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48892066",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7222491,\n   "lon": 37.681516,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48892897",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7216069,\n   "lon": 37.6777383,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48893448",\n   "kind": "organisation",\n   "name": "Сбербанк",\n   "office_type": "bank",\n   "lat": 55.721895,\n   "lon": 37.6952567,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "way/48894083",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6557488,\n   "lon": 37.4982313,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48894240",\n   "kind": "building",\n   "name": "Центральный российский дом знаний",\n   "office_type": null,\n   "lat": 55.7744919,\n   "lon": 37.6823389,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48894731",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7181328,\n   "lon": 37.6775614,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48898734",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7222803,\n   "lon": 37.6782563,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48898741",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7243038,\n   "lon": 37.6739942,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48898745",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7231879,\n   "lon": 37.6810251,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48899472",\n   "kind": "organisation",\n   "name": "Московские кабельные сети",\n   "office_type": "foundation",\n   "lat": 55.7777567,\n   "lon": 37.706724,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48902425",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6640416,\n   "lon": 37.4722532,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48902426",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6636764,\n   "lon": 37.4728992,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48902983",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6637405,\n   "lon": 37.4692932,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48904109",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7043983,\n   "lon": 37.6651885,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48904117",\n   "kind": "building",\n   "name": "Инспекция ФНС России № 25",\n   "office_type": "government",\n   "lat": 55.7058198,\n   "lon": 37.6643525,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48905239",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.784002,\n   "lon": 37.6831502,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48912247",\n   "kind": "building",\n   "name": "Сибинтек",\n   "office_type": null,\n   "lat": 55.6996298,\n   "lon": 37.6162639,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48912249",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6993019,\n   "lon": 37.6147562,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48918622",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7167398,\n   "lon": 37.6020513,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48918700",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7570647,\n   "lon": 37.5925904,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48918701",\n   "kind": "building",\n   "name": "Посольство Мьянмы",\n   "office_type": "diplomatic",\n   "lat": 55.7571432,\n   "lon": 37.5929392,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48920267",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7217919,\n   "lon": 37.6997113,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48920643",\n   "kind": "organisation",\n   "name": "Главный информационно-вычислительный центр НИИ Контур",\n   "office_type": "telecommunication",\n   "lat": 55.6867003,\n   "lon": 37.5178061,\n   "opening_hours": null\n  },\n  {\n   "id": "way/48955872",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7304933,\n   "lon": 37.5491367,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49008342",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7201311,\n   "lon": 37.5185254,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49008919",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7200834,\n   "lon": 37.5188578,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49009913",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7535996,\n   "lon": 37.6397736,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49023028",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7949727,\n   "lon": 37.7324867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49023029",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7950907,\n   "lon": 37.7320713,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49023181",\n   "kind": "building",\n   "name": "Офисный центр",\n   "office_type": null,\n   "lat": 55.7958302,\n   "lon": 37.7321017,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49069615",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7466555,\n   "lon": 37.7053498,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49093123",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Принт\\"",\n   "office_type": null,\n   "lat": 55.837997,\n   "lon": 37.4537303,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49093160",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8392007,\n   "lon": 37.4540983,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49140311",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432253,\n   "lon": 37.7064362,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49140317",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.743537,\n   "lon": 37.7056891,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49140329",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7479307,\n   "lon": 37.7002083,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49150971",\n   "kind": "organisation",\n   "name": "АО Институт пластмасс имени Г.С.Петрова",\n   "office_type": "research",\n   "lat": 55.7436193,\n   "lon": 37.7102489,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49150993",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт теории и истории архитектуры и градостроительства",\n   "office_type": "research",\n   "lat": 55.7439844,\n   "lon": 37.7069372,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49155799",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8386091,\n   "lon": 37.4402544,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49155801",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8393382,\n   "lon": 37.4404258,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49200698",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8818897,\n   "lon": 37.547678,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49200700",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8817225,\n   "lon": 37.5456415,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49251850",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7353571,\n   "lon": 37.7146734,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49251851",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7357534,\n   "lon": 37.7152818,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49260751",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7382541,\n   "lon": 37.7013977,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49260761",\n   "kind": "organisation",\n   "name": "Митрополия Русской Православной Старообрядческой Церкви",\n   "office_type": "religion",\n   "lat": 55.7384158,\n   "lon": 37.7026843,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49260779",\n   "kind": "building",\n   "name": "ПАО \\"Промсвязьбанк\\"",\n   "office_type": "company",\n   "lat": 55.7352789,\n   "lon": 37.7132336,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49260784",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7363045,\n   "lon": 37.7099645,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49260785",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7358685,\n   "lon": 37.7108778,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49260799",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7360338,\n   "lon": 37.7060324,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49260806",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7373629,\n   "lon": 37.7055962,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49324856",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7731959,\n   "lon": 37.6793658,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49333327",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7364135,\n   "lon": 37.7094509,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49343427",\n   "kind": "organisation",\n   "name": "ВНИИНЕФТЕМАШ",\n   "office_type": "research",\n   "lat": 55.7056736,\n   "lon": 37.6117791,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49406072",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7765263,\n   "lon": 37.6830102,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49428991",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7732429,\n   "lon": 37.6815501,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49436491",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7566803,\n   "lon": 37.6291092,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49436497",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7580954,\n   "lon": 37.6267079,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49436498",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Cabinet Lounge\\"",\n   "office_type": null,\n   "lat": 55.7583395,\n   "lon": 37.6261782,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49488039",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7739079,\n   "lon": 37.6801661,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49494827",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461991,\n   "lon": 37.683787,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49494831",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7459531,\n   "lon": 37.68906,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49494841",\n   "kind": "building",\n   "name": "Московская дирекция по управлению терминально-складским комплексом",\n   "office_type": "railway",\n   "lat": 55.7468619,\n   "lon": 37.6900576,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49522900",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7284676,\n   "lon": 37.7481364,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49522902",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7282292,\n   "lon": 37.7504866,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49589202",\n   "kind": "building",\n   "name": "Бизнес-центр «Станция»",\n   "office_type": null,\n   "lat": 55.7799586,\n   "lon": 37.6879532,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49591154",\n   "kind": "building",\n   "name": "Институт проблем комплексного освоения недр",\n   "office_type": "research",\n   "lat": 55.7647741,\n   "lon": 37.7106636,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49591239",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7660102,\n   "lon": 37.6105155,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49591265",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7654553,\n   "lon": 37.6110216,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49591455",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.776361,\n   "lon": 37.6077981,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49593070",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7748098,\n   "lon": 37.6120173,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49593075",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7743574,\n   "lon": 37.611265,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49655510",\n   "kind": "building",\n   "name": "",\n   "office_type": "administrative",\n   "lat": 55.8329281,\n   "lon": 37.6370034,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49677877",\n   "kind": "organisation",\n   "name": "Межотраслевой НИИ Интеграл",\n   "office_type": "research",\n   "lat": 55.7475519,\n   "lon": 37.7190485,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49681032",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7833731,\n   "lon": 37.7227226,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49789790",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7277028,\n   "lon": 37.7360409,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49798329",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7772397,\n   "lon": 37.6108306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49799895",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7752251,\n   "lon": 37.6094272,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49799926",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7754901,\n   "lon": 37.6092135,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49825009",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8334754,\n   "lon": 37.6324891,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49901415",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7585071,\n   "lon": 37.6643921,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49902706",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Басманный\\"",\n   "office_type": null,\n   "lat": 55.7651326,\n   "lon": 37.6616375,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49906341",\n   "kind": "organisation",\n   "name": "Центроспас",\n   "office_type": "foundation",\n   "lat": 55.7496328,\n   "lon": 37.6787448,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49921584",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7803146,\n   "lon": 37.6892664,\n   "opening_hours": null\n  },\n  {\n   "id": "way/49921587",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.779899,\n   "lon": 37.6895968,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50072166",\n   "kind": "organisation",\n   "name": "Управление Федеральной Почтовой Связи г. Москвы",\n   "office_type": "government",\n   "lat": 55.7920301,\n   "lon": 37.7965334,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50120312",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6802209,\n   "lon": 37.5908604,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50120318",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6797163,\n   "lon": 37.5894936,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50120320",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6797657,\n   "lon": 37.5884762,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50120322",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6805878,\n   "lon": 37.588546,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50120323",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6803853,\n   "lon": 37.588029,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50120327",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6806644,\n   "lon": 37.5914808,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50123058",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6743581,\n   "lon": 37.5491956,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50123757",\n   "kind": "building",\n   "name": "ОВД ВАО",\n   "office_type": null,\n   "lat": 55.7970908,\n   "lon": 37.7889905,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50213397",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8088398,\n   "lon": 37.5775913,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50217525",\n   "kind": "building",\n   "name": "Креди Агриколь",\n   "office_type": null,\n   "lat": 55.7595436,\n   "lon": 37.6319924,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50218016",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432325,\n   "lon": 37.6825553,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50218288",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7608431,\n   "lon": 37.6349802,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50219079",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8058216,\n   "lon": 37.5408626,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50223712",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8050877,\n   "lon": 37.5457854,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50223716",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8041837,\n   "lon": 37.5461236,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50263497",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.632862,\n   "lon": 37.831492,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50286024",\n   "kind": "building",\n   "name": "Научно-исследовательский и проектно-конструкторский институт информатизации, автоматизации и связи на железнодорожном транспорте",\n   "office_type": "research",\n   "lat": 55.7372668,\n   "lon": 37.6956912,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "way/50286156",\n   "kind": "building",\n   "name": "УВД по ЦАО",\n   "office_type": null,\n   "lat": 55.735107,\n   "lon": 37.6916754,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50287332",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7669971,\n   "lon": 37.6344712,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50287739",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7677146,\n   "lon": 37.634582,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50318878",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7004085,\n   "lon": 37.691028,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50319749",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7002194,\n   "lon": 37.6875062,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321462",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7023861,\n   "lon": 37.6936331,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321467",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7034449,\n   "lon": 37.6914702,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321468",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7036463,\n   "lon": 37.6926085,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321472",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7028461,\n   "lon": 37.6918015,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321477",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7031873,\n   "lon": 37.6922974,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321478",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7040115,\n   "lon": 37.6903023,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321488",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7021685,\n   "lon": 37.6968786,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321958",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7005435,\n   "lon": 37.6967706,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321965",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7057961,\n   "lon": 37.6921028,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321966",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7041813,\n   "lon": 37.6936637,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321975",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7047809,\n   "lon": 37.6918479,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50321976",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7050244,\n   "lon": 37.6909299,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50344802",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6805652,\n   "lon": 37.6258466,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50378916",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7622178,\n   "lon": 37.6337951,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50379934",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7642692,\n   "lon": 37.6324026,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50380253",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7622271,\n   "lon": 37.6312918,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50380301",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623309,\n   "lon": 37.6305721,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50380771",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7612264,\n   "lon": 37.6303778,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50380775",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7609642,\n   "lon": 37.6311569,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50381104",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.764403,\n   "lon": 37.6284012,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50381105",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7645306,\n   "lon": 37.6279705,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50381107",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7639319,\n   "lon": 37.6269257,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50417159",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.865742,\n   "lon": 37.4682123,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50548832",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7533091,\n   "lon": 37.723366,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50654372",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7825756,\n   "lon": 37.6628588,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50833973",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7831295,\n   "lon": 37.6327527,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50833977",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835674,\n   "lon": 37.6328462,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50833983",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7853357,\n   "lon": 37.6311693,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50833986",\n   "kind": "building",\n   "name": "НИиПИ градостроительства",\n   "office_type": "research",\n   "lat": 55.784908,\n   "lon": 37.6317445,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50833996",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7888007,\n   "lon": 37.6333703,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50843222",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8502065,\n   "lon": 37.6549583,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50848880",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7802609,\n   "lon": 37.8190681,\n   "opening_hours": null\n  },\n  {\n   "id": "way/50885565",\n   "kind": "building",\n   "name": "Торгово-промышленная палата РФ",\n   "office_type": "government",\n   "lat": 55.7546679,\n   "lon": 37.6262569,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51004884",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7826161,\n   "lon": 37.6315568,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51004890",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7836968,\n   "lon": 37.6333401,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51004891",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7862272,\n   "lon": 37.6324033,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51015389",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7570834,\n   "lon": 37.6212843,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51045653",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7815473,\n   "lon": 37.6305914,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51049275",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7600608,\n   "lon": 37.5881778,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51119842",\n   "kind": "building",\n   "name": "Faberlic",\n   "office_type": "company",\n   "lat": 55.585282,\n   "lon": 37.6214892,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51143046",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7469718,\n   "lon": 37.4965332,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51193751",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт зерна и продуктов его переработки",\n   "office_type": "research",\n   "lat": 55.8164696,\n   "lon": 37.5741769,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51205133",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7803119,\n   "lon": 37.6903159,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51258940",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7650046,\n   "lon": 37.6257273,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51303552",\n   "kind": "organisation",\n   "name": "Центральная оперативная таможня ФТС РФ",\n   "office_type": "government",\n   "lat": 55.7562557,\n   "lon": 37.7436902,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51304612",\n   "kind": "building",\n   "name": "АОСН «Ястреб» ЦСН СР МВД России",\n   "office_type": null,\n   "lat": 55.7467487,\n   "lon": 37.7276618,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51336835",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "union",\n   "lat": 55.7648128,\n   "lon": 37.635044,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51341034",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6689439,\n   "lon": 37.5475064,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51492940",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7465546,\n   "lon": 37.7217803,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51497725",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6068393,\n   "lon": 37.5050778,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51498325",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6339236,\n   "lon": 37.6171573,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51499181",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.637164,\n   "lon": 37.6235302,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51543678",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7890919,\n   "lon": 37.6321945,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51543680",\n   "kind": "building",\n   "name": "Доходный дом Солодовникова",\n   "office_type": null,\n   "lat": 55.7892703,\n   "lon": 37.6326047,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51544996",\n   "kind": "organisation",\n   "name": "АО «Объединение «ИНГЕОКОМ»",\n   "office_type": "company",\n   "lat": 55.7291914,\n   "lon": 37.5601251,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51545006",\n   "kind": "organisation",\n   "name": "Институт медицинской паразитологии и тропической медицины им. Е. И. Марциновского",\n   "office_type": "research",\n   "lat": 55.7276875,\n   "lon": 37.5649678,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51545032",\n   "kind": "organisation",\n   "name": "Посольство исламской республики Мавритания",\n   "office_type": "diplomatic",\n   "lat": 55.7323097,\n   "lon": 37.5653474,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51545048",\n   "kind": "organisation",\n   "name": "Объединённый читальный зал ГАРФ и РГАЭ",\n   "office_type": "government",\n   "lat": 55.7334219,\n   "lon": 37.5752932,\n   "opening_hours": "Mo-Th 10:00-14:00,15:00-18:00; Fr 10:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "way/51545698",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7335292,\n   "lon": 37.5668117,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51546901",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7263487,\n   "lon": 37.5761837,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51546902",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7266349,\n   "lon": 37.5767252,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51546909",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7310559,\n   "lon": 37.573889,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51551232",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7411498,\n   "lon": 37.5800517,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51598677",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8437827,\n   "lon": 37.4449003,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51632159",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6641987,\n   "lon": 37.778158,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51675685",\n   "kind": "organisation",\n   "name": "Cлужба диспетчеров ЖКХ",\n   "office_type": "property_management",\n   "lat": 55.6572219,\n   "lon": 37.7731826,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51690290",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6539321,\n   "lon": 37.7126136,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51702004",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6563182,\n   "lon": 37.7279451,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51702015",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6557824,\n   "lon": 37.7367894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51702045",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.661083,\n   "lon": 37.7183899,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51702046",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6614174,\n   "lon": 37.719052,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51702047",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6617205,\n   "lon": 37.7196529,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51702068",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6590626,\n   "lon": 37.7354236,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51702152",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.658808,\n   "lon": 37.7338342,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51702175",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6591261,\n   "lon": 37.7361475,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51709761",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6610926,\n   "lon": 37.7395404,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51709771",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.664612,\n   "lon": 37.7462112,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51709777",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6653647,\n   "lon": 37.7477225,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51709787",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.668041,\n   "lon": 37.7477343,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51760927",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7154628,\n   "lon": 37.6122661,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51783841",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6400294,\n   "lon": 37.5289616,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51974413",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6956259,\n   "lon": 37.7367824,\n   "opening_hours": null\n  },\n  {\n   "id": "way/51988396",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7439405,\n   "lon": 37.6301086,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52071493",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8117221,\n   "lon": 37.8102989,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52104714",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7312479,\n   "lon": 37.6473611,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52104745",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7310219,\n   "lon": 37.6466932,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106116",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7594742,\n   "lon": 37.5896706,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106118",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7605516,\n   "lon": 37.5867876,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106120",\n   "kind": "organisation",\n   "name": "Посольство Таджикистана",\n   "office_type": "diplomatic",\n   "lat": 55.7596929,\n   "lon": 37.5881771,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106122",\n   "kind": "building",\n   "name": "ГТРК Культура",\n   "office_type": null,\n   "lat": 55.7592523,\n   "lon": 37.5884305,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106123",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7595635,\n   "lon": 37.5864268,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106124",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7602186,\n   "lon": 37.5869775,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106127",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.75996,\n   "lon": 37.58596,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106137",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7624685,\n   "lon": 37.5877476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106138",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623729,\n   "lon": 37.5880767,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52106141",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.762255,\n   "lon": 37.5875964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52111022",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7612237,\n   "lon": 37.5929035,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52111042",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7613619,\n   "lon": 37.5934659,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52111064",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619817,\n   "lon": 37.5960234,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52122719",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7610972,\n   "lon": 37.600692,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52122730",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7587612,\n   "lon": 37.596315,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52122740",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7606075,\n   "lon": 37.6001857,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52122743",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7595415,\n   "lon": 37.6009787,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52122744",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7595751,\n   "lon": 37.6013291,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52122752",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7624943,\n   "lon": 37.5968344,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52122760",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7615082,\n   "lon": 37.5998942,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52132574",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7611704,\n   "lon": 37.6063647,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52132591",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7605901,\n   "lon": 37.6020888,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52132593",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7603805,\n   "lon": 37.6025466,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52132598",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591527,\n   "lon": 37.6040252,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52132601",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7605865,\n   "lon": 37.6057672,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52132603",\n   "kind": "building",\n   "name": "Посольство Азербайджана",\n   "office_type": "diplomatic",\n   "lat": 55.7604216,\n   "lon": 37.6052641,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52132604",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7601912,\n   "lon": 37.6054966,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52132608",\n   "kind": "building",\n   "name": "Министерство культуры РФ",\n   "office_type": "government",\n   "lat": 55.7623571,\n   "lon": 37.6054451,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52132623",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7622521,\n   "lon": 37.6065632,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52302907",\n   "kind": "organisation",\n   "name": "Федеральное государственное унитарное предприятие «Центр аэронавигационной информации»",\n   "office_type": "government",\n   "lat": 55.8544461,\n   "lon": 37.4455313,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52313282",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7713585,\n   "lon": 37.6754109,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52387366",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7533014,\n   "lon": 37.6392903,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52387379",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7526977,\n   "lon": 37.6419316,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52387381",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7515974,\n   "lon": 37.6414545,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52387382",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7512198,\n   "lon": 37.6420526,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52387383",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7513208,\n   "lon": 37.6422736,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52387398",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7508057,\n   "lon": 37.6429675,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52387418",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7533942,\n   "lon": 37.640512,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52387419",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7532193,\n   "lon": 37.6397848,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52392181",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7548446,\n   "lon": 37.6380307,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52402557",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8596376,\n   "lon": 37.6396694,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52402564",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8596367,\n   "lon": 37.6440141,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52404044",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7502125,\n   "lon": 37.6439654,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434620",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7392849,\n   "lon": 37.6046917,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434621",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390518,\n   "lon": 37.6053124,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434625",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7385159,\n   "lon": 37.6048348,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434632",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Депо\\"",\n   "office_type": null,\n   "lat": 55.7417212,\n   "lon": 37.6006833,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434640",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7418554,\n   "lon": 37.6034972,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434643",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7423483,\n   "lon": 37.5993199,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434644",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Остоженка 10\\"",\n   "office_type": null,\n   "lat": 55.7425962,\n   "lon": 37.599867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434650",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7412824,\n   "lon": 37.6028491,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434653",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7409982,\n   "lon": 37.6046683,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434662",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7426441,\n   "lon": 37.6032599,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434663",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7424879,\n   "lon": 37.6034451,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434900",\n   "kind": "organisation",\n   "name": "Посольство Бруней Даруссалам",\n   "office_type": "diplomatic",\n   "lat": 55.7418668,\n   "lon": 37.6062782,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434901",\n   "kind": "organisation",\n   "name": "Посольство Мадагаскара. Канцелярия",\n   "office_type": "diplomatic",\n   "lat": 55.7420572,\n   "lon": 37.606092,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434908",\n   "kind": "organisation",\n   "name": "Посольство Кот-д’Ивуара",\n   "office_type": "diplomatic",\n   "lat": 55.7384075,\n   "lon": 37.5995729,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434911",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7375478,\n   "lon": 37.6017887,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434912",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Бутик-офис\\"",\n   "office_type": null,\n   "lat": 55.7378099,\n   "lon": 37.6021524,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434914",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.737741,\n   "lon": 37.6012675,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434920",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7379455,\n   "lon": 37.6014928,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434921",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7381864,\n   "lon": 37.6017157,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434922",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7384412,\n   "lon": 37.601565,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434931",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7398951,\n   "lon": 37.6030253,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434941",\n   "kind": "organisation",\n   "name": "Посольство Ливана. Резиденция",\n   "office_type": "diplomatic",\n   "lat": 55.7416832,\n   "lon": 37.6065452,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52434942",\n   "kind": "organisation",\n   "name": "Посольство Мадагаскара. Резиденция",\n   "office_type": "diplomatic",\n   "lat": 55.7420934,\n   "lon": 37.6064413,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52442206",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7766919,\n   "lon": 37.688178,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52442207",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7769966,\n   "lon": 37.6888003,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52442747",\n   "kind": "organisation",\n   "name": "Инспекция ФНС России № 19 по г.Москве",\n   "office_type": "government",\n   "lat": 55.8099901,\n   "lon": 37.8220853,\n   "opening_hours": "Mo,We 09:00-18:00; Tu,Th 09:00-20:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "way/52445298",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7440551,\n   "lon": 37.6280929,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445302",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7411187,\n   "lon": 37.6288941,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445303",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7411065,\n   "lon": 37.6292722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445310",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7426666,\n   "lon": 37.6296617,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445314",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7428446,\n   "lon": 37.6290812,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445315",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7429547,\n   "lon": 37.6292503,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445316",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7430991,\n   "lon": 37.6291456,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445318",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7426946,\n   "lon": 37.6286579,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445323",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7429283,\n   "lon": 37.6286178,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445324",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432065,\n   "lon": 37.6291373,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445331",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432947,\n   "lon": 37.628929,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445333",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.743634,\n   "lon": 37.6286547,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52445334",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7437825,\n   "lon": 37.6284627,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52450886",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7431361,\n   "lon": 37.6305781,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52450890",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432869,\n   "lon": 37.6306321,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52453402",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7391005,\n   "lon": 37.6566823,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52453412",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.739337,\n   "lon": 37.656514,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52453575",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7415034,\n   "lon": 37.62821,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52454804",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7325363,\n   "lon": 37.6821569,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52456791",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7589112,\n   "lon": 37.5853992,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52456792",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591522,\n   "lon": 37.5853307,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52456794",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7592853,\n   "lon": 37.5856228,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52456796",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591796,\n   "lon": 37.5860993,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52456797",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7589092,\n   "lon": 37.5861125,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52456801",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7589828,\n   "lon": 37.5873185,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52456804",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7586221,\n   "lon": 37.5884661,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52456805",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588089,\n   "lon": 37.5878432,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52456806",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7585592,\n   "lon": 37.588022,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52462134",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7886033,\n   "lon": 37.6415494,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52462299",\n   "kind": "building",\n   "name": "Бизнес-центр Dominion Tower",\n   "office_type": null,\n   "lat": 55.7225359,\n   "lon": 37.6682682,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52462308",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7153883,\n   "lon": 37.6658103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52462357",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7144406,\n   "lon": 37.673977,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52463968",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Пересвет\\"",\n   "office_type": null,\n   "lat": 55.7229463,\n   "lon": 37.6702844,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52466862",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7996816,\n   "lon": 37.563639,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52466874",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7984071,\n   "lon": 37.5543394,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52466882",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7991837,\n   "lon": 37.5660667,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52467172",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7545623,\n   "lon": 37.631089,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52467173",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.755193,\n   "lon": 37.6307585,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52467175",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7546937,\n   "lon": 37.6286509,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52467176",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7545516,\n   "lon": 37.6296476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52467182",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7535671,\n   "lon": 37.6296976,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52467187",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7549234,\n   "lon": 37.6300389,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52473764",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8036794,\n   "lon": 37.5475086,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52475408",\n   "kind": "building",\n   "name": "Палаты тверского подворья",\n   "office_type": null,\n   "lat": 55.7626452,\n   "lon": 37.6249015,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52477085",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8203891,\n   "lon": 37.6575553,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52481298",\n   "kind": "organisation",\n   "name": "Главное управление ПФР №4",\n   "office_type": "government",\n   "lat": 55.71176,\n   "lon": 37.6881651,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52481301",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7110167,\n   "lon": 37.6840244,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52481313",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7103702,\n   "lon": 37.6843867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52482757",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7600972,\n   "lon": 37.6877621,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52482759",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7605497,\n   "lon": 37.6882515,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52495103",\n   "kind": "building",\n   "name": "Федеральное казначейство",\n   "office_type": "government",\n   "lat": 55.7552578,\n   "lon": 37.6275708,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52502143",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8219537,\n   "lon": 37.5012189,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52502148",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8227117,\n   "lon": 37.5012383,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52503596",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390163,\n   "lon": 37.6148642,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52503678",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8113754,\n   "lon": 37.7705703,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52504835",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7573215,\n   "lon": 37.636381,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52507994",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.756664,\n   "lon": 37.628293,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52508540",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7613721,\n   "lon": 37.5648684,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52508543",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.761446,\n   "lon": 37.5626958,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52509048",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.759138,\n   "lon": 37.5729579,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52511190",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7965205,\n   "lon": 37.5459231,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52511205",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.797175,\n   "lon": 37.5473004,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52520573",\n   "kind": "building",\n   "name": "Башня 2000",\n   "office_type": null,\n   "lat": 55.7463013,\n   "lon": 37.5461728,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52521807",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.783966,\n   "lon": 37.7162804,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52521845",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835541,\n   "lon": 37.7116015,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52521853",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.78557,\n   "lon": 37.7161219,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52563754",\n   "kind": "organisation",\n   "name": "НИИ Контур",\n   "office_type": "engineer",\n   "lat": 55.6843354,\n   "lon": 37.5174663,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52563762",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6815455,\n   "lon": 37.5124426,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52563767",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6806715,\n   "lon": 37.5108965,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52563768",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6811019,\n   "lon": 37.5116642,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52565342",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7055634,\n   "lon": 37.5479227,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52571598",\n   "kind": "organisation",\n   "name": "Постоянное представительство Республики Ингушетия при Президенте Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7368161,\n   "lon": 37.6576764,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52583199",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.766701,\n   "lon": 37.6004683,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52583204",\n   "kind": "organisation",\n   "name": "Посольство Нигерии",\n   "office_type": "diplomatic",\n   "lat": 55.7661302,\n   "lon": 37.5978864,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52583210",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7659801,\n   "lon": 37.5966813,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52583235",\n   "kind": "organisation",\n   "name": "Координационный совет генеральных прокуроров стран участниц СНГ - Секретариат",\n   "office_type": "government",\n   "lat": 55.767413,\n   "lon": 37.5983035,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52583238",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7688449,\n   "lon": 37.6004846,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52584109",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.792555,\n   "lon": 37.4904765,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52584120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7975375,\n   "lon": 37.5019341,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52588003",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7810975,\n   "lon": 37.5261155,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52609039",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7667153,\n   "lon": 37.6109525,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52609042",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.766291,\n   "lon": 37.6114482,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52609047",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7665232,\n   "lon": 37.6113857,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52609071",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7675091,\n   "lon": 37.6127695,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52609072",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.767431,\n   "lon": 37.6130609,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52609073",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7676649,\n   "lon": 37.613354,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52613447",\n   "kind": "organisation",\n   "name": "НИИ общей реаниматологии им. В. А. Неговского",\n   "office_type": "research",\n   "lat": 55.7672561,\n   "lon": 37.6135614,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52657951",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8226084,\n   "lon": 37.5072502,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52657968",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8241359,\n   "lon": 37.5037744,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673159",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7658712,\n   "lon": 37.6280405,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673160",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7660942,\n   "lon": 37.6280529,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673164",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7659518,\n   "lon": 37.6277613,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673166",\n   "kind": "building",\n   "name": "Федеральное агентство по рыболовству",\n   "office_type": "government",\n   "lat": 55.7660637,\n   "lon": 37.6273386,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673187",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7680024,\n   "lon": 37.6264863,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673197",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7669656,\n   "lon": 37.6257382,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673209",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7690089,\n   "lon": 37.6277036,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673210",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.768969,\n   "lon": 37.6272376,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673212",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7689701,\n   "lon": 37.6263594,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673219",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7669254,\n   "lon": 37.6296402,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52673241",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7688408,\n   "lon": 37.6300519,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52706587",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7239678,\n   "lon": 37.6261665,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52706592",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7256815,\n   "lon": 37.6260979,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52706631",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7294339,\n   "lon": 37.6473991,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52706633",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7295103,\n   "lon": 37.6468307,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52706640",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7292832,\n   "lon": 37.648236,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52706652",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7311177,\n   "lon": 37.6473964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52706654",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7314163,\n   "lon": 37.6465274,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52708468",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.713684,\n   "lon": 37.5684271,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52708472",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7155792,\n   "lon": 37.5699252,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52708473",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7146591,\n   "lon": 37.5683735,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52708474",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7147764,\n   "lon": 37.5687609,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52708476",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7149157,\n   "lon": 37.5690771,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52708477",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7152,\n   "lon": 37.5683574,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52721771",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7388227,\n   "lon": 37.6571153,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52736114",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7504787,\n   "lon": 37.5979538,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52736115",\n   "kind": "organisation",\n   "name": "Посольство Туркменистана. Консульский отдел",\n   "office_type": "diplomatic",\n   "lat": 55.7508105,\n   "lon": 37.5979234,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52736118",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7504998,\n   "lon": 37.5993,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738002",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7691538,\n   "lon": 37.6336132,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738007",\n   "kind": "building",\n   "name": "Московский кредитный банк",\n   "office_type": null,\n   "lat": 55.768147,\n   "lon": 37.6327154,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "way/52738008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7681756,\n   "lon": 37.6322237,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738013",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Чайка Плаза\\"",\n   "office_type": null,\n   "lat": 55.7678372,\n   "lon": 37.6322095,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738014",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7675344,\n   "lon": 37.6321175,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738016",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7676717,\n   "lon": 37.6320218,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738018",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7677689,\n   "lon": 37.6327654,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738019",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7674948,\n   "lon": 37.6326507,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738021",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7683653,\n   "lon": 37.6323152,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738030",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7677358,\n   "lon": 37.6351139,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738034",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7681767,\n   "lon": 37.63471,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52738036",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7691171,\n   "lon": 37.6339411,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52739515",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.765797,\n   "lon": 37.6394208,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52739517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.765853,\n   "lon": 37.6407177,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52739522",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7658775,\n   "lon": 37.6399796,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52739526",\n   "kind": "organisation",\n   "name": "Федеральная антимонопольная служба",\n   "office_type": "government",\n   "lat": 55.7682924,\n   "lon": 37.6394215,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749533",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7666542,\n   "lon": 37.6440644,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749553",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7663036,\n   "lon": 37.6436179,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749574",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7661844,\n   "lon": 37.646805,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749575",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7662612,\n   "lon": 37.6450836,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749580",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7659761,\n   "lon": 37.6465993,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749594",\n   "kind": "building",\n   "name": "Сургутнефтегаз",\n   "office_type": "insurance",\n   "lat": 55.7679046,\n   "lon": 37.6428618,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749600",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Усадьба Суворовых\\"",\n   "office_type": null,\n   "lat": 55.7679571,\n   "lon": 37.644218,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749603",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7673287,\n   "lon": 37.6427776,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749610",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.766935,\n   "lon": 37.6420562,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749611",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7668195,\n   "lon": 37.6439003,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749650",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7675916,\n   "lon": 37.6471608,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749675",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7670027,\n   "lon": 37.6457092,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749681",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7675245,\n   "lon": 37.6468861,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52749698",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7688655,\n   "lon": 37.6462308,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52769697",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7668038,\n   "lon": 37.6485788,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52769700",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7672551,\n   "lon": 37.6488499,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52769705",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7669674,\n   "lon": 37.6479995,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52769707",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7671513,\n   "lon": 37.649236,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52772323",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7121178,\n   "lon": 37.5923346,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52794175",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7643541,\n   "lon": 37.6577202,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52797050",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7641867,\n   "lon": 37.6509076,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52805419",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.76558,\n   "lon": 37.6477288,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52813016",\n   "kind": "organisation",\n   "name": "Посольство Эстонии в России",\n   "office_type": "diplomatic",\n   "lat": 55.7552978,\n   "lon": 37.6019907,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52813020",\n   "kind": "organisation",\n   "name": "Посольство Нидерландов",\n   "office_type": "diplomatic",\n   "lat": 55.7545308,\n   "lon": 37.6022684,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "way/52813032",\n   "kind": "building",\n   "name": "Роскино",\n   "office_type": "company",\n   "lat": 55.7563507,\n   "lon": 37.600926,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52813036",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7565632,\n   "lon": 37.6019581,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52813038",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7561209,\n   "lon": 37.6023581,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52813039",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7559513,\n   "lon": 37.6018178,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52823281",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7636921,\n   "lon": 37.6423376,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52823287",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7645488,\n   "lon": 37.6403457,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52825789",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.765137,\n   "lon": 37.6389015,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52825805",\n   "kind": "organisation",\n   "name": "Посольство республики Казахстан. Консульский отдел",\n   "office_type": "diplomatic",\n   "lat": 55.7647261,\n   "lon": 37.6404593,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52826339",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.786868,\n   "lon": 37.499473,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52829388",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7639359,\n   "lon": 37.6322751,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52829395",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7624348,\n   "lon": 37.6324327,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52829401",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7632051,\n   "lon": 37.6311977,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52829416",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7650654,\n   "lon": 37.6307517,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52829423",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7649551,\n   "lon": 37.6311656,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52829425",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7643251,\n   "lon": 37.6327617,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52829426",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7643767,\n   "lon": 37.632871,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52830229",\n   "kind": "organisation",\n   "name": "Федеральный научный центр пищевых систем им. В.М. Горбатова",\n   "office_type": "research",\n   "lat": 55.7314608,\n   "lon": 37.6789802,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52832290",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7785261,\n   "lon": 37.6752847,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52835463",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7628932,\n   "lon": 37.6280522,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52835490",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7649428,\n   "lon": 37.6265158,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52835491",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7649423,\n   "lon": 37.6261425,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52877787",\n   "kind": "organisation",\n   "name": "Министерство здравоохранения Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7653162,\n   "lon": 37.6196858,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52877800",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.764881,\n   "lon": 37.6219591,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52879923",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6920252,\n   "lon": 37.5791917,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52880390",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.765336,\n   "lon": 37.6133612,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52880392",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7656127,\n   "lon": 37.6135842,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52886846",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7654383,\n   "lon": 37.6114778,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52886871",\n   "kind": "building",\n   "name": "Бизнес-центр «Тверской»",\n   "office_type": null,\n   "lat": 55.7647864,\n   "lon": 37.6076579,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52889906",\n   "kind": "organisation",\n   "name": "Федеральное агентство по делам молодёжи",\n   "office_type": "government",\n   "lat": 55.7548829,\n   "lon": 37.6433521,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52889912",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7582349,\n   "lon": 37.6426211,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52890554",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7629013,\n   "lon": 37.6060743,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52890563",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7636836,\n   "lon": 37.6055104,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52890573",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7655525,\n   "lon": 37.6031185,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52897569",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7641391,\n   "lon": 37.5960813,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52909818",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7638373,\n   "lon": 37.5951157,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52909901",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7650579,\n   "lon": 37.5902748,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52909904",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7641948,\n   "lon": 37.5899906,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52917389",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7636924,\n   "lon": 37.5896477,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52917399",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7627985,\n   "lon": 37.588043,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52917408",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7627187,\n   "lon": 37.588388,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52923602",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7401297,\n   "lon": 37.5269361,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52933364",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7589833,\n   "lon": 37.6100208,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52944437",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619119,\n   "lon": 37.6151849,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52944439",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7616859,\n   "lon": 37.6150958,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52944442",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7622844,\n   "lon": 37.6147656,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52951259",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7612069,\n   "lon": 37.6226044,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52951299",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7614398,\n   "lon": 37.6227981,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52951312",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7614917,\n   "lon": 37.6225381,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52955652",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7624128,\n   "lon": 37.6294575,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52977907",\n   "kind": "organisation",\n   "name": "Русское географическое общество (Московская штаб-квартира)",\n   "office_type": "ngo",\n   "lat": 55.7579812,\n   "lon": 37.6271854,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52977911",\n   "kind": "organisation",\n   "name": "Русское географическое общество",\n   "office_type": "ngo",\n   "lat": 55.7577795,\n   "lon": 37.6270819,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52977915",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7575783,\n   "lon": 37.6273839,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52979059",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7571546,\n   "lon": 37.6218417,\n   "opening_hours": null\n  },\n  {\n   "id": "way/52979060",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7568477,\n   "lon": 37.6217473,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53016579",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7611469,\n   "lon": 37.630832,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53016580",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.761297,\n   "lon": 37.631025,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53016602",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.761909,\n   "lon": 37.6304497,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53016604",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7614511,\n   "lon": 37.6305439,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53016609",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7618916,\n   "lon": 37.6321928,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53016610",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619548,\n   "lon": 37.6319169,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53027128",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7610334,\n   "lon": 37.6372594,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53027145",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7618483,\n   "lon": 37.6365888,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53027147",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7621067,\n   "lon": 37.637788,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53027148",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.761549,\n   "lon": 37.6371561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53027150",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619308,\n   "lon": 37.6370665,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53027161",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7587812,\n   "lon": 37.6404142,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53034713",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7337003,\n   "lon": 37.6997378,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53034714",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7338767,\n   "lon": 37.6990669,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53039458",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7621446,\n   "lon": 37.6467641,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53077297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.758405,\n   "lon": 37.6509171,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53077299",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7583485,\n   "lon": 37.6522274,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53077302",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7561542,\n   "lon": 37.6518802,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53077311",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7581619,\n   "lon": 37.6523677,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53081767",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7623952,\n   "lon": 37.6804912,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088941",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7563232,\n   "lon": 37.6440257,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088946",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7561605,\n   "lon": 37.6440535,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088950",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.757528,\n   "lon": 37.6440031,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088956",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7574104,\n   "lon": 37.6442991,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088963",\n   "kind": "building",\n   "name": "Научное издательство «Большая Российская энциклопедия»",\n   "office_type": "publisher",\n   "lat": 55.7569772,\n   "lon": 37.6469309,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088966",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7567807,\n   "lon": 37.6466841,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088969",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7577665,\n   "lon": 37.6447943,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088970",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7576875,\n   "lon": 37.6451477,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088973",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7570401,\n   "lon": 37.6450932,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53088975",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7579617,\n   "lon": 37.645305,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53156453",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7552627,\n   "lon": 37.6341999,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53156500",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7590366,\n   "lon": 37.6302306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53156743",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7680351,\n   "lon": 37.679255,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53156747",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7676631,\n   "lon": 37.6788117,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53167715",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6889274,\n   "lon": 37.6142986,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53167716",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6890216,\n   "lon": 37.6147596,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53167717",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6891107,\n   "lon": 37.615229,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53167841",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6886836,\n   "lon": 37.6167639,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53167843",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6883506,\n   "lon": 37.6169585,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53192636",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8351883,\n   "lon": 37.6469768,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53192653",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8345097,\n   "lon": 37.6510388,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53205942",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7937939,\n   "lon": 37.8238498,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53205947",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7950815,\n   "lon": 37.8206548,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53219824",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.821902,\n   "lon": 37.5212774,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53227151",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7402674,\n   "lon": 37.6557857,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53264613",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7522375,\n   "lon": 37.6631728,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53268734",\n   "kind": "building",\n   "name": "Московское бюро информационного агенства \\"Синьхуа\\"",\n   "office_type": "news_agency",\n   "lat": 55.7361667,\n   "lon": 37.6805464,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53367607",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7697362,\n   "lon": 37.6773649,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53398989",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8060559,\n   "lon": 37.7630291,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53445882",\n   "kind": "organisation",\n   "name": "Центральный объединённый архив учреждений системы образования Москвы",\n   "office_type": "government",\n   "lat": 55.7440353,\n   "lon": 37.6661819,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53446406",\n   "kind": "organisation",\n   "name": "Учебно-методический центр Департамента образования г. Москвы",\n   "office_type": "government",\n   "lat": 55.7431656,\n   "lon": 37.6652275,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53446431",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449579,\n   "lon": 37.662299,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53446432",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449697,\n   "lon": 37.6625476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53521337",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7534524,\n   "lon": 37.7246502,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53548386",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.74238,\n   "lon": 37.6173378,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53548389",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7420315,\n   "lon": 37.6170113,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53548398",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7404591,\n   "lon": 37.6196516,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53548416",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7412039,\n   "lon": 37.6176021,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53548445",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7428271,\n   "lon": 37.6176502,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53548447",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7425946,\n   "lon": 37.6179305,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53551234",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Якиманка\\"",\n   "office_type": "yes",\n   "lat": 55.7378368,\n   "lon": 37.6156412,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53551238",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7365357,\n   "lon": 37.6148962,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53551241",\n   "kind": "building",\n   "name": "Институт физики атмосферы им. А.М. Обухова РАН",\n   "office_type": "research",\n   "lat": 55.7391091,\n   "lon": 37.6233565,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53551246",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7383427,\n   "lon": 37.620052,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53551248",\n   "kind": "building",\n   "name": "Всероссийский научно-исследовательский институт минерального сырья им. Н.М.Федоровского",\n   "office_type": "research",\n   "lat": 55.738306,\n   "lon": 37.6212211,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53551249",\n   "kind": "building",\n   "name": "Почвенный институт им. Докучаева",\n   "office_type": "research",\n   "lat": 55.7386141,\n   "lon": 37.6204275,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53551250",\n   "kind": "building",\n   "name": "Геологический институт РАН",\n   "office_type": "research",\n   "lat": 55.7389143,\n   "lon": 37.6213051,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53551255",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7367197,\n   "lon": 37.6252984,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53553532",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390856,\n   "lon": 37.6189481,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53553546",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7383119,\n   "lon": 37.6186287,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53553547",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7381222,\n   "lon": 37.6188936,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53553552",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7385108,\n   "lon": 37.6185053,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53553558",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7394371,\n   "lon": 37.6179703,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53553559",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7396583,\n   "lon": 37.6177389,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53553560",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7392876,\n   "lon": 37.6188808,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53591175",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7320433,\n   "lon": 37.658603,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53594886",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7061662,\n   "lon": 37.6873863,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53650437",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7310992,\n   "lon": 37.6569883,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53662110",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.739282,\n   "lon": 37.6114182,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53662132",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7418753,\n   "lon": 37.6152456,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53662150",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7405784,\n   "lon": 37.6156055,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53662181",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7416452,\n   "lon": 37.614616,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53662625",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7408475,\n   "lon": 37.6143101,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53662630",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7398466,\n   "lon": 37.6146163,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53721856",\n   "kind": "organisation",\n   "name": "Планета Железяка",\n   "office_type": "company",\n   "lat": 55.8970651,\n   "lon": 37.6075487,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "way/53721858",\n   "kind": "building",\n   "name": "Планета Железяка",\n   "office_type": "company",\n   "lat": 55.897405,\n   "lon": 37.6069182,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "way/53725264",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7780612,\n   "lon": 37.673269,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53735882",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7008419,\n   "lon": 37.6242252,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53735902",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "it",\n   "lat": 55.6964885,\n   "lon": 37.6253806,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53735931",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.691466,\n   "lon": 37.6232184,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53749261",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7087476,\n   "lon": 37.6147541,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53754817",\n   "kind": "organisation",\n   "name": "Центральный банк РФ",\n   "office_type": "company",\n   "lat": 55.7308008,\n   "lon": 37.6171822,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53754819",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.73424,\n   "lon": 37.6169466,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53754839",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7329413,\n   "lon": 37.6203843,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53754840",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7330835,\n   "lon": 37.6184638,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53756200",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7314587,\n   "lon": 37.6159905,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53756205",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7315616,\n   "lon": 37.6151252,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53756206",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7348631,\n   "lon": 37.6144762,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53756217",\n   "kind": "building",\n   "name": "Институт географии РАН",\n   "office_type": "research",\n   "lat": 55.7348044,\n   "lon": 37.6149202,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53767681",\n   "kind": "building",\n   "name": "Деловой центр \\"Денисовский 26\\"",\n   "office_type": null,\n   "lat": 55.766737,\n   "lon": 37.6764773,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53816164",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7336558,\n   "lon": 37.6280411,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53816167",\n   "kind": "building",\n   "name": "Бизнес-центр Lighthouse",\n   "office_type": null,\n   "lat": 55.730633,\n   "lon": 37.6282446,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53816168",\n   "kind": "building",\n   "name": "ПАО НК «РуссНефть»",\n   "office_type": "company",\n   "lat": 55.7318326,\n   "lon": 37.6274533,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53816170",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.731449,\n   "lon": 37.627545,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53816180",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7331089,\n   "lon": 37.6218401,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53816201",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.731391,\n   "lon": 37.6256917,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53816202",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7316146,\n   "lon": 37.6259082,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53816203",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7317605,\n   "lon": 37.6254599,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53830842",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7322581,\n   "lon": 37.6296935,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53830843",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7322455,\n   "lon": 37.6300498,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53830844",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7323964,\n   "lon": 37.6302922,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53830854",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7328923,\n   "lon": 37.6301397,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53830866",\n   "kind": "organisation",\n   "name": "Религиозная организация Российское Библейское Общество",\n   "office_type": "company",\n   "lat": 55.7310334,\n   "lon": 37.6331908,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53830905",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7331117,\n   "lon": 37.6340955,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53833256",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7795242,\n   "lon": 37.7112851,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53845503",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7549286,\n   "lon": 37.6815462,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53845517",\n   "kind": "building",\n   "name": "КРОК",\n   "office_type": "company",\n   "lat": 55.7538259,\n   "lon": 37.6812561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53845539",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7537222,\n   "lon": 37.6800722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53845549",\n   "kind": "organisation",\n   "name": "Центр организации дорожного движения Правительства Москвы",\n   "office_type": "government",\n   "lat": 55.7550662,\n   "lon": 37.6900294,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854184",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7473533,\n   "lon": 37.6338188,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854186",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7473363,\n   "lon": 37.6286273,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854187",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461844,\n   "lon": 37.6333792,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854190",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7470227,\n   "lon": 37.6324746,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854194",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.74632,\n   "lon": 37.6309039,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854203",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7472569,\n   "lon": 37.6290032,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854211",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7467812,\n   "lon": 37.6302529,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854212",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.747151,\n   "lon": 37.6292894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854213",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.747115,\n   "lon": 37.6295727,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854216",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7467289,\n   "lon": 37.631106,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854221",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7466883,\n   "lon": 37.6329207,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854222",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Энерджи Хаус\\"",\n   "office_type": null,\n   "lat": 55.7477648,\n   "lon": 37.6330729,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854225",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7479105,\n   "lon": 37.6288864,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53854226",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7476106,\n   "lon": 37.628901,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53888272",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7754289,\n   "lon": 37.6578121,\n   "opening_hours": null\n  },\n  {\n   "id": "way/53889555",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7751781,\n   "lon": 37.6602508,\n   "opening_hours": null\n  },\n  {\n   "id": "way/54601789",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7250022,\n   "lon": 37.6377568,\n   "opening_hours": null\n  },\n  {\n   "id": "way/54601792",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.724248,\n   "lon": 37.6370631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/54601796",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7260009,\n   "lon": 37.637546,\n   "opening_hours": null\n  },\n  {\n   "id": "way/54637544",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8962832,\n   "lon": 37.5544067,\n   "opening_hours": null\n  },\n  {\n   "id": "way/54818710",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7321985,\n   "lon": 37.6971011,\n   "opening_hours": null\n  },\n  {\n   "id": "way/54830109",\n   "kind": "organisation",\n   "name": "Офис продаж ЖК \\"Wave\\"",\n   "office_type": "estate_agent",\n   "lat": 55.636122,\n   "lon": 37.7116191,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55015763",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7820924,\n   "lon": 37.5870503,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55063058",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8342853,\n   "lon": 37.449301,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55063990",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8322452,\n   "lon": 37.451037,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55068587",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.825808,\n   "lon": 37.4467219,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55068754",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8262947,\n   "lon": 37.4464268,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55068945",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.826117,\n   "lon": 37.448937,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55069709",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8261758,\n   "lon": 37.4485641,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55069712",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8253677,\n   "lon": 37.4476755,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55141134",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8337433,\n   "lon": 37.4546499,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55272247",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8189127,\n   "lon": 37.7071316,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55422685",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8408644,\n   "lon": 37.4414207,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55503023",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7545616,\n   "lon": 37.5366479,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55695634",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7725769,\n   "lon": 37.540964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/55924296",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.727532,\n   "lon": 37.6216539,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56014614",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8183383,\n   "lon": 37.5182029,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56016994",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8184848,\n   "lon": 37.5221579,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56017916",\n   "kind": "building",\n   "name": "Hi-Tec House",\n   "office_type": null,\n   "lat": 55.818788,\n   "lon": 37.521314,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56031018",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6111181,\n   "lon": 37.5894338,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56075407",\n   "kind": "organisation",\n   "name": "Объединенный институт высоких температур РАН",\n   "office_type": "research",\n   "lat": 55.7557922,\n   "lon": 37.7110258,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56121187",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7536367,\n   "lon": 37.7072773,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56215931",\n   "kind": "organisation",\n   "name": "Мосавтостекло",\n   "office_type": "company",\n   "lat": 55.7237982,\n   "lon": 37.6498428,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56256137",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7068749,\n   "lon": 37.501449,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56460827",\n   "kind": "organisation",\n   "name": "Управление федерального казначейства по Московской области",\n   "office_type": "government",\n   "lat": 55.7252676,\n   "lon": 37.6525705,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56460837",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7216933,\n   "lon": 37.6486603,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56629809",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7847357,\n   "lon": 37.7190943,\n   "opening_hours": null\n  },\n  {\n   "id": "way/56629815",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7845164,\n   "lon": 37.7225819,\n   "opening_hours": null\n  },\n  {\n   "id": "way/57359889",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7305607,\n   "lon": 37.644726,\n   "opening_hours": null\n  },\n  {\n   "id": "way/58664065",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6110755,\n   "lon": 37.5148631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/58694250",\n   "kind": "building",\n   "name": "Мосводоканал, Производственно-Эксплуатационное управление Канализационной Сети",\n   "office_type": null,\n   "lat": 55.6061545,\n   "lon": 37.5088986,\n   "opening_hours": null\n  },\n  {\n   "id": "way/58742237",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.87698,\n   "lon": 37.7033534,\n   "opening_hours": null\n  },\n  {\n   "id": "way/58788605",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7388446,\n   "lon": 37.5825789,\n   "opening_hours": null\n  },\n  {\n   "id": "way/58821301",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7155942,\n   "lon": 37.6327002,\n   "opening_hours": null\n  },\n  {\n   "id": "way/58821334",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7165619,\n   "lon": 37.6337592,\n   "opening_hours": null\n  },\n  {\n   "id": "way/58821513",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7184064,\n   "lon": 37.6317277,\n   "opening_hours": null\n  },\n  {\n   "id": "way/58821520",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.718739,\n   "lon": 37.6295872,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60093709",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8246521,\n   "lon": 37.5013869,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60095352",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Головино Плаза\\"",\n   "office_type": null,\n   "lat": 55.8433013,\n   "lon": 37.5260554,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60095436",\n   "kind": "building",\n   "name": "Бизнес-центр",\n   "office_type": null,\n   "lat": 55.8341885,\n   "lon": 37.5074709,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60351024",\n   "kind": "building",\n   "name": "ТЦ \\"Домашний\\"",\n   "office_type": null,\n   "lat": 55.8079418,\n   "lon": 37.5388696,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60453478",\n   "kind": "organisation",\n   "name": "Министерство юстиции РФ",\n   "office_type": "government",\n   "lat": 55.7310107,\n   "lon": 37.6156158,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60550906",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8086921,\n   "lon": 37.5345607,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60863074",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8081711,\n   "lon": 37.5223639,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60863075",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8100561,\n   "lon": 37.5226466,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60903175",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7765873,\n   "lon": 37.6728614,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60911175",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7679285,\n   "lon": 37.6811862,\n   "opening_hours": null\n  },\n  {\n   "id": "way/60911217",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7715734,\n   "lon": 37.6778346,\n   "opening_hours": null\n  },\n  {\n   "id": "way/61231367",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7029586,\n   "lon": 37.6797894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/61450390",\n   "kind": "organisation",\n   "name": "Управа района Преображенское",\n   "office_type": "government",\n   "lat": 55.7958796,\n   "lon": 37.7257008,\n   "opening_hours": null\n  },\n  {\n   "id": "way/61671680",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7767951,\n   "lon": 37.6885032,\n   "opening_hours": null\n  },\n  {\n   "id": "way/61794400",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7395372,\n   "lon": 37.5073258,\n   "opening_hours": null\n  },\n  {\n   "id": "way/61794401",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7400869,\n   "lon": 37.5079455,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62044245",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7475733,\n   "lon": 37.6019072,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62044248",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7493137,\n   "lon": 37.6008217,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62060928",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7981779,\n   "lon": 37.7401754,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62116255",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8046481,\n   "lon": 37.74391,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62169335",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7552147,\n   "lon": 37.7387165,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62169337",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7551684,\n   "lon": 37.739895,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62198296",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.5866056,\n   "lon": 37.6917544,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62209257",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Интеграл\\"",\n   "office_type": null,\n   "lat": 55.7525887,\n   "lon": 37.7023348,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62333882",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8451235,\n   "lon": 37.6255064,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62669071",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.714095,\n   "lon": 37.5802319,\n   "opening_hours": null\n  },\n  {\n   "id": "way/62681917",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7499457,\n   "lon": 37.5466741,\n   "opening_hours": null\n  },\n  {\n   "id": "way/63357951",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7082963,\n   "lon": 37.6816111,\n   "opening_hours": null\n  },\n  {\n   "id": "way/63940100",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7048355,\n   "lon": 37.6823907,\n   "opening_hours": null\n  },\n  {\n   "id": "way/64266176",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7864527,\n   "lon": 37.7265182,\n   "opening_hours": null\n  },\n  {\n   "id": "way/65041543",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7043203,\n   "lon": 37.6834483,\n   "opening_hours": null\n  },\n  {\n   "id": "way/65041552",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7052458,\n   "lon": 37.6821335,\n   "opening_hours": null\n  },\n  {\n   "id": "way/65134315",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7776119,\n   "lon": 37.601227,\n   "opening_hours": null\n  },\n  {\n   "id": "way/65137208",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7585953,\n   "lon": 37.6025068,\n   "opening_hours": null\n  },\n  {\n   "id": "way/65138312",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7443131,\n   "lon": 37.5869782,\n   "opening_hours": null\n  },\n  {\n   "id": "way/65138313",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7443978,\n   "lon": 37.5877257,\n   "opening_hours": null\n  },\n  {\n   "id": "way/66297529",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.768048,\n   "lon": 37.5285895,\n   "opening_hours": null\n  },\n  {\n   "id": "way/66420167",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7070586,\n   "lon": 37.6901017,\n   "opening_hours": null\n  },\n  {\n   "id": "way/66420168",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7078602,\n   "lon": 37.6911293,\n   "opening_hours": null\n  },\n  {\n   "id": "way/66420170",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7074423,\n   "lon": 37.6906008,\n   "opening_hours": null\n  },\n  {\n   "id": "way/67251887",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7458297,\n   "lon": 37.6730252,\n   "opening_hours": null\n  },\n  {\n   "id": "way/67251891",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7458222,\n   "lon": 37.6726831,\n   "opening_hours": null\n  },\n  {\n   "id": "way/67251893",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7457177,\n   "lon": 37.6713443,\n   "opening_hours": null\n  },\n  {\n   "id": "way/67522506",\n   "kind": "organisation",\n   "name": "ПФР, ГУ №7",\n   "office_type": "government",\n   "lat": 55.790287,\n   "lon": 37.8109542,\n   "opening_hours": "Mo-Th 09:00-18:00;Fr 09:00-16:00"\n  },\n  {\n   "id": "way/67522568",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8036597,\n   "lon": 37.8340613,\n   "opening_hours": null\n  },\n  {\n   "id": "way/67522669",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Восточное Измайлово",\n   "office_type": "government",\n   "lat": 55.7890686,\n   "lon": 37.8092527,\n   "opening_hours": null\n  },\n  {\n   "id": "way/67625265",\n   "kind": "organisation",\n   "name": "Российский союз ветеранов Афганистана",\n   "office_type": "ngo",\n   "lat": 55.6107839,\n   "lon": 37.6654428,\n   "opening_hours": null\n  },\n  {\n   "id": "way/67625274",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.6117227,\n   "lon": 37.6657765,\n   "opening_hours": null\n  },\n  {\n   "id": "way/68047427",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7558851,\n   "lon": 37.5874862,\n   "opening_hours": null\n  },\n  {\n   "id": "way/68047474",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7560971,\n   "lon": 37.5973645,\n   "opening_hours": null\n  },\n  {\n   "id": "way/68744733",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7084708,\n   "lon": 37.6025084,\n   "opening_hours": null\n  },\n  {\n   "id": "way/68744734",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7087408,\n   "lon": 37.6019602,\n   "opening_hours": null\n  },\n  {\n   "id": "way/72412660",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.800497,\n   "lon": 37.7616338,\n   "opening_hours": null\n  },\n  {\n   "id": "way/72412833",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7968769,\n   "lon": 37.7501125,\n   "opening_hours": null\n  },\n  {\n   "id": "way/72412998",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8005452,\n   "lon": 37.7596233,\n   "opening_hours": null\n  },\n  {\n   "id": "way/74076054",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7267719,\n   "lon": 37.6113958,\n   "opening_hours": null\n  },\n  {\n   "id": "way/74337871",\n   "kind": "organisation",\n   "name": "ВНИИнефть",\n   "office_type": "research",\n   "lat": 55.8092276,\n   "lon": 37.5723456,\n   "opening_hours": null\n  },\n  {\n   "id": "way/74747124",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7501228,\n   "lon": 37.7873735,\n   "opening_hours": null\n  },\n  {\n   "id": "way/74747144",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7732492,\n   "lon": 37.7406528,\n   "opening_hours": null\n  },\n  {\n   "id": "way/75398773",\n   "kind": "organisation",\n   "name": "Мосэнергосбыт",\n   "office_type": "company",\n   "lat": 55.8107986,\n   "lon": 37.6498677,\n   "opening_hours": "Mo-Th 08:30-12:30,13:00-17:15; Fr 08:30-12:30,13:00-16:00; Sa-Su off"\n  },\n  {\n   "id": "way/75512627",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7111898,\n   "lon": 37.6309876,\n   "opening_hours": null\n  },\n  {\n   "id": "way/75516826",\n   "kind": "organisation",\n   "name": "Отдел внешних церковных связей",\n   "office_type": "religion",\n   "lat": 55.7107314,\n   "lon": 37.628729,\n   "opening_hours": null\n  },\n  {\n   "id": "way/75517203",\n   "kind": "building",\n   "name": "Резиденция Патриарха Московского и всея Руси",\n   "office_type": "religion",\n   "lat": 55.7110687,\n   "lon": 37.6278505,\n   "opening_hours": null\n  },\n  {\n   "id": "way/75616519",\n   "kind": "organisation",\n   "name": "ООО \\"Газпромсвязь\\"",\n   "office_type": "company",\n   "lat": 55.5937096,\n   "lon": 37.737793,\n   "opening_hours": null\n  },\n  {\n   "id": "way/75714717",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.729541,\n   "lon": 37.6447281,\n   "opening_hours": null\n  },\n  {\n   "id": "way/75765894",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447153,\n   "lon": 37.8080596,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76439939",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6624243,\n   "lon": 37.5556631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76586358",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8037571,\n   "lon": 37.5635161,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76598375",\n   "kind": "building",\n   "name": "Центр промышленности Республики Болгария",\n   "office_type": null,\n   "lat": 55.6098356,\n   "lon": 37.5762533,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76618917",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7385574,\n   "lon": 37.595936,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76633174",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7380823,\n   "lon": 37.5986708,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76633188",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7382716,\n   "lon": 37.5983391,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76633192",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7369298,\n   "lon": 37.5980807,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76633216",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7405752,\n   "lon": 37.5985101,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76633221",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7371566,\n   "lon": 37.5958214,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76633241",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7369323,\n   "lon": 37.5969429,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76633299",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Kesko House\\"",\n   "office_type": null,\n   "lat": 55.7384781,\n   "lon": 37.5985816,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76633307",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7371419,\n   "lon": 37.5959985,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76633316",\n   "kind": "building",\n   "name": "Особняк Абрикосовых",\n   "office_type": null,\n   "lat": 55.7372973,\n   "lon": 37.5956113,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76726552",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8242612,\n   "lon": 37.8114473,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76739042",\n   "kind": "organisation",\n   "name": "ГУП \\"Москоллектор\\"",\n   "office_type": "company",\n   "lat": 55.6713008,\n   "lon": 37.5450654,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76739049",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6695016,\n   "lon": 37.5481786,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76776620",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8045654,\n   "lon": 37.5628697,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76783731",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7497395,\n   "lon": 37.5834104,\n   "opening_hours": null\n  },\n  {\n   "id": "way/76828698",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6403861,\n   "lon": 37.5923344,\n   "opening_hours": null\n  },\n  {\n   "id": "way/77341416",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7810372,\n   "lon": 37.7076865,\n   "opening_hours": null\n  },\n  {\n   "id": "way/77341429",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7816191,\n   "lon": 37.7078508,\n   "opening_hours": null\n  },\n  {\n   "id": "way/77352020",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7356203,\n   "lon": 37.6875984,\n   "opening_hours": null\n  },\n  {\n   "id": "way/77489473",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8029282,\n   "lon": 37.4991511,\n   "opening_hours": null\n  },\n  {\n   "id": "way/77738222",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8942625,\n   "lon": 37.5243206,\n   "opening_hours": null\n  },\n  {\n   "id": "way/78287902",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8597441,\n   "lon": 37.6794824,\n   "opening_hours": null\n  },\n  {\n   "id": "way/78386080",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6668801,\n   "lon": 37.6128099,\n   "opening_hours": null\n  },\n  {\n   "id": "way/78446883",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7378183,\n   "lon": 37.486593,\n   "opening_hours": null\n  },\n  {\n   "id": "way/78606895",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7550122,\n   "lon": 37.6058886,\n   "opening_hours": null\n  },\n  {\n   "id": "way/78606898",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.754949,\n   "lon": 37.6048722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/78606904",\n   "kind": "building",\n   "name": "Московская биржа",\n   "office_type": "company",\n   "lat": 55.7553367,\n   "lon": 37.6063919,\n   "opening_hours": null\n  },\n  {\n   "id": "way/78675137",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7502992,\n   "lon": 37.5090798,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79246883",\n   "kind": "building",\n   "name": "Департамент здравоохранения г. Москвы",\n   "office_type": "government",\n   "lat": 55.7761832,\n   "lon": 37.608899,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418043",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7458236,\n   "lon": 37.6337472,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418049",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.747676,\n   "lon": 37.6263274,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418054",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.747631,\n   "lon": 37.6259724,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418057",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7458709,\n   "lon": 37.6333897,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418754",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461217,\n   "lon": 37.6309459,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418755",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461571,\n   "lon": 37.6325595,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418756",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.74609,\n   "lon": 37.6313169,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418764",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7480649,\n   "lon": 37.6267537,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418765",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.746611,\n   "lon": 37.6314762,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79418768",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7463158,\n   "lon": 37.6316179,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79561179",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7802005,\n   "lon": 37.6140721,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79649123",\n   "kind": "building",\n   "name": "ГУП Автокомбинат санитарного транспорта «Мосавтосантранс» – автобаза № 2",\n   "office_type": null,\n   "lat": 55.6157633,\n   "lon": 37.6097845,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79683958",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.76437,\n   "lon": 37.6718987,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79685502",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Mosenka Plaza 3\\"",\n   "office_type": null,\n   "lat": 55.7732556,\n   "lon": 37.6204494,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79697339",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8143953,\n   "lon": 37.6016446,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79697348",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8153337,\n   "lon": 37.601643,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79869864",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Сорус Холл\\"",\n   "office_type": null,\n   "lat": 55.8183103,\n   "lon": 37.7698616,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79887485",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7589517,\n   "lon": 37.6637435,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79896994",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8151354,\n   "lon": 37.7871,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79897052",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8152717,\n   "lon": 37.7826467,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79968238",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7419519,\n   "lon": 37.6281886,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79969277",\n   "kind": "building",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.743421,\n   "lon": 37.6270904,\n   "opening_hours": null\n  },\n  {\n   "id": "way/79969279",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7425439,\n   "lon": 37.6280507,\n   "opening_hours": null\n  },\n  {\n   "id": "way/80015521",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7532536,\n   "lon": 37.543124,\n   "opening_hours": null\n  },\n  {\n   "id": "way/80306303",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7378457,\n   "lon": 37.5919764,\n   "opening_hours": null\n  },\n  {\n   "id": "way/80499632",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7385652,\n   "lon": 37.5440411,\n   "opening_hours": null\n  },\n  {\n   "id": "way/80499645",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7414294,\n   "lon": 37.5553456,\n   "opening_hours": null\n  },\n  {\n   "id": "way/80518532",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.766924,\n   "lon": 37.5628571,\n   "opening_hours": null\n  },\n  {\n   "id": "way/80591070",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7844875,\n   "lon": 37.7095552,\n   "opening_hours": null\n  },\n  {\n   "id": "way/80591096",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Улей-Плаза\\"",\n   "office_type": null,\n   "lat": 55.7844271,\n   "lon": 37.709732,\n   "opening_hours": null\n  },\n  {\n   "id": "way/80664559",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7868917,\n   "lon": 37.7093427,\n   "opening_hours": null\n  },\n  {\n   "id": "way/80746007",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7770786,\n   "lon": 37.6680271,\n   "opening_hours": null\n  },\n  {\n   "id": "way/81023760",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7845209,\n   "lon": 37.7081948,\n   "opening_hours": null\n  },\n  {\n   "id": "way/81107236",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.6149759,\n   "lon": 37.599211,\n   "opening_hours": null\n  },\n  {\n   "id": "way/81245107",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6403581,\n   "lon": 37.664555,\n   "opening_hours": null\n  },\n  {\n   "id": "way/81252071",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7208443,\n   "lon": 37.6476222,\n   "opening_hours": null\n  },\n  {\n   "id": "way/81252072",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7212665,\n   "lon": 37.6470875,\n   "opening_hours": null\n  },\n  {\n   "id": "way/81252076",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7211583,\n   "lon": 37.6476163,\n   "opening_hours": null\n  },\n  {\n   "id": "way/81356767",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7842052,\n   "lon": 37.7006512,\n   "opening_hours": null\n  },\n  {\n   "id": "way/81356768",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7836607,\n   "lon": 37.7000606,\n   "opening_hours": null\n  },\n  {\n   "id": "way/81519822",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6237222,\n   "lon": 37.6640868,\n   "opening_hours": null\n  },\n  {\n   "id": "way/82307159",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8108229,\n   "lon": 37.655559,\n   "opening_hours": null\n  },\n  {\n   "id": "way/82512621",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8097243,\n   "lon": 37.5712972,\n   "opening_hours": null\n  },\n  {\n   "id": "way/82531853",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7799107,\n   "lon": 37.6949448,\n   "opening_hours": null\n  },\n  {\n   "id": "way/82531901",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7780293,\n   "lon": 37.6932111,\n   "opening_hours": null\n  },\n  {\n   "id": "way/82567953",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6812568,\n   "lon": 37.626595,\n   "opening_hours": null\n  },\n  {\n   "id": "way/82682631",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7755677,\n   "lon": 37.5860429,\n   "opening_hours": null\n  },\n  {\n   "id": "way/82682675",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7754305,\n   "lon": 37.5857293,\n   "opening_hours": null\n  },\n  {\n   "id": "way/82820977",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7520241,\n   "lon": 37.6109561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83040952",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7387906,\n   "lon": 37.6655852,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83040956",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7388283,\n   "lon": 37.6652085,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83097795",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7752246,\n   "lon": 37.5854469,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83113147",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7518446,\n   "lon": 37.7241947,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83127816",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.808892,\n   "lon": 37.6386826,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83178437",\n   "kind": "organisation",\n   "name": "ОАО Мосэнергосбыт",\n   "office_type": "company",\n   "lat": 55.6043684,\n   "lon": 37.5955156,\n   "opening_hours": "Mo-Fr 08:30-19:00"\n  },\n  {\n   "id": "way/83201422",\n   "kind": "building",\n   "name": "ГУП г. Москвы ДЕЗ района Чертаново Центральное",\n   "office_type": null,\n   "lat": 55.6189697,\n   "lon": 37.6032623,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83210865",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7962947,\n   "lon": 37.6398049,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83290054",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.780699,\n   "lon": 37.4921531,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83290263",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7804895,\n   "lon": 37.4888804,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83425143",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7402513,\n   "lon": 37.5001901,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83700548",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7861901,\n   "lon": 37.5823759,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83700627",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7832703,\n   "lon": 37.5829149,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83700685",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7857154,\n   "lon": 37.5830986,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83700800",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7850015,\n   "lon": 37.584234,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83742578",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7863914,\n   "lon": 37.5846472,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83742589",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.78922,\n   "lon": 37.5833282,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83855155",\n   "kind": "building",\n   "name": "Ордена Ленина трест «Мосстрой-4»",\n   "office_type": "construction_company",\n   "lat": 55.7061759,\n   "lon": 37.5273601,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83864330",\n   "kind": "building",\n   "name": "Метеорологическая обсерватория МГУ",\n   "office_type": "research",\n   "lat": 55.7069835,\n   "lon": 37.5219341,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83864333",\n   "kind": "building",\n   "name": "Отдел флоры",\n   "office_type": null,\n   "lat": 55.7067036,\n   "lon": 37.527415,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83864336",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7069305,\n   "lon": 37.5269821,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83864347",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.707908,\n   "lon": 37.5217817,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83894557",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7271822,\n   "lon": 37.6098918,\n   "opening_hours": null\n  },\n  {\n   "id": "way/83897499",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6158844,\n   "lon": 37.6171595,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "way/83903003",\n   "kind": "building",\n   "name": "Речная администрация Московского Бассейна",\n   "office_type": "government",\n   "lat": 55.8162407,\n   "lon": 37.4479953,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84098981",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7542841,\n   "lon": 37.678703,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84217333",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8004567,\n   "lon": 37.6392501,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84653588",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7548656,\n   "lon": 37.6280888,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84653602",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7554878,\n   "lon": 37.6283284,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84653673",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.755443,\n   "lon": 37.6287306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84675512",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7569568,\n   "lon": 37.6270035,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84675555",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7558139,\n   "lon": 37.6271177,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84675585",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7577605,\n   "lon": 37.6276561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84675586",\n   "kind": "building",\n   "name": "Федеральное казначейство",\n   "office_type": "government",\n   "lat": 55.755361,\n   "lon": 37.6256676,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84675631",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7562613,\n   "lon": 37.6275847,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84675635",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7571525,\n   "lon": 37.628081,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84675636",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7561655,\n   "lon": 37.626748,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84675646",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7582685,\n   "lon": 37.6264555,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84690589",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7564211,\n   "lon": 37.6262315,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84807892",\n   "kind": "organisation",\n   "name": "Межрегиональная инспекция ФНС России по крупнейшим налогоплательщикам № 6",\n   "office_type": "government",\n   "lat": 55.8360182,\n   "lon": 37.5162694,\n   "opening_hours": null\n  },\n  {\n   "id": "way/84890806",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7551346,\n   "lon": 37.7305805,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85224642",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7433677,\n   "lon": 37.5196483,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85227424",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7070564,\n   "lon": 37.6528301,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85301929",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7919581,\n   "lon": 37.5998107,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85385274",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7717201,\n   "lon": 37.5542704,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85385292",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7702573,\n   "lon": 37.5565855,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85397709",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6966285,\n   "lon": 37.7905947,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85397751",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6967206,\n   "lon": 37.7899889,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85414931",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7761368,\n   "lon": 37.5566156,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85596175",\n   "kind": "building",\n   "name": "Институт Цветметобработка",\n   "office_type": null,\n   "lat": 55.7389934,\n   "lon": 37.6224342,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85604737",\n   "kind": "organisation",\n   "name": "Москонцерт",\n   "office_type": "theatre",\n   "lat": 55.7845581,\n   "lon": 37.5708218,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85633431",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7095492,\n   "lon": 37.6353976,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85639296",\n   "kind": "organisation",\n   "name": "Представительство Республики Коми при Президенте РФ",\n   "office_type": "government",\n   "lat": 55.8174564,\n   "lon": 37.4626561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85703103",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6260083,\n   "lon": 37.6483037,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85929967",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7932382,\n   "lon": 37.6979964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85929970",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7920775,\n   "lon": 37.6959856,\n   "opening_hours": null\n  },\n  {\n   "id": "way/85951507",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8796409,\n   "lon": 37.4914477,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86110998",\n   "kind": "organisation",\n   "name": "MetroGroup Russia",\n   "office_type": "company",\n   "lat": 55.8757141,\n   "lon": 37.4451784,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86177106",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7863261,\n   "lon": 37.6314152,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86380786",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8284843,\n   "lon": 37.4666777,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86386202",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7878495,\n   "lon": 37.6544093,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86412526",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7512129,\n   "lon": 37.5825282,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86412535",\n   "kind": "organisation",\n   "name": "Резиденция посла Испании",\n   "office_type": "diplomatic",\n   "lat": 55.7501865,\n   "lon": 37.5883287,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86481978",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7309726,\n   "lon": 37.5136641,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86971068",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7959499,\n   "lon": 37.579138,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86985775",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7794311,\n   "lon": 37.6624903,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86985778",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7789089,\n   "lon": 37.6614184,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86985781",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7799552,\n   "lon": 37.6632657,\n   "opening_hours": null\n  },\n  {\n   "id": "way/86985813",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7800447,\n   "lon": 37.6643007,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87127252",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7396505,\n   "lon": 37.5498989,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87130508",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8506657,\n   "lon": 37.4551132,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87212924",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.652733,\n   "lon": 37.5287896,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87276275",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7529518,\n   "lon": 37.6071619,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87276277",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7531611,\n   "lon": 37.6070971,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87349420",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7773016,\n   "lon": 37.6642556,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87349423",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7956306,\n   "lon": 37.6556741,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87349435",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.778271,\n   "lon": 37.6655571,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87349444",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7765168,\n   "lon": 37.6614751,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87349450",\n   "kind": "building",\n   "name": "Электрояговая подстанция №1",\n   "office_type": null,\n   "lat": 55.7773558,\n   "lon": 37.6621771,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87349453",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7788049,\n   "lon": 37.6667858,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87349460",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7768327,\n   "lon": 37.6643907,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87349462",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7775199,\n   "lon": 37.6654748,\n   "opening_hours": null\n  },\n  {\n   "id": "way/87349465",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7769081,\n   "lon": 37.6648905,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88092710",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6200603,\n   "lon": 37.7202392,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88238915",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7050229,\n   "lon": 37.6209737,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88408309",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7767398,\n   "lon": 37.6535023,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88416425",\n   "kind": "building",\n   "name": "Московское территориальное управление Октябрьской железной дороги",\n   "office_type": "transport",\n   "lat": 55.7833697,\n   "lon": 37.6525465,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88418092",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7748802,\n   "lon": 37.631491,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88418093",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7754523,\n   "lon": 37.630899,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88418094",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7754174,\n   "lon": 37.6316183,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88423799",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7747006,\n   "lon": 37.6319331,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88423821",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.773448,\n   "lon": 37.6314453,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88423833",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7741663,\n   "lon": 37.6307532,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88436452",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.778372,\n   "lon": 37.647373,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88436461",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7787118,\n   "lon": 37.6468137,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88440977",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7746132,\n   "lon": 37.629335,\n   "opening_hours": null\n  },\n  {\n   "id": "way/88440985",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7748557,\n   "lon": 37.6302058,\n   "opening_hours": null\n  },\n  {\n   "id": "way/89488662",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7970197,\n   "lon": 37.5914756,\n   "opening_hours": null\n  },\n  {\n   "id": "way/89488675",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7967662,\n   "lon": 37.5913167,\n   "opening_hours": null\n  },\n  {\n   "id": "way/89571699",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6840183,\n   "lon": 37.5574931,\n   "opening_hours": null\n  },\n  {\n   "id": "way/89823247",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7439716,\n   "lon": 37.6289521,\n   "opening_hours": null\n  },\n  {\n   "id": "way/90265767",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8230188,\n   "lon": 37.6254596,\n   "opening_hours": null\n  },\n  {\n   "id": "way/90982315",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.781159,\n   "lon": 37.609663,\n   "opening_hours": null\n  },\n  {\n   "id": "way/90982316",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7809221,\n   "lon": 37.6133085,\n   "opening_hours": null\n  },\n  {\n   "id": "way/90982326",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7807341,\n   "lon": 37.6118903,\n   "opening_hours": null\n  },\n  {\n   "id": "way/91144732",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8964414,\n   "lon": 37.58162,\n   "opening_hours": null\n  },\n  {\n   "id": "way/91209651",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.820035,\n   "lon": 37.6720712,\n   "opening_hours": null\n  },\n  {\n   "id": "way/91251476",\n   "kind": "organisation",\n   "name": "Прокуратура ЮАО",\n   "office_type": "government",\n   "lat": 55.6657974,\n   "lon": 37.6518119,\n   "opening_hours": null\n  },\n  {\n   "id": "way/91322210",\n   "kind": "building",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7934524,\n   "lon": 37.590486,\n   "opening_hours": null\n  },\n  {\n   "id": "way/91322214",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7932101,\n   "lon": 37.5912414,\n   "opening_hours": null\n  },\n  {\n   "id": "way/91684477",\n   "kind": "organisation",\n   "name": "Военная прокуратура",\n   "office_type": "government",\n   "lat": 55.8891679,\n   "lon": 37.4493579,\n   "opening_hours": null\n  },\n  {\n   "id": "way/91888927",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.5888486,\n   "lon": 37.6120817,\n   "opening_hours": null\n  },\n  {\n   "id": "way/92227422",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7462378,\n   "lon": 37.6219366,\n   "opening_hours": null\n  },\n  {\n   "id": "way/92227423",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7480919,\n   "lon": 37.6210824,\n   "opening_hours": null\n  },\n  {\n   "id": "way/92227434",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461306,\n   "lon": 37.6212003,\n   "opening_hours": null\n  },\n  {\n   "id": "way/92227438",\n   "kind": "building",\n   "name": "ПАО «НК «Роснефть»",\n   "office_type": "company",\n   "lat": 55.7474081,\n   "lon": 37.6201168,\n   "opening_hours": null\n  },\n  {\n   "id": "way/92227439",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7463709,\n   "lon": 37.6204906,\n   "opening_hours": null\n  },\n  {\n   "id": "way/92337937",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.6826982,\n   "lon": 37.577065,\n   "opening_hours": null\n  },\n  {\n   "id": "way/92341461",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7256748,\n   "lon": 37.5274379,\n   "opening_hours": null\n  },\n  {\n   "id": "way/92508993",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8109056,\n   "lon": 37.6471118,\n   "opening_hours": null\n  },\n  {\n   "id": "way/93029727",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626585,\n   "lon": 37.6320221,\n   "opening_hours": null\n  },\n  {\n   "id": "way/93449674",\n   "kind": "building",\n   "name": "Акадо Телеком (ОАО \'Комкор\')",\n   "office_type": null,\n   "lat": 55.6037197,\n   "lon": 37.608091,\n   "opening_hours": null\n  },\n  {\n   "id": "way/93473895",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7614304,\n   "lon": 37.7383537,\n   "opening_hours": null\n  },\n  {\n   "id": "way/93699976",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6171885,\n   "lon": 37.7213006,\n   "opening_hours": null\n  },\n  {\n   "id": "way/93950050",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7188806,\n   "lon": 37.6618864,\n   "opening_hours": null\n  },\n  {\n   "id": "way/94206785",\n   "kind": "organisation",\n   "name": "Перовский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7461811,\n   "lon": 37.7833572,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "way/94253580",\n   "kind": "organisation",\n   "name": "Международный литературный фонд",\n   "office_type": "association",\n   "lat": 55.8055522,\n   "lon": 37.5312718,\n   "opening_hours": null\n  },\n  {\n   "id": "way/94818189",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных услуг района Зюзино",\n   "office_type": "government",\n   "lat": 55.6579495,\n   "lon": 37.593594,\n   "opening_hours": "PH,Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "way/95200175",\n   "kind": "building",\n   "name": "Европа",\n   "office_type": null,\n   "lat": 55.727863,\n   "lon": 37.6137925,\n   "opening_hours": null\n  },\n  {\n   "id": "way/95200178",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7279058,\n   "lon": 37.6121748,\n   "opening_hours": null\n  },\n  {\n   "id": "way/95200179",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7280361,\n   "lon": 37.6126615,\n   "opening_hours": null\n  },\n  {\n   "id": "way/95200534",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7254731,\n   "lon": 37.6153876,\n   "opening_hours": null\n  },\n  {\n   "id": "way/95429955",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7399288,\n   "lon": 37.5224456,\n   "opening_hours": null\n  },\n  {\n   "id": "way/95474064",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7272931,\n   "lon": 37.5348051,\n   "opening_hours": null\n  },\n  {\n   "id": "way/95500775",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7966496,\n   "lon": 37.463074,\n   "opening_hours": null\n  },\n  {\n   "id": "way/95528140",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7840702,\n   "lon": 37.4994884,\n   "opening_hours": null\n  },\n  {\n   "id": "way/95636388",\n   "kind": "organisation",\n   "name": "Институт системного программирования им. В.П. Иванникова РАН",\n   "office_type": "research",\n   "lat": 55.7450651,\n   "lon": 37.6643506,\n   "opening_hours": "Mo-Fr 00:00-24:00"\n  },\n  {\n   "id": "way/95665378",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819655,\n   "lon": 37.6299313,\n   "opening_hours": null\n  },\n  {\n   "id": "way/95749464",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7662284,\n   "lon": 37.7324529,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96132015",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Gorky Park Tower\\"",\n   "office_type": null,\n   "lat": 55.7208344,\n   "lon": 37.6007046,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96255256",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7848494,\n   "lon": 37.6770312,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96266243",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7666209,\n   "lon": 37.7327056,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96300055",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8106651,\n   "lon": 37.7205376,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96475640",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7510307,\n   "lon": 37.6071558,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96538379",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6217289,\n   "lon": 37.4764919,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96538714",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.621745,\n   "lon": 37.4770051,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96630839",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7526348,\n   "lon": 37.679287,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96630860",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7523494,\n   "lon": 37.679092,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96636931",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7889872,\n   "lon": 37.4895365,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96636933",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7885808,\n   "lon": 37.4890018,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96653512",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6751842,\n   "lon": 37.4954347,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96653516",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6754722,\n   "lon": 37.4954726,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96739851",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7076824,\n   "lon": 37.5980133,\n   "opening_hours": null\n  },\n  {\n   "id": "way/96935135",\n   "kind": "organisation",\n   "name": "Банк России",\n   "office_type": "government",\n   "lat": 55.7629594,\n   "lon": 37.6208663,\n   "opening_hours": null\n  },\n  {\n   "id": "way/97152731",\n   "kind": "organisation",\n   "name": "Дирекция аварийно-восстановительных средств МЖД",\n   "office_type": "company",\n   "lat": 55.7825806,\n   "lon": 37.6562804,\n   "opening_hours": null\n  },\n  {\n   "id": "way/97201246",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6253584,\n   "lon": 37.6614522,\n   "opening_hours": null\n  },\n  {\n   "id": "way/97334900",\n   "kind": "organisation",\n   "name": "Мои документы Царицыно",\n   "office_type": "government",\n   "lat": 55.635853,\n   "lon": 37.6749368,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "way/97378871",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.787037,\n   "lon": 37.6311816,\n   "opening_hours": null\n  },\n  {\n   "id": "way/97513489",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6229829,\n   "lon": 37.648933,\n   "opening_hours": null\n  },\n  {\n   "id": "way/97703890",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Меридио\\"",\n   "office_type": null,\n   "lat": 55.6221426,\n   "lon": 37.621314,\n   "opening_hours": null\n  },\n  {\n   "id": "way/97905165",\n   "kind": "organisation",\n   "name": "ИФНС № 24",\n   "office_type": "government",\n   "lat": 55.6475204,\n   "lon": 37.6635418,\n   "opening_hours": null\n  },\n  {\n   "id": "way/97992224",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7289372,\n   "lon": 37.6191918,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98040685",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7414381,\n   "lon": 37.63306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98207080",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7482682,\n   "lon": 37.5835971,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98213311",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819531,\n   "lon": 37.5871851,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98213372",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819267,\n   "lon": 37.5871243,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98311005",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7729272,\n   "lon": 37.5999668,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98395257",\n   "kind": "organisation",\n   "name": "Управа района Нагатино-Садовники",\n   "office_type": "government",\n   "lat": 55.6622244,\n   "lon": 37.6445515,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572152",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7829938,\n   "lon": 37.5967414,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572171",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7823687,\n   "lon": 37.5959699,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572179",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7820219,\n   "lon": 37.5965309,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572184",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.780522,\n   "lon": 37.595359,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572185",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7818925,\n   "lon": 37.5962758,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572194",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7830262,\n   "lon": 37.5972622,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572204",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7806163,\n   "lon": 37.5949888,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572213",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7806327,\n   "lon": 37.5940259,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572229",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819766,\n   "lon": 37.5969758,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572245",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7822443,\n   "lon": 37.5964512,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572249",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7814576,\n   "lon": 37.5955625,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572257",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7816564,\n   "lon": 37.5963223,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572258",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.782194,\n   "lon": 37.5971514,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572264",\n   "kind": "organisation",\n   "name": "Федеральный фонд обязательного медицинского страхования",\n   "office_type": "company",\n   "lat": 55.7826398,\n   "lon": 37.5972794,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572271",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7812546,\n   "lon": 37.5969037,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572274",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7826294,\n   "lon": 37.5964793,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572275",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7816168,\n   "lon": 37.5947223,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572280",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7813623,\n   "lon": 37.5951171,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572293",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7823502,\n   "lon": 37.596355,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98572298",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7812717,\n   "lon": 37.5940578,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98598805",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6807793,\n   "lon": 37.6812903,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98673235",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8382389,\n   "lon": 37.4769608,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98713297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6750488,\n   "lon": 37.7512517,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98731486",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7861896,\n   "lon": 37.7371931,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98731488",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7865971,\n   "lon": 37.7377228,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98787683",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8047477,\n   "lon": 37.4914188,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98959477",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7495261,\n   "lon": 37.6055306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98961541",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7465272,\n   "lon": 37.6023747,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98961542",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7456854,\n   "lon": 37.6030481,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98961546",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7468293,\n   "lon": 37.6023098,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98985297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7560576,\n   "lon": 37.6083492,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98985300",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7571171,\n   "lon": 37.6107692,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98985302",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7592323,\n   "lon": 37.6097795,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98985305",\n   "kind": "building",\n   "name": "ГУ МВД России по Московской области",\n   "office_type": "government",\n   "lat": 55.7563557,\n   "lon": 37.6089969,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98985310",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.756723,\n   "lon": 37.6093712,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98985311",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7570561,\n   "lon": 37.6115664,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98985312",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.756872,\n   "lon": 37.6111415,\n   "opening_hours": null\n  },\n  {\n   "id": "way/98991837",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588439,\n   "lon": 37.6088948,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99029120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8870539,\n   "lon": 37.523432,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99150783",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7567814,\n   "lon": 37.6053447,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99150785",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7566805,\n   "lon": 37.6057631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99150793",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7565283,\n   "lon": 37.6064456,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99208596",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7655535,\n   "lon": 37.7134516,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99208598",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7658304,\n   "lon": 37.7146566,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99241324",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6928973,\n   "lon": 37.586984,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99312800",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7574441,\n   "lon": 37.6035607,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99312804",\n   "kind": "building",\n   "name": "Департамент жилищной политики и жилищного фонда города Москвы",\n   "office_type": "government",\n   "lat": 55.7567627,\n   "lon": 37.6066875,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99698177",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8104393,\n   "lon": 37.6449021,\n   "opening_hours": null\n  },\n  {\n   "id": "way/99998405",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7356058,\n   "lon": 37.6961283,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100025948",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7541192,\n   "lon": 37.7394402,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100054548",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Богородское и Метрогородок",\n   "office_type": "government",\n   "lat": 55.8115463,\n   "lon": 37.733461,\n   "opening_hours": "Mo-Sa 08:00-20:00"\n  },\n  {\n   "id": "way/100107174",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.778547,\n   "lon": 37.7155305,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100134908",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7971448,\n   "lon": 37.6924262,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100134912",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7978463,\n   "lon": 37.6938879,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100134915",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.796014,\n   "lon": 37.6922706,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100480804",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7543462,\n   "lon": 37.6048029,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100480817",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7552221,\n   "lon": 37.6057736,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100480836",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588753,\n   "lon": 37.6058911,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100480855",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7570356,\n   "lon": 37.6068001,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100480873",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7572609,\n   "lon": 37.6064619,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100616325",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6902278,\n   "lon": 37.5783197,\n   "opening_hours": null\n  },\n  {\n   "id": "way/100704274",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7204377,\n   "lon": 37.6214077,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101077202",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6776054,\n   "lon": 37.4961964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101358831",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7727452,\n   "lon": 37.6877927,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101579740",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7458181,\n   "lon": 37.6019087,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101579762",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7484007,\n   "lon": 37.601445,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101579773",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7465037,\n   "lon": 37.6008876,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101579806",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.748333,\n   "lon": 37.6018472,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101579808",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7457363,\n   "lon": 37.6022509,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101579812",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.745836,\n   "lon": 37.6021258,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101579819",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7456387,\n   "lon": 37.6023446,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101579821",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7456403,\n   "lon": 37.6019861,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101579842",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7476586,\n   "lon": 37.6013693,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101591607",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7677299,\n   "lon": 37.5542479,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101592254",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7676249,\n   "lon": 37.5536989,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101592611",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7678665,\n   "lon": 37.5540797,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101616975",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7482292,\n   "lon": 37.6016996,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101820244",\n   "kind": "organisation",\n   "name": "Центр \\"Антистихия\\" МЧС России",\n   "office_type": "government",\n   "lat": 55.6242884,\n   "lon": 37.6591225,\n   "opening_hours": null\n  },\n  {\n   "id": "way/101846349",\n   "kind": "organisation",\n   "name": "Центральное управление филиала Энергучет МОЭСК",\n   "office_type": "energy_supplier",\n   "lat": 55.6603156,\n   "lon": 37.6338028,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-16:00"\n  },\n  {\n   "id": "way/102225977",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7194129,\n   "lon": 37.6116605,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102226041",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7186167,\n   "lon": 37.6152607,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102226052",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7208465,\n   "lon": 37.6112016,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102226082",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.720036,\n   "lon": 37.6105046,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102226084",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7169909,\n   "lon": 37.6115822,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102226093",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7208383,\n   "lon": 37.6120345,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102226095",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7164485,\n   "lon": 37.6151948,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102226118",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7205199,\n   "lon": 37.611735,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102716896",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7524188,\n   "lon": 37.455242,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102747694",\n   "kind": "building",\n   "name": "Бюро ЮНЕСКО в Москве",\n   "office_type": "ngo",\n   "lat": 55.7417464,\n   "lon": 37.5873063,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102747707",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7384248,\n   "lon": 37.5900831,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102747717",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7419117,\n   "lon": 37.5875929,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102747718",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.73999,\n   "lon": 37.5885406,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102749785",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7378367,\n   "lon": 37.5931508,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102749798",\n   "kind": "organisation",\n   "name": "Государственный центр русского фольклора",\n   "office_type": "research",\n   "lat": 55.7383888,\n   "lon": 37.5925895,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102784187",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7279025,\n   "lon": 37.6243065,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102784190",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7287032,\n   "lon": 37.6239971,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102784202",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.72781,\n   "lon": 37.6235621,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102784214",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7289132,\n   "lon": 37.623806,\n   "opening_hours": null\n  },\n  {\n   "id": "way/102938539",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7988097,\n   "lon": 37.5304067,\n   "opening_hours": null\n  },\n  {\n   "id": "way/103267476",\n   "kind": "organisation",\n   "name": "Компания ВТТ Высокие технологии тонера",\n   "office_type": "company",\n   "lat": 55.8054662,\n   "lon": 37.7911701,\n   "opening_hours": null\n  },\n  {\n   "id": "way/103267478",\n   "kind": "organisation",\n   "name": "Управа района Северное Измайлово",\n   "office_type": "government",\n   "lat": 55.804418,\n   "lon": 37.7892082,\n   "opening_hours": null\n  },\n  {\n   "id": "way/103329996",\n   "kind": "organisation",\n   "name": "Инженерная служба района Северное Измайлово",\n   "office_type": "property_management",\n   "lat": 55.8074196,\n   "lon": 37.801175,\n   "opening_hours": null\n  },\n  {\n   "id": "way/103565837",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.746022,\n   "lon": 37.6588342,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104105393",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Южное Медведково",\n   "office_type": "government",\n   "lat": 55.8718136,\n   "lon": 37.638693,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "way/104112615",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.78648,\n   "lon": 37.7369177,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104122834",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7986159,\n   "lon": 37.7057724,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104179361",\n   "kind": "building",\n   "name": "Посольство Габона",\n   "office_type": "diplomatic",\n   "lat": 55.7448382,\n   "lon": 37.5871924,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104179370",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449082,\n   "lon": 37.5839754,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104179382",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7437245,\n   "lon": 37.5874162,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104179427",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7451853,\n   "lon": 37.5868992,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104179436",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7441673,\n   "lon": 37.5862482,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104201985",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.739239,\n   "lon": 37.5882673,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104201989",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7391644,\n   "lon": 37.5890988,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104201990",\n   "kind": "building",\n   "name": "Посольство Гвинеи",\n   "office_type": "diplomatic",\n   "lat": 55.7392975,\n   "lon": 37.5893955,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104201991",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7421402,\n   "lon": 37.5815984,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104202004",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447906,\n   "lon": 37.5845767,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104202016",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7391683,\n   "lon": 37.5880463,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104202018",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7392616,\n   "lon": 37.5884266,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104202024",\n   "kind": "building",\n   "name": "Усадьба Самсонова",\n   "office_type": null,\n   "lat": 55.7394018,\n   "lon": 37.5887806,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104472815",\n   "kind": "organisation",\n   "name": "Управа района «Очаково-Матвеевское»",\n   "office_type": "government",\n   "lat": 55.6882182,\n   "lon": 37.4634923,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104708946",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.816168,\n   "lon": 37.6942811,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104728161",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7921566,\n   "lon": 37.5492426,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104728162",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7917813,\n   "lon": 37.5500275,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104739136",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7143654,\n   "lon": 37.5803081,\n   "opening_hours": null\n  },\n  {\n   "id": "way/104739138",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.714686,\n   "lon": 37.5811469,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105208003",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7422631,\n   "lon": 37.5966964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105296533",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7402601,\n   "lon": 37.4838791,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105351515",\n   "kind": "organisation",\n   "name": "Представительство Самарской области",\n   "office_type": "government",\n   "lat": 55.7941676,\n   "lon": 37.580298,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105407861",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7208204,\n   "lon": 37.6072638,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105407864",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.720225,\n   "lon": 37.6075063,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105658741",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8005089,\n   "lon": 37.7066762,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105713559",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7769898,\n   "lon": 37.5023776,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105920654",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7579828,\n   "lon": 37.7155195,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105920656",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7590196,\n   "lon": 37.7157497,\n   "opening_hours": null\n  },\n  {\n   "id": "way/105920657",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7582885,\n   "lon": 37.7156812,\n   "opening_hours": null\n  },\n  {\n   "id": "way/106182898",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6338277,\n   "lon": 37.7529434,\n   "opening_hours": null\n  },\n  {\n   "id": "way/106575353",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.6349372,\n   "lon": 37.770129,\n   "opening_hours": null\n  },\n  {\n   "id": "way/106619073",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7773591,\n   "lon": 37.5046022,\n   "opening_hours": null\n  },\n  {\n   "id": "way/106619078",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7775468,\n   "lon": 37.5031988,\n   "opening_hours": null\n  },\n  {\n   "id": "way/106701709",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7093452,\n   "lon": 37.6545722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/106994992",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7442059,\n   "lon": 37.6488864,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107040049",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7414437,\n   "lon": 37.5978288,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107040052",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7418653,\n   "lon": 37.5973971,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107328227",\n   "kind": "building",\n   "name": "Деловой центр «Гелиос Сити»",\n   "office_type": null,\n   "lat": 55.75015,\n   "lon": 37.7704095,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107331194",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7503145,\n   "lon": 37.7731327,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107331199",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7497363,\n   "lon": 37.769914,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107340831",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.735453,\n   "lon": 37.6140846,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107539464",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7606775,\n   "lon": 37.783726,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107572768",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.686864,\n   "lon": 37.5704293,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107673086",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7949065,\n   "lon": 37.7091518,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107699032",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7936182,\n   "lon": 37.6957896,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107699037",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7918729,\n   "lon": 37.6921175,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107699047",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7916592,\n   "lon": 37.6923836,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107699064",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7958,\n   "lon": 37.6984694,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107699071",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7931107,\n   "lon": 37.6940376,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107943503",\n   "kind": "organisation",\n   "name": "Управа района Измайлово Восточное",\n   "office_type": "government",\n   "lat": 55.7962056,\n   "lon": 37.8178335,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107957959",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7907694,\n   "lon": 37.775085,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107959081",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7912841,\n   "lon": 37.7722713,\n   "opening_hours": null\n  },\n  {\n   "id": "way/107984126",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7979199,\n   "lon": 37.59204,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108082413",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8124766,\n   "lon": 37.8147842,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108098316",\n   "kind": "building",\n   "name": "МОЭСК",\n   "office_type": null,\n   "lat": 55.8145283,\n   "lon": 37.7949938,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108201825",\n   "kind": "organisation",\n   "name": "Институт эпидемиологии и микробиологии имени Н.Ф. Гамалеи РАМН",\n   "office_type": "research",\n   "lat": 55.8018077,\n   "lon": 37.4531168,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108581515",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7953341,\n   "lon": 37.7713169,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108584080",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.795987,\n   "lon": 37.7737408,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108584081",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7967063,\n   "lon": 37.7718895,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108584593",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7966482,\n   "lon": 37.7729486,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108593197",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7915409,\n   "lon": 37.8161856,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108680685",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.701805,\n   "lon": 37.6579779,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108880167",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7357804,\n   "lon": 37.5867135,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108880195",\n   "kind": "organisation",\n   "name": "Федеральная служба по интеллектуальной собственности",\n   "office_type": "government",\n   "lat": 55.7314396,\n   "lon": 37.55122,\n   "opening_hours": "Mo-Th 09:30-18:15; Fr 09:30-17:00"\n  },\n  {\n   "id": "way/108880209",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7374694,\n   "lon": 37.586531,\n   "opening_hours": null\n  },\n  {\n   "id": "way/108880227",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7374953,\n   "lon": 37.5863034,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109067903",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7908463,\n   "lon": 37.7827664,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109067968",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7919629,\n   "lon": 37.78455,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109081780",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.778748,\n   "lon": 37.659407,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109089979",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7855808,\n   "lon": 37.612643,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109127643",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7815766,\n   "lon": 37.6129495,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109127646",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.786404,\n   "lon": 37.6088334,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109127674",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.7873963,\n   "lon": 37.6099667,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109127683",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7864835,\n   "lon": 37.6085169,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109200889",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7818172,\n   "lon": 37.6668351,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109209561",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7864375,\n   "lon": 37.7253708,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109209565",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7865683,\n   "lon": 37.7287655,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109215221",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7287857,\n   "lon": 37.5677475,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109232765",\n   "kind": "building",\n   "name": "Дендрарий Ботанического сада МГУ",\n   "office_type": null,\n   "lat": 55.7065704,\n   "lon": 37.5279466,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109393870",\n   "kind": "organisation",\n   "name": "Дорогомиловская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7391693,\n   "lon": 37.5335035,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109397423",\n   "kind": "organisation",\n   "name": "Представительство республики Калмыкия",\n   "office_type": "government",\n   "lat": 55.7364257,\n   "lon": 37.526998,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109530936",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7885936,\n   "lon": 37.5785225,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109554112",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7458927,\n   "lon": 37.6470016,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109743497",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7850643,\n   "lon": 37.6555391,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109743508",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7849085,\n   "lon": 37.6558643,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109743512",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7855188,\n   "lon": 37.6553351,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109743523",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7847181,\n   "lon": 37.6560888,\n   "opening_hours": null\n  },\n  {\n   "id": "way/109939339",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7801429,\n   "lon": 37.634732,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110275254",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7937087,\n   "lon": 37.6971212,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110591227",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7589469,\n   "lon": 37.6644727,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110591250",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7597603,\n   "lon": 37.6649721,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110601281",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7544971,\n   "lon": 37.6623845,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110666790",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7366163,\n   "lon": 37.6307021,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110666795",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7363576,\n   "lon": 37.6294028,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110675894",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7399338,\n   "lon": 37.6290278,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110675960",\n   "kind": "organisation",\n   "name": "Посольство Индонезии",\n   "office_type": "diplomatic",\n   "lat": 55.7379733,\n   "lon": 37.6300476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110894028",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7356746,\n   "lon": 37.6327529,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110894158",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7346346,\n   "lon": 37.6330643,\n   "opening_hours": null\n  },\n  {\n   "id": "way/110896237",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7388213,\n   "lon": 37.6310278,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111243095",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7070548,\n   "lon": 37.5975195,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111243102",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7086987,\n   "lon": 37.5978976,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111243103",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7081178,\n   "lon": 37.5983801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111243124",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7084829,\n   "lon": 37.5984542,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111243142",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7065022,\n   "lon": 37.5979357,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111585287",\n   "kind": "organisation",\n   "name": "Институт астрономии Российской академии наук",\n   "office_type": "research",\n   "lat": 55.7358836,\n   "lon": 37.6270308,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111585483",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7368778,\n   "lon": 37.6288782,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111587795",\n   "kind": "building",\n   "name": "Дом.РФ",\n   "office_type": null,\n   "lat": 55.7322523,\n   "lon": 37.6286483,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "way/111601797",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7929692,\n   "lon": 37.6863022,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111648714",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.714815,\n   "lon": 37.6771763,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111652510",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7524236,\n   "lon": 37.6697627,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111652512",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7522189,\n   "lon": 37.6683381,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111652514",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.751726,\n   "lon": 37.6695709,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111652525",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7527476,\n   "lon": 37.6692798,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111652528",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7526833,\n   "lon": 37.6688787,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111652529",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7531328,\n   "lon": 37.6672636,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111652541",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7528631,\n   "lon": 37.6679123,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111652553",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7526039,\n   "lon": 37.6698136,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111652567",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7520993,\n   "lon": 37.6699706,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111719883",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7060118,\n   "lon": 37.5974628,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111719884",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7069562,\n   "lon": 37.5983682,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111790462",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7442169,\n   "lon": 37.6385324,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111790463",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7446456,\n   "lon": 37.6369467,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111790466",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7441736,\n   "lon": 37.6391458,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111790467",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7443955,\n   "lon": 37.6383125,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111833111",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7074301,\n   "lon": 37.5988557,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111859143",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7368751,\n   "lon": 37.6894656,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111859163",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7364307,\n   "lon": 37.6921287,\n   "opening_hours": null\n  },\n  {\n   "id": "way/111996097",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7429677,\n   "lon": 37.528964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/112358863",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7764191,\n   "lon": 37.6521653,\n   "opening_hours": null\n  },\n  {\n   "id": "way/112435822",\n   "kind": "building",\n   "name": "Диспетчерский пункт ГКУ ЦОДД",\n   "office_type": null,\n   "lat": 55.8785717,\n   "lon": 37.7214389,\n   "opening_hours": null\n  },\n  {\n   "id": "way/112772089",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8355329,\n   "lon": 37.5060292,\n   "opening_hours": null\n  },\n  {\n   "id": "way/112984997",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7890322,\n   "lon": 37.7420253,\n   "opening_hours": null\n  },\n  {\n   "id": "way/113653319",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7717068,\n   "lon": 37.5772751,\n   "opening_hours": null\n  },\n  {\n   "id": "way/114984614",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.744589,\n   "lon": 37.6315609,\n   "opening_hours": null\n  },\n  {\n   "id": "way/114995690",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7385819,\n   "lon": 37.6718651,\n   "opening_hours": null\n  },\n  {\n   "id": "way/115750806",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7733314,\n   "lon": 37.5869715,\n   "opening_hours": null\n  },\n  {\n   "id": "way/115750909",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7731718,\n   "lon": 37.5872653,\n   "opening_hours": null\n  },\n  {\n   "id": "way/115827004",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.758607,\n   "lon": 37.7350315,\n   "opening_hours": null\n  },\n  {\n   "id": "way/115827008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591181,\n   "lon": 37.7370529,\n   "opening_hours": null\n  },\n  {\n   "id": "way/115827013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7571083,\n   "lon": 37.7367602,\n   "opening_hours": null\n  },\n  {\n   "id": "way/115827022",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7579212,\n   "lon": 37.7366121,\n   "opening_hours": null\n  },\n  {\n   "id": "way/115863603",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.759433,\n   "lon": 37.7547943,\n   "opening_hours": null\n  },\n  {\n   "id": "way/116966538",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7448323,\n   "lon": 37.6502579,\n   "opening_hours": null\n  },\n  {\n   "id": "way/117125908",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7847354,\n   "lon": 37.6838824,\n   "opening_hours": null\n  },\n  {\n   "id": "way/117419270",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7088761,\n   "lon": 37.6029854,\n   "opening_hours": null\n  },\n  {\n   "id": "way/117499896",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8496547,\n   "lon": 37.4550679,\n   "opening_hours": null\n  },\n  {\n   "id": "way/117824346",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7248228,\n   "lon": 37.643946,\n   "opening_hours": null\n  },\n  {\n   "id": "way/117824347",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7242631,\n   "lon": 37.6428417,\n   "opening_hours": null\n  },\n  {\n   "id": "way/117972245",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7625494,\n   "lon": 37.4787319,\n   "opening_hours": null\n  },\n  {\n   "id": "way/118939820",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7864623,\n   "lon": 37.6749932,\n   "opening_hours": null\n  },\n  {\n   "id": "way/118939826",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7862911,\n   "lon": 37.6750621,\n   "opening_hours": null\n  },\n  {\n   "id": "way/118939827",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7867508,\n   "lon": 37.6739208,\n   "opening_hours": null\n  },\n  {\n   "id": "way/119029564",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7701648,\n   "lon": 37.7117707,\n   "opening_hours": null\n  },\n  {\n   "id": "way/120689353",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7820526,\n   "lon": 37.7030063,\n   "opening_hours": null\n  },\n  {\n   "id": "way/121289693",\n   "kind": "building",\n   "name": "Бизнес-центр «The Yard»",\n   "office_type": null,\n   "lat": 55.7662758,\n   "lon": 37.526826,\n   "opening_hours": null\n  },\n  {\n   "id": "way/121351103",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7543704,\n   "lon": 37.6043957,\n   "opening_hours": null\n  },\n  {\n   "id": "way/121351105",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.754353,\n   "lon": 37.6059578,\n   "opening_hours": null\n  },\n  {\n   "id": "way/121352376",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7551991,\n   "lon": 37.6039736,\n   "opening_hours": null\n  },\n  {\n   "id": "way/121352380",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7553853,\n   "lon": 37.6040268,\n   "opening_hours": null\n  },\n  {\n   "id": "way/121352382",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7555046,\n   "lon": 37.6033563,\n   "opening_hours": null\n  },\n  {\n   "id": "way/122222238",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7339028,\n   "lon": 37.539124,\n   "opening_hours": null\n  },\n  {\n   "id": "way/122228837",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6834243,\n   "lon": 37.593758,\n   "opening_hours": null\n  },\n  {\n   "id": "way/122229818",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6798307,\n   "lon": 37.5949159,\n   "opening_hours": null\n  },\n  {\n   "id": "way/122231129",\n   "kind": "organisation",\n   "name": "Управа района Бирюлево Восточное",\n   "office_type": "government",\n   "lat": 55.5849362,\n   "lon": 37.6639411,\n   "opening_hours": null\n  },\n  {\n   "id": "way/122638764",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7062783,\n   "lon": 37.6535425,\n   "opening_hours": null\n  },\n  {\n   "id": "way/122638765",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7052568,\n   "lon": 37.6518583,\n   "opening_hours": null\n  },\n  {\n   "id": "way/122638768",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7051235,\n   "lon": 37.6529048,\n   "opening_hours": null\n  },\n  {\n   "id": "way/123075136",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8030169,\n   "lon": 37.5601195,\n   "opening_hours": null\n  },\n  {\n   "id": "way/123186490",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8825968,\n   "lon": 37.4856164,\n   "opening_hours": null\n  },\n  {\n   "id": "way/123314617",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7955503,\n   "lon": 37.6953324,\n   "opening_hours": null\n  },\n  {\n   "id": "way/123587403",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6804401,\n   "lon": 37.5867806,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124283219",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7703492,\n   "lon": 37.6409087,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124365450",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7830354,\n   "lon": 37.7055442,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124365451",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7870228,\n   "lon": 37.7070611,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124365454",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7920476,\n   "lon": 37.7097167,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124365455",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7836388,\n   "lon": 37.7037103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124365460",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7922142,\n   "lon": 37.7096238,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124365465",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.792887,\n   "lon": 37.7091349,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124365466",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7932876,\n   "lon": 37.7092918,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124365467",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7932075,\n   "lon": 37.7084032,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124365469",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7924276,\n   "lon": 37.709129,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124540748",\n   "kind": "organisation",\n   "name": "Отделение УФМС по району \\"Алексеевский\\"",\n   "office_type": "government",\n   "lat": 55.8082282,\n   "lon": 37.6437883,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124697446",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6851886,\n   "lon": 37.594563,\n   "opening_hours": null\n  },\n  {\n   "id": "way/124793957",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7025443,\n   "lon": 37.5920699,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125067048",\n   "kind": "organisation",\n   "name": "Типография Внешторгиздат",\n   "office_type": "company",\n   "lat": 55.889489,\n   "lon": 37.5786414,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125105439",\n   "kind": "organisation",\n   "name": "Институт глобального климата и экологии",\n   "office_type": "research",\n   "lat": 55.8140177,\n   "lon": 37.7174207,\n   "opening_hours": "Mo-Fr 08:00-20:00"\n  },\n  {\n   "id": "way/125111829",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.781969,\n   "lon": 37.5911389,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125308193",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Паскаль\\"",\n   "office_type": null,\n   "lat": 55.6944893,\n   "lon": 37.6620527,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125452592",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7693681,\n   "lon": 37.50773,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125667351",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7824188,\n   "lon": 37.5986774,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125667369",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Европа\\"",\n   "office_type": null,\n   "lat": 55.7828314,\n   "lon": 37.5985103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125674933",\n   "kind": "building",\n   "name": "Деловой центр \\"Чайка\\"",\n   "office_type": null,\n   "lat": 55.7825522,\n   "lon": 37.6298631,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125674936",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7780068,\n   "lon": 37.631683,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125674939",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7792678,\n   "lon": 37.631168,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125674944",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7791409,\n   "lon": 37.6336053,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125674949",\n   "kind": "building",\n   "name": "Кассы спорт-комплекса \\"Олимпийский\\"",\n   "office_type": null,\n   "lat": 55.7813953,\n   "lon": 37.6289022,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125674950",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7823855,\n   "lon": 37.6288833,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125674951",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7839467,\n   "lon": 37.6287329,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125674958",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7799229,\n   "lon": 37.6314469,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125849315",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7845937,\n   "lon": 37.6007717,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125849327",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7828562,\n   "lon": 37.5997895,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125849335",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7840232,\n   "lon": 37.6000341,\n   "opening_hours": null\n  },\n  {\n   "id": "way/125983154",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7980651,\n   "lon": 37.6878733,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126059620",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7746008,\n   "lon": 37.6300857,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126064401",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7713646,\n   "lon": 37.6331222,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126064402",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.771881,\n   "lon": 37.6358927,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126064405",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7697467,\n   "lon": 37.632862,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126064408",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7710367,\n   "lon": 37.6327352,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126064411",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7713763,\n   "lon": 37.6327142,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126064412",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7713176,\n   "lon": 37.6338571,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126064416",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7713693,\n   "lon": 37.6333901,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126097691",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8049068,\n   "lon": 37.5702106,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126097696",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8045917,\n   "lon": 37.5705789,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126097700",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8045163,\n   "lon": 37.5691105,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126101538",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8040187,\n   "lon": 37.5687013,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126318340",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.678243,\n   "lon": 37.5765815,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126382643",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7374901,\n   "lon": 37.6601946,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126464530",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.767836,\n   "lon": 37.5623245,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126464541",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.766539,\n   "lon": 37.562678,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126603519",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8083928,\n   "lon": 37.5662621,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126764679",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7120508,\n   "lon": 37.6295325,\n   "opening_hours": null\n  },\n  {\n   "id": "way/126880584",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.769083,\n   "lon": 37.5116751,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127192099",\n   "kind": "organisation",\n   "name": "КПП-4",\n   "office_type": "security",\n   "lat": 55.6653764,\n   "lon": 37.6598592,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127192631",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8688343,\n   "lon": 37.4988713,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127321476",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7824433,\n   "lon": 37.6935619,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127467642",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7273677,\n   "lon": 37.6460501,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127632239",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7924583,\n   "lon": 37.7114519,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127632244",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7906246,\n   "lon": 37.7123275,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127632245",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7914124,\n   "lon": 37.7103621,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127717485",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Троицкий\\"",\n   "office_type": null,\n   "lat": 55.7767784,\n   "lon": 37.6215173,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127742882",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7351175,\n   "lon": 37.590004,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127868206",\n   "kind": "building",\n   "name": "ОАО \\"МОЭК\\" филиал № 7 \\"Юго-Западный\\"",\n   "office_type": null,\n   "lat": 55.6568211,\n   "lon": 37.595878,\n   "opening_hours": null\n  },\n  {\n   "id": "way/127875665",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7781061,\n   "lon": 37.6283963,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128110843",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.5940283,\n   "lon": 37.7439844,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128202648",\n   "kind": "building",\n   "name": "ПАО «Новатэк»",\n   "office_type": "company",\n   "lat": 55.6717198,\n   "lon": 37.5197766,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128256913",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8214501,\n   "lon": 37.5093591,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128264421",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7791997,\n   "lon": 37.6316573,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128287757",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7160392,\n   "lon": 37.5682147,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128323709",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588287,\n   "lon": 37.5870884,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128480316",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8540185,\n   "lon": 37.6030162,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128480606",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8560468,\n   "lon": 37.6008944,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128545252",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7493539,\n   "lon": 37.7709941,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128764522",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7366385,\n   "lon": 37.620605,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128890854",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7116241,\n   "lon": 37.5584438,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128942734",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7134895,\n   "lon": 37.5686413,\n   "opening_hours": null\n  },\n  {\n   "id": "way/128945744",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7158513,\n   "lon": 37.5640111,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129110194",\n   "kind": "organisation",\n   "name": "Pony Express",\n   "office_type": "logistics",\n   "lat": 55.7690956,\n   "lon": 37.4993908,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129212968",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7435662,\n   "lon": 37.5969715,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129228244",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7424818,\n   "lon": 37.5947132,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129228259",\n   "kind": "organisation",\n   "name": "Главное управление МЧС России по городу Москве",\n   "office_type": "government",\n   "lat": 55.7421135,\n   "lon": 37.594256,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129228275",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7424183,\n   "lon": 37.5946294,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129228278",\n   "kind": "organisation",\n   "name": "Центр управления в кризисных ситуациях Главного управления МЧС России по г. Москве",\n   "office_type": "government",\n   "lat": 55.7424765,\n   "lon": 37.5930753,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129244134",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7912289,\n   "lon": 37.6922761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129244579",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7939324,\n   "lon": 37.6985944,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129247247",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7417828,\n   "lon": 37.5877557,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129371700",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6939401,\n   "lon": 37.6125444,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129379450",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7392607,\n   "lon": 37.5984333,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129406265",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7989312,\n   "lon": 37.5312056,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129408551",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7024599,\n   "lon": 37.662408,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129435121",\n   "kind": "building",\n   "name": "корпус Б",\n   "office_type": null,\n   "lat": 55.8101038,\n   "lon": 37.5090519,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129496253",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7712755,\n   "lon": 37.6385618,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129724345",\n   "kind": "organisation",\n   "name": "ОДС района Люблино",\n   "office_type": "property_management",\n   "lat": 55.6725675,\n   "lon": 37.7627401,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129776713",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.731561,\n   "lon": 37.621306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129776714",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7316857,\n   "lon": 37.6216955,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129778988",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.73079,\n   "lon": 37.6227483,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129780891",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7308498,\n   "lon": 37.6215698,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129920063",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7923625,\n   "lon": 37.6757938,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129920064",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7915096,\n   "lon": 37.6768784,\n   "opening_hours": null\n  },\n  {\n   "id": "way/129920094",\n   "kind": "organisation",\n   "name": "Инспекция ФНС №18 по г. Москве",\n   "office_type": "government",\n   "lat": 55.7876114,\n   "lon": 37.6720593,\n   "opening_hours": "Mo,We 09:00-18:00; Tu,Th 09:00-20:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "way/130002060",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7204715,\n   "lon": 37.6155763,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130004089",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7262395,\n   "lon": 37.6887935,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130004091",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7285364,\n   "lon": 37.6901591,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130015402",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7187196,\n   "lon": 37.6280962,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130015404",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.718901,\n   "lon": 37.6280955,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130015406",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7189201,\n   "lon": 37.6284308,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130017250",\n   "kind": "building",\n   "name": "Бизнес-центр \\"М-Стиль Офис\\"",\n   "office_type": null,\n   "lat": 55.7196029,\n   "lon": 37.6296233,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130067108",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7576898,\n   "lon": 37.6019134,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130126806",\n   "kind": "building",\n   "name": "Лидер-Групп",\n   "office_type": null,\n   "lat": 55.7678071,\n   "lon": 37.556533,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130128258",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7657732,\n   "lon": 37.5575185,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130128260",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7660302,\n   "lon": 37.5570511,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130154997",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7352272,\n   "lon": 37.6363039,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130224879",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7213285,\n   "lon": 37.7709208,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130228969",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7724457,\n   "lon": 37.7146161,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130228970",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.776326,\n   "lon": 37.7212576,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130228976",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7774604,\n   "lon": 37.7190844,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130229010",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7731557,\n   "lon": 37.713532,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130296400",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7986645,\n   "lon": 37.5660355,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130297622",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7742439,\n   "lon": 37.6743365,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130397891",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7061716,\n   "lon": 37.7722961,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130537916",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7783694,\n   "lon": 37.7139801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130537917",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7808804,\n   "lon": 37.714226,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130538869",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7754457,\n   "lon": 37.6886949,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130577084",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7984238,\n   "lon": 37.6072017,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130577096",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7973761,\n   "lon": 37.603769,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130581309",\n   "kind": "organisation",\n   "name": "Управа района Марьина роща города Москвы",\n   "office_type": "government",\n   "lat": 55.7950425,\n   "lon": 37.6088292,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45; Sa,Su off"\n  },\n  {\n   "id": "way/130642284",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7308644,\n   "lon": 37.6454151,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130645096",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7774693,\n   "lon": 37.5912761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130645633",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7716258,\n   "lon": 37.6159397,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130645635",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7714114,\n   "lon": 37.6136552,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130648749",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7705323,\n   "lon": 37.6066304,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130648752",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7704612,\n   "lon": 37.6062741,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130648757",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7706345,\n   "lon": 37.6070371,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130650121",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7723294,\n   "lon": 37.6081595,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130651971",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.772965,\n   "lon": 37.6111439,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130651972",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7721483,\n   "lon": 37.612147,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130652494",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7710816,\n   "lon": 37.6140035,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130652497",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7710383,\n   "lon": 37.6126451,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130706112",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7787198,\n   "lon": 37.6390789,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130734300",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7142873,\n   "lon": 37.5691343,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130734342",\n   "kind": "building",\n   "name": "Управление социальной защиты населения района Ховрино",\n   "office_type": "government",\n   "lat": 55.8594704,\n   "lon": 37.5054195,\n   "opening_hours": "Mo 11:00-13:45, 14:30-20:00; Tu-Th 09:00-13:45, 14:30-18:00; Fr 09:00-13:45, 14:30-16:45"\n  },\n  {\n   "id": "way/130846562",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7910275,\n   "lon": 37.6614058,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130850432",\n   "kind": "building",\n   "name": "Внешэкономбанк",\n   "office_type": null,\n   "lat": 55.7721381,\n   "lon": 37.644924,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130851795",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7665921,\n   "lon": 37.6653053,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130854476",\n   "kind": "building",\n   "name": "Бизнес-центр «Токмаков, 5 с1»",\n   "office_type": null,\n   "lat": 55.7683404,\n   "lon": 37.6701987,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130855517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7706323,\n   "lon": 37.6730354,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130860128",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7807949,\n   "lon": 37.6901871,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130862427",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7743064,\n   "lon": 37.6804057,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130862595",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7746207,\n   "lon": 37.6785205,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130862651",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7746939,\n   "lon": 37.6777424,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130950851",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7846908,\n   "lon": 37.6921045,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130950853",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7847578,\n   "lon": 37.6937973,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130979746",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7777631,\n   "lon": 37.6921807,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130979747",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7774373,\n   "lon": 37.6919559,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130981800",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7790858,\n   "lon": 37.6929626,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130995831",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7584395,\n   "lon": 37.7366228,\n   "opening_hours": null\n  },\n  {\n   "id": "way/130996609",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6971806,\n   "lon": 37.622053,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131014254",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7788378,\n   "lon": 37.6941083,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131014258",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7790855,\n   "lon": 37.6937619,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131014259",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7803198,\n   "lon": 37.6963696,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131014260",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7789861,\n   "lon": 37.6939141,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131014266",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7786157,\n   "lon": 37.6927303,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131014275",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7789249,\n   "lon": 37.6919226,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131014278",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7788441,\n   "lon": 37.6936771,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131014281",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7791651,\n   "lon": 37.6936423,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131014285",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7787305,\n   "lon": 37.6946445,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131110261",\n   "kind": "building",\n   "name": "Институт Св. Фомы",\n   "office_type": null,\n   "lat": 55.7758163,\n   "lon": 37.6898125,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131173264",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7645477,\n   "lon": 37.678488,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131173268",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7644578,\n   "lon": 37.67939,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131173276",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7663297,\n   "lon": 37.6763406,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131175197",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7665793,\n   "lon": 37.6792761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131182227",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.763577,\n   "lon": 37.679659,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131190141",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7593661,\n   "lon": 37.6646642,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131190148",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7601229,\n   "lon": 37.6627517,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131190152",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7600177,\n   "lon": 37.6635735,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131190156",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7597867,\n   "lon": 37.6641533,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131193125",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7537236,\n   "lon": 37.6600581,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131193127",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7529461,\n   "lon": 37.6649443,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131200016",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7516556,\n   "lon": 37.6765563,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131200018",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7517646,\n   "lon": 37.6760723,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131379377",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7786787,\n   "lon": 37.6942784,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131383705",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7523166,\n   "lon": 37.6640072,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131386605",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7488488,\n   "lon": 37.6616506,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131386609",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.748598,\n   "lon": 37.6619084,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131474105",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7669731,\n   "lon": 37.7329173,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131491817",\n   "kind": "organisation",\n   "name": "Генеральное консульство Республики Таджикистан",\n   "office_type": "diplomatic",\n   "lat": 55.7561879,\n   "lon": 37.5932041,\n   "opening_hours": "Mo-Fr 10:00-16:30"\n  },\n  {\n   "id": "way/131491819",\n   "kind": "organisation",\n   "name": "Резиденция посла Чили в России",\n   "office_type": "diplomatic",\n   "lat": 55.7559402,\n   "lon": 37.592943,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131491823",\n   "kind": "building",\n   "name": "Государственный дом радиовещания и звукозаписи",\n   "office_type": null,\n   "lat": 55.7594328,\n   "lon": 37.5875187,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131491841",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.758792,\n   "lon": 37.5957439,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131499277",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7479136,\n   "lon": 37.6761713,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131499278",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7476798,\n   "lon": 37.6742114,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131499293",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7475194,\n   "lon": 37.6761238,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131499297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7480982,\n   "lon": 37.6744107,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131499300",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7478106,\n   "lon": 37.6736484,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131499304",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7475569,\n   "lon": 37.6735624,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131499307",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.747991,\n   "lon": 37.6753521,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131499308",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7482206,\n   "lon": 37.6737434,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131499309",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.747443,\n   "lon": 37.6733287,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131544978",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6904193,\n   "lon": 37.6636261,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131565961",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7740647,\n   "lon": 37.7397042,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131589346",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7780268,\n   "lon": 37.7250193,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131589347",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.777872,\n   "lon": 37.7248359,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131589415",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7782663,\n   "lon": 37.7242683,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131650679",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.776873,\n   "lon": 37.7380446,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131650806",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7763164,\n   "lon": 37.7382537,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131653876",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7757156,\n   "lon": 37.7342752,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131655167",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7870903,\n   "lon": 37.7183622,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131655168",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7841395,\n   "lon": 37.7229284,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131655172",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7865228,\n   "lon": 37.7188171,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131655174",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7837743,\n   "lon": 37.7233586,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131655261",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.787324,\n   "lon": 37.717849,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131655262",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7870313,\n   "lon": 37.7191925,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131655266",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7875793,\n   "lon": 37.717295,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131655566",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7863035,\n   "lon": 37.7200663,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131655567",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7863519,\n   "lon": 37.7206046,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131746762",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7523246,\n   "lon": 37.6491489,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131833399",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8097487,\n   "lon": 37.6913423,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131833401",\n   "kind": "building",\n   "name": "ОАО \\"Оборонэнерго\\", филиал \\"Центральный\\"",\n   "office_type": "company",\n   "lat": 55.8093753,\n   "lon": 37.690379,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131833406",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8064055,\n   "lon": 37.6933843,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131833408",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8063213,\n   "lon": 37.6927847,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131888473",\n   "kind": "building",\n   "name": "Кристалл Плаза",\n   "office_type": null,\n   "lat": 55.7468387,\n   "lon": 37.6290998,\n   "opening_hours": null\n  },\n  {\n   "id": "way/131888476",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7464587,\n   "lon": 37.6290738,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132062315",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7438102,\n   "lon": 37.6304313,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132066885",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7815112,\n   "lon": 37.6392949,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132092890",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7712855,\n   "lon": 37.5383012,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132193355",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7822421,\n   "lon": 37.5922434,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132193364",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7823206,\n   "lon": 37.5927994,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132295300",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6567539,\n   "lon": 37.5577596,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132365077",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835238,\n   "lon": 37.7309597,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132365078",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7837506,\n   "lon": 37.7300779,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132365079",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7841145,\n   "lon": 37.7295295,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132365084",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.783413,\n   "lon": 37.7316284,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132469317",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.729749,\n   "lon": 37.6435848,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132673759",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449225,\n   "lon": 37.5790243,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132708594",\n   "kind": "organisation",\n   "name": "Юго-Западная транспортная прокуратура",\n   "office_type": "government",\n   "lat": 55.7204241,\n   "lon": 37.5631722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132715740",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7406454,\n   "lon": 37.5940448,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132715742",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7421608,\n   "lon": 37.5983593,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132742756",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7939274,\n   "lon": 37.6202138,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132766060",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7507075,\n   "lon": 37.6062056,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132919159",\n   "kind": "organisation",\n   "name": "Орифлэйм косметикс",\n   "office_type": "company",\n   "lat": 55.7220293,\n   "lon": 37.5600814,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132928517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7968343,\n   "lon": 37.5612617,\n   "opening_hours": null\n  },\n  {\n   "id": "way/132998233",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7807538,\n   "lon": 37.7104256,\n   "opening_hours": null\n  },\n  {\n   "id": "way/133000155",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7795544,\n   "lon": 37.712013,\n   "opening_hours": null\n  },\n  {\n   "id": "way/133000156",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7793408,\n   "lon": 37.7128918,\n   "opening_hours": null\n  },\n  {\n   "id": "way/133000160",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.779592,\n   "lon": 37.7123619,\n   "opening_hours": null\n  },\n  {\n   "id": "way/133392897",\n   "kind": "building",\n   "name": "НПП САПФИР",\n   "office_type": null,\n   "lat": 55.7833238,\n   "lon": 37.7343255,\n   "opening_hours": null\n  },\n  {\n   "id": "way/133636968",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7296452,\n   "lon": 37.657525,\n   "opening_hours": null\n  },\n  {\n   "id": "way/133854710",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7079896,\n   "lon": 37.6246412,\n   "opening_hours": null\n  },\n  {\n   "id": "way/134046007",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.784371,\n   "lon": 37.7082765,\n   "opening_hours": null\n  },\n  {\n   "id": "way/134113177",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7229511,\n   "lon": 37.7032083,\n   "opening_hours": null\n  },\n  {\n   "id": "way/134113178",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7231121,\n   "lon": 37.7041496,\n   "opening_hours": null\n  },\n  {\n   "id": "way/134113181",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7230644,\n   "lon": 37.7027669,\n   "opening_hours": null\n  },\n  {\n   "id": "way/134633730",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7213668,\n   "lon": 37.5904257,\n   "opening_hours": null\n  },\n  {\n   "id": "way/134823877",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7348058,\n   "lon": 37.6239602,\n   "opening_hours": null\n  },\n  {\n   "id": "way/134823878",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7345213,\n   "lon": 37.6240736,\n   "opening_hours": null\n  },\n  {\n   "id": "way/134874637",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7347771,\n   "lon": 37.6228193,\n   "opening_hours": null\n  },\n  {\n   "id": "way/135402017",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8182938,\n   "lon": 37.5646462,\n   "opening_hours": null\n  },\n  {\n   "id": "way/135406007",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8225503,\n   "lon": 37.5683822,\n   "opening_hours": null\n  },\n  {\n   "id": "way/135406008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8227646,\n   "lon": 37.5694107,\n   "opening_hours": null\n  },\n  {\n   "id": "way/135581382",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7708241,\n   "lon": 37.6290252,\n   "opening_hours": null\n  },\n  {\n   "id": "way/135640396",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7471609,\n   "lon": 37.5605369,\n   "opening_hours": null\n  },\n  {\n   "id": "way/135642145",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7494762,\n   "lon": 37.5364821,\n   "opening_hours": null\n  },\n  {\n   "id": "way/136514707",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7479771,\n   "lon": 37.6478046,\n   "opening_hours": null\n  },\n  {\n   "id": "way/136577528",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7477666,\n   "lon": 37.6539199,\n   "opening_hours": null\n  },\n  {\n   "id": "way/136577541",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7482591,\n   "lon": 37.651286,\n   "opening_hours": null\n  },\n  {\n   "id": "way/136714481",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.825144,\n   "lon": 37.5035396,\n   "opening_hours": null\n  },\n  {\n   "id": "way/136796949",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7451253,\n   "lon": 37.6628618,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137289399",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7408585,\n   "lon": 37.5934297,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137289414",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7378159,\n   "lon": 37.5945107,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137289419",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7386035,\n   "lon": 37.5990842,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137289437",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7384907,\n   "lon": 37.6012181,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137289438",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390159,\n   "lon": 37.5982316,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137289443",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7369315,\n   "lon": 37.5939034,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137372691",\n   "kind": "organisation",\n   "name": "Федеральный институт промышленной собственности",\n   "office_type": "government",\n   "lat": 55.7302136,\n   "lon": 37.548038,\n   "opening_hours": "Mo-Th 09:30-18:15; Fr 09:30-17:00; Sa,Su,PH off"\n  },\n  {\n   "id": "way/137667439",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8040181,\n   "lon": 37.5632162,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137928481",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7003874,\n   "lon": 37.6138293,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137928484",\n   "kind": "building",\n   "name": "КПП",\n   "office_type": null,\n   "lat": 55.6979901,\n   "lon": 37.6103234,\n   "opening_hours": null\n  },\n  {\n   "id": "way/137928490",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6991324,\n   "lon": 37.6145182,\n   "opening_hours": null\n  },\n  {\n   "id": "way/139055063",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7744348,\n   "lon": 37.6164318,\n   "opening_hours": null\n  },\n  {\n   "id": "way/139062980",\n   "kind": "organisation",\n   "name": "Институт биохимической физики им. Н. М. Эмануэля РАН",\n   "office_type": "research",\n   "lat": 55.7081685,\n   "lon": 37.5707314,\n   "opening_hours": "Mo-Tu 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "way/139063159",\n   "kind": "organisation",\n   "name": "Лаборатория взрывов и горения",\n   "office_type": "research",\n   "lat": 55.7090465,\n   "lon": 37.5714124,\n   "opening_hours": null\n  },\n  {\n   "id": "way/139069935",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Волконский\\"",\n   "office_type": null,\n   "lat": 55.7753735,\n   "lon": 37.6168486,\n   "opening_hours": null\n  },\n  {\n   "id": "way/139450248",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7711147,\n   "lon": 37.6709824,\n   "opening_hours": null\n  },\n  {\n   "id": "way/139452110",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7714944,\n   "lon": 37.6664132,\n   "opening_hours": null\n  },\n  {\n   "id": "way/139880865",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7788026,\n   "lon": 37.6127291,\n   "opening_hours": null\n  },\n  {\n   "id": "way/140050057",\n   "kind": "organisation",\n   "name": "ГКУ \\"УДМС\\"",\n   "office_type": "research",\n   "lat": 55.6637138,\n   "lon": 37.5490505,\n   "opening_hours": null\n  },\n  {\n   "id": "way/140326033",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7325348,\n   "lon": 37.533659,\n   "opening_hours": null\n  },\n  {\n   "id": "way/140326039",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7325258,\n   "lon": 37.5343647,\n   "opening_hours": null\n  },\n  {\n   "id": "way/141068416",\n   "kind": "organisation",\n   "name": "Курчатовский комплекс генетических исследований",\n   "office_type": "research",\n   "lat": 55.6167887,\n   "lon": 37.6147501,\n   "opening_hours": null\n  },\n  {\n   "id": "way/141693754",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7709121,\n   "lon": 37.6495238,\n   "opening_hours": null\n  },\n  {\n   "id": "way/141765723",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7679112,\n   "lon": 37.6521137,\n   "opening_hours": null\n  },\n  {\n   "id": "way/141765724",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7680025,\n   "lon": 37.6525002,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142235990",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8184467,\n   "lon": 37.6324081,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142387375",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7444734,\n   "lon": 37.5901979,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142387376",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.744883,\n   "lon": 37.5898002,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142394549",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.733744,\n   "lon": 37.6372478,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142467921",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8193433,\n   "lon": 37.6044812,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142471571",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8174301,\n   "lon": 37.6130174,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142585587",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7413798,\n   "lon": 37.5246231,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142585588",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.741044,\n   "lon": 37.5254561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142585593",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7418548,\n   "lon": 37.525329,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142585595",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7402658,\n   "lon": 37.5283718,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142585601",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7416706,\n   "lon": 37.5243629,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142585602",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7412891,\n   "lon": 37.52499,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142585604",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7413518,\n   "lon": 37.5240187,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142585606",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7418095,\n   "lon": 37.5247523,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142638120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6936619,\n   "lon": 37.6008888,\n   "opening_hours": null\n  },\n  {\n   "id": "way/142949876",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7511105,\n   "lon": 37.7544697,\n   "opening_hours": null\n  },\n  {\n   "id": "way/143189426",\n   "kind": "organisation",\n   "name": "ГУП \\"Московский метрополитен\\"",\n   "office_type": "company",\n   "lat": 55.7831077,\n   "lon": 37.6288387,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:30"\n  },\n  {\n   "id": "way/143189428",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7839367,\n   "lon": 37.6281282,\n   "opening_hours": null\n  },\n  {\n   "id": "way/143604911",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "educational_institution",\n   "lat": 55.7012778,\n   "lon": 37.5565375,\n   "opening_hours": null\n  },\n  {\n   "id": "way/143750481",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7499314,\n   "lon": 37.5260103,\n   "opening_hours": null\n  },\n  {\n   "id": "way/144236570",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Буракова, 29\\"",\n   "office_type": null,\n   "lat": 55.7570276,\n   "lon": 37.7354471,\n   "opening_hours": null\n  },\n  {\n   "id": "way/145181698",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6997375,\n   "lon": 37.6265349,\n   "opening_hours": null\n  },\n  {\n   "id": "way/146515662",\n   "kind": "building",\n   "name": "Бизнес-центр \\"На Звенигородке\\"",\n   "office_type": null,\n   "lat": 55.7613472,\n   "lon": 37.5516415,\n   "opening_hours": null\n  },\n  {\n   "id": "way/146841919",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Диагональ Хаус\\"",\n   "office_type": null,\n   "lat": 55.8039194,\n   "lon": 37.5821838,\n   "opening_hours": null\n  },\n  {\n   "id": "way/146926420",\n   "kind": "building",\n   "name": "Управление ГИБДД Московской области",\n   "office_type": "government",\n   "lat": 55.7839213,\n   "lon": 37.6364465,\n   "opening_hours": null\n  },\n  {\n   "id": "way/146992707",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.6443409,\n   "lon": 37.6164227,\n   "opening_hours": null\n  },\n  {\n   "id": "way/147587062",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7575036,\n   "lon": 37.6023021,\n   "opening_hours": null\n  },\n  {\n   "id": "way/148189793",\n   "kind": "organisation",\n   "name": "КПП-1",\n   "office_type": "security",\n   "lat": 55.6760924,\n   "lon": 37.6716476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/148189795",\n   "kind": "organisation",\n   "name": "КПП-3",\n   "office_type": "security",\n   "lat": 55.6681988,\n   "lon": 37.6622158,\n   "opening_hours": null\n  },\n  {\n   "id": "way/148470686",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7740277,\n   "lon": 37.6092603,\n   "opening_hours": null\n  },\n  {\n   "id": "way/148470689",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7737829,\n   "lon": 37.6097613,\n   "opening_hours": null\n  },\n  {\n   "id": "way/148707917",\n   "kind": "organisation",\n   "name": "Транссервис-95",\n   "office_type": "company",\n   "lat": 55.6493757,\n   "lon": 37.6251803,\n   "opening_hours": null\n  },\n  {\n   "id": "way/148748426",\n   "kind": "organisation",\n   "name": "Центральный научно-исследовательский геологоразведочный институт цветных и благородных металлов",\n   "office_type": "research",\n   "lat": 55.6134464,\n   "lon": 37.6129158,\n   "opening_hours": null\n  },\n  {\n   "id": "way/148787765",\n   "kind": "organisation",\n   "name": "3-й Центральный научно-исследовательский институт Министерства Обороны Российской Федерации",\n   "office_type": "research",\n   "lat": 55.8184676,\n   "lon": 37.7075022,\n   "opening_hours": null\n  },\n  {\n   "id": "way/149757436",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.7197804,\n   "lon": 37.6700405,\n   "opening_hours": null\n  },\n  {\n   "id": "way/149864235",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7804374,\n   "lon": 37.615143,\n   "opening_hours": null\n  },\n  {\n   "id": "way/149864236",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7805288,\n   "lon": 37.6146787,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150134797",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8078735,\n   "lon": 37.7032617,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150186773",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6981294,\n   "lon": 37.6249944,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150403447",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8269843,\n   "lon": 37.7177863,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150591339",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7829192,\n   "lon": 37.7046957,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150824915",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8153533,\n   "lon": 37.6878186,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150824920",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8140911,\n   "lon": 37.6897151,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150824925",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8164549,\n   "lon": 37.689595,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150824928",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8136683,\n   "lon": 37.6920925,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150824929",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8137552,\n   "lon": 37.6932278,\n   "opening_hours": null\n  },\n  {\n   "id": "way/150964762",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7446269,\n   "lon": 37.6712494,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151040660",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7138393,\n   "lon": 37.6635687,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151045120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7206,\n   "lon": 37.6470527,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151045124",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7210163,\n   "lon": 37.64693,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151045132",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7208157,\n   "lon": 37.6467593,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151060784",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7224543,\n   "lon": 37.6362503,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151060787",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7218108,\n   "lon": 37.6386706,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151060804",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7169908,\n   "lon": 37.633939,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151060822",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7211789,\n   "lon": 37.6360095,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151060833",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.721929,\n   "lon": 37.6377901,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151060838",\n   "kind": "organisation",\n   "name": "ГУП Моспродконтракт",\n   "office_type": "yes",\n   "lat": 55.7233344,\n   "lon": 37.6360452,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077466",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7469409,\n   "lon": 37.6445216,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077472",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7452117,\n   "lon": 37.6469594,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077473",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7453853,\n   "lon": 37.6469596,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077484",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7454631,\n   "lon": 37.6478501,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077486",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7457035,\n   "lon": 37.6473921,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077495",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7452687,\n   "lon": 37.6483091,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7450415,\n   "lon": 37.6484578,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077521",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7446746,\n   "lon": 37.6485793,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077531",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7455061,\n   "lon": 37.6465968,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077533",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.745082,\n   "lon": 37.6476493,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151077540",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447744,\n   "lon": 37.6477606,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151133363",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7997554,\n   "lon": 37.6127754,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151133364",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7996288,\n   "lon": 37.6132094,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151231755",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7464056,\n   "lon": 37.6443835,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151403546",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390332,\n   "lon": 37.6623697,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151403555",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7389816,\n   "lon": 37.6649697,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151403557",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390813,\n   "lon": 37.6657317,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151574816",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7773938,\n   "lon": 37.602727,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151574820",\n   "kind": "building",\n   "name": "РНКБ Банк",\n   "office_type": null,\n   "lat": 55.775564,\n   "lon": 37.605136,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa,Su off"\n  },\n  {\n   "id": "way/151574821",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7751217,\n   "lon": 37.6051025,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151574822",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7750221,\n   "lon": 37.6088812,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151574826",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7741232,\n   "lon": 37.610889,\n   "opening_hours": null\n  },\n  {\n   "id": "way/151815136",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7663419,\n   "lon": 37.5179096,\n   "opening_hours": null\n  },\n  {\n   "id": "way/152006794",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7074717,\n   "lon": 37.6512963,\n   "opening_hours": null\n  },\n  {\n   "id": "way/152270032",\n   "kind": "building",\n   "name": "Представительство республики Башкоторстан",\n   "office_type": "government",\n   "lat": 55.7682305,\n   "lon": 37.6720754,\n   "opening_hours": null\n  },\n  {\n   "id": "way/152715262",\n   "kind": "building",\n   "name": "Абсолют",\n   "office_type": null,\n   "lat": 55.7700766,\n   "lon": 37.6227684,\n   "opening_hours": "Sa 10:00-16:00; Mo-Fr 09:00-20:00; Su off"\n  },\n  {\n   "id": "way/152715265",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7700696,\n   "lon": 37.6248231,\n   "opening_hours": null\n  },\n  {\n   "id": "way/153312567",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6457448,\n   "lon": 37.5433658,\n   "opening_hours": null\n  },\n  {\n   "id": "way/153312579",\n   "kind": "building",\n   "name": "Издательство Наука",\n   "office_type": "publisher",\n   "lat": 55.649369,\n   "lon": 37.5303867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/153394143",\n   "kind": "organisation",\n   "name": "Посольство Швеции",\n   "office_type": "diplomatic",\n   "lat": 55.7166397,\n   "lon": 37.5159108,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154005966",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8048737,\n   "lon": 37.5800914,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154025374",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7639995,\n   "lon": 37.4952129,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154143441",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7995982,\n   "lon": 37.5656437,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154143444",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7996247,\n   "lon": 37.5660545,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154326122",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6553361,\n   "lon": 37.5779943,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154326131",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6532149,\n   "lon": 37.5646796,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154326144",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6570227,\n   "lon": 37.5681932,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154326163",\n   "kind": "organisation",\n   "name": "Центр молодёжного парламентаризма",\n   "office_type": "government",\n   "lat": 55.6545361,\n   "lon": 37.5755492,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154331579",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6578696,\n   "lon": 37.6147146,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154331582",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6584465,\n   "lon": 37.6155943,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154331583",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6580388,\n   "lon": 37.6154669,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154342426",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6738053,\n   "lon": 37.5981157,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154363966",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8149467,\n   "lon": 37.6962312,\n   "opening_hours": null\n  },\n  {\n   "id": "way/154491648",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6807373,\n   "lon": 37.4807679,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155053513",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6870116,\n   "lon": 37.6821806,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155217658",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Clever\\"",\n   "office_type": null,\n   "lat": 55.7047923,\n   "lon": 37.5978193,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155217689",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7042389,\n   "lon": 37.5676059,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155217723",\n   "kind": "organisation",\n   "name": "Институт металлургии и материаловедения им. А. А. Байкова РАН",\n   "office_type": "research",\n   "lat": 55.7001596,\n   "lon": 37.569452,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155217730",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6932041,\n   "lon": 37.550454,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155217743",\n   "kind": "organisation",\n   "name": "Вычислительный центр им. А. А. Дородницына РАН",\n   "office_type": "research",\n   "lat": 55.6944531,\n   "lon": 37.5650365,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713640",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7404716,\n   "lon": 37.5823476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713661",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7483677,\n   "lon": 37.5800263,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713665",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7482383,\n   "lon": 37.5800428,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713685",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7466265,\n   "lon": 37.5797206,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713722",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7392782,\n   "lon": 37.5832361,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713728",\n   "kind": "building",\n   "name": "Посольство Объединённых Арабских Эмиратов, Военный атташе",\n   "office_type": "diplomatic",\n   "lat": 55.7468936,\n   "lon": 37.5789551,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713765",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7396895,\n   "lon": 37.5831734,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713778",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.740555,\n   "lon": 37.5825122,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713808",\n   "kind": "organisation",\n   "name": "Посольство Турции",\n   "office_type": "diplomatic",\n   "lat": 55.7409792,\n   "lon": 37.5741261,\n   "opening_hours": "Mo-Fr 09:00-12:30"\n  },\n  {\n   "id": "way/155713983",\n   "kind": "organisation",\n   "name": "Посольство Бангладеш",\n   "office_type": "diplomatic",\n   "lat": 55.740098,\n   "lon": 37.5785233,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155713987",\n   "kind": "organisation",\n   "name": "Посольство Великобритании",\n   "office_type": "diplomatic",\n   "lat": 55.7505762,\n   "lon": 37.577,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-17:00;PH off"\n  },\n  {\n   "id": "way/155713994",\n   "kind": "organisation",\n   "name": "Посольство Таиланда",\n   "office_type": "diplomatic",\n   "lat": 55.7394398,\n   "lon": 37.5794815,\n   "opening_hours": "Mo-Fr 09:00-13:00, 14:00-17:00"\n  },\n  {\n   "id": "way/155714005",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7402592,\n   "lon": 37.5823306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155794664",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7767386,\n   "lon": 37.6688134,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155998606",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7393235,\n   "lon": 37.6402792,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155998613",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7392827,\n   "lon": 37.6408312,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155998617",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390616,\n   "lon": 37.6409501,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155998621",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7394539,\n   "lon": 37.6407598,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155998642",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7386226,\n   "lon": 37.6411328,\n   "opening_hours": null\n  },\n  {\n   "id": "way/155998649",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7365585,\n   "lon": 37.6427298,\n   "opening_hours": null\n  },\n  {\n   "id": "way/156300327",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7430456,\n   "lon": 37.6265317,\n   "opening_hours": null\n  },\n  {\n   "id": "way/156493104",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7359996,\n   "lon": 37.6239812,\n   "opening_hours": null\n  },\n  {\n   "id": "way/156786270",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7714979,\n   "lon": 37.6508193,\n   "opening_hours": null\n  },\n  {\n   "id": "way/156918821",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6891514,\n   "lon": 37.6160804,\n   "opening_hours": null\n  },\n  {\n   "id": "way/156918830",\n   "kind": "organisation",\n   "name": "НИИ вакуумной техники им. С.А. Векшинского",\n   "office_type": "research",\n   "lat": 55.6864977,\n   "lon": 37.6152678,\n   "opening_hours": null\n  },\n  {\n   "id": "way/156918838",\n   "kind": "building",\n   "name": "РЕСО Гарантия Центр выплат",\n   "office_type": "insurance",\n   "lat": 55.6881293,\n   "lon": 37.6156939,\n   "opening_hours": "Mo-Fr 08:30-20:30; Sa 09:00-18:00; Su 10:00-17:00"\n  },\n  {\n   "id": "way/157086671",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Западное Дегунино",\n   "office_type": "government",\n   "lat": 55.8724792,\n   "lon": 37.5093543,\n   "opening_hours": "08:00-20:00"\n  },\n  {\n   "id": "way/157165371",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6318889,\n   "lon": 37.7621477,\n   "opening_hours": null\n  },\n  {\n   "id": "way/157165397",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6363044,\n   "lon": 37.7558783,\n   "opening_hours": null\n  },\n  {\n   "id": "way/157210017",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6434354,\n   "lon": 37.6036237,\n   "opening_hours": null\n  },\n  {\n   "id": "way/157316172",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6854406,\n   "lon": 37.6216902,\n   "opening_hours": null\n  },\n  {\n   "id": "way/157316174",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6885482,\n   "lon": 37.6145281,\n   "opening_hours": null\n  },\n  {\n   "id": "way/157316175",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6879487,\n   "lon": 37.615027,\n   "opening_hours": null\n  },\n  {\n   "id": "way/157320097",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7225507,\n   "lon": 37.6357546,\n   "opening_hours": null\n  },\n  {\n   "id": "way/157320169",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7281729,\n   "lon": 37.6467672,\n   "opening_hours": null\n  },\n  {\n   "id": "way/157320183",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Оазис\\"",\n   "office_type": null,\n   "lat": 55.7284323,\n   "lon": 37.6202342,\n   "opening_hours": null\n  },\n  {\n   "id": "way/158190562",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8153218,\n   "lon": 37.6358802,\n   "opening_hours": null\n  },\n  {\n   "id": "way/158190563",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8156195,\n   "lon": 37.6368063,\n   "opening_hours": null\n  },\n  {\n   "id": "way/158266531",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7984666,\n   "lon": 37.5817049,\n   "opening_hours": null\n  },\n  {\n   "id": "way/158853572",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.5970992,\n   "lon": 37.6446424,\n   "opening_hours": null\n  },\n  {\n   "id": "way/159532065",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7332095,\n   "lon": 37.6177828,\n   "opening_hours": null\n  },\n  {\n   "id": "way/159539362",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7340055,\n   "lon": 37.6167777,\n   "opening_hours": null\n  },\n  {\n   "id": "way/159629159",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7296498,\n   "lon": 37.6304801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/159706338",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7906684,\n   "lon": 37.7040189,\n   "opening_hours": null\n  },\n  {\n   "id": "way/159959317",\n   "kind": "organisation",\n   "name": "Посольство Колумбии",\n   "office_type": "diplomatic",\n   "lat": 55.7389852,\n   "lon": 37.5798553,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160166330",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7350965,\n   "lon": 37.6145637,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160191345",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7373046,\n   "lon": 37.6021803,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160195717",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.6383579,\n   "lon": 37.7667373,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160232261",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7524322,\n   "lon": 37.5986038,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160232329",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7501903,\n   "lon": 37.6011814,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160232359",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7507377,\n   "lon": 37.601605,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160232360",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7523943,\n   "lon": 37.5981792,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160232368",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7476472,\n   "lon": 37.601052,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160247752",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7977923,\n   "lon": 37.5793172,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160272396",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7233655,\n   "lon": 37.6187121,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160272397",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7233562,\n   "lon": 37.6178801,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160272398",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7232773,\n   "lon": 37.6170676,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160277907",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7289746,\n   "lon": 37.6186381,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160282253",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6688937,\n   "lon": 37.7907038,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160317024",\n   "kind": "organisation",\n   "name": "посольство Исландии",\n   "office_type": "diplomatic",\n   "lat": 55.7565169,\n   "lon": 37.5911497,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160317025",\n   "kind": "organisation",\n   "name": "посольство Литвы",\n   "office_type": "diplomatic",\n   "lat": 55.7546387,\n   "lon": 37.5905894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160318329",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7468505,\n   "lon": 37.5815066,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160318339",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7469034,\n   "lon": 37.5806219,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160318340",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7490722,\n   "lon": 37.5816201,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160318341",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7468173,\n   "lon": 37.5799516,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160318342",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7486851,\n   "lon": 37.5810227,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160343044",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8363479,\n   "lon": 37.5113622,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160379357",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8127707,\n   "lon": 37.6329004,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160409250",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7264151,\n   "lon": 37.6256762,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160412228",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7275947,\n   "lon": 37.6460501,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160412239",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7265515,\n   "lon": 37.6486311,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160412242",\n   "kind": "building",\n   "name": "RRG",\n   "office_type": null,\n   "lat": 55.7288034,\n   "lon": 37.6592633,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160418444",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7861159,\n   "lon": 37.5615616,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160600506",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7258438,\n   "lon": 37.6242651,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160671131",\n   "kind": "organisation",\n   "name": "посольство ФРГ",\n   "office_type": "diplomatic",\n   "lat": 55.7185066,\n   "lon": 37.5168464,\n   "opening_hours": null\n  },\n  {\n   "id": "way/160673821",\n   "kind": "organisation",\n   "name": "Посольство ФРГ",\n   "office_type": "diplomatic",\n   "lat": 55.6714129,\n   "lon": 37.5274077,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161149000",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449879,\n   "lon": 37.4952965,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161196245",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432716,\n   "lon": 37.6527139,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161232038",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7438119,\n   "lon": 37.6484382,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161232039",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7444725,\n   "lon": 37.6485253,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161412432",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7550222,\n   "lon": 37.5567061,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161450858",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8186793,\n   "lon": 37.4503792,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161574652",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7226103,\n   "lon": 37.6466202,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161767029",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8028569,\n   "lon": 37.5654909,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161871024",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7050014,\n   "lon": 37.6228115,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161871073",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7067501,\n   "lon": 37.6249815,\n   "opening_hours": null\n  },\n  {\n   "id": "way/161938734",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8207022,\n   "lon": 37.5018093,\n   "opening_hours": null\n  },\n  {\n   "id": "way/162058122",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7539486,\n   "lon": 37.603667,\n   "opening_hours": null\n  },\n  {\n   "id": "way/162139505",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7475194,\n   "lon": 37.6712908,\n   "opening_hours": null\n  },\n  {\n   "id": "way/162139507",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7476054,\n   "lon": 37.6714002,\n   "opening_hours": null\n  },\n  {\n   "id": "way/162139510",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.747467,\n   "lon": 37.6726522,\n   "opening_hours": null\n  },\n  {\n   "id": "way/162139511",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7473524,\n   "lon": 37.6709666,\n   "opening_hours": null\n  },\n  {\n   "id": "way/162691913",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7695429,\n   "lon": 37.6344446,\n   "opening_hours": null\n  },\n  {\n   "id": "way/162691914",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7697721,\n   "lon": 37.6335298,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163008908",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6807995,\n   "lon": 37.5691143,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163057935",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6808503,\n   "lon": 37.5867152,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163553887",\n   "kind": "building",\n   "name": "корпус 5",\n   "office_type": null,\n   "lat": 55.8099913,\n   "lon": 37.5062219,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163638567",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8083194,\n   "lon": 37.5160699,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163638649",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8092405,\n   "lon": 37.5121489,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163738638",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7319715,\n   "lon": 37.619069,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163742855",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7339178,\n   "lon": 37.6198663,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163742858",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7328806,\n   "lon": 37.6200811,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163742859",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7326914,\n   "lon": 37.6201259,\n   "opening_hours": null\n  },\n  {\n   "id": "way/163742861",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7333969,\n   "lon": 37.6196873,\n   "opening_hours": null\n  },\n  {\n   "id": "way/164210798",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7343954,\n   "lon": 37.6217334,\n   "opening_hours": null\n  },\n  {\n   "id": "way/164210805",\n   "kind": "building",\n   "name": "Дом немецкой экономики",\n   "office_type": null,\n   "lat": 55.734814,\n   "lon": 37.6223837,\n   "opening_hours": null\n  },\n  {\n   "id": "way/164210807",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.734715,\n   "lon": 37.6217753,\n   "opening_hours": null\n  },\n  {\n   "id": "way/165212829",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7973853,\n   "lon": 37.5784952,\n   "opening_hours": null\n  },\n  {\n   "id": "way/165736621",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7448702,\n   "lon": 37.6300009,\n   "opening_hours": null\n  },\n  {\n   "id": "way/165814762",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7975415,\n   "lon": 37.5793419,\n   "opening_hours": null\n  },\n  {\n   "id": "way/166019621",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7860341,\n   "lon": 37.635824,\n   "opening_hours": null\n  },\n  {\n   "id": "way/166409735",\n   "kind": "organisation",\n   "name": "Эскиз",\n   "office_type": "company",\n   "lat": 55.7396578,\n   "lon": 37.7382586,\n   "opening_hours": null\n  },\n  {\n   "id": "way/166416428",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.736753,\n   "lon": 37.7297895,\n   "opening_hours": null\n  },\n  {\n   "id": "way/166416490",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.735064,\n   "lon": 37.7393336,\n   "opening_hours": null\n  },\n  {\n   "id": "way/166492540",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7408788,\n   "lon": 37.5638098,\n   "opening_hours": null\n  },\n  {\n   "id": "way/166492544",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7385803,\n   "lon": 37.5633877,\n   "opening_hours": null\n  },\n  {\n   "id": "way/166534132",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7377244,\n   "lon": 37.730213,\n   "opening_hours": null\n  },\n  {\n   "id": "way/166534156",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7384402,\n   "lon": 37.7304018,\n   "opening_hours": null\n  },\n  {\n   "id": "way/167291720",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8024364,\n   "lon": 37.6411406,\n   "opening_hours": null\n  },\n  {\n   "id": "way/167375732",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7422457,\n   "lon": 37.55903,\n   "opening_hours": null\n  },\n  {\n   "id": "way/167425386",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7077829,\n   "lon": 37.650474,\n   "opening_hours": null\n  },\n  {\n   "id": "way/167442902",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8679395,\n   "lon": 37.515188,\n   "opening_hours": null\n  },\n  {\n   "id": "way/167870725",\n   "kind": "building",\n   "name": "Ремонтно-строительное управление МВД РФ",\n   "office_type": null,\n   "lat": 55.8132493,\n   "lon": 37.8102352,\n   "opening_hours": null\n  },\n  {\n   "id": "way/168022311",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7282072,\n   "lon": 37.7737258,\n   "opening_hours": null\n  },\n  {\n   "id": "way/168357975",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7511816,\n   "lon": 37.7230256,\n   "opening_hours": null\n  },\n  {\n   "id": "way/168656742",\n   "kind": "building",\n   "name": "Краснодарский домик",\n   "office_type": null,\n   "lat": 55.7967775,\n   "lon": 37.6759875,\n   "opening_hours": null\n  },\n  {\n   "id": "way/168771619",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7080469,\n   "lon": 37.8088318,\n   "opening_hours": null\n  },\n  {\n   "id": "way/168840465",\n   "kind": "organisation",\n   "name": "Управа района Измайлово",\n   "office_type": "government",\n   "lat": 55.7913107,\n   "lon": 37.7886568,\n   "opening_hours": "Mo-Th 08:00-12:30,13:15-17:00; Fr 08:00-12:30,13:15-15:45"\n  },\n  {\n   "id": "way/168840481",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7943348,\n   "lon": 37.7844635,\n   "opening_hours": null\n  },\n  {\n   "id": "way/169270985",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7422071,\n   "lon": 37.6620002,\n   "opening_hours": null\n  },\n  {\n   "id": "way/169270989",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7423229,\n   "lon": 37.6623321,\n   "opening_hours": null\n  },\n  {\n   "id": "way/169272777",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7420421,\n   "lon": 37.66239,\n   "opening_hours": null\n  },\n  {\n   "id": "way/169347495",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Таволга\\"",\n   "office_type": null,\n   "lat": 55.8303827,\n   "lon": 37.5178405,\n   "opening_hours": null\n  },\n  {\n   "id": "way/169350450",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6331449,\n   "lon": 37.4637017,\n   "opening_hours": null\n  },\n  {\n   "id": "way/169458802",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8045364,\n   "lon": 37.5840508,\n   "opening_hours": null\n  },\n  {\n   "id": "way/170081763",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Каланчёвская плаза\\"",\n   "office_type": null,\n   "lat": 55.7775929,\n   "lon": 37.6494667,\n   "opening_hours": null\n  },\n  {\n   "id": "way/170183557",\n   "kind": "organisation",\n   "name": "НИИ Парашютостроения",\n   "office_type": "research",\n   "lat": 55.8154724,\n   "lon": 37.7871081,\n   "opening_hours": null\n  },\n  {\n   "id": "way/170679000",\n   "kind": "organisation",\n   "name": "Институт физики Земли им. О.Ю. Шмидта Российской академии наук (ИФЗ РАН)",\n   "office_type": "research",\n   "lat": 55.7636641,\n   "lon": 37.5804899,\n   "opening_hours": null\n  },\n  {\n   "id": "way/170957726",\n   "kind": "organisation",\n   "name": "Торговый дом Разгуляй",\n   "office_type": "company",\n   "lat": 55.7212561,\n   "lon": 37.7766957,\n   "opening_hours": null\n  },\n  {\n   "id": "way/171431730",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.739087,\n   "lon": 37.6574767,\n   "opening_hours": null\n  },\n  {\n   "id": "way/171610193",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7467993,\n   "lon": 37.7742475,\n   "opening_hours": null\n  },\n  {\n   "id": "way/171683457",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7241434,\n   "lon": 37.6106803,\n   "opening_hours": null\n  },\n  {\n   "id": "way/171878914",\n   "kind": "organisation",\n   "name": "Государственный институт лекарственных средств и надлежащих практик",\n   "office_type": "government",\n   "lat": 55.7336357,\n   "lon": 37.6613388,\n   "opening_hours": null\n  },\n  {\n   "id": "way/171901455",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7695586,\n   "lon": 37.6290812,\n   "opening_hours": null\n  },\n  {\n   "id": "way/171901462",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7693185,\n   "lon": 37.629784,\n   "opening_hours": null\n  },\n  {\n   "id": "way/171901466",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7680429,\n   "lon": 37.6267472,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172010806",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8008753,\n   "lon": 37.5872276,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172010811",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7999105,\n   "lon": 37.587373,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172010823",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8002221,\n   "lon": 37.5865151,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172073196",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7671088,\n   "lon": 37.6266063,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172073201",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7675238,\n   "lon": 37.6255426,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172549518",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7746909,\n   "lon": 37.6271414,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172700196",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8151984,\n   "lon": 37.5747942,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172700200",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8155261,\n   "lon": 37.5731466,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172839669",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7816926,\n   "lon": 37.6375788,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172839675",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7830175,\n   "lon": 37.6438977,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172839706",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.781079,\n   "lon": 37.6457981,\n   "opening_hours": null\n  },\n  {\n   "id": "way/172839709",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7845411,\n   "lon": 37.6368557,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173097294",\n   "kind": "building",\n   "name": "Яхтенный бизнес-центр",\n   "office_type": null,\n   "lat": 55.8329611,\n   "lon": 37.4822969,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173166365",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7462986,\n   "lon": 37.5984547,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173166370",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7467841,\n   "lon": 37.5990206,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173166376",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7455037,\n   "lon": 37.5995992,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173166386",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7465475,\n   "lon": 37.5985388,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173167622",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7465714,\n   "lon": 37.5976554,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173167625",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7470665,\n   "lon": 37.5975861,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173190494",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7889719,\n   "lon": 37.63845,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173497504",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7656619,\n   "lon": 37.6405443,\n   "opening_hours": null\n  },\n  {\n   "id": "way/173524268",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8897024,\n   "lon": 37.5810987,\n   "opening_hours": null\n  },\n  {\n   "id": "way/175033106",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6189719,\n   "lon": 37.5109604,\n   "opening_hours": null\n  },\n  {\n   "id": "way/175187062",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7742472,\n   "lon": 37.6625669,\n   "opening_hours": null\n  },\n  {\n   "id": "way/175187063",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7707475,\n   "lon": 37.6644016,\n   "opening_hours": null\n  },\n  {\n   "id": "way/175187064",\n   "kind": "building",\n   "name": "Пожарная часть № 14",\n   "office_type": null,\n   "lat": 55.7704633,\n   "lon": 37.6646834,\n   "opening_hours": null\n  },\n  {\n   "id": "way/175229976",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7243102,\n   "lon": 37.7915309,\n   "opening_hours": null\n  },\n  {\n   "id": "way/175332941",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7558601,\n   "lon": 37.5971935,\n   "opening_hours": null\n  },\n  {\n   "id": "way/175776140",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7673399,\n   "lon": 37.6623563,\n   "opening_hours": null\n  },\n  {\n   "id": "way/175776148",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7671324,\n   "lon": 37.6623562,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176025431",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7770989,\n   "lon": 37.6491149,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176256112",\n   "kind": "organisation",\n   "name": "Музейно-выставочный центр \\"РОСИЗО\\"",\n   "office_type": "government",\n   "lat": 55.7454322,\n   "lon": 37.5933155,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176299926",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7460931,\n   "lon": 37.5866584,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176299930",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461277,\n   "lon": 37.5864837,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176321351",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.742182,\n   "lon": 37.5867019,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176321356",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7428326,\n   "lon": 37.5857145,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176321357",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7427282,\n   "lon": 37.5879692,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176321358",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.742649,\n   "lon": 37.587328,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176321362",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7433364,\n   "lon": 37.5869692,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176371327",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7732235,\n   "lon": 37.6194429,\n   "opening_hours": null\n  },\n  {\n   "id": "way/176371332",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7722514,\n   "lon": 37.6199685,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177325783",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7074319,\n   "lon": 37.6047306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177526471",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7075827,\n   "lon": 37.6198584,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177575657",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8044642,\n   "lon": 37.5803858,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177657595",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7947078,\n   "lon": 37.7083016,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177657627",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7938433,\n   "lon": 37.7072783,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177682896",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7641957,\n   "lon": 37.5080897,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177682899",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7686438,\n   "lon": 37.5102838,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177913374",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7933108,\n   "lon": 37.7088196,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177913376",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7911115,\n   "lon": 37.7110404,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177944582",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7043582,\n   "lon": 37.6106481,\n   "opening_hours": null\n  },\n  {\n   "id": "way/177944584",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7040514,\n   "lon": 37.6109882,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178163702",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8053353,\n   "lon": 37.7053786,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178199299",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7933376,\n   "lon": 37.7084825,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178199318",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7930167,\n   "lon": 37.7082877,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178199321",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7928139,\n   "lon": 37.7083384,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178226591",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7891946,\n   "lon": 37.7073999,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178442441",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7982111,\n   "lon": 37.7406698,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178442454",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8065019,\n   "lon": 37.7456433,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178442474",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8033955,\n   "lon": 37.7476214,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178558387",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7821027,\n   "lon": 37.5963811,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178933857",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8364667,\n   "lon": 37.4816819,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178933858",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8377893,\n   "lon": 37.4807192,\n   "opening_hours": null\n  },\n  {\n   "id": "way/178945553",\n   "kind": "organisation",\n   "name": "ЕИРЦ района Западное Дегунино",\n   "office_type": "company",\n   "lat": 55.8732999,\n   "lon": 37.5161721,\n   "opening_hours": null\n  },\n  {\n   "id": "way/179341739",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7781775,\n   "lon": 37.6459108,\n   "opening_hours": null\n  },\n  {\n   "id": "way/179615802",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8082268,\n   "lon": 37.6911458,\n   "opening_hours": null\n  },\n  {\n   "id": "way/180989819",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7812378,\n   "lon": 37.696772,\n   "opening_hours": null\n  },\n  {\n   "id": "way/180989822",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7816392,\n   "lon": 37.69684,\n   "opening_hours": null\n  },\n  {\n   "id": "way/180989824",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7810434,\n   "lon": 37.696908,\n   "opening_hours": null\n  },\n  {\n   "id": "way/181513827",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7803499,\n   "lon": 37.7138625,\n   "opening_hours": null\n  },\n  {\n   "id": "way/182028224",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.6223628,\n   "lon": 37.7536665,\n   "opening_hours": null\n  },\n  {\n   "id": "way/183768728",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8864337,\n   "lon": 37.7363653,\n   "opening_hours": null\n  },\n  {\n   "id": "way/186251450",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7656076,\n   "lon": 37.6493363,\n   "opening_hours": null\n  },\n  {\n   "id": "way/187816033",\n   "kind": "building",\n   "name": "Конечная станция \\"Детский санаторий\\"",\n   "office_type": null,\n   "lat": 55.8293684,\n   "lon": 37.7672319,\n   "opening_hours": null\n  },\n  {\n   "id": "way/187945926",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8015553,\n   "lon": 37.5618256,\n   "opening_hours": null\n  },\n  {\n   "id": "way/187945927",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8012014,\n   "lon": 37.5621475,\n   "opening_hours": null\n  },\n  {\n   "id": "way/187958628",\n   "kind": "organisation",\n   "name": "Байкал Сервис ТК",\n   "office_type": "company",\n   "lat": 55.751079,\n   "lon": 37.7597951,\n   "opening_hours": "Mo-Fr 08:00-18:00"\n  },\n  {\n   "id": "way/187960003",\n   "kind": "building",\n   "name": "Бизнес-центр «Юникон»",\n   "office_type": null,\n   "lat": 55.7546262,\n   "lon": 37.7567098,\n   "opening_hours": null\n  },\n  {\n   "id": "way/188391513",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6618201,\n   "lon": 37.7199394,\n   "opening_hours": null\n  },\n  {\n   "id": "way/188564954",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8223221,\n   "lon": 37.7562344,\n   "opening_hours": null\n  },\n  {\n   "id": "way/188596108",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6107554,\n   "lon": 37.6484382,\n   "opening_hours": null\n  },\n  {\n   "id": "way/189921066",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7920043,\n   "lon": 37.7709022,\n   "opening_hours": null\n  },\n  {\n   "id": "way/190783190",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835426,\n   "lon": 37.7359148,\n   "opening_hours": null\n  },\n  {\n   "id": "way/190783191",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835397,\n   "lon": 37.7373554,\n   "opening_hours": null\n  },\n  {\n   "id": "way/191035365",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8355323,\n   "lon": 37.6461086,\n   "opening_hours": null\n  },\n  {\n   "id": "way/191035367",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8355193,\n   "lon": 37.6464566,\n   "opening_hours": null\n  },\n  {\n   "id": "way/191035368",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8358808,\n   "lon": 37.6484217,\n   "opening_hours": null\n  },\n  {\n   "id": "way/192393349",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7766987,\n   "lon": 37.6781819,\n   "opening_hours": null\n  },\n  {\n   "id": "way/192393351",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7769835,\n   "lon": 37.6768563,\n   "opening_hours": null\n  },\n  {\n   "id": "way/192393353",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7770239,\n   "lon": 37.6754038,\n   "opening_hours": null\n  },\n  {\n   "id": "way/192515033",\n   "kind": "organisation",\n   "name": "Филиал НТЦ Энергетики - ВНИИЭ",\n   "office_type": "research",\n   "lat": 55.657522,\n   "lon": 37.6376283,\n   "opening_hours": null\n  },\n  {\n   "id": "way/192625498",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Чайка Плаза 6\\"",\n   "office_type": null,\n   "lat": 55.7392895,\n   "lon": 37.5978021,\n   "opening_hours": null\n  },\n  {\n   "id": "way/192626130",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390923,\n   "lon": 37.5978798,\n   "opening_hours": null\n  },\n  {\n   "id": "way/193292260",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7601206,\n   "lon": 37.5830483,\n   "opening_hours": null\n  },\n  {\n   "id": "way/193437541",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8298369,\n   "lon": 37.6997979,\n   "opening_hours": null\n  },\n  {\n   "id": "way/193637898",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.803014,\n   "lon": 37.6576307,\n   "opening_hours": null\n  },\n  {\n   "id": "way/193856615",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432417,\n   "lon": 37.5575713,\n   "opening_hours": null\n  },\n  {\n   "id": "way/193864139",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7473062,\n   "lon": 37.5990452,\n   "opening_hours": null\n  },\n  {\n   "id": "way/194925725",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8597123,\n   "lon": 37.4806117,\n   "opening_hours": null\n  },\n  {\n   "id": "way/197511591",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7750578,\n   "lon": 37.630139,\n   "opening_hours": null\n  },\n  {\n   "id": "way/197511711",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7750516,\n   "lon": 37.6297845,\n   "opening_hours": null\n  },\n  {\n   "id": "way/197891060",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7763857,\n   "lon": 37.6290961,\n   "opening_hours": null\n  },\n  {\n   "id": "way/197900519",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8004267,\n   "lon": 37.7370498,\n   "opening_hours": null\n  },\n  {\n   "id": "way/197929393",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7746461,\n   "lon": 37.623568,\n   "opening_hours": null\n  },\n  {\n   "id": "way/198289562",\n   "kind": "building",\n   "name": "Здание конечной станции \\"Уссурийская\\"",\n   "office_type": null,\n   "lat": 55.8219539,\n   "lon": 37.8237203,\n   "opening_hours": null\n  },\n  {\n   "id": "way/200610180",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8159593,\n   "lon": 37.6949784,\n   "opening_hours": null\n  },\n  {\n   "id": "way/200657967",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7802467,\n   "lon": 37.6971841,\n   "opening_hours": null\n  },\n  {\n   "id": "way/200658407",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7804653,\n   "lon": 37.6975463,\n   "opening_hours": null\n  },\n  {\n   "id": "way/201896879",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8445835,\n   "lon": 37.5587855,\n   "opening_hours": null\n  },\n  {\n   "id": "way/202018174",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8075586,\n   "lon": 37.6220666,\n   "opening_hours": null\n  },\n  {\n   "id": "way/202721419",\n   "kind": "building",\n   "name": "Инспекция налоговой службы № 17 по г. Москве",\n   "office_type": "government",\n   "lat": 55.8046706,\n   "lon": 37.645259,\n   "opening_hours": null\n  },\n  {\n   "id": "way/203794026",\n   "kind": "organisation",\n   "name": "Управление Министерства внутренних дел по ЗАО г.Москвы",\n   "office_type": "government",\n   "lat": 55.6909172,\n   "lon": 37.475337,\n   "opening_hours": null\n  },\n  {\n   "id": "way/204059732",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.792229,\n   "lon": 37.7933969,\n   "opening_hours": null\n  },\n  {\n   "id": "way/204170758",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7961856,\n   "lon": 37.5865699,\n   "opening_hours": null\n  },\n  {\n   "id": "way/204855038",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619656,\n   "lon": 37.6375394,\n   "opening_hours": null\n  },\n  {\n   "id": "way/204855048",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7606004,\n   "lon": 37.635402,\n   "opening_hours": null\n  },\n  {\n   "id": "way/205734956",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7586433,\n   "lon": 37.6046215,\n   "opening_hours": null\n  },\n  {\n   "id": "way/205734957",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7576346,\n   "lon": 37.6046355,\n   "opening_hours": null\n  },\n  {\n   "id": "way/205833595",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8021493,\n   "lon": 37.5870852,\n   "opening_hours": null\n  },\n  {\n   "id": "way/206084443",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6458222,\n   "lon": 37.4688664,\n   "opening_hours": null\n  },\n  {\n   "id": "way/207277660",\n   "kind": "organisation",\n   "name": "Техноспас",\n   "office_type": "company",\n   "lat": 55.7655333,\n   "lon": 37.7190595,\n   "opening_hours": null\n  },\n  {\n   "id": "way/208151142",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8113515,\n   "lon": 37.5775186,\n   "opening_hours": null\n  },\n  {\n   "id": "way/209181175",\n   "kind": "building",\n   "name": "ОУФМС района Соколиная Гора",\n   "office_type": null,\n   "lat": 55.766687,\n   "lon": 37.7270156,\n   "opening_hours": null\n  },\n  {\n   "id": "way/210508755",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7722836,\n   "lon": 37.6068906,\n   "opening_hours": null\n  },\n  {\n   "id": "way/213469567",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6541989,\n   "lon": 37.5560751,\n   "opening_hours": null\n  },\n  {\n   "id": "way/213484273",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7508558,\n   "lon": 37.5255782,\n   "opening_hours": null\n  },\n  {\n   "id": "way/213900864",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.713476,\n   "lon": 37.6599075,\n   "opening_hours": null\n  },\n  {\n   "id": "way/214407578",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7420053,\n   "lon": 37.6031928,\n   "opening_hours": null\n  },\n  {\n   "id": "way/214407583",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7437708,\n   "lon": 37.5996126,\n   "opening_hours": null\n  },\n  {\n   "id": "way/214407585",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7437396,\n   "lon": 37.5994306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/214407586",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432261,\n   "lon": 37.5989712,\n   "opening_hours": null\n  },\n  {\n   "id": "way/214407591",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7435742,\n   "lon": 37.6026849,\n   "opening_hours": null\n  },\n  {\n   "id": "way/214676836",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7843697,\n   "lon": 37.6707138,\n   "opening_hours": null\n  },\n  {\n   "id": "way/215205224",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7075646,\n   "lon": 37.6547682,\n   "opening_hours": null\n  },\n  {\n   "id": "way/215550949",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7515995,\n   "lon": 37.6465282,\n   "opening_hours": null\n  },\n  {\n   "id": "way/215601987",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7774622,\n   "lon": 37.5329803,\n   "opening_hours": null\n  },\n  {\n   "id": "way/216913057",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6642581,\n   "lon": 37.4928748,\n   "opening_hours": null\n  },\n  {\n   "id": "way/216919550",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7175998,\n   "lon": 37.7984145,\n   "opening_hours": null\n  },\n  {\n   "id": "way/218655102",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7112924,\n   "lon": 37.5736115,\n   "opening_hours": null\n  },\n  {\n   "id": "way/218655103",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7111885,\n   "lon": 37.5741453,\n   "opening_hours": null\n  },\n  {\n   "id": "way/218680398",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7411041,\n   "lon": 37.6338583,\n   "opening_hours": null\n  },\n  {\n   "id": "way/218680400",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7406515,\n   "lon": 37.6342732,\n   "opening_hours": null\n  },\n  {\n   "id": "way/219053089",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7254233,\n   "lon": 37.6373469,\n   "opening_hours": null\n  },\n  {\n   "id": "way/219053090",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7257602,\n   "lon": 37.6374461,\n   "opening_hours": null\n  },\n  {\n   "id": "way/219055559",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7255906,\n   "lon": 37.6374235,\n   "opening_hours": null\n  },\n  {\n   "id": "way/219945335",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7672652,\n   "lon": 37.5549887,\n   "opening_hours": null\n  },\n  {\n   "id": "way/219964018",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591744,\n   "lon": 37.6357327,\n   "opening_hours": null\n  },\n  {\n   "id": "way/220178033",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7525913,\n   "lon": 37.5161408,\n   "opening_hours": null\n  },\n  {\n   "id": "way/220180880",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8056121,\n   "lon": 37.5678394,\n   "opening_hours": null\n  },\n  {\n   "id": "way/220218844",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6921759,\n   "lon": 37.6593498,\n   "opening_hours": null\n  },\n  {\n   "id": "way/220594550",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7983972,\n   "lon": 37.6703894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/221105520",\n   "kind": "building",\n   "name": "ЗАО \\"Ремстройтрест\\"",\n   "office_type": null,\n   "lat": 55.6591643,\n   "lon": 37.5202651,\n   "opening_hours": null\n  },\n  {\n   "id": "way/221122350",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6594893,\n   "lon": 37.5162866,\n   "opening_hours": null\n  },\n  {\n   "id": "way/221332005",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.729548,\n   "lon": 37.5687268,\n   "opening_hours": null\n  },\n  {\n   "id": "way/221404462",\n   "kind": "organisation",\n   "name": "Вневедомственная охрана СВАО",\n   "office_type": "security",\n   "lat": 55.8653875,\n   "lon": 37.6859092,\n   "opening_hours": null\n  },\n  {\n   "id": "way/223115819",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8204069,\n   "lon": 37.4459941,\n   "opening_hours": null\n  },\n  {\n   "id": "way/223645731",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6872038,\n   "lon": 37.5953543,\n   "opening_hours": null\n  },\n  {\n   "id": "way/224343561",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.803955,\n   "lon": 37.7067722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/225157172",\n   "kind": "organisation",\n   "name": "ФЕСКО Транспортная группа",\n   "office_type": "company",\n   "lat": 55.7358703,\n   "lon": 37.6433831,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "way/227029132",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7641408,\n   "lon": 37.6016867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/227527246",\n   "kind": "building",\n   "name": "Бизнес-центр «Газ Филд»",\n   "office_type": null,\n   "lat": 55.6576621,\n   "lon": 37.5294023,\n   "opening_hours": null\n  },\n  {\n   "id": "way/229048378",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7452927,\n   "lon": 37.830207,\n   "opening_hours": null\n  },\n  {\n   "id": "way/229427664",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7526448,\n   "lon": 37.6041681,\n   "opening_hours": null\n  },\n  {\n   "id": "way/230895986",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8103497,\n   "lon": 37.6560261,\n   "opening_hours": null\n  },\n  {\n   "id": "way/232072083",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7133482,\n   "lon": 37.8050029,\n   "opening_hours": null\n  },\n  {\n   "id": "way/232164803",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.748961,\n   "lon": 37.6814813,\n   "opening_hours": null\n  },\n  {\n   "id": "way/232408194",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7431253,\n   "lon": 37.6274428,\n   "opening_hours": null\n  },\n  {\n   "id": "way/232434687",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.712098,\n   "lon": 37.5961975,\n   "opening_hours": null\n  },\n  {\n   "id": "way/232435130",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7117083,\n   "lon": 37.5932739,\n   "opening_hours": null\n  },\n  {\n   "id": "way/232818584",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7761596,\n   "lon": 37.6887802,\n   "opening_hours": null\n  },\n  {\n   "id": "way/235139153",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7422719,\n   "lon": 37.5978811,\n   "opening_hours": null\n  },\n  {\n   "id": "way/236083573",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8007914,\n   "lon": 37.6374086,\n   "opening_hours": null\n  },\n  {\n   "id": "way/236083574",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8004978,\n   "lon": 37.6400049,\n   "opening_hours": null\n  },\n  {\n   "id": "way/236083575",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8008899,\n   "lon": 37.6372081,\n   "opening_hours": null\n  },\n  {\n   "id": "way/236083577",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8008145,\n   "lon": 37.6393743,\n   "opening_hours": null\n  },\n  {\n   "id": "way/237176019",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7419797,\n   "lon": 37.6470696,\n   "opening_hours": null\n  },\n  {\n   "id": "way/237176022",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7438061,\n   "lon": 37.6479966,\n   "opening_hours": null\n  },\n  {\n   "id": "way/237176047",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7416746,\n   "lon": 37.6473692,\n   "opening_hours": null\n  },\n  {\n   "id": "way/238232287",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8061135,\n   "lon": 37.617495,\n   "opening_hours": null\n  },\n  {\n   "id": "way/238756693",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.751106,\n   "lon": 37.5993293,\n   "opening_hours": null\n  },\n  {\n   "id": "way/239653878",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7859675,\n   "lon": 37.6344255,\n   "opening_hours": null\n  },\n  {\n   "id": "way/239966894",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7402187,\n   "lon": 37.5528245,\n   "opening_hours": null\n  },\n  {\n   "id": "way/239989598",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6429522,\n   "lon": 37.5900724,\n   "opening_hours": null\n  },\n  {\n   "id": "way/240636771",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.5938313,\n   "lon": 37.7453582,\n   "opening_hours": null\n  },\n  {\n   "id": "way/241232988",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6489209,\n   "lon": 37.7436902,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "way/241751693",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7509072,\n   "lon": 37.6839541,\n   "opening_hours": null\n  },\n  {\n   "id": "way/241751695",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7284944,\n   "lon": 37.6602921,\n   "opening_hours": null\n  },\n  {\n   "id": "way/241751701",\n   "kind": "organisation",\n   "name": "Гормост",\n   "office_type": "company",\n   "lat": 55.7505219,\n   "lon": 37.6809752,\n   "opening_hours": null\n  },\n  {\n   "id": "way/241819992",\n   "kind": "organisation",\n   "name": "Сытый слон",\n   "office_type": "company",\n   "lat": 55.7177423,\n   "lon": 37.7022331,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242346315",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8460286,\n   "lon": 37.5795461,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242346322",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8445771,\n   "lon": 37.576824,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242346323",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8442466,\n   "lon": 37.5780304,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242446768",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7907937,\n   "lon": 37.5747722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242449961",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7906143,\n   "lon": 37.7035703,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242682007",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7269338,\n   "lon": 37.6615719,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242682009",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.72695,\n   "lon": 37.6621562,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242682010",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7269257,\n   "lon": 37.6610021,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242682015",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.728229,\n   "lon": 37.6570709,\n   "opening_hours": null\n  },\n  {\n   "id": "way/242682017",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7288558,\n   "lon": 37.6584077,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243260960",\n   "kind": "building",\n   "name": "Павильон",\n   "office_type": null,\n   "lat": 55.7636085,\n   "lon": 37.6613787,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243512956",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7066043,\n   "lon": 37.691896,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243512959",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7079354,\n   "lon": 37.6884368,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243512967",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7075591,\n   "lon": 37.6878029,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243512968",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7074611,\n   "lon": 37.6874405,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243512978",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7068858,\n   "lon": 37.6926796,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243512985",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7065935,\n   "lon": 37.6910788,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243512995",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7071606,\n   "lon": 37.6910966,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243513006",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7071528,\n   "lon": 37.6937906,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243513011",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.707267,\n   "lon": 37.6883108,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243514340",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7067428,\n   "lon": 37.6884337,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243542014",\n   "kind": "building",\n   "name": "Башня А",\n   "office_type": null,\n   "lat": 55.7372789,\n   "lon": 37.5344829,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243542015",\n   "kind": "building",\n   "name": "Башня D",\n   "office_type": null,\n   "lat": 55.7362023,\n   "lon": 37.5316545,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243542016",\n   "kind": "building",\n   "name": "Корпус E",\n   "office_type": null,\n   "lat": 55.7362983,\n   "lon": 37.5329764,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243832995",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588067,\n   "lon": 37.6398662,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243848720",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7415513,\n   "lon": 37.6599942,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243848722",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7417335,\n   "lon": 37.6607408,\n   "opening_hours": null\n  },\n  {\n   "id": "way/243920374",\n   "kind": "organisation",\n   "name": "Спасо-хаус",\n   "office_type": "diplomatic",\n   "lat": 55.7508198,\n   "lon": 37.5884695,\n   "opening_hours": null\n  },\n  {\n   "id": "way/244725272",\n   "kind": "building",\n   "name": "Угрешский",\n   "office_type": null,\n   "lat": 55.7152479,\n   "lon": 37.6962504,\n   "opening_hours": null\n  },\n  {\n   "id": "way/244963462",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7881097,\n   "lon": 37.6313968,\n   "opening_hours": null\n  },\n  {\n   "id": "way/244963463",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7880487,\n   "lon": 37.6311157,\n   "opening_hours": null\n  },\n  {\n   "id": "way/244963471",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7874718,\n   "lon": 37.6318263,\n   "opening_hours": null\n  },\n  {\n   "id": "way/244963474",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7878417,\n   "lon": 37.6313036,\n   "opening_hours": null\n  },\n  {\n   "id": "way/245991120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.758094,\n   "lon": 37.6018014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/246898125",\n   "kind": "organisation",\n   "name": "Бюро мед-соц экспертизы №89",\n   "office_type": "yes",\n   "lat": 55.7091681,\n   "lon": 37.811852,\n   "opening_hours": null\n  },\n  {\n   "id": "way/247107152",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7567049,\n   "lon": 37.6031845,\n   "opening_hours": null\n  },\n  {\n   "id": "way/247212628",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7205558,\n   "lon": 37.6354225,\n   "opening_hours": null\n  },\n  {\n   "id": "way/247212630",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7208898,\n   "lon": 37.63578,\n   "opening_hours": null\n  },\n  {\n   "id": "way/247938636",\n   "kind": "organisation",\n   "name": "Российский институт стратегических исследований",\n   "office_type": "government",\n   "lat": 55.8568054,\n   "lon": 37.5065372,\n   "opening_hours": null\n  },\n  {\n   "id": "way/248104563",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7319337,\n   "lon": 37.6186876,\n   "opening_hours": null\n  },\n  {\n   "id": "way/249104978",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.782572,\n   "lon": 37.5994058,\n   "opening_hours": null\n  },\n  {\n   "id": "way/249167733",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.737307,\n   "lon": 37.6743178,\n   "opening_hours": null\n  },\n  {\n   "id": "way/249167735",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7364434,\n   "lon": 37.6751581,\n   "opening_hours": null\n  },\n  {\n   "id": "way/249172399",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7362748,\n   "lon": 37.6938894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/249172401",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7362014,\n   "lon": 37.6947063,\n   "opening_hours": null\n  },\n  {\n   "id": "way/249468659",\n   "kind": "building",\n   "name": "Посольство Бенина",\n   "office_type": "diplomatic",\n   "lat": 55.7695056,\n   "lon": 37.6090767,\n   "opening_hours": null\n  },\n  {\n   "id": "way/250567165",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7466544,\n   "lon": 37.6649131,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135845",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7903503,\n   "lon": 37.5815923,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135847",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7892392,\n   "lon": 37.5788764,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135849",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7895022,\n   "lon": 37.5782417,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135852",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7906408,\n   "lon": 37.5818316,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135853",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7905437,\n   "lon": 37.5808476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135856",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7905077,\n   "lon": 37.5772827,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135858",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7908669,\n   "lon": 37.5819173,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135862",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7899833,\n   "lon": 37.5774187,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135864",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7890089,\n   "lon": 37.5813414,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135866",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7901547,\n   "lon": 37.5798364,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135867",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7891178,\n   "lon": 37.5795,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135868",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7900847,\n   "lon": 37.5786016,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251135870",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7906,\n   "lon": 37.5822069,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251137900",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.791066,\n   "lon": 37.5796759,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251236181",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7723311,\n   "lon": 37.6113814,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251236182",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7726802,\n   "lon": 37.6113989,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251313996",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7545545,\n   "lon": 37.6066539,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251313997",\n   "kind": "building",\n   "name": "Никитская",\n   "office_type": null,\n   "lat": 55.7549098,\n   "lon": 37.607123,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251314000",\n   "kind": "building",\n   "name": "ФГБУ «ЦУРЭН»",\n   "office_type": "government",\n   "lat": 55.7552627,\n   "lon": 37.6068492,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "way/251318129",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7538241,\n   "lon": 37.6070171,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251318130",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7544844,\n   "lon": 37.6071132,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251757449",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.739284,\n   "lon": 37.6289609,\n   "opening_hours": null\n  },\n  {\n   "id": "way/251769531",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.743744,\n   "lon": 37.6272471,\n   "opening_hours": null\n  },\n  {\n   "id": "way/252143937",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7421733,\n   "lon": 37.628099,\n   "opening_hours": null\n  },\n  {\n   "id": "way/252270594",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7598028,\n   "lon": 37.578378,\n   "opening_hours": null\n  },\n  {\n   "id": "way/252270595",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7599694,\n   "lon": 37.5783176,\n   "opening_hours": null\n  },\n  {\n   "id": "way/253105988",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7589332,\n   "lon": 37.6097639,\n   "opening_hours": null\n  },\n  {\n   "id": "way/253530551",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7188703,\n   "lon": 37.6346694,\n   "opening_hours": null\n  },\n  {\n   "id": "way/253530556",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.722126,\n   "lon": 37.6370785,\n   "opening_hours": null\n  },\n  {\n   "id": "way/253638247",\n   "kind": "building",\n   "name": "офисный центр \\"На Научном\\"",\n   "office_type": null,\n   "lat": 55.6557411,\n   "lon": 37.5519135,\n   "opening_hours": null\n  },\n  {\n   "id": "way/253642538",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7550193,\n   "lon": 37.6040845,\n   "opening_hours": null\n  },\n  {\n   "id": "way/254832554",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7541933,\n   "lon": 37.6057991,\n   "opening_hours": null\n  },\n  {\n   "id": "way/254837173",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7555197,\n   "lon": 37.6061882,\n   "opening_hours": null\n  },\n  {\n   "id": "way/254837174",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7552921,\n   "lon": 37.6039846,\n   "opening_hours": null\n  },\n  {\n   "id": "way/254872690",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7535389,\n   "lon": 37.603542,\n   "opening_hours": null\n  },\n  {\n   "id": "way/254872692",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.753692,\n   "lon": 37.6034394,\n   "opening_hours": null\n  },\n  {\n   "id": "way/255045500",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.756337,\n   "lon": 37.6016531,\n   "opening_hours": null\n  },\n  {\n   "id": "way/255838301",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7578377,\n   "lon": 37.6017147,\n   "opening_hours": null\n  },\n  {\n   "id": "way/255838302",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7579603,\n   "lon": 37.6019955,\n   "opening_hours": null\n  },\n  {\n   "id": "way/255845109",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7582936,\n   "lon": 37.6029437,\n   "opening_hours": null\n  },\n  {\n   "id": "way/256044992",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7586183,\n   "lon": 37.6014857,\n   "opening_hours": null\n  },\n  {\n   "id": "way/256356846",\n   "kind": "building",\n   "name": "Центр социального обслуживания «Арбат». Филиал «Пресненский»",\n   "office_type": null,\n   "lat": 55.7595823,\n   "lon": 37.6017102,\n   "opening_hours": null\n  },\n  {\n   "id": "way/256363639",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8682643,\n   "lon": 37.582352,\n   "opening_hours": null\n  },\n  {\n   "id": "way/256390894",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8313585,\n   "lon": 37.5750589,\n   "opening_hours": null\n  },\n  {\n   "id": "way/257159573",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7276722,\n   "lon": 37.6453738,\n   "opening_hours": null\n  },\n  {\n   "id": "way/258640275",\n   "kind": "organisation",\n   "name": "Чертановский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.6507309,\n   "lon": 37.6115187,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "way/259779434",\n   "kind": "building",\n   "name": "Родионовский",\n   "office_type": null,\n   "lat": 55.710777,\n   "lon": 37.654172,\n   "opening_hours": null\n  },\n  {\n   "id": "way/259881378",\n   "kind": "organisation",\n   "name": "МОЭСК",\n   "office_type": "company",\n   "lat": 55.7130875,\n   "lon": 37.6456714,\n   "opening_hours": null\n  },\n  {\n   "id": "way/259881385",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7127246,\n   "lon": 37.6444739,\n   "opening_hours": null\n  },\n  {\n   "id": "way/260067409",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7995885,\n   "lon": 37.6127941,\n   "opening_hours": null\n  },\n  {\n   "id": "way/260067410",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7995152,\n   "lon": 37.613018,\n   "opening_hours": null\n  },\n  {\n   "id": "way/260246276",\n   "kind": "organisation",\n   "name": "НИИ технико-экономических исследований в химическом комплексе",\n   "office_type": "research",\n   "lat": 55.6614336,\n   "lon": 37.5552709,\n   "opening_hours": null\n  },\n  {\n   "id": "way/260269275",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6550761,\n   "lon": 37.4832902,\n   "opening_hours": null\n  },\n  {\n   "id": "way/260462280",\n   "kind": "organisation",\n   "name": "ФГУП Антидопинговый центр",\n   "office_type": "government",\n   "lat": 55.7613896,\n   "lon": 37.6718225,\n   "opening_hours": null\n  },\n  {\n   "id": "way/260600578",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8069499,\n   "lon": 37.6094479,\n   "opening_hours": null\n  },\n  {\n   "id": "way/261381559",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7079406,\n   "lon": 37.6099688,\n   "opening_hours": null\n  },\n  {\n   "id": "way/261381562",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7076997,\n   "lon": 37.6092116,\n   "opening_hours": null\n  },\n  {\n   "id": "way/261381563",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7073638,\n   "lon": 37.6094195,\n   "opening_hours": null\n  },\n  {\n   "id": "way/261381564",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7073879,\n   "lon": 37.6090133,\n   "opening_hours": null\n  },\n  {\n   "id": "way/261381569",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7070158,\n   "lon": 37.6097803,\n   "opening_hours": null\n  },\n  {\n   "id": "way/261488681",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.739835,\n   "lon": 37.5285492,\n   "opening_hours": null\n  },\n  {\n   "id": "way/261752407",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7745211,\n   "lon": 37.6297545,\n   "opening_hours": null\n  },\n  {\n   "id": "way/261952030",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7910425,\n   "lon": 37.6354606,\n   "opening_hours": null\n  },\n  {\n   "id": "way/262199961",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.704446,\n   "lon": 37.6153568,\n   "opening_hours": null\n  },\n  {\n   "id": "way/262199962",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7042594,\n   "lon": 37.6158329,\n   "opening_hours": null\n  },\n  {\n   "id": "way/262331860",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Огородный\\"",\n   "office_type": null,\n   "lat": 55.8186226,\n   "lon": 37.5950404,\n   "opening_hours": null\n  },\n  {\n   "id": "way/264313344",\n   "kind": "organisation",\n   "name": "Ростехнадзор",\n   "office_type": "government",\n   "lat": 55.8690291,\n   "lon": 37.4895229,\n   "opening_hours": null\n  },\n  {\n   "id": "way/265438946",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8094917,\n   "lon": 37.6159447,\n   "opening_hours": null\n  },\n  {\n   "id": "way/265438949",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8101006,\n   "lon": 37.6150243,\n   "opening_hours": null\n  },\n  {\n   "id": "way/265438967",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8062454,\n   "lon": 37.6178374,\n   "opening_hours": null\n  },\n  {\n   "id": "way/265898502",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7434976,\n   "lon": 37.6530821,\n   "opening_hours": null\n  },\n  {\n   "id": "way/265898503",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7439148,\n   "lon": 37.6483355,\n   "opening_hours": null\n  },\n  {\n   "id": "way/270788762",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7115527,\n   "lon": 37.6794751,\n   "opening_hours": null\n  },\n  {\n   "id": "way/271847439",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7302603,\n   "lon": 37.6461017,\n   "opening_hours": null\n  },\n  {\n   "id": "way/272484895",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.865085,\n   "lon": 37.6758468,\n   "opening_hours": null\n  },\n  {\n   "id": "way/277554021",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.8684021,\n   "lon": 37.5819477,\n   "opening_hours": null\n  },\n  {\n   "id": "way/279535755",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7328712,\n   "lon": 37.6251981,\n   "opening_hours": null\n  },\n  {\n   "id": "way/280141507",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7883531,\n   "lon": 37.5679557,\n   "opening_hours": null\n  },\n  {\n   "id": "way/280141509",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7885307,\n   "lon": 37.5674588,\n   "opening_hours": null\n  },\n  {\n   "id": "way/280141513",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7887086,\n   "lon": 37.5670505,\n   "opening_hours": null\n  },\n  {\n   "id": "way/281776082",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.710154,\n   "lon": 37.6782611,\n   "opening_hours": null\n  },\n  {\n   "id": "way/284423447",\n   "kind": "building",\n   "name": "Любимый носик",\n   "office_type": null,\n   "lat": 55.7067042,\n   "lon": 37.6716356,\n   "opening_hours": null\n  },\n  {\n   "id": "way/288175870",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6560136,\n   "lon": 37.5211515,\n   "opening_hours": null\n  },\n  {\n   "id": "way/290763178",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6656321,\n   "lon": 37.6203156,\n   "opening_hours": null\n  },\n  {\n   "id": "way/290803433",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7796592,\n   "lon": 37.5712306,\n   "opening_hours": null\n  },\n  {\n   "id": "way/291342109",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7726171,\n   "lon": 37.6153922,\n   "opening_hours": null\n  },\n  {\n   "id": "way/291726763",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7878478,\n   "lon": 37.4992857,\n   "opening_hours": null\n  },\n  {\n   "id": "way/292179843",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447722,\n   "lon": 37.5866096,\n   "opening_hours": null\n  },\n  {\n   "id": "way/292179845",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7451219,\n   "lon": 37.5864567,\n   "opening_hours": null\n  },\n  {\n   "id": "way/292180531",\n   "kind": "building",\n   "name": "Смоленский пассаж 2",\n   "office_type": null,\n   "lat": 55.7482373,\n   "lon": 37.5813706,\n   "opening_hours": null\n  },\n  {\n   "id": "way/292375657",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7352543,\n   "lon": 37.5883175,\n   "opening_hours": null\n  },\n  {\n   "id": "way/292375660",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7352268,\n   "lon": 37.5887805,\n   "opening_hours": null\n  },\n  {\n   "id": "way/292381278",\n   "kind": "building",\n   "name": "Davis",\n   "office_type": null,\n   "lat": 55.8045144,\n   "lon": 37.5903043,\n   "opening_hours": null\n  },\n  {\n   "id": "way/292808274",\n   "kind": "building",\n   "name": "КПП",\n   "office_type": null,\n   "lat": 55.7134848,\n   "lon": 37.6327202,\n   "opening_hours": null\n  },\n  {\n   "id": "way/294431543",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7044796,\n   "lon": 37.614087,\n   "opening_hours": null\n  },\n  {\n   "id": "way/294431554",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7041474,\n   "lon": 37.6137186,\n   "opening_hours": null\n  },\n  {\n   "id": "way/294431555",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7044179,\n   "lon": 37.6136283,\n   "opening_hours": null\n  },\n  {\n   "id": "way/294431557",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7059011,\n   "lon": 37.6131942,\n   "opening_hours": null\n  },\n  {\n   "id": "way/294431562",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7070183,\n   "lon": 37.6165518,\n   "opening_hours": null\n  },\n  {\n   "id": "way/294431564",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7063804,\n   "lon": 37.6124082,\n   "opening_hours": null\n  },\n  {\n   "id": "way/295060264",\n   "kind": "organisation",\n   "name": "Стройтрансгаз-М",\n   "office_type": "company",\n   "lat": 55.6665698,\n   "lon": 37.6128924,\n   "opening_hours": null\n  },\n  {\n   "id": "way/295248998",\n   "kind": "organisation",\n   "name": "Институт общей физики",\n   "office_type": "research",\n   "lat": 55.6954335,\n   "lon": 37.567155,\n   "opening_hours": null\n  },\n  {\n   "id": "way/296891267",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7413531,\n   "lon": 37.664443,\n   "opening_hours": null\n  },\n  {\n   "id": "way/296891273",\n   "kind": "building",\n   "name": "Дорожная инспекция ОАТИ",\n   "office_type": "government",\n   "lat": 55.7387198,\n   "lon": 37.6643219,\n   "opening_hours": null\n  },\n  {\n   "id": "way/296891718",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7462965,\n   "lon": 37.6674001,\n   "opening_hours": null\n  },\n  {\n   "id": "way/296987649",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7487861,\n   "lon": 37.6603839,\n   "opening_hours": null\n  },\n  {\n   "id": "way/297022249",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7460028,\n   "lon": 37.6673897,\n   "opening_hours": null\n  },\n  {\n   "id": "way/297086886",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7453137,\n   "lon": 37.6534765,\n   "opening_hours": null\n  },\n  {\n   "id": "way/297145270",\n   "kind": "building",\n   "name": "Автобаза МОЭСК",\n   "office_type": null,\n   "lat": 55.7049602,\n   "lon": 37.5871272,\n   "opening_hours": null\n  },\n  {\n   "id": "way/297179087",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7510364,\n   "lon": 37.6778115,\n   "opening_hours": null\n  },\n  {\n   "id": "way/297759264",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7293624,\n   "lon": 37.6262969,\n   "opening_hours": null\n  },\n  {\n   "id": "way/299126287",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7099271,\n   "lon": 37.6866629,\n   "opening_hours": null\n  },\n  {\n   "id": "way/299280470",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7101294,\n   "lon": 37.6471945,\n   "opening_hours": null\n  },\n  {\n   "id": "way/299381445",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.656092,\n   "lon": 37.5995051,\n   "opening_hours": null\n  },\n  {\n   "id": "way/299782864",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7929549,\n   "lon": 37.6966853,\n   "opening_hours": null\n  },\n  {\n   "id": "way/299782868",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7932213,\n   "lon": 37.6970302,\n   "opening_hours": null\n  },\n  {\n   "id": "way/299782872",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7925595,\n   "lon": 37.6953032,\n   "opening_hours": null\n  },\n  {\n   "id": "way/299906959",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6458603,\n   "lon": 37.6543217,\n   "opening_hours": null\n  },\n  {\n   "id": "way/299995582",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7150053,\n   "lon": 37.6726763,\n   "opening_hours": null\n  },\n  {\n   "id": "way/300123611",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7305738,\n   "lon": 37.6193231,\n   "opening_hours": null\n  },\n  {\n   "id": "way/300340509",\n   "kind": "organisation",\n   "name": "Посольство Кубы",\n   "office_type": "diplomatic",\n   "lat": 55.7320468,\n   "lon": 37.6236248,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "way/300619115",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8468878,\n   "lon": 37.4854261,\n   "opening_hours": null\n  },\n  {\n   "id": "way/301689702",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.748518,\n   "lon": 37.7625034,\n   "opening_hours": null\n  },\n  {\n   "id": "way/301689716",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7487266,\n   "lon": 37.7625966,\n   "opening_hours": null\n  },\n  {\n   "id": "way/301822032",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7508313,\n   "lon": 37.7730277,\n   "opening_hours": null\n  },\n  {\n   "id": "way/301822036",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7504842,\n   "lon": 37.7722307,\n   "opening_hours": null\n  },\n  {\n   "id": "way/301822038",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7508092,\n   "lon": 37.7720024,\n   "opening_hours": null\n  },\n  {\n   "id": "way/303655233",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7267639,\n   "lon": 37.6316372,\n   "opening_hours": null\n  },\n  {\n   "id": "way/303670944",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7788405,\n   "lon": 37.6916827,\n   "opening_hours": null\n  },\n  {\n   "id": "way/305258203",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.755491,\n   "lon": 37.7313544,\n   "opening_hours": null\n  },\n  {\n   "id": "way/306836838",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7485682,\n   "lon": 37.5865402,\n   "opening_hours": null\n  },\n  {\n   "id": "way/306836839",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7482629,\n   "lon": 37.5861395,\n   "opening_hours": null\n  },\n  {\n   "id": "way/306836840",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7485146,\n   "lon": 37.586197,\n   "opening_hours": null\n  },\n  {\n   "id": "way/307039429",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6915707,\n   "lon": 37.6060353,\n   "opening_hours": null\n  },\n  {\n   "id": "way/307158980",\n   "kind": "organisation",\n   "name": "Центральный спортивный клуб МЧС",\n   "office_type": "government",\n   "lat": 55.764272,\n   "lon": 37.7733808,\n   "opening_hours": null\n  },\n  {\n   "id": "way/307862058",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8322414,\n   "lon": 37.4719549,\n   "opening_hours": null\n  },\n  {\n   "id": "way/308378620",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.800641,\n   "lon": 37.6367913,\n   "opening_hours": null\n  },\n  {\n   "id": "way/308378621",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8001362,\n   "lon": 37.6373828,\n   "opening_hours": null\n  },\n  {\n   "id": "way/308378634",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8018385,\n   "lon": 37.6381357,\n   "opening_hours": null\n  },\n  {\n   "id": "way/308903318",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8010773,\n   "lon": 37.6372046,\n   "opening_hours": null\n  },\n  {\n   "id": "way/308967740",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7047211,\n   "lon": 37.6163325,\n   "opening_hours": null\n  },\n  {\n   "id": "way/308967741",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7045022,\n   "lon": 37.6167808,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310127264",\n   "kind": "organisation",\n   "name": "Российский федеральный центр судебной экспертизы",\n   "office_type": "government",\n   "lat": 55.7569048,\n   "lon": 37.6340603,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310261743",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7436946,\n   "lon": 37.6301943,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310276420",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7450426,\n   "lon": 37.6307008,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310343705",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7802502,\n   "lon": 37.7005975,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310374719",\n   "kind": "building",\n   "name": "Деловой центр \\"Калибр\\"",\n   "office_type": null,\n   "lat": 55.8093808,\n   "lon": 37.6284224,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310402008",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7750193,\n   "lon": 37.6803664,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310493746",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7394891,\n   "lon": 37.6338033,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310520523",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8098938,\n   "lon": 37.6045033,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310520538",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8099902,\n   "lon": 37.6042875,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310520551",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8096314,\n   "lon": 37.6052765,\n   "opening_hours": null\n  },\n  {\n   "id": "way/310520580",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8140597,\n   "lon": 37.6034657,\n   "opening_hours": null\n  },\n  {\n   "id": "way/311375928",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7498705,\n   "lon": 37.7746515,\n   "opening_hours": null\n  },\n  {\n   "id": "way/311375929",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7495858,\n   "lon": 37.7746476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/312097699",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8112775,\n   "lon": 37.6177634,\n   "opening_hours": null\n  },\n  {\n   "id": "way/312097814",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8113831,\n   "lon": 37.618042,\n   "opening_hours": null\n  },\n  {\n   "id": "way/312131702",\n   "kind": "building",\n   "name": "НИИ \\"Аэропроект\\"",\n   "office_type": "research",\n   "lat": 55.815601,\n   "lon": 37.498954,\n   "opening_hours": null\n  },\n  {\n   "id": "way/313608561",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7455491,\n   "lon": 37.675039,\n   "opening_hours": null\n  },\n  {\n   "id": "way/313611298",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7450432,\n   "lon": 37.6878107,\n   "opening_hours": null\n  },\n  {\n   "id": "way/314616106",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7143405,\n   "lon": 37.6614073,\n   "opening_hours": null\n  },\n  {\n   "id": "way/314973025",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7039913,\n   "lon": 37.7697074,\n   "opening_hours": null\n  },\n  {\n   "id": "way/314973026",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7037655,\n   "lon": 37.7681542,\n   "opening_hours": null\n  },\n  {\n   "id": "way/315311698",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6326525,\n   "lon": 37.8285337,\n   "opening_hours": null\n  },\n  {\n   "id": "way/315745538",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Тропикано\\"",\n   "office_type": null,\n   "lat": 55.7625379,\n   "lon": 37.5702676,\n   "opening_hours": null\n  },\n  {\n   "id": "way/316269736",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7523756,\n   "lon": 37.6352658,\n   "opening_hours": null\n  },\n  {\n   "id": "way/316439134",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6779719,\n   "lon": 37.6324645,\n   "opening_hours": null\n  },\n  {\n   "id": "way/316776213",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7528755,\n   "lon": 37.6832811,\n   "opening_hours": null\n  },\n  {\n   "id": "way/316793853",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835032,\n   "lon": 37.7202208,\n   "opening_hours": null\n  },\n  {\n   "id": "way/316828625",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6999642,\n   "lon": 37.6242264,\n   "opening_hours": null\n  },\n  {\n   "id": "way/317173750",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7179209,\n   "lon": 37.446286,\n   "opening_hours": null\n  },\n  {\n   "id": "way/317173754",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7174937,\n   "lon": 37.446241,\n   "opening_hours": null\n  },\n  {\n   "id": "way/317173755",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7177098,\n   "lon": 37.4462659,\n   "opening_hours": null\n  },\n  {\n   "id": "way/317668226",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7613322,\n   "lon": 37.7346182,\n   "opening_hours": null\n  },\n  {\n   "id": "way/317918942",\n   "kind": "building",\n   "name": "Центральная избирательная комиссия Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7575259,\n   "lon": 37.6265897,\n   "opening_hours": null\n  },\n  {\n   "id": "way/318342094",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Слободской\\"",\n   "office_type": null,\n   "lat": 55.7124932,\n   "lon": 37.6544765,\n   "opening_hours": null\n  },\n  {\n   "id": "way/318342096",\n   "kind": "building",\n   "name": "Симонов-плаза",\n   "office_type": null,\n   "lat": 55.7111912,\n   "lon": 37.6528303,\n   "opening_hours": null\n  },\n  {\n   "id": "way/318342097",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7107144,\n   "lon": 37.6533608,\n   "opening_hours": null\n  },\n  {\n   "id": "way/318342098",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Омега-2\\" корп. B",\n   "office_type": null,\n   "lat": 55.7102372,\n   "lon": 37.6528249,\n   "opening_hours": null\n  },\n  {\n   "id": "way/318858405",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7782918,\n   "lon": 37.6483907,\n   "opening_hours": null\n  },\n  {\n   "id": "way/319196593",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8099792,\n   "lon": 37.5719455,\n   "opening_hours": null\n  },\n  {\n   "id": "way/319196595",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8098059,\n   "lon": 37.5720668,\n   "opening_hours": null\n  },\n  {\n   "id": "way/319196597",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8099356,\n   "lon": 37.5716481,\n   "opening_hours": null\n  },\n  {\n   "id": "way/320168852",\n   "kind": "organisation",\n   "name": "ООО «СпецМонтажПроект»",\n   "office_type": "company",\n   "lat": 55.7357542,\n   "lon": 37.715839,\n   "opening_hours": null\n  },\n  {\n   "id": "way/320168853",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7360454,\n   "lon": 37.7144892,\n   "opening_hours": null\n  },\n  {\n   "id": "way/321790157",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7521574,\n   "lon": 37.6413589,\n   "opening_hours": null\n  },\n  {\n   "id": "way/321959547",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7590914,\n   "lon": 37.6290811,\n   "opening_hours": null\n  },\n  {\n   "id": "way/322142537",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7699096,\n   "lon": 37.6643535,\n   "opening_hours": null\n  },\n  {\n   "id": "way/323607951",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7849052,\n   "lon": 37.6726472,\n   "opening_hours": null\n  },\n  {\n   "id": "way/323766949",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7550332,\n   "lon": 37.5853866,\n   "opening_hours": null\n  },\n  {\n   "id": "way/325937079",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6778053,\n   "lon": 37.5747823,\n   "opening_hours": null\n  },\n  {\n   "id": "way/326009061",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6816996,\n   "lon": 37.5872691,\n   "opening_hours": null\n  },\n  {\n   "id": "way/326009067",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6809227,\n   "lon": 37.5897504,\n   "opening_hours": null\n  },\n  {\n   "id": "way/326498000",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7559995,\n   "lon": 37.5858869,\n   "opening_hours": null\n  },\n  {\n   "id": "way/326498002",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7563334,\n   "lon": 37.5867689,\n   "opening_hours": null\n  },\n  {\n   "id": "way/326498013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7558854,\n   "lon": 37.5860367,\n   "opening_hours": null\n  },\n  {\n   "id": "way/326498056",\n   "kind": "building",\n   "name": "Отдел социальной защиты населения района Арбат",\n   "office_type": null,\n   "lat": 55.7556386,\n   "lon": 37.5862755,\n   "opening_hours": null\n  },\n  {\n   "id": "way/327567712",\n   "kind": "organisation",\n   "name": "Резиденция посла Бельгии в России",\n   "office_type": "diplomatic",\n   "lat": 55.7550457,\n   "lon": 37.5949476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/327567726",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7551785,\n   "lon": 37.5970346,\n   "opening_hours": null\n  },\n  {\n   "id": "way/330097156",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7391718,\n   "lon": 37.5903239,\n   "opening_hours": null\n  },\n  {\n   "id": "way/330097160",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Дом Чайковского\\"",\n   "office_type": null,\n   "lat": 55.7415922,\n   "lon": 37.5963051,\n   "opening_hours": null\n  },\n  {\n   "id": "way/331213875",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт экспериментальной ветеринарии им. Я.Р. Коваленко",\n   "office_type": "research",\n   "lat": 55.720917,\n   "lon": 37.7735473,\n   "opening_hours": null\n  },\n  {\n   "id": "way/331353941",\n   "kind": "building",\n   "name": "ГАУ Медиацентр",\n   "office_type": null,\n   "lat": 55.7994288,\n   "lon": 37.7056579,\n   "opening_hours": null\n  },\n  {\n   "id": "way/331445512",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "administrative",\n   "lat": 55.6715874,\n   "lon": 37.7731189,\n   "opening_hours": null\n  },\n  {\n   "id": "way/331722140",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7182268,\n   "lon": 37.7038111,\n   "opening_hours": null\n  },\n  {\n   "id": "way/334195838",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626358,\n   "lon": 37.5879045,\n   "opening_hours": null\n  },\n  {\n   "id": "way/334329884",\n   "kind": "organisation",\n   "name": "ГУП \\"Москоллектор\\"",\n   "office_type": "company",\n   "lat": 55.7867203,\n   "lon": 37.6605321,\n   "opening_hours": null\n  },\n  {\n   "id": "way/334496706",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7042971,\n   "lon": 37.5685649,\n   "opening_hours": null\n  },\n  {\n   "id": "way/335073903",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6679699,\n   "lon": 37.5665087,\n   "opening_hours": null\n  },\n  {\n   "id": "way/335074429",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7291867,\n   "lon": 37.6401217,\n   "opening_hours": null\n  },\n  {\n   "id": "way/335226161",\n   "kind": "building",\n   "name": "Башня C",\n   "office_type": null,\n   "lat": 55.7364472,\n   "lon": 37.5325768,\n   "opening_hours": null\n  },\n  {\n   "id": "way/335317730",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7185113,\n   "lon": 37.6035888,\n   "opening_hours": null\n  },\n  {\n   "id": "way/335746189",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8121802,\n   "lon": 37.4568242,\n   "opening_hours": null\n  },\n  {\n   "id": "way/336407970",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7971609,\n   "lon": 37.6988559,\n   "opening_hours": null\n  },\n  {\n   "id": "way/336682866",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7022134,\n   "lon": 37.6948835,\n   "opening_hours": null\n  },\n  {\n   "id": "way/336743739",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8133603,\n   "lon": 37.503856,\n   "opening_hours": null\n  },\n  {\n   "id": "way/336745370",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8140932,\n   "lon": 37.5041052,\n   "opening_hours": null\n  },\n  {\n   "id": "way/337151437",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7556201,\n   "lon": 37.7418623,\n   "opening_hours": null\n  },\n  {\n   "id": "way/337690898",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7420298,\n   "lon": 37.5976832,\n   "opening_hours": null\n  },\n  {\n   "id": "way/337690903",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7416262,\n   "lon": 37.6002484,\n   "opening_hours": null\n  },\n  {\n   "id": "way/337690904",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7420719,\n   "lon": 37.5972003,\n   "opening_hours": null\n  },\n  {\n   "id": "way/338306412",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6977132,\n   "lon": 37.6252195,\n   "opening_hours": null\n  },\n  {\n   "id": "way/338419954",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7082019,\n   "lon": 37.615674,\n   "opening_hours": null\n  },\n  {\n   "id": "way/338985012",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7577874,\n   "lon": 37.5881259,\n   "opening_hours": null\n  },\n  {\n   "id": "way/339815594",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7136322,\n   "lon": 37.5922629,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340448664",\n   "kind": "building",\n   "name": "РегионФинансБанк",\n   "office_type": null,\n   "lat": 55.740566,\n   "lon": 37.6499701,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340451886",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7414366,\n   "lon": 37.6591819,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340453151",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8610774,\n   "lon": 37.4499766,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340460371",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7408376,\n   "lon": 37.662149,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340466861",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7435201,\n   "lon": 37.6509258,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340466898",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7456038,\n   "lon": 37.646842,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340466901",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449384,\n   "lon": 37.6482623,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340500620",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7452131,\n   "lon": 37.6473093,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340500621",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7433334,\n   "lon": 37.6499671,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340934853",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8054946,\n   "lon": 37.5924989,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340947236",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7970295,\n   "lon": 37.5381771,\n   "opening_hours": null\n  },\n  {\n   "id": "way/340976769",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7321989,\n   "lon": 37.6637635,\n   "opening_hours": null\n  },\n  {\n   "id": "way/342981170",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7964909,\n   "lon": 37.5869002,\n   "opening_hours": null\n  },\n  {\n   "id": "way/343478205",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7432656,\n   "lon": 37.626513,\n   "opening_hours": null\n  },\n  {\n   "id": "way/343478206",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.743295,\n   "lon": 37.6267201,\n   "opening_hours": null\n  },\n  {\n   "id": "way/343483360",\n   "kind": "organisation",\n   "name": "Институт геохимии и аналитической химии им. В. И. Вернадского",\n   "office_type": "research",\n   "lat": 55.7052612,\n   "lon": 37.550807,\n   "opening_hours": null\n  },\n  {\n   "id": "way/344304581",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7362647,\n   "lon": 37.6238447,\n   "opening_hours": null\n  },\n  {\n   "id": "way/345542273",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.750661,\n   "lon": 37.5844546,\n   "opening_hours": null\n  },\n  {\n   "id": "way/347869657",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8620631,\n   "lon": 37.5291854,\n   "opening_hours": null\n  },\n  {\n   "id": "way/349741206",\n   "kind": "building",\n   "name": "АО «Атомэнергопроект»",\n   "office_type": "architect",\n   "lat": 55.7743553,\n   "lon": 37.6813598,\n   "opening_hours": null\n  },\n  {\n   "id": "way/350093280",\n   "kind": "organisation",\n   "name": "Военторг-Ритейл",\n   "office_type": "company",\n   "lat": 55.8078842,\n   "lon": 37.5419729,\n   "opening_hours": null\n  },\n  {\n   "id": "way/350327311",\n   "kind": "organisation",\n   "name": "Московский центр качества образования",\n   "office_type": "government",\n   "lat": 55.7971146,\n   "lon": 37.5472983,\n   "opening_hours": "Mo-Th 09:00-16:00; Fr 09:00-12:00"\n  },\n  {\n   "id": "way/351630133",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7617034,\n   "lon": 37.5516212,\n   "opening_hours": null\n  },\n  {\n   "id": "way/352135197",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.83686,\n   "lon": 37.6356586,\n   "opening_hours": null\n  },\n  {\n   "id": "way/356067224",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7875096,\n   "lon": 37.6182281,\n   "opening_hours": null\n  },\n  {\n   "id": "way/358226403",\n   "kind": "building",\n   "name": "Башня B",\n   "office_type": null,\n   "lat": 55.7367421,\n   "lon": 37.5336019,\n   "opening_hours": null\n  },\n  {\n   "id": "way/360019372",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7314487,\n   "lon": 37.5806443,\n   "opening_hours": null\n  },\n  {\n   "id": "way/360029038",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7358839,\n   "lon": 37.5896628,\n   "opening_hours": null\n  },\n  {\n   "id": "way/360267360",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7833626,\n   "lon": 37.7299158,\n   "opening_hours": null\n  },\n  {\n   "id": "way/360694744",\n   "kind": "building",\n   "name": "Министерство здравоохранения Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7682145,\n   "lon": 37.6308245,\n   "opening_hours": null\n  },\n  {\n   "id": "way/361034529",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6970702,\n   "lon": 37.6083826,\n   "opening_hours": null\n  },\n  {\n   "id": "way/362310589",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7933127,\n   "lon": 37.6791707,\n   "opening_hours": null\n  },\n  {\n   "id": "way/362816111",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7963249,\n   "lon": 37.6820335,\n   "opening_hours": null\n  },\n  {\n   "id": "way/363128565",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8131887,\n   "lon": 37.6927965,\n   "opening_hours": null\n  },\n  {\n   "id": "way/363128567",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8132882,\n   "lon": 37.6942049,\n   "opening_hours": null\n  },\n  {\n   "id": "way/363128568",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8142014,\n   "lon": 37.6924196,\n   "opening_hours": null\n  },\n  {\n   "id": "way/363460987",\n   "kind": "building",\n   "name": "Администрация",\n   "office_type": null,\n   "lat": 55.7936624,\n   "lon": 37.6738422,\n   "opening_hours": null\n  },\n  {\n   "id": "way/364897860",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7048128,\n   "lon": 37.6525423,\n   "opening_hours": null\n  },\n  {\n   "id": "way/365987656",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7325917,\n   "lon": 37.5898126,\n   "opening_hours": null\n  },\n  {\n   "id": "way/366054149",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7041005,\n   "lon": 37.6146163,\n   "opening_hours": null\n  },\n  {\n   "id": "way/367416166",\n   "kind": "organisation",\n   "name": "ДЭЗ",\n   "office_type": "property_management",\n   "lat": 55.8330189,\n   "lon": 37.5561094,\n   "opening_hours": null\n  },\n  {\n   "id": "way/367420412",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7838153,\n   "lon": 37.7318175,\n   "opening_hours": null\n  },\n  {\n   "id": "way/367797361",\n   "kind": "organisation",\n   "name": "Радиал",\n   "office_type": "company",\n   "lat": 55.7527269,\n   "lon": 37.7505977,\n   "opening_hours": null\n  },\n  {\n   "id": "way/369219146",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.731746,\n   "lon": 37.6210049,\n   "opening_hours": null\n  },\n  {\n   "id": "way/369399563",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.5850256,\n   "lon": 37.4736116,\n   "opening_hours": null\n  },\n  {\n   "id": "way/370298019",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7064313,\n   "lon": 37.6476734,\n   "opening_hours": null\n  },\n  {\n   "id": "way/370298028",\n   "kind": "building",\n   "name": "Пандора",\n   "office_type": "company",\n   "lat": 55.7076074,\n   "lon": 37.6501921,\n   "opening_hours": null\n  },\n  {\n   "id": "way/370305997",\n   "kind": "building",\n   "name": "Бизнес-центр «Симоновский Плаза»",\n   "office_type": null,\n   "lat": 55.7102163,\n   "lon": 37.6521213,\n   "opening_hours": null\n  },\n  {\n   "id": "way/370306000",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7062952,\n   "lon": 37.6515984,\n   "opening_hours": null\n  },\n  {\n   "id": "way/370306028",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.706096,\n   "lon": 37.6527275,\n   "opening_hours": null\n  },\n  {\n   "id": "way/370306033",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7100718,\n   "lon": 37.6524676,\n   "opening_hours": null\n  },\n  {\n   "id": "way/370487953",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.693622,\n   "lon": 37.5420084,\n   "opening_hours": null\n  },\n  {\n   "id": "way/371265952",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7795822,\n   "lon": 37.6929933,\n   "opening_hours": null\n  },\n  {\n   "id": "way/372068255",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.782541,\n   "lon": 37.5992768,\n   "opening_hours": null\n  },\n  {\n   "id": "way/372470217",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8379799,\n   "lon": 37.6290152,\n   "opening_hours": null\n  },\n  {\n   "id": "way/372610347",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7143565,\n   "lon": 37.580725,\n   "opening_hours": null\n  },\n  {\n   "id": "way/372651469",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6838895,\n   "lon": 37.5881193,\n   "opening_hours": null\n  },\n  {\n   "id": "way/372651470",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6838721,\n   "lon": 37.5871179,\n   "opening_hours": null\n  },\n  {\n   "id": "way/372651471",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6836642,\n   "lon": 37.5877162,\n   "opening_hours": null\n  },\n  {\n   "id": "way/372756974",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7146166,\n   "lon": 37.6052119,\n   "opening_hours": null\n  },\n  {\n   "id": "way/372756975",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.714928,\n   "lon": 37.6048215,\n   "opening_hours": null\n  },\n  {\n   "id": "way/373649896",\n   "kind": "building",\n   "name": "Администрация 7АП",\n   "office_type": null,\n   "lat": 55.852809,\n   "lon": 37.5621404,\n   "opening_hours": null\n  },\n  {\n   "id": "way/374066823",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8244705,\n   "lon": 37.6130907,\n   "opening_hours": null\n  },\n  {\n   "id": "way/375867350",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7835131,\n   "lon": 37.6000384,\n   "opening_hours": null\n  },\n  {\n   "id": "way/377448683",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7153583,\n   "lon": 37.6460333,\n   "opening_hours": null\n  },\n  {\n   "id": "way/378433058",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7578269,\n   "lon": 37.5351258,\n   "opening_hours": null\n  },\n  {\n   "id": "way/379436139",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7364197,\n   "lon": 37.6429552,\n   "opening_hours": null\n  },\n  {\n   "id": "way/379469037",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.787597,\n   "lon": 37.7894078,\n   "opening_hours": null\n  },\n  {\n   "id": "way/381837959",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7135039,\n   "lon": 37.612398,\n   "opening_hours": null\n  },\n  {\n   "id": "way/382877242",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7875142,\n   "lon": 37.6357735,\n   "opening_hours": null\n  },\n  {\n   "id": "way/382877247",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7872786,\n   "lon": 37.6358125,\n   "opening_hours": null\n  },\n  {\n   "id": "way/384975606",\n   "kind": "building",\n   "name": "Прокуратура города Москвы",\n   "office_type": "government",\n   "lat": 55.732586,\n   "lon": 37.6691096,\n   "opening_hours": null\n  },\n  {\n   "id": "way/385851823",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7273323,\n   "lon": 37.6452965,\n   "opening_hours": null\n  },\n  {\n   "id": "way/385855369",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7254472,\n   "lon": 37.6376694,\n   "opening_hours": null\n  },\n  {\n   "id": "way/385986596",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7443371,\n   "lon": 37.6361735,\n   "opening_hours": null\n  },\n  {\n   "id": "way/385989725",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7466298,\n   "lon": 37.6349084,\n   "opening_hours": null\n  },\n  {\n   "id": "way/386297281",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7403377,\n   "lon": 37.6159386,\n   "opening_hours": null\n  },\n  {\n   "id": "way/386343964",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7535573,\n   "lon": 37.7082991,\n   "opening_hours": null\n  },\n  {\n   "id": "way/386343965",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7530277,\n   "lon": 37.7091147,\n   "opening_hours": null\n  },\n  {\n   "id": "way/386343971",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7532778,\n   "lon": 37.7086822,\n   "opening_hours": null\n  },\n  {\n   "id": "way/386351342",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7514314,\n   "lon": 37.6795635,\n   "opening_hours": null\n  },\n  {\n   "id": "way/386974431",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7895349,\n   "lon": 37.6338611,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387158356",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7853347,\n   "lon": 37.6298844,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387362473",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8695565,\n   "lon": 37.4580426,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387377651",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8378963,\n   "lon": 37.4771722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387617423",\n   "kind": "building",\n   "name": "Бизнес-центр «Прео 8»",\n   "office_type": null,\n   "lat": 55.7948161,\n   "lon": 37.7127812,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387641581",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7933378,\n   "lon": 37.7451687,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387678109",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6890985,\n   "lon": 37.5602624,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387694869",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7839334,\n   "lon": 37.6005097,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387694871",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7842499,\n   "lon": 37.6000235,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387694872",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7841981,\n   "lon": 37.6002508,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387694873",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7840506,\n   "lon": 37.5997628,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387694874",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7840822,\n   "lon": 37.6004697,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387694897",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7841142,\n   "lon": 37.5995674,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387749878",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7286831,\n   "lon": 37.6631514,\n   "opening_hours": null\n  },\n  {\n   "id": "way/387844289",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Favorit\\"",\n   "office_type": null,\n   "lat": 55.6738693,\n   "lon": 37.6195671,\n   "opening_hours": null\n  },\n  {\n   "id": "way/388428427",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7525045,\n   "lon": 37.6697787,\n   "opening_hours": null\n  },\n  {\n   "id": "way/388432361",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7529679,\n   "lon": 37.6645122,\n   "opening_hours": null\n  },\n  {\n   "id": "way/388432362",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.753181,\n   "lon": 37.6645826,\n   "opening_hours": null\n  },\n  {\n   "id": "way/388432363",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7552458,\n   "lon": 37.6599014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/389268468",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8095837,\n   "lon": 37.6908422,\n   "opening_hours": null\n  },\n  {\n   "id": "way/389448515",\n   "kind": "organisation",\n   "name": "Префектура Восточного административного округа",\n   "office_type": "government",\n   "lat": 55.79692,\n   "lon": 37.709138,\n   "opening_hours": null\n  },\n  {\n   "id": "way/390670593",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8128093,\n   "lon": 37.7361674,\n   "opening_hours": null\n  },\n  {\n   "id": "way/391363684",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8028119,\n   "lon": 37.8242474,\n   "opening_hours": null\n  },\n  {\n   "id": "way/399372359",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Чертаново Южное",\n   "office_type": "government",\n   "lat": 55.5928009,\n   "lon": 37.6093645,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "way/405488181",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8159297,\n   "lon": 37.4486505,\n   "opening_hours": null\n  },\n  {\n   "id": "way/406715364",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7065073,\n   "lon": 37.5970385,\n   "opening_hours": null\n  },\n  {\n   "id": "way/408786003",\n   "kind": "organisation",\n   "name": "Аналитическая лаборатория Департамента природопользования Москвы",\n   "office_type": "research",\n   "lat": 55.7148329,\n   "lon": 37.6050142,\n   "opening_hours": null\n  },\n  {\n   "id": "way/409865851",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7076105,\n   "lon": 37.6044039,\n   "opening_hours": null\n  },\n  {\n   "id": "way/410014562",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.704016,\n   "lon": 37.6114585,\n   "opening_hours": null\n  },\n  {\n   "id": "way/410553973",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7073761,\n   "lon": 37.5978738,\n   "opening_hours": null\n  },\n  {\n   "id": "way/411637834",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7383768,\n   "lon": 37.601892,\n   "opening_hours": null\n  },\n  {\n   "id": "way/411637839",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7390024,\n   "lon": 37.6011989,\n   "opening_hours": null\n  },\n  {\n   "id": "way/412016689",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7389767,\n   "lon": 37.5889372,\n   "opening_hours": null\n  },\n  {\n   "id": "way/412472996",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7401128,\n   "lon": 37.6279366,\n   "opening_hours": null\n  },\n  {\n   "id": "way/412738928",\n   "kind": "building",\n   "name": "Усадьба Давыдова",\n   "office_type": null,\n   "lat": 55.7416664,\n   "lon": 37.594528,\n   "opening_hours": null\n  },\n  {\n   "id": "way/413372212",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7371586,\n   "lon": 37.6350247,\n   "opening_hours": null\n  },\n  {\n   "id": "way/415477350",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.750905,\n   "lon": 37.6057583,\n   "opening_hours": null\n  },\n  {\n   "id": "way/415477363",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7519365,\n   "lon": 37.6075909,\n   "opening_hours": null\n  },\n  {\n   "id": "way/417393024",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7536693,\n   "lon": 37.6073417,\n   "opening_hours": null\n  },\n  {\n   "id": "way/420332156",\n   "kind": "organisation",\n   "name": "Шипиловский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.6169156,\n   "lon": 37.6951938,\n   "opening_hours": "Tu-Sa 09:00-14:00,15:00-18:00; Th[4] off \\"Санитарный день\\""\n  },\n  {\n   "id": "way/425516157",\n   "kind": "building",\n   "name": "Диспетчерская к/ст \\"138 квартал Выхино\\"",\n   "office_type": null,\n   "lat": 55.6930819,\n   "lon": 37.8169764,\n   "opening_hours": null\n  },\n  {\n   "id": "way/428108782",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7195171,\n   "lon": 37.6014915,\n   "opening_hours": null\n  },\n  {\n   "id": "way/428930473",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7129281,\n   "lon": 37.6832785,\n   "opening_hours": null\n  },\n  {\n   "id": "way/429124421",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7648638,\n   "lon": 37.5569465,\n   "opening_hours": null\n  },\n  {\n   "id": "way/429154268",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7652113,\n   "lon": 37.558725,\n   "opening_hours": null\n  },\n  {\n   "id": "way/430177573",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7522774,\n   "lon": 37.6650705,\n   "opening_hours": null\n  },\n  {\n   "id": "way/430378875",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.730998,\n   "lon": 37.5876954,\n   "opening_hours": null\n  },\n  {\n   "id": "way/430378880",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7312086,\n   "lon": 37.5874721,\n   "opening_hours": null\n  },\n  {\n   "id": "way/434330553",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.768696,\n   "lon": 37.6092988,\n   "opening_hours": null\n  },\n  {\n   "id": "way/435604935",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7258694,\n   "lon": 37.6105248,\n   "opening_hours": null\n  },\n  {\n   "id": "way/436565647",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7696445,\n   "lon": 37.61716,\n   "opening_hours": null\n  },\n  {\n   "id": "way/437649036",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7551267,\n   "lon": 37.5579761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/438429438",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7606617,\n   "lon": 37.6476487,\n   "opening_hours": null\n  },\n  {\n   "id": "way/438997308",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7734672,\n   "lon": 37.6392158,\n   "opening_hours": null\n  },\n  {\n   "id": "way/443319665",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8088792,\n   "lon": 37.8083188,\n   "opening_hours": null\n  },\n  {\n   "id": "way/443767519",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7558274,\n   "lon": 37.7429442,\n   "opening_hours": null\n  },\n  {\n   "id": "way/443767522",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7560249,\n   "lon": 37.7436959,\n   "opening_hours": null\n  },\n  {\n   "id": "way/444437262",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7546466,\n   "lon": 37.730676,\n   "opening_hours": null\n  },\n  {\n   "id": "way/445438693",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7536437,\n   "lon": 37.605005,\n   "opening_hours": null\n  },\n  {\n   "id": "way/446071847",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7779197,\n   "lon": 37.6745454,\n   "opening_hours": null\n  },\n  {\n   "id": "way/448842514",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7324106,\n   "lon": 37.5515466,\n   "opening_hours": null\n  },\n  {\n   "id": "way/448842515",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7327485,\n   "lon": 37.5521596,\n   "opening_hours": null\n  },\n  {\n   "id": "way/448842517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7321488,\n   "lon": 37.5509106,\n   "opening_hours": null\n  },\n  {\n   "id": "way/448961881",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7777796,\n   "lon": 37.6743404,\n   "opening_hours": null\n  },\n  {\n   "id": "way/449099050",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7531742,\n   "lon": 37.6490635,\n   "opening_hours": null\n  },\n  {\n   "id": "way/449533854",\n   "kind": "building",\n   "name": "Город Столиц - Москва",\n   "office_type": null,\n   "lat": 55.7475642,\n   "lon": 37.5387956,\n   "opening_hours": null\n  },\n  {\n   "id": "way/449695901",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819088,\n   "lon": 37.6933496,\n   "opening_hours": null\n  },\n  {\n   "id": "way/452293894",\n   "kind": "organisation",\n   "name": "Федеральное медико-биологическое агентство России",\n   "office_type": "government",\n   "lat": 55.8129345,\n   "lon": 37.4796226,\n   "opening_hours": null\n  },\n  {\n   "id": "way/452418764",\n   "kind": "organisation",\n   "name": "Государственный научно-исследовательский институт гражданской авиации",\n   "office_type": "research",\n   "lat": 55.812206,\n   "lon": 37.4827929,\n   "opening_hours": "Mo-Fr 08:30-17:30"\n  },\n  {\n   "id": "way/454302970",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7815564,\n   "lon": 37.6990805,\n   "opening_hours": null\n  },\n  {\n   "id": "way/454302985",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.781532,\n   "lon": 37.6988231,\n   "opening_hours": null\n  },\n  {\n   "id": "way/455957137",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7703282,\n   "lon": 37.6092373,\n   "opening_hours": null\n  },\n  {\n   "id": "way/456353059",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7513634,\n   "lon": 37.6644802,\n   "opening_hours": null\n  },\n  {\n   "id": "way/459216794",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7346766,\n   "lon": 37.7531168,\n   "opening_hours": null\n  },\n  {\n   "id": "way/460985904",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7864288,\n   "lon": 37.4716459,\n   "opening_hours": null\n  },\n  {\n   "id": "way/464462058",\n   "kind": "organisation",\n   "name": "Российский государственный архив экономики",\n   "office_type": "government",\n   "lat": 55.7322137,\n   "lon": 37.5734056,\n   "opening_hours": null\n  },\n  {\n   "id": "way/465220852",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7622123,\n   "lon": 37.6581075,\n   "opening_hours": null\n  },\n  {\n   "id": "way/467023847",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.756973,\n   "lon": 37.5074476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/467391743",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7647609,\n   "lon": 37.5703285,\n   "opening_hours": null\n  },\n  {\n   "id": "way/468011389",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7820117,\n   "lon": 37.5967859,\n   "opening_hours": null\n  },\n  {\n   "id": "way/468038375",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.762152,\n   "lon": 37.5813109,\n   "opening_hours": null\n  },\n  {\n   "id": "way/470483406",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8010824,\n   "lon": 37.6368119,\n   "opening_hours": null\n  },\n  {\n   "id": "way/470483407",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8002966,\n   "lon": 37.6395244,\n   "opening_hours": null\n  },\n  {\n   "id": "way/470483408",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8001081,\n   "lon": 37.6380689,\n   "opening_hours": null\n  },\n  {\n   "id": "way/470483409",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8002885,\n   "lon": 37.637907,\n   "opening_hours": null\n  },\n  {\n   "id": "way/470483411",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8004509,\n   "lon": 37.6376755,\n   "opening_hours": null\n  },\n  {\n   "id": "way/472626653",\n   "kind": "organisation",\n   "name": "Управление социальной защиты населения и центр социального обслуживания района Зябликово",\n   "office_type": "government",\n   "lat": 55.619224,\n   "lon": 37.7487755,\n   "opening_hours": null\n  },\n  {\n   "id": "way/474041406",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.6859875,\n   "lon": 37.7568741,\n   "opening_hours": null\n  },\n  {\n   "id": "way/477923147",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7438932,\n   "lon": 37.6289685,\n   "opening_hours": null\n  },\n  {\n   "id": "way/478307033",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7842199,\n   "lon": 37.7082028,\n   "opening_hours": null\n  },\n  {\n   "id": "way/481075439",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7612666,\n   "lon": 37.67989,\n   "opening_hours": null\n  },\n  {\n   "id": "way/482250380",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7430442,\n   "lon": 37.6288927,\n   "opening_hours": null\n  },\n  {\n   "id": "way/485473599",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6826426,\n   "lon": 37.6333225,\n   "opening_hours": null\n  },\n  {\n   "id": "way/489066735",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7786418,\n   "lon": 37.6743912,\n   "opening_hours": null\n  },\n  {\n   "id": "way/490319602",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.782725,\n   "lon": 37.7124757,\n   "opening_hours": null\n  },\n  {\n   "id": "way/495236507",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7419287,\n   "lon": 37.5243621,\n   "opening_hours": null\n  },\n  {\n   "id": "way/497946374",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7873932,\n   "lon": 37.7900968,\n   "opening_hours": null\n  },\n  {\n   "id": "way/497946375",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7874773,\n   "lon": 37.7889777,\n   "opening_hours": null\n  },\n  {\n   "id": "way/498545636",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.812761,\n   "lon": 37.598465,\n   "opening_hours": null\n  },\n  {\n   "id": "way/499939694",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7830127,\n   "lon": 37.7136007,\n   "opening_hours": null\n  },\n  {\n   "id": "way/500097771",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7476152,\n   "lon": 37.5983817,\n   "opening_hours": null\n  },\n  {\n   "id": "way/500110353",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7412525,\n   "lon": 37.5963366,\n   "opening_hours": null\n  },\n  {\n   "id": "way/500114231",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7404856,\n   "lon": 37.6002146,\n   "opening_hours": null\n  },\n  {\n   "id": "way/500498152",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7817133,\n   "lon": 37.6972478,\n   "opening_hours": null\n  },\n  {\n   "id": "way/500498160",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819145,\n   "lon": 37.6942454,\n   "opening_hours": null\n  },\n  {\n   "id": "way/501253280",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6886609,\n   "lon": 37.5735021,\n   "opening_hours": null\n  },\n  {\n   "id": "way/501633357",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8373075,\n   "lon": 37.6322812,\n   "opening_hours": null\n  },\n  {\n   "id": "way/501633358",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8370376,\n   "lon": 37.6325134,\n   "opening_hours": null\n  },\n  {\n   "id": "way/501679712",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7761689,\n   "lon": 37.6738707,\n   "opening_hours": null\n  },\n  {\n   "id": "way/501679720",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7761043,\n   "lon": 37.6747722,\n   "opening_hours": null\n  },\n  {\n   "id": "way/501762388",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7658386,\n   "lon": 37.6638611,\n   "opening_hours": null\n  },\n  {\n   "id": "way/502378037",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6543181,\n   "lon": 37.4961769,\n   "opening_hours": null\n  },\n  {\n   "id": "way/503235172",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8093219,\n   "lon": 37.6877693,\n   "opening_hours": null\n  },\n  {\n   "id": "way/504240806",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.698684,\n   "lon": 37.6116954,\n   "opening_hours": null\n  },\n  {\n   "id": "way/505771351",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.7210744,\n   "lon": 37.725898,\n   "opening_hours": null\n  },\n  {\n   "id": "way/507418183",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7058332,\n   "lon": 37.6031774,\n   "opening_hours": null\n  },\n  {\n   "id": "way/507497576",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7106376,\n   "lon": 37.5770738,\n   "opening_hours": null\n  },\n  {\n   "id": "way/507497577",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7109014,\n   "lon": 37.5776981,\n   "opening_hours": null\n  },\n  {\n   "id": "way/508234833",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7630508,\n   "lon": 37.6410024,\n   "opening_hours": null\n  },\n  {\n   "id": "way/508353050",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7557558,\n   "lon": 37.5803561,\n   "opening_hours": null\n  },\n  {\n   "id": "way/509948193",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7834456,\n   "lon": 37.677346,\n   "opening_hours": null\n  },\n  {\n   "id": "way/510031365",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7386866,\n   "lon": 37.6310491,\n   "opening_hours": null\n  },\n  {\n   "id": "way/512103113",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7393629,\n   "lon": 37.6291114,\n   "opening_hours": null\n  },\n  {\n   "id": "way/513730259",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7368168,\n   "lon": 37.62866,\n   "opening_hours": null\n  },\n  {\n   "id": "way/514859777",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7364494,\n   "lon": 37.6155486,\n   "opening_hours": null\n  },\n  {\n   "id": "way/515672089",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8131848,\n   "lon": 37.7365178,\n   "opening_hours": null\n  },\n  {\n   "id": "way/518538553",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6922279,\n   "lon": 37.6238475,\n   "opening_hours": null\n  },\n  {\n   "id": "way/520009375",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6785196,\n   "lon": 37.5085453,\n   "opening_hours": null\n  },\n  {\n   "id": "way/520635412",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7800624,\n   "lon": 37.6688443,\n   "opening_hours": null\n  },\n  {\n   "id": "way/521295436",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7862229,\n   "lon": 37.7234772,\n   "opening_hours": null\n  },\n  {\n   "id": "way/522136389",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8432325,\n   "lon": 37.5459688,\n   "opening_hours": null\n  },\n  {\n   "id": "way/522579457",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.769393,\n   "lon": 37.6082706,\n   "opening_hours": null\n  },\n  {\n   "id": "way/524364893",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8142657,\n   "lon": 37.780287,\n   "opening_hours": null\n  },\n  {\n   "id": "way/524697555",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8630134,\n   "lon": 37.5405273,\n   "opening_hours": null\n  },\n  {\n   "id": "way/524701452",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7362055,\n   "lon": 37.7102762,\n   "opening_hours": null\n  },\n  {\n   "id": "way/528438079",\n   "kind": "building",\n   "name": "Бюро пропусков",\n   "office_type": null,\n   "lat": 55.7269257,\n   "lon": 37.5875534,\n   "opening_hours": null\n  },\n  {\n   "id": "way/531331536",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8288869,\n   "lon": 37.6516213,\n   "opening_hours": null\n  },\n  {\n   "id": "way/534370311",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7342639,\n   "lon": 37.6301935,\n   "opening_hours": null\n  },\n  {\n   "id": "way/534370312",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7343541,\n   "lon": 37.6301225,\n   "opening_hours": null\n  },\n  {\n   "id": "way/535203485",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7117519,\n   "lon": 37.5131714,\n   "opening_hours": null\n  },\n  {\n   "id": "way/535203492",\n   "kind": "organisation",\n   "name": "посольство КНР",\n   "office_type": "diplomatic",\n   "lat": 55.7108127,\n   "lon": 37.5164699,\n   "opening_hours": null\n  },\n  {\n   "id": "way/535324267",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7896774,\n   "lon": 37.6356271,\n   "opening_hours": null\n  },\n  {\n   "id": "way/536342520",\n   "kind": "organisation",\n   "name": "Aitarget",\n   "office_type": "it",\n   "lat": 55.7325417,\n   "lon": 37.6460907,\n   "opening_hours": null\n  },\n  {\n   "id": "way/536771194",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7556059,\n   "lon": 37.6341644,\n   "opening_hours": null\n  },\n  {\n   "id": "way/536771197",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7555134,\n   "lon": 37.6339196,\n   "opening_hours": null\n  },\n  {\n   "id": "way/536888849",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7309669,\n   "lon": 37.6199499,\n   "opening_hours": null\n  },\n  {\n   "id": "way/536888852",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7333684,\n   "lon": 37.6199296,\n   "opening_hours": null\n  },\n  {\n   "id": "way/539238923",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.784549,\n   "lon": 37.6359968,\n   "opening_hours": null\n  },\n  {\n   "id": "way/539468773",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7692194,\n   "lon": 37.6298293,\n   "opening_hours": null\n  },\n  {\n   "id": "way/539698564",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7454348,\n   "lon": 37.59887,\n   "opening_hours": null\n  },\n  {\n   "id": "way/539821071",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.832653,\n   "lon": 37.5356184,\n   "opening_hours": null\n  },\n  {\n   "id": "way/539821073",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8314976,\n   "lon": 37.5327367,\n   "opening_hours": null\n  },\n  {\n   "id": "way/540515211",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7036973,\n   "lon": 37.6158967,\n   "opening_hours": null\n  },\n  {\n   "id": "way/540515212",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7038797,\n   "lon": 37.6158957,\n   "opening_hours": null\n  },\n  {\n   "id": "way/541958498",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7667558,\n   "lon": 37.6781802,\n   "opening_hours": null\n  },\n  {\n   "id": "way/543041795",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8820805,\n   "lon": 37.4546706,\n   "opening_hours": null\n  },\n  {\n   "id": "way/543778176",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8227136,\n   "lon": 37.4435396,\n   "opening_hours": null\n  },\n  {\n   "id": "way/544140924",\n   "kind": "building",\n   "name": "Пожарная часть №76, Управление по ЮВАО 24 ОФПС",\n   "office_type": null,\n   "lat": 55.6578599,\n   "lon": 37.7390154,\n   "opening_hours": null\n  },\n  {\n   "id": "way/544967043",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7629687,\n   "lon": 37.7818824,\n   "opening_hours": null\n  },\n  {\n   "id": "way/546597731",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8246609,\n   "lon": 37.5890134,\n   "opening_hours": null\n  },\n  {\n   "id": "way/546801520",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7301892,\n   "lon": 37.7396179,\n   "opening_hours": null\n  },\n  {\n   "id": "way/547671952",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.869019,\n   "lon": 37.6894439,\n   "opening_hours": null\n  },\n  {\n   "id": "way/548947686",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7776729,\n   "lon": 37.7195523,\n   "opening_hours": null\n  },\n  {\n   "id": "way/548947690",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7759866,\n   "lon": 37.7209491,\n   "opening_hours": null\n  },\n  {\n   "id": "way/550451598",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.838018,\n   "lon": 37.6282952,\n   "opening_hours": null\n  },\n  {\n   "id": "way/550451614",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8412856,\n   "lon": 37.6254333,\n   "opening_hours": null\n  },\n  {\n   "id": "way/550456798",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8083871,\n   "lon": 37.5622177,\n   "opening_hours": null\n  },\n  {\n   "id": "way/554996849",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7464209,\n   "lon": 37.661643,\n   "opening_hours": null\n  },\n  {\n   "id": "way/554996854",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7467418,\n   "lon": 37.6621572,\n   "opening_hours": null\n  },\n  {\n   "id": "way/554996855",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7467411,\n   "lon": 37.6618962,\n   "opening_hours": null\n  },\n  {\n   "id": "way/555604810",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588703,\n   "lon": 37.7361119,\n   "opening_hours": null\n  },\n  {\n   "id": "way/555604815",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7591517,\n   "lon": 37.7338288,\n   "opening_hours": null\n  },\n  {\n   "id": "way/555604816",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7588043,\n   "lon": 37.7341292,\n   "opening_hours": null\n  },\n  {\n   "id": "way/555604818",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7587707,\n   "lon": 37.7335463,\n   "opening_hours": null\n  },\n  {\n   "id": "way/555999537",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Алесеевский",\n   "office_type": "government",\n   "lat": 55.8225463,\n   "lon": 37.663858,\n   "opening_hours": null\n  },\n  {\n   "id": "way/556871645",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461542,\n   "lon": 37.6676564,\n   "opening_hours": null\n  },\n  {\n   "id": "way/557427228",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7800034,\n   "lon": 37.6943626,\n   "opening_hours": null\n  },\n  {\n   "id": "way/560305484",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7405414,\n   "lon": 37.6409419,\n   "opening_hours": null\n  },\n  {\n   "id": "way/565161237",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7566879,\n   "lon": 37.6517769,\n   "opening_hours": null\n  },\n  {\n   "id": "way/565161294",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7551558,\n   "lon": 37.6493894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/566377571",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.5930566,\n   "lon": 37.7438515,\n   "opening_hours": null\n  },\n  {\n   "id": "way/581636269",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7922838,\n   "lon": 37.6964686,\n   "opening_hours": null\n  },\n  {\n   "id": "way/583547836",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7268493,\n   "lon": 37.688741,\n   "opening_hours": null\n  },\n  {\n   "id": "way/584904915",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7159546,\n   "lon": 37.5660807,\n   "opening_hours": null\n  },\n  {\n   "id": "way/584904916",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7156922,\n   "lon": 37.5657458,\n   "opening_hours": null\n  },\n  {\n   "id": "way/585909639",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7065761,\n   "lon": 37.7438938,\n   "opening_hours": null\n  },\n  {\n   "id": "way/586387092",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7784668,\n   "lon": 37.5554065,\n   "opening_hours": null\n  },\n  {\n   "id": "way/588460406",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8251944,\n   "lon": 37.5172149,\n   "opening_hours": null\n  },\n  {\n   "id": "way/591006670",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8160903,\n   "lon": 37.5955192,\n   "opening_hours": null\n  },\n  {\n   "id": "way/593903274",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7734788,\n   "lon": 37.6815305,\n   "opening_hours": null\n  },\n  {\n   "id": "way/594672996",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7597564,\n   "lon": 37.5864754,\n   "opening_hours": null\n  },\n  {\n   "id": "way/594672998",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7598188,\n   "lon": 37.5860019,\n   "opening_hours": null\n  },\n  {\n   "id": "way/595399508",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7612719,\n   "lon": 37.7222993,\n   "opening_hours": null\n  },\n  {\n   "id": "way/605960104",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7853163,\n   "lon": 37.6940041,\n   "opening_hours": null\n  },\n  {\n   "id": "way/607164213",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7443711,\n   "lon": 37.6330069,\n   "opening_hours": null\n  },\n  {\n   "id": "way/607499726",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7848795,\n   "lon": 37.6926864,\n   "opening_hours": null\n  },\n  {\n   "id": "way/607499727",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.78509,\n   "lon": 37.6925629,\n   "opening_hours": null\n  },\n  {\n   "id": "way/609945076",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7760438,\n   "lon": 37.6242829,\n   "opening_hours": null\n  },\n  {\n   "id": "way/611160381",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8582965,\n   "lon": 37.6550672,\n   "opening_hours": null\n  },\n  {\n   "id": "way/616057793",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7264718,\n   "lon": 37.6653835,\n   "opening_hours": null\n  },\n  {\n   "id": "way/616259751",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6919252,\n   "lon": 37.624123,\n   "opening_hours": null\n  },\n  {\n   "id": "way/616867750",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7868429,\n   "lon": 37.6722189,\n   "opening_hours": null\n  },\n  {\n   "id": "way/621791680",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8224943,\n   "lon": 37.8138912,\n   "opening_hours": null\n  },\n  {\n   "id": "way/625292030",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7806391,\n   "lon": 37.5897931,\n   "opening_hours": null\n  },\n  {\n   "id": "way/627844613",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7979829,\n   "lon": 37.6317691,\n   "opening_hours": null\n  },\n  {\n   "id": "way/631813786",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.758939,\n   "lon": 37.6599452,\n   "opening_hours": null\n  },\n  {\n   "id": "way/632190265",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.758814,\n   "lon": 37.5884159,\n   "opening_hours": null\n  },\n  {\n   "id": "way/632190266",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7587645,\n   "lon": 37.5874309,\n   "opening_hours": null\n  },\n  {\n   "id": "way/632303558",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6072786,\n   "lon": 37.6557397,\n   "opening_hours": null\n  },\n  {\n   "id": "way/633039022",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7600929,\n   "lon": 37.5874029,\n   "opening_hours": null\n  },\n  {\n   "id": "way/636971341",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.762875,\n   "lon": 37.6272572,\n   "opening_hours": null\n  },\n  {\n   "id": "way/640129100",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.737094,\n   "lon": 37.6695835,\n   "opening_hours": null\n  },\n  {\n   "id": "way/641330825",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6788226,\n   "lon": 37.5091312,\n   "opening_hours": null\n  },\n  {\n   "id": "way/641330827",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6789738,\n   "lon": 37.5093511,\n   "opening_hours": null\n  },\n  {\n   "id": "way/641937406",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7451383,\n   "lon": 37.6462843,\n   "opening_hours": null\n  },\n  {\n   "id": "way/641937408",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447592,\n   "lon": 37.6470104,\n   "opening_hours": null\n  },\n  {\n   "id": "way/645359441",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7287315,\n   "lon": 37.6584179,\n   "opening_hours": null\n  },\n  {\n   "id": "way/645359444",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7284266,\n   "lon": 37.6595504,\n   "opening_hours": null\n  },\n  {\n   "id": "way/650087428",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7251927,\n   "lon": 37.7988665,\n   "opening_hours": null\n  },\n  {\n   "id": "way/650087435",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7244474,\n   "lon": 37.8011147,\n   "opening_hours": null\n  },\n  {\n   "id": "way/656592359",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7067972,\n   "lon": 37.6062793,\n   "opening_hours": null\n  },\n  {\n   "id": "way/668438418",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6831187,\n   "lon": 37.449408,\n   "opening_hours": null\n  },\n  {\n   "id": "way/671267517",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7115958,\n   "lon": 37.5733632,\n   "opening_hours": null\n  },\n  {\n   "id": "way/671267518",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7114764,\n   "lon": 37.5729353,\n   "opening_hours": null\n  },\n  {\n   "id": "way/671606393",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.716392,\n   "lon": 37.5953574,\n   "opening_hours": null\n  },\n  {\n   "id": "way/673085737",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7463114,\n   "lon": 37.631248,\n   "opening_hours": null\n  },\n  {\n   "id": "way/679418353",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8121781,\n   "lon": 37.6955754,\n   "opening_hours": null\n  },\n  {\n   "id": "way/681140440",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.762777,\n   "lon": 37.6277011,\n   "opening_hours": null\n  },\n  {\n   "id": "way/682991438",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7546048,\n   "lon": 37.7357867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/683115949",\n   "kind": "organisation",\n   "name": "Офис продаж «Селигер Сити»",\n   "office_type": "estate_agent",\n   "lat": 55.8623045,\n   "lon": 37.5414303,\n   "opening_hours": null\n  },\n  {\n   "id": "way/684009371",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7978452,\n   "lon": 37.6694014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/684545249",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7778583,\n   "lon": 37.6467267,\n   "opening_hours": null\n  },\n  {\n   "id": "way/685238622",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7048338,\n   "lon": 37.6450222,\n   "opening_hours": null\n  },\n  {\n   "id": "way/686018586",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7967067,\n   "lon": 37.5331286,\n   "opening_hours": null\n  },\n  {\n   "id": "way/686831803",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7290475,\n   "lon": 37.6265908,\n   "opening_hours": null\n  },\n  {\n   "id": "way/687395507",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7615058,\n   "lon": 37.600233,\n   "opening_hours": null\n  },\n  {\n   "id": "way/687395508",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7614037,\n   "lon": 37.600577,\n   "opening_hours": null\n  },\n  {\n   "id": "way/687788744",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7208847,\n   "lon": 37.4633038,\n   "opening_hours": null\n  },\n  {\n   "id": "way/690970538",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6719092,\n   "lon": 37.6132607,\n   "opening_hours": null\n  },\n  {\n   "id": "way/692561958",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7898648,\n   "lon": 37.7795052,\n   "opening_hours": null\n  },\n  {\n   "id": "way/692561959",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.790302,\n   "lon": 37.7790031,\n   "opening_hours": null\n  },\n  {\n   "id": "way/694994848",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7223245,\n   "lon": 37.5710556,\n   "opening_hours": null\n  },\n  {\n   "id": "way/694994849",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7222373,\n   "lon": 37.5697156,\n   "opening_hours": null\n  },\n  {\n   "id": "way/695807786",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7272228,\n   "lon": 37.5893337,\n   "opening_hours": null\n  },\n  {\n   "id": "way/699660908",\n   "kind": "organisation",\n   "name": "Восход-Авто",\n   "office_type": "company",\n   "lat": 55.6053574,\n   "lon": 37.5041432,\n   "opening_hours": null\n  },\n  {\n   "id": "way/699667957",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.6055862,\n   "lon": 37.5048987,\n   "opening_hours": null\n  },\n  {\n   "id": "way/706495344",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7807812,\n   "lon": 37.67326,\n   "opening_hours": null\n  },\n  {\n   "id": "way/706495356",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7829183,\n   "lon": 37.6765611,\n   "opening_hours": null\n  },\n  {\n   "id": "way/708012674",\n   "kind": "building",\n   "name": "Следственный комитет Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7634768,\n   "lon": 37.6810903,\n   "opening_hours": null\n  },\n  {\n   "id": "way/708546378",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7582785,\n   "lon": 37.6640882,\n   "opening_hours": null\n  },\n  {\n   "id": "way/711872855",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7535544,\n   "lon": 37.6027873,\n   "opening_hours": null\n  },\n  {\n   "id": "way/713708416",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.759349,\n   "lon": 37.6013016,\n   "opening_hours": null\n  },\n  {\n   "id": "way/713796365",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8138058,\n   "lon": 37.6952761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/713870976",\n   "kind": "building",\n   "name": "Бизнес-центр \\"10\\"",\n   "office_type": null,\n   "lat": 55.7322287,\n   "lon": 37.73186,\n   "opening_hours": null\n  },\n  {\n   "id": "way/714753796",\n   "kind": "organisation",\n   "name": "Консульство США",\n   "office_type": "diplomatic",\n   "lat": 55.7554482,\n   "lon": 37.5799746,\n   "opening_hours": null\n  },\n  {\n   "id": "way/716827297",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6600875,\n   "lon": 37.7213481,\n   "opening_hours": null\n  },\n  {\n   "id": "way/722221700",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7155537,\n   "lon": 37.5586985,\n   "opening_hours": null\n  },\n  {\n   "id": "way/722847617",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6800584,\n   "lon": 37.6924423,\n   "opening_hours": null\n  },\n  {\n   "id": "way/729353269",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7251542,\n   "lon": 37.7423903,\n   "opening_hours": null\n  },\n  {\n   "id": "way/736781663",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7019866,\n   "lon": 37.491528,\n   "opening_hours": null\n  },\n  {\n   "id": "way/737401378",\n   "kind": "organisation",\n   "name": "Рефсервис",\n   "office_type": "company",\n   "lat": 55.7828866,\n   "lon": 37.6619319,\n   "opening_hours": null\n  },\n  {\n   "id": "way/739091056",\n   "kind": "building",\n   "name": "Минстрой России",\n   "office_type": "government",\n   "lat": 55.7729786,\n   "lon": 37.6153825,\n   "opening_hours": null\n  },\n  {\n   "id": "way/743897851",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.5924561,\n   "lon": 37.6576166,\n   "opening_hours": null\n  },\n  {\n   "id": "way/749169895",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8072974,\n   "lon": 37.622694,\n   "opening_hours": null\n  },\n  {\n   "id": "way/755444730",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7837306,\n   "lon": 37.6567867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/759854483",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7658751,\n   "lon": 37.6466723,\n   "opening_hours": null\n  },\n  {\n   "id": "way/760014288",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7059381,\n   "lon": 37.6099646,\n   "opening_hours": null\n  },\n  {\n   "id": "way/760072789",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7839965,\n   "lon": 37.6733506,\n   "opening_hours": null\n  },\n  {\n   "id": "way/760941895",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.8013495,\n   "lon": 37.5585864,\n   "opening_hours": null\n  },\n  {\n   "id": "way/761845623",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8086712,\n   "lon": 37.629311,\n   "opening_hours": null\n  },\n  {\n   "id": "way/765536355",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7654285,\n   "lon": 37.6488413,\n   "opening_hours": null\n  },\n  {\n   "id": "way/765950793",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.870246,\n   "lon": 37.4870213,\n   "opening_hours": null\n  },\n  {\n   "id": "way/766569237",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8118154,\n   "lon": 37.4556703,\n   "opening_hours": null\n  },\n  {\n   "id": "way/767376225",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8363761,\n   "lon": 37.4994656,\n   "opening_hours": null\n  },\n  {\n   "id": "way/768103785",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7130548,\n   "lon": 37.5915903,\n   "opening_hours": null\n  },\n  {\n   "id": "way/768255064",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7399266,\n   "lon": 37.6018598,\n   "opening_hours": null\n  },\n  {\n   "id": "way/768255593",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.721533,\n   "lon": 37.5690115,\n   "opening_hours": null\n  },\n  {\n   "id": "way/770533814",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7816024,\n   "lon": 37.6355516,\n   "opening_hours": null\n  },\n  {\n   "id": "way/770533832",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7785101,\n   "lon": 37.6398654,\n   "opening_hours": null\n  },\n  {\n   "id": "way/773082530",\n   "kind": "organisation",\n   "name": "Lom24",\n   "office_type": "company",\n   "lat": 55.8596272,\n   "lon": 37.5792934,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "way/773724059",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.804006,\n   "lon": 37.663891,\n   "opening_hours": null\n  },\n  {\n   "id": "way/774536606",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7973012,\n   "lon": 37.6724071,\n   "opening_hours": null\n  },\n  {\n   "id": "way/774536607",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7958296,\n   "lon": 37.6620593,\n   "opening_hours": null\n  },\n  {\n   "id": "way/774536608",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7957065,\n   "lon": 37.6620143,\n   "opening_hours": null\n  },\n  {\n   "id": "way/779280103",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7127149,\n   "lon": 37.5912455,\n   "opening_hours": null\n  },\n  {\n   "id": "way/779280162",\n   "kind": "building",\n   "name": "Институт проблем экологии и эволюции им. А. Н. Северцова РАН",\n   "office_type": "research",\n   "lat": 55.711889,\n   "lon": 37.5889203,\n   "opening_hours": "Mo-Fr 09:30-17:30"\n  },\n  {\n   "id": "way/779616333",\n   "kind": "building",\n   "name": "Институт нефтехимического синтеза им. А. В. Топчиева РАН",\n   "office_type": "research",\n   "lat": 55.7141055,\n   "lon": 37.590842,\n   "opening_hours": null\n  },\n  {\n   "id": "way/779616342",\n   "kind": "organisation",\n   "name": "Институт общей и неорганической химии им. Н. С. Курнакова РАН",\n   "office_type": "research",\n   "lat": 55.7132926,\n   "lon": 37.5900726,\n   "opening_hours": null\n  },\n  {\n   "id": "way/779616344",\n   "kind": "organisation",\n   "name": "Институт физической химии и электрохимии имени А. Н. Фрумкина РАН",\n   "office_type": "research",\n   "lat": 55.7126805,\n   "lon": 37.5899096,\n   "opening_hours": null\n  },\n  {\n   "id": "way/780343507",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7155353,\n   "lon": 37.5944105,\n   "opening_hours": null\n  },\n  {\n   "id": "way/782265983",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7185079,\n   "lon": 37.5949917,\n   "opening_hours": null\n  },\n  {\n   "id": "way/782266013",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.71849,\n   "lon": 37.6021412,\n   "opening_hours": null\n  },\n  {\n   "id": "way/783028021",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7070764,\n   "lon": 37.6058262,\n   "opening_hours": null\n  },\n  {\n   "id": "way/783201463",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7071755,\n   "lon": 37.6096887,\n   "opening_hours": null\n  },\n  {\n   "id": "way/783454908",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7871772,\n   "lon": 37.7178538,\n   "opening_hours": null\n  },\n  {\n   "id": "way/784759334",\n   "kind": "building",\n   "name": "ОКО (Кристалл)",\n   "office_type": null,\n   "lat": 55.7497673,\n   "lon": 37.5338752,\n   "opening_hours": null\n  },\n  {\n   "id": "way/789024920",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7207021,\n   "lon": 37.6124741,\n   "opening_hours": null\n  },\n  {\n   "id": "way/789024922",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7205051,\n   "lon": 37.6107839,\n   "opening_hours": null\n  },\n  {\n   "id": "way/789713814",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7167276,\n   "lon": 37.5695907,\n   "opening_hours": null\n  },\n  {\n   "id": "way/789713815",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.716571,\n   "lon": 37.5689246,\n   "opening_hours": null\n  },\n  {\n   "id": "way/790434643",\n   "kind": "building",\n   "name": "Институт энергетических проблем химической физики РАН им. В. Л. Тальрозе",\n   "office_type": "research",\n   "lat": 55.7051857,\n   "lon": 37.5691862,\n   "opening_hours": null\n  },\n  {\n   "id": "way/791862947",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8591576,\n   "lon": 37.6848163,\n   "opening_hours": null\n  },\n  {\n   "id": "way/791952460",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6969994,\n   "lon": 37.5637828,\n   "opening_hours": null\n  },\n  {\n   "id": "way/791952461",\n   "kind": "building",\n   "name": "",\n   "office_type": "research",\n   "lat": 55.6985573,\n   "lon": 37.5664996,\n   "opening_hours": null\n  },\n  {\n   "id": "way/791952462",\n   "kind": "building",\n   "name": "",\n   "office_type": "research",\n   "lat": 55.698917,\n   "lon": 37.5669721,\n   "opening_hours": null\n  },\n  {\n   "id": "way/791952463",\n   "kind": "building",\n   "name": "Институт точной механики и вычислительной техники им. С. А. Лебедева",\n   "office_type": "research",\n   "lat": 55.6989363,\n   "lon": 37.5662262,\n   "opening_hours": null\n  },\n  {\n   "id": "way/792950883",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7122756,\n   "lon": 37.6275545,\n   "opening_hours": null\n  },\n  {\n   "id": "way/792950884",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7117358,\n   "lon": 37.6278843,\n   "opening_hours": null\n  },\n  {\n   "id": "way/792950890",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7117762,\n   "lon": 37.6295007,\n   "opening_hours": null\n  },\n  {\n   "id": "way/793424261",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7064849,\n   "lon": 37.6107768,\n   "opening_hours": null\n  },\n  {\n   "id": "way/793424270",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7058017,\n   "lon": 37.6128071,\n   "opening_hours": null\n  },\n  {\n   "id": "way/793646305",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7097423,\n   "lon": 37.6169077,\n   "opening_hours": null\n  },\n  {\n   "id": "way/795000663",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7248891,\n   "lon": 37.6101599,\n   "opening_hours": null\n  },\n  {\n   "id": "way/795321078",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7074708,\n   "lon": 37.6004688,\n   "opening_hours": null\n  },\n  {\n   "id": "way/795321079",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7079227,\n   "lon": 37.5997718,\n   "opening_hours": null\n  },\n  {\n   "id": "way/796314604",\n   "kind": "building",\n   "name": "Росэнергоатом",\n   "office_type": "company",\n   "lat": 55.7100363,\n   "lon": 37.6267419,\n   "opening_hours": null\n  },\n  {\n   "id": "way/796314607",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.706259,\n   "lon": 37.6253909,\n   "opening_hours": null\n  },\n  {\n   "id": "way/796314609",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Center-T\\"",\n   "office_type": null,\n   "lat": 55.7075061,\n   "lon": 37.6265177,\n   "opening_hours": null\n  },\n  {\n   "id": "way/796671035",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7037773,\n   "lon": 37.614713,\n   "opening_hours": null\n  },\n  {\n   "id": "way/797670343",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7609856,\n   "lon": 37.5196851,\n   "opening_hours": null\n  },\n  {\n   "id": "way/798376273",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8126441,\n   "lon": 37.6869967,\n   "opening_hours": null\n  },\n  {\n   "id": "way/802048846",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7159936,\n   "lon": 37.633747,\n   "opening_hours": null\n  },\n  {\n   "id": "way/803075408",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6534367,\n   "lon": 37.4950654,\n   "opening_hours": null\n  },\n  {\n   "id": "way/805678339",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7104972,\n   "lon": 37.5867099,\n   "opening_hours": null\n  },\n  {\n   "id": "way/809261886",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.80128,\n   "lon": 37.68764,\n   "opening_hours": null\n  },\n  {\n   "id": "way/809610199",\n   "kind": "building",\n   "name": "ОКО-II",\n   "office_type": null,\n   "lat": 55.7504871,\n   "lon": 37.5363831,\n   "opening_hours": null\n  },\n  {\n   "id": "way/812647010",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7964097,\n   "lon": 37.6640906,\n   "opening_hours": null\n  },\n  {\n   "id": "way/813851614",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7859892,\n   "lon": 37.6903081,\n   "opening_hours": null\n  },\n  {\n   "id": "way/813931154",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7991289,\n   "lon": 37.6725644,\n   "opening_hours": null\n  },\n  {\n   "id": "way/814267652",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "security",\n   "lat": 55.6828164,\n   "lon": 37.5426601,\n   "opening_hours": null\n  },\n  {\n   "id": "way/814636514",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7824924,\n   "lon": 37.6928014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/818074286",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7834174,\n   "lon": 37.5983789,\n   "opening_hours": null\n  },\n  {\n   "id": "way/819786155",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7582967,\n   "lon": 37.5863625,\n   "opening_hours": null\n  },\n  {\n   "id": "way/822838020",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8001807,\n   "lon": 37.6389997,\n   "opening_hours": null\n  },\n  {\n   "id": "way/822838021",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8009802,\n   "lon": 37.638277,\n   "opening_hours": null\n  },\n  {\n   "id": "way/822838022",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8004704,\n   "lon": 37.6372685,\n   "opening_hours": null\n  },\n  {\n   "id": "way/823553948",\n   "kind": "building",\n   "name": "Министерство Российской Федерации по развитию Дальнего Востока и Арктики",\n   "office_type": "government",\n   "lat": 55.7457851,\n   "lon": 37.6723461,\n   "opening_hours": null\n  },\n  {\n   "id": "way/824528656",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6530728,\n   "lon": 37.6944405,\n   "opening_hours": null\n  },\n  {\n   "id": "way/825189165",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7436576,\n   "lon": 37.6914575,\n   "opening_hours": null\n  },\n  {\n   "id": "way/832137711",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8044206,\n   "lon": 37.7039015,\n   "opening_hours": null\n  },\n  {\n   "id": "way/832137713",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8036964,\n   "lon": 37.7041472,\n   "opening_hours": null\n  },\n  {\n   "id": "way/835466032",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7793104,\n   "lon": 37.6764756,\n   "opening_hours": null\n  },\n  {\n   "id": "way/835611874",\n   "kind": "organisation",\n   "name": "Главное управление экономической безопасности и противодействия коррупции МВД РФ",\n   "office_type": "police",\n   "lat": 55.724733,\n   "lon": 37.6101492,\n   "opening_hours": null\n  },\n  {\n   "id": "way/838577773",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7596044,\n   "lon": 37.5916431,\n   "opening_hours": null\n  },\n  {\n   "id": "way/849480841",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7564864,\n   "lon": 37.7356025,\n   "opening_hours": null\n  },\n  {\n   "id": "way/849480857",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7573194,\n   "lon": 37.7363031,\n   "opening_hours": null\n  },\n  {\n   "id": "way/849488280",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8471727,\n   "lon": 37.5921227,\n   "opening_hours": null\n  },\n  {\n   "id": "way/850793246",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7789085,\n   "lon": 37.7147571,\n   "opening_hours": null\n  },\n  {\n   "id": "way/850793250",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7796408,\n   "lon": 37.7136761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/851733239",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7860402,\n   "lon": 37.7193047,\n   "opening_hours": null\n  },\n  {\n   "id": "way/852078756",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7978915,\n   "lon": 37.7043616,\n   "opening_hours": null\n  },\n  {\n   "id": "way/854912218",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7706639,\n   "lon": 37.6122389,\n   "opening_hours": null\n  },\n  {\n   "id": "way/865594604",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7763012,\n   "lon": 37.6693347,\n   "opening_hours": null\n  },\n  {\n   "id": "way/866730686",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8170817,\n   "lon": 37.7545857,\n   "opening_hours": null\n  },\n  {\n   "id": "way/868127395",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8031978,\n   "lon": 37.6819476,\n   "opening_hours": null\n  },\n  {\n   "id": "way/868244171",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7908587,\n   "lon": 37.6895181,\n   "opening_hours": null\n  },\n  {\n   "id": "way/868369964",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6910257,\n   "lon": 37.6714633,\n   "opening_hours": null\n  },\n  {\n   "id": "way/869879055",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7348436,\n   "lon": 37.6154433,\n   "opening_hours": null\n  },\n  {\n   "id": "way/871830346",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8979008,\n   "lon": 37.7953773,\n   "opening_hours": null\n  },\n  {\n   "id": "way/875297806",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8039079,\n   "lon": 37.706488,\n   "opening_hours": null\n  },\n  {\n   "id": "way/875621410",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.787744,\n   "lon": 37.635859,\n   "opening_hours": null\n  },\n  {\n   "id": "way/875900151",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8004349,\n   "lon": 37.7268355,\n   "opening_hours": null\n  },\n  {\n   "id": "way/876448398",\n   "kind": "organisation",\n   "name": "linxdatacenter",\n   "office_type": "telecommunication",\n   "lat": 55.8056547,\n   "lon": 37.5508871,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "way/877956160",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7456049,\n   "lon": 37.8311202,\n   "opening_hours": null\n  },\n  {\n   "id": "way/881413679",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.857882,\n   "lon": 37.6542294,\n   "opening_hours": null\n  },\n  {\n   "id": "way/881413681",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8581069,\n   "lon": 37.6534654,\n   "opening_hours": null\n  },\n  {\n   "id": "way/881677302",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8578804,\n   "lon": 37.6564061,\n   "opening_hours": null\n  },\n  {\n   "id": "way/882395241",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7639865,\n   "lon": 37.5704511,\n   "opening_hours": null\n  },\n  {\n   "id": "way/882704269",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.775914,\n   "lon": 37.673066,\n   "opening_hours": null\n  },\n  {\n   "id": "way/885242127",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7411854,\n   "lon": 37.8192775,\n   "opening_hours": null\n  },\n  {\n   "id": "way/887874847",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8123732,\n   "lon": 37.6012408,\n   "opening_hours": null\n  },\n  {\n   "id": "way/887874852",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8114711,\n   "lon": 37.6028618,\n   "opening_hours": null\n  },\n  {\n   "id": "way/887983715",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7826376,\n   "lon": 37.6352951,\n   "opening_hours": null\n  },\n  {\n   "id": "way/887983716",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7828211,\n   "lon": 37.6352928,\n   "opening_hours": null\n  },\n  {\n   "id": "way/888596972",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6845108,\n   "lon": 37.6219541,\n   "opening_hours": null\n  },\n  {\n   "id": "way/889430324",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7839206,\n   "lon": 37.677323,\n   "opening_hours": null\n  },\n  {\n   "id": "way/895643264",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7202628,\n   "lon": 37.7274855,\n   "opening_hours": null\n  },\n  {\n   "id": "way/896000448",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7525075,\n   "lon": 37.6417105,\n   "opening_hours": null\n  },\n  {\n   "id": "way/901188668",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7956506,\n   "lon": 37.5621862,\n   "opening_hours": null\n  },\n  {\n   "id": "way/901188669",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7957277,\n   "lon": 37.5621759,\n   "opening_hours": null\n  },\n  {\n   "id": "way/905133564",\n   "kind": "organisation",\n   "name": "Офис продаж Wellton Towers",\n   "office_type": "estate_agent",\n   "lat": 55.7746173,\n   "lon": 37.4743983,\n   "opening_hours": null\n  },\n  {\n   "id": "way/907845793",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Золотое кольцо\\"",\n   "office_type": null,\n   "lat": 55.7070622,\n   "lon": 37.6888364,\n   "opening_hours": null\n  },\n  {\n   "id": "way/912954480",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8088061,\n   "lon": 37.6586519,\n   "opening_hours": null\n  },\n  {\n   "id": "way/926960185",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7720535,\n   "lon": 37.5478217,\n   "opening_hours": null\n  },\n  {\n   "id": "way/927808755",\n   "kind": "building",\n   "name": "Action 44",\n   "office_type": null,\n   "lat": 55.7694235,\n   "lon": 37.5593484,\n   "opening_hours": null\n  },\n  {\n   "id": "way/927934767",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7444924,\n   "lon": 37.6629137,\n   "opening_hours": null\n  },\n  {\n   "id": "way/929102217",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7829558,\n   "lon": 37.5827041,\n   "opening_hours": null\n  },\n  {\n   "id": "way/933023157",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7633636,\n   "lon": 37.6832244,\n   "opening_hours": null\n  },\n  {\n   "id": "way/934038206",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7904054,\n   "lon": 37.7109939,\n   "opening_hours": null\n  },\n  {\n   "id": "way/934038207",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7900205,\n   "lon": 37.7115881,\n   "opening_hours": null\n  },\n  {\n   "id": "way/934038208",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7901755,\n   "lon": 37.7113866,\n   "opening_hours": null\n  },\n  {\n   "id": "way/934038209",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7901766,\n   "lon": 37.7109316,\n   "opening_hours": null\n  },\n  {\n   "id": "way/934038210",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7902358,\n   "lon": 37.7106929,\n   "opening_hours": null\n  },\n  {\n   "id": "way/934038212",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7896469,\n   "lon": 37.7115817,\n   "opening_hours": null\n  },\n  {\n   "id": "way/934144441",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8366618,\n   "lon": 37.6460867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/934305362",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8416876,\n   "lon": 37.6572951,\n   "opening_hours": null\n  },\n  {\n   "id": "way/934640085",\n   "kind": "building",\n   "name": "Творческая мастерская З.К. Церетели \\"Детская школа искусств\\"",\n   "office_type": null,\n   "lat": 55.7413466,\n   "lon": 37.627899,\n   "opening_hours": null\n  },\n  {\n   "id": "way/938285122",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7663187,\n   "lon": 37.7329283,\n   "opening_hours": null\n  },\n  {\n   "id": "way/938285130",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.765612,\n   "lon": 37.7354258,\n   "opening_hours": null\n  },\n  {\n   "id": "way/940477808",\n   "kind": "organisation",\n   "name": "Техническая инспекция ОАТИ",\n   "office_type": "government",\n   "lat": 55.7392124,\n   "lon": 37.6642032,\n   "opening_hours": null\n  },\n  {\n   "id": "way/940624209",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7959013,\n   "lon": 37.6620881,\n   "opening_hours": null\n  },\n  {\n   "id": "way/942338738",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7388152,\n   "lon": 37.6631322,\n   "opening_hours": null\n  },\n  {\n   "id": "way/943013022",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7921614,\n   "lon": 37.7068919,\n   "opening_hours": null\n  },\n  {\n   "id": "way/945755484",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8625596,\n   "lon": 37.4499648,\n   "opening_hours": null\n  },\n  {\n   "id": "way/949165966",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7841084,\n   "lon": 37.724276,\n   "opening_hours": null\n  },\n  {\n   "id": "way/949165968",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7842627,\n   "lon": 37.7250031,\n   "opening_hours": null\n  },\n  {\n   "id": "way/952713423",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7414395,\n   "lon": 37.583528,\n   "opening_hours": null\n  },\n  {\n   "id": "way/963164484",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8488747,\n   "lon": 37.6210551,\n   "opening_hours": null\n  },\n  {\n   "id": "way/971118063",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7089741,\n   "lon": 37.6596342,\n   "opening_hours": null\n  },\n  {\n   "id": "way/981382157",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7826999,\n   "lon": 37.6347751,\n   "opening_hours": null\n  },\n  {\n   "id": "way/986672025",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.778313,\n   "lon": 37.6491951,\n   "opening_hours": null\n  },\n  {\n   "id": "way/986672026",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7783931,\n   "lon": 37.6488734,\n   "opening_hours": null\n  },\n  {\n   "id": "way/990435948",\n   "kind": "building",\n   "name": "Офис продаж ЖК \\"Легендарный квартал\\"",\n   "office_type": "estate_agent",\n   "lat": 55.8480917,\n   "lon": 37.6241399,\n   "opening_hours": null\n  },\n  {\n   "id": "way/992221277",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7374824,\n   "lon": 37.7300279,\n   "opening_hours": null\n  },\n  {\n   "id": "way/992221278",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7380888,\n   "lon": 37.7302678,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319788",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7038676,\n   "lon": 37.6917947,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319789",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7036627,\n   "lon": 37.6916274,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319791",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7030372,\n   "lon": 37.6894405,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319794",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.700936,\n   "lon": 37.6899686,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319795",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7014018,\n   "lon": 37.6887731,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319796",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7020454,\n   "lon": 37.6907728,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319797",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7017648,\n   "lon": 37.6891794,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319798",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7015791,\n   "lon": 37.6900411,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319799",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7022474,\n   "lon": 37.6894813,\n   "opening_hours": null\n  },\n  {\n   "id": "way/993319800",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7020155,\n   "lon": 37.6894978,\n   "opening_hours": null\n  },\n  {\n   "id": "way/996708912",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7059211,\n   "lon": 37.6919223,\n   "opening_hours": null\n  },\n  {\n   "id": "way/998324827",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7089716,\n   "lon": 37.6851636,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1006800146",\n   "kind": "building",\n   "name": "Центральный научно-исследовательский институт эпидемиологии",\n   "office_type": "research",\n   "lat": 55.7432604,\n   "lon": 37.6409645,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1009809547",\n   "kind": "organisation",\n   "name": "ООО \\"Облачные технологии\\"",\n   "office_type": "it",\n   "lat": 55.762894,\n   "lon": 37.557018,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "way/1009810033",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7621605,\n   "lon": 37.5572426,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1013827767",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7026178,\n   "lon": 37.7311014,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1015112317",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7893366,\n   "lon": 37.5812894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1015112319",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7895492,\n   "lon": 37.5792787,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1015112320",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7899352,\n   "lon": 37.5781447,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1015112321",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.790278,\n   "lon": 37.5781761,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1015112327",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7851543,\n   "lon": 37.5838608,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1019682410",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7468258,\n   "lon": 37.4971197,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1020964789",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7626351,\n   "lon": 37.6322053,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1025492165",\n   "kind": "organisation",\n   "name": "ЖК \\"Режиссер\\"",\n   "office_type": "estate_agent",\n   "lat": 55.834381,\n   "lon": 37.6387737,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1035585623",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7504612,\n   "lon": 37.7708658,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1037165433",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7440524,\n   "lon": 37.7721538,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1041133293",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8485816,\n   "lon": 37.5830718,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1055478571",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7362124,\n   "lon": 37.7082215,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1056570453",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.779083,\n   "lon": 37.6942253,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1058956317",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7357476,\n   "lon": 37.7117291,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1058956318",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7356054,\n   "lon": 37.712126,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1061534276",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7550704,\n   "lon": 37.7401252,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1061534277",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7548747,\n   "lon": 37.7394802,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1062506601",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8368004,\n   "lon": 37.4516621,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1064649498",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7545715,\n   "lon": 37.7394691,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1065492788",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7315604,\n   "lon": 37.5508592,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1066253622",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7782879,\n   "lon": 37.6335508,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1068412743",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8213237,\n   "lon": 37.5627688,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1072076275",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7151739,\n   "lon": 37.6595132,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1076096690",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7451285,\n   "lon": 37.645862,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1076096702",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7458064,\n   "lon": 37.6476858,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1076096706",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7449934,\n   "lon": 37.6479695,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1076096713",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7445379,\n   "lon": 37.6482449,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1076096718",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7445373,\n   "lon": 37.6465934,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1077515384",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7518319,\n   "lon": 37.6448867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1081797214",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7735152,\n   "lon": 37.7232518,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1082296056",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7895141,\n   "lon": 37.5857097,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1091019125",\n   "kind": "organisation",\n   "name": "Конечная станция \\"улица Лебедева\\"",\n   "office_type": "transportation",\n   "lat": 55.6972636,\n   "lon": 37.5280174,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1091862055",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8414642,\n   "lon": 37.626266,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1094572408",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.85694,\n   "lon": 37.4804884,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1098573836",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6912974,\n   "lon": 37.656763,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1134291444",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7130955,\n   "lon": 37.57964,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1135283095",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7650355,\n   "lon": 37.6287023,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1150916669",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7880974,\n   "lon": 37.578367,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1155834189",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6697861,\n   "lon": 37.4689314,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1171996570",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7811078,\n   "lon": 37.5945644,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1171996571",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7816048,\n   "lon": 37.593755,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1179695832",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8284508,\n   "lon": 37.4444906,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1181750799",\n   "kind": "building",\n   "name": "Центр информационных технологий Центрального банка РФ",\n   "office_type": null,\n   "lat": 55.8508802,\n   "lon": 37.4483427,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1188275504",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7819615,\n   "lon": 37.6500386,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1188734283",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7289341,\n   "lon": 37.6182343,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1190127789",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8213742,\n   "lon": 37.4975657,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1193729783",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8756476,\n   "lon": 37.5488641,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1194664870",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8154039,\n   "lon": 37.6042095,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1194664871",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8160421,\n   "lon": 37.6035186,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1194664872",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8165243,\n   "lon": 37.6007018,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1194664873",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8164555,\n   "lon": 37.6021284,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1194664874",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8157641,\n   "lon": 37.6017092,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1205165299",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8545003,\n   "lon": 37.6046994,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1208129254",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "security",\n   "lat": 55.6991807,\n   "lon": 37.65473,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1211542617",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8112942,\n   "lon": 37.8043282,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1212143847",\n   "kind": "organisation",\n   "name": "Офис продаж ЖК \\"Level Южнопортовая\\"",\n   "office_type": "estate_agent",\n   "lat": 55.7064769,\n   "lon": 37.7002114,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1212474857",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8156448,\n   "lon": 37.7843615,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1216209524",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7971353,\n   "lon": 37.6892175,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1216391945",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7907446,\n   "lon": 37.6832991,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1217052792",\n   "kind": "building",\n   "name": "Бизнес-центр \\"Пальмира\\"",\n   "office_type": null,\n   "lat": 55.7783344,\n   "lon": 37.6939938,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1217052793",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7785755,\n   "lon": 37.6935674,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1217065218",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7789094,\n   "lon": 37.6939725,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1217112504",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7816924,\n   "lon": 37.6975624,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1217112505",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7815542,\n   "lon": 37.6972239,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1217112507",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7809433,\n   "lon": 37.6974462,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218184063",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7853218,\n   "lon": 37.6909509,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218217046",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7845843,\n   "lon": 37.6898864,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218257135",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7836542,\n   "lon": 37.7097423,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218257136",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7831065,\n   "lon": 37.709296,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218469078",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7735945,\n   "lon": 37.7240414,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218484711",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.780075,\n   "lon": 37.7159791,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218484712",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7796705,\n   "lon": 37.7143696,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218484713",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7798973,\n   "lon": 37.7150934,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218484714",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7806369,\n   "lon": 37.7154824,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218943677",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7457184,\n   "lon": 37.8241468,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1218954303",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7461198,\n   "lon": 37.8311998,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1220466087",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8064909,\n   "lon": 37.7055083,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1221919097",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7916881,\n   "lon": 37.8168145,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1224986924",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7427081,\n   "lon": 37.5028894,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1225858600",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6503,\n   "lon": 37.6976579,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1230556086",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8130798,\n   "lon": 37.6057811,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1231772485",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7862652,\n   "lon": 37.6087897,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1232521138",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7671661,\n   "lon": 37.6368208,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1234294632",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7733288,\n   "lon": 37.6775767,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1234550686",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7649152,\n   "lon": 37.681095,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1234594084",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7602981,\n   "lon": 37.6881166,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1235852722",\n   "kind": "organisation",\n   "name": "Институт физики твёрдого тела РАН",\n   "office_type": "research",\n   "lat": 55.6932134,\n   "lon": 37.5504782,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1236063671",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "security",\n   "lat": 55.6805563,\n   "lon": 37.5390825,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1236346713",\n   "kind": "building",\n   "name": "Бизнес-центр Orbital",\n   "office_type": null,\n   "lat": 55.7629044,\n   "lon": 37.523295,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1236346715",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7574207,\n   "lon": 37.5298986,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1241077918",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7700924,\n   "lon": 37.5902585,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1241077920",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7698817,\n   "lon": 37.5905862,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1242441863",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.823585,\n   "lon": 37.57626,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1251332631",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.762142,\n   "lon": 37.5552665,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1251332632",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.762339,\n   "lon": 37.5567183,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1251332633",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7620436,\n   "lon": 37.556544,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1252904101",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7781607,\n   "lon": 37.6490939,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1253396959",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник\\"",\n   "office_type": "property_management",\n   "lat": 55.8847843,\n   "lon": 37.551557,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1253785678",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "security",\n   "lat": 55.7463963,\n   "lon": 37.4757132,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1255506148",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7864077,\n   "lon": 37.6059394,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1255607274",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7808106,\n   "lon": 37.5956149,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1257342119",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7779184,\n   "lon": 37.5918189,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1257342120",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7770339,\n   "lon": 37.5905867,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1257472131",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7330865,\n   "lon": 37.6650461,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1258205683",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6716676,\n   "lon": 37.5268939,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1258446502",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8441085,\n   "lon": 37.5713742,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1258452589",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7555722,\n   "lon": 37.5553055,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1258522104",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8112682,\n   "lon": 37.6239737,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1258522105",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.810969,\n   "lon": 37.6239829,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1265302069",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.8298589,\n   "lon": 37.64905,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1270834184",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7558294,\n   "lon": 37.6058009,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1271395294",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7619273,\n   "lon": 37.6311597,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1273308267",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7406114,\n   "lon": 37.6000034,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1273318354",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7386712,\n   "lon": 37.5976759,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1279262549",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7447662,\n   "lon": 37.6303621,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1280857624",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7773511,\n   "lon": 37.6935411,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1282953785",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7912039,\n   "lon": 37.5418945,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1282985606",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7797683,\n   "lon": 37.6945553,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1290248549",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "security",\n   "lat": 55.675269,\n   "lon": 37.6557109,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1290248554",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "security",\n   "lat": 55.6711941,\n   "lon": 37.6552968,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1302143446",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "security",\n   "lat": 55.7343156,\n   "lon": 37.5782294,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1315791861",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.7430049,\n   "lon": 37.6016167,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1335366009",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6179882,\n   "lon": 37.6553352,\n   "opening_hours": null\n  },\n  {\n   "id": "way/1336267457",\n   "kind": "building",\n   "name": "",\n   "office_type": null,\n   "lat": 55.6427606,\n   "lon": 37.8000395,\n   "opening_hours": null\n  },\n  {\n   "id": "node/251544377",\n   "kind": "organisation",\n   "name": "Медведковский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.8770116,\n   "lon": 37.6458512,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/317232723",\n   "kind": "organisation",\n   "name": "ЦОДД",\n   "office_type": "yes",\n   "lat": 55.6906071,\n   "lon": 37.8210599,\n   "opening_hours": null\n  },\n  {\n   "id": "node/380729527",\n   "kind": "organisation",\n   "name": "Префектура ЮВАО г. Москвы",\n   "office_type": "government",\n   "lat": 55.7540334,\n   "lon": 37.7144778,\n   "opening_hours": null\n  },\n  {\n   "id": "node/432502061",\n   "kind": "organisation",\n   "name": "Московская Страховая Компания",\n   "office_type": "insurance",\n   "lat": 55.7790193,\n   "lon": 37.6017619,\n   "opening_hours": null\n  },\n  {\n   "id": "node/432507349",\n   "kind": "organisation",\n   "name": "Информстрах",\n   "office_type": "insurance",\n   "lat": 55.7792066,\n   "lon": 37.6028396,\n   "opening_hours": null\n  },\n  {\n   "id": "node/510113949",\n   "kind": "organisation",\n   "name": "посольство Эквадора",\n   "office_type": "diplomatic",\n   "lat": 55.7649729,\n   "lon": 37.6674539,\n   "opening_hours": null\n  },\n  {\n   "id": "node/526557513",\n   "kind": "organisation",\n   "name": "Бабушкинский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.8696497,\n   "lon": 37.6619057,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/530204987",\n   "kind": "organisation",\n   "name": "Центральная акцизная таможня",\n   "office_type": "government",\n   "lat": 55.7467403,\n   "lon": 37.6471724,\n   "opening_hours": null\n  },\n  {\n   "id": "node/530602344",\n   "kind": "organisation",\n   "name": "Нотариальная контора",\n   "office_type": "notary",\n   "lat": 55.7123358,\n   "lon": 37.7582432,\n   "opening_hours": null\n  },\n  {\n   "id": "node/555943220",\n   "kind": "organisation",\n   "name": "Дворец Бракосочетания №3",\n   "office_type": "government",\n   "lat": 55.7003427,\n   "lon": 37.7456191,\n   "opening_hours": "Tu-Sa 09:00-14:00,15:00-18:00"\n  },\n  {\n   "id": "node/571445187",\n   "kind": "organisation",\n   "name": "Московский центр качества образования",\n   "office_type": "government",\n   "lat": 55.8083802,\n   "lon": 37.5233786,\n   "opening_hours": null\n  },\n  {\n   "id": "node/588155402",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8704067,\n   "lon": 37.6622526,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/588155406",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "lawyer",\n   "lat": 55.8689992,\n   "lon": 37.6700894,\n   "opening_hours": "Mo-Fr 09:30-18:00;Sa 10:00-17:00; Su 10:00-14:00; 14:00-15:00 off"\n  },\n  {\n   "id": "node/593991927",\n   "kind": "organisation",\n   "name": "Юридическая консультация №17",\n   "office_type": "lawyer",\n   "lat": 55.8681323,\n   "lon": 37.6797799,\n   "opening_hours": null\n  },\n  {\n   "id": "node/610804802",\n   "kind": "organisation",\n   "name": "Бабушкинский военкомат",\n   "office_type": "government",\n   "lat": 55.802077,\n   "lon": 37.5090749,\n   "opening_hours": "Mo-We 10:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/621918658",\n   "kind": "organisation",\n   "name": "Посольство Ливии",\n   "office_type": "diplomatic",\n   "lat": 55.7194921,\n   "lon": 37.5197423,\n   "opening_hours": "Mo-Fr 09:00-15:00"\n  },\n  {\n   "id": "node/621974631",\n   "kind": "organisation",\n   "name": "Посольство Объединенных Арабских Эмиратов",\n   "office_type": "diplomatic",\n   "lat": 55.7185857,\n   "lon": 37.5142277,\n   "opening_hours": "Mo-Fr 09:00-15:00"\n  },\n  {\n   "id": "node/622000107",\n   "kind": "organisation",\n   "name": "Посольство Кувейта",\n   "office_type": "diplomatic",\n   "lat": 55.7200775,\n   "lon": 37.5188446,\n   "opening_hours": "Mo-Fr 09:00-15:00"\n  },\n  {\n   "id": "node/622006644",\n   "kind": "organisation",\n   "name": "Посольство Австралии",\n   "office_type": "diplomatic",\n   "lat": 55.7525862,\n   "lon": 37.6424555,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/660562337",\n   "kind": "organisation",\n   "name": "УФМС г. Москвы по ЮАО",\n   "office_type": "government",\n   "lat": 55.5953254,\n   "lon": 37.724038,\n   "opening_hours": null\n  },\n  {\n   "id": "node/666283542",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт транспортного строительства",\n   "office_type": "research",\n   "lat": 55.8582564,\n   "lon": 37.6409519,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698892110",\n   "kind": "organisation",\n   "name": "Консульство Австрии",\n   "office_type": "diplomatic",\n   "lat": 55.7429332,\n   "lon": 37.5904418,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698892145",\n   "kind": "organisation",\n   "name": "Посольство Судана",\n   "office_type": "diplomatic",\n   "lat": 55.769001,\n   "lon": 37.6067085,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698892149",\n   "kind": "organisation",\n   "name": "Посольство Словении",\n   "office_type": "diplomatic",\n   "lat": 55.7696838,\n   "lon": 37.6058863,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698892150",\n   "kind": "organisation",\n   "name": "Посольство Алжира",\n   "office_type": "diplomatic",\n   "lat": 55.7669296,\n   "lon": 37.6164169,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698892153",\n   "kind": "organisation",\n   "name": "Посольство Венесуэлы",\n   "office_type": "diplomatic",\n   "lat": 55.7754321,\n   "lon": 37.622791,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698892155",\n   "kind": "organisation",\n   "name": "Посольство Беларуси",\n   "office_type": "diplomatic",\n   "lat": 55.7582251,\n   "lon": 37.6380818,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698892169",\n   "kind": "organisation",\n   "name": "Посольство Республики Казахстан",\n   "office_type": "diplomatic",\n   "lat": 55.7638167,\n   "lon": 37.6417137,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698892171",\n   "kind": "organisation",\n   "name": "Посольство Швейцарии",\n   "office_type": "diplomatic",\n   "lat": 55.7652039,\n   "lon": 37.6410307,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698892188",\n   "kind": "organisation",\n   "name": "посольство Ливана",\n   "office_type": "diplomatic",\n   "lat": 55.7378714,\n   "lon": 37.593152,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698905994",\n   "kind": "organisation",\n   "name": "Посольство Анголы",\n   "office_type": "diplomatic",\n   "lat": 55.7194303,\n   "lon": 37.5124425,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698906178",\n   "kind": "organisation",\n   "name": "Посольство Брунея",\n   "office_type": "diplomatic",\n   "lat": 55.7415448,\n   "lon": 37.5662172,\n   "opening_hours": null\n  },\n  {\n   "id": "node/698906226",\n   "kind": "organisation",\n   "name": "посольство Алжира (консульский отдел)",\n   "office_type": "diplomatic",\n   "lat": 55.7761763,\n   "lon": 37.6480193,\n   "opening_hours": "Mo-Th 09:00-13:00"\n  },\n  {\n   "id": "node/699049187",\n   "kind": "organisation",\n   "name": "посольство Узбекистана",\n   "office_type": "diplomatic",\n   "lat": 55.7326469,\n   "lon": 37.6212885,\n   "opening_hours": null\n  },\n  {\n   "id": "node/699049191",\n   "kind": "organisation",\n   "name": "Посольство Израиля",\n   "office_type": "diplomatic",\n   "lat": 55.7337798,\n   "lon": 37.6236567,\n   "opening_hours": null\n  },\n  {\n   "id": "node/702748380",\n   "kind": "organisation",\n   "name": "МОФОМС",\n   "office_type": "government",\n   "lat": 55.7216968,\n   "lon": 37.6544792,\n   "opening_hours": null\n  },\n  {\n   "id": "node/765089053",\n   "kind": "organisation",\n   "name": "Газпром",\n   "office_type": "company",\n   "lat": 55.6657486,\n   "lon": 37.4864806,\n   "opening_hours": null\n  },\n  {\n   "id": "node/775957750",\n   "kind": "organisation",\n   "name": "управление Госрадиокомплект",\n   "office_type": "government",\n   "lat": 55.7508393,\n   "lon": 37.7081449,\n   "opening_hours": null\n  },\n  {\n   "id": "node/794595322",\n   "kind": "organisation",\n   "name": "Ассоциация международных автомобильных перевозчиков",\n   "office_type": "company",\n   "lat": 55.8037876,\n   "lon": 37.5285776,\n   "opening_hours": null\n  },\n  {\n   "id": "node/797196577",\n   "kind": "organisation",\n   "name": "Международная организация труда (МОТ) в РФ",\n   "office_type": "yes",\n   "lat": 55.7640154,\n   "lon": 37.6168294,\n   "opening_hours": null\n  },\n  {\n   "id": "node/797201499",\n   "kind": "organisation",\n   "name": "Офис Старшего советника по правам человекапри системе ООН в Российской Федерации",\n   "office_type": "company",\n   "lat": 55.7592077,\n   "lon": 37.6025124,\n   "opening_hours": null\n  },\n  {\n   "id": "node/798569021",\n   "kind": "organisation",\n   "name": "Инженерная служба района Вешняки",\n   "office_type": "property_management",\n   "lat": 55.7200421,\n   "lon": 37.8185836,\n   "opening_hours": null\n  },\n  {\n   "id": "node/798569022",\n   "kind": "organisation",\n   "name": "Управляющая организация ГУП \\"ДЕЗ района Вешняки\\"",\n   "office_type": "property_management",\n   "lat": 55.7201839,\n   "lon": 37.8180984,\n   "opening_hours": null\n  },\n  {\n   "id": "node/798569023",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Вешняки",\n   "office_type": "government",\n   "lat": 55.7345522,\n   "lon": 37.8335737,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/803662802",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7385638,\n   "lon": 37.5454036,\n   "opening_hours": null\n  },\n  {\n   "id": "node/838569813",\n   "kind": "organisation",\n   "name": "Посольство Южной Осетии",\n   "office_type": "diplomatic",\n   "lat": 55.7414437,\n   "lon": 37.60629,\n   "opening_hours": null\n  },\n  {\n   "id": "node/846812374",\n   "kind": "organisation",\n   "name": "посольство Чили",\n   "office_type": "diplomatic",\n   "lat": 55.7437458,\n   "lon": 37.5874667,\n   "opening_hours": null\n  },\n  {\n   "id": "node/850774994",\n   "kind": "organisation",\n   "name": "Инспекция федеральной налоговой службы №16 по г. Москве",\n   "office_type": "government",\n   "lat": 55.8781202,\n   "lon": 37.6990442,\n   "opening_hours": null\n  },\n  {\n   "id": "node/893134126",\n   "kind": "organisation",\n   "name": "Трудовая инспекция",\n   "office_type": "government",\n   "lat": 55.6075422,\n   "lon": 37.7083146,\n   "opening_hours": null\n  },\n  {\n   "id": "node/914269753",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7960799,\n   "lon": 37.79957,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/920582529",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7997227,\n   "lon": 37.7939924,\n   "opening_hours": null\n  },\n  {\n   "id": "node/921902301",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7971081,\n   "lon": 37.796059,\n   "opening_hours": null\n  },\n  {\n   "id": "node/925170722",\n   "kind": "organisation",\n   "name": "Комиссия по комплектованию дошкольных образовательных учреждений района Филёвский Парк",\n   "office_type": "government",\n   "lat": 55.736572,\n   "lon": 37.4792303,\n   "opening_hours": null\n  },\n  {\n   "id": "node/926954558",\n   "kind": "organisation",\n   "name": "Детвиль",\n   "office_type": "company",\n   "lat": 55.8823327,\n   "lon": 37.6344383,\n   "opening_hours": null\n  },\n  {\n   "id": "node/937260445",\n   "kind": "organisation",\n   "name": "Интерлан Коммуникейшнс",\n   "office_type": "telecommunication",\n   "lat": 55.8727745,\n   "lon": 37.6832264,\n   "opening_hours": "Mo-Fr 11:00-20:00; Sa-Su 11:00-19:00"\n  },\n  {\n   "id": "node/938471182",\n   "kind": "organisation",\n   "name": "Тушинский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.8523549,\n   "lon": 37.4470131,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/938526782",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7317935,\n   "lon": 37.5486456,\n   "opening_hours": null\n  },\n  {\n   "id": "node/942448646",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.828408,\n   "lon": 37.4512661,\n   "opening_hours": null\n  },\n  {\n   "id": "node/942482697",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.8305596,\n   "lon": 37.4535356,\n   "opening_hours": null\n  },\n  {\n   "id": "node/942567308",\n   "kind": "organisation",\n   "name": "Управление здравоохранения СЗАО г. Москвы",\n   "office_type": "government",\n   "lat": 55.8300829,\n   "lon": 37.4489438,\n   "opening_hours": null\n  },\n  {\n   "id": "node/947013372",\n   "kind": "organisation",\n   "name": "ИНКОМ",\n   "office_type": "estate_agent",\n   "lat": 55.8630656,\n   "lon": 37.6062183,\n   "opening_hours": null\n  },\n  {\n   "id": "node/954910738",\n   "kind": "organisation",\n   "name": "Министерство регионального развития",\n   "office_type": "government",\n   "lat": 55.6897778,\n   "lon": 37.5354141,\n   "opening_hours": null\n  },\n  {\n   "id": "node/954912010",\n   "kind": "organisation",\n   "name": "Газпром",\n   "office_type": "company",\n   "lat": 55.6887935,\n   "lon": 37.5335948,\n   "opening_hours": null\n  },\n  {\n   "id": "node/954912352",\n   "kind": "organisation",\n   "name": "Центр занятости населения",\n   "office_type": "government",\n   "lat": 55.6888522,\n   "lon": 37.5399604,\n   "opening_hours": null\n  },\n  {\n   "id": "node/956231549",\n   "kind": "organisation",\n   "name": "Производственно Коммерческая Дирекция",\n   "office_type": "company",\n   "lat": 55.7646985,\n   "lon": 37.5951558,\n   "opening_hours": null\n  },\n  {\n   "id": "node/956433457",\n   "kind": "organisation",\n   "name": "МЧС России",\n   "office_type": "government",\n   "lat": 55.6878688,\n   "lon": 37.5388913,\n   "opening_hours": null\n  },\n  {\n   "id": "node/962224344",\n   "kind": "organisation",\n   "name": "Спасские ворота",\n   "office_type": "insurance",\n   "lat": 55.6797584,\n   "lon": 37.6239561,\n   "opening_hours": null\n  },\n  {\n   "id": "node/964667536",\n   "kind": "organisation",\n   "name": "УФМС Чертаново Центральное",\n   "office_type": "government",\n   "lat": 55.6072352,\n   "lon": 37.5916817,\n   "opening_hours": null\n  },\n  {\n   "id": "node/967415335",\n   "kind": "organisation",\n   "name": "Теплопроект",\n   "office_type": "architect",\n   "lat": 55.8639267,\n   "lon": 37.6773341,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/967493362",\n   "kind": "organisation",\n   "name": "Муниципалитет района Лосиноостровский",\n   "office_type": "government",\n   "lat": 55.8564301,\n   "lon": 37.6701074,\n   "opening_hours": null\n  },\n  {\n   "id": "node/967493367",\n   "kind": "organisation",\n   "name": "Муниципалитет района Бабушкинский",\n   "office_type": "government",\n   "lat": 55.8564809,\n   "lon": 37.6701157,\n   "opening_hours": null\n  },\n  {\n   "id": "node/967493368",\n   "kind": "organisation",\n   "name": "Муниципалитет района Свиблово",\n   "office_type": "government",\n   "lat": 55.8565432,\n   "lon": 37.6701239,\n   "opening_hours": null\n  },\n  {\n   "id": "node/968170587",\n   "kind": "organisation",\n   "name": "НИ и ПИ Генплана Москвы",\n   "office_type": "company",\n   "lat": 55.7697327,\n   "lon": 37.5931612,\n   "opening_hours": null\n  },\n  {\n   "id": "node/968170600",\n   "kind": "organisation",\n   "name": "Моспроект",\n   "office_type": "architect",\n   "lat": 55.7701575,\n   "lon": 37.5928487,\n   "opening_hours": null\n  },\n  {\n   "id": "node/968170685",\n   "kind": "organisation",\n   "name": "Мосгосэкспертиза",\n   "office_type": "government",\n   "lat": 55.7713139,\n   "lon": 37.5906239,\n   "opening_hours": null\n  },\n  {\n   "id": "node/968170733",\n   "kind": "organisation",\n   "name": "Комитет по архитектуре и градостроительству Москвы",\n   "office_type": "government",\n   "lat": 55.7694219,\n   "lon": 37.5938188,\n   "opening_hours": null\n  },\n  {\n   "id": "node/968170787",\n   "kind": "organisation",\n   "name": "ОАО «Моспроект-2» имени М. В. Посохина",\n   "office_type": "architect",\n   "lat": 55.7703304,\n   "lon": 37.5909153,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/988853638",\n   "kind": "organisation",\n   "name": "1С",\n   "office_type": "company",\n   "lat": 55.7801738,\n   "lon": 37.6140109,\n   "opening_hours": null\n  },\n  {\n   "id": "node/993044951",\n   "kind": "organisation",\n   "name": "ГБУ \\"Автомобильные дороги\\"",\n   "office_type": "yes",\n   "lat": 55.7847585,\n   "lon": 37.4811889,\n   "opening_hours": null\n  },\n  {\n   "id": "node/994054736",\n   "kind": "organisation",\n   "name": "Богородский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.8135682,\n   "lon": 37.7202685,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/1017969474",\n   "kind": "organisation",\n   "name": "Управа района Обручевский",\n   "office_type": "government",\n   "lat": 55.6651751,\n   "lon": 37.517103,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1043580611",\n   "kind": "organisation",\n   "name": "Отдел жилищных субсидий \\"Бибирево\\"",\n   "office_type": "government",\n   "lat": 55.8954045,\n   "lon": 37.5946896,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1043580620",\n   "kind": "organisation",\n   "name": "Служба одного окна",\n   "office_type": "property_management",\n   "lat": 55.8954047,\n   "lon": 37.5947297,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1043580628",\n   "kind": "organisation",\n   "name": "Диспетчерская служба района Бибирево",\n   "office_type": "property_management",\n   "lat": 55.8954842,\n   "lon": 37.594692,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1055415374",\n   "kind": "organisation",\n   "name": "УФМС Царицыно",\n   "office_type": "government",\n   "lat": 55.6333038,\n   "lon": 37.6501629,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1064696301",\n   "kind": "organisation",\n   "name": "Пассажирское агентство метрополитена",\n   "office_type": "company",\n   "lat": 55.8968825,\n   "lon": 37.5878429,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1065506747",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Алтуфьевский",\n   "office_type": "government",\n   "lat": 55.8809,\n   "lon": 37.5894,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1086716562",\n   "kind": "organisation",\n   "name": "Интернет-провайдер \\"Старлинк\\"",\n   "office_type": "telecommunication",\n   "lat": 55.8962758,\n   "lon": 37.5880903,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1092115360",\n   "kind": "organisation",\n   "name": "УФМС Марфино",\n   "office_type": "government",\n   "lat": 55.8447457,\n   "lon": 37.586494,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1101780386",\n   "kind": "organisation",\n   "name": "Московское УВД на водном и воздушном транспорте",\n   "office_type": "government",\n   "lat": 55.7940852,\n   "lon": 37.4836431,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1106955937",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8134124,\n   "lon": 37.6372952,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/1112461355",\n   "kind": "organisation",\n   "name": "Государственная инспекция труда в Московской области",\n   "office_type": "government",\n   "lat": 55.6064358,\n   "lon": 37.7086885,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1112461368",\n   "kind": "organisation",\n   "name": "Государственная инспекция труда в городе Москве",\n   "office_type": "government",\n   "lat": 55.6065576,\n   "lon": 37.7086271,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1121305576",\n   "kind": "organisation",\n   "name": "XiTech.ru",\n   "office_type": "company",\n   "lat": 55.7667663,\n   "lon": 37.5262928,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/1132460159",\n   "kind": "organisation",\n   "name": "Авиа и ж/д билеты \\"Трансмост-тур\\"",\n   "office_type": "travel_agent",\n   "lat": 55.8852757,\n   "lon": 37.6020779,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1139440046",\n   "kind": "organisation",\n   "name": "ИФНС №26",\n   "office_type": "government",\n   "lat": 55.6444748,\n   "lon": 37.6163911,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1141840068",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.889187,\n   "lon": 37.5944235,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1167852511",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.7474988,\n   "lon": 37.8114095,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1211199997",\n   "kind": "organisation",\n   "name": "Административно-техническая инспекция СВАО",\n   "office_type": "government",\n   "lat": 55.8657325,\n   "lon": 37.6502078,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1212271025",\n   "kind": "organisation",\n   "name": "ОДС-17",\n   "office_type": "government",\n   "lat": 55.6086787,\n   "lon": 37.7409903,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1226435954",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6540034,\n   "lon": 37.6209084,\n   "opening_hours": "10:00-22:00"\n  },\n  {\n   "id": "node/1227893671",\n   "kind": "organisation",\n   "name": "Единая Россия - Отделение района Покровское-Стрешнево",\n   "office_type": "political_party",\n   "lat": 55.8330393,\n   "lon": 37.4480894,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1227895668",\n   "kind": "organisation",\n   "name": "Инженерная служба района Покровское-Стрешнево",\n   "office_type": "property_management",\n   "lat": 55.83423,\n   "lon": 37.44934,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1229873933",\n   "kind": "organisation",\n   "name": "ПАО «НК «Роснефть»",\n   "office_type": "company",\n   "lat": 55.7250042,\n   "lon": 37.6376504,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/1233280556",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6486306,\n   "lon": 37.7437736,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/1233359559",\n   "kind": "organisation",\n   "name": "ОДС № 3/2 ГУ «ИС Ховрино»",\n   "office_type": "company",\n   "lat": 55.87077,\n   "lon": 37.4892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1236178121",\n   "kind": "organisation",\n   "name": "ОДС-173",\n   "office_type": "property_management",\n   "lat": 55.6400467,\n   "lon": 37.5289281,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1236178145",\n   "kind": "organisation",\n   "name": "ГБУ \\"ИС района Коньково\\"",\n   "office_type": "yes",\n   "lat": 55.6324637,\n   "lon": 37.522823,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1244154259",\n   "kind": "organisation",\n   "name": "Центр занятости населения отдел Басманный ЦАО г. Москвы",\n   "office_type": "government",\n   "lat": 55.7586263,\n   "lon": 37.6520432,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1244160661",\n   "kind": "organisation",\n   "name": "Единый центр продаж ГУП г .Москвы «Дирекция гаражного строительства»",\n   "office_type": "estate_agent",\n   "lat": 55.7582991,\n   "lon": 37.6513164,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1245294415",\n   "kind": "organisation",\n   "name": "ДЕЗ района Сокольники",\n   "office_type": "property_management",\n   "lat": 55.7884225,\n   "lon": 37.6632646,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1250611645",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт проблем охраны труда",\n   "office_type": "research",\n   "lat": 55.7323265,\n   "lon": 37.5822896,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1252444913",\n   "kind": "organisation",\n   "name": "Управа района Дорогомилово",\n   "office_type": "government",\n   "lat": 55.7381945,\n   "lon": 37.5334659,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1253820852",\n   "kind": "organisation",\n   "name": "Голос Родины",\n   "office_type": "newspaper",\n   "lat": 55.7645148,\n   "lon": 37.644821,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1255469476",\n   "kind": "organisation",\n   "name": "3 СМ",\n   "office_type": "property_management",\n   "lat": 55.8714821,\n   "lon": 37.4854726,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1260491673",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "lawyer",\n   "lat": 55.8465457,\n   "lon": 37.4547527,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1277634699",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.8761998,\n   "lon": 37.7245057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1277634723",\n   "kind": "organisation",\n   "name": "Благотворительное общество многодетных и одиноких малообеспеченных семей «ЛОСЬ»",\n   "office_type": "association",\n   "lat": 55.8763099,\n   "lon": 37.7245284,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1277894653",\n   "kind": "organisation",\n   "name": "ЗАО «Мостоотряд №76»",\n   "office_type": "company",\n   "lat": 55.8766766,\n   "lon": 37.7235612,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1277894665",\n   "kind": "organisation",\n   "name": "СК «Строймост-1»",\n   "office_type": "company",\n   "lat": 55.8767126,\n   "lon": 37.7236011,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1277894686",\n   "kind": "organisation",\n   "name": "ООО «СУПРОТЕК-М»",\n   "office_type": "company",\n   "lat": 55.8766047,\n   "lon": 37.7234813,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1277894739",\n   "kind": "organisation",\n   "name": "«АнтейСтройИнвест»",\n   "office_type": "company",\n   "lat": 55.8766407,\n   "lon": 37.7235213,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1280821159",\n   "kind": "organisation",\n   "name": "Комстар",\n   "office_type": "company",\n   "lat": 55.7417438,\n   "lon": 37.5785585,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1282439374",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8865152,\n   "lon": 37.6624702,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/1287590471",\n   "kind": "organisation",\n   "name": "Окружная прокуратура СЗАО",\n   "office_type": "government",\n   "lat": 55.7968215,\n   "lon": 37.4532705,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1288276461",\n   "kind": "organisation",\n   "name": "Наша мама",\n   "office_type": "company",\n   "lat": 55.830457,\n   "lon": 37.4893649,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1288516431",\n   "kind": "organisation",\n   "name": "Институт развития информационного общества (ИРИО)",\n   "office_type": "company",\n   "lat": 55.7047402,\n   "lon": 37.6583898,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1288907571",\n   "kind": "organisation",\n   "name": "Управа района Щукино СЗАО",\n   "office_type": "government",\n   "lat": 55.7941152,\n   "lon": 37.4831307,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1301866949",\n   "kind": "organisation",\n   "name": "Architact.ru",\n   "office_type": "architect",\n   "lat": 55.7525835,\n   "lon": 37.6681198,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1313514295",\n   "kind": "organisation",\n   "name": "ЗАО «Банк Русский Стандарт»",\n   "office_type": "company",\n   "lat": 55.786404,\n   "lon": 37.7369305,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1325052029",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7801655,\n   "lon": 37.5903873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1326067090",\n   "kind": "organisation",\n   "name": "ГУП ДЕЗ",\n   "office_type": "property_management",\n   "lat": 55.8724604,\n   "lon": 37.6326062,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1333238867",\n   "kind": "organisation",\n   "name": "Хамовническая межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7214674,\n   "lon": 37.584667,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1335572334",\n   "kind": "organisation",\n   "name": "Управление социальной защиты населения района Южное Медведково",\n   "office_type": "government",\n   "lat": 55.8705331,\n   "lon": 37.6311214,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1338487651",\n   "kind": "organisation",\n   "name": "Инженерная служба района Проспект Вернадского",\n   "office_type": "property_management",\n   "lat": 55.6736801,\n   "lon": 37.507539,\n   "opening_hours": "Tu-Fr 09:00-21:00; Sa 09:00-15:00"\n  },\n  {\n   "id": "node/1338487657",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6755605,\n   "lon": 37.5053715,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1349413603",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8243713,\n   "lon": 37.5230509,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1351779698",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7720224,\n   "lon": 37.6359207,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/1351882529",\n   "kind": "organisation",\n   "name": "Государственная инспекция по маломерным судам МЧС России по г. Москве",\n   "office_type": "government",\n   "lat": 55.7618284,\n   "lon": 37.6496638,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1351882533",\n   "kind": "organisation",\n   "name": "Парамонов А. М.",\n   "office_type": "notary",\n   "lat": 55.7624367,\n   "lon": 37.6496956,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1355498349",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6411842,\n   "lon": 37.5226778,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1359360267",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8832742,\n   "lon": 37.6050156,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/1359360268",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8832949,\n   "lon": 37.6050644,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1361520255",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "lawyer",\n   "lat": 55.6524267,\n   "lon": 37.4813961,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1361990681",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8628522,\n   "lon": 37.6075803,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1368000668",\n   "kind": "organisation",\n   "name": "Институт экономики и комплексных проблем связи",\n   "office_type": "company",\n   "lat": 55.6662979,\n   "lon": 37.5759134,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1370762376",\n   "kind": "organisation",\n   "name": "Инком-Недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.8519168,\n   "lon": 37.4428805,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/1370993634",\n   "kind": "organisation",\n   "name": "ДЕЗ дмитровского района",\n   "office_type": "property_management",\n   "lat": 55.8948322,\n   "lon": 37.5424827,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1373831941",\n   "kind": "organisation",\n   "name": "комиссия по комплектованию детских садов",\n   "office_type": "educational_institution",\n   "lat": 55.6666121,\n   "lon": 37.5842668,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1389935959",\n   "kind": "organisation",\n   "name": "Учительская газета",\n   "office_type": "newspaper",\n   "lat": 55.7707481,\n   "lon": 37.6369225,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1401461423",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7158232,\n   "lon": 37.7976176,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/1408191312",\n   "kind": "organisation",\n   "name": "Альфастрахование",\n   "office_type": "insurance",\n   "lat": 55.7496962,\n   "lon": 37.7042511,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1409434494",\n   "kind": "organisation",\n   "name": "RU-Center",\n   "office_type": "it",\n   "lat": 55.8060427,\n   "lon": 37.5191794,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/1421168474",\n   "kind": "organisation",\n   "name": "Huawei Honor",\n   "office_type": "telecommunication",\n   "lat": 55.7716035,\n   "lon": 37.6381928,\n   "opening_hours": "Mo-Su 09:30-21:00"\n  },\n  {\n   "id": "node/1424361662",\n   "kind": "organisation",\n   "name": "ДЕЗ Алтуфьевский",\n   "office_type": "property_management",\n   "lat": 55.8854554,\n   "lon": 37.5960395,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1426845637",\n   "kind": "organisation",\n   "name": "Якорь",\n   "office_type": "insurance",\n   "lat": 55.7581425,\n   "lon": 37.6512205,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1432563047",\n   "kind": "organisation",\n   "name": "Hyundai Truck & Bus",\n   "office_type": "company",\n   "lat": 55.7616804,\n   "lon": 37.6583307,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1437413453",\n   "kind": "organisation",\n   "name": "ФГУП \\"Российские сети вещания и оповещения\\"",\n   "office_type": "government",\n   "lat": 55.778523,\n   "lon": 37.7156201,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1437413454",\n   "kind": "organisation",\n   "name": "Московский инструментальный завод",\n   "office_type": "company",\n   "lat": 55.7806241,\n   "lon": 37.7152121,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1447975689",\n   "kind": "organisation",\n   "name": "Диалог-Наука",\n   "office_type": "it",\n   "lat": 55.6826187,\n   "lon": 37.6247061,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1453496457",\n   "kind": "organisation",\n   "name": "Инком",\n   "office_type": "company",\n   "lat": 55.671767,\n   "lon": 37.5484884,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1460858010",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7240151,\n   "lon": 37.8252077,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/1466713733",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7242051,\n   "lon": 37.8247561,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/1480045964",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "lawyer",\n   "lat": 55.6286731,\n   "lon": 37.6637892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1490068287",\n   "kind": "organisation",\n   "name": "УФМС по району Вешняки",\n   "office_type": "government",\n   "lat": 55.7212012,\n   "lon": 37.8093114,\n   "opening_hours": "Mo 10:00-14:00,14:45-18:00; Tu,Th 10:00-14:00,14:45-20:00; We 10:00-14:00; Fr 09:00-14:00,14:45-16:45; Sa 09:00-13:00; Su off"\n  },\n  {\n   "id": "node/1490642882",\n   "kind": "organisation",\n   "name": "Мосэнергосбыт",\n   "office_type": "company",\n   "lat": 55.7842285,\n   "lon": 37.7389157,\n   "opening_hours": "Mo-Th 08:30-20:00; Fr 08:30-19:00"\n  },\n  {\n   "id": "node/1496633019",\n   "kind": "organisation",\n   "name": "Компания ЧА И",\n   "office_type": "company",\n   "lat": 55.721146,\n   "lon": 37.565152,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1507533146",\n   "kind": "organisation",\n   "name": "Дворец бракосочетания № 4",\n   "office_type": "government",\n   "lat": 55.7982304,\n   "lon": 37.5833737,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/1507817281",\n   "kind": "organisation",\n   "name": "Отдел развития персонала аптек Ригла",\n   "office_type": "company",\n   "lat": 55.8978437,\n   "lon": 37.5916299,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1509624523",\n   "kind": "organisation",\n   "name": "Твой юрист",\n   "office_type": "lawyer",\n   "lat": 55.7981222,\n   "lon": 37.5755881,\n   "opening_hours": "Mo-Sa 10:00-19:00"\n  },\n  {\n   "id": "node/1509782389",\n   "kind": "organisation",\n   "name": "Савеловский районный совет ветеранов войны и труда",\n   "office_type": "ngo",\n   "lat": 55.8020362,\n   "lon": 37.5669688,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1530991489",\n   "kind": "organisation",\n   "name": "Служба судебных приставов",\n   "office_type": "government",\n   "lat": 55.8266316,\n   "lon": 37.6587187,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1531266582",\n   "kind": "organisation",\n   "name": "Инспекция ФНС № 28",\n   "office_type": "government",\n   "lat": 55.6936475,\n   "lon": 37.5847639,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1541601266",\n   "kind": "organisation",\n   "name": "ИНКОМ-Недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.871035,\n   "lon": 37.658303,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1551010179",\n   "kind": "organisation",\n   "name": "Филиал Московская железная Дорога станция \\"Бескудниково\\"",\n   "office_type": "company",\n   "lat": 55.8817935,\n   "lon": 37.5701346,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1559711826",\n   "kind": "organisation",\n   "name": "Замоскворецкая межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7360212,\n   "lon": 37.6375115,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1559711844",\n   "kind": "organisation",\n   "name": "Следственный отдел по Замоскворецкому району г. Москвы",\n   "office_type": "government",\n   "lat": 55.7360024,\n   "lon": 37.6375372,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1561551444",\n   "kind": "organisation",\n   "name": "Бюро переводов \\"Профессионал\\"",\n   "office_type": "notary",\n   "lat": 55.7700621,\n   "lon": 37.6332875,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1573000627",\n   "kind": "organisation",\n   "name": "Управление специальных и технических мероприятий ГУВД Москвы",\n   "office_type": "government",\n   "lat": 55.7693564,\n   "lon": 37.6455181,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1579853078",\n   "kind": "organisation",\n   "name": "aitarget",\n   "office_type": "it",\n   "lat": 55.7007575,\n   "lon": 37.623464,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1581229801",\n   "kind": "organisation",\n   "name": "ВТБ Страхование",\n   "office_type": "insurance",\n   "lat": 55.7731602,\n   "lon": 37.6262908,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/1588042667",\n   "kind": "organisation",\n   "name": "Пенсионный фонд клиентская служба",\n   "office_type": "government",\n   "lat": 55.6278519,\n   "lon": 37.6154688,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1588066068",\n   "kind": "organisation",\n   "name": "Пенсионный фонд главное управление ПФР № 8",\n   "office_type": "government",\n   "lat": 55.7048795,\n   "lon": 37.6375683,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1593585128",\n   "kind": "organisation",\n   "name": "ДЕЗ \\"Люблино\\"",\n   "office_type": "property_management",\n   "lat": 55.684238,\n   "lon": 37.7418475,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1595306929",\n   "kind": "organisation",\n   "name": "Центр занятости ЮАО",\n   "office_type": "government",\n   "lat": 55.7044004,\n   "lon": 37.6651682,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1596071930",\n   "kind": "organisation",\n   "name": "Военкомат Бутырского района СВАО Москвы",\n   "office_type": "government",\n   "lat": 55.8606594,\n   "lon": 37.6030793,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1596071932",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8573635,\n   "lon": 37.5960757,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-18:00"\n  },\n  {\n   "id": "node/1596071937",\n   "kind": "organisation",\n   "name": "Отдел регистрации прав на недвижимость",\n   "office_type": "government",\n   "lat": 55.8592531,\n   "lon": 37.6020074,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1599296099",\n   "kind": "organisation",\n   "name": "Следственный комитет РФ",\n   "office_type": "government",\n   "lat": 55.7006759,\n   "lon": 37.7717448,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1599802680",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8084012,\n   "lon": 37.4632536,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1599802683",\n   "kind": "organisation",\n   "name": "Горячие туры",\n   "office_type": "travel_agent",\n   "lat": 55.808012,\n   "lon": 37.4642457,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1599802688",\n   "kind": "organisation",\n   "name": "МСК",\n   "office_type": "insurance",\n   "lat": 55.8083771,\n   "lon": 37.4632195,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1611514184",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6675357,\n   "lon": 37.552967,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/1626597681",\n   "kind": "organisation",\n   "name": "ОАО ТЦ \\"Геоинформатика\\"",\n   "office_type": "company",\n   "lat": 55.8037707,\n   "lon": 37.552553,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1628504132",\n   "kind": "organisation",\n   "name": "Леонова Д. Т.",\n   "office_type": "notary",\n   "lat": 55.766173,\n   "lon": 37.6460029,\n   "opening_hours": "Mo-Th 09:00-13:00,14:00-18:00; Fr 09:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/1628504229",\n   "kind": "organisation",\n   "name": "Нотариус Герасимова М.Д.",\n   "office_type": "notary",\n   "lat": 55.7714997,\n   "lon": 37.6371591,\n   "opening_hours": "Mo-Th 10:00-13:00,14:00-18:00; Fr 10:30-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/1629033574",\n   "kind": "organisation",\n   "name": "ОДС-13, Инженерная служба района Новогиреево, участок №13",\n   "office_type": "property_management",\n   "lat": 55.748799,\n   "lon": 37.818457,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1666923550",\n   "kind": "organisation",\n   "name": "Инженерная служба района Соколиная Гора",\n   "office_type": "property_management",\n   "lat": 55.7826289,\n   "lon": 37.711988,\n   "opening_hours": "Mo 09:00-18:00; Tu-Fr 09:00-20:00; Su 09:00-17:00"\n  },\n  {\n   "id": "node/1676369407",\n   "kind": "organisation",\n   "name": "Болгарский культурный институт",\n   "office_type": "educational_institution",\n   "lat": 55.7487905,\n   "lon": 37.6477769,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1684400406",\n   "kind": "organisation",\n   "name": "Департамент ОБДД МВД РФ",\n   "office_type": "government",\n   "lat": 55.7601506,\n   "lon": 37.6299204,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1684606595",\n   "kind": "organisation",\n   "name": "Типография №2",\n   "office_type": "printing_house",\n   "lat": 55.8155341,\n   "lon": 37.6377197,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1690110039",\n   "kind": "organisation",\n   "name": "Кашалот",\n   "office_type": "travel_agent",\n   "lat": 55.8744694,\n   "lon": 37.5882219,\n   "opening_hours": "Mo-Fr 09:30-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/1690132005",\n   "kind": "organisation",\n   "name": "Всё для вас",\n   "office_type": "newspaper",\n   "lat": 55.8708458,\n   "lon": 37.5902011,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1693972693",\n   "kind": "organisation",\n   "name": "Русское географическое общество (Московское отделение)",\n   "office_type": "ngo",\n   "lat": 55.7383062,\n   "lon": 37.6199456,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1695255932",\n   "kind": "organisation",\n   "name": "АО «Атомэнергопром»",\n   "office_type": "company",\n   "lat": 55.7380848,\n   "lon": 37.6195829,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1698543832",\n   "kind": "organisation",\n   "name": "Управа района Ясенево",\n   "office_type": "government",\n   "lat": 55.5992984,\n   "lon": 37.5385828,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1711235853",\n   "kind": "organisation",\n   "name": "Медицинская газета",\n   "office_type": "newspaper",\n   "lat": 55.7730337,\n   "lon": 37.6332835,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1711235866",\n   "kind": "organisation",\n   "name": "Территориальное управление Росимущества в городе Москве",\n   "office_type": "government",\n   "lat": 55.7714769,\n   "lon": 37.6469092,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1712912514",\n   "kind": "organisation",\n   "name": "ОАО Пьезо",\n   "office_type": "company",\n   "lat": 55.7890617,\n   "lon": 37.7088761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1716463761",\n   "kind": "organisation",\n   "name": "Министерство образования Московской области",\n   "office_type": "government",\n   "lat": 55.7330617,\n   "lon": 37.6183232,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1725834933",\n   "kind": "organisation",\n   "name": "ЕИРЦ района Ясенево",\n   "office_type": "government",\n   "lat": 55.6034195,\n   "lon": 37.5416976,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1726840442",\n   "kind": "organisation",\n   "name": "ОАО \\"ВНИПИэнергопром\\"",\n   "office_type": "research",\n   "lat": 55.7773148,\n   "lon": 37.7047097,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1727767320",\n   "kind": "organisation",\n   "name": "УФМС района Западное Дегунино",\n   "office_type": "government",\n   "lat": 55.8692259,\n   "lon": 37.5131017,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1730672601",\n   "kind": "organisation",\n   "name": "РЕСО страхование",\n   "office_type": "insurance",\n   "lat": 55.7713046,\n   "lon": 37.6322254,\n   "opening_hours": "Mo-Fr 11:00-20:00"\n  },\n  {\n   "id": "node/1734298025",\n   "kind": "organisation",\n   "name": "ИТАР-ТАСС автобаза",\n   "office_type": "company",\n   "lat": 55.8127985,\n   "lon": 37.6276015,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1734298029",\n   "kind": "organisation",\n   "name": "ИТАР-ТАСС типография",\n   "office_type": "newspaper",\n   "lat": 55.8128322,\n   "lon": 37.6276966,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1734298033",\n   "kind": "organisation",\n   "name": "Институт бизнеса и права",\n   "office_type": "company",\n   "lat": 55.8119946,\n   "lon": 37.6364622,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1734298037",\n   "kind": "organisation",\n   "name": "Мосводоканал специализированная автобаза",\n   "office_type": "company",\n   "lat": 55.8130343,\n   "lon": 37.6284114,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1734298039",\n   "kind": "organisation",\n   "name": "ОАО \\"Гипромез\\"",\n   "office_type": "research",\n   "lat": 55.8119497,\n   "lon": 37.6364524,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1734298043",\n   "kind": "organisation",\n   "name": "Росавтодор",\n   "office_type": "government",\n   "lat": 55.8105096,\n   "lon": 37.6346907,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1734298045",\n   "kind": "organisation",\n   "name": "Росреестр",\n   "office_type": "government",\n   "lat": 55.8118852,\n   "lon": 37.6364457,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1734298046",\n   "kind": "organisation",\n   "name": "Совинсервис",\n   "office_type": "company",\n   "lat": 55.8108625,\n   "lon": 37.6338437,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1736682447",\n   "kind": "organisation",\n   "name": "ОДС",\n   "office_type": "property_management",\n   "lat": 55.8681622,\n   "lon": 37.5356458,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1736682483",\n   "kind": "organisation",\n   "name": "Стройка",\n   "office_type": "company",\n   "lat": 55.8681847,\n   "lon": 37.5352193,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1736759500",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Бескудниковский",\n   "office_type": "government",\n   "lat": 55.8648349,\n   "lon": 37.5649363,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:00"\n  },\n  {\n   "id": "node/1736759501",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.864574,\n   "lon": 37.5651347,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1745421983",\n   "kind": "organisation",\n   "name": "Нотариус Демина",\n   "office_type": "notary",\n   "lat": 55.7735151,\n   "lon": 37.6501942,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1745422061",\n   "kind": "organisation",\n   "name": "ГУТА-Страхование",\n   "office_type": "insurance",\n   "lat": 55.7728776,\n   "lon": 37.6507534,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1746620020",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8492563,\n   "lon": 37.4410001,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/1770225058",\n   "kind": "organisation",\n   "name": "Единый информационно-расчетный центр Лосиноостровского района",\n   "office_type": "property_management",\n   "lat": 55.878704,\n   "lon": 37.693145,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1777917757",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.7688217,\n   "lon": 37.6384773,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1778780476",\n   "kind": "organisation",\n   "name": "Автофрамос",\n   "office_type": "company",\n   "lat": 55.7346676,\n   "lon": 37.662077,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1781424683",\n   "kind": "organisation",\n   "name": "Национальный телевизионный синдикат",\n   "office_type": "company",\n   "lat": 55.7690245,\n   "lon": 37.6391192,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1781695266",\n   "kind": "organisation",\n   "name": "Измайловский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7903043,\n   "lon": 37.7991665,\n   "opening_hours": "Tu-Sa 09:00-14:00,15:00-18:00"\n  },\n  {\n   "id": "node/1798692460",\n   "kind": "organisation",\n   "name": "Global Spirits",\n   "office_type": "insurance",\n   "lat": 55.7033466,\n   "lon": 37.6934392,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1798703907",\n   "kind": "organisation",\n   "name": "Посольство Саудовской Аравии",\n   "office_type": "diplomatic",\n   "lat": 55.7393075,\n   "lon": 37.583298,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1804170144",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7174728,\n   "lon": 37.782017,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1804212078",\n   "kind": "organisation",\n   "name": "УСЗН по району Перово",\n   "office_type": "government",\n   "lat": 55.7597589,\n   "lon": 37.7893807,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1804212079",\n   "kind": "organisation",\n   "name": "Прокуратура района Перово",\n   "office_type": "prosecutor",\n   "lat": 55.7596276,\n   "lon": 37.7886189,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1811735704",\n   "kind": "organisation",\n   "name": "ЗАО \\"Мосотделстрой-5\\"",\n   "office_type": "company",\n   "lat": 55.7464113,\n   "lon": 37.6473395,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1813892460",\n   "kind": "organisation",\n   "name": "UpJet",\n   "office_type": "company",\n   "lat": 55.771215,\n   "lon": 37.6327733,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1813974861",\n   "kind": "organisation",\n   "name": "АЛД Автомотив",\n   "office_type": "company",\n   "lat": 55.6935615,\n   "lon": 37.625028,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1821665221",\n   "kind": "organisation",\n   "name": "Abbott",\n   "office_type": "company",\n   "lat": 55.8213048,\n   "lon": 37.4979228,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1828670602",\n   "kind": "organisation",\n   "name": "Фомин В. А.",\n   "office_type": "lawyer",\n   "lat": 55.7684078,\n   "lon": 37.6257159,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1837797304",\n   "kind": "organisation",\n   "name": "Цезарь Сателлит",\n   "office_type": "company",\n   "lat": 55.7407358,\n   "lon": 37.5245694,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1837808551",\n   "kind": "organisation",\n   "name": "Volkswagen Group Finanz",\n   "office_type": "company",\n   "lat": 55.6562851,\n   "lon": 37.5274933,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/1837812744",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8854319,\n   "lon": 37.4468695,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1849508564",\n   "kind": "organisation",\n   "name": "Басманная межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7677438,\n   "lon": 37.6435297,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1849508568",\n   "kind": "organisation",\n   "name": "Московское общество защиты потребителей",\n   "office_type": "quango",\n   "lat": 55.7674675,\n   "lon": 37.644178,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1850660698",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.7834442,\n   "lon": 37.7201532,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/1868510870",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8646419,\n   "lon": 37.6835402,\n   "opening_hours": "Tu-Sa 10:00-20:00"\n  },\n  {\n   "id": "node/1879955449",\n   "kind": "organisation",\n   "name": "Культура",\n   "office_type": "newspaper",\n   "lat": 55.7903447,\n   "lon": 37.5898813,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1879955510",\n   "kind": "organisation",\n   "name": "Собеседник",\n   "office_type": "newspaper",\n   "lat": 55.7901459,\n   "lon": 37.5900741,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1891683162",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7399193,\n   "lon": 37.6142711,\n   "opening_hours": "Fr 09:00-17:00; Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/1892859898",\n   "kind": "organisation",\n   "name": "СУ-25 Мосасфальтстрой",\n   "office_type": "company",\n   "lat": 55.7670562,\n   "lon": 37.6693082,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1892859901",\n   "kind": "organisation",\n   "name": "СУ-37 СП ОАО \\"Дормост\\"",\n   "office_type": "company",\n   "lat": 55.7671029,\n   "lon": 37.669294,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1895352796",\n   "kind": "organisation",\n   "name": "ООО \\"НИИГАЗЭКОНОМИКА\\"",\n   "office_type": "company",\n   "lat": 55.766624,\n   "lon": 37.6651002,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1895352797",\n   "kind": "organisation",\n   "name": "Жилищник Басманного района ОДС №9",\n   "office_type": "property_management",\n   "lat": 55.7671222,\n   "lon": 37.6657133,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/1895369395",\n   "kind": "organisation",\n   "name": "Управление социальной защиты населения Басманного района города Москвы",\n   "office_type": "government",\n   "lat": 55.7654436,\n   "lon": 37.6637889,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1896793079",\n   "kind": "organisation",\n   "name": "Петрол Плюс",\n   "office_type": "company",\n   "lat": 55.8050775,\n   "lon": 37.5670803,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1897172513",\n   "kind": "organisation",\n   "name": "Управа района Раменки",\n   "office_type": "government",\n   "lat": 55.6944791,\n   "lon": 37.4980802,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1902113640",\n   "kind": "organisation",\n   "name": "Главное следственное управление ГУ МВД России по г. Москве",\n   "office_type": "government",\n   "lat": 55.7854739,\n   "lon": 37.5947769,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1909859232",\n   "kind": "organisation",\n   "name": "Ложкин Валерий Леонидович",\n   "office_type": "notary",\n   "lat": 55.7915559,\n   "lon": 37.6767924,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1909859236",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7915826,\n   "lon": 37.676941,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1913319856",\n   "kind": "organisation",\n   "name": "ОАО \\"Управление механизации строительства\\"",\n   "office_type": "company",\n   "lat": 55.8891903,\n   "lon": 37.6378043,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1913319858",\n   "kind": "organisation",\n   "name": "ОАО Завод световой рекламы Аркон",\n   "office_type": "company",\n   "lat": 55.8895461,\n   "lon": 37.634196,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1918857019",\n   "kind": "organisation",\n   "name": "SGS",\n   "office_type": "company",\n   "lat": 55.8720145,\n   "lon": 37.6717997,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1951722206",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8026389,\n   "lon": 37.5538713,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/1952548173",\n   "kind": "organisation",\n   "name": "Московская еврейская община",\n   "office_type": "foundation",\n   "lat": 55.7891166,\n   "lon": 37.606228,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1952556459",\n   "kind": "organisation",\n   "name": "МосЖилИнспекция",\n   "office_type": "government",\n   "lat": 55.8619065,\n   "lon": 37.5862343,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1955431347",\n   "kind": "organisation",\n   "name": "Арвал",\n   "office_type": "company",\n   "lat": 55.708845,\n   "lon": 37.654291,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1955575980",\n   "kind": "organisation",\n   "name": "Альянс",\n   "office_type": "insurance",\n   "lat": 55.693537,\n   "lon": 37.6245576,\n   "opening_hours": null\n  },\n  {\n   "id": "node/1962300300",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7703706,\n   "lon": 37.5965174,\n   "opening_hours": "\\"ежедневно с 9 до 22\\""\n  },\n  {\n   "id": "node/1966876890",\n   "kind": "organisation",\n   "name": "Мещанский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7753995,\n   "lon": 37.6328667,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/1997933514",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7857837,\n   "lon": 37.5955218,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/2006884253",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7880693,\n   "lon": 37.7490877,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2016194675",\n   "kind": "organisation",\n   "name": "Инком",\n   "office_type": "estate_agent",\n   "lat": 55.8674509,\n   "lon": 37.6717844,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/2016194715",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8686115,\n   "lon": 37.6691806,\n   "opening_hours": "09:00-18:00"\n  },\n  {\n   "id": "node/2019609770",\n   "kind": "organisation",\n   "name": "МАКС-М",\n   "office_type": "insurance",\n   "lat": 55.7652333,\n   "lon": 37.6222276,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2028517006",\n   "kind": "organisation",\n   "name": "РАТ",\n   "office_type": "company",\n   "lat": 55.7027559,\n   "lon": 37.691577,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2030668573",\n   "kind": "organisation",\n   "name": "Отдел приватизации",\n   "office_type": "government",\n   "lat": 55.8683878,\n   "lon": 37.6569103,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2032412951",\n   "kind": "organisation",\n   "name": "Лига Консультант",\n   "office_type": "company",\n   "lat": 55.780376,\n   "lon": 37.5909392,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2035584973",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8678787,\n   "lon": 37.6641223,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2035605211",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.8678229,\n   "lon": 37.6640913,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2035605212",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "translator",\n   "lat": 55.8678508,\n   "lon": 37.6641068,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2036568089",\n   "kind": "organisation",\n   "name": "Миэль",\n   "office_type": "estate_agent",\n   "lat": 55.8861048,\n   "lon": 37.6043508,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2045978863",\n   "kind": "organisation",\n   "name": "Агримодерн",\n   "office_type": "company",\n   "lat": 55.8304579,\n   "lon": 37.4479936,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2045983870",\n   "kind": "organisation",\n   "name": "Территориальный отдел квотирования СЗАО г. Москвы",\n   "office_type": "government",\n   "lat": 55.8318649,\n   "lon": 37.4494039,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2052899628",\n   "kind": "organisation",\n   "name": "Персонал-сервис",\n   "office_type": "employment_agency",\n   "lat": 55.8120694,\n   "lon": 37.6352509,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2052899629",\n   "kind": "organisation",\n   "name": "Страховое общество Россия",\n   "office_type": "insurance",\n   "lat": 55.8122952,\n   "lon": 37.6319266,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2052899631",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8118453,\n   "lon": 37.6352256,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2052899632",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8123459,\n   "lon": 37.631937,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2063627836",\n   "kind": "organisation",\n   "name": "Гагаринская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7039252,\n   "lon": 37.5759358,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2071843572",\n   "kind": "organisation",\n   "name": "Emirates",\n   "office_type": "company",\n   "lat": 55.7685526,\n   "lon": 37.6229967,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2073137685",\n   "kind": "organisation",\n   "name": "IFK Hotel Management",\n   "office_type": "company",\n   "lat": 55.7045136,\n   "lon": 37.584728,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/2073885832",\n   "kind": "organisation",\n   "name": "Dolby CIS Limited",\n   "office_type": "company",\n   "lat": 55.7475548,\n   "lon": 37.5812492,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2085387512",\n   "kind": "organisation",\n   "name": "юридическая консультация",\n   "office_type": "lawyer",\n   "lat": 55.663759,\n   "lon": 37.4758029,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2092870507",\n   "kind": "organisation",\n   "name": "ВСК",\n   "office_type": "insurance",\n   "lat": 55.8032055,\n   "lon": 37.582505,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2092870517",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8034934,\n   "lon": 37.5824511,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2101150834",\n   "kind": "organisation",\n   "name": "Отдел УФМС по г. Москве по району Текстильщики",\n   "office_type": "government",\n   "lat": 55.7163803,\n   "lon": 37.7448289,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2102020011",\n   "kind": "organisation",\n   "name": "Нотариус. Юридическая консультация",\n   "office_type": "notary",\n   "lat": 55.793049,\n   "lon": 37.798853,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2104339456",\n   "kind": "organisation",\n   "name": "ООО \\"МСД\\"",\n   "office_type": "company",\n   "lat": 55.7965129,\n   "lon": 37.781745,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2104739902",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7687247,\n   "lon": 37.6477892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2105789042",\n   "kind": "organisation",\n   "name": "УФМС \\"Богородское\\"",\n   "office_type": "government",\n   "lat": 55.8160026,\n   "lon": 37.6948896,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2108839041",\n   "kind": "organisation",\n   "name": "Жлобо Игорь Геннадьевич",\n   "office_type": "notary",\n   "lat": 55.8558332,\n   "lon": 37.6562093,\n   "opening_hours": "Mo-Fr 09:00-19:00; Fr 09:00-18:00; Sa 09:00-15:00; Su off"\n  },\n  {\n   "id": "node/2112514461",\n   "kind": "organisation",\n   "name": "Яндекс.Деньги",\n   "office_type": "company",\n   "lat": 55.734892,\n   "lon": 37.585825,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 09:00-18:00; Su off; PH off"\n  },\n  {\n   "id": "node/2123024147",\n   "kind": "organisation",\n   "name": "Прокуратура Московского метрополитена",\n   "office_type": "government",\n   "lat": 55.6575075,\n   "lon": 37.544841,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2128909791",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7511054,\n   "lon": 37.7840869,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/2134399864",\n   "kind": "organisation",\n   "name": "Популярная Механика",\n   "office_type": "newspaper",\n   "lat": 55.7982944,\n   "lon": 37.6046715,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2140237754",\n   "kind": "organisation",\n   "name": "Мосгорбти",\n   "office_type": "company",\n   "lat": 55.7974207,\n   "lon": 37.6035761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2140675006",\n   "kind": "organisation",\n   "name": "Дирекция по ремонту и техническому обслуживанию учреждений",\n   "office_type": "company",\n   "lat": 55.7695152,\n   "lon": 37.6288561,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2140675146",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7713344,\n   "lon": 37.6321431,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2141328696",\n   "kind": "organisation",\n   "name": "УФМС по району Марьина Роща",\n   "office_type": "government",\n   "lat": 55.7891244,\n   "lon": 37.6180103,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2145729980",\n   "kind": "organisation",\n   "name": "Главстрой",\n   "office_type": "company",\n   "lat": 55.7615313,\n   "lon": 37.6105685,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2145729994",\n   "kind": "organisation",\n   "name": "Российские сети вещания и оповещения",\n   "office_type": "company",\n   "lat": 55.762268,\n   "lon": 37.610142,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2145885693",\n   "kind": "organisation",\n   "name": "МетЛайф Алико",\n   "office_type": "insurance",\n   "lat": 55.8045274,\n   "lon": 37.5841715,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2162410813",\n   "kind": "organisation",\n   "name": "Народная футбольная лига",\n   "office_type": "association",\n   "lat": 55.7364187,\n   "lon": 37.4873323,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2167788777",\n   "kind": "organisation",\n   "name": "Военно-страховая компания",\n   "office_type": "company",\n   "lat": 55.763969,\n   "lon": 37.6284494,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2172106383",\n   "kind": "organisation",\n   "name": "Торрент фарма",\n   "office_type": "company",\n   "lat": 55.6689774,\n   "lon": 37.5663177,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2173372347",\n   "kind": "organisation",\n   "name": "ОДС №13",\n   "office_type": "property_management",\n   "lat": 55.8102204,\n   "lon": 37.7312466,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/2173372348",\n   "kind": "organisation",\n   "name": "ОДС №7",\n   "office_type": "property_management",\n   "lat": 55.8125685,\n   "lon": 37.7222252,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/2188527776",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7512893,\n   "lon": 37.8163389,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2196651533",\n   "kind": "organisation",\n   "name": "Аякс-Капиталъ",\n   "office_type": "accountant",\n   "lat": 55.7714689,\n   "lon": 37.6372132,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2199683992",\n   "kind": "organisation",\n   "name": "Центр социальной работы МВД России",\n   "office_type": "government",\n   "lat": 55.7543075,\n   "lon": 37.6034089,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2237958572",\n   "kind": "organisation",\n   "name": "Территориальный отдел Управления Роспотребнадзора",\n   "office_type": "government",\n   "lat": 55.6134792,\n   "lon": 37.7563057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2269076331",\n   "kind": "organisation",\n   "name": "ОДС № 1221",\n   "office_type": "property_management",\n   "lat": 55.6458709,\n   "lon": 37.5204266,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/2273605834",\n   "kind": "organisation",\n   "name": "Муниципалитет района Хорошёво-Мнёвники",\n   "office_type": "government",\n   "lat": 55.7863136,\n   "lon": 37.4803449,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2273605849",\n   "kind": "organisation",\n   "name": "Отдел регистрации прав на недвижимость по СЗАО г. Москвы",\n   "office_type": "government",\n   "lat": 55.7858327,\n   "lon": 37.4807925,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2273610642",\n   "kind": "organisation",\n   "name": "Хорошёвская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.786305,\n   "lon": 37.4808402,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2273616580",\n   "kind": "organisation",\n   "name": "Управление по работе с недвижимостью СЗАО г. Москвы",\n   "office_type": "government",\n   "lat": 55.7861336,\n   "lon": 37.4798992,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2277752698",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "estate_agent",\n   "lat": 55.7300015,\n   "lon": 37.6298168,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2287986729",\n   "kind": "organisation",\n   "name": "Управа района Москворечье-Сабурово",\n   "office_type": "government",\n   "lat": 55.645704,\n   "lon": 37.6575761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2290539043",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7056271,\n   "lon": 37.7597331,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2292506454",\n   "kind": "building",\n   "name": "Администрация",\n   "office_type": "yes",\n   "lat": 55.803277,\n   "lon": 37.5611857,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2297529545",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8021433,\n   "lon": 37.5828981,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2304274556",\n   "kind": "organisation",\n   "name": "Управление социальной защиты населения",\n   "office_type": "government",\n   "lat": 55.8655036,\n   "lon": 37.6788105,\n   "opening_hours": "Mo 11:00-20:00; We 09:00-18:00; Fr 09:00-16:45; 13:45-14:30 off"\n  },\n  {\n   "id": "node/2304275188",\n   "kind": "organisation",\n   "name": "Шахтспецпроект",\n   "office_type": "architect",\n   "lat": 55.8676747,\n   "lon": 37.6869549,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2305937874",\n   "kind": "organisation",\n   "name": "РЖД",\n   "office_type": "company",\n   "lat": 55.8037895,\n   "lon": 37.647121,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2305938092",\n   "kind": "organisation",\n   "name": "Первая страховая",\n   "office_type": "insurance",\n   "lat": 55.8041993,\n   "lon": 37.6428187,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2308476161",\n   "kind": "organisation",\n   "name": "Миэль",\n   "office_type": "estate_agent",\n   "lat": 55.803283,\n   "lon": 37.582233,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2310477769",\n   "kind": "organisation",\n   "name": "ООО «Комплексное обслуживание района»",\n   "office_type": "architect",\n   "lat": 55.8683212,\n   "lon": 37.6550763,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2317415001",\n   "kind": "organisation",\n   "name": "Посольство Тунисской республики",\n   "office_type": "diplomatic",\n   "lat": 55.7594823,\n   "lon": 37.5862593,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2320727422",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.685955,\n   "lon": 37.5345425,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2322527582",\n   "kind": "organisation",\n   "name": "ОДС 3",\n   "office_type": "property_management",\n   "lat": 55.874162,\n   "lon": 37.690289,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2322611721",\n   "kind": "organisation",\n   "name": "Управление социальной защиты населения",\n   "office_type": "government",\n   "lat": 55.8583104,\n   "lon": 37.6668159,\n   "opening_hours": "Mo,We 11:00-20:00; Fr 09:00-16:45; 13:45-14:30 off"\n  },\n  {\n   "id": "node/2326617536",\n   "kind": "organisation",\n   "name": "ООО \\"Сервис-Нафта\\"",\n   "office_type": "company",\n   "lat": 55.7578873,\n   "lon": 37.6459684,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2327255214",\n   "kind": "organisation",\n   "name": "ОУФМС «Лосиноостровский»",\n   "office_type": "government",\n   "lat": 55.8735574,\n   "lon": 37.6877377,\n   "opening_hours": "Mo 09:00-18:00; Tu 09:00-20:00; We 09:00-16:00; Th 09:00-20:00; Fr 09:00-16:45; Sa 09:00-18:00; Su off"\n  },\n  {\n   "id": "node/2328657311",\n   "kind": "organisation",\n   "name": "ГУТА Страхование",\n   "office_type": "insurance",\n   "lat": 55.7756507,\n   "lon": 37.6765368,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/2328735015",\n   "kind": "organisation",\n   "name": "ЗАО \\"ВОЛС-СЕРВИС\\"",\n   "office_type": "company",\n   "lat": 55.8809827,\n   "lon": 37.6788314,\n   "opening_hours": "Mo-Th 08:00-16:45; Fr 08:00-15:45"\n  },\n  {\n   "id": "node/2340717150",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8163747,\n   "lon": 37.6401409,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/2343644375",\n   "kind": "organisation",\n   "name": "Zurich",\n   "office_type": "insurance",\n   "lat": 55.7430405,\n   "lon": 37.5062203,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2349301394",\n   "kind": "organisation",\n   "name": "Yota",\n   "office_type": "telecommunication",\n   "lat": 55.7750456,\n   "lon": 37.5830733,\n   "opening_hours": "Mo-Su 08:00-22:00"\n  },\n  {\n   "id": "node/2349771979",\n   "kind": "organisation",\n   "name": "Бюро переводов \\"Трактат\\"",\n   "office_type": "translator",\n   "lat": 55.7126132,\n   "lon": 37.7240789,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/2349771982",\n   "kind": "organisation",\n   "name": "Бюро переводов \\"Трактат\\"",\n   "office_type": "translator",\n   "lat": 55.7759847,\n   "lon": 37.585296,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/2349771984",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.8593366,\n   "lon": 37.6168013,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/2350632807",\n   "kind": "organisation",\n   "name": "NSG",\n   "office_type": "company",\n   "lat": 55.7775697,\n   "lon": 37.7379557,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355331418",\n   "kind": "organisation",\n   "name": "Дети Марии",\n   "office_type": "ngo",\n   "lat": 55.7615739,\n   "lon": 37.6156356,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355373373",\n   "kind": "organisation",\n   "name": "БФ \\"Жизнь\\"",\n   "office_type": "ngo",\n   "lat": 55.8574689,\n   "lon": 37.5831359,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355373377",\n   "kind": "organisation",\n   "name": "Благотворительный фонд \\"Подари жизнь\\"",\n   "office_type": "ngo",\n   "lat": 55.722655,\n   "lon": 37.5670775,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355373381",\n   "kind": "organisation",\n   "name": "БФ \\"Подари надежду\\"",\n   "office_type": "ngo",\n   "lat": 55.749793,\n   "lon": 37.6835194,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355373385",\n   "kind": "organisation",\n   "name": "БФ \\"Созидание\\"",\n   "office_type": "ngo",\n   "lat": 55.7750675,\n   "lon": 37.8242806,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355373389",\n   "kind": "organisation",\n   "name": "Помоги.орг",\n   "office_type": "ngo",\n   "lat": 55.7735859,\n   "lon": 37.5992051,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355373408",\n   "kind": "organisation",\n   "name": "Центр равных возможностей \\"Вверх\\"",\n   "office_type": "ngo",\n   "lat": 55.7578532,\n   "lon": 37.6053595,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355412283",\n   "kind": "organisation",\n   "name": "Благотворительная организация «Нечаянная радость»",\n   "office_type": "ngo",\n   "lat": 55.7522883,\n   "lon": 37.7788806,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355462417",\n   "kind": "organisation",\n   "name": "БФ \\"Адреса милосердия\\"",\n   "office_type": "ngo",\n   "lat": 55.7645053,\n   "lon": 37.5796064,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355462418",\n   "kind": "organisation",\n   "name": "БФ \\"Галчонок\\"",\n   "office_type": "ngo",\n   "lat": 55.8413756,\n   "lon": 37.4498779,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355462419",\n   "kind": "organisation",\n   "name": "БФ \\"Старшие братья старшие сестры\\"",\n   "office_type": "ngo",\n   "lat": 55.7916639,\n   "lon": 37.6955817,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355462420",\n   "kind": "organisation",\n   "name": "Фонд помощи хосписам \\"Вера\\"",\n   "office_type": "ngo",\n   "lat": 55.7234777,\n   "lon": 37.5667524,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355462421",\n   "kind": "organisation",\n   "name": "Центр социокультурной анимации \\"Одухотворение\\"",\n   "office_type": "ngo",\n   "lat": 55.7164219,\n   "lon": 37.6109526,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355505491",\n   "kind": "organisation",\n   "name": "Международный социальный проект Натальи Фаустовой «Колыбельные для всей семьи»",\n   "office_type": "association",\n   "lat": 55.7604515,\n   "lon": 37.6807405,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355550314",\n   "kind": "organisation",\n   "name": "БФ \\"Детские сердца\\"",\n   "office_type": "ngo",\n   "lat": 55.7483279,\n   "lon": 37.5836067,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355571959",\n   "kind": "organisation",\n   "name": "Калин фонд",\n   "office_type": "ngo",\n   "lat": 55.707381,\n   "lon": 37.651043,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355581433",\n   "kind": "organisation",\n   "name": "БФ \\"Дети наши\\"",\n   "office_type": "ngo",\n   "lat": 55.7223853,\n   "lon": 37.65395,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355581434",\n   "kind": "organisation",\n   "name": "БФ \\"Димина мечта\\"",\n   "office_type": "ngo",\n   "lat": 55.8035335,\n   "lon": 37.454662,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355581436",\n   "kind": "organisation",\n   "name": "БФ \\"Подсолнух\\"",\n   "office_type": "ngo",\n   "lat": 55.7283534,\n   "lon": 37.5322694,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355581437",\n   "kind": "organisation",\n   "name": "Добровольческое движение Даниловцы",\n   "office_type": "ngo",\n   "lat": 55.7118875,\n   "lon": 37.6318323,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355581438",\n   "kind": "organisation",\n   "name": "Общественный фонд Смирнова",\n   "office_type": "ngo",\n   "lat": 55.6857911,\n   "lon": 37.5723905,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355581439",\n   "kind": "organisation",\n   "name": "Предание.ру",\n   "office_type": "ngo",\n   "lat": 55.7271077,\n   "lon": 37.6098517,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355582239",\n   "kind": "organisation",\n   "name": "Благотворительный фонд помощи взрослым «Живой»",\n   "office_type": "ngo",\n   "lat": 55.7435069,\n   "lon": 37.5592118,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355585988",\n   "kind": "organisation",\n   "name": "Благотворительный фонд «Добросердие»",\n   "office_type": "ngo",\n   "lat": 55.7558,\n   "lon": 37.596031,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355586392",\n   "kind": "organisation",\n   "name": "РОСТ",\n   "office_type": "ngo",\n   "lat": 55.6603474,\n   "lon": 37.617567,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355591412",\n   "kind": "organisation",\n   "name": "Благотворительнчй фонд помощи душевнобольным",\n   "office_type": "ngo",\n   "lat": 55.7026208,\n   "lon": 37.5667224,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355591694",\n   "kind": "organisation",\n   "name": "Благотворительный фонд «Большая перемена»",\n   "office_type": "ngo",\n   "lat": 55.660387,\n   "lon": 37.617659,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355592560",\n   "kind": "organisation",\n   "name": "Благотворительное сообщество переводчиков «Настоящее будущее»",\n   "office_type": "ngo",\n   "lat": 55.7478434,\n   "lon": 37.5924404,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355593207",\n   "kind": "organisation",\n   "name": "Благотворительный фонд \\"Время добра\\"",\n   "office_type": "ngo",\n   "lat": 55.7958694,\n   "lon": 37.5452347,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355593208",\n   "kind": "organisation",\n   "name": "Благотворительный фонд \\"Красота-детям\\"",\n   "office_type": "ngo",\n   "lat": 55.7046893,\n   "lon": 37.65827,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355593209",\n   "kind": "organisation",\n   "name": "БФ \\"Разные дети\\"",\n   "office_type": "ngo",\n   "lat": 55.7108645,\n   "lon": 37.7597047,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2355593478",\n   "kind": "organisation",\n   "name": "Благотворитльный фонд «Кто если не я?»",\n   "office_type": "ngo",\n   "lat": 55.7757987,\n   "lon": 37.6120426,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2356983194",\n   "kind": "organisation",\n   "name": "Федеральное агентство морского и речного транспорта",\n   "office_type": "government",\n   "lat": 55.761037,\n   "lon": 37.618552,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2358370634",\n   "kind": "organisation",\n   "name": "Транспортное агентство московского метрополитена",\n   "office_type": "logistics",\n   "lat": 55.7684119,\n   "lon": 37.6479266,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2362367787",\n   "kind": "organisation",\n   "name": "Гостехнадзор",\n   "office_type": "government",\n   "lat": 55.6802788,\n   "lon": 37.6000849,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2381366014",\n   "kind": "organisation",\n   "name": "Ростелеком",\n   "office_type": "telecommunication",\n   "lat": 55.7440965,\n   "lon": 37.5629169,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/2381366029",\n   "kind": "organisation",\n   "name": "Ростелеком",\n   "office_type": "telecommunication",\n   "lat": 55.7924342,\n   "lon": 37.6073835,\n   "opening_hours": "09:00-20:30"\n  },\n  {\n   "id": "node/2388112325",\n   "kind": "organisation",\n   "name": "Журнал «Retail&Loyalty»",\n   "office_type": "newspaper",\n   "lat": 55.6746854,\n   "lon": 37.5823588,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2389116207",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Люблинского района",\n   "office_type": "government",\n   "lat": 55.6593191,\n   "lon": 37.7538817,\n   "opening_hours": "Mo,We 10:00-18:00"\n  },\n  {\n   "id": "node/2397716862",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8218723,\n   "lon": 37.5725639,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/2400143242",\n   "kind": "organisation",\n   "name": "Федеральное бюджетное учреждение «Регистр сертификации на федеральном железнодорожном транспорте»",\n   "office_type": "government",\n   "lat": 55.8038849,\n   "lon": 37.6482398,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2402539684",\n   "kind": "organisation",\n   "name": "Корнеева Наталья Васильевна",\n   "office_type": "notary",\n   "lat": 55.7588364,\n   "lon": 37.6458316,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/2409792999",\n   "kind": "organisation",\n   "name": "Natalie tours",\n   "office_type": "travel_agency",\n   "lat": 55.7463693,\n   "lon": 37.6548924,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa 10:00-20:00; Su 10:00-18:00"\n  },\n  {\n   "id": "node/2409793048",\n   "kind": "organisation",\n   "name": "Петровское",\n   "office_type": "insurance",\n   "lat": 55.7464372,\n   "lon": 37.6548492,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/2420951515",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7697605,\n   "lon": 37.65583,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2425876063",\n   "kind": "organisation",\n   "name": "Отдел регистрации прав на недвижимость по ЦАО Москвы",\n   "office_type": "government",\n   "lat": 55.7694461,\n   "lon": 37.6561546,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2430091899",\n   "kind": "organisation",\n   "name": "Инженерная служба района Проспект Вернадского",\n   "office_type": "property_management",\n   "lat": 55.6751945,\n   "lon": 37.4955866,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2430101144",\n   "kind": "organisation",\n   "name": "ООО «Ремонтно-строительное предприятие»",\n   "office_type": "company",\n   "lat": 55.6754308,\n   "lon": 37.4954042,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2431522994",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт бетона и железобетона",\n   "office_type": "research",\n   "lat": 55.7217228,\n   "lon": 37.7750849,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2431522997",\n   "kind": "organisation",\n   "name": "НИИОСП имени Н. М. Герсеванова",\n   "office_type": "company",\n   "lat": 55.7222616,\n   "lon": 37.7735414,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2431523000",\n   "kind": "organisation",\n   "name": "НИЦ \\"Строительство\\"",\n   "office_type": "company",\n   "lat": 55.7208081,\n   "lon": 37.7775679,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2431523004",\n   "kind": "organisation",\n   "name": "ЦНИИСК им. Кучеренко",\n   "office_type": "company",\n   "lat": 55.7207507,\n   "lon": 37.777732,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2432212418",\n   "kind": "organisation",\n   "name": "Визовый центр Швеции и Норвегии",\n   "office_type": "visa",\n   "lat": 55.7200479,\n   "lon": 37.6100701,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/2432846953",\n   "kind": "organisation",\n   "name": "Мелодия",\n   "office_type": "company",\n   "lat": 55.7707904,\n   "lon": 37.4658813,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2434364589",\n   "kind": "organisation",\n   "name": "Двитекс",\n   "office_type": "lawyer",\n   "lat": 55.7388274,\n   "lon": 37.6141638,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2434364629",\n   "kind": "organisation",\n   "name": "Семенова Е. В.",\n   "office_type": "lawyer",\n   "lat": 55.7121082,\n   "lon": 37.7952219,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2434364645",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7154754,\n   "lon": 37.7961282,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2434364749",\n   "kind": "organisation",\n   "name": "нотариус Микаелян Л.С.",\n   "office_type": "lawyer",\n   "lat": 55.7766799,\n   "lon": 37.5547136,\n   "opening_hours": "Mo-Th 09:00-17:00;Fr 09:00-16:00"\n  },\n  {\n   "id": "node/2441356765",\n   "kind": "organisation",\n   "name": "Отделение УФМС по району Вешняки",\n   "office_type": "government",\n   "lat": 55.7345483,\n   "lon": 37.833922,\n   "opening_hours": "Mo 09:00-14:00,14:45-18:00; Tu,Th 11:00-14:00,14:45-20:00; We 09:00-13:00; Fr 09:00-14:00,14:45-16:45; Sa 09:00-16:00"\n  },\n  {\n   "id": "node/2441356766",\n   "kind": "organisation",\n   "name": "Отделение УФМС по району Новогиреево",\n   "office_type": "government",\n   "lat": 55.7486202,\n   "lon": 37.8075244,\n   "opening_hours": "Mo 09:00-14:00,14:45-18:00; Tu,Th 11:00-14:00,14:45-20:00; We 09:00-13:00; Fr 09:00-14:00,14:45-16:45; Sa 09:00-16:00"\n  },\n  {\n   "id": "node/2447120129",\n   "kind": "organisation",\n   "name": "Управа района Лианозово",\n   "office_type": "government",\n   "lat": 55.8967953,\n   "lon": 37.5857355,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2450396047",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Ивановское",\n   "office_type": "government",\n   "lat": 55.7642343,\n   "lon": 37.8333004,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2450396048",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Новогиреево",\n   "office_type": "government",\n   "lat": 55.7487114,\n   "lon": 37.8075295,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/2453677859",\n   "kind": "organisation",\n   "name": "Конечная",\n   "office_type": "company",\n   "lat": 55.6280653,\n   "lon": 37.6789578,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2457778015",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7642713,\n   "lon": 37.7032428,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/2457778017",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8119159,\n   "lon": 37.6364513,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2459146866",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8185109,\n   "lon": 37.5738458,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/2468990192",\n   "kind": "organisation",\n   "name": "ЦентрТрубПласт",\n   "office_type": "company",\n   "lat": 55.8952366,\n   "lon": 37.5224882,\n   "opening_hours": "Mo-Fr 08:00-18:00; Sa 09:00-15:00"\n  },\n  {\n   "id": "node/2473512083",\n   "kind": "organisation",\n   "name": "Посольство Молдовы",\n   "office_type": "diplomatic",\n   "lat": 55.7617972,\n   "lon": 37.6228668,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/2474342157",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7550509,\n   "lon": 37.6343407,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2477516812",\n   "kind": "organisation",\n   "name": "Визовый центр Испании BLS international",\n   "office_type": "visa",\n   "lat": 55.7278757,\n   "lon": 37.6140594,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/2479828026",\n   "kind": "organisation",\n   "name": "Интернет-магазин \\"CigarSmoker.ru\\"",\n   "office_type": "yes",\n   "lat": 55.7063059,\n   "lon": 37.5965269,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2480091072",\n   "kind": "organisation",\n   "name": "Секретариат Юго-западного викариатства города Москвы",\n   "office_type": "church",\n   "lat": 55.7119481,\n   "lon": 37.5744215,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2483431885",\n   "kind": "organisation",\n   "name": "Первая страховая компания",\n   "office_type": "insurance",\n   "lat": 55.764905,\n   "lon": 37.6087057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2483431886",\n   "kind": "organisation",\n   "name": "Ларт недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.7649334,\n   "lon": 37.608679,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2485412649",\n   "kind": "organisation",\n   "name": "Siemens",\n   "office_type": "yes",\n   "lat": 55.7134173,\n   "lon": 37.6321926,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2494183401",\n   "kind": "organisation",\n   "name": "Деловые Линии",\n   "office_type": "company",\n   "lat": 55.619014,\n   "lon": 37.8022384,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2506826091",\n   "kind": "organisation",\n   "name": "НПФ Благосостояние",\n   "office_type": "insurance",\n   "lat": 55.7687047,\n   "lon": 37.6063112,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2506826094",\n   "kind": "organisation",\n   "name": "Энерготехкомплект",\n   "office_type": "company",\n   "lat": 55.7683989,\n   "lon": 37.6075272,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2510079148",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7587681,\n   "lon": 37.6388306,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2512027275",\n   "kind": "organisation",\n   "name": "Детский фонд",\n   "office_type": "association",\n   "lat": 55.7591513,\n   "lon": 37.6382049,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2517166044",\n   "kind": "organisation",\n   "name": "Югория",\n   "office_type": "insurance",\n   "lat": 55.7886829,\n   "lon": 37.5899063,\n   "opening_hours": "Mo-Fr 09:00-18:00;Sa 10:00-15:00"\n  },\n  {\n   "id": "node/2551430141",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Восточное Дегунино",\n   "office_type": "government",\n   "lat": 55.8888369,\n   "lon": 37.55905,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/2551430148",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Свиблово",\n   "office_type": "government",\n   "lat": 55.8578342,\n   "lon": 37.6445931,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-16:00"\n  },\n  {\n   "id": "node/2551430152",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Тимирязевский",\n   "office_type": "government",\n   "lat": 55.8106287,\n   "lon": 37.5710378,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/2555369745",\n   "kind": "organisation",\n   "name": "Coral Travel",\n   "office_type": "travel_agent",\n   "lat": 55.7468522,\n   "lon": 37.545697,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/2555555552",\n   "kind": "organisation",\n   "name": "Пегас",\n   "office_type": "employment_agency",\n   "lat": 55.7480777,\n   "lon": 37.5441689,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2558642748",\n   "kind": "organisation",\n   "name": "ООО «Независимая оценка»",\n   "office_type": "company",\n   "lat": 55.6424845,\n   "lon": 37.676669,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2560868921",\n   "kind": "organisation",\n   "name": "Отделение Пенсионного фонда РФ по г. Москве и Московской области",\n   "office_type": "government",\n   "lat": 55.7605442,\n   "lon": 37.6018125,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2584297887",\n   "kind": "organisation",\n   "name": "Приемная Государственной Думы",\n   "office_type": "government",\n   "lat": 55.7528642,\n   "lon": 37.6109659,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2606638958",\n   "kind": "organisation",\n   "name": "Дом приёмов Правительства России",\n   "office_type": "government",\n   "lat": 55.7531654,\n   "lon": 37.603626,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2610722589",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг СЗАО",\n   "office_type": "government",\n   "lat": 55.8506103,\n   "lon": 37.4457767,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/2612117390",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8505554,\n   "lon": 37.444864,\n   "opening_hours": "10:00-22:00"\n  },\n  {\n   "id": "node/2616090383",\n   "kind": "organisation",\n   "name": "АТИ ЗАО",\n   "office_type": "foundation",\n   "lat": 55.7388271,\n   "lon": 37.4749337,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2625306937",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7641868,\n   "lon": 37.6058508,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/2632359530",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Тропарёво-Никулино",\n   "office_type": "government",\n   "lat": 55.6651996,\n   "lon": 37.4813873,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2632370131",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Лефортово",\n   "office_type": "government",\n   "lat": 55.7530435,\n   "lon": 37.7011605,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633398244",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Нагатино-Садовники",\n   "office_type": "government",\n   "lat": 55.6791576,\n   "lon": 37.6301206,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633412453",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Марьино",\n   "office_type": "government",\n   "lat": 55.6745165,\n   "lon": 37.7605867,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633420882",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Соколиная Гора",\n   "office_type": "government",\n   "lat": 55.782525,\n   "lon": 37.707396,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/2633424055",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Южнопортовый",\n   "office_type": "government",\n   "lat": 55.7312589,\n   "lon": 37.6663036,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633440868",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Фили-Давыдково",\n   "office_type": "government",\n   "lat": 55.7203709,\n   "lon": 37.4631099,\n   "opening_hours": "Mo-Su 08:00-22:00"\n  },\n  {\n   "id": "node/2633464881",\n   "kind": "organisation",\n   "name": "Vita Group",\n   "office_type": "company",\n   "lat": 55.7952518,\n   "lon": 37.6082295,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2633529765",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Ростокино",\n   "office_type": "government",\n   "lat": 55.8315185,\n   "lon": 37.6604691,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633607221",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Гагаринский",\n   "office_type": "government",\n   "lat": 55.6796939,\n   "lon": 37.5463971,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633671013",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Ломоносовский",\n   "office_type": "government",\n   "lat": 55.6761537,\n   "lon": 37.5364291,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633712219",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Текстильщики",\n   "office_type": "government",\n   "lat": 55.7108834,\n   "lon": 37.7478766,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633734045",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Братеево",\n   "office_type": "government",\n   "lat": 55.6359721,\n   "lon": 37.7523647,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633773572",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Измайлово",\n   "office_type": "government",\n   "lat": 55.7986071,\n   "lon": 37.8027238,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2633790475",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Капотня",\n   "office_type": "government",\n   "lat": 55.6382073,\n   "lon": 37.8008364,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2634797373",\n   "kind": "organisation",\n   "name": "МФЦ районов Нагорный и Нагатино-Садовники",\n   "office_type": "government",\n   "lat": 55.6790416,\n   "lon": 37.6301649,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2634802603",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Нижегородский",\n   "office_type": "government",\n   "lat": 55.7529806,\n   "lon": 37.701519,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2634814358",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Печатники",\n   "office_type": "government",\n   "lat": 55.6674824,\n   "lon": 37.7196187,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2634817379",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Тёплый Стан",\n   "office_type": "government",\n   "lat": 55.6343651,\n   "lon": 37.4743574,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2634845203",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Бескудниково",\n   "office_type": "government",\n   "lat": 55.8890978,\n   "lon": 37.5595925,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/2634861848",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Северное Измайлово",\n   "office_type": "government",\n   "lat": 55.8060749,\n   "lon": 37.8235274,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2634866805",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Замоскворечье",\n   "office_type": "government",\n   "lat": 55.7356188,\n   "lon": 37.6357204,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2634967457",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Сокол",\n   "office_type": "government",\n   "lat": 55.7971422,\n   "lon": 37.5183491,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2634981875",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Рязанский",\n   "office_type": "government",\n   "lat": 55.7159021,\n   "lon": 37.7847985,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2634995977",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Люблино",\n   "office_type": "government",\n   "lat": 55.68947,\n   "lon": 37.7370062,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:45"\n  },\n  {\n   "id": "node/2635212206",\n   "kind": "organisation",\n   "name": "Вернадский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.6650028,\n   "lon": 37.5083537,\n   "opening_hours": "Tu-Sa 9:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/2635219088",\n   "kind": "organisation",\n   "name": "Гагаринский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7013656,\n   "lon": 37.5671675,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/2639373866",\n   "kind": "organisation",\n   "name": "Кутузовский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7445308,\n   "lon": 37.5483052,\n   "opening_hours": "Tu-Sa 09:00-18:00"\n  },\n  {\n   "id": "node/2639398936",\n   "kind": "organisation",\n   "name": "Люблинский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7019108,\n   "lon": 37.7462779,\n   "opening_hours": "Tu-Sa 09:00-18:00"\n  },\n  {\n   "id": "node/2639560378",\n   "kind": "organisation",\n   "name": "Таганский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7389276,\n   "lon": 37.6702083,\n   "opening_hours": "Tu-Sa 09:00-18:00"\n  },\n  {\n   "id": "node/2639575390",\n   "kind": "organisation",\n   "name": "Тверской отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7752297,\n   "lon": 37.5912254,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/2639636398",\n   "kind": "organisation",\n   "name": "Хамовнический отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7400467,\n   "lon": 37.5739579,\n   "opening_hours": "Tu-Sa 09:00-18:00"\n  },\n  {\n   "id": "node/2639638836",\n   "kind": "organisation",\n   "name": "Хорошевский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7784849,\n   "lon": 37.4762351,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/2639644164",\n   "kind": "organisation",\n   "name": "Замоскворецкий отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.741189,\n   "lon": 37.6365747,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/2639652106",\n   "kind": "organisation",\n   "name": "Объединенный архив ЗАГС",\n   "office_type": "government",\n   "lat": 55.7658603,\n   "lon": 37.6449401,\n   "opening_hours": "Mo-Fr 09:00-12:45,14:00-17:30"\n  },\n  {\n   "id": "node/2639749381",\n   "kind": "organisation",\n   "name": "Дворец бракосочетания № 5",\n   "office_type": "government",\n   "lat": 55.7945682,\n   "lon": 37.749294,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/2640818655",\n   "kind": "organisation",\n   "name": "ООО \\"АС Эксперт\\"",\n   "office_type": "company",\n   "lat": 55.8333153,\n   "lon": 37.4959409,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2641981276",\n   "kind": "organisation",\n   "name": "Пестрецова Т. А.",\n   "office_type": "notary",\n   "lat": 55.74907,\n   "lon": 37.5822812,\n   "opening_hours": "Mo-Fr 10:00-17:15"\n  },\n  {\n   "id": "node/2651488099",\n   "kind": "organisation",\n   "name": "Анкор",\n   "office_type": "employment_agency",\n   "lat": 55.74626,\n   "lon": 37.6823863,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2670169698",\n   "kind": "organisation",\n   "name": "Визаход",\n   "office_type": "company",\n   "lat": 55.7736922,\n   "lon": 37.5771382,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2672659872",\n   "kind": "organisation",\n   "name": "ИнфраМенеджер",\n   "office_type": "it",\n   "lat": 55.6941885,\n   "lon": 37.5910391,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2673526388",\n   "kind": "organisation",\n   "name": "Ветеран",\n   "office_type": "newspaper",\n   "lat": 55.7746099,\n   "lon": 37.6291595,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2674886924",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7730511,\n   "lon": 37.7063033,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2675528372",\n   "kind": "organisation",\n   "name": "Пенсионный Фонд РФ Клиентская служба Новогиреево, Перово, Вешняки",\n   "office_type": "government",\n   "lat": 55.7499848,\n   "lon": 37.8011402,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/2675537446",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Перово",\n   "office_type": "government",\n   "lat": 55.7606891,\n   "lon": 37.7884372,\n   "opening_hours": "08:00-20:00"\n  },\n  {\n   "id": "node/2678465424",\n   "kind": "organisation",\n   "name": "Московская академическая коллегия адвокатов",\n   "office_type": "lawyer",\n   "lat": 55.7094564,\n   "lon": 37.6248285,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2678480876",\n   "kind": "organisation",\n   "name": "ОАО \\"Концерн радиостроения \\"Вега\\"",\n   "office_type": "company",\n   "lat": 55.7396482,\n   "lon": 37.5296504,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2693211447",\n   "kind": "organisation",\n   "name": "Региональная общественная организация охотников и рыболовов",\n   "office_type": "ngo",\n   "lat": 55.7587433,\n   "lon": 37.6465285,\n   "opening_hours": "Tu-Th 14:00-19:00"\n  },\n  {\n   "id": "node/2705160043",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Алтуфьевского района",\n   "office_type": "government",\n   "lat": 55.8814221,\n   "lon": 37.5888371,\n   "opening_hours": "08:00-20:00"\n  },\n  {\n   "id": "node/2715867508",\n   "kind": "organisation",\n   "name": "Север столицы",\n   "office_type": "newspaper",\n   "lat": 55.8009534,\n   "lon": 37.5169052,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2715867510",\n   "kind": "organisation",\n   "name": "HOMEWAY недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.8572175,\n   "lon": 37.5679894,\n   "opening_hours": "Mo-Sa 09:00-21:00"\n  },\n  {\n   "id": "node/2734701441",\n   "kind": "organisation",\n   "name": "Главное управление по вопросам миграции МВД РФ",\n   "office_type": "government",\n   "lat": 55.7604158,\n   "lon": 37.6440871,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2749487425",\n   "kind": "organisation",\n   "name": "Институт социально-экономических проблем народонаселения РАН",\n   "office_type": "research",\n   "lat": 55.6764466,\n   "lon": 37.5709749,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2749570354",\n   "kind": "organisation",\n   "name": "Business Solutions Adviser",\n   "office_type": "it",\n   "lat": 55.6293989,\n   "lon": 37.6211668,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2751628534",\n   "kind": "organisation",\n   "name": "ПФР Отдел Ярославский",\n   "office_type": "government",\n   "lat": 55.8732681,\n   "lon": 37.7046752,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:45; Sa,Su off"\n  },\n  {\n   "id": "node/2756094419",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Даниловский",\n   "office_type": "government",\n   "lat": 55.712912,\n   "lon": 37.6130602,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/2786384890",\n   "kind": "organisation",\n   "name": "Визовый центр Италии",\n   "office_type": "visa",\n   "lat": 55.7420284,\n   "lon": 37.6185801,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2794735938",\n   "kind": "organisation",\n   "name": "Байкал-Сервис, терминал \\"Дзержинский\\"",\n   "office_type": "transport",\n   "lat": 55.6404537,\n   "lon": 37.8247665,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2794789321",\n   "kind": "organisation",\n   "name": "ПЭК",\n   "office_type": "logistics",\n   "lat": 55.7174102,\n   "lon": 37.7577712,\n   "opening_hours": "Mo 08:00-24:00;Sa 00:00-16:00;Su 10:00-16:00;Tu-Fr 00:00-24:00"\n  },\n  {\n   "id": "node/2799251637",\n   "kind": "organisation",\n   "name": "Avon",\n   "office_type": "company",\n   "lat": 55.7307233,\n   "lon": 37.5763435,\n   "opening_hours": "Mo-Sa 09:00-21:00; Su 10:00-18:00"\n  },\n  {\n   "id": "node/2802164768",\n   "kind": "organisation",\n   "name": "Управление социальной защиты населения",\n   "office_type": "government",\n   "lat": 55.8061604,\n   "lon": 37.5826094,\n   "opening_hours": "Mo 11:00-13:45,14:30-20:00; We 09:00-13:45,14:30-18:00; Fr 09:00-13:45,14:30-16:45; Tu,Th 09:00-13:45,14:30-18:00"\n  },\n  {\n   "id": "node/2804034329",\n   "kind": "organisation",\n   "name": "миссия Мальтийского Ордена",\n   "office_type": "diplomatic",\n   "lat": 55.7485066,\n   "lon": 37.6085591,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2810086111",\n   "kind": "organisation",\n   "name": "Ассоциация активной помощи престарелым и детям",\n   "office_type": "ngo",\n   "lat": 55.7390335,\n   "lon": 37.5155871,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2810086113",\n   "kind": "organisation",\n   "name": "ООО \\"Интикома\\"",\n   "office_type": "company",\n   "lat": 55.7255963,\n   "lon": 37.5850882,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2815257266",\n   "kind": "organisation",\n   "name": "Юго-Восточное ТБТИ",\n   "office_type": "company",\n   "lat": 55.7495238,\n   "lon": 37.7262746,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2815357449",\n   "kind": "organisation",\n   "name": "Клава",\n   "office_type": "company",\n   "lat": 55.7511978,\n   "lon": 37.733431,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2820236144",\n   "kind": "organisation",\n   "name": "УК ЖилКомСервис",\n   "office_type": "property_management",\n   "lat": 55.7476304,\n   "lon": 37.8173037,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2820236145",\n   "kind": "organisation",\n   "name": "ГУП ДЕЗ района Новогиреево",\n   "office_type": "property_management",\n   "lat": 55.7486878,\n   "lon": 37.8193246,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2820260562",\n   "kind": "organisation",\n   "name": "ОДС-14, Инженерная служба района Новогиреево, участок №39",\n   "office_type": "property_management",\n   "lat": 55.7493961,\n   "lon": 37.8062413,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2820260563",\n   "kind": "organisation",\n   "name": "ОДС-12, Инженерная служба района Новогиреево, участок №43",\n   "office_type": "property_management",\n   "lat": 55.7536568,\n   "lon": 37.8222606,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2820260564",\n   "kind": "organisation",\n   "name": "ОДС-9, Инженерная служба района Новогиреево, участок №39",\n   "office_type": "property_management",\n   "lat": 55.7512774,\n   "lon": 37.8107635,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2820260565",\n   "kind": "organisation",\n   "name": "ОДС-8, Инженерная служба района Новогиреево, участок №38",\n   "office_type": "property_management",\n   "lat": 55.7494047,\n   "lon": 37.7908054,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2820260566",\n   "kind": "organisation",\n   "name": "ОДС-8, Инженерная служба района Новогиреево, участок №38",\n   "office_type": "property_management",\n   "lat": 55.7421366,\n   "lon": 37.7832693,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2830320020",\n   "kind": "organisation",\n   "name": "Центр перевода",\n   "office_type": "notary",\n   "lat": 55.764702,\n   "lon": 37.634318,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2830320021",\n   "kind": "organisation",\n   "name": "Департамент спорта города Москвы",\n   "office_type": "government",\n   "lat": 55.7644232,\n   "lon": 37.6325237,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2853492976",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8664054,\n   "lon": 37.6781112,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2855188627",\n   "kind": "organisation",\n   "name": "Сигни Груп",\n   "office_type": "company",\n   "lat": 55.7539675,\n   "lon": 37.7145025,\n   "opening_hours": "09:00-18:00"\n  },\n  {\n   "id": "node/2875600485",\n   "kind": "organisation",\n   "name": "Желдорэкспедиция",\n   "office_type": "company",\n   "lat": 55.7956305,\n   "lon": 37.6556843,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2875670725",\n   "kind": "organisation",\n   "name": "ОСАГО",\n   "office_type": "insurance",\n   "lat": 55.641366,\n   "lon": 37.6172103,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2898731512",\n   "kind": "organisation",\n   "name": "Конгресс-центр",\n   "office_type": "company",\n   "lat": 55.7495836,\n   "lon": 37.5456896,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2898731513",\n   "kind": "organisation",\n   "name": "Экспо Сервис",\n   "office_type": "company",\n   "lat": 55.7495849,\n   "lon": 37.5452405,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2911644912",\n   "kind": "organisation",\n   "name": "Серёгин и партнёры",\n   "office_type": "lawyer",\n   "lat": 55.7687794,\n   "lon": 37.6456223,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2912535231",\n   "kind": "organisation",\n   "name": "Налоговая инспекция №4",\n   "office_type": "government",\n   "lat": 55.7227251,\n   "lon": 37.5649325,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2919125134",\n   "kind": "organisation",\n   "name": "ОДС #1032",\n   "office_type": "property_management",\n   "lat": 55.642943,\n   "lon": 37.6100873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2922768877",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7862121,\n   "lon": 37.5956126,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2922768880",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "accountant",\n   "lat": 55.786261,\n   "lon": 37.5954577,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2928253702",\n   "kind": "organisation",\n   "name": "Архитектурное бюро КС",\n   "office_type": "architect",\n   "lat": 55.7970576,\n   "lon": 37.5139903,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/2929306132",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Хамовники",\n   "office_type": "government",\n   "lat": 55.7429908,\n   "lon": 37.5846651,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/2932443498",\n   "kind": "organisation",\n   "name": "Green Home",\n   "office_type": "company",\n   "lat": 55.7015538,\n   "lon": 37.5777256,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2935977472",\n   "kind": "organisation",\n   "name": "Европейская Медиагруппа",\n   "office_type": "company",\n   "lat": 55.7452437,\n   "lon": 37.6621316,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2949507367",\n   "kind": "organisation",\n   "name": "TELE2",\n   "office_type": "telecommunication",\n   "lat": 55.6124429,\n   "lon": 37.6981163,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/2956991672",\n   "kind": "organisation",\n   "name": "Гранд Тур Вояж",\n   "office_type": "travel_agent",\n   "lat": 55.7652261,\n   "lon": 37.6386081,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2957166870",\n   "kind": "organisation",\n   "name": "Научно-исследовательский центр информатизации Министерства иностранных дел Российской Федерации",\n   "office_type": "research",\n   "lat": 55.7466037,\n   "lon": 37.5852752,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2960001233",\n   "kind": "organisation",\n   "name": "Глаголева",\n   "office_type": "notary",\n   "lat": 55.6235937,\n   "lon": 37.6662693,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2960001333",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.6235336,\n   "lon": 37.6663158,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2960299558",\n   "kind": "organisation",\n   "name": "Поморское землячество в Москве",\n   "office_type": "ngo",\n   "lat": 55.7393321,\n   "lon": 37.5275574,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2984110946",\n   "kind": "organisation",\n   "name": "ООО ИТЦ СКАНЭКС",\n   "office_type": "company",\n   "lat": 55.6343198,\n   "lon": 37.4415716,\n   "opening_hours": "10:00-18:00"\n  },\n  {\n   "id": "node/2986384522",\n   "kind": "organisation",\n   "name": "Инженерная служба",\n   "office_type": "property_management",\n   "lat": 55.7951426,\n   "lon": 37.5833022,\n   "opening_hours": null\n  },\n  {\n   "id": "node/2990115914",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Арбат",\n   "office_type": "government",\n   "lat": 55.7475749,\n   "lon": 37.593536,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-16:00"\n  },\n  {\n   "id": "node/2991909339",\n   "kind": "organisation",\n   "name": "GiperActive digital agency",\n   "office_type": "company",\n   "lat": 55.7393257,\n   "lon": 37.6491055,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3000681428",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.6285866,\n   "lon": 37.6047543,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3006337933",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6473891,\n   "lon": 37.666274,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/3006979111",\n   "kind": "organisation",\n   "name": "ФБУЗ Центр гигиены и эпидемиологии в городе Москве",\n   "office_type": "government",\n   "lat": 55.7428628,\n   "lon": 37.6594985,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3013909193",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6124781,\n   "lon": 37.5418321,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00"\n  },\n  {\n   "id": "node/3017474733",\n   "kind": "organisation",\n   "name": "Нагатинская межрайонная прокуратура ЮАО города Москвы",\n   "office_type": "government",\n   "lat": 55.679779,\n   "lon": 37.6336064,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3027297185",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "translator",\n   "lat": 55.7046869,\n   "lon": 37.7746765,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3028683366",\n   "kind": "organisation",\n   "name": "Федеральная служба судебных приставов России",\n   "office_type": "government",\n   "lat": 55.7618049,\n   "lon": 37.6220983,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3032188252",\n   "kind": "organisation",\n   "name": "НПО ГеоТраст",\n   "office_type": "company",\n   "lat": 55.7628096,\n   "lon": 37.6638633,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3032938033",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6645783,\n   "lon": 37.6264861,\n   "opening_hours": "10:00-22:00"\n  },\n  {\n   "id": "node/3032954633",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6641241,\n   "lon": 37.6269863,\n   "opening_hours": "10:00-22:00"\n  },\n  {\n   "id": "node/3043238764",\n   "kind": "organisation",\n   "name": "ЖЭК-1",\n   "office_type": "property_management",\n   "lat": 55.7502749,\n   "lon": 37.5799994,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3043238765",\n   "kind": "organisation",\n   "name": "Инженерная служба района Арбат",\n   "office_type": "property_management",\n   "lat": 55.7502706,\n   "lon": 37.5801272,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3043238768",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "notary",\n   "lat": 55.752111,\n   "lon": 37.5821803,\n   "opening_hours": "Mo-Th 10:00-14:00,15:00-17:00"\n  },\n  {\n   "id": "node/3043881830",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.7466632,\n   "lon": 37.5767379,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3044662861",\n   "kind": "organisation",\n   "name": "Отделение Пенсионного фонда РФ (Управление № 1)",\n   "office_type": "government",\n   "lat": 55.7297773,\n   "lon": 37.6473746,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3044662862",\n   "kind": "organisation",\n   "name": "Отделение Пенсионного фонда РФ (Управление № 2)",\n   "office_type": "government",\n   "lat": 55.7306493,\n   "lon": 37.6478021,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3044662863",\n   "kind": "organisation",\n   "name": "Отделение Пенсионного фонда РФ (Управление № 3)",\n   "office_type": "government",\n   "lat": 55.7303441,\n   "lon": 37.6475823,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3045638228",\n   "kind": "organisation",\n   "name": "Совет ветеранов района Арбат",\n   "office_type": "ngo",\n   "lat": 55.7502203,\n   "lon": 37.5904611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3045638229",\n   "kind": "organisation",\n   "name": "Технопромэкспорт",\n   "office_type": "company",\n   "lat": 55.7516303,\n   "lon": 37.592373,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3045639877",\n   "kind": "organisation",\n   "name": "Афанасьева Н.В.",\n   "office_type": "lawyer",\n   "lat": 55.751065,\n   "lon": 37.5912641,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/3048481810",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6482102,\n   "lon": 37.7385132,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3052614758",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "lawyer",\n   "lat": 55.7843337,\n   "lon": 37.6617732,\n   "opening_hours": "Mo-Th 10:00-13:30,14:00-18:00; Fr 10:00-13:30,14:00-17:00"\n  },\n  {\n   "id": "node/3054994129",\n   "kind": "organisation",\n   "name": "Центр по проблемам экологии и продуктивности лесов РАН",\n   "office_type": "research",\n   "lat": 55.6551777,\n   "lon": 37.5336416,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3059957847",\n   "kind": "organisation",\n   "name": "Симоновская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7154755,\n   "lon": 37.6110812,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3060631530",\n   "kind": "organisation",\n   "name": "«Спа Стайл» - строительство саун.",\n   "office_type": "company",\n   "lat": 55.6340665,\n   "lon": 37.4419324,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3060750057",\n   "kind": "organisation",\n   "name": "Антонов, Богданов и партнеры",\n   "office_type": "lawyer",\n   "lat": 55.7173317,\n   "lon": 37.7870572,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3062640903",\n   "kind": "organisation",\n   "name": "Рязанский отдел ЗАГС",\n   "office_type": "government",\n   "lat": 55.7168024,\n   "lon": 37.7851502,\n   "opening_hours": "Tu-Sa 09:00-13:30,15:00-17:30"\n  },\n  {\n   "id": "node/3066427247",\n   "kind": "organisation",\n   "name": "ООО «Фарла» - подушки для беременных",\n   "office_type": "company",\n   "lat": 55.6460972,\n   "lon": 37.6378293,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-20:00"\n  },\n  {\n   "id": "node/3069191134",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7510308,\n   "lon": 37.5959143,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3069303514",\n   "kind": "organisation",\n   "name": "Петровка 38",\n   "office_type": "newspaper",\n   "lat": 55.7407959,\n   "lon": 37.617512,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3073265739",\n   "kind": "organisation",\n   "name": "Topcon Positioning Systems",\n   "office_type": "research",\n   "lat": 55.7218522,\n   "lon": 37.652097,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3073439709",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8673622,\n   "lon": 37.6749419,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3075401133",\n   "kind": "organisation",\n   "name": "ОДС #343",\n   "office_type": "property_management",\n   "lat": 55.6256643,\n   "lon": 37.6738685,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/3075402733",\n   "kind": "organisation",\n   "name": "ОДС",\n   "office_type": "property_management",\n   "lat": 55.6297401,\n   "lon": 37.6676997,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3077821383",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Коньково",\n   "office_type": "government",\n   "lat": 55.6448031,\n   "lon": 37.5198882,\n   "opening_hours": "08:00-20:00"\n  },\n  {\n   "id": "node/3078036633",\n   "kind": "organisation",\n   "name": "Миэль",\n   "office_type": "estate_agent",\n   "lat": 55.643053,\n   "lon": 37.6551353,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3078041933",\n   "kind": "organisation",\n   "name": "Инвент",\n   "office_type": "advertising_agency",\n   "lat": 55.7513808,\n   "lon": 37.6792322,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3079075220",\n   "kind": "organisation",\n   "name": "Ланит",\n   "office_type": "company",\n   "lat": 55.768844,\n   "lon": 37.6722486,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3080014607",\n   "kind": "organisation",\n   "name": "Главный радиочастотный центр",\n   "office_type": "government",\n   "lat": 55.7232483,\n   "lon": 37.6533733,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3080738886",\n   "kind": "organisation",\n   "name": "X5 Retail Group",\n   "office_type": "company",\n   "lat": 55.7315686,\n   "lon": 37.6983577,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3084378343",\n   "kind": "organisation",\n   "name": "Softline",\n   "office_type": "it",\n   "lat": 55.6945741,\n   "lon": 37.5640736,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3097126851",\n   "kind": "organisation",\n   "name": "Отдел экологического контроля Южного административного округа.",\n   "office_type": "government",\n   "lat": 55.6337362,\n   "lon": 37.6150483,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3102110849",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7444687,\n   "lon": 37.5640369,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3102112575",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7447055,\n   "lon": 37.5641994,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3115162033",\n   "kind": "organisation",\n   "name": "Советник Права",\n   "office_type": "lawyer",\n   "lat": 55.7681749,\n   "lon": 37.6450564,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3118452323",\n   "kind": "organisation",\n   "name": "Coral Travel",\n   "office_type": "travel_agent",\n   "lat": 55.7484904,\n   "lon": 37.5842906,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3123243611",\n   "kind": "organisation",\n   "name": "Общество инвалидов №2",\n   "office_type": "ngo",\n   "lat": 55.7169699,\n   "lon": 37.4534298,\n   "opening_hours": "We[1,3] 10:00-13:00"\n  },\n  {\n   "id": "node/3123243617",\n   "kind": "organisation",\n   "name": "Совет ветеранов РЭП-36",\n   "office_type": "ngo",\n   "lat": 55.7170336,\n   "lon": 37.4533946,\n   "opening_hours": "Tu[2,4] 10:00-12:00"\n  },\n  {\n   "id": "node/3129214757",\n   "kind": "organisation",\n   "name": "Отдел трудоустройства \\"Бабушкинский\\" Центра занятости",\n   "office_type": "government",\n   "lat": 55.863654,\n   "lon": 37.6656251,\n   "opening_hours": "Mo-Fr 09:00-13:00,13:45-17:00; Tu 12:00-13:00,13:45-20:00; Th 11:00-13:00,13:45-19:00"\n  },\n  {\n   "id": "node/3131052377",\n   "kind": "organisation",\n   "name": "ООО «Устрой Праздник»",\n   "office_type": "company",\n   "lat": 55.7972557,\n   "lon": 37.7699487,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3132936933",\n   "kind": "organisation",\n   "name": "ФГБНУ НИИ фармакологии им. В. В. Закусова",\n   "office_type": "research",\n   "lat": 55.8098554,\n   "lon": 37.5153309,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3132936934",\n   "kind": "organisation",\n   "name": "НИИ нормальной физиологии им. П. К. Анохина",\n   "office_type": "research",\n   "lat": 55.8093423,\n   "lon": 37.5158052,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3135016785",\n   "kind": "organisation",\n   "name": "RuDive",\n   "office_type": "travel_agent",\n   "lat": 55.7898106,\n   "lon": 37.7100689,\n   "opening_hours": "Mo-Fr 11:00-21:00; Sa 11:00-17:00"\n  },\n  {\n   "id": "node/3139686695",\n   "kind": "organisation",\n   "name": "Аттестационный научно-технический центр \\"Эксперт\\"",\n   "office_type": "company",\n   "lat": 55.7018601,\n   "lon": 37.658039,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3141818025",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7234354,\n   "lon": 37.8251176,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/3142790599",\n   "kind": "organisation",\n   "name": "Нотра",\n   "office_type": "estate_agent",\n   "lat": 55.7589256,\n   "lon": 37.58918,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3143594736",\n   "kind": "organisation",\n   "name": "студия \\"Местная еда\\"",\n   "office_type": "ngo",\n   "lat": 55.8097679,\n   "lon": 37.5125036,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3143971100",\n   "kind": "organisation",\n   "name": "ООО \\"Легион-Сервис\\"",\n   "office_type": "company",\n   "lat": 55.6360073,\n   "lon": 37.7430988,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3145596509",\n   "kind": "organisation",\n   "name": "Отдел УФМС России по г. Москве в ЮЗАО (отделение по району Обручевский)",\n   "office_type": "government",\n   "lat": 55.6633261,\n   "lon": 37.5192502,\n   "opening_hours": "Mo 09:00-14:00,14:45-18:00;Tu,Th 11:00-14:00,14:45-20:00;We 09:00-13:00;Fr 09:00-14:00,14:45-16:45;Sa 09:00-16:00"\n  },\n  {\n   "id": "node/3148066983",\n   "kind": "organisation",\n   "name": "Гудок",\n   "office_type": "newspaper",\n   "lat": 55.7689257,\n   "lon": 37.6704619,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3149355706",\n   "kind": "organisation",\n   "name": "Институт стекла",\n   "office_type": "research",\n   "lat": 55.7450657,\n   "lon": 37.7064794,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3158893743",\n   "kind": "organisation",\n   "name": "ДЭЗ района Богородское",\n   "office_type": "property_management",\n   "lat": 55.8120186,\n   "lon": 37.717092,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3159079179",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6487269,\n   "lon": 37.7449633,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/3161602788",\n   "kind": "organisation",\n   "name": "Mirantis",\n   "office_type": "it",\n   "lat": 55.7343605,\n   "lon": 37.6645311,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3165933537",\n   "kind": "organisation",\n   "name": "Бюро переводов \\"Алигат\\"",\n   "office_type": "translator",\n   "lat": 55.7399945,\n   "lon": 37.6539828,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3170513485",\n   "kind": "organisation",\n   "name": "Стейл",\n   "office_type": "logistics",\n   "lat": 55.6184986,\n   "lon": 37.4447446,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3177871161",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6540262,\n   "lon": 37.6476307,\n   "opening_hours": "Mo-Fr 09:00-20:00: Sa 10:00-19:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/3179833731",\n   "kind": "organisation",\n   "name": "Trimble RUS",\n   "office_type": "company",\n   "lat": 55.6998806,\n   "lon": 37.5616277,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3182511222",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Даниловского района",\n   "office_type": "government",\n   "lat": 55.7144926,\n   "lon": 37.6624697,\n   "opening_hours": "Mo 10:00-17:00; We 10:00-17:00"\n  },\n  {\n   "id": "node/3198157996",\n   "kind": "organisation",\n   "name": "Бизнес DEPO",\n   "office_type": "yes",\n   "lat": 55.8894649,\n   "lon": 37.5790628,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3199936579",\n   "kind": "organisation",\n   "name": "Инженерная служба Тимирязевского района",\n   "office_type": "property_management",\n   "lat": 55.8115266,\n   "lon": 37.5778529,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/3199936580",\n   "kind": "organisation",\n   "name": "Постоянное представительство Челябинской области при правительстве Российской Федерации",\n   "office_type": "government",\n   "lat": 55.8114095,\n   "lon": 37.577477,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3203085322",\n   "kind": "organisation",\n   "name": "Карабанов и партнёры",\n   "office_type": "lawyer",\n   "lat": 55.7570757,\n   "lon": 37.6370694,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3211585385",\n   "kind": "organisation",\n   "name": "Samsung R&D Institute",\n   "office_type": "research",\n   "lat": 55.7964887,\n   "lon": 37.6006114,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3212573828",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7636942,\n   "lon": 37.5599414,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3215034349",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.762894,\n   "lon": 37.5646408,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/3215335035",\n   "kind": "organisation",\n   "name": "Московское областное бюро технической инвентаризации",\n   "office_type": "company",\n   "lat": 55.8959261,\n   "lon": 37.4438972,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3218570854",\n   "kind": "organisation",\n   "name": "Межгосударственный авиационный комитет",\n   "office_type": "government",\n   "lat": 55.740774,\n   "lon": 37.6246314,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3223039085",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7963628,\n   "lon": 37.5809262,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3224502718",\n   "kind": "organisation",\n   "name": "ФКП \\"Росгосцирк\\"",\n   "office_type": "government",\n   "lat": 55.7606229,\n   "lon": 37.6223398,\n   "opening_hours": "09:00-18:00"\n  },\n  {\n   "id": "node/3228599362",\n   "kind": "organisation",\n   "name": "Хронос",\n   "office_type": "lawyer",\n   "lat": 55.6969309,\n   "lon": 37.5328715,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3234461962",\n   "kind": "organisation",\n   "name": "Поедем.ру",\n   "office_type": "travel_agent",\n   "lat": 55.7572405,\n   "lon": 37.6001411,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/3237900035",\n   "kind": "organisation",\n   "name": "Нотариус Похлебаева Н.Н.",\n   "office_type": "notary",\n   "lat": 55.6915175,\n   "lon": 37.5749479,\n   "opening_hours": "Tu-Th 10:00-17:00, 13:00-14:00, Fr 10:00-16:00, 13:00-14:00, Sa 10:00-14:00"\n  },\n  {\n   "id": "node/3249265164",\n   "kind": "organisation",\n   "name": "Источник Бэттэрис",\n   "office_type": "company",\n   "lat": 55.7599046,\n   "lon": 37.7645036,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/3252601667",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7784129,\n   "lon": 37.692761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3256119954",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Савёловский",\n   "office_type": "government",\n   "lat": 55.7985249,\n   "lon": 37.5827459,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/3279911458",\n   "kind": "organisation",\n   "name": "ООО «АрхиСтиль»",\n   "office_type": "company",\n   "lat": 55.6181886,\n   "lon": 37.5340508,\n   "opening_hours": "09:00-18:00"\n  },\n  {\n   "id": "node/3280224135",\n   "kind": "organisation",\n   "name": "ООО \\"Вайт Смайл\\"",\n   "office_type": "company",\n   "lat": 55.7800373,\n   "lon": 37.5902781,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3283852415",\n   "kind": "organisation",\n   "name": "ПБУ 24 - бухгалтерские услуги в Москве",\n   "office_type": "accountant",\n   "lat": 55.8759984,\n   "lon": 37.5886833,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3286282764",\n   "kind": "organisation",\n   "name": "МГТС",\n   "office_type": "telecommunication",\n   "lat": 55.8389443,\n   "lon": 37.4882132,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3286316070",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8394419,\n   "lon": 37.4868049,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3286316071",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8403534,\n   "lon": 37.4861622,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3312704248",\n   "kind": "organisation",\n   "name": "Инженерная служба",\n   "office_type": "property_management",\n   "lat": 55.7942734,\n   "lon": 37.5754975,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3313895341",\n   "kind": "organisation",\n   "name": "Негосударственный пенсионный фонд \\"Промагрофонд\\"",\n   "office_type": "financial",\n   "lat": 55.7273907,\n   "lon": 37.6471224,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/3313906082",\n   "kind": "organisation",\n   "name": "Негосударственный пенсионный фонд \\"Наследие\\"",\n   "office_type": "financial",\n   "lat": 55.727362,\n   "lon": 37.6471734,\n   "opening_hours": "Mo-Fr 9:00-18:00"\n  },\n  {\n   "id": "node/3333662925",\n   "kind": "organisation",\n   "name": "ООО \\"Компания МСС\\"",\n   "office_type": "company",\n   "lat": 55.6159311,\n   "lon": 37.7032651,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/3337558985",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7654583,\n   "lon": 37.6078103,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3343806639",\n   "kind": "organisation",\n   "name": "Центральный офис продаж ГК \\"Пионер\\"",\n   "office_type": "company",\n   "lat": 55.7626043,\n   "lon": 37.5703789,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa,Su 10:00-18:00"\n  },\n  {\n   "id": "node/3344160910",\n   "kind": "organisation",\n   "name": "Condé Nast Россия",\n   "office_type": "publisher",\n   "lat": 55.7616453,\n   "lon": 37.613188,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3345511288",\n   "kind": "organisation",\n   "name": "Клиентская служба ПФР",\n   "office_type": "government",\n   "lat": 55.8022818,\n   "lon": 37.4727669,\n   "opening_hours": "Mo-Th 09:00-12:30,13:15-18:00; Fr 09:00-12:30,13:15-17:00"\n  },\n  {\n   "id": "node/3357507961",\n   "kind": "organisation",\n   "name": "Научное шоу Ньютона",\n   "office_type": "company",\n   "lat": 55.7593313,\n   "lon": 37.8070173,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3360158575",\n   "kind": "organisation",\n   "name": "ИНЭУМ им. И. С. Брука",\n   "office_type": "research",\n   "lat": 55.7015684,\n   "lon": 37.5771673,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3360158577",\n   "kind": "organisation",\n   "name": "Центр исследований и разработок МЦСТ",\n   "office_type": "research",\n   "lat": 55.7015921,\n   "lon": 37.5771394,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3361508541",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7060961,\n   "lon": 37.7674403,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3368762076",\n   "kind": "organisation",\n   "name": "ФГУП Космическая связь",\n   "office_type": "yes",\n   "lat": 55.7170127,\n   "lon": 37.6115529,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3385549151",\n   "kind": "organisation",\n   "name": "Издательство «Перо»",\n   "office_type": "publisher",\n   "lat": 55.7347855,\n   "lon": 37.7098536,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3385561248",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7455909,\n   "lon": 37.6841571,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3403844614",\n   "kind": "organisation",\n   "name": "ООО «Дом каркасный»",\n   "office_type": "company",\n   "lat": 55.7335238,\n   "lon": 37.7127648,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa 10:00-18:00; Su off"\n  },\n  {\n   "id": "node/3404909402",\n   "kind": "organisation",\n   "name": "Институт географии РАН",\n   "office_type": "research",\n   "lat": 55.6952168,\n   "lon": 37.5684929,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3406888930",\n   "kind": "organisation",\n   "name": "Грифон инжиниринг",\n   "office_type": "company",\n   "lat": 55.8112044,\n   "lon": 37.833568,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3411234796",\n   "kind": "organisation",\n   "name": "Деловые линии",\n   "office_type": "company",\n   "lat": 55.8969596,\n   "lon": 37.5189562,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3414035650",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7719266,\n   "lon": 37.5932975,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3414094498",\n   "kind": "organisation",\n   "name": "ООО «MDtools»",\n   "office_type": "company",\n   "lat": 55.8732352,\n   "lon": 37.689612,\n   "opening_hours": "Mo-Fr 08:00-17:30; Sa,Su off"\n  },\n  {\n   "id": "node/3414536088",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7658475,\n   "lon": 37.602854,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/3423427032",\n   "kind": "organisation",\n   "name": "Нотариус Карнаухова А.Э.",\n   "office_type": "lawyer",\n   "lat": 55.7544325,\n   "lon": 37.5956605,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/3424330621",\n   "kind": "organisation",\n   "name": "ООО Кема Клаб",\n   "office_type": "company",\n   "lat": 55.7895352,\n   "lon": 37.7115622,\n   "opening_hours": "Mo-Fr 09:30-18:00; Sa off; Su off"\n  },\n  {\n   "id": "node/3428369999",\n   "kind": "organisation",\n   "name": "QIWI",\n   "office_type": "company",\n   "lat": 55.6372898,\n   "lon": 37.5976564,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3440390521",\n   "kind": "organisation",\n   "name": "Управление района Лианозово",\n   "office_type": "government",\n   "lat": 55.8934253,\n   "lon": 37.5856041,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:45"\n  },\n  {\n   "id": "node/3440392783",\n   "kind": "organisation",\n   "name": "УСЗН района\xa0Лианозово",\n   "office_type": "government",\n   "lat": 55.8930341,\n   "lon": 37.5856341,\n   "opening_hours": "Mo-Fr 08:00-12:00, Mo-Th 12:45-17:00, Fr 12:45-15:45"\n  },\n  {\n   "id": "node/3447562623",\n   "kind": "organisation",\n   "name": "Посольство Кении",\n   "office_type": "diplomatic",\n   "lat": 55.7423568,\n   "lon": 37.5967806,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3450970957",\n   "kind": "organisation",\n   "name": "ООО \\"НТЦ ИТ РОСА\\"",\n   "office_type": "company",\n   "lat": 55.7897083,\n   "lon": 37.5023618,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3463079466",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Басманного района",\n   "office_type": "government",\n   "lat": 55.780138,\n   "lon": 37.685547,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/3474436773",\n   "kind": "organisation",\n   "name": "Территориальная избирательная комиссия района Зюзино",\n   "office_type": "government",\n   "lat": 55.6553227,\n   "lon": 37.5779933,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3474436774",\n   "kind": "organisation",\n   "name": "Управа района Зюзино",\n   "office_type": "government",\n   "lat": 55.6553475,\n   "lon": 37.5778958,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3476977403",\n   "kind": "organisation",\n   "name": "Федеральная миграционная служба",\n   "office_type": "government",\n   "lat": 55.7454405,\n   "lon": 37.6483702,\n   "opening_hours": "Mo-Th 10:00-13:00,14:00-17:00; Fr 10:00-13:00,14:00-16:00; Sa 10:00-13:00"\n  },\n  {\n   "id": "node/3477342556",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.7413593,\n   "lon": 37.6500876,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/3477342562",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.7402084,\n   "lon": 37.6497755,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3477342564",\n   "kind": "organisation",\n   "name": "Управление капитального строительства",\n   "office_type": "government",\n   "lat": 55.7388519,\n   "lon": 37.6482203,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3477350083",\n   "kind": "organisation",\n   "name": "Администрация ГАУК г. Москвы \\"ПКиО \\"Таганский\\"",\n   "office_type": "company",\n   "lat": 55.7408638,\n   "lon": 37.6621877,\n   "opening_hours": "Mo-Th 09:00-18:00; PH closed; Fr 09:00-17:00"\n  },\n  {\n   "id": "node/3477480560",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7389608,\n   "lon": 37.6524836,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3478155496",\n   "kind": "organisation",\n   "name": "Homag",\n   "office_type": "company",\n   "lat": 55.7359522,\n   "lon": 37.6533256,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3478996650",\n   "kind": "organisation",\n   "name": "Правильные люди",\n   "office_type": "employment_agency",\n   "lat": 55.734431,\n   "lon": 37.6585218,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/3481829140",\n   "kind": "organisation",\n   "name": "Местная общественная организация воинов-интернационалистов Афганистана",\n   "office_type": "ngo",\n   "lat": 55.7333106,\n   "lon": 37.6567203,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3482033723",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.7386179,\n   "lon": 37.6531644,\n   "opening_hours": "Mo-Fr 09:30-20:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/3486439305",\n   "kind": "organisation",\n   "name": "Вокин Иннокентий Михайлович",\n   "office_type": "lawyer",\n   "lat": 55.7970699,\n   "lon": 37.6193318,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 10:00-16:00; Su off"\n  },\n  {\n   "id": "node/3486890994",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.699572,\n   "lon": 37.5040172,\n   "opening_hours": "Tu-Sa 10:00-18:30"\n  },\n  {\n   "id": "node/3498917593",\n   "kind": "organisation",\n   "name": "Визовый центр Финляндии",\n   "office_type": "visa",\n   "lat": 55.6722833,\n   "lon": 37.6309517,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/3509241704",\n   "kind": "organisation",\n   "name": "Падва и Эпштейн",\n   "office_type": "lawyer",\n   "lat": 55.7480636,\n   "lon": 37.6551307,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3510443331",\n   "kind": "organisation",\n   "name": "Hewlett Packard Enterprise",\n   "office_type": "it",\n   "lat": 55.822153,\n   "lon": 37.4992504,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/3513209808",\n   "kind": "organisation",\n   "name": "АО \\"Метрогипротранс\\"",\n   "office_type": "company",\n   "lat": 55.731773,\n   "lon": 37.6367298,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3515458540",\n   "kind": "organisation",\n   "name": "Tele2 Россия",\n   "office_type": "telecommunication",\n   "lat": 55.8377965,\n   "lon": 37.4807369,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/3515458542",\n   "kind": "organisation",\n   "name": "Tinkoff Bank",\n   "office_type": "company",\n   "lat": 55.8400346,\n   "lon": 37.4901984,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3515458543",\n   "kind": "organisation",\n   "name": "Kaspersky Lab",\n   "office_type": "company",\n   "lat": 55.8371489,\n   "lon": 37.4811768,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3515458544",\n   "kind": "organisation",\n   "name": "Kaspersky Lab",\n   "office_type": "company",\n   "lat": 55.8364982,\n   "lon": 37.4814664,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3521821943",\n   "kind": "organisation",\n   "name": "Московское агентство недвижимости",\n   "office_type": "estate_agent",\n   "lat": 55.7472851,\n   "lon": 37.5884008,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3524525876",\n   "kind": "organisation",\n   "name": "СОБЕРИЗАВОД",\n   "office_type": "company",\n   "lat": 55.6629749,\n   "lon": 37.7136392,\n   "opening_hours": "Mo-Fr 11:00-16:00"\n  },\n  {\n   "id": "node/3525634421",\n   "kind": "organisation",\n   "name": "Глория-Холод",\n   "office_type": "company",\n   "lat": 55.7075035,\n   "lon": 37.7003363,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/3526824259",\n   "kind": "organisation",\n   "name": "Посольство султаната Оман",\n   "office_type": "diplomatic",\n   "lat": 55.7400822,\n   "lon": 37.6185909,\n   "opening_hours": "Mo-Fr 09:00-15:00"\n  },\n  {\n   "id": "node/3549215601",\n   "kind": "organisation",\n   "name": "3DPrintus",\n   "office_type": "company",\n   "lat": 55.7501488,\n   "lon": 37.6459792,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/3552337974",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7679207,\n   "lon": 37.6057259,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3558389993",\n   "kind": "organisation",\n   "name": "Юридический центр «ЮрЛекс»",\n   "office_type": "lawyer",\n   "lat": 55.8018765,\n   "lon": 37.5505275,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/3558503402",\n   "kind": "organisation",\n   "name": "НТЦ \\"Модуль\\"",\n   "office_type": "research",\n   "lat": 55.8036816,\n   "lon": 37.5482932,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3558670189",\n   "kind": "organisation",\n   "name": "ЖСК \\"Тюльпан-2\\"",\n   "office_type": "company",\n   "lat": 55.801958,\n   "lon": 37.5507147,\n   "opening_hours": "Mo-We 19:00-20:00"\n  },\n  {\n   "id": "node/3558689521",\n   "kind": "organisation",\n   "name": "Инженерная служба района Аэропорт",\n   "office_type": "property_management",\n   "lat": 55.8075503,\n   "lon": 37.5331373,\n   "opening_hours": "Mo-Tu 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/3558832645",\n   "kind": "organisation",\n   "name": "Гематолог",\n   "office_type": "research",\n   "lat": 55.8188198,\n   "lon": 37.5196597,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3558832646",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.7995284,\n   "lon": 37.5500145,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3558832649",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.7993796,\n   "lon": 37.5498446,\n   "opening_hours": "Mo 11:00-13:00; We 11:00-13:00"\n  },\n  {\n   "id": "node/3560681365",\n   "kind": "organisation",\n   "name": "Торговый дом \\"Сотдел\\"",\n   "office_type": "company",\n   "lat": 55.6559202,\n   "lon": 37.819722,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/3585788593",\n   "kind": "organisation",\n   "name": "Ракурс",\n   "office_type": "company",\n   "lat": 55.8195968,\n   "lon": 37.6439134,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/3587241193",\n   "kind": "organisation",\n   "name": "Театральная жизнь",\n   "office_type": "newspaper",\n   "lat": 55.765313,\n   "lon": 37.625907,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3587243493",\n   "kind": "organisation",\n   "name": "Gala Form",\n   "office_type": "company",\n   "lat": 55.765518,\n   "lon": 37.625716,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3600089741",\n   "kind": "organisation",\n   "name": "Федеральная Служба Информации",\n   "office_type": "news_agency",\n   "lat": 55.7188155,\n   "lon": 37.6320249,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3600106784",\n   "kind": "organisation",\n   "name": "АЭИ ПРАЙМ",\n   "office_type": "news_agency",\n   "lat": 55.737623,\n   "lon": 37.5898761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3600111893",\n   "kind": "organisation",\n   "name": "Р-Спорт",\n   "office_type": "news_agency",\n   "lat": 55.7374962,\n   "lon": 37.5901711,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3600135675",\n   "kind": "organisation",\n   "name": "Интерфакс",\n   "office_type": "news_agency",\n   "lat": 55.7698707,\n   "lon": 37.5969736,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3600156087",\n   "kind": "organisation",\n   "name": "Медиа-холдинг \\"РБК\\"",\n   "office_type": "company",\n   "lat": 55.6599552,\n   "lon": 37.539418,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3600165421",\n   "kind": "organisation",\n   "name": "Информационное агентство \\"Regnum\\"",\n   "office_type": "news_agency",\n   "lat": 55.7407575,\n   "lon": 37.6102551,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3600190940",\n   "kind": "organisation",\n   "name": "Росбалт",\n   "office_type": "news_agency",\n   "lat": 55.7555968,\n   "lon": 37.5968842,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3600655658",\n   "kind": "organisation",\n   "name": "ООО \\"Авто Паспорт\\"",\n   "office_type": "company",\n   "lat": 55.6583468,\n   "lon": 37.585002,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3601438921",\n   "kind": "organisation",\n   "name": "Институт клинических исследований",\n   "office_type": "research",\n   "lat": 55.7175939,\n   "lon": 37.5115742,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3601438922",\n   "kind": "organisation",\n   "name": "ММС",\n   "office_type": "research",\n   "lat": 55.7175274,\n   "lon": 37.5114455,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/3605808494",\n   "kind": "organisation",\n   "name": "Французский визовый центр",\n   "office_type": "visa",\n   "lat": 55.7859075,\n   "lon": 37.6356259,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/3613293680",\n   "kind": "organisation",\n   "name": "Инфофлот",\n   "office_type": "travel_agent",\n   "lat": 55.8556027,\n   "lon": 37.4652827,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3616841994",\n   "kind": "organisation",\n   "name": "Тема",\n   "office_type": "company",\n   "lat": 55.765389,\n   "lon": 37.6332157,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3618470511",\n   "kind": "organisation",\n   "name": "Власова С.С.",\n   "office_type": "lawyer",\n   "lat": 55.7831211,\n   "lon": 37.7332937,\n   "opening_hours": "Mo-Th 10:00-18:00; Fr 10:00-16:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/3626257993",\n   "kind": "organisation",\n   "name": "Главное управление по вопросам миграции МВД России",\n   "office_type": "government",\n   "lat": 55.7680355,\n   "lon": 37.6474366,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3626284793",\n   "kind": "organisation",\n   "name": "Государственная инспекция по контролю за использованием объектов недвижимости г. Москвы",\n   "office_type": "government",\n   "lat": 55.7680833,\n   "lon": 37.6460149,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3626301493",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7688341,\n   "lon": 37.6427628,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3626427993",\n   "kind": "organisation",\n   "name": "Да!",\n   "office_type": "financial",\n   "lat": 55.769495,\n   "lon": 37.629492,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3630833651",\n   "kind": "organisation",\n   "name": "Продюсерский Центр RUSMUSIC",\n   "office_type": "company",\n   "lat": 55.7984358,\n   "lon": 37.5544343,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3631546952",\n   "kind": "organisation",\n   "name": "ООО «Джой Продактс»",\n   "office_type": "company",\n   "lat": 55.6822351,\n   "lon": 37.4847144,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3632368193",\n   "kind": "organisation",\n   "name": "Tescao",\n   "office_type": "union",\n   "lat": 55.7450265,\n   "lon": 37.6002787,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3632393293",\n   "kind": "organisation",\n   "name": "ЮрПартнер",\n   "office_type": "lawyer",\n   "lat": 55.7484544,\n   "lon": 37.5980044,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3632393593",\n   "kind": "organisation",\n   "name": "Бест новострой",\n   "office_type": "estate_agent",\n   "lat": 55.7487056,\n   "lon": 37.5984038,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3634312421",\n   "kind": "organisation",\n   "name": "ЛДПР (отделение в ЗАО)",\n   "office_type": "political_party",\n   "lat": 55.6792587,\n   "lon": 37.5267962,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3636284795",\n   "kind": "organisation",\n   "name": "Корсик В. К.",\n   "office_type": "notary",\n   "lat": 55.7625007,\n   "lon": 37.6254504,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/3636362793",\n   "kind": "organisation",\n   "name": "Центрдорстрой",\n   "office_type": "company",\n   "lat": 55.7628515,\n   "lon": 37.6256915,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3638722831",\n   "kind": "organisation",\n   "name": "Ростехнадзор",\n   "office_type": "government",\n   "lat": 55.7696743,\n   "lon": 37.5847414,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3639479342",\n   "kind": "organisation",\n   "name": "Сервисный центр \\"Московский транспорт\\"",\n   "office_type": "government",\n   "lat": 55.7669205,\n   "lon": 37.6651841,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/3640294193",\n   "kind": "organisation",\n   "name": "Общественная приёмная министра обороны РФ",\n   "office_type": "government",\n   "lat": 55.7489819,\n   "lon": 37.6015474,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3641948593",\n   "kind": "organisation",\n   "name": "Росфинмониторинг",\n   "office_type": "government",\n   "lat": 55.7672253,\n   "lon": 37.6408771,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3642483378",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Покровское-Стрешнево",\n   "office_type": "government",\n   "lat": 55.8269318,\n   "lon": 37.4444131,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/3642483379",\n   "kind": "organisation",\n   "name": "Мосэнергосбыт Клиентский офис \\"Северное Тушино\\"",\n   "office_type": "company",\n   "lat": 55.8601615,\n   "lon": 37.4482792,\n   "opening_hours": "Mo-Th 08:30-17:15; Fr 08:30-16:00"\n  },\n  {\n   "id": "node/3642979096",\n   "kind": "organisation",\n   "name": "Лемикс",\n   "office_type": "notary",\n   "lat": 55.8140737,\n   "lon": 37.5765254,\n   "opening_hours": "Mo-Sa 10:00-17:00"\n  },\n  {\n   "id": "node/3644652297",\n   "kind": "organisation",\n   "name": "Studio Ventuno",\n   "office_type": "company",\n   "lat": 55.7720489,\n   "lon": 37.5859157,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3646383656",\n   "kind": "organisation",\n   "name": "негосударственный пенсионный фонд \\"Лукойл-Гарант\\"",\n   "office_type": "company",\n   "lat": 55.7348137,\n   "lon": 37.5883421,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/3646383657",\n   "kind": "organisation",\n   "name": "Пенсионный фонд электроэнергетики",\n   "office_type": "company",\n   "lat": 55.7347887,\n   "lon": 37.5883513,\n   "opening_hours": "Mo-Th 09:30-17:00; Fr 09:30-16:00"\n  },\n  {\n   "id": "node/3646415993",\n   "kind": "organisation",\n   "name": "Cordiant",\n   "office_type": "research",\n   "lat": 55.7319153,\n   "lon": 37.5750176,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3646476447",\n   "kind": "organisation",\n   "name": "Русская факторинговая компания",\n   "office_type": "company",\n   "lat": 55.735866,\n   "lon": 37.5886242,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/3646476448",\n   "kind": "organisation",\n   "name": "Строй-инвест Атропат",\n   "office_type": "company",\n   "lat": 55.7358258,\n   "lon": 37.5886112,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3646476449",\n   "kind": "organisation",\n   "name": "Инвестиционная компания \\"Спарта-финанс\\"",\n   "office_type": "company",\n   "lat": 55.7358348,\n   "lon": 37.5885812,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3646541390",\n   "kind": "organisation",\n   "name": "Оргмонтажпроект",\n   "office_type": "company",\n   "lat": 55.7256166,\n   "lon": 37.5851913,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3647237361",\n   "kind": "organisation",\n   "name": "Телефон.ру",\n   "office_type": "telecommunication",\n   "lat": 55.7996529,\n   "lon": 37.5354018,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3650124638",\n   "kind": "organisation",\n   "name": "Редакция газеты Москва За Калужской заставой",\n   "office_type": "newspaper",\n   "lat": 55.6766901,\n   "lon": 37.5987708,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3655145825",\n   "kind": "organisation",\n   "name": "Арсеева Ольга Владимировна",\n   "office_type": "notary",\n   "lat": 55.7656259,\n   "lon": 37.6352434,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3657466497",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.6675553,\n   "lon": 37.5537277,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3658494025",\n   "kind": "organisation",\n   "name": "Представительство Красноярского края",\n   "office_type": "government",\n   "lat": 55.764941,\n   "lon": 37.622978,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3664063724",\n   "kind": "organisation",\n   "name": "Адвокатская контора",\n   "office_type": "lawyer",\n   "lat": 55.7643402,\n   "lon": 37.6574429,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3667121825",\n   "kind": "organisation",\n   "name": "ОДС Жилищник района Арбат",\n   "office_type": "property_management",\n   "lat": 55.7509445,\n   "lon": 37.591592,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3668833696",\n   "kind": "organisation",\n   "name": "Ирис Дизайн",\n   "office_type": "advertising_agency",\n   "lat": 55.724084,\n   "lon": 37.6325281,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3668847394",\n   "kind": "organisation",\n   "name": "Интернет-магазин \\"ТАБАКА.НЕТ\\"",\n   "office_type": "company",\n   "lat": 55.7240532,\n   "lon": 37.6324718,\n   "opening_hours": "Mo-Fr 10:00-22:00"\n  },\n  {\n   "id": "node/3670349304",\n   "kind": "organisation",\n   "name": "ГУ ИС ОДС-484",\n   "office_type": "government",\n   "lat": 55.608213,\n   "lon": 37.7532548,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3671706276",\n   "kind": "organisation",\n   "name": "Принтер-Плоттер.ру",\n   "office_type": "company",\n   "lat": 55.8625432,\n   "lon": 37.6590163,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3671757064",\n   "kind": "organisation",\n   "name": "Союз художников России",\n   "office_type": "ngo",\n   "lat": 55.7615408,\n   "lon": 37.6508718,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3672501999",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.652014,\n   "lon": 37.4815734,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3672502000",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.71734,\n   "lon": 37.59639,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3672505707",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.6028115,\n   "lon": 37.5290996,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/3672547380",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.860491,\n   "lon": 37.5673702,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3672554246",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.676676,\n   "lon": 37.759686,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3672862625",\n   "kind": "organisation",\n   "name": "Аргументы недели",\n   "office_type": "newspaper",\n   "lat": 55.762367,\n   "lon": 37.610458,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3673568557",\n   "kind": "organisation",\n   "name": "Festo",\n   "office_type": "company",\n   "lat": 55.6866824,\n   "lon": 37.4810055,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3678839337",\n   "kind": "organisation",\n   "name": "AKBmag.ru",\n   "office_type": "company",\n   "lat": 55.8180974,\n   "lon": 37.7310703,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3678859986",\n   "kind": "organisation",\n   "name": "AKBmag.ru",\n   "office_type": "company",\n   "lat": 55.6660033,\n   "lon": 37.5160367,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3679213635",\n   "kind": "organisation",\n   "name": "Военная коллегия адвокатов",\n   "office_type": "lawyer",\n   "lat": 55.7678307,\n   "lon": 37.6658655,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3679968560",\n   "kind": "organisation",\n   "name": "Альянс",\n   "office_type": "lawyer",\n   "lat": 55.7722147,\n   "lon": 37.5760042,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3680841352",\n   "kind": "organisation",\n   "name": "RFG Русская Фильм Группа",\n   "office_type": "company",\n   "lat": 55.786826,\n   "lon": 37.5805953,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3681904312",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7649691,\n   "lon": 37.7018144,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/3683780337",\n   "kind": "organisation",\n   "name": "Топ.Ри.Инвест",\n   "office_type": "company",\n   "lat": 55.7862041,\n   "lon": 37.5798051,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3689283566",\n   "kind": "organisation",\n   "name": "ООО \\"Империя кроссовок\\"",\n   "office_type": "company",\n   "lat": 55.8508163,\n   "lon": 37.6788358,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3691221956",\n   "kind": "organisation",\n   "name": "Фонд Сергея Безрукова",\n   "office_type": "theatre",\n   "lat": 55.708105,\n   "lon": 37.7675322,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3691311445",\n   "kind": "organisation",\n   "name": "Театр Михаила Звездинского",\n   "office_type": "company",\n   "lat": 55.7403235,\n   "lon": 37.5801796,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3692078151",\n   "kind": "organisation",\n   "name": "Плавбаза КЮМ",\n   "office_type": "company",\n   "lat": 55.8400967,\n   "lon": 37.4774669,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3696456384",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.7513621,\n   "lon": 37.7857421,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3699842355",\n   "kind": "organisation",\n   "name": "Прокуратура ЦАО",\n   "office_type": "government",\n   "lat": 55.7323448,\n   "lon": 37.5899782,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3701005144",\n   "kind": "organisation",\n   "name": "Согаз",\n   "office_type": "insurance",\n   "lat": 55.7688781,\n   "lon": 37.6421393,\n   "opening_hours": "Mo-Th 08:45-18:30; Fr 08:45-17:30"\n  },\n  {\n   "id": "node/3702356925",\n   "kind": "organisation",\n   "name": "Pronetworking",\n   "office_type": "company",\n   "lat": 55.7997382,\n   "lon": 37.5307598,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3702568305",\n   "kind": "organisation",\n   "name": "Гемотест",\n   "office_type": "research",\n   "lat": 55.7162817,\n   "lon": 37.6218724,\n   "opening_hours": "Mo-Fr 07:30-19:30; Sa-Su 07:30-14:30"\n  },\n  {\n   "id": "node/3702713431",\n   "kind": "organisation",\n   "name": "Messe Düsseldorf",\n   "office_type": "company",\n   "lat": 55.7326,\n   "lon": 37.5916039,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3702713456",\n   "kind": "organisation",\n   "name": "Росавтур",\n   "office_type": "company",\n   "lat": 55.732603,\n   "lon": 37.589312,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3704170104",\n   "kind": "organisation",\n   "name": "НИИ Ипромашпром",\n   "office_type": "research",\n   "lat": 55.7809879,\n   "lon": 37.6139941,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3704476092",\n   "kind": "organisation",\n   "name": "ИФНС России №34 по г. Москве",\n   "office_type": "government",\n   "lat": 55.7758118,\n   "lon": 37.5049612,\n   "opening_hours": "Mo,We 09:00-18:00; Tu,Th 09:00-20:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/3706836548",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Ломоносовский",\n   "office_type": "government",\n   "lat": 55.6804325,\n   "lon": 37.5155427,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3706836549",\n   "kind": "organisation",\n   "name": "Управа района Ломоносовский",\n   "office_type": "government",\n   "lat": 55.6770071,\n   "lon": 37.5196116,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3707616006",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7167765,\n   "lon": 37.7927559,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3708364249",\n   "kind": "organisation",\n   "name": "Московский социальный регистр",\n   "office_type": "government",\n   "lat": 55.7258556,\n   "lon": 37.6253143,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3711992727",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7942391,\n   "lon": 37.5867898,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3713494730",\n   "kind": "organisation",\n   "name": "ООО «ДипломТайм»",\n   "office_type": "company",\n   "lat": 55.7463412,\n   "lon": 37.5984443,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3716041751",\n   "kind": "organisation",\n   "name": "Аквасторож",\n   "office_type": "company",\n   "lat": 55.8758798,\n   "lon": 37.5950038,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3717996325",\n   "kind": "organisation",\n   "name": "Управа Войковского района",\n   "office_type": "government",\n   "lat": 55.815797,\n   "lon": 37.503953,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3718469074",\n   "kind": "organisation",\n   "name": "ООО \\"Бриколина\\"",\n   "office_type": "company",\n   "lat": 55.7049867,\n   "lon": 37.44658,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3723061611",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8697457,\n   "lon": 37.6610755,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3723933974",\n   "kind": "organisation",\n   "name": "3S-Telematica",\n   "office_type": "it",\n   "lat": 55.6797694,\n   "lon": 37.6230213,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3723978794",\n   "kind": "organisation",\n   "name": "Русский АвтоМотоКлуб (РАМК)",\n   "office_type": "company",\n   "lat": 55.8765403,\n   "lon": 37.5895685,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/3726001869",\n   "kind": "organisation",\n   "name": "МГТС",\n   "office_type": "telecommunication",\n   "lat": 55.740532,\n   "lon": 37.62502,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3732264699",\n   "kind": "organisation",\n   "name": "Инком",\n   "office_type": "estate_agent",\n   "lat": 55.7424249,\n   "lon": 37.5498402,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa,Su 10:00-17:00"\n  },\n  {\n   "id": "node/3734462801",\n   "kind": "organisation",\n   "name": "Терминал DPD",\n   "office_type": "logistics",\n   "lat": 55.8846578,\n   "lon": 37.6291103,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3737336775",\n   "kind": "organisation",\n   "name": "Типография Виктория-Друк",\n   "office_type": "company",\n   "lat": 55.7586241,\n   "lon": 37.7350183,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3737540625",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7170334,\n   "lon": 37.7922318,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/3742291396",\n   "kind": "organisation",\n   "name": "Центр занятости населелния СЗАО",\n   "office_type": "government",\n   "lat": 55.7747971,\n   "lon": 37.4921567,\n   "opening_hours": "Mo, We, Fr 09:00-17:00; Tu 09:00-20:00; Th 09:00-19:00"\n  },\n  {\n   "id": "node/3742677148",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7226202,\n   "lon": 37.8242236,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3744711928",\n   "kind": "organisation",\n   "name": "Департамент государственной службы и кадров МВД России",\n   "office_type": "government",\n   "lat": 55.7569872,\n   "lon": 37.6086066,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3747702232",\n   "kind": "organisation",\n   "name": "Отклик",\n   "office_type": "company",\n   "lat": 55.7439025,\n   "lon": 37.6267033,\n   "opening_hours": "09:00-18:00"\n  },\n  {\n   "id": "node/3749366418",\n   "kind": "organisation",\n   "name": "Coral Travel",\n   "office_type": "travel_agent",\n   "lat": 55.7363824,\n   "lon": 37.5903672,\n   "opening_hours": "Mo-Su 10:00-20:15"\n  },\n  {\n   "id": "node/3750376527",\n   "kind": "organisation",\n   "name": "Вычислительный центр МТУСИ",\n   "office_type": "research",\n   "lat": 55.7555384,\n   "lon": 37.7155918,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3753935396",\n   "kind": "organisation",\n   "name": "Налоговая инспекция",\n   "office_type": "government",\n   "lat": 55.7501472,\n   "lon": 37.7762343,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3754134881",\n   "kind": "organisation",\n   "name": "МосгорБТИ",\n   "office_type": "property_management",\n   "lat": 55.7947405,\n   "lon": 37.7100674,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/3758306926",\n   "kind": "organisation",\n   "name": "Вычислительный центр Министерства обороны России",\n   "office_type": "government",\n   "lat": 55.7289562,\n   "lon": 37.5806568,\n   "opening_hours": "\\"нет публичного доступа\\""\n  },\n  {\n   "id": "node/3758931768",\n   "kind": "organisation",\n   "name": "ООО \\"Ситибет\\"",\n   "office_type": "company",\n   "lat": 55.7372854,\n   "lon": 37.5908299,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3758975754",\n   "kind": "organisation",\n   "name": "Casa del Sol",\n   "office_type": "company",\n   "lat": 55.8503294,\n   "lon": 37.4442903,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3760414968",\n   "kind": "organisation",\n   "name": "Строительная Страховая Группа",\n   "office_type": "insurance",\n   "lat": 55.8292855,\n   "lon": 37.6369988,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3762194118",\n   "kind": "organisation",\n   "name": "РЕСО-гарантия",\n   "office_type": "insurance",\n   "lat": 55.6147424,\n   "lon": 37.6045585,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3766911682",\n   "kind": "organisation",\n   "name": "ООО \\"Тех Принт\\"",\n   "office_type": "company",\n   "lat": 55.7749107,\n   "lon": 37.6696706,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3775373106",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.7988369,\n   "lon": 37.5473484,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3784442838",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "property_management",\n   "lat": 55.8462308,\n   "lon": 37.4852671,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3784442844",\n   "kind": "organisation",\n   "name": "Инженерная служба района Головинский",\n   "office_type": "property_management",\n   "lat": 55.849696,\n   "lon": 37.5156066,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3786104997",\n   "kind": "organisation",\n   "name": "Интернет-магазин \\"Лепной Декор.ру\\"",\n   "office_type": "company",\n   "lat": 55.6340365,\n   "lon": 37.6447696,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3787556766",\n   "kind": "organisation",\n   "name": "Департамент городского имущества г. Москвы",\n   "office_type": "government",\n   "lat": 55.7501037,\n   "lon": 37.5344992,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3790152254",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.6106475,\n   "lon": 37.6015498,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3790542498",\n   "kind": "organisation",\n   "name": "УФМС",\n   "office_type": "government",\n   "lat": 55.6464714,\n   "lon": 37.4795682,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3792446184",\n   "kind": "organisation",\n   "name": "Адвокатский кабинет Нуцаловой А. А.",\n   "office_type": "lawyer",\n   "lat": 55.8467487,\n   "lon": 37.5688616,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/3796795786",\n   "kind": "organisation",\n   "name": "ООО \\"Омега-2\\"",\n   "office_type": "company",\n   "lat": 55.7461846,\n   "lon": 37.4924338,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3798171783",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7326379,\n   "lon": 37.8289441,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3799105553",\n   "kind": "organisation",\n   "name": "Жилищник Лосиноостровского района",\n   "office_type": "property_management",\n   "lat": 55.8854769,\n   "lon": 37.6853083,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3800654308",\n   "kind": "organisation",\n   "name": "Правовая звезда",\n   "office_type": "lawyer",\n   "lat": 55.7829447,\n   "lon": 37.6348027,\n   "opening_hours": "09:00-24:00"\n  },\n  {\n   "id": "node/3802217438",\n   "kind": "organisation",\n   "name": "Межтрансавто",\n   "office_type": "company",\n   "lat": 55.7874621,\n   "lon": 37.6322611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3804722756",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7062309,\n   "lon": 37.6570156,\n   "opening_hours": "09:00-22:00"\n  },\n  {\n   "id": "node/3805697945",\n   "kind": "organisation",\n   "name": "Акадо",\n   "office_type": "telecommunication",\n   "lat": 55.728502,\n   "lon": 37.6692432,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3806506328",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8272585,\n   "lon": 37.4454933,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3808286574",\n   "kind": "organisation",\n   "name": "НЦ \\"Износостойкость\\"",\n   "office_type": "research",\n   "lat": 55.7549947,\n   "lon": 37.7090764,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3820185276",\n   "kind": "organisation",\n   "name": "StepDir",\n   "office_type": "company",\n   "lat": 55.8026552,\n   "lon": 37.6469204,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3823427482",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8226747,\n   "lon": 37.8235518,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/3825362757",\n   "kind": "organisation",\n   "name": "roomolo.ru",\n   "office_type": "company",\n   "lat": 55.8531959,\n   "lon": 37.5588264,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3830519198",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8743386,\n   "lon": 37.5236757,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3835715209",\n   "kind": "organisation",\n   "name": "Договорно-правовой департамент МВД России",\n   "office_type": "government",\n   "lat": 55.7309818,\n   "lon": 37.6182893,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3835715213",\n   "kind": "organisation",\n   "name": "Финансово-экономический департамент МВД России",\n   "office_type": "government",\n   "lat": 55.7307744,\n   "lon": 37.6184932,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3840381335",\n   "kind": "organisation",\n   "name": "ООО «Директ-Медиа»",\n   "office_type": "company",\n   "lat": 55.6544719,\n   "lon": 37.5389971,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3846629557",\n   "kind": "organisation",\n   "name": "ГосВыписка",\n   "office_type": "company",\n   "lat": 55.7215731,\n   "lon": 37.5723748,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3854231660",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6744938,\n   "lon": 37.5047064,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/3860484646",\n   "kind": "organisation",\n   "name": "Водительская медкомиссия",\n   "office_type": "physician",\n   "lat": 55.7113125,\n   "lon": 37.6111882,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/3869738090",\n   "kind": "organisation",\n   "name": "ООО \\"ФЛЭТ и Ко\\"",\n   "office_type": "architect",\n   "lat": 55.7672884,\n   "lon": 37.5666002,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3880376941",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.7143794,\n   "lon": 37.6093183,\n   "opening_hours": "Mo-Th 11:00-15:00"\n  },\n  {\n   "id": "node/3884087523",\n   "kind": "organisation",\n   "name": "Нотариус Крупнова Лейла Геннадиевна",\n   "office_type": "lawyer",\n   "lat": 55.8041274,\n   "lon": 37.6355594,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/3887445393",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг районов Останкинский и Марьина Роща",\n   "office_type": "government",\n   "lat": 55.8093235,\n   "lon": 37.6121153,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3887492857",\n   "kind": "organisation",\n   "name": "Межрегиональная инспекция ФНС России по крупнейшим налогоплательщикам №3",\n   "office_type": "government",\n   "lat": 55.7902523,\n   "lon": 37.6411706,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/3887580581",\n   "kind": "organisation",\n   "name": "ООО \\"Лоджикс-Телеком\\"",\n   "office_type": "telecommunication",\n   "lat": 55.7755898,\n   "lon": 37.5859707,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3892962319",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.706251,\n   "lon": 37.6570515,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3897252403",\n   "kind": "organisation",\n   "name": "РЕСО страхование",\n   "office_type": "insurance",\n   "lat": 55.7626543,\n   "lon": 37.657711,\n   "opening_hours": "Mo-Th 09:30-20:00; Fr 09:30-19:30; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/3897252404",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7665646,\n   "lon": 37.647232,\n   "opening_hours": "Mo-Fr 09:30-20:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/3897252405",\n   "kind": "organisation",\n   "name": "РЕСО страхование",\n   "office_type": "insurance",\n   "lat": 55.7764997,\n   "lon": 37.6319429,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/3897252406",\n   "kind": "organisation",\n   "name": "Аэросистемы",\n   "office_type": "company",\n   "lat": 55.811309,\n   "lon": 37.5782272,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa 09:00-16:00"\n  },\n  {\n   "id": "node/3902576675",\n   "kind": "organisation",\n   "name": "Аквантум",\n   "office_type": "company",\n   "lat": 55.8100674,\n   "lon": 37.832966,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/3910617060",\n   "kind": "organisation",\n   "name": "Федеральное агентство по управлению государственным имуществом",\n   "office_type": "government",\n   "lat": 55.7636024,\n   "lon": 37.5895533,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3910664209",\n   "kind": "organisation",\n   "name": "Post@-Magazine",\n   "office_type": "company",\n   "lat": 55.7591686,\n   "lon": 37.5861924,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3910682236",\n   "kind": "organisation",\n   "name": "ИнтерСервисИнвест",\n   "office_type": "company",\n   "lat": 55.7590668,\n   "lon": 37.5871888,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3910740952",\n   "kind": "organisation",\n   "name": "Московский Союз Музыкантов",\n   "office_type": "association",\n   "lat": 55.7581707,\n   "lon": 37.5908931,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3922372306",\n   "kind": "organisation",\n   "name": "Институт Конфуция",\n   "office_type": "educational_institution",\n   "lat": 55.7785121,\n   "lon": 37.5970048,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3933846333",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7884541,\n   "lon": 37.7481102,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/3933853620",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7897095,\n   "lon": 37.7520382,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3933853621",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7897592,\n   "lon": 37.7522688,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3939278827",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7815057,\n   "lon": 37.5986676,\n   "opening_hours": "Mo-Su 09:00-19:00"\n  },\n  {\n   "id": "node/3939325503",\n   "kind": "organisation",\n   "name": "Визовый центр Словении",\n   "office_type": "visa",\n   "lat": 55.7200283,\n   "lon": 37.6102083,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/3939325504",\n   "kind": "organisation",\n   "name": "Визовый центр Эстонии",\n   "office_type": "visa",\n   "lat": 55.7200153,\n   "lon": 37.6103027,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/3943945850",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8204442,\n   "lon": 37.499226,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3946173966",\n   "kind": "organisation",\n   "name": "Октава недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.7696624,\n   "lon": 37.6468645,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3946801540",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6684718,\n   "lon": 37.5176778,\n   "opening_hours": "Mo-Fr 09:00-21:00;Sa-Su 10:00-21:00"\n  },\n  {\n   "id": "node/3952575085",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8070401,\n   "lon": 37.5025417,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00"\n  },\n  {\n   "id": "node/3953350501",\n   "kind": "organisation",\n   "name": "ТПО Резерв",\n   "office_type": "company",\n   "lat": 55.7676809,\n   "lon": 37.5981607,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3954037359",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7314317,\n   "lon": 37.5920534,\n   "opening_hours": "Mo-Fr 09:30-20:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/3956982665",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7934474,\n   "lon": 37.7998184,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/3956988837",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7935061,\n   "lon": 37.8007656,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/3957292291",\n   "kind": "organisation",\n   "name": "АльфаСтрахование",\n   "office_type": "insurance",\n   "lat": 55.7202427,\n   "lon": 37.615051,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/3958034973",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.8467788,\n   "lon": 37.4988688,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3960412788",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7884845,\n   "lon": 37.7482161,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/3968804755",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7521994,\n   "lon": 37.8181125,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3968804756",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7527866,\n   "lon": 37.818158,\n   "opening_hours": "09:00-22:00"\n  },\n  {\n   "id": "node/3968840857",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7526628,\n   "lon": 37.8181016,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3968909318",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "telecommunication",\n   "lat": 55.7503575,\n   "lon": 37.7847015,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3971310339",\n   "kind": "organisation",\n   "name": "Бюро судебно-медицинской экспертизы",\n   "office_type": "government",\n   "lat": 55.7512513,\n   "lon": 37.7794063,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3973513893",\n   "kind": "organisation",\n   "name": "Торговый дом ОАО «Кизлярский электромеханический завод»",\n   "office_type": "company",\n   "lat": 55.7696276,\n   "lon": 37.5164148,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3980376848",\n   "kind": "organisation",\n   "name": "Налоговая консультация",\n   "office_type": "tax_advisor",\n   "lat": 55.8776421,\n   "lon": 37.69969,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3980376853",\n   "kind": "organisation",\n   "name": "Центр занятости населения. Отдел трудоустройства \\"Лосиноостровский\\"",\n   "office_type": "employment_agency",\n   "lat": 55.8768888,\n   "lon": 37.7008035,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3980755657",\n   "kind": "organisation",\n   "name": "Бюро регистрации несчастных случаев ГУ МВД по г. Москве",\n   "office_type": "government",\n   "lat": 55.7763441,\n   "lon": 37.6290185,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3982871567",\n   "kind": "organisation",\n   "name": "Fitbar.ru",\n   "office_type": "yes",\n   "lat": 55.805369,\n   "lon": 37.6420685,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:00"\n  },\n  {\n   "id": "node/3987075859",\n   "kind": "organisation",\n   "name": "Объединённая Диспетчерская Служба № 19-1",\n   "office_type": "property_management",\n   "lat": 55.8525725,\n   "lon": 37.5209635,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/3997460123",\n   "kind": "organisation",\n   "name": "Schlumberger",\n   "office_type": "company",\n   "lat": 55.8223695,\n   "lon": 37.4990834,\n   "opening_hours": null\n  },\n  {\n   "id": "node/3998566804",\n   "kind": "organisation",\n   "name": "Приёмная МВД",\n   "office_type": "government",\n   "lat": 55.7741028,\n   "lon": 37.6238231,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4003215567",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.80886,\n   "lon": 37.7984178,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/4011057946",\n   "kind": "organisation",\n   "name": "ЗАО «Выбор»",\n   "office_type": "company",\n   "lat": 55.8117475,\n   "lon": 37.6363025,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4018351665",\n   "kind": "organisation",\n   "name": "служба эксплуатации коллекторов",\n   "office_type": "property_management",\n   "lat": 55.6086435,\n   "lon": 37.593722,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4029780887",\n   "kind": "organisation",\n   "name": "ФГУП \\"Президент-Сервис\\"",\n   "office_type": "government",\n   "lat": 55.7471245,\n   "lon": 37.5833419,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4029853916",\n   "kind": "organisation",\n   "name": "Московская геммологическая лаборатория",\n   "office_type": "company",\n   "lat": 55.7495424,\n   "lon": 37.5896749,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4029963737",\n   "kind": "organisation",\n   "name": "Арбат",\n   "office_type": "lawyer",\n   "lat": 55.7480703,\n   "lon": 37.5841978,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4029963740",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7481897,\n   "lon": 37.5836057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4029963742",\n   "kind": "organisation",\n   "name": "Московский комитет ветеранов войны",\n   "office_type": "ngo",\n   "lat": 55.7493778,\n   "lon": 37.5872635,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4030467256",\n   "kind": "organisation",\n   "name": "Департамент образования",\n   "office_type": "government",\n   "lat": 55.6214731,\n   "lon": 37.5853132,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4037407039",\n   "kind": "organisation",\n   "name": "Нотариальная контора на Шаболовке",\n   "office_type": "notary",\n   "lat": 55.7201206,\n   "lon": 37.6102352,\n   "opening_hours": "Mo-Th 09:00-14:00,15:00-18:00; Fr 09:00-14:00,15:00-17:00"\n  },\n  {\n   "id": "node/4039956672",\n   "kind": "organisation",\n   "name": "Следственное управление по Центральному административному округу г.Москвы",\n   "office_type": "government",\n   "lat": 55.7322717,\n   "lon": 37.5898879,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4043907864",\n   "kind": "organisation",\n   "name": "Издательский дом \\"Красивые Дома\\"",\n   "office_type": "publisher",\n   "lat": 55.7786194,\n   "lon": 37.6285074,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4047059114",\n   "kind": "organisation",\n   "name": "Техносерв",\n   "office_type": "company",\n   "lat": 55.7303806,\n   "lon": 37.7372511,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4047104060",\n   "kind": "organisation",\n   "name": "Гильдия Управляющих Документацией",\n   "office_type": "association",\n   "lat": 55.7206204,\n   "lon": 37.5999675,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4052719722",\n   "kind": "organisation",\n   "name": "Министерство Российской Федерации по развитию Дальнего Востока",\n   "office_type": "government",\n   "lat": 55.7388227,\n   "lon": 37.581627,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4059338665",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6757979,\n   "lon": 37.6648346,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/4060065382",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7438827,\n   "lon": 37.5827127,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4063672237",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7356693,\n   "lon": 37.591692,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4065376853",\n   "kind": "organisation",\n   "name": "Мосэнергосбыт",\n   "office_type": "company",\n   "lat": 55.5951761,\n   "lon": 37.6570384,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4066982689",\n   "kind": "organisation",\n   "name": "Яндекс",\n   "office_type": "company",\n   "lat": 55.7339688,\n   "lon": 37.5884817,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4066984889",\n   "kind": "organisation",\n   "name": "VK",\n   "office_type": "company",\n   "lat": 55.796902,\n   "lon": 37.5377716,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4068141512",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7997448,\n   "lon": 37.5351597,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4069997691",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6245784,\n   "lon": 37.6112681,\n   "opening_hours": "Mo-Fr 10:00-14:00, 15:00-19:00; Sa 09:00-15:00"\n  },\n  {\n   "id": "node/4071266491",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7530664,\n   "lon": 37.7138163,\n   "opening_hours": "Mo-Th 10:00-18:00; Fr 10:00-16:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/4071688791",\n   "kind": "organisation",\n   "name": "Исаева Зоя Владимировна",\n   "office_type": "notary",\n   "lat": 55.8026228,\n   "lon": 37.5317749,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/4072110190",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6714449,\n   "lon": 37.4463354,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/4072110290",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6715204,\n   "lon": 37.4457893,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4073630989",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "notary",\n   "lat": 55.7465655,\n   "lon": 37.7660887,\n   "opening_hours": "Mo-Fr 10:00-17:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/4073657792",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.6714714,\n   "lon": 37.4462389,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4073821789",\n   "kind": "organisation",\n   "name": "\\"НИ и ПИ Генплана Москвы\\"",\n   "office_type": "company",\n   "lat": 55.7666217,\n   "lon": 37.5834378,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4073821889",\n   "kind": "organisation",\n   "name": "\\"НИ и ПИ Генплана Москвы\\"",\n   "office_type": "company",\n   "lat": 55.7716926,\n   "lon": 37.5937533,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4074646492",\n   "kind": "organisation",\n   "name": "НТВ",\n   "office_type": "company",\n   "lat": 55.8166039,\n   "lon": 37.6195302,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4076834973",\n   "kind": "organisation",\n   "name": "Отдел УФМС по г. Москве в Северо-Западном административном округе",\n   "office_type": "government",\n   "lat": 55.8173861,\n   "lon": 37.4487847,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4079585689",\n   "kind": "organisation",\n   "name": "Пенсионный фонд Российской Федерации",\n   "office_type": "government",\n   "lat": 55.6642272,\n   "lon": 37.4722598,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4079941089",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7976417,\n   "lon": 37.5406962,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4080201754",\n   "kind": "organisation",\n   "name": "Кворум",\n   "office_type": "lawyer",\n   "lat": 55.755041,\n   "lon": 37.5905109,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4080855437",\n   "kind": "organisation",\n   "name": "Богородский СОБЕС",\n   "office_type": "government",\n   "lat": 55.8107276,\n   "lon": 37.7207422,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4081065696",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6839927,\n   "lon": 37.6215483,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4081065698",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6839265,\n   "lon": 37.6214344,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4081196093",\n   "kind": "organisation",\n   "name": "Консульский отдел посольства Хорватии",\n   "office_type": "diplomatic",\n   "lat": 55.7402931,\n   "lon": 37.598203,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4085211671",\n   "kind": "organisation",\n   "name": "Филиал адвокатской консультации МРКО",\n   "office_type": "lawyer",\n   "lat": 55.7546451,\n   "lon": 37.5871789,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4086257279",\n   "kind": "organisation",\n   "name": "Институт социально-политических исследований РАН",\n   "office_type": "research",\n   "lat": 55.7545697,\n   "lon": 37.5893385,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4086983089",\n   "kind": "organisation",\n   "name": "Московский дом ресторатора",\n   "office_type": "company",\n   "lat": 55.8052106,\n   "lon": 37.5858332,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4092709209",\n   "kind": "organisation",\n   "name": "Контек",\n   "office_type": "company",\n   "lat": 55.7330544,\n   "lon": 37.5943327,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4092709210",\n   "kind": "organisation",\n   "name": "Акимова А.А.",\n   "office_type": "notary",\n   "lat": 55.7330869,\n   "lon": 37.5941141,\n   "opening_hours": "Tu-Fr 10:00-17:00; Sa 10:30-16:00"\n  },\n  {\n   "id": "node/4099778314",\n   "kind": "organisation",\n   "name": "Общественная организация пенсионеров, ветеранов войны, труда, вооруженных сил и правоохранительных органов района Хамовники",\n   "office_type": "ngo",\n   "lat": 55.7324714,\n   "lon": 37.5937007,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4099778315",\n   "kind": "organisation",\n   "name": "Совет общественной организации инвалидов Великой Отечественной войны района Хамовники",\n   "office_type": "ngo",\n   "lat": 55.7325243,\n   "lon": 37.5937356,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4101227490",\n   "kind": "organisation",\n   "name": "Центр досуга и спорта Лосинка",\n   "office_type": "government",\n   "lat": 55.8714808,\n   "lon": 37.6911368,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4101227789",\n   "kind": "organisation",\n   "name": "Общественная приемная Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.8716905,\n   "lon": 37.6914599,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4103462055",\n   "kind": "organisation",\n   "name": "Адвокатская контора № 19",\n   "office_type": "lawyer",\n   "lat": 55.7825185,\n   "lon": 37.7071995,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4103696594",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6413062,\n   "lon": 37.5297932,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4103696595",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6412794,\n   "lon": 37.5298957,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4104580589",\n   "kind": "organisation",\n   "name": "Столичный филиал ПАО МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7975727,\n   "lon": 37.5797276,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4108074390",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6631765,\n   "lon": 37.4815817,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4108074489",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6629964,\n   "lon": 37.4815427,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/4109624322",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8084944,\n   "lon": 37.5307325,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4111206589",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7181415,\n   "lon": 37.6740799,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4112411291",\n   "kind": "organisation",\n   "name": "ООО\\"Автотюнинг\\"",\n   "office_type": "company",\n   "lat": 55.8575592,\n   "lon": 37.5831361,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4113603389",\n   "kind": "organisation",\n   "name": "Головинская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7983458,\n   "lon": 37.5324694,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4114718390",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "telecommunication",\n   "lat": 55.7960388,\n   "lon": 37.798993,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4115989790",\n   "kind": "organisation",\n   "name": "Научный центр по комплексным транспортным проблемам Министерства транспорта Российской Федерации",\n   "office_type": "research",\n   "lat": 55.7688971,\n   "lon": 37.6263801,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4115989791",\n   "kind": "organisation",\n   "name": "Управление внутренних дел на Московском метрополитене",\n   "office_type": "government",\n   "lat": 55.7679789,\n   "lon": 37.6255244,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4116098709",\n   "kind": "organisation",\n   "name": "CDC, группа компаний",\n   "office_type": "it",\n   "lat": 55.7410492,\n   "lon": 37.7855967,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4116180047",\n   "kind": "organisation",\n   "name": "ИФНС №21",\n   "office_type": "government",\n   "lat": 55.7098089,\n   "lon": 37.8024014,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4116972111",\n   "kind": "organisation",\n   "name": "ФК Групп",\n   "office_type": "lawyer",\n   "lat": 55.7067586,\n   "lon": 37.6586238,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4117004873",\n   "kind": "organisation",\n   "name": "SyrianAir",\n   "office_type": "company",\n   "lat": 55.7311424,\n   "lon": 37.5933594,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4117083908",\n   "kind": "organisation",\n   "name": "Наука права",\n   "office_type": "company",\n   "lat": 55.767157,\n   "lon": 37.6773062,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4117801489",\n   "kind": "organisation",\n   "name": "Посольство Люксембурга",\n   "office_type": "diplomatic",\n   "lat": 55.7439119,\n   "lon": 37.5965132,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/4118187289",\n   "kind": "organisation",\n   "name": "МБУ \\"Лидер\\"",\n   "office_type": "government",\n   "lat": 55.8582866,\n   "lon": 37.5159552,\n   "opening_hours": "Mo-Sa 14:00-21:00"\n  },\n  {\n   "id": "node/4118410725",\n   "kind": "organisation",\n   "name": "Творческая экспериментальная студия",\n   "office_type": "advertising_agency",\n   "lat": 55.7271321,\n   "lon": 37.6266891,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4118652990",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты населения района Аэропорт",\n   "office_type": "government",\n   "lat": 55.8082428,\n   "lon": 37.5335999,\n   "opening_hours": "Mo 11:00-13:45,14:30-20:00; Tu-Th 09:00-13:45,14:30-16:45; Fr 09:00-13:45,14:30-16:45"\n  },\n  {\n   "id": "node/4118653291",\n   "kind": "organisation",\n   "name": "Клиентская служба ПФР",\n   "office_type": "government",\n   "lat": 55.8084911,\n   "lon": 37.5339623,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00, Fr 09:00-16:45"\n  },\n  {\n   "id": "node/4118819890",\n   "kind": "organisation",\n   "name": "Управление Федеральной антимонопольной службы по г. Москве",\n   "office_type": "government",\n   "lat": 55.7681035,\n   "lon": 37.6470061,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/4119354791",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7451225,\n   "lon": 37.56451,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4120856490",\n   "kind": "organisation",\n   "name": "Мегафон",\n   "office_type": "telecommunication",\n   "lat": 55.7695026,\n   "lon": 37.5840052,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4124116617",\n   "kind": "organisation",\n   "name": "Профсоюз работников здравоохранения МО",\n   "office_type": "association",\n   "lat": 55.6792752,\n   "lon": 37.5228525,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4124173990",\n   "kind": "organisation",\n   "name": "Евразийская экономическая комиссия",\n   "office_type": "government",\n   "lat": 55.7395401,\n   "lon": 37.5840263,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4124619990",\n   "kind": "organisation",\n   "name": "Gowling WLG",\n   "office_type": "lawyer",\n   "lat": 55.7459004,\n   "lon": 37.5995242,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/4125584992",\n   "kind": "organisation",\n   "name": "Samsung Electronics",\n   "office_type": "company",\n   "lat": 55.7534671,\n   "lon": 37.6059176,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4126275125",\n   "kind": "organisation",\n   "name": "Юзетикс",\n   "office_type": "it",\n   "lat": 55.7588296,\n   "lon": 37.638462,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4127984319",\n   "kind": "organisation",\n   "name": "ООО «Молл лист»",\n   "office_type": "company",\n   "lat": 55.7806172,\n   "lon": 37.7156219,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4128591491",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник Головинского района",\n   "office_type": "property_management",\n   "lat": 55.8423969,\n   "lon": 37.5227534,\n   "opening_hours": "Mo-Fr 08:00-12:00,13:00-17:00"\n  },\n  {\n   "id": "node/4129975060",\n   "kind": "organisation",\n   "name": "Мосэнергосбыт Клиентский офис Алтуфьевский",\n   "office_type": "government",\n   "lat": 55.8830195,\n   "lon": 37.596998,\n   "opening_hours": "Mo-Th 08:30-20:00; Fr 08:30-19:00"\n  },\n  {\n   "id": "node/4130793889",\n   "kind": "organisation",\n   "name": "Hines International In",\n   "office_type": "architect",\n   "lat": 55.7686025,\n   "lon": 37.5903493,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4132722941",\n   "kind": "organisation",\n   "name": "ФГУП паспортно-визовый сервис",\n   "office_type": "government",\n   "lat": 55.7921876,\n   "lon": 37.5492199,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4132745179",\n   "kind": "organisation",\n   "name": "Некоммерческая организация международный фонд социально-экономических исследований",\n   "office_type": "ngo",\n   "lat": 55.7967348,\n   "lon": 37.5389009,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4132932199",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Донской",\n   "office_type": "government",\n   "lat": 55.7075456,\n   "lon": 37.6005296,\n   "opening_hours": "08:00-20:00"\n  },\n  {\n   "id": "node/4132978011",\n   "kind": "organisation",\n   "name": "Секретариат Совета министров обороны государств — участников Содружества Независимых Государств",\n   "office_type": "government",\n   "lat": 55.7963505,\n   "lon": 37.5353251,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4133086389",\n   "kind": "organisation",\n   "name": "Управление федеральной миграционной службы района Аэропорт (УФМС)",\n   "office_type": "government",\n   "lat": 55.7995658,\n   "lon": 37.5281509,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4133293040",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7997277,\n   "lon": 37.535208,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4135664335",\n   "kind": "organisation",\n   "name": "Фролова Т.В.",\n   "office_type": "notary",\n   "lat": 55.7386168,\n   "lon": 37.5889834,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/4135664336",\n   "kind": "organisation",\n   "name": "ФГБУ \\"Интеробразование\\"",\n   "office_type": "government",\n   "lat": 55.7340326,\n   "lon": 37.5921839,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4137283460",\n   "kind": "organisation",\n   "name": "Генмар",\n   "office_type": "architect",\n   "lat": 55.6813048,\n   "lon": 37.5892826,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4137330695",\n   "kind": "organisation",\n   "name": "Согаз",\n   "office_type": "insurance",\n   "lat": 55.6620605,\n   "lon": 37.5466099,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4138926196",\n   "kind": "organisation",\n   "name": "Колесник И. Е.",\n   "office_type": "notary",\n   "lat": 55.7388267,\n   "lon": 37.6271083,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/4139671804",\n   "kind": "organisation",\n   "name": "ООО \\"Технотрейд групп\\"",\n   "office_type": "company",\n   "lat": 55.8061587,\n   "lon": 37.6174885,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/4141017887",\n   "kind": "organisation",\n   "name": "Миэль",\n   "office_type": "estate_agent",\n   "lat": 55.7384967,\n   "lon": 37.6254901,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa,Su 10:00-16:00"\n  },\n  {\n   "id": "node/4141017888",\n   "kind": "organisation",\n   "name": "Посольство Республики Камерун",\n   "office_type": "diplomatic",\n   "lat": 55.7371277,\n   "lon": 37.6245683,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4142533663",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6746311,\n   "lon": 37.5582816,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa-Su 09:00-19:00"\n  },\n  {\n   "id": "node/4142624689",\n   "kind": "organisation",\n   "name": "ГК Ромашка Партнер",\n   "office_type": "company",\n   "lat": 55.8020071,\n   "lon": 37.5943078,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4142818367",\n   "kind": "organisation",\n   "name": "Армия спасения",\n   "office_type": "religion",\n   "lat": 55.7327089,\n   "lon": 37.6621966,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4142818373",\n   "kind": "organisation",\n   "name": "Умка",\n   "office_type": "association",\n   "lat": 55.731772,\n   "lon": 37.660726,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4142818374",\n   "kind": "organisation",\n   "name": "Центр инноваций и научно-технического творчества",\n   "office_type": "research",\n   "lat": 55.7329952,\n   "lon": 37.6611055,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4142818375",\n   "kind": "organisation",\n   "name": "Энергогарант",\n   "office_type": "insurance",\n   "lat": 55.733356,\n   "lon": 37.6599519,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/4144902390",\n   "kind": "organisation",\n   "name": "Кульков, Колотилов и партнеры",\n   "office_type": "lawyer",\n   "lat": 55.7393138,\n   "lon": 37.6122278,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4146820962",\n   "kind": "organisation",\n   "name": "Открытие Финансовые решения",\n   "office_type": "insurance",\n   "lat": 55.7348345,\n   "lon": 37.588172,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa 10:00-16:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/4147034394",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8239661,\n   "lon": 37.6281731,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4147052062",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7313112,\n   "lon": 37.6353635,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4148702246",\n   "kind": "organisation",\n   "name": "Ресо",\n   "office_type": "insurance",\n   "lat": 55.7579568,\n   "lon": 37.7156566,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4149712790",\n   "kind": "organisation",\n   "name": "Мортон. Отдел оформления",\n   "office_type": "estate_agent",\n   "lat": 55.7597908,\n   "lon": 37.7138153,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4151719991",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.684457,\n   "lon": 37.6241305,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4153060193",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6101224,\n   "lon": 37.6036807,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4153110390",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6100143,\n   "lon": 37.6032126,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4156181390",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8256,\n   "lon": 37.5711138,\n   "opening_hours": "Mo-Sa 10:00-20:00"\n  },\n  {\n   "id": "node/4158140160",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник района Отрадное\\"",\n   "office_type": "property_management",\n   "lat": 55.8632602,\n   "lon": 37.6215591,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4160912392",\n   "kind": "organisation",\n   "name": "Постоянное представительство Алтайского края в городе Москве",\n   "office_type": "government",\n   "lat": 55.740222,\n   "lon": 37.6198606,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:00"\n  },\n  {\n   "id": "node/4161014090",\n   "kind": "organisation",\n   "name": "Представительство правительства Мурманской области в Москве",\n   "office_type": "government",\n   "lat": 55.7565249,\n   "lon": 37.6066294,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4161937889",\n   "kind": "organisation",\n   "name": "Адвокат Бурмистров Станислав Александрович",\n   "office_type": "lawyer",\n   "lat": 55.7897769,\n   "lon": 37.5828061,\n   "opening_hours": "Tu,Th 10:00-17:00"\n  },\n  {\n   "id": "node/4162647064",\n   "kind": "organisation",\n   "name": "Департамент социальной защиты населения г. Москвы",\n   "office_type": "government",\n   "lat": 55.785581,\n   "lon": 37.6215733,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4169255910",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7142231,\n   "lon": 37.8210252,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4169258876",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7929109,\n   "lon": 37.4951507,\n   "opening_hours": "09:00-22:00"\n  },\n  {\n   "id": "node/4172545240",\n   "kind": "organisation",\n   "name": "Центральное управление федеральной службы по экологическому, технологическому и атомному надзору",\n   "office_type": "government",\n   "lat": 55.7572144,\n   "lon": 37.6079441,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4172980297",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6195685,\n   "lon": 37.752362,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4176123392",\n   "kind": "organisation",\n   "name": "РУСЗН района Академический",\n   "office_type": "government",\n   "lat": 55.6790962,\n   "lon": 37.5619009,\n   "opening_hours": "Mo 11:00-13:45,14:30-20:00; Tu-Th 09:00-13:45,14:30-18:00; Fr 09:00-13:45,14:30-16:45"\n  },\n  {\n   "id": "node/4177235410",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7799861,\n   "lon": 37.6335051,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4180536365",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6783536,\n   "lon": 37.5640697,\n   "opening_hours": "09:00-21:00"\n  },\n  {\n   "id": "node/4180874595",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг районов Левобережный, Молжаниновский, Ховрино",\n   "office_type": "government",\n   "lat": 55.8605337,\n   "lon": 37.4675129,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4181400890",\n   "kind": "organisation",\n   "name": "Профсоюз \\"МПО Газпрома\\"",\n   "office_type": "association",\n   "lat": 55.6641995,\n   "lon": 37.5623151,\n   "opening_hours": "Mo-Fr"\n  },\n  {\n   "id": "node/4182155695",\n   "kind": "organisation",\n   "name": "MS Motorservice International GmbH",\n   "office_type": "company",\n   "lat": 55.7915065,\n   "lon": 37.7055048,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/4182633393",\n   "kind": "organisation",\n   "name": "РЕСО",\n   "office_type": "insurance",\n   "lat": 55.7327443,\n   "lon": 37.5356717,\n   "opening_hours": "Mo-Su 07:00-23:00"\n  },\n  {\n   "id": "node/4193312315",\n   "kind": "organisation",\n   "name": "Zwilling",\n   "office_type": "company",\n   "lat": 55.8113412,\n   "lon": 37.8334916,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4194182596",\n   "kind": "organisation",\n   "name": "МНПП Сатурн",\n   "office_type": "company",\n   "lat": 55.7855568,\n   "lon": 37.7083778,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4198931791",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.6486247,\n   "lon": 37.7437276,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4202038726",\n   "kind": "organisation",\n   "name": "Отдел службы судебных приставов по Центральному административному округу",\n   "office_type": "government",\n   "lat": 55.7742287,\n   "lon": 37.6386882,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4202698495",\n   "kind": "organisation",\n   "name": "Главное управление Пенсионного фонда РФ №8 г. Москвы и Московской области",\n   "office_type": "government",\n   "lat": 55.6097234,\n   "lon": 37.7066016,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4203797291",\n   "kind": "organisation",\n   "name": "Позитив Текнолоджиз",\n   "office_type": "it",\n   "lat": 55.7948856,\n   "lon": 37.7125747,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4207050189",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6120723,\n   "lon": 37.6060036,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4207416694",\n   "kind": "organisation",\n   "name": "Росинтер Ресторантс",\n   "office_type": "company",\n   "lat": 55.7451475,\n   "lon": 37.7063861,\n   "opening_hours": "Mo-Fr 09:30-18:00"\n  },\n  {\n   "id": "node/4207471935",\n   "kind": "organisation",\n   "name": "Ростик Групп",\n   "office_type": "company",\n   "lat": 55.7452616,\n   "lon": 37.7063204,\n   "opening_hours": "Mo-Fr 09:30-18:00"\n  },\n  {\n   "id": "node/4208738076",\n   "kind": "organisation",\n   "name": "Городисский и Партнеры",\n   "office_type": "lawyer",\n   "lat": 55.7755848,\n   "lon": 37.6410909,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4210274700",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6506873,\n   "lon": 37.7433278,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4210274791",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.650673,\n   "lon": 37.7432158,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4215909596",\n   "kind": "organisation",\n   "name": "Патентное бюро \\"Табэлла\\"",\n   "office_type": "lawyer",\n   "lat": 55.8433098,\n   "lon": 37.5774446,\n   "opening_hours": "Mo-Fr 10:00-18:30"\n  },\n  {\n   "id": "node/4221468940",\n   "kind": "organisation",\n   "name": "ДатаПро",\n   "office_type": "it",\n   "lat": 55.7363635,\n   "lon": 37.7210695,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4221800092",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7510132,\n   "lon": 37.7840063,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4221814991",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.751031,\n   "lon": 37.7840611,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4222107992",\n   "kind": "organisation",\n   "name": "DPD",\n   "office_type": "company",\n   "lat": 55.6341573,\n   "lon": 37.6296058,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4222926789",\n   "kind": "organisation",\n   "name": "Мосэнергосбыт",\n   "office_type": "government",\n   "lat": 55.7462828,\n   "lon": 37.7900806,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4223923095",\n   "kind": "organisation",\n   "name": "Нотариус Бабак Светлана Геннадьевна",\n   "office_type": "notary",\n   "lat": 55.6965116,\n   "lon": 37.5622988,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4225209696",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6549644,\n   "lon": 37.7562605,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/4225317014",\n   "kind": "organisation",\n   "name": "Мосмонтаж",\n   "office_type": "company",\n   "lat": 55.7279659,\n   "lon": 37.6072093,\n   "opening_hours": "Mo-Th 10:00-19:00; Fr 10:00-18:00"\n  },\n  {\n   "id": "node/4225327295",\n   "kind": "organisation",\n   "name": "ДОМС",\n   "office_type": "property_management",\n   "lat": 55.8406565,\n   "lon": 37.4942904,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4225817001",\n   "kind": "organisation",\n   "name": "ГУ Банка России по Центральному федеральному округу",\n   "office_type": "government",\n   "lat": 55.7476185,\n   "lon": 37.6253138,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4226425389",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Ясенево",\n   "office_type": "government",\n   "lat": 55.6091917,\n   "lon": 37.5349132,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4227393892",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6096785,\n   "lon": 37.7199872,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4228584531",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных услуг района Дмитровский",\n   "office_type": "government",\n   "lat": 55.8898991,\n   "lon": 37.5373309,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4232039972",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг по району Гольяново",\n   "office_type": "government",\n   "lat": 55.8102036,\n   "lon": 37.7787892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4234177828",\n   "kind": "organisation",\n   "name": "iCore",\n   "office_type": "company",\n   "lat": 55.7173022,\n   "lon": 37.6777646,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4235158510",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8071591,\n   "lon": 37.6376352,\n   "opening_hours": "09:00-21:00"\n  },\n  {\n   "id": "node/4235395680",\n   "kind": "organisation",\n   "name": "МИПА - Институт ароматерапии",\n   "office_type": "government",\n   "lat": 55.6662877,\n   "lon": 37.5579582,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4235484693",\n   "kind": "organisation",\n   "name": "Управа района Тёплый стан",\n   "office_type": "government",\n   "lat": 55.6221512,\n   "lon": 37.5028883,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4235507694",\n   "kind": "organisation",\n   "name": "Объединенная Энергетическая Компания",\n   "office_type": "company",\n   "lat": 55.7218526,\n   "lon": 37.6998549,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4235874143",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7317009,\n   "lon": 37.6639474,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4235874147",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7319627,\n   "lon": 37.6642099,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4235874148",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7317909,\n   "lon": 37.6642841,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4239089489",\n   "kind": "organisation",\n   "name": "Управа района Марьино",\n   "office_type": "government",\n   "lat": 55.6529449,\n   "lon": 37.7437329,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:45"\n  },\n  {\n   "id": "node/4239918290",\n   "kind": "organisation",\n   "name": "Ушаковы, Путиловы",\n   "office_type": "lawyer",\n   "lat": 55.6704836,\n   "lon": 37.5508126,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4240940535",\n   "kind": "organisation",\n   "name": "Социально-реабилитационный центр для несовершеннолетних «Возраждение»",\n   "office_type": "government",\n   "lat": 55.8750352,\n   "lon": 37.7219485,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4242693635",\n   "kind": "organisation",\n   "name": "Комацу СНГ",\n   "office_type": "company",\n   "lat": 55.8049956,\n   "lon": 37.4909806,\n   "opening_hours": "Mo-Fr 09:00-13:00, 13:45-18:00"\n  },\n  {\n   "id": "node/4243080395",\n   "kind": "organisation",\n   "name": "Межрегиональное операционное управление Федерального казначейства",\n   "office_type": "government",\n   "lat": 55.7440183,\n   "lon": 37.6506218,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4244512793",\n   "kind": "organisation",\n   "name": "ФКУ \\"Центравтомагистраль\\"",\n   "office_type": "government",\n   "lat": 55.8087669,\n   "lon": 37.5070611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4248760026",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7255528,\n   "lon": 37.6511216,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4250571908",\n   "kind": "organisation",\n   "name": "Совет ветеранов района Марьино",\n   "office_type": "ngo",\n   "lat": 55.6490097,\n   "lon": 37.734473,\n   "opening_hours": "Tu,Th 10:00-14:00"\n  },\n  {\n   "id": "node/4250850790",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.8595222,\n   "lon": 37.5673804,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/4253298711",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7961407,\n   "lon": 37.6170549,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4254188136",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты населения Бескудниковского района",\n   "office_type": "government",\n   "lat": 55.8654831,\n   "lon": 37.5636842,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4254345192",\n   "kind": "organisation",\n   "name": "Кит",\n   "office_type": "logistics",\n   "lat": 55.7043699,\n   "lon": 37.4457838,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4258170494",\n   "kind": "organisation",\n   "name": "Посольство ЮАР",\n   "office_type": "diplomatic",\n   "lat": 55.758955,\n   "lon": 37.5943635,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4260087533",\n   "kind": "organisation",\n   "name": "Центральный научно-исследовательский институт связи",\n   "office_type": "research",\n   "lat": 55.7520118,\n   "lon": 37.7685983,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4264121958",\n   "kind": "organisation",\n   "name": "Открытые Медиа",\n   "office_type": "company",\n   "lat": 55.7437545,\n   "lon": 37.7098404,\n   "opening_hours": "10:00-18:00"\n  },\n  {\n   "id": "node/4264449792",\n   "kind": "organisation",\n   "name": "НИИ космической медицины ФГБУ ФНКЦ ФМБА России",\n   "office_type": "research",\n   "lat": 55.8019398,\n   "lon": 37.4545807,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4267850392",\n   "kind": "organisation",\n   "name": "Межрегиональный информационный центр",\n   "office_type": "government",\n   "lat": 55.7934597,\n   "lon": 37.5904515,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4268756211",\n   "kind": "organisation",\n   "name": "Башнефть",\n   "office_type": "company",\n   "lat": 55.7711874,\n   "lon": 37.5935685,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4269928494",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7742656,\n   "lon": 37.608717,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/4270927191",\n   "kind": "organisation",\n   "name": "АО \\"Атомэнергопроект\\"",\n   "office_type": "company",\n   "lat": 55.6109679,\n   "lon": 37.6112036,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4271047493",\n   "kind": "organisation",\n   "name": "Акционерное общество \\"Федеральная корпорация по развитию малого и среднего предпринимательства\\"",\n   "office_type": "government",\n   "lat": 55.7528551,\n   "lon": 37.6361525,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4271064291",\n   "kind": "organisation",\n   "name": "Социальный фонд России",\n   "office_type": "government",\n   "lat": 55.7528106,\n   "lon": 37.6361252,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:45"\n  },\n  {\n   "id": "node/4271309905",\n   "kind": "organisation",\n   "name": "ОАО \\"Электросеть\\"",\n   "office_type": "government",\n   "lat": 55.8910568,\n   "lon": 37.7151106,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-16:00"\n  },\n  {\n   "id": "node/4271332096",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7467234,\n   "lon": 37.6574962,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4278560089",\n   "kind": "organisation",\n   "name": "НППС\\"Творческие студии скульпторов\\" МСХ",\n   "office_type": "association",\n   "lat": 55.8448172,\n   "lon": 37.5791252,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4279220181",\n   "kind": "organisation",\n   "name": "Wayfinding.pro",\n   "office_type": "company",\n   "lat": 55.7754532,\n   "lon": 37.6029888,\n   "opening_hours": "Mo-Fr 11:00-20:00"\n  },\n  {\n   "id": "node/4280524656",\n   "kind": "organisation",\n   "name": "Представительство Правительства Рязанской Области",\n   "office_type": "government",\n   "lat": 55.7412473,\n   "lon": 37.6194508,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4281813405",\n   "kind": "organisation",\n   "name": "Известия",\n   "office_type": "publisher",\n   "lat": 55.7660306,\n   "lon": 37.6046679,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4284082646",\n   "kind": "organisation",\n   "name": "ГосТехНадзор №8",\n   "office_type": "government",\n   "lat": 55.752982,\n   "lon": 37.6448822,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4285776586",\n   "kind": "organisation",\n   "name": "ООО Адастра",\n   "office_type": "company",\n   "lat": 55.6892478,\n   "lon": 37.5605061,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4285828989",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Хорошёво-Мнёвники",\n   "office_type": "government",\n   "lat": 55.7750605,\n   "lon": 37.4730942,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4287108066",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7448795,\n   "lon": 37.4940826,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4287477211",\n   "kind": "organisation",\n   "name": "RU-Center",\n   "office_type": "government",\n   "lat": 55.777663,\n   "lon": 37.5030081,\n   "opening_hours": "Su-Th 09:00-19:00"\n  },\n  {\n   "id": "node/4287503589",\n   "kind": "organisation",\n   "name": "Центр юного анималиста \\"Art Zebra\\"",\n   "office_type": "government",\n   "lat": 55.7650988,\n   "lon": 37.5762824,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4290213689",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6745441,\n   "lon": 37.7605806,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4290214589",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.659321,\n   "lon": 37.7558855,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4293203450",\n   "kind": "organisation",\n   "name": "Викитревел",\n   "office_type": "company",\n   "lat": 55.7536184,\n   "lon": 37.6630332,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4293203451",\n   "kind": "organisation",\n   "name": "Визовый центр: Великобритания, Бельгия, Швейцария",\n   "office_type": "visa",\n   "lat": 55.7537495,\n   "lon": 37.6617913,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/4293413987",\n   "kind": "organisation",\n   "name": "Согаз",\n   "office_type": "insurance",\n   "lat": 55.7699508,\n   "lon": 37.6419429,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4294398084",\n   "kind": "organisation",\n   "name": "М Такси",\n   "office_type": "company",\n   "lat": 55.7528998,\n   "lon": 37.6644383,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4294497653",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "notary",\n   "lat": 55.7521057,\n   "lon": 37.6649192,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4295750831",\n   "kind": "organisation",\n   "name": "Художественная литература",\n   "office_type": "publisher",\n   "lat": 55.7697996,\n   "lon": 37.659801,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4298602377",\n   "kind": "organisation",\n   "name": "ООО \\"Компонент-Реактив\\"",\n   "office_type": "company",\n   "lat": 55.7566759,\n   "lon": 37.7250068,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/4298694435",\n   "kind": "organisation",\n   "name": "СервисСнабГаз",\n   "office_type": "company",\n   "lat": 55.6793478,\n   "lon": 37.523968,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4298694451",\n   "kind": "organisation",\n   "name": "Эско конфидеус",\n   "office_type": "company",\n   "lat": 55.6784374,\n   "lon": 37.5241195,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4298694469",\n   "kind": "organisation",\n   "name": "М.К.Марселия",\n   "office_type": "yes",\n   "lat": 55.6778188,\n   "lon": 37.5257476,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4298694615",\n   "kind": "organisation",\n   "name": "domtest.su",\n   "office_type": "company",\n   "lat": 55.6778687,\n   "lon": 37.5243596,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4298816692",\n   "kind": "organisation",\n   "name": "Телеканал «Звезда»",\n   "office_type": "company",\n   "lat": 55.8172565,\n   "lon": 37.6398863,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4300877889",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7648778,\n   "lon": 37.6162947,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4301803493",\n   "kind": "organisation",\n   "name": "Территория бизнеса",\n   "office_type": "association",\n   "lat": 55.7809762,\n   "lon": 37.6139151,\n   "opening_hours": "Mo-Fr 09:30-18:30"\n  },\n  {\n   "id": "node/4301832092",\n   "kind": "organisation",\n   "name": "Опора России",\n   "office_type": "association",\n   "lat": 55.7810072,\n   "lon": 37.6138121,\n   "opening_hours": "Mo-Fr 09:30-18:30"\n  },\n  {\n   "id": "node/4303002975",\n   "kind": "organisation",\n   "name": "Посольство Йеменской Республики",\n   "office_type": "diplomatic",\n   "lat": 55.7396084,\n   "lon": 37.5829028,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4303165697",\n   "kind": "organisation",\n   "name": "Мастерская рекламы",\n   "office_type": "advertising_agency",\n   "lat": 55.8120966,\n   "lon": 37.6499606,\n   "opening_hours": "Mo-Fr 11:00-22:00"\n  },\n  {\n   "id": "node/4303628892",\n   "kind": "organisation",\n   "name": "Федеральное агентство по управлению государственным имуществом",\n   "office_type": "government",\n   "lat": 55.7535871,\n   "lon": 37.6296757,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4305672398",\n   "kind": "organisation",\n   "name": "Катаржина",\n   "office_type": "association",\n   "lat": 55.893982,\n   "lon": 37.6118633,\n   "opening_hours": "Mo-Th 10:00-18:00; Fr 10:00-17:00; Sa,Su off"\n  },\n  {\n   "id": "node/4308709108",\n   "kind": "organisation",\n   "name": "Управление жилищно-коммунального хозяйства и благоустройства ЮАО г. Москвы",\n   "office_type": "government",\n   "lat": 55.7061561,\n   "lon": 37.6535841,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4312085795",\n   "kind": "organisation",\n   "name": "ЖЭК",\n   "office_type": "property_management",\n   "lat": 55.870353,\n   "lon": 37.595286,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4316261493",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7801869,\n   "lon": 37.6398223,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4316820204",\n   "kind": "organisation",\n   "name": "ООО «Домашние Деньги»",\n   "office_type": "company",\n   "lat": 55.7035242,\n   "lon": 37.6965862,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4316953090",\n   "kind": "organisation",\n   "name": "EcoFinance",\n   "office_type": "company",\n   "lat": 55.7397773,\n   "lon": 37.6052606,\n   "opening_hours": "Mo-Fr 10:30-19:30"\n  },\n  {\n   "id": "node/4317397995",\n   "kind": "organisation",\n   "name": "Федеральная служба исполнения наказаний Российской Федерации",\n   "office_type": "government",\n   "lat": 55.730923,\n   "lon": 37.6153983,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4320667358",\n   "kind": "organisation",\n   "name": "Регистрация на экскурсию на Останкинскую башню",\n   "office_type": "yes",\n   "lat": 55.8185382,\n   "lon": 37.6158771,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4324943891",\n   "kind": "organisation",\n   "name": "Уралсиб страхование",\n   "office_type": "insurance",\n   "lat": 55.8369798,\n   "lon": 37.6455181,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4327147177",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8459467,\n   "lon": 37.4775691,\n   "opening_hours": "Fr 10:00-17:00;Mo-Th 10:00-20:00"\n  },\n  {\n   "id": "node/4328042275",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8449248,\n   "lon": 37.5680319,\n   "opening_hours": "Mo-Th 09:00-20:00; Fr 09:00-19:00; Sa 09:00-18:00"\n  },\n  {\n   "id": "node/4328042279",\n   "kind": "organisation",\n   "name": "Статус-Риэлти",\n   "office_type": "estate_agent",\n   "lat": 55.8450273,\n   "lon": 37.5679948,\n   "opening_hours": "Mo-Sa 10:00-20:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/4328081397",\n   "kind": "organisation",\n   "name": "Забория",\n   "office_type": "company",\n   "lat": 55.7493927,\n   "lon": 37.6647377,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4329739289",\n   "kind": "organisation",\n   "name": "Бадзыма А.Р.",\n   "office_type": "notary",\n   "lat": 55.6444657,\n   "lon": 37.5190988,\n   "opening_hours": "Mo-Th 10:00-17:00; Fr 10:00-15:00; Sa-Su off"\n  },\n  {\n   "id": "node/4330824161",\n   "kind": "organisation",\n   "name": "СитиГИД",\n   "office_type": "company",\n   "lat": 55.6739129,\n   "lon": 37.5046725,\n   "opening_hours": "10:00-19:00"\n  },\n  {\n   "id": "node/4331433665",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7528932,\n   "lon": 37.8252143,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4331488238",\n   "kind": "organisation",\n   "name": "ЗАО \\"Хозяйственное управление ВВЦ\\", отдел благоустройства и озеленения",\n   "office_type": "company",\n   "lat": 55.8304657,\n   "lon": 37.6200468,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4332145990",\n   "kind": "organisation",\n   "name": "Международный комитет Красного Креста",\n   "office_type": "association",\n   "lat": 55.7772423,\n   "lon": 37.639202,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4332965790",\n   "kind": "organisation",\n   "name": "Правовой альянс",\n   "office_type": "lawyer",\n   "lat": 55.7486393,\n   "lon": 37.7001021,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4333930176",\n   "kind": "organisation",\n   "name": "Департамент ЖКХ и благоустройства г. Москвы",\n   "office_type": "government",\n   "lat": 55.7555711,\n   "lon": 37.6235572,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4335096541",\n   "kind": "organisation",\n   "name": "Социальный фонд России",\n   "office_type": "government",\n   "lat": 55.7258858,\n   "lon": 37.6111671,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4335187609",\n   "kind": "organisation",\n   "name": "ITE Moscow",\n   "office_type": "company",\n   "lat": 55.7862559,\n   "lon": 37.660815,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4337419789",\n   "kind": "organisation",\n   "name": "ЖЭК",\n   "office_type": "government",\n   "lat": 55.64814,\n   "lon": 37.4966312,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4337550091",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7925013,\n   "lon": 37.4943529,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4340466665",\n   "kind": "organisation",\n   "name": "Управление по вопросам миграции ГУ МВД России по городу Москве",\n   "office_type": "government",\n   "lat": 55.7425973,\n   "lon": 37.623842,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4341300806",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8401669,\n   "lon": 37.4863053,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4341300807",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.840115,\n   "lon": 37.4863452,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4343079602",\n   "kind": "organisation",\n   "name": "посольство Зимбабве",\n   "office_type": "diplomatic",\n   "lat": 55.727562,\n   "lon": 37.6156358,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4344137197",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник района Зябликово\\"",\n   "office_type": "property_management",\n   "lat": 55.6183142,\n   "lon": 37.7350977,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4344137198",\n   "kind": "organisation",\n   "name": "Посольство Перу",\n   "office_type": "diplomatic",\n   "lat": 55.7699704,\n   "lon": 37.5982812,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/4344169985",\n   "kind": "organisation",\n   "name": "Посольство Непала",\n   "office_type": "diplomatic",\n   "lat": 55.7404426,\n   "lon": 37.5822824,\n   "opening_hours": "Mo-Fr 10:00-12:00"\n  },\n  {\n   "id": "node/4344518792",\n   "kind": "organisation",\n   "name": "Агропромкредит",\n   "office_type": "bank",\n   "lat": 55.7627116,\n   "lon": 37.6484057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4344518798",\n   "kind": "organisation",\n   "name": "Департамент городского имущества города Москвы",\n   "office_type": "government",\n   "lat": 55.7706509,\n   "lon": 37.6128433,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4344939765",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7820528,\n   "lon": 37.7040932,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4346384015",\n   "kind": "organisation",\n   "name": "ZepterHome",\n   "office_type": "company",\n   "lat": 55.7779875,\n   "lon": 37.455303,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4349029774",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичной организации №3 и №4 Красносельского района",\n   "office_type": "ngo",\n   "lat": 55.7749705,\n   "lon": 37.6384719,\n   "opening_hours": "Mo-We 13:00-15:00"\n  },\n  {\n   "id": "node/4350064932",\n   "kind": "organisation",\n   "name": "Миэль",\n   "office_type": "estate_agent",\n   "lat": 55.7471176,\n   "lon": 37.659683,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4353691892",\n   "kind": "organisation",\n   "name": "Бубнов и Партнеры",\n   "office_type": "lawyer",\n   "lat": 55.6831694,\n   "lon": 37.5854846,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4353963702",\n   "kind": "organisation",\n   "name": "LEGIUS",\n   "office_type": "lawyer",\n   "lat": 55.7160512,\n   "lon": 37.6219287,\n   "opening_hours": "Mo-Th 10:00-18:00; Fr 10:00-17:00"\n  },\n  {\n   "id": "node/4354203194",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7834768,\n   "lon": 37.6327285,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/4354949300",\n   "kind": "organisation",\n   "name": "Посольство Никарагуа",\n   "office_type": "diplomatic",\n   "lat": 55.7211224,\n   "lon": 37.5150291,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4357819090",\n   "kind": "organisation",\n   "name": "Корейский культурный центр",\n   "office_type": "diplomatic",\n   "lat": 55.7618382,\n   "lon": 37.6450525,\n   "opening_hours": "Mo-Fr 09:00-21:00"\n  },\n  {\n   "id": "node/4359025395",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8018361,\n   "lon": 37.5321414,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4361748653",\n   "kind": "organisation",\n   "name": "Департамент города Москвы по конкурентной политике",\n   "office_type": "government",\n   "lat": 55.7611963,\n   "lon": 37.6474468,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4361816902",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Обручевский",\n   "office_type": "government",\n   "lat": 55.66515,\n   "lon": 37.514271,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/4363303418",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7798596,\n   "lon": 37.667474,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4365288832",\n   "kind": "organisation",\n   "name": "Профсоюз Трудящихся-мигрантов",\n   "office_type": "ngo",\n   "lat": 55.7578086,\n   "lon": 37.6332449,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4367972811",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7792865,\n   "lon": 37.7128969,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4370606593",\n   "kind": "organisation",\n   "name": "Пенсионный фонд",\n   "office_type": "government",\n   "lat": 55.8141769,\n   "lon": 37.626403,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4373597263",\n   "kind": "organisation",\n   "name": "ООО \\"Купонейшен\\"",\n   "office_type": "company",\n   "lat": 55.8048686,\n   "lon": 37.5899102,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4374752090",\n   "kind": "organisation",\n   "name": "Федеральный центр нормирования, стандартизации и технической оценки соответствия в строительстве",\n   "office_type": "government",\n   "lat": 55.7136109,\n   "lon": 37.7210514,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4377652390",\n   "kind": "organisation",\n   "name": "\\"Даркстор\\" Самокат",\n   "office_type": "yes",\n   "lat": 55.8281391,\n   "lon": 37.8111124,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4379084229",\n   "kind": "organisation",\n   "name": "РусГаз",\n   "office_type": "company",\n   "lat": 55.8126175,\n   "lon": 37.8330408,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4380511639",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7281733,\n   "lon": 37.5798879,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4383676490",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7234418,\n   "lon": 37.610324,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4383676545",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7235283,\n   "lon": 37.6103794,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4391394389",\n   "kind": "organisation",\n   "name": "Министерство юстиции Российской федерации",\n   "office_type": "government",\n   "lat": 55.6818902,\n   "lon": 37.5642188,\n   "opening_hours": "Mo-Th 09:00-13:00,14:00-18:00; Fr 09:00-13:00,14:00-16:45"\n  },\n  {\n   "id": "node/4391405091",\n   "kind": "organisation",\n   "name": "ОТЕКО",\n   "office_type": "logistics",\n   "lat": 55.7221785,\n   "lon": 37.630399,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4391423490",\n   "kind": "organisation",\n   "name": "промышленно-финансовая корпорация Алмак",\n   "office_type": "company",\n   "lat": 55.7225107,\n   "lon": 37.6320072,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4393077589",\n   "kind": "organisation",\n   "name": "Жилищник района Преображенское",\n   "office_type": "property_management",\n   "lat": 55.8030148,\n   "lon": 37.7197137,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4393092690",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Преображенское",\n   "office_type": "government",\n   "lat": 55.7992002,\n   "lon": 37.7181721,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4394476389",\n   "kind": "organisation",\n   "name": "Центр поддержки молодежных творческих инициатив",\n   "office_type": "government",\n   "lat": 55.7664009,\n   "lon": 37.646762,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:45"\n  },\n  {\n   "id": "node/4397565291",\n   "kind": "organisation",\n   "name": "Bombardier Transportation Signal",\n   "office_type": "company",\n   "lat": 55.85449,\n   "lon": 37.6690952,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4400926397",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7299243,\n   "lon": 37.6393621,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4400952811",\n   "kind": "organisation",\n   "name": "Федеральное агентство связи",\n   "office_type": "government",\n   "lat": 55.748191,\n   "lon": 37.6598668,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4402348196",\n   "kind": "organisation",\n   "name": "ЦПО ГУВД МО",\n   "office_type": "government",\n   "lat": 55.7638599,\n   "lon": 37.5409301,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4409223489",\n   "kind": "organisation",\n   "name": "National Instruments",\n   "office_type": "company",\n   "lat": 55.6708529,\n   "lon": 37.4455398,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4411939689",\n   "kind": "organisation",\n   "name": "ТаманьНефтеГаз",\n   "office_type": "company",\n   "lat": 55.7221538,\n   "lon": 37.6303303,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4412162290",\n   "kind": "organisation",\n   "name": "Деликатесы Стерео",\n   "office_type": "company",\n   "lat": 55.7341096,\n   "lon": 37.5483491,\n   "opening_hours": "Mo-Fr 08:00-19:00"\n  },\n  {\n   "id": "node/4417772415",\n   "kind": "organisation",\n   "name": "Инженерная служба СВАО",\n   "office_type": "property_management",\n   "lat": 55.8828268,\n   "lon": 37.6605664,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:45; 12:00-12:45 off"\n  },\n  {\n   "id": "node/4422579189",\n   "kind": "organisation",\n   "name": "АСК \\" Поместье\\"",\n   "office_type": "company",\n   "lat": 55.8456354,\n   "lon": 37.5797664,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4423519095",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7234835,\n   "lon": 37.5653173,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/4423519097",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7234561,\n   "lon": 37.5653538,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4423519100",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7235109,\n   "lon": 37.5652809,\n   "opening_hours": "Mo-Fr,Su 10:00-21:00"\n  },\n  {\n   "id": "node/4423519101",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7235383,\n   "lon": 37.5652444,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4425455092",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.647428,\n   "lon": 37.527776,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4427562291",\n   "kind": "organisation",\n   "name": "Управление финансового обеспечения Министерства обороны РФ по Москве и Московской области",\n   "office_type": "government",\n   "lat": 55.7413153,\n   "lon": 37.639846,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4432758299",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7247107,\n   "lon": 37.5774903,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4433647834",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6029931,\n   "lon": 37.529444,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-19:00"\n  },\n  {\n   "id": "node/4434642700",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8043591,\n   "lon": 37.513629,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4437338403",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7827156,\n   "lon": 37.5984659,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/4438993705",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.887009,\n   "lon": 37.5232757,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4440513189",\n   "kind": "organisation",\n   "name": "Ресо страхование",\n   "office_type": "insurance",\n   "lat": 55.779863,\n   "lon": 37.51551,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4442018892",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных услуг района Щукино",\n   "office_type": "government",\n   "lat": 55.8077914,\n   "lon": 37.4648088,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4451074143",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт молочной промышленности\\"",\n   "office_type": "research",\n   "lat": 55.7230539,\n   "lon": 37.6227584,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4451074145",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Черёмушки",\n   "office_type": "government",\n   "lat": 55.6706632,\n   "lon": 37.5698492,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4451897997",\n   "kind": "organisation",\n   "name": "ООО \\"Нерудпласт\\"",\n   "office_type": "company",\n   "lat": 55.8060652,\n   "lon": 37.6175427,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4452895668",\n   "kind": "organisation",\n   "name": "Пенсионный фонд района Академический",\n   "office_type": "government",\n   "lat": 55.6885885,\n   "lon": 37.6007069,\n   "opening_hours": "Mo-Th 09:00-12:30,13:15-18:00; Fr 09:00-12:30,13:15-16:45"\n  },\n  {\n   "id": "node/4453027689",\n   "kind": "organisation",\n   "name": "Пенсионный фонд",\n   "office_type": "government",\n   "lat": 55.8172498,\n   "lon": 37.6949735,\n   "opening_hours": "Mo-Fr 08:00-18:00"\n  },\n  {\n   "id": "node/4453422770",\n   "kind": "organisation",\n   "name": "HeadHunter",\n   "office_type": "company",\n   "lat": 55.8093238,\n   "lon": 37.6286244,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4453628428",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7522347,\n   "lon": 37.5943806,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa,Su 10:00-21:00"\n  },\n  {\n   "id": "node/4454072589",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Проспект Вернадского",\n   "office_type": "government",\n   "lat": 55.676219,\n   "lon": 37.5121332,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4454906980",\n   "kind": "organisation",\n   "name": "Базис-Про",\n   "office_type": "company",\n   "lat": 55.6830955,\n   "lon": 37.6329416,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa 11:00-15:00; Su off"\n  },\n  {\n   "id": "node/4457857489",\n   "kind": "organisation",\n   "name": "Юридическое Бюро Селютина Игоря Анатольевича",\n   "office_type": "lawyer",\n   "lat": 55.6468996,\n   "lon": 37.7515568,\n   "opening_hours": "Mo-Fr"\n  },\n  {\n   "id": "node/4458505815",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6176754,\n   "lon": 37.6975242,\n   "opening_hours": "Mo-Fr 10:00-14:00, 15:00-20:00; Sa 10:00-14:00, 15:00-18:00; Su 10:00-14:00, 15:00-16:00"\n  },\n  {\n   "id": "node/4460385395",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7185191,\n   "lon": 37.5716314,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/4462366736",\n   "kind": "organisation",\n   "name": "Агро-Строительные Технологии",\n   "office_type": "company",\n   "lat": 55.5904612,\n   "lon": 37.6313976,\n   "opening_hours": "8:00 - 17:00"\n  },\n  {\n   "id": "node/4463138252",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7747529,\n   "lon": 37.6824242,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4464050116",\n   "kind": "organisation",\n   "name": "Управа Академического района",\n   "office_type": "government",\n   "lat": 55.6807875,\n   "lon": 37.5800902,\n   "opening_hours": "Mo-Th 08:00-12:30,13:15-17:00; Fr 08:00-12:30,13:15-15:45"\n  },\n  {\n   "id": "node/4464077126",\n   "kind": "organisation",\n   "name": "Миграционная служба Академического района",\n   "office_type": "government",\n   "lat": 55.6806673,\n   "lon": 37.5799749,\n   "opening_hours": "Mo 09:00-14:00,14:45-18:00; Tu 11:00-14:00,14:45-20:00; We 09:00-13:00; Th 11:00-14:00,14:45-20:00; Fr 09:00-14:00,14:45-16:45; Sa 09:00-14:00,14:45-16:00"\n  },\n  {\n   "id": "node/4474609283",\n   "kind": "organisation",\n   "name": "Камень и Технологии",\n   "office_type": "company",\n   "lat": 55.6850391,\n   "lon": 37.7364753,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4474950188",\n   "kind": "organisation",\n   "name": "Пенсионный Фонд РФ, ГУ №3, Районное Управление Кузьминки",\n   "office_type": "government",\n   "lat": 55.71486,\n   "lon": 37.7680228,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4478706595",\n   "kind": "organisation",\n   "name": "НПП Пластикам",\n   "office_type": "company",\n   "lat": 55.8966531,\n   "lon": 37.5194359,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4481631166",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.6546478,\n   "lon": 37.7370962,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4481814895",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Аэропорт",\n   "office_type": "government",\n   "lat": 55.8060631,\n   "lon": 37.5249901,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4483676293",\n   "kind": "organisation",\n   "name": "Управа района Аэропорт",\n   "office_type": "government",\n   "lat": 55.8058326,\n   "lon": 37.5258374,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/4485070799",\n   "kind": "organisation",\n   "name": "Агентство стратегического развития \\"ЦЕНТР\\"",\n   "office_type": "company",\n   "lat": 55.7530816,\n   "lon": 37.5911915,\n   "opening_hours": "10:00-19:00"\n  },\n  {\n   "id": "node/4485983789",\n   "kind": "organisation",\n   "name": "Сила дезинфекции",\n   "office_type": "company",\n   "lat": 55.6141249,\n   "lon": 37.616426,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/4487346989",\n   "kind": "organisation",\n   "name": "Другой водитель",\n   "office_type": "company",\n   "lat": 55.7471791,\n   "lon": 37.5396663,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4490143629",\n   "kind": "organisation",\n   "name": "Postgres Professional",\n   "office_type": "it",\n   "lat": 55.691818,\n   "lon": 37.5646763,\n   "opening_hours": "Mo-Fr 10:00-19:00; PH off"\n  },\n  {\n   "id": "node/4492777230",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7074647,\n   "lon": 37.5903016,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4497276612",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.6547548,\n   "lon": 37.7372938,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4499862779",\n   "kind": "organisation",\n   "name": "Bain & Company",\n   "office_type": "consulting",\n   "lat": 55.7506317,\n   "lon": 37.5838892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4500123791",\n   "kind": "organisation",\n   "name": "ФАУ \\"Главгосэкспертиза России\\"",\n   "office_type": "government",\n   "lat": 55.7604096,\n   "lon": 37.6303869,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4501150890",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичной организации #5 района Фил.парк",\n   "office_type": "ngo",\n   "lat": 55.7406113,\n   "lon": 37.5004085,\n   "opening_hours": "Mo 11:00-14:00"\n  },\n  {\n   "id": "node/4501480492",\n   "kind": "organisation",\n   "name": "Центр продаж и обслуживания МГТС \\"Багратионовский\\"",\n   "office_type": "telecommunication",\n   "lat": 55.7434057,\n   "lon": 37.5007766,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/4501783389",\n   "kind": "organisation",\n   "name": "Главное контрольное управление Московской области",\n   "office_type": "government",\n   "lat": 55.7677091,\n   "lon": 37.6319731,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4501831793",\n   "kind": "organisation",\n   "name": "Министерство ЖКХ Московской области",\n   "office_type": "government",\n   "lat": 55.7709733,\n   "lon": 37.6005611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4501831794",\n   "kind": "organisation",\n   "name": "Министерство сельского хозяйства Московской области",\n   "office_type": "government",\n   "lat": 55.7710001,\n   "lon": 37.6007269,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4501832692",\n   "kind": "organisation",\n   "name": "Уполномоченный по правам человека Московской области",\n   "office_type": "government",\n   "lat": 55.7708044,\n   "lon": 37.6008922,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4501832693",\n   "kind": "organisation",\n   "name": "Уполномоченный по правам ребёнка Московской области",\n   "office_type": "government",\n   "lat": 55.7707077,\n   "lon": 37.6008135,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4501874490",\n   "kind": "organisation",\n   "name": "Инженерная служба Тверского района",\n   "office_type": "property_management",\n   "lat": 55.771695,\n   "lon": 37.6185211,\n   "opening_hours": "Mo-Fr 08:00-12:00,12:45-17:00"\n  },\n  {\n   "id": "node/4501874491",\n   "kind": "organisation",\n   "name": "Главное управление ветеринарии Московской области",\n   "office_type": "government",\n   "lat": 55.7708879,\n   "lon": 37.6008395,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4502612311",\n   "kind": "organisation",\n   "name": "СтройЭкоГрант",\n   "office_type": "company",\n   "lat": 55.8385977,\n   "lon": 37.5498601,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4506117295",\n   "kind": "organisation",\n   "name": "Отдел ФСБ по Юго-Западному административному округу",\n   "office_type": "government",\n   "lat": 55.6881944,\n   "lon": 37.5652801,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4506353189",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7307333,\n   "lon": 37.6448412,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4506470793",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Пресненский",\n   "office_type": "government",\n   "lat": 55.748156,\n   "lon": 37.536606,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4508520531",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7836882,\n   "lon": 37.7190208,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4508520532",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7836314,\n   "lon": 37.7190384,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/4512137290",\n   "kind": "organisation",\n   "name": "МГТС",\n   "office_type": "telecommunication",\n   "lat": 55.8256243,\n   "lon": 37.4483917,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4512246289",\n   "kind": "organisation",\n   "name": "ГУП Московской области Мострансавто",\n   "office_type": "government",\n   "lat": 55.7719606,\n   "lon": 37.6185502,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4512246290",\n   "kind": "organisation",\n   "name": "ГУП Московской области Мострансавто",\n   "office_type": "government",\n   "lat": 55.7723301,\n   "lon": 37.6194001,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4512725291",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.6745645,\n   "lon": 37.7611512,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4515530735",\n   "kind": "organisation",\n   "name": "Wheely",\n   "office_type": "company",\n   "lat": 55.80429,\n   "lon": 37.5833715,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4516890321",\n   "kind": "organisation",\n   "name": "МГТС",\n   "office_type": "telecommunication",\n   "lat": 55.864285,\n   "lon": 37.5664455,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4520193190",\n   "kind": "organisation",\n   "name": "АгроНИИПроект",\n   "office_type": "research",\n   "lat": 55.7254971,\n   "lon": 37.6514696,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4521752606",\n   "kind": "organisation",\n   "name": "Департамент труда и соц.защиты филиал Тверской",\n   "office_type": "government",\n   "lat": 55.7711662,\n   "lon": 37.6047376,\n   "opening_hours": "Mo-Th 09:00-20:00; Fr 09:00-18:45; Sa 09:00-17:00"\n  },\n  {\n   "id": "node/4523489592",\n   "kind": "organisation",\n   "name": "Винт",\n   "office_type": "research",\n   "lat": 55.7532124,\n   "lon": 37.5820163,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4525226989",\n   "kind": "organisation",\n   "name": "Генеральное консульство Италии",\n   "office_type": "diplomatic",\n   "lat": 55.741648,\n   "lon": 37.6145203,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4525340891",\n   "kind": "organisation",\n   "name": "Русское Радио",\n   "office_type": "company",\n   "lat": 55.7885057,\n   "lon": 37.5006282,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4525965715",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7343003,\n   "lon": 37.6191575,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4526041152",\n   "kind": "organisation",\n   "name": "Главпулторг",\n   "office_type": "company",\n   "lat": 55.5902252,\n   "lon": 37.6706498,\n   "opening_hours": "09:00-18:00"\n  },\n  {\n   "id": "node/4528064341",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.822721,\n   "lon": 37.5896627,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4528342197",\n   "kind": "organisation",\n   "name": "Транс-КТП, ООО",\n   "office_type": "company",\n   "lat": 55.6548448,\n   "lon": 37.6015315,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4540199361",\n   "kind": "organisation",\n   "name": "Титанмед",\n   "office_type": "company",\n   "lat": 55.8206169,\n   "lon": 37.6575239,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4541598720",\n   "kind": "organisation",\n   "name": "Новартис",\n   "office_type": "company",\n   "lat": 55.80447,\n   "lon": 37.520098,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4541598722",\n   "kind": "organisation",\n   "name": "KupiVip",\n   "office_type": "company",\n   "lat": 55.7580913,\n   "lon": 37.6784009,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4544295489",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Красносельский",\n   "office_type": "government",\n   "lat": 55.7861563,\n   "lon": 37.6602454,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4547855937",\n   "kind": "organisation",\n   "name": "Новопашина У. С.",\n   "office_type": "notary",\n   "lat": 55.7810615,\n   "lon": 37.5996607,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4547863292",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6773811,\n   "lon": 37.5656675,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4549854109",\n   "kind": "organisation",\n   "name": "Factor Group",\n   "office_type": "company",\n   "lat": 55.8096042,\n   "lon": 37.4832715,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/4549854112",\n   "kind": "organisation",\n   "name": "Advanced Traiding",\n   "office_type": "company",\n   "lat": 55.8096931,\n   "lon": 37.4832635,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4549854113",\n   "kind": "organisation",\n   "name": "группа компаний Вагонмаш",\n   "office_type": "company",\n   "lat": 55.8097624,\n   "lon": 37.4832608,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4549874313",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Головинского района",\n   "office_type": "government",\n   "lat": 55.8401781,\n   "lon": 37.4922431,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4550973993",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Нагатинский Затон",\n   "office_type": "government",\n   "lat": 55.6781332,\n   "lon": 37.6549438,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4551688692",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6771055,\n   "lon": 37.7645335,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4551713391",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6769905,\n   "lon": 37.7648129,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4553819915",\n   "kind": "organisation",\n   "name": "НТВ-Плюс",\n   "office_type": "telecommunication",\n   "lat": 55.6287847,\n   "lon": 37.6210762,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4554204978",\n   "kind": "organisation",\n   "name": "ООО \\"ОнЛайн Трейд\\"",\n   "office_type": "company",\n   "lat": 55.8097609,\n   "lon": 37.4833976,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4555839216",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7061028,\n   "lon": 37.5938103,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4557393190",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7072964,\n   "lon": 37.6697738,\n   "opening_hours": "Fr 10:00-19:00;Mo-Th 10:00-20:00;Sa 10:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/4563580825",\n   "kind": "organisation",\n   "name": "Юрэнерго",\n   "office_type": "company",\n   "lat": 55.7190113,\n   "lon": 37.6138879,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/4568748318",\n   "kind": "organisation",\n   "name": "ГК Пионер",\n   "office_type": "company",\n   "lat": 55.7311707,\n   "lon": 37.573902,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4569239515",\n   "kind": "organisation",\n   "name": "Лаборатория Экспертных Исследований",\n   "office_type": "company",\n   "lat": 55.7780283,\n   "lon": 37.6285631,\n   "opening_hours": "Mo-Fr 11:00-19:00"\n  },\n  {\n   "id": "node/4572773589",\n   "kind": "organisation",\n   "name": "Гриненко Светлана Владимировна",\n   "office_type": "notary",\n   "lat": 55.7381461,\n   "lon": 37.5949985,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/4574264191",\n   "kind": "organisation",\n   "name": "Смирнов Максим Борисович",\n   "office_type": "lawyer",\n   "lat": 55.7970384,\n   "lon": 37.7157258,\n   "opening_hours": "Mo-Su 09:00-23:00"\n  },\n  {\n   "id": "node/4575703595",\n   "kind": "organisation",\n   "name": "Союз юристов",\n   "office_type": "lawyer",\n   "lat": 55.793499,\n   "lon": 37.6937493,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4577713875",\n   "kind": "organisation",\n   "name": "Отдел трудоустройства Пресненский",\n   "office_type": "government",\n   "lat": 55.7739355,\n   "lon": 37.5845363,\n   "opening_hours": "Mo-Tu 09:00-17:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/4579337813",\n   "kind": "organisation",\n   "name": "Headway Communications",\n   "office_type": "company",\n   "lat": 55.773928,\n   "lon": 37.5839404,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4579337816",\n   "kind": "organisation",\n   "name": "Волонтеры в помощь детям-сиротам",\n   "office_type": "association",\n   "lat": 55.7740129,\n   "lon": 37.5840892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4580200734",\n   "kind": "organisation",\n   "name": "МГСН",\n   "office_type": "estate_agent",\n   "lat": 55.7955092,\n   "lon": 37.7085668,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/4580223125",\n   "kind": "organisation",\n   "name": "Кузнецова Наталия Эдуардовна",\n   "office_type": "lawyer",\n   "lat": 55.7955469,\n   "lon": 37.7085067,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/4580379490",\n   "kind": "organisation",\n   "name": "Нотариус Куренцова Н.В.",\n   "office_type": "notary",\n   "lat": 55.7889174,\n   "lon": 37.4867806,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/4580483607",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8468541,\n   "lon": 37.5672716,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4580551989",\n   "kind": "organisation",\n   "name": "Открытие Брокер",\n   "office_type": "company",\n   "lat": 55.7289362,\n   "lon": 37.6435278,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4580669493",\n   "kind": "organisation",\n   "name": "ИФНС России № 1",\n   "office_type": "government",\n   "lat": 55.7614431,\n   "lon": 37.6588336,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4580998026",\n   "kind": "organisation",\n   "name": "Отделение пенсионного фонда",\n   "office_type": "government",\n   "lat": 55.8208643,\n   "lon": 37.5085249,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4581080450",\n   "kind": "organisation",\n   "name": "ООО «Витаслим24»",\n   "office_type": "company",\n   "lat": 55.7873719,\n   "lon": 37.6343081,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4581176081",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.6094575,\n   "lon": 37.7049788,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4582886895",\n   "kind": "organisation",\n   "name": "Налоговая инспекция №27",\n   "office_type": "government",\n   "lat": 55.6680699,\n   "lon": 37.5632098,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4584007451",\n   "kind": "organisation",\n   "name": "РусХим",\n   "office_type": "company",\n   "lat": 55.7563881,\n   "lon": 37.69377,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa 11:00-19:00"\n  },\n  {\n   "id": "node/4584073808",\n   "kind": "organisation",\n   "name": "ФГУП ГНЦ «НИОПИК»",\n   "office_type": "research",\n   "lat": 55.7669168,\n   "lon": 37.5907614,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4587000775",\n   "kind": "organisation",\n   "name": "Schwarzkopf & Henkel, ZAO",\n   "office_type": "company",\n   "lat": 55.7682456,\n   "lon": 37.627633,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4597870880",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6899217,\n   "lon": 37.5805484,\n   "opening_hours": "Mo-Th 09:00-20:00; Fr 09:00-19:00; Sa 09:00-18:00"\n  },\n  {\n   "id": "node/4597871092",\n   "kind": "organisation",\n   "name": "Бюро переводов, адвокаты",\n   "office_type": "lawyer",\n   "lat": 55.6897558,\n   "lon": 37.5791804,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4597871098",\n   "kind": "organisation",\n   "name": "Ваш эксперт",\n   "office_type": "estate_agent",\n   "lat": 55.6899541,\n   "lon": 37.5793106,\n   "opening_hours": "Mo-Sa 10:00-19:00"\n  },\n  {\n   "id": "node/4597871101",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6886083,\n   "lon": 37.5752871,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4598362294",\n   "kind": "organisation",\n   "name": "ОДС №3",\n   "office_type": "government",\n   "lat": 55.6600616,\n   "lon": 37.516136,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4598423826",\n   "kind": "organisation",\n   "name": "Gloria Jeans",\n   "office_type": "company",\n   "lat": 55.7602601,\n   "lon": 37.6625719,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4601699556",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Коптево",\n   "office_type": "government",\n   "lat": 55.8351821,\n   "lon": 37.5245453,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4601898590",\n   "kind": "organisation",\n   "name": "Щекощихина С. А.",\n   "office_type": "notary",\n   "lat": 55.7330831,\n   "lon": 37.4640622,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4603995963",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7474212,\n   "lon": 37.5830223,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4608733509",\n   "kind": "organisation",\n   "name": "Общество защиты прав потребителей",\n   "office_type": "government",\n   "lat": 55.7418269,\n   "lon": 37.6525035,\n   "opening_hours": "10:00-20:00"\n  },\n  {\n   "id": "node/4608993624",\n   "kind": "organisation",\n   "name": "Институт Теоретической и Экспериментальной Физики",\n   "office_type": "research",\n   "lat": 55.67764,\n   "lon": 37.5869356,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4609307004",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7454519,\n   "lon": 37.5646163,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4611431884",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6667149,\n   "lon": 37.5521026,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/4611456384",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7010749,\n   "lon": 37.58072,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/4611473952",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6829385,\n   "lon": 37.5859434,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/4611530714",\n   "kind": "organisation",\n   "name": "Сделано Лазером",\n   "office_type": "company",\n   "lat": 55.6827208,\n   "lon": 37.6178592,\n   "opening_hours": "Mo-Fr 10:00-22:00"\n  },\n  {\n   "id": "node/4612141189",\n   "kind": "organisation",\n   "name": "Independent Media",\n   "office_type": "publisher",\n   "lat": 55.7983103,\n   "lon": 37.6043567,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/4616110670",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.6687443,\n   "lon": 37.5189575,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4616110671",\n   "kind": "organisation",\n   "name": "Адвокаты",\n   "office_type": "lawyer",\n   "lat": 55.6687444,\n   "lon": 37.5189576,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4616111611",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "advertising_agency",\n   "lat": 55.6687437,\n   "lon": 37.5189576,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4616124839",\n   "kind": "organisation",\n   "name": "УФМС района Обручевский (миграционная служба)",\n   "office_type": "government",\n   "lat": 55.665127,\n   "lon": 37.5142306,\n   "opening_hours": "Mo 09:00-14:00,14:45-18:00; Tu,Th 11:00-14:00,14:45-20:00; We 09:00-13:00; Fr 09:00-14:00,14:45-16:45; Sa 09:00-14:00,14:45-16:00"\n  },\n  {\n   "id": "node/4616321031",\n   "kind": "organisation",\n   "name": "ВинХимКолор",\n   "office_type": "company",\n   "lat": 55.6779145,\n   "lon": 37.5448611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4618650343",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "translator",\n   "lat": 55.7823479,\n   "lon": 37.5996303,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4618650363",\n   "kind": "organisation",\n   "name": "ФГУП \\"Росморпорт\\"",\n   "office_type": "government",\n   "lat": 55.7815504,\n   "lon": 37.6010375,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4618650596",\n   "kind": "organisation",\n   "name": "Молодая гвардия",\n   "office_type": "publisher",\n   "lat": 55.7817204,\n   "lon": 37.6015391,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4619148239",\n   "kind": "organisation",\n   "name": "Матвеенко и партнёры",\n   "office_type": "lawyer",\n   "lat": 55.6822349,\n   "lon": 37.523164,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4619148240",\n   "kind": "organisation",\n   "name": "Отделение ЕР по ЮЗАО",\n   "office_type": "political_party",\n   "lat": 55.6822757,\n   "lon": 37.5231882,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4620022590",\n   "kind": "organisation",\n   "name": "Китайский визовый центр",\n   "office_type": "visa",\n   "lat": 55.7568592,\n   "lon": 37.6322471,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4620335791",\n   "kind": "organisation",\n   "name": "Департамент транспорта и развития дорожно-транспортной инфраструктуры города Москвы",\n   "office_type": "government",\n   "lat": 55.7743106,\n   "lon": 37.6129329,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4620456294",\n   "kind": "organisation",\n   "name": "Центр занятости населения",\n   "office_type": "government",\n   "lat": 55.6816925,\n   "lon": 37.5988876,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4620564918",\n   "kind": "organisation",\n   "name": "Газета \\"Церковный Вестник\\"",\n   "office_type": "newspaper",\n   "lat": 55.7299311,\n   "lon": 37.5613483,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4620705890",\n   "kind": "organisation",\n   "name": "Управа района Котловка",\n   "office_type": "government",\n   "lat": 55.6818078,\n   "lon": 37.5996745,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4625336391",\n   "kind": "organisation",\n   "name": "Росгеология",\n   "office_type": "government",\n   "lat": 55.7641457,\n   "lon": 37.5510349,\n   "opening_hours": "Mo-Fr 09:00-13:00, 14:00-18:00"\n  },\n  {\n   "id": "node/4625575936",\n   "kind": "organisation",\n   "name": "Поверенный",\n   "office_type": "insurance",\n   "lat": 55.681521,\n   "lon": 37.5229222,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4625631671",\n   "kind": "organisation",\n   "name": "Jeco Pigment",\n   "office_type": "chemist",\n   "lat": 55.821049,\n   "lon": 37.5200863,\n   "opening_hours": "Mo-Th 9:30-18:00; Fr 9:30-17:00"\n  },\n  {\n   "id": "node/4625880608",\n   "kind": "organisation",\n   "name": "Министерство природных ресурсов и экологии Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7614847,\n   "lon": 37.5806529,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4627082790",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8119188,\n   "lon": 37.8041024,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4631738934",\n   "kind": "organisation",\n   "name": "ЕСП-Полимер",\n   "office_type": "doityourself",\n   "lat": 55.6725287,\n   "lon": 37.544084,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4632210769",\n   "kind": "organisation",\n   "name": "КапиталАльянс",\n   "office_type": "company",\n   "lat": 55.7896729,\n   "lon": 37.8100267,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4632279889",\n   "kind": "organisation",\n   "name": "СинтезХим",\n   "office_type": "company",\n   "lat": 55.7781635,\n   "lon": 37.7326718,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4632424435",\n   "kind": "organisation",\n   "name": "КолорХим",\n   "office_type": "chemist",\n   "lat": 55.7217705,\n   "lon": 37.7751832,\n   "opening_hours": "Mo-Th 09:00-17:00; Fr 09:00-16:00; Sa-Su off"\n  },\n  {\n   "id": "node/4632459517",\n   "kind": "organisation",\n   "name": "НПП ГЦ",\n   "office_type": "company",\n   "lat": 55.6667961,\n   "lon": 37.6267557,\n   "opening_hours": "Mo-Fr 9:00-17:00; Sa 10:00-14:00; Su Off"\n  },\n  {\n   "id": "node/4634559823",\n   "kind": "organisation",\n   "name": "ГирМет",\n   "office_type": "company",\n   "lat": 55.747927,\n   "lon": 37.6623811,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4634744885",\n   "kind": "organisation",\n   "name": "WebMoney",\n   "office_type": "company",\n   "lat": 55.7287268,\n   "lon": 37.6176847,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/4635240438",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6516153,\n   "lon": 37.742366,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/4637255136",\n   "kind": "organisation",\n   "name": "Полипроф-Л",\n   "office_type": "company",\n   "lat": 55.8626442,\n   "lon": 37.6269722,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4643042838",\n   "kind": "organisation",\n   "name": "Офис продаж ЛСР",\n   "office_type": "estate_agent",\n   "lat": 55.7053211,\n   "lon": 37.6353447,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4646432090",\n   "kind": "organisation",\n   "name": "Медстрах",\n   "office_type": "government",\n   "lat": 55.7825499,\n   "lon": 37.6629071,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4646564489",\n   "kind": "organisation",\n   "name": "ОЛРР по ВАО ГУ Росгвардии по г. Москве",\n   "office_type": "government",\n   "lat": 55.7452378,\n   "lon": 37.7893183,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4648209598",\n   "kind": "organisation",\n   "name": "ЗАО \\"Сарден\\"",\n   "office_type": "company",\n   "lat": 55.8116097,\n   "lon": 37.6502008,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4653226658",\n   "kind": "organisation",\n   "name": "РСВО",\n   "office_type": "company",\n   "lat": 55.7329868,\n   "lon": 37.7010113,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4653903007",\n   "kind": "organisation",\n   "name": "посольство Марокко",\n   "office_type": "diplomatic",\n   "lat": 55.7437442,\n   "lon": 37.5942242,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/4653903010",\n   "kind": "organisation",\n   "name": "Посольство Австрии",\n   "office_type": "diplomatic",\n   "lat": 55.7436241,\n   "lon": 37.5948854,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4653903017",\n   "kind": "organisation",\n   "name": "Российский фонд культуры",\n   "office_type": "charity",\n   "lat": 55.7455496,\n   "lon": 37.6013741,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4657139496",\n   "kind": "organisation",\n   "name": "Инспекция Федеральной налоговой службы № 2 по г.Москве",\n   "office_type": "government",\n   "lat": 55.7845778,\n   "lon": 37.6419969,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4657139497",\n   "kind": "organisation",\n   "name": "Инспекция Федеральной налоговой службы № 8 по г. Москве",\n   "office_type": "government",\n   "lat": 55.7844526,\n   "lon": 37.64208,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4659855911",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6766784,\n   "lon": 37.5266229,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4659855912",\n   "kind": "organisation",\n   "name": "Инвест Консалтинг",\n   "office_type": "company",\n   "lat": 55.6766542,\n   "lon": 37.5266631,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4659855913",\n   "kind": "organisation",\n   "name": "Бюро переводов",\n   "office_type": "company",\n   "lat": 55.6766149,\n   "lon": 37.5267543,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4659855914",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6765952,\n   "lon": 37.5267999,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4659856316",\n   "kind": "organisation",\n   "name": "Конечная станция \\"Улица Кравченко\\"",\n   "office_type": "company",\n   "lat": 55.6750079,\n   "lon": 37.5250075,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4659856321",\n   "kind": "organisation",\n   "name": "Мэрри Лэйн Групп",\n   "office_type": "estate_agent",\n   "lat": 55.6744192,\n   "lon": 37.5231514,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4659856336",\n   "kind": "organisation",\n   "name": "НТЦС",\n   "office_type": "company",\n   "lat": 55.6733022,\n   "lon": 37.5256163,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4661164692",\n   "kind": "organisation",\n   "name": "Saveliev, Batanov & Partners",\n   "office_type": "lawyer",\n   "lat": 55.7543042,\n   "lon": 37.641779,\n   "opening_hours": "Mo-Fr"\n  },\n  {\n   "id": "node/4663234754",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7300936,\n   "lon": 37.6442508,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/4664376466",\n   "kind": "organisation",\n   "name": "Инком",\n   "office_type": "estate_agent",\n   "lat": 55.8077937,\n   "lon": 37.7981738,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/4666300600",\n   "kind": "organisation",\n   "name": "Всероссийский государственный центр качества и стандартизации лекарственных средств для животных и кормов",\n   "office_type": "government",\n   "lat": 55.7637748,\n   "lon": 37.5540885,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4675039664",\n   "kind": "organisation",\n   "name": "Московская городская радиотрансляционная сеть",\n   "office_type": "company",\n   "lat": 55.8078143,\n   "lon": 37.5217177,\n   "opening_hours": "Mo-Th 08:30-17:30, Fr 08:30-15:00"\n  },\n  {\n   "id": "node/4675485599",\n   "kind": "organisation",\n   "name": "Издательство \\"Высшая школа\\"",\n   "office_type": "publisher",\n   "lat": 55.7671082,\n   "lon": 37.6200895,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4678212193",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7825286,\n   "lon": 37.5992845,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682236677",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Аэропорт",\n   "office_type": "government",\n   "lat": 55.7994823,\n   "lon": 37.5277234,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4682250506",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Бабушкинского района",\n   "office_type": "government",\n   "lat": 55.8562883,\n   "lon": 37.6700653,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682250507",\n   "kind": "organisation",\n   "name": "Управа Бабушкинского района",\n   "office_type": "government",\n   "lat": 55.8562073,\n   "lon": 37.6700169,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682250508",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Лосиностровский",\n   "office_type": "government",\n   "lat": 55.8565923,\n   "lon": 37.6695846,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682250509",\n   "kind": "organisation",\n   "name": "Управа Лосиноостровского района",\n   "office_type": "government",\n   "lat": 55.8561479,\n   "lon": 37.6700115,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682250510",\n   "kind": "organisation",\n   "name": "Управа района Свиблово",\n   "office_type": "government",\n   "lat": 55.8560914,\n   "lon": 37.6699849,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682255627",\n   "kind": "organisation",\n   "name": "Администрация муниципалного округа Свиблово",\n   "office_type": "government",\n   "lat": 55.8565097,\n   "lon": 37.6695666,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682258477",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Бабушкинский",\n   "office_type": "government",\n   "lat": 55.8564499,\n   "lon": 37.6695399,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682278442",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Беговой",\n   "office_type": "government",\n   "lat": 55.7896776,\n   "lon": 37.5828621,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682287951",\n   "kind": "organisation",\n   "name": "Управление федеральной миграционной службы района Богородское и Метрогородок (УФМС)",\n   "office_type": "government",\n   "lat": 55.8114427,\n   "lon": 37.7334393,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682296143",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Бутырского района",\n   "office_type": "government",\n   "lat": 55.8285291,\n   "lon": 37.5801863,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682296144",\n   "kind": "organisation",\n   "name": "Миграционная служба района Бутырский (УФМС)",\n   "office_type": "government",\n   "lat": 55.8284303,\n   "lon": 37.5803819,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682296145",\n   "kind": "organisation",\n   "name": "Управа Бутырского района",\n   "office_type": "government",\n   "lat": 55.8287378,\n   "lon": 37.5804014,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682298400",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Войковского района",\n   "office_type": "government",\n   "lat": 55.8152337,\n   "lon": 37.5146313,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4682306094",\n   "kind": "organisation",\n   "name": "УФМС района Восточное Измайлово",\n   "office_type": "government",\n   "lat": 55.7890276,\n   "lon": 37.8094508,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682318360",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Дорогомилово",\n   "office_type": "government",\n   "lat": 55.7450688,\n   "lon": 37.5660969,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682365140",\n   "kind": "organisation",\n   "name": "УФМС района Западное Дегунино",\n   "office_type": "government",\n   "lat": 55.8726049,\n   "lon": 37.5093955,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682373916",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Котловка",\n   "office_type": "government",\n   "lat": 55.6795857,\n   "lon": 37.6064258,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682377646",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Кузьминки",\n   "office_type": "government",\n   "lat": 55.6984369,\n   "lon": 37.7557038,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682377647",\n   "kind": "organisation",\n   "name": "УФМС района Кузьминки",\n   "office_type": "government",\n   "lat": 55.6983826,\n   "lon": 37.7556581,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682385276",\n   "kind": "organisation",\n   "name": "УФМС районов Левобережный, Молжаниновский и Ховрино",\n   "office_type": "government",\n   "lat": 55.8603739,\n   "lon": 37.4676036,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682387547",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Лианозово",\n   "office_type": "government",\n   "lat": 55.8955966,\n   "lon": 37.5735422,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4682393289",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Лосиноостровский",\n   "office_type": "government",\n   "lat": 55.870988,\n   "lon": 37.6827831,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682394234",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Марфино",\n   "office_type": "government",\n   "lat": 55.8443208,\n   "lon": 37.5782189,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682394782",\n   "kind": "organisation",\n   "name": "УФМС района Марфино",\n   "office_type": "government",\n   "lat": 55.8441954,\n   "lon": 37.5780483,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682401092",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Мещанский",\n   "office_type": "government",\n   "lat": 55.784502,\n   "lon": 37.6369581,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682411310",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Орехово-Борисово Южное",\n   "office_type": "government",\n   "lat": 55.5964586,\n   "lon": 37.7216544,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682411311",\n   "kind": "organisation",\n   "name": "УФМС района Орехово-Борисово Южное",\n   "office_type": "government",\n   "lat": 55.5964657,\n   "lon": 37.7213135,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682422790",\n   "kind": "organisation",\n   "name": "УФМС района Очаково-Матвеевское",\n   "office_type": "government",\n   "lat": 55.6829066,\n   "lon": 37.4590974,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682436912",\n   "kind": "organisation",\n   "name": "УФМС района Проспект Вернадского",\n   "office_type": "government",\n   "lat": 55.6760804,\n   "lon": 37.5123601,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682437721",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Раменки",\n   "office_type": "government",\n   "lat": 55.6948637,\n   "lon": 37.4956382,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4682442491",\n   "kind": "organisation",\n   "name": "УФМС района Раменки",\n   "office_type": "government",\n   "lat": 55.6947745,\n   "lon": 37.4956757,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4682453581",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Тверской",\n   "office_type": "government",\n   "lat": 55.7677681,\n   "lon": 37.6048065,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682458469",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Хорошёвский",\n   "office_type": "government",\n   "lat": 55.7868719,\n   "lon": 37.511985,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682465495",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Чертаново Северное",\n   "office_type": "government",\n   "lat": 55.6230907,\n   "lon": 37.6117777,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/4682472472",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Якиманка",\n   "office_type": "government",\n   "lat": 55.7334025,\n   "lon": 37.6097952,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682475382",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Ярославский",\n   "office_type": "government",\n   "lat": 55.866845,\n   "lon": 37.7093187,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682504546",\n   "kind": "organisation",\n   "name": "Управа района Богородское",\n   "office_type": "government",\n   "lat": 55.8109684,\n   "lon": 37.7058668,\n   "opening_hours": "Mo-We 08:00-12:30,13:15-17:00; Fr 08:00-12:30,13:15-15:45"\n  },\n  {\n   "id": "node/4682514521",\n   "kind": "organisation",\n   "name": "Управа района Капотня",\n   "office_type": "government",\n   "lat": 55.6401629,\n   "lon": 37.7971747,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682514522",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Капотня",\n   "office_type": "government",\n   "lat": 55.6399904,\n   "lon": 37.7970912,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682521666",\n   "kind": "organisation",\n   "name": "Управа района Текстильщики",\n   "office_type": "government",\n   "lat": 55.7013956,\n   "lon": 37.74659,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682522076",\n   "kind": "organisation",\n   "name": "Управа района Нагорный",\n   "office_type": "government",\n   "lat": 55.6560321,\n   "lon": 37.6143204,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682522077",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Нагорный",\n   "office_type": "government",\n   "lat": 55.6560272,\n   "lon": 37.6144994,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682524834",\n   "kind": "organisation",\n   "name": "Управа района Люблино",\n   "office_type": "government",\n   "lat": 55.6898046,\n   "lon": 37.7370175,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682524835",\n   "kind": "organisation",\n   "name": "УФМС района Люблино",\n   "office_type": "government",\n   "lat": 55.689784,\n   "lon": 37.7372182,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682524836",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципалього округа Люблино",\n   "office_type": "government",\n   "lat": 55.6895295,\n   "lon": 37.7372502,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682525999",\n   "kind": "organisation",\n   "name": "Управа района Пресненский",\n   "office_type": "government",\n   "lat": 55.7592492,\n   "lon": 37.5565376,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682526000",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Пресненское",\n   "office_type": "government",\n   "lat": 55.7592912,\n   "lon": 37.556974,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682529008",\n   "kind": "organisation",\n   "name": "Управа района Хамовники",\n   "office_type": "government",\n   "lat": 55.7433067,\n   "lon": 37.59703,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682529009",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Хамовники",\n   "office_type": "government",\n   "lat": 55.7433643,\n   "lon": 37.5971676,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682529010",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Хамовники",\n   "office_type": "government",\n   "lat": 55.7432527,\n   "lon": 37.5970236,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682535133",\n   "kind": "organisation",\n   "name": "Управа района Кузьминки",\n   "office_type": "government",\n   "lat": 55.6987539,\n   "lon": 37.7708628,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682537651",\n   "kind": "organisation",\n   "name": "Управа района Коньково",\n   "office_type": "government",\n   "lat": 55.6327791,\n   "lon": 37.5234444,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682540937",\n   "kind": "organisation",\n   "name": "Управа района Перово",\n   "office_type": "government",\n   "lat": 55.7504928,\n   "lon": 37.7869818,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682541117",\n   "kind": "organisation",\n   "name": "Управа района Новогиреево",\n   "office_type": "government",\n   "lat": 55.7505042,\n   "lon": 37.787048,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682545285",\n   "kind": "organisation",\n   "name": "Управа района Отрадное",\n   "office_type": "government",\n   "lat": 55.8584187,\n   "lon": 37.6113227,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682548443",\n   "kind": "organisation",\n   "name": "Управа района Лефортово",\n   "office_type": "government",\n   "lat": 55.7525844,\n   "lon": 37.7027213,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682549942",\n   "kind": "organisation",\n   "name": "Управа района Нагатинский Затон",\n   "office_type": "government",\n   "lat": 55.6762755,\n   "lon": 37.692818,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682550728",\n   "kind": "organisation",\n   "name": "Управа района Выхино-Жулебино",\n   "office_type": "government",\n   "lat": 55.713257,\n   "lon": 37.8049228,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682551549",\n   "kind": "organisation",\n   "name": "Управа района Якиманка",\n   "office_type": "government",\n   "lat": 55.7252077,\n   "lon": 37.6234919,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682551550",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Якиманка",\n   "office_type": "government",\n   "lat": 55.7251984,\n   "lon": 37.6237591,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682560657",\n   "kind": "organisation",\n   "name": "Управа района Проспект Вернадского",\n   "office_type": "government",\n   "lat": 55.6767758,\n   "lon": 37.4961409,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682560658",\n   "kind": "organisation",\n   "name": "ИФНС №29 ЗАО, район Проспект Вернадского",\n   "office_type": "government",\n   "lat": 55.6766957,\n   "lon": 37.4961162,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682566247",\n   "kind": "organisation",\n   "name": "Управа района Сокол",\n   "office_type": "government",\n   "lat": 55.8046886,\n   "lon": 37.50324,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682566248",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Сокол",\n   "office_type": "government",\n   "lat": 55.8048403,\n   "lon": 37.5031726,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682566249",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Сокол",\n   "office_type": "government",\n   "lat": 55.8049365,\n   "lon": 37.5030999,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682566250",\n   "kind": "organisation",\n   "name": "Инженерная служба района Сокол",\n   "office_type": "property_management",\n   "lat": 55.8050357,\n   "lon": 37.5028041,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682570665",\n   "kind": "organisation",\n   "name": "Управа района Чертаново Северное",\n   "office_type": "government",\n   "lat": 55.6339177,\n   "lon": 37.6168898,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682570666",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Чертаново Северное",\n   "office_type": "government",\n   "lat": 55.6340613,\n   "lon": 37.616939,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682570667",\n   "kind": "organisation",\n   "name": "Нагатинская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.6338577,\n   "lon": 37.6168166,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682572707",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Бирюлево Воточное",\n   "office_type": "government",\n   "lat": 55.5851705,\n   "lon": 37.661291,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682573768",\n   "kind": "organisation",\n   "name": "Управа района Орехово-Борисово Южное",\n   "office_type": "government",\n   "lat": 55.6057189,\n   "lon": 37.7181069,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682574060",\n   "kind": "organisation",\n   "name": "Управа района Филевский Парк",\n   "office_type": "government",\n   "lat": 55.7484459,\n   "lon": 37.5002045,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682576231",\n   "kind": "organisation",\n   "name": "Управа Хорошевского района",\n   "office_type": "government",\n   "lat": 55.7791572,\n   "lon": 37.5197694,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682576232",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Хорошевский",\n   "office_type": "government",\n   "lat": 55.7860211,\n   "lon": 37.522752,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682576758",\n   "kind": "organisation",\n   "name": "Управа Алексеевского района",\n   "office_type": "government",\n   "lat": 55.8021536,\n   "lon": 37.636754,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682577735",\n   "kind": "organisation",\n   "name": "Управа Останкинского района",\n   "office_type": "government",\n   "lat": 55.8240519,\n   "lon": 37.6203334,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682578616",\n   "kind": "organisation",\n   "name": "Управа Ярославского района",\n   "office_type": "government",\n   "lat": 55.8699372,\n   "lon": 37.7122894,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682578701",\n   "kind": "organisation",\n   "name": "Управа района Медведково Северное",\n   "office_type": "government",\n   "lat": 55.8810267,\n   "lon": 37.6521541,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682578702",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Северное Медведково",\n   "office_type": "government",\n   "lat": 55.8814368,\n   "lon": 37.6492945,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682582505",\n   "kind": "organisation",\n   "name": "Управа района Орехово-Борисово Северное",\n   "office_type": "government",\n   "lat": 55.6100909,\n   "lon": 37.6979984,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682582506",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Орехово-Борисово Южное",\n   "office_type": "government",\n   "lat": 55.6071045,\n   "lon": 37.7017845,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682583708",\n   "kind": "organisation",\n   "name": "Управа района Соколиная Гора",\n   "office_type": "government",\n   "lat": 55.7828722,\n   "lon": 37.7253135,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682583709",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Соколиная Гора",\n   "office_type": "government",\n   "lat": 55.7839338,\n   "lon": 37.710295,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682603734",\n   "kind": "organisation",\n   "name": "Управа Мещанского района",\n   "office_type": "government",\n   "lat": 55.7747252,\n   "lon": 37.6319741,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682603735",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Мещанский",\n   "office_type": "government",\n   "lat": 55.7746805,\n   "lon": 37.6319794,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682603736",\n   "kind": "organisation",\n   "name": "Управа района Арбат",\n   "office_type": "government",\n   "lat": 55.7487427,\n   "lon": 37.5875621,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682603737",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Арбат",\n   "office_type": "government",\n   "lat": 55.7488275,\n   "lon": 37.587478,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682603827",\n   "kind": "organisation",\n   "name": "Управа района Восточное Дегунино",\n   "office_type": "government",\n   "lat": 55.879359,\n   "lon": 37.564989,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682603828",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Восточное Дегунино",\n   "office_type": "government",\n   "lat": 55.877932,\n   "lon": 37.5662682,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682605599",\n   "kind": "organisation",\n   "name": "Управа Южнопортового района",\n   "office_type": "government",\n   "lat": 55.7029643,\n   "lon": 37.6796906,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682605998",\n   "kind": "organisation",\n   "name": "Управа района Фили-Давыдково",\n   "office_type": "government",\n   "lat": 55.734493,\n   "lon": 37.4810763,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682605999",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Фили-Давыдково",\n   "office_type": "government",\n   "lat": 55.7333323,\n   "lon": 37.4687844,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682606121",\n   "kind": "organisation",\n   "name": "Управа Бескудниковского района",\n   "office_type": "government",\n   "lat": 55.8639516,\n   "lon": 37.561897,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682607204",\n   "kind": "organisation",\n   "name": "Управа Савеловского района",\n   "office_type": "government",\n   "lat": 55.7940746,\n   "lon": 37.5685856,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/4682607205",\n   "kind": "organisation",\n   "name": "Совет депутатов муниципального округа Савеловский",\n   "office_type": "government",\n   "lat": 55.7924719,\n   "lon": 37.5699179,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682607966",\n   "kind": "organisation",\n   "name": "Управа района Метрогородок",\n   "office_type": "government",\n   "lat": 55.8249823,\n   "lon": 37.7502847,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682608000",\n   "kind": "organisation",\n   "name": "Управа района Беговой",\n   "office_type": "government",\n   "lat": 55.7847674,\n   "lon": 37.5715843,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682608001",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Беговой",\n   "office_type": "government",\n   "lat": 55.7876836,\n   "lon": 37.5728772,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682609509",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Северное Измайлово",\n   "office_type": "government",\n   "lat": 55.8044534,\n   "lon": 37.8014448,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682610556",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Сокольники",\n   "office_type": "government",\n   "lat": 55.7884806,\n   "lon": 37.6829601,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682612381",\n   "kind": "organisation",\n   "name": "Управа Головинского района",\n   "office_type": "government",\n   "lat": 55.8512804,\n   "lon": 37.4798782,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682612609",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Головинский",\n   "office_type": "government",\n   "lat": 55.8514169,\n   "lon": 37.4797922,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682612610",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Левобережный",\n   "office_type": "government",\n   "lat": 55.851398,\n   "lon": 37.4802336,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682612611",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Ховрино",\n   "office_type": "government",\n   "lat": 55.8511124,\n   "lon": 37.4798857,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682612612",\n   "kind": "organisation",\n   "name": "Управа района Ховрино",\n   "office_type": "government",\n   "lat": 55.8515765,\n   "lon": 37.4796388,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682612613",\n   "kind": "organisation",\n   "name": "Управа района Левобережный",\n   "office_type": "government",\n   "lat": 55.8515198,\n   "lon": 37.4797062,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682613865",\n   "kind": "organisation",\n   "name": "Управа района Царицыно",\n   "office_type": "government",\n   "lat": 55.6231696,\n   "lon": 37.6623383,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682613866",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Царицыно",\n   "office_type": "government",\n   "lat": 55.6230617,\n   "lon": 37.6622129,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682613924",\n   "kind": "organisation",\n   "name": "Управа района Марфино",\n   "office_type": "government",\n   "lat": 55.8314916,\n   "lon": 37.5929096,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682613925",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Марфино",\n   "office_type": "government",\n   "lat": 55.8314223,\n   "lon": 37.5929476,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682616980",\n   "kind": "organisation",\n   "name": "Управа Таганского района",\n   "office_type": "government",\n   "lat": 55.7368042,\n   "lon": 37.6600908,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:45"\n  },\n  {\n   "id": "node/4682618528",\n   "kind": "organisation",\n   "name": "Управа района Черёмушки",\n   "office_type": "government",\n   "lat": 55.6743285,\n   "lon": 37.5492466,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682618529",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Обручевский",\n   "office_type": "government",\n   "lat": 55.6689271,\n   "lon": 37.5475691,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682618559",\n   "kind": "organisation",\n   "name": "Управа Тверского района",\n   "office_type": "government",\n   "lat": 55.7753016,\n   "lon": 37.5938794,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682618560",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Тверской",\n   "office_type": "government",\n   "lat": 55.7752754,\n   "lon": 37.5938069,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682619591",\n   "kind": "organisation",\n   "name": "Управа Тимирязевского района",\n   "office_type": "government",\n   "lat": 55.8115661,\n   "lon": 37.5613254,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682619592",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Тимирязевский",\n   "office_type": "government",\n   "lat": 55.81152,\n   "lon": 37.56102,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682620043",\n   "kind": "organisation",\n   "name": "Управа Гагаринского района",\n   "office_type": "government",\n   "lat": 55.689581,\n   "lon": 37.5441113,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682620044",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Гагаринский",\n   "office_type": "government",\n   "lat": 55.6942471,\n   "lon": 37.5524182,\n   "opening_hours": "Mo 15:00-19:00; Th 08:30-12:30; Sa,Su off"\n  },\n  {\n   "id": "node/4682621855",\n   "kind": "organisation",\n   "name": "Управа района Замоскворечье",\n   "office_type": "government",\n   "lat": 55.7354414,\n   "lon": 37.6356526,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682621856",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Замоскворечье",\n   "office_type": "government",\n   "lat": 55.7354299,\n   "lon": 37.6357677,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682622893",\n   "kind": "organisation",\n   "name": "Управа Басманного района",\n   "office_type": "government",\n   "lat": 55.7705671,\n   "lon": 37.6685701,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682622894",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Басманный",\n   "office_type": "government",\n   "lat": 55.7704925,\n   "lon": 37.6686773,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682623968",\n   "kind": "organisation",\n   "name": "Управа Даниловского района",\n   "office_type": "government",\n   "lat": 55.7104736,\n   "lon": 37.6238542,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682623969",\n   "kind": "organisation",\n   "name": "Управа Донского района",\n   "office_type": "government",\n   "lat": 55.6975841,\n   "lon": 37.6196312,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682623970",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Донской",\n   "office_type": "government",\n   "lat": 55.6930131,\n   "lon": 37.6130358,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682627269",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Ивановское",\n   "office_type": "government",\n   "lat": 55.7663346,\n   "lon": 37.8318371,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682629045",\n   "kind": "organisation",\n   "name": "Управа района Тропарево-Никулино",\n   "office_type": "government",\n   "lat": 55.6547414,\n   "lon": 37.4915531,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682629046",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Тропарево-Никулино",\n   "office_type": "government",\n   "lat": 55.6662045,\n   "lon": 37.4660537,\n   "opening_hours": "Mo 15:00-17:00"\n  },\n  {\n   "id": "node/4682631588",\n   "kind": "organisation",\n   "name": "Управа района Чертаново Южное",\n   "office_type": "government",\n   "lat": 55.6062154,\n   "lon": 37.6098245,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682631689",\n   "kind": "organisation",\n   "name": "Муниципалитет муниципального округа Чертаново Южное",\n   "office_type": "government",\n   "lat": 55.5921229,\n   "lon": 37.6141163,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682631724",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Войковский",\n   "office_type": "government",\n   "lat": 55.8158736,\n   "lon": 37.5042342,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682633122",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Коптево",\n   "office_type": "government",\n   "lat": 55.825065,\n   "lon": 37.5222247,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4682633123",\n   "kind": "organisation",\n   "name": "Управа района Коптево",\n   "office_type": "government",\n   "lat": 55.8243508,\n   "lon": 37.5225846,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4683435865",\n   "kind": "organisation",\n   "name": "Департамент финансов города Москвы",\n   "office_type": "government",\n   "lat": 55.7758988,\n   "lon": 37.5930725,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4683528001",\n   "kind": "organisation",\n   "name": "Инженерная служба района Замоскворечье",\n   "office_type": "property_management",\n   "lat": 55.7345215,\n   "lon": 37.6387865,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4683854791",\n   "kind": "organisation",\n   "name": "МР Групп",\n   "office_type": "estate_agent",\n   "lat": 55.8045312,\n   "lon": 37.5902146,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/4684137693",\n   "kind": "organisation",\n   "name": "Инженерная служба Красносельского района",\n   "office_type": "property_management",\n   "lat": 55.7799307,\n   "lon": 37.6695099,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684141141",\n   "kind": "organisation",\n   "name": "Инженерная служба Мещанского района",\n   "office_type": "property_management",\n   "lat": 55.7782747,\n   "lon": 37.6378104,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684143115",\n   "kind": "organisation",\n   "name": "Инженерная служба Пресненского района",\n   "office_type": "property_management",\n   "lat": 55.7625442,\n   "lon": 37.5692333,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684147661",\n   "kind": "organisation",\n   "name": "Инженерная служба Таганского района",\n   "office_type": "property_management",\n   "lat": 55.7323863,\n   "lon": 37.662645,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684155370",\n   "kind": "organisation",\n   "name": "Инженерная служба района Хамовники",\n   "office_type": "property_management",\n   "lat": 55.7244093,\n   "lon": 37.5743062,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684155706",\n   "kind": "organisation",\n   "name": "Инженерная служба Тверского района",\n   "office_type": "property_management",\n   "lat": 55.7615611,\n   "lon": 37.6139274,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684158733",\n   "kind": "organisation",\n   "name": "Инженерная служба района Якиманка",\n   "office_type": "property_management",\n   "lat": 55.7281389,\n   "lon": 37.6139597,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684162330",\n   "kind": "organisation",\n   "name": "Инженерная служба района Аэропорт ОДС № 1",\n   "office_type": "property_management",\n   "lat": 55.8078056,\n   "lon": 37.5335149,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684164712",\n   "kind": "organisation",\n   "name": "Инженерная служба района Беговой",\n   "office_type": "property_management",\n   "lat": 55.7867995,\n   "lon": 37.572693,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684166583",\n   "kind": "organisation",\n   "name": "Инженерная служба Бескудниковского района",\n   "office_type": "property_management",\n   "lat": 55.8694667,\n   "lon": 37.5588999,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684190842",\n   "kind": "organisation",\n   "name": "Инженерная служба района Восточное Дегунино",\n   "office_type": "property_management",\n   "lat": 55.8607204,\n   "lon": 37.5659874,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684193139",\n   "kind": "organisation",\n   "name": "Инженерная служба района Войковский",\n   "office_type": "property_management",\n   "lat": 55.8208586,\n   "lon": 37.492027,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684215034",\n   "kind": "organisation",\n   "name": "Инженерная служба района Дмитровский",\n   "office_type": "property_management",\n   "lat": 55.8876597,\n   "lon": 37.5218993,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684216962",\n   "kind": "organisation",\n   "name": "Инженерная служба района Западное Дегунино",\n   "office_type": "property_management",\n   "lat": 55.8732452,\n   "lon": 37.5162985,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684216969",\n   "kind": "organisation",\n   "name": "Инженерная служба района Савеловский",\n   "office_type": "property_management",\n   "lat": 55.7968486,\n   "lon": 37.5625567,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684224922",\n   "kind": "organisation",\n   "name": "Инженерная служба района Коптево",\n   "office_type": "property_management",\n   "lat": 55.8319163,\n   "lon": 37.5199735,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684228810",\n   "kind": "organisation",\n   "name": "Инженерная служба Левобережного района",\n   "office_type": "property_management",\n   "lat": 55.8646319,\n   "lon": 37.4752909,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684234329",\n   "kind": "organisation",\n   "name": "Инженерная служба района Сокол",\n   "office_type": "property_management",\n   "lat": 55.8012745,\n   "lon": 37.4980596,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684234948",\n   "kind": "organisation",\n   "name": "Инженерная служба района Тимирязевский",\n   "office_type": "property_management",\n   "lat": 55.8105409,\n   "lon": 37.5709762,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684236577",\n   "kind": "organisation",\n   "name": "Инженерная служба района Ховрино",\n   "office_type": "property_management",\n   "lat": 55.8613316,\n   "lon": 37.5001576,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684239552",\n   "kind": "organisation",\n   "name": "Инженерная служба Хорошевского района",\n   "office_type": "property_management",\n   "lat": 55.7783053,\n   "lon": 37.526545,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684245444",\n   "kind": "organisation",\n   "name": "Инженерная служба района Южное Медведково",\n   "office_type": "property_management",\n   "lat": 55.8739275,\n   "lon": 37.6291551,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684245523",\n   "kind": "organisation",\n   "name": "Инженерная служба района Алексеевский",\n   "office_type": "property_management",\n   "lat": 55.8085391,\n   "lon": 37.6424656,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684247506",\n   "kind": "organisation",\n   "name": "Инженерная служба Алтуфьевского района",\n   "office_type": "property_management",\n   "lat": 55.8811855,\n   "lon": 37.5891789,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684248495",\n   "kind": "organisation",\n   "name": "Инженерная служба Бабушкинского района",\n   "office_type": "property_management",\n   "lat": 55.8638303,\n   "lon": 37.6650969,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684250419",\n   "kind": "organisation",\n   "name": "Инженерная служба района Бибирево",\n   "office_type": "property_management",\n   "lat": 55.8906154,\n   "lon": 37.5924308,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684255345",\n   "kind": "organisation",\n   "name": "Инженерная служба Бутырского района",\n   "office_type": "property_management",\n   "lat": 55.8173989,\n   "lon": 37.5954007,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684259826",\n   "kind": "organisation",\n   "name": "Инженерная служба Лосиноостровского района",\n   "office_type": "property_management",\n   "lat": 55.8786118,\n   "lon": 37.6930499,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684261316",\n   "kind": "organisation",\n   "name": "Инженерная служба района Марфино",\n   "office_type": "property_management",\n   "lat": 55.8313169,\n   "lon": 37.5930248,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684263597",\n   "kind": "organisation",\n   "name": "Инженерная служба района Марьина Роща",\n   "office_type": "property_management",\n   "lat": 55.7958828,\n   "lon": 37.6153643,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684267584",\n   "kind": "organisation",\n   "name": "Инженерная служба района Останкинский",\n   "office_type": "property_management",\n   "lat": 55.8209533,\n   "lon": 37.6237508,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684268892",\n   "kind": "organisation",\n   "name": "Инженерная служба района Отрадное",\n   "office_type": "property_management",\n   "lat": 55.8633063,\n   "lon": 37.6217604,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684270761",\n   "kind": "organisation",\n   "name": "Инженерная служба района Северное Медведково",\n   "office_type": "property_management",\n   "lat": 55.8870019,\n   "lon": 37.6500128,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684271177",\n   "kind": "organisation",\n   "name": "Инженерная служба района Ростокино",\n   "office_type": "property_management",\n   "lat": 55.8395159,\n   "lon": 37.6598605,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684271354",\n   "kind": "organisation",\n   "name": "Инженерная служба района Свиблово",\n   "office_type": "property_management",\n   "lat": 55.8526477,\n   "lon": 37.6453959,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684279898",\n   "kind": "organisation",\n   "name": "Инженерная служба Ярославского района",\n   "office_type": "property_management",\n   "lat": 55.866967,\n   "lon": 37.7090181,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684287046",\n   "kind": "organisation",\n   "name": "Инженерная служба района Богородское",\n   "office_type": "property_management",\n   "lat": 55.8136393,\n   "lon": 37.7239024,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684292879",\n   "kind": "organisation",\n   "name": "Опорный пункт",\n   "office_type": "property_management",\n   "lat": 55.7234256,\n   "lon": 37.8284945,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684293167",\n   "kind": "organisation",\n   "name": "Инженерная служба района Воточное Измайлово",\n   "office_type": "property_management",\n   "lat": 55.7949716,\n   "lon": 37.8142761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684296289",\n   "kind": "organisation",\n   "name": "Инженерная служба района Гольяново",\n   "office_type": "property_management",\n   "lat": 55.8144305,\n   "lon": 37.7901242,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684297204",\n   "kind": "organisation",\n   "name": "Инженерная служба района Перово",\n   "office_type": "property_management",\n   "lat": 55.7552807,\n   "lon": 37.7956621,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684298678",\n   "kind": "organisation",\n   "name": "Инженерная служба района Ивановское",\n   "office_type": "property_management",\n   "lat": 55.764094,\n   "lon": 37.8333015,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684303401",\n   "kind": "organisation",\n   "name": "Инженерная служба района Измайлово",\n   "office_type": "property_management",\n   "lat": 55.7959456,\n   "lon": 37.7974448,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684305545",\n   "kind": "organisation",\n   "name": "Инженерная служба района Метрогородок",\n   "office_type": "property_management",\n   "lat": 55.8230358,\n   "lon": 37.7575909,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684309435",\n   "kind": "organisation",\n   "name": "Инженерная служба района Новогиреево",\n   "office_type": "property_management",\n   "lat": 55.7480719,\n   "lon": 37.8003899,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684318336",\n   "kind": "organisation",\n   "name": "Инженерная служба района Преображенское",\n   "office_type": "property_management",\n   "lat": 55.8005798,\n   "lon": 37.7194597,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4684337962",\n   "kind": "organisation",\n   "name": "Инженерная служба района Сокольники",\n   "office_type": "property_management",\n   "lat": 55.7884213,\n   "lon": 37.6829747,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4687047586",\n   "kind": "organisation",\n   "name": "IKa film",\n   "office_type": "company",\n   "lat": 55.7130102,\n   "lon": 37.5007695,\n   "opening_hours": "11:00-19:00"\n  },\n  {\n   "id": "node/4690945933",\n   "kind": "organisation",\n   "name": "Аутодеск",\n   "office_type": "company",\n   "lat": 55.7281432,\n   "lon": 37.449654,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4690945936",\n   "kind": "organisation",\n   "name": "Управление городским имуществомв Западном административном округе",\n   "office_type": "government",\n   "lat": 55.7288893,\n   "lon": 37.4495937,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4690945937",\n   "kind": "organisation",\n   "name": "Отдел государственной статистики в Западном административном округе",\n   "office_type": "government",\n   "lat": 55.7289936,\n   "lon": 37.4495454,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4690945963",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7300697,\n   "lon": 37.4504145,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4690945964",\n   "kind": "organisation",\n   "name": "Московская областная коллегия адвокатов",\n   "office_type": "lawyer",\n   "lat": 55.7298605,\n   "lon": 37.4521371,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4690945983",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.7319177,\n   "lon": 37.4603219,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4692951474",\n   "kind": "organisation",\n   "name": "Посольство Лаоса",\n   "office_type": "diplomatic",\n   "lat": 55.7588168,\n   "lon": 37.5903992,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4692951510",\n   "kind": "organisation",\n   "name": "Россотрудничество",\n   "office_type": "government",\n   "lat": 55.7532664,\n   "lon": 37.6028743,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4693241496",\n   "kind": "organisation",\n   "name": "Российский союз промышленников и предпринимателей",\n   "office_type": "ngo",\n   "lat": 55.744626,\n   "lon": 37.6435823,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/4693248789",\n   "kind": "organisation",\n   "name": "ДЭЗ",\n   "office_type": "property_management",\n   "lat": 55.6402957,\n   "lon": 37.6645264,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4693248989",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6373311,\n   "lon": 37.6640668,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4693433691",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник р-на Москворечье-Сабурово",\n   "office_type": "property_management",\n   "lat": 55.6398376,\n   "lon": 37.6923308,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4694651290",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8084567,\n   "lon": 37.7991898,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4694653089",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8086977,\n   "lon": 37.7984192,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/4695536792",\n   "kind": "organisation",\n   "name": "ООО \\"ЛЕДлтд\\"",\n   "office_type": "company",\n   "lat": 55.8639271,\n   "lon": 37.6496217,\n   "opening_hours": "Mo-Th 10:00-19:00; Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4696196986",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6585465,\n   "lon": 37.7446583,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/4701533537",\n   "kind": "organisation",\n   "name": "Международный учебно-научный лазерный центр МГУ им. М. В. Ломоносова",\n   "office_type": "research",\n   "lat": 55.7065536,\n   "lon": 37.520208,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4706909178",\n   "kind": "organisation",\n   "name": "\\"Жилищник\\" Академического района. Инженерная служба",\n   "office_type": "property_management",\n   "lat": 55.6879722,\n   "lon": 37.5840568,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4708975157",\n   "kind": "organisation",\n   "name": "FS-Property",\n   "office_type": "estate_agent",\n   "lat": 55.756862,\n   "lon": 37.6173919,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4709826251",\n   "kind": "organisation",\n   "name": "ООО «Био-Формула»",\n   "office_type": "company",\n   "lat": 55.7681802,\n   "lon": 37.6839841,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4711159391",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.641006,\n   "lon": 37.5308744,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4713953389",\n   "kind": "organisation",\n   "name": "Пункт выдачи и приема груза \\"Деловые линии\\"",\n   "office_type": "logistics",\n   "lat": 55.8962631,\n   "lon": 37.5208642,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4714488189",\n   "kind": "organisation",\n   "name": "Управление социальной защиты населения района Марьина Роща",\n   "office_type": "government",\n   "lat": 55.8061175,\n   "lon": 37.6119247,\n   "opening_hours": "Mo 11:00-13:45,14:30-20:00; We 09:00-13:45,14:30-18:00; Fr 09:00-13:45,14:30-16:45"\n  },\n  {\n   "id": "node/4715223889",\n   "kind": "organisation",\n   "name": "Александр Липатников",\n   "office_type": "lawyer",\n   "lat": 55.7628381,\n   "lon": 37.6261814,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa,Su 10:00-20:00"\n  },\n  {\n   "id": "node/4715450792",\n   "kind": "organisation",\n   "name": "Управление организации дознания ГУ МВД РФ по г. Москве",\n   "office_type": "government",\n   "lat": 55.7118696,\n   "lon": 37.6189465,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4716381689",\n   "kind": "organisation",\n   "name": "Росно",\n   "office_type": "insurance",\n   "lat": 55.7805127,\n   "lon": 37.6382761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4716997995",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8862335,\n   "lon": 37.6624582,\n   "opening_hours": "Fr 10:00-19:00;Mo-Th 10:00-20:00;Sa 10:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/4717109690",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.6119642,\n   "lon": 37.6059597,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4717778090",\n   "kind": "organisation",\n   "name": "ИнГеоСервис",\n   "office_type": "government",\n   "lat": 55.6898073,\n   "lon": 37.5272478,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4720464696",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7262736,\n   "lon": 37.4585583,\n   "opening_hours": "Mo-Fr 09:00-20:00;Sa-Su 09:00-18:00"\n  },\n  {\n   "id": "node/4728946287",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7088439,\n   "lon": 37.7333215,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4728952170",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.7155356,\n   "lon": 37.7108887,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4735613221",\n   "kind": "organisation",\n   "name": "Шухов Лаб",\n   "office_type": "research",\n   "lat": 55.761556,\n   "lon": 37.6332889,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4735923237",\n   "kind": "organisation",\n   "name": "Альфа-Капитал",\n   "office_type": "company",\n   "lat": 55.765431,\n   "lon": 37.591048,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4737683052",\n   "kind": "organisation",\n   "name": "ООО «ИД \\"Гребенников\\"»",\n   "office_type": "company",\n   "lat": 55.8007841,\n   "lon": 37.5021701,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4738369768",\n   "kind": "organisation",\n   "name": "НИИ проблем хранения Росрезерва",\n   "office_type": "research",\n   "lat": 55.7519339,\n   "lon": 37.6773012,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4739741991",\n   "kind": "organisation",\n   "name": "УФМС по р-ну Зюзино",\n   "office_type": "government",\n   "lat": 55.6622087,\n   "lon": 37.6080553,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4739760458",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6700031,\n   "lon": 37.587629,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4741377947",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8892946,\n   "lon": 37.7205639,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4743123184",\n   "kind": "organisation",\n   "name": "Дезцентр",\n   "office_type": "company",\n   "lat": 55.6317453,\n   "lon": 37.6152364,\n   "opening_hours": "Mo-Su 09:00-23:00"\n  },\n  {\n   "id": "node/4748871341",\n   "kind": "organisation",\n   "name": "НТК Солтек",\n   "office_type": "company",\n   "lat": 55.8757702,\n   "lon": 37.5951481,\n   "opening_hours": "10:00-18:00"\n  },\n  {\n   "id": "node/4750706631",\n   "kind": "organisation",\n   "name": "ООО \\"Кинопортал\\"",\n   "office_type": "company",\n   "lat": 55.720507,\n   "lon": 37.5192296,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4750941725",\n   "kind": "organisation",\n   "name": "Департамент культурного наследия г. Москвы",\n   "office_type": "government",\n   "lat": 55.7427121,\n   "lon": 37.6291187,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4751311821",\n   "kind": "organisation",\n   "name": "Мосгосстройнадзор",\n   "office_type": "government",\n   "lat": 55.7443069,\n   "lon": 37.5592393,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/4751589113",\n   "kind": "organisation",\n   "name": "Российский дом международного научно-технического сотрудничества",\n   "office_type": "foundation",\n   "lat": 55.7589327,\n   "lon": 37.6059424,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4751589114",\n   "kind": "organisation",\n   "name": "Министерство образования и науки Российской Федерации",\n   "office_type": "government",\n   "lat": 55.758869,\n   "lon": 37.6060724,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4755659223",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7217096,\n   "lon": 37.8138137,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/4756271896",\n   "kind": "organisation",\n   "name": "Отделение Росреестра",\n   "office_type": "government",\n   "lat": 55.6793248,\n   "lon": 37.6301354,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4759598408",\n   "kind": "organisation",\n   "name": "Химкинское управление ЗАГС",\n   "office_type": "government",\n   "lat": 55.8896519,\n   "lon": 37.4438851,\n   "opening_hours": "Tu-Fr 09:00-13:00,14:00-18:00; Sa 09:00-13:00,14:00-17:30; Th[-1] off \\"санитарный день, приёма нет\\""\n  },\n  {\n   "id": "node/4761184422",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7720831,\n   "lon": 37.6632359,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4762121977",\n   "kind": "organisation",\n   "name": "Кард Онлайн",\n   "office_type": "company",\n   "lat": 55.8509291,\n   "lon": 37.6697763,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4763136172",\n   "kind": "organisation",\n   "name": "Департамент культуры г. Москвы",\n   "office_type": "government",\n   "lat": 55.7644313,\n   "lon": 37.6146442,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4763136193",\n   "kind": "organisation",\n   "name": "Тверская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7650004,\n   "lon": 37.6148753,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4764720982",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.759712,\n   "lon": 37.7618089,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/4769136130",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.87967,\n   "lon": 37.5925989,\n   "opening_hours": "09:30-20:00"\n  },\n  {\n   "id": "node/4771032072",\n   "kind": "organisation",\n   "name": "Объединённая энергетическая компания, Департамент систем противопожарной защиты и инженерных систем",\n   "office_type": "company",\n   "lat": 55.7217786,\n   "lon": 37.6249713,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4771032073",\n   "kind": "organisation",\n   "name": "Третейский суд. Архив",\n   "office_type": "government",\n   "lat": 55.7194597,\n   "lon": 37.6246575,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4771663558",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7932764,\n   "lon": 37.8120963,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/4775243630",\n   "kind": "organisation",\n   "name": "Lucullus",\n   "office_type": "company",\n   "lat": 55.7664917,\n   "lon": 37.5933453,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4776519117",\n   "kind": "organisation",\n   "name": "АйТи Культура",\n   "office_type": "company",\n   "lat": 55.7850901,\n   "lon": 37.6595069,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4776559971",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7100504,\n   "lon": 37.5987104,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4780742621",\n   "kind": "organisation",\n   "name": "Всероссийский Научно-исследовательский институт по эксплуатации атомных электростанций",\n   "office_type": "engineer",\n   "lat": 55.7026005,\n   "lon": 37.8266639,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4781399471",\n   "kind": "organisation",\n   "name": "ИФНС России №3 по г. Москве",\n   "office_type": "government",\n   "lat": 55.7600171,\n   "lon": 37.5545606,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4783959607",\n   "kind": "organisation",\n   "name": "Мосэнергосбыт Клиентский офис \\"Тимирязевский\\"",\n   "office_type": "company",\n   "lat": 55.8462501,\n   "lon": 37.5706273,\n   "opening_hours": "Mo-Th 08:30-20:00; Fr 08:30-19:00"\n  },\n  {\n   "id": "node/4784517424",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.6494401,\n   "lon": 37.7707066,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4786504125",\n   "kind": "organisation",\n   "name": "АйТи Культура",\n   "office_type": "company",\n   "lat": 55.7702664,\n   "lon": 37.6868456,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4786659331",\n   "kind": "organisation",\n   "name": "АО \\"МирТелеКом имени Э. К. Первышина\\"",\n   "office_type": "telecommunication",\n   "lat": 55.7050186,\n   "lon": 37.6210627,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/4787660883",\n   "kind": "organisation",\n   "name": "АльфаСтрахование",\n   "office_type": "insurance",\n   "lat": 55.6794646,\n   "lon": 37.5635558,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/4787744731",\n   "kind": "organisation",\n   "name": "Теле2",\n   "office_type": "telecommunication",\n   "lat": 55.6772113,\n   "lon": 37.5661466,\n   "opening_hours": "Mo-Su 08:30-21:00"\n  },\n  {\n   "id": "node/4789082796",\n   "kind": "organisation",\n   "name": "Департамент по делам гражданской обороны, чрезвычайным ситуациям и пожарной безопасности города Москвы",\n   "office_type": "government",\n   "lat": 55.7624552,\n   "lon": 37.6107536,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4789870553",\n   "kind": "organisation",\n   "name": "Atlant Clinical CRO",\n   "office_type": "company",\n   "lat": 55.7621457,\n   "lon": 37.6366242,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4790422691",\n   "kind": "organisation",\n   "name": "Главное управление Пенсионного фонда РФ № 10 по Москве и Московской области. Управление по приёму населения. Клиентская служба \\"Замоскворечье\\"",\n   "office_type": "government",\n   "lat": 55.7205144,\n   "lon": 37.6186042,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/4790436940",\n   "kind": "organisation",\n   "name": "Московское городское БТИ",\n   "office_type": "government",\n   "lat": 55.762919,\n   "lon": 37.6068212,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4790476981",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт социальных систем",\n   "office_type": "research",\n   "lat": 55.7208037,\n   "lon": 37.620362,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4792450121",\n   "kind": "organisation",\n   "name": "Филиал ФСС №25",\n   "office_type": "government",\n   "lat": 55.7423936,\n   "lon": 37.5858087,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4793324121",\n   "kind": "organisation",\n   "name": "Офицеры России",\n   "office_type": "ngo",\n   "lat": 55.7419298,\n   "lon": 37.6480618,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/4794210908",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.6587154,\n   "lon": 37.7451063,\n   "opening_hours": "Mo-Fr 11:00-20:00;Sa-Su 11:00-19:00"\n  },\n  {\n   "id": "node/4799394516",\n   "kind": "organisation",\n   "name": "Жилищник района Тверской",\n   "office_type": "property_management",\n   "lat": 55.7680315,\n   "lon": 37.5976932,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4801212317",\n   "kind": "organisation",\n   "name": "Sanamus",\n   "office_type": "company",\n   "lat": 55.7811542,\n   "lon": 37.6375776,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4801596146",\n   "kind": "organisation",\n   "name": "Леконс",\n   "office_type": "employment_agency",\n   "lat": 55.7738448,\n   "lon": 37.5925608,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 11:00-17:00"\n  },\n  {\n   "id": "node/4804137535",\n   "kind": "organisation",\n   "name": "ФГБУ \\"Автотранспортный комбинат\\"",\n   "office_type": "government",\n   "lat": 55.742534,\n   "lon": 37.6379988,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4806151943",\n   "kind": "organisation",\n   "name": "ДойчеКлимат.РУ",\n   "office_type": "company",\n   "lat": 55.7302962,\n   "lon": 37.647532,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4807527784",\n   "kind": "organisation",\n   "name": "Управление Государственной службы занятости населения г. Москвы. Отдел \\"Южный\\".",\n   "office_type": "government",\n   "lat": 55.6099575,\n   "lon": 37.732776,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4807527787",\n   "kind": "organisation",\n   "name": "Департамент торговли и услуг города Москвы",\n   "office_type": "government",\n   "lat": 55.7575758,\n   "lon": 37.633076,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4807738043",\n   "kind": "organisation",\n   "name": "Миэль",\n   "office_type": "estate_agent",\n   "lat": 55.8192198,\n   "lon": 37.4965558,\n   "opening_hours": "Mo-Fr 09:30-19:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/4811528098",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7215693,\n   "lon": 37.622847,\n   "opening_hours": "Mo-Th 09:00-13:00, 14:00-17:00; Fr 09:00-14:00; Sa 10:00-14:00"\n  },\n  {\n   "id": "node/4811528780",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7215329,\n   "lon": 37.6228574,\n   "opening_hours": "Mo-Th 09:00-14:00, 15:00-17:00; Fr 09:00-15:00"\n  },\n  {\n   "id": "node/4811677313",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6291053,\n   "lon": 37.4856879,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4812069623",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7329584,\n   "lon": 37.5356914,\n   "opening_hours": "Mo-Su 07:00-23:00"\n  },\n  {\n   "id": "node/4813948524",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.6309535,\n   "lon": 37.6587886,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/4817326222",\n   "kind": "organisation",\n   "name": "Саранов А.Н.",\n   "office_type": "lawyer",\n   "lat": 55.6143693,\n   "lon": 37.602798,\n   "opening_hours": "Mo-Th 09:00-17:00; Fr 09:00-16:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/4818155507",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8965227,\n   "lon": 37.5549748,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4818155727",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.8974038,\n   "lon": 37.5541478,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4818156605",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.8936032,\n   "lon": 37.5339659,\n   "opening_hours": "Mo-Su 11:00-20:00; 13:00-14:00 off \\"Обед\\"; Mo-Fr 18:00-20:00 \\"Приём жителей\\""\n  },\n  {\n   "id": "node/4820600673",\n   "kind": "organisation",\n   "name": "Департамент труда и социальной защиты населения города Москвы",\n   "office_type": "government",\n   "lat": 55.7693654,\n   "lon": 37.655692,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4821286521",\n   "kind": "organisation",\n   "name": "ООО «МК развитие»",\n   "office_type": "financial",\n   "lat": 55.7482496,\n   "lon": 37.5401354,\n   "opening_hours": "09:00-18:00"\n  },\n  {\n   "id": "node/4826106185",\n   "kind": "organisation",\n   "name": "Казначейство",\n   "office_type": "government",\n   "lat": 55.8881163,\n   "lon": 37.4444194,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4826742785",\n   "kind": "organisation",\n   "name": "Экотовары",\n   "office_type": "company",\n   "lat": 55.8583781,\n   "lon": 37.6694572,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa,Su 10:00-18:00"\n  },\n  {\n   "id": "node/4828467163",\n   "kind": "organisation",\n   "name": "Курьер Сервис Экспресс",\n   "office_type": "logistics",\n   "lat": 55.7230051,\n   "lon": 37.6099194,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4832795921",\n   "kind": "organisation",\n   "name": "РусАсфальт",\n   "office_type": "company",\n   "lat": 55.6847886,\n   "lon": 37.624125,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4836730301",\n   "kind": "organisation",\n   "name": "ГазФонд",\n   "office_type": "company",\n   "lat": 55.6559532,\n   "lon": 37.6072928,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4838230922",\n   "kind": "organisation",\n   "name": "БИК СНГ",\n   "office_type": "government",\n   "lat": 55.8561392,\n   "lon": 37.6002032,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4848034221",\n   "kind": "organisation",\n   "name": "Студия \\"Союзмультфильм\\"",\n   "office_type": "company",\n   "lat": 55.821065,\n   "lon": 37.6015741,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4854442995",\n   "kind": "organisation",\n   "name": "Пари",\n   "office_type": "insurance",\n   "lat": 55.7915612,\n   "lon": 37.5811909,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4867936368",\n   "kind": "organisation",\n   "name": "АО «Доринж-ПБ»",\n   "office_type": "company",\n   "lat": 55.8470606,\n   "lon": 37.6882066,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4868390721",\n   "kind": "organisation",\n   "name": "Резиденция посла Австралии",\n   "office_type": "diplomatic",\n   "lat": 55.7385372,\n   "lon": 37.5914133,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4869613923",\n   "kind": "organisation",\n   "name": "Гамма",\n   "office_type": "company",\n   "lat": 55.8004302,\n   "lon": 37.798509,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4869704921",\n   "kind": "organisation",\n   "name": "Служба специальных объектов ГУСП при Президенте РФ",\n   "office_type": "government",\n   "lat": 55.7079335,\n   "lon": 37.5009162,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4869734221",\n   "kind": "organisation",\n   "name": "Спецавтобаза Федеральной Службы Охраны РФ",\n   "office_type": "government",\n   "lat": 55.7080099,\n   "lon": 37.5042874,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4869863521",\n   "kind": "organisation",\n   "name": "Штаб Космических войск Воздушно-космических сил РФ",\n   "office_type": "government",\n   "lat": 55.6544421,\n   "lon": 37.5354259,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4869863621",\n   "kind": "organisation",\n   "name": "Главное Управление Научно-исследовательской деятельности и технологического сопровождения передовых технологий Министерства Обороны РФ",\n   "office_type": "government",\n   "lat": 55.6546954,\n   "lon": 37.5347682,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4870396924",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7716521,\n   "lon": 37.5990657,\n   "opening_hours": "Mo-Fr 10:00-14:00,15:00-18:00"\n  },\n  {\n   "id": "node/4871439025",\n   "kind": "organisation",\n   "name": "Московская областная коллегия адвокатов. Филиал #1",\n   "office_type": "lawyer",\n   "lat": 55.7699676,\n   "lon": 37.5980314,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 11:00-14:00"\n  },\n  {\n   "id": "node/4871439123",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7706626,\n   "lon": 37.5996044,\n   "opening_hours": "Mo-Fr 09:00-13:00, 14:00-18:00"\n  },\n  {\n   "id": "node/4872719622",\n   "kind": "organisation",\n   "name": "ООО «НИКАСТРОЙ»",\n   "office_type": "company",\n   "lat": 55.7217219,\n   "lon": 37.6930126,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4873588824",\n   "kind": "organisation",\n   "name": "Министерство промышленности и торговли Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7522209,\n   "lon": 37.6335075,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4877108290",\n   "kind": "organisation",\n   "name": "Ресо-страхование",\n   "office_type": "insurance",\n   "lat": 55.8304106,\n   "lon": 37.5719826,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4879931398",\n   "kind": "organisation",\n   "name": "Амтел-Сервис",\n   "office_type": "it",\n   "lat": 55.8090599,\n   "lon": 37.5119956,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4881743325",\n   "kind": "organisation",\n   "name": "Эксплуатационно-техническое управление при Главном Управлении специальных программ Президента РФ",\n   "office_type": "government",\n   "lat": 55.6502148,\n   "lon": 37.5491188,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4881899223",\n   "kind": "organisation",\n   "name": "ДЭЗ",\n   "office_type": "property_management",\n   "lat": 55.7313891,\n   "lon": 37.5338047,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4881905222",\n   "kind": "organisation",\n   "name": "Мосводосток",\n   "office_type": "government",\n   "lat": 55.7338316,\n   "lon": 37.5390523,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4881905921",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.7329273,\n   "lon": 37.535884,\n   "opening_hours": "Mo-Su 07:00-23:00"\n  },\n  {\n   "id": "node/4882971022",\n   "kind": "organisation",\n   "name": "Бурейникова Л.Л.",\n   "office_type": "lawyer",\n   "lat": 55.7756606,\n   "lon": 37.6067633,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa-Su off"\n  },\n  {\n   "id": "node/4885591362",\n   "kind": "organisation",\n   "name": "Международное общество \\"Мемориал\\"",\n   "office_type": "ngo",\n   "lat": 55.7721288,\n   "lon": 37.6092485,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4885969495",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.767812,\n   "lon": 37.6012659,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4885969516",\n   "kind": "organisation",\n   "name": "Itec",\n   "office_type": "company",\n   "lat": 55.7675381,\n   "lon": 37.6025959,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4887670049",\n   "kind": "organisation",\n   "name": "Центр судебно-медицинской экспертизы",\n   "office_type": "government",\n   "lat": 55.7696397,\n   "lon": 37.6853479,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4888068431",\n   "kind": "organisation",\n   "name": "Второе территориальное управление (дополнительный офис) ГБУ МосгорБТИ",\n   "office_type": "company",\n   "lat": 55.6781878,\n   "lon": 37.5722605,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4888321938",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7719898,\n   "lon": 37.6056138,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4889829407",\n   "kind": "organisation",\n   "name": "Сибирский центр фармакологии и биотехнологии",\n   "office_type": "company",\n   "lat": 55.7388675,\n   "lon": 37.5811851,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4891020840",\n   "kind": "organisation",\n   "name": "Районный совет ветеранов района Дмитровский",\n   "office_type": "ngo",\n   "lat": 55.8934279,\n   "lon": 37.5338171,\n   "opening_hours": "Mo-We 11:00-14:00"\n  },\n  {\n   "id": "node/4891309463",\n   "kind": "organisation",\n   "name": "Адвокаты",\n   "office_type": "lawyer",\n   "lat": 55.7744293,\n   "lon": 37.598776,\n   "opening_hours": "Mo-Fr 11:00-20:00"\n  },\n  {\n   "id": "node/4897208704",\n   "kind": "organisation",\n   "name": "Сбербанк Страхование",\n   "office_type": "insurance",\n   "lat": 55.7211876,\n   "lon": 37.6105124,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4903807222",\n   "kind": "organisation",\n   "name": "Ультра Телеком",\n   "office_type": "company",\n   "lat": 55.6124958,\n   "lon": 37.6142291,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4905159101",\n   "kind": "organisation",\n   "name": "ABBYY",\n   "office_type": "it",\n   "lat": 55.8567518,\n   "lon": 37.5990996,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4909714221",\n   "kind": "organisation",\n   "name": "ФГКУ Специальное Управление Федеральной Противопожарной Службы МЧС России",\n   "office_type": "government",\n   "lat": 55.7001242,\n   "lon": 37.495568,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/4911986652",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8897985,\n   "lon": 37.5375393,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4912077944",\n   "kind": "organisation",\n   "name": "Амтел-Связь",\n   "office_type": "telecommunication",\n   "lat": 55.8091391,\n   "lon": 37.5121478,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4912077945",\n   "kind": "organisation",\n   "name": "Дозор-Телепорт",\n   "office_type": "telecommunication",\n   "lat": 55.8092145,\n   "lon": 37.5120405,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4913374710",\n   "kind": "organisation",\n   "name": "РЕСО-гарантия",\n   "office_type": "insurance",\n   "lat": 55.782943,\n   "lon": 37.7026856,\n   "opening_hours": "Mo-Fr 09:30-20:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/4913732822",\n   "kind": "organisation",\n   "name": "Вневедомственная охрана",\n   "office_type": "security",\n   "lat": 55.723569,\n   "lon": 37.7695028,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4916258313",\n   "kind": "organisation",\n   "name": "Центр занятости населения",\n   "office_type": "government",\n   "lat": 55.8891046,\n   "lon": 37.541561,\n   "opening_hours": "Mo,We,Fr 09:00-17:00; Tu 09:00-20:00; Th 09:00-19:00; 13:00-13:45 off"\n  },\n  {\n   "id": "node/4919012183",\n   "kind": "organisation",\n   "name": "МГТС",\n   "office_type": "telecommunication",\n   "lat": 55.8074752,\n   "lon": 37.5399637,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-18:00; PH off"\n  },\n  {\n   "id": "node/4923789421",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6073704,\n   "lon": 37.531944,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4929197520",\n   "kind": "organisation",\n   "name": "ГБУ «Жилищник района Аэропорт»",\n   "office_type": "property_management",\n   "lat": 55.8075917,\n   "lon": 37.5254305,\n   "opening_hours": "Mo-Tu 08:00-12:00,13:00-17:00; Fr 08:00-12:00,13:00-15:45"\n  },\n  {\n   "id": "node/4929249657",\n   "kind": "organisation",\n   "name": "Объединённая диспетчерская служба №2",\n   "office_type": "property_management",\n   "lat": 55.8067534,\n   "lon": 37.5220942,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4934370323",\n   "kind": "organisation",\n   "name": "Государственное бюджетное учреждение города Москвы \\"Центральный государственный архив города Москвы\\"",\n   "office_type": "government",\n   "lat": 55.7452187,\n   "lon": 37.6840173,\n   "opening_hours": "Mo-Fr 08:00-12:00,12:45-17:00"\n  },\n  {\n   "id": "node/4934404121",\n   "kind": "organisation",\n   "name": "Государственное бюджетное учреждение города Москвы \\"Центральный государственный архив трудовых отношений города Москвы\\"",\n   "office_type": "government",\n   "lat": 55.7453832,\n   "lon": 37.6825691,\n   "opening_hours": "Mo-Fr 08:00-12:00,12:45-17:00"\n  },\n  {\n   "id": "node/4936299492",\n   "kind": "organisation",\n   "name": "Совет муниципальных образований города Москвы",\n   "office_type": "government",\n   "lat": 55.7689949,\n   "lon": 37.6102106,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4936299493",\n   "kind": "organisation",\n   "name": "Общественная палата города Москвы",\n   "office_type": "government",\n   "lat": 55.769108,\n   "lon": 37.610112,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4936299494",\n   "kind": "organisation",\n   "name": "Уполномоченный по правам человека в городе Москве",\n   "office_type": "government",\n   "lat": 55.7691503,\n   "lon": 37.6095807,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4936299495",\n   "kind": "organisation",\n   "name": "Приёмная Думы",\n   "office_type": "government",\n   "lat": 55.7692318,\n   "lon": 37.6096183,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4936299496",\n   "kind": "organisation",\n   "name": "Аппарат Московской городской Думы",\n   "office_type": "government",\n   "lat": 55.7685063,\n   "lon": 37.611891,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4940050093",\n   "kind": "organisation",\n   "name": "Службы Единого Окна - УФК по г. Москве",\n   "office_type": "government",\n   "lat": 55.7076079,\n   "lon": 37.6139608,\n   "opening_hours": "Mo-Th 09:30-12:45,14:15-17:00; Fr 09:30-12:45,14:15-16:00"\n  },\n  {\n   "id": "node/4944960712",\n   "kind": "organisation",\n   "name": "Правда.Ру",\n   "office_type": "mass_media",\n   "lat": 55.7651886,\n   "lon": 37.6611746,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4947355466",\n   "kind": "organisation",\n   "name": "Национальная система платежных карт",\n   "office_type": "company",\n   "lat": 55.7411162,\n   "lon": 37.6336951,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4948246231",\n   "kind": "organisation",\n   "name": "ТруботоргРус",\n   "office_type": "company",\n   "lat": 55.6829829,\n   "lon": 37.6335243,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4948278023",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "lawyer",\n   "lat": 55.7805225,\n   "lon": 37.588497,\n   "opening_hours": "Mo-Th 10:00-13:30, 14:00-18:00; Fr 10:00-13:00, 14:00-17:00"\n  },\n  {\n   "id": "node/4948502413",\n   "kind": "organisation",\n   "name": "АО \\"Союзтехноком\\"",\n   "office_type": "company",\n   "lat": 55.7367894,\n   "lon": 37.5245749,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4956254221",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.5946083,\n   "lon": 37.7451248,\n   "opening_hours": "Mo-Tu 10:00-14:00, 15:00-18:00; Th 10:00-14:00, 15:00-18:00; Sa 11:00-14:00; Fr 10:00-14:00, 15:00-17:00"\n  },\n  {\n   "id": "node/4958777296",\n   "kind": "organisation",\n   "name": "Общественная палата г. Москвы",\n   "office_type": "government",\n   "lat": 55.7795071,\n   "lon": 37.5771966,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4958844697",\n   "kind": "organisation",\n   "name": "Совет ветеранов блокадного Ленинграда",\n   "office_type": "ngo",\n   "lat": 55.7377184,\n   "lon": 37.6899377,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4959777521",\n   "kind": "organisation",\n   "name": "Российское военно-историческое общество",\n   "office_type": "government",\n   "lat": 55.7652914,\n   "lon": 37.6127921,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4960395687",\n   "kind": "organisation",\n   "name": "Объединенный консалтинговый центр \\"Сокол\\"",\n   "office_type": "lawyer",\n   "lat": 55.8074535,\n   "lon": 37.519391,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4960497811",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.8048491,\n   "lon": 37.5203178,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/4960497812",\n   "kind": "organisation",\n   "name": "Фатыхова С.М.",\n   "office_type": "notary",\n   "lat": 55.8051034,\n   "lon": 37.5201026,\n   "opening_hours": "Mo-Su 09:00-13:00,14:00-18:00; Fr 09:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/4964372676",\n   "kind": "organisation",\n   "name": "Филиал ФГУП \\"Почта России\\" - Управления Федеральной почтовой связи по Московской области",\n   "office_type": "government",\n   "lat": 55.7896685,\n   "lon": 37.8111433,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4968280661",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.5969485,\n   "lon": 37.5116989,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/4968280698",\n   "kind": "organisation",\n   "name": "Bayplus.ru",\n   "office_type": "company",\n   "lat": 55.7737769,\n   "lon": 37.6822458,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4972313160",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.820613,\n   "lon": 37.6419758,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/4972313161",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8208337,\n   "lon": 37.6421541,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4972313521",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.820792,\n   "lon": 37.6421728,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/4974148723",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8834047,\n   "lon": 37.5330471,\n   "opening_hours": "Mo-Fr 10:00-18:00, Sa 10:00-16:00; Su off"\n  },\n  {\n   "id": "node/4974912621",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7961575,\n   "lon": 37.617025,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/4975712909",\n   "kind": "organisation",\n   "name": "ОДС № 4 ГБУ \\"Жилищник Басманного района\\"",\n   "office_type": "property_management",\n   "lat": 55.7601854,\n   "lon": 37.6437051,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4976217718",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.6290597,\n   "lon": 37.7414381,\n   "opening_hours": "Mo-Fr 10:00-13:00,14:00-19:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/4986436626",\n   "kind": "organisation",\n   "name": "Комеди Клаб продакшн",\n   "office_type": "company",\n   "lat": 55.7950817,\n   "lon": 37.5651238,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4986934921",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7555205,\n   "lon": 37.6760049,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4991373840",\n   "kind": "organisation",\n   "name": "Посольство Республики Мали",\n   "office_type": "diplomatic",\n   "lat": 55.7383885,\n   "lon": 37.6307273,\n   "opening_hours": null\n  },\n  {\n   "id": "node/4991926079",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.6345435,\n   "lon": 37.6587965,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/4992881987",\n   "kind": "organisation",\n   "name": "Гидроинжстрой",\n   "office_type": "company",\n   "lat": 55.5941028,\n   "lon": 37.7472855,\n   "opening_hours": "Mo-Fr 09:00-18:00, Sa-Su 09:00-15:00"\n  },\n  {\n   "id": "node/4992882002",\n   "kind": "organisation",\n   "name": "Газпромтелеком",\n   "office_type": "company",\n   "lat": 55.5919808,\n   "lon": 37.7549829,\n   "opening_hours": "Mo-Th 09:00-18:00, Fr 09:00-16:45"\n  },\n  {\n   "id": "node/4996469024",\n   "kind": "organisation",\n   "name": "РУСАЛ",\n   "office_type": "company",\n   "lat": 55.7379384,\n   "lon": 37.5058787,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/4999574621",\n   "kind": "organisation",\n   "name": "School Gosh",\n   "office_type": "company",\n   "lat": 55.7668719,\n   "lon": 37.6258514,\n   "opening_hours": "Mo-Fr 13:00-20:00"\n  },\n  {\n   "id": "node/5003980344",\n   "kind": "organisation",\n   "name": "Металь интернейшенл",\n   "office_type": "physician",\n   "lat": 55.8023487,\n   "lon": 37.5163582,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa 09:00-17:00"\n  },\n  {\n   "id": "node/5005142021",\n   "kind": "organisation",\n   "name": "Гринпис России",\n   "office_type": "ngo",\n   "lat": 55.7829906,\n   "lon": 37.572945,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5015175324",\n   "kind": "organisation",\n   "name": "Маздовод.Ру",\n   "office_type": "company",\n   "lat": 55.6863776,\n   "lon": 37.5120474,\n   "opening_hours": "10:00-20:00"\n  },\n  {\n   "id": "node/5018641686",\n   "kind": "organisation",\n   "name": "Жилищник Дмитровского района",\n   "office_type": "property_management",\n   "lat": 55.883367,\n   "lon": 37.5233236,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5021064736",\n   "kind": "organisation",\n   "name": "Иванова М.Н.",\n   "office_type": "notary",\n   "lat": 55.7370457,\n   "lon": 37.6284374,\n   "opening_hours": "Mo-Fr 10:00-14:00,15:00-17:30"\n  },\n  {\n   "id": "node/5021583221",\n   "kind": "organisation",\n   "name": "Визовый центр стран Азии",\n   "office_type": "visa",\n   "lat": 55.7375099,\n   "lon": 37.6291662,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5021888227",\n   "kind": "organisation",\n   "name": "ПрофТехСнаб",\n   "office_type": "company",\n   "lat": 55.7638198,\n   "lon": 37.5262475,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5022545022",\n   "kind": "organisation",\n   "name": "Уолт Дисней Компани СНГ",\n   "office_type": "company",\n   "lat": 55.7499552,\n   "lon": 37.584,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5022604849",\n   "kind": "organisation",\n   "name": "Департамент развития новых территорий города Москвы",\n   "office_type": "government",\n   "lat": 55.7358244,\n   "lon": 37.6309714,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5024639407",\n   "kind": "organisation",\n   "name": "Дирекция по обеспечению деятельности государственных учреждений Департамента образования города Москвы",\n   "office_type": "government",\n   "lat": 55.7350678,\n   "lon": 37.6296034,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5026087521",\n   "kind": "organisation",\n   "name": "Главное Управление ПФР России 10",\n   "office_type": "government",\n   "lat": 55.7368802,\n   "lon": 37.5705967,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/5026762985",\n   "kind": "organisation",\n   "name": "Отдел квотирования рабочих мест по работе с иноплательщиками",\n   "office_type": "employment_agency",\n   "lat": 55.8075353,\n   "lon": 37.5186525,\n   "opening_hours": "Mo-Tu 10:00-13:00,13:45-17:00; We off; Th 10:00-13:00,13:45-17:00; Fr 10:00-13:00,13:45-16:00"\n  },\n  {\n   "id": "node/5027107517",\n   "kind": "organisation",\n   "name": "Интертраст",\n   "office_type": "company",\n   "lat": 55.8034761,\n   "lon": 37.5314706,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5027107518",\n   "kind": "organisation",\n   "name": "ЖКХ \\"Советсткое искусство\\"",\n   "office_type": "property_management",\n   "lat": 55.8034637,\n   "lon": 37.531504,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5027107556",\n   "kind": "organisation",\n   "name": "Интертраст",\n   "office_type": "company",\n   "lat": 55.8040488,\n   "lon": 37.5319947,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5027252242",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6830196,\n   "lon": 37.6660793,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/5027732021",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт автоматики им. Н.Л.Духова",\n   "office_type": "research",\n   "lat": 55.6232649,\n   "lon": 37.6686336,\n   "opening_hours": "Mo-Th 08:15-17:15; Fr 08:15-14:45"\n  },\n  {\n   "id": "node/5028885329",\n   "kind": "organisation",\n   "name": "Миэль",\n   "office_type": "estate_agent",\n   "lat": 55.8039777,\n   "lon": 37.532933,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/5028885460",\n   "kind": "organisation",\n   "name": "Сюртель",\n   "office_type": "security",\n   "lat": 55.8044294,\n   "lon": 37.5315465,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5028885507",\n   "kind": "organisation",\n   "name": "Тарасова И.В.",\n   "office_type": "notary",\n   "lat": 55.8054401,\n   "lon": 37.531175,\n   "opening_hours": "Mo 13:00-17:00; Tu-Fr 11:00-17:00"\n  },\n  {\n   "id": "node/5028885508",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8054526,\n   "lon": 37.5311335,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5031115686",\n   "kind": "organisation",\n   "name": "Вторая московская областная коллегия адвокатов МОКА2",\n   "office_type": "lawyer",\n   "lat": 55.8071397,\n   "lon": 37.5259067,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5032420921",\n   "kind": "organisation",\n   "name": "Федеральная налоговая служба",\n   "office_type": "government",\n   "lat": 55.7498173,\n   "lon": 37.7114308,\n   "opening_hours": "Mo,We 09:00-18:00; Tu,Th 09:00-20:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/5035803514",\n   "kind": "organisation",\n   "name": "Академия Н.С. Михалкова",\n   "office_type": "company",\n   "lat": 55.8066328,\n   "lon": 37.5346359,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5035803721",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.8067354,\n   "lon": 37.5351577,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5036103321",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Пресненский",\n   "office_type": "government",\n   "lat": 55.7673883,\n   "lon": 37.5551697,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/5038900121",\n   "kind": "organisation",\n   "name": "МейТан",\n   "office_type": "company",\n   "lat": 55.6749325,\n   "lon": 37.6606798,\n   "opening_hours": "Tu-Fr 12:00-21:00; Sa-Su 11:00-18:00"\n  },\n  {\n   "id": "node/5039755668",\n   "kind": "organisation",\n   "name": "Торговая компания ООО \\"Пако\\"",\n   "office_type": "company",\n   "lat": 55.7282459,\n   "lon": 37.7616191,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5041988683",\n   "kind": "organisation",\n   "name": "Национальный институт корпоративной реформы",\n   "office_type": "ngo",\n   "lat": 55.7357459,\n   "lon": 37.6245609,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5044901491",\n   "kind": "organisation",\n   "name": "Сущий",\n   "office_type": "company",\n   "lat": 55.7360212,\n   "lon": 37.7182809,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/5044921766",\n   "kind": "organisation",\n   "name": "Совет ветеранов района Аэропорт",\n   "office_type": "ngo",\n   "lat": 55.8092154,\n   "lon": 37.5327523,\n   "opening_hours": "Mo-Tu 11:00-15:00"\n  },\n  {\n   "id": "node/5045257696",\n   "kind": "organisation",\n   "name": "Деловая Русь",\n   "office_type": "company",\n   "lat": 55.8030119,\n   "lon": 37.532707,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5045258127",\n   "kind": "organisation",\n   "name": "Проектно-изыскательский и научно-исследовательский институт морского транспорта АО «Союзморниипроект»",\n   "office_type": "research",\n   "lat": 55.8642456,\n   "lon": 37.6769527,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5045431732",\n   "kind": "organisation",\n   "name": "УФМС",\n   "office_type": "government",\n   "lat": 55.8053829,\n   "lon": 37.5375841,\n   "opening_hours": "Mo 09:00-18:00, Tu 11:00-20:00, We 09:00-13:00, Th 11:00-20:00, Fr 09:00-16:45"\n  },\n  {\n   "id": "node/5046221162",\n   "kind": "organisation",\n   "name": "Пенсионный фонд",\n   "office_type": "ngo",\n   "lat": 55.7503761,\n   "lon": 37.7666831,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5047666341",\n   "kind": "organisation",\n   "name": "Bette GmbH & Co.KG (представительство)",\n   "office_type": "company",\n   "lat": 55.7696741,\n   "lon": 37.6265062,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5048226135",\n   "kind": "organisation",\n   "name": "ООО «Гермсат» Speedcast",\n   "office_type": "yes",\n   "lat": 55.8095508,\n   "lon": 37.5358457,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/5048231560",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник района Бирюлево-Западное\\"",\n   "office_type": "property_management",\n   "lat": 55.5922563,\n   "lon": 37.6510826,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/5048231567",\n   "kind": "organisation",\n   "name": "Спецтранс Веста",\n   "office_type": "company",\n   "lat": 55.62441,\n   "lon": 37.8266251,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5048493221",\n   "kind": "organisation",\n   "name": "РБК",\n   "office_type": "estate_agent",\n   "lat": 55.7716202,\n   "lon": 37.6569211,\n   "opening_hours": "Mo-Su 11:00-18:00"\n  },\n  {\n   "id": "node/5049660043",\n   "kind": "organisation",\n   "name": "МЧС",\n   "office_type": "government",\n   "lat": 55.8003753,\n   "lon": 37.5481846,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5049660045",\n   "kind": "organisation",\n   "name": "«Жилищник района Аэропорт» ОДС № 5",\n   "office_type": "government",\n   "lat": 55.8001169,\n   "lon": 37.5476435,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5049660095",\n   "kind": "organisation",\n   "name": "GoldenTab",\n   "office_type": "company",\n   "lat": 55.799767,\n   "lon": 37.5470548,\n   "opening_hours": "Mo-Fr 10:30-17:00"\n  },\n  {\n   "id": "node/5050097916",\n   "kind": "organisation",\n   "name": "Аэротелеком",\n   "office_type": "company",\n   "lat": 55.6193397,\n   "lon": 37.6099447,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5051609021",\n   "kind": "organisation",\n   "name": "Редакция газеты \\"Красная звезда\\"",\n   "office_type": "newspaper",\n   "lat": 55.7765453,\n   "lon": 37.5370651,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5052285574",\n   "kind": "organisation",\n   "name": "Биокомплекс",\n   "office_type": "company",\n   "lat": 55.8100978,\n   "lon": 37.8080213,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5054515622",\n   "kind": "organisation",\n   "name": "Фабрика Браслетов",\n   "office_type": "company",\n   "lat": 55.733913,\n   "lon": 37.5528076,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 12:00-18:00"\n  },\n  {\n   "id": "node/5054669706",\n   "kind": "organisation",\n   "name": "Объединённая диспетчерская служба №2",\n   "office_type": "property_management",\n   "lat": 55.800731,\n   "lon": 37.5156209,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5054671130",\n   "kind": "organisation",\n   "name": "Пункт оповещения",\n   "office_type": "government",\n   "lat": 55.802199,\n   "lon": 37.5166872,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5057321125",\n   "kind": "organisation",\n   "name": "ИВК - Информационная Внедренческая Компания\\"",\n   "office_type": "company",\n   "lat": 55.8033081,\n   "lon": 37.5824363,\n   "opening_hours": "Mo-Fr 09:30-18:30"\n  },\n  {\n   "id": "node/5057733048",\n   "kind": "organisation",\n   "name": "Объединение \\"Выставочные залы Москвы\\"",\n   "office_type": "government",\n   "lat": 55.7683026,\n   "lon": 37.6189753,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5057873537",\n   "kind": "organisation",\n   "name": "МАКС",\n   "office_type": "insurance",\n   "lat": 55.7317915,\n   "lon": 37.6255144,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5058934759",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.5911469,\n   "lon": 37.759881,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-19:00"\n  },\n  {\n   "id": "node/5059086721",\n   "kind": "organisation",\n   "name": "Институт биохимии им. А. Н. Баха",\n   "office_type": "research",\n   "lat": 55.7114545,\n   "lon": 37.589718,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5059544926",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6820955,\n   "lon": 37.4855571,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5059549129",\n   "kind": "organisation",\n   "name": "ЮниКредит Лизинг",\n   "office_type": "company",\n   "lat": 55.7375966,\n   "lon": 37.6016504,\n   "opening_hours": "Mo-Fr 09:00-17:45"\n  },\n  {\n   "id": "node/5060968724",\n   "kind": "organisation",\n   "name": "Бюро переводов «ПереводПро.ру»",\n   "office_type": "company",\n   "lat": 55.8007027,\n   "lon": 37.4754371,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5063798422",\n   "kind": "organisation",\n   "name": "ИННО Имплант РУ",\n   "office_type": "company",\n   "lat": 55.780089,\n   "lon": 37.7106937,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5064014162",\n   "kind": "organisation",\n   "name": "Посольство Бахрейна",\n   "office_type": "diplomatic",\n   "lat": 55.7274122,\n   "lon": 37.6156908,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5068878700",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.6914368,\n   "lon": 37.6234328,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5072905571",\n   "kind": "organisation",\n   "name": "Федеральная налоговая служба",\n   "office_type": "government",\n   "lat": 55.774667,\n   "lon": 37.5493542,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5072956123",\n   "kind": "organisation",\n   "name": "Московская городская телефонная сеть",\n   "office_type": "telecommunication",\n   "lat": 55.7358231,\n   "lon": 37.617458,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5072956139",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7329387,\n   "lon": 37.6152839,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5073481545",\n   "kind": "organisation",\n   "name": "Каскадёр",\n   "office_type": "theatre",\n   "lat": 55.7612613,\n   "lon": 37.5972041,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5074033135",\n   "kind": "organisation",\n   "name": "Совет депутатов муниципального округа Аэропорт",\n   "office_type": "government",\n   "lat": 55.8060359,\n   "lon": 37.5251031,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5074033140",\n   "kind": "organisation",\n   "name": "Территориальная избирательная комиссия района Аэропорт",\n   "office_type": "government",\n   "lat": 55.8059638,\n   "lon": 37.52534,\n   "opening_hours": "Mo-Fr 15:00-19:00, Sa 10:00-14:00"\n  },\n  {\n   "id": "node/5074033141",\n   "kind": "organisation",\n   "name": "Избирательная комиссия муниципального округа Аэропорт",\n   "office_type": "government",\n   "lat": 55.8059313,\n   "lon": 37.5254702,\n   "opening_hours": "Mo-Fr 15:00-19:00, Sa 10:00-14:00"\n  },\n  {\n   "id": "node/5074033142",\n   "kind": "organisation",\n   "name": "Комиссия по делам несовершеннолетних",\n   "office_type": "government",\n   "lat": 55.8058995,\n   "lon": 37.5255791,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/5081625762",\n   "kind": "organisation",\n   "name": "Viagria.ru",\n   "office_type": "company",\n   "lat": 55.6758877,\n   "lon": 37.661734,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081943347",\n   "kind": "organisation",\n   "name": "Управление по обеспечению деятельности мировых судей города Москвы",\n   "office_type": "government",\n   "lat": 55.7532478,\n   "lon": 37.5765193,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081943348",\n   "kind": "organisation",\n   "name": "Управление записи актов гражданского состояния города Москвы",\n   "office_type": "government",\n   "lat": 55.7521007,\n   "lon": 37.5844235,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954313",\n   "kind": "organisation",\n   "name": "Главное контрольное управление города Москвы",\n   "office_type": "government",\n   "lat": 55.7534113,\n   "lon": 37.5763595,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954314",\n   "kind": "organisation",\n   "name": "Государственная жилищная инспекция города Москвы",\n   "office_type": "government",\n   "lat": 55.7779801,\n   "lon": 37.632167,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954315",\n   "kind": "organisation",\n   "name": "Государственная инспекция города Москвы по качеству сельскохозяйственной продукции, сырья и продовольствия",\n   "office_type": "government",\n   "lat": 55.7672815,\n   "lon": 37.584342,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954316",\n   "kind": "organisation",\n   "name": "Государственная инспекция по контролю за использованием объектов недвижимости города Москвы",\n   "office_type": "government",\n   "lat": 55.7595168,\n   "lon": 37.630313,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954317",\n   "kind": "organisation",\n   "name": "Департамент внешнеэкономических и международных связей города Москвы",\n   "office_type": "government",\n   "lat": 55.760609,\n   "lon": 37.6072347,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954318",\n   "kind": "organisation",\n   "name": "Департамент городского имущества г. Москвы",\n   "office_type": "government",\n   "lat": 55.7338916,\n   "lon": 37.6356388,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954319",\n   "kind": "organisation",\n   "name": "Департамент градостроительной политики города Москвы",\n   "office_type": "government",\n   "lat": 55.7570183,\n   "lon": 37.6098999,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954320",\n   "kind": "organisation",\n   "name": "Департамент здравоохранения города Москвы",\n   "office_type": "government",\n   "lat": 55.7735123,\n   "lon": 37.6071697,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954921",\n   "kind": "organisation",\n   "name": "Департамент информационных технологий города Москвы",\n   "office_type": "government",\n   "lat": 55.7688788,\n   "lon": 37.656018,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954923",\n   "kind": "organisation",\n   "name": "Департамент культуры города Москвы",\n   "office_type": "government",\n   "lat": 55.7615752,\n   "lon": 37.6205597,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954924",\n   "kind": "organisation",\n   "name": "Департамент науки, промышленной политики и предпринимательства города Москвы",\n   "office_type": "government",\n   "lat": 55.760515,\n   "lon": 37.6070699,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954925",\n   "kind": "organisation",\n   "name": "Департамент национальной политики и межрегиональных связей города Москвы",\n   "office_type": "government",\n   "lat": 55.7532631,\n   "lon": 37.5761597,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954926",\n   "kind": "organisation",\n   "name": "Департамент образования и науки города Москвы",\n   "office_type": "government",\n   "lat": 55.7744217,\n   "lon": 37.6397019,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954927",\n   "kind": "organisation",\n   "name": "Департамент природопользования и охраны окружающей среды города Москвы",\n   "office_type": "government",\n   "lat": 55.7520048,\n   "lon": 37.595115,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954928",\n   "kind": "organisation",\n   "name": "Департамент развития новых территорий города Москвы",\n   "office_type": "government",\n   "lat": 55.7528861,\n   "lon": 37.607192,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954929",\n   "kind": "organisation",\n   "name": "Департамент региональной безопасности и противодействия коррупции города Москвы",\n   "office_type": "government",\n   "lat": 55.7531992,\n   "lon": 37.5762287,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954930",\n   "kind": "organisation",\n   "name": "Департамент средств массовой информации и рекламы города Москвы",\n   "office_type": "government",\n   "lat": 55.7535334,\n   "lon": 37.5762394,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954931",\n   "kind": "organisation",\n   "name": "Департамент строительства города Москвы",\n   "office_type": "government",\n   "lat": 55.7622613,\n   "lon": 37.6146517,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954932",\n   "kind": "organisation",\n   "name": "Департамент территориальных органов исполнительной власти города Москвы",\n   "office_type": "government",\n   "lat": 55.7608424,\n   "lon": 37.6074262,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954933",\n   "kind": "organisation",\n   "name": "Департамент торговли и услуг города Москвы",\n   "office_type": "government",\n   "lat": 55.7655434,\n   "lon": 37.6031301,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954934",\n   "kind": "organisation",\n   "name": "Московское объединение ветеринарии",\n   "office_type": "government",\n   "lat": 55.7182,\n   "lon": 37.6055,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:45"\n  },\n  {\n   "id": "node/5081954935",\n   "kind": "organisation",\n   "name": "Комитет города Москвы по обеспечению реализации инвестиционных проектов в строительстве и контролю в области долевого строительства",\n   "office_type": "government",\n   "lat": 55.7529677,\n   "lon": 37.6073005,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954936",\n   "kind": "organisation",\n   "name": "Комитет государственных услуг города Москвы",\n   "office_type": "government",\n   "lat": 55.7609467,\n   "lon": 37.6075741,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954937",\n   "kind": "organisation",\n   "name": "Комитет общественных связей города Москвы",\n   "office_type": "government",\n   "lat": 55.7533398,\n   "lon": 37.5764322,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954938",\n   "kind": "organisation",\n   "name": "Московская административная дорожная инспекция",\n   "office_type": "government",\n   "lat": 55.7792131,\n   "lon": 37.6450382,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954939",\n   "kind": "organisation",\n   "name": "Объединение административно-технических инспекций города Москвы",\n   "office_type": "government",\n   "lat": 55.7536518,\n   "lon": 37.576094,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081954940",\n   "kind": "organisation",\n   "name": "Управление делами Мэра и Правительства Москвы",\n   "office_type": "government",\n   "lat": 55.7612888,\n   "lon": 37.6091726,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5081984338",\n   "kind": "organisation",\n   "name": "ООО \\"НСС\\"",\n   "office_type": "company",\n   "lat": 55.7169563,\n   "lon": 37.6238209,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5085589186",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Таганский",\n   "office_type": "government",\n   "lat": 55.7463409,\n   "lon": 37.6826072,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138164",\n   "kind": "organisation",\n   "name": "Военкомат района Замоскворечье",\n   "office_type": "government",\n   "lat": 55.7382551,\n   "lon": 37.5982685,\n   "opening_hours": "Mo-Th 10:00-13:00,14:45-18:00; Fr 10:00-13:00,14:45-17:00"\n  },\n  {\n   "id": "node/5086138166",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Тверского района (ОВК Тверского района)",\n   "office_type": "government",\n   "lat": 55.7562711,\n   "lon": 37.5489397,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138172",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Савёловского района",\n   "office_type": "government",\n   "lat": 55.7793789,\n   "lon": 37.5852021,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138176",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Измайловского района",\n   "office_type": "government",\n   "lat": 55.794388,\n   "lon": 37.7894492,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138177",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7454862,\n   "lon": 37.7929582,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138178",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Лефортовского района",\n   "office_type": "government",\n   "lat": 55.7167516,\n   "lon": 37.7409626,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138180",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Царицынского района",\n   "office_type": "government",\n   "lat": 55.6611602,\n   "lon": 37.6260855,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138183",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Черёмушкинского района",\n   "office_type": "government",\n   "lat": 55.6935419,\n   "lon": 37.5639124,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138184",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Гагаринского района",\n   "office_type": "government",\n   "lat": 55.6934119,\n   "lon": 37.563671,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138200",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат района Раменки",\n   "office_type": "government",\n   "lat": 55.6846658,\n   "lon": 37.4834897,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5086138201",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Тушинского района",\n   "office_type": "government",\n   "lat": 55.8292134,\n   "lon": 37.4490464,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5089481080",\n   "kind": "organisation",\n   "name": "РусГидро",\n   "office_type": "company",\n   "lat": 55.7682571,\n   "lon": 37.6054287,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5092713590",\n   "kind": "organisation",\n   "name": "Жилищник района Аэропорт",\n   "office_type": "property_management",\n   "lat": 55.8075631,\n   "lon": 37.5330529,\n   "opening_hours": "Mo-Tu 08:00-12:00,13:00-17:00; Fr 08:00-12:00,13:00-15:45"\n  },\n  {\n   "id": "node/5093092437",\n   "kind": "organisation",\n   "name": "Отдел жилищных субсидий «Аэропорт» для юр. лиц",\n   "office_type": "government",\n   "lat": 55.8072682,\n   "lon": 37.5379793,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:45"\n  },\n  {\n   "id": "node/5093092438",\n   "kind": "organisation",\n   "name": "Центр физической культуры и спорта северного административного округа",\n   "office_type": "government",\n   "lat": 55.8074603,\n   "lon": 37.5382105,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5095040224",\n   "kind": "organisation",\n   "name": "Совет ветеранов микрорайона 1",\n   "office_type": "ngo",\n   "lat": 55.7364364,\n   "lon": 37.6508448,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5095040254",\n   "kind": "organisation",\n   "name": "Массив-Паркет",\n   "office_type": "company",\n   "lat": 55.7351315,\n   "lon": 37.6520472,\n   "opening_hours": "Mo-Fr 08:00-19:00; Sa,Su 09:00-16:00"\n  },\n  {\n   "id": "node/5095040264",\n   "kind": "organisation",\n   "name": "ЗАО \\"НФК\\"",\n   "office_type": "company",\n   "lat": 55.7372399,\n   "lon": 37.6523044,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5095399818",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7347638,\n   "lon": 37.6521204,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5096491698",\n   "kind": "organisation",\n   "name": "Жилищник-1",\n   "office_type": "property_management",\n   "lat": 55.7570047,\n   "lon": 37.6359294,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5098363133",\n   "kind": "organisation",\n   "name": "Единый информационно-расчетный центр района Северное Чертаново",\n   "office_type": "utility_charge_accounting",\n   "lat": 55.6279283,\n   "lon": 37.604091,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5098756511",\n   "kind": "organisation",\n   "name": "Всероссийский студенческий союз",\n   "office_type": "ngo",\n   "lat": 55.7151168,\n   "lon": 37.6737419,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5098756512",\n   "kind": "organisation",\n   "name": "Конструкторское бюро по архитектурно-строительным системам и новым технологиям им. А.А. Якушева",\n   "office_type": "company",\n   "lat": 55.7151878,\n   "lon": 37.6739967,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5103213144",\n   "kind": "organisation",\n   "name": "ООО \\"Московия Центр Кадастрового Учёта\\"",\n   "office_type": "company",\n   "lat": 55.7893238,\n   "lon": 37.5833253,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5103375603",\n   "kind": "organisation",\n   "name": "ООО \\"Основа\\"",\n   "office_type": "company",\n   "lat": 55.8083365,\n   "lon": 37.5160725,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/5103763706",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6355946,\n   "lon": 37.7482322,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/5103763827",\n   "kind": "organisation",\n   "name": "Inventive Retail Group",\n   "office_type": "company",\n   "lat": 55.7731289,\n   "lon": 37.677784,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5104212702",\n   "kind": "organisation",\n   "name": "УВО по ЦАО ФГКУ «УВО ВНГ России по г. Москве» финансово-экономический отдел",\n   "office_type": "government",\n   "lat": 55.7558039,\n   "lon": 37.6025057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5104212706",\n   "kind": "organisation",\n   "name": "Управление по ЦАО ГУ МЧС",\n   "office_type": "government",\n   "lat": 55.7555101,\n   "lon": 37.6027644,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5104212707",\n   "kind": "organisation",\n   "name": "21 пожарно-спасательный отряд ФПС",\n   "office_type": "government",\n   "lat": 55.7554509,\n   "lon": 37.6027561,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5104427122",\n   "kind": "organisation",\n   "name": "Консульский отдел Посольства Беларуси",\n   "office_type": "diplomatic",\n   "lat": 55.7581034,\n   "lon": 37.6374137,\n   "opening_hours": "Mo,Tu,Th,Fr 09:30-12:30"\n  },\n  {\n   "id": "node/5104427221",\n   "kind": "organisation",\n   "name": "Деловой и культурный комплекс Посольства Беларуси",\n   "office_type": "yes",\n   "lat": 55.7587431,\n   "lon": 37.6367599,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5104499524",\n   "kind": "organisation",\n   "name": "ФГБУ \\"Главэкспертцентр\\"",\n   "office_type": "government",\n   "lat": 55.7263888,\n   "lon": 37.6069393,\n   "opening_hours": "Mo-Th 09:30-13:00,14:00-17:30; Fr 09:30-13:00,14:00-16:15"\n  },\n  {\n   "id": "node/5104704508",\n   "kind": "organisation",\n   "name": "Группа Компаний \\"Московия\\"",\n   "office_type": "company",\n   "lat": 55.7537977,\n   "lon": 37.5766327,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5106953143",\n   "kind": "organisation",\n   "name": "Нотариальная контора Новопашиной Ульяны Семеневны",\n   "office_type": "notary",\n   "lat": 55.7811436,\n   "lon": 37.6001863,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5111896287",\n   "kind": "organisation",\n   "name": "Чернигов И.О.",\n   "office_type": "lawyer",\n   "lat": 55.7671564,\n   "lon": 37.6082973,\n   "opening_hours": "Mo-Fr 11:00-21:00; Sa,Su 10:00-20:00"\n  },\n  {\n   "id": "node/5116773524",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6327086,\n   "lon": 37.5197727,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5117994924",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7825642,\n   "lon": 37.722936,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5118895722",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7873101,\n   "lon": 37.4787812,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa 09:00-16:00"\n  },\n  {\n   "id": "node/5119536726",\n   "kind": "organisation",\n   "name": "Стройсервис",\n   "office_type": "company",\n   "lat": 55.7085019,\n   "lon": 37.6655002,\n   "opening_hours": "Su-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5121656622",\n   "kind": "organisation",\n   "name": "Самокат",\n   "office_type": "publisher",\n   "lat": 55.7382823,\n   "lon": 37.6263733,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5128231322",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7506736,\n   "lon": 37.7896767,\n   "opening_hours": "Mo-Su 11:00-20:00"\n  },\n  {\n   "id": "node/5129194365",\n   "kind": "organisation",\n   "name": "Объединённый военный комиссариат Академического района",\n   "office_type": "government",\n   "lat": 55.6880168,\n   "lon": 37.5675908,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5129194366",\n   "kind": "organisation",\n   "name": "Зюзинская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.6878754,\n   "lon": 37.568323,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5130449522",\n   "kind": "organisation",\n   "name": "Московская Дирекция пассажирских обустройств",\n   "office_type": "company",\n   "lat": 55.7964268,\n   "lon": 37.6329959,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5130466121",\n   "kind": "organisation",\n   "name": "МДПОч",\n   "office_type": "company",\n   "lat": 55.8423783,\n   "lon": 37.5304569,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5131957909",\n   "kind": "organisation",\n   "name": "ОДС-6",\n   "office_type": "property_management",\n   "lat": 55.7695524,\n   "lon": 37.6171325,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5134380906",\n   "kind": "organisation",\n   "name": "КПРФ",\n   "office_type": "political_party",\n   "lat": 55.6470272,\n   "lon": 37.603664,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5137499423",\n   "kind": "organisation",\n   "name": "Google",\n   "office_type": "it",\n   "lat": 55.7467448,\n   "lon": 37.6264988,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5137519922",\n   "kind": "organisation",\n   "name": "Рамблер",\n   "office_type": "company",\n   "lat": 55.6985512,\n   "lon": 37.6248251,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5140609323",\n   "kind": "organisation",\n   "name": "БМЗ",\n   "office_type": "company",\n   "lat": 55.8433398,\n   "lon": 37.524753,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/5141360352",\n   "kind": "organisation",\n   "name": "Air Astana",\n   "office_type": "company",\n   "lat": 55.7615927,\n   "lon": 37.605418,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5141360353",\n   "kind": "organisation",\n   "name": "Новые облачные технологии",\n   "office_type": "company",\n   "lat": 55.7615738,\n   "lon": 37.6056097,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5141360355",\n   "kind": "organisation",\n   "name": "Всероссийское общество автомобилистов",\n   "office_type": "ngo",\n   "lat": 55.7614259,\n   "lon": 37.6061985,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5141360380",\n   "kind": "organisation",\n   "name": "ООО \\"Академия частной жизни\\"",\n   "office_type": "company",\n   "lat": 55.7429665,\n   "lon": 37.6621293,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5141534797",\n   "kind": "organisation",\n   "name": "Бюро Кабельных Технологий",\n   "office_type": "company",\n   "lat": 55.7540924,\n   "lon": 37.6428442,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5142822222",\n   "kind": "organisation",\n   "name": "АО «Инфраструктурный проект»(ISPGroup)",\n   "office_type": "company",\n   "lat": 55.6221571,\n   "lon": 37.6212978,\n   "opening_hours": "Mo-Fr 08:00-18:30"\n  },\n  {\n   "id": "node/5144881922",\n   "kind": "organisation",\n   "name": "Современный Сервис",\n   "office_type": "company",\n   "lat": 55.7779852,\n   "lon": 37.5557923,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/5145226411",\n   "kind": "organisation",\n   "name": "Cakeland",\n   "office_type": "company",\n   "lat": 55.8062582,\n   "lon": 37.6219068,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 11:00-17:00"\n  },\n  {\n   "id": "node/5149934825",\n   "kind": "organisation",\n   "name": "Русал",\n   "office_type": "company",\n   "lat": 55.7475827,\n   "lon": 37.6518717,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5151779450",\n   "kind": "organisation",\n   "name": "Магазин софта",\n   "office_type": "company",\n   "lat": 55.8130243,\n   "lon": 37.6387384,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5155009891",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7091338,\n   "lon": 37.6209148,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5155307425",\n   "kind": "organisation",\n   "name": "Байлар",\n   "office_type": "estate_agent",\n   "lat": 55.7165325,\n   "lon": 37.610863,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5155784283",\n   "kind": "organisation",\n   "name": "Либерально-демократическая партия России",\n   "office_type": "political_party",\n   "lat": 55.7910184,\n   "lon": 37.4970401,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5155784285",\n   "kind": "organisation",\n   "name": "ЮВЕГА-Сити",\n   "office_type": "estate_agent",\n   "lat": 55.7897444,\n   "lon": 37.4962632,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5160650711",\n   "kind": "organisation",\n   "name": "Allbearing.ru",\n   "office_type": "company",\n   "lat": 55.7076833,\n   "lon": 37.69664,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5161366423",\n   "kind": "organisation",\n   "name": "Джиэльти - дизайн",\n   "office_type": "company",\n   "lat": 55.7010121,\n   "lon": 37.503551,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5161593994",\n   "kind": "organisation",\n   "name": "Объединенная судостроительная корпорация",\n   "office_type": "company",\n   "lat": 55.7408229,\n   "lon": 37.6334788,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5162527021",\n   "kind": "organisation",\n   "name": "Москоувайн",\n   "office_type": "company",\n   "lat": 55.7428733,\n   "lon": 37.5986156,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/5162599424",\n   "kind": "organisation",\n   "name": "IBM",\n   "office_type": "it",\n   "lat": 55.7463975,\n   "lon": 37.5363771,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5164986821",\n   "kind": "organisation",\n   "name": "SKF Противомоскитные системы",\n   "office_type": "company",\n   "lat": 55.6803035,\n   "lon": 37.6918728,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5169109506",\n   "kind": "organisation",\n   "name": "Зюзинский межрайонный следственный отдел",\n   "office_type": "government",\n   "lat": 55.5910014,\n   "lon": 37.5391982,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5170196474",\n   "kind": "organisation",\n   "name": "Институт Африки РАН",\n   "office_type": "research",\n   "lat": 55.7623435,\n   "lon": 37.5911593,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5172555021",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7089941,\n   "lon": 37.6584253,\n   "opening_hours": "Mo-Th 10:00-18:00; Fr 10:00-14:00"\n  },\n  {\n   "id": "node/5174725122",\n   "kind": "organisation",\n   "name": "Никифоров В.Н.",\n   "office_type": "notary",\n   "lat": 55.7172368,\n   "lon": 37.7857789,\n   "opening_hours": "Mo-Su 10:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/5176321688",\n   "kind": "organisation",\n   "name": "Федеральная служба по экологическому, технологическому, и атомному надзору",\n   "office_type": "government",\n   "lat": 55.7686933,\n   "lon": 37.6639095,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5176456934",\n   "kind": "organisation",\n   "name": "Российский Союз Боевых Искусств",\n   "office_type": "ngo",\n   "lat": 55.7643109,\n   "lon": 37.6718453,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5180960835",\n   "kind": "organisation",\n   "name": "Дар Арктики",\n   "office_type": "yes",\n   "lat": 55.8018464,\n   "lon": 37.5504416,\n   "opening_hours": "Mo-Fr 11:00-18:00; Sa 12:00-16:00"\n  },\n  {\n   "id": "node/5180960836",\n   "kind": "organisation",\n   "name": "МБР-Авиа",\n   "office_type": "travel_agent",\n   "lat": 55.8018373,\n   "lon": 37.5505226,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5180960907",\n   "kind": "organisation",\n   "name": "ООО «Командир»",\n   "office_type": "company",\n   "lat": 55.8021745,\n   "lon": 37.5512944,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5181573645",\n   "kind": "organisation",\n   "name": "Инсайт Груп",\n   "office_type": "company",\n   "lat": 55.7593817,\n   "lon": 37.6323637,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/5181712732",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7292247,\n   "lon": 37.6227786,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5182606929",\n   "kind": "organisation",\n   "name": "Федеральное агентство лесного хозяйства",\n   "office_type": "government",\n   "lat": 55.7334542,\n   "lon": 37.6288001,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5184187523",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8106346,\n   "lon": 37.8155793,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/5186059815",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8180439,\n   "lon": 37.5734474,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5188229103",\n   "kind": "organisation",\n   "name": "ООО \\"Акцент-СБ\\"",\n   "office_type": "company",\n   "lat": 55.7491778,\n   "lon": 37.7640188,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5189353529",\n   "kind": "organisation",\n   "name": "Альфа Трейд",\n   "office_type": "company",\n   "lat": 55.8786638,\n   "lon": 37.5697724,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5189582125",\n   "kind": "organisation",\n   "name": "ТелемедХелп",\n   "office_type": "company",\n   "lat": 55.7927046,\n   "lon": 37.5272768,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5192481122",\n   "kind": "organisation",\n   "name": "ИскраУралТел",\n   "office_type": "company",\n   "lat": 55.7263749,\n   "lon": 37.6511547,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5193872716",\n   "kind": "organisation",\n   "name": "Офис продаж \\"Парк легенд\\"",\n   "office_type": "estate_agent",\n   "lat": 55.7029748,\n   "lon": 37.6483333,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5193952522",\n   "kind": "organisation",\n   "name": "MaxStyle Digital Agency",\n   "office_type": "company",\n   "lat": 55.6479961,\n   "lon": 37.5414438,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5194103200",\n   "kind": "organisation",\n   "name": "Мир Гравировки",\n   "office_type": "company",\n   "lat": 55.7811251,\n   "lon": 37.6036075,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5196240023",\n   "kind": "organisation",\n   "name": "Институт территориального планирования \\"Урбаника\\"",\n   "office_type": "company",\n   "lat": 55.7804416,\n   "lon": 37.5872266,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5198355693",\n   "kind": "organisation",\n   "name": "Солидарность паблишерс",\n   "office_type": "company",\n   "lat": 55.8058997,\n   "lon": 37.5288511,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/5198355694",\n   "kind": "organisation",\n   "name": "Межправительственный совет дорожников",\n   "office_type": "government",\n   "lat": 55.8058353,\n   "lon": 37.5288006,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5198355695",\n   "kind": "organisation",\n   "name": "Региональный фонд содействия ветеранам МВД",\n   "office_type": "government",\n   "lat": 55.8058091,\n   "lon": 37.5287754,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5199351203",\n   "kind": "organisation",\n   "name": "Измайловская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7946561,\n   "lon": 37.7788067,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5200543422",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8546444,\n   "lon": 37.4774561,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5200569822",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8546846,\n   "lon": 37.4776126,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/5200575322",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8546042,\n   "lon": 37.4772996,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5208119223",\n   "kind": "organisation",\n   "name": "Мегатехникс",\n   "office_type": "company",\n   "lat": 55.8046418,\n   "lon": 37.5304858,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5209132421",\n   "kind": "organisation",\n   "name": "ПОКСИДОМ ООО",\n   "office_type": "company",\n   "lat": 55.6491526,\n   "lon": 37.6250229,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/5209211217",\n   "kind": "organisation",\n   "name": "Инком недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.8060475,\n   "lon": 37.5167306,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa 09:30-19:30; Su 10:00-18:00"\n  },\n  {\n   "id": "node/5218234049",\n   "kind": "organisation",\n   "name": "ЗАО «Генеральная Дирекция Центр»",\n   "office_type": "company",\n   "lat": 55.7637356,\n   "lon": 37.6021229,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5218469164",\n   "kind": "organisation",\n   "name": "Фонд развития интернет-инициатив",\n   "office_type": "company",\n   "lat": 55.7633425,\n   "lon": 37.6332459,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5218798145",\n   "kind": "organisation",\n   "name": "Бутик-офис",\n   "office_type": "company",\n   "lat": 55.7378338,\n   "lon": 37.6018507,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5219218621",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.726973,\n   "lon": 37.6261807,\n   "opening_hours": "Mo-Fr 10:00-17:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5219421023",\n   "kind": "organisation",\n   "name": "Профессии будущего",\n   "office_type": "government",\n   "lat": 55.7814816,\n   "lon": 37.6306585,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5219955621",\n   "kind": "organisation",\n   "name": "ФБУ Государственная комиссия по запасам полезных ископаемых",\n   "office_type": "government",\n   "lat": 55.731084,\n   "lon": 37.6214698,\n   "opening_hours": "Mo-Fr 08:00-12:00,13:00-18:00"\n  },\n  {\n   "id": "node/5220093396",\n   "kind": "organisation",\n   "name": "ESA Moscow",\n   "office_type": "company",\n   "lat": 55.7655199,\n   "lon": 37.6350403,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5222849879",\n   "kind": "organisation",\n   "name": "Городская реклама и информация",\n   "office_type": "government",\n   "lat": 55.8066181,\n   "lon": 37.5173486,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/5223487436",\n   "kind": "organisation",\n   "name": "Nolwo",\n   "office_type": "company",\n   "lat": 55.8029248,\n   "lon": 37.5255746,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5223487440",\n   "kind": "organisation",\n   "name": "АО «Поликон»",\n   "office_type": "company",\n   "lat": 55.8029136,\n   "lon": 37.525625,\n   "opening_hours": "Mo-Fr 10:00-14:00,15:00-19:00"\n  },\n  {\n   "id": "node/5223762230",\n   "kind": "organisation",\n   "name": "Центр обслуживания клиентов МОЭСК",\n   "office_type": "company",\n   "lat": 55.8116118,\n   "lon": 37.5856962,\n   "opening_hours": "Mo-Fr 09:00-17:00; Sa 09:00-14:00"\n  },\n  {\n   "id": "node/5223762231",\n   "kind": "organisation",\n   "name": "МОЭСК Северные электрические сети",\n   "office_type": "company",\n   "lat": 55.8115619,\n   "lon": 37.585329,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5223945721",\n   "kind": "organisation",\n   "name": "ОАО «Московская объединенная энергетическая компания»",\n   "office_type": "company",\n   "lat": 55.7982262,\n   "lon": 37.5937342,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5223967821",\n   "kind": "organisation",\n   "name": "Времена Года",\n   "office_type": "company",\n   "lat": 55.8014862,\n   "lon": 37.595655,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5224960915",\n   "kind": "organisation",\n   "name": "Администрация",\n   "office_type": "yes",\n   "lat": 55.8150159,\n   "lon": 37.5889831,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5225513427",\n   "kind": "organisation",\n   "name": "Межрегиональное бюро кадастровых работ",\n   "office_type": "company",\n   "lat": 55.8041515,\n   "lon": 37.5247252,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/5225926322",\n   "kind": "organisation",\n   "name": "Московское городское региональное отделение Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.7781831,\n   "lon": 37.6161679,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5225947513",\n   "kind": "organisation",\n   "name": "2ГИС",\n   "office_type": "it",\n   "lat": 55.6952686,\n   "lon": 37.6250816,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5226140695",\n   "kind": "organisation",\n   "name": "Игл текнолоджиз",\n   "office_type": "company",\n   "lat": 55.8053399,\n   "lon": 37.5273668,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5226847082",\n   "kind": "organisation",\n   "name": "Старград",\n   "office_type": "company",\n   "lat": 55.7313642,\n   "lon": 37.5649158,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5227014555",\n   "kind": "organisation",\n   "name": "РЕСО",\n   "office_type": "insurance",\n   "lat": 55.7075775,\n   "lon": 37.6207818,\n   "opening_hours": "Mo-Fr 09:30-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/5230399260",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8085711,\n   "lon": 37.5304858,\n   "opening_hours": "Mo-Sa 10:00-21:00; Su 10:00-20:00"\n  },\n  {\n   "id": "node/5230399264",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.8086302,\n   "lon": 37.5308226,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/5230921935",\n   "kind": "organisation",\n   "name": "Грин Хилл",\n   "office_type": "company",\n   "lat": 55.8024923,\n   "lon": 37.5437714,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5230921976",\n   "kind": "organisation",\n   "name": "Валтекс-НТ",\n   "office_type": "company",\n   "lat": 55.8021118,\n   "lon": 37.5436826,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5232309434",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7240256,\n   "lon": 37.8257896,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5232641520",\n   "kind": "organisation",\n   "name": "М.П.О. Аудит",\n   "office_type": "company",\n   "lat": 55.8016824,\n   "lon": 37.5175706,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5232641733",\n   "kind": "organisation",\n   "name": "Financial broker",\n   "office_type": "financial",\n   "lat": 55.8071323,\n   "lon": 37.5168354,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/5236281739",\n   "kind": "organisation",\n   "name": "Агентство промышленного развития города Москвы",\n   "office_type": "government",\n   "lat": 55.7602888,\n   "lon": 37.6070367,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5239507431",\n   "kind": "organisation",\n   "name": "probok.net",\n   "office_type": "ngo",\n   "lat": 55.721356,\n   "lon": 37.6261128,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5240047797",\n   "kind": "organisation",\n   "name": "Общественная приёмная Зюганова Л.А.",\n   "office_type": "yes",\n   "lat": 55.8063524,\n   "lon": 37.5168939,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5240047799",\n   "kind": "organisation",\n   "name": "Объединённая диспетчерская служба № 3",\n   "office_type": "property_management",\n   "lat": 55.8063071,\n   "lon": 37.5168538,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5240047801",\n   "kind": "organisation",\n   "name": "Сервис-ККМ",\n   "office_type": "company",\n   "lat": 55.8056663,\n   "lon": 37.5169112,\n   "opening_hours": "Mo-Tu 09:30-18:00; Fr 09:30-17:00"\n  },\n  {\n   "id": "node/5240047802",\n   "kind": "organisation",\n   "name": "Альфа-Сервис",\n   "office_type": "company",\n   "lat": 55.8056468,\n   "lon": 37.5169891,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5240047815",\n   "kind": "organisation",\n   "name": "Дизайн-студия Renova",\n   "office_type": "architect",\n   "lat": 55.8055109,\n   "lon": 37.5187536,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5240047816",\n   "kind": "organisation",\n   "name": "Art Holz",\n   "office_type": "architect",\n   "lat": 55.8051689,\n   "lon": 37.5183976,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5240047818",\n   "kind": "organisation",\n   "name": "ПАО «Научно-производственная корпорация \\"Иркут\\"»",\n   "office_type": "company",\n   "lat": 55.8046137,\n   "lon": 37.525067,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5240254511",\n   "kind": "organisation",\n   "name": "Московская межрегиональная транспортная прокуратура",\n   "office_type": "government",\n   "lat": 55.7819521,\n   "lon": 37.6777848,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5241292721",\n   "kind": "organisation",\n   "name": "Филиал РАО \\"ЕЭС России\\"",\n   "office_type": "company",\n   "lat": 55.66207,\n   "lon": 37.5345592,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5243127261",\n   "kind": "organisation",\n   "name": "МойТехосмотр",\n   "office_type": "insurance",\n   "lat": 55.8865203,\n   "lon": 37.4930614,\n   "opening_hours": "Mo-Su 09:30-19:00"\n  },\n  {\n   "id": "node/5244040523",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "estate_agent",\n   "lat": 55.744171,\n   "lon": 37.5605599,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5244167451",\n   "kind": "organisation",\n   "name": "Бакулина Юлия Николаевна",\n   "office_type": "notary",\n   "lat": 55.7638038,\n   "lon": 37.6072883,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5247160683",\n   "kind": "organisation",\n   "name": "Управление cлужбы занятости",\n   "office_type": "government",\n   "lat": 55.7928391,\n   "lon": 37.793186,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5249045952",\n   "kind": "organisation",\n   "name": "Сидорук Наталья Николаевна",\n   "office_type": "notary",\n   "lat": 55.8032205,\n   "lon": 37.5189207,\n   "opening_hours": "Mo-Fr 10:00-13:00,14:00-18:00; Sa 11:00-15:00"\n  },\n  {\n   "id": "node/5249270707",\n   "kind": "organisation",\n   "name": "ГКУ «АМПП»",\n   "office_type": "government",\n   "lat": 55.7774719,\n   "lon": 37.5749674,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5249589547",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7751312,\n   "lon": 37.5837431,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5249950235",\n   "kind": "organisation",\n   "name": "Центральный научно-исследовательский институт «Курс»",\n   "office_type": "research",\n   "lat": 55.7787,\n   "lon": 37.73694,\n   "opening_hours": "Mo-Th 08:30-17:15; Fr 08:30-16:00"\n  },\n  {\n   "id": "node/5250465866",\n   "kind": "organisation",\n   "name": "Бирюкова Ольга Михайловна",\n   "office_type": "notary",\n   "lat": 55.8077328,\n   "lon": 37.5319904,\n   "opening_hours": "Mo 14:00-20:00; Tu-Th 10:00-13:00,14:00-17:00; Fr 10:00-15:00"\n  },\n  {\n   "id": "node/5250465867",\n   "kind": "organisation",\n   "name": "Адвокат Родина С.В.",\n   "office_type": "lawyer",\n   "lat": 55.8077538,\n   "lon": 37.5320074,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5250465868",\n   "kind": "organisation",\n   "name": "КБ ЭГИДА",\n   "office_type": "lawyer",\n   "lat": 55.8077744,\n   "lon": 37.5320308,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:00"\n  },\n  {\n   "id": "node/5251030543",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6525712,\n   "lon": 37.7563023,\n   "opening_hours": "Mo-Th 10:00-19:00; Fr 10:00-19:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/5251038258",\n   "kind": "organisation",\n   "name": "Центр страхования",\n   "office_type": "insurance",\n   "lat": 55.6526265,\n   "lon": 37.7562289,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/5252876721",\n   "kind": "organisation",\n   "name": "Проходная \\"МКБ Компас\\"",\n   "office_type": "company",\n   "lat": 55.7409757,\n   "lon": 37.7897158,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5257252697",\n   "kind": "organisation",\n   "name": "Ресо",\n   "office_type": "insurance",\n   "lat": 55.7819196,\n   "lon": 37.7242264,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5257367721",\n   "kind": "organisation",\n   "name": "Сбербанк",\n   "office_type": "bank",\n   "lat": 55.7414645,\n   "lon": 37.5311218,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5257691121",\n   "kind": "organisation",\n   "name": "Фонд социального страхования. Филиал 29",\n   "office_type": "government",\n   "lat": 55.8071639,\n   "lon": 37.6457461,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/5259575001",\n   "kind": "organisation",\n   "name": "Плотникова А. В.",\n   "office_type": "notary",\n   "lat": 55.8005327,\n   "lon": 37.5485158,\n   "opening_hours": "Mo-Fr 10:00-14:00,15:00-19:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/5259575003",\n   "kind": "organisation",\n   "name": "Марс-секьюрити",\n   "office_type": "company",\n   "lat": 55.8004845,\n   "lon": 37.5485817,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5259671023",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.6428144,\n   "lon": 37.6062401,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa, Su 09:00-15:00"\n  },\n  {\n   "id": "node/5260031307",\n   "kind": "organisation",\n   "name": "Kraftway",\n   "office_type": "it",\n   "lat": 55.8069355,\n   "lon": 37.6458308,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5260198489",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7082268,\n   "lon": 37.6592651,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/5261051222",\n   "kind": "organisation",\n   "name": "Исследовательский центр частного права им. С. С. Алексеева",\n   "office_type": "company",\n   "lat": 55.7550756,\n   "lon": 37.626691,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5262175421",\n   "kind": "organisation",\n   "name": "Центр урегулирования страховых случаев",\n   "office_type": "company",\n   "lat": 55.7913172,\n   "lon": 37.6031123,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/5263138521",\n   "kind": "organisation",\n   "name": "Трансбизнескар",\n   "office_type": "company",\n   "lat": 55.6742723,\n   "lon": 37.6342428,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5263138522",\n   "kind": "organisation",\n   "name": "Такси \\"Шоколад\\"",\n   "office_type": "company",\n   "lat": 55.6742717,\n   "lon": 37.6329601,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5263150921",\n   "kind": "organisation",\n   "name": "Ирбис-СВ",\n   "office_type": "company",\n   "lat": 55.7884538,\n   "lon": 37.6626895,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5263151021",\n   "kind": "organisation",\n   "name": "МЦК",\n   "office_type": "company",\n   "lat": 55.6835475,\n   "lon": 37.629998,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5263299321",\n   "kind": "organisation",\n   "name": "Такси 2412",\n   "office_type": "company",\n   "lat": 55.6594313,\n   "lon": 37.5349074,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/5263709121",\n   "kind": "organisation",\n   "name": "Такси \\"Лето\\"/Прайм",\n   "office_type": "company",\n   "lat": 55.8290052,\n   "lon": 37.6509814,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5265181256",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.6531885,\n   "lon": 37.7349982,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5270737123",\n   "kind": "organisation",\n   "name": "Московский научно-практический центр наркологии",\n   "office_type": "government",\n   "lat": 55.6715844,\n   "lon": 37.7527091,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/5271307221",\n   "kind": "organisation",\n   "name": "Радио Шансон",\n   "office_type": "telecommunication",\n   "lat": 55.8113567,\n   "lon": 37.6423211,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/5271905223",\n   "kind": "organisation",\n   "name": "Простор",\n   "office_type": "estate_agent",\n   "lat": 55.7583967,\n   "lon": 37.5887135,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5273471522",\n   "kind": "organisation",\n   "name": "Консультант Плюс",\n   "office_type": "company",\n   "lat": 55.678761,\n   "lon": 37.5748133,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5274311625",\n   "kind": "organisation",\n   "name": "Фонд поддержки социальных благотворительных инициатив",\n   "office_type": "ngo",\n   "lat": 55.8049317,\n   "lon": 37.5498689,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5275595463",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.8929201,\n   "lon": 37.6547409,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5281696419",\n   "kind": "organisation",\n   "name": "Радченко И.В.",\n   "office_type": "notary",\n   "lat": 55.7569743,\n   "lon": 37.6381559,\n   "opening_hours": "Mo-Th 10:00-18:00; Fr 10:00-17:00"\n  },\n  {\n   "id": "node/5282334339",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.6995115,\n   "lon": 37.579314,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5283680221",\n   "kind": "organisation",\n   "name": "ГеоБренд",\n   "office_type": "company",\n   "lat": 55.7600166,\n   "lon": 37.7644477,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:00"\n  },\n  {\n   "id": "node/5283748322",\n   "kind": "organisation",\n   "name": "Соловьёв Игорь Алексеевич",\n   "office_type": "notary",\n   "lat": 55.7744514,\n   "lon": 37.5931702,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5284896338",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "translator",\n   "lat": 55.7569677,\n   "lon": 37.6382187,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5286249917",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7055294,\n   "lon": 37.5777045,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5289122521",\n   "kind": "organisation",\n   "name": "Московский электромеханический ремонтный завод МЭМРЗ",\n   "office_type": "company",\n   "lat": 55.8025799,\n   "lon": 37.6489788,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5290442712",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6557655,\n   "lon": 37.7318372,\n   "opening_hours": "Mo-Fr 09:00-21:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/5290666554",\n   "kind": "organisation",\n   "name": "Манго Телеком",\n   "office_type": "company",\n   "lat": 55.6624831,\n   "lon": 37.5470417,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5292056070",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7587451,\n   "lon": 37.7518255,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5292540296",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных услуг района Филёвский парк",\n   "office_type": "government",\n   "lat": 55.7522728,\n   "lon": 37.5006858,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/5293758297",\n   "kind": "organisation",\n   "name": "Обьединеная диспетчерская служба",\n   "office_type": "property_management",\n   "lat": 55.7431417,\n   "lon": 37.7677211,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5296326921",\n   "kind": "organisation",\n   "name": "ООО \\"МТГ. Бизнес-решения\\"",\n   "office_type": "company",\n   "lat": 55.7205956,\n   "lon": 37.774306,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5300666423",\n   "kind": "organisation",\n   "name": "РЕСО",\n   "office_type": "insurance",\n   "lat": 55.7218289,\n   "lon": 37.6733255,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5301815572",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.8101573,\n   "lon": 37.63211,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5302593721",\n   "kind": "organisation",\n   "name": "Ориентир",\n   "office_type": "estate_agent",\n   "lat": 55.7237393,\n   "lon": 37.6080251,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5303619329",\n   "kind": "organisation",\n   "name": "Союз ветеранов Афганистана",\n   "office_type": "ngo",\n   "lat": 55.730008,\n   "lon": 37.5573825,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5303619330",\n   "kind": "organisation",\n   "name": "Военно-спортивный клуб \\"Герат\\"",\n   "office_type": "company",\n   "lat": 55.7300347,\n   "lon": 37.5573541,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5305719156",\n   "kind": "organisation",\n   "name": "Центральный Объединенный Архив Департамента Здравоохранения г. Москвы",\n   "office_type": "government",\n   "lat": 55.6840659,\n   "lon": 37.7409795,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5307261001",\n   "kind": "organisation",\n   "name": "ГБУ «Жилищник»",\n   "office_type": "property_management",\n   "lat": 55.807793,\n   "lon": 37.5335756,\n   "opening_hours": "Mo-Su 08:00-17:00"\n  },\n  {\n   "id": "node/5308193121",\n   "kind": "organisation",\n   "name": "РусЭнергоСервис",\n   "office_type": "company",\n   "lat": 55.7408007,\n   "lon": 37.671975,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5308358474",\n   "kind": "organisation",\n   "name": "Резиденция посла Нигерии",\n   "office_type": "diplomatic",\n   "lat": 55.757914,\n   "lon": 37.5904512,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5312278166",\n   "kind": "organisation",\n   "name": "ЖСК «Советский писатель»",\n   "office_type": "property_management",\n   "lat": 55.8025115,\n   "lon": 37.5350584,\n   "opening_hours": "Tu 14:00-18:00; We 15:00-19:00"\n  },\n  {\n   "id": "node/5312278180",\n   "kind": "organisation",\n   "name": "LinkLine",\n   "office_type": "telecommunication",\n   "lat": 55.8025364,\n   "lon": 37.5350863,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5312348196",\n   "kind": "organisation",\n   "name": "Арт-родник",\n   "office_type": "publisher",\n   "lat": 55.8029784,\n   "lon": 37.535822,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5315478721",\n   "kind": "organisation",\n   "name": "ОБИТ",\n   "office_type": "telecommunication",\n   "lat": 55.7132486,\n   "lon": 37.6204373,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5317006661",\n   "kind": "organisation",\n   "name": "Эж-ЮРИСТ",\n   "office_type": "newspaper",\n   "lat": 55.8073668,\n   "lon": 37.5408632,\n   "opening_hours": "Mo-Th 09:15-18:00; Fr 09:15-16:45"\n  },\n  {\n   "id": "node/5317006663",\n   "kind": "organisation",\n   "name": "Издательский дом «Экономическая газета»",\n   "office_type": "newspaper",\n   "lat": 55.8074503,\n   "lon": 37.5410311,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5317006664",\n   "kind": "organisation",\n   "name": "ООО «Медиагруппа «Журналист»",\n   "office_type": "newspaper",\n   "lat": 55.8074939,\n   "lon": 37.5411216,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5317957423",\n   "kind": "organisation",\n   "name": "Бросвил",\n   "office_type": "company",\n   "lat": 55.7461098,\n   "lon": 37.8311196,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5318264921",\n   "kind": "organisation",\n   "name": "Диспетчерская ГБУ \\"Жилищник Ярославского района\\"",\n   "office_type": "property_management",\n   "lat": 55.8741357,\n   "lon": 37.711315,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5318435752",\n   "kind": "organisation",\n   "name": "ООО \\"ТСЖ Аэробус\\"",\n   "office_type": "property_management",\n   "lat": 55.8067969,\n   "lon": 37.5439831,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-19:00"\n  },\n  {\n   "id": "node/5318507449",\n   "kind": "organisation",\n   "name": "Еврейская община «Сокол-Аэропорт»",\n   "office_type": "ngo",\n   "lat": 55.8062293,\n   "lon": 37.5454248,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5318507451",\n   "kind": "organisation",\n   "name": "Олимп Стройсервис",\n   "office_type": "architect",\n   "lat": 55.8064506,\n   "lon": 37.5459753,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5318678346",\n   "kind": "organisation",\n   "name": "Администрация ЖК \\"Аэробус\\"",\n   "office_type": "property_management",\n   "lat": 55.8067713,\n   "lon": 37.5440237,\n   "opening_hours": "Mo-Fr 10:00-14:00,15:00-19:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/5320068822",\n   "kind": "organisation",\n   "name": "Майтони",\n   "office_type": "company",\n   "lat": 55.6822235,\n   "lon": 37.4848328,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5322272421",\n   "kind": "organisation",\n   "name": "Даталайн",\n   "office_type": "company",\n   "lat": 55.7704437,\n   "lon": 37.7157699,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5322312592",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8136179,\n   "lon": 37.575606,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5322312594",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8137626,\n   "lon": 37.5767192,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00"\n  },\n  {\n   "id": "node/5322383870",\n   "kind": "organisation",\n   "name": "Посольство Монголии",\n   "office_type": "diplomatic",\n   "lat": 55.7501248,\n   "lon": 37.5869617,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5323246721",\n   "kind": "organisation",\n   "name": "МСМ-групп",\n   "office_type": "company",\n   "lat": 55.7349434,\n   "lon": 37.5661515,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5324297105",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.8526298,\n   "lon": 37.6813969,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45; Sa-Su off"\n  },\n  {\n   "id": "node/5324297106",\n   "kind": "organisation",\n   "name": "Московский лицей",\n   "office_type": "publishing",\n   "lat": 55.8518183,\n   "lon": 37.6795623,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5327043023",\n   "kind": "organisation",\n   "name": "ООО\\"Вербета\\"",\n   "office_type": "company",\n   "lat": 55.7962703,\n   "lon": 37.5612181,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5328209498",\n   "kind": "organisation",\n   "name": "Федерация хоккея России",\n   "office_type": "company",\n   "lat": 55.7021177,\n   "lon": 37.6535441,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5328224922",\n   "kind": "organisation",\n   "name": "Adconsult",\n   "office_type": "company",\n   "lat": 55.8106081,\n   "lon": 37.6316563,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5329509623",\n   "kind": "organisation",\n   "name": "Институт управления образованием РАО",\n   "office_type": "research",\n   "lat": 55.761782,\n   "lon": 37.6478884,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5331561040",\n   "kind": "organisation",\n   "name": "Геометрия",\n   "office_type": "company",\n   "lat": 55.7426596,\n   "lon": 37.5453515,\n   "opening_hours": "Mo-Th 09:00-20:00; Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5332088935",\n   "kind": "organisation",\n   "name": "Представительство Ханты-Мансийского автономного округа - Югры",\n   "office_type": "government",\n   "lat": 55.7451089,\n   "lon": 37.5962019,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5333589346",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичной организации №7 района Сокол",\n   "office_type": "ngo",\n   "lat": 55.7981771,\n   "lon": 37.5090272,\n   "opening_hours": "Mo 11:00-14:00; Th 11:00-14:00"\n  },\n  {\n   "id": "node/5333589347",\n   "kind": "organisation",\n   "name": "Руспломба",\n   "office_type": "company",\n   "lat": 55.798199,\n   "lon": 37.5089125,\n   "opening_hours": "Mo-Th 08:30-17:30; Fr 08:30-16:00"\n  },\n  {\n   "id": "node/5333967572",\n   "kind": "organisation",\n   "name": "Творческое пространство ООО \\"Юпитер\\"",\n   "office_type": "company",\n   "lat": 55.7972067,\n   "lon": 37.5094687,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5334710944",\n   "kind": "organisation",\n   "name": "ОДС-9",\n   "office_type": "company",\n   "lat": 55.8150655,\n   "lon": 37.7023449,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5335119820",\n   "kind": "organisation",\n   "name": "Столичная Сберегательная Компания",\n   "office_type": "company",\n   "lat": 55.7755729,\n   "lon": 37.5950539,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5336819024",\n   "kind": "organisation",\n   "name": "Администрация совхоза имени Ленина",\n   "office_type": "government",\n   "lat": 55.5852196,\n   "lon": 37.731285,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5336819126",\n   "kind": "organisation",\n   "name": "Мои документы",\n   "office_type": "government",\n   "lat": 55.5852711,\n   "lon": 37.731289,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5336928444",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7100616,\n   "lon": 37.6204859,\n   "opening_hours": "Mo-Fr 09:00-22:00; Sa,Su 9:00-22:00"\n  },\n  {\n   "id": "node/5337863422",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7826201,\n   "lon": 37.7227445,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/5338565833",\n   "kind": "organisation",\n   "name": "Архив кадровой документации северного административного округа",\n   "office_type": "government",\n   "lat": 55.7953175,\n   "lon": 37.5054026,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5338565863",\n   "kind": "organisation",\n   "name": "Гриф-Гарант",\n   "office_type": "security",\n   "lat": 55.7955956,\n   "lon": 37.5043757,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5338846125",\n   "kind": "organisation",\n   "name": "3 региональный отдел надзорной деятельности и профилактической работы Управления по САО Главного управления МЧС России по г. Москве",\n   "office_type": "government",\n   "lat": 55.7959189,\n   "lon": 37.5070829,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/5338846126",\n   "kind": "organisation",\n   "name": "Юридический центр Zeta",\n   "office_type": "lawyer",\n   "lat": 55.7958123,\n   "lon": 37.5071514,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/5338846127",\n   "kind": "organisation",\n   "name": "Адвокатский кабинет Заксона Ю.С.",\n   "office_type": "lawyer",\n   "lat": 55.7958427,\n   "lon": 37.507258,\n   "opening_hours": "Mo-Fr 11:00-18:00"\n  },\n  {\n   "id": "node/5338846128",\n   "kind": "organisation",\n   "name": "Бабашева Т.В.",\n   "office_type": "notary",\n   "lat": 55.7958527,\n   "lon": 37.5071236,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 11:00-15:00"\n  },\n  {\n   "id": "node/5338846129",\n   "kind": "organisation",\n   "name": "Оценочная компания Буданов В.С.",\n   "office_type": "company",\n   "lat": 55.7958788,\n   "lon": 37.5072232,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5338846130",\n   "kind": "organisation",\n   "name": "Оскар Авиа Груп",\n   "office_type": "company",\n   "lat": 55.7959065,\n   "lon": 37.5074544,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5338846131",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичная организацая №4 района Сокол",\n   "office_type": "ngo",\n   "lat": 55.796447,\n   "lon": 37.5038247,\n   "opening_hours": "Mo[1,3] 16:00-18:00; Th[1,3] 16:00-18:00"\n  },\n  {\n   "id": "node/5338846132",\n   "kind": "organisation",\n   "name": "«Жилищник района Сокол» ОДС № 3",\n   "office_type": "property_management",\n   "lat": 55.7970228,\n   "lon": 37.5052028,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5340495783",\n   "kind": "organisation",\n   "name": "ЖСК «Ракета»",\n   "office_type": "property_management",\n   "lat": 55.800847,\n   "lon": 37.5073095,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5341356344",\n   "kind": "organisation",\n   "name": "Управляющая компания Южуралзолото \\"ЮГК\\"",\n   "office_type": "company",\n   "lat": 55.7998093,\n   "lon": 37.5052458,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5342491421",\n   "kind": "organisation",\n   "name": "Магеллан",\n   "office_type": "ngo",\n   "lat": 55.7510849,\n   "lon": 37.7318916,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5347242623",\n   "kind": "organisation",\n   "name": "Приёмная ЛДПР",\n   "office_type": "political_party",\n   "lat": 55.7879723,\n   "lon": 37.6307756,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5349329279",\n   "kind": "organisation",\n   "name": "GERIT.RU",\n   "office_type": "company",\n   "lat": 55.7377905,\n   "lon": 37.628615,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5351383621",\n   "kind": "organisation",\n   "name": "Бентли Системс",\n   "office_type": "company",\n   "lat": 55.7222037,\n   "lon": 37.6391091,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5351454460",\n   "kind": "organisation",\n   "name": "ФМС Отделение по району Сокол",\n   "office_type": "government",\n   "lat": 55.8024332,\n   "lon": 37.5104339,\n   "opening_hours": "Mo 09:00-14:00,14:45-18:00; Tu 11:00-14:00,14:45-20:00; We 09:00-13:00; Th 11:00-14:00,14:45-20:00; Fr 09:00-14:00,14:45-16:45; Su 09:00-14:00,14:45-16:00"\n  },\n  {\n   "id": "node/5352012821",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт автоматизации управления в непромышленной сфере им. В. В. Соломатина",\n   "office_type": "company",\n   "lat": 55.6619451,\n   "lon": 37.6023952,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5352162621",\n   "kind": "organisation",\n   "name": "ГБУ «ЦЭИИС»",\n   "office_type": "government",\n   "lat": 55.7316743,\n   "lon": 37.7413601,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5352800144",\n   "kind": "organisation",\n   "name": "ОДС №18 ГБУ \\"Жилищник Мещанского района\\"",\n   "office_type": "property_management",\n   "lat": 55.778875,\n   "lon": 37.6381007,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5352800145",\n   "kind": "organisation",\n   "name": "ОДС №4 ГБУ \\"Жилищник Красносельского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7762819,\n   "lon": 37.6391843,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5352919348",\n   "kind": "organisation",\n   "name": "Факультут медицинского права",\n   "office_type": "lawyer",\n   "lat": 55.7994641,\n   "lon": 37.5050116,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5353164001",\n   "kind": "organisation",\n   "name": "Моушн Вью",\n   "office_type": "advertising_agency",\n   "lat": 55.8029657,\n   "lon": 37.5321653,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5356800502",\n   "kind": "organisation",\n   "name": "Люблинский центр услуг связи (МГТС)",\n   "office_type": "telecommunication",\n   "lat": 55.7040698,\n   "lon": 37.7401797,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5358365329",\n   "kind": "organisation",\n   "name": "МИЦ",\n   "office_type": "estate_agent",\n   "lat": 55.7320923,\n   "lon": 37.6477827,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5368356557",\n   "kind": "organisation",\n   "name": "Приёмная комиссия",\n   "office_type": "yes",\n   "lat": 55.801285,\n   "lon": 37.5269372,\n   "opening_hours": "Mo-Th 09:30-13:00,14:00-17:30; Fr 09:30-13:00,14:00-16:30"\n  },\n  {\n   "id": "node/5368356573",\n   "kind": "organisation",\n   "name": "Инспекция федеральной налоговой службы № 14 по г.Москве",\n   "office_type": "government",\n   "lat": 55.8007863,\n   "lon": 37.5189392,\n   "opening_hours": "Mo-Fr 09:00-13:00,13:45-18:00; Sa 09:00-13:00,13:45-16:45"\n  },\n  {\n   "id": "node/5370348265",\n   "kind": "organisation",\n   "name": "Аппарат совета депутатов муниципального округа Академический",\n   "office_type": "government",\n   "lat": 55.6869955,\n   "lon": 37.5664911,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5374361826",\n   "kind": "organisation",\n   "name": "Следственный комитет",\n   "office_type": "government",\n   "lat": 55.7437438,\n   "lon": 37.5509035,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5375745714",\n   "kind": "organisation",\n   "name": "Карпов Н.В. И Карпова Е.Н.",\n   "office_type": "notary",\n   "lat": 55.804599,\n   "lon": 37.5102438,\n   "opening_hours": "Mo-Th 09:00-13:00,14:00-18:00; Fr 09:00-13:00,14:00-17:00; Sa-Su off"\n  },\n  {\n   "id": "node/5378214849",\n   "kind": "organisation",\n   "name": "IceCro",\n   "office_type": "company",\n   "lat": 55.8361115,\n   "lon": 37.5509208,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5378271785",\n   "kind": "organisation",\n   "name": "ТСЖ «Дом 75 на Ленинградском проспекте»",\n   "office_type": "property_management",\n   "lat": 55.8046156,\n   "lon": 37.5104837,\n   "opening_hours": "Tu 10:00-13:00; We 19:30-21:00"\n  },\n  {\n   "id": "node/5382881842",\n   "kind": "organisation",\n   "name": "Центр социального обслуживания",\n   "office_type": "government",\n   "lat": 55.626183,\n   "lon": 37.5104248,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5382888339",\n   "kind": "organisation",\n   "name": "Нотариальная контора",\n   "office_type": "lawyer",\n   "lat": 55.7322843,\n   "lon": 37.663728,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5382888432",\n   "kind": "organisation",\n   "name": "Корона, визовый центр стран Азии",\n   "office_type": "visa",\n   "lat": 55.7377603,\n   "lon": 37.6285387,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5382888526",\n   "kind": "organisation",\n   "name": "Интернет-компания \\"Стрела\\"",\n   "office_type": "telecommunication",\n   "lat": 55.6401193,\n   "lon": 37.5132936,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5382890430",\n   "kind": "organisation",\n   "name": "ОДС №630 района Тёплый Стан",\n   "office_type": "government",\n   "lat": 55.6268598,\n   "lon": 37.5095264,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5382890434",\n   "kind": "organisation",\n   "name": "Стройматериалы",\n   "office_type": "company",\n   "lat": 55.6311749,\n   "lon": 37.5087198,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5382892023",\n   "kind": "organisation",\n   "name": "Инспекция ФНС № 28",\n   "office_type": "government",\n   "lat": 55.6594222,\n   "lon": 37.6057917,\n   "opening_hours": "Mo,We 09:00-18:00; Tu,Th 09:00-20:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/5393827944",\n   "kind": "organisation",\n   "name": "Аутентика",\n   "office_type": "company",\n   "lat": 55.7580839,\n   "lon": 37.6643449,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5395224871",\n   "kind": "organisation",\n   "name": "Давкар",\n   "office_type": "company",\n   "lat": 55.801982,\n   "lon": 37.5181397,\n   "opening_hours": "Mo-Fr 09:30-18:00"\n  },\n  {\n   "id": "node/5395224872",\n   "kind": "organisation",\n   "name": "Адрем принт",\n   "office_type": "advertising_agency",\n   "lat": 55.8025286,\n   "lon": 37.5182752,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5395224879",\n   "kind": "organisation",\n   "name": "ЖСК «Медик»",\n   "office_type": "property_management",\n   "lat": 55.8024333,\n   "lon": 37.5188702,\n   "opening_hours": "Mo 18:00-21:00"\n  },\n  {\n   "id": "node/5395464939",\n   "kind": "organisation",\n   "name": "We-Run",\n   "office_type": "company",\n   "lat": 55.8062046,\n   "lon": 37.6177745,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5396188995",\n   "kind": "organisation",\n   "name": "ОДС № 6 ГБУ \\"Жилищник Басманного района\\"",\n   "office_type": "property_management",\n   "lat": 55.7659342,\n   "lon": 37.6437239,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5403771421",\n   "kind": "organisation",\n   "name": "Stormin",\n   "office_type": "company",\n   "lat": 55.7211457,\n   "lon": 37.652325,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5403990623",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.6464799,\n   "lon": 37.7461375,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5406036121",\n   "kind": "organisation",\n   "name": "ОАО «ВНИПИнефть»",\n   "office_type": "company",\n   "lat": 55.7748005,\n   "lon": 37.687064,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5406836422",\n   "kind": "organisation",\n   "name": "Леви Штраусс Москва",\n   "office_type": "company",\n   "lat": 55.8220607,\n   "lon": 37.4991337,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5408324970",\n   "kind": "organisation",\n   "name": "Оптимторг",\n   "office_type": "logistics",\n   "lat": 55.5947954,\n   "lon": 37.7430229,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5408430499",\n   "kind": "organisation",\n   "name": "Центр \\"Амурский тигр\\"",\n   "office_type": "association",\n   "lat": 55.7669821,\n   "lon": 37.6419698,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5408431159",\n   "kind": "organisation",\n   "name": "Полномочное представительство Республики Бурятия при Президенте Российской Федерации",\n   "office_type": "government",\n   "lat": 55.768408,\n   "lon": 37.6428281,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5409660845",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8018443,\n   "lon": 37.5323648,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/5418200725",\n   "kind": "organisation",\n   "name": "Ясеневая 14 офис продаж",\n   "office_type": "estate_agent",\n   "lat": 55.6023586,\n   "lon": 37.728397,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5418889452",\n   "kind": "organisation",\n   "name": "Государственный научно-исследовательский институт системного анализа Счётной палаты Российской Федерации",\n   "office_type": "research",\n   "lat": 55.7241926,\n   "lon": 37.5612608,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5423546714",\n   "kind": "organisation",\n   "name": "Государственный Российский Дом народного творчества имени В. Д. Поленова",\n   "office_type": "government",\n   "lat": 55.7599439,\n   "lon": 37.6402786,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5423546753",\n   "kind": "organisation",\n   "name": "Секретариат организации договора о коллективной безопасности",\n   "office_type": "government",\n   "lat": 55.7604239,\n   "lon": 37.6392272,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5427999521",\n   "kind": "organisation",\n   "name": "Магазин проката",\n   "office_type": "company",\n   "lat": 55.8923406,\n   "lon": 37.7508298,\n   "opening_hours": "Mo-Su 07:00-23:00"\n  },\n  {\n   "id": "node/5429514021",\n   "kind": "organisation",\n   "name": "СЭС Дезслужба Москвы",\n   "office_type": "company",\n   "lat": 55.7405973,\n   "lon": 37.6155922,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5429605321",\n   "kind": "organisation",\n   "name": "СЭС Дезслужба Москвы",\n   "office_type": "company",\n   "lat": 55.7766829,\n   "lon": 37.645919,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5429613021",\n   "kind": "organisation",\n   "name": "СЭС Дезслужба Москвы",\n   "office_type": "company",\n   "lat": 55.786579,\n   "lon": 37.6078781,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5438585756",\n   "kind": "organisation",\n   "name": "Министерство РФ по делам Северного Кавказа",\n   "office_type": "government",\n   "lat": 55.7553331,\n   "lon": 37.5867793,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5440130221",\n   "kind": "organisation",\n   "name": "Нотариус Елин",\n   "office_type": "lawyer",\n   "lat": 55.6808499,\n   "lon": 37.5368401,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5440699308",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.6635396,\n   "lon": 37.6462868,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5443944670",\n   "kind": "organisation",\n   "name": "Постоянное представительство Республики Северная Осетия-Алания при Президенте РФ",\n   "office_type": "government",\n   "lat": 55.7550312,\n   "lon": 37.6488027,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5446107119",\n   "kind": "organisation",\n   "name": "КонЭксперт",\n   "office_type": "estate_agent",\n   "lat": 55.8033465,\n   "lon": 37.5343085,\n   "opening_hours": "Mo-Fr 11:00-19:00"\n  },\n  {\n   "id": "node/5446108025",\n   "kind": "organisation",\n   "name": "Галар",\n   "office_type": "publisher",\n   "lat": 55.803242,\n   "lon": 37.5342342,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5448418935",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "estate_agent",\n   "lat": 55.5875526,\n   "lon": 37.7340109,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5450631123",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6125201,\n   "lon": 37.7203133,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/5450658321",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.612461,\n   "lon": 37.7199708,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/5450805023",\n   "kind": "organisation",\n   "name": "Транспортная компания \\"Деловые линии\\"",\n   "office_type": "logistics",\n   "lat": 55.6451072,\n   "lon": 37.8297621,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5452118923",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичной организации № 6 района Сокол САО г. Москвы",\n   "office_type": "ngo",\n   "lat": 55.8041428,\n   "lon": 37.5132904,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5460737324",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7738632,\n   "lon": 37.6558463,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/5463090649",\n   "kind": "organisation",\n   "name": "Lingvo Art",\n   "office_type": "translator",\n   "lat": 55.7247824,\n   "lon": 37.562858,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/5463090651",\n   "kind": "organisation",\n   "name": "Лингво Арт",\n   "office_type": "notary",\n   "lat": 55.7248043,\n   "lon": 37.5628231,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5463090654",\n   "kind": "organisation",\n   "name": "Отдел лицензионно-разрешительной работы по ЦАО г. Москвы",\n   "office_type": "government",\n   "lat": 55.7248927,\n   "lon": 37.5630712,\n   "opening_hours": "Tu,Th 10:00-17:00, Sa[1,3] 10:00-15:00"\n  },\n  {\n   "id": "node/5463090655",\n   "kind": "organisation",\n   "name": "Отдел организации миграционного контроля",\n   "office_type": "government",\n   "lat": 55.7249131,\n   "lon": 37.5630538,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5464423445",\n   "kind": "organisation",\n   "name": "Инспекция ФНС России № 47 по г. Москве",\n   "office_type": "government",\n   "lat": 55.7778945,\n   "lon": 37.6016065,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5467471775",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7674512,\n   "lon": 37.7292492,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5468680723",\n   "kind": "organisation",\n   "name": "Федерация Окинава каратэ и кобудо России",\n   "office_type": "company",\n   "lat": 55.8524418,\n   "lon": 37.6457833,\n   "opening_hours": "Mo-Su 08:00-23:00"\n  },\n  {\n   "id": "node/5477972501",\n   "kind": "organisation",\n   "name": "Yakima Quality Hops Россия",\n   "office_type": "company",\n   "lat": 55.8682966,\n   "lon": 37.5823706,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5478104821",\n   "kind": "organisation",\n   "name": "СовКомФлот",\n   "office_type": "company",\n   "lat": 55.7686244,\n   "lon": 37.5901995,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5486099810",\n   "kind": "organisation",\n   "name": "Нотариус Сопина Татьяна Ивановна",\n   "office_type": "notary",\n   "lat": 55.5962107,\n   "lon": 37.599272,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5487426911",\n   "kind": "organisation",\n   "name": "Мосводосток ГУП Эгтр № 2",\n   "office_type": "company",\n   "lat": 55.8028013,\n   "lon": 37.5721546,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5489990798",\n   "kind": "organisation",\n   "name": "Фабрика обуви «Kapika»",\n   "office_type": "company",\n   "lat": 55.8787977,\n   "lon": 37.49297,\n   "opening_hours": "Mo-Fr 09:30-18:00"\n  },\n  {\n   "id": "node/5491621123",\n   "kind": "organisation",\n   "name": "Корона-сервис",\n   "office_type": "company",\n   "lat": 55.6057614,\n   "lon": 37.7474097,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5496984921",\n   "kind": "organisation",\n   "name": "Ресо-гарантия",\n   "office_type": "insurance",\n   "lat": 55.6078022,\n   "lon": 37.7550219,\n   "opening_hours": "Mo-Fr 09:30-13:00,14:00-20:00; Sa 10:00-13:00,14:00-18:00; Su 10:00-13:00,14:00-16:00"\n  },\n  {\n   "id": "node/5500202966",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.8698826,\n   "lon": 37.6300675,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5502200880",\n   "kind": "organisation",\n   "name": "Intellex",\n   "office_type": "it",\n   "lat": 55.7865069,\n   "lon": 37.6610282,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5502200882",\n   "kind": "organisation",\n   "name": "ООО ВегМаркет",\n   "office_type": "company",\n   "lat": 55.7864858,\n   "lon": 37.6611073,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5502200883",\n   "kind": "organisation",\n   "name": "ООО Верифилд",\n   "office_type": "surveyor",\n   "lat": 55.7865213,\n   "lon": 37.6611261,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5503255725",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6175582,\n   "lon": 37.7424411,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5506782937",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты населения Даниловского района",\n   "office_type": "government",\n   "lat": 55.7054065,\n   "lon": 37.6580831,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5506782938",\n   "kind": "organisation",\n   "name": "Управление социальной защиты населения Южного административного округа",\n   "office_type": "government",\n   "lat": 55.7053105,\n   "lon": 37.6583058,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5507944279",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7238595,\n   "lon": 37.7685472,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5511246142",\n   "kind": "organisation",\n   "name": "NAYADA",\n   "office_type": "company",\n   "lat": 55.6086008,\n   "lon": 37.7825698,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5513091821",\n   "kind": "organisation",\n   "name": "Управа Дмитровского района",\n   "office_type": "government",\n   "lat": 55.8875697,\n   "lon": 37.5219027,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5513091822",\n   "kind": "organisation",\n   "name": "Управа района Западное Дегунино",\n   "office_type": "government",\n   "lat": 55.8630108,\n   "lon": 37.5405446,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5513091823",\n   "kind": "organisation",\n   "name": "Управа Останкинского района",\n   "office_type": "government",\n   "lat": 55.8008785,\n   "lon": 37.6339906,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5513091824",\n   "kind": "organisation",\n   "name": "Управа района Ростокино",\n   "office_type": "government",\n   "lat": 55.8367452,\n   "lon": 37.6679313,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5513091825",\n   "kind": "organisation",\n   "name": "Управа Нижегородского района",\n   "office_type": "government",\n   "lat": 55.7340916,\n   "lon": 37.7054146,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5513091826",\n   "kind": "organisation",\n   "name": "Управа района Лефортово",\n   "office_type": "government",\n   "lat": 55.7587056,\n   "lon": 37.6962515,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5513091827",\n   "kind": "organisation",\n   "name": "Управа района Зябликово",\n   "office_type": "government",\n   "lat": 55.6195882,\n   "lon": 37.755107,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5513091828",\n   "kind": "organisation",\n   "name": "Управа района Чертаново Центральное",\n   "office_type": "government",\n   "lat": 55.6158458,\n   "lon": 37.5827107,\n   "opening_hours": "Mo-Th 08:00-13:00,13:45-17:00; Fr 08:00-13:00,13:45-15:45; Sa,Su off"\n  },\n  {\n   "id": "node/5514969927",\n   "kind": "organisation",\n   "name": "Европлан",\n   "office_type": "company",\n   "lat": 55.7281414,\n   "lon": 37.6201293,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5515177370",\n   "kind": "organisation",\n   "name": "Нечаева М. И.",\n   "office_type": "notary",\n   "lat": 55.7677969,\n   "lon": 37.6246145,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5520175235",\n   "kind": "organisation",\n   "name": "Управление Росреестра по г. Москве",\n   "office_type": "government",\n   "lat": 55.7089322,\n   "lon": 37.6235765,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5520175236",\n   "kind": "organisation",\n   "name": "Инспекция ФНС России №10 по г. Москве",\n   "office_type": "government",\n   "lat": 55.7085302,\n   "lon": 37.6236328,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5523038842",\n   "kind": "organisation",\n   "name": "Посольство Гватемалы",\n   "office_type": "diplomatic",\n   "lat": 55.7286874,\n   "lon": 37.6168882,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/5523060842",\n   "kind": "organisation",\n   "name": "Посольство Катара",\n   "office_type": "diplomatic",\n   "lat": 55.7286852,\n   "lon": 37.6157286,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5523060851",\n   "kind": "organisation",\n   "name": "Посольство Парагвая",\n   "office_type": "diplomatic",\n   "lat": 55.7286766,\n   "lon": 37.6160948,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/5523082538",\n   "kind": "organisation",\n   "name": "Посольство Экваториальной Гвинеи",\n   "office_type": "diplomatic",\n   "lat": 55.7324492,\n   "lon": 37.6218841,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5523082897",\n   "kind": "organisation",\n   "name": "Посольство Мальты",\n   "office_type": "diplomatic",\n   "lat": 55.7286995,\n   "lon": 37.616519,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5523096737",\n   "kind": "organisation",\n   "name": "Посольство Черногории",\n   "office_type": "diplomatic",\n   "lat": 55.7276799,\n   "lon": 37.6156924,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5524528348",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8941715,\n   "lon": 37.4498165,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/5524594022",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7156306,\n   "lon": 37.508278,\n   "opening_hours": "Mo-Su 08:00-22:00"\n  },\n  {\n   "id": "node/5525466214",\n   "kind": "organisation",\n   "name": "Жилищник Даниловского района, ЮАО",\n   "office_type": "property_management",\n   "lat": 55.7172184,\n   "lon": 37.6181522,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:00-15:45"\n  },\n  {\n   "id": "node/5525470786",\n   "kind": "organisation",\n   "name": "Инженерная служба Даниловского района",\n   "office_type": "property_management",\n   "lat": 55.7173692,\n   "lon": 37.6181523,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5525515214",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.7169213,\n   "lon": 37.6181384,\n   "opening_hours": "Mo,We 11:00-13:00"\n  },\n  {\n   "id": "node/5534731524",\n   "kind": "organisation",\n   "name": "Центр информационных технологий «Умный город»",\n   "office_type": "company",\n   "lat": 55.8339867,\n   "lon": 37.6308977,\n   "opening_hours": "Tu-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5538543791",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7977728,\n   "lon": 37.5581595,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/5541234491",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник района Лефортово\\"",\n   "office_type": "property_management",\n   "lat": 55.7487941,\n   "lon": 37.715005,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5542022422",\n   "kind": "organisation",\n   "name": "Видовая комиссия автомототуризма",\n   "office_type": "ngo",\n   "lat": 55.743702,\n   "lon": 37.6614461,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5542022428",\n   "kind": "organisation",\n   "name": "Федерация сортивного туризма Москвы",\n   "office_type": "ngo",\n   "lat": 55.7439105,\n   "lon": 37.6614951,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5543690921",\n   "kind": "organisation",\n   "name": "Общественная приёмная ЛДПР",\n   "office_type": "political_party",\n   "lat": 55.7718899,\n   "lon": 37.6605416,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5548442522",\n   "kind": "organisation",\n   "name": "Департамент Дистанционного Образования ИМЦ (Институт Мировых Цивилизаций)",\n   "office_type": "company",\n   "lat": 55.7718815,\n   "lon": 37.6605727,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5550914039",\n   "kind": "organisation",\n   "name": "Российский научный фонд",\n   "office_type": "yes",\n   "lat": 55.7521542,\n   "lon": 37.6398378,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5551583491",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник района Хамовники\\"",\n   "office_type": "property_management",\n   "lat": 55.7226253,\n   "lon": 37.5717777,\n   "opening_hours": "Mo-Th 08:00-17:00, Fr 08:00-15:45; 12:00-12:45 off \\"Обед\\""\n  },\n  {\n   "id": "node/5552241034",\n   "kind": "organisation",\n   "name": "АО \\"Фирма континент\\"",\n   "office_type": "yes",\n   "lat": 55.8060695,\n   "lon": 37.516674,\n   "opening_hours": "Mo-Fr 11:00-16:00"\n  },\n  {\n   "id": "node/5552749746",\n   "kind": "organisation",\n   "name": "АО \\"Бюро комплексного проектирования\\" Трансинжстрой",\n   "office_type": "yes",\n   "lat": 55.7511496,\n   "lon": 37.6447546,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5566445678",\n   "kind": "organisation",\n   "name": "Профессиональный союз работников культуры",\n   "office_type": "association",\n   "lat": 55.7669932,\n   "lon": 37.5726289,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5568401321",\n   "kind": "organisation",\n   "name": "ФССП",\n   "office_type": "government",\n   "lat": 55.6734609,\n   "lon": 37.6143151,\n   "opening_hours": "Tu,Th 09:00-10:45,11:00-13:00,13:45-15:45,16:00-18:00"\n  },\n  {\n   "id": "node/5575972221",\n   "kind": "organisation",\n   "name": "РЭУ-53",\n   "office_type": "government",\n   "lat": 55.7958545,\n   "lon": 37.6982819,\n   "opening_hours": "Mo-Tu, Th 08:00-12:30, 13:30-17:00; Fr 08:00-12:30, 13:30-15:45"\n  },\n  {\n   "id": "node/5576599722",\n   "kind": "organisation",\n   "name": "Управление медицинского обеспечения ДТ МВД России",\n   "office_type": "government",\n   "lat": 55.7949415,\n   "lon": 37.483654,\n   "opening_hours": "Mo-Th 09:00-13:00, 13:45-18:00; Fr 09:00-13:00, 13:45-16:45"\n  },\n  {\n   "id": "node/5576599723",\n   "kind": "organisation",\n   "name": "ФКУЗ «ЦМСЧ МВД России»",\n   "office_type": "government",\n   "lat": 55.7947554,\n   "lon": 37.4836654,\n   "opening_hours": "Mo-Th 09:00-13:00, 13:45-18:00; Fr 09:00-13:00, 13:45-16:45"\n  },\n  {\n   "id": "node/5576608601",\n   "kind": "organisation",\n   "name": "Департамент по материально-техническому и медицинскому обеспечению МВД России",\n   "office_type": "government",\n   "lat": 55.7306515,\n   "lon": 37.6183803,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5580323728",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8274346,\n   "lon": 37.4437505,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5580650221",\n   "kind": "organisation",\n   "name": "М.видео",\n   "office_type": "company",\n   "lat": 55.7740592,\n   "lon": 37.6717827,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/5590307377",\n   "kind": "organisation",\n   "name": "Районная диспетчерская служба № 1",\n   "office_type": "property_management",\n   "lat": 55.8185044,\n   "lon": 37.5239173,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5590307378",\n   "kind": "organisation",\n   "name": "Совет ветеранов № 9",\n   "office_type": "ngo",\n   "lat": 55.8184562,\n   "lon": 37.52386,\n   "opening_hours": "Tu,Th 11:00-13:00"\n  },\n  {\n   "id": "node/5590307384",\n   "kind": "organisation",\n   "name": "ООО «ОВСВ «Защита»",\n   "office_type": "yes",\n   "lat": 55.8177464,\n   "lon": 37.5234676,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5590307385",\n   "kind": "organisation",\n   "name": "ООО «Центр безопасности»",\n   "office_type": "yes",\n   "lat": 55.8177327,\n   "lon": 37.5235165,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5590307386",\n   "kind": "organisation",\n   "name": "ООО «Охрана»",\n   "office_type": "yes",\n   "lat": 55.8177187,\n   "lon": 37.5235741,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5590307387",\n   "kind": "organisation",\n   "name": "ООО «Стандарт безопасности»",\n   "office_type": "yes",\n   "lat": 55.8177251,\n   "lon": 37.5234416,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5590307388",\n   "kind": "organisation",\n   "name": "ООО «Оскар-КСБ»",\n   "office_type": "yes",\n   "lat": 55.8177095,\n   "lon": 37.5235001,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5590307389",\n   "kind": "organisation",\n   "name": "ООО «Евротехцентр»",\n   "office_type": "yes",\n   "lat": 55.8176971,\n   "lon": 37.5235568,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5590307390",\n   "kind": "organisation",\n   "name": "ООО «Системы безопасности XXI век»",\n   "office_type": "yes",\n   "lat": 55.8177046,\n   "lon": 37.5236307,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5593111721",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8509884,\n   "lon": 37.5967946,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/5596475712",\n   "kind": "organisation",\n   "name": "Первый Визовый Центр",\n   "office_type": "insurance",\n   "lat": 55.7376692,\n   "lon": 37.5051821,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/5598600688",\n   "kind": "organisation",\n   "name": "Гледен Инвест",\n   "office_type": "company",\n   "lat": 55.7353018,\n   "lon": 37.5851177,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5598600689",\n   "kind": "organisation",\n   "name": "Росспиртпром",\n   "office_type": "company",\n   "lat": 55.7423392,\n   "lon": 37.5293188,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5600977922",\n   "kind": "organisation",\n   "name": "ООО \\"МИТМИТМИТ\\"",\n   "office_type": "company",\n   "lat": 55.7440304,\n   "lon": 37.7547333,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5604093257",\n   "kind": "organisation",\n   "name": "ООО \\"Лонсдейлит\\"",\n   "office_type": "company",\n   "lat": 55.796506,\n   "lon": 37.7817468,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5604857121",\n   "kind": "organisation",\n   "name": "Гипроречтранс",\n   "office_type": "company",\n   "lat": 55.7890262,\n   "lon": 37.7421939,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5605935348",\n   "kind": "organisation",\n   "name": "Центр независимых испытаний и эксертизы в строительстве",\n   "office_type": "association",\n   "lat": 55.8007058,\n   "lon": 37.5242101,\n   "opening_hours": "Mo-Th 09:00-18:00, Fr 09:00-15:00; Sa-Su off"\n  },\n  {\n   "id": "node/5605935349",\n   "kind": "organisation",\n   "name": "Инженерная служба Хорошевского района",\n   "office_type": "property_management",\n   "lat": 55.8007096,\n   "lon": 37.524257,\n   "opening_hours": "Mo-Th 08:00-17:00, Fr 08:00-15:45; 12:00-12:45 off \\"Перерыв\\"; Sa-Su off"\n  },\n  {\n   "id": "node/5605935350",\n   "kind": "organisation",\n   "name": "Можайская земля",\n   "office_type": "company",\n   "lat": 55.8006802,\n   "lon": 37.5245628,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5605935351",\n   "kind": "organisation",\n   "name": "ВТБ",\n   "office_type": "insurance",\n   "lat": 55.8006749,\n   "lon": 37.5246352,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5605935352",\n   "kind": "organisation",\n   "name": "Креатив-Маркет",\n   "office_type": "event_agency",\n   "lat": 55.8005543,\n   "lon": 37.5247881,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5605935358",\n   "kind": "organisation",\n   "name": "Китони",\n   "office_type": "newspaper",\n   "lat": 55.8005852,\n   "lon": 37.5243281,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5605935360",\n   "kind": "organisation",\n   "name": "Слово",\n   "office_type": "newspaper",\n   "lat": 55.800583,\n   "lon": 37.5243576,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5606937267",\n   "kind": "organisation",\n   "name": "ООО \\"РФЛ-Логистик\\"",\n   "office_type": "yes",\n   "lat": 55.8193965,\n   "lon": 37.5267474,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5606937268",\n   "kind": "organisation",\n   "name": "ООО \\"Руссия Мессервис\\"",\n   "office_type": "yes",\n   "lat": 55.8193868,\n   "lon": 37.5267821,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5606937269",\n   "kind": "organisation",\n   "name": "ООО \\"ВИА Сервис\\"",\n   "office_type": "yes",\n   "lat": 55.8193758,\n   "lon": 37.5268156,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5606937270",\n   "kind": "organisation",\n   "name": "ООО \\"Руссия логистик\\"",\n   "office_type": "yes",\n   "lat": 55.8193643,\n   "lon": 37.5268503,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5607227812",\n   "kind": "organisation",\n   "name": "ГУП САО \\"Центр развития жилищного строительства\\" (отдел коммерческого консалтинга)",\n   "office_type": "yes",\n   "lat": 55.817846,\n   "lon": 37.5244631,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5607339660",\n   "kind": "organisation",\n   "name": "РЕСО-МЕД",\n   "office_type": "insurance",\n   "lat": 55.8085875,\n   "lon": 37.5304987,\n   "opening_hours": "Mo-Su 11:00-20:00"\n  },\n  {\n   "id": "node/5609989086",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.5870148,\n   "lon": 37.7249238,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5609989090",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.5843847,\n   "lon": 37.7236631,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5610264933",\n   "kind": "organisation",\n   "name": "ФБК",\n   "office_type": "company",\n   "lat": 55.7692401,\n   "lon": 37.6461487,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5610265140",\n   "kind": "organisation",\n   "name": "Постоянное представительство Республики Саха (Якутия) при Президенте РФ",\n   "office_type": "government",\n   "lat": 55.7687022,\n   "lon": 37.6467414,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5617146026",\n   "kind": "organisation",\n   "name": "Климатдисконт",\n   "office_type": "company",\n   "lat": 55.8822398,\n   "lon": 37.4864225,\n   "opening_hours": "Mo-Su 09:30-18:00"\n  },\n  {\n   "id": "node/5617161767",\n   "kind": "organisation",\n   "name": "Cherbrooke",\n   "office_type": "company",\n   "lat": 55.8820773,\n   "lon": 37.4866666,\n   "opening_hours": "Mo-Fr 09:30-18:00"\n  },\n  {\n   "id": "node/5619874694",\n   "kind": "organisation",\n   "name": "Стройкомплектсервис",\n   "office_type": "company",\n   "lat": 55.6943983,\n   "lon": 37.5686382,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5620060952",\n   "kind": "organisation",\n   "name": "ЖСК Фианит",\n   "office_type": "property_management",\n   "lat": 55.8814873,\n   "lon": 37.4899782,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5621239023",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник Выхино района Выхино-Жулебино",\n   "office_type": "property_management",\n   "lat": 55.6937101,\n   "lon": 37.820628,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5621686360",\n   "kind": "organisation",\n   "name": "Федерация независимых профсоюзов России",\n   "office_type": "ngo",\n   "lat": 55.7031471,\n   "lon": 37.567623,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5621705180",\n   "kind": "organisation",\n   "name": "ФГУП \\"ЖКУ РАН\\"",\n   "office_type": "property_management",\n   "lat": 55.6947278,\n   "lon": 37.5680521,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5621908125",\n   "kind": "organisation",\n   "name": "YRD Center",\n   "office_type": "company",\n   "lat": 55.6969843,\n   "lon": 37.5637442,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5621909919",\n   "kind": "organisation",\n   "name": "Гипротрубопровод",\n   "office_type": "research",\n   "lat": 55.7020895,\n   "lon": 37.5775207,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5622050518",\n   "kind": "organisation",\n   "name": "Московское аэрогеодезическое предприятие",\n   "office_type": "geodesist",\n   "lat": 55.7138439,\n   "lon": 37.7214176,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5622505640",\n   "kind": "organisation",\n   "name": "ГИПРОНИИ РАН",\n   "office_type": "research",\n   "lat": 55.6955975,\n   "lon": 37.5653951,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5622507400",\n   "kind": "organisation",\n   "name": "Секция прикладных проблем при президиуме РАН",\n   "office_type": "research",\n   "lat": 55.6954499,\n   "lon": 37.5656167,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5622509974",\n   "kind": "organisation",\n   "name": "ИВП РАН",\n   "office_type": "research",\n   "lat": 55.6956619,\n   "lon": 37.5653132,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5622509975",\n   "kind": "organisation",\n   "name": "ИПНГ РАН",\n   "office_type": "research",\n   "lat": 55.6955501,\n   "lon": 37.5654826,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5622697150",\n   "kind": "organisation",\n   "name": "Ассоциация российских фармацевтических производителей",\n   "office_type": "association",\n   "lat": 55.7619067,\n   "lon": 37.6339651,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5622717686",\n   "kind": "organisation",\n   "name": "Ассоциация юристов России",\n   "office_type": "ngo",\n   "lat": 55.7621333,\n   "lon": 37.6354952,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5624028379",\n   "kind": "organisation",\n   "name": "Объединённый центр исследований и разработок Роснефть",\n   "office_type": "research",\n   "lat": 55.6967534,\n   "lon": 37.5632858,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5625566606",\n   "kind": "organisation",\n   "name": "ГУП Мосгортранс Южный филиал",\n   "office_type": "company",\n   "lat": 55.6882084,\n   "lon": 37.6639391,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5626780996",\n   "kind": "organisation",\n   "name": "Совет ветеранов #7",\n   "office_type": "ngo",\n   "lat": 55.6946546,\n   "lon": 37.5692673,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5628637941",\n   "kind": "organisation",\n   "name": "ООО \\"ГиперСталь\\"",\n   "office_type": "company",\n   "lat": 55.7355979,\n   "lon": 37.7150935,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5628655074",\n   "kind": "organisation",\n   "name": "Red Taxi",\n   "office_type": "company",\n   "lat": 55.8461044,\n   "lon": 37.4753329,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5630823205",\n   "kind": "organisation",\n   "name": "Жилищник",\n   "office_type": "property_management",\n   "lat": 55.758062,\n   "lon": 37.5519991,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5630830946",\n   "kind": "organisation",\n   "name": "ОДС 18 Жилищник Басманного района",\n   "office_type": "property_management",\n   "lat": 55.7767688,\n   "lon": 37.7052646,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5630830947",\n   "kind": "organisation",\n   "name": "Издательский дом Питер",\n   "office_type": "publisher",\n   "lat": 55.7772228,\n   "lon": 37.7045779,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5630830949",\n   "kind": "organisation",\n   "name": "Национальный расчётный депозитарий",\n   "office_type": "financial",\n   "lat": 55.7720846,\n   "lon": 37.6746167,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5630832106",\n   "kind": "organisation",\n   "name": "Первая грузовая компания",\n   "office_type": "company",\n   "lat": 55.7649453,\n   "lon": 37.6593429,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5630832538",\n   "kind": "organisation",\n   "name": "НПЦ Перспектива",\n   "office_type": "company",\n   "lat": 55.7773378,\n   "lon": 37.7047942,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5630832539",\n   "kind": "organisation",\n   "name": "Национальный клиринговый центр",\n   "office_type": "financial",\n   "lat": 55.7722109,\n   "lon": 37.6749478,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5631002122",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7406741,\n   "lon": 37.5969598,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5631131046",\n   "kind": "organisation",\n   "name": "Вокруг света",\n   "office_type": "publisher",\n   "lat": 55.8322495,\n   "lon": 37.4510349,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5636144781",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты населения Южнопортового района",\n   "office_type": "government",\n   "lat": 55.7111142,\n   "lon": 37.6682791,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5636823923",\n   "kind": "organisation",\n   "name": "Мегал",\n   "office_type": "company",\n   "lat": 55.7228099,\n   "lon": 37.6268517,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5639806766",\n   "kind": "organisation",\n   "name": "Информатика и управление",\n   "office_type": "research",\n   "lat": 55.6939249,\n   "lon": 37.5629372,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5640663124",\n   "kind": "organisation",\n   "name": "Викивиза",\n   "office_type": "company",\n   "lat": 55.7536049,\n   "lon": 37.6629495,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/5641718579",\n   "kind": "organisation",\n   "name": "Холдинг Славянский Деловой Мир",\n   "office_type": "company",\n   "lat": 55.696031,\n   "lon": 37.5754899,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5643856965",\n   "kind": "organisation",\n   "name": "Спортмастер",\n   "office_type": "company",\n   "lat": 55.8074529,\n   "lon": 37.5449625,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5645581608",\n   "kind": "organisation",\n   "name": "ООО «Гллосс Лейбл»",\n   "office_type": "company",\n   "lat": 55.6815579,\n   "lon": 37.652716,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5645762405",\n   "kind": "organisation",\n   "name": "Обьединение инжинеров",\n   "office_type": "company",\n   "lat": 55.7841209,\n   "lon": 37.7025962,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5645767478",\n   "kind": "organisation",\n   "name": "Тепло-арт",\n   "office_type": "company",\n   "lat": 55.7840877,\n   "lon": 37.70268,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5645767480",\n   "kind": "organisation",\n   "name": "Центр реабилитации депо Москва-Сортировочная",\n   "office_type": "company",\n   "lat": 55.7608311,\n   "lon": 37.7307604,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5647613843",\n   "kind": "organisation",\n   "name": "Институт системного анализа РАН",\n   "office_type": "research",\n   "lat": 55.698757,\n   "lon": 37.5791652,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5647648694",\n   "kind": "organisation",\n   "name": "Международный НИИ проблем управления",\n   "office_type": "research",\n   "lat": 55.6988681,\n   "lon": 37.579202,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5648924222",\n   "kind": "organisation",\n   "name": "Технопромимпорт",\n   "office_type": "company",\n   "lat": 55.7754052,\n   "lon": 37.5495856,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5649397505",\n   "kind": "organisation",\n   "name": "представительство АО \\"Мотор Сич\\"",\n   "office_type": "company",\n   "lat": 55.7965548,\n   "lon": 37.5118968,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5649397514",\n   "kind": "organisation",\n   "name": "ЗАО \\"Двигатели «Владимир Климов-Мотор Сич»\\"",\n   "office_type": "company",\n   "lat": 55.7965791,\n   "lon": 37.5122705,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5649398223",\n   "kind": "organisation",\n   "name": "Первая Строительная компания «Атлант»",\n   "office_type": "company",\n   "lat": 55.7979513,\n   "lon": 37.5146857,\n   "opening_hours": "Mo-Th 09:30-18:00; Fr 09:30-17:30"\n  },\n  {\n   "id": "node/5649965021",\n   "kind": "organisation",\n   "name": "РЕСО гарантия",\n   "office_type": "insurance",\n   "lat": 55.7046239,\n   "lon": 37.6543024,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5653954289",\n   "kind": "organisation",\n   "name": "КПК \\"Эталон Сбережений\\"",\n   "office_type": "company",\n   "lat": 55.7862355,\n   "lon": 37.6741523,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5654516255",\n   "kind": "organisation",\n   "name": "MDM техно",\n   "office_type": "company",\n   "lat": 55.8159054,\n   "lon": 37.5286017,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/5655544225",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7503551,\n   "lon": 37.7846751,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5656991003",\n   "kind": "organisation",\n   "name": "Транскор",\n   "office_type": "company",\n   "lat": 55.7467588,\n   "lon": 37.7743786,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5656991004",\n   "kind": "organisation",\n   "name": "Творческие коллективные мастерские МОСХ России",\n   "office_type": "company",\n   "lat": 55.8149303,\n   "lon": 37.6418261,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5656992329",\n   "kind": "organisation",\n   "name": "Экономос Экопроект Маркел Груп",\n   "office_type": "company",\n   "lat": 55.7467633,\n   "lon": 37.7744675,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5656992721",\n   "kind": "organisation",\n   "name": "Газинжинирингавтоматизация",\n   "office_type": "company",\n   "lat": 55.7468042,\n   "lon": 37.7745346,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5661000926",\n   "kind": "organisation",\n   "name": "Nailico.ru",\n   "office_type": "company",\n   "lat": 55.7583553,\n   "lon": 37.7610238,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5663124624",\n   "kind": "organisation",\n   "name": "Cummins",\n   "office_type": "company",\n   "lat": 55.6553344,\n   "lon": 37.4976898,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5671111817",\n   "kind": "organisation",\n   "name": "1С",\n   "office_type": "company",\n   "lat": 55.779863,\n   "lon": 37.6104176,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5673674922",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник района Ясенево",\n   "office_type": "property_management",\n   "lat": 55.6136248,\n   "lon": 37.5419692,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5675750021",\n   "kind": "organisation",\n   "name": "Солар Секьюрити",\n   "office_type": "company",\n   "lat": 55.7982896,\n   "lon": 37.5800919,\n   "opening_hours": "Mo-Fr 10:00-18:30"\n  },\n  {\n   "id": "node/5676621643",\n   "kind": "organisation",\n   "name": "Союз ветеранов Группы войск в Германии",\n   "office_type": "ngo",\n   "lat": 55.7911582,\n   "lon": 37.5921853,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5676621644",\n   "kind": "organisation",\n   "name": "Norita",\n   "office_type": "online_shop",\n   "lat": 55.7911838,\n   "lon": 37.5921377,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5686220345",\n   "kind": "organisation",\n   "name": "Автотранспортный комбинат, ФГБУ",\n   "office_type": "government",\n   "lat": 55.7648702,\n   "lon": 37.542179,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5686977621",\n   "kind": "organisation",\n   "name": "Укулеле.ру",\n   "office_type": "company",\n   "lat": 55.7459314,\n   "lon": 37.6598714,\n   "opening_hours": "Tu-Fr 12:00-22:00; Sa-Su 10:00-19:00"\n  },\n  {\n   "id": "node/5691266021",\n   "kind": "organisation",\n   "name": "Юридический центр",\n   "office_type": "lawyer",\n   "lat": 55.8262957,\n   "lon": 37.5831017,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/5693096277",\n   "kind": "organisation",\n   "name": "Level Travel",\n   "office_type": "it",\n   "lat": 55.7791265,\n   "lon": 37.6929908,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5693181921",\n   "kind": "organisation",\n   "name": "АльфаСтрахование",\n   "office_type": "insurance",\n   "lat": 55.7106198,\n   "lon": 37.7516989,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5693227737",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7105234,\n   "lon": 37.7515697,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5694089626",\n   "kind": "organisation",\n   "name": "МГТС",\n   "office_type": "telecommunication",\n   "lat": 55.6201269,\n   "lon": 37.7202398,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/5698890480",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.6397359,\n   "lon": 37.6065096,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5701395621",\n   "kind": "organisation",\n   "name": "ООО \\"Атон\\"",\n   "office_type": "company",\n   "lat": 55.7440646,\n   "lon": 37.6349568,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5706095021",\n   "kind": "organisation",\n   "name": "Отдел по вопросам легализации и апостиля",\n   "office_type": "government",\n   "lat": 55.7397894,\n   "lon": 37.583937,\n   "opening_hours": "Mo-Th 10:00-13:00,14:00-17:00; Fr 10:00-13:00,14:00-16:00"\n  },\n  {\n   "id": "node/5706095221",\n   "kind": "organisation",\n   "name": "Институт профессиональных бухгалтеров России",\n   "office_type": "ngo",\n   "lat": 55.7676025,\n   "lon": 37.6019896,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5715235510",\n   "kind": "organisation",\n   "name": "Комитет солдатских матерей России",\n   "office_type": "ngo",\n   "lat": 55.7800733,\n   "lon": 37.7204585,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5715689421",\n   "kind": "organisation",\n   "name": "Бутырский межрайонный следственный отдел",\n   "office_type": "government",\n   "lat": 55.829887,\n   "lon": 37.5760193,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5715689422",\n   "kind": "organisation",\n   "name": "Бутырская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.8299053,\n   "lon": 37.5750538,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5723112662",\n   "kind": "organisation",\n   "name": "Интеллектуальные социальные системы",\n   "office_type": "it",\n   "lat": 55.7702194,\n   "lon": 37.6644296,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5723793121",\n   "kind": "organisation",\n   "name": "РГС-Жизнь",\n   "office_type": "insurance",\n   "lat": 55.7451648,\n   "lon": 37.6225787,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5727172721",\n   "kind": "organisation",\n   "name": "ПТЛ Инжиниринг",\n   "office_type": "company",\n   "lat": 55.7678493,\n   "lon": 37.6741994,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/5728928547",\n   "kind": "organisation",\n   "name": "Центр независимой диагностики",\n   "office_type": "educational_institution",\n   "lat": 55.6316558,\n   "lon": 37.6516485,\n   "opening_hours": "Mo-Sa 09:00-17:30"\n  },\n  {\n   "id": "node/5742875982",\n   "kind": "organisation",\n   "name": "Элитаудит",\n   "office_type": "company",\n   "lat": 55.7742916,\n   "lon": 37.7108276,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5742900868",\n   "kind": "organisation",\n   "name": "УК Трек",\n   "office_type": "property_management",\n   "lat": 55.7746251,\n   "lon": 37.7106933,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5743934170",\n   "kind": "organisation",\n   "name": "Конечная станция Нагатино",\n   "office_type": "company",\n   "lat": 55.6862484,\n   "lon": 37.691402,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5749899138",\n   "kind": "organisation",\n   "name": "Центральное таможенное управление",\n   "office_type": "government",\n   "lat": 55.775718,\n   "lon": 37.6541239,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5749996321",\n   "kind": "organisation",\n   "name": "Фонд имени И.М. Смоктуновского «Золотой Пеликан»",\n   "office_type": "ngo",\n   "lat": 55.8203197,\n   "lon": 37.6040269,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5750163401",\n   "kind": "organisation",\n   "name": "Ресо",\n   "office_type": "insurance",\n   "lat": 55.8023092,\n   "lon": 37.5336821,\n   "opening_hours": "Mo-Fr 09:30-13:00; 14:00-20:00"\n  },\n  {\n   "id": "node/5751062986",\n   "kind": "organisation",\n   "name": "Служба «одного окна» по жилищным вопросам Департамента имущества Москвы",\n   "office_type": "government",\n   "lat": 55.7345808,\n   "lon": 37.6352241,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/5751605867",\n   "kind": "organisation",\n   "name": "Северный округ",\n   "office_type": "estate_agent",\n   "lat": 55.8110798,\n   "lon": 37.5726856,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/5755585468",\n   "kind": "organisation",\n   "name": "ОАО \\"ГИПРОНИИАВИАПРОМ\\"",\n   "office_type": "research",\n   "lat": 55.7955608,\n   "lon": 37.5618725,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5756068373",\n   "kind": "organisation",\n   "name": "Представительство Администрации Кемеровской области при Правительстве РФ",\n   "office_type": "government",\n   "lat": 55.7427627,\n   "lon": 37.6313496,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5756636335",\n   "kind": "organisation",\n   "name": "Медикал-групп",\n   "office_type": "company",\n   "lat": 55.6835785,\n   "lon": 37.6604805,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5758595084",\n   "kind": "organisation",\n   "name": "Борлас",\n   "office_type": "it",\n   "lat": 55.701029,\n   "lon": 37.6255482,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5760402567",\n   "kind": "organisation",\n   "name": "Главное управление ПФР № 10 по г. Москве и Московской области",\n   "office_type": "government",\n   "lat": 55.7327119,\n   "lon": 37.6262426,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5766219398",\n   "kind": "organisation",\n   "name": "Группа компаний \\"Ренова\\"",\n   "office_type": "company",\n   "lat": 55.7328939,\n   "lon": 37.6252824,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5767489953",\n   "kind": "organisation",\n   "name": "Коллегия налоговых консультантов",\n   "office_type": "lawyer",\n   "lat": 55.7648975,\n   "lon": 37.6149639,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5767733060",\n   "kind": "organisation",\n   "name": "Toyway",\n   "office_type": "company",\n   "lat": 55.725625,\n   "lon": 37.7796778,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5768533046",\n   "kind": "organisation",\n   "name": "Северная окружная организация Всероссийского общества инвалидов",\n   "office_type": "association",\n   "lat": 55.8470226,\n   "lon": 37.5723785,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5771727439",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7681636,\n   "lon": 37.6018211,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5772737549",\n   "kind": "organisation",\n   "name": "ОДС Жилищник Пресненского района",\n   "office_type": "property_management",\n   "lat": 55.758892,\n   "lon": 37.5921947,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5772756372",\n   "kind": "organisation",\n   "name": "Резиденция посла Республики Корея",\n   "office_type": "diplomatic",\n   "lat": 55.7601205,\n   "lon": 37.5934821,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5773685153",\n   "kind": "organisation",\n   "name": "Строй Инженер Проект",\n   "office_type": "company",\n   "lat": 55.8377241,\n   "lon": 37.6356809,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/5779263907",\n   "kind": "organisation",\n   "name": "ОДС Жилищник Бескудниковского района",\n   "office_type": "property_management",\n   "lat": 55.8686459,\n   "lon": 37.5555503,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5783903087",\n   "kind": "organisation",\n   "name": "Криотек",\n   "office_type": "company",\n   "lat": 55.6960913,\n   "lon": 37.5671831,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5785109792",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.8183591,\n   "lon": 37.7824125,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5787683501",\n   "kind": "organisation",\n   "name": "Центральная научно-исследовательская лаборатория психофизиологии и психологии труда ФСБ России",\n   "office_type": "government",\n   "lat": 55.7610045,\n   "lon": 37.6311927,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5793439255",\n   "kind": "organisation",\n   "name": "Капитал Инвест",\n   "office_type": "estate_agent",\n   "lat": 55.7707972,\n   "lon": 37.630219,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5794762500",\n   "kind": "organisation",\n   "name": "ООО «Гипсокартонные технологии»",\n   "office_type": "company",\n   "lat": 55.8820092,\n   "lon": 37.5475252,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5794762548",\n   "kind": "organisation",\n   "name": "ООО \\"7К-Развитие\\"",\n   "office_type": "company",\n   "lat": 55.897153,\n   "lon": 37.5423217,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5795158145",\n   "kind": "organisation",\n   "name": "Эваком",\n   "office_type": "company",\n   "lat": 55.8736313,\n   "lon": 37.5532578,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5796971428",\n   "kind": "organisation",\n   "name": "ДРТ",\n   "office_type": "company",\n   "lat": 55.7782218,\n   "lon": 37.5869563,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5796971429",\n   "kind": "organisation",\n   "name": "Яков и партнеры",\n   "office_type": "consulting",\n   "lat": 55.7778115,\n   "lon": 37.5867042,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5796971430",\n   "kind": "organisation",\n   "name": "Технологии доверия",\n   "office_type": "company",\n   "lat": 55.7784104,\n   "lon": 37.5858271,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5798322380",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник района Кузьминки",\n   "office_type": "property_management",\n   "lat": 55.7008802,\n   "lon": 37.7842323,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/5798448254",\n   "kind": "organisation",\n   "name": "Russian Cycle Touring Club",\n   "office_type": "association",\n   "lat": 55.7438868,\n   "lon": 37.6614838,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5801801397",\n   "kind": "organisation",\n   "name": "Artox media",\n   "office_type": "advertising_agency",\n   "lat": 55.7113298,\n   "lon": 37.6532706,\n   "opening_hours": "Fr 10:00-18:00; Mo-Th 10:00-19:00"\n  },\n  {\n   "id": "node/5804107581",\n   "kind": "organisation",\n   "name": "Московская биржа",\n   "office_type": "company",\n   "lat": 55.7528366,\n   "lon": 37.6095378,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5804107582",\n   "kind": "organisation",\n   "name": "АФК «Система»",\n   "office_type": "company",\n   "lat": 55.756278,\n   "lon": 37.6133868,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5804924768",\n   "kind": "organisation",\n   "name": "Научно-производственное предприятие Волна",\n   "office_type": "research",\n   "lat": 55.6852752,\n   "lon": 37.6217555,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5805012474",\n   "kind": "organisation",\n   "name": "Российское историческое общество",\n   "office_type": "association",\n   "lat": 55.7545668,\n   "lon": 37.6544353,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5805016453",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7690699,\n   "lon": 37.6700353,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5810154856",\n   "kind": "organisation",\n   "name": "Посольство Индии",\n   "office_type": "diplomatic",\n   "lat": 55.7528217,\n   "lon": 37.649914,\n   "opening_hours": "Mo-Fr 09:30-18:00"\n  },\n  {\n   "id": "node/5812410627",\n   "kind": "organisation",\n   "name": "Администрация Музея Победы",\n   "office_type": "yes",\n   "lat": 55.7295328,\n   "lon": 37.5107253,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5814464448",\n   "kind": "organisation",\n   "name": "Информационно-аналитический центр в сфере здравоохранения г. Москвы",\n   "office_type": "government",\n   "lat": 55.7522539,\n   "lon": 37.6630747,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5821410818",\n   "kind": "organisation",\n   "name": "Кравец С. И.",\n   "office_type": "notary",\n   "lat": 55.7395882,\n   "lon": 37.5246097,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5821527353",\n   "kind": "organisation",\n   "name": "Первый канал",\n   "office_type": "company",\n   "lat": 55.8206645,\n   "lon": 37.6043822,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/5823745053",\n   "kind": "organisation",\n   "name": "Инженерная служба района Южнопортовый",\n   "office_type": "property_management",\n   "lat": 55.7175248,\n   "lon": 37.6754299,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5827447933",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.8081847,\n   "lon": 37.5326537,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5827562285",\n   "kind": "organisation",\n   "name": "Yota",\n   "office_type": "telecommunication",\n   "lat": 55.6777539,\n   "lon": 37.466838,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/5829969962",\n   "kind": "organisation",\n   "name": "Westline",\n   "office_type": "company",\n   "lat": 55.8682928,\n   "lon": 37.5836264,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5831623245",\n   "kind": "organisation",\n   "name": "Ресурсконтроль",\n   "office_type": "company",\n   "lat": 55.634151,\n   "lon": 37.441026,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/5833221382",\n   "kind": "organisation",\n   "name": "Группа компаний \\"Юни\\"",\n   "office_type": "company",\n   "lat": 55.6558911,\n   "lon": 37.5521412,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5833349428",\n   "kind": "organisation",\n   "name": "ОAО \\"Мосгипробум\\"",\n   "office_type": "company",\n   "lat": 55.8158897,\n   "lon": 37.5274969,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5835909578",\n   "kind": "organisation",\n   "name": "Юридическое партнёрство Лидер",\n   "office_type": "lawyer",\n   "lat": 55.7852649,\n   "lon": 37.5949653,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5837199487",\n   "kind": "organisation",\n   "name": "Научно-исследовательский клинический институт оториноларингологии им. Л. И. Свержевского",\n   "office_type": "research",\n   "lat": 55.6909412,\n   "lon": 37.5328765,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:00"\n  },\n  {\n   "id": "node/5837982052",\n   "kind": "organisation",\n   "name": "Группа компаний «Щит»",\n   "office_type": "company",\n   "lat": 55.7532306,\n   "lon": 37.7233964,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5839453101",\n   "kind": "organisation",\n   "name": "LR Health&B",\n   "office_type": "company",\n   "lat": 55.7198502,\n   "lon": 37.6280423,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5842011580",\n   "kind": "organisation",\n   "name": "Гарантпост",\n   "office_type": "company",\n   "lat": 55.6983886,\n   "lon": 37.6225524,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5846276439",\n   "kind": "organisation",\n   "name": "Молодёжный центр инноваций Богоявленского кафедрального собора и компании \\"Бекхофф Автоматизация\\"",\n   "office_type": "yes",\n   "lat": 55.7650054,\n   "lon": 37.6607855,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5846276440",\n   "kind": "organisation",\n   "name": "Представительство компании Beckhoff",\n   "office_type": "company",\n   "lat": 55.7650552,\n   "lon": 37.6607238,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5846276441",\n   "kind": "organisation",\n   "name": "Центр молодёжного инновационного творчества \\"Цифровая земля\\"",\n   "office_type": "yes",\n   "lat": 55.7649194,\n   "lon": 37.6629165,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5848914077",\n   "kind": "organisation",\n   "name": "Мособлгеотрест",\n   "office_type": "company",\n   "lat": 55.7964263,\n   "lon": 37.5867605,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5849918302",\n   "kind": "organisation",\n   "name": "Крона-Маркет",\n   "office_type": "estate_agent",\n   "lat": 55.7424109,\n   "lon": 37.7869031,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5852227942",\n   "kind": "organisation",\n   "name": "KFC",\n   "office_type": "company",\n   "lat": 55.8239799,\n   "lon": 37.5042558,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5852227956",\n   "kind": "organisation",\n   "name": "DMG Mori Rus",\n   "office_type": "company",\n   "lat": 55.8234722,\n   "lon": 37.5020751,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5852227957",\n   "kind": "organisation",\n   "name": "Strabag",\n   "office_type": "company",\n   "lat": 55.8235596,\n   "lon": 37.5024855,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5853756205",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.8721267,\n   "lon": 37.5979722,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5859574885",\n   "kind": "organisation",\n   "name": "Балтика-Транс Логистика",\n   "office_type": "company",\n   "lat": 55.7147935,\n   "lon": 37.6774974,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/5862932772",\n   "kind": "organisation",\n   "name": "Новая газета",\n   "office_type": "newspaper",\n   "lat": 55.7622076,\n   "lon": 37.6401418,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5862993078",\n   "kind": "organisation",\n   "name": "ГКУ \\"Центр реформы в ЖКХ\\"",\n   "office_type": "government",\n   "lat": 55.7695798,\n   "lon": 37.6610852,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5862993079",\n   "kind": "organisation",\n   "name": "ГБУ \\"Городская реклама и информация\\"",\n   "office_type": "government",\n   "lat": 55.7695232,\n   "lon": 37.6610879,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5863435089",\n   "kind": "organisation",\n   "name": "Адвокатская контора С. А. Романовского",\n   "office_type": "lawyer",\n   "lat": 55.8089568,\n   "lon": 37.4922422,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5865274258",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7574511,\n   "lon": 37.63361,\n   "opening_hours": "Mo-Su 08:00-21:30"\n  },\n  {\n   "id": "node/5867528029",\n   "kind": "organisation",\n   "name": "ООО «Аккаунт»",\n   "office_type": "lawyer",\n   "lat": 55.761766,\n   "lon": 37.6581736,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5878419648",\n   "kind": "organisation",\n   "name": "Спорт-экспресс",\n   "office_type": "newspaper",\n   "lat": 55.7196931,\n   "lon": 37.6306254,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5879603646",\n   "kind": "organisation",\n   "name": "Федеральное казначейство",\n   "office_type": "government",\n   "lat": 55.7587502,\n   "lon": 37.6292521,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5882916189",\n   "kind": "organisation",\n   "name": "Фотостудия Крафт",\n   "office_type": "company",\n   "lat": 55.7885152,\n   "lon": 37.7059755,\n   "opening_hours": "Mo-Su 07:00-23:00"\n  },\n  {\n   "id": "node/5885170014",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.707155,\n   "lon": 37.7305122,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5890514897",\n   "kind": "organisation",\n   "name": "ИнтармИТ",\n   "office_type": "it",\n   "lat": 55.8064603,\n   "lon": 37.5052983,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5890514898",\n   "kind": "organisation",\n   "name": "Эталон-ИКТ",\n   "office_type": "it",\n   "lat": 55.8065523,\n   "lon": 37.504684,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5891302686",\n   "kind": "organisation",\n   "name": "Artec3D",\n   "office_type": "company",\n   "lat": 55.7437748,\n   "lon": 37.5911309,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5895662421",\n   "kind": "organisation",\n   "name": "ООО «МВО Перспектива»",\n   "office_type": "company",\n   "lat": 55.7128855,\n   "lon": 37.7226127,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5901927452",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.839288,\n   "lon": 37.5277251,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5903732501",\n   "kind": "organisation",\n   "name": "ООО «КАМОС ЛАЙН»",\n   "office_type": "company",\n   "lat": 55.7616369,\n   "lon": 37.7124391,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5904931343",\n   "kind": "organisation",\n   "name": "Постоянное представительство Республики Дагестан при Президенте РФ",\n   "office_type": "government",\n   "lat": 55.7598594,\n   "lon": 37.650345,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5904932613",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7578295,\n   "lon": 37.6331721,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5905765267",\n   "kind": "organisation",\n   "name": "Артгаз",\n   "office_type": "company",\n   "lat": 55.760273,\n   "lon": 37.7646371,\n   "opening_hours": "Mo-Th 08:30-17:30; Fr 08:30-16:30"\n  },\n  {\n   "id": "node/5912429376",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7932968,\n   "lon": 37.5753142,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/5914023116",\n   "kind": "organisation",\n   "name": "Df group",\n   "office_type": "company",\n   "lat": 55.743968,\n   "lon": 37.561625,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/5914077167",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Ясенево",\n   "office_type": "government",\n   "lat": 55.6193795,\n   "lon": 37.5097322,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/5914645519",\n   "kind": "organisation",\n   "name": "Федеральное агентство по делам молодежи",\n   "office_type": "government",\n   "lat": 55.7583812,\n   "lon": 37.6507433,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5915398885",\n   "kind": "organisation",\n   "name": "НаукаСофт",\n   "office_type": "company",\n   "lat": 55.8066818,\n   "lon": 37.6256139,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5915764088",\n   "kind": "organisation",\n   "name": "ООО \\"ПромСнабРезерв\\"",\n   "office_type": "company",\n   "lat": 55.8030079,\n   "lon": 37.5843816,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5915990096",\n   "kind": "organisation",\n   "name": "SFP media production",\n   "office_type": "company",\n   "lat": 55.7766492,\n   "lon": 37.6997073,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5918034186",\n   "kind": "organisation",\n   "name": "Федеральная палата адвокатов РФ",\n   "office_type": "lawyer",\n   "lat": 55.7464384,\n   "lon": 37.5868538,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5918319670",\n   "kind": "organisation",\n   "name": "ООО «Метакоммерс»",\n   "office_type": "company",\n   "lat": 55.7142528,\n   "lon": 37.6552082,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5919961841",\n   "kind": "organisation",\n   "name": "Интернет-магазин Fissman",\n   "office_type": "company",\n   "lat": 55.7663342,\n   "lon": 37.5264672,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5922136712",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7509727,\n   "lon": 37.8182216,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5922548471",\n   "kind": "organisation",\n   "name": "AGenYZ",\n   "office_type": "company",\n   "lat": 55.709676,\n   "lon": 37.4426914,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/5923035837",\n   "kind": "organisation",\n   "name": "Вагонная ремонтная компания-1",\n   "office_type": "company",\n   "lat": 55.7613807,\n   "lon": 37.6469627,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5924222491",\n   "kind": "organisation",\n   "name": "ПФР Тимирязевский",\n   "office_type": "government",\n   "lat": 55.8083503,\n   "lon": 37.5662845,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5926296695",\n   "kind": "organisation",\n   "name": "activday.ru Товары для активной жизни",\n   "office_type": "company",\n   "lat": 55.8593486,\n   "lon": 37.5853991,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5927590351",\n   "kind": "organisation",\n   "name": "Бюро переводов TLS",\n   "office_type": "lawyer",\n   "lat": 55.7590025,\n   "lon": 37.6522743,\n   "opening_hours": "Mo-Th 09:30-18:30; Fr 09:30-18:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/5931296889",\n   "kind": "organisation",\n   "name": "Московский Юрист",\n   "office_type": "lawyer",\n   "lat": 55.8176347,\n   "lon": 37.656001,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5931296890",\n   "kind": "organisation",\n   "name": "Московский Юрист",\n   "office_type": "lawyer",\n   "lat": 55.7094147,\n   "lon": 37.6546584,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5931299885",\n   "kind": "organisation",\n   "name": "Московский Юрист",\n   "office_type": "lawyer",\n   "lat": 55.7785496,\n   "lon": 37.5878021,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5931343785",\n   "kind": "organisation",\n   "name": "Московский Юрист",\n   "office_type": "lawyer",\n   "lat": 55.7594191,\n   "lon": 37.6661928,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5931383385",\n   "kind": "organisation",\n   "name": "Московский Юрист",\n   "office_type": "lawyer",\n   "lat": 55.7534695,\n   "lon": 37.6257547,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5931383485",\n   "kind": "organisation",\n   "name": "Московский Юрист",\n   "office_type": "lawyer",\n   "lat": 55.7573718,\n   "lon": 37.6166799,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5932638883",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7600684,\n   "lon": 37.6543924,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5938631885",\n   "kind": "organisation",\n   "name": "Планета Закона",\n   "office_type": "lawyer",\n   "lat": 55.770077,\n   "lon": 37.6326576,\n   "opening_hours": "Mo-Su 08:00-23:00"\n  },\n  {\n   "id": "node/5939110622",\n   "kind": "organisation",\n   "name": "Совет Ветеранов",\n   "office_type": "ngo",\n   "lat": 55.7665436,\n   "lon": 37.4886349,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5939766820",\n   "kind": "organisation",\n   "name": "ТранЭкспресс",\n   "office_type": "translator",\n   "lat": 55.7830305,\n   "lon": 37.5729565,\n   "opening_hours": "Mo-Fr 09:30-18:30"\n  },\n  {\n   "id": "node/5939766821",\n   "kind": "organisation",\n   "name": "Гринга",\n   "office_type": "company",\n   "lat": 55.7829551,\n   "lon": 37.572864,\n   "opening_hours": "Mo-Th 10:00-19:00; Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5939766822",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7829362,\n   "lon": 37.5728211,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/5940452186",\n   "kind": "organisation",\n   "name": "ООО «АРД ЛЕКС»",\n   "office_type": "lawyer",\n   "lat": 55.7471473,\n   "lon": 37.5834304,\n   "opening_hours": "Mo-Sa 10:00-19:00"\n  },\n  {\n   "id": "node/5942476974",\n   "kind": "organisation",\n   "name": "Сбербанк Лизинг",\n   "office_type": "company",\n   "lat": 55.7271675,\n   "lon": 37.5350864,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5944064886",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник района Текстильщики\\"",\n   "office_type": "property_management",\n   "lat": 55.6981894,\n   "lon": 37.7470243,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5944858580",\n   "kind": "organisation",\n   "name": "Департамент развития новых территорий города Москвы",\n   "office_type": "government",\n   "lat": 55.7558916,\n   "lon": 37.6568399,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5951389952",\n   "kind": "organisation",\n   "name": "ООО «Насосы Ампика»",\n   "office_type": "company",\n   "lat": 55.7229997,\n   "lon": 37.7841973,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5951450682",\n   "kind": "organisation",\n   "name": "Sweet Home",\n   "office_type": "property_management",\n   "lat": 55.6718257,\n   "lon": 37.5405233,\n   "opening_hours": "09:00-18:00"\n  },\n  {\n   "id": "node/5954737746",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7819995,\n   "lon": 37.556831,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5956382601",\n   "kind": "organisation",\n   "name": "Центр внешнеполитического сотрудничества имени Е.М. Примакова",\n   "office_type": "ngo",\n   "lat": 55.7587245,\n   "lon": 37.6553419,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5960712887",\n   "kind": "organisation",\n   "name": "АО КГ \\"Постпроцессор\\"",\n   "office_type": "company",\n   "lat": 55.7715873,\n   "lon": 37.568052,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/5961332787",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7883433,\n   "lon": 37.7438831,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5963365086",\n   "kind": "organisation",\n   "name": "ООО Арема",\n   "office_type": "company",\n   "lat": 55.8121057,\n   "lon": 37.6363949,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/5963449992",\n   "kind": "organisation",\n   "name": "Консульство Мальты",\n   "office_type": "diplomatic",\n   "lat": 55.7213222,\n   "lon": 37.6055311,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5963524101",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6696674,\n   "lon": 37.5512116,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5967509205",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7496908,\n   "lon": 37.5872527,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5971645625",\n   "kind": "organisation",\n   "name": "Solid IT",\n   "office_type": "it",\n   "lat": 55.7588656,\n   "lon": 37.5883242,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5974196190",\n   "kind": "organisation",\n   "name": "Dicis",\n   "office_type": "company",\n   "lat": 55.7781612,\n   "lon": 37.6998504,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5978203386",\n   "kind": "organisation",\n   "name": "Poster POS",\n   "office_type": "company",\n   "lat": 55.7765565,\n   "lon": 37.676098,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/5979303253",\n   "kind": "organisation",\n   "name": "Полюс",\n   "office_type": "company",\n   "lat": 55.766377,\n   "lon": 37.5890887,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5982587847",\n   "kind": "organisation",\n   "name": "Центр строительной экспертизы",\n   "office_type": "architect",\n   "lat": 55.7180973,\n   "lon": 37.6239759,\n   "opening_hours": "Mo-Fr 09:30-18:00"\n  },\n  {\n   "id": "node/5982745507",\n   "kind": "organisation",\n   "name": "Сити Фабрика",\n   "office_type": "company",\n   "lat": 55.7169826,\n   "lon": 37.6964349,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5982854054",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.857074,\n   "lon": 37.5591901,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/5984018974",\n   "kind": "organisation",\n   "name": "Мослифт",\n   "office_type": "company",\n   "lat": 55.7832779,\n   "lon": 37.5729337,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5985281485",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.644164,\n   "lon": 37.4734334,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5985673785",\n   "kind": "organisation",\n   "name": "DesignChat",\n   "office_type": "company",\n   "lat": 55.7666616,\n   "lon": 37.6489781,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/5986154985",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7420614,\n   "lon": 37.5356756,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/5989203548",\n   "kind": "organisation",\n   "name": "Агуччи",\n   "office_type": "company",\n   "lat": 55.849578,\n   "lon": 37.5963337,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5992467298",\n   "kind": "organisation",\n   "name": "ЮрбериПроф",\n   "office_type": "lawyer",\n   "lat": 55.7757569,\n   "lon": 37.5533092,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5992996647",\n   "kind": "organisation",\n   "name": "Группа компаний ТрансСпецНаз",\n   "office_type": "logistics",\n   "lat": 55.8336525,\n   "lon": 37.4549713,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5994115692",\n   "kind": "organisation",\n   "name": "Союз архитекторов России",\n   "office_type": "ngo",\n   "lat": 55.7600111,\n   "lon": 37.5882572,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5996509062",\n   "kind": "organisation",\n   "name": "Уолт Дисней Студиос Сони Пикчерс Релизинг",\n   "office_type": "company",\n   "lat": 55.745512,\n   "lon": 37.6629272,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5997561294",\n   "kind": "organisation",\n   "name": "ПСИ",\n   "office_type": "company",\n   "lat": 55.7310761,\n   "lon": 37.5610625,\n   "opening_hours": null\n  },\n  {\n   "id": "node/5998578185",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7399224,\n   "lon": 37.660006,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 11:00-18:00; Su 11:00-16:00"\n  },\n  {\n   "id": "node/6007533113",\n   "kind": "organisation",\n   "name": "CreditControl",\n   "office_type": "financial",\n   "lat": 55.708365,\n   "lon": 37.6521403,\n   "opening_hours": "09:00-21:00"\n  },\n  {\n   "id": "node/6009754293",\n   "kind": "organisation",\n   "name": "ООО \\"Межрегиональный центр охраны труда \\"ЭКСПЕРТИЗА\\"",\n   "office_type": "company",\n   "lat": 55.7613769,\n   "lon": 37.5816231,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6010685695",\n   "kind": "organisation",\n   "name": "ГлавБэкери",\n   "office_type": "company",\n   "lat": 55.7847098,\n   "lon": 37.6738952,\n   "opening_hours": "Mo-Su 08:00-22:00"\n  },\n  {\n   "id": "node/6011799924",\n   "kind": "organisation",\n   "name": "ВентРесурс",\n   "office_type": "company",\n   "lat": 55.8144969,\n   "lon": 37.7724042,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6014994642",\n   "kind": "organisation",\n   "name": "Российский общественно-политический центр",\n   "office_type": "government",\n   "lat": 55.7581711,\n   "lon": 37.6312389,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6015673186",\n   "kind": "organisation",\n   "name": "МХ",\n   "office_type": "security",\n   "lat": 55.7826015,\n   "lon": 37.5986164,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6017492809",\n   "kind": "organisation",\n   "name": "ООО Независимый консалтинговый центр \\"Эталонъ\\"",\n   "office_type": "company",\n   "lat": 55.7338929,\n   "lon": 37.6644802,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6023226147",\n   "kind": "organisation",\n   "name": "Церих Кэпитал Менеджмент",\n   "office_type": "company",\n   "lat": 55.7427816,\n   "lon": 37.5987486,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6023226687",\n   "kind": "organisation",\n   "name": "Агрохимэкспорт",\n   "office_type": "company",\n   "lat": 55.7469515,\n   "lon": 37.6019284,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6024715390",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7917625,\n   "lon": 37.4565634,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/6027117159",\n   "kind": "organisation",\n   "name": "Право дольщика",\n   "office_type": "lawyer",\n   "lat": 55.6463612,\n   "lon": 37.4817501,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6031414278",\n   "kind": "organisation",\n   "name": "СептикиТопас.ру",\n   "office_type": "company",\n   "lat": 55.6760155,\n   "lon": 37.6249807,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6035100884",\n   "kind": "organisation",\n   "name": "Emirates",\n   "office_type": "company",\n   "lat": 55.7726836,\n   "lon": 37.6305017,\n   "opening_hours": "Mo-Fr 09:30-19:00"\n  },\n  {\n   "id": "node/6037644485",\n   "kind": "organisation",\n   "name": "ОКТА Медиа",\n   "office_type": "advertising_agency",\n   "lat": 55.7613763,\n   "lon": 37.5515838,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/6039177662",\n   "kind": "organisation",\n   "name": "Central Yard II",\n   "office_type": "company",\n   "lat": 55.7766465,\n   "lon": 37.6769159,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6041254126",\n   "kind": "organisation",\n   "name": "ООО \\"Техно-Сервис\\"",\n   "office_type": "company",\n   "lat": 55.7205896,\n   "lon": 37.7745949,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6041808009",\n   "kind": "organisation",\n   "name": "Единство-Персонал",\n   "office_type": "employment_agency",\n   "lat": 55.7648213,\n   "lon": 37.6541818,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6042607379",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7385682,\n   "lon": 37.6589964,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6042611290",\n   "kind": "organisation",\n   "name": "МТТ",\n   "office_type": "telecommunication",\n   "lat": 55.7380578,\n   "lon": 37.6602101,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6042611291",\n   "kind": "organisation",\n   "name": "ПАО \\"Мосэнергосбыт\\"",\n   "office_type": "company",\n   "lat": 55.7376289,\n   "lon": 37.6609451,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6042611514",\n   "kind": "organisation",\n   "name": "Федеральная служба по экологическому, технологическому и атомному надзору",\n   "office_type": "government",\n   "lat": 55.7394107,\n   "lon": 37.6657502,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6043635819",\n   "kind": "organisation",\n   "name": "ООО «Мототека»",\n   "office_type": "company",\n   "lat": 55.747283,\n   "lon": 37.5393445,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6045685968",\n   "kind": "organisation",\n   "name": "Горсерв",\n   "office_type": "company",\n   "lat": 55.7183163,\n   "lon": 37.7770232,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6045808005",\n   "kind": "organisation",\n   "name": "Водоканалсбыт",\n   "office_type": "company",\n   "lat": 55.7107993,\n   "lon": 37.8005212,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6047018186",\n   "kind": "organisation",\n   "name": "ПЕНТАдизайн-М",\n   "office_type": "company",\n   "lat": 55.6600506,\n   "lon": 37.7464426,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/6047018389",\n   "kind": "organisation",\n   "name": "Миэль",\n   "office_type": "estate_agent",\n   "lat": 55.6603732,\n   "lon": 37.7478792,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6048401364",\n   "kind": "organisation",\n   "name": "ООО \\"Мобильное Измерение\\"",\n   "office_type": "company",\n   "lat": 55.7769513,\n   "lon": 37.676385,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6048902011",\n   "kind": "organisation",\n   "name": "Ясность",\n   "office_type": "company",\n   "lat": 55.775761,\n   "lon": 37.5385015,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6049643814",\n   "kind": "organisation",\n   "name": "ВСК",\n   "office_type": "insurance",\n   "lat": 55.6643689,\n   "lon": 37.7542866,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/6049646988",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "notary",\n   "lat": 55.664911,\n   "lon": 37.7541602,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/6049688813",\n   "kind": "organisation",\n   "name": "Хатрако",\n   "office_type": "company",\n   "lat": 55.6663696,\n   "lon": 37.7566121,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6049762489",\n   "kind": "organisation",\n   "name": "Home net telecom",\n   "office_type": "telecommunication",\n   "lat": 55.6683628,\n   "lon": 37.7577882,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/6050068446",\n   "kind": "organisation",\n   "name": "Регионмонтаж",\n   "office_type": "company",\n   "lat": 55.6661326,\n   "lon": 37.7548666,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6051024754",\n   "kind": "organisation",\n   "name": "RosCase",\n   "office_type": "company",\n   "lat": 55.7563904,\n   "lon": 37.6328862,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa,Su 11:00-17:00"\n  },\n  {\n   "id": "node/6051438186",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7873034,\n   "lon": 37.4789436,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6051438187",\n   "kind": "organisation",\n   "name": "ЗАО Р-ФАРМ",\n   "office_type": "company",\n   "lat": 55.7875812,\n   "lon": 37.4764278,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6052790486",\n   "kind": "organisation",\n   "name": "ОДС № 11 ГБУ \\"Жилищник района Кузьминки\\"",\n   "office_type": "property_management",\n   "lat": 55.6957702,\n   "lon": 37.7613715,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6052989073",\n   "kind": "organisation",\n   "name": "Красная Звезда",\n   "office_type": "company",\n   "lat": 55.6943869,\n   "lon": 37.7569748,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6054425438",\n   "kind": "organisation",\n   "name": "Посольство Абхазии",\n   "office_type": "diplomatic",\n   "lat": 55.7446749,\n   "lon": 37.5970333,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6054468615",\n   "kind": "organisation",\n   "name": "SteelRex",\n   "office_type": "company",\n   "lat": 55.8506332,\n   "lon": 37.589626,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6054998265",\n   "kind": "organisation",\n   "name": "Посольство Мадагаскара",\n   "office_type": "diplomatic",\n   "lat": 55.7421552,\n   "lon": 37.6060979,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6054998266",\n   "kind": "organisation",\n   "name": "Посольство Камбоджи",\n   "office_type": "diplomatic",\n   "lat": 55.7460963,\n   "lon": 37.59553,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6055327085",\n   "kind": "organisation",\n   "name": "НПК Негатоскоп.Ру",\n   "office_type": "company",\n   "lat": 55.7909644,\n   "lon": 37.5773798,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6058115352",\n   "kind": "organisation",\n   "name": "Управление Федеральной службы государственной регистрации, кадастра и картографии по Московской области",\n   "office_type": "government",\n   "lat": 55.733387,\n   "lon": 37.5218028,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6058582781",\n   "kind": "organisation",\n   "name": "ПЭКО",\n   "office_type": "lawyer",\n   "lat": 55.7155474,\n   "lon": 37.4767417,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6059959035",\n   "kind": "organisation",\n   "name": "Mondigo",\n   "office_type": "yes",\n   "lat": 55.7085603,\n   "lon": 37.6534046,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-15:00; Su off"\n  },\n  {\n   "id": "node/6061295312",\n   "kind": "organisation",\n   "name": "Мобильные ТелеСистемы",\n   "office_type": "telecommunication",\n   "lat": 55.740205,\n   "lon": 37.6557013,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6062754455",\n   "kind": "organisation",\n   "name": "Кенгуру.Про",\n   "office_type": "company",\n   "lat": 55.7251218,\n   "lon": 37.5622548,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6064246214",\n   "kind": "organisation",\n   "name": "Мобильные ТелеСистемы",\n   "office_type": "telecommunication",\n   "lat": 55.7393216,\n   "lon": 37.6574394,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6065468887",\n   "kind": "organisation",\n   "name": "Star Talk",\n   "office_type": "company",\n   "lat": 55.7476334,\n   "lon": 37.6735835,\n   "opening_hours": "Mo-Sa 10:30-19:30"\n  },\n  {\n   "id": "node/6066175744",\n   "kind": "organisation",\n   "name": "Федеральное агентство по образованию Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7202385,\n   "lon": 37.6234558,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6066175749",\n   "kind": "organisation",\n   "name": "Vira Realtime",\n   "office_type": "it",\n   "lat": 55.8163402,\n   "lon": 37.8332072,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6068375686",\n   "kind": "organisation",\n   "name": "Посольство Федеративной Республики Германия Резиденция",\n   "office_type": "diplomatic",\n   "lat": 55.756994,\n   "lon": 37.58814,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6070264805",\n   "kind": "organisation",\n   "name": "Управление образования",\n   "office_type": "government",\n   "lat": 55.7017477,\n   "lon": 37.7756744,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6070474550",\n   "kind": "organisation",\n   "name": "Городской психолого-педагогический центр",\n   "office_type": "government",\n   "lat": 55.7016632,\n   "lon": 37.7756489,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6070474558",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты населения района Кузьминки",\n   "office_type": "government",\n   "lat": 55.7032249,\n   "lon": 37.772924,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6073883028",\n   "kind": "organisation",\n   "name": "Сервисторг",\n   "office_type": "company",\n   "lat": 55.7624558,\n   "lon": 37.7275288,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:00; Sa 09:00-16:00"\n  },\n  {\n   "id": "node/6074161486",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7499801,\n   "lon": 37.8008678,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6074667098",\n   "kind": "organisation",\n   "name": "Побратим",\n   "office_type": "newspaper",\n   "lat": 55.7446922,\n   "lon": 37.5134544,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6074667099",\n   "kind": "organisation",\n   "name": "Инвалиды войны",\n   "office_type": "ngo",\n   "lat": 55.7447677,\n   "lon": 37.5133632,\n   "opening_hours": "Tu-Th 10:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/6074667100",\n   "kind": "organisation",\n   "name": "Столица",\n   "office_type": "ngo",\n   "lat": 55.7448432,\n   "lon": 37.5131741,\n   "opening_hours": "Tu-Th 10:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/6075668596",\n   "kind": "organisation",\n   "name": "Метал-лайн",\n   "office_type": "company",\n   "lat": 55.7656698,\n   "lon": 37.7265669,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6076122186",\n   "kind": "organisation",\n   "name": "Волкова Юлия Станиславовна",\n   "office_type": "notary",\n   "lat": 55.7994002,\n   "lon": 37.7123158,\n   "opening_hours": "Mo-Th 11:00-13:30,14:15-20:00; Fr 11:00-13:30,14:15-17:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/6079974486",\n   "kind": "organisation",\n   "name": "Arena Space",\n   "office_type": "company",\n   "lat": 55.6561286,\n   "lon": 37.5411123,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/6080606352",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.6127147,\n   "lon": 37.7200475,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6081732967",\n   "kind": "organisation",\n   "name": "АНО МСЭБ",\n   "office_type": "company",\n   "lat": 55.6907264,\n   "lon": 37.6269296,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6081776147",\n   "kind": "organisation",\n   "name": "ОДС № 2 ГБУ \\"Жилищник Басманного района\\"",\n   "office_type": "property_management",\n   "lat": 55.7553377,\n   "lon": 37.6409921,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6082544527",\n   "kind": "organisation",\n   "name": "HouseLaw",\n   "office_type": "lawyer",\n   "lat": 55.7819275,\n   "lon": 37.6970813,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6083103686",\n   "kind": "organisation",\n   "name": "Диаметраль",\n   "office_type": "lawyer",\n   "lat": 55.749783,\n   "lon": 37.5378886,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6086142585",\n   "kind": "organisation",\n   "name": "Корбайт",\n   "office_type": "company",\n   "lat": 55.7212281,\n   "lon": 37.6520598,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6088093627",\n   "kind": "organisation",\n   "name": "RusVolvo",\n   "office_type": "company",\n   "lat": 55.7202309,\n   "lon": 37.7163863,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6095584334",\n   "kind": "organisation",\n   "name": "ООО \\"АПРИОРИ ГРУПП\\"",\n   "office_type": "company",\n   "lat": 55.692406,\n   "lon": 37.6632317,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6100250413",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7509238,\n   "lon": 37.8199714,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/6100330146",\n   "kind": "organisation",\n   "name": "ТК \\"Гонец\\"",\n   "office_type": "company",\n   "lat": 55.6037704,\n   "lon": 37.790731,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6103171976",\n   "kind": "organisation",\n   "name": "ЦТП № 6 Развилка МУП Видновское ПТО ГХ",\n   "office_type": "company",\n   "lat": 55.5886634,\n   "lon": 37.7562128,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6105232248",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.6560628,\n   "lon": 37.6491082,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6106776786",\n   "kind": "organisation",\n   "name": "Аура",\n   "office_type": "notary",\n   "lat": 55.6143481,\n   "lon": 37.7321267,\n   "opening_hours": "Tu-Fr 10:00-13:00,14:00-17:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/6111087131",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6732547,\n   "lon": 37.6615164,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6111428385",\n   "kind": "organisation",\n   "name": "KNX-Trade",\n   "office_type": "company",\n   "lat": 55.655685,\n   "lon": 37.5519771,\n   "opening_hours": "Mo-Fr 12:00-18:00"\n  },\n  {\n   "id": "node/6111603916",\n   "kind": "organisation",\n   "name": "Ресо",\n   "office_type": "insurance",\n   "lat": 55.6359243,\n   "lon": 37.7199195,\n   "opening_hours": "Mo-Fr 09:30-20:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/6112327787",\n   "kind": "organisation",\n   "name": "ООО Столичный Обмен Жилья",\n   "office_type": "lawyer",\n   "lat": 55.747473,\n   "lon": 37.673064,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6112669260",\n   "kind": "organisation",\n   "name": "Студия дизайна Arstelle Design",\n   "office_type": "company",\n   "lat": 55.8015827,\n   "lon": 37.5812978,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6112826987",\n   "kind": "organisation",\n   "name": "АО \\"Особые экономические зоны\\"",\n   "office_type": "company",\n   "lat": 55.758388,\n   "lon": 37.5998054,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6112826997",\n   "kind": "organisation",\n   "name": "ОДС № 3 ГБУ \\"Жилищник района Якиманка\\"",\n   "office_type": "property_management",\n   "lat": 55.7315385,\n   "lon": 37.6087962,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6112922085",\n   "kind": "organisation",\n   "name": "Квик Софт",\n   "office_type": "company",\n   "lat": 55.7885859,\n   "lon": 37.4681501,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6114043830",\n   "kind": "organisation",\n   "name": "ОДС № 404 ГБУ \\"Жилищник района Дорогомилово\\"",\n   "office_type": "property_management",\n   "lat": 55.7380653,\n   "lon": 37.5217827,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6118622761",\n   "kind": "organisation",\n   "name": "Федеральная служба войск национальной гвардии",\n   "office_type": "government",\n   "lat": 55.7578453,\n   "lon": 37.6377059,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6125651053",\n   "kind": "organisation",\n   "name": "Мечел",\n   "office_type": "company",\n   "lat": 55.7964971,\n   "lon": 37.5498211,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6125651789",\n   "kind": "organisation",\n   "name": "Исследовательский центр Пиво и напитки XXI век",\n   "office_type": "research",\n   "lat": 55.7987037,\n   "lon": 37.5703776,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6126489124",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6115599,\n   "lon": 37.7199615,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/6126502722",\n   "kind": "organisation",\n   "name": "Ресо",\n   "office_type": "insurance",\n   "lat": 55.6206525,\n   "lon": 37.7179648,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6129031284",\n   "kind": "organisation",\n   "name": "Жилищник района Орехово-Борисово Южное",\n   "office_type": "government",\n   "lat": 55.6078774,\n   "lon": 37.7084671,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6129142182",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6106832,\n   "lon": 37.720989,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6130392330",\n   "kind": "organisation",\n   "name": "ООО «ГРК-Групп»",\n   "office_type": "company",\n   "lat": 55.6144733,\n   "lon": 37.6176051,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6132073885",\n   "kind": "organisation",\n   "name": "1й Национальный альянс",\n   "office_type": "company",\n   "lat": 55.7099445,\n   "lon": 37.6550958,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6132808310",\n   "kind": "organisation",\n   "name": "Simplemeals",\n   "office_type": "company",\n   "lat": 55.8833422,\n   "lon": 37.6794626,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6137399352",\n   "kind": "organisation",\n   "name": "Шоли Механикс",\n   "office_type": "company",\n   "lat": 55.7238415,\n   "lon": 37.6368445,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6142584250",\n   "kind": "organisation",\n   "name": "Горная долина",\n   "office_type": "company",\n   "lat": 55.8856939,\n   "lon": 37.5342313,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6143223734",\n   "kind": "organisation",\n   "name": "Турчина М. А.",\n   "office_type": "notary",\n   "lat": 55.8704804,\n   "lon": 37.560153,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6144618578",\n   "kind": "organisation",\n   "name": "Ру-Синтез",\n   "office_type": "company",\n   "lat": 55.7105215,\n   "lon": 37.4442841,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6148078943",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8225064,\n   "lon": 37.8236103,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/6149354402",\n   "kind": "organisation",\n   "name": "Dinoel",\n   "office_type": "company",\n   "lat": 55.6602026,\n   "lon": 37.7415929,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/6151264710",\n   "kind": "organisation",\n   "name": "Богатырев & Партнеры",\n   "office_type": "lawyer",\n   "lat": 55.7827873,\n   "lon": 37.5983658,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6152205218",\n   "kind": "organisation",\n   "name": "ПИК-Комфорт",\n   "office_type": "company",\n   "lat": 55.7121251,\n   "lon": 37.7499764,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6158353571",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8030171,\n   "lon": 37.6368431,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6160009106",\n   "kind": "organisation",\n   "name": "Ренессанс страхование",\n   "office_type": "insurance",\n   "lat": 55.5855713,\n   "lon": 37.7295456,\n   "opening_hours": "Fr 09:00-16:45;Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/6162042486",\n   "kind": "organisation",\n   "name": "TraceAir",\n   "office_type": "company",\n   "lat": 55.707256,\n   "lon": 37.6495922,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/6167958256",\n   "kind": "organisation",\n   "name": "ЭлектроПласт",\n   "office_type": "company",\n   "lat": 55.6114082,\n   "lon": 37.6137962,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6176251891",\n   "kind": "organisation",\n   "name": "МОЭК Филиал № 2 Предприятие № 5 «Савёловское»",\n   "office_type": "administrative",\n   "lat": 55.8104646,\n   "lon": 37.537622,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6177029391",\n   "kind": "organisation",\n   "name": "Газпром-медиа",\n   "office_type": "company",\n   "lat": 55.7573224,\n   "lon": 37.565901,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6186812279",\n   "kind": "organisation",\n   "name": "Творческий союз художников России",\n   "office_type": "ngo",\n   "lat": 55.7419127,\n   "lon": 37.6214468,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6191270886",\n   "kind": "organisation",\n   "name": "ПроОдинДом",\n   "office_type": "company",\n   "lat": 55.7885869,\n   "lon": 37.4890077,\n   "opening_hours": "Mo-Sa 10:00-21:00"\n  },\n  {\n   "id": "node/6193233295",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7733582,\n   "lon": 37.6063862,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6195423785",\n   "kind": "organisation",\n   "name": "Федеральный экологический оператор",\n   "office_type": "company",\n   "lat": 55.7390615,\n   "lon": 37.6201574,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6200811012",\n   "kind": "organisation",\n   "name": "Мусор-Рин",\n   "office_type": "company",\n   "lat": 55.7114119,\n   "lon": 37.7052686,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6202028477",\n   "kind": "organisation",\n   "name": "Резиденция посла Великобритании",\n   "office_type": "diplomatic",\n   "lat": 55.7470897,\n   "lon": 37.6166993,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6203968904",\n   "kind": "organisation",\n   "name": "Группа Компаний СИМБИО",\n   "office_type": "company",\n   "lat": 55.8403118,\n   "lon": 37.4903763,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6204078820",\n   "kind": "organisation",\n   "name": "ОАО Моснефтегазстройкомпонент",\n   "office_type": "company",\n   "lat": 55.7259704,\n   "lon": 37.7371654,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6204209959",\n   "kind": "organisation",\n   "name": "ГКУ \\"АМПП\\"",\n   "office_type": "administrative",\n   "lat": 55.7400242,\n   "lon": 37.7441638,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6205258085",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7351156,\n   "lon": 37.4599839,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/6206469996",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6917965,\n   "lon": 37.7283624,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6206469997",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6917164,\n   "lon": 37.728357,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/6208540085",\n   "kind": "organisation",\n   "name": "Mover",\n   "office_type": "company",\n   "lat": 55.7880221,\n   "lon": 37.5828759,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6209910406",\n   "kind": "organisation",\n   "name": "ООО \\"Роял Косметикс\\"",\n   "office_type": "company",\n   "lat": 55.7176871,\n   "lon": 37.5120863,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6211344203",\n   "kind": "organisation",\n   "name": "GTD Russia",\n   "office_type": "company",\n   "lat": 55.7741405,\n   "lon": 37.5310091,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6212361482",\n   "kind": "organisation",\n   "name": "BerryBoutique",\n   "office_type": "company",\n   "lat": 55.8124092,\n   "lon": 37.5324798,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6215506850",\n   "kind": "organisation",\n   "name": "Транслогистик",\n   "office_type": "logistics",\n   "lat": 55.7702465,\n   "lon": 37.5988882,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6215522639",\n   "kind": "organisation",\n   "name": "ООО НИЛ \\"СТРОЙМАТЕРИАЛЫ\\"",\n   "office_type": "company",\n   "lat": 55.7687687,\n   "lon": 37.5068368,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6221133336",\n   "kind": "organisation",\n   "name": "Офис продаж balance",\n   "office_type": "estate_agent",\n   "lat": 55.7201328,\n   "lon": 37.7735415,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6222477285",\n   "kind": "organisation",\n   "name": "Сенсор Слип",\n   "office_type": "company",\n   "lat": 55.6121745,\n   "lon": 37.6151946,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/6225567567",\n   "kind": "organisation",\n   "name": "ОДС № 3 ГБУ \\"Жилищник Басманного района\\"",\n   "office_type": "property_management",\n   "lat": 55.7631621,\n   "lon": 37.6362661,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6225902487",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7623374,\n   "lon": 37.6360394,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6226510253",\n   "kind": "organisation",\n   "name": "Стилот",\n   "office_type": "company",\n   "lat": 55.6443037,\n   "lon": 37.4946885,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6226688754",\n   "kind": "organisation",\n   "name": "ОООП Новые возможности",\n   "office_type": "company",\n   "lat": 55.6437116,\n   "lon": 37.6706599,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6226689458",\n   "kind": "organisation",\n   "name": "Управление гидрометеорологии",\n   "office_type": "government",\n   "lat": 55.6437529,\n   "lon": 37.6699756,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6226807730",\n   "kind": "organisation",\n   "name": "ЛесПро",\n   "office_type": "company",\n   "lat": 55.7130551,\n   "lon": 37.7983273,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6226950097",\n   "kind": "organisation",\n   "name": "Академкапстрой",\n   "office_type": "company",\n   "lat": 55.6954946,\n   "lon": 37.565682,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6227937785",\n   "kind": "organisation",\n   "name": "Движение «Альтернатива»",\n   "office_type": "ngo",\n   "lat": 55.7412151,\n   "lon": 37.609179,\n   "opening_hours": "Mo-Su 11:00-19:00"\n  },\n  {\n   "id": "node/6229113209",\n   "kind": "organisation",\n   "name": "Радуга",\n   "office_type": "company",\n   "lat": 55.7393394,\n   "lon": 37.6290808,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6230822078",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7793241,\n   "lon": 37.7216844,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6231780513",\n   "kind": "organisation",\n   "name": "Литлидер",\n   "office_type": "company",\n   "lat": 55.630378,\n   "lon": 37.4745073,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6241070485",\n   "kind": "organisation",\n   "name": "IQSocial",\n   "office_type": "company",\n   "lat": 55.7924458,\n   "lon": 37.6079433,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6247266276",\n   "kind": "organisation",\n   "name": "Sititek",\n   "office_type": "company",\n   "lat": 55.7974544,\n   "lon": 37.7442215,\n   "opening_hours": "Mo-Fr 09:00-20:00, Su-Sa 11:00-17:00"\n  },\n  {\n   "id": "node/6255457722",\n   "kind": "organisation",\n   "name": "Альт групп",\n   "office_type": "company",\n   "lat": 55.8157416,\n   "lon": 37.5235723,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6255468772",\n   "kind": "organisation",\n   "name": "Нотариус Кутилина Н.В.",\n   "office_type": "notary",\n   "lat": 55.8157498,\n   "lon": 37.5234699,\n   "opening_hours": "Mo-Fr 10:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/6257539685",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.894101,\n   "lon": 37.4511074,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/6263326920",\n   "kind": "organisation",\n   "name": "Генетэк",\n   "office_type": "company",\n   "lat": 55.6119982,\n   "lon": 37.6127698,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6265767313",\n   "kind": "organisation",\n   "name": "Blacklight",\n   "office_type": "advertising_agency",\n   "lat": 55.7765443,\n   "lon": 37.674914,\n   "opening_hours": "Mo-Fr 11:00-19:30"\n  },\n  {\n   "id": "node/6265827729",\n   "kind": "organisation",\n   "name": "Министерство финансов РФ",\n   "office_type": "government",\n   "lat": 55.755312,\n   "lon": 37.6264089,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6267762636",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7174103,\n   "lon": 37.7887698,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6270330605",\n   "kind": "organisation",\n   "name": "Научный центр экспертизы средств медицинского применения",\n   "office_type": "research",\n   "lat": 55.7676317,\n   "lon": 37.6164365,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6274337004",\n   "kind": "organisation",\n   "name": "Центральное Хованское кладбище",\n   "office_type": "government",\n   "lat": 55.606557,\n   "lon": 37.4468029,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6274776085",\n   "kind": "organisation",\n   "name": "Лоджикал импорт",\n   "office_type": "company",\n   "lat": 55.6860906,\n   "lon": 37.7380049,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6274776186",\n   "kind": "organisation",\n   "name": "Car price",\n   "office_type": "company",\n   "lat": 55.6861856,\n   "lon": 37.7364585,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6274869586",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7471378,\n   "lon": 37.6794143,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/6278366936",\n   "kind": "organisation",\n   "name": "ЛК ТК Сервис",\n   "office_type": "logistics",\n   "lat": 55.7444061,\n   "lon": 37.7390906,\n   "opening_hours": "Mo-Sa 09:00-18:00"\n  },\n  {\n   "id": "node/6280581634",\n   "kind": "organisation",\n   "name": "Навикон",\n   "office_type": "it",\n   "lat": 55.7147571,\n   "lon": 37.6770296,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6280805485",\n   "kind": "organisation",\n   "name": "Тянь Жень",\n   "office_type": "company",\n   "lat": 55.7604676,\n   "lon": 37.765339,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6281337549",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8303015,\n   "lon": 37.4539134,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00"\n  },\n  {\n   "id": "node/6282545986",\n   "kind": "organisation",\n   "name": "Цветкова Е.С.",\n   "office_type": "notary",\n   "lat": 55.7546373,\n   "lon": 37.5993079,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6283375436",\n   "kind": "organisation",\n   "name": "Управление Судебного Департамента при ВС РФ в МО",\n   "office_type": "government",\n   "lat": 55.7523679,\n   "lon": 37.6549564,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6283521285",\n   "kind": "organisation",\n   "name": "Центральный НИИ экономики, информатики и систем управления",\n   "office_type": "research",\n   "lat": 55.7591146,\n   "lon": 37.5988962,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6283928585",\n   "kind": "organisation",\n   "name": "Квартал Риэлти",\n   "office_type": "estate_agent",\n   "lat": 55.6707718,\n   "lon": 37.5148832,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6285635676",\n   "kind": "organisation",\n   "name": "Промсвязьналадка",\n   "office_type": "telecommunication",\n   "lat": 55.764403,\n   "lon": 37.6549272,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6285635747",\n   "kind": "organisation",\n   "name": "Клуб Приключение",\n   "office_type": "company",\n   "lat": 55.762075,\n   "lon": 37.649533,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6287817785",\n   "kind": "organisation",\n   "name": "ИнЭнерджи",\n   "office_type": "company",\n   "lat": 55.6435534,\n   "lon": 37.6356724,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6287918286",\n   "kind": "organisation",\n   "name": "Гиперкоптер",\n   "office_type": "company",\n   "lat": 55.6436813,\n   "lon": 37.6356869,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6288024987",\n   "kind": "organisation",\n   "name": "Топаз",\n   "office_type": "company",\n   "lat": 55.6437131,\n   "lon": 37.6361335,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6288054686",\n   "kind": "organisation",\n   "name": "КераТех",\n   "office_type": "company",\n   "lat": 55.6437221,\n   "lon": 37.6358425,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6291334425",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "translator",\n   "lat": 55.7588471,\n   "lon": 37.6458415,\n   "opening_hours": "Mo-Th 10:00-19:00; Fr 10:00-18:00"\n  },\n  {\n   "id": "node/6292689094",\n   "kind": "organisation",\n   "name": "Представительство Правительства г. Санкт-Петербурга в г. Москве",\n   "office_type": "government",\n   "lat": 55.7611777,\n   "lon": 37.5929752,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6293651596",\n   "kind": "organisation",\n   "name": "Федерация баскетбола Москвы",\n   "office_type": "government",\n   "lat": 55.7676146,\n   "lon": 37.647177,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6294334161",\n   "kind": "organisation",\n   "name": "Департамент жилищно-коммунального хозяйства города Москвы",\n   "office_type": "government",\n   "lat": 55.7624348,\n   "lon": 37.6003996,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6294334179",\n   "kind": "organisation",\n   "name": "Главное управление Пенсионного фонда РФ №10 г. Москвы и Московской области",\n   "office_type": "government",\n   "lat": 55.762846,\n   "lon": 37.5975604,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6299089851",\n   "kind": "organisation",\n   "name": "Государственная фельдъегерская служба Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7531113,\n   "lon": 37.6378781,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6300869366",\n   "kind": "organisation",\n   "name": "Aristo Developers",\n   "office_type": "company",\n   "lat": 55.7542815,\n   "lon": 37.5575497,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6301297192",\n   "kind": "organisation",\n   "name": "Sharkom",\n   "office_type": "company",\n   "lat": 55.7962075,\n   "lon": 37.6887881,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6303242468",\n   "kind": "organisation",\n   "name": "Энергогарант",\n   "office_type": "insurance",\n   "lat": 55.745697,\n   "lon": 37.6334873,\n   "opening_hours": "Mo-Th 09:00-19:00; Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6305881890",\n   "kind": "organisation",\n   "name": "Завод металлоконструкций ПроектРесурс",\n   "office_type": "company",\n   "lat": 55.622856,\n   "lon": 37.6039131,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6306852262",\n   "kind": "organisation",\n   "name": "Нечаева Т. Е.",\n   "office_type": "notary",\n   "lat": 55.73446,\n   "lon": 37.6418249,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6306853722",\n   "kind": "organisation",\n   "name": "ОДС № 2 ГБУ \\"Жилищник района Замоскворечье\\"",\n   "office_type": "property_management",\n   "lat": 55.7428744,\n   "lon": 37.6401392,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6308074488",\n   "kind": "organisation",\n   "name": "Издательство «Ломоносовъ»",\n   "office_type": "publisher",\n   "lat": 55.7414135,\n   "lon": 37.5901354,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/6310988385",\n   "kind": "organisation",\n   "name": "GMCS",\n   "office_type": "company",\n   "lat": 55.6636369,\n   "lon": 37.4694743,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6313045293",\n   "kind": "organisation",\n   "name": "Мещанская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7703928,\n   "lon": 37.6250182,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6313147357",\n   "kind": "organisation",\n   "name": "Институт минералогии, геохимии и кристаллохимии редких элементов",\n   "office_type": "research",\n   "lat": 55.7185146,\n   "lon": 37.4458713,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/6315255197",\n   "kind": "organisation",\n   "name": "Консульский отдел",\n   "office_type": "diplomatic",\n   "lat": 55.7524455,\n   "lon": 37.6540795,\n   "opening_hours": "Mo-Fr 10:00-13:00,15:00-17:00"\n  },\n  {\n   "id": "node/6315572923",\n   "kind": "organisation",\n   "name": "Контур центр",\n   "office_type": "company",\n   "lat": 55.7810721,\n   "lon": 37.6038659,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6315898271",\n   "kind": "organisation",\n   "name": "ООО ТД ТМЗ",\n   "office_type": "company",\n   "lat": 55.7153771,\n   "lon": 37.5223478,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6318511651",\n   "kind": "organisation",\n   "name": "Sberbank CIB",\n   "office_type": "financial",\n   "lat": 55.754826,\n   "lon": 37.6090971,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6318602749",\n   "kind": "organisation",\n   "name": "Приёмная ГУ МВД по МО",\n   "office_type": "government",\n   "lat": 55.7544018,\n   "lon": 37.6514799,\n   "opening_hours": "Mo-Th 10:00-17:00; We 10:00-20:00; Fr 10:00-16:00; Sa[1] 10:00-13:00; 13:00-14:00 off; PH off"\n  },\n  {\n   "id": "node/6320259785",\n   "kind": "organisation",\n   "name": "Информтехника",\n   "office_type": "it",\n   "lat": 55.7853043,\n   "lon": 37.659406,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6322501978",\n   "kind": "organisation",\n   "name": "Главное управление пенсионного фонда РФ",\n   "office_type": "government",\n   "lat": 55.7628551,\n   "lon": 37.5975523,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6328334086",\n   "kind": "organisation",\n   "name": "Деловые линии",\n   "office_type": "logistics",\n   "lat": 55.78232,\n   "lon": 37.6776744,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6330118844",\n   "kind": "organisation",\n   "name": "Объединённая диспетчерская служба №2",\n   "office_type": "property_management",\n   "lat": 55.7529339,\n   "lon": 37.6479041,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6330254245",\n   "kind": "organisation",\n   "name": "ИнфоМонтажЦентр",\n   "office_type": "it",\n   "lat": 55.8065331,\n   "lon": 37.504855,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6330630316",\n   "kind": "organisation",\n   "name": "ЖК LIFE-Варшавская",\n   "office_type": "company",\n   "lat": 55.652846,\n   "lon": 37.6233679,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6331010684",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8485223,\n   "lon": 37.5908616,\n   "opening_hours": "9:00-21:00"\n  },\n  {\n   "id": "node/6331011286",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8484907,\n   "lon": 37.5906631,\n   "opening_hours": "9:00-21:00"\n  },\n  {\n   "id": "node/6331011287",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.84856,\n   "lon": 37.590533,\n   "opening_hours": "9:00-21:00"\n  },\n  {\n   "id": "node/6331011288",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8485133,\n   "lon": 37.5905317,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/6334108887",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "employment_agency",\n   "lat": 55.6955267,\n   "lon": 37.4971889,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6334568018",\n   "kind": "organisation",\n   "name": "Ли Вест",\n   "office_type": "company",\n   "lat": 55.7407901,\n   "lon": 37.6262229,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6334938892",\n   "kind": "organisation",\n   "name": "Профлоджистик-ТБ",\n   "office_type": "company",\n   "lat": 55.7421906,\n   "lon": 37.5015284,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6339486944",\n   "kind": "organisation",\n   "name": "Фонд перспективных исследований",\n   "office_type": "research",\n   "lat": 55.7324001,\n   "lon": 37.5516826,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6340331885",\n   "kind": "organisation",\n   "name": "GlowByte Consulting",\n   "office_type": "company",\n   "lat": 55.7593045,\n   "lon": 37.666119,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6340530418",\n   "kind": "organisation",\n   "name": "Т‑Страхование",\n   "office_type": "insurance",\n   "lat": 55.8041483,\n   "lon": 37.5641414,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6341147566",\n   "kind": "organisation",\n   "name": "Ferrum Design",\n   "office_type": "company",\n   "lat": 55.7026906,\n   "lon": 37.6711562,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6341309506",\n   "kind": "organisation",\n   "name": "Est-a-Tet",\n   "office_type": "estate_agent",\n   "lat": 55.7286756,\n   "lon": 37.6107743,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6344439616",\n   "kind": "organisation",\n   "name": "Зубастик",\n   "office_type": "company",\n   "lat": 55.8410092,\n   "lon": 37.5741483,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6347418555",\n   "kind": "organisation",\n   "name": "Международный союз музыкальных деятелей",\n   "office_type": "ngo",\n   "lat": 55.7570945,\n   "lon": 37.6054957,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6349281706",\n   "kind": "organisation",\n   "name": "Автономная некоммерческая организация «Московский институт судебных экспертиз»",\n   "office_type": "ngo",\n   "lat": 55.770835,\n   "lon": 37.650292,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6349706114",\n   "kind": "organisation",\n   "name": "Альфа страхование",\n   "office_type": "insurance",\n   "lat": 55.7894353,\n   "lon": 37.5365543,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6355886086",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.6299361,\n   "lon": 37.8280815,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6355997059",\n   "kind": "organisation",\n   "name": "Лысякова Ольга Сергеевна",\n   "office_type": "notary",\n   "lat": 55.7106372,\n   "lon": 37.5989203,\n   "opening_hours": "Mo-Th 10:00-19:00; Fr 10:00-16:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/6360168500",\n   "kind": "organisation",\n   "name": "Балтийский лизинг",\n   "office_type": "company",\n   "lat": 55.7626881,\n   "lon": 37.6774122,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6364373385",\n   "kind": "organisation",\n   "name": "УК ЮНИ-ДОМ",\n   "office_type": "property_management",\n   "lat": 55.7443758,\n   "lon": 37.771406,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/6366067100",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг",\n   "office_type": "government",\n   "lat": 55.8298481,\n   "lon": 37.6344958,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6370394792",\n   "kind": "organisation",\n   "name": "Мамси",\n   "office_type": "company",\n   "lat": 55.7065258,\n   "lon": 37.5971523,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6370396086",\n   "kind": "organisation",\n   "name": "Reposh",\n   "office_type": "company",\n   "lat": 55.7064341,\n   "lon": 37.5970641,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6371947385",\n   "kind": "organisation",\n   "name": "Бурделов и партнеры",\n   "office_type": "lawyer",\n   "lat": 55.8089621,\n   "lon": 37.4969856,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6375839822",\n   "kind": "organisation",\n   "name": "Соврыбфлот",\n   "office_type": "company",\n   "lat": 55.7668516,\n   "lon": 37.625961,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6376808567",\n   "kind": "organisation",\n   "name": "ВТБ",\n   "office_type": "bank",\n   "lat": 55.7623027,\n   "lon": 37.6237428,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6378207427",\n   "kind": "organisation",\n   "name": "Б2Б.КЭШ ЛИМИТЕД (представительство)",\n   "office_type": "company",\n   "lat": 55.603755,\n   "lon": 37.6076436,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6378471226",\n   "kind": "organisation",\n   "name": "Стройтэкс",\n   "office_type": "company",\n   "lat": 55.7620718,\n   "lon": 37.6268716,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6384348299",\n   "kind": "organisation",\n   "name": "Пресс-бюро Службы внешней разведки России",\n   "office_type": "government",\n   "lat": 55.737379,\n   "lon": 37.5956011,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6385366335",\n   "kind": "organisation",\n   "name": "Первая нерудная компания",\n   "office_type": "company",\n   "lat": 55.7401974,\n   "lon": 37.5966042,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6387935560",\n   "kind": "organisation",\n   "name": "Щетинин И. А.",\n   "office_type": "notary",\n   "lat": 55.7437041,\n   "lon": 37.5968523,\n   "opening_hours": "Mo-Th 10:00-14:30,15:00-18:00; Fr 10:00-14:30,15:00-17:00"\n  },\n  {\n   "id": "node/6391637199",\n   "kind": "organisation",\n   "name": "Институт США и Канады Российской академии наук",\n   "office_type": "research",\n   "lat": 55.7545286,\n   "lon": 37.5982846,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6393166585",\n   "kind": "organisation",\n   "name": "Федеральный Комитет по защите прав потребителей и контролю соблюдения гражданских прав",\n   "office_type": "lawyer",\n   "lat": 55.7722006,\n   "lon": 37.5971735,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/6397254499",\n   "kind": "organisation",\n   "name": "Сергеева И. А.",\n   "office_type": "notary",\n   "lat": 55.7315649,\n   "lon": 37.6211961,\n   "opening_hours": "Mo-Th 09:00-13:00,14:00-17:30; Fr 09:00-13:00,14:00-16:30"\n  },\n  {\n   "id": "node/6397956685",\n   "kind": "organisation",\n   "name": "Российская Автомобильная Федерация",\n   "office_type": "association",\n   "lat": 55.7498526,\n   "lon": 37.6450416,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6398408746",\n   "kind": "organisation",\n   "name": "Адвокат Ясюченя О. В.",\n   "office_type": "lawyer",\n   "lat": 55.8463136,\n   "lon": 37.5757447,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6401012645",\n   "kind": "organisation",\n   "name": "Капитонова Т. С.",\n   "office_type": "notary",\n   "lat": 55.734284,\n   "lon": 37.6240151,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/6402524948",\n   "kind": "organisation",\n   "name": "Посольство Республики Намибия",\n   "office_type": "diplomatic",\n   "lat": 55.7337268,\n   "lon": 37.6219873,\n   "opening_hours": "Mo-Fr 09:00-13:00, 14:00-16:30"\n  },\n  {\n   "id": "node/6402527175",\n   "kind": "organisation",\n   "name": "Газметаллпроект",\n   "office_type": "company",\n   "lat": 55.7347462,\n   "lon": 37.6240244,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6404027694",\n   "kind": "organisation",\n   "name": "Представительство Администрации Краснодарского края при Правительстве РФ",\n   "office_type": "government",\n   "lat": 55.7332798,\n   "lon": 37.6213637,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6405653085",\n   "kind": "organisation",\n   "name": "Роснефть",\n   "office_type": "company",\n   "lat": 55.7756065,\n   "lon": 37.5531512,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6407632976",\n   "kind": "organisation",\n   "name": "Московский фонд реновации жилой застройки",\n   "office_type": "government",\n   "lat": 55.7558012,\n   "lon": 37.6280264,\n   "opening_hours": "Mo-Th 08:30-17:30; Fr 08:30-16:15"\n  },\n  {\n   "id": "node/6411339686",\n   "kind": "organisation",\n   "name": "OAO Дукс",\n   "office_type": "company",\n   "lat": 55.7862827,\n   "lon": 37.5820092,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6415560137",\n   "kind": "organisation",\n   "name": "СтройМир",\n   "office_type": "company",\n   "lat": 55.6950325,\n   "lon": 37.5911465,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6417245050",\n   "kind": "organisation",\n   "name": "Image Pos",\n   "office_type": "company",\n   "lat": 55.8595383,\n   "lon": 37.5783491,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6417245059",\n   "kind": "organisation",\n   "name": "Транскиносервис",\n   "office_type": "company",\n   "lat": 55.8614771,\n   "lon": 37.5799477,\n   "opening_hours": "09:00-21:00"\n  },\n  {\n   "id": "node/6418326333",\n   "kind": "organisation",\n   "name": "Simetra",\n   "office_type": "research",\n   "lat": 55.7625563,\n   "lon": 37.6396312,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6420366286",\n   "kind": "organisation",\n   "name": "АО \\"ЭР-Телеком Холдинг\\"",\n   "office_type": "company",\n   "lat": 55.7440843,\n   "lon": 37.6348735,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6425559347",\n   "kind": "organisation",\n   "name": "Пенсионный фонд",\n   "office_type": "government",\n   "lat": 55.8585448,\n   "lon": 37.5145971,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:45"\n  },\n  {\n   "id": "node/6426282602",\n   "kind": "organisation",\n   "name": "Центр дизайна и ремёсел",\n   "office_type": "company",\n   "lat": 55.7189312,\n   "lon": 37.6017165,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6426286691",\n   "kind": "organisation",\n   "name": "Столичные кабельные сети",\n   "office_type": "telecommunication",\n   "lat": 55.7185956,\n   "lon": 37.602092,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6426308024",\n   "kind": "organisation",\n   "name": "Центр патриотического воспитания и школьного спорта Департамента образования г. Москвы",\n   "office_type": "government",\n   "lat": 55.7189188,\n   "lon": 37.6073497,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6428749763",\n   "kind": "organisation",\n   "name": "Районное общество инвалидов Якиманка",\n   "office_type": "ngo",\n   "lat": 55.735364,\n   "lon": 37.6168057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6428780495",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7372089,\n   "lon": 37.6153887,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6429802356",\n   "kind": "organisation",\n   "name": "Культурно-деловой центр российских корейцев",\n   "office_type": "ngo",\n   "lat": 55.7973174,\n   "lon": 37.5336474,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6430278915",\n   "kind": "organisation",\n   "name": "Деловой центр \\"Ленинградский\\"",\n   "office_type": "company",\n   "lat": 55.7993656,\n   "lon": 37.5327502,\n   "opening_hours": "08:00-22:00"\n  },\n  {\n   "id": "node/6430298781",\n   "kind": "organisation",\n   "name": "МЧС",\n   "office_type": "government",\n   "lat": 55.8471912,\n   "lon": 37.5728694,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6432064996",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7465219,\n   "lon": 37.6738927,\n   "opening_hours": "Fr 10:00-19:00; Mo-Th 10:00-20:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/6432473591",\n   "kind": "organisation",\n   "name": "Горэнергосбыт - Северное отделение",\n   "office_type": "company",\n   "lat": 55.858319,\n   "lon": 37.4701515,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6432738982",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.6916242,\n   "lon": 37.6224087,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/6432745992",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.8923944,\n   "lon": 37.5548931,\n   "opening_hours": "Mo-Th,Sa-Su 10:00-21:00"\n  },\n  {\n   "id": "node/6432745993",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.766815,\n   "lon": 37.540111,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/6432745994",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.598044,\n   "lon": 37.726044,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/6432745997",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.686216,\n   "lon": 37.736401,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/6432745998",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.8899519,\n   "lon": 37.633962,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/6432745999",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.687819,\n   "lon": 37.488108,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/6432746000",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.8055761,\n   "lon": 37.586323,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/6432746001",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.7870905,\n   "lon": 37.7178995,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/6433659356",\n   "kind": "organisation",\n   "name": "Жилищник района Тропарёво-Никулино",\n   "office_type": "property_management",\n   "lat": 55.6626569,\n   "lon": 37.4889448,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6436000785",\n   "kind": "organisation",\n   "name": "Ваш страховой магазин",\n   "office_type": "insurance",\n   "lat": 55.8118563,\n   "lon": 37.6983685,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/6436810195",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6155523,\n   "lon": 37.7417225,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6438111467",\n   "kind": "organisation",\n   "name": "ООО АФ \\"Трай - МАК Аудит\\"",\n   "office_type": "accountant",\n   "lat": 55.7153285,\n   "lon": 37.5224782,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6438112517",\n   "kind": "organisation",\n   "name": "SMP Racing",\n   "office_type": "quango",\n   "lat": 55.7153602,\n   "lon": 37.522446,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6438120706",\n   "kind": "organisation",\n   "name": "СБМ Группа компаний",\n   "office_type": "company",\n   "lat": 55.7155264,\n   "lon": 37.522733,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6438280380",\n   "kind": "organisation",\n   "name": "Мослифт специализированное управление #5",\n   "office_type": "company",\n   "lat": 55.6901799,\n   "lon": 37.5603769,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6438700786",\n   "kind": "organisation",\n   "name": "Юрстарт",\n   "office_type": "lawyer",\n   "lat": 55.7529106,\n   "lon": 37.6649345,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6442444928",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8225813,\n   "lon": 37.6222666,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6442588482",\n   "kind": "organisation",\n   "name": "Ассоциация банков России",\n   "office_type": "association",\n   "lat": 55.7365643,\n   "lon": 37.6146944,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6443398340",\n   "kind": "organisation",\n   "name": "Nika Estate",\n   "office_type": "estate_agent",\n   "lat": 55.7641166,\n   "lon": 37.6006624,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6447712448",\n   "kind": "organisation",\n   "name": "Институт государства и права РАН",\n   "office_type": "research",\n   "lat": 55.7502025,\n   "lon": 37.6055279,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6451244752",\n   "kind": "organisation",\n   "name": "Московская коллегия адвокатов «На Малой Дмитровке»",\n   "office_type": "lawyer",\n   "lat": 55.7711905,\n   "lon": 37.6058124,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6454313363",\n   "kind": "organisation",\n   "name": "Амарилис",\n   "office_type": "company",\n   "lat": 55.8554701,\n   "lon": 37.4451862,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6457497889",\n   "kind": "organisation",\n   "name": "Гринворкс",\n   "office_type": "company",\n   "lat": 55.7341855,\n   "lon": 37.6101627,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su 00:00-24:00"\n  },\n  {\n   "id": "node/6459145585",\n   "kind": "organisation",\n   "name": "Havas Creative Group Russia",\n   "office_type": "company",\n   "lat": 55.7323034,\n   "lon": 37.6469474,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6473447014",\n   "kind": "organisation",\n   "name": "Российский союз ветеранов",\n   "office_type": "ngo",\n   "lat": 55.7453112,\n   "lon": 37.6019043,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6475038467",\n   "kind": "organisation",\n   "name": "Следственный департамент МВД России",\n   "office_type": "government",\n   "lat": 55.7571398,\n   "lon": 37.6089263,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6476592623",\n   "kind": "organisation",\n   "name": "Совет пенсионеров, ветеранов войны, труда, Вооруженных Сил и правоохранительных органов района Дорогомилово",\n   "office_type": "ngo",\n   "lat": 55.738524,\n   "lon": 37.5168789,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6477416444",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.8047904,\n   "lon": 37.5666814,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/6477555485",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7926099,\n   "lon": 37.7836026,\n   "opening_hours": "Mo-Su 12:00-19:00"\n  },\n  {\n   "id": "node/6482038987",\n   "kind": "organisation",\n   "name": "Акустическая Заморозка",\n   "office_type": "company",\n   "lat": 55.8050882,\n   "lon": 37.4544893,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6482133285",\n   "kind": "organisation",\n   "name": "Фомос-Материалс",\n   "office_type": "company",\n   "lat": 55.789156,\n   "lon": 37.7088938,\n   "opening_hours": "Mo-Fr 09:30-18:00"\n  },\n  {\n   "id": "node/6483497285",\n   "kind": "organisation",\n   "name": "Клинский институт охраны и условий труда",\n   "office_type": "company",\n   "lat": 55.7867721,\n   "lon": 37.5203057,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6483803185",\n   "kind": "organisation",\n   "name": "Ресо",\n   "office_type": "insurance",\n   "lat": 55.6068457,\n   "lon": 37.7018781,\n   "opening_hours": "Mo-Fr 09:30-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/6483838986",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6126386,\n   "lon": 37.6959935,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6485288284",\n   "kind": "organisation",\n   "name": "Строй Хауз",\n   "office_type": "company",\n   "lat": 55.7792978,\n   "lon": 37.6333448,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6488376616",\n   "kind": "organisation",\n   "name": "Amtel Properties",\n   "office_type": "company",\n   "lat": 55.7371669,\n   "lon": 37.5244877,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6489431658",\n   "kind": "organisation",\n   "name": "Alex Beauty Concept",\n   "office_type": "company",\n   "lat": 55.7386437,\n   "lon": 37.5318624,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6489925802",\n   "kind": "organisation",\n   "name": "Клуб Путешественников «Спорт-Марафон»",\n   "office_type": "association",\n   "lat": 55.7052069,\n   "lon": 37.6667283,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/6489977788",\n   "kind": "organisation",\n   "name": "Бейт Аарон",\n   "office_type": "company",\n   "lat": 55.6520961,\n   "lon": 37.4730837,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/6490175473",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7946795,\n   "lon": 37.6987629,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6490338685",\n   "kind": "organisation",\n   "name": "Remagic Group",\n   "office_type": "company",\n   "lat": 55.7045766,\n   "lon": 37.6163389,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6491072261",\n   "kind": "organisation",\n   "name": "Представительство Республики Карелия при Президенте РФ",\n   "office_type": "government",\n   "lat": 55.7594323,\n   "lon": 37.6373443,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6491072281",\n   "kind": "organisation",\n   "name": "АО \\"Зарубежнефть\\"",\n   "office_type": "company",\n   "lat": 55.7597025,\n   "lon": 37.6379317,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6491333583",\n   "kind": "organisation",\n   "name": "Интернет-правосудие | WEBJUSTICE",\n   "office_type": "lawyer",\n   "lat": 55.777773,\n   "lon": 37.502992,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6492838405",\n   "kind": "organisation",\n   "name": "Группа Компаний \\"ОСЧ\\"",\n   "office_type": "company",\n   "lat": 55.7430904,\n   "lon": 37.6740195,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6495945649",\n   "kind": "organisation",\n   "name": "Римский офис продаж",\n   "office_type": "estate_agent",\n   "lat": 55.5963033,\n   "lon": 37.7669259,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6497195385",\n   "kind": "organisation",\n   "name": "общежитие НИЯУ МИФИ",\n   "office_type": "company",\n   "lat": 55.6457893,\n   "lon": 37.6495901,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6497914787",\n   "kind": "organisation",\n   "name": "ДизайнДепо",\n   "office_type": "company",\n   "lat": 55.7413378,\n   "lon": 37.6088749,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6503588486",\n   "kind": "organisation",\n   "name": "Рег.ру",\n   "office_type": "company",\n   "lat": 55.7913636,\n   "lon": 37.523879,\n   "opening_hours": "Mo-Sa 09:00-20:00"\n  },\n  {\n   "id": "node/6503703053",\n   "kind": "organisation",\n   "name": "Бюро переводов \\"Диалог\\"",\n   "office_type": "translator",\n   "lat": 55.7660942,\n   "lon": 37.6042633,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6506035890",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7944854,\n   "lon": 37.5891006,\n   "opening_hours": "07:00-23:00"\n  },\n  {\n   "id": "node/6506982356",\n   "kind": "organisation",\n   "name": "ТД Подшипник Стандарт",\n   "office_type": "company",\n   "lat": 55.6081079,\n   "lon": 37.7846436,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6507708486",\n   "kind": "organisation",\n   "name": "ОБР",\n   "office_type": "company",\n   "lat": 55.682738,\n   "lon": 37.6332163,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6507952686",\n   "kind": "organisation",\n   "name": "Автоматика-Сервис",\n   "office_type": "company",\n   "lat": 55.6429792,\n   "lon": 37.8089525,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6511585226",\n   "kind": "organisation",\n   "name": "Autstok",\n   "office_type": "it",\n   "lat": 55.7327897,\n   "lon": 37.5933065,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/6512358386",\n   "kind": "organisation",\n   "name": "japshop.ru",\n   "office_type": "company",\n   "lat": 55.8446395,\n   "lon": 37.5785106,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6513011086",\n   "kind": "organisation",\n   "name": "Пенсионный фонд",\n   "office_type": "government",\n   "lat": 55.6529153,\n   "lon": 37.5238973,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6517552282",\n   "kind": "organisation",\n   "name": "Мурманская транспортная компания",\n   "office_type": "company",\n   "lat": 55.8530438,\n   "lon": 37.5705761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6518568773",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7677797,\n   "lon": 37.8314667,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/6519267287",\n   "kind": "organisation",\n   "name": "Иконалаб",\n   "office_type": "company",\n   "lat": 55.6658936,\n   "lon": 37.4877672,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/6522736136",\n   "kind": "organisation",\n   "name": "Sopytka",\n   "office_type": "company",\n   "lat": 55.7277912,\n   "lon": 37.6882665,\n   "opening_hours": "Mo-Su 10:00-17:00"\n  },\n  {\n   "id": "node/6524867385",\n   "kind": "organisation",\n   "name": "ООО ЮК \\"Юридиция\\"",\n   "office_type": "lawyer",\n   "lat": 55.8079452,\n   "lon": 37.636025,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa-Su 10:00-16:00"\n  },\n  {\n   "id": "node/6528003645",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7175181,\n   "lon": 37.7935316,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/6528003646",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7175425,\n   "lon": 37.7936742,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6528003649",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7176196,\n   "lon": 37.7940966,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6528278385",\n   "kind": "organisation",\n   "name": "Компания ЕТ",\n   "office_type": "company",\n   "lat": 55.7587679,\n   "lon": 37.7341218,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6534199355",\n   "kind": "organisation",\n   "name": "Государственный НИИ операционных систем",\n   "office_type": "research",\n   "lat": 55.7365492,\n   "lon": 37.6447123,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6539131447",\n   "kind": "organisation",\n   "name": "slysky",\n   "office_type": "company",\n   "lat": 55.7340211,\n   "lon": 37.5511103,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/6541838985",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7432668,\n   "lon": 37.5764836,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6543942459",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6548243,\n   "lon": 37.762985,\n   "opening_hours": "Mo-Fr 09:00-20:00,14:00-15:00; Sa 10:00-19:00,14:00-15:00; Su 10:00-16:00,14:00-15:00"\n  },\n  {\n   "id": "node/6552325637",\n   "kind": "organisation",\n   "name": "Компания «Вектор Тепла»",\n   "office_type": "company",\n   "lat": 55.6794578,\n   "lon": 37.4483878,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6555483217",\n   "kind": "organisation",\n   "name": "Мишанин Ю. В.",\n   "office_type": "notary",\n   "lat": 55.7707029,\n   "lon": 37.5926051,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6556211065",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7754522,\n   "lon": 37.5860564,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6557983735",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7188214,\n   "lon": 37.5110391,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/6557983737",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.856517,\n   "lon": 37.4933808,\n   "opening_hours": "Mo-Fr 09:30-20:00; Sa 10:00-18:00; Su 10:15-16:00"\n  },\n  {\n   "id": "node/6559548012",\n   "kind": "organisation",\n   "name": "Альт",\n   "office_type": "company",\n   "lat": 55.8648936,\n   "lon": 37.5943026,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6560021866",\n   "kind": "organisation",\n   "name": "Aryan",\n   "office_type": "company",\n   "lat": 55.7249926,\n   "lon": 37.7600094,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa 09:00-13:00"\n  },\n  {\n   "id": "node/6565488398",\n   "kind": "organisation",\n   "name": "Барт",\n   "office_type": "company",\n   "lat": 55.8926278,\n   "lon": 37.5888279,\n   "opening_hours": "Mo-Fr 08:30-17:30"\n  },\n  {\n   "id": "node/6567540285",\n   "kind": "organisation",\n   "name": "Эзвелл",\n   "office_type": "company",\n   "lat": 55.7394641,\n   "lon": 37.6132816,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/6569581176",\n   "kind": "organisation",\n   "name": "Лэксан",\n   "office_type": "lawyer",\n   "lat": 55.8349269,\n   "lon": 37.6580478,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6569999763",\n   "kind": "organisation",\n   "name": "Металл Трейд",\n   "office_type": "company",\n   "lat": 55.8744797,\n   "lon": 37.5877779,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6570297541",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.6853632,\n   "lon": 37.5699699,\n   "opening_hours": "Mo,Th,Fr,Sa,Su 10:00-19:00"\n  },\n  {\n   "id": "node/6570666078",\n   "kind": "organisation",\n   "name": "Комбинат промышленных и бытовых услуг",\n   "office_type": "company",\n   "lat": 55.6782882,\n   "lon": 37.5823533,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6570688310",\n   "kind": "organisation",\n   "name": "Специализированное управление #35 Гордорстрой",\n   "office_type": "company",\n   "lat": 55.6782863,\n   "lon": 37.5800733,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6570689847",\n   "kind": "organisation",\n   "name": "Ассоциация \\"Народные художественные промыслы России\\"",\n   "office_type": "company",\n   "lat": 55.6780907,\n   "lon": 37.5770446,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6570720215",\n   "kind": "organisation",\n   "name": "Институт русского языка и культуры МГУ",\n   "office_type": "company",\n   "lat": 55.6775681,\n   "lon": 37.5761593,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6571248960",\n   "kind": "organisation",\n   "name": "Теле2",\n   "office_type": "telecommunication",\n   "lat": 55.7101811,\n   "lon": 37.6749683,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6576327385",\n   "kind": "organisation",\n   "name": "Группа компаний ОТК",\n   "office_type": "company",\n   "lat": 55.7621845,\n   "lon": 37.6366155,\n   "opening_hours": "Mo-Th 09:00-19:00; Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6577568102",\n   "kind": "organisation",\n   "name": "Трансском",\n   "office_type": "company",\n   "lat": 55.7808651,\n   "lon": 37.6877554,\n   "opening_hours": "Mo-Fr 11:00-19:00"\n  },\n  {\n   "id": "node/6579966563",\n   "kind": "organisation",\n   "name": "ГБУ «Жилищник Красносельского района»",\n   "office_type": "property_management",\n   "lat": 55.7798596,\n   "lon": 37.6688281,\n   "opening_hours": "Mo-Fr 08:00-17:00; Sa 08:00-15:45"\n  },\n  {\n   "id": "node/6580085386",\n   "kind": "organisation",\n   "name": "Управа Красносельского района",\n   "office_type": "government",\n   "lat": 55.7793124,\n   "lon": 37.6683506,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6581232917",\n   "kind": "organisation",\n   "name": "Геленк",\n   "office_type": "company",\n   "lat": 55.7599164,\n   "lon": 37.7144809,\n   "opening_hours": "Mo-Sa 10:00-19:00"\n  },\n  {\n   "id": "node/6581508662",\n   "kind": "organisation",\n   "name": "By-f",\n   "office_type": "company",\n   "lat": 55.6512549,\n   "lon": 37.4830488,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/6582767186",\n   "kind": "organisation",\n   "name": "ООО \\"Поликварцит\\"",\n   "office_type": "company",\n   "lat": 55.6137247,\n   "lon": 37.6314236,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6582767187",\n   "kind": "organisation",\n   "name": "Р.Инвестмент Групп",\n   "office_type": "estate_agent",\n   "lat": 55.6138468,\n   "lon": 37.6313616,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6583656639",\n   "kind": "organisation",\n   "name": "Борис-88",\n   "office_type": "company",\n   "lat": 55.7005959,\n   "lon": 37.5894508,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/6586598276",\n   "kind": "organisation",\n   "name": "Техсервис",\n   "office_type": "company",\n   "lat": 55.6760882,\n   "lon": 37.6249636,\n   "opening_hours": "Mo-Th 10:00-18:00; Fr 10:00-17:30"\n  },\n  {\n   "id": "node/6587182786",\n   "kind": "organisation",\n   "name": "РН-Транс",\n   "office_type": "company",\n   "lat": 55.7190201,\n   "lon": 37.5999259,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6587192221",\n   "kind": "organisation",\n   "name": "РН-Сервис",\n   "office_type": "company",\n   "lat": 55.7188889,\n   "lon": 37.600642,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6592193574",\n   "kind": "organisation",\n   "name": "База ГУССТ № 1",\n   "office_type": "company",\n   "lat": 55.652365,\n   "lon": 37.6931758,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6593151290",\n   "kind": "organisation",\n   "name": "Московская баскетбольная академия",\n   "office_type": "company",\n   "lat": 55.6915474,\n   "lon": 37.6194267,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6593151291",\n   "kind": "organisation",\n   "name": "Отдел жилищных субсидий Доеско",\n   "office_type": "company",\n   "lat": 55.691899,\n   "lon": 37.619332,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6593166883",\n   "kind": "organisation",\n   "name": "ПИК-Комфорт служба эксплуатации",\n   "office_type": "property_management",\n   "lat": 55.6932737,\n   "lon": 37.6180896,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6593696732",\n   "kind": "organisation",\n   "name": "Межрегиональная общественная организация Академия Военных Наук",\n   "office_type": "ngo",\n   "lat": 55.7163233,\n   "lon": 37.5217081,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6593703999",\n   "kind": "organisation",\n   "name": "НИИ Военной Истории",\n   "office_type": "research",\n   "lat": 55.7163936,\n   "lon": 37.5216062,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6595783287",\n   "kind": "organisation",\n   "name": "Сбербанк-АСТ",\n   "office_type": "company",\n   "lat": 55.7319566,\n   "lon": 37.5623224,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6598168675",\n   "kind": "organisation",\n   "name": "ООО «МПК-Групп»",\n   "office_type": "company",\n   "lat": 55.6255217,\n   "lon": 37.6185968,\n   "opening_hours": "Mo-Fr 08:00-17:00; Sa 08:00-14:00"\n  },\n  {\n   "id": "node/6614851193",\n   "kind": "organisation",\n   "name": "ООО \\"Марк энд Оукс\\"",\n   "office_type": "lawyer",\n   "lat": 55.8192436,\n   "lon": 37.5649227,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/6617755666",\n   "kind": "organisation",\n   "name": "Адвокатский кабинет № 2075",\n   "office_type": "lawyer",\n   "lat": 55.7411449,\n   "lon": 37.6174074,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6618728557",\n   "kind": "organisation",\n   "name": "Binom Systems",\n   "office_type": "it",\n   "lat": 55.7839305,\n   "lon": 37.702088,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/6625058442",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7631387,\n   "lon": 37.6473275,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6630668244",\n   "kind": "organisation",\n   "name": "Deutsches Historisches Institut Moskau",\n   "office_type": "research",\n   "lat": 55.7389381,\n   "lon": 37.6561546,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6630668245",\n   "kind": "organisation",\n   "name": "Институт научной информации по общественным наукам РАН",\n   "office_type": "research",\n   "lat": 55.6802582,\n   "lon": 37.5713164,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6631613605",\n   "kind": "organisation",\n   "name": "ФГУП \\"Администрация гражданских аэропортов (аэродромов)\\"",\n   "office_type": "government",\n   "lat": 55.8221622,\n   "lon": 37.4915091,\n   "opening_hours": "Mo-Fr 08:00-05:00"\n  },\n  {\n   "id": "node/6640030366",\n   "kind": "organisation",\n   "name": "Главное бюро медико-социальной экспертизы по г. Москве",\n   "office_type": "government",\n   "lat": 55.7810931,\n   "lon": 37.5736941,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6644719313",\n   "kind": "organisation",\n   "name": "ОДС № 7",\n   "office_type": "property_management",\n   "lat": 55.7636277,\n   "lon": 37.6507567,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6649829013",\n   "kind": "organisation",\n   "name": "ООО «Драйвбелтсистем»",\n   "office_type": "company",\n   "lat": 55.8838624,\n   "lon": 37.6819129,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6652130498",\n   "kind": "organisation",\n   "name": "ГБОУ Жилищник района Нагатинский Затон",\n   "office_type": "property_management",\n   "lat": 55.6836253,\n   "lon": 37.7070915,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6652130499",\n   "kind": "organisation",\n   "name": "ОДС-302",\n   "office_type": "property_management",\n   "lat": 55.6835997,\n   "lon": 37.7069924,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6652473722",\n   "kind": "organisation",\n   "name": "Финанс Френдс",\n   "office_type": "accountant",\n   "lat": 55.7147765,\n   "lon": 37.624495,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6652892407",\n   "kind": "organisation",\n   "name": "Клининг Чистая Москва",\n   "office_type": "company",\n   "lat": 55.6806998,\n   "lon": 37.5691921,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/6653402780",\n   "kind": "organisation",\n   "name": "Жилищник района Братеево",\n   "office_type": "property_management",\n   "lat": 55.6355447,\n   "lon": 37.7283476,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6653891578",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник Бабушкинского района",\n   "office_type": "property_management",\n   "lat": 55.8637215,\n   "lon": 37.6654565,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6654749088",\n   "kind": "organisation",\n   "name": "ГТС Складские системы",\n   "office_type": "company",\n   "lat": 55.674147,\n   "lon": 37.4408518,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6655919472",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7676015,\n   "lon": 37.5996512,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/6656080139",\n   "kind": "organisation",\n   "name": "Веста-СБ",\n   "office_type": "company",\n   "lat": 55.8735077,\n   "lon": 37.6337125,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6656930342",\n   "kind": "organisation",\n   "name": "HR Sales",\n   "office_type": "employment_agency",\n   "lat": 55.7100993,\n   "lon": 37.6543426,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6657383829",\n   "kind": "organisation",\n   "name": "Hoops",\n   "office_type": "company",\n   "lat": 55.75669,\n   "lon": 37.7729896,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6658417287",\n   "kind": "organisation",\n   "name": "HinteJinro",\n   "office_type": "company",\n   "lat": 55.7098551,\n   "lon": 37.6571302,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6662492985",\n   "kind": "organisation",\n   "name": "Акционерное Общество «Единая Электронная Торговая Площадка»",\n   "office_type": "company",\n   "lat": 55.728097,\n   "lon": 37.6457724,\n   "opening_hours": "Mo-Fr 09:00-18:30"\n  },\n  {\n   "id": "node/6665202140",\n   "kind": "organisation",\n   "name": "Визовый центр Болгарии в Москве",\n   "office_type": "visa",\n   "lat": 55.7626135,\n   "lon": 37.6254374,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/6669053785",\n   "kind": "organisation",\n   "name": "Группа компаний Родина",\n   "office_type": "company",\n   "lat": 55.7712234,\n   "lon": 37.6060792,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6673311285",\n   "kind": "organisation",\n   "name": "группа компаний МПЦ",\n   "office_type": "company",\n   "lat": 55.8537838,\n   "lon": 37.5771571,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6673777104",\n   "kind": "organisation",\n   "name": "Инженерная служба Даниловского района",\n   "office_type": "property_management",\n   "lat": 55.7090977,\n   "lon": 37.6174924,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6681225194",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7284727,\n   "lon": 37.6227614,\n   "opening_hours": "Mo-Su 08:30-21:30"\n  },\n  {\n   "id": "node/6681225195",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7269749,\n   "lon": 37.6260164,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/6681317618",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8231136,\n   "lon": 37.5008225,\n   "opening_hours": "Sa 10:00-19:00; Su 12:30-18:00; Tu-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/6685059983",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7709941,\n   "lon": 37.6284689,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6685102910",\n   "kind": "organisation",\n   "name": "НИиПИ Градостроительного и системного проектирования",\n   "office_type": "research",\n   "lat": 55.7707806,\n   "lon": 37.6299079,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6687577222",\n   "kind": "organisation",\n   "name": "Бест-Недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.7592731,\n   "lon": 37.5855414,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6687722386",\n   "kind": "organisation",\n   "name": "ООО \\"Колибри транспорт\\"",\n   "office_type": "company",\n   "lat": 55.7646647,\n   "lon": 37.6344493,\n   "opening_hours": "Mo-Fr 00:00-24:00"\n  },\n  {\n   "id": "node/6689748485",\n   "kind": "organisation",\n   "name": "Арти",\n   "office_type": "company",\n   "lat": 55.8128176,\n   "lon": 37.6323142,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6689811285",\n   "kind": "organisation",\n   "name": "Индивидуальный предприниматель Румянцев Игорь Маратович",\n   "office_type": "company",\n   "lat": 55.721718,\n   "lon": 37.7751754,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/6693517072",\n   "kind": "organisation",\n   "name": "ID-partment",\n   "office_type": "advertising_agency",\n   "lat": 55.7546132,\n   "lon": 37.6054797,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6695345813",\n   "kind": "organisation",\n   "name": "Anextour",\n   "office_type": "travel_agent",\n   "lat": 55.6422748,\n   "lon": 37.6045477,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6697475068",\n   "kind": "organisation",\n   "name": "ГК РСУ 4",\n   "office_type": "company",\n   "lat": 55.7299561,\n   "lon": 37.6320935,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6701011701",\n   "kind": "organisation",\n   "name": "ОДС № 7 ГБУ \\"Жилищник Пресненского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7573103,\n   "lon": 37.6046509,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6702414638",\n   "kind": "organisation",\n   "name": "Javad",\n   "office_type": "yes",\n   "lat": 55.7983297,\n   "lon": 37.5209111,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6703292063",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7082038,\n   "lon": 37.6157069,\n   "opening_hours": "Mo-Fr 09:00-20:30; Sa-Su 10:00-20:30"\n  },\n  {\n   "id": "node/6704055289",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.6828847,\n   "lon": 37.5671355,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6708024083",\n   "kind": "organisation",\n   "name": "Дезинфекционная станция №4",\n   "office_type": "government",\n   "lat": 55.6955009,\n   "lon": 37.6222574,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6710129645",\n   "kind": "organisation",\n   "name": "Контект",\n   "office_type": "translator",\n   "lat": 55.726509,\n   "lon": 37.6286766,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6712015174",\n   "kind": "organisation",\n   "name": "Карго - мир",\n   "office_type": "company",\n   "lat": 55.754414,\n   "lon": 37.5571564,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6715041253",\n   "kind": "organisation",\n   "name": "Всероссийская общественная организация \\"Русское географическое общество\\"",\n   "office_type": "ngo",\n   "lat": 55.7577138,\n   "lon": 37.6271134,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6718444071",\n   "kind": "organisation",\n   "name": "EKA-Процессинг",\n   "office_type": "company",\n   "lat": 55.754432,\n   "lon": 37.5828069,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6718444082",\n   "kind": "organisation",\n   "name": "Промсырьеимпорт",\n   "office_type": "company",\n   "lat": 55.7540683,\n   "lon": 37.5814712,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6718987685",\n   "kind": "organisation",\n   "name": "LegisVia",\n   "office_type": "lawyer",\n   "lat": 55.7705987,\n   "lon": 37.6249009,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6720548485",\n   "kind": "organisation",\n   "name": "Nuffic Neso Russia",\n   "office_type": "educational_institution",\n   "lat": 55.7484433,\n   "lon": 37.6484014,\n   "opening_hours": "Mo-Fr 14:00-18:00"\n  },\n  {\n   "id": "node/6722440985",\n   "kind": "organisation",\n   "name": "Пенсионный фонд России, клиентская служба Красносельский",\n   "office_type": "government",\n   "lat": 55.7789569,\n   "lon": 37.6429076,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/6723117796",\n   "kind": "organisation",\n   "name": "ООО \\"Все кондиционеры\\"",\n   "office_type": "company",\n   "lat": 55.8528765,\n   "lon": 37.567404,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6723505958",\n   "kind": "organisation",\n   "name": "ДСЛ-Сервис",\n   "office_type": "company",\n   "lat": 55.7256803,\n   "lon": 37.5734674,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6724056199",\n   "kind": "organisation",\n   "name": "Федеральная антимонопольная служба",\n   "office_type": "government",\n   "lat": 55.763593,\n   "lon": 37.5868598,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6724056200",\n   "kind": "organisation",\n   "name": "ФГУП «ЦНИИ «Центр»",\n   "office_type": "research",\n   "lat": 55.7629984,\n   "lon": 37.5855535,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6726025556",\n   "kind": "organisation",\n   "name": "Офис продаж «Симфония набережных»",\n   "office_type": "estate_agent",\n   "lat": 55.7440651,\n   "lon": 37.6441777,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6726237915",\n   "kind": "organisation",\n   "name": "Лэйбл Хоум",\n   "office_type": "company",\n   "lat": 55.7916432,\n   "lon": 37.7485823,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/6726586597",\n   "kind": "organisation",\n   "name": "Сатон-Холидж",\n   "office_type": "logistics",\n   "lat": 55.6543422,\n   "lon": 37.5579311,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6727029708",\n   "kind": "organisation",\n   "name": "ОДС 369 ГБУ \\"Жилищник района Дорогомилово\\"",\n   "office_type": "property_management",\n   "lat": 55.736559,\n   "lon": 37.5224036,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6727547854",\n   "kind": "organisation",\n   "name": "Johnson Controls",\n   "office_type": "company",\n   "lat": 55.7358697,\n   "lon": 37.5262713,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6727547881",\n   "kind": "organisation",\n   "name": "Электронный щит",\n   "office_type": "telecommunication",\n   "lat": 55.7360577,\n   "lon": 37.5251824,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6727696328",\n   "kind": "organisation",\n   "name": "Союз театральных деятелей Российской Федерации",\n   "office_type": "ngo",\n   "lat": 55.7667799,\n   "lon": 37.611056,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6731638972",\n   "kind": "organisation",\n   "name": "Сберагро",\n   "office_type": "company",\n   "lat": 55.766838,\n   "lon": 37.6527295,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6731905485",\n   "kind": "organisation",\n   "name": "Ателье 3д услуг",\n   "office_type": "company",\n   "lat": 55.7531117,\n   "lon": 37.6702083,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6734499403",\n   "kind": "organisation",\n   "name": "Альтернативный маркетинг",\n   "office_type": "company",\n   "lat": 55.6827774,\n   "lon": 37.6184349,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6734833163",\n   "kind": "organisation",\n   "name": "ООО «СТРОЙИНВЕСТ»",\n   "office_type": "company",\n   "lat": 55.7153071,\n   "lon": 37.6966339,\n   "opening_hours": "Mo-Fr 08:30-18:00"\n  },\n  {\n   "id": "node/6735072336",\n   "kind": "organisation",\n   "name": "Aleks Group",\n   "office_type": "lawyer",\n   "lat": 55.7625081,\n   "lon": 37.6523148,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6735423307",\n   "kind": "organisation",\n   "name": "Нимрод",\n   "office_type": "company",\n   "lat": 55.7790092,\n   "lon": 37.5869998,\n   "opening_hours": "Mo-Su 10:00-23:00"\n  },\n  {\n   "id": "node/6736223685",\n   "kind": "organisation",\n   "name": "Корп Софт",\n   "office_type": "company",\n   "lat": 55.7808868,\n   "lon": 37.6119362,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6737899540",\n   "kind": "organisation",\n   "name": "Cetera Labs",\n   "office_type": "it",\n   "lat": 55.6746483,\n   "lon": 37.5824852,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6737940607",\n   "kind": "organisation",\n   "name": "Vikilini",\n   "office_type": "company",\n   "lat": 55.7327751,\n   "lon": 37.5649462,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6738074093",\n   "kind": "organisation",\n   "name": "Elite Coffee Collection",\n   "office_type": "company",\n   "lat": 55.6682151,\n   "lon": 37.7392289,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6738633185",\n   "kind": "organisation",\n   "name": "1С Автоматизация",\n   "office_type": "it",\n   "lat": 55.7496791,\n   "lon": 37.7256967,\n   "opening_hours": "Mo-Fr 09:15-19:30"\n  },\n  {\n   "id": "node/6738796695",\n   "kind": "organisation",\n   "name": "НТА-Пром",\n   "office_type": "company",\n   "lat": 55.6562682,\n   "lon": 37.5276578,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6739888858",\n   "kind": "organisation",\n   "name": "ФГУП \\"Торговый дом \\"Кремлевский\\"",\n   "office_type": "government",\n   "lat": 55.7602352,\n   "lon": 37.6526383,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6739888859",\n   "kind": "organisation",\n   "name": "ФГУП \\"Госзагрансобственность\\"",\n   "office_type": "government",\n   "lat": 55.7603016,\n   "lon": 37.6526946,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6742082233",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.8732734,\n   "lon": 37.5574708,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6745453126",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.728813,\n   "lon": 37.6111203,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6745753066",\n   "kind": "organisation",\n   "name": "Армгласс",\n   "office_type": "company",\n   "lat": 55.7782277,\n   "lon": 37.6996178,\n   "opening_hours": "Mo-Su 08:00-19:00"\n  },\n  {\n   "id": "node/6746227454",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.840805,\n   "lon": 37.4873589,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6746284529",\n   "kind": "organisation",\n   "name": "Пэк:Easyway",\n   "office_type": "logistics",\n   "lat": 55.7566369,\n   "lon": 37.7509868,\n   "opening_hours": "Mo-Fr 10:00-19:00;Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/6746284530",\n   "kind": "organisation",\n   "name": "ПЭК:Easyway",\n   "office_type": "logistics",\n   "lat": 55.7907616,\n   "lon": 37.4938793,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/6746284533",\n   "kind": "organisation",\n   "name": "ПЭК:Easyway",\n   "office_type": "logistics",\n   "lat": 55.771661,\n   "lon": 37.598726,\n   "opening_hours": "Fr 10:00-20:00;Mo-Th 10:00-21:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/6746284535",\n   "kind": "organisation",\n   "name": "ПЭК:Easyway",\n   "office_type": "logistics",\n   "lat": 55.7818699,\n   "lon": 37.707248,\n   "opening_hours": "Mo-Fr 10:00-21:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/6746284537",\n   "kind": "organisation",\n   "name": "ПЭК:Easyway",\n   "office_type": "logistics",\n   "lat": 55.640421,\n   "lon": 37.61026,\n   "opening_hours": "Mo-Fr 10:00-21:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/6746284539",\n   "kind": "organisation",\n   "name": "ПЭК:Easyway",\n   "office_type": "logistics",\n   "lat": 55.617271,\n   "lon": 37.715686,\n   "opening_hours": "Mo-Fr 10:00-19:30;Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/6746284547",\n   "kind": "organisation",\n   "name": "ПЭК:Easyway",\n   "office_type": "logistics",\n   "lat": 55.6333729,\n   "lon": 37.6580161,\n   "opening_hours": "Mo-Fr 09:00-20:00;Sa-Su 10:00-19:00"\n  },\n  {\n   "id": "node/6746284560",\n   "kind": "organisation",\n   "name": "ПЭК:Easyway",\n   "office_type": "logistics",\n   "lat": 55.603446,\n   "lon": 37.662506,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/6746284591",\n   "kind": "organisation",\n   "name": "ПЭК:Easyway",\n   "office_type": "logistics",\n   "lat": 55.750454,\n   "lon": 37.789896,\n   "opening_hours": "Mo-Fr 10:00-20:30;Sa-Su 11:00-18:00"\n  },\n  {\n   "id": "node/6749185237",\n   "kind": "organisation",\n   "name": "Отдел комплектации",\n   "office_type": "company",\n   "lat": 55.6341476,\n   "lon": 37.6294594,\n   "opening_hours": "Mo-Fr 08:30-17:30; Su 10:00-15:00"\n  },\n  {\n   "id": "node/6749531205",\n   "kind": "organisation",\n   "name": "ОА \\"Гипрокислород\\"",\n   "office_type": "company",\n   "lat": 55.8124403,\n   "lon": 37.5317787,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6749531210",\n   "kind": "organisation",\n   "name": "МЗ Инжиниринг",\n   "office_type": "company",\n   "lat": 55.8124293,\n   "lon": 37.5320883,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6749531214",\n   "kind": "organisation",\n   "name": "ООО «Киностудия \\"Звезда\\"»",\n   "office_type": "company",\n   "lat": 55.8074248,\n   "lon": 37.5421331,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6750180100",\n   "kind": "organisation",\n   "name": "Механика",\n   "office_type": "company",\n   "lat": 55.80858,\n   "lon": 37.5180622,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6753485986",\n   "kind": "organisation",\n   "name": "Объединенная энергетическая компания",\n   "office_type": "company",\n   "lat": 55.7484952,\n   "lon": 37.6288819,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6764738185",\n   "kind": "organisation",\n   "name": "ППК \\"РЭО\\"",\n   "office_type": "company",\n   "lat": 55.7498972,\n   "lon": 37.5377382,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6774653250",\n   "kind": "organisation",\n   "name": "Память",\n   "office_type": "company",\n   "lat": 55.6929918,\n   "lon": 37.5631556,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6776776185",\n   "kind": "organisation",\n   "name": "Телеканал ТНТ",\n   "office_type": "television",\n   "lat": 55.7900816,\n   "lon": 37.6315374,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6776782385",\n   "kind": "organisation",\n   "name": "Телеканал Матч",\n   "office_type": "television",\n   "lat": 55.7900467,\n   "lon": 37.6313266,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6776782386",\n   "kind": "organisation",\n   "name": "Телеканал 2х2",\n   "office_type": "television",\n   "lat": 55.7900262,\n   "lon": 37.6316092,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6776782387",\n   "kind": "organisation",\n   "name": "Телеканал Пятница",\n   "office_type": "television",\n   "lat": 55.7898302,\n   "lon": 37.6309746,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6776782388",\n   "kind": "organisation",\n   "name": "Телеканал ТВ-3",\n   "office_type": "television",\n   "lat": 55.7898974,\n   "lon": 37.6312011,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6776786285",\n   "kind": "organisation",\n   "name": "Телеканал Супер",\n   "office_type": "television",\n   "lat": 55.7899713,\n   "lon": 37.6314024,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6776946685",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7894032,\n   "lon": 37.6300737,\n   "opening_hours": "Mo-Fr 09:30-20:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/6780568785",\n   "kind": "organisation",\n   "name": "ФАУ «Главгосэкспертиза России»",\n   "office_type": "government",\n   "lat": 55.7335224,\n   "lon": 37.6114351,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6782814414",\n   "kind": "organisation",\n   "name": "Aim Factory",\n   "office_type": "company",\n   "lat": 55.7335722,\n   "lon": 37.4802582,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/6782933385",\n   "kind": "organisation",\n   "name": "Норд",\n   "office_type": "lawyer",\n   "lat": 55.7413496,\n   "lon": 37.5314069,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6785042685",\n   "kind": "organisation",\n   "name": "Государственная инспекция по контролю за использованием объектов недвижимости",\n   "office_type": "government",\n   "lat": 55.7989385,\n   "lon": 37.52827,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6787775469",\n   "kind": "organisation",\n   "name": "Russdveri",\n   "office_type": "company",\n   "lat": 55.8583792,\n   "lon": 37.6856594,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/6788047035",\n   "kind": "organisation",\n   "name": "Райдекс",\n   "office_type": "company",\n   "lat": 55.7621526,\n   "lon": 37.5698289,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/6790080114",\n   "kind": "organisation",\n   "name": "Rebelstore",\n   "office_type": "company",\n   "lat": 55.7828134,\n   "lon": 37.7177082,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/6792885464",\n   "kind": "organisation",\n   "name": "Сити-Тест",\n   "office_type": "company",\n   "lat": 55.7632037,\n   "lon": 37.5248474,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6795322216",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник Диспетчерская района Нагатино-Садовники",\n   "office_type": "property_management",\n   "lat": 55.6621901,\n   "lon": 37.6291061,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6798175043",\n   "kind": "organisation",\n   "name": "Стройгарантсервис",\n   "office_type": "company",\n   "lat": 55.7693773,\n   "lon": 37.5079072,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6799413974",\n   "kind": "organisation",\n   "name": "Stonex group",\n   "office_type": "company",\n   "lat": 55.6733259,\n   "lon": 37.5841308,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/6800354370",\n   "kind": "organisation",\n   "name": "Российский дорожный научно-исследовательский институт",\n   "office_type": "research",\n   "lat": 55.8445906,\n   "lon": 37.5011498,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6801999750",\n   "kind": "organisation",\n   "name": "MDT",\n   "office_type": "company",\n   "lat": 55.7902306,\n   "lon": 37.5537273,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/6802180457",\n   "kind": "organisation",\n   "name": "Мостотрест",\n   "office_type": "company",\n   "lat": 55.7483638,\n   "lon": 37.6562351,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6804715219",\n   "kind": "organisation",\n   "name": "ХолодПарк",\n   "office_type": "company",\n   "lat": 55.7653415,\n   "lon": 37.6386242,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/6805054274",\n   "kind": "organisation",\n   "name": "АС-Групп",\n   "office_type": "company",\n   "lat": 55.6816301,\n   "lon": 37.6332153,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:30"\n  },\n  {\n   "id": "node/6805172400",\n   "kind": "organisation",\n   "name": "Текно-медикал",\n   "office_type": "company",\n   "lat": 55.6756421,\n   "lon": 37.5059613,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/6806548487",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7279295,\n   "lon": 37.6141043,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/6807372316",\n   "kind": "organisation",\n   "name": "Отдел по ЗАО УФСБ РФ",\n   "office_type": "government",\n   "lat": 55.7380971,\n   "lon": 37.5331868,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6814430385",\n   "kind": "organisation",\n   "name": "ГК Мангазея",\n   "office_type": "company",\n   "lat": 55.7115555,\n   "lon": 37.5584106,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6819572035",\n   "kind": "organisation",\n   "name": "ЛюксДом",\n   "office_type": "estate_agent",\n   "lat": 55.7268696,\n   "lon": 37.574387,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6821259442",\n   "kind": "organisation",\n   "name": "Красные крыши",\n   "office_type": "company",\n   "lat": 55.7691677,\n   "lon": 37.6668373,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6824846164",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник района Чертаново Северное",\n   "office_type": "property_management",\n   "lat": 55.6286963,\n   "lon": 37.6163087,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6835786785",\n   "kind": "organisation",\n   "name": "ООО Сисмекс Рус",\n   "office_type": "company",\n   "lat": 55.7655766,\n   "lon": 37.5273873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6840407997",\n   "kind": "organisation",\n   "name": "Кэшелот",\n   "office_type": "company",\n   "lat": 55.8467229,\n   "lon": 37.6555806,\n   "opening_hours": "Mo-Su 09:00-19:00"\n  },\n  {\n   "id": "node/6849473047",\n   "kind": "organisation",\n   "name": "Kanzler",\n   "office_type": "company",\n   "lat": 55.8524784,\n   "lon": 37.5690933,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/6859731485",\n   "kind": "organisation",\n   "name": "ИФНС России № 36 по г. Москве",\n   "office_type": "government",\n   "lat": 55.6879737,\n   "lon": 37.5409439,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6860438113",\n   "kind": "organisation",\n   "name": "SvetOnOff",\n   "office_type": "company",\n   "lat": 55.7342262,\n   "lon": 37.8286972,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/6861387685",\n   "kind": "organisation",\n   "name": "ХитЛайф",\n   "office_type": "company",\n   "lat": 55.7705403,\n   "lon": 37.5275388,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/6866514383",\n   "kind": "organisation",\n   "name": "Глубокий колодец",\n   "office_type": "company",\n   "lat": 55.8915823,\n   "lon": 37.6226163,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/6868031110",\n   "kind": "organisation",\n   "name": "ООО ПКФ \\"РЕМЕС\\"",\n   "office_type": "company",\n   "lat": 55.7167637,\n   "lon": 37.7267584,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6869203547",\n   "kind": "organisation",\n   "name": "Hotlead",\n   "office_type": "company",\n   "lat": 55.7818234,\n   "lon": 37.6699612,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/6870806924",\n   "kind": "organisation",\n   "name": "ООО \\"Эталон Фрезерная Компания\\"",\n   "office_type": "company",\n   "lat": 55.7863701,\n   "lon": 37.7040517,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/6874868185",\n   "kind": "organisation",\n   "name": "Институт российской истории Российской академии наук",\n   "office_type": "research",\n   "lat": 55.6888395,\n   "lon": 37.5729583,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6876297482",\n   "kind": "organisation",\n   "name": "Институт Археологии РАН",\n   "office_type": "research",\n   "lat": 55.6889739,\n   "lon": 37.5725663,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6877568880",\n   "kind": "organisation",\n   "name": "City",\n   "office_type": "company",\n   "lat": 55.8181458,\n   "lon": 37.7342491,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/6878281232",\n   "kind": "organisation",\n   "name": "ООО \\"Каркас Комплект\\"",\n   "office_type": "company",\n   "lat": 55.8065285,\n   "lon": 37.5040466,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6878419831",\n   "kind": "organisation",\n   "name": "Корвет-М",\n   "office_type": "company",\n   "lat": 55.6866188,\n   "lon": 37.614048,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6883658344",\n   "kind": "organisation",\n   "name": "Мостранссклад",\n   "office_type": "company",\n   "lat": 55.7276086,\n   "lon": 37.6982132,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6883658345",\n   "kind": "organisation",\n   "name": "Автопитер",\n   "office_type": "company",\n   "lat": 55.7274557,\n   "lon": 37.6994772,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6883694847",\n   "kind": "organisation",\n   "name": "Институт экономической политики им. Е. Т. Гайдара",\n   "office_type": "research",\n   "lat": 55.7573828,\n   "lon": 37.6076549,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6898467285",\n   "kind": "organisation",\n   "name": "Что делать Консалт",\n   "office_type": "company",\n   "lat": 55.7989389,\n   "lon": 37.5604315,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6899065382",\n   "kind": "organisation",\n   "name": "Люкс Персонал",\n   "office_type": "employment_agency",\n   "lat": 55.7859496,\n   "lon": 37.6598498,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6902480685",\n   "kind": "organisation",\n   "name": "АО ЭМИ",\n   "office_type": "company",\n   "lat": 55.7706216,\n   "lon": 37.6533042,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6904885765",\n   "kind": "organisation",\n   "name": "Контур Экстерн",\n   "office_type": "company",\n   "lat": 55.7875786,\n   "lon": 37.591362,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 11:00-17:00"\n  },\n  {\n   "id": "node/6905215030",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6752677,\n   "lon": 37.7238592,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6905334375",\n   "kind": "organisation",\n   "name": "Бокс Склад",\n   "office_type": "company",\n   "lat": 55.7439029,\n   "lon": 37.6268478,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6906126530",\n   "kind": "organisation",\n   "name": "Тевекс",\n   "office_type": "company",\n   "lat": 55.6083755,\n   "lon": 37.6319199,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:00"\n  },\n  {\n   "id": "node/6907994336",\n   "kind": "organisation",\n   "name": "Live Concert",\n   "office_type": "company",\n   "lat": 55.7584204,\n   "lon": 37.6132652,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/6910263729",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7972058,\n   "lon": 37.7994758,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6912255979",\n   "kind": "organisation",\n   "name": "Кофактор",\n   "office_type": "advertising_agency",\n   "lat": 55.6827273,\n   "lon": 37.5769641,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6918453219",\n   "kind": "organisation",\n   "name": "Министерство экономического развития России",\n   "office_type": "government",\n   "lat": 55.7480483,\n   "lon": 37.5354069,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6918453220",\n   "kind": "organisation",\n   "name": "Диспетчерская служба №2",\n   "office_type": "property_management",\n   "lat": 55.7548788,\n   "lon": 37.5900516,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6920210435",\n   "kind": "organisation",\n   "name": "Арт",\n   "office_type": "estate_agent",\n   "lat": 55.7717462,\n   "lon": 37.6159792,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa 11:00-15:00"\n  },\n  {\n   "id": "node/6920685185",\n   "kind": "organisation",\n   "name": "Нокиа",\n   "office_type": "company",\n   "lat": 55.7457607,\n   "lon": 37.6609801,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6924921185",\n   "kind": "organisation",\n   "name": "Теле-кастинг",\n   "office_type": "company",\n   "lat": 55.8256754,\n   "lon": 37.5055906,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6938073285",\n   "kind": "organisation",\n   "name": "ООО \\"ПрофДокумент\\"",\n   "office_type": "company",\n   "lat": 55.8912332,\n   "lon": 37.5900984,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/6944140785",\n   "kind": "organisation",\n   "name": "Представительство Ростовской области в г. Москве",\n   "office_type": "government",\n   "lat": 55.7443989,\n   "lon": 37.6292581,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6944681693",\n   "kind": "organisation",\n   "name": "Московско-Павелецкая дистанция гражданских сооружений",\n   "office_type": "company",\n   "lat": 55.5923465,\n   "lon": 37.6576335,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6944759783",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.5936616,\n   "lon": 37.6561237,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6954888785",\n   "kind": "organisation",\n   "name": "ООО \\"Пасека\\"",\n   "office_type": "company",\n   "lat": 55.8053093,\n   "lon": 37.5862274,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/6958036941",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.6917162,\n   "lon": 37.5357779,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6958083744",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6917716,\n   "lon": 37.5356704,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-19:00"\n  },\n  {\n   "id": "node/6958090474",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6918442,\n   "lon": 37.5355995,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6965710185",\n   "kind": "organisation",\n   "name": "Lash for you",\n   "office_type": "company",\n   "lat": 55.7183035,\n   "lon": 37.6728016,\n   "opening_hours": "Mo-Sa 09:00-18:00"\n  },\n  {\n   "id": "node/6966408908",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8092128,\n   "lon": 37.798394,\n   "opening_hours": "Mo-Sa 11:00-21:00"\n  },\n  {\n   "id": "node/6966408911",\n   "kind": "organisation",\n   "name": "ПЭК:EASYWAY",\n   "office_type": "logistics",\n   "lat": 55.6616204,\n   "lon": 37.4815381,\n   "opening_hours": "Mo-Sa 12:00-20:00"\n  },\n  {\n   "id": "node/6966408916",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "company",\n   "lat": 55.677067,\n   "lon": 37.659739,\n   "opening_hours": "Mo-Su 11:00-21:00"\n  },\n  {\n   "id": "node/6966408917",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7529119,\n   "lon": 37.8026933,\n   "opening_hours": "Mo-Su 11:00-21:00"\n  },\n  {\n   "id": "node/6967245879",\n   "kind": "organisation",\n   "name": "За и Против",\n   "office_type": "lawyer",\n   "lat": 55.7615376,\n   "lon": 37.6379706,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6967521519",\n   "kind": "organisation",\n   "name": "ОДС № 1 ГБУ \\"Жилищник Красносельского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7645602,\n   "lon": 37.631721,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6970241519",\n   "kind": "organisation",\n   "name": "Головинский межрайонный следственный отдел",\n   "office_type": "government",\n   "lat": 55.8400139,\n   "lon": 37.546581,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6975023025",\n   "kind": "organisation",\n   "name": "Финко",\n   "office_type": "accountant",\n   "lat": 55.689788,\n   "lon": 37.5790827,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6976109201",\n   "kind": "organisation",\n   "name": "ООО «ПСП «Структура»",\n   "office_type": "company",\n   "lat": 55.8642197,\n   "lon": 37.6506805,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/6980614271",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6896378,\n   "lon": 37.5742807,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6982474389",\n   "kind": "organisation",\n   "name": "Мрспроектстройиндустрия",\n   "office_type": "architect",\n   "lat": 55.7597997,\n   "lon": 37.5205328,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6982716857",\n   "kind": "organisation",\n   "name": "Управляющая компания 10 Пресненского района",\n   "office_type": "property_management",\n   "lat": 55.7574636,\n   "lon": 37.5221268,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6986636163",\n   "kind": "organisation",\n   "name": "FineLux",\n   "office_type": "company",\n   "lat": 55.8825874,\n   "lon": 37.7331467,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/6989048960",\n   "kind": "organisation",\n   "name": "ДиПОС",\n   "office_type": "company",\n   "lat": 55.7639996,\n   "lon": 37.6098117,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6989541980",\n   "kind": "organisation",\n   "name": "Ск Столица",\n   "office_type": "company",\n   "lat": 55.7705062,\n   "lon": 37.5752729,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6991563610",\n   "kind": "organisation",\n   "name": "Vamvidnee",\n   "office_type": "company",\n   "lat": 55.6996689,\n   "lon": 37.6955299,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6993770449",\n   "kind": "organisation",\n   "name": "Системный оператор Единой энергетической системы",\n   "office_type": "company",\n   "lat": 55.7514524,\n   "lon": 37.6351342,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6994668467",\n   "kind": "organisation",\n   "name": "Посольство Новой Зеландии",\n   "office_type": "diplomatic",\n   "lat": 55.7358234,\n   "lon": 37.599297,\n   "opening_hours": "Mo-Fr 09:00-12:30,13:30-17:30"\n  },\n  {\n   "id": "node/6995764154",\n   "kind": "organisation",\n   "name": "Лавка чудес",\n   "office_type": "company",\n   "lat": 55.7060655,\n   "lon": 37.6468356,\n   "opening_hours": null\n  },\n  {\n   "id": "node/6998075946",\n   "kind": "organisation",\n   "name": "Крановый завод 1974",\n   "office_type": "company",\n   "lat": 55.6833506,\n   "lon": 37.6622767,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7001149797",\n   "kind": "organisation",\n   "name": "Диуса пет",\n   "office_type": "company",\n   "lat": 55.7494991,\n   "lon": 37.6808487,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7001759339",\n   "kind": "organisation",\n   "name": "Техническая инспекция ЕЭС",\n   "office_type": "company",\n   "lat": 55.7515143,\n   "lon": 37.6356867,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7003929886",\n   "kind": "organisation",\n   "name": "Смарт-Т",\n   "office_type": "company",\n   "lat": 55.7217925,\n   "lon": 37.6834295,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7004872749",\n   "kind": "organisation",\n   "name": "Centershot",\n   "office_type": "company",\n   "lat": 55.8127706,\n   "lon": 37.6043351,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7005924720",\n   "kind": "organisation",\n   "name": "ГК Ланит",\n   "office_type": "it",\n   "lat": 55.8070395,\n   "lon": 37.6222032,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7008024129",\n   "kind": "organisation",\n   "name": "ООО \\"КМ Кран\\"",\n   "office_type": "company",\n   "lat": 55.762263,\n   "lon": 37.5696879,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7009848859",\n   "kind": "organisation",\n   "name": "ОДС ГБУ \\"Жилищник Мещанского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7716247,\n   "lon": 37.6248586,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7012442293",\n   "kind": "organisation",\n   "name": "Главное медицинское управление Управления делами Президента Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7570824,\n   "lon": 37.6266557,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7012442324",\n   "kind": "organisation",\n   "name": "Межрегиональная инспекция ФНС по крупнейшим налогоплательщикам №2",\n   "office_type": "government",\n   "lat": 55.7564304,\n   "lon": 37.6272941,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7012442353",\n   "kind": "organisation",\n   "name": "Министерство труда и социальной защиты Российской Федерации",\n   "office_type": "government",\n   "lat": 55.756484,\n   "lon": 37.62843,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7012442354",\n   "kind": "organisation",\n   "name": "Совет при Президенте РФ по развитию гражданского общества и правам человека",\n   "office_type": "government",\n   "lat": 55.7566855,\n   "lon": 37.6292118,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7012442355",\n   "kind": "organisation",\n   "name": "Управление Президента РФ по обеспечению конституционных прав граждан",\n   "office_type": "government",\n   "lat": 55.756764,\n   "lon": 37.6290965,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7013194485",\n   "kind": "organisation",\n   "name": "Гильдия",\n   "office_type": "company",\n   "lat": 55.6343882,\n   "lon": 37.769141,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/7019741027",\n   "kind": "organisation",\n   "name": "АСТ",\n   "office_type": "company",\n   "lat": 55.8370528,\n   "lon": 37.4532681,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7024438302",\n   "kind": "organisation",\n   "name": "Страховой дом ВСК",\n   "office_type": "insurance",\n   "lat": 55.8793327,\n   "lon": 37.5468345,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-17:00; Su off"\n  },\n  {\n   "id": "node/7027167016",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7691428,\n   "lon": 37.6017501,\n   "opening_hours": "Mo-Fr 09:30-20:00"\n  },\n  {\n   "id": "node/7028360460",\n   "kind": "organisation",\n   "name": "Визовый центр Австрии",\n   "office_type": "visa",\n   "lat": 55.6724011,\n   "lon": 37.6313803,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/7028360461",\n   "kind": "organisation",\n   "name": "Визовый центр Литвы",\n   "office_type": "visa",\n   "lat": 55.6724752,\n   "lon": 37.6309162,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/7034471305",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7740047,\n   "lon": 37.6177078,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7036853119",\n   "kind": "organisation",\n   "name": "Представительство Калининградской области при Правительстве РФ",\n   "office_type": "government",\n   "lat": 55.760196,\n   "lon": 37.5734299,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7042443594",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7488545,\n   "lon": 37.5832012,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7044118329",\n   "kind": "organisation",\n   "name": "Центральная пригородная пассажирская компания",\n   "office_type": "company",\n   "lat": 55.7717386,\n   "lon": 37.6622232,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7046078918",\n   "kind": "organisation",\n   "name": "Федеральная таможенная служба Российской федерации",\n   "office_type": "government",\n   "lat": 55.7699737,\n   "lon": 37.6611406,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7046400301",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.893264,\n   "lon": 37.724481,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400304",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.8836997,\n   "lon": 37.7068994,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400306",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.7314978,\n   "lon": 37.4642613,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400307",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.688086,\n   "lon": 37.56362,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400315",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.6332168,\n   "lon": 37.4762099,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400329",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.7921244,\n   "lon": 37.6281236,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400330",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.774558,\n   "lon": 37.593858,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400331",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.7862818,\n   "lon": 37.7336141,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400338",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.6726964,\n   "lon": 37.7909759,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400343",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.8428492,\n   "lon": 37.4910982,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400361",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.6643137,\n   "lon": 37.753028,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7046400392",\n   "kind": "organisation",\n   "name": "Абакус центр",\n   "office_type": "company",\n   "lat": 55.587423,\n   "lon": 37.651563,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7049184046",\n   "kind": "organisation",\n   "name": "Федеральная служба по экологическому, технологическому, и атомному надзору",\n   "office_type": "government",\n   "lat": 55.7706995,\n   "lon": 37.6643777,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7049195882",\n   "kind": "organisation",\n   "name": "Российское общество оценщиков",\n   "office_type": "company",\n   "lat": 55.7703578,\n   "lon": 37.6634211,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7049201483",\n   "kind": "organisation",\n   "name": "Межрегиональный центр библиотечного сотрудничества",\n   "office_type": "company",\n   "lat": 55.7702953,\n   "lon": 37.6634098,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7049346113",\n   "kind": "organisation",\n   "name": "Грибаков, Поляк и партнёры",\n   "office_type": "lawyer",\n   "lat": 55.7394347,\n   "lon": 37.6646698,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa 11:00-18:00"\n  },\n  {\n   "id": "node/7050400713",\n   "kind": "organisation",\n   "name": "ФГБУ Научный центр экспертизы средств медицинского применения",\n   "office_type": "government",\n   "lat": 55.7466036,\n   "lon": 37.5874364,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7051782710",\n   "kind": "organisation",\n   "name": "ООО «Офислорд»",\n   "office_type": "company",\n   "lat": 55.7498224,\n   "lon": 37.5375294,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7052621828",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7722512,\n   "lon": 37.6781885,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/7053065378",\n   "kind": "organisation",\n   "name": "Представительство Тамбовской области при Правительстве РФ",\n   "office_type": "government",\n   "lat": 55.7442242,\n   "lon": 37.587706,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7054792585",\n   "kind": "organisation",\n   "name": "АрхиСтиль",\n   "office_type": "company",\n   "lat": 55.5905336,\n   "lon": 37.6017123,\n   "opening_hours": "Mo-Sa 09:00-19:00"\n  },\n  {\n   "id": "node/7056375244",\n   "kind": "organisation",\n   "name": "Центр инноваций «Объединённые кондитеры»",\n   "office_type": "research",\n   "lat": 55.775518,\n   "lon": 37.6762515,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7057095690",\n   "kind": "organisation",\n   "name": "Иванов Б.С.",\n   "office_type": "notary",\n   "lat": 55.7780313,\n   "lon": 37.6724054,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7057170496",\n   "kind": "organisation",\n   "name": "Сфера",\n   "office_type": "company",\n   "lat": 55.6290487,\n   "lon": 37.7411143,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7058162916",\n   "kind": "organisation",\n   "name": "ОДС № 6 ГБУ \\"Жилищник района Хамовники\\"",\n   "office_type": "property_management",\n   "lat": 55.7425227,\n   "lon": 37.5834185,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7059996045",\n   "kind": "organisation",\n   "name": "Московская городская нотариальная палата",\n   "office_type": "notary",\n   "lat": 55.7474224,\n   "lon": 37.5371882,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7061642788",\n   "kind": "organisation",\n   "name": "МосАкрил",\n   "office_type": "company",\n   "lat": 55.7478963,\n   "lon": 37.7039252,\n   "opening_hours": "Mo-Su 08:00-23:00"\n  },\n  {\n   "id": "node/7062865111",\n   "kind": "organisation",\n   "name": "Сельэлектросетьстрой",\n   "office_type": "company",\n   "lat": 55.7612448,\n   "lon": 37.6410913,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7063533118",\n   "kind": "organisation",\n   "name": "Holz House",\n   "office_type": "company",\n   "lat": 55.7396,\n   "lon": 37.613494,\n   "opening_hours": "Mo-Fr 09:30-18:30"\n  },\n  {\n   "id": "node/7065885702",\n   "kind": "organisation",\n   "name": "Dexmedical",\n   "office_type": "company",\n   "lat": 55.7964956,\n   "lon": 37.609633,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7070243440",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт МВД России",\n   "office_type": "research",\n   "lat": 55.7329143,\n   "lon": 37.6227021,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/7079861398",\n   "kind": "organisation",\n   "name": "ActiveCloud",\n   "office_type": "telecommunication",\n   "lat": 55.7220379,\n   "lon": 37.6518122,\n   "opening_hours": "Mo-Fr 08:00-19:00"\n  },\n  {\n   "id": "node/7082005940",\n   "kind": "organisation",\n   "name": "Realboxing.ru",\n   "office_type": "company",\n   "lat": 55.8562986,\n   "lon": 37.5296912,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7082117595",\n   "kind": "organisation",\n   "name": "Атмосфера успеха",\n   "office_type": "company",\n   "lat": 55.676284,\n   "lon": 37.57762,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7082377628",\n   "kind": "organisation",\n   "name": "Лимпопо",\n   "office_type": "company",\n   "lat": 55.8819407,\n   "lon": 37.5468371,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7083477931",\n   "kind": "organisation",\n   "name": "Журнал «Химия и жизнь»",\n   "office_type": "newspaper",\n   "lat": 55.7136841,\n   "lon": 37.5923408,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7083665132",\n   "kind": "organisation",\n   "name": "O2 Consulting",\n   "office_type": "company",\n   "lat": 55.7493884,\n   "lon": 37.5368673,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/7088079027",\n   "kind": "organisation",\n   "name": "Управление Федеральной службы государственной регистрации, кадастра и картографии по Московской области",\n   "office_type": "government",\n   "lat": 55.8533464,\n   "lon": 37.6561023,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/7088566319",\n   "kind": "organisation",\n   "name": "Московский музыкальный театр «На Басманной»",\n   "office_type": "theatre",\n   "lat": 55.7716941,\n   "lon": 37.6639867,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7090815991",\n   "kind": "organisation",\n   "name": "Промбурвод",\n   "office_type": "company",\n   "lat": 55.7637522,\n   "lon": 37.648454,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7092685185",\n   "kind": "organisation",\n   "name": "Доставка Гуру",\n   "office_type": "company",\n   "lat": 55.7279081,\n   "lon": 37.7145849,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7098028328",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7159907,\n   "lon": 37.8153597,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7098028330",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7160194,\n   "lon": 37.8154509,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7098028332",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7160632,\n   "lon": 37.8155073,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7098332546",\n   "kind": "organisation",\n   "name": "Представительство Омской области при Правительстве Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7655448,\n   "lon": 37.6477848,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7111073485",\n   "kind": "organisation",\n   "name": "maak Media",\n   "office_type": "company",\n   "lat": 55.7523756,\n   "lon": 37.4553323,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7112745841",\n   "kind": "organisation",\n   "name": "АНО «Музыкальное сердце театра»",\n   "office_type": "theatre",\n   "lat": 55.763353,\n   "lon": 37.6014429,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7112745879",\n   "kind": "organisation",\n   "name": "Театр «Бродячий вертеп»",\n   "office_type": "theatre",\n   "lat": 55.790539,\n   "lon": 37.597531,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7112753905",\n   "kind": "organisation",\n   "name": "Московский театр марионеток «Фигаро»",\n   "office_type": "theatre",\n   "lat": 55.759461,\n   "lon": 37.629709,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7115068170",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.8656204,\n   "lon": 37.6060919,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7116074185",\n   "kind": "organisation",\n   "name": "MyUSACorporation",\n   "office_type": "company",\n   "lat": 55.75764,\n   "lon": 37.6173328,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/7120352885",\n   "kind": "organisation",\n   "name": "Никифорова С. А.",\n   "office_type": "notary",\n   "lat": 55.8059813,\n   "lon": 37.5062057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7126077485",\n   "kind": "organisation",\n   "name": "Калинка",\n   "office_type": "estate_agent",\n   "lat": 55.7387501,\n   "lon": 37.6023553,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7131079577",\n   "kind": "organisation",\n   "name": "Гелиос групп",\n   "office_type": "accountant",\n   "lat": 55.7064671,\n   "lon": 37.617985,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7131916965",\n   "kind": "organisation",\n   "name": "ОДС № 5 ГБУ \\"Жилищник Басманного района\\"",\n   "office_type": "property_management",\n   "lat": 55.7626702,\n   "lon": 37.6448243,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7136843594",\n   "kind": "organisation",\n   "name": "РЕСО-мед",\n   "office_type": "insurance",\n   "lat": 55.7826648,\n   "lon": 37.6645848,\n   "opening_hours": "10:00-20:00"\n  },\n  {\n   "id": "node/7137031936",\n   "kind": "organisation",\n   "name": "Альфа регистр",\n   "office_type": "company",\n   "lat": 55.7512114,\n   "lon": 37.7544121,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7138856815",\n   "kind": "organisation",\n   "name": "Металл-стандарт",\n   "office_type": "company",\n   "lat": 55.7784269,\n   "lon": 37.6997953,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7140976685",\n   "kind": "organisation",\n   "name": "ВТБ Медстрахование",\n   "office_type": "insurance",\n   "lat": 55.5940786,\n   "lon": 37.5962269,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7147302432",\n   "kind": "organisation",\n   "name": "Ваш квартирный вопрос",\n   "office_type": "estate_agent",\n   "lat": 55.7680911,\n   "lon": 37.6331252,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7147302472",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.767806,\n   "lon": 37.6351194,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7148546607",\n   "kind": "organisation",\n   "name": "Конверсология",\n   "office_type": "marketing",\n   "lat": 55.7472755,\n   "lon": 37.5395621,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/7149384091",\n   "kind": "organisation",\n   "name": "Casting Moscow",\n   "office_type": "company",\n   "lat": 55.7204395,\n   "lon": 37.6166276,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7149802834",\n   "kind": "organisation",\n   "name": "Институт законодательства и сравнительного правоведения при Правительстве РФ",\n   "office_type": "research",\n   "lat": 55.7656867,\n   "lon": 37.6493646,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7150702435",\n   "kind": "organisation",\n   "name": "Всероссийский институт аграрных проблем и информатики им. А.А. Никонова",\n   "office_type": "research",\n   "lat": 55.7664908,\n   "lon": 37.6488099,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7151467498",\n   "kind": "organisation",\n   "name": "ROIF Expert",\n   "office_type": "marketing",\n   "lat": 55.8075789,\n   "lon": 37.5706011,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7153732445",\n   "kind": "organisation",\n   "name": "Садрин",\n   "office_type": "company",\n   "lat": 55.8165598,\n   "lon": 37.6494272,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7154194998",\n   "kind": "organisation",\n   "name": "Гаро",\n   "office_type": "company",\n   "lat": 55.8819389,\n   "lon": 37.548039,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7156550185",\n   "kind": "organisation",\n   "name": "Repometr",\n   "office_type": "company",\n   "lat": 55.8109685,\n   "lon": 37.5145858,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7156550285",\n   "kind": "organisation",\n   "name": "Sape",\n   "office_type": "company",\n   "lat": 55.8111904,\n   "lon": 37.514671,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7157582185",\n   "kind": "organisation",\n   "name": "ХексРэй",\n   "office_type": "company",\n   "lat": 55.670606,\n   "lon": 37.445083,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/7160173535",\n   "kind": "organisation",\n   "name": "Terrasun",\n   "office_type": "estate_agent",\n   "lat": 55.7493871,\n   "lon": 37.5373539,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7162041568",\n   "kind": "organisation",\n   "name": "Роспотребнадзор",\n   "office_type": "government",\n   "lat": 55.789723,\n   "lon": 37.5967633,\n   "opening_hours": "Mo-Th 09:00-17:00; Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/7162148862",\n   "kind": "organisation",\n   "name": "БК-01",\n   "office_type": "company",\n   "lat": 55.6644201,\n   "lon": 37.7529471,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/7162321584",\n   "kind": "organisation",\n   "name": "Leyli",\n   "office_type": "company",\n   "lat": 55.6612167,\n   "lon": 37.5096302,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7166603213",\n   "kind": "organisation",\n   "name": "Инлайн технолоджис",\n   "office_type": "it",\n   "lat": 55.804652,\n   "lon": 37.4896628,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7168832250",\n   "kind": "organisation",\n   "name": "Дуэт",\n   "office_type": "yes",\n   "lat": 55.7913442,\n   "lon": 37.5257031,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7182315984",\n   "kind": "organisation",\n   "name": "Технологический центр управления пригородным пассажирским комплексом Московской железной дороги",\n   "office_type": "company",\n   "lat": 55.7755321,\n   "lon": 37.6531318,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7185289331",\n   "kind": "organisation",\n   "name": "Азбука жилья",\n   "office_type": "estate_agent",\n   "lat": 55.7771153,\n   "lon": 37.5869067,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa,Su 10:00-18:00"\n  },\n  {\n   "id": "node/7193277498",\n   "kind": "organisation",\n   "name": "Комитет за гражданские права",\n   "office_type": "ngo",\n   "lat": 55.8839461,\n   "lon": 37.6659071,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7193931495",\n   "kind": "organisation",\n   "name": "Вира-АртСтрой",\n   "office_type": "company",\n   "lat": 55.7880838,\n   "lon": 37.4726856,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/7198100307",\n   "kind": "organisation",\n   "name": "СиТек",\n   "office_type": "it",\n   "lat": 55.8047645,\n   "lon": 37.4905239,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7198100308",\n   "kind": "organisation",\n   "name": "Системный Софт",\n   "office_type": "it",\n   "lat": 55.8049016,\n   "lon": 37.4905822,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7198100309",\n   "kind": "organisation",\n   "name": "Univef",\n   "office_type": "it",\n   "lat": 55.8048533,\n   "lon": 37.4908663,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7198107924",\n   "kind": "organisation",\n   "name": "Бухпрофи",\n   "office_type": "accountant",\n   "lat": 55.8032509,\n   "lon": 37.4908493,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7201363974",\n   "kind": "organisation",\n   "name": "Технопром",\n   "office_type": "it",\n   "lat": 55.8031906,\n   "lon": 37.4910613,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7201718374",\n   "kind": "organisation",\n   "name": "ОДС № 2 ГБУ \\"Жилищник Мещанского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7684963,\n   "lon": 37.6268394,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7201747805",\n   "kind": "organisation",\n   "name": "Московское городское бюро недвижимости",\n   "office_type": "estate_agent",\n   "lat": 55.7688531,\n   "lon": 37.6295122,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/7204105284",\n   "kind": "organisation",\n   "name": "МОЭСК",\n   "office_type": "company",\n   "lat": 55.6187851,\n   "lon": 37.7061645,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7205265272",\n   "kind": "organisation",\n   "name": "Рабочие Руки",\n   "office_type": "company",\n   "lat": 55.7742081,\n   "lon": 37.5319061,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/7205732703",\n   "kind": "organisation",\n   "name": "ООО \\"Агентство \\"Домовой и Партнеры\\"",\n   "office_type": "company",\n   "lat": 55.6963043,\n   "lon": 37.6201859,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7205859999",\n   "kind": "organisation",\n   "name": "Институт энергетической стратегии",\n   "office_type": "research",\n   "lat": 55.7504372,\n   "lon": 37.6452005,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7208206867",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7088409,\n   "lon": 37.733245,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7213600138",\n   "kind": "organisation",\n   "name": "Wood-Brus",\n   "office_type": "company",\n   "lat": 55.7802768,\n   "lon": 37.6964832,\n   "opening_hours": "Mo-Sa 10:00-19:00"\n  },\n  {\n   "id": "node/7218108679",\n   "kind": "organisation",\n   "name": "Московский союз художников \\"Объединение художников графического станкового искусства\\"",\n   "office_type": "company",\n   "lat": 55.7920601,\n   "lon": 37.567032,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7218460966",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.79276,\n   "lon": 37.5633281,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7218476696",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7937934,\n   "lon": 37.5644488,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/7218522988",\n   "kind": "organisation",\n   "name": "НИИЦ авиационной, космической медицины и военной эргономики Министерства обороны РФ",\n   "office_type": "research",\n   "lat": 55.794353,\n   "lon": 37.5596944,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7219020996",\n   "kind": "organisation",\n   "name": "Центральный НИИ радиоэлектронных систем",\n   "office_type": "research",\n   "lat": 55.7866532,\n   "lon": 37.634367,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7220821010",\n   "kind": "organisation",\n   "name": "Корпорация \\"Галактика\\"",\n   "office_type": "it",\n   "lat": 55.7934938,\n   "lon": 37.5597153,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7220821011",\n   "kind": "organisation",\n   "name": "Информзащита",\n   "office_type": "it",\n   "lat": 55.7934653,\n   "lon": 37.5597885,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7220821012",\n   "kind": "organisation",\n   "name": "Нептун Ко Лтд.",\n   "office_type": "company",\n   "lat": 55.7934413,\n   "lon": 37.5598556,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7221158435",\n   "kind": "organisation",\n   "name": "IT Com",\n   "office_type": "company",\n   "lat": 55.7956561,\n   "lon": 37.5622026,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7221169271",\n   "kind": "organisation",\n   "name": "Шарапова Е.П.",\n   "office_type": "notary",\n   "lat": 55.7943561,\n   "lon": 37.716307,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/7221647175",\n   "kind": "organisation",\n   "name": "Геотехдизайн",\n   "office_type": "company",\n   "lat": 55.7960053,\n   "lon": 37.5603061,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7221647176",\n   "kind": "organisation",\n   "name": "Спецпромресурс",\n   "office_type": "company",\n   "lat": 55.7959732,\n   "lon": 37.5603505,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7221681196",\n   "kind": "organisation",\n   "name": "НПО Специальных Материалов",\n   "office_type": "research",\n   "lat": 55.7965965,\n   "lon": 37.5585068,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7222120220",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты населения Савеловского района",\n   "office_type": "government",\n   "lat": 55.7973735,\n   "lon": 37.559995,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7222120249",\n   "kind": "organisation",\n   "name": "ФГУП \\"Охрана\\" Росгвардии",\n   "office_type": "company",\n   "lat": 55.7987384,\n   "lon": 37.5543822,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7227418773",\n   "kind": "organisation",\n   "name": "Bricks IT",\n   "office_type": "it",\n   "lat": 55.7236519,\n   "lon": 37.6553585,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7228749861",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6802778,\n   "lon": 37.5569773,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/7231274163",\n   "kind": "organisation",\n   "name": "ООО «Юрфлекс»",\n   "office_type": "lawyer",\n   "lat": 55.5868123,\n   "lon": 37.6015777,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7235353320",\n   "kind": "organisation",\n   "name": "Computer Support Services",\n   "office_type": "company",\n   "lat": 55.8045898,\n   "lon": 37.5615702,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7235646182",\n   "kind": "organisation",\n   "name": "Машиностроительное конструкторское бюро \\"Искра\\"",\n   "office_type": "company",\n   "lat": 55.8046081,\n   "lon": 37.5627313,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7235646184",\n   "kind": "organisation",\n   "name": "Московское ПО «Металлпластизделие» (Галантерея)",\n   "office_type": "ngo",\n   "lat": 55.8047349,\n   "lon": 37.5689441,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7235661285",\n   "kind": "organisation",\n   "name": "Металлпластизделие",\n   "office_type": "company",\n   "lat": 55.8048781,\n   "lon": 37.5688698,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7236662964",\n   "kind": "organisation",\n   "name": "Cognitive Technologies",\n   "office_type": "it",\n   "lat": 55.7907295,\n   "lon": 37.6610242,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7240911525",\n   "kind": "organisation",\n   "name": "ПСК Палекс",\n   "office_type": "company",\n   "lat": 55.657592,\n   "lon": 37.5294452,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7241703081",\n   "kind": "organisation",\n   "name": "Вега аст",\n   "office_type": "company",\n   "lat": 55.7008653,\n   "lon": 37.6227559,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7241782463",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник район Аэропорт ОДС №4",\n   "office_type": "company",\n   "lat": 55.8023116,\n   "lon": 37.5392928,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7243041495",\n   "kind": "organisation",\n   "name": "Vanilla lab",\n   "office_type": "company",\n   "lat": 55.7778882,\n   "lon": 37.6726809,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7243135534",\n   "kind": "organisation",\n   "name": "Вторлом",\n   "office_type": "company",\n   "lat": 55.8595774,\n   "lon": 37.5794435,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7247106891",\n   "kind": "organisation",\n   "name": "Галика",\n   "office_type": "company",\n   "lat": 55.7146183,\n   "lon": 37.5810182,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7247118952",\n   "kind": "organisation",\n   "name": "ТСЖ Малахит-26",\n   "office_type": "company",\n   "lat": 55.7956611,\n   "lon": 37.5660133,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7248430085",\n   "kind": "organisation",\n   "name": "LM Soft",\n   "office_type": "it",\n   "lat": 55.8047702,\n   "lon": 37.4902868,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7255673694",\n   "kind": "organisation",\n   "name": "Червов А. М.",\n   "office_type": "notary",\n   "lat": 55.82831,\n   "lon": 37.4908702,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7258009957",\n   "kind": "organisation",\n   "name": "Всероссийское Общество Охраны Памятников Истории и Культуры - Московское городское отделение",\n   "office_type": "ngo",\n   "lat": 55.7915825,\n   "lon": 37.6799137,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7258010000",\n   "kind": "organisation",\n   "name": "ФГБУ Национальный Парк Лосиный Остров",\n   "office_type": "company",\n   "lat": 55.7942888,\n   "lon": 37.6576945,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7261601229",\n   "kind": "organisation",\n   "name": "Дом для ПК",\n   "office_type": "it",\n   "lat": 55.7633842,\n   "lon": 37.5407437,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7261601230",\n   "kind": "organisation",\n   "name": "Sailing Art",\n   "office_type": "company",\n   "lat": 55.7636009,\n   "lon": 37.5408362,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/7261601231",\n   "kind": "organisation",\n   "name": "Гуманитарные технологии управления",\n   "office_type": "consulting",\n   "lat": 55.7631641,\n   "lon": 37.5406636,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/7263436085",\n   "kind": "organisation",\n   "name": "F class",\n   "office_type": "company",\n   "lat": 55.8017987,\n   "lon": 37.5846922,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7265739617",\n   "kind": "organisation",\n   "name": "Управа района Богородское",\n   "office_type": "government",\n   "lat": 55.8164684,\n   "lon": 37.6927108,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7265739618",\n   "kind": "organisation",\n   "name": "Совет депутатов муниципального округа Богородское",\n   "office_type": "government",\n   "lat": 55.8162635,\n   "lon": 37.6926156,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:45"\n  },\n  {\n   "id": "node/7265739649",\n   "kind": "organisation",\n   "name": "Россети - Управленеие кабельных сетей восточного округа",\n   "office_type": "company",\n   "lat": 55.8157287,\n   "lon": 37.6917003,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7265739650",\n   "kind": "organisation",\n   "name": "Россети 18 ремонтно-эксплуатационный район",\n   "office_type": "company",\n   "lat": 55.8156877,\n   "lon": 37.6918179,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7271204210",\n   "kind": "organisation",\n   "name": "Ренессанс страхование",\n   "office_type": "insurance",\n   "lat": 55.8599703,\n   "lon": 37.5666118,\n   "opening_hours": "Mo-Th 10:00-19:00; Fr 10:00-17:45; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/7271204253",\n   "kind": "organisation",\n   "name": "Ренессанс страхование",\n   "office_type": "insurance",\n   "lat": 55.7769411,\n   "lon": 37.6308158,\n   "opening_hours": "Mo-Fr 09:30-18:30"\n  },\n  {\n   "id": "node/7279128485",\n   "kind": "organisation",\n   "name": "Мальтийская служба помощи (Москва)",\n   "office_type": "ngo",\n   "lat": 55.8842684,\n   "lon": 37.5206828,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7291604570",\n   "kind": "organisation",\n   "name": "Приоритет",\n   "office_type": "company",\n   "lat": 55.5964615,\n   "lon": 37.7236034,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7297174129",\n   "kind": "organisation",\n   "name": "Совет ветеранов № 2",\n   "office_type": "ngo",\n   "lat": 55.6079342,\n   "lon": 37.7091417,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7297174130",\n   "kind": "organisation",\n   "name": "Архив кадровой документации ЮАО",\n   "office_type": "government",\n   "lat": 55.6063699,\n   "lon": 37.7087367,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7297623919",\n   "kind": "organisation",\n   "name": "ПАО «Мосэнергосбыт»",\n   "office_type": "company",\n   "lat": 55.7042492,\n   "lon": 37.5864071,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7299186134",\n   "kind": "organisation",\n   "name": "Роскоммунмашстрой",\n   "office_type": "company",\n   "lat": 55.7622137,\n   "lon": 37.6329401,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7302696904",\n   "kind": "organisation",\n   "name": "Районный совет ветеранов района Лефортово, первичная органищация",\n   "office_type": "ngo",\n   "lat": 55.7617247,\n   "lon": 37.7121726,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7306033928",\n   "kind": "organisation",\n   "name": "SDM Climate",\n   "office_type": "company",\n   "lat": 55.776514,\n   "lon": 37.5715539,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7309191215",\n   "kind": "organisation",\n   "name": "ЛомСервис",\n   "office_type": "company",\n   "lat": 55.8597234,\n   "lon": 37.5801328,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7309601287",\n   "kind": "organisation",\n   "name": "Вторая прокуратура по надзору за исполнением законов на особо режимных объектах Московской области",\n   "office_type": "government",\n   "lat": 55.7881538,\n   "lon": 37.5775538,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7312617354",\n   "kind": "organisation",\n   "name": "ОДС-13 ГБУ \\"Жилищник района Богородское\\"",\n   "office_type": "property_management",\n   "lat": 55.810094,\n   "lon": 37.730822,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7315432009",\n   "kind": "organisation",\n   "name": "ОДС № 3 ГБУ \\"Жилищник района Сокольники\\"",\n   "office_type": "property_management",\n   "lat": 55.7898974,\n   "lon": 37.664446,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7315492255",\n   "kind": "organisation",\n   "name": "Совет ветеранов #13 первичной организации Бауманского района",\n   "office_type": "government",\n   "lat": 55.7749414,\n   "lon": 37.7055148,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7318731403",\n   "kind": "organisation",\n   "name": "Фонд Андрея Рылькова",\n   "office_type": "ngo",\n   "lat": 55.7820551,\n   "lon": 37.7406039,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7320525077",\n   "kind": "organisation",\n   "name": "Архив кадровой документации ВАО",\n   "office_type": "government",\n   "lat": 55.78032,\n   "lon": 37.7351296,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7320525080",\n   "kind": "organisation",\n   "name": "Гидромашсервис",\n   "office_type": "company",\n   "lat": 55.7972797,\n   "lon": 37.771908,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7322162151",\n   "kind": "organisation",\n   "name": "Трансмашхолдинг",\n   "office_type": "company",\n   "lat": 55.726009,\n   "lon": 37.5758734,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7325408685",\n   "kind": "organisation",\n   "name": "Щеглов и партнёры",\n   "office_type": "lawyer",\n   "lat": 55.8006072,\n   "lon": 37.5552052,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7326254165",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7464968,\n   "lon": 37.5813893,\n   "opening_hours": "Mo-Sa 10:00-19:00"\n  },\n  {\n   "id": "node/7327883169",\n   "kind": "organisation",\n   "name": "ООО ЧОП «Страж»",\n   "office_type": "security",\n   "lat": 55.707926,\n   "lon": 37.604611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7330002514",\n   "kind": "organisation",\n   "name": "Амонт",\n   "office_type": "company",\n   "lat": 55.8180793,\n   "lon": 37.6321057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7330187914",\n   "kind": "organisation",\n   "name": "Офис продаж ЖК \\"Орехово-Борисово\\"",\n   "office_type": "estate_agent",\n   "lat": 55.6045434,\n   "lon": 37.7249211,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7333102459",\n   "kind": "organisation",\n   "name": "Порт Прокат",\n   "office_type": "company",\n   "lat": 55.863982,\n   "lon": 37.4838464,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7334712603",\n   "kind": "organisation",\n   "name": "Федеральное Космическое Агентство Роскосмос ОАО \\"ЭХО\\"",\n   "office_type": "company",\n   "lat": 55.7983448,\n   "lon": 37.554307,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7334712654",\n   "kind": "organisation",\n   "name": "Гормедтехника Департамента здравоохранения города Москвы",\n   "office_type": "company",\n   "lat": 55.7947706,\n   "lon": 37.5759219,\n   "opening_hours": "Mo-Su 06:30-12:00"\n  },\n  {\n   "id": "node/7334712665",\n   "kind": "organisation",\n   "name": "Администрация парка \\"Новослободский\\"",\n   "office_type": "company",\n   "lat": 55.7901384,\n   "lon": 37.5941031,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7335143368",\n   "kind": "organisation",\n   "name": "Территориальный центр социального обслуживания Кузьминки филиал Рязанский",\n   "office_type": "government",\n   "lat": 55.7167387,\n   "lon": 37.7897175,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7341815472",\n   "kind": "organisation",\n   "name": "Сретенка",\n   "office_type": "architect",\n   "lat": 55.7708703,\n   "lon": 37.6357905,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7341815476",\n   "kind": "organisation",\n   "name": "Agenda",\n   "office_type": "advertising_agency",\n   "lat": 55.771454,\n   "lon": 37.6366312,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/7341815477",\n   "kind": "organisation",\n   "name": "Сайт Мастер",\n   "office_type": "it",\n   "lat": 55.7717219,\n   "lon": 37.6368007,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7348263685",\n   "kind": "organisation",\n   "name": "ООО «МЭК»",\n   "office_type": "company",\n   "lat": 55.6449883,\n   "lon": 37.5995025,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7350255185",\n   "kind": "organisation",\n   "name": "Кадастр Химки",\n   "office_type": "geodesist",\n   "lat": 55.8944497,\n   "lon": 37.4492103,\n   "opening_hours": "Mo-Fr 10:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/7351150120",\n   "kind": "organisation",\n   "name": "ЖСК \\"Тагил\\"",\n   "office_type": "yes",\n   "lat": 55.8727956,\n   "lon": 37.6831631,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7357078487",\n   "kind": "organisation",\n   "name": "АО \\"Мосгаз\\" управление №1",\n   "office_type": "company",\n   "lat": 55.8029016,\n   "lon": 37.6405925,\n   "opening_hours": "Mo-Th 10:00-12:00,12:45-17:00; Fr 10:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/7360122397",\n   "kind": "organisation",\n   "name": "Представительство Правительства Калужской области при Правительстве РФ",\n   "office_type": "government",\n   "lat": 55.7441295,\n   "lon": 37.58633,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7360122436",\n   "kind": "organisation",\n   "name": "Райффайзен Капитал",\n   "office_type": "company",\n   "lat": 55.7442967,\n   "lon": 37.5845021,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7360141677",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник района Сокольники\\"",\n   "office_type": "property_management",\n   "lat": 55.7886635,\n   "lon": 37.6748014,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7360185412",\n   "kind": "organisation",\n   "name": "Вера-1",\n   "office_type": "logistics",\n   "lat": 55.8045779,\n   "lon": 37.6516244,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7360185413",\n   "kind": "organisation",\n   "name": "Баварес",\n   "office_type": "company",\n   "lat": 55.8038679,\n   "lon": 37.6509002,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7360185415",\n   "kind": "organisation",\n   "name": "ООО \\"Cleaning company\\"",\n   "office_type": "company",\n   "lat": 55.8010865,\n   "lon": 37.6464289,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7365291657",\n   "kind": "organisation",\n   "name": "МК-Логистика",\n   "office_type": "logistics",\n   "lat": 55.7711883,\n   "lon": 37.6330022,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7365291658",\n   "kind": "organisation",\n   "name": "ПриборКомплект",\n   "office_type": "company",\n   "lat": 55.7711725,\n   "lon": 37.633222,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7365291659",\n   "kind": "organisation",\n   "name": "Business Class",\n   "office_type": "company",\n   "lat": 55.7712029,\n   "lon": 37.6328224,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7365291663",\n   "kind": "organisation",\n   "name": "Альфа Реестр",\n   "office_type": "company",\n   "lat": 55.7713368,\n   "lon": 37.6333204,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7366890679",\n   "kind": "organisation",\n   "name": "Lingvoservice",\n   "office_type": "translator",\n   "lat": 55.6116414,\n   "lon": 37.7281679,\n   "opening_hours": "Mo-Fr 09:30-13:00,13:30-18:00"\n  },\n  {\n   "id": "node/7366890680",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.6116505,\n   "lon": 37.7282256,\n   "opening_hours": "Mo-Fr 09:30-13:00,13:30-18:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/7369268541",\n   "kind": "organisation",\n   "name": "Федерация вертолетного спорта России",\n   "office_type": "association",\n   "lat": 55.8232296,\n   "lon": 37.4412346,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7369268542",\n   "kind": "organisation",\n   "name": "Федерация авиации общего назначения России",\n   "office_type": "association",\n   "lat": 55.8233411,\n   "lon": 37.440961,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7369501290",\n   "kind": "organisation",\n   "name": "Express.ru",\n   "office_type": "logistics",\n   "lat": 55.8630576,\n   "lon": 37.4658385,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7370125041",\n   "kind": "organisation",\n   "name": "Русский дом",\n   "office_type": "publisher",\n   "lat": 55.7653713,\n   "lon": 37.6269507,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7373549193",\n   "kind": "organisation",\n   "name": "International Outsourcing Group",\n   "office_type": "company",\n   "lat": 55.7435803,\n   "lon": 37.5889747,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7373549255",\n   "kind": "organisation",\n   "name": "Национальный научный центр наркологии",\n   "office_type": "research",\n   "lat": 55.7443193,\n   "lon": 37.5893261,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7379039455",\n   "kind": "organisation",\n   "name": "InboxMarket",\n   "office_type": "company",\n   "lat": 55.6960258,\n   "lon": 37.6195679,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7379080792",\n   "kind": "organisation",\n   "name": "Московская Федерация Профсоюзов",\n   "office_type": "ngo",\n   "lat": 55.753998,\n   "lon": 37.5767674,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/7379529616",\n   "kind": "organisation",\n   "name": "АО Мосводоканал Район канализационной сети №10",\n   "office_type": "company",\n   "lat": 55.8026114,\n   "lon": 37.6394383,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7397131033",\n   "kind": "organisation",\n   "name": "ОДС-2",\n   "office_type": "engineer",\n   "lat": 55.7913348,\n   "lon": 37.6732443,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7397131040",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7897978,\n   "lon": 37.6781592,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7404805605",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичной организации №1 района Сокольники ВАО",\n   "office_type": "association",\n   "lat": 55.7912315,\n   "lon": 37.6752519,\n   "opening_hours": "Th 11:00-13:00; Su 11:00-13:00"\n  },\n  {\n   "id": "node/7404805606",\n   "kind": "organisation",\n   "name": "Общество инвалидов",\n   "office_type": "association",\n   "lat": 55.7912406,\n   "lon": 37.6752946,\n   "opening_hours": "We 11:00-13:00"\n  },\n  {\n   "id": "node/7410663788",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт ФГУП «Гознак»",\n   "office_type": "research",\n   "lat": 55.7180932,\n   "lon": 37.6197557,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7417184552",\n   "kind": "organisation",\n   "name": "ОДС № 854",\n   "office_type": "property_management",\n   "lat": 55.7621895,\n   "lon": 37.5899175,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7419749260",\n   "kind": "organisation",\n   "name": "Герадез",\n   "office_type": "company",\n   "lat": 55.8436611,\n   "lon": 37.528529,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7422319795",\n   "kind": "organisation",\n   "name": "ООО «ЛФБ Консалтинг»",\n   "office_type": "lawyer",\n   "lat": 55.75162,\n   "lon": 37.5440221,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7425533542",\n   "kind": "organisation",\n   "name": "Представительство губернатора Свердловской области",\n   "office_type": "government",\n   "lat": 55.7410286,\n   "lon": 37.6047098,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7451899672",\n   "kind": "organisation",\n   "name": "ООО \\"Форпост\\"",\n   "office_type": "company",\n   "lat": 55.842994,\n   "lon": 37.4412753,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7456670981",\n   "kind": "organisation",\n   "name": "ОАО Агрохиминвест",\n   "office_type": "company",\n   "lat": 55.789163,\n   "lon": 37.6646337,\n   "opening_hours": "09:00-18:00"\n  },\n  {\n   "id": "node/7456670982",\n   "kind": "organisation",\n   "name": "Российский союз предприятий и организаций химического комплекса",\n   "office_type": "company",\n   "lat": 55.7892826,\n   "lon": 37.664944,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7456687235",\n   "kind": "organisation",\n   "name": "Мировые судьи Судебных участков №№ 104. 105 Района Сокольники",\n   "office_type": "lawyer",\n   "lat": 55.7889877,\n   "lon": 37.6632516,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/7471160706",\n   "kind": "organisation",\n   "name": "Master Content",\n   "office_type": "company",\n   "lat": 55.876207,\n   "lon": 37.5949291,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa-Su 11:00-18:00"\n  },\n  {\n   "id": "node/7490448159",\n   "kind": "organisation",\n   "name": "ООО \\"РФК\\"",\n   "office_type": "company",\n   "lat": 55.7154748,\n   "lon": 37.522528,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7500915829",\n   "kind": "organisation",\n   "name": "ООО Стройресурс Инновации",\n   "office_type": "company",\n   "lat": 55.7848813,\n   "lon": 37.6770476,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7502939816",\n   "kind": "organisation",\n   "name": "Инком-Недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.6148401,\n   "lon": 37.7139106,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/7505927613",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.602983,\n   "lon": 37.529268,\n   "opening_hours": "Mo-Fr 09:30-13:00,13:30-18:00;Sa 10:00-13:00,13:00-15:00"\n  },\n  {\n   "id": "node/7505927614",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.8142545,\n   "lon": 37.5760075,\n   "opening_hours": "Mo-Fr 10:00-13:00,14:00-19:00"\n  },\n  {\n   "id": "node/7505927679",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.741445,\n   "lon": 37.632287,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/7505927680",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.714663,\n   "lon": 37.790893,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/7505927785",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.808594,\n   "lon": 37.639312,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/7507039882",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.7521874,\n   "lon": 37.5013697,\n   "opening_hours": "Mo-Fr 09:00-20:00;Sa 10:00-15:00"\n  },\n  {\n   "id": "node/7507039884",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.7210739,\n   "lon": 37.8152418,\n   "opening_hours": "Mo-Fr 10:00-13:00,13:30-18:30"\n  },\n  {\n   "id": "node/7507064085",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.8503184,\n   "lon": 37.5206709,\n   "opening_hours": "Mo-Fr 09:30-13:00,14:00-18:30"\n  },\n  {\n   "id": "node/7507064184",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.686061,\n   "lon": 37.541243,\n   "opening_hours": "Mo-Fr 10:00-18:00;Sa 10:00-15:00"\n  },\n  {\n   "id": "node/7507064185",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.865152,\n   "lon": 37.472081,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/7507064186",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.825108,\n   "lon": 37.571228,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/7507064189",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.753124,\n   "lon": 37.779682,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/7507064190",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.8695684,\n   "lon": 37.6669264,\n   "opening_hours": "Mo-Fr 11:00-18:00; Sa 11:00-15:00"\n  },\n  {\n   "id": "node/7507064192",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.7641198,\n   "lon": 37.7344161,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/7537400926",\n   "kind": "organisation",\n   "name": "NewSpire",\n   "office_type": "advertising_agency",\n   "lat": 55.8439598,\n   "lon": 37.5262108,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/7548326385",\n   "kind": "organisation",\n   "name": "ООО «ТехноНИКОЛЬ-Строительные Системы»",\n   "office_type": "company",\n   "lat": 55.785551,\n   "lon": 37.6312579,\n   "opening_hours": "Mo-Fr 08:00-19:00; Sa-Su 09:00-18:00"\n  },\n  {\n   "id": "node/7551278125",\n   "kind": "organisation",\n   "name": "ТрансКонтейнер",\n   "office_type": "logistics",\n   "lat": 55.7729267,\n   "lon": 37.5997156,\n   "opening_hours": "Fr 09:00-16:45; Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/7551278126",\n   "kind": "organisation",\n   "name": "ТрансКонтейнер",\n   "office_type": "logistics",\n   "lat": 55.7764961,\n   "lon": 37.6522131,\n   "opening_hours": "Fr 08:00-15:45; Mo-Th 08:00-17:00"\n  },\n  {\n   "id": "node/7556452185",\n   "kind": "organisation",\n   "name": "Новотранс",\n   "office_type": "company",\n   "lat": 55.7952792,\n   "lon": 37.5469249,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7556725845",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7831237,\n   "lon": 37.6595468,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7559993085",\n   "kind": "organisation",\n   "name": "Гет Телеком",\n   "office_type": "telecommunication",\n   "lat": 55.7549701,\n   "lon": 37.5570277,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/7562039998",\n   "kind": "organisation",\n   "name": "Совет ветеранов №4",\n   "office_type": "ngo",\n   "lat": 55.7978928,\n   "lon": 37.6873454,\n   "opening_hours": "Tu 12:00-14:00"\n  },\n  {\n   "id": "node/7562040005",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7946836,\n   "lon": 37.6893147,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/7562040006",\n   "kind": "organisation",\n   "name": "ОДС 6",\n   "office_type": "property_management",\n   "lat": 55.7942796,\n   "lon": 37.6903882,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7562972789",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8965072,\n   "lon": 37.5548903,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/7580137314",\n   "kind": "organisation",\n   "name": "Otto Group Russia",\n   "office_type": "company",\n   "lat": 55.7975209,\n   "lon": 37.578392,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7593315185",\n   "kind": "organisation",\n   "name": "Центральный банк РФ",\n   "office_type": "government",\n   "lat": 55.7241633,\n   "lon": 37.6053222,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7593315186",\n   "kind": "organisation",\n   "name": "Российское Технологическое Общество",\n   "office_type": "government",\n   "lat": 55.724124,\n   "lon": 37.6052815,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7593357385",\n   "kind": "organisation",\n   "name": "Центральный банк РФ",\n   "office_type": "government",\n   "lat": 55.726889,\n   "lon": 37.6093367,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7596820418",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.719045,\n   "lon": 37.608709,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7605890791",\n   "kind": "organisation",\n   "name": "Главдоставка",\n   "office_type": "logistics",\n   "lat": 55.6589702,\n   "lon": 37.7199563,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/7605890846",\n   "kind": "organisation",\n   "name": "Главдоставка",\n   "office_type": "logistics",\n   "lat": 55.6773725,\n   "lon": 37.7810222,\n   "opening_hours": "Mo-Sa 08:00-17:00"\n  },\n  {\n   "id": "node/7605890851",\n   "kind": "organisation",\n   "name": "Главдоставка",\n   "office_type": "logistics",\n   "lat": 55.6211112,\n   "lon": 37.7850938,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/7608214787",\n   "kind": "organisation",\n   "name": "Инженерно - техническая служба",\n   "office_type": "engineer",\n   "lat": 55.7978399,\n   "lon": 37.6709699,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7609624693",\n   "kind": "organisation",\n   "name": "Мосгоруслуга",\n   "office_type": "company",\n   "lat": 55.7374738,\n   "lon": 37.6586396,\n   "opening_hours": "Mo-Th 09:00-12:30; Mo-Th 13:30-18:00; Fr 09:00-12:30; Fr 13:30-17:00"\n  },\n  {\n   "id": "node/7620684102",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7522362,\n   "lon": 37.7164158,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7620684104",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.752426,\n   "lon": 37.7162925,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7623371785",\n   "kind": "organisation",\n   "name": "Statex",\n   "office_type": "company",\n   "lat": 55.7810989,\n   "lon": 37.5166176,\n   "opening_hours": "Mo-Fr 09:00-17:30"\n  },\n  {\n   "id": "node/7625250491",\n   "kind": "organisation",\n   "name": "ООО \\"Вода Отечества\\"",\n   "office_type": "company",\n   "lat": 55.7942244,\n   "lon": 37.6920879,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625250492",\n   "kind": "organisation",\n   "name": "ФКУ Росавтодор. Узел связи и автоматизации",\n   "office_type": "company",\n   "lat": 55.794037,\n   "lon": 37.6921399,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625250498",\n   "kind": "organisation",\n   "name": "Инженерная служба района Сокольники",\n   "office_type": "property_management",\n   "lat": 55.7912173,\n   "lon": 37.6890528,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625250528",\n   "kind": "organisation",\n   "name": "Бизнес центр \\"Фортуна\\"",\n   "office_type": "company",\n   "lat": 55.7923379,\n   "lon": 37.6950695,\n   "opening_hours": "Mo-Su 08:00-21:00"\n  },\n  {\n   "id": "node/7625250530",\n   "kind": "organisation",\n   "name": "Прокуратура ВАО Москвы",\n   "office_type": "government",\n   "lat": 55.7906377,\n   "lon": 37.6832147,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625250531",\n   "kind": "organisation",\n   "name": "Департамент городского имущества города Москвы ВАО",\n   "office_type": "government",\n   "lat": 55.7906646,\n   "lon": 37.6832906,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625250532",\n   "kind": "organisation",\n   "name": "Отдел государственной статистики ВАО",\n   "office_type": "government",\n   "lat": 55.7906942,\n   "lon": 37.6833805,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625250535",\n   "kind": "organisation",\n   "name": "Управление по ВАО главного управления МЧС России по городу Москве",\n   "office_type": "government",\n   "lat": 55.7893398,\n   "lon": 37.6818746,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625250536",\n   "kind": "organisation",\n   "name": "Федеральная палата пожарно-спасательной отрасли",\n   "office_type": "government",\n   "lat": 55.7886732,\n   "lon": 37.683421,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625741852",\n   "kind": "organisation",\n   "name": "Совет депутатов муниципального округа Сокольники",\n   "office_type": "government",\n   "lat": 55.7884917,\n   "lon": 37.6830169,\n   "opening_hours": "Th 16:00-18:00"\n  },\n  {\n   "id": "node/7625741853",\n   "kind": "organisation",\n   "name": "НПО \\"Пульс\\"",\n   "office_type": "company",\n   "lat": 55.7884843,\n   "lon": 37.683518,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625741854",\n   "kind": "organisation",\n   "name": "ВАО ГБУ Жилищник",\n   "office_type": "property_management",\n   "lat": 55.7883136,\n   "lon": 37.6825464,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625741857",\n   "kind": "organisation",\n   "name": "Тихорецкий машиностроительный завод им. В. В. Воровского Московское представительство",\n   "office_type": "company",\n   "lat": 55.7871065,\n   "lon": 37.6847994,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625741858",\n   "kind": "organisation",\n   "name": "ООО \\"Росток\\"",\n   "office_type": "company",\n   "lat": 55.7870884,\n   "lon": 37.6848695,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625741859",\n   "kind": "organisation",\n   "name": "АНО Центр сертификации изделий медицинской оптики",\n   "office_type": "company",\n   "lat": 55.787071,\n   "lon": 37.684929,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625741860",\n   "kind": "organisation",\n   "name": "УЗТС - Ульяновский завод тяжелых и уникальных станков (Московское представительство)",\n   "office_type": "company",\n   "lat": 55.7870526,\n   "lon": 37.6850084,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625741864",\n   "kind": "organisation",\n   "name": "ОДС 4",\n   "office_type": "property_management",\n   "lat": 55.7853087,\n   "lon": 37.6874119,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625741865",\n   "kind": "organisation",\n   "name": "Совет ветеранов войны и труда №3",\n   "office_type": "ngo",\n   "lat": 55.7851892,\n   "lon": 37.6876594,\n   "opening_hours": "Th 11:00-13:00"\n  },\n  {\n   "id": "node/7625741866",\n   "kind": "organisation",\n   "name": "Центральная энергетическая таможня",\n   "office_type": "company",\n   "lat": 55.7849086,\n   "lon": 37.6880809,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7625741873",\n   "kind": "organisation",\n   "name": "Мировой судья судебного участка №297 района Соколиная Гора",\n   "office_type": "lawyer",\n   "lat": 55.7837675,\n   "lon": 37.6963087,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7634538194",\n   "kind": "organisation",\n   "name": "Государственное казенное учреждение г. Москвы \\"Инженерная служба района Южное Тушино\\"",\n   "office_type": "property_management",\n   "lat": 55.8378209,\n   "lon": 37.4404984,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7640415419",\n   "kind": "organisation",\n   "name": "Сургутнефтегаз",\n   "office_type": "company",\n   "lat": 55.7658693,\n   "lon": 37.6394284,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7644067802",\n   "kind": "organisation",\n   "name": "Санмейт",\n   "office_type": "company",\n   "lat": 55.7501859,\n   "lon": 37.787438,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7644571095",\n   "kind": "organisation",\n   "name": "Гольфстрим",\n   "office_type": "company",\n   "lat": 55.8024885,\n   "lon": 37.5838369,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7650141378",\n   "kind": "organisation",\n   "name": "Северсталь Менеджмент",\n   "office_type": "company",\n   "lat": 55.817903,\n   "lon": 37.5211859,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7653056438",\n   "kind": "organisation",\n   "name": "Импресарио",\n   "office_type": "theatre",\n   "lat": 55.7565348,\n   "lon": 37.6435372,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7653798203",\n   "kind": "organisation",\n   "name": "Иви",\n   "office_type": "it",\n   "lat": 55.8034669,\n   "lon": 37.5843063,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/7653798209",\n   "kind": "organisation",\n   "name": "Национальное объединение внутренних аудиторов и контролеров",\n   "office_type": "association",\n   "lat": 55.804097,\n   "lon": 37.584309,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7660544845",\n   "kind": "organisation",\n   "name": "адвокат Соловьев Алексей Александрович",\n   "office_type": "lawyer",\n   "lat": 55.7777744,\n   "lon": 37.6034572,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7661714953",\n   "kind": "organisation",\n   "name": "Нотариус Емельянова Г. В.",\n   "office_type": "notary",\n   "lat": 55.7743908,\n   "lon": 37.6306587,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7666981652",\n   "kind": "organisation",\n   "name": "СоюзДорЭнерго",\n   "office_type": "association",\n   "lat": 55.7731085,\n   "lon": 37.6188344,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7670786758",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.887097,\n   "lon": 37.5233343,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7670786776",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.7209379,\n   "lon": 37.4519956,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/7673081176",\n   "kind": "organisation",\n   "name": "Фолдин",\n   "office_type": "company",\n   "lat": 55.8003134,\n   "lon": 37.6394676,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7682427485",\n   "kind": "organisation",\n   "name": "Нотариус",\n   "office_type": "notary",\n   "lat": 55.674044,\n   "lon": 37.7494813,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/7682601860",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты Лефортово",\n   "office_type": "government",\n   "lat": 55.7543394,\n   "lon": 37.7100573,\n   "opening_hours": "Mo 11:00-20:00; Tu-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/7687041235",\n   "kind": "organisation",\n   "name": "Techtrends HTC-ЭКО",\n   "office_type": "company",\n   "lat": 55.8001525,\n   "lon": 37.5924495,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7689311331",\n   "kind": "organisation",\n   "name": "Администрация северного морского пути",\n   "office_type": "government",\n   "lat": 55.7458797,\n   "lon": 37.6744688,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7692198142",\n   "kind": "organisation",\n   "name": "ОДС № 9 ГБУ \\"Жилищник Таганского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7479985,\n   "lon": 37.6603402,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7693025927",\n   "kind": "organisation",\n   "name": "АО «Российские космические системы»",\n   "office_type": "company",\n   "lat": 55.745657,\n   "lon": 37.7225366,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7695363621",\n   "kind": "organisation",\n   "name": "Офис продаж ЖК \\"Метрополия\\"",\n   "office_type": "estate_agent",\n   "lat": 55.7192565,\n   "lon": 37.6935312,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7702286577",\n   "kind": "organisation",\n   "name": "Контрольно-счетная палата Московской области",\n   "office_type": "government",\n   "lat": 55.7676702,\n   "lon": 37.6319128,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7707655679",\n   "kind": "organisation",\n   "name": "Буки веди",\n   "office_type": "company",\n   "lat": 55.720249,\n   "lon": 37.6306039,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7711797414",\n   "kind": "organisation",\n   "name": "ОДС № 20 ГБУ \\"Жилищник Басманного района\\"",\n   "office_type": "property_management",\n   "lat": 55.7574515,\n   "lon": 37.6503624,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7713736224",\n   "kind": "organisation",\n   "name": "Shtaine",\n   "office_type": "company",\n   "lat": 55.7319853,\n   "lon": 37.682042,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7713774521",\n   "kind": "organisation",\n   "name": "Стройкомпозит-Н",\n   "office_type": "company",\n   "lat": 55.6247745,\n   "lon": 37.6757681,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7713961172",\n   "kind": "organisation",\n   "name": "Агентство по распространению зарубежных изданий",\n   "office_type": "company",\n   "lat": 55.7817993,\n   "lon": 37.6394085,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7719831024",\n   "kind": "organisation",\n   "name": "ГУП Моссвет",\n   "office_type": "company",\n   "lat": 55.8020878,\n   "lon": 37.6401104,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7719831025",\n   "kind": "organisation",\n   "name": "Страховая группа \\"Спасские ворота\\"",\n   "office_type": "insurance",\n   "lat": 55.8020489,\n   "lon": 37.6411451,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7721223563",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "coworking",\n   "lat": 55.7975164,\n   "lon": 37.6321,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7721223566",\n   "kind": "organisation",\n   "name": "Торгметрострой ЗАО \\"Томес\\"",\n   "office_type": "company",\n   "lat": 55.797906,\n   "lon": 37.6318692,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7724419593",\n   "kind": "organisation",\n   "name": "ООО «ДекоЛайф»",\n   "office_type": "company",\n   "lat": 55.7317276,\n   "lon": 37.5595055,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7730320700",\n   "kind": "organisation",\n   "name": "АО НИИ \\"Кулон\\"",\n   "office_type": "company",\n   "lat": 55.806495,\n   "lon": 37.6234779,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7730320706",\n   "kind": "organisation",\n   "name": "ECP-Logistic",\n   "office_type": "logistics",\n   "lat": 55.8055604,\n   "lon": 37.6228051,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7730780478",\n   "kind": "organisation",\n   "name": "Siegenia",\n   "office_type": "company",\n   "lat": 55.7862776,\n   "lon": 37.7088389,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7731542342",\n   "kind": "organisation",\n   "name": "ООО «Точдеталь»",\n   "office_type": "company",\n   "lat": 55.682853,\n   "lon": 37.6622573,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7733241429",\n   "kind": "organisation",\n   "name": "ТИСО",\n   "office_type": "company",\n   "lat": 55.8064564,\n   "lon": 37.5940368,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7734075229",\n   "kind": "organisation",\n   "name": "Новейшие технологии связи",\n   "office_type": "telecommunication",\n   "lat": 55.7986619,\n   "lon": 37.6959739,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7734075230",\n   "kind": "organisation",\n   "name": "ООО Управляющая кампания \\"Изумрудный Город\\"",\n   "office_type": "property_management",\n   "lat": 55.7985248,\n   "lon": 37.6960186,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7734075236",\n   "kind": "organisation",\n   "name": "ОДС-5",\n   "office_type": "property_management",\n   "lat": 55.7956173,\n   "lon": 37.694552,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7734075238",\n   "kind": "organisation",\n   "name": "ООО \\"ОЛиКС-строй\\"",\n   "office_type": "company",\n   "lat": 55.7953425,\n   "lon": 37.6976148,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7734616142",\n   "kind": "organisation",\n   "name": "First National Consulting Group",\n   "office_type": "company",\n   "lat": 55.7589011,\n   "lon": 37.6646036,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7737843873",\n   "kind": "organisation",\n   "name": "Архитектурная Мастерская Николая Лызлова",\n   "office_type": "architect",\n   "lat": 55.7731985,\n   "lon": 37.5974021,\n   "opening_hours": "Mo-Fr 09:00-18:30; PH closed"\n  },\n  {\n   "id": "node/7739414950",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "educational_institution",\n   "lat": 55.7926143,\n   "lon": 37.7330855,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7740432919",\n   "kind": "organisation",\n   "name": "ООО \\"Энвибокс\\"",\n   "office_type": "it",\n   "lat": 55.803299,\n   "lon": 37.5554619,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7741235276",\n   "kind": "organisation",\n   "name": "Дирекция по ремонту тягового подвижного состава",\n   "office_type": "company",\n   "lat": 55.7721499,\n   "lon": 37.650938,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7741261510",\n   "kind": "organisation",\n   "name": "Трансэнерго",\n   "office_type": "company",\n   "lat": 55.772292,\n   "lon": 37.6497306,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7741269179",\n   "kind": "organisation",\n   "name": "Желдоручёт",\n   "office_type": "company",\n   "lat": 55.7720335,\n   "lon": 37.649911,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7741338485",\n   "kind": "organisation",\n   "name": "PlayVista",\n   "office_type": "advertising_agency",\n   "lat": 55.8174687,\n   "lon": 37.6554493,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/7745532993",\n   "kind": "organisation",\n   "name": "Пионер-Сервис",\n   "office_type": "property_management",\n   "lat": 55.847266,\n   "lon": 37.6315746,\n   "opening_hours": "Tu 09:00-13:00,14:00-20:00; We,Th 09:00-13:00,14:00-18:00; Fr 09:00-13:00,14:00-16:30"\n  },\n  {\n   "id": "node/7753554192",\n   "kind": "organisation",\n   "name": "Бюро судебных экспертиз и независимой оценки",\n   "office_type": "association",\n   "lat": 55.7706184,\n   "lon": 37.6554412,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7755681969",\n   "kind": "organisation",\n   "name": "ОАО \\"Манотомь\\"",\n   "office_type": "company",\n   "lat": 55.7970592,\n   "lon": 37.6984355,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7755681970",\n   "kind": "organisation",\n   "name": "Манометр сервис",\n   "office_type": "company",\n   "lat": 55.7970085,\n   "lon": 37.6984975,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7755681971",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник. Единая диспетчерская служба",\n   "office_type": "property_management",\n   "lat": 55.7954353,\n   "lon": 37.6996636,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7755681972",\n   "kind": "organisation",\n   "name": "Печатный двор",\n   "office_type": "typography",\n   "lat": 55.795473,\n   "lon": 37.6996033,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7755681976",\n   "kind": "organisation",\n   "name": "Межрайонная инспекция гостехнадзора города Москвы №3",\n   "office_type": "property_management",\n   "lat": 55.7958417,\n   "lon": 37.6983064,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7755709795",\n   "kind": "organisation",\n   "name": "Бизнес центр \\"На Стромынке\\"",\n   "office_type": "company",\n   "lat": 55.7954897,\n   "lon": 37.6979513,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/7755709796",\n   "kind": "organisation",\n   "name": "Территориальное агентство по развитию предпринимательства по ВАО",\n   "office_type": "lawyer",\n   "lat": 55.7954121,\n   "lon": 37.6977672,\n   "opening_hours": "Mo-Su 10:00-17:00"\n  },\n  {\n   "id": "node/7755873582",\n   "kind": "organisation",\n   "name": "Боненкамп",\n   "office_type": "company",\n   "lat": 55.6188824,\n   "lon": 37.4455945,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7758302986",\n   "kind": "organisation",\n   "name": "Фонд Елены Образцовой",\n   "office_type": "association",\n   "lat": 55.7659764,\n   "lon": 37.662726,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7760351735",\n   "kind": "organisation",\n   "name": "ТвоёКино",\n   "office_type": "company",\n   "lat": 55.715932,\n   "lon": 37.646558,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7760843030",\n   "kind": "organisation",\n   "name": "Decolife",\n   "office_type": "company",\n   "lat": 55.7316424,\n   "lon": 37.5604122,\n   "opening_hours": "08:00-20:00"\n  },\n  {\n   "id": "node/7765019061",\n   "kind": "organisation",\n   "name": "Инс-брокер",\n   "office_type": "insurance",\n   "lat": 55.8642154,\n   "lon": 37.6515451,\n   "opening_hours": "10:00-20:00"\n  },\n  {\n   "id": "node/7770963669",\n   "kind": "organisation",\n   "name": "Ночлежка",\n   "office_type": "ngo",\n   "lat": 55.7875715,\n   "lon": 37.5862395,\n   "opening_hours": "Mo-Fr 10:00-14:00, 15:00-17:00"\n  },\n  {\n   "id": "node/7772179502",\n   "kind": "organisation",\n   "name": "Универмаг Мск",\n   "office_type": "company",\n   "lat": 55.7838061,\n   "lon": 37.6565882,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7780981883",\n   "kind": "organisation",\n   "name": "Брандтрейд",\n   "office_type": "company",\n   "lat": 55.7581181,\n   "lon": 37.7333293,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/7781680618",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7786381,\n   "lon": 37.6518805,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/7785208860",\n   "kind": "organisation",\n   "name": "Всё для саймов",\n   "office_type": "409",\n   "lat": 55.6212296,\n   "lon": 37.7455169,\n   "opening_hours": "Mo-Su 11:00-19:00"\n  },\n  {\n   "id": "node/7787668485",\n   "kind": "organisation",\n   "name": "Фонд \\"Небесный мост\\"",\n   "office_type": "ngo",\n   "lat": 55.7578819,\n   "lon": 37.6052689,\n   "opening_hours": "Mo-Fr 11:00-19:00"\n  },\n  {\n   "id": "node/7791040015",\n   "kind": "organisation",\n   "name": "Центральный банк Российской Федерации",\n   "office_type": "bank",\n   "lat": 55.7194599,\n   "lon": 37.6150979,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7791424886",\n   "kind": "organisation",\n   "name": "Посольство Боснии и Герцеговины",\n   "office_type": "diplomatic",\n   "lat": 55.7212526,\n   "lon": 37.5154405,\n   "opening_hours": "Mo-Fr 10:00-14:00"\n  },\n  {\n   "id": "node/7791472085",\n   "kind": "organisation",\n   "name": "Посольство республики Руанда",\n   "office_type": "diplomatic",\n   "lat": 55.721327,\n   "lon": 37.5153155,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7791814831",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.811917,\n   "lon": 37.8040168,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/7798145532",\n   "kind": "organisation",\n   "name": "Окей полиграфия",\n   "office_type": "company",\n   "lat": 55.7880173,\n   "lon": 37.6622473,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7810650305",\n   "kind": "organisation",\n   "name": "ОДС № 1 ГБУ \\"Жилищник района Сокол\\"",\n   "office_type": "property_management",\n   "lat": 55.7951371,\n   "lon": 37.5159033,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7816508277",\n   "kind": "organisation",\n   "name": "Роскосмос",\n   "office_type": "government",\n   "lat": 55.7836254,\n   "lon": 37.6303879,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7820734585",\n   "kind": "organisation",\n   "name": "Oxe Capital",\n   "office_type": "estate_agent",\n   "lat": 55.7390787,\n   "lon": 37.6007379,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7820805985",\n   "kind": "organisation",\n   "name": "Европейский банк развития",\n   "office_type": "bank",\n   "lat": 55.7393958,\n   "lon": 37.604773,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7826222677",\n   "kind": "organisation",\n   "name": "Мезон",\n   "office_type": "estate_agent",\n   "lat": 55.7617202,\n   "lon": 37.5925434,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/7826287613",\n   "kind": "organisation",\n   "name": "Резиденция посла Алжира",\n   "office_type": "diplomatic",\n   "lat": 55.760611,\n   "lon": 37.592833,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7831814506",\n   "kind": "organisation",\n   "name": "Amorim Flooring Rus",\n   "office_type": "company",\n   "lat": 55.752977,\n   "lon": 37.6646224,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7832365441",\n   "kind": "organisation",\n   "name": "Biomag",\n   "office_type": "water_utility",\n   "lat": 55.7096636,\n   "lon": 37.6164633,\n   "opening_hours": "Mo-Su 08:00-19:00"\n  },\n  {\n   "id": "node/7832365998",\n   "kind": "organisation",\n   "name": "NetDolgov.org",\n   "office_type": "company",\n   "lat": 55.777694,\n   "lon": 37.6490039,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa 11:00-18:00"\n  },\n  {\n   "id": "node/7832400974",\n   "kind": "organisation",\n   "name": "Альфавита",\n   "office_type": "company",\n   "lat": 55.7441971,\n   "lon": 37.627278,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/7832400978",\n   "kind": "organisation",\n   "name": "MrCable",\n   "office_type": "telecommunication",\n   "lat": 55.822789,\n   "lon": 37.6198367,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7839198261",\n   "kind": "organisation",\n   "name": "ДМК Пресс",\n   "office_type": "company",\n   "lat": 55.6751475,\n   "lon": 37.6618444,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/7843095061",\n   "kind": "organisation",\n   "name": "Мистер Бош",\n   "office_type": "company",\n   "lat": 55.7859688,\n   "lon": 37.6604805,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7844583131",\n   "kind": "organisation",\n   "name": "Союз маркшейдеров России",\n   "office_type": "ngo",\n   "lat": 55.7654641,\n   "lon": 37.6640752,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7844583132",\n   "kind": "organisation",\n   "name": "Союз реставраторов России",\n   "office_type": "ngo",\n   "lat": 55.7658595,\n   "lon": 37.6628856,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7844583133",\n   "kind": "organisation",\n   "name": "Центр медицинской инспекции",\n   "office_type": "government",\n   "lat": 55.7668531,\n   "lon": 37.6544956,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7845490012",\n   "kind": "organisation",\n   "name": "MrArito",\n   "office_type": "company",\n   "lat": 55.7861252,\n   "lon": 37.6604233,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7845823451",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник",\n   "office_type": "property_management",\n   "lat": 55.8630892,\n   "lon": 37.5403501,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7846119186",\n   "kind": "organisation",\n   "name": "Корпорация Ремонтов",\n   "office_type": "company",\n   "lat": 55.8120372,\n   "lon": 37.636468,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7849926089",\n   "kind": "organisation",\n   "name": "Росстин",\n   "office_type": "company",\n   "lat": 55.6836198,\n   "lon": 37.5303912,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7849926091",\n   "kind": "organisation",\n   "name": "СтепИнформ",\n   "office_type": "company",\n   "lat": 55.6840039,\n   "lon": 37.5311208,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7849926123",\n   "kind": "organisation",\n   "name": "ШатлАвиа",\n   "office_type": "company",\n   "lat": 55.6862969,\n   "lon": 37.5246499,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7853544151",\n   "kind": "organisation",\n   "name": "АльфаСтрахование",\n   "office_type": "insurance",\n   "lat": 55.8606432,\n   "lon": 37.483191,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7853544164",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.8608735,\n   "lon": 37.4815562,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7853544170",\n   "kind": "organisation",\n   "name": "ВСК",\n   "office_type": "insurance",\n   "lat": 55.8570217,\n   "lon": 37.4797397,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7865956906",\n   "kind": "organisation",\n   "name": "Лексгарант",\n   "office_type": "insurance",\n   "lat": 55.7591569,\n   "lon": 37.635629,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7866004883",\n   "kind": "organisation",\n   "name": "Инженерная служба ГБУ \\"Жилищник района Дорогомилово\\"",\n   "office_type": "property_management",\n   "lat": 55.7382307,\n   "lon": 37.5216258,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7866619115",\n   "kind": "organisation",\n   "name": "ОДС",\n   "office_type": "property_management",\n   "lat": 55.7135713,\n   "lon": 37.6062965,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7872134069",\n   "kind": "organisation",\n   "name": "Кредит консалтинг",\n   "office_type": "financial",\n   "lat": 55.6147142,\n   "lon": 37.6045862,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7875388134",\n   "kind": "organisation",\n   "name": "Comeforta",\n   "office_type": "company",\n   "lat": 55.8210636,\n   "lon": 37.7298445,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7876663231",\n   "kind": "organisation",\n   "name": "Банкрот-консалтинг",\n   "office_type": "financial_advisor",\n   "lat": 55.7311576,\n   "lon": 37.5838357,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7876748090",\n   "kind": "organisation",\n   "name": "ООО Бюро Экос",\n   "office_type": "company",\n   "lat": 55.7318447,\n   "lon": 37.5818913,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/7876927701",\n   "kind": "organisation",\n   "name": "Офисный центр П13",\n   "office_type": "yes",\n   "lat": 55.7290881,\n   "lon": 37.5703318,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7881502277",\n   "kind": "organisation",\n   "name": "TUI",\n   "office_type": "travel_agent",\n   "lat": 55.77168,\n   "lon": 37.6337643,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 11:00-17:00"\n  },\n  {\n   "id": "node/7881519527",\n   "kind": "organisation",\n   "name": "Аудит Консалт XXI",\n   "office_type": "company",\n   "lat": 55.7700981,\n   "lon": 37.6361216,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/7882094501",\n   "kind": "organisation",\n   "name": "АО \\"АФК \\"Черноземье\\"",\n   "office_type": "company",\n   "lat": 55.7695045,\n   "lon": 37.6292749,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/7883089058",\n   "kind": "organisation",\n   "name": "Яндекс Про Центр для водителей",\n   "office_type": "educational_institution",\n   "lat": 55.8144571,\n   "lon": 37.6046317,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/7884512730",\n   "kind": "organisation",\n   "name": "Gogol-school",\n   "office_type": "educational_institution",\n   "lat": 55.7622672,\n   "lon": 37.6615462,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7884817885",\n   "kind": "organisation",\n   "name": "Йота",\n   "office_type": "telecommunication",\n   "lat": 55.5845129,\n   "lon": 37.7251915,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/7887115185",\n   "kind": "organisation",\n   "name": "Йота",\n   "office_type": "telecommunication",\n   "lat": 55.6585385,\n   "lon": 37.741451,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/7887526985",\n   "kind": "organisation",\n   "name": "Йота",\n   "office_type": "telecommunication",\n   "lat": 55.6409389,\n   "lon": 37.5310795,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/7889363349",\n   "kind": "organisation",\n   "name": "Рустест-М",\n   "office_type": "company",\n   "lat": 55.7447345,\n   "lon": 37.4942282,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7889480281",\n   "kind": "organisation",\n   "name": "ГБУ \\"Озеленение\\"",\n   "office_type": "company",\n   "lat": 55.8174532,\n   "lon": 37.6131141,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7892314786",\n   "kind": "organisation",\n   "name": "Йота",\n   "office_type": "telecommunication",\n   "lat": 55.8976184,\n   "lon": 37.6659085,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/7892375645",\n   "kind": "organisation",\n   "name": "Росреестр",\n   "office_type": "government",\n   "lat": 55.8128727,\n   "lon": 37.5760826,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7896036320",\n   "kind": "organisation",\n   "name": "ОАО\\"НПП\\"Радий\\"",\n   "office_type": "company",\n   "lat": 55.811163,\n   "lon": 37.5212831,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7896450512",\n   "kind": "organisation",\n   "name": "ДЭП",\n   "office_type": "company",\n   "lat": 55.6125452,\n   "lon": 37.6145129,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7908836992",\n   "kind": "organisation",\n   "name": "Научный центр экспертизы средств медицинского применения",\n   "office_type": "government",\n   "lat": 55.8045869,\n   "lon": 37.491242,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7914319811",\n   "kind": "organisation",\n   "name": "Моспром",\n   "office_type": "association",\n   "lat": 55.7545785,\n   "lon": 37.6067698,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7922165789",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7827948,\n   "lon": 37.6644018,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/7926383484",\n   "kind": "organisation",\n   "name": "Пенсионный фонд РФ",\n   "office_type": "government",\n   "lat": 55.81698,\n   "lon": 37.8139329,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7935652802",\n   "kind": "organisation",\n   "name": "Отделение по вопросам миграции ОМВД России по району Якиманка г. Москвы",\n   "office_type": "government",\n   "lat": 55.7364005,\n   "lon": 37.6198804,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7942483224",\n   "kind": "organisation",\n   "name": "Rocketdata",\n   "office_type": "company",\n   "lat": 55.7790442,\n   "lon": 37.6335295,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/7942484116",\n   "kind": "organisation",\n   "name": "Партия Закона",\n   "office_type": "lawyer",\n   "lat": 55.739355,\n   "lon": 37.614136,\n   "opening_hours": "Mo-Fr 09:00-18:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/7942484117",\n   "kind": "organisation",\n   "name": "Московская юридическая компания",\n   "office_type": "lawyer",\n   "lat": 55.7441314,\n   "lon": 37.5417182,\n   "opening_hours": "Mo-Fr 10:00-21:00;Sa-Su 10:00-19:00"\n  },\n  {\n   "id": "node/7942484136",\n   "kind": "organisation",\n   "name": "Полизнак",\n   "office_type": "company",\n   "lat": 55.712527,\n   "lon": 37.686823,\n   "opening_hours": "Mo-Fr 10:00-19:00;Sa 11:00-17:00"\n  },\n  {\n   "id": "node/7942486814",\n   "kind": "organisation",\n   "name": "Династия Прайд",\n   "office_type": "lawyer",\n   "lat": 55.7441314,\n   "lon": 37.5416645,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-19:00"\n  },\n  {\n   "id": "node/7944140393",\n   "kind": "organisation",\n   "name": "Медиана-фильтр",\n   "office_type": "company",\n   "lat": 55.7863878,\n   "lon": 37.7184704,\n   "opening_hours": "Mo-Su 09:00-17:00"\n  },\n  {\n   "id": "node/7948226512",\n   "kind": "organisation",\n   "name": "ЗАО Трест Мосремстроймонтаж",\n   "office_type": "company",\n   "lat": 55.796026,\n   "lon": 37.7071702,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7948226513",\n   "kind": "organisation",\n   "name": "АО ЦНИИ \\"Буревестник\\" Московский филиал",\n   "office_type": "company",\n   "lat": 55.7961089,\n   "lon": 37.7071635,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7962093883",\n   "kind": "organisation",\n   "name": "ГУП СППМ",\n   "office_type": "company",\n   "lat": 55.8962818,\n   "lon": 37.5815678,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7962151856",\n   "kind": "organisation",\n   "name": "Балт Бизнес Брокер",\n   "office_type": "company",\n   "lat": 55.7431698,\n   "lon": 37.5770124,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7968489295",\n   "kind": "organisation",\n   "name": "Утконос",\n   "office_type": "company",\n   "lat": 55.7645663,\n   "lon": 37.5716892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7970684396",\n   "kind": "organisation",\n   "name": "Kuznetski Most 12",\n   "office_type": "estate_agent",\n   "lat": 55.761782,\n   "lon": 37.6214374,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7970684405",\n   "kind": "organisation",\n   "name": "Приемная Федеральной службы безопасности РФ",\n   "office_type": "government",\n   "lat": 55.7618968,\n   "lon": 37.6250584,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/7978733251",\n   "kind": "organisation",\n   "name": "Electrolux",\n   "office_type": "company",\n   "lat": 55.7313451,\n   "lon": 37.6456082,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7999017524",\n   "kind": "organisation",\n   "name": "Кибер-Инструмент",\n   "office_type": "company",\n   "lat": 55.7556712,\n   "lon": 37.7340299,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:00"\n  },\n  {\n   "id": "node/7999018920",\n   "kind": "organisation",\n   "name": "РесурсТранс",\n   "office_type": "company",\n   "lat": 55.840898,\n   "lon": 37.6717865,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7999018921",\n   "kind": "organisation",\n   "name": "ООО Грузовые метизы",\n   "office_type": "company",\n   "lat": 55.8402534,\n   "lon": 37.6734495,\n   "opening_hours": null\n  },\n  {\n   "id": "node/7999509633",\n   "kind": "organisation",\n   "name": "Институт авиационного приборостроения «Навигатор»",\n   "office_type": "company",\n   "lat": 55.7976622,\n   "lon": 37.5319925,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8001377398",\n   "kind": "organisation",\n   "name": "ООО \\"Бизнеспроф\\"",\n   "office_type": "company",\n   "lat": 55.7614297,\n   "lon": 37.6343644,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8001377401",\n   "kind": "organisation",\n   "name": "Юта",\n   "office_type": "estate_agent",\n   "lat": 55.7614591,\n   "lon": 37.6344274,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/8001377402",\n   "kind": "organisation",\n   "name": "Модельная школа Анны Айс",\n   "office_type": "yes",\n   "lat": 55.761438,\n   "lon": 37.6344046,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8002704677",\n   "kind": "organisation",\n   "name": "Федеральное агентство связи \\"Россвязь\\"",\n   "office_type": "government",\n   "lat": 55.8078393,\n   "lon": 37.5215597,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8002743796",\n   "kind": "organisation",\n   "name": "AV Power",\n   "office_type": "company",\n   "lat": 55.8112906,\n   "lon": 37.523877,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8002743797",\n   "kind": "organisation",\n   "name": "Фудсервис",\n   "office_type": "company",\n   "lat": 55.811398,\n   "lon": 37.5239407,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8002743798",\n   "kind": "organisation",\n   "name": "НИИ Часпром",\n   "office_type": "company",\n   "lat": 55.810577,\n   "lon": 37.5245432,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8002750607",\n   "kind": "organisation",\n   "name": "Техстройконтракт",\n   "office_type": "company",\n   "lat": 55.8125669,\n   "lon": 37.5304047,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8003908064",\n   "kind": "organisation",\n   "name": "Ведомственная охрана",\n   "office_type": "security",\n   "lat": 55.8646854,\n   "lon": 37.6858983,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8004002503",\n   "kind": "organisation",\n   "name": "Центр морского права",\n   "office_type": "lawyer",\n   "lat": 55.7988083,\n   "lon": 37.5699202,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8006077873",\n   "kind": "organisation",\n   "name": "Инженерная служба района Марьина Роща",\n   "office_type": "property_management",\n   "lat": 55.7922278,\n   "lon": 37.6010927,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8006077874",\n   "kind": "organisation",\n   "name": "Пункт оповещения №17 Участок оповещения №5",\n   "office_type": "government",\n   "lat": 55.7921149,\n   "lon": 37.601109,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8006077875",\n   "kind": "organisation",\n   "name": "Моя работа. Центр занятости населения города Москвы",\n   "office_type": "government",\n   "lat": 55.792342,\n   "lon": 37.6016251,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45; Sa,Su off"\n  },\n  {\n   "id": "node/8006077877",\n   "kind": "organisation",\n   "name": "МИИТ Учебный центр подготовки специалистов в области транспортной безопасности",\n   "office_type": "educational_institution",\n   "lat": 55.7904071,\n   "lon": 37.6053025,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8006157843",\n   "kind": "organisation",\n   "name": "АО Молодая гвардия",\n   "office_type": "publisher",\n   "lat": 55.8064965,\n   "lon": 37.5875766,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8006157844",\n   "kind": "organisation",\n   "name": "4S",\n   "office_type": "company",\n   "lat": 55.805849,\n   "lon": 37.5925396,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8006896630",\n   "kind": "organisation",\n   "name": "ГлавДоставка",\n   "office_type": "logistics",\n   "lat": 55.8307459,\n   "lon": 37.4992186,\n   "opening_hours": "Mo-Fr 09:00-19:00;Sa 10:00-16:00"\n  },\n  {\n   "id": "node/8006911974",\n   "kind": "organisation",\n   "name": "Аско-страхование",\n   "office_type": "insurance",\n   "lat": 55.6501587,\n   "lon": 37.5390387,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/8009242328",\n   "kind": "organisation",\n   "name": "Информавтодор",\n   "office_type": "company",\n   "lat": 55.811443,\n   "lon": 37.6248389,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8009242329",\n   "kind": "organisation",\n   "name": "Дороги России",\n   "office_type": "company",\n   "lat": 55.8114261,\n   "lon": 37.6248725,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8009789666",\n   "kind": "organisation",\n   "name": "Альта Софт",\n   "office_type": "it",\n   "lat": 55.8072052,\n   "lon": 37.629694,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8018028847",\n   "kind": "organisation",\n   "name": "БЛ Групп",\n   "office_type": "company",\n   "lat": 55.8064947,\n   "lon": 37.6537443,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8035236212",\n   "kind": "organisation",\n   "name": "НПО Экран",\n   "office_type": "company",\n   "lat": 55.8265296,\n   "lon": 37.6536081,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8039663817",\n   "kind": "organisation",\n   "name": "Фишер Крепежные Системы Рус",\n   "office_type": "company",\n   "lat": 55.8477378,\n   "lon": 37.4742794,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8041749710",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.8454904,\n   "lon": 37.585457,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8043900848",\n   "kind": "organisation",\n   "name": "Alexander Volkov architects",\n   "office_type": "architect",\n   "lat": 55.757336,\n   "lon": 37.6399031,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8043900857",\n   "kind": "organisation",\n   "name": "Московский Союз художников",\n   "office_type": "ngo",\n   "lat": 55.7572311,\n   "lon": 37.6406206,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8044622268",\n   "kind": "organisation",\n   "name": "Ingrad",\n   "office_type": "company",\n   "lat": 55.773933,\n   "lon": 37.6093072,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8044714569",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7174921,\n   "lon": 37.6600934,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8048602802",\n   "kind": "organisation",\n   "name": "SDR-Group",\n   "office_type": "company",\n   "lat": 55.7580076,\n   "lon": 37.6016307,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8058669733",\n   "kind": "organisation",\n   "name": "Московская дирекция по тепловодоснабжению РЖД",\n   "office_type": "company",\n   "lat": 55.7753435,\n   "lon": 37.6656631,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8058669734",\n   "kind": "organisation",\n   "name": "Московская химико-техническая лаборатория РЖД",\n   "office_type": "research",\n   "lat": 55.7749996,\n   "lon": 37.6658508,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8058669736",\n   "kind": "organisation",\n   "name": "Железнодорожная торговая компания. Московский филиал",\n   "office_type": "company",\n   "lat": 55.774028,\n   "lon": 37.6649147,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8058706720",\n   "kind": "organisation",\n   "name": "Промтехвзрыв",\n   "office_type": "company",\n   "lat": 55.7722192,\n   "lon": 37.6604515,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8062485441",\n   "kind": "organisation",\n   "name": "Forte Group",\n   "office_type": "estate_agent",\n   "lat": 55.7427492,\n   "lon": 37.527248,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8065673628",\n   "kind": "organisation",\n   "name": "Настенька",\n   "office_type": "company",\n   "lat": 55.826028,\n   "lon": 37.626285,\n   "opening_hours": "Mo-Su 08:00-22:00"\n  },\n  {\n   "id": "node/8065696527",\n   "kind": "organisation",\n   "name": "Миал-С",\n   "office_type": "company",\n   "lat": 55.7457385,\n   "lon": 37.6281899,\n   "opening_hours": "Mo-Fr 09:30-18:30"\n  },\n  {\n   "id": "node/8065696549",\n   "kind": "organisation",\n   "name": "Первый Визовый Центр",\n   "office_type": "insurance",\n   "lat": 55.6602852,\n   "lon": 37.7415669,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8065703063",\n   "kind": "organisation",\n   "name": "Бюро перевозок",\n   "office_type": "company",\n   "lat": 55.8400335,\n   "lon": 37.6736802,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8065706595",\n   "kind": "organisation",\n   "name": "Nice Home Studio",\n   "office_type": "company",\n   "lat": 55.761145,\n   "lon": 37.769466,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8065709488",\n   "kind": "organisation",\n   "name": "Авторитет",\n   "office_type": "company",\n   "lat": 55.649627,\n   "lon": 37.53967,\n   "opening_hours": "Mo-Sa 09:00-18:00"\n  },\n  {\n   "id": "node/8065721269",\n   "kind": "organisation",\n   "name": "Талор",\n   "office_type": "company",\n   "lat": 55.7080346,\n   "lon": 37.830053,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/8065721270",\n   "kind": "organisation",\n   "name": "СитиСберъ",\n   "office_type": "company",\n   "lat": 55.6594152,\n   "lon": 37.6079446,\n   "opening_hours": "Mo-Fr 09:30-18:30;Sa 09:30-15:00"\n  },\n  {\n   "id": "node/8065737323",\n   "kind": "organisation",\n   "name": "Чистый лист",\n   "office_type": "lawyer",\n   "lat": 55.7390906,\n   "lon": 37.6224768,\n   "opening_hours": "Mo-Fr 10:00-19:00;Sa 13:00-19:00"\n  },\n  {\n   "id": "node/8065737325",\n   "kind": "organisation",\n   "name": "Чистый Лист",\n   "office_type": "lawyer",\n   "lat": 55.6709234,\n   "lon": 37.445628,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/8068206547",\n   "kind": "organisation",\n   "name": "Оборонэнерго",\n   "office_type": "energy_supplier",\n   "lat": 55.7843902,\n   "lon": 37.6706412,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8068206554",\n   "kind": "organisation",\n   "name": "Альта форум",\n   "office_type": "educational_institution",\n   "lat": 55.7793901,\n   "lon": 37.6676895,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8068206555",\n   "kind": "organisation",\n   "name": "ИДПО \\"Таможпромсоцбизнес\\"",\n   "office_type": "company",\n   "lat": 55.7760767,\n   "lon": 37.6697253,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8068212087",\n   "kind": "organisation",\n   "name": "МОЭК-Проект",\n   "office_type": "company",\n   "lat": 55.7763279,\n   "lon": 37.6692773,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8068212152",\n   "kind": "organisation",\n   "name": "Инспекция по проведению плановых проверок и целевых обследований",\n   "office_type": "surveyor",\n   "lat": 55.7819384,\n   "lon": 37.6702456,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8068588423",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7318155,\n   "lon": 37.6357849,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8076880753",\n   "kind": "organisation",\n   "name": "zaycev.net",\n   "office_type": "company",\n   "lat": 55.8052169,\n   "lon": 37.5267242,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8083664905",\n   "kind": "organisation",\n   "name": "Bustren",\n   "office_type": "company",\n   "lat": 55.8081035,\n   "lon": 37.5267626,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8083744077",\n   "kind": "organisation",\n   "name": "ЭМСК",\n   "office_type": "company",\n   "lat": 55.7794725,\n   "lon": 37.6523515,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8085346796",\n   "kind": "organisation",\n   "name": "Правильные люди",\n   "office_type": "company",\n   "lat": 55.7769698,\n   "lon": 37.5788024,\n   "opening_hours": "Mo-Fr 11:00-15:00"\n  },\n  {\n   "id": "node/8085346797",\n   "kind": "organisation",\n   "name": "Рубикон",\n   "office_type": "logistics",\n   "lat": 55.7765715,\n   "lon": 37.5772977,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8085943251",\n   "kind": "organisation",\n   "name": "Сваи Сервис",\n   "office_type": "company",\n   "lat": 55.78034,\n   "lon": 37.6964342,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8087158048",\n   "kind": "organisation",\n   "name": "MSK-IX",\n   "office_type": "telecommunication",\n   "lat": 55.8008079,\n   "lon": 37.5546177,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8087198519",\n   "kind": "organisation",\n   "name": "Совет ветеранов Гольяново",\n   "office_type": "ngo",\n   "lat": 55.8207528,\n   "lon": 37.8199156,\n   "opening_hours": "Mo-Th 11:00-14:00"\n  },\n  {\n   "id": "node/8087198522",\n   "kind": "organisation",\n   "name": "Жилищник Гольяново",\n   "office_type": "property_management",\n   "lat": 55.8225011,\n   "lon": 37.8137062,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/8087286107",\n   "kind": "organisation",\n   "name": "Работа.ру",\n   "office_type": "company",\n   "lat": 55.7817457,\n   "lon": 37.5810428,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8087505828",\n   "kind": "organisation",\n   "name": "ФГУП Издательство Известия управления делами президента РФ",\n   "office_type": "company",\n   "lat": 55.7871207,\n   "lon": 37.5861698,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8087505829",\n   "kind": "organisation",\n   "name": "Издательский дом \\"Панорама\\"",\n   "office_type": "company",\n   "lat": 55.7888636,\n   "lon": 37.5864693,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8091021406",\n   "kind": "organisation",\n   "name": "Спецжелезобетонстрой",\n   "office_type": "company",\n   "lat": 55.7738727,\n   "lon": 37.6724316,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/8091035422",\n   "kind": "organisation",\n   "name": "ФертиХелп",\n   "office_type": "consulting",\n   "lat": 55.7740688,\n   "lon": 37.6699036,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8093019888",\n   "kind": "organisation",\n   "name": "Московско-Ярославская транспортная прокуратура",\n   "office_type": "government",\n   "lat": 55.780599,\n   "lon": 37.654156,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8093019889",\n   "kind": "organisation",\n   "name": "Филиал ОАО РЖД Центральная дирекция пассажирских обустройств",\n   "office_type": "company",\n   "lat": 55.7834301,\n   "lon": 37.6513739,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8095995354",\n   "kind": "organisation",\n   "name": "Технический центр автоматики и телемеханики",\n   "office_type": "company",\n   "lat": 55.784599,\n   "lon": 37.6542746,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8097082291",\n   "kind": "organisation",\n   "name": "Редакция журнала \\"Защита и карантин растений\\"",\n   "office_type": "newspaper",\n   "lat": 55.7836263,\n   "lon": 37.6568449,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8097082292",\n   "kind": "organisation",\n   "name": "Профсоюз сотрудников Московской железной дороги",\n   "office_type": "ngo",\n   "lat": 55.7826982,\n   "lon": 37.6578007,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099767391",\n   "kind": "organisation",\n   "name": "Территориальное управление Октябрьской железной дороги",\n   "office_type": "company",\n   "lat": 55.7827515,\n   "lon": 37.6531371,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099832498",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник Басманного района, ОДС-6",\n   "office_type": "property_management",\n   "lat": 55.773905,\n   "lon": 37.6725084,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099876060",\n   "kind": "organisation",\n   "name": "Московское железнодорожное агентство",\n   "office_type": "company",\n   "lat": 55.775104,\n   "lon": 37.6600046,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099876064",\n   "kind": "organisation",\n   "name": "ООО РЖД-охрана",\n   "office_type": "company",\n   "lat": 55.7736927,\n   "lon": 37.6650671,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099897701",\n   "kind": "organisation",\n   "name": "ОДС-6 ГБУ \\"Жилищник Красносельского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7720369,\n   "lon": 37.6597773,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099897702",\n   "kind": "organisation",\n   "name": "Тоннельная ассоциация России",\n   "office_type": "company",\n   "lat": 55.7721018,\n   "lon": 37.6597747,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099897703",\n   "kind": "organisation",\n   "name": "ООО Колумб РУ",\n   "office_type": "company",\n   "lat": 55.7721983,\n   "lon": 37.6606813,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099966652",\n   "kind": "organisation",\n   "name": "ЗАО \\"Мик-Информ\\"",\n   "office_type": "research",\n   "lat": 55.7671939,\n   "lon": 37.6575047,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099966653",\n   "kind": "organisation",\n   "name": "АО РЖД Логистика",\n   "office_type": "company",\n   "lat": 55.7672048,\n   "lon": 37.6575577,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099966654",\n   "kind": "organisation",\n   "name": "ПАО Моспромтранспроект",\n   "office_type": "company",\n   "lat": 55.7672143,\n   "lon": 37.6576038,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099966655",\n   "kind": "organisation",\n   "name": "Центр продажи услуг",\n   "office_type": "company",\n   "lat": 55.7668106,\n   "lon": 37.6578816,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099966656",\n   "kind": "organisation",\n   "name": "Дирекция тяги",\n   "office_type": "company",\n   "lat": 55.7664771,\n   "lon": 37.6578895,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8099966657",\n   "kind": "organisation",\n   "name": "Министерство транспорта Российской федерации",\n   "office_type": "government",\n   "lat": 55.766392,\n   "lon": 37.6580674,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8100832461",\n   "kind": "organisation",\n   "name": "Участок № 3 ГБУ \\"Жилищник Пресненского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7741095,\n   "lon": 37.5819379,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8104893854",\n   "kind": "organisation",\n   "name": "Посольство Республики Южный Судан",\n   "office_type": "diplomatic",\n   "lat": 55.7351227,\n   "lon": 37.6145325,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8106211970",\n   "kind": "organisation",\n   "name": "Адвокатский кабинет №801",\n   "office_type": "lawyer",\n   "lat": 55.7958811,\n   "lon": 37.699038,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8106211971",\n   "kind": "organisation",\n   "name": "ЗАО Объединенный Проектный Институт Гипрохиммонтаж",\n   "office_type": "company",\n   "lat": 55.7958326,\n   "lon": 37.699018,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8106211972",\n   "kind": "organisation",\n   "name": "Жилищная инспекция по ВАО Москвы",\n   "office_type": "property_management",\n   "lat": 55.7958023,\n   "lon": 37.6991474,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8106211973",\n   "kind": "organisation",\n   "name": "Дирекция заказчика жилищно-коммунального хозяйства и благоустройства ВАО",\n   "office_type": "property_management",\n   "lat": 55.7957668,\n   "lon": 37.6991936,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8106211974",\n   "kind": "organisation",\n   "name": "ООО МВК Сервиспроект",\n   "office_type": "company",\n   "lat": 55.7957356,\n   "lon": 37.6992337,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8107071421",\n   "kind": "organisation",\n   "name": "ГУП Мосводосток ЭГТР-3",\n   "office_type": "property_management",\n   "lat": 55.7906265,\n   "lon": 37.7123114,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8109566625",\n   "kind": "organisation",\n   "name": "Бюро научно-технической информации",\n   "office_type": "company",\n   "lat": 55.8109976,\n   "lon": 37.8307128,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8110613001",\n   "kind": "organisation",\n   "name": "ФГКУ \\"УВО ВНГ России по городу Москве\\"",\n   "office_type": "security",\n   "lat": 55.7959867,\n   "lon": 37.7071842,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8112003634",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник района Богородское ОДС-30",\n   "office_type": "property_management",\n   "lat": 55.81167,\n   "lon": 37.7099377,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8112236445",\n   "kind": "organisation",\n   "name": "ОАО НТПП Геофизика Арт-2000",\n   "office_type": "company",\n   "lat": 55.8198381,\n   "lon": 37.7811336,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8112236446",\n   "kind": "organisation",\n   "name": "ОАО НПП Геофизика Космос-2000",\n   "office_type": "company",\n   "lat": 55.8200839,\n   "lon": 37.7815311,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8112239631",\n   "kind": "organisation",\n   "name": "Геофизика Космос",\n   "office_type": "company",\n   "lat": 55.8212393,\n   "lon": 37.7842691,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8115503171",\n   "kind": "organisation",\n   "name": "Совет ветеранов войн и военной службы Военно-Воздушных сил",\n   "office_type": "ngo",\n   "lat": 55.7971813,\n   "lon": 37.5503683,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8116377257",\n   "kind": "organisation",\n   "name": "Мосинжпроект",\n   "office_type": "engineer",\n   "lat": 55.7704284,\n   "lon": 37.5923283,\n   "opening_hours": "Mo-Fr 08:30-17:30"\n  },\n  {\n   "id": "node/8116462940",\n   "kind": "organisation",\n   "name": "Метрогипротранс",\n   "office_type": "company",\n   "lat": 55.731654,\n   "lon": 37.6359209,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8118274408",\n   "kind": "organisation",\n   "name": "Vekons",\n   "office_type": "company",\n   "lat": 55.7187338,\n   "lon": 37.7443644,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8122042676",\n   "kind": "organisation",\n   "name": "ОДС № 2 ГБУ \\"Жилищник Пресненского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7680187,\n   "lon": 37.5736257,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8129648838",\n   "kind": "organisation",\n   "name": "Вертолеты России",\n   "office_type": "company",\n   "lat": 55.7296703,\n   "lon": 37.6353219,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8129648870",\n   "kind": "organisation",\n   "name": "Местная районная организация \\"Замосковречье\\"",\n   "office_type": "government",\n   "lat": 55.7295554,\n   "lon": 37.6324761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8129648885",\n   "kind": "organisation",\n   "name": "Управлением Федеральной службы по надзору в сфере защиты прав потребителей и благополучия человека по железнодорожному транспорту",\n   "office_type": "government",\n   "lat": 55.7271697,\n   "lon": 37.6389469,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8129648886",\n   "kind": "organisation",\n   "name": "Центр гигиены и эпидемиологии по железнодорожному транспорту",\n   "office_type": "government",\n   "lat": 55.7270693,\n   "lon": 37.638967,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8146186057",\n   "kind": "organisation",\n   "name": "Градиентех",\n   "office_type": "it",\n   "lat": 55.7147791,\n   "lon": 37.6773655,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8146853174",\n   "kind": "organisation",\n   "name": "Zetta страхование",\n   "office_type": "company",\n   "lat": 55.7921726,\n   "lon": 37.7095564,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8146893188",\n   "kind": "organisation",\n   "name": "Новый регистратор",\n   "office_type": "company",\n   "lat": 55.7925563,\n   "lon": 37.709968,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8148301498",\n   "kind": "organisation",\n   "name": "Межгосударственная телерадиокомпания «Мир»",\n   "office_type": "telecommunication",\n   "lat": 55.8064139,\n   "lon": 37.7055636,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8149143298",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8734545,\n   "lon": 37.6456447,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8152353729",\n   "kind": "organisation",\n   "name": "Управление социальной защиты населения Центрального административного округа",\n   "office_type": "government",\n   "lat": 55.787635,\n   "lon": 37.6362929,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8154382970",\n   "kind": "organisation",\n   "name": "ПРОконтроль",\n   "office_type": "company",\n   "lat": 55.7024851,\n   "lon": 37.6767379,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8154393310",\n   "kind": "organisation",\n   "name": "Агентство недвижимости",\n   "office_type": "estate_agent",\n   "lat": 55.6651387,\n   "lon": 37.5478417,\n   "opening_hours": "Mo-Su 08:00-21:00"\n  },\n  {\n   "id": "node/8154393581",\n   "kind": "organisation",\n   "name": "Фаворит",\n   "office_type": "company",\n   "lat": 55.725222,\n   "lon": 37.7822882,\n   "opening_hours": "Mo-Sa 00:00-24:00"\n  },\n  {\n   "id": "node/8154393595",\n   "kind": "organisation",\n   "name": "Фабрика Потолков",\n   "office_type": "company",\n   "lat": 55.64607,\n   "lon": 37.677236,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8154394700",\n   "kind": "organisation",\n   "name": "Kurttrans",\n   "office_type": "company",\n   "lat": 55.634054,\n   "lon": 37.628412,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/8154400401",\n   "kind": "organisation",\n   "name": "CityTelecom.ru",\n   "office_type": "telecommunication",\n   "lat": 55.674375,\n   "lon": 37.6187809,\n   "opening_hours": "Fr 09:00-17:00;Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/8157750495",\n   "kind": "organisation",\n   "name": "Современные энергоэффективные технологии",\n   "office_type": "company",\n   "lat": 55.8111664,\n   "lon": 37.8331268,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8173499080",\n   "kind": "organisation",\n   "name": "ЭЗОИС",\n   "office_type": "company",\n   "lat": 55.8119634,\n   "lon": 37.7619619,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8175951989",\n   "kind": "organisation",\n   "name": "Федерация дзюдо Москвы",\n   "office_type": "ngo",\n   "lat": 55.7987447,\n   "lon": 37.5712252,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8177456689",\n   "kind": "organisation",\n   "name": "Объединенная авиастроительная корпорация",\n   "office_type": "company",\n   "lat": 55.7294251,\n   "lon": 37.6345714,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8179283858",\n   "kind": "organisation",\n   "name": "Amway",\n   "office_type": "company",\n   "lat": 55.7857704,\n   "lon": 37.6604805,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8184957703",\n   "kind": "organisation",\n   "name": "Архитектурно-планировочное управление городского округа Химки",\n   "office_type": "government",\n   "lat": 55.8872791,\n   "lon": 37.4578751,\n   "opening_hours": "Mo-Th 09:00-13:00,14:00-18:00 open \\"График работы\\"; Fr 09:00-13:00,14:00-16:45 open \\"График работы\\" || Mo,Tu 10:00-13:00 open \\"Приём по вопросам перепланировок\\"|| Th 14:30-17:00 open \\"Приём по вопросам перепланировок\\""\n  },\n  {\n   "id": "node/8185422651",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.763894,\n   "lon": 37.5640301,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8191912931",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8179158,\n   "lon": 37.4999025,\n   "opening_hours": "PH,Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8191912972",\n   "kind": "organisation",\n   "name": "Ингосстрах-Жизнь",\n   "office_type": "insurance",\n   "lat": 55.8216656,\n   "lon": 37.4996799,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8192074782",\n   "kind": "organisation",\n   "name": "Офис продаж жилого комплекса \\"Любовь и голуби\\"",\n   "office_type": "estate_agent",\n   "lat": 55.8629751,\n   "lon": 37.5344654,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8194402489",\n   "kind": "organisation",\n   "name": "Эксперт Банкрот",\n   "office_type": "lawyer",\n   "lat": 55.6826616,\n   "lon": 37.662031,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8194406804",\n   "kind": "organisation",\n   "name": "Эксперт Банкрот",\n   "office_type": "lawyer",\n   "lat": 55.7826256,\n   "lon": 37.7233818,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa 09:00-15:00"\n  },\n  {\n   "id": "node/8194427095",\n   "kind": "organisation",\n   "name": "Эксперт Банкрот",\n   "office_type": "lawyer",\n   "lat": 55.8120783,\n   "lon": 37.6363478,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8194447971",\n   "kind": "organisation",\n   "name": "Эксперт Банкрот",\n   "office_type": "lawyer",\n   "lat": 55.8031595,\n   "lon": 37.5836396,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa-Su 11:00-15:00"\n  },\n  {\n   "id": "node/8194509025",\n   "kind": "organisation",\n   "name": "Эксперт Банкрот",\n   "office_type": "lawyer",\n   "lat": 55.675509,\n   "lon": 37.5059063,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8195853695",\n   "kind": "organisation",\n   "name": "Ericsson",\n   "office_type": "company",\n   "lat": 55.8214352,\n   "lon": 37.4984256,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8199967962",\n   "kind": "organisation",\n   "name": "ОДС № 8 ГБУ \\"Жилищник Пресненского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7641951,\n   "lon": 37.5653189,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8205992224",\n   "kind": "organisation",\n   "name": "Фонд развития цифровой экономики",\n   "office_type": "government",\n   "lat": 55.7628233,\n   "lon": 37.5700128,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8205992254",\n   "kind": "organisation",\n   "name": "О. В. Конина",\n   "office_type": "notary",\n   "lat": 55.7626845,\n   "lon": 37.5688729,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8209006504",\n   "kind": "organisation",\n   "name": "Сплайн",\n   "office_type": "company",\n   "lat": 55.7770128,\n   "lon": 37.6753485,\n   "opening_hours": "Mo-Su 09:30-18:00"\n  },\n  {\n   "id": "node/8210279520",\n   "kind": "organisation",\n   "name": "АО «Ирмаст-Холдинг»",\n   "office_type": "company",\n   "lat": 55.7750079,\n   "lon": 37.6803441,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8210279521",\n   "kind": "organisation",\n   "name": "Центроргтрудавтотранс",\n   "office_type": "company",\n   "lat": 55.7750479,\n   "lon": 37.6803803,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8219367132",\n   "kind": "organisation",\n   "name": "Стогруз",\n   "office_type": "logistics",\n   "lat": 55.7802876,\n   "lon": 37.6866929,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8219367153",\n   "kind": "organisation",\n   "name": "ЗАО \\"Мягкая тара-2\\"",\n   "office_type": "company",\n   "lat": 55.7781419,\n   "lon": 37.6824617,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8224219784",\n   "kind": "organisation",\n   "name": "Бета Консалтанс",\n   "office_type": "insurance",\n   "lat": 55.6057775,\n   "lon": 37.72439,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8224219787",\n   "kind": "organisation",\n   "name": "Главное управление ПФР № 10 по г. Москве и Московской области. Управление назначения, перерасчета, выплаты пенсий, ЕДВ и других социальных выплат",\n   "office_type": "government",\n   "lat": 55.7815619,\n   "lon": 37.6970584,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8232480946",\n   "kind": "organisation",\n   "name": "ОДС № 7 ГБУ \\"Жилищник Мещанского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7851353,\n   "lon": 37.6394016,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8237601287",\n   "kind": "organisation",\n   "name": "Проектстройсервис",\n   "office_type": "company",\n   "lat": 55.736604,\n   "lon": 37.718084,\n   "opening_hours": "Mo-Fr,Su 09:00-20:00;Sa 09:00-19:00"\n  },\n  {\n   "id": "node/8237603239",\n   "kind": "organisation",\n   "name": "Агентство Регистрации и Содействия Бизнесу",\n   "office_type": "lawyer",\n   "lat": 55.838681,\n   "lon": 37.6541376,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8237674240",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6644398,\n   "lon": 37.469033,\n   "opening_hours": "Mo-Fr 11:00-21:00; Sa-Su 11:00-19:00"\n  },\n  {\n   "id": "node/8238451588",\n   "kind": "organisation",\n   "name": "ТБН",\n   "office_type": "company",\n   "lat": 55.7276554,\n   "lon": 37.6451495,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8242481950",\n   "kind": "organisation",\n   "name": "Евразийская экономическая комиссия",\n   "office_type": "government",\n   "lat": 55.7283532,\n   "lon": 37.6447794,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8244655530",\n   "kind": "organisation",\n   "name": "Шраубен М.У.Н.",\n   "office_type": "company",\n   "lat": 55.8084036,\n   "lon": 37.5871052,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8244655531",\n   "kind": "organisation",\n   "name": "Fotolab",\n   "office_type": "company",\n   "lat": 55.8084895,\n   "lon": 37.586216,\n   "opening_hours": "Mo-Sa 09:00-21:30 Su 10:00-21:30"\n  },\n  {\n   "id": "node/8244655538",\n   "kind": "organisation",\n   "name": "Детская литература",\n   "office_type": "publisher",\n   "lat": 55.8076439,\n   "lon": 37.586967,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/8247866217",\n   "kind": "organisation",\n   "name": "ООО \\"Новый Мир\\"",\n   "office_type": "company",\n   "lat": 55.7595802,\n   "lon": 37.6371942,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8253845157",\n   "kind": "organisation",\n   "name": "Буровая техника ВНИИБТ",\n   "office_type": "company",\n   "lat": 55.7266942,\n   "lon": 37.6449254,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8253845158",\n   "kind": "organisation",\n   "name": "Росатом",\n   "office_type": "company",\n   "lat": 55.724342,\n   "lon": 37.6432631,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8254943547",\n   "kind": "organisation",\n   "name": "ABC",\n   "office_type": "company",\n   "lat": 55.8119456,\n   "lon": 37.6040554,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8261912222",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "government",\n   "lat": 55.7924034,\n   "lon": 37.569931,\n   "opening_hours": "Mo-Th 11:00-15:00"\n  },\n  {\n   "id": "node/8268932069",\n   "kind": "organisation",\n   "name": "Московский драматический театр художественной публицистики",\n   "office_type": "company",\n   "lat": 55.7822877,\n   "lon": 37.6739873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8268932080",\n   "kind": "organisation",\n   "name": "Центр по работе с населением ЦАО города Москвы, филиал \\"Красносельский\\"",\n   "office_type": "government",\n   "lat": 55.7819935,\n   "lon": 37.6741187,\n   "opening_hours": "Mo-Fr 10:15-19:00"\n  },\n  {\n   "id": "node/8268934184",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7835297,\n   "lon": 37.6770759,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8270161408",\n   "kind": "organisation",\n   "name": "Отдел жилищных субсидий \\"Красносельский\\"",\n   "office_type": "government",\n   "lat": 55.7799935,\n   "lon": 37.6694047,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8270161411",\n   "kind": "organisation",\n   "name": "Отдел лицензионно-разрешительной работы по центральному административному округу г. Москвы",\n   "office_type": "lawyer",\n   "lat": 55.7818751,\n   "lon": 37.6739417,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8270288065",\n   "kind": "organisation",\n   "name": "Административно-техническая инспекция по центральному-административному округу, отдел \\"Басманный\\"",\n   "office_type": "government",\n   "lat": 55.7704585,\n   "lon": 37.668997,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8270288066",\n   "kind": "organisation",\n   "name": "Дирекция заказчика Жилищно-коммунального хозяйства и благоустройства ЦАО",\n   "office_type": "government",\n   "lat": 55.7704524,\n   "lon": 37.6688603,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8271502772",\n   "kind": "organisation",\n   "name": "Клиентская служба \\"Басманный\\"",\n   "office_type": "government",\n   "lat": 55.7685264,\n   "lon": 37.6562257,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/8271502781",\n   "kind": "organisation",\n   "name": "Желдоррассчет",\n   "office_type": "company",\n   "lat": 55.7691933,\n   "lon": 37.657595,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8272938366",\n   "kind": "organisation",\n   "name": "Федеральная служба по труду и занятости",\n   "office_type": "government",\n   "lat": 55.7660503,\n   "lon": 37.6415782,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8272938373",\n   "kind": "organisation",\n   "name": "Дворец бракосочетания № 1 \\"Грибоедовский\\"",\n   "office_type": "government",\n   "lat": 55.7659681,\n   "lon": 37.6448625,\n   "opening_hours": "Mo-Th 09:00-18:00, Sa 09:00-23:59, Su 00:00-06:00, Fr 09:00-20:00, Su 10:00-19:00"\n  },\n  {\n   "id": "node/8274140393",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт судебной экспертизы",\n   "office_type": "company",\n   "lat": 55.7916147,\n   "lon": 37.5856233,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8278903811",\n   "kind": "organisation",\n   "name": "Управляющая компания \\"Ивушка\\"",\n   "office_type": "property_management",\n   "lat": 55.7947707,\n   "lon": 37.5716825,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8281001166",\n   "kind": "organisation",\n   "name": "ГКУ\\"Инженерная служба Красносельского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7827681,\n   "lon": 37.6701102,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8281001167",\n   "kind": "organisation",\n   "name": "ОДС № 8 ГБУ \\"Жилищник Красносельского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7828103,\n   "lon": 37.670019,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8281001168",\n   "kind": "organisation",\n   "name": "Аргументы и факты",\n   "office_type": "newspaper",\n   "lat": 55.7672673,\n   "lon": 37.6431057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8282135342",\n   "kind": "organisation",\n   "name": "Правое дело",\n   "office_type": "lawyer",\n   "lat": 55.7705191,\n   "lon": 37.6327039,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8302334541",\n   "kind": "organisation",\n   "name": "Упаковочные системы",\n   "office_type": "company",\n   "lat": 55.8394552,\n   "lon": 37.6357484,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8306618217",\n   "kind": "organisation",\n   "name": "Дипломат-Консалтинг",\n   "office_type": "lawyer",\n   "lat": 55.7265317,\n   "lon": 37.6272886,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-16:00"\n  },\n  {\n   "id": "node/8307245121",\n   "kind": "organisation",\n   "name": "Дирекция образовательных программ в сфере культуры и и скусства",\n   "office_type": "government",\n   "lat": 55.776079,\n   "lon": 37.6326545,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8307464032",\n   "kind": "organisation",\n   "name": "Участок № 10 ГБУ \\"Жилищник района Тверской\\"",\n   "office_type": "property_management",\n   "lat": 55.7824016,\n   "lon": 37.5938442,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8307565221",\n   "kind": "organisation",\n   "name": "Урал Ресурс",\n   "office_type": "company",\n   "lat": 55.7765903,\n   "lon": 37.676045,\n   "opening_hours": "Mo-Fr 09:00-13:00, 14:00-18:00"\n  },\n  {\n   "id": "node/8308203711",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "association",\n   "lat": 55.780397,\n   "lon": 37.6642911,\n   "opening_hours": "Mo-Th 11:00-15:00"\n  },\n  {\n   "id": "node/8313879273",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.8678591,\n   "lon": 37.6672648,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8334009825",\n   "kind": "organisation",\n   "name": "Российский фонд прямых инвестиций",\n   "office_type": "government",\n   "lat": 55.7469164,\n   "lon": 37.539088,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8341433239",\n   "kind": "organisation",\n   "name": "Московская административная дорожная инспекция",\n   "office_type": "government",\n   "lat": 55.8243431,\n   "lon": 37.6600894,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8356769817",\n   "kind": "organisation",\n   "name": "Трунов, Айвар и партнеры",\n   "office_type": "lawyer",\n   "lat": 55.8089355,\n   "lon": 37.4922063,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/8356769917",\n   "kind": "organisation",\n   "name": "Общественный пункт охраны правопорядка № 56",\n   "office_type": "yes",\n   "lat": 55.8088324,\n   "lon": 37.4920481,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8356769918",\n   "kind": "organisation",\n   "name": "Жилищная инспекция по САО",\n   "office_type": "government",\n   "lat": 55.8081489,\n   "lon": 37.4922231,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8356818517",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник района Сокол\\"",\n   "office_type": "property_management",\n   "lat": 55.8014761,\n   "lon": 37.4979581,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8356823117",\n   "kind": "organisation",\n   "name": "Аргумент",\n   "office_type": "lawyer",\n   "lat": 55.8001378,\n   "lon": 37.49904,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8356840218",\n   "kind": "organisation",\n   "name": "ГБУ Жилищник района Сокол",\n   "office_type": "property_management",\n   "lat": 55.7987961,\n   "lon": 37.5012703,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8358123482",\n   "kind": "organisation",\n   "name": "Центр современных технологий безопасности информации",\n   "office_type": "it",\n   "lat": 55.8373955,\n   "lon": 37.5496092,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8364370485",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты населения Тимирязевского района",\n   "office_type": "government",\n   "lat": 55.812449,\n   "lon": 37.5762624,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8364370516",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичной организации №6 района Тимирязевский САО",\n   "office_type": "association",\n   "lat": 55.8113631,\n   "lon": 37.5771099,\n   "opening_hours": "Mo-Th 11:00-15:00"\n  },\n  {\n   "id": "node/8364382217",\n   "kind": "organisation",\n   "name": "Климби",\n   "office_type": "company",\n   "lat": 55.8113197,\n   "lon": 37.5768878,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8364382218",\n   "kind": "organisation",\n   "name": "\\"Жилищник\\" Тимирязевского района",\n   "office_type": "property_management",\n   "lat": 55.8114091,\n   "lon": 37.5768149,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/8364799117",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7343471,\n   "lon": 37.475293,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8370846287",\n   "kind": "organisation",\n   "name": "АПМ-1",\n   "office_type": "company",\n   "lat": 55.819871,\n   "lon": 37.6183194,\n   "opening_hours": "Mo-Fr 09:00-18:00;Sa-Su 11:00-16:00"\n  },\n  {\n   "id": "node/8370846289",\n   "kind": "organisation",\n   "name": "Гостиный Двор",\n   "office_type": "estate_agent",\n   "lat": 55.7861435,\n   "lon": 37.7232581,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8370846303",\n   "kind": "organisation",\n   "name": "City service",\n   "office_type": "company",\n   "lat": 55.8198921,\n   "lon": 37.6183194,\n   "opening_hours": "Mo-Fr 09:00-18:00;Sa-Su 10:00-16:00"\n  },\n  {\n   "id": "node/8370848125",\n   "kind": "organisation",\n   "name": "Kupribrow",\n   "office_type": "educational_institution",\n   "lat": 55.704523,\n   "lon": 37.598368,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/8370848135",\n   "kind": "organisation",\n   "name": "iNSTANS",\n   "office_type": "insurance",\n   "lat": 55.8931917,\n   "lon": 37.7245992,\n   "opening_hours": "Mo-Fr 09:00-21:00;Sa 10:00-19:00;Su 12:00-19:00"\n  },\n  {\n   "id": "node/8370848138",\n   "kind": "organisation",\n   "name": "Агентство недвижимости Ивановой Дарии",\n   "office_type": "estate_agent",\n   "lat": 55.782646,\n   "lon": 37.598761,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/8370848144",\n   "kind": "organisation",\n   "name": "МосСанЭксперт",\n   "office_type": "company",\n   "lat": 55.8124007,\n   "lon": 37.7710417,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/8370848148",\n   "kind": "organisation",\n   "name": "Транс Трек-ДВ",\n   "office_type": "logistics",\n   "lat": 55.7960041,\n   "lon": 37.6509243,\n   "opening_hours": "Fr 09:00-17:00;Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/8370849456",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7213518,\n   "lon": 37.7555117,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/8371953466",\n   "kind": "organisation",\n   "name": "Аналитический центр Минсельхоза России",\n   "office_type": "government",\n   "lat": 55.7431462,\n   "lon": 37.6273933,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8377639676",\n   "kind": "organisation",\n   "name": "Спецремкомплект",\n   "office_type": "company",\n   "lat": 55.7776056,\n   "lon": 37.6041238,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8383939795",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8174871,\n   "lon": 37.5736472,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8384652391",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8952145,\n   "lon": 37.5424746,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8386248940",\n   "kind": "organisation",\n   "name": "Steel heart",\n   "office_type": "company",\n   "lat": 55.7095744,\n   "lon": 37.6138997,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8390522305",\n   "kind": "organisation",\n   "name": "Cuteam",\n   "office_type": "coworking",\n   "lat": 55.7479515,\n   "lon": 37.6914444,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/8390987295",\n   "kind": "organisation",\n   "name": "Трансэнерго",\n   "office_type": "company",\n   "lat": 55.7787392,\n   "lon": 37.6468648,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8391779062",\n   "kind": "organisation",\n   "name": "Научно-исследовательский кинофотоинститут",\n   "office_type": "research",\n   "lat": 55.799891,\n   "lon": 37.5308847,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8391990768",\n   "kind": "organisation",\n   "name": "Дирекция скоростного сообщения",\n   "office_type": "company",\n   "lat": 55.7823744,\n   "lon": 37.6452173,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8392053632",\n   "kind": "organisation",\n   "name": "Инспекция водных ресурсов",\n   "office_type": "government",\n   "lat": 55.7883634,\n   "lon": 37.6407614,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8392053708",\n   "kind": "organisation",\n   "name": "Дирекция специальных перевозок",\n   "office_type": "company",\n   "lat": 55.7823812,\n   "lon": 37.6452515,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8392869058",\n   "kind": "organisation",\n   "name": "Кристон-кадастр",\n   "office_type": "company",\n   "lat": 55.7100611,\n   "lon": 37.6184055,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8396897887",\n   "kind": "organisation",\n   "name": "Click-Express",\n   "office_type": "company",\n   "lat": 55.8175446,\n   "lon": 37.6559376,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/8403186931",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.688977,\n   "lon": 37.5713169,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8404167917",\n   "kind": "organisation",\n   "name": "ОАО НПО \\"Родина\\"",\n   "office_type": "company",\n   "lat": 55.7647883,\n   "lon": 37.5557467,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/8405595618",\n   "kind": "organisation",\n   "name": "ФБК",\n   "office_type": "ngo",\n   "lat": 55.7086424,\n   "lon": 37.6529827,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8409352315",\n   "kind": "organisation",\n   "name": "Жилищник района Очаково-Матвеевское",\n   "office_type": "property_management",\n   "lat": 55.682401,\n   "lon": 37.4591544,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8412505013",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7795484,\n   "lon": 37.7070966,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8417281799",\n   "kind": "organisation",\n   "name": "Центр по работе с населением района Ростокино г. Москвы",\n   "office_type": "association",\n   "lat": 55.8365016,\n   "lon": 37.6668888,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8417281800",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичной организации №3 района Ростокино СВАО",\n   "office_type": "association",\n   "lat": 55.8365362,\n   "lon": 37.6668204,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8417767244",\n   "kind": "organisation",\n   "name": "Дайлет",\n   "office_type": "estate_agent",\n   "lat": 55.75357,\n   "lon": 37.5892987,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/8426387122",\n   "kind": "organisation",\n   "name": "РЕСО-Мед",\n   "office_type": "insurance",\n   "lat": 55.7052524,\n   "lon": 37.6209509,\n   "opening_hours": "Tu, Fr 10:00-13:00, 14:00-18:00"\n  },\n  {\n   "id": "node/8427900918",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.64129,\n   "lon": 37.5013887,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8431342553",\n   "kind": "organisation",\n   "name": "Управление лесного хозяйства",\n   "office_type": "government",\n   "lat": 55.8138333,\n   "lon": 37.4919873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8431854959",\n   "kind": "organisation",\n   "name": "Отдел по формированию пенсионных прав отдельных категорий застрахованных лиц",\n   "office_type": "government",\n   "lat": 55.8171017,\n   "lon": 37.4877966,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8434616970",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "yes",\n   "lat": 55.7132021,\n   "lon": 37.6286466,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8435683394",\n   "kind": "organisation",\n   "name": "Центральное управление государственного речного надзора",\n   "office_type": "government",\n   "lat": 55.8548229,\n   "lon": 37.4668442,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8441125959",\n   "kind": "organisation",\n   "name": "Налоговая инспекция № 36",\n   "office_type": "government",\n   "lat": 55.7080006,\n   "lon": 37.498183,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8441125960",\n   "kind": "organisation",\n   "name": "Налоговая инспекция № 29",\n   "office_type": "government",\n   "lat": 55.7081004,\n   "lon": 37.4979926,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8441427600",\n   "kind": "organisation",\n   "name": "Трансмобильность",\n   "office_type": "company",\n   "lat": 55.7828043,\n   "lon": 37.6559521,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8445715532",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7904686,\n   "lon": 37.6602745,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8445715561",\n   "kind": "organisation",\n   "name": "Объединённый экзаминационный центр Кембриджского универститета",\n   "office_type": "company",\n   "lat": 55.7831708,\n   "lon": 37.6612428,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8459095961",\n   "kind": "organisation",\n   "name": "Еврогород",\n   "office_type": "property_management",\n   "lat": 55.87812,\n   "lon": 37.5378129,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8460641131",\n   "kind": "organisation",\n   "name": "Bask",\n   "office_type": "company",\n   "lat": 55.8073817,\n   "lon": 37.6256365,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8460641160",\n   "kind": "organisation",\n   "name": "Особое конструкторское бюро средств автоматизации",\n   "office_type": "company",\n   "lat": 55.806625,\n   "lon": 37.629008,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8460641165",\n   "kind": "organisation",\n   "name": "ооо \\"интернет для каждого\\"",\n   "office_type": "company",\n   "lat": 55.8071488,\n   "lon": 37.6298019,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8460641166",\n   "kind": "organisation",\n   "name": "Куппер",\n   "office_type": "company",\n   "lat": 55.8070975,\n   "lon": 37.6299039,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8460641167",\n   "kind": "organisation",\n   "name": "ПАО \\"Калибр\\"",\n   "office_type": "company",\n   "lat": 55.8070433,\n   "lon": 37.6298355,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8460641349",\n   "kind": "organisation",\n   "name": "Интехника",\n   "office_type": "company",\n   "lat": 55.8047513,\n   "lon": 37.6287425,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8460641350",\n   "kind": "organisation",\n   "name": "Derren",\n   "office_type": "company",\n   "lat": 55.8046601,\n   "lon": 37.6285815,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8460684804",\n   "kind": "organisation",\n   "name": "АМК Групп",\n   "office_type": "graphic_design",\n   "lat": 55.7590158,\n   "lon": 37.7338743,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/8462369059",\n   "kind": "organisation",\n   "name": "Новая изба",\n   "office_type": "company",\n   "lat": 55.8071638,\n   "lon": 37.6168998,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8463340990",\n   "kind": "organisation",\n   "name": "IBS",\n   "office_type": "company",\n   "lat": 55.8061238,\n   "lon": 37.5906792,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8473641179",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7584151,\n   "lon": 37.7320759,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8475102306",\n   "kind": "organisation",\n   "name": "Управление №10 по эксплуатации и ремонту газового хозяйства",\n   "office_type": "company",\n   "lat": 55.8158362,\n   "lon": 37.7091806,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8475102308",\n   "kind": "organisation",\n   "name": "Управление аварийно-восстановительных работ по газоснабжению",\n   "office_type": "company",\n   "lat": 55.8156922,\n   "lon": 37.7097063,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8476507417",\n   "kind": "organisation",\n   "name": "Ресо",\n   "office_type": "insurance",\n   "lat": 55.6795371,\n   "lon": 37.5461435,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8476507418",\n   "kind": "organisation",\n   "name": "Ликвид недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.6793806,\n   "lon": 37.5458691,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8486682317",\n   "kind": "organisation",\n   "name": "Управление по вопросам миграции",\n   "office_type": "government",\n   "lat": 55.7519438,\n   "lon": 37.6412798,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8488538015",\n   "kind": "organisation",\n   "name": "Контора дорожного мастера",\n   "office_type": "company",\n   "lat": 55.7785793,\n   "lon": 37.6497857,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8489068344",\n   "kind": "organisation",\n   "name": "Ростоппром",\n   "office_type": "company",\n   "lat": 55.7672047,\n   "lon": 37.6511241,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8489068408",\n   "kind": "organisation",\n   "name": "Московский проектно-изыскательный институт \\"Мосжелдорпроект\\"",\n   "office_type": "engineering",\n   "lat": 55.7752628,\n   "lon": 37.650699,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8491539065",\n   "kind": "organisation",\n   "name": "Streetbasket",\n   "office_type": "yes",\n   "lat": 55.767563,\n   "lon": 37.6472296,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8491539066",\n   "kind": "organisation",\n   "name": "Российская федерация баскетбола",\n   "office_type": "government",\n   "lat": 55.7675887,\n   "lon": 37.6472081,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8495964506",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "association",\n   "lat": 55.7703883,\n   "lon": 37.6493298,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8495964553",\n   "kind": "organisation",\n   "name": "Управление по безопасности крупных международных и массовых спортивных мероприятий",\n   "office_type": "government",\n   "lat": 55.7680104,\n   "lon": 37.6475045,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8496587089",\n   "kind": "organisation",\n   "name": "Стоянов, Соколов и партнеры",\n   "office_type": "lawyer",\n   "lat": 55.7095013,\n   "lon": 37.5860711,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8497684164",\n   "kind": "organisation",\n   "name": "Представительство Кузбасса при правительстве Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7090758,\n   "lon": 37.5785456,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8499187314",\n   "kind": "organisation",\n   "name": "Главное управление по обслуживанию дипломатического корпуса при Министерстве иностранных дел Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7760858,\n   "lon": 37.6475555,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8499207065",\n   "kind": "organisation",\n   "name": "Департамент управления дочерними и зависимыми обществами",\n   "office_type": "company",\n   "lat": 55.7732601,\n   "lon": 37.6522078,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8499207079",\n   "kind": "organisation",\n   "name": "Информтехника",\n   "office_type": "research",\n   "lat": 55.7720819,\n   "lon": 37.6513897,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8499207084",\n   "kind": "organisation",\n   "name": "Научно-исследовательский и проектно-конструкторский институт информатизации, автоматизации и связи на железнодорожном транспорте",\n   "office_type": "research",\n   "lat": 55.7721467,\n   "lon": 37.6504737,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8504175008",\n   "kind": "organisation",\n   "name": "ОДС № 1156",\n   "office_type": "property_management",\n   "lat": 55.8644754,\n   "lon": 37.4774471,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8504175009",\n   "kind": "organisation",\n   "name": "ОДС № 968",\n   "office_type": "property_management",\n   "lat": 55.8642482,\n   "lon": 37.477777,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8504175010",\n   "kind": "organisation",\n   "name": "ООО \\"ДЕЗ Левобережный\\"",\n   "office_type": "property_management",\n   "lat": 55.8590762,\n   "lon": 37.4742579,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8507561288",\n   "kind": "organisation",\n   "name": "Энергия",\n   "office_type": "logistics",\n   "lat": 55.6027084,\n   "lon": 37.6517209,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8509324567",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.759965,\n   "lon": 37.5783373,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/8511850969",\n   "kind": "organisation",\n   "name": "Альфа страхование",\n   "office_type": "insurance",\n   "lat": 55.7934529,\n   "lon": 37.7143444,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8512645081",\n   "kind": "organisation",\n   "name": "Центр гигиены и эпидемиологии в Московской области",\n   "office_type": "government",\n   "lat": 55.8969975,\n   "lon": 37.4430668,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8515930488",\n   "kind": "organisation",\n   "name": "Фонд социального страхования Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7715757,\n   "lon": 37.6477801,\n   "opening_hours": "Mo-Th 09:00-12:30,13:15-18:00; Fr 09:00-12:30,13:15-16:45"\n  },\n  {\n   "id": "node/8515930494",\n   "kind": "organisation",\n   "name": "Управление служебными зданиями",\n   "office_type": "government",\n   "lat": 55.7715644,\n   "lon": 37.6471049,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8517392270",\n   "kind": "organisation",\n   "name": "Централизованная бухгалтерия №1 ЦОУО",\n   "office_type": "yes",\n   "lat": 55.7643819,\n   "lon": 37.5633197,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8519517970",\n   "kind": "organisation",\n   "name": "Федеральная служба по надзору в сфере защиты прав потребителей и благополучия человека",\n   "office_type": "government",\n   "lat": 55.897216,\n   "lon": 37.4423163,\n   "opening_hours": "Mo-Fr 09:00-13:00,13:45-17:00; PH off"\n  },\n  {\n   "id": "node/8521187094",\n   "kind": "organisation",\n   "name": "Стекольная компания",\n   "office_type": "company",\n   "lat": 55.7223035,\n   "lon": 37.678476,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8521187108",\n   "kind": "organisation",\n   "name": "Компания по установке натяжных потолков",\n   "office_type": "company",\n   "lat": 55.7581869,\n   "lon": 37.6824049,\n   "opening_hours": "Mo-Sa 09:00-21:00"\n  },\n  {\n   "id": "node/8521187109",\n   "kind": "organisation",\n   "name": "Империя радости",\n   "office_type": "company",\n   "lat": 55.7496789,\n   "lon": 37.5376117,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/8521187111",\n   "kind": "organisation",\n   "name": "Гильдия Инженеров",\n   "office_type": "company",\n   "lat": 55.7463499,\n   "lon": 37.6822177,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8521191336",\n   "kind": "organisation",\n   "name": "Headway",\n   "office_type": "estate_agent",\n   "lat": 55.7496304,\n   "lon": 37.5379148,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/8521191349",\n   "kind": "organisation",\n   "name": "Тесла Телекоммуникации",\n   "office_type": "telecommunication",\n   "lat": 55.7958141,\n   "lon": 37.5032258,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/8521191350",\n   "kind": "organisation",\n   "name": "Разгруз",\n   "office_type": "logistics",\n   "lat": 55.665103,\n   "lon": 37.471126,\n   "opening_hours": "Mo-Su 08:00-23:00"\n  },\n  {\n   "id": "node/8521191355",\n   "kind": "organisation",\n   "name": "Банкрот-Сервис",\n   "office_type": "lawyer",\n   "lat": 55.7626045,\n   "lon": 37.6575226,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/8521191364",\n   "kind": "organisation",\n   "name": "arb design",\n   "office_type": "company",\n   "lat": 55.842944,\n   "lon": 37.480073,\n   "opening_hours": "Mo-Su 10:00-13:00,14:00-20:00"\n  },\n  {\n   "id": "node/8521191375",\n   "kind": "organisation",\n   "name": "Строймаксимум",\n   "office_type": "company",\n   "lat": 55.7586355,\n   "lon": 37.5511944,\n   "opening_hours": "Mo-Sa 09:00-19:00"\n  },\n  {\n   "id": "node/8521191380",\n   "kind": "organisation",\n   "name": "Московская тепловая компания",\n   "office_type": "company",\n   "lat": 55.6833189,\n   "lon": 37.6626134,\n   "opening_hours": "Mo-Fr 09:00-19:00;Sa 10:00-19:00"\n  },\n  {\n   "id": "node/8521191390",\n   "kind": "organisation",\n   "name": "Unlockbest",\n   "office_type": "company",\n   "lat": 55.6098234,\n   "lon": 37.605072,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/8521191402",\n   "kind": "organisation",\n   "name": "Klary Nails",\n   "office_type": "educational_institution",\n   "lat": 55.8133668,\n   "lon": 37.6950252,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/8521191410",\n   "kind": "organisation",\n   "name": "Dellon",\n   "office_type": "educational_institution",\n   "lat": 55.6052404,\n   "lon": 37.7384233,\n   "opening_hours": "Mo-Fr 09:00-19:00;Sa-Su 09:00-18:00"\n  },\n  {\n   "id": "node/8536326573",\n   "kind": "organisation",\n   "name": "Адвокатское бюро «Казаков и Партнёры»",\n   "office_type": "lawyer",\n   "lat": 55.77052,\n   "lon": 37.6063019,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8543792318",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7262295,\n   "lon": 37.6810509,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8545158232",\n   "kind": "organisation",\n   "name": "ПИК комфорт",\n   "office_type": "property_management",\n   "lat": 55.5851395,\n   "lon": 37.6020096,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8555150389",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7819128,\n   "lon": 37.634253,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8558934489",\n   "kind": "organisation",\n   "name": "Совет ветеранов войны, труда, вооружённых сил и правоохранительных органов",\n   "office_type": "association",\n   "lat": 55.8873308,\n   "lon": 37.4425992,\n   "opening_hours": "Tu-Th 10:00-13:00; PH off"\n  },\n  {\n   "id": "node/8561283637",\n   "kind": "organisation",\n   "name": "АО \\"Авиасалон\\"",\n   "office_type": "company",\n   "lat": 55.7246767,\n   "lon": 37.5735654,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8566375799",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8946172,\n   "lon": 37.452108,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8567296919",\n   "kind": "organisation",\n   "name": "Приёмная депутата Паниной Елены Владимировны",\n   "office_type": "government",\n   "lat": 55.704464,\n   "lon": 37.6536768,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8568458842",\n   "kind": "organisation",\n   "name": "ООО «ЗВИ»",\n   "office_type": "company",\n   "lat": 55.7925526,\n   "lon": 37.5281945,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8569051017",\n   "kind": "organisation",\n   "name": "Следственный комитет Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7259087,\n   "lon": 37.5854494,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8569767785",\n   "kind": "organisation",\n   "name": "Алмазная палата России",\n   "office_type": "company",\n   "lat": 55.7897576,\n   "lon": 37.5455376,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8570499614",\n   "kind": "organisation",\n   "name": "Торговый комплекс «Вятский»",\n   "office_type": "company",\n   "lat": 55.8007872,\n   "lon": 37.5778127,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8576451494",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.8693036,\n   "lon": 37.527996,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8576451514",\n   "kind": "organisation",\n   "name": "Глобус",\n   "office_type": "logistics",\n   "lat": 55.8876817,\n   "lon": 37.6404162,\n   "opening_hours": "Mo-Su 09:00-00:00"\n  },\n  {\n   "id": "node/8578772355",\n   "kind": "organisation",\n   "name": "Политехресурс",\n   "office_type": "company",\n   "lat": 55.7472063,\n   "lon": 37.6305085,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8582209380",\n   "kind": "organisation",\n   "name": "Правовые ответы",\n   "office_type": "lawyer",\n   "lat": 55.7578735,\n   "lon": 37.80196,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8582691985",\n   "kind": "organisation",\n   "name": "Ассоциация экономического сотрудничества со странами Африки",\n   "office_type": "association",\n   "lat": 55.7535519,\n   "lon": 37.5812134,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/8586593642",\n   "kind": "organisation",\n   "name": "Freedom international group",\n   "office_type": "company",\n   "lat": 55.7556199,\n   "lon": 37.5846443,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8589386838",\n   "kind": "organisation",\n   "name": "Tablogix",\n   "office_type": "company",\n   "lat": 55.7619556,\n   "lon": 37.4581486,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8589968673",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6120842,\n   "lon": 37.7463809,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8590456239",\n   "kind": "organisation",\n   "name": "Научно-исследовательский клинический институт отолорингологии им Л. И. Свержевского",\n   "office_type": "research",\n   "lat": 55.7733525,\n   "lon": 37.5440175,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa,Su 09:00-16:00"\n  },\n  {\n   "id": "node/8596146193",\n   "kind": "organisation",\n   "name": "Инж магистраль",\n   "office_type": "company",\n   "lat": 55.7721498,\n   "lon": 37.5454652,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8601148583",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "company",\n   "lat": 55.8031496,\n   "lon": 37.5550058,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa,Su 10:00-19:00"\n  },\n  {\n   "id": "node/8610122459",\n   "kind": "organisation",\n   "name": "РТ-Техприемка",\n   "office_type": "company",\n   "lat": 55.7721123,\n   "lon": 37.5736977,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8614547347",\n   "kind": "organisation",\n   "name": "Дирекция единого заказчика по строительству, капитальному и текущему ремонту объектов образования и науки",\n   "office_type": "government",\n   "lat": 55.6749649,\n   "lon": 37.5515109,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8616790386",\n   "kind": "organisation",\n   "name": "Группа компаний ДУКС",\n   "office_type": "company",\n   "lat": 55.784667,\n   "lon": 37.579127,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8616790494",\n   "kind": "organisation",\n   "name": "Dr. Web",\n   "office_type": "it",\n   "lat": 55.7824,\n   "lon": 37.5821163,\n   "opening_hours": "Mo-Su 09:30-18:00"\n  },\n  {\n   "id": "node/8617061813",\n   "kind": "organisation",\n   "name": "Пятёрочка",\n   "office_type": "company",\n   "lat": 55.7973008,\n   "lon": 37.5784537,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8617421962",\n   "kind": "organisation",\n   "name": "ARO group",\n   "office_type": "company",\n   "lat": 55.7734146,\n   "lon": 37.6312523,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8617461317",\n   "kind": "organisation",\n   "name": "Военный комиссариат Останкинского района северо-восточного административного округа города Москвы",\n   "office_type": "government",\n   "lat": 55.7674478,\n   "lon": 37.6290314,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8618100300",\n   "kind": "organisation",\n   "name": "Московский спорт",\n   "office_type": "government",\n   "lat": 55.7880513,\n   "lon": 37.7305967,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8629644307",\n   "kind": "organisation",\n   "name": "Роскадастр",\n   "office_type": "government",\n   "lat": 55.7641268,\n   "lon": 37.6422527,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8634183432",\n   "kind": "organisation",\n   "name": "Жилищник участок №3",\n   "office_type": "property_management",\n   "lat": 55.8833634,\n   "lon": 37.4519779,\n   "opening_hours": "Mo,We 09:00-12:00; Tu 15:00-19:00; PH off"\n  },\n  {\n   "id": "node/8634490317",\n   "kind": "organisation",\n   "name": "ФС-Групп",\n   "office_type": "company",\n   "lat": 55.7637846,\n   "lon": 37.526679,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8635394131",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.8654387,\n   "lon": 37.6046455,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/8635394136",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.6937029,\n   "lon": 37.5479436,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00;Sa 10:00-15:00"\n  },\n  {\n   "id": "node/8635394137",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.6828456,\n   "lon": 37.5740334,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/8635394138",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.599117,\n   "lon": 37.531207,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/8635394149",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.677209,\n   "lon": 37.452785,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/8635395211",\n   "kind": "organisation",\n   "name": "Школа-студия Шаталова-Лысенковой",\n   "office_type": "company",\n   "lat": 55.7272974,\n   "lon": 37.6099133,\n   "opening_hours": "We-Su 10:30-17:00"\n  },\n  {\n   "id": "node/8635395214",\n   "kind": "organisation",\n   "name": "Хозяйственный магазин",\n   "office_type": "company",\n   "lat": 55.7393413,\n   "lon": 37.4899274,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/8635395215",\n   "kind": "organisation",\n   "name": "Хобби-Экспресс",\n   "office_type": "company",\n   "lat": 55.7093341,\n   "lon": 37.6544675,\n   "opening_hours": "Fr 09:00-13:00,14:00-17:00;Mo-Th 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/8635426730",\n   "kind": "organisation",\n   "name": "Банкрот Экспресс",\n   "office_type": "company",\n   "lat": 55.6403399,\n   "lon": 37.533001,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/8635426742",\n   "kind": "organisation",\n   "name": "Kibrology",\n   "office_type": "educational_institution",\n   "lat": 55.7543188,\n   "lon": 37.5121146,\n   "opening_hours": "Mo-Fr 10:00-21:00"\n  },\n  {\n   "id": "node/8635426747",\n   "kind": "organisation",\n   "name": "Gartel",\n   "office_type": "telecommunication",\n   "lat": 55.7765775,\n   "lon": 37.6790071,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/8635426758",\n   "kind": "organisation",\n   "name": "Адвокатский кабинет Шутовой Н.В.",\n   "office_type": "lawyer",\n   "lat": 55.7640819,\n   "lon": 37.5791135,\n   "opening_hours": "Mo-Fr 09:00-21:00;Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8640835008",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "estate_agent",\n   "lat": 55.788803,\n   "lon": 37.4753027,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8647827682",\n   "kind": "organisation",\n   "name": "Общество охотников и рыболовов восточного административного округа",\n   "office_type": "association",\n   "lat": 55.7817373,\n   "lon": 37.6766719,\n   "opening_hours": "Tu-Th 10:00-18:00"\n  },\n  {\n   "id": "node/8647921326",\n   "kind": "organisation",\n   "name": "Ароса",\n   "office_type": "company",\n   "lat": 55.7766356,\n   "lon": 37.6435281,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8648191349",\n   "kind": "organisation",\n   "name": "SberAutoTech",\n   "office_type": "company",\n   "lat": 55.693597,\n   "lon": 37.6655745,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8653270825",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7964096,\n   "lon": 37.7080402,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8656051421",\n   "kind": "organisation",\n   "name": "АО Авиационная промышленность",\n   "office_type": "company",\n   "lat": 55.7691171,\n   "lon": 37.640805,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8657534455",\n   "kind": "organisation",\n   "name": "Главкосмос",\n   "office_type": "company",\n   "lat": 55.7834732,\n   "lon": 37.6320375,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8657618596",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт природных, синтетических алмазов и инструмента",\n   "office_type": "research",\n   "lat": 55.7892215,\n   "lon": 37.6327631,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8659444617",\n   "kind": "organisation",\n   "name": "Страховая группа СОГАЗ",\n   "office_type": "insurance",\n   "lat": 55.6643793,\n   "lon": 37.5744012,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8664452733",\n   "kind": "organisation",\n   "name": "Силтэк",\n   "office_type": "company",\n   "lat": 55.7655924,\n   "lon": 37.6406032,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8674284361",\n   "kind": "organisation",\n   "name": "ОДС № 4 ГБУ \\"Жилищник района Сокол\\"",\n   "office_type": "property_management",\n   "lat": 55.8003154,\n   "lon": 37.4992803,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8677638961",\n   "kind": "organisation",\n   "name": "Персона",\n   "office_type": "travel_agent",\n   "lat": 55.7697388,\n   "lon": 37.6366771,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/8677638970",\n   "kind": "organisation",\n   "name": "Нотариус Гончаров Ф. Ю.",\n   "office_type": "notary",\n   "lat": 55.768755,\n   "lon": 37.6405154,\n   "opening_hours": "Mo-Th 09:30-17:30; Fr 09:30-17:00"\n  },\n  {\n   "id": "node/8677638971",\n   "kind": "organisation",\n   "name": "СОГАЗ-Жизнь",\n   "office_type": "insurance",\n   "lat": 55.7695441,\n   "lon": 37.6406139,\n   "opening_hours": "Mo-Th 09:00-12:30,13:15-18:00; Fr 09:00-12:30,13:15-16:45"\n  },\n  {\n   "id": "node/8678169837",\n   "kind": "organisation",\n   "name": "ОДС-8",\n   "office_type": "property_management",\n   "lat": 55.7658896,\n   "lon": 37.6557375,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8678169870",\n   "kind": "organisation",\n   "name": "Парламентское собрание союза Белоруси и России",\n   "office_type": "government",\n   "lat": 55.7689089,\n   "lon": 37.6447472,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8678169871",\n   "kind": "organisation",\n   "name": "Министерство транспорта РФ",\n   "office_type": "government",\n   "lat": 55.7698821,\n   "lon": 37.6439881,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8678169872",\n   "kind": "organisation",\n   "name": "Федеральный центр нормирования",\n   "office_type": "government",\n   "lat": 55.7695909,\n   "lon": 37.6450892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8678169873",\n   "kind": "organisation",\n   "name": "ОДС №1000",\n   "office_type": "property_management",\n   "lat": 55.7698006,\n   "lon": 37.6488993,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/8678169875",\n   "kind": "organisation",\n   "name": "Центр фирменного транспортного обслуживания ржд",\n   "office_type": "company",\n   "lat": 55.7715244,\n   "lon": 37.6522091,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8679103837",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7925031,\n   "lon": 37.4943016,\n   "opening_hours": "Mo-Fr 10:00-12:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8679266072",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7077128,\n   "lon": 37.6206306,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/8679297322",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.8084625,\n   "lon": 37.5102121,\n   "opening_hours": "Mo-Sa 10:00-19:00"\n  },\n  {\n   "id": "node/8679393758",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6151321,\n   "lon": 37.7214921,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/8680246958",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6690686,\n   "lon": 37.4534162,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/8682722899",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7400853,\n   "lon": 37.784092,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/8683003076",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7791667,\n   "lon": 37.6071907,\n   "opening_hours": "Mo-Fr 09:30-17:30"\n  },\n  {\n   "id": "node/8684306021",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.8241808,\n   "lon": 37.6344316,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/8684320061",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7640491,\n   "lon": 37.5576914,\n   "opening_hours": "Mo-Fr 10:00-20:00 Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8690911182",\n   "kind": "organisation",\n   "name": "ОДС №17",\n   "office_type": "property_management",\n   "lat": 55.7746767,\n   "lon": 37.7061027,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8693258071",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7406593,\n   "lon": 37.6575702,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8694279766",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6203926,\n   "lon": 37.6088257,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8696721358",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.724646,\n   "lon": 37.6426586,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:00"\n  },\n  {\n   "id": "node/8704582304",\n   "kind": "organisation",\n   "name": "Отдел по работе с обращениями граждан",\n   "office_type": "company",\n   "lat": 55.7691624,\n   "lon": 37.6529454,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8704582306",\n   "kind": "organisation",\n   "name": "Отдел служебных билетов",\n   "office_type": "company",\n   "lat": 55.7690413,\n   "lon": 37.6549248,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8705265688",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.8458403,\n   "lon": 37.4776236,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/8705321494",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7897927,\n   "lon": 37.6786778,\n   "opening_hours": "Mo-Th 10:00-19:00; Fr 10:00-17:45"\n  },\n  {\n   "id": "node/8705569159",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7021053,\n   "lon": 37.6948577,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/8706186042",\n   "kind": "organisation",\n   "name": "ТСЖ \\"Колизей\\"",\n   "office_type": "property_management",\n   "lat": 55.6995314,\n   "lon": 37.5116844,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8706186043",\n   "kind": "organisation",\n   "name": "ТСЖ \\"Достояние\\"",\n   "office_type": "property_management",\n   "lat": 55.6965231,\n   "lon": 37.5071272,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8706186044",\n   "kind": "organisation",\n   "name": "ТСЖ \\"Солнечный берег\\"",\n   "office_type": "property_management",\n   "lat": 55.6964518,\n   "lon": 37.4993484,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8707023163",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.8707426,\n   "lon": 37.677966,\n   "opening_hours": "Mo-Fr 10:00-12:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/8709508496",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7207929,\n   "lon": 37.774229,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-19:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/8709603495",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6542051,\n   "lon": 37.7638963,\n   "opening_hours": "Mo-Fr 09:00-14:00, 15:00-20:00: Ыф 10:00-14:00, 15:00-19:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/8709607208",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7826864,\n   "lon": 37.7049984,\n   "opening_hours": "Mo-Fr 09:00-14:00, 15:00-20:00; Sa 10:00-14:00, 15:00-19:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/8709612320",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.673881,\n   "lon": 37.6615658,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/8709625697",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6498157,\n   "lon": 37.5295808,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/8710379349",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7039382,\n   "lon": 37.5088508,\n   "opening_hours": "Mo-Fr 09:00-20:00: Sa 10:00-19:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/8710415283",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6331393,\n   "lon": 37.5214481,\n   "opening_hours": "Mo-Fr 09:00-20:00: Sa 10:00-19:00"\n  },\n  {\n   "id": "node/8710421849",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.8656164,\n   "lon": 37.6060202,\n   "opening_hours": "Mo-Fr 09:00-20:00: Sa 10:00-19:00"\n  },\n  {\n   "id": "node/8710715183",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7817518,\n   "lon": 37.5346153,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/8710757648",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6584194,\n   "lon": 37.6184163,\n   "opening_hours": "Mo-Sa 09:00-20:00: Su 09^00-17:00"\n  },\n  {\n   "id": "node/8713090762",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7605011,\n   "lon": 37.5997636,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/8713122911",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7422988,\n   "lon": 37.5568889,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/8713123471",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7742717,\n   "lon": 37.6481132,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/8713132487",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.7504021,\n   "lon": 37.7788827,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-19:00; Su 10:00-18:00"\n  },\n  {\n   "id": "node/8713146662",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.6858079,\n   "lon": 37.5478134,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/8713151437",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.8551278,\n   "lon": 37.5646555,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/8715324108",\n   "kind": "organisation",\n   "name": "Восточный окружной комитет КПРФ",\n   "office_type": "political_party",\n   "lat": 55.8083049,\n   "lon": 37.7294084,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8715324114",\n   "kind": "organisation",\n   "name": "Центр по работе с населением \\"Преображенец\\"",\n   "office_type": "government",\n   "lat": 55.8083418,\n   "lon": 37.7293481,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8715324702",\n   "kind": "organisation",\n   "name": "Научная лаборатория кафедры теории и методики велосипедного спорта",\n   "office_type": "research",\n   "lat": 55.800497,\n   "lon": 37.7615708,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8723755185",\n   "kind": "organisation",\n   "name": "Инженерная служба района \\"Савеовский\\"",\n   "office_type": "property_management",\n   "lat": 55.7998111,\n   "lon": 37.5753853,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/8723755186",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник Савеловского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7998487,\n   "lon": 37.5753906,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/8723755187",\n   "kind": "organisation",\n   "name": "Пункт оповещения №4",\n   "office_type": "government",\n   "lat": 55.7975805,\n   "lon": 37.5689694,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8723755188",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичной организации №3 района Савеловский САО г. Москвы",\n   "office_type": "association",\n   "lat": 55.7975827,\n   "lon": 37.5690834,\n   "opening_hours": "Mo-Th 11:00-15:00"\n  },\n  {\n   "id": "node/8723755189",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник Савеловского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7975948,\n   "lon": 37.5690204,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/8723755196",\n   "kind": "organisation",\n   "name": "Авиационно-космическое научно-исследовательское испытательное производственное объединение",\n   "office_type": "company",\n   "lat": 55.8029589,\n   "lon": 37.5725904,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8724507311",\n   "kind": "organisation",\n   "name": "Бюджеткин",\n   "office_type": "company",\n   "lat": 55.8121491,\n   "lon": 37.6353943,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8724555471",\n   "kind": "organisation",\n   "name": "ФГУП «РОСТЭК»",\n   "office_type": "company",\n   "lat": 55.82365,\n   "lon": 37.6612321,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8725494639",\n   "kind": "organisation",\n   "name": "Якушенко Е. А.",\n   "office_type": "notary",\n   "lat": 55.7729116,\n   "lon": 37.5769034,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8726076226",\n   "kind": "organisation",\n   "name": "Авиационно-космическое научно-исследовательское испытательное производственное объединение",\n   "office_type": "company",\n   "lat": 55.8003206,\n   "lon": 37.5388697,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8726076227",\n   "kind": "organisation",\n   "name": "Городской психолого-педагогический центр",\n   "office_type": "government",\n   "lat": 55.7931618,\n   "lon": 37.5062218,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/8726076228",\n   "kind": "organisation",\n   "name": "АО \\"ДХЛ интернешнл\\"",\n   "office_type": "company",\n   "lat": 55.8059135,\n   "lon": 37.5501215,\n   "opening_hours": "Mo-Fr 08:30-18:30"\n  },\n  {\n   "id": "node/8728811034",\n   "kind": "organisation",\n   "name": "SAP",\n   "office_type": "it",\n   "lat": 55.7346389,\n   "lon": 37.6460776,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8731799322",\n   "kind": "organisation",\n   "name": "Московская областная коллегия адвокатов №7",\n   "office_type": "lawyer",\n   "lat": 55.7404132,\n   "lon": 37.6617042,\n   "opening_hours": "Mo-Fr 10:00-18:00; PH closed"\n  },\n  {\n   "id": "node/8739564181",\n   "kind": "organisation",\n   "name": "Paker и Waterman",\n   "office_type": "company",\n   "lat": 55.8032521,\n   "lon": 37.5824395,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/8741759429",\n   "kind": "organisation",\n   "name": "Социальный фонд России - клиентская служба Беговой, Хорошевский",\n   "office_type": "government",\n   "lat": 55.7865401,\n   "lon": 37.5604896,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45; Sa,Su off"\n  },\n  {\n   "id": "node/8741759430",\n   "kind": "organisation",\n   "name": "ОДС № 1245",\n   "office_type": "property_management",\n   "lat": 55.7751934,\n   "lon": 37.5521748,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8746266862",\n   "kind": "organisation",\n   "name": "Российский государственный архив новейшей истории",\n   "office_type": "government",\n   "lat": 55.7482476,\n   "lon": 37.6223454,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8747512325",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.622698,\n   "lon": 37.6058743,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/8750268747",\n   "kind": "organisation",\n   "name": "Ашробот",\n   "office_type": "yes",\n   "lat": 55.7760812,\n   "lon": 37.6777777,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8750547851",\n   "kind": "organisation",\n   "name": "Департамент информационных технологий, связи и защиты информации МВД России",\n   "office_type": "government",\n   "lat": 55.7290804,\n   "lon": 37.6339124,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8750717156",\n   "kind": "organisation",\n   "name": "Всероссийское общество глухих. Московское городское правление",\n   "office_type": "ngo",\n   "lat": 55.7698345,\n   "lon": 37.6341284,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8752557688",\n   "kind": "organisation",\n   "name": "СИОР",\n   "office_type": "company",\n   "lat": 55.8043623,\n   "lon": 37.5861275,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/8752557689",\n   "kind": "organisation",\n   "name": "Центр Бытовых Услуг",\n   "office_type": "company",\n   "lat": 55.7362654,\n   "lon": 37.6565436,\n   "opening_hours": "Mo-Su 07:00-00:00"\n  },\n  {\n   "id": "node/8752557694",\n   "kind": "organisation",\n   "name": "Московская Макулатура",\n   "office_type": "company",\n   "lat": 55.6557731,\n   "lon": 37.6844251,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/8752557703",\n   "kind": "organisation",\n   "name": "Цифровые Технологии",\n   "office_type": "company",\n   "lat": 55.7205602,\n   "lon": 37.6360864,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/8752557704",\n   "kind": "organisation",\n   "name": "Центр Бытовых Услуг",\n   "office_type": "company",\n   "lat": 55.756935,\n   "lon": 37.535106,\n   "opening_hours": "Mo-Su 07:00-00:00"\n  },\n  {\n   "id": "node/8752557705",\n   "kind": "organisation",\n   "name": "Центр Бытовых Услуг",\n   "office_type": "company",\n   "lat": 55.7091014,\n   "lon": 37.6540197,\n   "opening_hours": "Mo-Su 07:00-00:00"\n  },\n  {\n   "id": "node/8752557706",\n   "kind": "organisation",\n   "name": "Центр Бытовых Услуг",\n   "office_type": "company",\n   "lat": 55.6612142,\n   "lon": 37.5105912,\n   "opening_hours": "Mo-Su 07:00-00:00"\n  },\n  {\n   "id": "node/8752557714",\n   "kind": "organisation",\n   "name": "Коллегия Правовой Поддержки",\n   "office_type": "lawyer",\n   "lat": 55.7565738,\n   "lon": 37.6063728,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/8752581918",\n   "kind": "organisation",\n   "name": "ИнтСтайл",\n   "office_type": "company",\n   "lat": 55.6530236,\n   "lon": 37.7705321,\n   "opening_hours": "Mo-Fr 09:00-19:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8752581925",\n   "kind": "organisation",\n   "name": "АвтоЭкс-групп",\n   "office_type": "insurance",\n   "lat": 55.8641653,\n   "lon": 37.6521313,\n   "opening_hours": "Mo-Fr 08:00-19:00"\n  },\n  {\n   "id": "node/8752581933",\n   "kind": "organisation",\n   "name": "Royal Preschool Education",\n   "office_type": "company",\n   "lat": 55.7234215,\n   "lon": 37.6538678,\n   "opening_hours": "Mo-Fr 08:00-19:30"\n  },\n  {\n   "id": "node/8752581967",\n   "kind": "organisation",\n   "name": "Blumart",\n   "office_type": "company",\n   "lat": 55.6502439,\n   "lon": 37.831172,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/8754760534",\n   "kind": "organisation",\n   "name": "Департамент по делам гражданской обороны, чрезвычайным ситуациям и пожарной безопасности города Москвы",\n   "office_type": "government",\n   "lat": 55.7795794,\n   "lon": 37.6677632,\n   "opening_hours": "Mo-Th 08:00-12:15,13:00-16:00; Fr 08:00-12:15,13:00-15:00"\n  },\n  {\n   "id": "node/8754760535",\n   "kind": "organisation",\n   "name": "АО \\"СМНУ-Связь\\"",\n   "office_type": "company",\n   "lat": 55.7795206,\n   "lon": 37.6676466,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8755670118",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7193101,\n   "lon": 37.4613286,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8758277929",\n   "kind": "organisation",\n   "name": "TezTour",\n   "office_type": "travel_agent",\n   "lat": 55.740674,\n   "lon": 37.6576916,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8758277930",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7408971,\n   "lon": 37.657739,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8758277939",\n   "kind": "organisation",\n   "name": "TUI",\n   "office_type": "travel_agent",\n   "lat": 55.7412155,\n   "lon": 37.6581931,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8758277946",\n   "kind": "organisation",\n   "name": "Coral Travel",\n   "office_type": "travel_agent",\n   "lat": 55.7411686,\n   "lon": 37.658588,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8758277951",\n   "kind": "organisation",\n   "name": "Pegas Touristik",\n   "office_type": "travel_agent",\n   "lat": 55.7411931,\n   "lon": 37.6583956,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8758277964",\n   "kind": "organisation",\n   "name": "1001ТУР",\n   "office_type": "travel_agent",\n   "lat": 55.7413415,\n   "lon": 37.6586905,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8758277970",\n   "kind": "organisation",\n   "name": "Мостурфлот",\n   "office_type": "travel_agent",\n   "lat": 55.7408463,\n   "lon": 37.6577752,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa-Su 09:00-18:00"\n  },\n  {\n   "id": "node/8766243994",\n   "kind": "organisation",\n   "name": "ГБУ Личность",\n   "office_type": "government",\n   "lat": 55.6348885,\n   "lon": 37.6752879,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8772075053",\n   "kind": "organisation",\n   "name": "X5 Group",\n   "office_type": "company",\n   "lat": 55.7289452,\n   "lon": 37.620441,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8776503185",\n   "kind": "organisation",\n   "name": "APRELarchitects",\n   "office_type": "architect",\n   "lat": 55.7594575,\n   "lon": 37.6456911,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8777424052",\n   "kind": "organisation",\n   "name": "ОДС № 28",\n   "office_type": "property_management",\n   "lat": 55.8185235,\n   "lon": 37.4887499,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8779237606",\n   "kind": "organisation",\n   "name": "Согаз-мед",\n   "office_type": "insurance",\n   "lat": 55.7040694,\n   "lon": 37.6686873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8789982293",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6829602,\n   "lon": 37.5497216,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/8805289978",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.658606,\n   "lon": 37.7416088,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/8810943068",\n   "kind": "organisation",\n   "name": "Архивная служба Управления делами Московской железной дороги. Объединенный архив",\n   "office_type": "yes",\n   "lat": 55.771068,\n   "lon": 37.5599545,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8810943069",\n   "kind": "organisation",\n   "name": "Расчетно-кассовый центр Центрального банка РФ",\n   "office_type": "bank",\n   "lat": 55.7726982,\n   "lon": 37.5488274,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8810943070",\n   "kind": "organisation",\n   "name": "Агентство по закупкам (контрактная служба) Департамента здравоохранения города Москвы",\n   "office_type": "government",\n   "lat": 55.7766741,\n   "lon": 37.5540563,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8811274922",\n   "kind": "organisation",\n   "name": "Инспекция № 9 по г. Москве (ЦАО) Федеральной налоговой службы России",\n   "office_type": "government",\n   "lat": 55.7055501,\n   "lon": 37.7238804,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8824605017",\n   "kind": "organisation",\n   "name": "ТехМашПоставка",\n   "office_type": "company",\n   "lat": 55.7597951,\n   "lon": 37.744504,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/8825725922",\n   "kind": "organisation",\n   "name": "Главное производственно-коммерческое управление по обслуживанию дипломатического корпуса",\n   "office_type": "government",\n   "lat": 55.742651,\n   "lon": 37.6061502,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8829912979",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7392335,\n   "lon": 37.6663638,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/8829940033",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "educational_institution",\n   "lat": 55.7387372,\n   "lon": 37.6670838,\n   "opening_hours": "Mo-Su 10:00-23:00"\n  },\n  {\n   "id": "node/8829940056",\n   "kind": "organisation",\n   "name": "Адвокатская контора №10",\n   "office_type": "lawyer",\n   "lat": 55.7389548,\n   "lon": 37.6649064,\n   "opening_hours": "Mo-Th 10:00-20:00; Fr 10:00-19:00"\n  },\n  {\n   "id": "node/8829940068",\n   "kind": "organisation",\n   "name": "Таганский детский фонд",\n   "office_type": "charity",\n   "lat": 55.7384032,\n   "lon": 37.6645512,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8829940079",\n   "kind": "organisation",\n   "name": "Городской центр профессионального и карьерного развития",\n   "office_type": "government",\n   "lat": 55.7390927,\n   "lon": 37.6672873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8836849696",\n   "kind": "organisation",\n   "name": "ГБУ \\"Центр по работе с населением ЦАО\\". Филиал \\"Перекрёсток\\"",\n   "office_type": "government",\n   "lat": 55.7351095,\n   "lon": 37.6680081,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8838959987",\n   "kind": "organisation",\n   "name": "Московско-Курское территориальное управление Московской железной дороги",\n   "office_type": "company",\n   "lat": 55.7828359,\n   "lon": 37.6580107,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8839129797",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.6913399,\n   "lon": 37.5495261,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/8839129803",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.7465748,\n   "lon": 37.5599251,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/8839129804",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.787231,\n   "lon": 37.634373,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/8839129805",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.7093448,\n   "lon": 37.620838,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/8839129806",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.7311458,\n   "lon": 37.6350885,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/8839129807",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.7801481,\n   "lon": 37.5995171,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/8839209395",\n   "kind": "organisation",\n   "name": "Easy Visa",\n   "office_type": "visa",\n   "lat": 55.7540034,\n   "lon": 37.6353729,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/8839210228",\n   "kind": "organisation",\n   "name": "Элероз",\n   "office_type": "company",\n   "lat": 55.7471757,\n   "lon": 37.5835231,\n   "opening_hours": "Mo-Sa 10:00-20:00"\n  },\n  {\n   "id": "node/8839210252",\n   "kind": "organisation",\n   "name": "Remont-Boyler",\n   "office_type": "company",\n   "lat": 55.845818,\n   "lon": 37.585706,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/8839210255",\n   "kind": "organisation",\n   "name": "Лайф Стайл",\n   "office_type": "estate_agent",\n   "lat": 55.7227311,\n   "lon": 37.5659063,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/8839210259",\n   "kind": "organisation",\n   "name": "Риелтор Алексей Барсуков",\n   "office_type": "estate_agent",\n   "lat": 55.6592458,\n   "lon": 37.7465022,\n   "opening_hours": "Mo-Fr 10:00-21:00;Sa-Su 11:00-18:00"\n  },\n  {\n   "id": "node/8839215312",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7566902,\n   "lon": 37.5618642,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/8839215313",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7347582,\n   "lon": 37.6652795,\n   "opening_hours": "Fr 10:00-19:00;Mo-Th 10:00-20:00;Sa 10:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839215315",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.888356,\n   "lon": 37.5886997,\n   "opening_hours": "We-Su 11:00-20:00"\n  },\n  {\n   "id": "node/8839218891",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7630015,\n   "lon": 37.7819931,\n   "opening_hours": "Fr 10:00-19:00;Mo-Th 10:00-20:00;Sa 10:00-18:00;Su 11:00-18:00"\n  },\n  {\n   "id": "node/8839218892",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7735225,\n   "lon": 37.8259733,\n   "opening_hours": "Mo-Fr 09:00-20:00;Sa 09:00-18:00;Su 10:00-18:00"\n  },\n  {\n   "id": "node/8839218893",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.707278,\n   "lon": 37.809965,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218894",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6725978,\n   "lon": 37.5146815,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-18:00"\n  },\n  {\n   "id": "node/8839218897",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8697701,\n   "lon": 37.6312986,\n   "opening_hours": "Tu-Sa 10:00-19:00"\n  },\n  {\n   "id": "node/8839218898",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8861372,\n   "lon": 37.6061635,\n   "opening_hours": "Mo-Fr 10:00-19:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/8839218900",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7487835,\n   "lon": 37.5439551,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/8839218902",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6832858,\n   "lon": 37.5415621,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00"\n  },\n  {\n   "id": "node/8839218903",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.643769,\n   "lon": 37.496075,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00"\n  },\n  {\n   "id": "node/8839218905",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7833171,\n   "lon": 37.6387738,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218906",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6870371,\n   "lon": 37.4653395,\n   "opening_hours": "Mo-Fr 09:00-20:00;Sa-Su 09:00-18:00"\n  },\n  {\n   "id": "node/8839218907",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.779859,\n   "lon": 37.599372,\n   "opening_hours": "Fr 10:00-18:00;Mo-Th 10:00-19:00"\n  },\n  {\n   "id": "node/8839218908",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7335017,\n   "lon": 37.7125777,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218910",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7815976,\n   "lon": 37.4797511,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8839218911",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6289817,\n   "lon": 37.7412665,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218913",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.696387,\n   "lon": 37.494827,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218915",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6936073,\n   "lon": 37.7352214,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8839218917",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.784195,\n   "lon": 37.566357,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00"\n  },\n  {\n   "id": "node/8839218919",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8052833,\n   "lon": 37.7089137,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-18:00"\n  },\n  {\n   "id": "node/8839218920",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8319617,\n   "lon": 37.5199681,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218921",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.767726,\n   "lon": 37.5661369,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218924",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8599568,\n   "lon": 37.6591882,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218925",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.867359,\n   "lon": 37.486697,\n   "opening_hours": "Fr 09:00-19:00; Mo-Th 09:00-20:00; Sa 09:00-18:00; Su 10:00-18:00"\n  },\n  {\n   "id": "node/8839218926",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.861313,\n   "lon": 37.5654664,\n   "opening_hours": "Mo-Fr 10:00-19:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/8839218927",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6231045,\n   "lon": 37.6040554,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/8839218928",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8775102,\n   "lon": 37.5391913,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00"\n  },\n  {\n   "id": "node/8839218932",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6683593,\n   "lon": 37.5599571,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00"\n  },\n  {\n   "id": "node/8839218933",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.6096537,\n   "lon": 37.7250526,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8839218937",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6565423,\n   "lon": 37.6201272,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218941",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7766031,\n   "lon": 37.6880622,\n   "opening_hours": "Fr 09:00-19:00; Mo-Th 09:00-20:00; Sa 09:00-18:00; Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218943",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.5964075,\n   "lon": 37.5997316,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8839218944",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6595681,\n   "lon": 37.6012873,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/8842508417",\n   "kind": "organisation",\n   "name": "Каскад Телеком",\n   "office_type": "telecommunication",\n   "lat": 55.8596444,\n   "lon": 37.6441008,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8850851036",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7076609,\n   "lon": 37.5886884,\n   "opening_hours": "Mo-Th 10:00-13:30,14:30-17:00; Fr 10:00-14:00"\n  },\n  {\n   "id": "node/8866153817",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6994896,\n   "lon": 37.7360793,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8866966943",\n   "kind": "organisation",\n   "name": "Моспромпроект",\n   "office_type": "engineer",\n   "lat": 55.7717994,\n   "lon": 37.5912805,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8868336918",\n   "kind": "organisation",\n   "name": "Офис продаж квартир ЖК \\"Silver\\"",\n   "office_type": "estate_agent",\n   "lat": 55.847859,\n   "lon": 37.6511309,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8870174582",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8474324,\n   "lon": 37.440581,\n   "opening_hours": "Mo-Tu,Th 10:00-14:00,15:00-18:00; We,Sa-Su off; Fr 10:00-14:00,15:00-17:00"\n  },\n  {\n   "id": "node/8870941617",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.7862111,\n   "lon": 37.4479018,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8875626157",\n   "kind": "organisation",\n   "name": "Мосэнергопроект",\n   "office_type": "company",\n   "lat": 55.7693586,\n   "lon": 37.672897,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8879438201",\n   "kind": "organisation",\n   "name": "CBSD Thunderbird",\n   "office_type": "company",\n   "lat": 55.8461673,\n   "lon": 37.5646001,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8879639741",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.6906395,\n   "lon": 37.5516347,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/8880487535",\n   "kind": "organisation",\n   "name": "Национальный Центр Банкротств",\n   "office_type": "company",\n   "lat": 55.7575284,\n   "lon": 37.5748428,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8882918778",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.5864413,\n   "lon": 37.6479876,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/8894683625",\n   "kind": "organisation",\n   "name": "Старт-1",\n   "office_type": "company",\n   "lat": 55.7085491,\n   "lon": 37.6691104,\n   "opening_hours": "Mo-Fr 10:30-18:30"\n  },\n  {\n   "id": "node/8895498792",\n   "kind": "organisation",\n   "name": "Фундамент",\n   "office_type": "architect",\n   "lat": 55.7228529,\n   "lon": 37.6172594,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa,Su 11:00-19:00+"\n  },\n  {\n   "id": "node/8899871344",\n   "kind": "organisation",\n   "name": "Следственное управление по Восточному административному округу",\n   "office_type": "government",\n   "lat": 55.7949117,\n   "lon": 37.7088265,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8907170148",\n   "kind": "organisation",\n   "name": "Центр гигиены и эпидемиологии в городе Москве",\n   "office_type": "government",\n   "lat": 55.8100496,\n   "lon": 37.6335168,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8909994011",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6083691,\n   "lon": 37.6058041,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa-Su 09:00-20:00"\n  },\n  {\n   "id": "node/8911099128",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.8713608,\n   "lon": 37.4853363,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8912000197",\n   "kind": "organisation",\n   "name": "Центр защиты прав граждан (Справедливая Россия)",\n   "office_type": "political_party",\n   "lat": 55.8176097,\n   "lon": 37.5030759,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/8912303149",\n   "kind": "organisation",\n   "name": "ООО «Капитал Металл»",\n   "office_type": "company",\n   "lat": 55.681787,\n   "lon": 37.6332056,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8913661020",\n   "kind": "organisation",\n   "name": "ООО «МТС ДИДЖИТАЛ»",\n   "office_type": "company",\n   "lat": 55.6917322,\n   "lon": 37.6609993,\n   "opening_hours": "Mo-Su 07:10-23:00"\n  },\n  {\n   "id": "node/8916327381",\n   "kind": "organisation",\n   "name": "ДИАЭМ",\n   "office_type": "yes",\n   "lat": 55.8777278,\n   "lon": 37.6849168,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8938894069",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7376093,\n   "lon": 37.4902154,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8938901972",\n   "kind": "organisation",\n   "name": "Капитал Медицинское Страхование",\n   "office_type": "insurance",\n   "lat": 55.6927293,\n   "lon": 37.5469511,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/8938928418",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.693092,\n   "lon": 37.470331,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8938928782",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.61492,\n   "lon": 37.720848,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/8938928801",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6028068,\n   "lon": 37.7138961,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/8938929409",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.645279,\n   "lon": 37.6265377,\n   "opening_hours": "Mo-Fr 09:00-21:00;Sa-Su 10:00-21:00"\n  },\n  {\n   "id": "node/8938929462",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7178742,\n   "lon": 37.6465791,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/8938929487",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.8601239,\n   "lon": 37.4815911,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8938929527",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6898705,\n   "lon": 37.4927356,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8938929706",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.6707072,\n   "lon": 37.7705696,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/8938929752",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.870769,\n   "lon": 37.485071,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/8942661931",\n   "kind": "organisation",\n   "name": "Верум",\n   "office_type": "lawyer",\n   "lat": 55.7153238,\n   "lon": 37.6181357,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8949627672",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7456575,\n   "lon": 37.7900045,\n   "opening_hours": "Mo-Sa 11:00-19:00; Su 11:00-15:00"\n  },\n  {\n   "id": "node/8963815024",\n   "kind": "organisation",\n   "name": "Нагиев и Партнёры",\n   "office_type": "lawyer",\n   "lat": 55.7390586,\n   "lon": 37.657351,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/8963815033",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7394384,\n   "lon": 37.6580194,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/8971324117",\n   "kind": "organisation",\n   "name": "ООО \\"Вектор\\"",\n   "office_type": "company",\n   "lat": 55.8048025,\n   "lon": 37.6424405,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8971324133",\n   "kind": "organisation",\n   "name": "Мостранспроект",\n   "office_type": "company",\n   "lat": 55.8155423,\n   "lon": 37.5731751,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8974834308",\n   "kind": "organisation",\n   "name": "Институт социально-экономических и политических исследований",\n   "office_type": "association",\n   "lat": 55.7420236,\n   "lon": 37.6282489,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8976794369",\n   "kind": "organisation",\n   "name": "Многогруз",\n   "office_type": "moving_company",\n   "lat": 55.7094729,\n   "lon": 37.6527763,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8977672935",\n   "kind": "organisation",\n   "name": "Правительство Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7547717,\n   "lon": 37.5738054,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8980758574",\n   "kind": "organisation",\n   "name": "Центр гигиены и эпидемиологии в городе Москве",\n   "office_type": "government",\n   "lat": 55.8024312,\n   "lon": 37.6382992,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8986883559",\n   "kind": "organisation",\n   "name": "Rep-In",\n   "office_type": "lawyer",\n   "lat": 55.7108341,\n   "lon": 37.6633487,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8986883561",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7102227,\n   "lon": 37.6632428,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8986883578",\n   "kind": "organisation",\n   "name": "Кашурин Юрий Николаевич",\n   "office_type": "notary",\n   "lat": 55.7099217,\n   "lon": 37.6571304,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8986883580",\n   "kind": "organisation",\n   "name": "Tradition Club",\n   "office_type": "translator",\n   "lat": 55.7089956,\n   "lon": 37.6584736,\n   "opening_hours": "Mo-Th 10:00-18:00; Fr 10L00-14:00"\n  },\n  {\n   "id": "node/8987983760",\n   "kind": "organisation",\n   "name": "Всероссийское общество слепых - Лоссиноостровская",\n   "office_type": "ngo",\n   "lat": 55.8706575,\n   "lon": 37.6851437,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8996057897",\n   "kind": "organisation",\n   "name": "Совет ветеранов района Бабушкинский",\n   "office_type": "ngo",\n   "lat": 55.8777168,\n   "lon": 37.6794925,\n   "opening_hours": null\n  },\n  {\n   "id": "node/8997259707",\n   "kind": "organisation",\n   "name": "Луговский Александр Александрович",\n   "office_type": "notary",\n   "lat": 55.7074787,\n   "lon": 37.660209,\n   "opening_hours": "Mo-Fr 09:00-13:00,13:40-17:00"\n  },\n  {\n   "id": "node/9009604972",\n   "kind": "organisation",\n   "name": "АльфаФуд Инокс",\n   "office_type": "company",\n   "lat": 55.6951686,\n   "lon": 37.6254441,\n   "opening_hours": "Mo-Fr 08:30-17:30"\n  },\n  {\n   "id": "node/9020131142",\n   "kind": "organisation",\n   "name": "Stage Entertainment",\n   "office_type": "company",\n   "lat": 55.7183354,\n   "lon": 37.5610139,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9020384226",\n   "kind": "organisation",\n   "name": "Хоровой театр Бориса Певзнера",\n   "office_type": "theatre",\n   "lat": 55.7605019,\n   "lon": 37.6227379,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9022840292",\n   "kind": "organisation",\n   "name": "Объединенная зерновая компания",\n   "office_type": "company",\n   "lat": 55.7713192,\n   "lon": 37.6480396,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9022840293",\n   "kind": "organisation",\n   "name": "Росфинмониторинг",\n   "office_type": "government",\n   "lat": 55.7684917,\n   "lon": 37.6443744,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9035698555",\n   "kind": "organisation",\n   "name": "Гражданское содействие",\n   "office_type": "ngo",\n   "lat": 55.7878271,\n   "lon": 37.6227783,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9048542260",\n   "kind": "organisation",\n   "name": "Авторусь Логистика",\n   "office_type": "company",\n   "lat": 55.6259229,\n   "lon": 37.525825,\n   "opening_hours": "Mo-Fr 09:00-18:00, Sa 09:00-15:00"\n  },\n  {\n   "id": "node/9049315104",\n   "kind": "organisation",\n   "name": "Единый заказчик в сфере строительства",\n   "office_type": "government",\n   "lat": 55.7456351,\n   "lon": 37.6754089,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9049315105",\n   "kind": "organisation",\n   "name": "ФГУП \\"Росгосэкспертиза\\"",\n   "office_type": "government",\n   "lat": 55.7456487,\n   "lon": 37.6755993,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9049315107",\n   "kind": "organisation",\n   "name": "Управление Роскомнадзора по Центральному федеральному округу",\n   "office_type": "government",\n   "lat": 55.7458978,\n   "lon": 37.6747316,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9049315108",\n   "kind": "organisation",\n   "name": "Женщины Бизнеса",\n   "office_type": "ngo",\n   "lat": 55.7457657,\n   "lon": 37.6727656,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9051099625",\n   "kind": "organisation",\n   "name": "МФМК",\n   "office_type": "company",\n   "lat": 55.6429493,\n   "lon": 37.634339,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9053789560",\n   "kind": "organisation",\n   "name": "ООО «ОКС»",\n   "office_type": "company",\n   "lat": 55.7408875,\n   "lon": 37.5245199,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9056973883",\n   "kind": "organisation",\n   "name": "Yandex Self-Driving Group",\n   "office_type": "company",\n   "lat": 55.6994108,\n   "lon": 37.4625206,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9060199127",\n   "kind": "organisation",\n   "name": "Сбербанк",\n   "office_type": "bank",\n   "lat": 55.7425091,\n   "lon": 37.5305843,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9060446717",\n   "kind": "organisation",\n   "name": "АО \\"Кронштадт\\"",\n   "office_type": "company",\n   "lat": 55.8046758,\n   "lon": 37.4900378,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/9060898757",\n   "kind": "organisation",\n   "name": "Институт Конфуция",\n   "office_type": "educational_institution",\n   "lat": 55.7333105,\n   "lon": 37.5920207,\n   "opening_hours": "Mo-Th 11:00-18:00; Fr 11:00-17:00"\n  },\n  {\n   "id": "node/9074456133",\n   "kind": "organisation",\n   "name": "Чё за Квест",\n   "office_type": "company",\n   "lat": 55.7591546,\n   "lon": 37.650345,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9077880586",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7812986,\n   "lon": 37.6339665,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9077896522",\n   "kind": "organisation",\n   "name": "Посольство Замбии",\n   "office_type": "diplomatic",\n   "lat": 55.7826729,\n   "lon": 37.634655,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9079208296",\n   "kind": "organisation",\n   "name": "ООО «Технопрок»",\n   "office_type": "company",\n   "lat": 55.6122437,\n   "lon": 37.7028275,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/9094158517",\n   "kind": "organisation",\n   "name": "Skyeng",\n   "office_type": "company",\n   "lat": 55.7444121,\n   "lon": 37.6632549,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/9106435117",\n   "kind": "organisation",\n   "name": "Студия Артемия Лебедева",\n   "office_type": "company",\n   "lat": 55.7618388,\n   "lon": 37.5585212,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9110464473",\n   "kind": "organisation",\n   "name": "УК \\"Дома Москвы\\"",\n   "office_type": "property_management",\n   "lat": 55.7903801,\n   "lon": 37.6826731,\n   "opening_hours": "Tu 09:00-14:00; Th 14:00-17:00"\n  },\n  {\n   "id": "node/9113248837",\n   "kind": "organisation",\n   "name": "Пресненская межрайонная прокуратура",\n   "office_type": "government",\n   "lat": 55.7617783,\n   "lon": 37.5737377,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9116063138",\n   "kind": "organisation",\n   "name": "Конструкторское бюро автоматизированных систем и технологий",\n   "office_type": "engineer",\n   "lat": 55.8462915,\n   "lon": 37.5853685,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9118172578",\n   "kind": "organisation",\n   "name": "БИИКС",\n   "office_type": "company",\n   "lat": 55.7375081,\n   "lon": 37.5329246,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9120231038",\n   "kind": "organisation",\n   "name": "Московская административно-дорожная инспекция",\n   "office_type": "government",\n   "lat": 55.7755414,\n   "lon": 37.6910268,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9120354955",\n   "kind": "organisation",\n   "name": "ОДС-16",\n   "office_type": "property_management",\n   "lat": 55.7759617,\n   "lon": 37.6929078,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9126582861",\n   "kind": "organisation",\n   "name": "Высота-сервис",\n   "office_type": "property_management",\n   "lat": 55.6660594,\n   "lon": 37.7610009,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9127352093",\n   "kind": "organisation",\n   "name": "абц",\n   "office_type": "consulting",\n   "lat": 55.7398441,\n   "lon": 37.659933,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/9130297826",\n   "kind": "organisation",\n   "name": "Прокуратура ВАО Москвы",\n   "office_type": "government",\n   "lat": 55.8215782,\n   "lon": 37.6669465,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9137344617",\n   "kind": "organisation",\n   "name": "Совет ветеранов Черёмушкинского района",\n   "office_type": "ngo",\n   "lat": 55.672648,\n   "lon": 37.5552322,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9138619600",\n   "kind": "organisation",\n   "name": "Альфа страхование",\n   "office_type": "insurance",\n   "lat": 55.6261105,\n   "lon": 37.666484,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9157627041",\n   "kind": "organisation",\n   "name": "Хелп Консалтинг",\n   "office_type": "lawyer",\n   "lat": 55.7929296,\n   "lon": 37.591556,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/9158941341",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "security",\n   "lat": 55.6672033,\n   "lon": 37.761145,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9159616475",\n   "kind": "organisation",\n   "name": "Объединённая диспетчерская служба №14",\n   "office_type": "property_management",\n   "lat": 55.8559517,\n   "lon": 37.5645384,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9163512484",\n   "kind": "organisation",\n   "name": "Uzman",\n   "office_type": "company",\n   "lat": 55.7391269,\n   "lon": 37.5713351,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9169909277",\n   "kind": "organisation",\n   "name": "Либерально-демократическая партия России",\n   "office_type": "political_party",\n   "lat": 55.7681643,\n   "lon": 37.6346487,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9171364855",\n   "kind": "organisation",\n   "name": "Центр развития \\"Среда возможностей\\"",\n   "office_type": "government",\n   "lat": 55.7495319,\n   "lon": 37.5341666,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9177040683",\n   "kind": "organisation",\n   "name": "Правовед",\n   "office_type": "lawyer",\n   "lat": 55.7398001,\n   "lon": 37.6565861,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9177040686",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7398953,\n   "lon": 37.6567476,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/9181274051",\n   "kind": "organisation",\n   "name": "Представительство Вологодской области",\n   "office_type": "government",\n   "lat": 55.7440883,\n   "lon": 37.5954187,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9182822865",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6403395,\n   "lon": 37.6076922,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9187076704",\n   "kind": "organisation",\n   "name": "Ауком",\n   "office_type": "company",\n   "lat": 55.7684408,\n   "lon": 37.6576663,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9190250037",\n   "kind": "organisation",\n   "name": "МБЦентр",\n   "office_type": "company",\n   "lat": 55.6830826,\n   "lon": 37.6180757,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/9201165063",\n   "kind": "organisation",\n   "name": "Всероссийское общество слепых, местная организация Царицыно",\n   "office_type": "ngo",\n   "lat": 55.6339004,\n   "lon": 37.6546016,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9213664198",\n   "kind": "organisation",\n   "name": "Frame Bakery",\n   "office_type": "company",\n   "lat": 55.6952236,\n   "lon": 37.5628754,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9227776994",\n   "kind": "organisation",\n   "name": "Ай Ти Оценка",\n   "office_type": "consulting",\n   "lat": 55.7384593,\n   "lon": 37.6589914,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9227809722",\n   "kind": "organisation",\n   "name": "Жилищник Таганского района",\n   "office_type": "property_management",\n   "lat": 55.7368303,\n   "lon": 37.6594886,\n   "opening_hours": "Mo-Fr 08:00-12:00; Mo-Th 12:45-17:00; Fr 12:45-15:45"\n  },\n  {\n   "id": "node/9227809763",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты населения Таганского района",\n   "office_type": "government",\n   "lat": 55.7368767,\n   "lon": 37.6601237,\n   "opening_hours": "Mo 11:00-13:45; Tu-Fr 09:00-13:45; Mo 14:30-20:00; Tu-Th 14:30-18:00; Fr 14:30-16:45"\n  },\n  {\n   "id": "node/9227809777",\n   "kind": "organisation",\n   "name": "Совет депутатов",\n   "office_type": "government",\n   "lat": 55.7368063,\n   "lon": 37.6599602,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:45"\n  },\n  {\n   "id": "node/9227809785",\n   "kind": "organisation",\n   "name": "ЦМК-ТУР",\n   "office_type": "travel_agent",\n   "lat": 55.7384783,\n   "lon": 37.6589333,\n   "opening_hours": "Mo-Fr 10:00-18:30"\n  },\n  {\n   "id": "node/9227809804",\n   "kind": "organisation",\n   "name": "Отдел жилищных субсидий \\"Таганский\\"",\n   "office_type": "government",\n   "lat": 55.7367106,\n   "lon": 37.6597043,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/9237777917",\n   "kind": "organisation",\n   "name": "Консультант плюс",\n   "office_type": "company",\n   "lat": 55.6822341,\n   "lon": 37.5627041,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9240751809",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8965447,\n   "lon": 37.5547821,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9240751810",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8965252,\n   "lon": 37.5548385,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9242593935",\n   "kind": "organisation",\n   "name": "УФК, Отдел №28",\n   "office_type": "government",\n   "lat": 55.8330379,\n   "lon": 37.4579139,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9250233462",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7519702,\n   "lon": 37.8130272,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9258217921",\n   "kind": "organisation",\n   "name": "Бюро пропусков БЦ \\"Ферро-плаза\\"",\n   "office_type": "security",\n   "lat": 55.6795044,\n   "lon": 37.5668224,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9258217922",\n   "kind": "organisation",\n   "name": "Сибур Диджитал",\n   "office_type": "company",\n   "lat": 55.6795033,\n   "lon": 37.5671947,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9258325272",\n   "kind": "organisation",\n   "name": "Стратегия ЗИП",\n   "office_type": "company",\n   "lat": 55.7659482,\n   "lon": 37.6472796,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9271772508",\n   "kind": "organisation",\n   "name": "KPMG",\n   "office_type": "consulting",\n   "lat": 55.7470929,\n   "lon": 37.5367339,\n   "opening_hours": "Mo-Fr 09:00-18:00+"\n  },\n  {\n   "id": "node/9273626587",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7683775,\n   "lon": 37.8323943,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9283525885",\n   "kind": "organisation",\n   "name": "Световое проектирование",\n   "office_type": "architect",\n   "lat": 55.7170995,\n   "lon": 37.6241662,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9283552382",\n   "kind": "organisation",\n   "name": "МАКА",\n   "office_type": "lawyer",\n   "lat": 55.7169667,\n   "lon": 37.6237691,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9292677985",\n   "kind": "organisation",\n   "name": "Сбер-Сервис",\n   "office_type": "company",\n   "lat": 55.7429245,\n   "lon": 37.7174873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9296210793",\n   "kind": "organisation",\n   "name": "Шайкевич М. Л.",\n   "office_type": "notary",\n   "lat": 55.7690394,\n   "lon": 37.6044242,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9318670626",\n   "kind": "organisation",\n   "name": "Экспресс-Л",\n   "office_type": "logistics",\n   "lat": 55.7298416,\n   "lon": 37.6381463,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9318813851",\n   "kind": "organisation",\n   "name": "Kelly Services",\n   "office_type": "company",\n   "lat": 55.7805766,\n   "lon": 37.6337235,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9332889616",\n   "kind": "organisation",\n   "name": "Посольство Буркина-Фасо",\n   "office_type": "diplomatic",\n   "lat": 55.7286597,\n   "lon": 37.6171354,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9339645440",\n   "kind": "organisation",\n   "name": "Раира",\n   "office_type": "insurance",\n   "lat": 55.6944692,\n   "lon": 37.7486117,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9339894582",\n   "kind": "organisation",\n   "name": "Согласие",\n   "office_type": "insurance",\n   "lat": 55.8101625,\n   "lon": 37.8103273,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9341706847",\n   "kind": "organisation",\n   "name": "посольство Доминиканской Республики",\n   "office_type": "diplomatic",\n   "lat": 55.7580204,\n   "lon": 37.6459336,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9342121434",\n   "kind": "organisation",\n   "name": "Посольство Гамбии",\n   "office_type": "diplomatic",\n   "lat": 55.7286477,\n   "lon": 37.6174471,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9342121435",\n   "kind": "organisation",\n   "name": "посольство Гондураса",\n   "office_type": "diplomatic",\n   "lat": 55.7273739,\n   "lon": 37.6164375,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9344869570",\n   "kind": "organisation",\n   "name": "Местное отделение партии \\"Единая Россия\\"",\n   "office_type": "political_party",\n   "lat": 55.877221,\n   "lon": 37.5438385,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9344889354",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.8142628,\n   "lon": 37.5639376,\n   "opening_hours": "Mo-Th 11:00-15:00"\n  },\n  {\n   "id": "node/9346261141",\n   "kind": "organisation",\n   "name": "Следственное управление по центральному административному округу",\n   "office_type": "government",\n   "lat": 55.736812,\n   "lon": 37.6595268,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9346261142",\n   "kind": "organisation",\n   "name": "5 региональный отдел надзорной деятельности и профилактической работы Управления по ЦАО Главного управления МСЧ России по г.Москве",\n   "office_type": "government",\n   "lat": 55.7369737,\n   "lon": 37.6603132,\n   "opening_hours": "Mo-Th 09:00-15:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/9346261143",\n   "kind": "organisation",\n   "name": "Совет Ветеранов Таганского Района",\n   "office_type": "ngo",\n   "lat": 55.7369463,\n   "lon": 37.6602469,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9346261144",\n   "kind": "organisation",\n   "name": "Территориальная избирательная комиссия Таганского района",\n   "office_type": "government",\n   "lat": 55.7370253,\n   "lon": 37.6604049,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9346565520",\n   "kind": "organisation",\n   "name": "Axioma GIS",\n   "office_type": "company",\n   "lat": 55.7467393,\n   "lon": 37.5910071,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/9348331871",\n   "kind": "organisation",\n   "name": "Южнопортовый",\n   "office_type": "yes",\n   "lat": 55.7040522,\n   "lon": 37.6902385,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9349724153",\n   "kind": "organisation",\n   "name": "Посольство Панамы",\n   "office_type": "diplomatic",\n   "lat": 55.7140691,\n   "lon": 37.5931577,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9350985187",\n   "kind": "organisation",\n   "name": "Pravo.ru",\n   "office_type": "company",\n   "lat": 55.7415884,\n   "lon": 37.6013759,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9351069491",\n   "kind": "organisation",\n   "name": "посольство Сенегала",\n   "office_type": "diplomatic",\n   "lat": 55.7286508,\n   "lon": 37.6177087,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9351069492",\n   "kind": "organisation",\n   "name": "Посольство Уругвая",\n   "office_type": "diplomatic",\n   "lat": 55.7273299,\n   "lon": 37.6156979,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9353002090",\n   "kind": "organisation",\n   "name": "Честное слово",\n   "office_type": "company",\n   "lat": 55.7689285,\n   "lon": 37.6151266,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9370923464",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7671705,\n   "lon": 37.8295075,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9371857676",\n   "kind": "organisation",\n   "name": "Zelenko group",\n   "office_type": "company",\n   "lat": 55.6642592,\n   "lon": 37.5988903,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9378808767",\n   "kind": "organisation",\n   "name": "Роскачество",\n   "office_type": "government",\n   "lat": 55.7102861,\n   "lon": 37.5902045,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9381497907",\n   "kind": "organisation",\n   "name": "Мосгорсвет",\n   "office_type": "company",\n   "lat": 55.7608011,\n   "lon": 37.6350014,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9383383071",\n   "kind": "organisation",\n   "name": "Жилой комплекс \\"На Ивановской\\"",\n   "office_type": "company",\n   "lat": 55.8559148,\n   "lon": 37.4406639,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9383383072",\n   "kind": "organisation",\n   "name": "Экопарк Тимирязево",\n   "office_type": "company",\n   "lat": 55.8559116,\n   "lon": 37.4406331,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9383383073",\n   "kind": "organisation",\n   "name": "Ларус капитал",\n   "office_type": "company",\n   "lat": 55.855909,\n   "lon": 37.4406056,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9388400851",\n   "kind": "organisation",\n   "name": "Радиум",\n   "office_type": "consulting",\n   "lat": 55.7643067,\n   "lon": 37.6488967,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/9397162312",\n   "kind": "organisation",\n   "name": "Всероссийский центр изучения общественного мнения",\n   "office_type": "research",\n   "lat": 55.7397195,\n   "lon": 37.5885563,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9399354782",\n   "kind": "organisation",\n   "name": "Конечная станция Якорная",\n   "office_type": "yes",\n   "lat": 55.6875173,\n   "lon": 37.6836853,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9402083217",\n   "kind": "organisation",\n   "name": "Московская объединенная энергетическая компания",\n   "office_type": "company",\n   "lat": 55.7050682,\n   "lon": 37.6530322,\n   "opening_hours": "Mo-Fr 08:00-17:00"\n  },\n  {\n   "id": "node/9403045769",\n   "kind": "organisation",\n   "name": "Сенат",\n   "office_type": "tax_advisor",\n   "lat": 55.7668395,\n   "lon": 37.6004622,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9413293248",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг Академический",\n   "office_type": "government",\n   "lat": 55.6807527,\n   "lon": 37.5799879,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/9415302647",\n   "kind": "organisation",\n   "name": "Хальме И. Ю.",\n   "office_type": "notary",\n   "lat": 55.6875236,\n   "lon": 37.5413048,\n   "opening_hours": "Tu-Fr 10:00-13:00,14:00-18:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/9420890920",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.743271,\n   "lon": 37.4989489,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9420895917",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7433667,\n   "lon": 37.498853,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9421035072",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "translator",\n   "lat": 55.6740207,\n   "lon": 37.7494808,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9421035078",\n   "kind": "organisation",\n   "name": "Lanport",\n   "office_type": "provider",\n   "lat": 55.6731427,\n   "lon": 37.7496222,\n   "opening_hours": "Mo-Fr 09:00-21:00;Sa-Su 11:00-20:00"\n  },\n  {\n   "id": "node/9421035086",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "translator",\n   "lat": 55.6734819,\n   "lon": 37.7516761,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 10:00-14:00"\n  },\n  {\n   "id": "node/9421035089",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.6734562,\n   "lon": 37.7517097,\n   "opening_hours": "Mo-We 09:30-18:00; Th-Fr 09:30-16:00"\n  },\n  {\n   "id": "node/9429741473",\n   "kind": "organisation",\n   "name": "Главное бюро медико-социальной экспертизы по г. Москве. Филиал № 76",\n   "office_type": "government",\n   "lat": 55.6767979,\n   "lon": 37.5000767,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9444060711",\n   "kind": "organisation",\n   "name": "Wellside",\n   "office_type": "estate_agent",\n   "lat": 55.7478681,\n   "lon": 37.5391249,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9447207984",\n   "kind": "organisation",\n   "name": "Элемент Лизинг",\n   "office_type": "company",\n   "lat": 55.7426374,\n   "lon": 37.5278994,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9469170719",\n   "kind": "organisation",\n   "name": "Мигрон",\n   "office_type": "company",\n   "lat": 55.7408234,\n   "lon": 37.6599263,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9487763122",\n   "kind": "organisation",\n   "name": "ООО \\"Технол\\"",\n   "office_type": "company",\n   "lat": 55.8004412,\n   "lon": 37.6367381,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9490585320",\n   "kind": "organisation",\n   "name": "ЦНИИчермет им. И.П.Бардина",\n   "office_type": "research",\n   "lat": 55.7635729,\n   "lon": 37.6844269,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9495244300",\n   "kind": "organisation",\n   "name": "Эрдиан кар сервис",\n   "office_type": "company",\n   "lat": 55.797444,\n   "lon": 37.6508465,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9495244301",\n   "kind": "organisation",\n   "name": "Центр управления телекоммуникационными ресурсами.",\n   "office_type": "telecommunication",\n   "lat": 55.7973053,\n   "lon": 37.6509619,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9496122320",\n   "kind": "organisation",\n   "name": "Центральная бухгалтерия МГТУ им. Баумана",\n   "office_type": "accountant",\n   "lat": 55.7651714,\n   "lon": 37.682223,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9496122321",\n   "kind": "organisation",\n   "name": "Российскй государственный архив фотодокументов",\n   "office_type": "government",\n   "lat": 55.7679418,\n   "lon": 37.686249,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9496122327",\n   "kind": "organisation",\n   "name": "Московско-Окское бассейновое водное управление",\n   "office_type": "government",\n   "lat": 55.7826135,\n   "lon": 37.6645634,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9496122357",\n   "kind": "organisation",\n   "name": "ФГУП Президент сервис",\n   "office_type": "government",\n   "lat": 55.7717907,\n   "lon": 37.6436381,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9496447276",\n   "kind": "organisation",\n   "name": "Удальцова 79-81",\n   "office_type": "property_management",\n   "lat": 55.679902,\n   "lon": 37.4952597,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9504183004",\n   "kind": "organisation",\n   "name": "Абсолют Страхование",\n   "office_type": "insurance",\n   "lat": 55.7104826,\n   "lon": 37.6548243,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9504534019",\n   "kind": "organisation",\n   "name": "НПО Геофизика-НВ",\n   "office_type": "company",\n   "lat": 55.7938343,\n   "lon": 37.6964655,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9507783864",\n   "kind": "organisation",\n   "name": "Dream catchers",\n   "office_type": "company",\n   "lat": 55.7910877,\n   "lon": 37.6700203,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9507783865",\n   "kind": "organisation",\n   "name": "Бисан",\n   "office_type": "company",\n   "lat": 55.790716,\n   "lon": 37.6703207,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9507883230",\n   "kind": "organisation",\n   "name": "Материальный склад ст. Москва-3",\n   "office_type": "company",\n   "lat": 55.7969599,\n   "lon": 37.6519853,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9510795174",\n   "kind": "organisation",\n   "name": "FP models",\n   "office_type": "company",\n   "lat": 55.7797374,\n   "lon": 37.7117521,\n   "opening_hours": "Mo-Su 11:00-20:00"\n  },\n  {\n   "id": "node/9519022396",\n   "kind": "organisation",\n   "name": "Федеральный центр оценки безопасности и качества зерна и продуктов его переработки",\n   "office_type": "government",\n   "lat": 55.7733205,\n   "lon": 37.6443435,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9519022398",\n   "kind": "organisation",\n   "name": "ГлавУпДК при МИД России",\n   "office_type": "company",\n   "lat": 55.7730987,\n   "lon": 37.6437977,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9519225258",\n   "kind": "organisation",\n   "name": "Пассажирское агентство Мосгортранс",\n   "office_type": "yes",\n   "lat": 55.7585448,\n   "lon": 37.6295191,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9522086793",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.6816223,\n   "lon": 37.4934922,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9523725285",\n   "kind": "organisation",\n   "name": "Familia",\n   "office_type": "company",\n   "lat": 55.7622411,\n   "lon": 37.5700743,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9524454605",\n   "kind": "organisation",\n   "name": "Ферон",\n   "office_type": "company",\n   "lat": 55.7921877,\n   "lon": 37.5406936,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9525541571",\n   "kind": "organisation",\n   "name": "Паралимпийский комитет России",\n   "office_type": "ngo",\n   "lat": 55.7656037,\n   "lon": 37.6357028,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9526600851",\n   "kind": "organisation",\n   "name": "Проектно-конструкторско-технологическое бюро по нормированию материально-технических ресурсов",\n   "office_type": "company",\n   "lat": 55.7652921,\n   "lon": 37.6779771,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9529035688",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6991659,\n   "lon": 37.8175753,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9529138043",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7130438,\n   "lon": 37.513327,\n   "opening_hours": "Fr 10:00-19:00; Mo-Th,Sa-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9529138049",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8751212,\n   "lon": 37.6763088,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9529138097",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.789441,\n   "lon": 37.782898,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9531730654",\n   "kind": "organisation",\n   "name": "АО \\"Воентелеком\\"",\n   "office_type": "telecommunication",\n   "lat": 55.8076186,\n   "lon": 37.6919003,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9531925698",\n   "kind": "organisation",\n   "name": "Управление градостроительного регулирования Северного административного округа г. Москвы",\n   "office_type": "government",\n   "lat": 55.8090642,\n   "lon": 37.5739396,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9531972968",\n   "kind": "organisation",\n   "name": "Овионт-информ",\n   "office_type": "it",\n   "lat": 55.8085996,\n   "lon": 37.5709583,\n   "opening_hours": "Mo-Fr 09:30-18:30"\n  },\n  {\n   "id": "node/9537343181",\n   "kind": "organisation",\n   "name": "Visa Fly",\n   "office_type": "visa",\n   "lat": 55.7226447,\n   "lon": 37.6228412,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9540337802",\n   "kind": "organisation",\n   "name": "DNS",\n   "office_type": "company",\n   "lat": 55.7503213,\n   "lon": 37.7709538,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9542931194",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.8291899,\n   "lon": 37.5695853,\n   "opening_hours": "Mo, We 11:00-15:00"\n  },\n  {\n   "id": "node/9544943400",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Покровское-Стрешнево",\n   "office_type": "government",\n   "lat": 55.8334,\n   "lon": 37.4558,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9553034313",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8647494,\n   "lon": 37.544902,\n   "opening_hours": "PH,Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9554923979",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.654449,\n   "lon": 37.7344778,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9556470234",\n   "kind": "organisation",\n   "name": "ОДС № 2 ГБУ \\"Жилищник района Восточное Дегунино\\"",\n   "office_type": "property_management",\n   "lat": 55.8668113,\n   "lon": 37.5661451,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9568429772",\n   "kind": "organisation",\n   "name": "Аппарат Совета депутатов муниципального округа Западное Дегунино",\n   "office_type": "government",\n   "lat": 55.8629,\n   "lon": 37.5407,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9568429773",\n   "kind": "organisation",\n   "name": "Аппарат Совета депутатов муниципального округа Дмитровский в городе Москве",\n   "office_type": "government",\n   "lat": 55.8876,\n   "lon": 37.522,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9570747446",\n   "kind": "organisation",\n   "name": "Unic Lab",\n   "office_type": "it",\n   "lat": 55.7527865,\n   "lon": 37.6678442,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9572648619",\n   "kind": "organisation",\n   "name": "Аппарат Совета депутатов Бескудниковский",\n   "office_type": "government",\n   "lat": 55.8648,\n   "lon": 37.565,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9572648620",\n   "kind": "organisation",\n   "name": "Аппарат Совета депутатов муниципального округа Восточное Дегунино",\n   "office_type": "government",\n   "lat": 55.8775,\n   "lon": 37.5664,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9577613280",\n   "kind": "organisation",\n   "name": "Аппарат Совета депутатов муниципального округа Отрадное",\n   "office_type": "government",\n   "lat": 55.8636,\n   "lon": 37.5939,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9577613285",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Бибирево",\n   "office_type": "government",\n   "lat": 55.8877,\n   "lon": 37.5973,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9589797317",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6757947,\n   "lon": 37.7725391,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9594076645",\n   "kind": "organisation",\n   "name": "Деметра-Холдинг",\n   "office_type": "company",\n   "lat": 55.7470701,\n   "lon": 37.5363067,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9599221003",\n   "kind": "organisation",\n   "name": "Юнайтед Норд",\n   "office_type": "property_management",\n   "lat": 55.8180862,\n   "lon": 37.569187,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9600132217",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Дорогомилово",\n   "office_type": "government",\n   "lat": 55.7366,\n   "lon": 37.5227,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9600132218",\n   "kind": "organisation",\n   "name": "Аппарат Совета депутатов муниципального округа Филевский парк",\n   "office_type": "government",\n   "lat": 55.7382,\n   "lon": 37.4949,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9600132219",\n   "kind": "organisation",\n   "name": "Аппарат Совета депутатов муниципального округа Фили-Давыдково",\n   "office_type": "government",\n   "lat": 55.7333,\n   "lon": 37.4687,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9602886016",\n   "kind": "organisation",\n   "name": "ТрансКонтейнер",\n   "office_type": "logistics",\n   "lat": 55.7953919,\n   "lon": 37.6919796,\n   "opening_hours": "Fr 08:00-15:45; Mo-Th 08:00-17:00"\n  },\n  {\n   "id": "node/9602887142",\n   "kind": "organisation",\n   "name": "Домовёнок",\n   "office_type": "company",\n   "lat": 55.7895752,\n   "lon": 37.6964481,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9603155076",\n   "kind": "organisation",\n   "name": "Научно-технический центр «Орион»",\n   "office_type": "research",\n   "lat": 55.7915701,\n   "lon": 37.61016,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9603155077",\n   "kind": "organisation",\n   "name": "Научно-технический центр «Атлас»",\n   "office_type": "research",\n   "lat": 55.7926146,\n   "lon": 37.6107297,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9605011316",\n   "kind": "organisation",\n   "name": "Сеоклаб",\n   "office_type": "company",\n   "lat": 55.7990044,\n   "lon": 37.6954852,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/9609061055",\n   "kind": "organisation",\n   "name": "Coalco",\n   "office_type": "company",\n   "lat": 55.7771809,\n   "lon": 37.5861243,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9610759342",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6932173,\n   "lon": 37.5338222,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9612448450",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6700642,\n   "lon": 37.5521627,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612448457",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.793682,\n   "lon": 37.5900838,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612448513",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7947014,\n   "lon": 37.4919578,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612448516",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7928949,\n   "lon": 37.4915153,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612450130",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7047597,\n   "lon": 37.6850957,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612450155",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7573798,\n   "lon": 37.6594698,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612450160",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8090216,\n   "lon": 37.4623302,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612450163",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6097371,\n   "lon": 37.7199027,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612450242",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8094964,\n   "lon": 37.4640736,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450252",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7068422,\n   "lon": 37.5917333,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9612450265",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7897659,\n   "lon": 37.7495036,\n   "opening_hours": "Mo-Su 10:00-21:45"\n  },\n  {\n   "id": "node/9612450307",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7883574,\n   "lon": 37.4547368,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612450319",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8898286,\n   "lon": 37.5370592,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612450363",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8957484,\n   "lon": 37.6821855,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612450373",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6778173,\n   "lon": 37.467142,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450466",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7090289,\n   "lon": 37.6208675,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612450567",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6066479,\n   "lon": 37.536394,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450568",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8108951,\n   "lon": 37.8006747,\n   "opening_hours": "Fr-Sa 10:00-23:00; Mo-Th,Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450572",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8448421,\n   "lon": 37.6595557,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450573",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8379204,\n   "lon": 37.5769812,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450574",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8636747,\n   "lon": 37.6023093,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450585",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7054428,\n   "lon": 37.4793541,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612450587",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7774359,\n   "lon": 37.522999,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450590",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7085906,\n   "lon": 37.6565301,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612450591",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7434769,\n   "lon": 37.5077909,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450592",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7050438,\n   "lon": 37.6399219,\n   "opening_hours": "Fr-Sa 10:00-23:00; Mo-Th,Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450594",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6975077,\n   "lon": 37.5000957,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450595",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8639802,\n   "lon": 37.5457898,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450616",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6920626,\n   "lon": 37.5283983,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450617",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6903164,\n   "lon": 37.6017755,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450620",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7087236,\n   "lon": 37.6218572,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612450622",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7446228,\n   "lon": 37.5659546,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612450948",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8619934,\n   "lon": 37.6877457,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612451399",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7022463,\n   "lon": 37.7956027,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612451552",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.790621,\n   "lon": 37.558195,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612451748",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8689168,\n   "lon": 37.6649764,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612451753",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8866742,\n   "lon": 37.6626053,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612451858",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7041642,\n   "lon": 37.7655191,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9612451860",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8564041,\n   "lon": 37.6531693,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612451888",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6044752,\n   "lon": 37.489627,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612451975",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7996301,\n   "lon": 37.4830824,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612451976",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8137406,\n   "lon": 37.7345663,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612452032",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.612329,\n   "lon": 37.605966,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452067",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6933234,\n   "lon": 37.5339961,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612452076",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8403212,\n   "lon": 37.4920973,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452089",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6390804,\n   "lon": 37.7591702,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452143",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7924878,\n   "lon": 37.7873442,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452169",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.796223,\n   "lon": 37.7995587,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612452170",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8551244,\n   "lon": 37.653231,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612452264",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8653168,\n   "lon": 37.7053839,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452316",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6588932,\n   "lon": 37.7418754,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612452410",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6043176,\n   "lon": 37.490834,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612452413",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.845842,\n   "lon": 37.6619321,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612452414",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7902771,\n   "lon": 37.5310403,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612452415",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8231543,\n   "lon": 37.497341,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612452416",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7492017,\n   "lon": 37.5396261,\n   "opening_hours": "Fr-Sa 10:00-23:00; Mo-Th,Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452418",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7280058,\n   "lon": 37.475988,\n   "opening_hours": "Fr-Sa 10:00-23:00; Mo-Th,Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452422",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6120761,\n   "lon": 37.7330402,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452423",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6214568,\n   "lon": 37.7138221,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452424",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7473644,\n   "lon": 37.7071971,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452425",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6596482,\n   "lon": 37.7479479,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612452427",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7826082,\n   "lon": 37.7209595,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9612452452",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6560304,\n   "lon": 37.5414366,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452469",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7166851,\n   "lon": 37.7458262,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452554",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8528165,\n   "lon": 37.5857493,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612452559",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6555068,\n   "lon": 37.8276095,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612452560",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6229909,\n   "lon": 37.4413553,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612452645",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8078625,\n   "lon": 37.573218,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612452931",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6571274,\n   "lon": 37.6077166,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612453040",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.60032,\n   "lon": 37.473212,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612453154",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7844588,\n   "lon": 37.4571347,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453388",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8670446,\n   "lon": 37.4932319,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612453429",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8940279,\n   "lon": 37.4498525,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453446",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6492143,\n   "lon": 37.7711302,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453458",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.743261,\n   "lon": 37.4990577,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612453461",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6475873,\n   "lon": 37.5956842,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453462",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8105123,\n   "lon": 37.7976894,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612453475",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7782113,\n   "lon": 37.5210598,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612453524",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6737746,\n   "lon": 37.7607179,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612453557",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8770186,\n   "lon": 37.5232995,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/9612453559",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8879483,\n   "lon": 37.5888419,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453575",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7972797,\n   "lon": 37.7171293,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612453587",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7759515,\n   "lon": 37.6604408,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612453589",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8068254,\n   "lon": 37.6377332,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612453602",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8927284,\n   "lon": 37.7268979,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9612453615",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7186386,\n   "lon": 37.6765823,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612453641",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6756138,\n   "lon": 37.5058249,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453645",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.824102,\n   "lon": 37.6345253,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612453706",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8057658,\n   "lon": 37.5152233,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453723",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8265985,\n   "lon": 37.4446222,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9612453786",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7720773,\n   "lon": 37.6784679,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453790",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6939644,\n   "lon": 37.7229255,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453796",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8551922,\n   "lon": 37.477541,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453920",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6374787,\n   "lon": 37.6001528,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453967",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6330351,\n   "lon": 37.5212878,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612453975",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.594704,\n   "lon": 37.5989619,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612453991",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6400841,\n   "lon": 37.7584058,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612454011",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6255821,\n   "lon": 37.7617961,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9612454072",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.728256,\n   "lon": 37.58031,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9612461281",\n   "kind": "organisation",\n   "name": "ПЭК",\n   "office_type": "logistics",\n   "lat": 55.6418552,\n   "lon": 37.8295594,\n   "opening_hours": "Mo-Su 09:00-00:00"\n  },\n  {\n   "id": "node/9625662160",\n   "kind": "organisation",\n   "name": "ООО \\"Сигнум Плюс",\n   "office_type": "company",\n   "lat": 55.795027,\n   "lon": 37.7375892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9629143241",\n   "kind": "organisation",\n   "name": "Олимпия бизнес трэвел",\n   "office_type": "travel_agent",\n   "lat": 55.6804335,\n   "lon": 37.554189,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9633683718",\n   "kind": "organisation",\n   "name": "Точка",\n   "office_type": "company",\n   "lat": 55.7457646,\n   "lon": 37.6283173,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9643651788",\n   "kind": "organisation",\n   "name": "Совет ветеранов района Черёмушки",\n   "office_type": "government",\n   "lat": 55.6775663,\n   "lon": 37.5593376,\n   "opening_hours": "Tu 14:00-16:00; PH closed"\n  },\n  {\n   "id": "node/9643677454",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.679579,\n   "lon": 37.5568649,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9644977759",\n   "kind": "organisation",\n   "name": "Фазотрон-НИИР",\n   "office_type": "company",\n   "lat": 55.7994673,\n   "lon": 37.5376359,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9645278960",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8550161,\n   "lon": 37.4746281,\n   "opening_hours": "Mo-Sa 10:00-20:00; Su 10:00-19:00"\n  },\n  {\n   "id": "node/9648037487",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7692333,\n   "lon": 37.5834373,\n   "opening_hours": "Fr 10:00-19:00;Mo-Th 10:00-20:00;Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/9648037512",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.8486413,\n   "lon": 37.5970173,\n   "opening_hours": "Fr 09:00-16:30;Mo-Th 09:00-17:30"\n  },\n  {\n   "id": "node/9648120024",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6045964,\n   "lon": 37.722711,\n   "opening_hours": "Fr 09:00-19:00;Mo-Th 09:00-20:00;Sa 09:00-18:00;Su 10:00-17:00"\n  },\n  {\n   "id": "node/9648120032",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7865311,\n   "lon": 37.6787603,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9648120033",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.6939281,\n   "lon": 37.7233064,\n   "opening_hours": "Mo-Fr 10:00-20:00;Sa 10:00-18:00"\n  },\n  {\n   "id": "node/9666510182",\n   "kind": "organisation",\n   "name": "Главное управление обустройства войск",\n   "office_type": "company",\n   "lat": 55.7947903,\n   "lon": 37.6541403,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9670205029",\n   "kind": "organisation",\n   "name": "Жилищник района Южное Тушино",\n   "office_type": "property_management",\n   "lat": 55.8431497,\n   "lon": 37.4404991,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/9670889068",\n   "kind": "organisation",\n   "name": "Литера Про",\n   "office_type": "translator",\n   "lat": 55.8140257,\n   "lon": 37.5765476,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/9680063614",\n   "kind": "organisation",\n   "name": "Русская факторинговая компания",\n   "office_type": "company",\n   "lat": 55.7345634,\n   "lon": 37.6331815,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9680063616",\n   "kind": "organisation",\n   "name": "Gaskar Group",\n   "office_type": "company",\n   "lat": 55.7357534,\n   "lon": 37.6348284,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9680467676",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.6715118,\n   "lon": 37.5535548,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9680467688",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.7422765,\n   "lon": 37.5600538,\n   "opening_hours": "Tu-Sa 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/9680467690",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.7931431,\n   "lon": 37.6226365,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 11:00-18:00"\n  },\n  {\n   "id": "node/9680537723",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.7178651,\n   "lon": 37.785134,\n   "opening_hours": "Mo-Fr 10:30-19:00;Sa 11:00-15:00"\n  },\n  {\n   "id": "node/9680537724",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.8579998,\n   "lon": 37.5854623,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/9680539122",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.7983471,\n   "lon": 37.7986389,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/9680539140",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.8388708,\n   "lon": 37.4885568,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa-Su 10:00-16:30"\n  },\n  {\n   "id": "node/9680539153",\n   "kind": "organisation",\n   "name": "Росгосстрах",\n   "office_type": "insurance",\n   "lat": 55.6667105,\n   "lon": 37.5158992,\n   "opening_hours": "Mo-Fr 13:00-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/9685411290",\n   "kind": "organisation",\n   "name": "Инженерная служба Гагаринского района",\n   "office_type": "property_management",\n   "lat": 55.6865797,\n   "lon": 37.5446549,\n   "opening_hours": "Mo 15:00-17:00"\n  },\n  {\n   "id": "node/9685427718",\n   "kind": "organisation",\n   "name": "Совет ветеранов № 6 Гагаринского района",\n   "office_type": "association",\n   "lat": 55.6864688,\n   "lon": 37.5468444,\n   "opening_hours": "Sa 14:00-18:00"\n  },\n  {\n   "id": "node/9689555059",\n   "kind": "organisation",\n   "name": "Серния Инжиниринг",\n   "office_type": "company",\n   "lat": 55.6894426,\n   "lon": 37.5606009,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9689785662",\n   "kind": "organisation",\n   "name": "Федеральное бюро медико-социальной экспертизы по г. Москве. Филиал №45",\n   "office_type": "government",\n   "lat": 55.8129855,\n   "lon": 37.5720459,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9689785664",\n   "kind": "organisation",\n   "name": "Адвокатская контора",\n   "office_type": "lawyer",\n   "lat": 55.8117135,\n   "lon": 37.5730062,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9689785671",\n   "kind": "organisation",\n   "name": "\\"Жилищник\\" Тимирязевского района. ОДФ №5",\n   "office_type": "property_management",\n   "lat": 55.812352,\n   "lon": 37.5736251,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9689785687",\n   "kind": "organisation",\n   "name": "Отдел жилищных субсидий \\"Тимирязевский\\"",\n   "office_type": "government",\n   "lat": 55.8116035,\n   "lon": 37.5748166,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:45"\n  },\n  {\n   "id": "node/9693047694",\n   "kind": "organisation",\n   "name": "Федеральный исследовательский центр химической физики\xa0им. Н.Н. Семёнова РАН",\n   "office_type": "research",\n   "lat": 55.7088162,\n   "lon": 37.5682428,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9700887039",\n   "kind": "organisation",\n   "name": "Домион",\n   "office_type": "architect",\n   "lat": 55.7077484,\n   "lon": 37.5699291,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9705074805",\n   "kind": "organisation",\n   "name": "ФГУП \\"Канал им. Москвы\\"",\n   "office_type": "transportation",\n   "lat": 55.8186564,\n   "lon": 37.4502388,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9705265817",\n   "kind": "organisation",\n   "name": "МСК Энерго",\n   "office_type": "company",\n   "lat": 55.7509248,\n   "lon": 37.5255186,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9706791719",\n   "kind": "organisation",\n   "name": "Scripta",\n   "office_type": "yes",\n   "lat": 55.6635175,\n   "lon": 37.4854277,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9708644526",\n   "kind": "organisation",\n   "name": "Аэроприбор-Восход",\n   "office_type": "research",\n   "lat": 55.786513,\n   "lon": 37.7279359,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9708644561",\n   "kind": "organisation",\n   "name": "ПромЭлектроАвтоматика",\n   "office_type": "company",\n   "lat": 55.7815493,\n   "lon": 37.698795,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9709230773",\n   "kind": "organisation",\n   "name": "Московский олимпийский центр водного спорта",\n   "office_type": "company",\n   "lat": 55.7867995,\n   "lon": 37.7313477,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9714209237",\n   "kind": "organisation",\n   "name": "Жилищник Донского района",\n   "office_type": "property_management",\n   "lat": 55.7171969,\n   "lon": 37.5931286,\n   "opening_hours": "Mo-Th 08:00-17:00; Fr 08:00-15:45"\n  },\n  {\n   "id": "node/9718226373",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.764762,\n   "lon": 37.701345,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718226378",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7085695,\n   "lon": 37.6218197,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718226382",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.671311,\n   "lon": 37.445784,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718226385",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7770935,\n   "lon": 37.5242972,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718226398",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6778077,\n   "lon": 37.6648278,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718226405",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6194924,\n   "lon": 37.5083783,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718226413",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.609653,\n   "lon": 37.719847,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718259054",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.7614365,\n   "lon": 37.5702703,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/9718259055",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.7384896,\n   "lon": 37.66157,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/9718273028",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7124545,\n   "lon": 37.8167439,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718273039",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.704733,\n   "lon": 37.6850698,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718273045",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7090742,\n   "lon": 37.6208836,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718273059",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6126654,\n   "lon": 37.7193555,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273061",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6175036,\n   "lon": 37.5068253,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718273066",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8510654,\n   "lon": 37.5977227,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273074",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6121261,\n   "lon": 37.6060349,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273082",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.897908,\n   "lon": 37.5880614,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/9718273084",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.610993,\n   "lon": 37.6060805,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718273089",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.886426,\n   "lon": 37.660355,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718273148",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8776867,\n   "lon": 37.7313691,\n   "opening_hours": "Mo-Su 11:00-16:00"\n  },\n  {\n   "id": "node/9718273346",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6066903,\n   "lon": 37.5364208,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273394",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6493687,\n   "lon": 37.7704436,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718273400",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.656109,\n   "lon": 37.5693396,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273402",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8377879,\n   "lon": 37.5770375,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273405",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8633812,\n   "lon": 37.5448805,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718273484",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8878942,\n   "lon": 37.6392728,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718273512",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8459083,\n   "lon": 37.6617202,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273603",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8405621,\n   "lon": 37.4915099,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273649",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6224427,\n   "lon": 37.6057667,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273721",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6327172,\n   "lon": 37.5196731,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718273766",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6207888,\n   "lon": 37.472088,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273772",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7927924,\n   "lon": 37.7871752,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273808",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8227535,\n   "lon": 37.8233689,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718273809",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6053252,\n   "lon": 37.4904209,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718273862",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6123231,\n   "lon": 37.7315274,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273887",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6960399,\n   "lon": 37.6657596,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273898",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8705752,\n   "lon": 37.6382777,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718273943",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6072752,\n   "lon": 37.5326067,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274041",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7298998,\n   "lon": 37.7299798,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718274153",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8655245,\n   "lon": 37.7044639,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274212",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6561817,\n   "lon": 37.5404334,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274213",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7216978,\n   "lon": 37.8132302,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274226",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8562174,\n   "lon": 37.6535046,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274232",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6128078,\n   "lon": 37.6973748,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274249",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6940082,\n   "lon": 37.7229577,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274251",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7939218,\n   "lon": 37.4921912,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718274253",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7085468,\n   "lon": 37.6563209,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274257",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6208328,\n   "lon": 37.7148789,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718274269",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6745369,\n   "lon": 37.5043818,\n   "opening_hours": "Mo-Sa 09:00-21:00;Su 10:00-20:00"\n  },\n  {\n   "id": "node/9718274343",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7525498,\n   "lon": 37.8179884,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718274361",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8702938,\n   "lon": 37.6629326,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274432",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8202703,\n   "lon": 37.4992159,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718274506",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6196105,\n   "lon": 37.7523494,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718274512",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8529234,\n   "lon": 37.5861222,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274513",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7930201,\n   "lon": 37.493645,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718274515",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7649933,\n   "lon": 37.5572991,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718274540",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8879965,\n   "lon": 37.5888392,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274544",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6401931,\n   "lon": 37.7577782,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274568",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6739561,\n   "lon": 37.7608439,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274576",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8539517,\n   "lon": 37.4754032,\n   "opening_hours": "Mo-Su 09:00-21:30"\n  },\n  {\n   "id": "node/9718274580",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8780839,\n   "lon": 37.5226074,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274584",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8165422,\n   "lon": 37.734789,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274591",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8629703,\n   "lon": 37.6825529,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274604",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.797052,\n   "lon": 37.7166036,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274605",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8060868,\n   "lon": 37.637583,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274771",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7045224,\n   "lon": 37.7661172,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9718274813",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8755034,\n   "lon": 37.665934,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718274831",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7934468,\n   "lon": 37.5918487,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718274833",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8832774,\n   "lon": 37.6047421,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274884",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7305341,\n   "lon": 37.7312645,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718274889",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7507701,\n   "lon": 37.8201663,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718274890",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8105922,\n   "lon": 37.8304258,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718274891",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6403218,\n   "lon": 37.6077837,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718274912",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7800395,\n   "lon": 37.6013035,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274947",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8556303,\n   "lon": 37.4767148,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274972",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8242406,\n   "lon": 37.6348311,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9718274977",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7826037,\n   "lon": 37.72104,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274980",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8917297,\n   "lon": 37.727783,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9718274981",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7442242,\n   "lon": 37.5092205,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718274989",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6597753,\n   "lon": 37.7503511,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718275455",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7073575,\n   "lon": 37.457065,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9718678567",\n   "kind": "organisation",\n   "name": "Московский художественно-производственный комбинат",\n   "office_type": "company",\n   "lat": 55.811284,\n   "lon": 37.5603917,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9720409962",\n   "kind": "organisation",\n   "name": "BNS Club",\n   "office_type": "company",\n   "lat": 55.7802238,\n   "lon": 37.69207,\n   "opening_hours": "Mo-Su 10:00-18:30"\n  },\n  {\n   "id": "node/9725505258",\n   "kind": "organisation",\n   "name": "Famall",\n   "office_type": "company",\n   "lat": 55.8481704,\n   "lon": 37.6293403,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9726743634",\n   "kind": "organisation",\n   "name": "Юридическая консультация № 5",\n   "office_type": "lawyer",\n   "lat": 55.7706164,\n   "lon": 37.6160357,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9733008117",\n   "kind": "organisation",\n   "name": "ЦИТМ Экспонента",\n   "office_type": "company",\n   "lat": 55.7038094,\n   "lon": 37.6946249,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9735580775",\n   "kind": "organisation",\n   "name": "Филиал дирекции по строительству сетей",\n   "office_type": "company",\n   "lat": 55.7797717,\n   "lon": 37.688659,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9735580786",\n   "kind": "organisation",\n   "name": "Служба автоматики и связи",\n   "office_type": "engineer",\n   "lat": 55.7778145,\n   "lon": 37.6743065,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9740048163",\n   "kind": "organisation",\n   "name": "Отделение социальной реабилитации детей и подростков инвалидов",\n   "office_type": "government",\n   "lat": 55.8122576,\n   "lon": 37.5637029,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/9741583421",\n   "kind": "organisation",\n   "name": "Международный культурно-деловой центр \\"ГРААЛЬ\\"",\n   "office_type": "company",\n   "lat": 55.7916268,\n   "lon": 37.5820908,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9741740955",\n   "kind": "organisation",\n   "name": "ОДС № 2 ГБУ \\"Жилищник района Покровское-Стрешнево\\"",\n   "office_type": "property_management",\n   "lat": 55.82762,\n   "lon": 37.4514686,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9743066955",\n   "kind": "organisation",\n   "name": "НИИЯФ МГУ",\n   "office_type": "research",\n   "lat": 55.6994247,\n   "lon": 37.5285399,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9749036895",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6514648,\n   "lon": 37.824643,\n   "opening_hours": "Mo-Su 08:00-19:00"\n  },\n  {\n   "id": "node/9749402010",\n   "kind": "organisation",\n   "name": "ОДС № 1082",\n   "office_type": "property_management",\n   "lat": 55.679261,\n   "lon": 37.5549179,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9749580256",\n   "kind": "organisation",\n   "name": "Столичная судоходная компания",\n   "office_type": "company",\n   "lat": 55.7441143,\n   "lon": 37.6568895,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9750333122",\n   "kind": "organisation",\n   "name": "Пятая газета",\n   "office_type": "newspaper",\n   "lat": 55.8125515,\n   "lon": 37.5850707,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9750333124",\n   "kind": "organisation",\n   "name": "Надежный страховщик",\n   "office_type": "insurance",\n   "lat": 55.8125048,\n   "lon": 37.5850788,\n   "opening_hours": "Mo-Fr 11:00-19:00; Sa 11:00-17:00"\n  },\n  {\n   "id": "node/9751005517",\n   "kind": "organisation",\n   "name": "Винсер",\n   "office_type": "company",\n   "lat": 55.7720619,\n   "lon": 37.5759413,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9751372016",\n   "kind": "organisation",\n   "name": "АО «Авиапром»",\n   "office_type": "company",\n   "lat": 55.7694808,\n   "lon": 37.6416117,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9754181569",\n   "kind": "organisation",\n   "name": "Яндекс",\n   "office_type": "company",\n   "lat": 55.7500989,\n   "lon": 37.5347601,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/9759748784",\n   "kind": "organisation",\n   "name": "Простор недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.7783413,\n   "lon": 37.7042892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9759801381",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7816914,\n   "lon": 37.7054467,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9759801382",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7816502,\n   "lon": 37.7054163,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9768692297",\n   "kind": "organisation",\n   "name": "Жилищник района Южнопортовый",\n   "office_type": "property_management",\n   "lat": 55.7010849,\n   "lon": 37.6867813,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/9775169631",\n   "kind": "organisation",\n   "name": "СПИД.центр",\n   "office_type": "charity",\n   "lat": 55.7521648,\n   "lon": 37.6688402,\n   "opening_hours": "Th-Su 13:00-21:00"\n  },\n  {\n   "id": "node/9783486440",\n   "kind": "organisation",\n   "name": "Инкотек карты и атласы",\n   "office_type": "company",\n   "lat": 55.7745386,\n   "lon": 37.6739472,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9786036617",\n   "kind": "organisation",\n   "name": "ФГУП Охрана, отдел по ЦАО",\n   "office_type": "company",\n   "lat": 55.7442738,\n   "lon": 37.655892,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9786036620",\n   "kind": "organisation",\n   "name": "Банк непрофильных активов Траст",\n   "office_type": "company",\n   "lat": 55.744772,\n   "lon": 37.6560136,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9786547182",\n   "kind": "organisation",\n   "name": "Машиноимпорт",\n   "office_type": "company",\n   "lat": 55.7469959,\n   "lon": 37.661337,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9787495657",\n   "kind": "organisation",\n   "name": "АО Управление технической эксплуатации ВДНХ",\n   "office_type": "engineer",\n   "lat": 55.8281277,\n   "lon": 37.623716,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9790942988",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7723464,\n   "lon": 37.6782556,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9791085033",\n   "kind": "organisation",\n   "name": "Питч-энд-патт",\n   "office_type": "company",\n   "lat": 55.6846457,\n   "lon": 37.5460224,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9795700643",\n   "kind": "organisation",\n   "name": "Департамент капитального строительства ГлавУпДК при МИД России",\n   "office_type": "government",\n   "lat": 55.7414015,\n   "lon": 37.6061885,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9797333502",\n   "kind": "organisation",\n   "name": "R-Studios",\n   "office_type": "media",\n   "lat": 55.7405447,\n   "lon": 37.7057594,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9799108845",\n   "kind": "organisation",\n   "name": "Ipsos",\n   "office_type": "company",\n   "lat": 55.7856905,\n   "lon": 37.6603899,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9805728653",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.617992,\n   "lon": 37.507714,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9805728654",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.801408,\n   "lon": 37.5318893,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/9805728656",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.790525,\n   "lon": 37.529413,\n   "opening_hours": "Fr-Sa 10:00-23:00;Mo-Th,Su 10:00-22:00"\n  },\n  {\n   "id": "node/9805728657",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7493602,\n   "lon": 37.5395697,\n   "opening_hours": "Fr-Sa 10:00-23:00;Mo-Th,Su 10:00-22:00"\n  },\n  {\n   "id": "node/9805728658",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.71083,\n   "lon": 37.676486,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/9805728659",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.850418,\n   "lon": 37.4428949,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9805728660",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.5868936,\n   "lon": 37.7254575,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9805728661",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6922017,\n   "lon": 37.5279933,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/9805728663",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.7479064,\n   "lon": 37.7051908,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9805728664",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6699462,\n   "lon": 37.5521895,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9807930725",\n   "kind": "organisation",\n   "name": "РЭУ 52",\n   "office_type": "property_management",\n   "lat": 55.7847499,\n   "lon": 37.6837733,\n   "opening_hours": "Mo 15:00-16:00; Tu 16:00-17:00"\n  },\n  {\n   "id": "node/9810728907",\n   "kind": "organisation",\n   "name": "Karcher",\n   "office_type": "company",\n   "lat": 55.7508027,\n   "lon": 37.7728493,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9810728908",\n   "kind": "organisation",\n   "name": "Грузовичкоф",\n   "office_type": "company",\n   "lat": 55.750268,\n   "lon": 37.7707454,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9813059593",\n   "kind": "organisation",\n   "name": "gazteplo.ru",\n   "office_type": "company",\n   "lat": 55.8551358,\n   "lon": 37.4444476,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:00"\n  },\n  {\n   "id": "node/9822426333",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.6402948,\n   "lon": 37.5332331,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9823034321",\n   "kind": "organisation",\n   "name": "Miralls",\n   "office_type": "company",\n   "lat": 55.8483592,\n   "lon": 37.6192063,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9824031117",\n   "kind": "organisation",\n   "name": "Президентский фонд культурных инициатив",\n   "office_type": "government",\n   "lat": 55.744786,\n   "lon": 37.6303133,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9824031217",\n   "kind": "organisation",\n   "name": "Фонд президентских грантов",\n   "office_type": "government",\n   "lat": 55.7508499,\n   "lon": 37.584525,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9826120335",\n   "kind": "organisation",\n   "name": "Кошкина Елена Сергеевна",\n   "office_type": "lawyer",\n   "lat": 55.7149447,\n   "lon": 37.6060457,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9828233722",\n   "kind": "organisation",\n   "name": "Центр московского долголетия",\n   "office_type": "company",\n   "lat": 55.7465115,\n   "lon": 37.6736198,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9831178717",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.8709209,\n   "lon": 37.6768514,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9831370427",\n   "kind": "organisation",\n   "name": "ПФР",\n   "office_type": "government",\n   "lat": 55.6795799,\n   "lon": 37.749934,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9833079276",\n   "kind": "organisation",\n   "name": "Proffinans",\n   "office_type": "company",\n   "lat": 55.7430994,\n   "lon": 37.6337099,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9833263577",\n   "kind": "organisation",\n   "name": "Главный информационно-вычислительный центр Министерства культуры Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7665083,\n   "lon": 37.6616693,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9835508762",\n   "kind": "organisation",\n   "name": "Центр корпоративного учёта и отчётности Желдоручёта",\n   "office_type": "company",\n   "lat": 55.7642381,\n   "lon": 37.6510511,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9835568393",\n   "kind": "organisation",\n   "name": "Видео8",\n   "office_type": "company",\n   "lat": 55.7869903,\n   "lon": 37.6739095,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9836931769",\n   "kind": "organisation",\n   "name": "vfs.global",\n   "office_type": "visa",\n   "lat": 55.6724103,\n   "lon": 37.6314461,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9841433493",\n   "kind": "organisation",\n   "name": "Байбараш О.В.",\n   "office_type": "lawyer",\n   "lat": 55.6808333,\n   "lon": 37.5645263,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9841791694",\n   "kind": "organisation",\n   "name": "НПО \\"Креативная механика\\"",\n   "office_type": "company",\n   "lat": 55.7411321,\n   "lon": 37.7430046,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9846450817",\n   "kind": "organisation",\n   "name": "Ансамбль народного танца «Каблучок» имени Киры Черданцевой",\n   "office_type": "yes",\n   "lat": 55.8385958,\n   "lon": 37.5472279,\n   "opening_hours": "Mo, We, Fr 19:00-21:00"\n  },\n  {\n   "id": "node/9853289942",\n   "kind": "organisation",\n   "name": "Московский Государственный академический камерный хор",\n   "office_type": "yes",\n   "lat": 55.7692334,\n   "lon": 37.6633592,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9853909305",\n   "kind": "organisation",\n   "name": "Русский хор",\n   "office_type": "yes",\n   "lat": 55.7692695,\n   "lon": 37.6634207,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9854308927",\n   "kind": "organisation",\n   "name": "Lysterra",\n   "office_type": "company",\n   "lat": 55.7115252,\n   "lon": 37.498945,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9855376438",\n   "kind": "organisation",\n   "name": "Виртуозы Москвы",\n   "office_type": "yes",\n   "lat": 55.7333232,\n   "lon": 37.6465908,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9856341836",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.7722003,\n   "lon": 37.497388,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9856607028",\n   "kind": "organisation",\n   "name": "ОДС № 1 ГБУ \\"Жилищник Басманного района\\"",\n   "office_type": "property_management",\n   "lat": 55.7542434,\n   "lon": 37.6389335,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9856607063",\n   "kind": "organisation",\n   "name": "Мосаэро",\n   "office_type": "company",\n   "lat": 55.7531921,\n   "lon": 37.6396993,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9857550264",\n   "kind": "organisation",\n   "name": "Stanfood",\n   "office_type": "company",\n   "lat": 55.8343122,\n   "lon": 37.6320131,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/9859321310",\n   "kind": "organisation",\n   "name": "Реюнико",\n   "office_type": "company",\n   "lat": 55.7233877,\n   "lon": 37.6554613,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9860933922",\n   "kind": "organisation",\n   "name": "Skupkamd.ru",\n   "office_type": "company",\n   "lat": 55.7704815,\n   "lon": 37.6225111,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/9862243269",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7380012,\n   "lon": 37.4908354,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9862403304",\n   "kind": "organisation",\n   "name": "Amio",\n   "office_type": "advertising_agency",\n   "lat": 55.7947818,\n   "lon": 37.7105136,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9862403305",\n   "kind": "organisation",\n   "name": "Коллегия адвокатов Артура Катанского",\n   "office_type": "lawyer",\n   "lat": 55.7947366,\n   "lon": 37.7105118,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9862403306",\n   "kind": "organisation",\n   "name": "Адвокатский кабинет Захаров Н.В.",\n   "office_type": "lawyer",\n   "lat": 55.7946913,\n   "lon": 37.7105118,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9862403308",\n   "kind": "organisation",\n   "name": "Проектная организация",\n   "office_type": "yes",\n   "lat": 55.794524,\n   "lon": 37.7105038,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-17:00"\n  },\n  {\n   "id": "node/9863715200",\n   "kind": "organisation",\n   "name": "Нотариус г. Москвы Калинина Ольга Александровна",\n   "office_type": "lawyer",\n   "lat": 55.6866704,\n   "lon": 37.7520221,\n   "opening_hours": "Mo-Fr 09:00-14:00,15:00-17:00"\n  },\n  {\n   "id": "node/9866025281",\n   "kind": "organisation",\n   "name": "Technoflot",\n   "office_type": "company",\n   "lat": 55.794731,\n   "lon": 37.507862,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9866025284",\n   "kind": "organisation",\n   "name": "le 16ème bureau",\n   "office_type": "architect",\n   "lat": 55.7945954,\n   "lon": 37.5082915,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9867036322",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8900557,\n   "lon": 37.5372148,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/9868431777",\n   "kind": "organisation",\n   "name": "Дезконтроль",\n   "office_type": "company",\n   "lat": 55.7453362,\n   "lon": 37.6467334,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9868431778",\n   "kind": "organisation",\n   "name": "Представительство Ульяновской области при Правительстве РФ",\n   "office_type": "government",\n   "lat": 55.7454841,\n   "lon": 37.6466596,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9868431810",\n   "kind": "organisation",\n   "name": "Концерн Радиоэлектронные технологии",\n   "office_type": "company",\n   "lat": 55.7442982,\n   "lon": 37.6478451,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9870500568",\n   "kind": "organisation",\n   "name": "Технолог",\n   "office_type": "company",\n   "lat": 55.8540089,\n   "lon": 37.5179163,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9876392979",\n   "kind": "organisation",\n   "name": "Посольство Королевства Бахрейн",\n   "office_type": "diplomatic",\n   "lat": 55.7510105,\n   "lon": 37.5654037,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9883144017",\n   "kind": "organisation",\n   "name": "Химлаб",\n   "office_type": "company",\n   "lat": 55.804451,\n   "lon": 37.7560158,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9884802982",\n   "kind": "organisation",\n   "name": "Мосинжпроект",\n   "office_type": "company",\n   "lat": 55.7871877,\n   "lon": 37.5348776,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9885173596",\n   "kind": "organisation",\n   "name": "Представительство Донецкой Народной республики в Москве",\n   "office_type": "diplomatic",\n   "lat": 55.7776922,\n   "lon": 37.6389227,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9885437819",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.6721554,\n   "lon": 37.7615521,\n   "opening_hours": "Mo-Th 10:00-18:00; Fr 10:00-17:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/9888057217",\n   "kind": "organisation",\n   "name": "Московская городская детская музыкальная школа имени Гнесиных",\n   "office_type": "government",\n   "lat": 55.7441327,\n   "lon": 37.4872156,\n   "opening_hours": "Mo-Sa 09:00-21:00"\n  },\n  {\n   "id": "node/9888459498",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7702672,\n   "lon": 37.682244,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9888459499",\n   "kind": "organisation",\n   "name": "Королайна Инжиниринг",\n   "office_type": "yes",\n   "lat": 55.7702332,\n   "lon": 37.6819611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9891382261",\n   "kind": "organisation",\n   "name": "Экон-бухгалтер",\n   "office_type": "yes",\n   "lat": 55.7963371,\n   "lon": 37.7050088,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9908253742",\n   "kind": "organisation",\n   "name": "Росэнергоатом",\n   "office_type": "company",\n   "lat": 55.7099159,\n   "lon": 37.625159,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9917810417",\n   "kind": "organisation",\n   "name": "Союз десантников России",\n   "office_type": "ngo",\n   "lat": 55.7599112,\n   "lon": 37.6863509,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/9918850466",\n   "kind": "organisation",\n   "name": "PrimeCord",\n   "office_type": "company",\n   "lat": 55.7429077,\n   "lon": 37.5268131,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9926515146",\n   "kind": "organisation",\n   "name": "Неософт",\n   "office_type": "company",\n   "lat": 55.7461927,\n   "lon": 37.6547758,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/9929588819",\n   "kind": "organisation",\n   "name": "ANK Translations",\n   "office_type": "translator",\n   "lat": 55.6704671,\n   "lon": 37.5679609,\n   "opening_hours": "Mo-Fr 10:00-13:00, 14:00-17:00"\n  },\n  {\n   "id": "node/9933988521",\n   "kind": "organisation",\n   "name": "Визовый центр Италии Almaviva Russia",\n   "office_type": "visa",\n   "lat": 55.7278962,\n   "lon": 37.6139498,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9938894936",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.8488045,\n   "lon": 37.4408518,\n   "opening_hours": "Mo-Fr 09:30-20:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/9948239165",\n   "kind": "organisation",\n   "name": "ТСН \\"Сосновая Аллея\\"",\n   "office_type": "property_management",\n   "lat": 55.8178655,\n   "lon": 37.464791,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9949746848",\n   "kind": "organisation",\n   "name": "Жилищник района Южное Тушино",\n   "office_type": "property_management",\n   "lat": 55.8457108,\n   "lon": 37.4549971,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9949746882",\n   "kind": "organisation",\n   "name": "Офис продаж ЖК «Царская площадь»",\n   "office_type": "estate_agent",\n   "lat": 55.7856661,\n   "lon": 37.563435,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9954759774",\n   "kind": "organisation",\n   "name": "Госфильмофонд",\n   "office_type": "yes",\n   "lat": 55.7629201,\n   "lon": 37.6050057,\n   "opening_hours": "Mo-Th 09:00-13:00; 13:45-18:00; Fr 09:00-13:00; 13:45-16:45"\n  },\n  {\n   "id": "node/9955033810",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6836784,\n   "lon": 37.4909085,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9957506953",\n   "kind": "organisation",\n   "name": "АО Финансовый брокер Авгкст",\n   "office_type": "company",\n   "lat": 55.7664248,\n   "lon": 37.6105873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9959392756",\n   "kind": "organisation",\n   "name": "Zaochnik",\n   "office_type": "company",\n   "lat": 55.7542688,\n   "lon": 37.6349944,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9960687524",\n   "kind": "organisation",\n   "name": "АМОДО",\n   "office_type": "architect",\n   "lat": 55.8396597,\n   "lon": 37.4427061,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9973372464",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "travel_agent",\n   "lat": 55.6838806,\n   "lon": 37.491654,\n   "opening_hours": "Mo-Fr 11:00-20:00; Sa 11:00-17:00"\n  },\n  {\n   "id": "node/9977242741",\n   "kind": "organisation",\n   "name": "Международная ассоциация Союзов архитекторов",\n   "office_type": "ngo",\n   "lat": 55.7592482,\n   "lon": 37.5925796,\n   "opening_hours": null\n  },\n  {\n   "id": "node/9986797917",\n   "kind": "organisation",\n   "name": "Илья Кудрин",\n   "office_type": "estate_agent",\n   "lat": 55.7581213,\n   "lon": 37.633979,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10001405483",\n   "kind": "organisation",\n   "name": "Федеральная пробирная палата",\n   "office_type": "government",\n   "lat": 55.7620628,\n   "lon": 37.5960034,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10002011651",\n   "kind": "organisation",\n   "name": "Жилищник района Северное Тушино",\n   "office_type": "property_management",\n   "lat": 55.8550235,\n   "lon": 37.4459921,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10007401762",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.8795569,\n   "lon": 37.5560828,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10012848682",\n   "kind": "organisation",\n   "name": "Медгамал",\n   "office_type": "research",\n   "lat": 55.80158,\n   "lon": 37.45277,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10019428030",\n   "kind": "organisation",\n   "name": "Спецмашмонтаж",\n   "office_type": "company",\n   "lat": 55.7495858,\n   "lon": 37.6056392,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10024296731",\n   "kind": "organisation",\n   "name": "Bosfor pictures",\n   "office_type": "company",\n   "lat": 55.7562561,\n   "lon": 37.6547398,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10025656493",\n   "kind": "organisation",\n   "name": "Финам",\n   "office_type": "company",\n   "lat": 55.739444,\n   "lon": 37.5372255,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10025656495",\n   "kind": "organisation",\n   "name": "Управа района Дорогомилово",\n   "office_type": "government",\n   "lat": 55.7391035,\n   "lon": 37.5376084,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10025656496",\n   "kind": "organisation",\n   "name": "Д. С. Бублий",\n   "office_type": "notary",\n   "lat": 55.7389155,\n   "lon": 37.5387999,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10027575306",\n   "kind": "organisation",\n   "name": "Delomant Group",\n   "office_type": "lawyer",\n   "lat": 55.7409033,\n   "lon": 37.539209,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10028341315",\n   "kind": "organisation",\n   "name": "ОДС № 588 ГБУ \\"Жилищник района Дорогомилово\\"",\n   "office_type": "property_management",\n   "lat": 55.7412891,\n   "lon": 37.5401651,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10028347617",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7406775,\n   "lon": 37.540089,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10028347618",\n   "kind": "organisation",\n   "name": "В. Д. Цабрия",\n   "office_type": "notary",\n   "lat": 55.740845,\n   "lon": 37.5398606,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10029522905",\n   "kind": "organisation",\n   "name": "Все вместе",\n   "office_type": "charity",\n   "lat": 55.7806746,\n   "lon": 37.6393627,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/10034932308",\n   "kind": "organisation",\n   "name": "Комплексный вычислительный центр Дмнг СФГ",\n   "office_type": "company",\n   "lat": 55.7077524,\n   "lon": 37.5982743,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10035605817",\n   "kind": "organisation",\n   "name": "Леджит",\n   "office_type": "company",\n   "lat": 55.7626216,\n   "lon": 37.6371328,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10046688631",\n   "kind": "organisation",\n   "name": "Ренессанс страхование",\n   "office_type": "insurance",\n   "lat": 55.757848,\n   "lon": 37.57314,\n   "opening_hours": "Fr 09:00-17:00;Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/10046688633",\n   "kind": "organisation",\n   "name": "Ренессанс страхование",\n   "office_type": "insurance",\n   "lat": 55.7090591,\n   "lon": 37.5930262,\n   "opening_hours": "Fr 09:30-17:15;Mo-Th 09:30-18:30"\n  },\n  {\n   "id": "node/10046688634",\n   "kind": "organisation",\n   "name": "Ренессанс страхование",\n   "office_type": "insurance",\n   "lat": 55.729311,\n   "lon": 37.609173,\n   "opening_hours": "Fr 09:30-17:15;Mo-Th 09:30-18:30"\n  },\n  {\n   "id": "node/10047202832",\n   "kind": "organisation",\n   "name": "Maryway",\n   "office_type": "company",\n   "lat": 55.746899,\n   "lon": 37.663112,\n   "opening_hours": "Mo-Su 11:00-20:00"\n  },\n  {\n   "id": "node/10047911090",\n   "kind": "organisation",\n   "name": "ВСК",\n   "office_type": "insurance",\n   "lat": 55.8215078,\n   "lon": 37.5106306,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10050187469",\n   "kind": "organisation",\n   "name": "Пункт явки и отдыха локомотивных бригад на ст. Белокаменная",\n   "office_type": "tranportation",\n   "lat": 55.8286617,\n   "lon": 37.7034514,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10054793440",\n   "kind": "organisation",\n   "name": "Музейно-парковый комплекс \\"Северное Тушино\\"",\n   "office_type": "government",\n   "lat": 55.8610255,\n   "lon": 37.4498364,\n   "opening_hours": "Mo-Th 09:00-13:00,13:45-18:00; Fr 09:00-13:00,13:45-16:45"\n  },\n  {\n   "id": "node/10056664270",\n   "kind": "organisation",\n   "name": "Институт Мосинжпроект",\n   "office_type": "engineer",\n   "lat": 55.7968463,\n   "lon": 37.5988553,\n   "opening_hours": "Mo-Fr 08:30-17:30"\n  },\n  {\n   "id": "node/10056664271",\n   "kind": "organisation",\n   "name": "Институт Мосинжпроект",\n   "office_type": "engineer",\n   "lat": 55.7961684,\n   "lon": 37.5997531,\n   "opening_hours": "Mo-Fr 08:30-17:30"\n  },\n  {\n   "id": "node/10063587262",\n   "kind": "organisation",\n   "name": "СПБ Биржа",\n   "office_type": "company",\n   "lat": 55.7777296,\n   "lon": 37.6024225,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10068900164",\n   "kind": "organisation",\n   "name": "ООО \\"Помаринка Сервис\\"",\n   "office_type": "company",\n   "lat": 55.7491239,\n   "lon": 37.8141557,\n   "opening_hours": "Mo-Sa 09:00-22:00"\n  },\n  {\n   "id": "node/10069372273",\n   "kind": "organisation",\n   "name": "ОДС № 10 ГБУ \\"Жилищник Басманного района\\"",\n   "office_type": "property_management",\n   "lat": 55.7601492,\n   "lon": 37.6586786,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10069488656",\n   "kind": "organisation",\n   "name": "ОДС № 19 ГБУ \\"Жилищник Басманного района\\"",\n   "office_type": "property_management",\n   "lat": 55.7619081,\n   "lon": 37.6566803,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10077643983",\n   "kind": "organisation",\n   "name": "Отдел социальной защиты населения района Дорогомилово",\n   "office_type": "government",\n   "lat": 55.7401114,\n   "lon": 37.543357,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10078484381",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.6116136,\n   "lon": 37.6015716,\n   "opening_hours": "PH,Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/10086579611",\n   "kind": "organisation",\n   "name": "Интек-М",\n   "office_type": "telecommunication",\n   "lat": 55.8962158,\n   "lon": 37.7134389,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10090014079",\n   "kind": "organisation",\n   "name": "Газпром Телеком",\n   "office_type": "company",\n   "lat": 55.6599002,\n   "lon": 37.538065,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10090569961",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник\\" Останкинского района",\n   "office_type": "property_management",\n   "lat": 55.8198423,\n   "lon": 37.6350242,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10101228053",\n   "kind": "organisation",\n   "name": "ППК Профиль",\n   "office_type": "company",\n   "lat": 55.7882572,\n   "lon": 37.6353534,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10108907717",\n   "kind": "organisation",\n   "name": "Неосфера",\n   "office_type": "company",\n   "lat": 55.7167531,\n   "lon": 37.633706,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/10113485945",\n   "kind": "organisation",\n   "name": "ООО \\"ПОЛИМЕР МЕ\\"",\n   "office_type": "company",\n   "lat": 55.7522947,\n   "lon": 37.5972097,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10123104418",\n   "kind": "organisation",\n   "name": "Телеграф",\n   "office_type": "coworking",\n   "lat": 55.7924605,\n   "lon": 37.6086142,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10127729135",\n   "kind": "organisation",\n   "name": "Учебно-методический центр по гражданской обороне и чрезвычайным ситуациям Северо-Западного административного округа",\n   "office_type": "educational_institution",\n   "lat": 55.8452346,\n   "lon": 37.4422773,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10127749267",\n   "kind": "organisation",\n   "name": "Совет ветеранов первичной организации №4 района Южное Тушино Северо-Западного административного округа города Москвы",\n   "office_type": "ngo",\n   "lat": 55.8454327,\n   "lon": 37.4423199,\n   "opening_hours": "Mo-Th 11:00-15:00"\n  },\n  {\n   "id": "node/10128641500",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "translator",\n   "lat": 55.7942754,\n   "lon": 37.7164319,\n   "opening_hours": "Mo-Fr 10:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/10129622420",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.8459131,\n   "lon": 37.44524,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10133179098",\n   "kind": "organisation",\n   "name": "Satellit center",\n   "office_type": "security",\n   "lat": 55.8179395,\n   "lon": 37.5092541,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10135070834",\n   "kind": "organisation",\n   "name": "Прогрессивная молодежь",\n   "office_type": "political_party",\n   "lat": 55.8454426,\n   "lon": 37.4424659,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10146889639",\n   "kind": "organisation",\n   "name": "Давыдов Александр Игоревич",\n   "office_type": "lawyer",\n   "lat": 55.7706263,\n   "lon": 37.6788208,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10146889640",\n   "kind": "organisation",\n   "name": "Судебный центр оценки и экспертизы",\n   "office_type": "surveyor",\n   "lat": 55.7706343,\n   "lon": 37.6787761,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10146889641",\n   "kind": "organisation",\n   "name": "Смирнова Ольга Владимировна",\n   "office_type": "notary",\n   "lat": 55.7705922,\n   "lon": 37.6788203,\n   "opening_hours": "Mo-Fr 09:30-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/10148063231",\n   "kind": "organisation",\n   "name": "Макс-М",\n   "office_type": "insurance",\n   "lat": 55.8027765,\n   "lon": 37.5380978,\n   "opening_hours": "Mo-Th 10:00-14:00,14:45-19:00; Fr 10:00-14:00,14:45-18:00"\n  },\n  {\n   "id": "node/10167446717",\n   "kind": "organisation",\n   "name": "Окна Калева",\n   "office_type": "company",\n   "lat": 55.6470427,\n   "lon": 37.6188444,\n   "opening_hours": "Su-Sa 09:00-21:00"\n  },\n  {\n   "id": "node/10178006917",\n   "kind": "organisation",\n   "name": "ООО \\"Аргус-Альбион\\"",\n   "office_type": "company",\n   "lat": 55.7526049,\n   "lon": 37.7717531,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10183282306",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.891829,\n   "lon": 37.7276677,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/10183283181",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6564177,\n   "lon": 37.5692967,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10183283187",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8881664,\n   "lon": 37.6623425,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/10183283188",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6733254,\n   "lon": 37.6124078,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10183283189",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.707299,\n   "lon": 37.456828,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10183283191",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6536924,\n   "lon": 37.6207146,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10183283192",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8037669,\n   "lon": 37.8007981,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10183283193",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7309449,\n   "lon": 37.6402545,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10183283196",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6009825,\n   "lon": 37.6092401,\n   "opening_hours": "Mo-Sa 10:00-20:00"\n  },\n  {\n   "id": "node/10198888583",\n   "kind": "organisation",\n   "name": "Центральное конструкторское бюро тяжелых путевых машин",\n   "office_type": "company",\n   "lat": 55.7871343,\n   "lon": 37.6848583,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10201995173",\n   "kind": "organisation",\n   "name": "KR Недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.7353087,\n   "lon": 37.585051,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10207144317",\n   "kind": "organisation",\n   "name": "Элеосмед",\n   "office_type": "company",\n   "lat": 55.7094905,\n   "lon": 37.5904267,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10207943482",\n   "kind": "organisation",\n   "name": "ОДС № 21",\n   "office_type": "property_management",\n   "lat": 55.7735268,\n   "lon": 37.7093523,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10209727267",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6638573,\n   "lon": 37.5115487,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10209727273",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6308717,\n   "lon": 37.6590592,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/10209727281",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.6373561,\n   "lon": 37.6002305,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10209727305",\n   "kind": "organisation",\n   "name": "Билайн",\n   "office_type": "telecommunication",\n   "lat": 55.8107595,\n   "lon": 37.8012702,\n   "opening_hours": "Fr-Sa 10:00-23:00; Mo-Th,Su 10:00-22:00"\n  },\n  {\n   "id": "node/10214406637",\n   "kind": "organisation",\n   "name": "Мосинтер",\n   "office_type": "telecommunication",\n   "lat": 55.7900622,\n   "lon": 37.7127962,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10214406639",\n   "kind": "organisation",\n   "name": "Геокадастр",\n   "office_type": "geodesist",\n   "lat": 55.7900645,\n   "lon": 37.7128646,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10233800618",\n   "kind": "organisation",\n   "name": "Администрация парка",\n   "office_type": "yes",\n   "lat": 55.8668496,\n   "lon": 37.6827763,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10235020415",\n   "kind": "organisation",\n   "name": "Комбинат питания «Кремлёвский»",\n   "office_type": "company",\n   "lat": 55.7540954,\n   "lon": 37.6306334,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10235020416",\n   "kind": "organisation",\n   "name": "Главное управление специальных программ Президента Российской Федерации",\n   "office_type": "government",\n   "lat": 55.7559476,\n   "lon": 37.6292463,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10240176163",\n   "kind": "organisation",\n   "name": "Комитет солдатских матерей Москвы",\n   "office_type": "ngo",\n   "lat": 55.7828028,\n   "lon": 37.7156702,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10242170039",\n   "kind": "organisation",\n   "name": "Научно-исследовательский и конструкторский институт химического машиностроения",\n   "office_type": "research",\n   "lat": 55.801183,\n   "lon": 37.5855482,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10242556233",\n   "kind": "organisation",\n   "name": "МосВодоСтрой",\n   "office_type": "company",\n   "lat": 55.7918741,\n   "lon": 37.6026808,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10242556248",\n   "kind": "organisation",\n   "name": "Следственный отдел ОМВД России по Тверскому району Г. Москвы",\n   "office_type": "government",\n   "lat": 55.7900947,\n   "lon": 37.5978622,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10243833213",\n   "kind": "organisation",\n   "name": "Визовый центр Болгарии",\n   "office_type": "visa",\n   "lat": 55.6724582,\n   "lon": 37.6317756,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/10243833214",\n   "kind": "organisation",\n   "name": "Визовый центр Чехии",\n   "office_type": "visa",\n   "lat": 55.6724604,\n   "lon": 37.631928,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/10243833215",\n   "kind": "organisation",\n   "name": "Визовый центр Польши",\n   "office_type": "visa",\n   "lat": 55.6724572,\n   "lon": 37.6318495,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/10243833216",\n   "kind": "organisation",\n   "name": "Визовый центр Канады и Новой Зеландии",\n   "office_type": "visa",\n   "lat": 55.672526,\n   "lon": 37.6318266,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/10243833217",\n   "kind": "organisation",\n   "name": "Многофункциональный центр предоставления государственных и муниципальных услуг района Марьина роща",\n   "office_type": "government",\n   "lat": 55.7943508,\n   "lon": 37.6043022,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/10247174534",\n   "kind": "organisation",\n   "name": "Пенсионный фонд Российской Федерации Бутырский Марфино",\n   "office_type": "government",\n   "lat": 55.8260908,\n   "lon": 37.5867578,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10253751374",\n   "kind": "organisation",\n   "name": "Инженерная служба района Беговой",\n   "office_type": "property_management",\n   "lat": 55.7769517,\n   "lon": 37.5770254,\n   "opening_hours": "Mo-Th 08:00-12:00,13:00-17:00; Fr 08:00-12:00,13:00-15:00"\n  },\n  {\n   "id": "node/10255485712",\n   "kind": "organisation",\n   "name": "Московско-Смоленская дистанция электроснабжения",\n   "office_type": "government",\n   "lat": 55.7815961,\n   "lon": 37.5852008,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10255485755",\n   "kind": "organisation",\n   "name": "Жилищник района Беговой",\n   "office_type": "property_management",\n   "lat": 55.7863094,\n   "lon": 37.5722377,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/10255485761",\n   "kind": "organisation",\n   "name": "Управление на транспорте МВД России по центральному федеральному округу",\n   "office_type": "government",\n   "lat": 55.7858464,\n   "lon": 37.5709502,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10258960537",\n   "kind": "organisation",\n   "name": "Научно-исследовательский институт дальней радиосвязи",\n   "office_type": "research",\n   "lat": 55.8035528,\n   "lon": 37.5529486,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10259383369",\n   "kind": "organisation",\n   "name": "Ant technologies",\n   "office_type": "it",\n   "lat": 55.8070907,\n   "lon": 37.5667579,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10261366365",\n   "kind": "organisation",\n   "name": "Совет ветеранов №7 Бутырского района",\n   "office_type": "ngo",\n   "lat": 55.8163306,\n   "lon": 37.5836743,\n   "opening_hours": "Tu 11:00-14:00; Th 14:00-17:00"\n  },\n  {\n   "id": "node/10261366366",\n   "kind": "organisation",\n   "name": "Совет ветеранов №5 Бутырского района",\n   "office_type": "ngo",\n   "lat": 55.8163487,\n   "lon": 37.5836691,\n   "opening_hours": "Mo,We 12:00-15:00"\n  },\n  {\n   "id": "node/10261366367",\n   "kind": "organisation",\n   "name": "Совет ветеранов №4 Бутырского района",\n   "office_type": "ngo",\n   "lat": 55.8163669,\n   "lon": 37.5836639,\n   "opening_hours": "Tu,Th 14:00-17:00"\n  },\n  {\n   "id": "node/10261366368",\n   "kind": "organisation",\n   "name": "Совет ветеранов №3 Бутырского района",\n   "office_type": "ngo",\n   "lat": 55.8163851,\n   "lon": 37.5836588,\n   "opening_hours": "Mo 11:00-14:00; We 15:00-18:00"\n  },\n  {\n   "id": "node/10261366369",\n   "kind": "organisation",\n   "name": "Организация инвалидов \\"Бутырский\\"",\n   "office_type": "ngo",\n   "lat": 55.8162939,\n   "lon": 37.5836875,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10261367312",\n   "kind": "organisation",\n   "name": "Жилищник Бутырского района ОДФ №551",\n   "office_type": "property_management",\n   "lat": 55.817717,\n   "lon": 37.5840232,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10264799124",\n   "kind": "organisation",\n   "name": "Всероссийское физкультурно-спортивное общество \\"Динамо\\"",\n   "office_type": "government",\n   "lat": 55.7882639,\n   "lon": 37.5668371,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10266358656",\n   "kind": "organisation",\n   "name": "ОДС № 557 ГБУ \\"Жилищник Бутырского района\\"",\n   "office_type": "property_management",\n   "lat": 55.816937,\n   "lon": 37.5906739,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10266362653",\n   "kind": "organisation",\n   "name": "Pac Group",\n   "office_type": "company",\n   "lat": 55.727783,\n   "lon": 37.6243908,\n   "opening_hours": "09:00-20:00"\n  },\n  {\n   "id": "node/10267867020",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7421047,\n   "lon": 37.6597816,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10271419902",\n   "kind": "organisation",\n   "name": "Московский государственный академический симфонический оркестр под управлением Павла Когана",\n   "office_type": "company",\n   "lat": 55.7770429,\n   "lon": 37.6814412,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10273072878",\n   "kind": "organisation",\n   "name": "Россети Московский регион",\n   "office_type": "engineer",\n   "lat": 55.77843,\n   "lon": 37.6672026,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10274685442",\n   "kind": "organisation",\n   "name": "Всероссийский научно-исследовательский институт геофизических методов разведки",\n   "office_type": "company",\n   "lat": 55.7788697,\n   "lon": 37.6666018,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10279037250",\n   "kind": "organisation",\n   "name": "Центральный НИИ организации и информатизации здравоохранения Минздрава России",\n   "office_type": "research",\n   "lat": 55.8143208,\n   "lon": 37.5924334,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10279037280",\n   "kind": "organisation",\n   "name": "Отдел Военного комиссариата - Бутырский",\n   "office_type": "government",\n   "lat": 55.8140369,\n   "lon": 37.5915025,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10281347986",\n   "kind": "organisation",\n   "name": "ГеоКомпани",\n   "office_type": "geodesist",\n   "lat": 55.6483438,\n   "lon": 37.5022703,\n   "opening_hours": "Mo-Fr 08:00-18:00; Sa-Su 09:00-17:00"\n  },\n  {\n   "id": "node/10281348007",\n   "kind": "organisation",\n   "name": "Голд Бизнес",\n   "office_type": "lawyer",\n   "lat": 55.73863,\n   "lon": 37.628663,\n   "opening_hours": "Fr 09:30-17:00; Mo-Th 09:30-18:30"\n  },\n  {\n   "id": "node/10281368728",\n   "kind": "organisation",\n   "name": "Всп33",\n   "office_type": "company",\n   "lat": 55.820356,\n   "lon": 37.510088,\n   "opening_hours": "Mo-Su 08:00-17:00"\n  },\n  {\n   "id": "node/10281368730",\n   "kind": "organisation",\n   "name": "Ретро Арт",\n   "office_type": "company",\n   "lat": 55.665115,\n   "lon": 37.631865,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/10281368736",\n   "kind": "organisation",\n   "name": "Exmail",\n   "office_type": "logistics",\n   "lat": 55.7756075,\n   "lon": 37.5860551,\n   "opening_hours": "Mo-Su 08:00-21:00"\n  },\n  {\n   "id": "node/10281368737",\n   "kind": "organisation",\n   "name": "Dalli",\n   "office_type": "logistics",\n   "lat": 55.801878,\n   "lon": 37.5921893,\n   "opening_hours": "Mo-Fr 09:00-22:00"\n  },\n  {\n   "id": "node/10281368779",\n   "kind": "organisation",\n   "name": "Коллегия адвокатов Легион",\n   "office_type": "lawyer",\n   "lat": 55.706325,\n   "lon": 37.617872,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/10281368780",\n   "kind": "organisation",\n   "name": "Ермак Дом",\n   "office_type": "construction_company",\n   "lat": 55.661593,\n   "lon": 37.508792,\n   "opening_hours": "Mo-Su 11:00-22:00"\n  },\n  {\n   "id": "node/10281368784",\n   "kind": "organisation",\n   "name": "Borzova Design Group",\n   "office_type": "architect",\n   "lat": 55.7557318,\n   "lon": 37.6579656,\n   "opening_hours": "Mo-Su 08:00-18:00"\n  },\n  {\n   "id": "node/10281368790",\n   "kind": "organisation",\n   "name": "Skn-Dom",\n   "office_type": "estate_agent",\n   "lat": 55.607963,\n   "lon": 37.719992,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/10281368794",\n   "kind": "organisation",\n   "name": "Кортекс",\n   "office_type": "company",\n   "lat": 55.739274,\n   "lon": 37.611687,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10281368796",\n   "kind": "organisation",\n   "name": "Наш штат",\n   "office_type": "moving_company",\n   "lat": 55.7521573,\n   "lon": 37.6648852,\n   "opening_hours": "Mo-Su 07:00-21:00"\n  },\n  {\n   "id": "node/10281368805",\n   "kind": "organisation",\n   "name": "Пск профиль",\n   "office_type": "company",\n   "lat": 55.6607936,\n   "lon": 37.5387758,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10281368822",\n   "kind": "organisation",\n   "name": "Diamond Personnel R&C",\n   "office_type": "employment_agency",\n   "lat": 55.7697871,\n   "lon": 37.6204303,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10282313977",\n   "kind": "organisation",\n   "name": "Образцова Елена Петровна",\n   "office_type": "notary",\n   "lat": 55.7904409,\n   "lon": 37.6769083,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa-Su 10:00-16:00"\n  },\n  {\n   "id": "node/10282426922",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7210561,\n   "lon": 37.8229881,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-17:00; Su 10:00-14:00"\n  },\n  {\n   "id": "node/10285492511",\n   "kind": "organisation",\n   "name": "Постоянное представительство главы Удмуртской республики при президенте Российской федерации",\n   "office_type": "government",\n   "lat": 55.786338,\n   "lon": 37.6115119,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10286309491",\n   "kind": "organisation",\n   "name": "Секретариат делегации Российской Федерации в Европейской комиссии за демократию через право (Венецианская комиссия)",\n   "office_type": "government",\n   "lat": 55.7656331,\n   "lon": 37.6490307,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10286572967",\n   "kind": "organisation",\n   "name": "Филина Елена Анательевна",\n   "office_type": "notary",\n   "lat": 55.7810622,\n   "lon": 37.5833595,\n   "opening_hours": "Mo-Fr 09:00-17:00"\n  },\n  {\n   "id": "node/10290077109",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.7394209,\n   "lon": 37.4840103,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10290077209",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.7394664,\n   "lon": 37.4841423,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10291017665",\n   "kind": "organisation",\n   "name": "НИИ СК - НИИ стартовых комплексов им. В.П. Бармина",\n   "office_type": "research",\n   "lat": 55.7832439,\n   "lon": 37.6304135,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10291507228",\n   "kind": "organisation",\n   "name": "Народные художественные промыслы России",\n   "office_type": "association",\n   "lat": 55.7618953,\n   "lon": 37.6481523,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10293704597",\n   "kind": "organisation",\n   "name": "Фабрика-прачечная №55",\n   "office_type": "company",\n   "lat": 55.8135236,\n   "lon": 37.6320845,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10296038380",\n   "kind": "organisation",\n   "name": "ОДС ТСЖ \\"Наш Дом\\"",\n   "office_type": "property_management",\n   "lat": 55.8014578,\n   "lon": 37.5700974,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10296500492",\n   "kind": "organisation",\n   "name": "Научный центр оперативного мониторинга земли",\n   "office_type": "research",\n   "lat": 55.859749,\n   "lon": 37.6251671,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10297885866",\n   "kind": "organisation",\n   "name": "LastPrint",\n   "office_type": "company",\n   "lat": 55.8682681,\n   "lon": 37.4982761,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/10300375728",\n   "kind": "organisation",\n   "name": "Падва и партнеры",\n   "office_type": "lawyer",\n   "lat": 55.769268,\n   "lon": 37.6268823,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10300700538",\n   "kind": "organisation",\n   "name": "Федеральная служба по ветеринарному и фитосанитарному надзору",\n   "office_type": "government",\n   "lat": 55.7703302,\n   "lon": 37.6453896,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10308758299",\n   "kind": "organisation",\n   "name": "Вертолетная сервисная компания",\n   "office_type": "company",\n   "lat": 55.7297231,\n   "lon": 37.6345387,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10308922824",\n   "kind": "organisation",\n   "name": "Chalet Studio",\n   "office_type": "coworking",\n   "lat": 55.7624264,\n   "lon": 37.6069577,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10309021859",\n   "kind": "organisation",\n   "name": "Феникс",\n   "office_type": "lawyer",\n   "lat": 55.771541,\n   "lon": 37.6297791,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10312023947",\n   "kind": "organisation",\n   "name": "Управа Тверского района города Москвы",\n   "office_type": "government",\n   "lat": 55.7722222,\n   "lon": 37.620028,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10312103767",\n   "kind": "organisation",\n   "name": "\\"Жиличник\\" района Щукинский",\n   "office_type": "property_management",\n   "lat": 55.8006885,\n   "lon": 37.4859591,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10538986512",\n   "kind": "organisation",\n   "name": "РусБизнесОценка",\n   "office_type": "company",\n   "lat": 55.7035129,\n   "lon": 37.6915477,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa-Su,PH off"\n  },\n  {\n   "id": "node/10539246258",\n   "kind": "organisation",\n   "name": "Консалт плюс",\n   "office_type": "lawyer",\n   "lat": 55.7034512,\n   "lon": 37.6914653,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa-Su,PH off"\n  },\n  {\n   "id": "node/10539246259",\n   "kind": "organisation",\n   "name": "Ligir",\n   "office_type": "consulting",\n   "lat": 55.7035123,\n   "lon": 37.6915101,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10542156141",\n   "kind": "organisation",\n   "name": "Постнаука",\n   "office_type": "publisher",\n   "lat": 55.7582276,\n   "lon": 37.6259022,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10548655061",\n   "kind": "organisation",\n   "name": "ГустоЛайф",\n   "office_type": "company",\n   "lat": 55.7035008,\n   "lon": 37.691499,\n   "opening_hours": "Mo-Fr 08:30-17:30; Sa-Su,PH off"\n  },\n  {\n   "id": "node/10556619154",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6100192,\n   "lon": 37.5363725,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/10563067487",\n   "kind": "organisation",\n   "name": "ТрансТелеКом",\n   "office_type": "telecommunication",\n   "lat": 55.7763173,\n   "lon": 37.6682393,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10563070024",\n   "kind": "organisation",\n   "name": "Greeenway",\n   "office_type": "company",\n   "lat": 55.7717032,\n   "lon": 37.660933,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10566781190",\n   "kind": "organisation",\n   "name": "Патрубки и РТИ",\n   "office_type": "ngo",\n   "lat": 55.7962913,\n   "lon": 37.4599076,\n   "opening_hours": "ПН-ПТ 9:00-18:00"\n  },\n  {\n   "id": "node/10576238946",\n   "kind": "organisation",\n   "name": "Инспекция Ростехнадзора",\n   "office_type": "government",\n   "lat": 55.8026008,\n   "lon": 37.5697204,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10576681479",\n   "kind": "organisation",\n   "name": "Эксперт Клининг",\n   "office_type": "company",\n   "lat": 55.7263956,\n   "lon": 37.6887087,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10584410871",\n   "kind": "organisation",\n   "name": "Прием металлолома",\n   "office_type": "company",\n   "lat": 55.6469826,\n   "lon": 37.562897,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/10587668562",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "ngo",\n   "lat": 55.8063218,\n   "lon": 37.5808701,\n   "opening_hours": "Mo-Th 11:00-15:00"\n  },\n  {\n   "id": "node/10588759491",\n   "kind": "organisation",\n   "name": "ВНИПИнефть",\n   "office_type": "company",\n   "lat": 55.7349096,\n   "lon": 37.6145523,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10589349846",\n   "kind": "organisation",\n   "name": "Участковый пункт полиции 28",\n   "office_type": "government",\n   "lat": 55.8264855,\n   "lon": 37.8232321,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10589369905",\n   "kind": "organisation",\n   "name": "Совет ветеранов",\n   "office_type": "government",\n   "lat": 55.8264794,\n   "lon": 37.822558,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10593741730",\n   "kind": "organisation",\n   "name": "Nextons",\n   "office_type": "lawyer",\n   "lat": 55.7786427,\n   "lon": 37.5879418,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10596739206",\n   "kind": "organisation",\n   "name": "Роскартография",\n   "office_type": "geodesist",\n   "lat": 55.7133724,\n   "lon": 37.7215269,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10596952414",\n   "kind": "organisation",\n   "name": "Инспекция по контролю за состоянием художественного оформления и рекламы",\n   "office_type": "government",\n   "lat": 55.7101229,\n   "lon": 37.6648611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10603563334",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8941332,\n   "lon": 37.4495924,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/10603563361",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.674805,\n   "lon": 37.503982,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/10603563392",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6538059,\n   "lon": 37.6179331,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/10603563400",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7536758,\n   "lon": 37.7779216,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/10603591511",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8643339,\n   "lon": 37.5923449,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/10603591526",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8158897,\n   "lon": 37.5276446,\n   "opening_hours": "Mo-Sa 10:00-20:00"\n  },\n  {\n   "id": "node/10603591528",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6562407,\n   "lon": 37.5690607,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/10603591532",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6048085,\n   "lon": 37.7124971,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/10607125790",\n   "kind": "organisation",\n   "name": "Приоритет",\n   "office_type": "estate_agent",\n   "lat": 55.7647473,\n   "lon": 37.6546996,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/10628561078",\n   "kind": "organisation",\n   "name": "Смородина",\n   "office_type": "it",\n   "lat": 55.767904,\n   "lon": 37.6749569,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10628574036",\n   "kind": "organisation",\n   "name": "Мосводосбыт",\n   "office_type": "company",\n   "lat": 55.7713019,\n   "lon": 37.6754934,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10642640881",\n   "kind": "organisation",\n   "name": "Яндекс",\n   "office_type": "company",\n   "lat": 55.751227,\n   "lon": 37.5337101,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10646591013",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.780271,\n   "lon": 37.6003298,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10648617054",\n   "kind": "organisation",\n   "name": "Росгеолфонд",\n   "office_type": "government",\n   "lat": 55.7700375,\n   "lon": 37.515218,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10648656384",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7395822,\n   "lon": 37.6699374,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10652913823",\n   "kind": "organisation",\n   "name": "Принтсувенир",\n   "office_type": "company",\n   "lat": 55.7665426,\n   "lon": 37.7330301,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10652913848",\n   "kind": "organisation",\n   "name": "Гранд-Альфа",\n   "office_type": "company",\n   "lat": 55.7662527,\n   "lon": 37.7326169,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10655293054",\n   "kind": "organisation",\n   "name": "Дизайн стиль",\n   "office_type": "graphic_design",\n   "lat": 55.7586973,\n   "lon": 37.7336591,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/10656181312",\n   "kind": "organisation",\n   "name": "Росинфра",\n   "office_type": "company",\n   "lat": 55.7707223,\n   "lon": 37.6068681,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10657572425",\n   "kind": "organisation",\n   "name": "InterMotion",\n   "office_type": "advertising_agency",\n   "lat": 55.8607184,\n   "lon": 37.6558557,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10660795105",\n   "kind": "organisation",\n   "name": "Tapper",\n   "office_type": "it",\n   "lat": 55.7496426,\n   "lon": 37.537641,\n   "opening_hours": "Su-Th 10:00-19:00"\n  },\n  {\n   "id": "node/10669523005",\n   "kind": "organisation",\n   "name": "Моснедвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.7667251,\n   "lon": 37.6028659,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10669779169",\n   "kind": "organisation",\n   "name": "Капитал Life",\n   "office_type": "insurance",\n   "lat": 55.7256374,\n   "lon": 37.6431245,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10669808008",\n   "kind": "organisation",\n   "name": "Капитал Life",\n   "office_type": "insurance",\n   "lat": 55.7740899,\n   "lon": 37.6290697,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10669808021",\n   "kind": "organisation",\n   "name": "Капитал Life",\n   "office_type": "insurance",\n   "lat": 55.745065,\n   "lon": 37.622514,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10669953173",\n   "kind": "organisation",\n   "name": "Лонмади",\n   "office_type": "company",\n   "lat": 55.6044,\n   "lon": 37.6365,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10669995278",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.7656178,\n   "lon": 37.6114342,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10669995279",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.7432413,\n   "lon": 37.585693,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10669995280",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.7496636,\n   "lon": 37.5371772,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10669995282",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.8701705,\n   "lon": 37.6590245,\n   "opening_hours": "Fr-Sa 10:00-17:00; Mo-Th 10:00-18:00"\n  },\n  {\n   "id": "node/10669995283",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.7660692,\n   "lon": 37.5328748,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10669995284",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.708657,\n   "lon": 37.4442225,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10669995285",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.7790681,\n   "lon": 37.5871521,\n   "opening_hours": "Mo-Fr 10:00-14:00,15:00-18:00"\n  },\n  {\n   "id": "node/10669995286",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.7471397,\n   "lon": 37.5365603,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/10669995287",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.765894,\n   "lon": 37.6045516,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10669995288",\n   "kind": "organisation",\n   "name": "Awatera",\n   "office_type": "translator",\n   "lat": 55.7421989,\n   "lon": 37.6530346,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/10671874146",\n   "kind": "organisation",\n   "name": "Greenko",\n   "office_type": "company",\n   "lat": 55.8124086,\n   "lon": 37.4562092,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10679361858",\n   "kind": "organisation",\n   "name": "Приёмная ЛДПР",\n   "office_type": "political_party",\n   "lat": 55.8819714,\n   "lon": 37.527729,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10687848042",\n   "kind": "organisation",\n   "name": "Først",\n   "office_type": "estate_agent",\n   "lat": 55.7053668,\n   "lon": 37.6328788,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/10698200823",\n   "kind": "organisation",\n   "name": "Beauty Toria",\n   "office_type": "coworking",\n   "lat": 55.7642132,\n   "lon": 37.6088767,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10698200830",\n   "kind": "organisation",\n   "name": "Особые подарки",\n   "office_type": "company",\n   "lat": 55.763565,\n   "lon": 37.6099321,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10705457667",\n   "kind": "organisation",\n   "name": "Ngenix",\n   "office_type": "company",\n   "lat": 55.8005309,\n   "lon": 37.554921,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10716324050",\n   "kind": "organisation",\n   "name": "Управление лицензирования и аккредитации",\n   "office_type": "government",\n   "lat": 55.7830863,\n   "lon": 37.7055489,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10718433393",\n   "kind": "organisation",\n   "name": "РЖД-Транспортная безопасность",\n   "office_type": "company",\n   "lat": 55.7705363,\n   "lon": 37.6608696,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10718451854",\n   "kind": "organisation",\n   "name": "МосжилНИИпроект",\n   "office_type": "company",\n   "lat": 55.7709098,\n   "lon": 37.6609791,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10719137723",\n   "kind": "organisation",\n   "name": "Совет ветеранов района \\"Бутырский\\"",\n   "office_type": "ngo",\n   "lat": 55.8158011,\n   "lon": 37.5899309,\n   "opening_hours": "Mo-Th 11:00-15:00"\n  },\n  {\n   "id": "node/10722898461",\n   "kind": "organisation",\n   "name": "Экобур",\n   "office_type": "company",\n   "lat": 55.7829566,\n   "lon": 37.716059,\n   "opening_hours": "Mo-Su 09:00-19:00"\n  },\n  {\n   "id": "node/10722898462",\n   "kind": "organisation",\n   "name": "Экобур",\n   "office_type": "company",\n   "lat": 55.7618228,\n   "lon": 37.6579786,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/10722898558",\n   "kind": "organisation",\n   "name": "Пресс Бюро",\n   "office_type": "advertising_agency",\n   "lat": 55.7915679,\n   "lon": 37.585323,\n   "opening_hours": "Mo-Fr 11:00-19:00"\n  },\n  {\n   "id": "node/10722898559",\n   "kind": "organisation",\n   "name": "Ора Консалтинг",\n   "office_type": "accountant",\n   "lat": 55.7746134,\n   "lon": 37.5484425,\n   "opening_hours": "Fr 09:00-17:00; Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/10722898615",\n   "kind": "organisation",\n   "name": "Плюсофон",\n   "office_type": "telecommunication",\n   "lat": 55.710673,\n   "lon": 37.6695255,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/10722898635",\n   "kind": "organisation",\n   "name": "Dk Entertainment",\n   "office_type": "company",\n   "lat": 55.7996347,\n   "lon": 37.5306487,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/10722898643",\n   "kind": "organisation",\n   "name": "Megagroup.ru",\n   "office_type": "it",\n   "lat": 55.7564802,\n   "lon": 37.6326665,\n   "opening_hours": "Tu,Th 09:30-18:00"\n  },\n  {\n   "id": "node/10722898644",\n   "kind": "organisation",\n   "name": "Юридический центр",\n   "office_type": "lawyer",\n   "lat": 55.7642298,\n   "lon": 37.614964,\n   "opening_hours": "Mo-Fr 08:30-19:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/10722898664",\n   "kind": "organisation",\n   "name": "Atmokids",\n   "office_type": "company",\n   "lat": 55.6870295,\n   "lon": 37.5854945,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10722898675",\n   "kind": "organisation",\n   "name": "Веванта",\n   "office_type": "construction_company",\n   "lat": 55.6494141,\n   "lon": 37.5402831,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/10723963436",\n   "kind": "organisation",\n   "name": "Кристи",\n   "office_type": "educational_institution",\n   "lat": 55.7098752,\n   "lon": 37.6070833,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/10727531643",\n   "kind": "organisation",\n   "name": "БКС Мир инвестиций",\n   "office_type": "financial_advisor",\n   "lat": 55.8042314,\n   "lon": 37.5204331,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/10734504225",\n   "kind": "organisation",\n   "name": "ПЭК",\n   "office_type": "logistics",\n   "lat": 55.83479,\n   "lon": 37.512703,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/10737232390",\n   "kind": "organisation",\n   "name": "Филиал РПОХБВ Роспотребнадзора",\n   "office_type": "government",\n   "lat": 55.7463756,\n   "lon": 37.5107105,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10742401042",\n   "kind": "organisation",\n   "name": "Городские квартиры",\n   "office_type": "estate_agent",\n   "lat": 55.8584364,\n   "lon": 37.5060622,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa-Su 11:00-16:00"\n  },\n  {\n   "id": "node/10743223895",\n   "kind": "organisation",\n   "name": "Региональная общественная организация многодетных семей",\n   "office_type": "ngo",\n   "lat": 55.7710884,\n   "lon": 37.602776,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10748305963",\n   "kind": "organisation",\n   "name": "ООО \\"ОРП-Инжиниринг\\"",\n   "office_type": "company",\n   "lat": 55.7569662,\n   "lon": 37.6769713,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10761038582",\n   "kind": "organisation",\n   "name": "Территориальная первичная ветеранская организация №1 района Северное Тушино",\n   "office_type": "ngo",\n   "lat": 55.8567973,\n   "lon": 37.4454484,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10761051949",\n   "kind": "organisation",\n   "name": "Территориальная первичная ветеранская организация №3 района Северное Тушино",\n   "office_type": "ngo",\n   "lat": 55.8568059,\n   "lon": 37.445419,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10761115952",\n   "kind": "organisation",\n   "name": "Станкомаш",\n   "office_type": "company",\n   "lat": 55.8507342,\n   "lon": 37.6683774,\n   "opening_hours": "Mo-Th 08:30-17:30, Fr 08:30-16:30"\n  },\n  {\n   "id": "node/10766576298",\n   "kind": "organisation",\n   "name": "Центр социально-трудовой адаптации и профориентации",\n   "office_type": "government",\n   "lat": 55.7753712,\n   "lon": 37.6398245,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10768881492",\n   "kind": "organisation",\n   "name": "Дирекция объединения Выставочные залы Москвы",\n   "office_type": "company",\n   "lat": 55.7362099,\n   "lon": 37.6511128,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/10770404624",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7692303,\n   "lon": 37.5910252,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10771845805",\n   "kind": "organisation",\n   "name": "Бондарева И.А.",\n   "office_type": "notary",\n   "lat": 55.8372276,\n   "lon": 37.5013912,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10773865478",\n   "kind": "organisation",\n   "name": "Школьный межрайонный совет СВАО",\n   "office_type": "ngo",\n   "lat": 55.8126095,\n   "lon": 37.592079,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10773931194",\n   "kind": "organisation",\n   "name": "НТЦ “Вулкан”",\n   "office_type": "it",\n   "lat": 55.7888118,\n   "lon": 37.7291611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10777809968",\n   "kind": "organisation",\n   "name": "Мосгорспорт",\n   "office_type": "government",\n   "lat": 55.7705032,\n   "lon": 37.7029789,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10779210740",\n   "kind": "organisation",\n   "name": "Независимая газета",\n   "office_type": "newspaper",\n   "lat": 55.7626509,\n   "lon": 37.6338896,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10780715528",\n   "kind": "organisation",\n   "name": "Благотворительный фонд «Зажги свечу»",\n   "office_type": "charity",\n   "lat": 55.8037515,\n   "lon": 37.5707333,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10786728049",\n   "kind": "organisation",\n   "name": "Москоллектор",\n   "office_type": "company",\n   "lat": 55.7740205,\n   "lon": 37.6391307,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10786728122",\n   "kind": "organisation",\n   "name": "Служба финансового контроля",\n   "office_type": "government",\n   "lat": 55.7754861,\n   "lon": 37.6398857,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10787474574",\n   "kind": "organisation",\n   "name": "ФГУП \\"Оборонпромэкология\\"",\n   "office_type": "military",\n   "lat": 55.7423324,\n   "lon": 37.6417431,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10787918260",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7405837,\n   "lon": 37.6718027,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10787918277",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6909757,\n   "lon": 37.6587242,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/10787918287",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.715511,\n   "lon": 37.612306,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10788242011",\n   "kind": "organisation",\n   "name": "Единый визовый центр",\n   "office_type": "government",\n   "lat": 55.7529694,\n   "lon": 37.5758949,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-19:00"\n  },\n  {\n   "id": "node/10788242012",\n   "kind": "organisation",\n   "name": "Единый визовый центр",\n   "office_type": "visa",\n   "lat": 55.7274968,\n   "lon": 37.6088941,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa-Su 09:00-19:00"\n  },\n  {\n   "id": "node/10788242014",\n   "kind": "organisation",\n   "name": "Единый визовый центр",\n   "office_type": "visa",\n   "lat": 55.7731591,\n   "lon": 37.6219886,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/10788242015",\n   "kind": "organisation",\n   "name": "Единый визовый центр",\n   "office_type": "visa",\n   "lat": 55.7393413,\n   "lon": 37.52684,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/10788242016",\n   "kind": "organisation",\n   "name": "Единый визовый центр",\n   "office_type": "visa",\n   "lat": 55.741071,\n   "lon": 37.659206,\n   "opening_hours": "Mo 10:00-19:00; Sa 09:00-18:00; Tu-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/10789680684",\n   "kind": "organisation",\n   "name": "Кадровый центр Департамента культуры города Москвы",\n   "office_type": "government",\n   "lat": 55.8141988,\n   "lon": 37.5950767,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10791178575",\n   "kind": "organisation",\n   "name": "Администрация муниципального округа Южное Медведково",\n   "office_type": "government",\n   "lat": 55.8781125,\n   "lon": 37.6262319,\n   "opening_hours": "Mo-Th 08:00-12:00,13:00-17:00; Fr 08:00-12:00,13:00-15:45"\n  },\n  {\n   "id": "node/10795069846",\n   "kind": "organisation",\n   "name": "Pro poest",\n   "office_type": "educational_institution",\n   "lat": 55.7513518,\n   "lon": 37.6067518,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10797115740",\n   "kind": "organisation",\n   "name": "ОДС № 4 ГБУ \\"Жилищник Таганского района\\"",\n   "office_type": "property_management",\n   "lat": 55.7427718,\n   "lon": 37.6460762,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10798140379",\n   "kind": "organisation",\n   "name": "МОЭК",\n   "office_type": "company",\n   "lat": 55.8052517,\n   "lon": 37.5740053,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10799043627",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.804397,\n   "lon": 37.583279,\n   "opening_hours": "10:00-21:00"\n  },\n  {\n   "id": "node/10803614872",\n   "kind": "organisation",\n   "name": "Абсолют Страхование",\n   "office_type": "insurance",\n   "lat": 55.7103319,\n   "lon": 37.6531071,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10803845901",\n   "kind": "organisation",\n   "name": "МЦСТ",\n   "office_type": "research",\n   "lat": 55.6398336,\n   "lon": 37.5230646,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10807442036",\n   "kind": "organisation",\n   "name": "СберЛизинг",\n   "office_type": "financial_advisor",\n   "lat": 55.7364798,\n   "lon": 37.5343662,\n   "opening_hours": "Fr 09:00-17:00; Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/10827526196",\n   "kind": "organisation",\n   "name": "Освобождение",\n   "office_type": "lawyer",\n   "lat": 55.7302471,\n   "lon": 37.6198243,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10828724807",\n   "kind": "organisation",\n   "name": "ППК Роскадастр",\n   "office_type": "government",\n   "lat": 55.7495438,\n   "lon": 37.7113826,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10831673695",\n   "kind": "organisation",\n   "name": "Экотехсервис",\n   "office_type": "company",\n   "lat": 55.8447337,\n   "lon": 37.5731081,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10831746305",\n   "kind": "organisation",\n   "name": "Юридическая Фирма по Экстрадиции",\n   "office_type": "lawyer",\n   "lat": 55.7095354,\n   "lon": 37.646676,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10840475333",\n   "kind": "organisation",\n   "name": "ОДС № 714",\n   "office_type": "property_management",\n   "lat": 55.7442589,\n   "lon": 37.5431451,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10844094861",\n   "kind": "organisation",\n   "name": "Газпромбанк Автолизинг",\n   "office_type": "financial_advisor",\n   "lat": 55.7820124,\n   "lon": 37.629869,\n   "opening_hours": "Fr 09:00-16:45; Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/10845639164",\n   "kind": "organisation",\n   "name": "Российское авторское общество",\n   "office_type": "company",\n   "lat": 55.760697,\n   "lon": 37.5988519,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10845639165",\n   "kind": "organisation",\n   "name": "Группа \\"ОНЭКСИМ\\"",\n   "office_type": "company",\n   "lat": 55.7603439,\n   "lon": 37.5992677,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10846185776",\n   "kind": "organisation",\n   "name": "Мосметрострой",\n   "office_type": "company",\n   "lat": 55.771776,\n   "lon": 37.6200287,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10856195510",\n   "kind": "organisation",\n   "name": "Lunda",\n   "office_type": "water_utility",\n   "lat": 55.8313133,\n   "lon": 37.4463744,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10857574244",\n   "kind": "organisation",\n   "name": "Ингосстрах",\n   "office_type": "insurance",\n   "lat": 55.7120379,\n   "lon": 37.6124432,\n   "opening_hours": "Mo-Th 10:00-19:00; Fr-Sa 10:00-18:00; Su off"\n  },\n  {\n   "id": "node/10870326136",\n   "kind": "organisation",\n   "name": "ОДС № 586 ГБУ \\"Жилищник района Дорогомилово\\"",\n   "office_type": "property_management",\n   "lat": 55.7407931,\n   "lon": 37.5464147,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10870865329",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7383944,\n   "lon": 37.5166042,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/10870883807",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8064576,\n   "lon": 37.5039491,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/10887055301",\n   "kind": "organisation",\n   "name": "Миронова Ирина Вячеславовна",\n   "office_type": "notary",\n   "lat": 55.7863883,\n   "lon": 37.5602729,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10915647181",\n   "kind": "organisation",\n   "name": "Медный Всадник",\n   "office_type": "company",\n   "lat": 55.7763837,\n   "lon": 37.6907538,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10916290332",\n   "kind": "organisation",\n   "name": "Издательство Видар",\n   "office_type": "publisher",\n   "lat": 55.7552523,\n   "lon": 37.6450956,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10919371005",\n   "kind": "organisation",\n   "name": "Юниметалл",\n   "office_type": "company",\n   "lat": 55.7366683,\n   "lon": 37.7371167,\n   "opening_hours": "Su-Sa 10:00-20:00"\n  },\n  {\n   "id": "node/10922965252",\n   "kind": "organisation",\n   "name": "Московская осетинская община",\n   "office_type": "quango",\n   "lat": 55.7840652,\n   "lon": 37.5973192,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10926491487",\n   "kind": "organisation",\n   "name": "Мои документы Москворечье-Сабурово",\n   "office_type": "government",\n   "lat": 55.6341392,\n   "lon": 37.6572043,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/10926751046",\n   "kind": "organisation",\n   "name": "Netwell",\n   "office_type": "it",\n   "lat": 55.723921,\n   "lon": 37.6537813,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10928207055",\n   "kind": "organisation",\n   "name": "Тсн Недвижимость",\n   "office_type": "estate_agent",\n   "lat": 55.800509,\n   "lon": 37.6366764,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-17:00"\n  },\n  {\n   "id": "node/10930392567",\n   "kind": "organisation",\n   "name": "Адвокат Федосеева С. Н.",\n   "office_type": "lawyer",\n   "lat": 55.8111353,\n   "lon": 37.5770055,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/10932266017",\n   "kind": "organisation",\n   "name": "Адвокат по гражданским делам Голованова",\n   "office_type": "lawyer",\n   "lat": 55.8127248,\n   "lon": 37.5761873,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10932266024",\n   "kind": "organisation",\n   "name": "Positive2B",\n   "office_type": "advertising_agency",\n   "lat": 55.8114965,\n   "lon": 37.5767599,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/10963592236",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7950604,\n   "lon": 37.6127119,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10963635089",\n   "kind": "organisation",\n   "name": "Столица",\n   "office_type": "company",\n   "lat": 55.7941089,\n   "lon": 37.6095733,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10963641377",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7943972,\n   "lon": 37.6094064,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10964974016",\n   "kind": "organisation",\n   "name": "Сарапульский радиозавод",\n   "office_type": "company",\n   "lat": 55.7935989,\n   "lon": 37.6091357,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10965165384",\n   "kind": "organisation",\n   "name": "Совет ветеранов ПО № 3",\n   "office_type": "ngo",\n   "lat": 55.7959016,\n   "lon": 37.6154937,\n   "opening_hours": "Tu,Th 12:00-15:00"\n  },\n  {\n   "id": "node/10967807877",\n   "kind": "organisation",\n   "name": "Элемент",\n   "office_type": "company",\n   "lat": 55.8152062,\n   "lon": 37.598758,\n   "opening_hours": "Mo-Fr 10:00-17:00"\n  },\n  {\n   "id": "node/10967807879",\n   "kind": "organisation",\n   "name": "Комплексные Системы и Сети",\n   "office_type": "company",\n   "lat": 55.8152439,\n   "lon": 37.5988157,\n   "opening_hours": "Mo-Fr 08:00-19:00"\n  },\n  {\n   "id": "node/10968881805",\n   "kind": "organisation",\n   "name": "ООО \\"Неал\\"",\n   "office_type": "company",\n   "lat": 55.884348,\n   "lon": 37.6318707,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/10968895497",\n   "kind": "organisation",\n   "name": "Бюро переводов",\n   "office_type": "translator",\n   "lat": 55.7963355,\n   "lon": 37.6170519,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10969632457",\n   "kind": "organisation",\n   "name": "ОДС №4",\n   "office_type": "property_maintenance",\n   "lat": 55.7531139,\n   "lon": 37.7148321,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/10969632458",\n   "kind": "organisation",\n   "name": "Управление городским имуществом в Юго-Восточном округе",\n   "office_type": "government",\n   "lat": 55.7526221,\n   "lon": 37.7147495,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/10969632460",\n   "kind": "organisation",\n   "name": "Кирилл и Мефодий",\n   "office_type": "translator",\n   "lat": 55.7523965,\n   "lon": 37.7155467,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10969632462",\n   "kind": "organisation",\n   "name": "Верас АСЦ",\n   "office_type": "lawyer",\n   "lat": 55.7524657,\n   "lon": 37.7158832,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/10976286603",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7125163,\n   "lon": 37.6457128,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10976288017",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7464253,\n   "lon": 37.6751017,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/10976288029",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8091241,\n   "lon": 37.7298456,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10976288037",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7391434,\n   "lon": 37.4846703,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10976288060",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8786867,\n   "lon": 37.4793593,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/10977464876",\n   "kind": "organisation",\n   "name": "КНГК-ИНПЗ",\n   "office_type": "company",\n   "lat": 55.7168823,\n   "lon": 37.6288846,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10980292851",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "estate_agent",\n   "lat": 55.6377067,\n   "lon": 37.7944437,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10981411672",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.6378642,\n   "lon": 37.7944467,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10981411674",\n   "kind": "organisation",\n   "name": "Адвокат",\n   "office_type": "lawyer",\n   "lat": 55.637744,\n   "lon": 37.7944401,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10996192592",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "estate_agent",\n   "lat": 55.6894032,\n   "lon": 37.5765487,\n   "opening_hours": null\n  },\n  {\n   "id": "node/10996192596",\n   "kind": "organisation",\n   "name": "Эльгрант",\n   "office_type": "lawyer",\n   "lat": 55.6897926,\n   "lon": 37.5791602,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11007197221",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6398161,\n   "lon": 37.5330385,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11007197228",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6740151,\n   "lon": 37.6130837,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11007197230",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7322876,\n   "lon": 37.7321362,\n   "opening_hours": "Mo-Fr 11:00-20:00; Sa 12:00-17:00"\n  },\n  {\n   "id": "node/11007197240",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6405821,\n   "lon": 37.6032239,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11007197241",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.799903,\n   "lon": 37.5268506,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11007197242",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6886911,\n   "lon": 37.5807846,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/11007197259",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.735326,\n   "lon": 37.6139367,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11007197261",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6264499,\n   "lon": 37.666522,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11014205596",\n   "kind": "organisation",\n   "name": "Мировой судья, судебный участок №263",\n   "office_type": "government",\n   "lat": 55.8979335,\n   "lon": 37.4529317,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11015594410",\n   "kind": "organisation",\n   "name": "Турпомощь",\n   "office_type": "association",\n   "lat": 55.7690719,\n   "lon": 37.6443368,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11016702133",\n   "kind": "organisation",\n   "name": "МС Прайс",\n   "office_type": "company",\n   "lat": 55.8040142,\n   "lon": 37.5632338,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11028232832",\n   "kind": "organisation",\n   "name": "ТехАргос",\n   "office_type": "it",\n   "lat": 55.8057431,\n   "lon": 37.5889385,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11028232833",\n   "kind": "organisation",\n   "name": "Сигнатек",\n   "office_type": "it",\n   "lat": 55.7315188,\n   "lon": 37.6444441,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11028232834",\n   "kind": "organisation",\n   "name": "Бастион",\n   "office_type": "it",\n   "lat": 55.7506508,\n   "lon": 37.5392184,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11028232835",\n   "kind": "organisation",\n   "name": "Гарда Технологии",\n   "office_type": "it",\n   "lat": 55.8056587,\n   "lon": 37.5888714,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11028232837",\n   "kind": "organisation",\n   "name": "Основа Лаб",\n   "office_type": "it",\n   "lat": 55.7758233,\n   "lon": 37.6775583,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11028232838",\n   "kind": "organisation",\n   "name": "Цитадель",\n   "office_type": "it",\n   "lat": 55.697359,\n   "lon": 37.5025773,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11028247329",\n   "kind": "organisation",\n   "name": "T.Hunter",\n   "office_type": "it",\n   "lat": 55.8154987,\n   "lon": 37.5184804,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11037404187",\n   "kind": "organisation",\n   "name": "Союз женщин России",\n   "office_type": "ngo",\n   "lat": 55.7631161,\n   "lon": 37.610136,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11039748830",\n   "kind": "organisation",\n   "name": "Sok Рыбаков Тауэр",\n   "office_type": "coworking",\n   "lat": 55.7882994,\n   "lon": 37.5678381,\n   "opening_hours": "Mo-Su 09:00-19:00"\n  },\n  {\n   "id": "node/11046042481",\n   "kind": "organisation",\n   "name": "Обит",\n   "office_type": "it",\n   "lat": 55.7371728,\n   "lon": 37.6356026,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11056877822",\n   "kind": "organisation",\n   "name": "Матиосов В. В.",\n   "office_type": "notary",\n   "lat": 55.7962921,\n   "lon": 37.617273,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11057051023",\n   "kind": "organisation",\n   "name": "Призыва.Нет",\n   "office_type": "lawyer",\n   "lat": 55.7654918,\n   "lon": 37.6387587,\n   "opening_hours": "Mo-Fr 09:00-19:00+; Sa 10:00-16:00; Su off"\n  },\n  {\n   "id": "node/11066379088",\n   "kind": "organisation",\n   "name": "Российский Красный Крест",\n   "office_type": "ngo",\n   "lat": 55.6918026,\n   "lon": 37.5724446,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11066426805",\n   "kind": "organisation",\n   "name": "КОМПО Технолоджис",\n   "office_type": "company",\n   "lat": 55.8788844,\n   "lon": 37.7296678,\n   "opening_hours": "Mo-Sa 09:00-19:00"\n  },\n  {\n   "id": "node/11067289419",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8544289,\n   "lon": 37.4769858,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11067289422",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7719792,\n   "lon": 37.5814577,\n   "opening_hours": "Fr 10:00-18:00; Mo-Th 10:00-19:00"\n  },\n  {\n   "id": "node/11067289427",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7559618,\n   "lon": 37.634764,\n   "opening_hours": "Mo-Fr 09:30-22:00; Sa-Su 11:00-21:00"\n  },\n  {\n   "id": "node/11067289435",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8234674,\n   "lon": 37.5000285,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11069717720",\n   "kind": "organisation",\n   "name": "УФНС России по городу Москве",\n   "office_type": "government",\n   "lat": 55.7746964,\n   "lon": 37.5517952,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11071976149",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.7705324,\n   "lon": 37.5837712,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11083589890",\n   "kind": "organisation",\n   "name": "Bizznes",\n   "office_type": "company",\n   "lat": 55.6835124,\n   "lon": 37.6611006,\n   "opening_hours": "Mo-Su 11:00-17:00"\n  },\n  {\n   "id": "node/11083605305",\n   "kind": "organisation",\n   "name": "Правовой Альянс 24",\n   "office_type": "lawyer",\n   "lat": 55.7390785,\n   "lon": 37.6293138,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11083605314",\n   "kind": "organisation",\n   "name": "Формат",\n   "office_type": "advertising_agency",\n   "lat": 55.67498,\n   "lon": 37.5435448,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11083605322",\n   "kind": "organisation",\n   "name": "Пара рук",\n   "office_type": "company",\n   "lat": 55.60916,\n   "lon": 37.652815,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/11083605323",\n   "kind": "organisation",\n   "name": "Expressrms",\n   "office_type": "logistics",\n   "lat": 55.6457332,\n   "lon": 37.6278359,\n   "opening_hours": "Mo-Fr 09:00-22:00; Sa 10:00-19:00"\n  },\n  {\n   "id": "node/11083605342",\n   "kind": "organisation",\n   "name": "Hualian machinery",\n   "office_type": "company",\n   "lat": 55.7578205,\n   "lon": 37.769757,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11083605355",\n   "kind": "organisation",\n   "name": "Wilstream",\n   "office_type": "company",\n   "lat": 55.7257446,\n   "lon": 37.6486847,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/11083605358",\n   "kind": "organisation",\n   "name": "Экология воды",\n   "office_type": "company",\n   "lat": 55.7955668,\n   "lon": 37.6994615,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/11083605361",\n   "kind": "organisation",\n   "name": "Майлс",\n   "office_type": "logistics",\n   "lat": 55.7763211,\n   "lon": 37.6754478,\n   "opening_hours": "Fr 09:00-17:00; Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/11083605365",\n   "kind": "organisation",\n   "name": "Твоё время",\n   "office_type": "coworking",\n   "lat": 55.7694506,\n   "lon": 37.6448464,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/11083605368",\n   "kind": "organisation",\n   "name": "Группа Партнер",\n   "office_type": "construction_company",\n   "lat": 55.770638,\n   "lon": 37.5249839,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11083605373",\n   "kind": "organisation",\n   "name": "Mod",\n   "office_type": "estate_agent",\n   "lat": 55.762185,\n   "lon": 37.5641227,\n   "opening_hours": "Mo-Su 11:00-19:00"\n  },\n  {\n   "id": "node/11083605387",\n   "kind": "organisation",\n   "name": "Европейский переезд",\n   "office_type": "logistics",\n   "lat": 55.8588639,\n   "lon": 37.5765467,\n   "opening_hours": "Mo-Su 08:00-23:55"\n  },\n  {\n   "id": "node/11083605391",\n   "kind": "organisation",\n   "name": "Группа Партнер",\n   "office_type": "construction_company",\n   "lat": 55.6565479,\n   "lon": 37.733928,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11083605392",\n   "kind": "organisation",\n   "name": "Группа Партнер",\n   "office_type": "construction_company",\n   "lat": 55.7245936,\n   "lon": 37.7213001,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11083605393",\n   "kind": "organisation",\n   "name": "Группа Партнер",\n   "office_type": "construction_company",\n   "lat": 55.6914987,\n   "lon": 37.4401135,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11083605396",\n   "kind": "organisation",\n   "name": "Megagroup",\n   "office_type": "it",\n   "lat": 55.7495911,\n   "lon": 37.5383573,\n   "opening_hours": "Tu,Th 09:30-18:00"\n  },\n  {\n   "id": "node/11083649936",\n   "kind": "organisation",\n   "name": "Сибур Полилаб",\n   "office_type": "research",\n   "lat": 55.7784828,\n   "lon": 37.637921,\n   "opening_hours": "Fr 09:00-16:45; Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/11084060797",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.7600581,\n   "lon": 37.7724862,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11085084468",\n   "kind": "organisation",\n   "name": "Центр экологической безопасности гражданской авиации",\n   "office_type": "consulting",\n   "lat": 55.7528029,\n   "lon": 37.7089288,\n   "opening_hours": "Mo-Fr 09:30-20:00"\n  },\n  {\n   "id": "node/11086472329",\n   "kind": "organisation",\n   "name": "Представительство университета Тохоку",\n   "office_type": "administrative",\n   "lat": 55.6986434,\n   "lon": 37.5344351,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11089137308",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.802902,\n   "lon": 37.5391462,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11099693213",\n   "kind": "organisation",\n   "name": "TomTom",\n   "office_type": "company",\n   "lat": 55.761598,\n   "lon": 37.6588338,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11101213766",\n   "kind": "organisation",\n   "name": "ЦИТ \\"Роскадастр-Инфотех\\"",\n   "office_type": "government",\n   "lat": 55.771651,\n   "lon": 37.650839,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11104660370",\n   "kind": "organisation",\n   "name": "Страна Девелопмент",\n   "office_type": "estate_agent",\n   "lat": 55.6903045,\n   "lon": 37.6576835,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11108426376",\n   "kind": "organisation",\n   "name": "ОДС №4",\n   "office_type": "property_maintenance",\n   "lat": 55.8109026,\n   "lon": 37.5575935,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11114747960",\n   "kind": "organisation",\n   "name": "Ассоциация российских банков",\n   "office_type": "ngo",\n   "lat": 55.7564402,\n   "lon": 37.5931764,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11116148180",\n   "kind": "organisation",\n   "name": "Гера",\n   "office_type": "construction_company",\n   "lat": 55.8057861,\n   "lon": 37.580751,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11118842262",\n   "kind": "organisation",\n   "name": "ФГБУ \\"ДОД Росимущества\\"",\n   "office_type": "government",\n   "lat": 55.7505988,\n   "lon": 37.7088961,\n   "opening_hours": "Mo-Fr 08:00-22:00"\n  },\n  {\n   "id": "node/11126334601",\n   "kind": "organisation",\n   "name": "Офис продаж ЖК \\"Преображенская площадь\\"",\n   "office_type": "estate_agent",\n   "lat": 55.7962179,\n   "lon": 37.7126039,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11136087553",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.8054839,\n   "lon": 37.5584713,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11136087554",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.8053723,\n   "lon": 37.5588924,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11149816848",\n   "kind": "organisation",\n   "name": "Юнистрах",\n   "office_type": "insurance",\n   "lat": 55.7499402,\n   "lon": 37.5375838,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11150676306",\n   "kind": "organisation",\n   "name": "Союз ПНК",\n   "office_type": "company",\n   "lat": 55.7626256,\n   "lon": 37.6564054,\n   "opening_hours": "Mo-Fr 09:30-18:30"\n  },\n  {\n   "id": "node/11155987770",\n   "kind": "organisation",\n   "name": "Hr-profi",\n   "office_type": "employment_agency",\n   "lat": 55.749515,\n   "lon": 37.537707,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/11155987774",\n   "kind": "organisation",\n   "name": "Квартиры Москвы",\n   "office_type": "estate_agent",\n   "lat": 55.7402865,\n   "lon": 37.6299495,\n   "opening_hours": "Mo-Fr 09:00-18:30; Sa-Su 11:00-13:30"\n  },\n  {\n   "id": "node/11155987781",\n   "kind": "organisation",\n   "name": "Visa Vezde",\n   "office_type": "visa",\n   "lat": 55.7635673,\n   "lon": 37.6191777,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11155987783",\n   "kind": "organisation",\n   "name": "Адвокат Репников И.В.",\n   "office_type": "lawyer",\n   "lat": 55.7492137,\n   "lon": 37.5367455,\n   "opening_hours": "Mo-Sa 10:00-18:00"\n  },\n  {\n   "id": "node/11155987784",\n   "kind": "organisation",\n   "name": "Отечественная коллегия адвокатов",\n   "office_type": "lawyer",\n   "lat": 55.875986,\n   "lon": 37.589095,\n   "opening_hours": "Mo-Su 09:00-13:00,14:00-19:00"\n  },\n  {\n   "id": "node/11155987791",\n   "kind": "organisation",\n   "name": "Нью Вэй Лоджистик",\n   "office_type": "logistics",\n   "lat": 55.747067,\n   "lon": 37.534833,\n   "opening_hours": "Mo-Fr 09:00-12:00,13:00-18:00"\n  },\n  {\n   "id": "node/11155987794",\n   "kind": "organisation",\n   "name": "Консалт партнер",\n   "office_type": "company",\n   "lat": 55.7553724,\n   "lon": 37.7495116,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11159472347",\n   "kind": "organisation",\n   "name": "Бюро судебно-медицинской экспертизы",\n   "office_type": "government",\n   "lat": 55.7731183,\n   "lon": 37.6599486,\n   "opening_hours": "Mo 13:00-17:30; Tu-Th 09:00-17:30"\n  },\n  {\n   "id": "node/11161936633",\n   "kind": "organisation",\n   "name": "Содружество ветеранов военной службы и боевых действий",\n   "office_type": "ngo",\n   "lat": 55.7224335,\n   "lon": 37.7845788,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11164485904",\n   "kind": "organisation",\n   "name": "Аронов & партнёры",\n   "office_type": "lawyer",\n   "lat": 55.7100322,\n   "lon": 37.6105412,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11166782925",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.8882687,\n   "lon": 37.5887936,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/11166896577",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.8036976,\n   "lon": 37.7528458,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11166935325",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.7325228,\n   "lon": 37.6713911,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/11166938165",\n   "kind": "organisation",\n   "name": "CarPrice",\n   "office_type": "company",\n   "lat": 55.5988452,\n   "lon": 37.7252317,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/11169439079",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7697086,\n   "lon": 37.6203337,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/11169534490",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7282164,\n   "lon": 37.7760864,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11169554268",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7290644,\n   "lon": 37.7203569,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11169554307",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6531254,\n   "lon": 37.7229621,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11169554312",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7282121,\n   "lon": 37.4520914,\n   "opening_hours": "Mo-Fr 10:00-22:00; Sa-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11171107790",\n   "kind": "organisation",\n   "name": "СДЭК",\n   "office_type": "logistics",\n   "lat": 55.7880264,\n   "lon": 37.6786906,\n   "opening_hours": "Mo-Fr 10:00-21:00; Sa,Su 10:00-20:00"\n  },\n  {\n   "id": "node/11174761628",\n   "kind": "organisation",\n   "name": "Посольство Демократической Республики Конго в Российской федерации",\n   "office_type": "diplomatic",\n   "lat": 55.7776961,\n   "lon": 37.639191,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11175623339",\n   "kind": "organisation",\n   "name": "Посольство Албании",\n   "office_type": "diplomatic",\n   "lat": 55.7277821,\n   "lon": 37.615701,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11176880099",\n   "kind": "organisation",\n   "name": "Офис продаж ЖК \\"39 проспект\\"",\n   "office_type": "estate_agent",\n   "lat": 55.727505,\n   "lon": 37.7585357,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11178629622",\n   "kind": "organisation",\n   "name": "Ресо Гарантия",\n   "office_type": "insurance",\n   "lat": 55.6467459,\n   "lon": 37.6048208,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/11180764416",\n   "kind": "organisation",\n   "name": "Жилищная инспекция по ЮЗАО г.Москвы",\n   "office_type": "government",\n   "lat": 55.6481874,\n   "lon": 37.6026471,\n   "opening_hours": "Tu 15:00-19:00; Th 08:00-12:00"\n  },\n  {\n   "id": "node/11185440228",\n   "kind": "organisation",\n   "name": "Росреестр",\n   "office_type": "government",\n   "lat": 55.817832,\n   "lon": 37.6482223,\n   "opening_hours": "Mo-Th 09:00-18:00; Fr 09:00-16:45"\n  },\n  {\n   "id": "node/11188175437",\n   "kind": "organisation",\n   "name": "ДокМобиль",\n   "office_type": "insurance",\n   "lat": 55.7481038,\n   "lon": 37.5566533,\n   "opening_hours": "Mo-Sa 08:00-23:00"\n  },\n  {\n   "id": "node/11188812114",\n   "kind": "organisation",\n   "name": "Посольство Боливии",\n   "office_type": "diplomatic",\n   "lat": 55.7122591,\n   "lon": 37.6165695,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11188812120",\n   "kind": "organisation",\n   "name": "Посольство Бурунди",\n   "office_type": "diplomatic",\n   "lat": 55.7284283,\n   "lon": 37.6131685,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11193678251",\n   "kind": "organisation",\n   "name": "Ассоциация армянских юристов",\n   "office_type": "lawyer",\n   "lat": 55.7606268,\n   "lon": 37.6353581,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11199078901",\n   "kind": "organisation",\n   "name": "Российский НИИ культурного и природного наследия им. Д.С. Лихачева",\n   "office_type": "research",\n   "lat": 55.7437464,\n   "lon": 37.6102392,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11200051464",\n   "kind": "organisation",\n   "name": "Информатика и управление",\n   "office_type": "research",\n   "lat": 55.6938999,\n   "lon": 37.5645444,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11200063532",\n   "kind": "organisation",\n   "name": "Юридическое бюро Ефремова",\n   "office_type": "lawyer",\n   "lat": 55.6904871,\n   "lon": 37.5675246,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su off"\n  },\n  {\n   "id": "node/11200154539",\n   "kind": "organisation",\n   "name": "Level",\n   "office_type": "estate_agent",\n   "lat": 55.6888663,\n   "lon": 37.5716636,\n   "opening_hours": "Mo-Su,PH 09:00-21:00"\n  },\n  {\n   "id": "node/11203518587",\n   "kind": "organisation",\n   "name": "Дом.рф",\n   "office_type": "company",\n   "lat": 55.7335811,\n   "lon": 37.5902876,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11214954079",\n   "kind": "organisation",\n   "name": "АНО \\"Центр политических стратегий\\"",\n   "office_type": "political_party",\n   "lat": 55.7183631,\n   "lon": 37.6296769,\n   "opening_hours": "Mo,Fr-Su off; Tu-Th 10:00-15:00"\n  },\n  {\n   "id": "node/11218001204",\n   "kind": "organisation",\n   "name": "Инстройсервис",\n   "office_type": "company",\n   "lat": 55.6938178,\n   "lon": 37.5710356,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11220318405",\n   "kind": "organisation",\n   "name": "Каркасник",\n   "office_type": "company",\n   "lat": 55.7480456,\n   "lon": 37.6519762,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11227994353",\n   "kind": "organisation",\n   "name": "ИТЕМА",\n   "office_type": "travel_agent",\n   "lat": 55.7941118,\n   "lon": 37.6921325,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11228539349",\n   "kind": "organisation",\n   "name": "Cocar",\n   "office_type": "company",\n   "lat": 55.7239445,\n   "lon": 37.6370573,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/11228611354",\n   "kind": "organisation",\n   "name": "Тыл",\n   "office_type": "company",\n   "lat": 55.6379426,\n   "lon": 37.5293939,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/11237663055",\n   "kind": "organisation",\n   "name": "7 региональный отдел надзорной деятельности и профилактической работы Управления по ЦАО Главного управления МЧС России по г. Москве",\n   "office_type": "government",\n   "lat": 55.7673978,\n   "lon": 37.633439,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11238285502",\n   "kind": "organisation",\n   "name": "Визовый центр Греции",\n   "office_type": "company",\n   "lat": 55.7443858,\n   "lon": 37.6305422,\n   "opening_hours": "Mo-Fr 09:00-16:00"\n  },\n  {\n   "id": "node/11238718755",\n   "kind": "organisation",\n   "name": "Vector 100",\n   "office_type": "graphic_design",\n   "lat": 55.8180069,\n   "lon": 37.5967437,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11238718768",\n   "kind": "organisation",\n   "name": "Оберег-Групп",\n   "office_type": "security",\n   "lat": 55.8151618,\n   "lon": 37.6010151,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11242897444",\n   "kind": "organisation",\n   "name": "ФГУП «Охрана» Росгвардии",\n   "office_type": "government",\n   "lat": 55.7747808,\n   "lon": 37.6119921,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11242932977",\n   "kind": "organisation",\n   "name": "Сурцукова Н.В.",\n   "office_type": "notary",\n   "lat": 55.6390608,\n   "lon": 37.5361976,\n   "opening_hours": "Mo-Th 09:00-13:00,14:00-17:00; Fr 09:00-13:00,14:00-16:00; Sa-Su off"\n  },\n  {\n   "id": "node/11242932983",\n   "kind": "organisation",\n   "name": "Газпромбанк Лизинг",\n   "office_type": "company",\n   "lat": 55.6383163,\n   "lon": 37.5396577,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11242932991",\n   "kind": "organisation",\n   "name": "Строймонтаж",\n   "office_type": "company",\n   "lat": 55.6392806,\n   "lon": 37.5347947,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11243625861",\n   "kind": "organisation",\n   "name": "ПрофСанЭксперт",\n   "office_type": "company",\n   "lat": 55.7130122,\n   "lon": 37.7226941,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/11243922238",\n   "kind": "organisation",\n   "name": "Win-Win",\n   "office_type": "translator",\n   "lat": 55.6933697,\n   "lon": 37.5337923,\n   "opening_hours": "Su-Th 10:00-19:00; Fr-Sa 23:00-00:00"\n  },\n  {\n   "id": "node/11243953537",\n   "kind": "organisation",\n   "name": "Win-Win",\n   "office_type": "translator",\n   "lat": 55.7498698,\n   "lon": 37.5373192,\n   "opening_hours": "Su-Th 09:00-19:00; Fr-Sa 23:00-00:00"\n  },\n  {\n   "id": "node/11243963337",\n   "kind": "organisation",\n   "name": "Win-Win",\n   "office_type": "translator",\n   "lat": 55.7663025,\n   "lon": 37.6105781,\n   "opening_hours": "Su-Th 09:00-19:00; Fr-Sa 23:00-00:00"\n  },\n  {\n   "id": "node/11243974137",\n   "kind": "organisation",\n   "name": "Win-Win",\n   "office_type": "translator",\n   "lat": 55.744021,\n   "lon": 37.627186,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11247690843",\n   "kind": "organisation",\n   "name": "Планкин А.А.",\n   "office_type": "notary",\n   "lat": 55.6409306,\n   "lon": 37.5289463,\n   "opening_hours": "Mo,Su off; Tu-Fr 09:30-13:00,14:00-18:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/11247690844",\n   "kind": "organisation",\n   "name": "Стройреконструкция",\n   "office_type": "company",\n   "lat": 55.6409044,\n   "lon": 37.5289173,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/11249007667",\n   "kind": "organisation",\n   "name": "DataCatalog",\n   "office_type": "company",\n   "lat": 55.8085127,\n   "lon": 37.6297907,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11249147344",\n   "kind": "organisation",\n   "name": "Архив внешней политики Российской империи",\n   "office_type": "government",\n   "lat": 55.7271358,\n   "lon": 37.6257572,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11253509906",\n   "kind": "organisation",\n   "name": "Социальный фонд России",\n   "office_type": "government",\n   "lat": 55.8452044,\n   "lon": 37.5561739,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11256097630",\n   "kind": "organisation",\n   "name": "Институт географии РАН",\n   "office_type": "research",\n   "lat": 55.7384579,\n   "lon": 37.6199636,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11256143951",\n   "kind": "organisation",\n   "name": "Институт Цветметобработка",\n   "office_type": "research",\n   "lat": 55.7387343,\n   "lon": 37.6226766,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11259513493",\n   "kind": "organisation",\n   "name": "Московская городская Дума",\n   "office_type": "government",\n   "lat": 55.7686675,\n   "lon": 37.6094601,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11259997939",\n   "kind": "organisation",\n   "name": "Win-Win",\n   "office_type": "translator",\n   "lat": 55.7614469,\n   "lon": 37.6238701,\n   "opening_hours": "Su-Th 09:00-18:00; Fr-Sa 00:00-24:00"\n  },\n  {\n   "id": "node/11260011137",\n   "kind": "organisation",\n   "name": "Win-Win",\n   "office_type": "translator",\n   "lat": 55.7314131,\n   "lon": 37.6352896,\n   "opening_hours": "Su-Th 09:00-18:00; Fr-Sa 00:00-24:00"\n  },\n  {\n   "id": "node/11260015337",\n   "kind": "organisation",\n   "name": "Win-Win",\n   "office_type": "translator",\n   "lat": 55.7730088,\n   "lon": 37.5915895,\n   "opening_hours": "Su-Th 09:00-18:00"\n  },\n  {\n   "id": "node/11260029837",\n   "kind": "organisation",\n   "name": "Win-Win",\n   "office_type": "translator",\n   "lat": 55.7469649,\n   "lon": 37.5989983,\n   "opening_hours": "Su-Th 09:00-18:00"\n  },\n  {\n   "id": "node/11260174838",\n   "kind": "organisation",\n   "name": "Идеи Сруба",\n   "office_type": "company",\n   "lat": 55.7263843,\n   "lon": 37.6813683,\n   "opening_hours": "Mo-Su 09:00-19:00"\n  },\n  {\n   "id": "node/11261229007",\n   "kind": "organisation",\n   "name": "Эй Си Эм",\n   "office_type": "advertising_agency",\n   "lat": 55.8047957,\n   "lon": 37.5659157,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11261229010",\n   "kind": "organisation",\n   "name": "АМДтехнологии",\n   "office_type": "engineer",\n   "lat": 55.8048681,\n   "lon": 37.5660847,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11261229011",\n   "kind": "organisation",\n   "name": "Икар",\n   "office_type": "engineer",\n   "lat": 55.8046495,\n   "lon": 37.5660633,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11261229012",\n   "kind": "organisation",\n   "name": "AT Consulting",\n   "office_type": "it",\n   "lat": 55.8058788,\n   "lon": 37.5672448,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11261229014",\n   "kind": "organisation",\n   "name": "Germes",\n   "office_type": "graphic_design",\n   "lat": 55.8048259,\n   "lon": 37.570159,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11261229015",\n   "kind": "organisation",\n   "name": "Юнистрой-проект",\n   "office_type": "architect",\n   "lat": 55.804945,\n   "lon": 37.5700611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11261229017",\n   "kind": "organisation",\n   "name": "GTU Group",\n   "office_type": "it",\n   "lat": 55.8042455,\n   "lon": 37.5706981,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11261589963",\n   "kind": "organisation",\n   "name": "Валидата",\n   "office_type": "it",\n   "lat": 55.8055427,\n   "lon": 37.5683042,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11261589964",\n   "kind": "organisation",\n   "name": "Enstruct",\n   "office_type": "construction_company",\n   "lat": 55.8054402,\n   "lon": 37.5683847,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11261589965",\n   "kind": "organisation",\n   "name": "AROUND",\n   "office_type": "company",\n   "lat": 55.8053542,\n   "lon": 37.5655523,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11266901722",\n   "kind": "organisation",\n   "name": "Журавлева М. Н.",\n   "office_type": "notary",\n   "lat": 55.8198639,\n   "lon": 37.576587,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11266901723",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8193927,\n   "lon": 37.5775287,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11271731820",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7105012,\n   "lon": 37.6549428,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11271731828",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7231187,\n   "lon": 37.4436067,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 11:00-17:00"\n  },\n  {\n   "id": "node/11271731830",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7743381,\n   "lon": 37.5203045,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11271731834",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7279065,\n   "lon": 37.5602093,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/11271731837",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8691134,\n   "lon": 37.4737065,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11273200536",\n   "kind": "organisation",\n   "name": "Представительство Правительства Тюменской области в органах государственной власти РФ",\n   "office_type": "government",\n   "lat": 55.7317333,\n   "lon": 37.6174678,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11273200537",\n   "kind": "organisation",\n   "name": "Представительство Ямало-Ненецкого автономного округа при Правительстве РФ",\n   "office_type": "government",\n   "lat": 55.7318609,\n   "lon": 37.6181088,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11274072969",\n   "kind": "organisation",\n   "name": "WAPP",\n   "office_type": "company",\n   "lat": 55.7424325,\n   "lon": 37.7075968,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11283108050",\n   "kind": "organisation",\n   "name": "РЭК-5",\n   "office_type": "engineer",\n   "lat": 55.7291627,\n   "lon": 37.6941401,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11291982424",\n   "kind": "organisation",\n   "name": "ФинТех",\n   "office_type": "it",\n   "lat": 55.7344804,\n   "lon": 37.6150856,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11292540557",\n   "kind": "organisation",\n   "name": "Приоритет № 38",\n   "office_type": "lawyer",\n   "lat": 55.6386546,\n   "lon": 37.5368579,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 10:00-13:00; Su off"\n  },\n  {\n   "id": "node/11292540559",\n   "kind": "organisation",\n   "name": "Жилищник района Коньково",\n   "office_type": "government",\n   "lat": 55.6388989,\n   "lon": 37.5366806,\n   "opening_hours": "Mo-Fr 08:00-17:00; Sa-Su off"\n  },\n  {\n   "id": "node/11292540560",\n   "kind": "organisation",\n   "name": "3-е межокружное территориальное управление МАДИ",\n   "office_type": "government",\n   "lat": 55.6389651,\n   "lon": 37.5365442,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11292804669",\n   "kind": "organisation",\n   "name": "Группа \\"Эталон\\"",\n   "office_type": "company",\n   "lat": 55.8044552,\n   "lon": 37.5833387,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11298745831",\n   "kind": "organisation",\n   "name": "Asterisk",\n   "office_type": "lawyer",\n   "lat": 55.7346329,\n   "lon": 37.6170775,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11299514090",\n   "kind": "organisation",\n   "name": "Стиль жизни",\n   "office_type": "company",\n   "lat": 55.771826,\n   "lon": 37.6789473,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/11299519773",\n   "kind": "organisation",\n   "name": "Транспортные технологии",\n   "office_type": "company",\n   "lat": 55.7420969,\n   "lon": 37.5353681,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11304357745",\n   "kind": "organisation",\n   "name": "Graver one",\n   "office_type": "company",\n   "lat": 55.7350452,\n   "lon": 37.6636004,\n   "opening_hours": "Mo-Sa 11:00-19:00"\n  },\n  {\n   "id": "node/11304357751",\n   "kind": "organisation",\n   "name": "Эхо",\n   "office_type": "company",\n   "lat": 55.767973,\n   "lon": 37.629611,\n   "opening_hours": "Mo-We 12:00-06:00, Th-Su 15:00-10:00"\n  },\n  {\n   "id": "node/11304366476",\n   "kind": "organisation",\n   "name": "Команда Х",\n   "office_type": "company",\n   "lat": 55.8343793,\n   "lon": 37.5183695,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11304366479",\n   "kind": "organisation",\n   "name": "Краскова, Майданчук и партнеры",\n   "office_type": "lawyer",\n   "lat": 55.6634836,\n   "lon": 37.5622397,\n   "opening_hours": "Mo-Fr 08:00-19:00"\n  },\n  {\n   "id": "node/11304366481",\n   "kind": "organisation",\n   "name": "LikePrint",\n   "office_type": "advertising_agency",\n   "lat": 55.7777361,\n   "lon": 37.7366504,\n   "opening_hours": "Mo-Fr 10:00-18:30"\n  },\n  {\n   "id": "node/11309988269",\n   "kind": "organisation",\n   "name": "Личные Наличные",\n   "office_type": "financial",\n   "lat": 55.7496506,\n   "lon": 37.5382595,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11329730695",\n   "kind": "organisation",\n   "name": "Частный детектив Lapin Security",\n   "office_type": "lawyer",\n   "lat": 55.7376005,\n   "lon": 37.6360739,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/11332840311",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6880332,\n   "lon": 37.482959,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 11:00-17:00"\n  },\n  {\n   "id": "node/11332840312",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8250922,\n   "lon": 37.595298,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/11332840314",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7406838,\n   "lon": 37.6302515,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11335417354",\n   "kind": "organisation",\n   "name": "Российское Газовое Общество",\n   "office_type": "ngo",\n   "lat": 55.6847382,\n   "lon": 37.5456843,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11335428128",\n   "kind": "organisation",\n   "name": "Еврогаз",\n   "office_type": "company",\n   "lat": 55.6847469,\n   "lon": 37.5456151,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11335483538",\n   "kind": "organisation",\n   "name": "Защита",\n   "office_type": "security",\n   "lat": 55.8328882,\n   "lon": 37.4484109,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11336414498",\n   "kind": "organisation",\n   "name": "Инспекция Федеральной налоговой службы № 15 по г. Москве",\n   "office_type": "government",\n   "lat": 55.812143,\n   "lon": 37.5937262,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11337774030",\n   "kind": "organisation",\n   "name": "Жилищник района Перово",\n   "office_type": "property_management",\n   "lat": 55.7521377,\n   "lon": 37.7888542,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11343283486",\n   "kind": "organisation",\n   "name": "Управление капитального строительства",\n   "office_type": "government",\n   "lat": 55.7608804,\n   "lon": 37.6005162,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11343283494",\n   "kind": "organisation",\n   "name": "Представительство ПАО \\"Татнефть\\" в г. Москве",\n   "office_type": "company",\n   "lat": 55.7615497,\n   "lon": 37.6012847,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11343283640",\n   "kind": "organisation",\n   "name": "Интус",\n   "office_type": "engineering",\n   "lat": 55.7652099,\n   "lon": 37.5989538,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11347799644",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.855254,\n   "lon": 37.4428557,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11350382276",\n   "kind": "organisation",\n   "name": "Центр подбора персонала Транспортного комплекса города Москвы",\n   "office_type": "employment_agency",\n   "lat": 55.8025699,\n   "lon": 37.7445522,\n   "opening_hours": "Mo-Fr 08:00-20:00; Sa 09:00-15:00"\n  },\n  {\n   "id": "node/11357446698",\n   "kind": "organisation",\n   "name": "ОДС-5",\n   "office_type": "engineer",\n   "lat": 55.7962544,\n   "lon": 37.7383322,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11357525665",\n   "kind": "organisation",\n   "name": "Евро Трэвэл 1",\n   "office_type": "property_management",\n   "lat": 55.8417835,\n   "lon": 37.549683,\n   "opening_hours": "Mo-Th 08:00-13:00,14:00-17:00; Fr 08:00-13:00,14:00-15:45"\n  },\n  {\n   "id": "node/11359855443",\n   "kind": "organisation",\n   "name": "Мосгаз Участок #3",\n   "office_type": "yes",\n   "lat": 55.7459846,\n   "lon": 37.5116371,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11360026916",\n   "kind": "organisation",\n   "name": "Московская энергетическая дирекция",\n   "office_type": "yes",\n   "lat": 55.7458067,\n   "lon": 37.5113052,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11360549612",\n   "kind": "organisation",\n   "name": "Чистый Выкуп By Smile025",\n   "office_type": "company",\n   "lat": 55.7354758,\n   "lon": 37.4998163,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11366013040",\n   "kind": "organisation",\n   "name": "Потолочные решения",\n   "office_type": "company",\n   "lat": 55.7549377,\n   "lon": 37.7486479,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11371298867",\n   "kind": "organisation",\n   "name": "Московский филиал «Центратомтехэнерго» АО «Атомтехэнерго»",\n   "office_type": "engineering",\n   "lat": 55.7348292,\n   "lon": 37.6594095,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11372902905",\n   "kind": "organisation",\n   "name": "Гравитон",\n   "office_type": "it",\n   "lat": 55.7422696,\n   "lon": 37.5591523,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11377632988",\n   "kind": "organisation",\n   "name": "Бизнес Центр Рубин",\n   "office_type": "yes",\n   "lat": 55.7416851,\n   "lon": 37.501397,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11378275394",\n   "kind": "organisation",\n   "name": "Офис продаж ЖК «Portland»",\n   "office_type": "estate_agent",\n   "lat": 55.6907732,\n   "lon": 37.7163856,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11379425092",\n   "kind": "organisation",\n   "name": "АО Вэлдан",\n   "office_type": "company",\n   "lat": 55.854182,\n   "lon": 37.5934553,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11380187617",\n   "kind": "organisation",\n   "name": "Российский союз писателей",\n   "office_type": "ngo",\n   "lat": 55.7716081,\n   "lon": 37.6038261,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11385704615",\n   "kind": "organisation",\n   "name": "ПСК Союз тест",\n   "office_type": "company",\n   "lat": 55.7477202,\n   "lon": 37.5865696,\n   "opening_hours": "Mo-Fr 09:00-17:30"\n  },\n  {\n   "id": "node/11387451667",\n   "kind": "organisation",\n   "name": "Hd Artel",\n   "office_type": "advertising_agency",\n   "lat": 55.8655085,\n   "lon": 37.4742716,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11391929979",\n   "kind": "organisation",\n   "name": "Оператор информационной системы",\n   "office_type": "company",\n   "lat": 55.7448319,\n   "lon": 37.5438157,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11395033353",\n   "kind": "organisation",\n   "name": "ГБУ \\"Жилищник Савеловского района\\"",\n   "office_type": "property_management",\n   "lat": 55.8239332,\n   "lon": 37.5618964,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-12:00,12:45-15:45"\n  },\n  {\n   "id": "node/11395564250",\n   "kind": "organisation",\n   "name": "Центральная городская детская библиотека им. А.П. Гайдара. Административно-производственный отдел, отдел комплектования",\n   "office_type": "yes",\n   "lat": 55.7714075,\n   "lon": 37.5763884,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11395564251",\n   "kind": "organisation",\n   "name": "Фонд развития гражданского общества",\n   "office_type": "association",\n   "lat": 55.7717288,\n   "lon": 37.5764152,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11397664250",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6954877,\n   "lon": 37.4984729,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11397664258",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6854748,\n   "lon": 37.5720518,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11410794569",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "accountant",\n   "lat": 55.8160219,\n   "lon": 37.5701965,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11411728636",\n   "kind": "organisation",\n   "name": "СПМ-жилстрой",\n   "office_type": "company",\n   "lat": 55.8045222,\n   "lon": 37.5833259,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11413539471",\n   "kind": "organisation",\n   "name": "МегаФон",\n   "office_type": "telecommunication",\n   "lat": 55.8106552,\n   "lon": 37.7978929,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11423171142",\n   "kind": "organisation",\n   "name": "Нотариус Н. Б. Савченко",\n   "office_type": "lawyer",\n   "lat": 55.7910424,\n   "lon": 37.6006758,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 10:00-17:00"\n  },\n  {\n   "id": "node/11426810304",\n   "kind": "organisation",\n   "name": "Deco Residence",\n   "office_type": "estate_agent",\n   "lat": 55.7109906,\n   "lon": 37.6200872,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/11427615247",\n   "kind": "organisation",\n   "name": "МАКОМНЕТ",\n   "office_type": "telecommunication",\n   "lat": 55.7838812,\n   "lon": 37.6318129,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11432927838",\n   "kind": "organisation",\n   "name": "Жилищник Красносельского района",\n   "office_type": "property_management",\n   "lat": 55.7693193,\n   "lon": 37.6395692,\n   "opening_hours": "Mo-Th 08:00-12:00,12:45-17:00; Fr 08:00-15:45"\n  },\n  {\n   "id": "node/11432961900",\n   "kind": "organisation",\n   "name": "ГЖА.рф",\n   "office_type": "estate_agent",\n   "lat": 55.7693593,\n   "lon": 37.6404342,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11434818084",\n   "kind": "organisation",\n   "name": "Acibadem",\n   "office_type": "company",\n   "lat": 55.735912,\n   "lon": 37.6132072,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11435197764",\n   "kind": "organisation",\n   "name": "КП НТЦ \\"ЭКО\\"",\n   "office_type": "research",\n   "lat": 55.7706637,\n   "lon": 37.6412737,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11435856876",\n   "kind": "organisation",\n   "name": "Центр Арбитража и права на Порываевой",\n   "office_type": "lawyer",\n   "lat": 55.7738628,\n   "lon": 37.6500271,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/11435889113",\n   "kind": "organisation",\n   "name": "Желдорипотека",\n   "office_type": "company",\n   "lat": 55.7721226,\n   "lon": 37.6498742,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11435889114",\n   "kind": "organisation",\n   "name": "Inex",\n   "office_type": "it",\n   "lat": 55.7722327,\n   "lon": 37.649783,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11435889115",\n   "kind": "organisation",\n   "name": "Абк-Инвест",\n   "office_type": "company",\n   "lat": 55.7724334,\n   "lon": 37.6496851,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11445088202",\n   "kind": "organisation",\n   "name": "Жилищник Красносельского района",\n   "office_type": "property_management",\n   "lat": 55.7852046,\n   "lon": 37.6566656,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11449072937",\n   "kind": "organisation",\n   "name": "Фонд по профилактике социально-значимых заболеваний \\"Шаги\\"",\n   "office_type": "charity",\n   "lat": 55.7115101,\n   "lon": 37.7392361,\n   "opening_hours": "Mo-Fr 14:00-19:30"\n  },\n  {\n   "id": "node/11449390683",\n   "kind": "organisation",\n   "name": "Телега добра",\n   "office_type": "charity",\n   "lat": 55.8463998,\n   "lon": 37.4461033,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11450064316",\n   "kind": "organisation",\n   "name": "СМ-знак",\n   "office_type": "company",\n   "lat": 55.7869307,\n   "lon": 37.6625302,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11451555371",\n   "kind": "organisation",\n   "name": "ОДС № 242",\n   "office_type": "property_management",\n   "lat": 55.6408292,\n   "lon": 37.5215914,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/11453975747",\n   "kind": "organisation",\n   "name": "Ветторг",\n   "office_type": "company",\n   "lat": 55.7764527,\n   "lon": 37.6700022,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11456154081",\n   "kind": "organisation",\n   "name": "AdMos",\n   "office_type": "advertising_agency",\n   "lat": 55.777724,\n   "lon": 37.5035048,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11459314144",\n   "kind": "organisation",\n   "name": "ОГРК",\n   "office_type": "estate_agent",\n   "lat": 55.7660933,\n   "lon": 37.6332218,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11459367116",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.8199011,\n   "lon": 37.5764662,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367117",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.8269058,\n   "lon": 37.4445606,\n   "opening_hours": "Mo-Fr 09:30-21:00"\n  },\n  {\n   "id": "node/11459367118",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.7860997,\n   "lon": 37.7199751,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367119",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.831074,\n   "lon": 37.634676,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367120",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.6338543,\n   "lon": 37.4435439,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367121",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.8630124,\n   "lon": 37.5475735,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11459367122",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.7083035,\n   "lon": 37.5070319,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367123",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.753149,\n   "lon": 37.715831,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367124",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.798136,\n   "lon": 37.4970031,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367125",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.8538628,\n   "lon": 37.4760309,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367126",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.6250006,\n   "lon": 37.6758904,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367127",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.7433229,\n   "lon": 37.4998408,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367129",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.7766892,\n   "lon": 37.6756383,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367130",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.6493294,\n   "lon": 37.5404923,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367131",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.6944013,\n   "lon": 37.6657891,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367135",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.7323902,\n   "lon": 37.7320933,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367136",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.7027329,\n   "lon": 37.581541,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11459367137",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.7306897,\n   "lon": 37.6260442,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11459425843",\n   "kind": "organisation",\n   "name": "ARES Logistics",\n   "office_type": "logistics",\n   "lat": 55.676373,\n   "lon": 37.7777009,\n   "opening_hours": "07:00 - 23:00"\n  },\n  {\n   "id": "node/11467550491",\n   "kind": "organisation",\n   "name": "AST",\n   "office_type": "advertising_agency",\n   "lat": 55.770939,\n   "lon": 37.6556799,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11467550492",\n   "kind": "organisation",\n   "name": "РЖД-ТехСервис",\n   "office_type": "company",\n   "lat": 55.770942,\n   "lon": 37.6553057,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11467550493",\n   "kind": "organisation",\n   "name": "Carrot Media",\n   "office_type": "company",\n   "lat": 55.7709458,\n   "lon": 37.6555029,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11470380091",\n   "kind": "organisation",\n   "name": "Гольфстрим",\n   "office_type": "charity",\n   "lat": 55.8012983,\n   "lon": 37.5714478,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11481235073",\n   "kind": "organisation",\n   "name": "РосПрофЖел",\n   "office_type": "union",\n   "lat": 55.7718027,\n   "lon": 37.6653197,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11481235078",\n   "kind": "organisation",\n   "name": "ПГК",\n   "office_type": "company",\n   "lat": 55.7717892,\n   "lon": 37.6655102,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11481235090",\n   "kind": "organisation",\n   "name": "Нотариус Аверина Елена Леонтьевна",\n   "office_type": "notary",\n   "lat": 55.7720924,\n   "lon": 37.669901,\n   "opening_hours": "Mo-Th 10:00-13:00,14:00-18:00; We,Fr 09:00-13:00,14:00-17:00"\n  },\n  {\n   "id": "node/11482403561",\n   "kind": "organisation",\n   "name": "TheOffice",\n   "office_type": "coworking",\n   "lat": 55.6951904,\n   "lon": 37.6256593,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/11482410976",\n   "kind": "organisation",\n   "name": "An.ru",\n   "office_type": "estate_agent",\n   "lat": 55.6636985,\n   "lon": 37.7534491,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/11482411010",\n   "kind": "organisation",\n   "name": "Призма",\n   "office_type": "coworking",\n   "lat": 55.7651819,\n   "lon": 37.6084676,\n   "opening_hours": "Mo-Sa 09:00-21:00"\n  },\n  {\n   "id": "node/11482411012",\n   "kind": "organisation",\n   "name": "Колибри клининг",\n   "office_type": "company",\n   "lat": 55.6443377,\n   "lon": 37.4945033,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11482576222",\n   "kind": "organisation",\n   "name": "ПЭК",\n   "office_type": "logistics",\n   "lat": 55.8822175,\n   "lon": 37.5144428,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/11482609442",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8050881,\n   "lon": 37.517176,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/11482609443",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6368043,\n   "lon": 37.6694992,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11482609444",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.657779,\n   "lon": 37.799077,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11482609464",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6900422,\n   "lon": 37.7259597,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11482682576",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8972944,\n   "lon": 37.5849881,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/11482682578",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8089034,\n   "lon": 37.8125108,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11483224641",\n   "kind": "organisation",\n   "name": "Кушнарёв&Никитин и партнёры",\n   "office_type": "lawyer",\n   "lat": 55.7971146,\n   "lon": 37.5768672,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11495762743",\n   "kind": "organisation",\n   "name": "Качай Права!",\n   "office_type": "lawyer",\n   "lat": 55.8088023,\n   "lon": 37.564816,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa,Su 11:00-21:00"\n  },\n  {\n   "id": "node/11498105570",\n   "kind": "organisation",\n   "name": "Мегапарк",\n   "office_type": "company",\n   "lat": 55.8146181,\n   "lon": 37.5634266,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/11498105573",\n   "kind": "organisation",\n   "name": "Игрология",\n   "office_type": "company",\n   "lat": 55.8143917,\n   "lon": 37.5625965,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11504820232",\n   "kind": "organisation",\n   "name": "Правовой центр помощи призывникам Султана Мирзаева",\n   "office_type": "lawyer",\n   "lat": 55.7873485,\n   "lon": 37.6119746,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11505302840",\n   "kind": "organisation",\n   "name": "Местное отделение политической партии \\"Единая Россия\\"",\n   "office_type": "political_party",\n   "lat": 55.7935335,\n   "lon": 37.5126511,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11505370011",\n   "kind": "organisation",\n   "name": "Дубровская, Кузнецова и партнеры",\n   "office_type": "lawyer",\n   "lat": 55.7862626,\n   "lon": 37.613014,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11505370023",\n   "kind": "organisation",\n   "name": "Квайссер Фарма",\n   "office_type": "company",\n   "lat": 55.7866223,\n   "lon": 37.61176,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11506329325",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "government",\n   "lat": 55.7990527,\n   "lon": 37.5002861,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11510811673",\n   "kind": "organisation",\n   "name": "М. Тех",\n   "office_type": "it",\n   "lat": 55.7735693,\n   "lon": 37.6693171,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/11512192672",\n   "kind": "organisation",\n   "name": "Фирн М",\n   "office_type": "company",\n   "lat": 55.7857876,\n   "lon": 37.6141123,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11515114859",\n   "kind": "organisation",\n   "name": "Novard",\n   "office_type": "company",\n   "lat": 55.7860613,\n   "lon": 37.6093474,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11515114862",\n   "kind": "organisation",\n   "name": "Цифровая логистика",\n   "office_type": "logistics",\n   "lat": 55.7862532,\n   "lon": 37.6087915,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11517878855",\n   "kind": "organisation",\n   "name": "Рассчётный центр",\n   "office_type": "government",\n   "lat": 55.8299882,\n   "lon": 37.5746423,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11520396946",\n   "kind": "organisation",\n   "name": "Audio Art",\n   "office_type": "advertising_agency",\n   "lat": 55.7862656,\n   "lon": 37.6326022,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11520396947",\n   "kind": "organisation",\n   "name": "Каскад-Холдинг",\n   "office_type": "company",\n   "lat": 55.78482,\n   "lon": 37.6296866,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11520396949",\n   "kind": "organisation",\n   "name": "Культурное представительство при посольстве Ирана",\n   "office_type": "diplomatic",\n   "lat": 55.786142,\n   "lon": 37.6318766,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11520396950",\n   "kind": "organisation",\n   "name": "Адвокатское бюро \\"Марусенко и партнёры\\"",\n   "office_type": "lawyer",\n   "lat": 55.7866434,\n   "lon": 37.6320121,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11520396951",\n   "kind": "organisation",\n   "name": "Проектно-конструкторское бюро московского метрополитена",\n   "office_type": "company",\n   "lat": 55.7861827,\n   "lon": 37.6314931,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11520396953",\n   "kind": "organisation",\n   "name": "Трансинфосеть",\n   "office_type": "it",\n   "lat": 55.7860666,\n   "lon": 37.6306897,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11526115264",\n   "kind": "organisation",\n   "name": "Сити-Жалюзи",\n   "office_type": "company",\n   "lat": 55.6486279,\n   "lon": 37.7360322,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11526519007",\n   "kind": "organisation",\n   "name": "Нотариус Фельде Марина Геннадиевна",\n   "office_type": "notary",\n   "lat": 55.7859142,\n   "lon": 37.6344851,\n   "opening_hours": "Mo-Fr 10:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/11529276502",\n   "kind": "organisation",\n   "name": "АльфаСтрахование",\n   "office_type": "insurance",\n   "lat": 55.7868719,\n   "lon": 37.637127,\n   "opening_hours": "Mo-Fr 09:00-18:30; Sa 10:00-16:30"\n  },\n  {\n   "id": "node/11530572884",\n   "kind": "organisation",\n   "name": "ФГУП «НТЦ «Заря»",\n   "office_type": "research",\n   "lat": 55.7698479,\n   "lon": 37.5907344,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11530681511",\n   "kind": "organisation",\n   "name": "Маяк",\n   "office_type": "yes",\n   "lat": 55.7700948,\n   "lon": 37.5901927,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11530721773",\n   "kind": "organisation",\n   "name": "Гравиметрическая лаборатория ППК \\"Роскадастр\\"",\n   "office_type": "government",\n   "lat": 55.7809928,\n   "lon": 37.6388705,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11532513149",\n   "kind": "organisation",\n   "name": "finoarte",\n   "office_type": "company",\n   "lat": 55.7863139,\n   "lon": 37.6389764,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/11532513150",\n   "kind": "organisation",\n   "name": "rutube",\n   "office_type": "telecommunication",\n   "lat": 55.7861442,\n   "lon": 37.6389308,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11533324185",\n   "kind": "organisation",\n   "name": "ОДС № 1158",\n   "office_type": "property_management",\n   "lat": 55.6342575,\n   "lon": 37.5224888,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11545493668",\n   "kind": "organisation",\n   "name": "Федеральная служба по надзору в сфере природопользования",\n   "office_type": "government",\n   "lat": 55.7511565,\n   "lon": 37.5892496,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11545531790",\n   "kind": "organisation",\n   "name": "Департамент торговли и услуг г. Москвы",\n   "office_type": "government",\n   "lat": 55.7500002,\n   "lon": 37.5345647,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11545588138",\n   "kind": "organisation",\n   "name": "ОДС №552",\n   "office_type": "property_maintenance",\n   "lat": 55.8236462,\n   "lon": 37.5762838,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11545721369",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6351185,\n   "lon": 37.7362669,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/11545721377",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7620703,\n   "lon": 37.6777276,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/11545721382",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6260304,\n   "lon": 37.6121287,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11545721394",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7516018,\n   "lon": 37.8200055,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11545721397",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6561862,\n   "lon": 37.6081833,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-15:00"\n  },\n  {\n   "id": "node/11545745874",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6723224,\n   "lon": 37.6113821,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11553381321",\n   "kind": "organisation",\n   "name": "Онлайн Лоджик",\n   "office_type": "it",\n   "lat": 55.6923877,\n   "lon": 37.5288816,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11553628299",\n   "kind": "organisation",\n   "name": "Las Vegas",\n   "office_type": "estate_agent",\n   "lat": 55.7322116,\n   "lon": 37.7321723,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/11559312277",\n   "kind": "organisation",\n   "name": "Управление ФСБ России по г. Москве и Московской области",\n   "office_type": "government",\n   "lat": 55.7642313,\n   "lon": 37.6300541,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11561513203",\n   "kind": "organisation",\n   "name": "Соф-Транс",\n   "office_type": "construction_company",\n   "lat": 55.792803,\n   "lon": 37.5715175,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11573889202",\n   "kind": "organisation",\n   "name": "Eco Village",\n   "office_type": "company",\n   "lat": 55.7878167,\n   "lon": 37.6406327,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11573889220",\n   "kind": "organisation",\n   "name": "Злато-Окна",\n   "office_type": "company",\n   "lat": 55.7886085,\n   "lon": 37.6414588,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11573889221",\n   "kind": "organisation",\n   "name": "OZERO",\n   "office_type": "architect",\n   "lat": 55.7886032,\n   "lon": 37.6411289,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11573889222",\n   "kind": "organisation",\n   "name": "Sibcoffee",\n   "office_type": "company",\n   "lat": 55.7886198,\n   "lon": 37.6412804,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11577681538",\n   "kind": "organisation",\n   "name": "Atlas Language Service",\n   "office_type": "company",\n   "lat": 55.7874676,\n   "lon": 37.6358315,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11577681539",\n   "kind": "organisation",\n   "name": "Нотариус Маскинская Татьяна Всеволодовна",\n   "office_type": "notary",\n   "lat": 55.7874299,\n   "lon": 37.6357068,\n   "opening_hours": "Mo-Fr 10:30-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/11577681543",\n   "kind": "organisation",\n   "name": "ФинессКо",\n   "office_type": "estate_agent",\n   "lat": 55.7873907,\n   "lon": 37.6356009,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11577681544",\n   "kind": "organisation",\n   "name": "Лингво Сервис",\n   "office_type": "company",\n   "lat": 55.7874103,\n   "lon": 37.6358275,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/11577719406",\n   "kind": "organisation",\n   "name": "Человеческие ресурсы",\n   "office_type": "it",\n   "lat": 55.7109043,\n   "lon": 37.6562872,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11593054742",\n   "kind": "organisation",\n   "name": "Моя работа",\n   "office_type": "employment_agency",\n   "lat": 55.7845388,\n   "lon": 37.6369567,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/11596428029",\n   "kind": "organisation",\n   "name": "Вологодский текстильный комбинат",\n   "office_type": "company",\n   "lat": 55.7814513,\n   "lon": 37.6392339,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11596428055",\n   "kind": "organisation",\n   "name": "Дзержинское межрайонное общество охотников и рыболовов",\n   "office_type": "quango",\n   "lat": 55.7828329,\n   "lon": 37.6386733,\n   "opening_hours": "Mo-Th 10:00-14:00,15:00-18:30"\n  },\n  {\n   "id": "node/11596428059",\n   "kind": "organisation",\n   "name": "Центр правовой помощи мигрантам В. В. Коженова",\n   "office_type": "lawyer",\n   "lat": 55.782808,\n   "lon": 37.6388209,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11596428060",\n   "kind": "organisation",\n   "name": "Ремстройсервис",\n   "office_type": "construction_company",\n   "lat": 55.7828186,\n   "lon": 37.638735,\n   "opening_hours": "Mo-Fr 09:00-21:00"\n  },\n  {\n   "id": "node/11598676504",\n   "kind": "organisation",\n   "name": "Центральная дирекция пассажирских обустройств РЖД",\n   "office_type": "company",\n   "lat": 55.7812933,\n   "lon": 37.6387384,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11598676518",\n   "kind": "organisation",\n   "name": "Удостоверяющий центр \\"Основание\\"",\n   "office_type": "it",\n   "lat": 55.7817409,\n   "lon": 37.6394016,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11602527685",\n   "kind": "organisation",\n   "name": "Правительственный вокзал",\n   "office_type": "railway",\n   "lat": 55.7820471,\n   "lon": 37.645517,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11608159138",\n   "kind": "organisation",\n   "name": "Рикши Рекламы",\n   "office_type": "advertising_agency",\n   "lat": 55.7590912,\n   "lon": 37.7325869,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11608159141",\n   "kind": "organisation",\n   "name": "Кош-Ка",\n   "office_type": "company",\n   "lat": 55.7650869,\n   "lon": 37.7270588,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11608853178",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.8632066,\n   "lon": 37.6823799,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11618355901",\n   "kind": "organisation",\n   "name": "Русзолото",\n   "office_type": "yes",\n   "lat": 55.7362717,\n   "lon": 37.5009358,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11622123925",\n   "kind": "organisation",\n   "name": "Amaks",\n   "office_type": "company",\n   "lat": 55.6888317,\n   "lon": 37.6245904,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11623725780",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "accountant",\n   "lat": 55.8296199,\n   "lon": 37.5787407,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11623725791",\n   "kind": "organisation",\n   "name": "Управление по развитию массового спорта СВАО г. Москвы",\n   "office_type": "government",\n   "lat": 55.8285902,\n   "lon": 37.5803567,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11627138797",\n   "kind": "organisation",\n   "name": "Адвокат",\n   "office_type": "lawyer",\n   "lat": 55.7506851,\n   "lon": 37.5172954,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11627702249",\n   "kind": "organisation",\n   "name": "Валерий Летенков",\n   "office_type": "estate_agent",\n   "lat": 55.8462681,\n   "lon": 37.585469,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11627702250",\n   "kind": "organisation",\n   "name": "Клаксон",\n   "office_type": "newspaper",\n   "lat": 55.8463276,\n   "lon": 37.5854114,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11627702256",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.7999694,\n   "lon": 37.5729632,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11627702257",\n   "kind": "organisation",\n   "name": "Единая Россия",\n   "office_type": "political_party",\n   "lat": 55.8122259,\n   "lon": 37.5694844,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11629962591",\n   "kind": "organisation",\n   "name": "Nakleyka",\n   "office_type": "advertising_agency",\n   "lat": 55.6896937,\n   "lon": 37.5917844,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11634256714",\n   "kind": "organisation",\n   "name": "Мир Новостей",\n   "office_type": "yes",\n   "lat": 55.7550653,\n   "lon": 37.6440051,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11634256715",\n   "kind": "organisation",\n   "name": "ФинМониторинг - ЦФО",\n   "office_type": "government",\n   "lat": 55.7550641,\n   "lon": 37.6439255,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11646021575",\n   "kind": "organisation",\n   "name": "19g Atelier",\n   "office_type": "company",\n   "lat": 55.7566999,\n   "lon": 37.6754722,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11648769074",\n   "kind": "organisation",\n   "name": "РИНФИН",\n   "office_type": "lawyer",\n   "lat": 55.7722549,\n   "lon": 37.6476213,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11652654047",\n   "kind": "organisation",\n   "name": "УФНС 30",\n   "office_type": "government",\n   "lat": 55.739782,\n   "lon": 37.4735355,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11654578339",\n   "kind": "organisation",\n   "name": "Администрация парка \\"Фили\\"",\n   "office_type": "yes",\n   "lat": 55.7458055,\n   "lon": 37.4873524,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11658339724",\n   "kind": "organisation",\n   "name": "Ботаник",\n   "office_type": "estate_agent",\n   "lat": 55.8488671,\n   "lon": 37.5874284,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11659696074",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.8493618,\n   "lon": 37.5888567,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/11659742386",\n   "kind": "organisation",\n   "name": "Russian Rhythmic Gymnastics Federation",\n   "office_type": "government",\n   "lat": 55.7142583,\n   "lon": 37.5671672,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11667211075",\n   "kind": "organisation",\n   "name": "Поместье",\n   "office_type": "construction_company",\n   "lat": 55.843035,\n   "lon": 37.5796741,\n   "opening_hours": "Mo-Fr 10:30-19:00"\n  },\n  {\n   "id": "node/11669106230",\n   "kind": "organisation",\n   "name": "Автономика",\n   "office_type": "company",\n   "lat": 55.7566853,\n   "lon": 37.587192,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11670452379",\n   "kind": "organisation",\n   "name": "Славия",\n   "office_type": "telecommunication",\n   "lat": 55.8439943,\n   "lon": 37.5810018,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11670452381",\n   "kind": "organisation",\n   "name": "Экотел",\n   "office_type": "telecommunication",\n   "lat": 55.8436554,\n   "lon": 37.5809696,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11673238665",\n   "kind": "organisation",\n   "name": "Zenapart",\n   "office_type": "company",\n   "lat": 55.8457471,\n   "lon": 37.5824368,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11673307423",\n   "kind": "organisation",\n   "name": "Сонсу до",\n   "office_type": "company",\n   "lat": 55.7987472,\n   "lon": 37.5244441,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/11674203158",\n   "kind": "organisation",\n   "name": "Алые паруса",\n   "office_type": "lawyer",\n   "lat": 55.8432293,\n   "lon": 37.5775753,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11676400765",\n   "kind": "organisation",\n   "name": "Дирекция парка",\n   "office_type": "yes",\n   "lat": 55.7865175,\n   "lon": 37.5307775,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11676404431",\n   "kind": "organisation",\n   "name": "Российская Премьер-Лига",\n   "office_type": "ngo",\n   "lat": 55.7364435,\n   "lon": 37.6599821,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11678180870",\n   "kind": "organisation",\n   "name": "Yulisbook",\n   "office_type": "publisher",\n   "lat": 55.8435342,\n   "lon": 37.5771327,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11678180871",\n   "kind": "organisation",\n   "name": "Consul Print",\n   "office_type": "advertising_agency",\n   "lat": 55.8431961,\n   "lon": 37.5776021,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11678180872",\n   "kind": "organisation",\n   "name": "Крайс",\n   "office_type": "company",\n   "lat": 55.843154,\n   "lon": 37.5776343,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11678603302",\n   "kind": "organisation",\n   "name": "Автоправо24",\n   "office_type": "lawyer",\n   "lat": 55.8473252,\n   "lon": 37.5760129,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/11682710291",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.8467734,\n   "lon": 37.5703816,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11691624908",\n   "kind": "organisation",\n   "name": "Общество охотников и рыболовов административных и правоохранительных органов",\n   "office_type": "association",\n   "lat": 55.7956377,\n   "lon": 37.7055824,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11695271482",\n   "kind": "organisation",\n   "name": "Expert Decora",\n   "office_type": "company",\n   "lat": 55.8443331,\n   "lon": 37.5698787,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11696599907",\n   "kind": "organisation",\n   "name": "Да Винчи",\n   "office_type": "architect",\n   "lat": 55.8112508,\n   "lon": 37.5148934,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/11696599910",\n   "kind": "organisation",\n   "name": "Сделай сделку",\n   "office_type": "lawyer",\n   "lat": 55.7468195,\n   "lon": 37.5393901,\n   "opening_hours": "Mo-Fr 09:00-21:00; Sa 09:00-18:00"\n  },\n  {\n   "id": "node/11696599928",\n   "kind": "organisation",\n   "name": "Миллениум",\n   "office_type": "construction_company",\n   "lat": 55.7962363,\n   "lon": 37.7006793,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11696599930",\n   "kind": "organisation",\n   "name": "Литерра",\n   "office_type": "translator",\n   "lat": 55.7309087,\n   "lon": 37.6413999,\n   "opening_hours": "Mo-Su 09:00-19:00"\n  },\n  {\n   "id": "node/11696599964",\n   "kind": "organisation",\n   "name": "brainbox.Vc",\n   "office_type": "company",\n   "lat": 55.7494085,\n   "lon": 37.5375018,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11696599965",\n   "kind": "organisation",\n   "name": "MGK",\n   "office_type": "lawyer",\n   "lat": 55.6141015,\n   "lon": 37.6260227,\n   "opening_hours": "Mo-Sa 09:00-19:00"\n  },\n  {\n   "id": "node/11696646869",\n   "kind": "organisation",\n   "name": "Дом полимеров",\n   "office_type": "company",\n   "lat": 55.8304552,\n   "lon": 37.6286163,\n   "opening_hours": "Fr-Sa 12:00-22:00; Tu-Th,Su 12:00-21:00"\n  },\n  {\n   "id": "node/11696776476",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6890458,\n   "lon": 37.529483,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11696776477",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.621444,\n   "lon": 37.6113542,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11696776486",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6223,\n   "lon": 37.6660788,\n   "opening_hours": "Mo-Fr 09:00-20:00"\n  },\n  {\n   "id": "node/11700585496",\n   "kind": "organisation",\n   "name": "iKey",\n   "office_type": "company",\n   "lat": 55.7044926,\n   "lon": 37.500832,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11708896726",\n   "kind": "organisation",\n   "name": "Московская печатная фабрика (Гознак)",\n   "office_type": "government",\n   "lat": 55.7203432,\n   "lon": 37.6193535,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11709472066",\n   "kind": "organisation",\n   "name": "3data",\n   "office_type": "it",\n   "lat": 55.8445439,\n   "lon": 37.5588548,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11715869683",\n   "kind": "organisation",\n   "name": "Кредитный оздоровитель",\n   "office_type": "financial",\n   "lat": 55.7097555,\n   "lon": 37.6185388,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11715869684",\n   "kind": "organisation",\n   "name": "Бюро",\n   "office_type": "estate_agent",\n   "lat": 55.7097691,\n   "lon": 37.6185385,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11715869685",\n   "kind": "organisation",\n   "name": "Правовой центр собственников недвижимости",\n   "office_type": "lawyer",\n   "lat": 55.7097853,\n   "lon": 37.6185382,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11717433851",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.6101923,\n   "lon": 37.7292475,\n   "opening_hours": "Mo-Fr 09:30-18:00; Sa 10:00-18:00; Su 10:00-16:00"\n  },\n  {\n   "id": "node/11725139313",\n   "kind": "organisation",\n   "name": "ООО \\"Лабсиз\\"",\n   "office_type": "company",\n   "lat": 55.717652,\n   "lon": 37.5116761,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11726854953",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6118645,\n   "lon": 37.5372523,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11726854959",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6193296,\n   "lon": 37.7518499,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11731518432",\n   "kind": "organisation",\n   "name": "ОДС 3040",\n   "office_type": "government",\n   "lat": 55.7360387,\n   "lon": 37.4922611,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11736391876",\n   "kind": "organisation",\n   "name": "IG production",\n   "office_type": "company",\n   "lat": 55.7093332,\n   "lon": 37.6179829,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11737235288",\n   "kind": "organisation",\n   "name": "Марсель",\n   "office_type": "yes",\n   "lat": 55.7967128,\n   "lon": 37.6900021,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11749637211",\n   "kind": "organisation",\n   "name": "Ēniteō",\n   "office_type": "estate_agent",\n   "lat": 55.6809575,\n   "lon": 37.5947532,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11751804639",\n   "kind": "organisation",\n   "name": "Бизнес Телеком Коммерц",\n   "office_type": "telecommunication",\n   "lat": 55.8370664,\n   "lon": 37.6339245,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11751804640",\n   "kind": "organisation",\n   "name": "Курсив",\n   "office_type": "publisher",\n   "lat": 55.8370732,\n   "lon": 37.6340666,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11754814361",\n   "kind": "organisation",\n   "name": "GetCourse",\n   "office_type": "it",\n   "lat": 55.7601333,\n   "lon": 37.561988,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11755821785",\n   "kind": "organisation",\n   "name": "Point Design",\n   "office_type": "architect",\n   "lat": 55.7498761,\n   "lon": 37.5710647,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11761035184",\n   "kind": "organisation",\n   "name": "Приёмная ФТС",\n   "office_type": "yes",\n   "lat": 55.7504673,\n   "lon": 37.508522,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11768746421",\n   "kind": "organisation",\n   "name": "Альфа-страхование",\n   "office_type": "insurance",\n   "lat": 55.8746032,\n   "lon": 37.52426,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11780110802",\n   "kind": "organisation",\n   "name": "Ваша кадровая служба",\n   "office_type": "company",\n   "lat": 55.7960453,\n   "lon": 37.7072274,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11780110818",\n   "kind": "organisation",\n   "name": "ООО \\"Космические дали\\"",\n   "office_type": "company",\n   "lat": 55.796252,\n   "lon": 37.7080159,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11781399788",\n   "kind": "organisation",\n   "name": "Time saving machine",\n   "office_type": "company",\n   "lat": 55.7113132,\n   "lon": 37.6115057,\n   "opening_hours": "Mo-Fr 09:00-19:00; Sa-Su off"\n  },\n  {\n   "id": "node/11782593625",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8534854,\n   "lon": 37.6068808,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11782606850",\n   "kind": "organisation",\n   "name": "Поинтер",\n   "office_type": "it",\n   "lat": 55.7974944,\n   "lon": 37.4971737,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11782622913",\n   "kind": "organisation",\n   "name": "ТрансКонтейнер",\n   "office_type": "logistics",\n   "lat": 55.7842177,\n   "lon": 37.6736024,\n   "opening_hours": "Mo-Su 08:00-20:00"\n  },\n  {\n   "id": "node/11784420728",\n   "kind": "organisation",\n   "name": "Detrox",\n   "office_type": "company",\n   "lat": 55.8348658,\n   "lon": 37.64247,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11784697014",\n   "kind": "organisation",\n   "name": "3DProScan",\n   "office_type": "geodesist",\n   "lat": 55.8327577,\n   "lon": 37.6511376,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11801706033",\n   "kind": "organisation",\n   "name": "Лаборатория нанокомпозитов на основе промышленных полимеров",\n   "office_type": "research",\n   "lat": 55.7527656,\n   "lon": 37.6524398,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11804699391",\n   "kind": "organisation",\n   "name": "Центр противопожарной пропаганды и общественных связей \u200bФедеральной противопожарной службы",\n   "office_type": "government",\n   "lat": 55.7784601,\n   "lon": 37.6293942,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11804979174",\n   "kind": "organisation",\n   "name": "Консалт Про",\n   "office_type": "consulting",\n   "lat": 55.7686788,\n   "lon": 37.6352414,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11806819644",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.5973848,\n   "lon": 37.5276854,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11806819649",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8554338,\n   "lon": 37.4754472,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 11:00-16:00"\n  },\n  {\n   "id": "node/11806819657",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7500899,\n   "lon": 37.7872256,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11806819660",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6532533,\n   "lon": 37.6465711,\n   "opening_hours": "Mo-Sa 10:00-22:00"\n  },\n  {\n   "id": "node/11806886775",\n   "kind": "organisation",\n   "name": "Open Service",\n   "office_type": "company",\n   "lat": 55.6319152,\n   "lon": 37.6179096,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11806910903",\n   "kind": "organisation",\n   "name": "Eft Group",\n   "office_type": "company",\n   "lat": 55.8044142,\n   "lon": 37.5904384,\n   "opening_hours": "Mo-Fr 09:30-18:00"\n  },\n  {\n   "id": "node/11807108493",\n   "kind": "organisation",\n   "name": "Art Area",\n   "office_type": "yes",\n   "lat": 55.7391092,\n   "lon": 37.5172345,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11807108496",\n   "kind": "organisation",\n   "name": "Enigma Design Studio",\n   "office_type": "yes",\n   "lat": 55.7391128,\n   "lon": 37.5171596,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11807108498",\n   "kind": "organisation",\n   "name": "MSF studio",\n   "office_type": "yes",\n   "lat": 55.7390958,\n   "lon": 37.5171468,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11809705881",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7337426,\n   "lon": 37.620213,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11809705899",\n   "kind": "organisation",\n   "name": "Московский авиационный центр",\n   "office_type": "government",\n   "lat": 55.7338265,\n   "lon": 37.6196927,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11813596866",\n   "kind": "organisation",\n   "name": "Российский союз строителей",\n   "office_type": "ngo",\n   "lat": 55.729023,\n   "lon": 37.618766,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11824276466",\n   "kind": "organisation",\n   "name": "Нетрис",\n   "office_type": "it",\n   "lat": 55.7318058,\n   "lon": 37.6271036,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11824276467",\n   "kind": "organisation",\n   "name": "АНТОР",\n   "office_type": "it",\n   "lat": 55.7056271,\n   "lon": 37.6118579,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11831260468",\n   "kind": "organisation",\n   "name": "Bentonit",\n   "office_type": "yes",\n   "lat": 55.7634674,\n   "lon": 37.6076429,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11831405881",\n   "kind": "organisation",\n   "name": "ПАО \\"Транснефть\\"",\n   "office_type": "company",\n   "lat": 55.7484348,\n   "lon": 37.5419971,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11831405882",\n   "kind": "organisation",\n   "name": "ПАО \\"Транснефть\\"",\n   "office_type": "company",\n   "lat": 55.73133,\n   "lon": 37.6219082,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11843758293",\n   "kind": "organisation",\n   "name": "Транзит",\n   "office_type": "logistics",\n   "lat": 55.8912424,\n   "lon": 37.755509,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/11843797406",\n   "kind": "organisation",\n   "name": "Новые технологии",\n   "office_type": "architect",\n   "lat": 55.6502404,\n   "lon": 37.5412648,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11843797424",\n   "kind": "organisation",\n   "name": "Спецсервисгаз",\n   "office_type": "company",\n   "lat": 55.6712078,\n   "lon": 37.5856179,\n   "opening_hours": "Mo-Sa 10:00-20:00; Su 10:00-19:00"\n  },\n  {\n   "id": "node/11843797430",\n   "kind": "organisation",\n   "name": "Лотос",\n   "office_type": "company",\n   "lat": 55.6927565,\n   "lon": 37.6597167,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11843797431",\n   "kind": "organisation",\n   "name": "Транзит",\n   "office_type": "logistics",\n   "lat": 55.7918484,\n   "lon": 37.6031703,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11843797444",\n   "kind": "organisation",\n   "name": "Синергия",\n   "office_type": "company",\n   "lat": 55.670751,\n   "lon": 37.6342248,\n   "opening_hours": "Mo-Su 08:00-22:00"\n  },\n  {\n   "id": "node/11844583170",\n   "kind": "organisation",\n   "name": "Моспароходство",\n   "office_type": "company",\n   "lat": 55.7441771,\n   "lon": 37.5417382,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11868431743",\n   "kind": "organisation",\n   "name": "Terradeck",\n   "office_type": "construction_company",\n   "lat": 55.8042575,\n   "lon": 37.6190952,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/11873547830",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7041224,\n   "lon": 37.6255184,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11873547836",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.833722,\n   "lon": 37.5232378,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/11873547840",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8558181,\n   "lon": 37.5588146,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11873547844",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6714269,\n   "lon": 37.5534689,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11873547846",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8501481,\n   "lon": 37.4786493,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11873547860",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.69905,\n   "lon": 37.5634146,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11873547862",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.886513,\n   "lon": 37.44824,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/11873547867",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.60656,\n   "lon": 37.6634953,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/11873574776",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.601113,\n   "lon": 37.6090776,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11873574780",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8007934,\n   "lon": 37.5936291,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11873574783",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7806596,\n   "lon": 37.6024069,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11873574807",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6642201,\n   "lon": 37.5984067,\n   "opening_hours": "Mo-Fr 09:00-22:00"\n  },\n  {\n   "id": "node/11873574818",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6823903,\n   "lon": 37.6939673,\n   "opening_hours": "Fr 10:00-19:00; Mo-Th 10:00-20:00; Sa 11:00-17:00"\n  },\n  {\n   "id": "node/11873574824",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6766159,\n   "lon": 37.7685392,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 11:00-15:00"\n  },\n  {\n   "id": "node/11873574827",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6828467,\n   "lon": 37.5496446,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/11873574842",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8848671,\n   "lon": 37.6039991,\n   "opening_hours": "Mo-Fr 09:00-21:00"\n  },\n  {\n   "id": "node/11873574843",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7914216,\n   "lon": 37.5842789,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/11873574848",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6331879,\n   "lon": 37.5215845,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11873574857",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7014931,\n   "lon": 37.5039361,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa-Su 10:00-15:00"\n  },\n  {\n   "id": "node/11873574861",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7772116,\n   "lon": 37.4804329,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11873574884",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7695185,\n   "lon": 37.6282666,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11873574898",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6559997,\n   "lon": 37.6154441,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/11873574901",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.655619,\n   "lon": 37.4930946,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/11873574902",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7092604,\n   "lon": 37.7646366,\n   "opening_hours": "Mo-Fr 11:00-20:00; Sa 12:00-17:00"\n  },\n  {\n   "id": "node/11876207410",\n   "kind": "organisation",\n   "name": "Александр Лебедев и Партнёры",\n   "office_type": "lawyer",\n   "lat": 55.8119587,\n   "lon": 37.4855962,\n   "opening_hours": "Mo-Fr 10:00-20:00"\n  },\n  {\n   "id": "node/11877112365",\n   "kind": "organisation",\n   "name": "Bauten",\n   "office_type": "company",\n   "lat": 55.6297536,\n   "lon": 37.4686084,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/11881212849",\n   "kind": "organisation",\n   "name": "Rhyme",\n   "office_type": "architect",\n   "lat": 55.7477574,\n   "lon": 37.5966759,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11889150084",\n   "kind": "organisation",\n   "name": "BBBends.com",\n   "office_type": "company",\n   "lat": 55.8350089,\n   "lon": 37.6426256,\n   "opening_hours": "Mo-Fr 12:00-17:00"\n  },\n  {\n   "id": "node/11905837216",\n   "kind": "organisation",\n   "name": "IGA Systems",\n   "office_type": "it",\n   "lat": 55.7526733,\n   "lon": 37.7021955,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/11906771490",\n   "kind": "organisation",\n   "name": "Autocapital",\n   "office_type": "company",\n   "lat": 55.6177331,\n   "lon": 37.7152826,\n   "opening_hours": "Mo-Su 10:00-19:00"\n  },\n  {\n   "id": "node/11909522482",\n   "kind": "organisation",\n   "name": "Globecast Russia",\n   "office_type": "company",\n   "lat": 55.7479909,\n   "lon": 37.5399077,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11910307101",\n   "kind": "organisation",\n   "name": "штаб УД РАН",\n   "office_type": "yes",\n   "lat": 55.718664,\n   "lon": 37.5924017,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11910307137",\n   "kind": "organisation",\n   "name": "технический надзор УД РАН",\n   "office_type": "yes",\n   "lat": 55.7186418,\n   "lon": 37.5924553,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11918301014",\n   "kind": "organisation",\n   "name": "Росатом",\n   "office_type": "company",\n   "lat": 55.7519188,\n   "lon": 37.6355553,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11925664843",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8959693,\n   "lon": 37.7093997,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa-Su 11:00-16:00"\n  },\n  {\n   "id": "node/11925664868",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7932798,\n   "lon": 37.6099337,\n   "opening_hours": "Mo-Fr 09:00-19:00"\n  },\n  {\n   "id": "node/11925687180",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6649227,\n   "lon": 37.7543203,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11931688185",\n   "kind": "organisation",\n   "name": "РЕСО-Гарантия",\n   "office_type": "insurance",\n   "lat": 55.8894645,\n   "lon": 37.636081,\n   "opening_hours": "Mo-Th 09:30-18:00"\n  },\n  {\n   "id": "node/11934047389",\n   "kind": "organisation",\n   "name": "MoskitCo",\n   "office_type": "company",\n   "lat": 55.8908363,\n   "lon": 37.6337984,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11940570117",\n   "kind": "organisation",\n   "name": "Зетта Страхование",\n   "office_type": "insurance",\n   "lat": 55.7379566,\n   "lon": 37.6380712,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11942707342",\n   "kind": "organisation",\n   "name": "Алтуфьево Коворкинг",\n   "office_type": "coworking",\n   "lat": 55.8960462,\n   "lon": 37.5856769,\n   "opening_hours": "Tu-Sa 12:00-22:00; Su 12:00-20:00"\n  },\n  {\n   "id": "node/11942926526",\n   "kind": "organisation",\n   "name": "Hub",\n   "office_type": "coworking",\n   "lat": 55.8863794,\n   "lon": 37.5853685,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11948211150",\n   "kind": "organisation",\n   "name": "Зона-сварки",\n   "office_type": "company",\n   "lat": 55.6858573,\n   "lon": 37.5449192,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11958118153",\n   "kind": "organisation",\n   "name": "Адвокат Лычагин А.А.",\n   "office_type": "lawyer",\n   "lat": 55.7802748,\n   "lon": 37.6326262,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11958262600",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.5912777,\n   "lon": 37.761117,\n   "opening_hours": "Mo-Fr,Su 10:00-19:00"\n  },\n  {\n   "id": "node/11958262606",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6342902,\n   "lon": 37.4407679,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/11958262608",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7109351,\n   "lon": 37.8120178,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11958352402",\n   "kind": "organisation",\n   "name": "Страна Девелопмент",\n   "office_type": "estate_agent",\n   "lat": 55.73667,\n   "lon": 37.5895876,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11958352405",\n   "kind": "organisation",\n   "name": "Страна Девелопмент",\n   "office_type": "estate_agent",\n   "lat": 55.6707192,\n   "lon": 37.4454295,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11958455640",\n   "kind": "organisation",\n   "name": "KapSEO",\n   "office_type": "company",\n   "lat": 55.8093519,\n   "lon": 37.7583662,\n   "opening_hours": "Mo-Fr 11:00-19:30, 13:00-13:30 off"\n  },\n  {\n   "id": "node/11958535200",\n   "kind": "organisation",\n   "name": "Жилфонд",\n   "office_type": "estate_agent",\n   "lat": 55.7099051,\n   "lon": 37.6527128,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa-Su 11:00-16:00"\n  },\n  {\n   "id": "node/11958541455",\n   "kind": "organisation",\n   "name": "Modis",\n   "office_type": "company",\n   "lat": 55.7095387,\n   "lon": 37.625571,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11959910969",\n   "kind": "organisation",\n   "name": "Диалект",\n   "office_type": "translator",\n   "lat": 55.7915322,\n   "lon": 37.6768566,\n   "opening_hours": "Su-Th 10:00-18:00"\n  },\n  {\n   "id": "node/11968892169",\n   "kind": "organisation",\n   "name": "Судебный юрист Сергей Иванов",\n   "office_type": "lawyer",\n   "lat": 55.7541201,\n   "lon": 37.8226763,\n   "opening_hours": "Su-Th 10:00-20:00; Fr-Sa 12:00-17:00"\n  },\n  {\n   "id": "node/11975171671",\n   "kind": "organisation",\n   "name": "Vysotsky",\n   "office_type": "estate_agent",\n   "lat": 55.5984145,\n   "lon": 37.6021961,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11975242283",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7379025,\n   "lon": 37.6866717,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11975242284",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6449662,\n   "lon": 37.4861884,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11975242285",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.870773,\n   "lon": 37.6775801,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/11975242286",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7786904,\n   "lon": 37.5418067,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 11:00-18:00"\n  },\n  {\n   "id": "node/11975242290",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.723521,\n   "lon": 37.651516,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11975242295",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7656838,\n   "lon": 37.6113285,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11975242304",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.783508,\n   "lon": 37.7205437,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/11975242305",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6129059,\n   "lon": 37.7195969,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11975242306",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6684866,\n   "lon": 37.5185479,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/11975242307",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6886847,\n   "lon": 37.4660954,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11975242308",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7067436,\n   "lon": 37.4669892,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/11981437029",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "insurance",\n   "lat": 55.802677,\n   "lon": 37.5266635,\n   "opening_hours": "Mo-Sa 11:00-17:30; Su 11:00-16:30"\n  },\n  {\n   "id": "node/11982493732",\n   "kind": "organisation",\n   "name": "Центр Правовой Помощи",\n   "office_type": "lawyer",\n   "lat": 55.7684701,\n   "lon": 37.5682395,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11982521345",\n   "kind": "organisation",\n   "name": "Содействие",\n   "office_type": "coworking",\n   "lat": 55.7686001,\n   "lon": 37.5674701,\n   "opening_hours": null\n  },\n  {\n   "id": "node/11987597369",\n   "kind": "organisation",\n   "name": "АО \\"ИнфоТеКС\\"",\n   "office_type": "company",\n   "lat": 55.8571217,\n   "lon": 37.5999061,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/11990326369",\n   "kind": "organisation",\n   "name": "Desoutter Tools",\n   "office_type": "company",\n   "lat": 55.7991253,\n   "lon": 37.6726759,\n   "opening_hours": "Fr,Th,We,Tu,Mo 09:00-18:00"\n  },\n  {\n   "id": "node/11991887541",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6635121,\n   "lon": 37.7304947,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11991887544",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6960315,\n   "lon": 37.6121364,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11991887550",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.725372,\n   "lon": 37.74557,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/11991903399",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8001129,\n   "lon": 37.5153966,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11991903401",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6823082,\n   "lon": 37.4847706,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/11991903405",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8030328,\n   "lon": 37.4556558,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11991903418",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7995174,\n   "lon": 37.4888266,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11991903419",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8344661,\n   "lon": 37.4545813,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11991903437",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6081002,\n   "lon": 37.7332692,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11991903443",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7755659,\n   "lon": 37.5861678,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11991903449",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7441218,\n   "lon": 37.6798214,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/11992711526",\n   "kind": "organisation",\n   "name": "Игроник",\n   "office_type": "advertising_agency",\n   "lat": 55.7777402,\n   "lon": 37.5870456,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/12000429187",\n   "kind": "organisation",\n   "name": "Gosurf",\n   "office_type": "yes",\n   "lat": 55.7592376,\n   "lon": 37.666221,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12009398993",\n   "kind": "organisation",\n   "name": "Городской психолого-педагогический центр, территориальное отделение Покровское-Стрешнево",\n   "office_type": "government",\n   "lat": 55.8302548,\n   "lon": 37.4575169,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12009505069",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "estate_agent",\n   "lat": 55.7926677,\n   "lon": 37.709721,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12010102101",\n   "kind": "organisation",\n   "name": "Администрация Преображенского рынка",\n   "office_type": "company",\n   "lat": 55.7939806,\n   "lon": 37.7196157,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12012237435",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6514076,\n   "lon": 37.6121487,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12012237443",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8255945,\n   "lon": 37.5161777,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12012237453",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6190655,\n   "lon": 37.509867,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12012237458",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6712089,\n   "lon": 37.5541084,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12012240669",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7520448,\n   "lon": 37.6000106,\n   "opening_hours": "Mo-Su 09:00-22:00"\n  },\n  {\n   "id": "node/12012240681",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7515078,\n   "lon": 37.8205032,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12012240685",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.6763556,\n   "lon": 37.6651465,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/12012240688",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.864668,\n   "lon": 37.6049132,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/12012240691",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.8410233,\n   "lon": 37.4850115,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12016320037",\n   "kind": "organisation",\n   "name": "Адвокатская контора №1",\n   "office_type": "lawyer",\n   "lat": 55.7309895,\n   "lon": 37.6413554,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/12020685627",\n   "kind": "organisation",\n   "name": "Межрегиональная инспекция по крупнейшим налогоплательщикам № 9",\n   "office_type": "government",\n   "lat": 55.7950574,\n   "lon": 37.7256247,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12020685628",\n   "kind": "organisation",\n   "name": "Межрегиональная инспекция по крупнейшим налогоплательщикам № 7",\n   "office_type": "government",\n   "lat": 55.795268,\n   "lon": 37.725506,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12020685629",\n   "kind": "organisation",\n   "name": "Межрегиональная инспекция по крупнейшим налогоплательщикам № 6",\n   "office_type": "government",\n   "lat": 55.795124,\n   "lon": 37.7251846,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12020685630",\n   "kind": "organisation",\n   "name": "Межрегиональная инспекция по крупнейшим налогоплательщикам № 5",\n   "office_type": "government",\n   "lat": 55.7952109,\n   "lon": 37.7251398,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12020685631",\n   "kind": "organisation",\n   "name": "Межрегиональная инспекция по крупнейшим налогоплательщикам № 4",\n   "office_type": "government",\n   "lat": 55.7949337,\n   "lon": 37.7248245,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12020685632",\n   "kind": "organisation",\n   "name": "Межрегиональная инспекция по крупнейшим налогоплательщикам № 3",\n   "office_type": "government",\n   "lat": 55.7951538,\n   "lon": 37.7248258,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12021268315",\n   "kind": "organisation",\n   "name": "Единая россия",\n   "office_type": "political_party",\n   "lat": 55.7039927,\n   "lon": 37.6531653,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12026320354",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.7508314,\n   "lon": 37.7195398,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa 10:00-15:00"\n  },\n  {\n   "id": "node/12032894060",\n   "kind": "organisation",\n   "name": "Настроение",\n   "office_type": "estate_agent",\n   "lat": 55.8500675,\n   "lon": 37.6829596,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12032894061",\n   "kind": "organisation",\n   "name": "ГК ФСК",\n   "office_type": "estate_agent",\n   "lat": 55.762047,\n   "lon": 37.633321,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12032894062",\n   "kind": "organisation",\n   "name": "Роттердам",\n   "office_type": "estate_agent",\n   "lat": 55.68588,\n   "lon": 37.624741,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12032894063",\n   "kind": "organisation",\n   "name": "Sydney City",\n   "office_type": "estate_agent",\n   "lat": 55.7646634,\n   "lon": 37.50972,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12032894064",\n   "kind": "organisation",\n   "name": "Движение",\n   "office_type": "estate_agent",\n   "lat": 55.6691733,\n   "lon": 37.4474176,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12032894065",\n   "kind": "organisation",\n   "name": "Архитектор",\n   "office_type": "estate_agent",\n   "lat": 55.6561301,\n   "lon": 37.5277573,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12032894066",\n   "kind": "organisation",\n   "name": "The Lake",\n   "office_type": "estate_agent",\n   "lat": 55.8518411,\n   "lon": 37.6485511,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12032924169",\n   "kind": "organisation",\n   "name": "Рихард",\n   "office_type": "estate_agent",\n   "lat": 55.7821551,\n   "lon": 37.5099292,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12032924992",\n   "kind": "organisation",\n   "name": "Первый ДСК",\n   "office_type": "estate_agent",\n   "lat": 55.7514113,\n   "lon": 37.5967775,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12032924994",\n   "kind": "organisation",\n   "name": "1-й Ясеневский",\n   "office_type": "estate_agent",\n   "lat": 55.605736,\n   "lon": 37.478375,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12043939291",\n   "kind": "organisation",\n   "name": "Ресо-мед",\n   "office_type": "insurance",\n   "lat": 55.6534336,\n   "lon": 37.5785095,\n   "opening_hours": "Mo-Fr 10:00-16:00"\n  },\n  {\n   "id": "node/12059107616",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8720657,\n   "lon": 37.5134064,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/12059107618",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6473667,\n   "lon": 37.6188494,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12059265106",\n   "kind": "organisation",\n   "name": "Адвокат Лычагин А.А.",\n   "office_type": "lawyer",\n   "lat": 55.7509543,\n   "lon": 37.6552454,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/12059265118",\n   "kind": "organisation",\n   "name": "arcsinus",\n   "office_type": "it",\n   "lat": 55.7652589,\n   "lon": 37.6220236,\n   "opening_hours": "Mo-Fr 09:00-13:00,14:00-18:00"\n  },\n  {\n   "id": "node/12063084670",\n   "kind": "organisation",\n   "name": "Charm Rus Distribution",\n   "office_type": "company",\n   "lat": 55.7858151,\n   "lon": 37.6746596,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12068128099",\n   "kind": "organisation",\n   "name": "ООО «Форвард-Групп»",\n   "office_type": "company",\n   "lat": 55.6545289,\n   "lon": 37.7125511,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12069309185",\n   "kind": "organisation",\n   "name": "Формула Рекламы",\n   "office_type": "advertising_agency",\n   "lat": 55.7554568,\n   "lon": 37.5565887,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12071074318",\n   "kind": "organisation",\n   "name": "Мосгражданпроект",\n   "office_type": "company",\n   "lat": 55.6511742,\n   "lon": 37.5487214,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12071074319",\n   "kind": "organisation",\n   "name": "Московская областная государственная экспертиза",\n   "office_type": "government",\n   "lat": 55.6514451,\n   "lon": 37.5481474,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12071296394",\n   "kind": "organisation",\n   "name": "IBG",\n   "office_type": "insurance",\n   "lat": 55.7631357,\n   "lon": 37.5082041,\n   "opening_hours": "Fr 09:00-17:00; Mo-Th 09:00-18:00"\n  },\n  {\n   "id": "node/12073970016",\n   "kind": "organisation",\n   "name": "Союз Журналистов России",\n   "office_type": "yes",\n   "lat": 55.7378144,\n   "lon": 37.5892027,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12074784800",\n   "kind": "organisation",\n   "name": "Синимекс",\n   "office_type": "it",\n   "lat": 55.7376218,\n   "lon": 37.6353197,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12078184369",\n   "kind": "organisation",\n   "name": "Адвокатская контора 4",\n   "office_type": "lawyer",\n   "lat": 55.7990296,\n   "lon": 37.5368216,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12079980631",\n   "kind": "organisation",\n   "name": "Совет ветеранов Ломоносовского района. Первичная организация № 4",\n   "office_type": "ngo",\n   "lat": 55.6816202,\n   "lon": 37.5319378,\n   "opening_hours": "Mo, We 11:00-13:00"\n  },\n  {\n   "id": "node/12080010395",\n   "kind": "organisation",\n   "name": "Всероссийкое общество слепых \\"Ломоносовский округ\\"",\n   "office_type": "ngo",\n   "lat": 55.6814676,\n   "lon": 37.5320746,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12084666609",\n   "kind": "organisation",\n   "name": "Lada Академия",\n   "office_type": "company",\n   "lat": 55.8674875,\n   "lon": 37.4570274,\n   "opening_hours": "Mo-Fr 09:00-16:45"\n  },\n  {\n   "id": "node/12089119229",\n   "kind": "organisation",\n   "name": "MusBooking",\n   "office_type": "it",\n   "lat": 55.7094773,\n   "lon": 37.6463763,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12089549441",\n   "kind": "organisation",\n   "name": "Управление вневедомственной охраны войск Национальной гвардии России по Москве",\n   "office_type": "security",\n   "lat": 55.6985235,\n   "lon": 37.7323052,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12089577262",\n   "kind": "organisation",\n   "name": "ФГУП \\"Охрана\\"",\n   "office_type": "security",\n   "lat": 55.6934134,\n   "lon": 37.7419907,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12091117790",\n   "kind": "organisation",\n   "name": "Нотариус Фурчакова Татьяна Евгеньевна",\n   "office_type": "lawyer",\n   "lat": 55.6868375,\n   "lon": 37.7522716,\n   "opening_hours": "Mo-Th 09:00-13:00,14:00-17:00; Fr 09:00-13:00,14:00-16:00"\n  },\n  {\n   "id": "node/12096755837",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.824238,\n   "lon": 37.6503428,\n   "opening_hours": "Mo-Fr 11:00-21:00"\n  },\n  {\n   "id": "node/12096755841",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.792597,\n   "lon": 37.4906328,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/12096755847",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.679582,\n   "lon": 37.5462297,\n   "opening_hours": "Mo-Sa 10:00-19:00"\n  },\n  {\n   "id": "node/12096755851",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8897581,\n   "lon": 37.594083,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/12096755855",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8862625,\n   "lon": 37.6567485,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 11:00-18:00"\n  },\n  {\n   "id": "node/12096755856",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7324202,\n   "lon": 37.6406992,\n   "opening_hours": "Mo-Su 10:00-18:00"\n  },\n  {\n   "id": "node/12096755884",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7434713,\n   "lon": 37.5076032,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12096755893",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7061144,\n   "lon": 37.6787217,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12096755894",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.724804,\n   "lon": 37.7444902,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/12096765851",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7110902,\n   "lon": 37.6750857,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12096765864",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7888778,\n   "lon": 37.6791959,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12096765869",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7019773,\n   "lon": 37.7650287,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12096807304",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.5857275,\n   "lon": 37.7241036,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/12097514864",\n   "kind": "organisation",\n   "name": "Объединенная Вагонная Компания",\n   "office_type": "company",\n   "lat": 55.7514569,\n   "lon": 37.5958438,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12097514865",\n   "kind": "organisation",\n   "name": "FESCO",\n   "office_type": "company",\n   "lat": 55.7391246,\n   "lon": 37.6304403,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12097530418",\n   "kind": "organisation",\n   "name": "Бухгалтерское агентство Валерии Кривонос",\n   "office_type": "accountant",\n   "lat": 55.776303,\n   "lon": 37.6737928,\n   "opening_hours": "Mo-Su 09:00-18:30"\n  },\n  {\n   "id": "node/12099562006",\n   "kind": "organisation",\n   "name": "Студия красоты",\n   "office_type": "coworking",\n   "lat": 55.7110635,\n   "lon": 37.6112564,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12100585817",\n   "kind": "organisation",\n   "name": "Алмаз сервис",\n   "office_type": "company",\n   "lat": 55.6457224,\n   "lon": 37.7552964,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12106517544",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7501057,\n   "lon": 37.7198621,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/12110530696",\n   "kind": "organisation",\n   "name": "Whoosh",\n   "office_type": "it",\n   "lat": 55.7755453,\n   "lon": 37.6030738,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12112273407",\n   "kind": "organisation",\n   "name": "DML",\n   "office_type": "logistics",\n   "lat": 55.8827821,\n   "lon": 37.5269722,\n   "opening_hours": "Mo-Fr 09:00-17:00; PH closed"\n  },\n  {\n   "id": "node/12117405113",\n   "kind": "organisation",\n   "name": "Линк-Сервис",\n   "office_type": "it",\n   "lat": 55.7501789,\n   "lon": 37.6771367,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12118424797",\n   "kind": "organisation",\n   "name": "Управляющая компания «Сходненская-1»",\n   "office_type": "property_management",\n   "lat": 55.8544915,\n   "lon": 37.4473557,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12121654301",\n   "kind": "organisation",\n   "name": "Защита",\n   "office_type": "ngo",\n   "lat": 55.7274387,\n   "lon": 37.5871619,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12122236463",\n   "kind": "organisation",\n   "name": "VK",\n   "office_type": "company",\n   "lat": 55.8046548,\n   "lon": 37.5186037,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12122427429",\n   "kind": "organisation",\n   "name": "VK",\n   "office_type": "company",\n   "lat": 55.8263183,\n   "lon": 37.5708362,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12122538626",\n   "kind": "organisation",\n   "name": "VK",\n   "office_type": "company",\n   "lat": 55.7361536,\n   "lon": 37.6422031,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12123900986",\n   "kind": "organisation",\n   "name": "Транзит",\n   "office_type": "logistics",\n   "lat": 55.7950813,\n   "lon": 37.7131543,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12124735949",\n   "kind": "organisation",\n   "name": "Жилищник района Южное Тушино",\n   "office_type": "property_management",\n   "lat": 55.8475293,\n   "lon": 37.4458547,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12136529994",\n   "kind": "organisation",\n   "name": "СВД-Риэлти",\n   "office_type": "estate_agent",\n   "lat": 55.8131789,\n   "lon": 37.8102792,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12137002601",\n   "kind": "organisation",\n   "name": "ФГБНУ «МГНЦ им. Н.П. Бочкова»",\n   "office_type": "research",\n   "lat": 55.6544993,\n   "lon": 37.6395984,\n   "opening_hours": "Mo-We, Fr 09:00-19:00; Th 09:00-17:00"\n  },\n  {\n   "id": "node/12137002602",\n   "kind": "organisation",\n   "name": "ФГБУ «НМИЦ эндокринологии» Минздрава России (ЭНЦ)",\n   "office_type": "research",\n   "lat": 55.6551121,\n   "lon": 37.6386883,\n   "opening_hours": "Mo-Fr 08:30-17:00"\n  },\n  {\n   "id": "node/12138283112",\n   "kind": "organisation",\n   "name": "Яковлев&Партнёры",\n   "office_type": "lawyer",\n   "lat": 55.7597377,\n   "lon": 37.6532341,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12141876500",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6561599,\n   "lon": 37.5416232,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12141981508",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6195048,\n   "lon": 37.734098,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12141981509",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8210644,\n   "lon": 37.5916641,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12141981513",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6303375,\n   "lon": 37.5172704,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12141981516",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6827388,\n   "lon": 37.544328,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12141981517",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6378951,\n   "lon": 37.540582,\n   "opening_hours": "Mo-Sa 11:00-21:00"\n  },\n  {\n   "id": "node/12141981518",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.777102,\n   "lon": 37.6280665,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12141981519",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8126124,\n   "lon": 37.6513664,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12141981523",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8705752,\n   "lon": 37.4755835,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12141981524",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6783394,\n   "lon": 37.6975261,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12141981529",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8561928,\n   "lon": 37.555398,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12141981535",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.713081,\n   "lon": 37.454241,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12141994358",\n   "kind": "organisation",\n   "name": "Такси Ритм",\n   "office_type": "taxi",\n   "lat": 55.6366476,\n   "lon": 37.6461393,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12141994359",\n   "kind": "organisation",\n   "name": "Такси Ритм",\n   "office_type": "taxi",\n   "lat": 55.6556702,\n   "lon": 37.7315729,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12141994360",\n   "kind": "organisation",\n   "name": "Такси Ритм",\n   "office_type": "taxi",\n   "lat": 55.860866,\n   "lon": 37.5788534,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12141994361",\n   "kind": "organisation",\n   "name": "Такси Ритм",\n   "office_type": "taxi",\n   "lat": 55.8138461,\n   "lon": 37.785008,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12141994362",\n   "kind": "organisation",\n   "name": "Такси Ритм",\n   "office_type": "taxi",\n   "lat": 55.8768321,\n   "lon": 37.4973356,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12148398744",\n   "kind": "organisation",\n   "name": "МТС",\n   "office_type": "telecommunication",\n   "lat": 55.7618669,\n   "lon": 37.6087338,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12165290240",\n   "kind": "organisation",\n   "name": "ТМ Электроникс",\n   "office_type": "company",\n   "lat": 55.8897805,\n   "lon": 37.5731808,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/12166473556",\n   "kind": "organisation",\n   "name": "T2",\n   "office_type": "telecommunication",\n   "lat": 55.8562874,\n   "lon": 37.6534107,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/12168659537",\n   "kind": "organisation",\n   "name": "Нотариec Костиков А. И.",\n   "office_type": "notary",\n   "lat": 55.8643768,\n   "lon": 37.6632223,\n   "opening_hours": "Mo-Th 09:30-18:30; Fr 09:30-13:30,14:30-17:30; Sa 10:00-16:00"\n  },\n  {\n   "id": "node/12169790421",\n   "kind": "organisation",\n   "name": "Эволюция",\n   "office_type": "financial_advisor",\n   "lat": 55.7330235,\n   "lon": 37.6434229,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12170604710",\n   "kind": "organisation",\n   "name": "Ростра",\n   "office_type": "company",\n   "lat": 55.637451,\n   "lon": 37.5287004,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/12170604711",\n   "kind": "organisation",\n   "name": "Комвид",\n   "office_type": "company",\n   "lat": 55.6374389,\n   "lon": 37.5287588,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/12170604713",\n   "kind": "organisation",\n   "name": "Стилот",\n   "office_type": "company",\n   "lat": 55.6376911,\n   "lon": 37.5292468,\n   "opening_hours": "Mo-Fr 09:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/12170604714",\n   "kind": "organisation",\n   "name": "Литлидер",\n   "office_type": "company",\n   "lat": 55.6377098,\n   "lon": 37.5292623,\n   "opening_hours": "Mo-Fr 08:30-17:30; Sa-Su off"\n  },\n  {\n   "id": "node/12170604716",\n   "kind": "organisation",\n   "name": "Угра",\n   "office_type": "company",\n   "lat": 55.6377804,\n   "lon": 37.529304,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12170604718",\n   "kind": "organisation",\n   "name": "Текта-Обручева",\n   "office_type": "company",\n   "lat": 55.6378437,\n   "lon": 37.5293702,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12180145460",\n   "kind": "organisation",\n   "name": "AJM",\n   "office_type": "company",\n   "lat": 55.6141651,\n   "lon": 37.5152233,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/12186432114",\n   "kind": "organisation",\n   "name": "Посольство Хорватии",\n   "office_type": "diplomatic",\n   "lat": 55.7405009,\n   "lon": 37.5977683,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12187705657",\n   "kind": "organisation",\n   "name": "Щербакова Наталья Николаевна",\n   "office_type": "notary",\n   "lat": 55.7756605,\n   "lon": 37.5951613,\n   "opening_hours": "Mo-Th 10:00-19:00, Fr 10:00-17:00"\n  },\n  {\n   "id": "node/12192667416",\n   "kind": "organisation",\n   "name": "Христианско-пресвитерианское общество",\n   "office_type": "religion",\n   "lat": 55.6020539,\n   "lon": 37.5550998,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12197697204",\n   "kind": "organisation",\n   "name": "Грин Код",\n   "office_type": "company",\n   "lat": 55.7848322,\n   "lon": 37.6837456,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12198623443",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7907311,\n   "lon": 37.5502079,\n   "opening_hours": "Mo-Sa 10:30-20:00"\n  },\n  {\n   "id": "node/12198623451",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8435075,\n   "lon": 37.4825642,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/12198623452",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6778316,\n   "lon": 37.5066987,\n   "opening_hours": "Mo-Su 10:00-21:00"\n  },\n  {\n   "id": "node/12198623454",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7428911,\n   "lon": 37.5582631,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/12198623458",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.811088,\n   "lon": 37.7995209,\n   "opening_hours": "Mo-Sa 10:00-21:00; Su 11:30-19:00"\n  },\n  {\n   "id": "node/12198873872",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7906794,\n   "lon": 37.4997904,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12198873879",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6028163,\n   "lon": 37.5293457,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12198873880",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7055805,\n   "lon": 37.688773,\n   "opening_hours": "Mo-Su 09:00-19:00"\n  },\n  {\n   "id": "node/12198873887",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.5910482,\n   "lon": 37.5390151,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12198873888",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7953905,\n   "lon": 37.5731719,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12198873894",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.6794988,\n   "lon": 37.7379084,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12198873896",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8945353,\n   "lon": 37.6133305,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12198873897",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7013812,\n   "lon": 37.5672662,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12198877504",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7199264,\n   "lon": 37.7437792,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12198984509",\n   "kind": "organisation",\n   "name": "id`ea",\n   "office_type": "advertising_agency",\n   "lat": 55.788235,\n   "lon": 37.567873,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12198984512",\n   "kind": "organisation",\n   "name": "Айвори Групп",\n   "office_type": "company",\n   "lat": 55.6962637,\n   "lon": 37.469709,\n   "opening_hours": "Mo-Fr 09:00-20:00; Sa 10:00-18:00; Su 11:00-17:00"\n  },\n  {\n   "id": "node/12198984515",\n   "kind": "organisation",\n   "name": "Компания эвакуации авто",\n   "office_type": "company",\n   "lat": 55.669726,\n   "lon": 37.531751,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/12198984537",\n   "kind": "organisation",\n   "name": "Долгов Авто",\n   "office_type": "company",\n   "lat": 55.6721123,\n   "lon": 37.6310141,\n   "opening_hours": "Mo-Fr 10:00-19:00; Sa 12:00-15:00"\n  },\n  {\n   "id": "node/12208384606",\n   "kind": "organisation",\n   "name": "Институт Прокуратуры Российской Федерации",\n   "office_type": "educational_institution",\n   "lat": 55.7634279,\n   "lon": 37.5546346,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12212671029",\n   "kind": "organisation",\n   "name": "Бармина Анна Владимировна",\n   "office_type": "notary",\n   "lat": 55.7859928,\n   "lon": 37.5735648,\n   "opening_hours": "Mo-Fr 10:00-17:00; We 11:00-19:00"\n  },\n  {\n   "id": "node/12215883525",\n   "kind": "organisation",\n   "name": "WestMotors",\n   "office_type": "company",\n   "lat": 55.7212926,\n   "lon": 37.6368942,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 13:00-17:00"\n  },\n  {\n   "id": "node/12230451728",\n   "kind": "organisation",\n   "name": "Tele2",\n   "office_type": "telecommunication",\n   "lat": 55.6125579,\n   "lon": 37.6968572,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12233205858",\n   "kind": "organisation",\n   "name": "Российский союз научных и инженерных общественных объединений",\n   "office_type": "ngo",\n   "lat": 55.7402623,\n   "lon": 37.6053737,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12237934007",\n   "kind": "organisation",\n   "name": "BeBrand",\n   "office_type": "company",\n   "lat": 55.6923982,\n   "lon": 37.6226404,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12237934012",\n   "kind": "organisation",\n   "name": "BeBrand",\n   "office_type": "company",\n   "lat": 55.8158716,\n   "lon": 37.5272824,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12238142129",\n   "kind": "organisation",\n   "name": "Чистый лист",\n   "office_type": "lawyer",\n   "lat": 55.7898564,\n   "lon": 37.6786879,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/12238142133",\n   "kind": "organisation",\n   "name": "Чистый лист",\n   "office_type": "lawyer",\n   "lat": 55.7008996,\n   "lon": 37.7335283,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/12238142143",\n   "kind": "organisation",\n   "name": "Чистый лист",\n   "office_type": "lawyer",\n   "lat": 55.7670726,\n   "lon": 37.8304795,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/12248402311",\n   "kind": "organisation",\n   "name": "Представительство Европейского Союза",\n   "office_type": "diplomatic",\n   "lat": 55.7439076,\n   "lon": 37.6196185,\n   "opening_hours": "Mo-Fr 09:00-17:30"\n  },\n  {\n   "id": "node/12258217304",\n   "kind": "organisation",\n   "name": "Вистерма",\n   "office_type": "company",\n   "lat": 55.8013699,\n   "lon": 37.5614465,\n   "opening_hours": "Su-Th 08:00-17:00"\n  },\n  {\n   "id": "node/12273135989",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "notary",\n   "lat": 55.6795142,\n   "lon": 37.5636299,\n   "opening_hours": "Mo-Fr 10:00-18:00; Sa-Su off"\n  },\n  {\n   "id": "node/12273183580",\n   "kind": "organisation",\n   "name": "Азбука вкуса",\n   "office_type": "company",\n   "lat": 55.7422698,\n   "lon": 37.5239267,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12281093173",\n   "kind": "organisation",\n   "name": "Nova",\n   "office_type": "estate_agent",\n   "lat": 55.7240836,\n   "lon": 37.504134,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12281093175",\n   "kind": "organisation",\n   "name": "Октябрьская 98",\n   "office_type": "estate_agent",\n   "lat": 55.800372,\n   "lon": 37.615174,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12281093176",\n   "kind": "organisation",\n   "name": "Квартал Румянцево",\n   "office_type": "estate_agent",\n   "lat": 55.6344508,\n   "lon": 37.4538045,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12281093179",\n   "kind": "organisation",\n   "name": "Вереск",\n   "office_type": "estate_agent",\n   "lat": 55.784195,\n   "lon": 37.496836,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12281093185",\n   "kind": "organisation",\n   "name": "Новоданиловская 8",\n   "office_type": "estate_agent",\n   "lat": 55.7319779,\n   "lon": 37.6476708,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12281093186",\n   "kind": "organisation",\n   "name": "Квартал Герцена",\n   "office_type": "estate_agent",\n   "lat": 55.5849948,\n   "lon": 37.7093664,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12281093188",\n   "kind": "organisation",\n   "name": "Квартал Домашний",\n   "office_type": "estate_agent",\n   "lat": 55.6455498,\n   "lon": 37.7073961,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12284775577",\n   "kind": "organisation",\n   "name": "Галчонок",\n   "office_type": "charity",\n   "lat": 55.7493048,\n   "lon": 37.5995796,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12287558696",\n   "kind": "organisation",\n   "name": "Юматекс",\n   "office_type": "research",\n   "lat": 55.7075044,\n   "lon": 37.7250277,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12290298171",\n   "kind": "organisation",\n   "name": "СберТех",\n   "office_type": "it",\n   "lat": 55.6961919,\n   "lon": 37.6256514,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12294330391",\n   "kind": "organisation",\n   "name": "INpacks",\n   "office_type": "company",\n   "lat": 55.7081066,\n   "lon": 37.6949055,\n   "opening_hours": "10:00 - 20:00"\n  },\n  {\n   "id": "node/12296421701",\n   "kind": "organisation",\n   "name": "Мой гектар",\n   "office_type": "company",\n   "lat": 55.7762654,\n   "lon": 37.627327,\n   "opening_hours": "Mo-Su 09:00-20:00"\n  },\n  {\n   "id": "node/12298956260",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.5925756,\n   "lon": 37.6126229,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12298956267",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8563596,\n   "lon": 37.5096433,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12298956275",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.7020746,\n   "lon": 37.7471137,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12299657856",\n   "kind": "organisation",\n   "name": "ООО \\"МОСГЕОПРОЕКТ\\"",\n   "office_type": "engineer",\n   "lat": 55.6467016,\n   "lon": 37.5260894,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12309783460",\n   "kind": "organisation",\n   "name": "Департамент информационных технологий г. Москвы",\n   "office_type": "government",\n   "lat": 55.7589464,\n   "lon": 37.6563933,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12309914524",\n   "kind": "organisation",\n   "name": "Kelin",\n   "office_type": "lawyer",\n   "lat": 55.7625311,\n   "lon": 37.6365304,\n   "opening_hours": "Mo-Fr 10:00-19:00"\n  },\n  {\n   "id": "node/12314105589",\n   "kind": "organisation",\n   "name": "Качай права!",\n   "office_type": "lawyer",\n   "lat": 55.7345506,\n   "lon": 37.6257824,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12325546118",\n   "kind": "organisation",\n   "name": "Селенкова В.С.",\n   "office_type": "notary",\n   "lat": 55.6444537,\n   "lon": 37.5190692,\n   "opening_hours": "Mo,Su off; Tu-Th 10:00-17:00; Fr-Sa 10:00-15:00"\n  },\n  {\n   "id": "node/12336636510",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.591279,\n   "lon": 37.444503,\n   "opening_hours": "Mo-Su 09:00-21:00"\n  },\n  {\n   "id": "node/12336636515",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6659387,\n   "lon": 37.4693935,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/12336636525",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.7582189,\n   "lon": 37.5521691,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/12336636529",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.6831553,\n   "lon": 37.6661212,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa-Su 10:00-18:00"\n  },\n  {\n   "id": "node/12336636534",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.73962,\n   "lon": 37.6051766,\n   "opening_hours": "Mo-Fr 10:00-18:00"\n  },\n  {\n   "id": "node/12336636536",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.8706934,\n   "lon": 37.6888593,\n   "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-18:00"\n  },\n  {\n   "id": "node/12336636539",\n   "kind": "organisation",\n   "name": "Самолет Плюс",\n   "office_type": "estate_agent",\n   "lat": 55.738245,\n   "lon": 37.5803425,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/12349139278",\n   "kind": "organisation",\n   "name": "Амаль",\n   "office_type": "company",\n   "lat": 55.789505,\n   "lon": 37.569817,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12357733694",\n   "kind": "organisation",\n   "name": "АдминВПС",\n   "office_type": "it",\n   "lat": 55.7084283,\n   "lon": 37.6532061,\n   "opening_hours": "Mo-Su 09:00-18:00"\n  },\n  {\n   "id": "node/12361482178",\n   "kind": "organisation",\n   "name": "Центральное диспетчерское управление топливно-энергетического комплекса",\n   "office_type": "surveyor",\n   "lat": 55.8043892,\n   "lon": 37.5520201,\n   "opening_hours": "Mo-Th 08:30-17:30; Fr 08:30-16:15"\n  },\n  {\n   "id": "node/12366974766",\n   "kind": "organisation",\n   "name": "Вектор Прайм",\n   "office_type": "lawyer",\n   "lat": 55.8866781,\n   "lon": 37.6158272,\n   "opening_hours": "Mo-Su 10:00-20:00"\n  },\n  {\n   "id": "node/12371592565",\n   "kind": "organisation",\n   "name": "Осипова С.А.",\n   "office_type": "notary",\n   "lat": 55.6320597,\n   "lon": 37.524392,\n   "opening_hours": "Mo-Th 10:00-13:00,14:00-17:00; Sa-Su off"\n  },\n  {\n   "id": "node/12371592580",\n   "kind": "organisation",\n   "name": "Партнер",\n   "office_type": "company",\n   "lat": 55.6325118,\n   "lon": 37.525224,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12374498668",\n   "kind": "organisation",\n   "name": "Комплексная лаборатория исследования внеземных территорий",\n   "office_type": "research",\n   "lat": 55.764837,\n   "lon": 37.6627952,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12384871484",\n   "kind": "organisation",\n   "name": "Кошелев",\n   "office_type": "company",\n   "lat": 55.7317794,\n   "lon": 37.5878616,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12387095501",\n   "kind": "organisation",\n   "name": "Achteck",\n   "office_type": "company",\n   "lat": 55.8449529,\n   "lon": 37.6342656,\n   "opening_hours": "24/7"\n  },\n  {\n   "id": "node/12392638205",\n   "kind": "organisation",\n   "name": "Про налог",\n   "office_type": "accountant",\n   "lat": 55.787215,\n   "lon": 37.6710449,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12394748309",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "lawyer",\n   "lat": 55.7884177,\n   "lon": 37.6715787,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12396503252",\n   "kind": "organisation",\n   "name": "Абсида",\n   "office_type": "engineer",\n   "lat": 55.6814277,\n   "lon": 37.6043624,\n   "opening_hours": "Mo-Fr 09:00-17:00; Sa-Su off"\n  },\n  {\n   "id": "node/12411881216",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8920764,\n   "lon": 37.7482113,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12411881235",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.880894,\n   "lon": 37.564492,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12411881242",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8709663,\n   "lon": 37.4848419,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12411881243",\n   "kind": "organisation",\n   "name": "ПЭК: Easyway",\n   "office_type": "logistics",\n   "lat": 55.8737106,\n   "lon": 37.5390038,\n   "opening_hours": "Mo-Su 10:00-22:00"\n  },\n  {\n   "id": "node/12430845798",\n   "kind": "organisation",\n   "name": "Komatsu",\n   "office_type": "company",\n   "lat": 55.7800832,\n   "lon": 37.6056969,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12434071661",\n   "kind": "organisation",\n   "name": "Московский фонд мира",\n   "office_type": "quango",\n   "lat": 55.7782987,\n   "lon": 37.6098463,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12434301438",\n   "kind": "organisation",\n   "name": "СоюзДонСтрой",\n   "office_type": "construction_company",\n   "lat": 55.7711404,\n   "lon": 37.6136631,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12438086984",\n   "kind": "organisation",\n   "name": "Прокошенкова Елена Евгеньевна",\n   "office_type": "notary",\n   "lat": 55.7730423,\n   "lon": 37.5918779,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12438086987",\n   "kind": "organisation",\n   "name": "СитиАрбитр",\n   "office_type": "lawyer",\n   "lat": 55.7739544,\n   "lon": 37.5889648,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12438974428",\n   "kind": "organisation",\n   "name": "Мосремонт",\n   "office_type": "government",\n   "lat": 55.7727867,\n   "lon": 37.609669,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12439578169",\n   "kind": "organisation",\n   "name": "Chattaev Protection",\n   "office_type": "lawyer",\n   "lat": 55.7922676,\n   "lon": 37.6093072,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12439613705",\n   "kind": "organisation",\n   "name": "",\n   "office_type": "company",\n   "lat": 55.7908162,\n   "lon": 37.6094359,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12444375574",\n   "kind": "organisation",\n   "name": "УК Первая",\n   "office_type": "financial",\n   "lat": 55.7498166,\n   "lon": 37.5377846,\n   "opening_hours": "Mo-Fr 09:00-18:00"\n  },\n  {\n   "id": "node/12447366880",\n   "kind": "organisation",\n   "name": "Bell Integrator",\n   "office_type": "company",\n   "lat": 55.7028968,\n   "lon": 37.6914476,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12450032785",\n   "kind": "organisation",\n   "name": "Psycho Place",\n   "office_type": "coworking",\n   "lat": 55.7408565,\n   "lon": 37.6301292,\n   "opening_hours": "Mo-Su 08:00-22:00"\n  },\n  {\n   "id": "node/12451062065",\n   "kind": "organisation",\n   "name": "Доходные материалы",\n   "office_type": "company",\n   "lat": 55.7583341,\n   "lon": 37.7725495,\n   "opening_hours": null\n  },\n  {\n   "id": "node/12455575201",\n   "kind": "organisation",\n   "name": "Призыва нет",\n   "office_type": "lawyer",\n   "lat": 55.7637738,\n   "lon": 37.6077733,\n   "opening_hours": null\n  }\n ],\n "routes": {\n  "17": {\n   "buildings": 26,\n   "organisations": 86,\n   "tagged_hours": 42,\n   "parsed_hours": 36,\n   "hour_signals": [\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      16.0,\n      14.0,\n      0.0\n     ],\n     [\n      32.0,\n      15.0,\n      0.0\n     ],\n     [\n      34.0,\n      2.0,\n      0.0\n     ],\n     [\n      34.0,\n      0.0,\n      0.0\n     ],\n     [\n      34.0,\n      0.0,\n      0.0\n     ],\n     [\n      34.0,\n      0.0,\n      0.0\n     ],\n     [\n      34.0,\n      0.0,\n      0.0\n     ],\n     [\n      34.0,\n      0.0,\n      0.0\n     ],\n     [\n      34.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.5,\n      0.0,\n      13.0\n     ],\n     [\n      21.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      0.0,\n      5.0\n     ],\n     [\n      6.0,\n      0.0,\n      10.0\n     ],\n     [\n      0.0,\n      0.0,\n      6.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      17.0,\n      15.0,\n      0.0\n     ],\n     [\n      34.0,\n      16.0,\n      0.0\n     ],\n     [\n      36.0,\n      2.0,\n      0.0\n     ],\n     [\n      36.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.5,\n      0.0,\n      1.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      36.0,\n      1.0,\n      0.0\n     ],\n     [\n      36.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.5,\n      0.0,\n      1.0\n     ],\n     [\n      22.5,\n      0.0,\n      13.0\n     ],\n     [\n      21.0,\n      0.0,\n      1.0\n     ],\n     [\n      16.0,\n      0.0,\n      5.0\n     ],\n     [\n      6.0,\n      0.0,\n      10.0\n     ],\n     [\n      0.0,\n      0.0,\n      6.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      17.0,\n      15.0,\n      0.0\n     ],\n     [\n      34.0,\n      16.0,\n      0.0\n     ],\n     [\n      36.0,\n      2.0,\n      0.0\n     ],\n     [\n      36.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.5,\n      0.0,\n      1.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      36.0,\n      1.0,\n      0.0\n     ],\n     [\n      36.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.5,\n      0.0,\n      1.0\n     ],\n     [\n      22.5,\n      0.0,\n      13.0\n     ],\n     [\n      21.0,\n      0.0,\n      1.0\n     ],\n     [\n      16.0,\n      0.0,\n      5.0\n     ],\n     [\n      6.0,\n      0.0,\n      10.0\n     ],\n     [\n      0.0,\n      0.0,\n      6.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      17.0,\n      15.0,\n      0.0\n     ],\n     [\n      34.0,\n      16.0,\n      0.0\n     ],\n     [\n      36.0,\n      2.0,\n      0.0\n     ],\n     [\n      36.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.5,\n      0.0,\n      1.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      36.0,\n      1.0,\n      0.0\n     ],\n     [\n      36.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.5,\n      0.0,\n      1.0\n     ],\n     [\n      22.5,\n      0.0,\n      13.0\n     ],\n     [\n      21.0,\n      0.0,\n      1.0\n     ],\n     [\n      16.0,\n      0.0,\n      5.0\n     ],\n     [\n      6.0,\n      0.0,\n      10.0\n     ],\n     [\n      0.0,\n      0.0,\n      6.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      16.5,\n      14.0,\n      0.0\n     ],\n     [\n      33.0,\n      16.0,\n      0.0\n     ],\n     [\n      35.0,\n      2.0,\n      0.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      34.0,\n      0.0,\n      2.0\n     ],\n     [\n      33.5,\n      1.0,\n      0.0\n     ],\n     [\n      35.0,\n      1.0,\n      0.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      33.0,\n      0.0,\n      3.0\n     ],\n     [\n      22.0,\n      0.0,\n      10.0\n     ],\n     [\n      20.0,\n      0.0,\n      2.0\n     ],\n     [\n      16.0,\n      0.0,\n      4.0\n     ],\n     [\n      6.0,\n      0.0,\n      10.0\n     ],\n     [\n      0.0,\n      0.0,\n      6.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.0,\n      1.0,\n      0.0\n     ],\n     [\n      10.0,\n      9.0,\n      0.0\n     ],\n     [\n      27.0,\n      17.0,\n      0.0\n     ],\n     [\n      29.0,\n      2.0,\n      0.0\n     ],\n     [\n      29.0,\n      0.0,\n      0.0\n     ],\n     [\n      28.5,\n      0.0,\n      1.0\n     ],\n     [\n      28.0,\n      0.0,\n      0.0\n     ],\n     [\n      27.5,\n      1.0,\n      3.0\n     ],\n     [\n      22.0,\n      0.0,\n      4.0\n     ],\n     [\n      20.5,\n      0.0,\n      2.0\n     ],\n     [\n      17.0,\n      0.0,\n      3.0\n     ],\n     [\n      16.0,\n      0.0,\n      1.0\n     ],\n     [\n      14.0,\n      0.0,\n      2.0\n     ],\n     [\n      6.0,\n      0.0,\n      8.0\n     ],\n     [\n      0.0,\n      0.0,\n      6.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.0,\n      1.0,\n      0.0\n     ],\n     [\n      6.0,\n      5.0,\n      0.0\n     ],\n     [\n      20.0,\n      14.0,\n      0.0\n     ],\n     [\n      21.0,\n      1.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      0.0\n     ],\n     [\n      19.0,\n      0.0,\n      2.0\n     ],\n     [\n      18.0,\n      0.0,\n      1.0\n     ],\n     [\n      16.0,\n      0.0,\n      2.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      14.0,\n      0.0,\n      2.0\n     ],\n     [\n      6.0,\n      0.0,\n      8.0\n     ],\n     [\n      0.0,\n      0.0,\n      6.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ]\n   ],\n   "poi_ids": [\n    "relation/1517647",\n    "relation/5559920",\n    "relation/7006925",\n    "relation/7763757",\n    "relation/8714888",\n    "relation/8718673",\n    "relation/8722473",\n    "relation/11385219",\n    "relation/12085584",\n    "relation/15692038",\n    "relation/15692039",\n    "relation/15692040",\n    "way/28828645",\n    "way/28836552",\n    "way/30426860",\n    "way/31019687",\n    "way/40999582",\n    "way/44661134",\n    "way/53192636",\n    "way/53192653",\n    "way/90265767",\n    "way/104105393",\n    "way/142235990",\n    "way/191035368",\n    "way/374066823",\n    "way/531331536",\n    "way/611160381",\n    "way/881413679",\n    "way/881413681",\n    "way/881677302",\n    "way/1265302069",\n    "node/526557513",\n    "node/588155402",\n    "node/588155406",\n    "node/926954558",\n    "node/967493362",\n    "node/967493367",\n    "node/967493368",\n    "node/1326067090",\n    "node/1335572334",\n    "node/1541601266",\n    "node/1913319856",\n    "node/2016194715",\n    "node/2030668573",\n    "node/2035584973",\n    "node/2035605211",\n    "node/2035605212",\n    "node/2108839041",\n    "node/2310477769",\n    "node/2322611721",\n    "node/3129214757",\n    "node/3671706276",\n    "node/3723061611",\n    "node/4147034394",\n    "node/4320667358",\n    "node/4397565291",\n    "node/4682250506",\n    "node/4682250507",\n    "node/4682250508",\n    "node/4682250509",\n    "node/4682250510",\n    "node/4682255627",\n    "node/4682258477",\n    "node/4682577735",\n    "node/4684245444",\n    "node/4684248495",\n    "node/4684267584",\n    "node/4684270761",\n    "node/4684271177",\n    "node/4762121977",\n    "node/4826742785",\n    "node/4972313160",\n    "node/4972313161",\n    "node/4972313521",\n    "node/5263709121",\n    "node/6442444928",\n    "node/6569581176",\n    "node/6653891578",\n    "node/6656080139",\n    "node/7330002514",\n    "node/7507064190",\n    "node/7832400978",\n    "node/7999018920",\n    "node/8035236212",\n    "node/8065673628",\n    "node/8149143298",\n    "node/8237603239",\n    "node/8313879273",\n    "node/8370846287",\n    "node/8370846303",\n    "node/8576451514",\n    "node/8684306021",\n    "node/8839218897",\n    "node/8839218924",\n    "node/9612450572",\n    "node/9612451748",\n    "node/9612452413",\n    "node/9612453645",\n    "node/9718273484",\n    "node/9718273512",\n    "node/9718273898",\n    "node/9718274226",\n    "node/9718274361",\n    "node/9718274972",\n    "node/10090569961",\n    "node/10657572425",\n    "node/10669995282",\n    "node/10761115952",\n    "node/11784697014",\n    "node/11931688185",\n    "node/12096755837",\n    "node/12168659537"\n   ]\n  },\n  "1": {\n   "buildings": 13,\n   "organisations": 36,\n   "tagged_hours": 19,\n   "parsed_hours": 18,\n   "hour_signals": [\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.5,\n      1.0,\n      0.0\n     ],\n     [\n      7.5,\n      7.0,\n      0.0\n     ],\n     [\n      16.0,\n      8.0,\n      0.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      12.5,\n      0.0,\n      4.0\n     ],\n     [\n      11.0,\n      0.0,\n      1.0\n     ],\n     [\n      7.0,\n      0.0,\n      4.0\n     ],\n     [\n      3.0,\n      0.0,\n      4.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.5,\n      1.0,\n      0.0\n     ],\n     [\n      8.5,\n      8.0,\n      0.0\n     ],\n     [\n      17.0,\n      8.0,\n      0.0\n     ],\n     [\n      17.0,\n      0.0,\n      0.0\n     ],\n     [\n      17.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.5,\n      0.0,\n      1.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      18.0,\n      2.0,\n      0.0\n     ],\n     [\n      18.0,\n      0.0,\n      0.0\n     ],\n     [\n      17.5,\n      0.0,\n      1.0\n     ],\n     [\n      14.5,\n      0.0,\n      3.0\n     ],\n     [\n      12.0,\n      0.0,\n      2.0\n     ],\n     [\n      7.0,\n      0.0,\n      5.0\n     ],\n     [\n      3.0,\n      0.0,\n      4.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.5,\n      1.0,\n      0.0\n     ],\n     [\n      8.5,\n      8.0,\n      0.0\n     ],\n     [\n      17.0,\n      8.0,\n      0.0\n     ],\n     [\n      17.0,\n      0.0,\n      0.0\n     ],\n     [\n      17.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.5,\n      0.0,\n      1.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      17.0,\n      1.0,\n      0.0\n     ],\n     [\n      17.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.5,\n      0.0,\n      1.0\n     ],\n     [\n      12.5,\n      0.0,\n      4.0\n     ],\n     [\n      11.0,\n      0.0,\n      1.0\n     ],\n     [\n      7.0,\n      0.0,\n      4.0\n     ],\n     [\n      3.0,\n      0.0,\n      4.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.5,\n      2.0,\n      0.0\n     ],\n     [\n      9.5,\n      8.0,\n      0.0\n     ],\n     [\n      18.0,\n      8.0,\n      0.0\n     ],\n     [\n      18.0,\n      0.0,\n      0.0\n     ],\n     [\n      17.0,\n      0.0,\n      1.0\n     ],\n     [\n      16.5,\n      0.0,\n      1.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      17.0,\n      1.0,\n      0.0\n     ],\n     [\n      17.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.5,\n      0.0,\n      1.0\n     ],\n     [\n      13.5,\n      0.0,\n      3.0\n     ],\n     [\n      12.0,\n      0.0,\n      1.0\n     ],\n     [\n      7.0,\n      0.0,\n      5.0\n     ],\n     [\n      3.0,\n      0.0,\n      4.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.5,\n      1.0,\n      0.0\n     ],\n     [\n      8.5,\n      8.0,\n      0.0\n     ],\n     [\n      17.0,\n      8.0,\n      0.0\n     ],\n     [\n      17.0,\n      0.0,\n      0.0\n     ],\n     [\n      17.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.5,\n      0.0,\n      1.0\n     ],\n     [\n      16.0,\n      0.0,\n      0.0\n     ],\n     [\n      17.0,\n      1.0,\n      0.0\n     ],\n     [\n      16.75,\n      0.0,\n      1.0\n     ],\n     [\n      15.5,\n      0.0,\n      1.0\n     ],\n     [\n      12.5,\n      0.0,\n      3.0\n     ],\n     [\n      10.0,\n      0.0,\n      2.0\n     ],\n     [\n      7.0,\n      0.0,\n      3.0\n     ],\n     [\n      3.0,\n      0.0,\n      4.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      4.5,\n      5.0,\n      0.0\n     ],\n     [\n      13.0,\n      8.0,\n      0.0\n     ],\n     [\n      13.0,\n      0.0,\n      0.0\n     ],\n     [\n      13.0,\n      0.0,\n      0.0\n     ],\n     [\n      12.5,\n      0.0,\n      1.0\n     ],\n     [\n      12.0,\n      0.0,\n      0.0\n     ],\n     [\n      11.0,\n      1.0,\n      2.0\n     ],\n     [\n      11.0,\n      0.0,\n      0.0\n     ],\n     [\n      10.5,\n      0.0,\n      1.0\n     ],\n     [\n      7.0,\n      0.0,\n      3.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      1.0\n     ],\n     [\n      3.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      2.0,\n      0.0\n     ],\n     [\n      10.0,\n      8.0,\n      0.0\n     ],\n     [\n      10.0,\n      0.0,\n      0.0\n     ],\n     [\n      10.0,\n      0.0,\n      0.0\n     ],\n     [\n      10.0,\n      0.0,\n      0.0\n     ],\n     [\n      10.0,\n      0.0,\n      0.0\n     ],\n     [\n      9.0,\n      0.0,\n      1.0\n     ],\n     [\n      8.0,\n      0.0,\n      1.0\n     ],\n     [\n      7.0,\n      0.0,\n      1.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      1.0\n     ],\n     [\n      3.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ]\n   ],\n   "poi_ids": [\n    "way/24913831",\n    "way/25418521",\n    "way/29325295",\n    "way/29325346",\n    "way/29325348",\n    "way/29325350",\n    "way/36743674",\n    "way/36743698",\n    "way/37790290",\n    "way/56031018",\n    "way/81107236",\n    "way/83178437",\n    "way/154331579",\n    "way/154331582",\n    "way/154331583",\n    "way/157210017",\n    "way/258640275",\n    "way/299381445",\n    "node/964667536",\n    "node/2919125134",\n    "node/3000681428",\n    "node/3428369999",\n    "node/4018351665",\n    "node/4528342197",\n    "node/4682522076",\n    "node/4682522077",\n    "node/4739741991",\n    "node/4836730301",\n    "node/5098363133",\n    "node/5134380906",\n    "node/5259671023",\n    "node/5382892023",\n    "node/5698890480",\n    "node/6695345813",\n    "node/6746284537",\n    "node/7140976685",\n    "node/7348263685",\n    "node/8065721270",\n    "node/8839218944",\n    "node/9182822865",\n    "node/9612452931",\n    "node/9612453920",\n    "node/9718274891",\n    "node/10209727281",\n    "node/11007197240",\n    "node/11178629622",\n    "node/11180764416",\n    "node/11545721397",\n    "node/12012237435"\n   ]\n  },\n  "12": {\n   "buildings": 218,\n   "organisations": 201,\n   "tagged_hours": 98,\n   "parsed_hours": 98,\n   "hour_signals": [\n    [\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      13.0,\n      11.0,\n      0.0\n     ],\n     [\n      53.25,\n      41.0,\n      0.0\n     ],\n     [\n      91.5,\n      37.0,\n      0.0\n     ],\n     [\n      95.0,\n      3.0,\n      0.0\n     ],\n     [\n      93.25,\n      4.0,\n      4.0\n     ],\n     [\n      94.75,\n      1.0,\n      1.0\n     ],\n     [\n      96.0,\n      1.0,\n      0.0\n     ],\n     [\n      96.0,\n      0.0,\n      0.0\n     ],\n     [\n      96.0,\n      0.0,\n      0.0\n     ],\n     [\n      88.75,\n      0.0,\n      8.0\n     ],\n     [\n      63.0,\n      0.0,\n      25.0\n     ],\n     [\n      48.0,\n      0.0,\n      16.0\n     ],\n     [\n      33.0,\n      0.0,\n      14.0\n     ],\n     [\n      16.75,\n      0.0,\n      17.0\n     ],\n     [\n      5.0,\n      0.0,\n      11.0\n     ],\n     [\n      3.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      13.0,\n      11.0,\n      0.0\n     ],\n     [\n      56.25,\n      44.0,\n      0.0\n     ],\n     [\n      94.5,\n      37.0,\n      0.0\n     ],\n     [\n      97.0,\n      2.0,\n      0.0\n     ],\n     [\n      95.25,\n      4.0,\n      4.0\n     ],\n     [\n      96.75,\n      1.0,\n      1.0\n     ],\n     [\n      97.0,\n      1.0,\n      1.0\n     ],\n     [\n      98.0,\n      1.0,\n      0.0\n     ],\n     [\n      98.0,\n      0.0,\n      0.0\n     ],\n     [\n      90.75,\n      0.0,\n      8.0\n     ],\n     [\n      64.0,\n      0.0,\n      26.0\n     ],\n     [\n      49.0,\n      0.0,\n      16.0\n     ],\n     [\n      33.0,\n      0.0,\n      15.0\n     ],\n     [\n      16.75,\n      0.0,\n      17.0\n     ],\n     [\n      5.0,\n      0.0,\n      11.0\n     ],\n     [\n      3.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      13.0,\n      11.0,\n      0.0\n     ],\n     [\n      56.25,\n      44.0,\n      0.0\n     ],\n     [\n      94.5,\n      37.0,\n      0.0\n     ],\n     [\n      97.0,\n      2.0,\n      0.0\n     ],\n     [\n      95.25,\n      4.0,\n      4.0\n     ],\n     [\n      96.75,\n      1.0,\n      1.0\n     ],\n     [\n      97.0,\n      1.0,\n      1.0\n     ],\n     [\n      98.0,\n      1.0,\n      0.0\n     ],\n     [\n      98.0,\n      0.0,\n      0.0\n     ],\n     [\n      90.75,\n      0.0,\n      8.0\n     ],\n     [\n      63.0,\n      0.0,\n      27.0\n     ],\n     [\n      48.0,\n      0.0,\n      16.0\n     ],\n     [\n      33.0,\n      0.0,\n      14.0\n     ],\n     [\n      16.75,\n      0.0,\n      17.0\n     ],\n     [\n      5.0,\n      0.0,\n      11.0\n     ],\n     [\n      3.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      13.0,\n      11.0,\n      0.0\n     ],\n     [\n      56.25,\n      44.0,\n      0.0\n     ],\n     [\n      94.5,\n      37.0,\n      0.0\n     ],\n     [\n      97.0,\n      2.0,\n      0.0\n     ],\n     [\n      95.25,\n      4.0,\n      4.0\n     ],\n     [\n      96.75,\n      1.0,\n      1.0\n     ],\n     [\n      97.0,\n      1.0,\n      1.0\n     ],\n     [\n      98.0,\n      1.0,\n      0.0\n     ],\n     [\n      98.0,\n      0.0,\n      0.0\n     ],\n     [\n      90.75,\n      0.0,\n      8.0\n     ],\n     [\n      64.0,\n      0.0,\n      26.0\n     ],\n     [\n      49.0,\n      0.0,\n      16.0\n     ],\n     [\n      33.0,\n      0.0,\n      15.0\n     ],\n     [\n      16.75,\n      0.0,\n      17.0\n     ],\n     [\n      5.0,\n      0.0,\n      11.0\n     ],\n     [\n      3.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      13.0,\n      11.0,\n      0.0\n     ],\n     [\n      56.25,\n      44.0,\n      0.0\n     ],\n     [\n      94.5,\n      37.0,\n      0.0\n     ],\n     [\n      97.0,\n      2.0,\n      0.0\n     ],\n     [\n      95.25,\n      4.0,\n      4.0\n     ],\n     [\n      96.75,\n      1.0,\n      1.0\n     ],\n     [\n      97.0,\n      1.0,\n      1.0\n     ],\n     [\n      97.5,\n      1.0,\n      2.0\n     ],\n     [\n      91.25,\n      0.0,\n      7.0\n     ],\n     [\n      82.5,\n      0.0,\n      7.0\n     ],\n     [\n      63.0,\n      0.0,\n      19.0\n     ],\n     [\n      45.0,\n      0.0,\n      19.0\n     ],\n     [\n      33.0,\n      0.0,\n      11.0\n     ],\n     [\n      16.75,\n      0.0,\n      17.0\n     ],\n     [\n      5.0,\n      0.0,\n      11.0\n     ],\n     [\n      3.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      4.0,\n      1.0,\n      0.0\n     ],\n     [\n      31.0,\n      27.0,\n      0.0\n     ],\n     [\n      56.5,\n      26.0,\n      0.0\n     ],\n     [\n      59.0,\n      2.0,\n      0.0\n     ],\n     [\n      60.0,\n      1.0,\n      0.0\n     ],\n     [\n      60.0,\n      0.0,\n      0.0\n     ],\n     [\n      59.0,\n      0.0,\n      1.0\n     ],\n     [\n      55.25,\n      1.0,\n      7.0\n     ],\n     [\n      52.0,\n      0.0,\n      1.0\n     ],\n     [\n      51.0,\n      0.0,\n      1.0\n     ],\n     [\n      43.0,\n      0.0,\n      8.0\n     ],\n     [\n      37.5,\n      0.0,\n      6.0\n     ],\n     [\n      32.0,\n      0.0,\n      5.0\n     ],\n     [\n      15.75,\n      0.0,\n      17.0\n     ],\n     [\n      5.0,\n      0.0,\n      10.0\n     ],\n     [\n      3.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      0.0,\n      0.0\n     ],\n     [\n      4.0,\n      1.0,\n      0.0\n     ],\n     [\n      24.0,\n      20.0,\n      0.0\n     ],\n     [\n      45.0,\n      21.0,\n      0.0\n     ],\n     [\n      46.0,\n      1.0,\n      0.0\n     ],\n     [\n      47.0,\n      1.0,\n      0.0\n     ],\n     [\n      47.0,\n      0.0,\n      0.0\n     ],\n     [\n      47.0,\n      0.0,\n      0.0\n     ],\n     [\n      47.0,\n      0.0,\n      0.0\n     ],\n     [\n      47.0,\n      0.0,\n      0.0\n     ],\n     [\n      45.0,\n      0.0,\n      2.0\n     ],\n     [\n      41.0,\n      0.0,\n      4.0\n     ],\n     [\n      37.0,\n      0.0,\n      4.0\n     ],\n     [\n      32.0,\n      0.0,\n      5.0\n     ],\n     [\n      15.75,\n      0.0,\n      17.0\n     ],\n     [\n      5.0,\n      0.0,\n      10.0\n     ],\n     [\n      3.0,\n      0.0,\n      2.0\n     ]\n    ]\n   ],\n   "poi_ids": [\n    "relation/2746582",\n    "relation/3905188",\n    "relation/7326540",\n    "relation/7328019",\n    "relation/8463134",\n    "relation/8685001",\n    "relation/8878388",\n    "relation/11292828",\n    "relation/11647935",\n    "relation/11840862",\n    "relation/12681154",\n    "relation/12681155",\n    "relation/12681158",\n    "relation/15043796",\n    "relation/15043807",\n    "relation/15043808",\n    "relation/15043809",\n    "relation/15257667",\n    "relation/15268234",\n    "relation/15268237",\n    "relation/15268248",\n    "relation/16190693",\n    "relation/17329527",\n    "relation/18372355",\n    "way/23372126",\n    "way/23414592",\n    "way/23768262",\n    "way/23769066",\n    "way/27555530",\n    "way/29544221",\n    "way/29544297",\n    "way/29544371",\n    "way/29544372",\n    "way/29544871",\n    "way/30185148",\n    "way/30185173",\n    "way/32244153",\n    "way/33456225",\n    "way/35547549",\n    "way/35547550",\n    "way/35547552",\n    "way/42987779",\n    "way/44581783",\n    "way/45831773",\n    "way/45831961",\n    "way/45831963",\n    "way/45832828",\n    "way/47572010",\n    "way/48346402",\n    "way/48830577",\n    "way/48892066",\n    "way/48892897",\n    "way/48894731",\n    "way/48898734",\n    "way/48898741",\n    "way/49069615",\n    "way/49140329",\n    "way/49494827",\n    "way/49494831",\n    "way/49494841",\n    "way/49677877",\n    "way/49681032",\n    "way/49906341",\n    "way/50072166",\n    "way/50123757",\n    "way/50218016",\n    "way/50548832",\n    "way/51303552",\n    "way/52462299",\n    "way/52462357",\n    "way/52463968",\n    "way/52481301",\n    "way/52521807",\n    "way/52521845",\n    "way/52521853",\n    "way/53205942",\n    "way/53205947",\n    "way/53521337",\n    "way/53591175",\n    "way/53650437",\n    "way/56121187",\n    "way/56629809",\n    "way/56629815",\n    "way/62169335",\n    "way/62169337",\n    "way/62209257",\n    "way/64266176",\n    "way/67251887",\n    "way/67251891",\n    "way/67251893",\n    "way/67522506",\n    "way/83040952",\n    "way/83040956",\n    "way/83113147",\n    "way/84890806",\n    "way/93473895",\n    "way/95749464",\n    "way/96266243",\n    "way/98731486",\n    "way/98731488",\n    "way/100025948",\n    "way/100107174",\n    "way/104112615",\n    "way/107943503",\n    "way/107957959",\n    "way/107959081",\n    "way/108581515",\n    "way/108584080",\n    "way/108593197",\n    "way/109067903",\n    "way/109067968",\n    "way/109209561",\n    "way/109209565",\n    "way/111648714",\n    "way/114995690",\n    "way/115827004",\n    "way/115827008",\n    "way/115827013",\n    "way/115827022",\n    "way/130228970",\n    "way/130228976",\n    "way/130537916",\n    "way/130537917",\n    "way/130995831",\n    "way/131474105",\n    "way/131499277",\n    "way/131499278",\n    "way/131499293",\n    "way/131499297",\n    "way/131499300",\n    "way/131499304",\n    "way/131499307",\n    "way/131499308",\n    "way/131499309",\n    "way/131589346",\n    "way/131589347",\n    "way/131589415",\n    "way/131655168",\n    "way/131655174",\n    "way/131655566",\n    "way/131655567",\n    "way/132365077",\n    "way/132365078",\n    "way/132365079",\n    "way/132365084",\n    "way/133000155",\n    "way/133000156",\n    "way/133000160",\n    "way/133392897",\n    "way/133636968",\n    "way/144236570",\n    "way/149757436",\n    "way/150964762",\n    "way/151403557",\n    "way/160412242",\n    "way/162139510",\n    "way/168357975",\n    "way/168840465",\n    "way/168840481",\n    "way/171878914",\n    "way/181513827",\n    "way/189921066",\n    "way/190783190",\n    "way/190783191",\n    "way/204059732",\n    "way/209181175",\n    "way/232164803",\n    "way/241751693",\n    "way/241751695",\n    "way/241751701",\n    "way/242682007",\n    "way/242682009",\n    "way/242682010",\n    "way/242682015",\n    "way/242682017",\n    "way/249167733",\n    "way/249167735",\n    "way/270788762",\n    "way/281776082",\n    "way/299995582",\n    "way/305258203",\n    "way/313608561",\n    "way/313611298",\n    "way/316793853",\n    "way/317668226",\n    "way/337151437",\n    "way/340976769",\n    "way/360267360",\n    "way/367420412",\n    "way/384975606",\n    "way/386343964",\n    "way/386343965",\n    "way/386343971",\n    "way/387749878",\n    "way/428930473",\n    "way/443767519",\n    "way/443767522",\n    "way/444437262",\n    "way/490319602",\n    "way/499939694",\n    "way/521295436",\n    "way/548947686",\n    "way/548947690",\n    "way/555604810",\n    "way/555604815",\n    "way/555604816",\n    "way/555604818",\n    "way/616057793",\n    "way/640129100",\n    "way/645359441",\n    "way/645359444",\n    "way/682991438",\n    "way/692561958",\n    "way/692561959",\n    "way/823553948",\n    "way/825189165",\n    "way/849480841",\n    "way/849480857",\n    "way/850793246",\n    "way/850793250",\n    "way/851733239",\n    "way/938285122",\n    "way/938285130",\n    "way/949165966",\n    "way/949165968",\n    "way/1061534276",\n    "way/1061534277",\n    "way/1064649498",\n    "way/1081797214",\n    "way/1218469078",\n    "way/1218484711",\n    "way/1218484712",\n    "way/1218484713",\n    "way/1218484714",\n    "way/1221919097",\n    "way/1257472131",\n    "node/380729527",\n    "node/775957750",\n    "node/914269753",\n    "node/921902301",\n    "node/1313514295",\n    "node/1408191312",\n    "node/1437413453",\n    "node/1437413454",\n    "node/1490642882",\n    "node/1666923550",\n    "node/1778780476",\n    "node/1781695266",\n    "node/1850660698",\n    "node/2006884253",\n    "node/2102020011",\n    "node/2104339456",\n    "node/2355373381",\n    "node/2632370131",\n    "node/2633424055",\n    "node/2634802603",\n    "node/2639560378",\n    "node/2651488099",\n    "node/2815257266",\n    "node/2855188627",\n    "node/3149355706",\n    "node/3161602788",\n    "node/3385561248",\n    "node/3478996650",\n    "node/3481829140",\n    "node/3618470511",\n    "node/3737336775",\n    "node/3750376527",\n    "node/3805697945",\n    "node/3933846333",\n    "node/3933853620",\n    "node/3933853621",\n    "node/3956982665",\n    "node/3956988837",\n    "node/3960412788",\n    "node/4071266491",\n    "node/4111206589",\n    "node/4114718390",\n    "node/4127984319",\n    "node/4142818367",\n    "node/4142818373",\n    "node/4142818374",\n    "node/4142818375",\n    "node/4207416694",\n    "node/4207471935",\n    "node/4234177828",\n    "node/4235874143",\n    "node/4235874147",\n    "node/4235874148",\n    "node/4298602377",\n    "node/4332965790",\n    "node/4367972811",\n    "node/4508520531",\n    "node/4508520532",\n    "node/4632210769",\n    "node/4682548443",\n    "node/4682583708",\n    "node/4684147661",\n    "node/4684293167",\n    "node/4684303401",\n    "node/4771663558",\n    "node/4934370323",\n    "node/4934404121",\n    "node/4964372676",\n    "node/5032420921",\n    "node/5085589186",\n    "node/5086138176",\n    "node/5098756511",\n    "node/5098756512",\n    "node/5117994924",\n    "node/5199351203",\n    "node/5247160683",\n    "node/5249950235",\n    "node/5257252697",\n    "node/5300666423",\n    "node/5308193121",\n    "node/5337863422",\n    "node/5382888339",\n    "node/5467471775",\n    "node/5541234491",\n    "node/5604093257",\n    "node/5645767480",\n    "node/5715235510",\n    "node/5823745053",\n    "node/5837982052",\n    "node/5859574885",\n    "node/5961332787",\n    "node/6017492809",\n    "node/6042611514",\n    "node/6065468887",\n    "node/6073883028",\n    "node/6075668596",\n    "node/6112327787",\n    "node/6230822078",\n    "node/6274869586",\n    "node/6280581634",\n    "node/6432064996",\n    "node/6477555485",\n    "node/6492838405",\n    "node/6528278385",\n    "node/6738633185",\n    "node/6790080114",\n    "node/6910263729",\n    "node/6965710185",\n    "node/7001149797",\n    "node/7046400331",\n    "node/7061642788",\n    "node/7318731403",\n    "node/7320525077",\n    "node/7507064192",\n    "node/7620684102",\n    "node/7620684104",\n    "node/7682601860",\n    "node/7689311331",\n    "node/7780981883",\n    "node/7999017524",\n    "node/8146186057",\n    "node/8194406804",\n    "node/8370846289",\n    "node/8390522305",\n    "node/8460684804",\n    "node/8473641179",\n    "node/8521187094",\n    "node/8521187111",\n    "node/8824605017",\n    "node/8829912979",\n    "node/8829940033",\n    "node/8829940068",\n    "node/8829940079",\n    "node/8836849696",\n    "node/8839215313",\n    "node/9049315104",\n    "node/9049315105",\n    "node/9049315107",\n    "node/9049315108",\n    "node/9510795174",\n    "node/9529138097",\n    "node/9612450265",\n    "node/9612452143",\n    "node/9612452169",\n    "node/9612452424",\n    "node/9612452427",\n    "node/9612453615",\n    "node/9708644526",\n    "node/9709230773",\n    "node/9718273772",\n    "node/9718274977",\n    "node/9805728658",\n    "node/9805728663",\n    "node/9828233722",\n    "node/10240176163",\n    "node/10648656384",\n    "node/10652913823",\n    "node/10652913848",\n    "node/10655293054",\n    "node/10722898461",\n    "node/10787918260",\n    "node/10828724807",\n    "node/10969632457",\n    "node/10969632458",\n    "node/10969632460",\n    "node/10969632462",\n    "node/10976288017",\n    "node/11085084468",\n    "node/11118842262",\n    "node/11166935325",\n    "node/11304357745",\n    "node/11371298867",\n    "node/11459367118",\n    "node/11459367123",\n    "node/11608159138",\n    "node/11608159141",\n    "node/11905837216",\n    "node/11975242304",\n    "node/11991903449",\n    "node/12026320354",\n    "node/12096765851",\n    "node/12106517544",\n    "node/12117405113"\n   ]\n  },\n  "11": {\n   "buildings": 164,\n   "organisations": 155,\n   "tagged_hours": 75,\n   "parsed_hours": 75,\n   "hour_signals": [\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      7.0,\n      8.0,\n      0.0\n     ],\n     [\n      42.0,\n      35.0,\n      0.0\n     ],\n     [\n      70.0,\n      27.0,\n      0.0\n     ],\n     [\n      73.0,\n      3.0,\n      0.0\n     ],\n     [\n      73.0,\n      1.0,\n      2.0\n     ],\n     [\n      72.0,\n      2.0,\n      2.0\n     ],\n     [\n      73.75,\n      2.0,\n      0.0\n     ],\n     [\n      74.0,\n      0.0,\n      0.0\n     ],\n     [\n      74.0,\n      0.0,\n      0.0\n     ],\n     [\n      68.75,\n      0.0,\n      6.0\n     ],\n     [\n      48.0,\n      0.0,\n      20.0\n     ],\n     [\n      38.0,\n      0.0,\n      10.0\n     ],\n     [\n      28.0,\n      0.0,\n      10.0\n     ],\n     [\n      8.75,\n      0.0,\n      20.0\n     ],\n     [\n      1.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      1.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      7.0,\n      8.0,\n      0.0\n     ],\n     [\n      43.0,\n      36.0,\n      0.0\n     ],\n     [\n      71.0,\n      27.0,\n      0.0\n     ],\n     [\n      74.0,\n      3.0,\n      0.0\n     ],\n     [\n      74.0,\n      1.0,\n      2.0\n     ],\n     [\n      73.0,\n      2.0,\n      2.0\n     ],\n     [\n      73.75,\n      2.0,\n      1.0\n     ],\n     [\n      75.0,\n      1.0,\n      0.0\n     ],\n     [\n      75.0,\n      0.0,\n      0.0\n     ],\n     [\n      69.75,\n      0.0,\n      6.0\n     ],\n     [\n      49.0,\n      0.0,\n      20.0\n     ],\n     [\n      39.0,\n      0.0,\n      10.0\n     ],\n     [\n      28.0,\n      0.0,\n      11.0\n     ],\n     [\n      8.75,\n      0.0,\n      20.0\n     ],\n     [\n      1.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      1.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      7.0,\n      8.0,\n      0.0\n     ],\n     [\n      43.0,\n      36.0,\n      0.0\n     ],\n     [\n      71.0,\n      27.0,\n      0.0\n     ],\n     [\n      74.0,\n      3.0,\n      0.0\n     ],\n     [\n      74.0,\n      1.0,\n      2.0\n     ],\n     [\n      73.0,\n      2.0,\n      2.0\n     ],\n     [\n      73.75,\n      2.0,\n      1.0\n     ],\n     [\n      75.0,\n      1.0,\n      0.0\n     ],\n     [\n      75.0,\n      0.0,\n      0.0\n     ],\n     [\n      69.75,\n      0.0,\n      6.0\n     ],\n     [\n      49.0,\n      0.0,\n      20.0\n     ],\n     [\n      39.0,\n      0.0,\n      10.0\n     ],\n     [\n      28.0,\n      0.0,\n      11.0\n     ],\n     [\n      8.75,\n      0.0,\n      20.0\n     ],\n     [\n      1.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      1.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      7.0,\n      0.0\n     ],\n     [\n      42.0,\n      36.0,\n      0.0\n     ],\n     [\n      70.0,\n      27.0,\n      0.0\n     ],\n     [\n      73.0,\n      3.0,\n      0.0\n     ],\n     [\n      73.5,\n      1.0,\n      1.0\n     ],\n     [\n      72.25,\n      1.0,\n      2.0\n     ],\n     [\n      72.75,\n      2.0,\n      1.0\n     ],\n     [\n      74.0,\n      1.0,\n      0.0\n     ],\n     [\n      74.0,\n      0.0,\n      0.0\n     ],\n     [\n      69.75,\n      0.0,\n      5.0\n     ],\n     [\n      49.0,\n      0.0,\n      20.0\n     ],\n     [\n      39.0,\n      0.0,\n      10.0\n     ],\n     [\n      28.0,\n      0.0,\n      11.0\n     ],\n     [\n      8.75,\n      0.0,\n      20.0\n     ],\n     [\n      1.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      1.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      7.0,\n      8.0,\n      0.0\n     ],\n     [\n      43.0,\n      36.0,\n      0.0\n     ],\n     [\n      71.0,\n      27.0,\n      0.0\n     ],\n     [\n      74.0,\n      3.0,\n      0.0\n     ],\n     [\n      74.0,\n      1.0,\n      2.0\n     ],\n     [\n      73.0,\n      2.0,\n      2.0\n     ],\n     [\n      73.75,\n      2.0,\n      1.0\n     ],\n     [\n      74.25,\n      1.0,\n      3.0\n     ],\n     [\n      68.75,\n      0.0,\n      4.0\n     ],\n     [\n      64.5,\n      0.0,\n      4.0\n     ],\n     [\n      48.0,\n      0.0,\n      16.0\n     ],\n     [\n      36.0,\n      0.0,\n      12.0\n     ],\n     [\n      28.0,\n      0.0,\n      8.0\n     ],\n     [\n      8.75,\n      0.0,\n      20.0\n     ],\n     [\n      1.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      1.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      2.0,\n      0.0\n     ],\n     [\n      22.0,\n      20.0,\n      0.0\n     ],\n     [\n      46.0,\n      24.0,\n      0.0\n     ],\n     [\n      49.0,\n      3.0,\n      0.0\n     ],\n     [\n      50.0,\n      1.0,\n      0.0\n     ],\n     [\n      50.0,\n      0.0,\n      0.0\n     ],\n     [\n      49.0,\n      0.0,\n      1.0\n     ],\n     [\n      47.0,\n      1.0,\n      3.0\n     ],\n     [\n      43.0,\n      0.0,\n      4.0\n     ],\n     [\n      39.0,\n      0.0,\n      4.0\n     ],\n     [\n      35.0,\n      0.0,\n      4.0\n     ],\n     [\n      32.0,\n      0.0,\n      3.0\n     ],\n     [\n      25.0,\n      0.0,\n      7.0\n     ],\n     [\n      8.75,\n      0.0,\n      17.0\n     ],\n     [\n      1.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      1.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      2.0,\n      0.0\n     ],\n     [\n      20.0,\n      18.0,\n      0.0\n     ],\n     [\n      41.0,\n      21.0,\n      0.0\n     ],\n     [\n      42.0,\n      1.0,\n      0.0\n     ],\n     [\n      43.0,\n      1.0,\n      0.0\n     ],\n     [\n      43.0,\n      0.0,\n      0.0\n     ],\n     [\n      43.0,\n      0.0,\n      0.0\n     ],\n     [\n      43.0,\n      0.0,\n      0.0\n     ],\n     [\n      41.0,\n      0.0,\n      2.0\n     ],\n     [\n      37.0,\n      0.0,\n      4.0\n     ],\n     [\n      35.0,\n      0.0,\n      2.0\n     ],\n     [\n      32.0,\n      0.0,\n      3.0\n     ],\n     [\n      25.0,\n      0.0,\n      7.0\n     ],\n     [\n      8.75,\n      0.0,\n      17.0\n     ],\n     [\n      1.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      1.0\n     ]\n    ]\n   ],\n   "poi_ids": [\n    "relation/1204804",\n    "relation/1204805",\n    "relation/5559920",\n    "relation/7006925",\n    "relation/7326540",\n    "relation/7328019",\n    "relation/8718673",\n    "relation/11679775",\n    "relation/12085584",\n    "relation/12681154",\n    "relation/12681155",\n    "relation/12681158",\n    "relation/14239439",\n    "relation/14930869",\n    "relation/14930872",\n    "relation/14930873",\n    "relation/15043796",\n    "relation/15043807",\n    "relation/15043808",\n    "relation/15043809",\n    "relation/15080647",\n    "relation/15257667",\n    "relation/16109093",\n    "relation/16109097",\n    "relation/16109098",\n    "relation/16109099",\n    "relation/16109101",\n    "relation/16421395",\n    "relation/16421396",\n    "relation/16421397",\n    "relation/17950695",\n    "relation/17950696",\n    "relation/17950697",\n    "way/23768262",\n    "way/23769066",\n    "way/28828645",\n    "way/28860303",\n    "way/29854197",\n    "way/29854216",\n    "way/30426860",\n    "way/40999582",\n    "way/44581783",\n    "way/45831773",\n    "way/45831961",\n    "way/45831963",\n    "way/48582126",\n    "way/48582143",\n    "way/48582172",\n    "way/48582174",\n    "way/48582176",\n    "way/48582179",\n    "way/48582184",\n    "way/48582185",\n    "way/48582186",\n    "way/48582193",\n    "way/48830577",\n    "way/49681032",\n    "way/50072166",\n    "way/50123757",\n    "way/52477085",\n    "way/52521807",\n    "way/52521845",\n    "way/52521853",\n    "way/53205942",\n    "way/53205947",\n    "way/56629809",\n    "way/56629815",\n    "way/64266176",\n    "way/67522506",\n    "way/80664559",\n    "way/90265767",\n    "way/91209651",\n    "way/98731486",\n    "way/98731488",\n    "way/104112615",\n    "way/104122834",\n    "way/104708946",\n    "way/105658741",\n    "way/107673086",\n    "way/107943503",\n    "way/107957959",\n    "way/107959081",\n    "way/108581515",\n    "way/108584080",\n    "way/108593197",\n    "way/109067903",\n    "way/109067968",\n    "way/109209561",\n    "way/109209565",\n    "way/124365454",\n    "way/124365460",\n    "way/124365465",\n    "way/124365466",\n    "way/124365467",\n    "way/124365469",\n    "way/127632239",\n    "way/127632244",\n    "way/127632245",\n    "way/130537917",\n    "way/131589415",\n    "way/131655167",\n    "way/131655168",\n    "way/131655172",\n    "way/131655174",\n    "way/131655261",\n    "way/131655262",\n    "way/131655266",\n    "way/131655566",\n    "way/131655567",\n    "way/131833399",\n    "way/132365077",\n    "way/132365078",\n    "way/132365079",\n    "way/132365084",\n    "way/133392897",\n    "way/142235990",\n    "way/150134797",\n    "way/150824915",\n    "way/150824920",\n    "way/150824925",\n    "way/150824928",\n    "way/150824929",\n    "way/154363966",\n    "way/168840465",\n    "way/168840481",\n    "way/177657595",\n    "way/177657627",\n    "way/177913374",\n    "way/177913376",\n    "way/178163702",\n    "way/178199299",\n    "way/178199318",\n    "way/178199321",\n    "way/181513827",\n    "way/189921066",\n    "way/190783190",\n    "way/190783191",\n    "way/200610180",\n    "way/204059732",\n    "way/224343561",\n    "way/316793853",\n    "way/331353941",\n    "way/360267360",\n    "way/363128565",\n    "way/363128567",\n    "way/363128568",\n    "way/367420412",\n    "way/374066823",\n    "way/387617423",\n    "way/389448515",\n    "way/490319602",\n    "way/499939694",\n    "way/521295436",\n    "way/531331536",\n    "way/555999537",\n    "way/679418353",\n    "way/692561958",\n    "way/692561959",\n    "way/713796365",\n    "way/783454908",\n    "way/798376273",\n    "way/832137711",\n    "way/832137713",\n    "way/851733239",\n    "way/875297806",\n    "way/934038206",\n    "way/934038207",\n    "way/934038208",\n    "way/934038209",\n    "way/934038210",\n    "way/934038212",\n    "way/943013022",\n    "way/949165966",\n    "way/949165968",\n    "way/1218484711",\n    "way/1218484712",\n    "way/1218484713",\n    "way/1218484714",\n    "way/1220466087",\n    "way/1221919097",\n    "way/1265302069",\n    "node/914269753",\n    "node/921902301",\n    "node/1313514295",\n    "node/1437413454",\n    "node/1490642882",\n    "node/1530991489",\n    "node/1666923550",\n    "node/1712912514",\n    "node/1781695266",\n    "node/1850660698",\n    "node/2006884253",\n    "node/2102020011",\n    "node/2104339456",\n    "node/2105789042",\n    "node/3135016785",\n    "node/3424330621",\n    "node/3618470511",\n    "node/3754134881",\n    "node/3933846333",\n    "node/3933853620",\n    "node/3933853621",\n    "node/3956982665",\n    "node/3956988837",\n    "node/3960412788",\n    "node/4114718390",\n    "node/4127984319",\n    "node/4147034394",\n    "node/4203797291",\n    "node/4320667358",\n    "node/4393092690",\n    "node/4453027689",\n    "node/4508520531",\n    "node/4508520532",\n    "node/4540199361",\n    "node/4574264191",\n    "node/4580200734",\n    "node/4580223125",\n    "node/4632210769",\n    "node/4682504546",\n    "node/4682577735",\n    "node/4682583708",\n    "node/4682583709",\n    "node/4684267584",\n    "node/4684293167",\n    "node/4684303401",\n    "node/4684318336",\n    "node/4771663558",\n    "node/4964372676",\n    "node/4972313160",\n    "node/4972313161",\n    "node/4972313521",\n    "node/5086138176",\n    "node/5117994924",\n    "node/5199351203",\n    "node/5247160683",\n    "node/5249950235",\n    "node/5257252697",\n    "node/5263709121",\n    "node/5334710944",\n    "node/5337863422",\n    "node/5604093257",\n    "node/5715235510",\n    "node/5961332787",\n    "node/6076122186",\n    "node/6230822078",\n    "node/6432746001",\n    "node/6436000785",\n    "node/6442444928",\n    "node/6477555485",\n    "node/6482133285",\n    "node/6790080114",\n    "node/6910263729",\n    "node/7046400331",\n    "node/7221169271",\n    "node/7265739617",\n    "node/7265739618",\n    "node/7265739649",\n    "node/7265739650",\n    "node/7318731403",\n    "node/7320525077",\n    "node/7330002514",\n    "node/7832400978",\n    "node/7944140393",\n    "node/7948226512",\n    "node/7948226513",\n    "node/8035236212",\n    "node/8065673628",\n    "node/8107071421",\n    "node/8110613001",\n    "node/8112003634",\n    "node/8146853174",\n    "node/8146893188",\n    "node/8148301498",\n    "node/8194406804",\n    "node/8341433239",\n    "node/8370846287",\n    "node/8370846289",\n    "node/8370846303",\n    "node/8511850969",\n    "node/8521191402",\n    "node/8653270825",\n    "node/8684306021",\n    "node/8724555471",\n    "node/8839218919",\n    "node/8899871344",\n    "node/9130297826",\n    "node/9529138097",\n    "node/9612450265",\n    "node/9612452143",\n    "node/9612452169",\n    "node/9612452427",\n    "node/9612453575",\n    "node/9612453645",\n    "node/9708644526",\n    "node/9709230773",\n    "node/9718273772",\n    "node/9718274604",\n    "node/9718274972",\n    "node/9718274977",\n    "node/9862403304",\n    "node/9862403305",\n    "node/9862403306",\n    "node/9862403308",\n    "node/10090569961",\n    "node/10128641500",\n    "node/10214406637",\n    "node/10214406639",\n    "node/10240176163",\n    "node/10722898461",\n    "node/11126334601",\n    "node/11459367118",\n    "node/11780110802",\n    "node/11780110818",\n    "node/11975242304",\n    "node/12009505069",\n    "node/12010102101",\n    "node/12096755837",\n    "node/12123900986"\n   ]\n  },\n  "25": {\n   "buildings": 49,\n   "organisations": 66,\n   "tagged_hours": 27,\n   "parsed_hours": 27,\n   "hour_signals": [\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      10.0,\n      7.0,\n      0.0\n     ],\n     [\n      21.0,\n      11.0,\n      0.0\n     ],\n     [\n      22.0,\n      1.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      1.0\n     ],\n     [\n      14.0,\n      0.0,\n      7.0\n     ],\n     [\n      11.0,\n      0.0,\n      3.0\n     ],\n     [\n      9.0,\n      0.0,\n      2.0\n     ],\n     [\n      2.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      2.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      11.0,\n      8.0,\n      0.0\n     ],\n     [\n      22.0,\n      11.0,\n      0.0\n     ],\n     [\n      23.0,\n      1.0,\n      0.0\n     ],\n     [\n      24.0,\n      1.0,\n      0.0\n     ],\n     [\n      24.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      2.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      1.0\n     ],\n     [\n      14.0,\n      0.0,\n      7.0\n     ],\n     [\n      11.0,\n      0.0,\n      3.0\n     ],\n     [\n      9.0,\n      0.0,\n      2.0\n     ],\n     [\n      2.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      2.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      10.0,\n      7.0,\n      0.0\n     ],\n     [\n      21.0,\n      11.0,\n      0.0\n     ],\n     [\n      23.0,\n      2.0,\n      0.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      1.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      1.0\n     ],\n     [\n      14.0,\n      0.0,\n      7.0\n     ],\n     [\n      11.0,\n      0.0,\n      3.0\n     ],\n     [\n      9.0,\n      0.0,\n      2.0\n     ],\n     [\n      2.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      2.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      10.0,\n      7.0,\n      0.0\n     ],\n     [\n      21.0,\n      11.0,\n      0.0\n     ],\n     [\n      23.0,\n      2.0,\n      0.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      1.0\n     ],\n     [\n      23.0,\n      1.0,\n      0.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      24.0,\n      1.0,\n      0.0\n     ],\n     [\n      22.0,\n      0.0,\n      2.0\n     ],\n     [\n      14.0,\n      0.0,\n      8.0\n     ],\n     [\n      11.0,\n      0.0,\n      3.0\n     ],\n     [\n      9.0,\n      0.0,\n      2.0\n     ],\n     [\n      2.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      2.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      3.0,\n      3.0,\n      0.0\n     ],\n     [\n      10.0,\n      7.0,\n      0.0\n     ],\n     [\n      20.0,\n      10.0,\n      0.0\n     ],\n     [\n      21.0,\n      1.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.0,\n      0.0,\n      0.0\n     ],\n     [\n      20.75,\n      0.0,\n      1.0\n     ],\n     [\n      20.0,\n      0.0,\n      0.0\n     ],\n     [\n      19.75,\n      0.0,\n      1.0\n     ],\n     [\n      13.0,\n      0.0,\n      6.0\n     ],\n     [\n      11.0,\n      0.0,\n      2.0\n     ],\n     [\n      9.0,\n      0.0,\n      2.0\n     ],\n     [\n      2.0,\n      0.0,\n      7.0\n     ],\n     [\n      0.0,\n      0.0,\n      2.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      2.0,\n      0.0\n     ],\n     [\n      4.0,\n      2.0,\n      0.0\n     ],\n     [\n      13.0,\n      9.0,\n      0.0\n     ],\n     [\n      14.0,\n      1.0,\n      0.0\n     ],\n     [\n      14.0,\n      0.0,\n      0.0\n     ],\n     [\n      14.0,\n      0.0,\n      0.0\n     ],\n     [\n      14.0,\n      0.0,\n      0.0\n     ],\n     [\n      14.0,\n      0.0,\n      0.0\n     ],\n     [\n      10.0,\n      0.0,\n      4.0\n     ],\n     [\n      10.0,\n      0.0,\n      0.0\n     ],\n     [\n      9.0,\n      0.0,\n      1.0\n     ],\n     [\n      9.0,\n      0.0,\n      0.0\n     ],\n     [\n      7.0,\n      0.0,\n      2.0\n     ],\n     [\n      2.0,\n      0.0,\n      5.0\n     ],\n     [\n      0.0,\n      0.0,\n      2.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      2.0,\n      0.0\n     ],\n     [\n      4.0,\n      2.0,\n      0.0\n     ],\n     [\n      13.0,\n      9.0,\n      0.0\n     ],\n     [\n      15.0,\n      2.0,\n      0.0\n     ],\n     [\n      15.0,\n      0.0,\n      0.0\n     ],\n     [\n      14.0,\n      0.0,\n      1.0\n     ],\n     [\n      14.0,\n      0.0,\n      0.0\n     ],\n     [\n      14.0,\n      0.0,\n      0.0\n     ],\n     [\n      11.0,\n      0.0,\n      3.0\n     ],\n     [\n      11.0,\n      0.0,\n      0.0\n     ],\n     [\n      9.0,\n      0.0,\n      2.0\n     ],\n     [\n      9.0,\n      0.0,\n      0.0\n     ],\n     [\n      7.0,\n      0.0,\n      2.0\n     ],\n     [\n      2.0,\n      0.0,\n      5.0\n     ],\n     [\n      0.0,\n      0.0,\n      2.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ]\n   ],\n   "poi_ids": [\n    "relation/1204804",\n    "relation/1204805",\n    "relation/5559920",\n    "relation/7006925",\n    "relation/8718673",\n    "relation/11257406",\n    "relation/12085584",\n    "relation/14760679",\n    "relation/14966578",\n    "relation/15057044",\n    "relation/17950695",\n    "relation/17950696",\n    "relation/17950697",\n    "way/28828645",\n    "way/29854157",\n    "way/29854159",\n    "way/30426860",\n    "way/40999582",\n    "way/47066460",\n    "way/52477085",\n    "way/90265767",\n    "way/91209651",\n    "way/100134908",\n    "way/111601797",\n    "way/125983154",\n    "way/129920063",\n    "way/129920064",\n    "way/131833399",\n    "way/131833401",\n    "way/131833406",\n    "way/131833408",\n    "way/142235990",\n    "way/150824915",\n    "way/150824920",\n    "way/150824925",\n    "way/150824928",\n    "way/150824929",\n    "way/168656742",\n    "way/179615802",\n    "way/362310589",\n    "way/362816111",\n    "way/363128565",\n    "way/363128568",\n    "way/363460987",\n    "way/374066823",\n    "way/389268468",\n    "way/503235172",\n    "way/531331536",\n    "way/555999537",\n    "way/798376273",\n    "way/809261886",\n    "way/868127395",\n    "way/1216209524",\n    "way/1216391945",\n    "way/1265302069",\n    "node/1530991489",\n    "node/1909859232",\n    "node/1909859236",\n    "node/4147034394",\n    "node/4320667358",\n    "node/4540199361",\n    "node/4682577735",\n    "node/4682610556",\n    "node/4684267584",\n    "node/4684337962",\n    "node/4972313160",\n    "node/4972313161",\n    "node/4972313521",\n    "node/5263709121",\n    "node/6301297192",\n    "node/6442444928",\n    "node/7258009957",\n    "node/7265739617",\n    "node/7265739618",\n    "node/7265739649",\n    "node/7265739650",\n    "node/7330002514",\n    "node/7360141677",\n    "node/7397131033",\n    "node/7397131040",\n    "node/7404805605",\n    "node/7404805606",\n    "node/7562039998",\n    "node/7562040005",\n    "node/7562040006",\n    "node/7625250530",\n    "node/7625250531",\n    "node/7625250532",\n    "node/7625250535",\n    "node/7625250536",\n    "node/7625741852",\n    "node/7625741853",\n    "node/7625741854",\n    "node/7832400978",\n    "node/8035236212",\n    "node/8065673628",\n    "node/8341433239",\n    "node/8370846287",\n    "node/8370846303",\n    "node/8684306021",\n    "node/8705321494",\n    "node/8724555471",\n    "node/9110464473",\n    "node/9130297826",\n    "node/9531730654",\n    "node/9612453645",\n    "node/9718274972",\n    "node/10090569961",\n    "node/10282313977",\n    "node/11171107790",\n    "node/11737235288",\n    "node/11959910969",\n    "node/12096755837",\n    "node/12096765864",\n    "node/12238142129"\n   ]\n  },\n  "50": {\n   "buildings": 386,\n   "organisations": 336,\n   "tagged_hours": 116,\n   "parsed_hours": 114,\n   "hour_signals": [\n    [\n     [\n      6.0,\n      0.0,\n      1.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      10.0,\n      0.0\n     ],\n     [\n      66.75,\n      55.0,\n      0.0\n     ],\n     [\n      105.75,\n      35.0,\n      0.0\n     ],\n     [\n      112.0,\n      6.0,\n      0.0\n     ],\n     [\n      110.5,\n      1.0,\n      2.0\n     ],\n     [\n      109.0,\n      2.0,\n      4.0\n     ],\n     [\n      109.5,\n      2.0,\n      1.0\n     ],\n     [\n      110.0,\n      1.0,\n      1.0\n     ],\n     [\n      109.0,\n      0.0,\n      1.0\n     ],\n     [\n      99.5,\n      0.0,\n      11.0\n     ],\n     [\n      66.5,\n      0.0,\n      34.0\n     ],\n     [\n      46.0,\n      0.0,\n      19.0\n     ],\n     [\n      24.5,\n      0.0,\n      21.0\n     ],\n     [\n      12.0,\n      0.0,\n      12.0\n     ],\n     [\n      7.0,\n      0.0,\n      5.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      6.0,\n      0.0,\n      1.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      10.0,\n      0.0\n     ],\n     [\n      69.75,\n      58.0,\n      0.0\n     ],\n     [\n      108.75,\n      35.0,\n      0.0\n     ],\n     [\n      114.0,\n      5.0,\n      0.0\n     ],\n     [\n      112.5,\n      1.0,\n      2.0\n     ],\n     [\n      109.5,\n      1.0,\n      5.0\n     ],\n     [\n      109.5,\n      2.0,\n      1.0\n     ],\n     [\n      111.0,\n      2.0,\n      1.0\n     ],\n     [\n      110.0,\n      0.0,\n      1.0\n     ],\n     [\n      100.0,\n      0.0,\n      12.0\n     ],\n     [\n      66.5,\n      0.0,\n      34.0\n     ],\n     [\n      46.0,\n      0.0,\n      19.0\n     ],\n     [\n      24.5,\n      0.0,\n      21.0\n     ],\n     [\n      12.0,\n      0.0,\n      12.0\n     ],\n     [\n      7.0,\n      0.0,\n      5.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      6.0,\n      0.0,\n      1.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      10.0,\n      0.0\n     ],\n     [\n      69.75,\n      58.0,\n      0.0\n     ],\n     [\n      108.75,\n      35.0,\n      0.0\n     ],\n     [\n      114.0,\n      5.0,\n      0.0\n     ],\n     [\n      112.5,\n      1.0,\n      2.0\n     ],\n     [\n      109.5,\n      1.0,\n      5.0\n     ],\n     [\n      109.5,\n      2.0,\n      1.0\n     ],\n     [\n      111.0,\n      2.0,\n      1.0\n     ],\n     [\n      110.0,\n      0.0,\n      1.0\n     ],\n     [\n      100.0,\n      0.0,\n      12.0\n     ],\n     [\n      65.5,\n      0.0,\n      35.0\n     ],\n     [\n      45.0,\n      0.0,\n      19.0\n     ],\n     [\n      24.5,\n      0.0,\n      20.0\n     ],\n     [\n      12.0,\n      0.0,\n      12.0\n     ],\n     [\n      7.0,\n      0.0,\n      5.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      6.0,\n      0.0,\n      1.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      10.0,\n      0.0\n     ],\n     [\n      69.75,\n      58.0,\n      0.0\n     ],\n     [\n      108.75,\n      35.0,\n      0.0\n     ],\n     [\n      114.0,\n      5.0,\n      0.0\n     ],\n     [\n      112.5,\n      1.0,\n      2.0\n     ],\n     [\n      109.5,\n      1.0,\n      5.0\n     ],\n     [\n      109.5,\n      2.0,\n      1.0\n     ],\n     [\n      111.0,\n      2.0,\n      1.0\n     ],\n     [\n      110.0,\n      0.0,\n      1.0\n     ],\n     [\n      100.0,\n      0.0,\n      12.0\n     ],\n     [\n      66.5,\n      0.0,\n      34.0\n     ],\n     [\n      46.0,\n      0.0,\n      19.0\n     ],\n     [\n      24.5,\n      0.0,\n      21.0\n     ],\n     [\n      12.0,\n      0.0,\n      12.0\n     ],\n     [\n      7.0,\n      0.0,\n      5.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      6.0,\n      0.0,\n      1.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      16.0,\n      10.0,\n      0.0\n     ],\n     [\n      67.75,\n      56.0,\n      0.0\n     ],\n     [\n      105.75,\n      34.0,\n      0.0\n     ],\n     [\n      110.0,\n      4.0,\n      0.0\n     ],\n     [\n      108.5,\n      1.0,\n      2.0\n     ],\n     [\n      106.5,\n      1.0,\n      4.0\n     ],\n     [\n      107.5,\n      2.0,\n      0.0\n     ],\n     [\n      107.0,\n      1.0,\n      4.0\n     ],\n     [\n      102.25,\n      0.0,\n      8.0\n     ],\n     [\n      89.5,\n      0.0,\n      9.0\n     ],\n     [\n      64.0,\n      0.0,\n      26.0\n     ],\n     [\n      44.0,\n      0.0,\n      19.0\n     ],\n     [\n      24.5,\n      0.0,\n      19.0\n     ],\n     [\n      12.0,\n      0.0,\n      12.0\n     ],\n     [\n      7.0,\n      0.0,\n      5.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      6.0,\n      0.0,\n      1.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      8.0,\n      2.0,\n      0.0\n     ],\n     [\n      28.5,\n      21.0,\n      0.0\n     ],\n     [\n      47.0,\n      18.0,\n      0.0\n     ],\n     [\n      50.0,\n      3.0,\n      0.0\n     ],\n     [\n      50.0,\n      0.0,\n      0.0\n     ],\n     [\n      49.5,\n      0.0,\n      1.0\n     ],\n     [\n      49.0,\n      0.0,\n      0.0\n     ],\n     [\n      47.25,\n      1.0,\n      5.0\n     ],\n     [\n      44.0,\n      0.0,\n      1.0\n     ],\n     [\n      41.5,\n      0.0,\n      3.0\n     ],\n     [\n      33.5,\n      0.0,\n      8.0\n     ],\n     [\n      29.0,\n      0.0,\n      4.0\n     ],\n     [\n      21.5,\n      0.0,\n      8.0\n     ],\n     [\n      11.0,\n      0.0,\n      10.0\n     ],\n     [\n      7.0,\n      0.0,\n      4.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      6.0,\n      0.0,\n      1.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.0,\n      0.0,\n      0.0\n     ],\n     [\n      7.0,\n      1.0,\n      0.0\n     ],\n     [\n      22.5,\n      16.0,\n      0.0\n     ],\n     [\n      37.0,\n      14.0,\n      0.0\n     ],\n     [\n      38.0,\n      1.0,\n      0.0\n     ],\n     [\n      38.0,\n      0.0,\n      0.0\n     ],\n     [\n      38.0,\n      0.0,\n      0.0\n     ],\n     [\n      38.0,\n      0.0,\n      0.0\n     ],\n     [\n      38.0,\n      0.0,\n      0.0\n     ],\n     [\n      38.0,\n      0.0,\n      0.0\n     ],\n     [\n      37.0,\n      0.0,\n      1.0\n     ],\n     [\n      31.5,\n      0.0,\n      6.0\n     ],\n     [\n      28.0,\n      0.0,\n      3.0\n     ],\n     [\n      20.5,\n      0.0,\n      8.0\n     ],\n     [\n      11.0,\n      0.0,\n      9.0\n     ],\n     [\n      7.0,\n      0.0,\n      4.0\n     ],\n     [\n      7.0,\n      0.0,\n      0.0\n     ]\n    ]\n   ],\n   "poi_ids": [\n    "relation/149242",\n    "relation/149243",\n    "relation/149244",\n    "relation/1397821",\n    "relation/1761513",\n    "relation/1853036",\n    "relation/2386975",\n    "relation/3618426",\n    "relation/3789120",\n    "relation/5424719",\n    "relation/5428217",\n    "relation/5428219",\n    "relation/5428220",\n    "relation/7761398",\n    "relation/7895066",\n    "relation/7895073",\n    "relation/8463134",\n    "relation/9201158",\n    "relation/9620168",\n    "relation/10433069",\n    "relation/11344329",\n    "relation/11345933",\n    "relation/11345934",\n    "relation/11855557",\n    "relation/11855559",\n    "relation/11863749",\n    "relation/12432667",\n    "relation/12743120",\n    "relation/12743122",\n    "relation/12743123",\n    "relation/12900987",\n    "relation/13465821",\n    "relation/13465823",\n    "relation/13533473",\n    "relation/13783336",\n    "relation/13838105",\n    "relation/13883096",\n    "relation/14033236",\n    "relation/14097875",\n    "relation/14110459",\n    "relation/14153409",\n    "relation/14209002",\n    "relation/14212251",\n    "relation/14212253",\n    "relation/14213232",\n    "relation/14214913",\n    "relation/14271754",\n    "relation/14277909",\n    "relation/14923956",\n    "relation/14924020",\n    "relation/14946967",\n    "relation/14946968",\n    "relation/14946969",\n    "relation/14956592",\n    "relation/14987149",\n    "relation/14987150",\n    "relation/15019422",\n    "relation/15025366",\n    "relation/15025368",\n    "relation/15046087",\n    "relation/15046094",\n    "relation/15046096",\n    "relation/15047543",\n    "relation/15047549",\n    "relation/15047550",\n    "relation/15322772",\n    "relation/15322773",\n    "relation/15322775",\n    "relation/15322795",\n    "relation/15322817",\n    "relation/15322818",\n    "relation/15322820",\n    "relation/15322821",\n    "relation/15326525",\n    "relation/15327986",\n    "relation/15342436",\n    "relation/15451609",\n    "relation/15844900",\n    "relation/16380400",\n    "relation/16380406",\n    "relation/16930258",\n    "relation/16930262",\n    "relation/16930275",\n    "relation/16930276",\n    "relation/16930278",\n    "relation/17024308",\n    "relation/17024309",\n    "relation/17051423",\n    "relation/17051424",\n    "relation/17051425",\n    "relation/17051426",\n    "relation/17088305",\n    "relation/17088308",\n    "relation/17305925",\n    "relation/17329527",\n    "relation/17940120",\n    "relation/18267726",\n    "relation/18274494",\n    "relation/18274495",\n    "relation/18274496",\n    "way/28568800",\n    "way/29070586",\n    "way/30437389",\n    "way/31950856",\n    "way/32244153",\n    "way/32276135",\n    "way/33456225",\n    "way/34319623",\n    "way/34844420",\n    "way/35168339",\n    "way/35168341",\n    "way/35168343",\n    "way/35168352",\n    "way/35168355",\n    "way/35168374",\n    "way/35168375",\n    "way/35168378",\n    "way/35168379",\n    "way/35168380",\n    "way/35168383",\n    "way/35168391",\n    "way/35168393",\n    "way/35168396",\n    "way/35168399",\n    "way/35168404",\n    "way/35168409",\n    "way/35168417",\n    "way/35547549",\n    "way/35547550",\n    "way/35547552",\n    "way/35610495",\n    "way/35610503",\n    "way/35610504",\n    "way/35610506",\n    "way/35610513",\n    "way/35693533",\n    "way/37001671",\n    "way/37001672",\n    "way/37130923",\n    "way/37130969",\n    "way/37130977",\n    "way/37130979",\n    "way/37130980",\n    "way/37130982",\n    "way/37130984",\n    "way/37130985",\n    "way/37130990",\n    "way/37130991",\n    "way/37130993",\n    "way/37130999",\n    "way/37131000",\n    "way/37735415",\n    "way/37756919",\n    "way/37756923",\n    "way/37756924",\n    "way/37756966",\n    "way/37757008",\n    "way/37757037",\n    "way/37757039",\n    "way/37757051",\n    "way/38966159",\n    "way/40604732",\n    "way/44771613",\n    "way/44771617",\n    "way/44778324",\n    "way/44778327",\n    "way/45105693",\n    "way/45243241",\n    "way/45829255",\n    "way/45829258",\n    "way/45831631",\n    "way/46414092",\n    "way/46414098",\n    "way/46414205",\n    "way/46414225",\n    "way/46414229",\n    "way/46414232",\n    "way/46414240",\n    "way/46414241",\n    "way/46414248",\n    "way/46603828",\n    "way/46603838",\n    "way/46737429",\n    "way/46737486",\n    "way/46740665",\n    "way/46740669",\n    "way/46740674",\n    "way/46740675",\n    "way/46740679",\n    "way/46740695",\n    "way/46740699",\n    "way/46740700",\n    "way/46740701",\n    "way/46740714",\n    "way/46740716",\n    "way/46740725",\n    "way/46854271",\n    "way/48274534",\n    "way/48648090",\n    "way/48894240",\n    "way/49324856",\n    "way/49406072",\n    "way/49428991",\n    "way/49488039",\n    "way/49591455",\n    "way/49677877",\n    "way/49798329",\n    "way/49799926",\n    "way/50548832",\n    "way/50654372",\n    "way/50833973",\n    "way/50833977",\n    "way/50833986",\n    "way/51004884",\n    "way/51004890",\n    "way/51045653",\n    "way/52313282",\n    "way/52482757",\n    "way/52482759",\n    "way/52832290",\n    "way/53081767",\n    "way/53156743",\n    "way/53156747",\n    "way/53367607",\n    "way/53521337",\n    "way/53725264",\n    "way/53767681",\n    "way/53845549",\n    "way/53888272",\n    "way/53889555",\n    "way/56075407",\n    "way/56121187",\n    "way/60903175",\n    "way/60911175",\n    "way/60911217",\n    "way/62209257",\n    "way/65134315",\n    "way/79246883",\n    "way/79561179",\n    "way/80746007",\n    "way/83113147",\n    "way/84890806",\n    "way/85301929",\n    "way/86985775",\n    "way/86985778",\n    "way/86985781",\n    "way/86985813",\n    "way/87349420",\n    "way/87349435",\n    "way/87349444",\n    "way/87349450",\n    "way/87349453",\n    "way/87349460",\n    "way/87349462",\n    "way/87349465",\n    "way/88408309",\n    "way/88418092",\n    "way/88418093",\n    "way/88418094",\n    "way/88423799",\n    "way/88423833",\n    "way/88436452",\n    "way/88436461",\n    "way/88440977",\n    "way/88440985",\n    "way/90982315",\n    "way/90982316",\n    "way/90982326",\n    "way/95665378",\n    "way/98572152",\n    "way/98572171",\n    "way/98572179",\n    "way/98572184",\n    "way/98572185",\n    "way/98572194",\n    "way/98572204",\n    "way/98572229",\n    "way/98572245",\n    "way/98572249",\n    "way/98572257",\n    "way/98572258",\n    "way/98572264",\n    "way/98572271",\n    "way/98572274",\n    "way/98572275",\n    "way/98572280",\n    "way/98572293",\n    "way/98572298",\n    "way/109081780",\n    "way/109089979",\n    "way/109127643",\n    "way/109127646",\n    "way/109127674",\n    "way/109127683",\n    "way/109200889",\n    "way/109939339",\n    "way/112358863",\n    "way/125667351",\n    "way/125667369",\n    "way/125674933",\n    "way/125674936",\n    "way/125674939",\n    "way/125674944",\n    "way/125674949",\n    "way/125674950",\n    "way/125674951",\n    "way/125674958",\n    "way/125849315",\n    "way/125849327",\n    "way/125849335",\n    "way/126059620",\n    "way/127717485",\n    "way/127875665",\n    "way/128264421",\n    "way/130297622",\n    "way/130706112",\n    "way/130855517",\n    "way/130862427",\n    "way/130862595",\n    "way/130862651",\n    "way/131173264",\n    "way/131173268",\n    "way/131173276",\n    "way/131175197",\n    "way/131182227",\n    "way/132066885",\n    "way/139055063",\n    "way/139069935",\n    "way/139450248",\n    "way/139880865",\n    "way/143189426",\n    "way/143189428",\n    "way/146926420",\n    "way/149864235",\n    "way/149864236",\n    "way/151574816",\n    "way/151574820",\n    "way/151574821",\n    "way/152270032",\n    "way/155794664",\n    "way/168357975",\n    "way/170081763",\n    "way/172549518",\n    "way/172839669",\n    "way/172839675",\n    "way/172839706",\n    "way/172839709",\n    "way/175187062",\n    "way/176025431",\n    "way/178558387",\n    "way/179341739",\n    "way/192393349",\n    "way/192393351",\n    "way/192393353",\n    "way/197511591",\n    "way/197511711",\n    "way/197891060",\n    "way/197929393",\n    "way/249104978",\n    "way/261752407",\n    "way/305258203",\n    "way/310402008",\n    "way/318858405",\n    "way/349741206",\n    "way/372068255",\n    "way/375867350",\n    "way/386343964",\n    "way/386343965",\n    "way/386343971",\n    "way/387694869",\n    "way/387694871",\n    "way/387694872",\n    "way/387694873",\n    "way/387694874",\n    "way/387694897",\n    "way/444437262",\n    "way/446071847",\n    "way/448961881",\n    "way/468011389",\n    "way/481075439",\n    "way/489066735",\n    "way/501679712",\n    "way/501679720",\n    "way/520635412",\n    "way/539238923",\n    "way/541958498",\n    "way/593903274",\n    "way/609945076",\n    "way/684545249",\n    "way/706495344",\n    "way/708012674",\n    "way/737401378",\n    "way/770533814",\n    "way/770533832",\n    "way/818074286",\n    "way/835466032",\n    "way/865594604",\n    "way/882704269",\n    "way/887983715",\n    "way/887983716",\n    "way/933023157",\n    "way/981382157",\n    "way/986672025",\n    "way/986672026",\n    "way/1066253622",\n    "way/1171996570",\n    "way/1171996571",\n    "way/1188275504",\n    "way/1231772485",\n    "way/1234294632",\n    "way/1234550686",\n    "way/1234594084",\n    "way/1252904101",\n    "way/1255506148",\n    "way/1255607274",\n    "node/380729527",\n    "node/432502061",\n    "node/432507349",\n    "node/698892153",\n    "node/698906226",\n    "node/775957750",\n    "node/988853638",\n    "node/1745421983",\n    "node/1745422061",\n    "node/1902113640",\n    "node/1952548173",\n    "node/1966876890",\n    "node/1997933514",\n    "node/2328657311",\n    "node/2355505491",\n    "node/2355593478",\n    "node/2381366029",\n    "node/2632370131",\n    "node/2634802603",\n    "node/2673526388",\n    "node/2815257266",\n    "node/2855188627",\n    "node/2922768877",\n    "node/2922768880",\n    "node/3079075220",\n    "node/3704170104",\n    "node/3750376527",\n    "node/3766911682",\n    "node/3800654308",\n    "node/3808286574",\n    "node/3897252405",\n    "node/3922372306",\n    "node/3939278827",\n    "node/3980755657",\n    "node/4043907864",\n    "node/4071266491",\n    "node/4117083908",\n    "node/4177235410",\n    "node/4208738076",\n    "node/4279220181",\n    "node/4298602377",\n    "node/4301803493",\n    "node/4301832092",\n    "node/4316261493",\n    "node/4332145990",\n    "node/4354203194",\n    "node/4363303418",\n    "node/4437338403",\n    "node/4463138252",\n    "node/4547855937",\n    "node/4569239515",\n    "node/4584007451",\n    "node/4618650343",\n    "node/4618650363",\n    "node/4618650596",\n    "node/4646432090",\n    "node/4678212193",\n    "node/4682401092",\n    "node/4682548443",\n    "node/4682603734",\n    "node/4682603735",\n    "node/4684137693",\n    "node/4684141141",\n    "node/4709826251",\n    "node/4716381689",\n    "node/4786504125",\n    "node/4801212317",\n    "node/4882971022",\n    "node/4887670049",\n    "node/4968280698",\n    "node/5032420921",\n    "node/5048493221",\n    "node/5081954314",\n    "node/5081954938",\n    "node/5103763827",\n    "node/5106953143",\n    "node/5194103200",\n    "node/5219421023",\n    "node/5225926322",\n    "node/5262175421",\n    "node/5352800144",\n    "node/5352800145",\n    "node/5429605321",\n    "node/5429613021",\n    "node/5460737324",\n    "node/5464423445",\n    "node/5513091826",\n    "node/5541234491",\n    "node/5543690921",\n    "node/5548442522",\n    "node/5580650221",\n    "node/5630830949",\n    "node/5630832539",\n    "node/5671111817",\n    "node/5727172721",\n    "node/5749899138",\n    "node/5835909578",\n    "node/5837982052",\n    "node/5978203386",\n    "node/6015673186",\n    "node/6039177662",\n    "node/6048401364",\n    "node/6151264710",\n    "node/6241070485",\n    "node/6265767313",\n    "node/6315572923",\n    "node/6360168500",\n    "node/6485288284",\n    "node/6579966563",\n    "node/6580085386",\n    "node/6722440985",\n    "node/6736223685",\n    "node/6738633185",\n    "node/6869203547",\n    "node/7034471305",\n    "node/7052621828",\n    "node/7056375244",\n    "node/7057095690",\n    "node/7112745879",\n    "node/7136843594",\n    "node/7162041568",\n    "node/7182315984",\n    "node/7243041495",\n    "node/7271204253",\n    "node/7334712665",\n    "node/7551278126",\n    "node/7620684102",\n    "node/7620684104",\n    "node/7660544845",\n    "node/7661714953",\n    "node/7682601860",\n    "node/7713961172",\n    "node/7741235276",\n    "node/7741261510",\n    "node/7741269179",\n    "node/7781680618",\n    "node/7816508277",\n    "node/7832365998",\n    "node/7922165789",\n    "node/7942483224",\n    "node/8006077873",\n    "node/8006077874",\n    "node/8006077875",\n    "node/8006077877",\n    "node/8058669733",\n    "node/8058669734",\n    "node/8058669736",\n    "node/8058706720",\n    "node/8068206554",\n    "node/8068206555",\n    "node/8068212087",\n    "node/8068212152",\n    "node/8083744077",\n    "node/8091021406",\n    "node/8091035422",\n    "node/8099832498",\n    "node/8099876060",\n    "node/8099876064",\n    "node/8099897701",\n    "node/8099897702",\n    "node/8099897703",\n    "node/8209006504",\n    "node/8210279520",\n    "node/8210279521",\n    "node/8219367153",\n    "node/8270161408",\n    "node/8281001166",\n    "node/8281001167",\n    "node/8307245121",\n    "node/8307464032",\n    "node/8307565221",\n    "node/8308203711",\n    "node/8370848138",\n    "node/8377639676",\n    "node/8390987295",\n    "node/8391990768",\n    "node/8392053708",\n    "node/8488538015",\n    "node/8489068408",\n    "node/8499187314",\n    "node/8499207065",\n    "node/8499207079",\n    "node/8499207084",\n    "node/8521187108",\n    "node/8555150389",\n    "node/8635426747",\n    "node/8647921326",\n    "node/8657534455",\n    "node/8678169875",\n    "node/8683003076",\n    "node/8713123471",\n    "node/8750268747",\n    "node/8754760534",\n    "node/8754760535",\n    "node/8839129807",\n    "node/8839218905",\n    "node/8839218907",\n    "node/8875626157",\n    "node/9077880586",\n    "node/9077896522",\n    "node/9318813851",\n    "node/9490585320",\n    "node/9496122320",\n    "node/9496122321",\n    "node/9496122327",\n    "node/9519022396",\n    "node/9519022398",\n    "node/9526600851",\n    "node/9603155076",\n    "node/9603155077",\n    "node/9612453587",\n    "node/9612453786",\n    "node/9718274912",\n    "node/9735580786",\n    "node/9783486440",\n    "node/9790942988",\n    "node/9885173596",\n    "node/9888459498",\n    "node/9888459499",\n    "node/9917810417",\n    "node/10029522905",\n    "node/10063587262",\n    "node/10123104418",\n    "node/10146889639",\n    "node/10146889640",\n    "node/10146889641",\n    "node/10242556233",\n    "node/10242556248",\n    "node/10271419902",\n    "node/10273072878",\n    "node/10274685442",\n    "node/10285492511",\n    "node/10291017665",\n    "node/10563067487",\n    "node/10628561078",\n    "node/10628574036",\n    "node/10646591013",\n    "node/10669808008",\n    "node/10766576298",\n    "node/10786728122",\n    "node/10828724807",\n    "node/10844094861",\n    "node/10922965252",\n    "node/10963635089",\n    "node/10963641377",\n    "node/10964974016",\n    "node/10969632457",\n    "node/10969632458",\n    "node/10969632460",\n    "node/10969632462",\n    "node/11028232837",\n    "node/11083605361",\n    "node/11083649936",\n    "node/11085084468",\n    "node/11101213766",\n    "node/11118842262",\n    "node/11159472347",\n    "node/11174761628",\n    "node/11299514090",\n    "node/11423171142",\n    "node/11427615247",\n    "node/11435856876",\n    "node/11435889113",\n    "node/11435889114",\n    "node/11435889115",\n    "node/11453975747",\n    "node/11459367123",\n    "node/11459367129",\n    "node/11467550491",\n    "node/11467550492",\n    "node/11467550493",\n    "node/11481235090",\n    "node/11504820232",\n    "node/11505370011",\n    "node/11505370023",\n    "node/11510811673",\n    "node/11512192672",\n    "node/11515114859",\n    "node/11515114862",\n    "node/11520396947",\n    "node/11530721773",\n    "node/11545721377",\n    "node/11593054742",\n    "node/11596428029",\n    "node/11596428055",\n    "node/11596428059",\n    "node/11596428060",\n    "node/11598676504",\n    "node/11598676518",\n    "node/11602527685",\n    "node/11648769074",\n    "node/11804699391",\n    "node/11843797431",\n    "node/11873574783",\n    "node/11905837216",\n    "node/11925664868",\n    "node/11958118153",\n    "node/12026320354",\n    "node/12097530418",\n    "node/12106517544",\n    "node/12110530696",\n    "node/12141981518",\n    "node/12296421701",\n    "node/12430845798",\n    "node/12434071661",\n    "node/12439578169",\n    "node/12439613705"\n   ]\n  },\n  "7": {\n   "buildings": 451,\n   "organisations": 398,\n   "tagged_hours": 143,\n   "parsed_hours": 142,\n   "hour_signals": [\n    [\n     [\n      5.0,\n      0.0,\n      1.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      24.0,\n      19.0,\n      0.0\n     ],\n     [\n      77.5,\n      56.0,\n      0.0\n     ],\n     [\n      126.5,\n      47.0,\n      0.0\n     ],\n     [\n      132.0,\n      5.0,\n      0.0\n     ],\n     [\n      129.75,\n      0.0,\n      3.0\n     ],\n     [\n      127.75,\n      4.0,\n      6.0\n     ],\n     [\n      128.25,\n      4.0,\n      2.0\n     ],\n     [\n      130.0,\n      3.0,\n      2.0\n     ],\n     [\n      128.0,\n      0.0,\n      2.0\n     ],\n     [\n      115.5,\n      0.0,\n      14.0\n     ],\n     [\n      84.0,\n      0.0,\n      32.0\n     ],\n     [\n      66.0,\n      0.0,\n      16.0\n     ],\n     [\n      37.0,\n      0.0,\n      29.0\n     ],\n     [\n      15.0,\n      0.0,\n      22.0\n     ],\n     [\n      8.0,\n      0.0,\n      7.0\n     ],\n     [\n      6.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      5.0,\n      0.0,\n      1.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      24.0,\n      19.0,\n      0.0\n     ],\n     [\n      82.5,\n      61.0,\n      0.0\n     ],\n     [\n      132.5,\n      48.0,\n      0.0\n     ],\n     [\n      138.0,\n      5.0,\n      0.0\n     ],\n     [\n      135.75,\n      0.0,\n      3.0\n     ],\n     [\n      131.25,\n      3.0,\n      9.0\n     ],\n     [\n      129.25,\n      4.0,\n      3.0\n     ],\n     [\n      133.0,\n      5.0,\n      2.0\n     ],\n     [\n      133.0,\n      1.0,\n      1.0\n     ],\n     [\n      118.0,\n      0.0,\n      18.0\n     ],\n     [\n      85.0,\n      0.0,\n      32.0\n     ],\n     [\n      67.0,\n      0.0,\n      16.0\n     ],\n     [\n      37.0,\n      0.0,\n      30.0\n     ],\n     [\n      15.0,\n      0.0,\n      22.0\n     ],\n     [\n      8.0,\n      0.0,\n      7.0\n     ],\n     [\n      6.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      5.0,\n      0.0,\n      1.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      24.0,\n      19.0,\n      0.0\n     ],\n     [\n      80.5,\n      59.0,\n      0.0\n     ],\n     [\n      130.5,\n      48.0,\n      0.0\n     ],\n     [\n      137.0,\n      6.0,\n      0.0\n     ],\n     [\n      134.75,\n      0.0,\n      3.0\n     ],\n     [\n      129.25,\n      3.0,\n      10.0\n     ],\n     [\n      128.25,\n      4.0,\n      2.0\n     ],\n     [\n      132.0,\n      5.0,\n      2.0\n     ],\n     [\n      131.0,\n      0.0,\n      1.0\n     ],\n     [\n      118.0,\n      0.0,\n      16.0\n     ],\n     [\n      84.0,\n      0.0,\n      33.0\n     ],\n     [\n      66.0,\n      0.0,\n      16.0\n     ],\n     [\n      37.0,\n      0.0,\n      29.0\n     ],\n     [\n      15.0,\n      0.0,\n      22.0\n     ],\n     [\n      8.0,\n      0.0,\n      7.0\n     ],\n     [\n      6.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      5.0,\n      0.0,\n      1.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      23.0,\n      18.0,\n      0.0\n     ],\n     [\n      79.5,\n      59.0,\n      0.0\n     ],\n     [\n      129.5,\n      48.0,\n      0.0\n     ],\n     [\n      136.0,\n      6.0,\n      0.0\n     ],\n     [\n      134.25,\n      0.0,\n      2.0\n     ],\n     [\n      128.5,\n      2.0,\n      10.0\n     ],\n     [\n      128.25,\n      5.0,\n      2.0\n     ],\n     [\n      132.0,\n      5.0,\n      2.0\n     ],\n     [\n      132.0,\n      1.0,\n      1.0\n     ],\n     [\n      119.0,\n      0.0,\n      16.0\n     ],\n     [\n      85.0,\n      0.0,\n      33.0\n     ],\n     [\n      67.0,\n      0.0,\n      16.0\n     ],\n     [\n      37.0,\n      0.0,\n      30.0\n     ],\n     [\n      15.0,\n      0.0,\n      22.0\n     ],\n     [\n      8.0,\n      0.0,\n      7.0\n     ],\n     [\n      6.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      5.0,\n      0.0,\n      1.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      24.0,\n      19.0,\n      0.0\n     ],\n     [\n      79.5,\n      58.0,\n      0.0\n     ],\n     [\n      126.5,\n      45.0,\n      0.0\n     ],\n     [\n      131.0,\n      4.0,\n      0.0\n     ],\n     [\n      128.75,\n      0.0,\n      3.0\n     ],\n     [\n      124.75,\n      3.0,\n      8.0\n     ],\n     [\n      125.25,\n      4.0,\n      1.0\n     ],\n     [\n      125.5,\n      4.0,\n      8.0\n     ],\n     [\n      120.25,\n      0.0,\n      7.0\n     ],\n     [\n      105.25,\n      0.0,\n      13.0\n     ],\n     [\n      80.5,\n      0.0,\n      23.0\n     ],\n     [\n      63.0,\n      0.0,\n      16.0\n     ],\n     [\n      37.0,\n      0.0,\n      26.0\n     ],\n     [\n      15.0,\n      0.0,\n      22.0\n     ],\n     [\n      8.0,\n      0.0,\n      7.0\n     ],\n     [\n      6.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      5.0,\n      0.0,\n      1.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      15.0,\n      10.0,\n      0.0\n     ],\n     [\n      41.0,\n      26.0,\n      0.0\n     ],\n     [\n      70.0,\n      29.0,\n      0.0\n     ],\n     [\n      73.0,\n      3.0,\n      0.0\n     ],\n     [\n      73.0,\n      0.0,\n      0.0\n     ],\n     [\n      71.5,\n      0.0,\n      3.0\n     ],\n     [\n      70.0,\n      0.0,\n      0.0\n     ],\n     [\n      71.75,\n      3.0,\n      2.0\n     ],\n     [\n      68.0,\n      0.0,\n      3.0\n     ],\n     [\n      62.5,\n      0.0,\n      7.0\n     ],\n     [\n      52.0,\n      0.0,\n      9.0\n     ],\n     [\n      51.0,\n      0.0,\n      1.0\n     ],\n     [\n      34.0,\n      0.0,\n      17.0\n     ],\n     [\n      15.0,\n      0.0,\n      19.0\n     ],\n     [\n      8.0,\n      0.0,\n      7.0\n     ],\n     [\n      6.0,\n      0.0,\n      2.0\n     ]\n    ],\n    [\n     [\n      5.0,\n      0.0,\n      1.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      5.0,\n      0.0,\n      0.0\n     ],\n     [\n      13.0,\n      8.0,\n      0.0\n     ],\n     [\n      34.0,\n      21.0,\n      0.0\n     ],\n     [\n      62.0,\n      28.0,\n      0.0\n     ],\n     [\n      64.0,\n      2.0,\n      0.0\n     ],\n     [\n      64.0,\n      0.0,\n      0.0\n     ],\n     [\n      63.0,\n      0.0,\n      1.0\n     ],\n     [\n      63.0,\n      0.0,\n      0.0\n     ],\n     [\n      63.0,\n      0.0,\n      0.0\n     ],\n     [\n      62.0,\n      0.0,\n      1.0\n     ],\n     [\n      58.0,\n      0.0,\n      4.0\n     ],\n     [\n      51.0,\n      0.0,\n      7.0\n     ],\n     [\n      50.0,\n      0.0,\n      1.0\n     ],\n     [\n      34.0,\n      0.0,\n      16.0\n     ],\n     [\n      15.0,\n      0.0,\n      19.0\n     ],\n     [\n      8.0,\n      0.0,\n      7.0\n     ],\n     [\n      6.0,\n      0.0,\n      2.0\n     ]\n    ]\n   ],\n   "poi_ids": [\n    "relation/149242",\n    "relation/149243",\n    "relation/149244",\n    "relation/164345",\n    "relation/1393275",\n    "relation/1394790",\n    "relation/1397821",\n    "relation/1410498",\n    "relation/1606951",\n    "relation/1708760",\n    "relation/1761513",\n    "relation/1787566",\n    "relation/2386975",\n    "relation/2801667",\n    "relation/3551173",\n    "relation/3618426",\n    "relation/3789120",\n    "relation/5424719",\n    "relation/5428217",\n    "relation/5428219",\n    "relation/5428220",\n    "relation/5642647",\n    "relation/6414818",\n    "relation/6780160",\n    "relation/6780161",\n    "relation/7895066",\n    "relation/7895073",\n    "relation/9201158",\n    "relation/9620168",\n    "relation/10433069",\n    "relation/10976811",\n    "relation/10976813",\n    "relation/10976815",\n    "relation/10976817",\n    "relation/10976819",\n    "relation/10991376",\n    "relation/11084392",\n    "relation/11084394",\n    "relation/11092579",\n    "relation/11208825",\n    "relation/11257406",\n    "relation/11344329",\n    "relation/11345933",\n    "relation/11345934",\n    "relation/11679775",\n    "relation/11855557",\n    "relation/11855559",\n    "relation/11863749",\n    "relation/12432667",\n    "relation/12743120",\n    "relation/12743122",\n    "relation/12743123",\n    "relation/13353101",\n    "relation/13533473",\n    "relation/13783336",\n    "relation/13968086",\n    "relation/13968260",\n    "relation/14110459",\n    "relation/14209002",\n    "relation/14212251",\n    "relation/14212253",\n    "relation/14213232",\n    "relation/14214913",\n    "relation/14239439",\n    "relation/14271754",\n    "relation/14760679",\n    "relation/14923956",\n    "relation/14924020",\n    "relation/14930869",\n    "relation/14930872",\n    "relation/14946967",\n    "relation/14946968",\n    "relation/14946969",\n    "relation/14966578",\n    "relation/15025366",\n    "relation/15025368",\n    "relation/15046087",\n    "relation/15046094",\n    "relation/15046096",\n    "relation/15047543",\n    "relation/15047549",\n    "relation/15047550",\n    "relation/15057044",\n    "relation/15080647",\n    "relation/15080649",\n    "relation/15326525",\n    "relation/15327986",\n    "relation/15342436",\n    "relation/15537987",\n    "relation/15537988",\n    "relation/15537989",\n    "relation/15844900",\n    "relation/16109093",\n    "relation/16109097",\n    "relation/16109098",\n    "relation/16109099",\n    "relation/16109101",\n    "relation/16238171",\n    "relation/16380400",\n    "relation/16380406",\n    "relation/16421395",\n    "relation/16421396",\n    "relation/16421397",\n    "relation/17024308",\n    "relation/17024309",\n    "relation/17051423",\n    "relation/17051424",\n    "relation/17051425",\n    "relation/17051426",\n    "relation/17088305",\n    "relation/17088308",\n    "relation/17305925",\n    "relation/17309820",\n    "relation/17704247",\n    "relation/17940120",\n    "relation/18267726",\n    "relation/18274494",\n    "relation/18274495",\n    "relation/18274496",\n    "relation/18293383",\n    "way/28684103",\n    "way/28860303",\n    "way/29854193",\n    "way/29854197",\n    "way/29854216",\n    "way/31891521",\n    "way/35159721",\n    "way/35168339",\n    "way/35168341",\n    "way/35168343",\n    "way/35168352",\n    "way/35168355",\n    "way/35168374",\n    "way/35168375",\n    "way/35168378",\n    "way/35168379",\n    "way/35168380",\n    "way/35168383",\n    "way/35168391",\n    "way/35168393",\n    "way/35168396",\n    "way/35168399",\n    "way/35168404",\n    "way/35168409",\n    "way/35168417",\n    "way/35610495",\n    "way/35610503",\n    "way/35610504",\n    "way/35610506",\n    "way/35610513",\n    "way/36419089",\n    "way/36812846",\n    "way/37001618",\n    "way/37001671",\n    "way/37001672",\n    "way/37002213",\n    "way/37130969",\n    "way/37130977",\n    "way/37130979",\n    "way/37130980",\n    "way/37130982",\n    "way/37130984",\n    "way/37130985",\n    "way/37130990",\n    "way/37130991",\n    "way/37130993",\n    "way/37453476",\n    "way/37735415",\n    "way/37756919",\n    "way/37756923",\n    "way/37756924",\n    "way/37756966",\n    "way/37757008",\n    "way/37757037",\n    "way/37757039",\n    "way/37757051",\n    "way/37764334",\n    "way/37914835",\n    "way/38573818",\n    "way/38573823",\n    "way/38573825",\n    "way/38966159",\n    "way/40398160",\n    "way/40398164",\n    "way/40604732",\n    "way/42921011",\n    "way/42921013",\n    "way/42921014",\n    "way/44353019",\n    "way/44771613",\n    "way/44771617",\n    "way/44778324",\n    "way/44778327",\n    "way/45105693",\n    "way/45782178",\n    "way/45829255",\n    "way/45829258",\n    "way/46414092",\n    "way/46414098",\n    "way/46603828",\n    "way/46603838",\n    "way/46737429",\n    "way/46739091",\n    "way/46739092",\n    "way/46739093",\n    "way/46739099",\n    "way/46740665",\n    "way/46740669",\n    "way/46740674",\n    "way/46740675",\n    "way/46740679",\n    "way/46740695",\n    "way/46740699",\n    "way/46740700",\n    "way/46740701",\n    "way/46740714",\n    "way/46740716",\n    "way/46740725",\n    "way/46854271",\n    "way/47066458",\n    "way/47066460",\n    "way/48296120",\n    "way/48296123",\n    "way/48582176",\n    "way/48582193",\n    "way/48585189",\n    "way/48585194",\n    "way/48585197",\n    "way/48648090",\n    "way/48905239",\n    "way/49591455",\n    "way/49798329",\n    "way/49799926",\n    "way/50654372",\n    "way/50833973",\n    "way/50833977",\n    "way/50833986",\n    "way/51004884",\n    "way/51004890",\n    "way/51045653",\n    "way/52832290",\n    "way/53725264",\n    "way/53888272",\n    "way/53889555",\n    "way/55015763",\n    "way/55272247",\n    "way/79246883",\n    "way/79561179",\n    "way/80746007",\n    "way/82682631",\n    "way/82682675",\n    "way/83097795",\n    "way/85929967",\n    "way/85929970",\n    "way/86985775",\n    "way/86985778",\n    "way/86985781",\n    "way/86985813",\n    "way/87349420",\n    "way/87349435",\n    "way/87349444",\n    "way/87349450",\n    "way/87349453",\n    "way/87349460",\n    "way/87349462",\n    "way/87349465",\n    "way/88408309",\n    "way/88418092",\n    "way/88418093",\n    "way/88418094",\n    "way/88423799",\n    "way/88423833",\n    "way/88436452",\n    "way/88436461",\n    "way/88440977",\n    "way/88440985",\n    "way/90982315",\n    "way/90982316",\n    "way/90982326",\n    "way/95665378",\n    "way/96255256",\n    "way/98213311",\n    "way/98213372",\n    "way/98572152",\n    "way/98572171",\n    "way/98572179",\n    "way/98572184",\n    "way/98572185",\n    "way/98572194",\n    "way/98572204",\n    "way/98572213",\n    "way/98572229",\n    "way/98572245",\n    "way/98572249",\n    "way/98572257",\n    "way/98572258",\n    "way/98572264",\n    "way/98572271",\n    "way/98572274",\n    "way/98572275",\n    "way/98572280",\n    "way/98572293",\n    "way/98572298",\n    "way/100054548",\n    "way/100134908",\n    "way/100134912",\n    "way/100134915",\n    "way/104122834",\n    "way/104708946",\n    "way/105658741",\n    "way/107673086",\n    "way/107699032",\n    "way/107699037",\n    "way/107699047",\n    "way/107699064",\n    "way/107699071",\n    "way/109081780",\n    "way/109089979",\n    "way/109127643",\n    "way/109127646",\n    "way/109127674",\n    "way/109127683",\n    "way/109200889",\n    "way/109939339",\n    "way/110275254",\n    "way/111601797",\n    "way/112358863",\n    "way/115750806",\n    "way/115750909",\n    "way/118939820",\n    "way/118939826",\n    "way/118939827",\n    "way/123314617",\n    "way/124365454",\n    "way/124365460",\n    "way/124365465",\n    "way/124365466",\n    "way/124365467",\n    "way/124365469",\n    "way/125105439",\n    "way/125111829",\n    "way/125667351",\n    "way/125667369",\n    "way/125674933",\n    "way/125674936",\n    "way/125674939",\n    "way/125674944",\n    "way/125674949",\n    "way/125674950",\n    "way/125674951",\n    "way/125674958",\n    "way/125849315",\n    "way/125849327",\n    "way/125849335",\n    "way/126059620",\n    "way/127632239",\n    "way/127632245",\n    "way/127717485",\n    "way/127875665",\n    "way/128264421",\n    "way/129244134",\n    "way/129244579",\n    "way/129920063",\n    "way/129920064",\n    "way/129920094",\n    "way/130645096",\n    "way/130706112",\n    "way/131833399",\n    "way/132066885",\n    "way/132193355",\n    "way/132193364",\n    "way/139055063",\n    "way/139069935",\n    "way/139880865",\n    "way/143189426",\n    "way/143189428",\n    "way/146926420",\n    "way/148787765",\n    "way/149864235",\n    "way/149864236",\n    "way/150134797",\n    "way/150824915",\n    "way/150824920",\n    "way/150824925",\n    "way/150824928",\n    "way/150824929",\n    "way/154363966",\n    "way/155794664",\n    "way/170081763",\n    "way/172549518",\n    "way/172839669",\n    "way/172839675",\n    "way/172839706",\n    "way/172839709",\n    "way/175187062",\n    "way/176025431",\n    "way/177657595",\n    "way/177657627",\n    "way/177913374",\n    "way/177913376",\n    "way/178163702",\n    "way/178199299",\n    "way/178199318",\n    "way/178199321",\n    "way/178558387",\n    "way/179341739",\n    "way/197511591",\n    "way/197511711",\n    "way/197891060",\n    "way/197929393",\n    "way/200610180",\n    "way/214676836",\n    "way/224343561",\n    "way/249104978",\n    "way/261752407",\n    "way/299782864",\n    "way/299782868",\n    "way/299782872",\n    "way/318858405",\n    "way/323607951",\n    "way/331353941",\n    "way/336407970",\n    "way/362310589",\n    "way/363128565",\n    "way/363128567",\n    "way/363128568",\n    "way/372068255",\n    "way/375867350",\n    "way/387617423",\n    "way/387694869",\n    "way/387694871",\n    "way/387694872",\n    "way/387694873",\n    "way/387694874",\n    "way/387694897",\n    "way/389448515",\n    "way/390670593",\n    "way/446071847",\n    "way/448961881",\n    "way/468011389",\n    "way/489066735",\n    "way/509948193",\n    "way/515672089",\n    "way/520635412",\n    "way/539238923",\n    "way/581636269",\n    "way/609945076",\n    "way/616867750",\n    "way/625292030",\n    "way/679418353",\n    "way/684545249",\n    "way/706495344",\n    "way/706495356",\n    "way/713796365",\n    "way/737401378",\n    "way/760072789",\n    "way/770533814",\n    "way/770533832",\n    "way/818074286",\n    "way/832137711",\n    "way/832137713",\n    "way/835466032",\n    "way/852078756",\n    "way/865594604",\n    "way/868244171",\n    "way/875297806",\n    "way/887983715",\n    "way/887983716",\n    "way/889430324",\n    "way/943013022",\n    "way/981382157",\n    "way/986672025",\n    "way/986672026",\n    "way/1066253622",\n    "way/1171996570",\n    "way/1171996571",\n    "way/1188275504",\n    "way/1216209524",\n    "way/1216391945",\n    "way/1220466087",\n    "way/1231772485",\n    "way/1252904101",\n    "way/1255506148",\n    "way/1255607274",\n    "way/1257342119",\n    "way/1257342120",\n    "node/432502061",\n    "node/432507349",\n    "node/698892153",\n    "node/698906226",\n    "node/988853638",\n    "node/994054736",\n    "node/1325052029",\n    "node/1745421983",\n    "node/1745422061",\n    "node/1902113640",\n    "node/1909859232",\n    "node/1909859236",\n    "node/1952548173",\n    "node/1966876890",\n    "node/1997933514",\n    "node/2032412951",\n    "node/2105789042",\n    "node/2173372347",\n    "node/2349301394",\n    "node/2349771982",\n    "node/2355462419",\n    "node/2355593478",\n    "node/2639575390",\n    "node/2670169698",\n    "node/2673526388",\n    "node/2922768877",\n    "node/2922768880",\n    "node/3280224135",\n    "node/3678839337",\n    "node/3704170104",\n    "node/3754134881",\n    "node/3800654308",\n    "node/3887580581",\n    "node/3897252405",\n    "node/3922372306",\n    "node/3939278827",\n    "node/3980755657",\n    "node/4043907864",\n    "node/4177235410",\n    "node/4182155695",\n    "node/4203797291",\n    "node/4208738076",\n    "node/4301803493",\n    "node/4301832092",\n    "node/4316261493",\n    "node/4332145990",\n    "node/4354203194",\n    "node/4363303418",\n    "node/4393092690",\n    "node/4437338403",\n    "node/4453027689",\n    "node/4547855937",\n    "node/4569239515",\n    "node/4574264191",\n    "node/4575703595",\n    "node/4577713875",\n    "node/4579337813",\n    "node/4579337816",\n    "node/4580200734",\n    "node/4580223125",\n    "node/4618650343",\n    "node/4618650363",\n    "node/4618650596",\n    "node/4646432090",\n    "node/4678212193",\n    "node/4682287951",\n    "node/4682401092",\n    "node/4682504546",\n    "node/4682603734",\n    "node/4682603735",\n    "node/4682610556",\n    "node/4682618559",\n    "node/4682618560",\n    "node/4683435865",\n    "node/4684137693",\n    "node/4684141141",\n    "node/4684287046",\n    "node/4684318336",\n    "node/4684337962",\n    "node/4716381689",\n    "node/4801212317",\n    "node/4948278023",\n    "node/5048493221",\n    "node/5081954314",\n    "node/5081954938",\n    "node/5086138172",\n    "node/5106953143",\n    "node/5194103200",\n    "node/5196240023",\n    "node/5219421023",\n    "node/5225926322",\n    "node/5240254511",\n    "node/5249589547",\n    "node/5334710944",\n    "node/5352800144",\n    "node/5352800145",\n    "node/5429605321",\n    "node/5429613021",\n    "node/5460737324",\n    "node/5543690921",\n    "node/5548442522",\n    "node/5575972221",\n    "node/5653954289",\n    "node/5671111817",\n    "node/5749899138",\n    "node/5796971428",\n    "node/5796971429",\n    "node/5796971430",\n    "node/5835909578",\n    "node/5931299885",\n    "node/6010685695",\n    "node/6015673186",\n    "node/6076122186",\n    "node/6151264710",\n    "node/6301297192",\n    "node/6315572923",\n    "node/6328334086",\n    "node/6436000785",\n    "node/6485288284",\n    "node/6490175473",\n    "node/6556211065",\n    "node/6579966563",\n    "node/6580085386",\n    "node/6722440985",\n    "node/6735423307",\n    "node/6736223685",\n    "node/6869203547",\n    "node/6877568880",\n    "node/7034471305",\n    "node/7057095690",\n    "node/7136843594",\n    "node/7182315984",\n    "node/7185289331",\n    "node/7221169271",\n    "node/7243041495",\n    "node/7258009957",\n    "node/7265739617",\n    "node/7265739618",\n    "node/7265739649",\n    "node/7265739650",\n    "node/7271204253",\n    "node/7360141677",\n    "node/7397131040",\n    "node/7404805605",\n    "node/7404805606",\n    "node/7500915829",\n    "node/7551278126",\n    "node/7562040005",\n    "node/7562040006",\n    "node/7625250491",\n    "node/7625250492",\n    "node/7625250498",\n    "node/7625250528",\n    "node/7625250530",\n    "node/7625250531",\n    "node/7625250532",\n    "node/7625250535",\n    "node/7625250536",\n    "node/7625741852",\n    "node/7625741853",\n    "node/7625741854",\n    "node/7625741857",\n    "node/7625741858",\n    "node/7625741859",\n    "node/7625741860",\n    "node/7660544845",\n    "node/7661714953",\n    "node/7713961172",\n    "node/7734075230",\n    "node/7734075236",\n    "node/7734075238",\n    "node/7741235276",\n    "node/7741261510",\n    "node/7741269179",\n    "node/7755681969",\n    "node/7755681970",\n    "node/7755681971",\n    "node/7755681972",\n    "node/7755681976",\n    "node/7755709795",\n    "node/7755709796",\n    "node/7781680618",\n    "node/7816508277",\n    "node/7832365998",\n    "node/7875388134",\n    "node/7922165789",\n    "node/7942483224",\n    "node/7948226512",\n    "node/7948226513",\n    "node/8058669733",\n    "node/8058669734",\n    "node/8058706720",\n    "node/8068206547",\n    "node/8068206554",\n    "node/8068206555",\n    "node/8068212087",\n    "node/8068212152",\n    "node/8083744077",\n    "node/8085346796",\n    "node/8085346797",\n    "node/8099876060",\n    "node/8099897701",\n    "node/8099897702",\n    "node/8099897703",\n    "node/8100832461",\n    "node/8106211970",\n    "node/8106211971",\n    "node/8106211972",\n    "node/8106211973",\n    "node/8106211974",\n    "node/8110613001",\n    "node/8112003634",\n    "node/8146853174",\n    "node/8146893188",\n    "node/8148301498",\n    "node/8268932069",\n    "node/8268932080",\n    "node/8268934184",\n    "node/8270161408",\n    "node/8270161411",\n    "node/8281001166",\n    "node/8281001167",\n    "node/8307245121",\n    "node/8307464032",\n    "node/8308203711",\n    "node/8370848138",\n    "node/8377639676",\n    "node/8390987295",\n    "node/8391990768",\n    "node/8392053708",\n    "node/8475102306",\n    "node/8475102308",\n    "node/8488538015",\n    "node/8489068408",\n    "node/8499187314",\n    "node/8499207065",\n    "node/8499207079",\n    "node/8499207084",\n    "node/8511850969",\n    "node/8521191402",\n    "node/8555150389",\n    "node/8647827682",\n    "node/8647921326",\n    "node/8653270825",\n    "node/8657534455",\n    "node/8678169875",\n    "node/8683003076",\n    "node/8705321494",\n    "node/8713123471",\n    "node/8754760534",\n    "node/8754760535",\n    "node/8839129807",\n    "node/8839218905",\n    "node/8839218907",\n    "node/8839218919",\n    "node/8899871344",\n    "node/9077880586",\n    "node/9077896522",\n    "node/9110464473",\n    "node/9318813851",\n    "node/9496122327",\n    "node/9504534019",\n    "node/9519022396",\n    "node/9519022398",\n    "node/9602886016",\n    "node/9609061055",\n    "node/9612451976",\n    "node/9612453575",\n    "node/9612453587",\n    "node/9648120032",\n    "node/9718274584",\n    "node/9718274604",\n    "node/9718274912",\n    "node/9735580786",\n    "node/9807930725",\n    "node/9835568393",\n    "node/9862403304",\n    "node/9862403305",\n    "node/9862403306",\n    "node/9862403308",\n    "node/9885173596",\n    "node/9891382261",\n    "node/10029522905",\n    "node/10128641500",\n    "node/10198888583",\n    "node/10253751374",\n    "node/10255485712",\n    "node/10273072878",\n    "node/10274685442",\n    "node/10281368736",\n    "node/10282313977",\n    "node/10285492511",\n    "node/10286572967",\n    "node/10291017665",\n    "node/10563067487",\n    "node/10593741730",\n    "node/10646591013",\n    "node/10669808008",\n    "node/10669995285",\n    "node/10766576298",\n    "node/10786728122",\n    "node/10844094861",\n    "node/10922965252",\n    "node/11083605358",\n    "node/11083649936",\n    "node/11101213766",\n    "node/11126334601",\n    "node/11159472347",\n    "node/11171107790",\n    "node/11174761628",\n    "node/11227994353",\n    "node/11427615247",\n    "node/11435856876",\n    "node/11435889113",\n    "node/11435889114",\n    "node/11435889115",\n    "node/11453975747",\n    "node/11467550491",\n    "node/11467550492",\n    "node/11467550493",\n    "node/11504820232",\n    "node/11505370011",\n    "node/11505370023",\n    "node/11512192672",\n    "node/11515114859",\n    "node/11515114862",\n    "node/11520396947",\n    "node/11530721773",\n    "node/11593054742",\n    "node/11596428029",\n    "node/11596428055",\n    "node/11596428059",\n    "node/11596428060",\n    "node/11598676504",\n    "node/11598676518",\n    "node/11602527685",\n    "node/11648769074",\n    "node/11691624908",\n    "node/11696599928",\n    "node/11737235288",\n    "node/11780110802",\n    "node/11780110818",\n    "node/11782622913",\n    "node/11804699391",\n    "node/11873574783",\n    "node/11958118153",\n    "node/11959910969",\n    "node/11991903443",\n    "node/11992711526",\n    "node/12009505069",\n    "node/12010102101",\n    "node/12063084670",\n    "node/12096765864",\n    "node/12123900986",\n    "node/12141981518",\n    "node/12197697204",\n    "node/12238142129",\n    "node/12296421701",\n    "node/12392638205",\n    "node/12394748309",\n    "node/12430845798",\n    "node/12434071661",\n    "node/12438086987"\n   ]\n  },\n  "28": {\n   "buildings": 19,\n   "organisations": 62,\n   "tagged_hours": 26,\n   "parsed_hours": 25,\n   "hour_signals": [\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.5,\n      2.0,\n      0.0\n     ],\n     [\n      12.0,\n      10.0,\n      0.0\n     ],\n     [\n      24.0,\n      12.0,\n      0.0\n     ],\n     [\n      24.0,\n      0.0,\n      0.0\n     ],\n     [\n      24.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.25,\n      1.0,\n      3.0\n     ],\n     [\n      23.25,\n      3.0,\n      1.0\n     ],\n     [\n      24.0,\n      0.0,\n      0.0\n     ],\n     [\n      24.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.5,\n      0.0,\n      2.0\n     ],\n     [\n      13.0,\n      0.0,\n      9.0\n     ],\n     [\n      11.0,\n      0.0,\n      2.0\n     ],\n     [\n      6.0,\n      0.0,\n      5.0\n     ],\n     [\n      3.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.5,\n      2.0,\n      0.0\n     ],\n     [\n      11.0,\n      9.0,\n      0.0\n     ],\n     [\n      24.0,\n      13.0,\n      0.0\n     ],\n     [\n      25.0,\n      1.0,\n      0.0\n     ],\n     [\n      25.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.25,\n      1.0,\n      4.0\n     ],\n     [\n      23.25,\n      3.0,\n      1.0\n     ],\n     [\n      24.0,\n      0.0,\n      0.0\n     ],\n     [\n      24.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.5,\n      0.0,\n      2.0\n     ],\n     [\n      14.0,\n      0.0,\n      8.0\n     ],\n     [\n      12.0,\n      0.0,\n      2.0\n     ],\n     [\n      6.0,\n      0.0,\n      6.0\n     ],\n     [\n      3.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.5,\n      2.0,\n      0.0\n     ],\n     [\n      12.0,\n      10.0,\n      0.0\n     ],\n     [\n      24.0,\n      12.0,\n      0.0\n     ],\n     [\n      24.0,\n      0.0,\n      0.0\n     ],\n     [\n      24.0,\n      0.0,\n      0.0\n     ],\n     [\n      20.25,\n      1.0,\n      4.0\n     ],\n     [\n      23.0,\n      2.0,\n      0.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.5,\n      0.0,\n      2.0\n     ],\n     [\n      13.0,\n      0.0,\n      8.0\n     ],\n     [\n      11.5,\n      1.0,\n      2.0\n     ],\n     [\n      7.0,\n      0.0,\n      5.0\n     ],\n     [\n      3.0,\n      0.0,\n      4.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.5,\n      2.0,\n      0.0\n     ],\n     [\n      11.0,\n      9.0,\n      0.0\n     ],\n     [\n      22.0,\n      11.0,\n      0.0\n     ],\n     [\n      23.0,\n      1.0,\n      0.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.25,\n      1.0,\n      2.0\n     ],\n     [\n      22.25,\n      2.0,\n      1.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.5,\n      0.0,\n      1.0\n     ],\n     [\n      14.0,\n      0.0,\n      8.0\n     ],\n     [\n      12.0,\n      0.0,\n      2.0\n     ],\n     [\n      6.0,\n      0.0,\n      6.0\n     ],\n     [\n      3.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.5,\n      2.0,\n      0.0\n     ],\n     [\n      12.0,\n      10.0,\n      0.0\n     ],\n     [\n      23.0,\n      11.0,\n      0.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      21.25,\n      1.0,\n      2.0\n     ],\n     [\n      22.25,\n      2.0,\n      1.0\n     ],\n     [\n      23.0,\n      0.0,\n      0.0\n     ],\n     [\n      22.75,\n      0.0,\n      1.0\n     ],\n     [\n      20.5,\n      0.0,\n      2.0\n     ],\n     [\n      13.0,\n      0.0,\n      7.0\n     ],\n     [\n      10.0,\n      0.0,\n      3.0\n     ],\n     [\n      6.0,\n      0.0,\n      4.0\n     ],\n     [\n      3.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.0,\n      1.0,\n      0.0\n     ],\n     [\n      4.0,\n      3.0,\n      0.0\n     ],\n     [\n      12.0,\n      8.0,\n      0.0\n     ],\n     [\n      12.0,\n      0.0,\n      0.0\n     ],\n     [\n      12.0,\n      0.0,\n      0.0\n     ],\n     [\n      12.0,\n      0.0,\n      0.0\n     ],\n     [\n      12.0,\n      0.0,\n      0.0\n     ],\n     [\n      12.0,\n      0.0,\n      0.0\n     ],\n     [\n      12.0,\n      0.0,\n      0.0\n     ],\n     [\n      12.0,\n      0.0,\n      0.0\n     ],\n     [\n      10.0,\n      0.0,\n      2.0\n     ],\n     [\n      8.0,\n      0.0,\n      2.0\n     ],\n     [\n      5.0,\n      0.0,\n      3.0\n     ],\n     [\n      3.0,\n      0.0,\n      2.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ],\n     [\n      1.0,\n      1.0,\n      0.0\n     ],\n     [\n      4.0,\n      3.0,\n      0.0\n     ],\n     [\n      11.0,\n      7.0,\n      0.0\n     ],\n     [\n      11.0,\n      0.0,\n      0.0\n     ],\n     [\n      11.0,\n      0.0,\n      0.0\n     ],\n     [\n      11.0,\n      0.0,\n      0.0\n     ],\n     [\n      10.25,\n      1.0,\n      1.0\n     ],\n     [\n      11.0,\n      0.0,\n      0.0\n     ],\n     [\n      10.0,\n      0.0,\n      1.0\n     ],\n     [\n      10.0,\n      0.0,\n      0.0\n     ],\n     [\n      9.0,\n      0.0,\n      1.0\n     ],\n     [\n      8.0,\n      0.0,\n      1.0\n     ],\n     [\n      5.0,\n      0.0,\n      3.0\n     ],\n     [\n      3.0,\n      0.0,\n      2.0\n     ],\n     [\n      0.0,\n      0.0,\n      3.0\n     ],\n     [\n      0.0,\n      0.0,\n      0.0\n     ]\n    ]\n   ],\n   "poi_ids": [\n    "relation/1819045",\n    "relation/2018324",\n    "relation/8118911",\n    "relation/11254965",\n    "relation/13486046",\n    "relation/13486048",\n    "relation/13687010",\n    "relation/17734525",\n    "relation/17734526",\n    "relation/17734527",\n    "relation/17734537",\n    "way/27715421",\n    "way/27800666",\n    "way/27830835",\n    "way/29276623",\n    "way/32203093",\n    "way/40687051",\n    "way/98787683",\n    "way/108201825",\n    "way/129435121",\n    "way/163553887",\n    "way/163638649",\n    "way/335746189",\n    "way/452293894",\n    "way/452418764",\n    "way/766569237",\n    "node/610804802",\n    "node/1287590471",\n    "node/1599802680",\n    "node/1599802683",\n    "node/1599802688",\n    "node/2355581434",\n    "node/3952575085",\n    "node/4242693635",\n    "node/4244512793",\n    "node/4264449792",\n    "node/4346384015",\n    "node/4434642700",\n    "node/4442018892",\n    "node/4549854109",\n    "node/4549854112",\n    "node/4549854113",\n    "node/4554204978",\n    "node/4682566247",\n    "node/4682566248",\n    "node/4682566249",\n    "node/4682566250",\n    "node/4879931398",\n    "node/4912077944",\n    "node/4912077945",\n    "node/5351454460",\n    "node/5375745714",\n    "node/5378271785",\n    "node/5452118923",\n    "node/5863435089",\n    "node/5890514897",\n    "node/5890514898",\n    "node/6024715390",\n    "node/6330254245",\n    "node/6371947385",\n    "node/6482038987",\n    "node/6878281232",\n    "node/7120352885",\n    "node/7198100308",\n    "node/7198100309",\n    "node/8356769817",\n    "node/8356769917",\n    "node/8356769918",\n    "node/8679297322",\n    "node/8870941617",\n    "node/9612450160",\n    "node/9612450242",\n    "node/9612450307",\n    "node/9612453154",\n    "node/9612453706",\n    "node/10012848682",\n    "node/10566781190",\n    "node/10671874146",\n    "node/10870883807",\n    "node/11876207410",\n    "node/11991903405"\n   ]\n  },\n  "26": {\n   "buildings": 172,\n   "organisations": 177,\n   "tagged_hours": 86,\n   "parsed_hours": 83,\n   "hour_signals": [\n    [\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      9.5,\n      8.0,\n      0.0\n     ],\n     [\n      50.5,\n      42.0,\n      0.0\n     ],\n     [\n      77.0,\n      25.0,\n      0.0\n     ],\n     [\n      79.0,\n      2.0,\n      0.0\n     ],\n     [\n      77.75,\n      0.0,\n      3.0\n     ],\n     [\n      71.25,\n      2.0,\n      7.0\n     ],\n     [\n      75.5,\n      8.0,\n      2.0\n     ],\n     [\n      78.0,\n      2.0,\n      1.0\n     ],\n     [\n      76.0,\n      0.0,\n      2.0\n     ],\n     [\n      67.0,\n      0.0,\n      10.0\n     ],\n     [\n      45.5,\n      0.0,\n      21.0\n     ],\n     [\n      37.0,\n      0.0,\n      8.0\n     ],\n     [\n      26.5,\n      0.0,\n      11.0\n     ],\n     [\n      8.0,\n      0.0,\n      18.0\n     ],\n     [\n      2.0,\n      0.0,\n      6.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      9.5,\n      8.0,\n      0.0\n     ],\n     [\n      50.5,\n      42.0,\n      0.0\n     ],\n     [\n      78.0,\n      26.0,\n      0.0\n     ],\n     [\n      80.0,\n      2.0,\n      0.0\n     ],\n     [\n      78.75,\n      0.0,\n      3.0\n     ],\n     [\n      71.25,\n      2.0,\n      8.0\n     ],\n     [\n      76.5,\n      9.0,\n      2.0\n     ],\n     [\n      78.0,\n      1.0,\n      1.0\n     ],\n     [\n      76.0,\n      0.0,\n      2.0\n     ],\n     [\n      68.0,\n      0.0,\n      9.0\n     ],\n     [\n      45.5,\n      0.0,\n      22.0\n     ],\n     [\n      37.0,\n      0.0,\n      8.0\n     ],\n     [\n      26.5,\n      0.0,\n      11.0\n     ],\n     [\n      8.0,\n      0.0,\n      18.0\n     ],\n     [\n      2.0,\n      0.0,\n      6.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      9.5,\n      8.0,\n      0.0\n     ],\n     [\n      51.5,\n      43.0,\n      0.0\n     ],\n     [\n      79.5,\n      27.0,\n      0.0\n     ],\n     [\n      81.0,\n      1.0,\n      0.0\n     ],\n     [\n      79.75,\n      0.0,\n      3.0\n     ],\n     [\n      71.25,\n      2.0,\n      9.0\n     ],\n     [\n      77.25,\n      8.0,\n      1.0\n     ],\n     [\n      78.0,\n      1.0,\n      1.0\n     ],\n     [\n      76.0,\n      0.0,\n      2.0\n     ],\n     [\n      67.0,\n      0.0,\n      10.0\n     ],\n     [\n      44.5,\n      0.0,\n      22.0\n     ],\n     [\n      36.0,\n      0.0,\n      8.0\n     ],\n     [\n      26.5,\n      0.0,\n      10.0\n     ],\n     [\n      8.0,\n      0.0,\n      18.0\n     ],\n     [\n      2.0,\n      0.0,\n      6.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      9.5,\n      8.0,\n      0.0\n     ],\n     [\n      50.5,\n      42.0,\n      0.0\n     ],\n     [\n      78.5,\n      27.0,\n      0.0\n     ],\n     [\n      81.0,\n      2.0,\n      0.0\n     ],\n     [\n      79.75,\n      0.0,\n      3.0\n     ],\n     [\n      72.25,\n      2.0,\n      8.0\n     ],\n     [\n      77.5,\n      9.0,\n      2.0\n     ],\n     [\n      79.0,\n      1.0,\n      1.0\n     ],\n     [\n      77.0,\n      0.0,\n      2.0\n     ],\n     [\n      68.0,\n      0.0,\n      10.0\n     ],\n     [\n      45.5,\n      0.0,\n      22.0\n     ],\n     [\n      37.0,\n      0.0,\n      8.0\n     ],\n     [\n      26.5,\n      0.0,\n      11.0\n     ],\n     [\n      8.0,\n      0.0,\n      18.0\n     ],\n     [\n      2.0,\n      0.0,\n      6.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      9.5,\n      8.0,\n      0.0\n     ],\n     [\n      51.5,\n      43.0,\n      0.0\n     ],\n     [\n      79.5,\n      27.0,\n      0.0\n     ],\n     [\n      80.0,\n      0.0,\n      0.0\n     ],\n     [\n      78.75,\n      0.0,\n      3.0\n     ],\n     [\n      71.25,\n      2.0,\n      8.0\n     ],\n     [\n      76.5,\n      9.0,\n      2.0\n     ],\n     [\n      78.5,\n      1.0,\n      2.0\n     ],\n     [\n      71.0,\n      0.0,\n      10.0\n     ],\n     [\n      59.75,\n      0.0,\n      8.0\n     ],\n     [\n      41.0,\n      0.0,\n      18.0\n     ],\n     [\n      36.0,\n      0.0,\n      5.0\n     ],\n     [\n      26.5,\n      0.0,\n      10.0\n     ],\n     [\n      8.0,\n      0.0,\n      18.0\n     ],\n     [\n      2.0,\n      0.0,\n      6.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.5,\n      5.0,\n      0.0\n     ],\n     [\n      21.0,\n      14.0,\n      0.0\n     ],\n     [\n      42.5,\n      22.0,\n      0.0\n     ],\n     [\n      44.0,\n      1.0,\n      0.0\n     ],\n     [\n      44.0,\n      0.0,\n      0.0\n     ],\n     [\n      44.0,\n      0.0,\n      0.0\n     ],\n     [\n      44.25,\n      2.0,\n      1.0\n     ],\n     [\n      41.0,\n      0.0,\n      4.0\n     ],\n     [\n      39.0,\n      0.0,\n      2.0\n     ],\n     [\n      36.0,\n      0.0,\n      3.0\n     ],\n     [\n      31.0,\n      0.0,\n      5.0\n     ],\n     [\n      28.0,\n      0.0,\n      3.0\n     ],\n     [\n      22.5,\n      0.0,\n      6.0\n     ],\n     [\n      8.0,\n      0.0,\n      14.0\n     ],\n     [\n      2.0,\n      0.0,\n      6.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ]\n    ],\n    [\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ],\n     [\n      6.5,\n      5.0,\n      0.0\n     ],\n     [\n      19.0,\n      12.0,\n      0.0\n     ],\n     [\n      34.5,\n      16.0,\n      0.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      35.0,\n      0.0,\n      0.0\n     ],\n     [\n      32.0,\n      0.0,\n      3.0\n     ],\n     [\n      31.0,\n      0.0,\n      1.0\n     ],\n     [\n      28.0,\n      0.0,\n      3.0\n     ],\n     [\n      22.5,\n      0.0,\n      6.0\n     ],\n     [\n      8.0,\n      0.0,\n      14.0\n     ],\n     [\n      2.0,\n      0.0,\n      6.0\n     ],\n     [\n      2.0,\n      0.0,\n      0.0\n     ]\n    ]\n   ],\n   "poi_ids": [\n    "relation/380619",\n    "relation/1112174",\n    "relation/2024252",\n    "relation/2024253",\n    "relation/2167545",\n    "relation/2849452",\n    "relation/2849453",\n    "relation/2849454",\n    "relation/2849458",\n    "relation/3683969",\n    "relation/6090248",\n    "relation/6675398",\n    "relation/6913343",\n    "relation/10969772",\n    "relation/11380679",\n    "relation/13450864",\n    "relation/13450866",\n    "relation/17647229",\n    "relation/18134838",\n    "relation/18134840",\n    "relation/18176576",\n    "relation/18178315",\n    "way/29128003",\n    "way/29128009",\n    "way/30098150",\n    "way/30098648",\n    "way/30385694",\n    "way/32606552",\n    "way/32609212",\n    "way/35559239",\n    "way/35559313",\n    "way/35559537",\n    "way/35559683",\n    "way/35559684",\n    "way/35560443",\n    "way/35574498",\n    "way/35574613",\n    "way/35574639",\n    "way/35574640",\n    "way/35574646",\n    "way/35574731",\n    "way/35574739",\n    "way/35574741",\n    "way/35574743",\n    "way/35574830",\n    "way/35574912",\n    "way/35667968",\n    "way/35774131",\n    "way/35977722",\n    "way/36507990",\n    "way/36508120",\n    "way/39129833",\n    "way/40009388",\n    "way/40703530",\n    "way/40735575",\n    "way/40813032",\n    "way/45819989",\n    "way/45819990",\n    "way/45820297",\n    "way/45820299",\n    "way/45848495",\n    "way/46094867",\n    "way/46094874",\n    "way/46094878",\n    "way/46570571",\n    "way/46657148",\n    "way/46657149",\n    "way/48056959",\n    "way/48135360",\n    "way/48399949",\n    "way/48400215",\n    "way/48400598",\n    "way/48401008",\n    "way/48401017",\n    "way/48401043",\n    "way/48504331",\n    "way/48504405",\n    "way/48912247",\n    "way/48912249",\n    "way/48918622",\n    "way/49343427",\n    "way/50120312",\n    "way/50120318",\n    "way/50120320",\n    "way/50120322",\n    "way/50120323",\n    "way/50120327",\n    "way/51760927",\n    "way/53749261",\n    "way/53754817",\n    "way/60453478",\n    "way/68744733",\n    "way/68744734",\n    "way/74076054",\n    "way/83894557",\n    "way/89571699",\n    "way/92337937",\n    "way/95200175",\n    "way/95200178",\n    "way/95200179",\n    "way/95200534",\n    "way/97992224",\n    "way/102225977",\n    "way/102226041",\n    "way/102226052",\n    "way/102226082",\n    "way/102226084",\n    "way/102226093",\n    "way/102226095",\n    "way/102226118",\n    "way/105407861",\n    "way/105407864",\n    "way/117419270",\n    "way/122228837",\n    "way/122229818",\n    "way/123587403",\n    "way/124697446",\n    "way/126318340",\n    "way/129371700",\n    "way/130002060",\n    "way/137928481",\n    "way/137928484",\n    "way/137928490",\n    "way/142638120",\n    "way/157320183",\n    "way/160272396",\n    "way/160272397",\n    "way/160272398",\n    "way/160277907",\n    "way/163008908",\n    "way/163057935",\n    "way/171683457",\n    "way/177325783",\n    "way/177944582",\n    "way/177944584",\n    "way/223645731",\n    "way/261381559",\n    "way/261381562",\n    "way/261381563",\n    "way/261381564",\n    "way/261381569",\n    "way/262199961",\n    "way/262199962",\n    "way/294431543",\n    "way/294431554",\n    "way/294431555",\n    "way/294431557",\n    "way/294431562",\n    "way/294431564",\n    "way/307039429",\n    "way/308967740",\n    "way/308967741",\n    "way/325937079",\n    "way/326009061",\n    "way/326009067",\n    "way/335317730",\n    "way/338419954",\n    "way/361034529",\n    "way/366054149",\n    "way/370487953",\n    "way/372651469",\n    "way/372651470",\n    "way/372651471",\n    "way/372756974",\n    "way/372756975",\n    "way/381837959",\n    "way/408786003",\n    "way/409865851",\n    "way/410014562",\n    "way/428108782",\n    "way/435604935",\n    "way/504240806",\n    "way/507418183",\n    "way/540515211",\n    "way/540515212",\n    "way/656592359",\n    "way/760014288",\n    "way/782266013",\n    "way/783028021",\n    "way/783201463",\n    "way/789024920",\n    "way/789024922",\n    "way/793424261",\n    "way/793424270",\n    "way/793646305",\n    "way/795000663",\n    "way/795321078",\n    "way/795321079",\n    "way/796671035",\n    "way/835611874",\n    "way/1188734283",\n    "node/954910738",\n    "node/954912010",\n    "node/954912352",\n    "node/956433457",\n    "node/2355462421",\n    "node/2355581439",\n    "node/2388112325",\n    "node/2432212418",\n    "node/2477516812",\n    "node/2749487425",\n    "node/2756094419",\n    "node/3059957847",\n    "node/3368762076",\n    "node/3860484646",\n    "node/3880376941",\n    "node/3939325503",\n    "node/3939325504",\n    "node/3957292291",\n    "node/4037407039",\n    "node/4132932199",\n    "node/4137283460",\n    "node/4176123392",\n    "node/4180536365",\n    "node/4225317014",\n    "node/4317397995",\n    "node/4335096541",\n    "node/4343079602",\n    "node/4353691892",\n    "node/4383676490",\n    "node/4383676545",\n    "node/4391394389",\n    "node/4452895668",\n    "node/4464050116",\n    "node/4464077126",\n    "node/4547863292",\n    "node/4563580825",\n    "node/4608993624",\n    "node/4611473952",\n    "node/4634744885",\n    "node/4682620043",\n    "node/4682623969",\n    "node/4682623970",\n    "node/4684158733",\n    "node/4776559971",\n    "node/4787660883",\n    "node/4787744731",\n    "node/4828467163",\n    "node/4888068431",\n    "node/4897208704",\n    "node/4940050093",\n    "node/5064014162",\n    "node/5081954934",\n    "node/5104499524",\n    "node/5155307425",\n    "node/5273471522",\n    "node/5302593721",\n    "node/5514969927",\n    "node/5523038842",\n    "node/5523060842",\n    "node/5523060851",\n    "node/5523082897",\n    "node/5523096737",\n    "node/5837199487",\n    "node/5963449992",\n    "node/6341309506",\n    "node/6355997059",\n    "node/6426282602",\n    "node/6426286691",\n    "node/6426308024",\n    "node/6490338685",\n    "node/6570666078",\n    "node/6570688310",\n    "node/6570689847",\n    "node/6570720215",\n    "node/6630668245",\n    "node/6652892407",\n    "node/6673777104",\n    "node/6703292063",\n    "node/6704055289",\n    "node/6737899540",\n    "node/6745453126",\n    "node/6806548487",\n    "node/6859731485",\n    "node/6912255979",\n    "node/6958036941",\n    "node/6958083744",\n    "node/6958090474",\n    "node/7082117595",\n    "node/7131079577",\n    "node/7149384091",\n    "node/7228749861",\n    "node/7327883169",\n    "node/7379039455",\n    "node/7507064184",\n    "node/7593315185",\n    "node/7593315186",\n    "node/7593357385",\n    "node/7596820418",\n    "node/7791040015",\n    "node/7832365441",\n    "node/7866619115",\n    "node/8386248940",\n    "node/8635394137",\n    "node/8635395211",\n    "node/8713146662",\n    "node/8789982293",\n    "node/8839129797",\n    "node/8895498792",\n    "node/8938901972",\n    "node/9237777917",\n    "node/9258217921",\n    "node/9258217922",\n    "node/9332889616",\n    "node/9342121434",\n    "node/9342121435",\n    "node/9351069491",\n    "node/9351069492",\n    "node/9413293248",\n    "node/9415302647",\n    "node/9610759342",\n    "node/9612450616",\n    "node/9612450617",\n    "node/9612452067",\n    "node/9629143241",\n    "node/9643677454",\n    "node/9685411290",\n    "node/9685427718",\n    "node/9791085033",\n    "node/9805728661",\n    "node/9826120335",\n    "node/9841433493",\n    "node/9933988521",\n    "node/10046688634",\n    "node/10281368779",\n    "node/10722898664",\n    "node/10723963436",\n    "node/10787918287",\n    "node/10788242012",\n    "node/10857574244",\n    "node/11164485904",\n    "node/11175623339",\n    "node/11188812120",\n    "node/11243922238",\n    "node/11335417354",\n    "node/11335428128",\n    "node/11553381321",\n    "node/11629962591",\n    "node/11736391876",\n    "node/11749637211",\n    "node/11781399788",\n    "node/11813596866",\n    "node/11824276467",\n    "node/11873574827",\n    "node/11948211150",\n    "node/11991887544",\n    "node/12099562006",\n    "node/12141981516",\n    "node/12273135989"\n   ]\n  }\n },\n "limitations": [\n  "Incomplete mapped inventory, not employee counts.",\n  "Building centres, approximate 500m radius; retrospective 2026 GTFS.",\n  "Buildings and organisations kept separate, never summed as workers.",\n  "Weekly public opening hours, not employee shifts; unsupported syntax unknown.",\n  "Holiday and transferred workday hours approximated by effective weekday."\n ]\n}\n', 'ml/forecast/office_competition.py': '"""Historical offices and public opening hours versus verified 0.88092 baseline."""\nimport argparse\nimport json\nimport pickle\nfrom datetime import date\nfrom pathlib import Path\nimport numpy as np\nfrom . import network_competition as nc\nfrom .network_regimes import state,STATE_NAMES,NORMAL\nfrom .hourly_experts import HourlyExperts,HourlyExpertEnsemble,combine\nfrom .weather_competition import corrected,weather_rows,WEIGHTS\nfrom .core import grid,load_labels,sha256\nfrom .run import FIRST,FINAL_CUTOFF\nfrom ml.submit.adapter import read_sample,write_submission\n\nfrom .education_features import calendar_features\nfrom .office_features import REGISTRY,density_features,hours_features\nfrom .education_profiles import EducationProfiles,Ensemble as SchoolEnsemble,join as school_join\nSCHOOL_CHOICES={r:(\'modular_0.5\' if r==\'28\' else \'incumbent\') for r in (\'1\',\'11\',\'12\',\'17\',\'25\',\'26\',\'28\')}\nBEST_HASH=\'471a8043f43e0d382bbceea0f99419b10db3f506e7a5120160be4b047d94bece\'\nTARGETS=(\'1\',\'11\',\'12\',\'17\',\'25\',\'26\',\'28\')\n\ndef active(key):return key[0] in TARGETS\n\ndef matrix(keys,base,weather,arm):\n    x=nc.features(keys,base,weather,True,True)\n    x=np.column_stack((x,np.asarray([calendar_features(k) for k in keys])))\n    if arm in (\'density\',\'hours\'):x=np.column_stack((x,np.asarray([density_features(k) for k in keys])))\n    if arm==\'hours\':x=np.column_stack((x,np.asarray([hours_features(k) for k in keys])))\n    return x\n\ndef fit_pair(bank,cutoff,weather):\n    import lightgbm as lgb\n    keys,bases,y,blocks=bank.training(cutoff);base=bases[\'network\']\n    scale=np.asarray([max(50,base[k]) for k in keys]);target=(y-np.asarray([base[k] for k in keys]))/scale\n    models={}\n    state_column=nc.features(keys[:1],base,weather,True,True).shape[1]-1\n    for name in (\'control\',\'density\',\'hours\'):\n        model=lgb.LGBMRegressor(objective=\'regression_l1\',n_estimators=220,learning_rate=.025,num_leaves=12,min_child_samples=120,reg_lambda=10.,max_depth=5,n_jobs=2,verbosity=-1,random_state=2025)\n        model.fit(matrix(keys,base,weather,name),target,sample_weight=scale,categorical_feature=[0,1,2,3,4,5,state_column])\n        models[name]=model\n    return models,blocks\n\ndef variants(keys,base,incumbent,models,weather):\n    result={\'incumbent\':incumbent}\n    for name in (\'control\',\'density\',\'hours\'):\n        raw=models[name].predict(matrix(keys,base,weather,name))\n        for w in WEIGHTS:\n            p=corrected(keys,base,raw,w)\n            result[f\'{name}_{w}\']={k:p[k] if active(k) else incumbent[k] for k in keys}\n    return result\n\ndef choose(keys,truth,preds,oct_keys,oct_preds):\n    selected={};audit={}\n    for route in TARGETS:\n        groups=[[k for k in keys if k[0]==route and k[1].month==m] for m in (9,10)]\n        errors={n:[sum(abs(p[k]-truth.get(k,0)) for k in g) for g in groups] for n,p in preds.items()}\n        oe={n:sum(abs(p[k]-truth.get(k,0)) for k in oct_keys if k[0]==route) for n,p in oct_preds.items()}\n        def beats(n,c):return all(a<=.995*b for a,b in zip(errors[n],errors[c])) and oe[n]<=1.005*oe[c]\n        calendars=[f\'density_{w}\' for w in WEIGHTS if beats(f\'density_{w}\',\'incumbent\') and beats(f\'density_{w}\',f\'control_{w}\')]\n        best_calendar=min([\'incumbent\']+calendars,key=lambda n:sum(errors[n]))\n        education=[f\'hours_{w}\' for w in WEIGHTS if beats(f\'hours_{w}\',\'incumbent\') and beats(f\'hours_{w}\',f\'density_{w}\') and beats(f\'hours_{w}\',best_calendar)]\n        selected[route]=min([\'incumbent\']+calendars+education,key=lambda n:sum(errors[n]))\n        audit[route]={\'errors_sep_oct\':errors,\'errors_october_refit\':oe,\'accepted_density\':calendars,\'accepted_hours\':education,\'selected\':selected[route]}\n    return selected,audit\n\ndef join(keys,preds,selected):return {k:preds[selected.get(k[0],\'incumbent\')][k] for k in keys}\n\nclass OfficeEnsemble:\n    def __init__(self,incumbent,reference,regimes,models,weather,selected):\n        self.incumbent,self.reference,self.regimes=incumbent,reference,regimes\n        self.models,self.weather,self.selected=models,weather,selected\n    def predict(self,keys):\n        keys=list(keys);old=self.incumbent.predict(keys)\n        base=self.regimes.apply(keys,self.reference.predict(keys))\n        replacements={k:(k[0],date(2025,11,2),k[2]) for k in keys\n                      if k[0] in (\'7\',\'50\') and k[1] in (date(2025,11,3),date(2025,11,4))}\n        proxies=list(dict.fromkeys(replacements.values()))\n        if proxies:\n            restricted=self.regimes.apply(proxies,self.reference.predict(proxies))\n            base.update({k:restricted[v] for k,v in replacements.items()})\n        return join(keys,variants(keys,base,old,self.models,self.weather),self.selected)\n\ndef main(argv=None):\n    ap=argparse.ArgumentParser()\n    for name in (\'train\',\'test\',\'sample\',\'weather\',\'baseline-dir\',\'out\'):ap.add_argument(\'--\'+name,type=Path,required=True)\n    ap.add_argument(\'--sample-from-reference\',action=\'store_true\');args=ap.parse_args(argv)\n    inputs=nc.verify_competition_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')},args.sample_from_reference)\n    values=load_labels([args.train,args.test]);assert all(FIRST<=k[1]<=FINAL_CUTOFF for k in values)\n    previous=json.loads((args.baseline_dir/\'hourly_experts_report.json\').read_text())\n    ref=previous[\'reference\'];choices=previous[\'selection\'][\'by_route\'];hourly=previous[\'hourly_experts\'][\'choices\']\n    weather=weather_rows(json.loads(args.weather.read_text()));bank=nc.HistoryBank(values,ref[\'choices\'])\n    args.out.mkdir(parents=True,exist_ok=True);folds={};held={}\n    for label,cutoff,start,end in [(\'sep_oct\',date(2025,8,31),date(2025,9,1),date(2025,10,31)),(\'october_refit\',date(2025,9,30),date(2025,10,1),date(2025,10,31))]:\n        print(\'Offices:\',label,flush=True)\n        keys=grid(bank.routes,start,end);bases=bank.predict_bases(cutoff,keys)\n        oldmodels,_=nc.fit_models(bank,cutoff,weather)\n        incumbent=nc.assemble(keys,nc.predict_models(keys,bases,oldmodels,weather),choices)\n        hp=HourlyExperts().fit({k:v for k,v in values.items() if k[1]<=cutoff},FIRST,cutoff)\n        incumbent=combine(keys,hp.variants(keys,incumbent),hourly)\n        expected=previous[\'folds\'][label if label==\'sep_oct\' else \'october_refit_31d\'][\'hourly_on_network_weather\'][\'selected\'][\'all\']\n        if nc.metrics(keys,values,incumbent)[\'all\']!=expected:raise RuntimeError(\'Historical baseline did not reproduce\')\n        history={k:v for k,v in values.items() if k[1]<=cutoff}\n        incumbent=school_join(keys,EducationProfiles().fit(history,cutoff).variants(keys,incumbent),SCHOOL_CHOICES)\n        expected_new={\'sep_oct\':.9063526771493093,\'october_refit\':.906346458573585}[label]\n        if abs(nc.metrics(keys,values,incumbent)[\'all\'][\'wape_score\']-expected_new)>1e-12:raise RuntimeError(\'School baseline mismatch\')\n        models,blocks=fit_pair(bank,cutoff,weather)\n        preds=variants(keys,bases[\'network\'],incumbent,models,weather)\n        held[label]=(keys,preds)\n        folds[label]={\'metrics\':{n:nc.metrics(keys,values,p) for n,p in preds.items()},\'training_blocks\':blocks}\n    keys,preds=held[\'sep_oct\'];oct_keys,oct_preds=held[\'october_refit\']\n    selected,audit=choose(keys,values,preds,oct_keys,oct_preds)\n    for label,(ks,ps) in held.items():folds[label][\'selected\']=nc.metrics(ks,values,join(ks,ps,selected))\n    # Fit all final model components from organizer labels, never load supplied pickle files.\n    _,sample=read_sample(args.sample);reference,regimes=bank.fitted(FINAL_CUTOFF)\n    oldmodels,_=nc.fit_models(bank,FINAL_CUTOFF,weather)\n    incumbent=nc.HolidayServiceEnsemble(nc.NetworkWeatherEnsemble(reference,regimes,oldmodels,weather,choices),reference,regimes)\n    incumbent=HourlyExpertEnsemble(incumbent,HourlyExperts().fit(values,FIRST,FINAL_CUTOFF),hourly)\n    incumbent=SchoolEnsemble(incumbent,EducationProfiles().fit(values,FINAL_CUTOFF),SCHOOL_CHOICES)\n    before=incumbent.predict(sample)\n    write_submission(args.sample,before,args.out/\'submission_best_088092.csv\')\n    if sha256(args.out/\'submission_best_088092.csv\')!=BEST_HASH:raise RuntimeError(\'Final baseline hash mismatch\')\n    models,_=fit_pair(bank,FINAL_CUTOFF,weather)\n    ensemble=OfficeEnsemble(incumbent,reference,regimes,models,weather,selected);after=ensemble.predict(sample)\n    assert all(after[k]==before[k] for k in sample if not active(k))\n    artifact=pickle.dumps(ensemble,protocol=5)\n    assert pickle.loads(artifact).predict(sample)==after\n    (args.out/\'model.local.pkl\').write_bytes(artifact)\n    output=write_submission(args.sample,after,args.out/\'submission_offices.csv\')\n    report={\'source_mode\':\'competition_retrospective\',\'inputs\':inputs,\'registry\':REGISTRY,\'selected\':selected,\'selection\':audit,\'folds\':folds,\'gate\':\'Density must beat incumbent and matched control by >=0.5% per month; opening hours must beat incumbent, matched density and best admitted density. Oct refit deterioration <=0.5%. Routes 7/50/5 protected.\',\'selection_warning\':\'Sep-Oct reused in research. This is exploratory validation, not an independent test.\',\'best_platform_score_user_reported\':.88092,\'new_platform_score\':None,\'primary_submission\':\'submission_offices.csv\',\'output\':output,\'changed_rows\':{r:sum(before[k]!=after[k] for k in sample if k[0]==r) for r in sorted({k[0] for k in sample},key=int)},\'outputs_sha256\':{n:sha256(args.out/n) for n in (\'submission_best_088092.csv\',\'submission_offices.csv\')},\'weather_sha256\':sha256(args.weather),\'libraries\':previous[\'libraries\']}\n    (args.out/\'office_report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\')\n    print(\'Selected:\',selected,\'changed rows:\',sum(report[\'changed_rows\'].values()),flush=True)\n    return report\n\nif __name__==\'__main__\':\n    from ml.forecast.office_competition import main as run_main\n    run_main()\n', 'ml/forecast/test_offices.py': "import unittest\nfrom datetime import date\nimport numpy as np\nfrom ml.forecast.office_hours import parse_hours,hourly_signals\nclass HoursTests(unittest.TestCase):\n    def test_weekday_and_weekend(self):\n        a=parse_hours('Mo-Fr 09:00-18:00; Sa 10:30-13:00; Su off')\n        self.assertEqual(a.sum(),5*540+150);self.assertEqual(hourly_signals(a)[5,10,0],.5)\n        self.assertEqual(hourly_signals(a)[0,9,1],1);self.assertEqual(hourly_signals(a)[0,18,2],1)\n    def test_unknown_is_not_closed(self):\n        for s in (None,'PH off','Mo-Fr 22:00-02:00','by appointment','Mo-Fr 09:00-18:00; PH off','Mo 25:00-26:00','Mo 09:00-18:00; Mo off'):\n            self.assertIsNone(parse_hours(s))\n    def test_continuous_has_no_arrivals(self):\n        x=hourly_signals(parse_hours('24/7'));self.assertEqual(x[:,:,0].sum(),168);self.assertEqual(x[:,:,1:].sum(),0)\n    def test_break_and_midnight(self):\n        x=hourly_signals(parse_hours('Mo 09:00-12:00,13:00-24:00'))\n        self.assertEqual(x[0,:,1].sum(),2);self.assertEqual(x[1,0,2],1)\n    def test_hour_neighbors_cross_dates(self):\n        from unittest.mock import patch\n        from ml.forecast.office_features import hours_features\n        signals=np.zeros((7,24,3));signals[1,0,1]=2;signals[6,23,2]=3\n        registry={'x':{'organisations':4,'parsed_hours':4,'hour_signals':signals.tolist()}}\n        with patch('ml.forecast.office_features.REGISTRY',{'routes':registry}):\n            self.assertAlmostEqual(hours_features(('x',date(2025,11,17),23))[-2],np.log1p(2))\n            self.assertAlmostEqual(hours_features(('x',date(2025,11,17),0))[-1],np.log1p(3))\n    def test_registry_and_features(self):\n        from ml.forecast.office_features import REGISTRY,density_features,hours_features\n        self.assertEqual(REGISTRY['snapshot'],'2025-01-01T00:00:00Z')\n        for r,p in REGISTRY['routes'].items():\n            self.assertLessEqual(p['parsed_hours'],p['tagged_hours']);self.assertLessEqual(p['tagged_hours'],p['organisations'])\n            a=density_features((r,date(2025,11,17),8))+hours_features((r,date(2025,11,17),8))\n            self.assertEqual(len(a),16);self.assertTrue(np.isfinite(a).all())\n        self.assertEqual(hours_features(('5',date(2025,11,17),8)),[0.]*8)\nif __name__=='__main__':unittest.main()\n"}
SOURCE_HASHES = {'ml/forecast/core.py': '819cfdc88d5b6b96bf80fb0e019eeef0678628b617d2dd077409d26265369b5f', 'ml/forecast/models.py': 'd02f6120641005057c411d120894617027ff8b6a10a0afba37a8e5612b5e2009', 'ml/forecast/route_specific.py': '3f9a0f92eb4bdccf0a4ad777efc06dcf24cf09ac03718c4eb678c553855f4a4c', 'ml/forecast/test_route_specific.py': 'a4e356ccbfafade57b8bd33821f63895a866c13f98ba80832bead1caeba7b5f5', 'ml/forecast/run.py': 'ce421c6fc8d77406a3817c0a15253f3464d360cc7c69bb78e8bd131edef46072', 'ml/forecast/prepare_inputs.py': '9fdb609444a3e443961504e76bda2ccb1d02dadb3ad88bb0420f70fe01711dd5', 'ml/forecast/test_pipeline.py': '267a014e6b267f4c792b7a90c8679c98ee94e3e36e070baa706a89f5fa8f399b', 'ml/forecast/download_yandex.py': '300dec623de4bf983a7960e23bbd85a128c120d40e25059d1a000d898c2ff37d', 'ml/forecast/test_download_yandex.py': '9ca4e25061931b50b2b1730e4085f44add1c9d9a0071cd1290779e5b939666d1', 'ml/submit/adapter.py': '9767bfd9573ea7637669e37bac42e75e45205f29d533d24d1d397f646047d97d', 'ml/forecast/weather_competition.py': 'cd21d93a65ae8f8ee66f54005ece5063e26e15cff64cd64c4e6dcf45e1d4dc11', 'ml/forecast/reference_route_model.json': 'd346c229308017f192e614a6dde8a1f88026c014325e664fe8ed611325baf894', 'ml/forecast/test_weather_competition.py': '69a0829537f335cae8d4f00e218fff3af3030069fbba319a94343791693db09d', 'ml/forecast/weather_source.json': '0a6f625573c314cfde5991f1da6c505dd5c74841141e7c94c07a9594bcbee3aa', 'ml/forecast/network_regimes.py': '4cdf72b14761f337a68d191ee8aaae816bc14cdc089170bb20199e5003e32116', 'ml/forecast/network_competition.py': '40e8902a8c0be0d2bb1818ff2049389f485c19d8a27b33053fc4bcfc83f50ebb', 'ml/forecast/test_network_competition.py': 'bd4852e10fab2e3b896d2f75b38e751f60b19bf0046888e81f8ef60019858ee4', 'ml/forecast/network_sources.json': 'e3443edfa338b44d93430f39741fb6f7b924fca00475b8642cba43d10b44ca6a', 'ml/forecast/research_audit.json': 'f947e15e786cc979ac9acf57c11f153f3c7fdffc8828aaa3a620dbd34a4821cb', 'ml/forecast/hourly_experts.py': '7661cd5ea210164959b0f995b9c64b1cd3f5fe0231881b87692f44ea35d28e11', 'ml/forecast/test_hourly_experts.py': '249664d4c5b914b0fee7776adfa15c7ef7390664c5e7aad2cbb2f97d83d0d4ea', 'ml/forecast/education_competition.py': 'e9cd5b4ecf8c9270820bbcd2bf3fa4eabc252fa5d8adcc1e8e8fa0189fb9a3e0', 'ml/forecast/education_features.py': '2a492e272b98db0b76b71ada0b53ae3a1e885780da52b42f7731be7661616530', 'ml/forecast/education_registry.json': '525a2c8d61ffef9c938906ad055a9ae776e79f8e86e0b5683ee0ee947d6f0644', 'ml/forecast/build_education_registry.py': 'bfd33fbcb355091853b0263c225dabaf59efcaaf52bc8fe022b13ca636c90b5e', 'ml/forecast/test_education.py': '8eea644a2ecaec5d57d0df5107a2fbb9297367b72e1d2dfac9c455e37e1edf2e', 'ml/forecast/education_profiles.py': '704a13512547a5aed38d3b022adfdb3819b63bbb72ac7226020c38df5aabc243', 'ml/forecast/office_hours.py': 'a6dee768bd27d4466eaced2503b9c0d194424e803cfa8bab210fc7eec02fb837', 'ml/forecast/office_features.py': '0e064a7056d3e0c131d7c4029f256aa0e5e91015eda6297a22617e510327d278', 'ml/forecast/office_registry.json': '762f9a7a7351969629b34ff6a2e4598dcac422951e4ae384b54d06a18833c973', 'ml/forecast/office_competition.py': 'ef5f51471a82ae14797ed34dbed099674646a9ea11ad09e3679988c970df51d3', 'ml/forecast/test_offices.py': '9fae3a6eb68ef3551d4a122bbe198f2a6990c9379507b52f5a9b2a3f879c22d7'}
for name, source in SOURCES.items():
    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_HASHES[name]:
        raise RuntimeError('Изменился встроенный исходник: ' + name)
    target = RUNTIME / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
print('Развёрнуты и проверены', len(SOURCES), 'модулей DS‑1.')

def run_module(module, *arguments):
    if not globals().get('MODEL_READY', False):
        raise RuntimeError('Сначала успешно выполните ячейку настройки среды.')
    command = [str(TRAIN_PYTHON), '-u', '-m', module, *map(str, arguments)]
    with subprocess.Popen(command, cwd=RUNTIME, env=MODEL_PROCESS_ENV,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if result:
        raise RuntimeError(f'{module} завершился с ошибкой {result}. См. вывод выше.')

if RUN_TESTS:
    run_module('unittest', 'discover', '-s', 'ml/forecast', '-p', 'test_*.py', '-v')


## 3. Скачать архив с Яндекс Диска и найти входы
Официальный публичный API не требует токена. Скачивание идёт потоком в `/tmp`,
с прогрессом, повторными попытками, проверкой размера и хеша. При повторном запуске
проверенный архив используется снова; частичная загрузка докачивается, если сервер
поддерживает Range. В память целый архив не загружается.

Распаковываются только небольшие метки и шаблон; сырые транзакции остаются в ZIP.
При сетевой ошибке ячейка остановится с объяснением, а не начнёт обучение без данных.
Официальный способ получения ссылки: https://yandex.cloud/en/docs/datasphere/operations/data/connect-to-ya-disk


In [ ]:
SEARCH_ROOT = INPUT_ROOT
if DATA_SOURCE == 'yandex':
    import importlib.util
    spec = importlib.util.spec_from_file_location('ds1_yandex', RUNTIME / 'ml/forecast/download_yandex.py')
    downloader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(downloader)
    downloaded_archive = downloader.download(YANDEX_URL, DOWNLOAD_CACHE)
    SEARCH_ROOT = downloaded_archive.parent
elif DATA_SOURCE != 'input':
    raise ValueError("DATA_SOURCE должен быть 'yandex' или 'input'")
if not SEARCH_ROOT.is_dir():
    raise FileNotFoundError('Нет входной папки. Для ручного режима добавьте набор через Add Input.')
required = {'train': 'labels_day_train.csv', 'test': 'labels_day_test.csv',
            'sample': 'test_submission.csv'}
files = {}
for role, name in required.items():
    if INPUT_FILES[role]:
        path = Path(INPUT_FILES[role])
        if not path.is_file():
            raise FileNotFoundError(path)
        files[role] = path
    else:
        matches = sorted(SEARCH_ROOT.rglob(name))
        if len(matches) > 1:
            raise RuntimeError(f'Несколько {name}: {matches}. Укажите INPUT_FILES[{role!r}].')
        if matches:
            files[role] = matches[0]

missing = set(required) - set(files)
archive_candidates = {role: [] for role in missing}
if missing:
    for archive in sorted(SEARCH_ROOT.rglob('*.zip')):
        with zipfile.ZipFile(archive) as bundle:
            for info in bundle.infolist():
                for role in missing:
                    if Path(info.filename).name == required[role]:
                        archive_candidates[role].append((archive, info.filename, info.file_size))
    for role in sorted(missing):
        matches = archive_candidates[role]
        if len(matches) != 1:
            raise RuntimeError(f'{required[role]}: найдено вариантов в ZIP: {len(matches)}. '
                               'Добавьте этот файл отдельно и укажите путь в INPUT_FILES.')
        archive, member, size = matches[0]
        if size > 20_000_000:
            raise ValueError('Неожиданно большой файл меток/шаблона: ' + member)
        with zipfile.ZipFile(archive) as bundle:
            path = EXTRACTED / required[role]
            path.write_bytes(bundle.read(member))
            files[role] = path

for role, path in files.items():
    print(role, '→', path, '|', f'{path.stat().st_size:,}', 'байт')
# Описание внутри исходного ZIP тоже можно прочитать без извлечения raw CSV.
for archive in sorted(SEARCH_ROOT.rglob('*.zip')):
    with zipfile.ZipFile(archive) as bundle:
        descriptions_in_zip = [i for i in bundle.infolist()
                               if Path(i.filename).name.lower() == 'readme.md' and i.file_size < 200_000]
        if len(descriptions_in_zip) == 1:
            description_path = EXTRACTED / 'README.md'
            description_path.write_bytes(bundle.read(descriptions_in_zip[0]))
            print('Описание из архива:\n', description_path.read_text(encoding='utf-8-sig'))
descriptions = sorted(SEARCH_ROOT.rglob('README.md'))
if len(descriptions) == 1 and descriptions[0].stat().st_size < 200_000:
    print('\nОписание набора:\n', descriptions[0].read_text(encoding='utf-8-sig'))
else:
    print('Описание не показано автоматически; сверьте README организаторов отдельно.')


## Обучение и проверка
Воспроизводим исходную модель, добавляем уже принятый учебный профиль 28, затем проверяем офисы.

In [ ]:
import base64, zlib
WEATHER = EXTRACTED / 'moscow_2025_era5.json'
WEATHER.write_bytes(zlib.decompress(base64.b64decode('...')))
assert hashlib.sha256(WEATHER.read_bytes()).hexdigest() == 'e6330c286fde16bdd97ed815376b7d221a383e3ad8a041a51b3df70bef80f9e0'
run_module('ml.forecast.network_competition', '--train', files['train'], '--test', files['test'],
           '--sample', files['sample'], '--weather', WEATHER,
           '--reference', RUNTIME / 'ml/forecast/reference_route_model.json', '--out', OUT, '--hourly-experts')

OFFICE_OUT = WORK/'office_results'
run_module('ml.forecast.office_competition', '--train', files['train'], '--test', files['test'],
           '--sample', files['sample'], '--weather', WEATHER, '--baseline-dir', OUT, '--out', OFFICE_OUT)


## Результат
Основной CSV: submission_offices.csv. Если изменений нет, повторная отправка не нужна.

In [ ]:
from IPython.display import display, Markdown, FileLink
import shutil
report = json.loads((OFFICE_OUT/'office_report.json').read_text())
print('Варианты:', report['selected'])
print('Изменено строк:', report['changed_rows'])
print('Локальный score до:', report['folds']['sep_oct']['metrics']['incumbent']['all']['wape_score'])
print('Локальный score после:', report['folds']['sep_oct']['selected']['all']['wape_score'])
if sum(report['changed_rows'].values()):
    display(Markdown('Отправлять **submission_offices.csv**. Платформенный score нового файла неизвестен.'))
else:
    display(Markdown('**Выигрыш не принят. CSV совпадает с лучшим 0,88092; повторно отправлять не нужно.**'))
for name in ('submission_offices.csv','submission_best_088092.csv','office_report.json'):
    target=WORK/name
    shutil.copy2(OFFICE_OUT/name,target)
    display(FileLink(str(target.relative_to(Path.cwd())) if target.is_relative_to(Path.cwd()) else str(target)))
